In [ ]:
import torch
import re

# load movie dialog data
with open("./data/scripts/clean_cornell_dialogue.txt", "r", encoding="utf-8") as f:
	text = f.read()

print(text[:100])


They do not! They do to! I hope so. She okay? Let's go. Wow Okay -- you're gonna need to learn how t


In [ ]:
def pretokenization(raw_text):
	text = raw_text.lower()
	
	pattern = re.compile(
		r"'s|'t|'re|'ve|'m|'ll|'d|" # common apostrophe suffixes
		r" ?[a-z]+|"				# lowercase 
		r" ?\d+|"					# digits
		r" ?[^\s\d[a-z]]+|"			# punctuation not space, letter, or digit
		r"\s+(?!\S)|\s+"			# white space
	)
	
	tokens = pattern.findall(text)

	return tokens

def get_init_vocab(tokens):
	vocab = {}
	for token in tokens:
		space_word =" " +  " ".join(list(token)) + " </w>"
		vocab[space_word] = vocab.get(space_word, 0) + 1
	return vocab

def get_stats(vocab):
	pairs = {}
	for word, freq in vocab.items():
		sym = word.split()
		word_pairs = [(sym[i], sym[i+1]) for i in range(len(sym)-1)]
		for pair in word_pairs:
			pairs[pair] = pairs.get(pair, 0) + freq
	return pairs

def merge_vocab(best_pair, vocab_in):
	vocab_out = {}
	merged_pair = best_pair.replace(" ", "")

	escape_pair = re.escape(best_pair) # e\ </w>
	pattern = re.compile(r'(?<!\S)' + escape_pair + r'(?!\S)')

	for word, freq in vocab_in.items():
		if best_pair in word:
			new_word = pattern.sub(merged_pair, word)
			vocab_out[new_word] = freq
		else:
			vocab_out[word] = freq
	return vocab_out


def train_bpe(raw_text, num_merges):
	# 1. pretokenization
	tokens = pretokenization(raw_text)		

	# 2. init vocab with tokens
	vocab = get_init_vocab(tokens)			
	
	# 3. BPE training loop
	for i in range(num_merges):
		pairs = get_stats(vocab)

		if not pairs:
			break

		# find the best pair
		best_pair_tuple = max(pairs, key=pairs.get)
		best_pair = ' '.join(best_pair_tuple)
		vocab = merge_vocab(best_pair, vocab)
	
	return vocab

final_vocab = train_bpe(text, 1000)

Debug: Starting pretokenization...
Debug: Text length is 17146310
Debug: About to run findall...
Debug: findall finished!


In [18]:
# print(final_vocab)


final_vocab = {' they</w>': 201, '   do</w>': 22640, '   not</w>': 19395, '   they</w>': 16788, '   to</w>': 80556, '   i</w>': 139527, '   hope</w>': 1020, '   so</w>': 13348, '   she</w>': 12107, '   okay</w>': 4554, '   let</w>': 6111, " 's</w>": 66884, '   go</w>': 9891, '   w ow</w>': 260, '   </w>': 136245, '   you</w>': 147641, " 're</w>": 21768, '   gonna</w>': 4316, '   need</w>': 3976, '   le ar n</w>': 426, '   how</w>': 11269, '   lie</w>': 459, '   no</w>': 19259, " 'm</w>": 23191, '   ki d ding</w>': 400, '   know</w>': 21637, '   some times</w>': 710, '   just</w>': 15742, '   be come</w>': 363, '   this</w>': 24428, ' per son a</w>': 1, '   and</w>': 45284, '   don</w>': 24545, " 't</w>": 55654, '   qu it</w>': 324, '   like</w>': 14937, '   my</w>': 20658, '   fe ar</w>': 273, '   of</w>': 39311, '   we ar ing</w>': 256, '   pa st el s</w>': 2, '   the</w>': 98584, ' real</w>': 28, '   what</w>': 37356, '   good</w>': 7413, '   stuff</w>': 1153, '   figu red</w>': 326, " 'd</w>": 7493, '   get</w>': 14105, '   ev en tually</w>': 88, '   thank</w>': 1920, '   god</w>': 2600, '   if</w>': 13065, '   had</w>': 5549, '   hear</w>': 1845, '   one</w>': 10666, '   more</w>': 4480, '   story</w>': 932, '   about</w>': 13973, '   your</w>': 20873, '   co i f fu re</w>': 3, '   me</w>': 31827, '   en d less</w>': 17, ' bl on de</w>': 2, '   bab ble</w>': 5, '   bor ing</w>': 116, '   myself</w>': 1224, '   cra p</w>': 216, '   listen</w>': 1843, '   then</w>': 6016, '   gu i ll er m o</w>': 1, '   says</w>': 1266, ' if</w>': 188, '   any</w>': 4215, '   li gh ter</w>': 22, '   look</w>': 5890, '   an</w>': 6996, '   ex tr a</w>': 163, '   on</w>': 19547, '   9 0 2 1 0</w>': 1, '   always</w>': 2574, '   been</w>': 6502, '   se l fi sh</w>': 47, '   but</w>': 17016, '   that</w>': 46323, '   all</w>': 15350, '   say</w>': 6120, '   well</w>': 9808, '   never</w>': 5424, '   wanted</w>': 1806, '   out</w>': 13347, '   with</w>': 17196, ' e</w>': 154, '   did</w>': 8622, '   was</w>': 19679, '   loo ked</w>': 451, '   for</w>': 23341, '   back</w>': 6242, '   at</w>': 10684, '   par ty</w>': 656, '   see med</w>': 192, '   be</w>': 19395, ' o c cu pi ed</w>': 3, '   t ons</w>': 35, '   have</w>': 21456, '   fu n</w>': 518, '   tonight</w>': 1390, '   believe</w>': 2534, '   we</w>': 29059, '   sh are</w>': 222, '   art</w>': 230, '   in st ru ctor</w>': 6, '   cha sti ty</w>': 9, '   looks</w>': 1007, '   things</w>': 2778, '   wor ked</w>': 427, '   huh</w>': 1652, '   hi</w>': 955, '   who</w>': 8317, '   knows</w>': 1000, " 've</w>": 10767, '   ever</w>': 3007, '   heard</w>': 1449, '   her</w>': 9282, '   is</w>': 29107, '   di p</w>': 20, '   before</w>': 2901, '   da ting</w>': 59, '   a</w>': 70784, '   guy</w>': 2702, '   s mo kes</w>': 27, '   kind</w>': 2075, '   li kes</w>': 273, '   pretty</w>': 1330, '   on es</w>': 344, '   le s bi an</w>': 19, '   found</w>': 1269, '   pi c ture</w>': 449, '   j ar ed</w>': 2, '   le to</w>': 4, '   in</w>': 34060, '   d ra w ers</w>': 13, '   sure</w>': 4372, '   har bor ing</w>': 5, '   same</w>': 1491, ' se x</w>': 6, '   ten den ci es</w>': 6, '   wor kin</w>': 78, '   it</w>': 65878, '   doesn</w>': 2306, '   se em</w>': 550, '   go in</w>': 455, '   him</w>': 12109, '   really</w>': 4565, '   wanna</w>': 954, '   can</w>': 17900, '   un less</w>': 390, '   si ster</w>': 507, '   go es</w>': 714, '   e b er</w>': 1, '   de ep</w>': 274, '   con di tion er</w>': 7, '   every</w>': 1871, '   two</w>': 3792, '   days</w>': 1104, '   use</w>': 1152, '   bl ow dr y er</w>': 1, '   without</w>': 1225, '   di f fu s er</w>': 1, '   att ac h ment</w>': 5, '   ha ir</w>': 459, '   s we et</w>': 371, '   word</w>': 932, '   as</w>': 7808, '   gen t le man</w>': 137, '   coun ted</w>': 23, '   help</w>': 2624, '   cause</w>': 495, '   th u g</w>': 5, '   are</w>': 16530, '   ob vi ou s ly</w>': 179, '   fa i ling</w>': 16, '   aren</w>': 978, '   going</w>': 8859, '   our</w>': 4089, '   da te</w>': 372, '   got</w>': 11462, '   something</w>': 5544, '   mind</w>': 1777, '   where</w>': 6763, '   there</w>': 14362, '   someone</w>': 1781, '   think</w>': 10763, '   might</w>': 1927, '   little</w>': 4503, '   find</w>': 2906, '   w en ch</w>': 8, '   pl an</w>': 495, '   pro gre ss ing</w>': 3, '   forget</w>': 1040, '   fr en ch</w>': 244, '   because</w>': 3323, '   such</w>': 1063, '   nice</w>': 1812, '   want</w>': 11046, '   thou gh</w>': 658, '   u se ful</w>': 56, '   st or es</w>': 31, '   much</w>': 3619, '   does</w>': 2438, '   cha m pa g n e</w>': 81, '   co st</w>': 240, '   ch at</w>': 56, '   life</w>': 2943, '   point</w>': 1009, '   head</w>': 1240, '   right</w>': 10012, '   see</w>': 8189, '   ready</w>': 879, '   qui z</w>': 9, '   c</w>': 644, ' e s c</w>': 1, '   ma</w>': 517, '   te te</w>': 1, '   go sh</w>': 79, '   only</w>': 4063, '   could</w>': 5968, '   k at</w>': 38, '   bo y friend</w>': 204, '   sha me</w>': 114, '   un so l ved</w>': 10, '   my st er y</w>': 73, '   used</w>': 1340, '   po pu l ar</w>': 80, '   when</w>': 7176, '   star ted</w>': 598, '   hi gh</w>': 671, '   school</w>': 1070, '   si ck</w>': 652, '   or</w>': 6474, '   why</w>': 8653, '   see ms</w>': 574, '   ea sy</w>': 834, '   enough</w>': 1828, '   thing</w>': 4180, '   ca mer on</w>': 26, '   mer cy</w>': 59, '   par ti cu l ar ly</w>': 73, '   hi de ous</w>': 15, '   b re ed</w>': 25, '   lo s er</w>': 89, '   until</w>': 1032, '   fa u lt</w>': 359, '   didn</w>': 5921, '   pro per</w>': 102, '   in t ro du ction</w>': 12, '   as king</w>': 495, '   cu te</w>': 199, '   name</w>': 2347, '   again</w>': 2367, ' b out</w>': 270, '   try</w>': 1565, '   some</w>': 6518, '   cu i s ine</w>': 7, '   sa tur day</w>': 144, '   night</w>': 2817, '   ha cking</w>': 10, '   ga g ging</w>': 5, '   spi tting</w>': 13, '   part</w>': 1013, '   please</w>': 2495, '   thought</w>': 3133, '   start</w>': 1166, '   pr on un ci ation</w>': 1, '   make</w>': 4754, '   qui ck</w>': 266, '   ro x an n e</w>': 1, '   k or r ine</w>': 1, '   an dre w</w>': 44, '   bar re t t</w>': 15, '   ha ving</w>': 845, '   in cre di b ly</w>': 43, '   h or r en d ous</w>': 3, '   pu b li c</w>': 253, '   brea k</w>': 684, '   up</w>': 11754, '   qu a d</w>': 2, '   re</w>': 165, '   so ph om ore</w>': 9, '   pro m</w>': 81, '   home</w>': 2207, ' il</w>': 90, '   twenty</w>': 1006, '   minutes</w>': 948, '   give</w>': 3472, '   pri v ate</w>': 322, '   l ine</w>': 598, '   jo ey</w>': 130, '   won der</w>': 384, '   guys</w>': 1507, '   supposed</w>': 937, '   ac tually</w>': 866, '   bi an c a</w>': 22, '   hi gh li gh ts</w>': 6, '   d or se y</w>': 7, '   in c lu de</w>': 36, '   do or</w>': 848, ' op en ing</w>': 5, '   co at</w>': 121, ' ho l ding</w>': 3, '   com b in ation</w>': 45, '   he</w>': 27216, '   differ ent</w>': 812, '   o i ly</w>': 4, '   d ry</w>': 140, '   pr ac ti ca lly</w>': 104, '   pro po sed</w>': 34, '   der ma to lo gi st</w>': 2, '   mean</w>': 5063, '   d r</w>': 824, '   b on ch ow s k i</w>': 1, '   great</w>': 2235, '   exactly</w>': 986, '   re l ev ant</w>': 17, '   con ver sa tion</w>': 150, '   would</w>': 6274, '   getting</w>': 1690, '   dr ink</w>': 837, '   here</w>': 13113, '   chan ge</w>': 774, '   deal</w>': 1035, '   t</w>': 240, '   talk</w>': 3205, '   con cen tra ting</w>': 13, '   a w fu lly</w>': 58, '   hard</w>': 1122, '   con si der ing</w>': 68, '   g y m</w>': 25, '   cla ss</w>': 334, '   hey</w>': 2562, '   ch ee ks</w>': 17, '   a g ent</w>': 321, '   sho t</w>': 804, '   being</w>': 1574, '   p ra da</w>': 3, '   next</w>': 1334, '   year</w>': 936, '   ne at</w>': 52, '   g ay</w>': 87, '   c ru i se</w>': 23, " 'll</w>": 14619, '   uni for m</w>': 79, '   que en</w>': 158, '   har ry</w>': 475, '   yeah</w>': 6901, '   se ars</w>': 9, '   ca ta lo g</w>': 4, '   tu be</w>': 36, '   so ck</w>': 21, '   gi g</w>': 46, '   hu ge</w>': 102, '   a d</w>': 57, '   wee k</w>': 897, '   ho pe fu lly</w>': 24, '   bo ge y</w>': 9, '   l ow en bra u</w>': 1, '   ex pen si ve</w>': 84, '   per m</w>': 7, '   pa tri ck</w>': 82, '   woman</w>': 1247, '   comp le te</w>': 172, '   f ru it</w>': 55, ' loo p</w>': 1, '   comp le tely</w>': 305, '   da ma ge</w>': 127, '   s end</w>': 638, '   ther a p y</w>': 73, '   for ever</w>': 230, '   set</w>': 799, '   beau ti ful</w>': 851, '   last</w>': 2628, '   guess</w>': 1942, '   will</w>': 6578, '   ex per i en ces</w>': 22, '   tr ust</w>': 745, '   people</w>': 4145, '   keep</w>': 2299, '   lo cked</w>': 191, '   away</w>': 2405, '   dar k</w>': 384, '   ex per i ence</w>': 219, '   anything</w>': 3433, '   pro te c ting</w>': 57, '   stu p id</w>': 596, '   re pe at</w>': 99, '   mi sta kes</w>': 67, '   own</w>': 1719, '   de ci si ons</w>': 40, '   in st ead</w>': 279, '   hel p ing</w>': 127, '   da ddy</w>': 621, '   hold</w>': 960, '   ho sta ge</w>': 47, '   tell</w>': 6805, '   after</w>': 2461, '   s wor e</w>': 41, ' every one</w>': 8, '   else</w>': 1867, '   doing</w>': 3100, '   haven</w>': 1439, '   since</w>': 1150, '   ex ce pt</w>': 408, '   st un ning</w>': 5, '   ga st r o</w>': 3, ' in te st in al</w>': 1, '   di sp la y</w>': 41, '   once</w>': 1256, '   af ter war ds</w>': 45, '   told</w>': 2717, '   any more</w>': 758, '   wasn</w>': 1706, '   pi ssed</w>': 132, '   bro ke</w>': 321, '   said</w>': 4081, '   every one</w>': 768, '   now</w>': 9849, '   ha te</w>': 804, '   to tal</w>': 130, '   ba be</w>': 120, '   9 </w>': 49, ' th</w>': 173, '   mon th</w>': 353, '   went</w>': 1479, '   wi sh</w>': 784, '   lu x u ry</w>': 18, '   as ked</w>': 798, '   won</w>': 2799, '   care</w>': 1616, '   fir m</w>': 93, '   beli ever</w>': 13, '   rea s ons</w>': 119, '   s</w>': 170, '   sit</w>': 857, '   su si e</w>': 74, '   w el come</w>': 301, '   act</w>': 422, '   too</w>': 4936, '   to ta lly</w>': 266, '   a pe shit</w>': 4, '   so ci al</w>': 106, '   ad vi ce</w>': 185, '   from</w>': 6905, '   un b al an c ed</w>': 3, '   f rea k</w>': 109, '   friend</w>': 1433, '   man de ll a</w>': 2, '   a ll ow ed</w>': 129, '   should</w>': 3577, '   ob se ss</w>': 1, '   over</w>': 4121, '   dead</w>': 2030, '   sha ke spe are</w>': 38, '   maybe</w>': 3928, '   even</w>': 3218, '   me ans</w>': 685, '   lea st</w>': 847, '   cl ou ted</w>': 1, '   f en</w>': 1, '   su cked</w>': 39, '   he dge</w>': 5, ' pi g</w>': 2, '   w re tch ed</w>': 21, '   l ow en st e in</w>': 3, '   nor ma l</w>': 210, '   bu sy</w>': 318, '   list en ing</w>': 240, '   bi tch es</w>': 35, '   pro z ac </w>': 5, '   ru in ing</w>': 27, '   t or ture</w>': 59, '   su ck</w>': 99, '   oh</w>': 7772, '   bo ther ing</w>': 72, '   ask</w>': 1871, '   gi g g le pu ss</w>': 4, '   pla ying</w>': 424, '   c lu b</w>': 241, '   s k un k</w>': 16, '   be coming</w>': 76, '   br a</w>': 22, '   po ten ti al</w>': 49, '   s m ack</w>': 25, '   way</w>': 4952, '   no where</w>': 142, '   cap tain</w>': 732, '   o pp re ssion</w>': 6, '   men</w>': 1257, '   mi ss ing</w>': 252, '   fine</w>': 1798, '   pri son er</w>': 54, '   house</w>': 1731, '   da u gh ter</w>': 488, '   po s se ssion</w>': 64, '   end</w>': 866, ' ho t</w>': 10, '   ro d</w>': 57, '   what ever</w>': 857, '   ho t</w>': 517, '   b end</w>': 48, '   ru les</w>': 196, '   has</w>': 3641, '   dis cu ss</w>': 158, '   tomorrow</w>': 1221, '   sc are</w>': 136, '   them</w>': 6102, '   pro mi se</w>': 501, '   bo ys</w>': 589, '   pre s ent</w>': 256, '   minute</w>': 1090, '   we ar</w>': 347, '   be lly</w>': 50, '   star ting</w>': 225, '   ex pe ct</w>': 433, '   knew</w>': 1526, '   for b id</w>': 22, ' g l ori a</w>': 1, '   st e in em</w>': 2, '   isn</w>': 2392, '   o ther wi se</w>': 159, '   kn own</w>': 438, '   or g y</w>': 9, '   must</w>': 2672, '   were</w>': 6257, '   att e mp ting</w>': 15, '   s ma ll</w>': 465, '   stu dy</w>': 170, '   gr ou p</w>': 193, '   friends</w>': 1020, '   fa ir</w>': 330, '   mu t ant</w>': 22, '   ne i ther</w>': 269, '   sle ep</w>': 848, '   star ts</w>': 162, '   dis cu ssion</w>': 50, '   u p set</w>': 274, '   boy</w>': 1566, '   s ent</w>': 528, ' em</w>': 1430, '   through</w>': 1812, '   pa du a</w>': 1, '   gir ls</w>': 514, '   ta ll</w>': 85, '   de c ent</w>': 99, '   body</w>': 643, '   other</w>': 2763, '   k in da</w>': 322, '   sh or t</w>': 275, '   under se x ed</w>': 1, '   f an</w>': 106, '   cou p le</w>': 777, '   min ors</w>': 12, '   come</w>': 6715, '   pe g ged</w>': 12, '   p re</w>': 65, ' te en</w>': 2, ' bu tt on</w>': 4, '   r ing</w>': 274, '   plea sure</w>': 262, '   b ru ci e</w>': 1, '   best</w>': 1398, '   case</w>': 1033, '   sc en ar i o</w>': 18, '   pa y ro ll</w>': 24, '   a while</w>': 125, '   hu mi li a ted</w>': 21, '   s ac ri fi ce</w>': 48, '   yourself</w>': 1791, '   al t ar</w>': 11, '   di g ni ty</w>': 31, '   sc ore</w>': 91, '   m</w>': 286, '   bu t th o l us</w>': 1, '   ex t re m us</w>': 1, '   ma king</w>': 760, '   pro gre ss</w>': 69, '   hell</w>': 2308, ' guy</w>': 13, '   pi cks</w>': 35, '   girl</w>': 1937, '   car ri es</w>': 17, '   while</w>': 1203, '   talking</w>': 1863, '   ex t reme ly</w>': 85, '   un for t un ate</w>': 51, '   man e u ver</w>': 16, '   whole</w>': 1438, '   already</w>': 1147, '   fa v ori te</w>': 181, '   b and</w>': 144, '   as sa il</w>': 1, '   e ars</w>': 131, '   for ty</w>': 307, ' one</w>': 201, '   un c le</w>': 313, '   l un g</w>': 26, '   can c er</w>': 111, '   i s su e</w>': 114, '   nu mber</w>': 678, '   ha tes</w>': 105, '   s mo k ers</w>': 4, '   pi ss</w>': 119, '   hi m self</w>': 585, '   jo y</w>': 60, '   u l ti ma te</w>': 33, '   ki ss</w>': 313, '   ass</w>': 915, '   b ent</w>': 33, ' wi de</w>': 5, '   bl ow</w>': 354, '   go l d en</w>': 61, '   o pp ort uni ty</w>': 125, '   k at ar in a</w>': 3, '   cho ice</w>': 352, '   be si d es</w>': 396, '   en e my</w>': 156, '   or che stra ting</w>': 1, '   ba ttle</w>': 104, '   po si tion</w>': 209, '   p ower</w>': 602, '   pre t end</w>': 121, '   ca lling</w>': 422, '   sho ts</w>': 85, '   se tting</w>': 60, '   time</w>': 6604, '   w o o</w>': 5, '   in vo l ved</w>': 295, '   gotta</w>': 1788, '   few</w>': 1177, '   c li en ts</w>': 55, '   wa ll</w>': 229, '   st re et</w>': 566, '   ha ted</w>': 116, '   those</w>': 2572, '   ou tr an k</w>': 2, '   st ri c tly</w>': 43, ' li st</w>': 6, '   by</w>': 4065, '   side</w>': 736, '   his</w>': 7316, '   re pu ta tion</w>': 96, '   ser i ous</w>': 583, '   man</w>': 5847, '   wh ac ked</w>': 27, '   so ld</w>': 161, '   li ver</w>': 24, '   bl ack</w>': 613, '   mar k et</w>': 121, '   bu y</w>': 664, '   new</w>': 2172, '   sp ea k ers</w>': 5, '   fe l ons</w>': 3, '   hon ors</w>': 14, '   bi o lo g y</w>': 9, '   c ri min al</w>': 140, '   l it</w>': 32, '   sta te</w>': 446, '   t ro op er</w>': 12, '   fi re</w>': 648, '   al ca tra z</w>': 1, '   th ri ves</w>': 1, '   dan ger</w>': 168, '   makes</w>': 934, '   un li ke ly</w>': 21, '   still</w>': 2801, '   t each</w>': 232, '   da z z le</w>': 2, '   char m</w>': 42, '   fa lls</w>': 91, '   love</w>': 3192, '   me w ling</w>': 1, '   ra m pa li an</w>': 1, '   w re tch</w>': 4, '   h er self</w>': 190, '   min or</w>': 52, '   en coun ter</w>': 19, '   sh re w</w>': 1, '   con se cra te</w>': 1, '   chan ce</w>': 818, '   sig ned</w>': 102, '   tu tor</w>': 8, '   mo m</w>': 891, '   can ad a</w>': 38, '   per ma</w>': 1, ' shit</w>': 44, ' gr in</w>': 1, '   mor on</w>': 52, '   tw el ve</w>': 319, '   mo de l</w>': 74, '   mo st ly</w>': 148, '   re gi on al</w>': 12, '   ru mor ed</w>': 4, '   big</w>': 2217, '   coming</w>': 1457, '   shit</w>': 2413, ' ea ting</w>': 6, '   gr in</w>': 4, '   b red</w>': 12, '   their</w>': 2387, '   mo th ers</w>': 37, '   li ked</w>': 308, '   g ran d mo th ers</w>': 1, '   gen e</w>': 30, '   po o l</w>': 143, '   r ar e ly</w>': 24, '   di lu ted</w>': 1, '   ha ir cu t</w>': 29, '   matter</w>': 1337, '   o l der</w>': 146, '   im possi bi li ty</w>': 5, '   stra t for d</w>': 8, '   bur n</w>': 161, '   p ine</w>': 18, '   per i sh</w>': 9, '   these</w>': 2813, '   seen</w>': 1434, '   h or se</w>': 238, '   j ack</w>': 847, '   off</w>': 4008, '   cl int</w>': 6, '   ea st w o od</w>': 4, '   thous and</w>': 859, '   most</w>': 1194, '   ev il</w>': 276, '   many</w>': 1289, '   thir ty</w>': 537, ' two</w>': 184, '   old</w>': 2352, '   out nu mb er ed</w>': 6, '   c ows</w>': 27, '   live</w>': 1351, '   nor th</w>': 191, '     </w>': 294, '   which</w>': 1428, '   da k o ta</w>': 14, ' on</w>': 536, '   t our</w>': 80, '   wor st</w>': 185, '   ki ssed</w>': 54, '   ma kin</w>': 70, '   hea d way</w>': 2, '   nee ds</w>': 477, '   co o l</w>': 493, '   day</w>': 2550, '   su n s</w>': 8, '   di re ct</w>': 94, '   qu o te</w>': 46, '   de ci ded</w>': 265, '   na il</w>': 78, '   dr un k</w>': 295, '   remember</w>': 2053, '   par ti al</w>': 20, '   no o d les</w>': 7, '   bo ok</w>': 615, '   around</w>': 2788, '   chi cks</w>': 47, '   play</w>': 1151, '   in st ru men ts</w>': 23, '   th a i</w>': 10, '   fo od</w>': 454, '   fe min i st</w>': 7, '   pro se</w>': 5, ' an g ry</w>': 2, '   st in k y</w>': 7, '   mu si c</w>': 365, '   in die</w>': 2, ' ro ck</w>': 6, '   per su a sion</w>': 4, '   re tri ev ed</w>': 3, '   cer tain</w>': 319, '   pi e ces</w>': 134, '   in for ma tion</w>': 361, '   miss</w>': 1279, '   hel p ful</w>': 46, '   n on</w>': 110, '   pri son</w>': 253, ' mo vi e</w>': 5, '   t y pe</w>': 213, '   leave</w>': 1891, '   alone</w>': 1085, '   ya</w>': 1336, '   run ning</w>': 583, '   rest</w>': 813, '   no ti c ed</w>': 158, '   f ea tu red</w>': 2, '   k mar t</w>': 1, '   sp read</w>': 66, '   el b ow</w>': 14, '   t ough</w>': 306, '   v in ta ge</w>': 6, '   rea ding</w>': 224, '   sa ss y</w>': 10, '   bar bi e</w>': 14, '   n</w>': 103, '   k en</w>': 31, ' the</w>': 675, '   li m o</w>': 32, '   f l ow ers</w>': 121, '   another</w>': 1717, '   hundred</w>': 1151, '   tu x</w>': 6, '   hu man</w>': 458, '   bu cks</w>': 315, '   u pped</w>': 6, '   pri ce</w>': 240, '   und er</w>': 891, '   cont ro l</w>': 438, '   ac ts</w>': 33, '   cra z ed</w>': 7, '   i ma ge</w>': 78, '   wa tch ing</w>': 321, '   bi tch</w>': 433, '   tra sh</w>': 51, '   car</w>': 1581, '   coun t</w>': 265, '   sh ell</w>': 36, '   fif ty</w>': 551, '   re su l ts</w>': 46, '   take</w>': 5655, '   ne go ti ation</w>': 14, '   ge ts</w>': 830, '   ca tch</w>': 429, '   p ay</w>': 890, '   ver on a</w>': 16, '   pi ck</w>': 634, '   ta b</w>': 19, '   ca ke</w>': 83, '   money</w>': 2770, '   sp ar k y</w>': 6, '   le gs</w>': 166, '   r ack</w>': 21, '   hi gh er</w>': 79, '   better</w>': 2743, '   fuck</w>': 2062, '   hea vi ly</w>': 19, '   in ve sted</w>': 15, '   took</w>': 1103, '   ba th es</w>': 2, '   together</w>': 1154, '   ki ds</w>': 899, '   uh</w>': 2019, '   hel p in</w>': 6, '   re c ru it</w>': 12, '   job</w>': 1608, '   pu r po se</w>': 133, '   in s an e</w>': 173, '   run</w>': 1223, '   idea</w>': 1305, '   inter e sted</w>': 391, '   no pe</w>': 144, '   came</w>': 1632, '   lo st</w>': 884, '   hon ey</w>': 707, '   pro gre ssed</w>': 3, '   fu ll</w>': 590, '   ha ll u c in a tions</w>': 12, '   wi lli am</w>': 112, '   meet</w>': 1076, '   us</w>': 5880, '   looking</w>': 1413, '   wrong</w>': 1913, '   per spe c tive</w>': 22, '   sta te ment</w>': 87, '   dre ss</w>': 242, '   anyway</w>': 1050, '   s ound</w>': 506, '   be tty</w>': 190, '   ar chi e</w>': 17, '   ta king</w>': 771, '   ver on i c a</w>': 60, '   da tes</w>': 48, '   i ma g ine</w>': 332, '   ba sti on</w>': 3, '   com mer ci al</w>': 46, '   ex c ess</w>': 11, '   pu ked</w>': 2, '   re je c ted</w>': 19, '   fa v or</w>': 314, '   b ack fi red</w>': 1, '   done</w>': 1746, '   offi ci ally</w>': 33, '   o pp osed</w>': 33, '   su bur b an</w>': 9, '   ac ti vi ty</w>': 42, '   car es</w>': 137, '   work</w>': 2777, ' any</w>': 40, '   pre ci ous</w>': 67, '   ti ar a</w>': 3, '   app re ci ate</w>': 259, '   e f for ts</w>': 24, '   t ow ard</w>': 108, '   sp ee dy</w>': 3, '   dea th</w>': 796, '   con su m ing</w>': 5, '   he ter o se x u a li ty</w>': 2, '   pro ven</w>': 22, '   gone</w>': 1072, '   ra ging</w>': 6, '   f it</w>': 193, '   s ar ah</w>': 95, '   la w r ence</w>': 16, '   in si sts</w>': 12, '   ma le</w>': 82, ' d om in a ted</w>': 1, '   pu king</w>': 8, '   fr at</w>': 7, '   go lf</w>': 63, '   t ea m</w>': 262, '   f ou l</w>': 36, '   du r ing</w>': 210, '   se x</w>': 460, '   rea li ze</w>': 285, '   in sti tu tion</w>': 44, '   se ver e ly</w>': 8, '   l ac king</w>': 16, '   ki lling</w>': 307, '   be y on d</w>': 169, '   s co pe</w>': 22, '   te en age</w>': 18, '   ob se ssi ons</w>': 4, '   ven tur ing</w>': 2, '   f ar</w>': 870, '   pa st</w>': 397, '   da y time</w>': 14, '   show</w>': 1439, '   fo d der</w>': 2, '   en ter ing</w>': 45, '   world</w>': 1668, '   very</w>': 3869, '   att e mp ted</w>': 18, '   s l it</w>': 20, '   e at</w>': 801, '   star ving</w>': 44, '   s l ow</w>': 256, '   die</w>': 1007, '   b lo ck</w>': 107, '   e</w>': 116, '   in cap able</w>': 15, '   inter e st ing</w>': 307, '   p at</w>': 53, '   por n</w>': 25, '   mo vi es</w>': 242, '   ran do m</w>': 44, '   s kid</w>': 6, '   d are</w>': 96, '   f re sh man</w>': 14, '   ye ar bo ok</w>': 7, '   ve g gi e</w>': 1, '   bur ger</w>': 34, '   bur n t</w>': 30, '   ob je ct</w>': 80, '   g ri ll</w>': 9, '   fuck ed</w>': 360, '   f ell</w>': 199, '   ca sh</w>': 300, '   as sho le</w>': 287, '   pa id</w>': 345, '   f en der</w>': 12, '   st r at</w>': 1, '   b ought</w>': 277, '   down</w>': 4472, '   pa y ment</w>': 51, '   b on us</w>': 33, '   sle e p ing</w>': 190, '   per son</w>': 722, '   tru ly</w>': 144, ' up</w>': 340, '   wait</w>': 1949, '   wor se</w>': 400, '   ad or able</w>': 31, '   li ved</w>': 274, '   g ran d father</w>': 84, '   di ed</w>': 608, '   sta y ed</w>': 127, '   ja il</w>': 295, '   mar il y n</w>': 10, '   man son</w>': 12, '   sle pt</w>': 106, '   spi ce</w>': 32, '   sp ent</w>': 211, '   si tting</w>': 312, '   g ran d ma</w>': 76, '   cou ch</w>': 50, '   wh ee l</w>': 95, '   for t un e</w>': 121, '   g ran d mother</w>': 64, '   sorry</w>': 3272, '   questi on ed</w>': 30, '   mo ti ves</w>': 17, '   s cu r v y</w>': 6, '   con vi c ted</w>': 19, '   nothing</w>': 3179, '   comp any</w>': 461, '   ans w er</w>': 646, '   questi on</w>': 775, '   anyone</w>': 1023, '   mo tive</w>': 39, '   c rea te</w>': 93, '   d ra ma</w>': 24, '   ru m or</w>': 34, '   tra di tion</w>': 43, '   re que st</w>': 85, '   com m and</w>': 177, '   a ma z in g ly</w>': 5, '   self</w>': 232, ' as su red</w>': 1, '   se x y</w>': 65, '   real</w>': 1828, '   p ea s</w>': 5, '   true</w>': 920, '   car e er</w>': 168, '   he ar say</w>': 4, '   du ck</w>': 57, '   fa ll ac y</w>': 1, '   di sa pp o in ted</w>': 51, '   sc re w ed</w>': 87, '   di sa pp o int</w>': 37, '   co ver ed</w>': 112, '   yes</w>': 7273, '   ac ting</w>': 207, '   ex cu se</w>': 647, '   so ft</w>': 131, '   da z z l ed</w>': 4, '   w it</w>': 37, '   cha p in</w>': 7, '   call</w>': 2947, '   ri di cu l ous</w>': 196, '   w in</w>': 333, '   re spe ct</w>': 305, '   par tri dge</w>': 5, '   fami ly</w>': 904, '   c li mb </w>': 84, '   sta y in</w>': 25, '   put</w>': 2511, '   f oo t</w>': 167, '   loo kin</w>': 153, '   an g le</w>': 58, '   bad</w>': 2075, '   afraid</w>': 1014, '   he i gh ts</w>': 25, '   su n sh ine</w>': 25, '   left</w>': 1651, '   sp run g</w>': 6, '   di ck head</w>': 15, '   c ru i sed</w>': 2, '   than</w>': 2721, '   u p ch u ck</w>': 2, '   re f le x</w>': 5, '   e f fe ct</w>': 75, '   what so ever</w>': 27, '   p an ti es</w>': 31, '   un w el come</w>': 2, '   t wi st</w>': 29, '   who le some</w>': 7, '   plea s ant</w>': 60, '   po e try</w>': 54, '   fe min ine</w>': 12, '   my sti qu e</w>': 1, '   co p y</w>': 175, '   of f en se</w>': 83, '   wants</w>': 1053, '   dad</w>': 1074, '   pa in</w>': 287, '   ge tt in</w>': 184, '   st ri ke</w>': 118, '   per mi ssion</w>': 92, '   father</w>': 1798, '   wouldn</w>': 2034, '   app ro ve</w>': 30, '   w er en</w>': 504, '   saw</w>': 1557, '   wa ke</w>': 301, '   ab o ve</w>': 129, ' co o l</w>': 6, ' la id</w>': 1, '   ma in l ine</w>': 1, '   te qui l a</w>': 14, '   af fe ction</w>': 22, '   bl ind</w>': 163, '   ha tr ed</w>': 19, '   wor ds</w>': 448, '   shi t f ac ed</w>': 2, '   pa tr on i z ing</w>': 6, '   con cu ssion</w>': 12, '   do g</w>': 535, '   w o ke</w>': 104, '   ve ge ta ble</w>': 20, '   differ ence</w>': 332, '   fun ny</w>': 721, ' i</w>': 3008, '   tra sh ed</w>': 10, '   ra in co a ts</w>': 2, '   bi k in i</w>': 9, '   kill</w>': 1963, '   sor t</w>': 626, '   de p ends</w>': 130, '   to pi c</w>': 15, '   f en d ers</w>': 1, '   whi p</w>': 41, '   into</w>': 3062, '   ver ba l</w>': 21, '   fr en zy</w>': 5, '   tal k er</w>': 8, '   la und ro m at</w>': 5, '   fo ll ow ing</w>': 157, '   se ven</w>': 441, ' thir ty</w>': 83, '   v om it</w>': 30, '   p on i es</w>': 12, '   f l at</w>': 98, '   be er</w>': 234, '   e yes</w>': 709, '   h and</w>': 818, '   sp end</w>': 288, '   doll ar</w>': 208, '   tr ack</w>': 162, '   war ran t</w>': 63, '   st rong</w>': 261, '   e mo tion</w>': 29, '   lot</w>': 1973, '   7 </w>': 76, ' el even</w>': 15, '   bur n side</w>': 1, '   sc re w boy</w>': 1, '   pl ac es</w>': 185, '   fri day</w>': 136, '   mi ssion</w>': 216, '   att en tion</w>': 229, '   s w ea ting</w>': 24, '   pi g</w>': 132, '   w o</w>': 5, ' man</w>': 84, '   do in</w>': 378, '   chri st</w>': 634, '   chan ged</w>': 331, '   che ck</w>': 769, '   fa th ers</w>': 25, '   ad m it</w>': 219, '   da u gh t ers</w>': 34, '   cap able</w>': 83, '   li ves</w>': 508, '   spe c ta t ors</w>': 3, '   le ts</w>': 87, '   in n in gs</w>': 5, '   b lea ch ers</w>': 6, '   years</w>': 2461, '   able</w>': 545, '   wa tch</w>': 913, '   ga me</w>': 663, '   im pre ssed</w>': 91, '   ru b bed</w>': 12, '   be at</w>': 398, '   par ts</w>': 104, '   dan ce</w>': 298, '   understand</w>': 1945, '   a ll u re</w>': 1, '   de h y d ra ted</w>': 2, '   hi p</w>': 41, '   bi k ers</w>': 4, '   sp er m</w>': 19, '   ea st</w>': 138, '   co a st</w>': 88, '   cho i ces</w>': 52, '   ei gh te en</w>': 119, ' five</w>': 368, '   par ent</w>': 32, '   a gre e</w>': 198, '   p uni sh ing</w>': 12, '   se i z u re</w>': 9, '   in su ran ce</w>': 152, '   co ver</w>': 246, '   p ms</w>': 25, '   wh ose</w>': 214, '   di ary</w>': 36, '   d ev o ted</w>': 22, '   g ro om ing</w>': 5, '   ti ps</w>': 36, '   u</w>': 131, '   0</w>': 19, '   am</w>': 3409, '   feel</w>': 2217, ' he in ous</w>': 1, '   ter m</w>': 60, '   of ten</w>': 189, '   te m pe stu ous</w>': 1, '   per ce i ve</w>': 4, '   some what</w>': 24, '   ma in tain</w>': 36, '   ki cked</w>': 89, '   ba lls</w>': 135, '   mer e ly</w>': 53, '   spe c ta tor</w>': 2, '   com par ed</w>': 39, '   ex pre ssion</w>': 42, '   today</w>': 1083, '   ev en ts</w>': 61, '   qui te</w>': 779, '   mi ld</w>': 10, '   bo b by</w>': 189, '   ri ctor</w>': 1, '   gon a d</w>': 1, '   re tri ev al</w>': 21, '   op er ation</w>': 128, '   ex pre ss ing</w>': 8, '   op in i on</w>': 153, '   ter r ori st</w>': 25, '   ac tion</w>': 172, '   ter r ori z ing</w>': 4, '   ms</w>': 108, '   b la i se</w>': 1, '   m ac be th</w>': 3, '   pi c tur es</w>': 260, '   ti red</w>': 461, '   brea thing</w>': 82, '   r en e w</w>': 7, '   th y</w>': 92, '   for ce</w>': 256, ' pa y in</w>': 2, '   lo se</w>': 524, '   co zy</w>': 24, '   st in gs</w>': 8, '   li kin</w>': 3, '   pre f er</w>': 127, '   si mp ly</w>': 161, '   al ter na tive</w>': 36, '   la w</w>': 352, '   a ll ows</w>': 20, '   te lling</w>': 730, ' n on</w>': 4, ' s mo k er</w>': 2, '   ta me</w>': 10, '   wi ld</w>': 184, '   bea st</w>': 68, '   pa w n</w>': 17, '   pl ow</w>': 4, '   who ever</w>': 154, '   sp ea k</w>': 572, '   cor re c tly</w>': 14, '   pu re</w>': 104, '   pu r er</w>': 4, '   chi ck</w>': 100, '   three</w>': 1979, '   ti ts</w>': 85, '   si tu ation</w>': 292, '   ma j or</w>': 354, '   j on es</w>': 99, '   stan d in</w>': 15, '   wai t in</w>': 51, '   de men ted</w>': 9, '   pre sti ge</w>': 4, '   ti t le</w>': 88, '   b en e fi ts</w>': 28, '   p ack age</w>': 65, '   a bu sed</w>': 7, '   s li gh tly</w>': 35, '   p sy cho ti c</w>': 28, '   dri ving</w>': 258, '   he mor r ho id</w>': 1, '   lo ss</w>': 77, '   ab s ence</w>': 26, '   t ou ch</w>': 449, '   f l u</w>': 20, '   as in ine</w>': 4, '   fee ling</w>': 554, '   t ee th</w>': 172, '   z i pp er</w>': 6, '   bra tw u r st</w>': 2, '   ea ting</w>': 175, '   l un ch</w>': 313, '   ex po sed</w>': 32, '   f re sh men</w>': 2, '   mi ssed</w>': 247, '   ab so lu tely</w>': 387, '   happened</w>': 1825, '   ru th</w>': 43, '   ki ss ing</w>': 51, '   happen s</w>': 529, '   kee ps</w>': 202, '   el b ows</w>': 16, '   pl ac en ta</w>': 1, '   pi ra te</w>': 59, '   ra ther</w>': 426, '   ra vi sh ed</w>': 1, '   bri ti sh</w>': 122, '   re ar</w>': 26, '   ad mi ra l</w>': 162, '   tu me sc ent</w>': 1, '   jesus</w>': 997, '   gr ab </w>': 149, '   s an d wi ch</w>': 51, '   wom en</w>': 652, '   di la ting</w>': 2, '   coun try</w>': 648, '   sy n on y m</w>': 1, '   th ro b b ing</w>': 7, '   c ry</w>': 158, '   mi c r ow a ve</w>': 19, '   al so</w>': 566, '   s m ell</w>': 240, '   se a</w>': 190, '   s me lls</w>': 87, '   c un t</w>': 38, '   sig n</w>': 371, '   wee ks</w>': 485, '   ago</w>': 1114, '   al on so</w>': 2, '   ne ar</w>': 329, '   land</w>': 305, '   crazy</w>': 1060, '   d ev il</w>': 168, '   chi ld</w>': 507, '   face</w>': 924, '   ah</w>': 636, '   har m</w>': 103, '   chi ck en</w>': 161, '   dr in king</w>': 167, '   g l ory</w>': 65, '   sp ain</w>': 23, '   co l on</w>': 17, '   ba st ard</w>': 283, '   wa ter</w>': 668, '   pu tri d</w>': 3, '   bar re ls</w>': 13, '   he at</w>': 146, '   la s</w>': 51, '   min as</w>': 1, ' god</w>': 42, ' wi lls</w>': 1, ' it</w>': 513, '   a si a</w>': 24, '   we st</w>': 230, '   pro ve</w>': 263, '   fa i th</w>': 180, '   con si der</w>': 172, '   h er e sy</w>': 12, '   con si der ed</w>': 78, '   h er e ti cal</w>': 2, '   ch oo se</w>': 129, '   car pen ter</w>': 57, '   son</w>': 1197, '   re v ea l</w>': 27, '   in ten ded</w>': 42, '   pro x i mi ty</w>': 5, '   wai ted</w>': 107, '   por tu gu e se</w>': 5, '   dis co ver ed</w>': 81, ' s k in ned</w>': 3, '   po pu la tions</w>': 1, '   bring</w>': 938, '   inter e sts</w>': 75, '   go ld</w>': 250, '   tra de</w>': 131, '   ex ce ll en cy</w>': 33, '   ac cor ding</w>': 113, '   mar c o</w>': 13, '   po l o</w>': 17, '   k in g do m</w>': 36, '   ch in a</w>': 98, '   ri che st</w>': 18, '   me an est</w>': 8, '   bu il din gs</w>': 47, '   ro of ed</w>': 1, '   fo llow</w>': 429, '   o th ers</w>': 394, '   e min ence</w>': 9, '   se ttle</w>': 137, '   j our ne y</w>': 65, '   ri s k</w>': 180, '   possi ble</w>': 468, '   sen or</w>': 13, '   ex per i en c ed</w>': 33, '   con cer n</w>': 106, '   cre w</w>': 177, '   wi lling</w>': 167, '   con s ci ence</w>': 57, '   re li ed</w>': 4, '   u p on</w>': 146, '   ju d g ment</w>': 39, '   can not</w>': 349, '   i g n ore</w>': 59, '   ca l cu la tions</w>': 11, '   ci r cu m f er ence</w>': 1, '   ear th</w>': 381, '   app ro x i ma tely</w>': 25, '   2 2 </w>': 33, ' 0 0 0</w>': 171, '   lea gu es</w>': 19, '   o c ean</w>': 107, '   un c ro s sa ble</w>': 2, '   un for t un a tely</w>': 73, '   pre ci se ly</w>': 57, '   op in i ons</w>': 18, '   di ff er</w>': 9, '   fami li ar</w>': 133, '   ar i sto t le</w>': 6, '   er a th o st en e</w>': 1, '   p to le me us</w>': 1, '   vo y age</w>': 29, '   six</w>': 925, '   sa i ling</w>': 38, '   wa st e</w>': 185, '   ar o ja z</w>': 1, '   m ine</w>': 806, '   reme mb er ed</w>': 72, '   tra ge dy</w>': 40, '   con tr ary</w>': 39, '   pre par ing</w>': 28, '   c ro ss</w>': 194, '   ri d</w>': 190, '   pro ph et</w>': 7, '   s an che z</w>': 20, '   in de ed</w>': 139, '   mer cen ar i es</w>': 3, '   sta tes</w>': 157, '   w el f are</w>': 26, '   pro sp er i ty</w>': 3, '   mer cen ary</w>': 10, '   con v in ce</w>': 74, '   king</w>': 485, '   por tu ga l</w>': 6, '   ab sur d</w>': 38, '   no tions</w>': 15, '   na tu ra lly</w>': 119, '   de pl ore</w>': 2, '   di sp u te</w>': 11, '   ge o gra ph y</w>': 10, '   ou rs</w>': 122, '   rea son</w>': 696, '   pro po si tion</w>': 40, '   m om en ts</w>': 54, '   in side</w>': 695, '   stay</w>': 1652, '   everything</w>': 2244, '   f ree</w>': 543, '   ri ch es</w>': 5, '   ri ch</w>': 307, '   bu si er</w>': 4, '   tri ed</w>': 685, '   ta k en</w>': 481, '   ar ran ge</w>': 58, '   f er n an do</w>': 6, '   di e go</w>': 83, '   ser vi ce</w>': 287, '   u su ally</w>': 236, '   bea tri x</w>': 1, '   de ci de</w>': 176, '   s we ar</w>': 355, '   per ha ps</w>': 539, '   me ant</w>': 337, '   ar gu e</w>': 70, '   mar ry</w>': 324, '   lea ving</w>': 445, '   gi ven</w>': 337, '   ne w s</w>': 538, '   ma in land</w>': 22, '   s ea man</w>': 2, '   con gra tu la tions</w>': 117, '   se ar ch</w>': 142, '   vi cer o y</w>': 5, '   in di es</w>': 4, '   app o in t ment</w>': 86, '   le tt ers</w>': 155, '   de</w>': 215, '   bo ba di ll a</w>': 3, '   bar to lo me</w>': 1, '   gi ac om o</w>': 2, '   may</w>': 1446, '   wh om</w>': 125, '   for give</w>': 266, '   po si tions</w>': 17, '   so on</w>': 857, '   app o in ting</w>': 2, '   go ver n ors</w>': 2, '   i s lan ds</w>': 28, '   first</w>': 2769, '   com es</w>': 757, '   yet</w>': 1168, '   be g in ning</w>': 231, '   ab o ard</w>': 87, '   shi ps</w>': 68, '   as ks</w>': 64, '   vi sit</w>': 207, '   ad dre ss</w>': 198, '   he ar ing</w>': 142, '   pa s sa ge</w>': 39, '   ex pl ore</w>': 31, '   ho ly</w>': 232, '   sa in ts</w>': 24, '   hea ven</w>': 198, '   st</w>': 124, '   chri st op her</w>': 8, '   c rea ted</w>': 107, '   ro of s</w>': 2, '   t ow ers</w>': 14, '   p al ac es</w>': 7, '   sp ir es</w>': 1, '   win d ow</w>': 294, '   d rea m er</w>': 16, '   gu ar ds</w>': 80, '   ri se</w>': 58, '   dan ger ous</w>': 287, '   o c cu pa tion</w>': 16, '   h y po c ri sy</w>': 6, '   long</w>': 2636, '   spe ci al</w>': 431, '   tal ent</w>': 103, '   ju d g es</w>': 15, '   th i ev es</w>': 28, '   ju dge</w>': 202, '   de ar</w>': 544, '   c ri sto ba l</w>': 1, '   l ack</w>': 60, '   no t ar i es</w>': 2, '   cont act</w>': 183, '   ad min i stra tion</w>': 25, '   com mon er</w>': 4, '   de f end</w>': 60, '   ad mi r ably</w>': 1, '   s in</w>': 74, '   na ke d ne ss</w>': 3, '   na ture</w>': 184, '   en ding</w>': 43, '   su mm er</w>': 197, '   t re es</w>': 89, '   fi lled</w>': 87, '   b lo ss om s</w>': 3, '   f ru i ts</w>': 3, '   ac ce pt</w>': 186, '   pro po sa l</w>': 29, '   a m bi ti ous</w>': 21, '   a m bi tion</w>': 24, '   vi r tu e</w>': 29, '   a m ong</w>': 106, '   no b les</w>': 10, '   bar ga in ing</w>': 6, '   re mind</w>': 128, '   bar ga in</w>': 52, '   f ought</w>': 87, '   ri s ks</w>': 33, '   pro f it</w>': 46, '   ser v ant</w>': 34, '   le ar ned</w>': 193, '   lan gu age</w>': 130, '   u ta p an</w>': 2, '   under stan ds</w>': 38, '   p ea ce</w>': 237, '   chi e f</w>': 306, '   me di c ine</w>': 85, '   ad mi re</w>': 71, '   thous an ds</w>': 100, '   bu i ld</w>': 177, '   for t</w>': 67, '   i s land</w>': 248, '   tri be</w>': 26, '   cu b a</w>': 49, '   im pre ssion</w>': 71, '   wor l ds</w>': 41, '   di sa gre e</w>': 20, '   supp ose</w>': 670, '   bo th</w>': 875, '   ab so lu tion</w>': 2, '   beli ev ed</w>': 128, '   b ound</w>': 57, '   o a th</w>': 44, '   cer ti tu d es</w>': 1, '   fri gh ten ing</w>': 28, '   t wi ce</w>': 223, '   di stan ce</w>': 89, '   li ed</w>': 155, '   lon ger</w>': 321, '   saying</w>': 1155, '   be tra y ed</w>': 48, '   sin ned</w>': 10, '   no m ine</w>': 2, '   pa tr is</w>': 2, '   et</w>': 21, '   fi li us</w>': 1, '   spi ri tu s</w>': 1, '   s an c t i</w>': 1, '   li es</w>': 110, '   damn</w>': 1032, '   the ori es</w>': 34, '   ba sed</w>': 69, '   sa fe ty</w>': 91, '   stu di es</w>': 30, '   in ten ds</w>': 8, '   mu st n</w>': 72, '   de sp a ir</w>': 17, '   me an ing</w>': 160, '   con tra di c ted</w>': 3, '   e ter ni ty</w>': 22, '   car ried</w>': 78, '   pa ssion</w>': 58, '   mar ch en a</w>': 2, '   o ver wh el m</w>': 5, '   e s d ra s</w>': 2, '   je w</w>': 65, '   to sc an e ll i</w>': 1, '   mar in</w>': 1, '   t y r</w>': 1, '   i g nor an ce</w>': 24, '   7 5 0</w>': 7, '   can ary</w>': 12, '   in fini te</w>': 13, '   op en</w>': 890, '   r ou te</w>': 82, '   m om ent</w>': 540, '   ways</w>': 233, '   rea ch ing</w>': 21, '   sa il</w>': 49, '   di f fi cu l ty</w>': 21, '   f ool</w>': 312, '   man age</w>': 90, '   t ea ch er</w>': 121, '   cho s en</w>': 50, '   b right</w>': 139, '   bro th ers</w>': 202, '   ra i sed</w>': 66, '   ma je st y</w>': 160, '   cont ent</w>': 26, '   read</w>': 936, '   n or</w>': 103, '   re turn</w>': 260, '   s an to</w>': 4, '   d om in go</w>': 8, '   co lon i es</w>': 11, '   cont in ent</w>': 14, '   na ked</w>': 100, '   thou gh ts</w>': 106, '   thin ks</w>': 452, ' what</w>': 558, '   in cl in ation</w>': 6, '   f re e ly</w>': 17, '   su re ly</w>': 122, '   ver di ct</w>': 12, '   coun ci l</w>': 45, '   im pre g n able</w>': 5, '   g ran ad a</w>': 5, '   mo ors</w>': 9, '   s an t an ge l</w>': 1, '   te lls</w>': 207, '   hon or</w>': 312, '   sin cer i ty</w>': 6, '   re gre t</w>': 84, '   he ld</w>': 150, '   de ten tion</w>': 11, '   de pri ved</w>': 12, '   pri vi le g es</w>': 10, '   po s se ssi ons</w>': 11, '   re tur ned</w>': 55, '   ju d ged</w>': 10, '   sa v a ger y</w>': 2, '   mon ke ys</w>': 25, '   b ru ta li ty</w>': 5, '   cha o s</w>': 37, '   tri b es</w>': 8, '   fi gh ting</w>': 216, '   each</w>': 822, '   jo in ing</w>': 24, '   for ces</w>': 59, '   aga in st</w>': 687, '   mo x i c a</w>': 2, '   ra i se</w>': 169, '   c ru sa de</w>': 5, '   for est</w>': 55, '   s wee p</w>': 38, '   bra ve</w>': 97, '   s wa llow</w>': 51, '   g ri e f</w>': 35, '   ac comp li sh</w>': 19, '   w ar</w>': 720, '   out nu mber</w>': 2, '   ten</w>': 1045, '   sh ou l d ers</w>': 42, '   cou sin s</w>': 24, '   wa sh</w>': 130, '   b loo d</w>': 619, '   in di ans</w>': 36, '   in di an</w>': 123, '   vi ce</w>': 92, '   d ra w ing</w>': 48, '   i s th m us</w>': 2, '   e ight</w>': 503, '   qu ad ran t</w>': 17, '   men de z</w>': 2, '   sir</w>': 2914, '   si ght</w>': 125, '   du e</w>': 157, '   p in z on</w>': 2, '   for w ard</w>': 166, '   b loo dy</w>': 130, '   mar i a</w>': 29, '   list en ed</w>': 47, '   half</w>': 947, '   ne ar ly</w>': 99, '   tur ning</w>': 125, '   ver ge</w>': 17, '   mu t in y</w>': 9, '   ho p es</w>': 35, '   a live</w>': 630, '   ma d</w>': 358, '   ch ea ted</w>': 42, '   im me di a tely</w>': 138, '   to l er ate</w>': 23, '   im per t in ence</w>': 2, '   ta s k</w>': 45, '   re pl ac ed</w>': 29, '   su g ge st</w>': 109, '   bro ther</w>': 890, '   bu y l</w>': 1, '   or der ed</w>': 102, '   ex e cu tion</w>': 14, '   five</w>': 1266, '   me mb ers</w>': 59, '   no bi li ty</w>': 13, '   ex pe c ting</w>': 130, '   im me di ate</w>': 29, '   pro fi ts</w>': 26, '   ship</w>': 571, '   re tur n s</w>': 33, '   car go</w>': 24, '   d ying</w>': 254, '   pro ves</w>': 29, '   pi ty</w>': 91, '   mon k</w>': 19, '   al thou gh</w>': 90, '   de man ds</w>': 23, '   g ran ted</w>': 51, '   t rea sur er</w>': 7, '   offi c ers</w>': 78, '   ki ll er</w>': 281, '   du ty</w>': 155, '   ar rest</w>': 173, ' so</w>': 167, '   fa m ous</w>': 154, '   ca u ght</w>': 357, '   men tal</w>': 63, '   ho spi tal</w>': 418, '   a bu se</w>': 35, '   ci gar e tt es</w>': 111, '   bor n</w>': 312, '   ci gar e tt e</w>': 112, '   b la med</w>': 21, '   mother</w>': 1650, '   bl in d ne ss</w>': 3, ' bad</w>': 16, '   doctor</w>': 923, '   ga ve</w>': 865, '   d ru gs</w>': 211, '   made</w>': 1925, '   c z e ch</w>': 25, '   re pu b li c</w>': 13, '   gi ving</w>': 369, '   bi r th</w>': 80, '   fucking</w>': 1723, '   in du ce</w>': 8, '   de gra ded</w>': 1, '   killed</w>': 1334, ' e st ee m</w>': 2, '   youn g</w>': 811, '   par en ts</w>': 416, '   b ack gr ound</w>': 58, '   u p br in ging</w>': 10, '   e mi l</w>': 12, ' per c ent</w>': 5, '   la w y er</w>': 264, '   bi g ge st</w>': 137, '   ne go ti ate</w>': 38, '   per c ent</w>': 195, ' half</w>': 36, '   cu t</w>': 823, '   fo cu sed</w>': 13, '   mo vi e</w>': 471, '   ri gh ts</w>': 100, '   worry</w>': 996, '   di sa pp ear ed</w>': 110, '   every where</w>': 145, '   c z e cho s lo v a ki a</w>': 7, '   o le g</w>': 9, '   se ver i ty</w>': 1, '   re cen tly</w>': 79, ' de lu si ons</w>': 1, '   par an o i a</w>': 21, '   br ought</w>': 531, '   ma il</w>': 90, '   clo th es</w>': 343, '   in vo king</w>': 2, '   re pre sen ted</w>': 9, '   coun se l</w>': 32, '   ca mer a</w>': 141, '   att or ne y</w>': 132, '   da ph n e</w>': 32, '   d ra g</w>': 88, ' do</w>': 147, '   proble ms</w>': 307, '   p our ing</w>': 20, '   ki tch en</w>': 155, '   co f fe e</w>': 417, '   p ac ked</w>': 56, '   gla d</w>': 516, '   m et</w>': 728, '   im mi gra tion</w>': 17, '   st all</w>': 23, '   pro c ess</w>': 86, '   e d die</w>': 276, '   re com men ded</w>': 25, '   gir l friend</w>': 218, ' thous and</w>': 55, '   mi les</w>': 415, '   lu d wi g</w>': 11, '   j ea l ous</w>': 126, '   al right</w>': 869, '   hi ding</w>': 162, '   tru st ing</w>': 26, '   de sp er ate</w>': 81, '   co p</w>': 351, '   wh ore</w>': 93, '   trying</w>': 1519, '   pro sti tu te</w>': 21, '   couldn</w>': 1294, '   gre en</w>': 205, '   c ard</w>': 228, '   app ro ac h ed</w>': 14, '   be l ow</w>': 78, '   ta ble</w>': 283, '   turn</w>': 916, '   tri cks</w>': 78, '   sta tion</w>': 263, '   e mp ty</w>': 162, '   qu ar t ers</w>': 48, '   pr ac ti c ing</w>': 33, '   pu tting</w>': 243, '   fir es</w>': 29, '   some where</w>': 506, '   place</w>': 2344, '   ki ll in gs</w>': 12, '   stop</w>': 1823, '   sh ower</w>': 124, ' before</w>': 26, '   cu sto dy</w>': 43, '   di v or c ed</w>': 81, '   mar ried</w>': 860, '   co op er ate</w>': 42, '   da</w>': 87, '   poli ce</w>': 780, '   de part ment</w>': 237, '   ju sti fi able</w>': 3, '   h om i ci de</w>': 68, ' my</w>': 194, '   sh ar ed</w>': 23, '   ra p ing</w>': 11, '   gu n</w>': 716, '   cha ir</w>': 127, '   happen ing</w>': 328, '   tru th</w>': 740, ' is</w>': 190, '   town</w>': 874, '   s lo v a ki a</w>': 1, '   s ou th</w>': 220, '   ci vi li an</w>': 35, '   f l ed</w>': 12, '   i ll e ga lly</w>': 11, '   de por t</w>': 3, '   wh e ther</w>': 227, '   ta min a</w>': 3, '   bro k en</w>': 208, '   the ir s</w>': 31, '   part ner</w>': 220, '   be ep</w>': 8, '   pre ss</w>': 209, '   con f er ence</w>': 76, '   bri e f ed</w>': 8, '   j or dy</w>': 1, '   bri e f s</w>': 7, '   you rs</w>': 742, '   de pu ty</w>': 59, '   mar sha ll</w>': 19, '   in c lu ded</w>': 19, '   de ci sion</w>': 149, '   su sp en ded</w>': 25, '   tri al</w>': 155, '   fi ght</w>': 576, '   ha dn</w>': 252, '   go tt en</w>': 264, '   fr on t</w>': 540, '   pa ge</w>': 133, '   h er o</w>': 152, '   ea si er</w>': 131, '   we ight</w>': 122, '   co ver age</w>': 14, '   gi m me</w>': 202, '   shi e ld</w>': 43, '   probably</w>': 1102, '   in no c ent</w>': 170, '   for got</w>': 272, '   han d cu ff ed</w>': 8, '   t ree</w>': 163, '   mu g</w>': 32, '   de fine</w>': 28, '   ro d ne y</w>': 2, '   sp or ts</w>': 48, '   wi t ne ss</w>': 179, '   tr ouble</w>': 816, ' cle an ed</w>': 2, '   la d der</w>': 36, '   2 0</w>': 93, '   ro ck</w>': 226, '   tra in ing</w>': 158, '   sto pped</w>': 286, '   cle an ed</w>': 73, '   in ve sti ga tion</w>': 109, '   called</w>': 1237, ' cause</w>': 553, '   imp ort ant</w>': 738, '   fin ding</w>': 111, '   re por ter</w>': 77, '   ori g in</w>': 19, '   s w ing</w>': 63, '   a part ment</w>': 338, '   stra ight</w>': 463, '   ho over</w>': 30, '   fini sh ed</w>': 272, '   d</w>': 467, ' a</w>': 887, '   vi de o ta pe</w>': 7, '   de po si tion</w>': 27, '   wi se guy</w>': 5, '   so l ve</w>': 51, '   f le mm ing</w>': 5, '   some th in</w>': 278, '   al er t</w>': 51, '   me di a</w>': 55, '   k or f in</w>': 2, '   o ver time</w>': 17, '   re la x</w>': 288, '   fuck ers</w>': 39, '   mo ther fuck er</w>': 126, '   fi l m ing</w>': 7, '   h it</w>': 833, '   vi de o ca mer a</w>': 3, ' ere</w>': 31, '   che cked</w>': 147, ' p d</w>': 1, '   un kn own</w>': 50, '   pr in ts</w>': 51, '   kid</w>': 1112, ' d</w>': 275, '   ta kin</w>': 63, '   ba th</w>': 91, '   ti m er</w>': 16, '   pro po se</w>': 38, '   me an while</w>': 36, '   mor gu e</w>': 46, '   cho pped</w>': 30, '   sp le en</w>': 9, '   he art</w>': 762, '   p an</w>': 30, '   hou rs</w>': 766, '   wal kin</w>': 33, '   bu y in</w>': 11, '   qu art</w>': 5, '   mi l k</w>': 108, '   ro b bed</w>': 66, ' 3 </w>': 40, '   pi e ce</w>': 471, '   be e f</w>': 59, '   thin kin</w>': 104, '   so on er</w>': 171, '   la ter</w>': 804, '   f rea k y</w>': 14, '   mar tell</w>': 1, '   vo d k a</w>': 30, '   ton i c</w>': 11, '   wa sh es</w>': 12, '   sa l on</w>': 17, '   6 3 </w>': 6, ' r d</w>': 33, '   ma di son</w>': 32, '   qui ck ly</w>': 142, ' ha i red</w>': 10, '   sc ary</w>': 82, '   secon d</w>': 799, ' sh or ter</w>': 1, '   w re st l er</w>': 13, '   de sc ri be</w>': 70, '   cou s in</w>': 133, '   wor ks</w>': 408, '   su d d en</w>': 130, '   wor king</w>': 885, '   po or</w>': 493, '   g</w>': 124, '   e d w ard</w>': 87, '   ho te l</w>': 403, '   he ar n</w>': 1, '   de te c tive</w>': 230, '   no th in</w>': 335, '   questi ons</w>': 422, '   pu pp y</w>': 32, '   ru g</w>': 46, '   ke pt</w>': 340, '   stand</w>': 689, '   li mb </w>': 26, '   somebody</w>': 1089, '   je op ar di ze</w>': 11, '   le m me</w>': 117, '   t v </w>': 320, '   se cre t</w>': 402, '   fuck in</w>': 884, '   wa tch es</w>': 34, '   te l ev i sion</w>': 164, '   fa me</w>': 37, '   i tty</w>': 1, '   bi tty</w>': 4, '   ci ty</w>': 588, '   sh ou l da</w>': 83, '   te ll in</w>': 101, '   pro fe ssi on al</w>': 124, '   ou tta</w>': 395, '   han g</w>': 390, '   su g ge sts</w>': 14, '   pa ss</w>': 276, '   den y</w>': 82, '   b it</w>': 698, '   nobody</w>': 1083, ' in c lu ding</w>': 2, '   pro fe ssi on al s</w>': 19, '   a ma te u rs</w>': 12, '   war m</w>': 162, '   sa vi or</w>': 10, '   sa ve</w>': 492, '   bur ning</w>': 101, '   bu il ding</w>': 323, '   in ti mi da te</w>': 16, '   ce le bri ty</w>': 26, '   se es</w>': 152, '   differ en tly</w>': 36, '   lea ds</w>': 131, '   c r ack</w>': 102, '   ra w</w>': 43, '   th rea ten ed</w>': 51, '   pu ll</w>': 456, '   gla d ly</w>': 18, '   ki ck</w>': 183, '   ru sh</w>': 94, '   s mo ke</w>': 206, '   d rea med</w>': 54, '   ki cking</w>': 40, '   pu lling</w>': 97, '   ye lling</w>': 50, ' f re e ze</w>': 1, '   d rea m</w>': 421, '   course</w>': 1981, '   thir st y</w>': 37, '   su spe c ts</w>': 30, '   le on</w>': 133, '   hea ds</w>': 154, '   ta i ls</w>': 13, '   f li p</w>': 40, '   co in</w>': 28, '   ri de</w>': 417, '   al ong</w>': 638, '   i dea s</w>': 145, '   e sc or t</w>': 44, '   fo l ks</w>': 230, '   poli sh</w>': 38, '   po land</w>': 5, '   ea st er n</w>': 32, '   e u ro p ean</w>': 22, '   s la v s</w>': 1, '   ger man s</w>': 56, '   ru ssi ans</w>': 41, '   in ten se</w>': 23, '   per son ally</w>': 152, '   e u ro pe</w>': 100, '   ro man i a</w>': 2, '   h un g ary</w>': 4, '   ri tu al</w>': 21, '   me s sa ge</w>': 284, '   bu ri al</w>': 20, '   ri tes</w>': 11, '   ser i ou s ly</w>': 186, '   hu mi li ate</w>': 13, '   fun er al</w>': 106, '   con de m ning</w>': 1, '   out side</w>': 538, '   l ead</w>': 248, '   vi si ting</w>': 44, '   su spe ct</w>': 167, '   ei ther</w>': 713, '   pre t ti est</w>': 10, '   h mm m m</w>': 14, '   su per</w>': 72, '   st ab bed</w>': 31, '   ti p</w>': 113, '   k ni fe</w>': 140, '   sp ine</w>': 17, '   in di ca tor</w>': 2, '   per son al</w>': 321, '   st e pp in</w>': 8, '   c ri me</w>': 237, '   sc en e</w>': 173, '   re por t</w>': 368, '   problem</w>': 1047, '   com men d able</w>': 1, '   na h</w>': 160, '   cre d it</w>': 161, '   mu st a</w>': 52, '   ex pla in</w>': 405, '   ca me llo</w>': 1, '   p un ch ing</w>': 10, '   ho le</w>': 191, '   f lo or</w>': 300, '   w ra pped</w>': 30, '   r out ine</w>': 72, '   gi ves</w>': 237, '   s mar ter</w>': 43, '   a mer i c ans</w>': 75, '   f ed</w>': 74, '   baby</w>': 1122, '   sh ows</w>': 146, ' de te c tive</w>': 2, '   go o d b ye</w>': 339, '   ni co le tt e</w>': 2, '   ju ry</w>': 103, '   bu ll shit</w>': 520, '   fi l m</w>': 221, '   no ose</w>': 8, '   ne ck</w>': 210, '   ta b lo i ds</w>': 8, '   en ac t men ts</w>': 1, '   ac tor</w>': 101, '   pla ys</w>': 111, '   ac comp li sh ment</w>': 5, '   wri te</w>': 526, '   bo o ks</w>': 289, ' who</w>': 119, '   ni ck y</w>': 91, '   late</w>': 873, '   lu ck</w>': 439, '   pa ir s</w>': 10, '   sho es</w>': 170, '   te st</w>': 222, ' how</w>': 178, '   til</w>': 55, '   n ay</w>': 23, '   a head</w>': 564, '   l en s</w>': 12, '   sho e</w>': 84, '   mar ri a g es</w>': 14, '   mi d dle</w>': 336, '   an ch or</w>': 23, '   e mer gen cy</w>': 150, '   ph one</w>': 792, '   shi r t</w>': 99, '   ru th less</w>': 13, '   ta u ght</w>': 161, '   mer ci less</w>': 5, '   ho l ding</w>': 203, '   ev i d ence</w>': 219, '   sti ck</w>': 313, '   mi c ro ph one</w>': 11, '   pa tr on i ze</w>': 7, ' turn</w>': 16, '   ca mer as</w>': 38, '   e mb ar ra ss</w>': 24, '   s wee the art</w>': 168, '   co ll ea gu es</w>': 19, '   s na p</w>': 53, '   u m</w>': 286, ' now</w>': 112, '   j ack son</w>': 49, '   hur t</w>': 884, '   in ju red</w>': 37, '   cor re ct</w>': 198, '   some how</w>': 188, '   re la ted</w>': 48, '   d ru g</w>': 152, '   bo di es</w>': 144, '   vi c ti ms</w>': 82, '   ear ly</w>': 326, '   mu r der</w>': 335, '   wor ried</w>': 346, '   t ou ch ed</w>': 100, '   pri ck</w>': 80, '   k ne es</w>': 83, '   be h ind</w>': 628, '   uni qu e</w>': 38, '   an ti gu a</w>': 5, '   bo ttle</w>': 145, ' su s an</w>': 5, '   1 5 </w>': 59, '   a und re a</w>': 1, '   thinking</w>': 972, '   c li c he</w>': 4, '   tal ked</w>': 353, '   mo o dy</w>': 12, '   cou pl a</w>': 42, '   ar ti c les</w>': 22, '   me da l</w>': 28, '   pla qu e</w>': 9, '   reme mb ers</w>': 26, '   everybody</w>': 928, '   hea v y</w>': 170, '   w ea p ons</w>': 137, '   ci g ar</w>': 41, '   bo om</w>': 62, '   sho t gu n</w>': 54, '   cou l da</w>': 66, '   bl own</w>': 67, '   p ani ck y</w>': 2, '   ta kes</w>': 415, '   ha b it</w>': 58, '   pro po sing</w>': 21, '   b on es</w>': 122, '   sa u sa ge</w>': 16, '   bu tch er</w>': 30, '   in te st in es</w>': 5, '   f ac t ory</w>': 55, '   sur r en der ing</w>': 6, '   four</w>': 1073, '   o</w>': 713, ' clo ck</w>': 291, '   ban k</w>': 357, '   doll ars</w>': 742, '   con di tion</w>': 154, '   ex c lu si vi ty</w>': 1, '   sur r en der</w>': 62, '   4 5 </w>': 22, '   bro ad way</w>': 18, '   l ower</w>': 72, '   sh er at on</w>': 10, '   room</w>': 1513, '   2 1 0</w>': 3, '   tri g ger</w>': 66, '   fi re man</w>': 27, '   b et</w>': 654, '   any body</w>': 809, '   tom my</w>': 195, '   wal k</w>': 777, '   le ss ons</w>': 51, '   ri g</w>': 43, '   pu r se</w>': 47, '   pa y in</w>': 18, '   run s</w>': 172, '   con fi den ti al</w>': 28, '   dre ss er</w>': 11, '   out call</w>': 2, '   und re ssed</w>': 21, '   business</w>': 1321, ' ch ya</w>': 1, '   sch oo l girl</w>': 4, '   m om my</w>': 145, '   en g li sh</w>': 279, '   plan ning</w>': 129, '   p ra gu e</w>': 17, '   tra ve lling</w>': 20, '   do cu men ts</w>': 36, '   jo in</w>': 260, '   f is</w>': 1, '   ar e a</w>': 191, ' hundred</w>': 88, '   car r ying</w>': 138, '   ho li day</w>': 58, '   uni ted</w>': 127, '   ted</w>': 210, '   bu n dy</w>': 21, '   ser i al</w>': 66, '   mi lli on</w>': 595, '   inter vi e w</w>': 108, '   mon i c a</w>': 34, '   wri ting</w>': 203, '   pre si dent</w>': 513, '   pa ys</w>': 62, '   ma ga z ine</w>': 110, '   y or k</w>': 484, '   fin est</w>': 50, '   tra i tor</w>': 22, '   mu r der er</w>': 94, '   di re ctor</w>': 114, '   ru ssi an</w>': 130, '   vi de o</w>': 112, '   ti t les</w>': 8, '   di re c ted</w>': 19, '   ra z gu l</w>': 1, ' this</w>': 213, '   a mer i can</w>': 322, '   vi o l ence</w>': 61, '   qui et</w>': 327, '   han ded</w>': 33, '   ta pe</w>': 260, '   pro je ct</w>': 110, ' ac tion</w>': 8, '   shut</w>': 967, '   ba th room</w>': 199, '   ge or ge</w>': 573, '   mi cha el</w>': 325, '   par an o id</w>': 53, ' fif ty</w>': 70, '   su c c ess</w>': 82, '   mu r d ers</w>': 75, '   pre ten ded</w>': 13, '   ac qui tt ed</w>': 2, '   p sy chi a tri sts</w>': 10, '   cer ti f y</w>': 1, '   s an e</w>': 32, '   d ouble</w>': 200, '   je op ar dy</w>': 6, '   er a se</w>': 18, '   an g ry</w>': 199, '   mi lo s</w>': 4, '   li ght</w>': 514, '   thir d</w>': 271, '   h ome si ck</w>': 7, '   move</w>': 907, '   rea l er</w>': 2, '   che mi ca ls</w>': 25, ' for</w>': 107, '   s mo king</w>': 114, '   stu pi d ne ss</w>': 1, '   car ry</w>': 272, '   ba g</w>': 291, '   re fu se</w>': 74, '   ba gs</w>': 75, '   vi de o ca mer as</w>': 1, '   co l or</w>': 148, '   vi e w fin der</w>': 2, '   st ab i li z ation</w>': 2, '   so l ar i z ation</w>': 1, '   vi sion</w>': 75, '   u g ly</w>': 140, '   al most</w>': 645, '   thr own</w>': 78, '   times</w>': 847, '   s qu are</w>': 130, '   do cu ment</w>': 33, '   tri p</w>': 323, '   a mer i c a</w>': 238, '   m ou th</w>': 420, '   fi x ing</w>': 30, '   to i le ts</w>': 8, '   pl u mber</w>': 14, '   h a</w>': 281, '   cl er k</w>': 39, '   sh ouldn</w>': 594, '   sur pri se</w>': 241, '   vi e w er</w>': 9, '   di sc re tion</w>': 20, '   ad vi sed</w>': 20, '   whi s k ey</w>': 57, '   ro ber t</w>': 146, '   as ha med</w>': 92, '   f oo ta ge</w>': 25, '   cu t l er</w>': 1, '   tri es</w>': 66, '   con vi ct</w>': 31, '   pa u lie</w>': 52, '   o ver tur es</w>': 2, '   clo set</w>': 55, '   wal ked</w>': 204, '   morning</w>': 1243, '   mar ri age</w>': 215, '   won der ful</w>': 450, '   fran k</w>': 550, '   ca pr a</w>': 4, '   whi te</w>': 667, ' to</w>': 212, '   ch ea p</w>': 159, '   na tion al</w>': 125, ' he</w>': 359, '   dri ve</w>': 578, '   pre c in ct</w>': 17, '   co ll ar</w>': 46, '   pri ori ti es</w>': 15, '   bur ned</w>': 116, '   the m se l ves</w>': 157, '   thanks</w>': 1523, '   ci vi l</w>': 65, '   ro b</w>': 150, '   lu ck y</w>': 454, '   wal king</w>': 229, '   su e</w>': 125, '   vi o la ted</w>': 17, '   st ab </w>': 29, '   fi re men</w>': 16, '   g un s</w>': 215, '   happy</w>': 876, '   sp are</w>': 129, '   wife</w>': 1236, '   lea v in</w>': 28, '   ac c el er ant</w>': 1, '   re li gi ous</w>': 48, '   a the i st</w>': 6, '   for e head</w>': 24, '   ma x</w>': 280, '   ge tter</w>': 2, '   bo o by</w>': 10, '   tra pped</w>': 61, '   la y</w>': 197, '   hur ts</w>': 101, '   a w</w>': 168, '   f all</w>': 357, '   as le ep</w>': 208, '   gar ci a</w>': 5, '   li v in</w>': 38, '   k ar lo v a</w>': 1, '   lan d l or d</w>': 21, ' in</w>': 336, '   sc ar ed</w>': 554, '   te st ing</w>': 27, '   per m ea te</w>': 1, '   no st ri ls</w>': 9, '   no se</w>': 232, '   cl ean</w>': 372, '   pi cking</w>': 121, '   sc en ts</w>': 2, '   ton y</w>': 104, '   2 </w>': 98, ' 1 </w>': 66, '   ro o ki e</w>': 11, '   w ou l da</w>': 59, '   na w</w>': 116, '   c r ow d</w>': 93, '   su spi ci ous</w>': 63, '   au to gra p h</w>': 29, '   sh oo t</w>': 577, '   co ld</w>': 480, '   st oo p</w>': 6, '   l ev el</w>': 168, '   vi de o ta p ed</w>': 1, '   s oun ded</w>': 60, '   re gi st er ed</w>': 28, '   fran c is</w>': 41, '   app ly</w>': 35, '   sa m</w>': 378, '   pre v ent</w>': 41, '   c ri min al s</w>': 41, '   pro fi ting</w>': 1, '   c ri mes</w>': 38, '   ne go ti a tions</w>': 14, '   per man en tly</w>': 26, '   dis ru p ted</w>': 4, '   se lling</w>': 100, '   pa in t in gs</w>': 19, '   ha sn</w>': 337, ' in ci dent</w>': 3, '   ju mp</w>': 191, '   ar ti st</w>': 93, '   vi c ti m</w>': 99, '   ma st ers</w>': 25, '   gra du ate</w>': 38, '   sa v v y</w>': 8, '   men ta lly</w>': 28, '   in com pe t ent</w>': 16, '   fin ger</w>': 115, '   pu lled</w>': 194, '   mor ally</w>': 8, '   re sp on si ble</w>': 156, '   p sy chi a tri st</w>': 57, '   re sp on si bi li ty</w>': 113, '   in si st</w>': 75, '   me di ca tion</w>': 28, '   re la tion ship</w>': 185, '   pen ny</w>': 80, '   lo af ers</w>': 7, '   fif ti es</w>': 11, '   ac tions</w>': 39, '   cont in u ed</w>': 55, '   s my s lo v </w>': 3, '   li ber ty</w>': 47, '   com m uni ca tion</w>': 36, '   e qui p ment</w>': 85, '   el en a</w>': 28, '   sp ace</w>': 259, '   sta t tion</w>': 1, '   5 </w>': 82, '   l oun ge</w>': 20, '   ro ger</w>': 122, '   st r en g th</w>': 127, '   par ti cu l ar</w>': 116, '   f re e ze</w>': 56, '   fa il ed</w>': 61, '   figu re</w>': 574, '   fi x</w>': 264, '   ha l</w>': 61, '   as se mb ly</w>': 17, '   wi se</w>': 107, '   uni ts</w>': 45, '   sp ar es</w>': 2, '   pro ce du re</w>': 51, '   re por ted</w>': 40, '   a o</w>': 4, ' un it</w>': 3, '   fa il</w>': 77, '   wi de</w>': 75, '   a wa ke</w>': 109, '   rea son ably</w>': 13, '   re place</w>': 48, '   sa fe</w>': 530, '   li ke ly</w>': 101, '   to l er an ces</w>': 3, '   ge ar</w>': 80, '   l ow</w>': 241, '   com pu t ers</w>': 55, '   lo gi ca lly</w>': 9, '   im possi ble</w>': 212, '   in con ce i v able</w>': 9, '   f an ta sti c</w>': 93, '   st ran ge</w>': 339, '   sen se</w>': 584, '   a part</w>': 183, '   con ce i v able</w>': 6, '   app ar en tly</w>': 123, '   beau ti es</w>': 7, '   tra in ed</w>': 72, '   se par a tely</w>': 6, '   f act</w>': 609, '   ru m our</w>': 8, '   or bi tal</w>': 4, ' out</w>': 116, '   sin i ster</w>': 6, '   ex plan ation</w>': 80, '   spe ci a li z ed</w>': 4, '   sp l it</w>': 175, '   gr ou ps</w>': 26, '   fu ss</w>': 33, '   stra i gh ten</w>': 37, '   ought</w>': 314, '   hou st on</w>': 37, '   char ged</w>': 50, '   ent</w>': 1, '   ac coun ting</w>': 19, '   offi ces</w>': 42, '   ye st er day</w>': 359, '   fin ally</w>': 317, '   offi ce</w>': 639, '   re ce i ved</w>': 61, '   a gs</w>': 3, ' 1 9 </w>': 3, '   no ti fi ca tion</w>': 3, '   men tion</w>': 205, '   as su med</w>': 35, '   pa ying</w>': 161, '   gra de</w>': 86, '   che qu e</w>': 11, ' 1 8 </w>': 6, '   pa p ers</w>': 231, '   offi ci al</w>': 75, ' gra ding</w>': 1, '   sa l ary</w>': 48, '   che qu es</w>': 2, '   an no ying</w>': 19, '   da ve</w>': 154, '   se c</w>': 50, '   car e fu lly</w>': 96, '   re lea se</w>': 83, '   hi ber na tion</w>': 5, '   or der</w>': 428, '   cen tra l</w>': 89, '   dis con ne ction</w>': 2, '   c r ying</w>': 144, '   en th u si as m</w>': 11, '   con f i</w>': 2, '   d ence</w>': 1, '   pre par ed</w>': 115, '   in st ru c tions</w>': 63, '   sha ll</w>': 494, '   for c ed</w>': 90, '   dis con ne ct</w>': 12, '   ac cor dan ce</w>': 6, '   su b</w>': 66, ' r out ine</w>': 7, ' 1 5 3 2 </w>': 1, ' 4 </w>': 45, '   in cap ac i ta ted</w>': 2, '   com pu ter</w>': 237, '   as su me</w>': 126, '   un qu o te</w>': 4, '   ther e fore</w>': 51, '   o ver ri de</w>': 14, '   au th ori ty</w>': 81, '   in te l</w>': 16, '   li gen tly</w>': 1, '   ex er ci se</w>': 60, '   man u al</w>': 37, '   t one</w>': 52, '   vo ice</w>': 273, '   st re ss</w>': 45, '   pi ll</w>': 36, '   as ser t</w>': 6, '   st re ss ful</w>': 9, '   s wi tch</w>': 100, '   de ter min ed</w>': 43, '   re vi ve</w>': 5, '   han dle</w>': 368, ' bo ard</w>': 12, '   me mor y</w>': 215, '   st ore</w>': 321, '   han d ling</w>': 39, '   re qui re men ts</w>': 5, '   re pa ir ing</w>': 4, '   an ten na</w>': 11, '   for go tt en</w>': 120, '   re vi ved</w>': 6, '   mon th s</w>': 591, '   re li e f</w>': 37, '   g rea te st</w>': 169, '   con fi d ence</w>': 84, '   fu lly</w>': 78, '   re st or ed</w>': 15, '   mi su n der stan ding</w>': 21, '   plea sed</w>': 101, '   in te g ri ty</w>': 29, '   re li ab i li ty</w>': 3, '   cer ta in ly</w>': 388, '   dis con ne c ted</w>': 11, '   te mp or ar i ly</w>': 15, '   en ti re</w>': 237, '   hi story</w>': 307, '   sin c ere</w>': 26, '   com pe t ent</w>': 13, '   as sure</w>': 75, ' uni ts</w>': 7, '   ac coun t</w>': 194, '   questi on ing</w>': 40, '   im min ent</w>': 4, '   fa i lu re</w>': 78, '   un it</w>': 86, '   op er a tion al</w>': 23, '   with in</w>': 184, '   se v enty</w>': 158, '   f</w>': 116, ' p</w>': 122, ' c</w>': 146, '   imp en ding</w>': 4, '   ori en ta tion</w>': 7, '   inter ru pt</w>': 30, '   fe sti vi ti es</w>': 4, '   kn ow ing</w>': 183, '   a spe ct</w>': 5, '   si lly</w>': 220, '   con in u ed</w>': 1, '   men tion ing</w>': 16, ' b ye</w>': 120, '   bi r th day</w>': 228, '   dar ling</w>': 310, '   lo ve ly</w>': 199, '   mrs</w>': 930, '   br own</w>': 150, '   f oo ling</w>': 34, '   f li ght</w>': 156, '   ber ke le y</w>': 22, '   con ta min ation</w>': 8, '   re tur ning</w>': 24, '   mar s</w>': 61, '   happ en</w>': 896, '   e pi de mi c</w>': 12, '   gre g or</w>': 22, '   see ing</w>': 379, '   j un e</w>': 51, '   o d d</w>': 88, '   cla vi us</w>': 7, '   a ir</w>': 468, '   tr ans m it</w>': 17, '   re fu sa l</w>': 6, '   de li gh t ful</w>': 21, '   age</w>': 258, '   gr ow ing</w>': 89, '   fa st</w>': 468, '   char m ing</w>': 98, '   se ction</w>': 84, '   do cking</w>': 4, '   do cu men ta tion</w>': 6, '   plea ant</w>': 1, '   under stan ding</w>': 80, '   ter ri b ly</w>': 79, '   do ck</w>': 39, '   pl enty</w>': 221, '   su d den ly</w>': 166, '   ran g</w>': 16, '   mr</w>': 3711, '   mi ll er</w>': 79, '   per mi tt ed</w>': 14, '   ea si ly</w>': 84, '   ba se</w>': 125, '   f lo y d</w>': 12, '   f ac ts</w>': 94, '   fran k ly</w>': 90, '   re li able</w>': 28, '   in te lli g ence</w>': 99, '   re por ts</w>': 60, '   app er en tly</w>': 1, '   pre ss ing</w>': 21, '   re ti c ent</w>': 1, '   stra i gh t for w ard</w>': 5, '   for t un a tely</w>': 19, '   sa fe ly</w>': 40, '   a fa id</w>': 1, '   row</w>': 75, '   den ying</w>': 13, '   vi o la tion</w>': 39, ' s</w>': 775, '   con ven tion</w>': 45, '   wh en ever</w>': 120, '   re cor ding</w>': 25, '   re p ea ts</w>': 6, '   lin es</w>': 142, ' as</w>': 58, '   mo on</w>': 151, '   si mp son</w>': 14, '   lo g</w>': 38, '   su n</w>': 231, '   g lan ce</w>': 13, ' p ow er ed</w>': 7, '   de li ber a tely</w>': 33, '   bu ry</w>': 73, '   p ow er ed</w>': 6, '   d ev ice</w>': 59, '   co l our</w>': 6, '   tom b</w>': 30, '   sh ine</w>': 35, '   sur v ey</w>': 9, ' mar k er</w>': 2, '   in er t</w>': 1, '   en er g y</w>': 127, '   s our ces</w>': 32, '   de te c ted</w>': 10, '   sur face</w>': 72, '   bar e ly</w>': 91, '   sc ra tch</w>': 67, '   la s er</w>': 39, '   dri ll</w>': 38, '   c lu e</w>': 61, '   de for ma tion</w>': 1, '   be tw e en</w>': 594, '   fi ll</w>': 141, '   bu ried</w>': 133, ' sur face</w>': 1, '   st ru c ture</w>': 44, '   w ra ps</w>': 8, '   de e med</w>': 12, '   ne ce ss ary</w>': 133, '   re que sted</w>': 31, '   for ma l</w>': 26, '   se cu ri ty</w>': 284, '   o a th s</w>': 2, '   ob ta in ed</w>': 7, '   kn ow le dge</w>': 104, '   ev ent</w>': 95, '   a de qu ate</w>': 7, '   c on</w>': 74, '   si der ation</w>': 1, '   an n oun ce ment</w>': 15, '   com for ta ble</w>': 144, '   con cer ned</w>': 169, '   st ori es</w>': 186, '   al ar m</w>': 85, '   au di ting</w>': 2, '   ter ri t ori al</w>': 8, '   ad min i stra tor</w>': 7, '   he ars</w>': 43, '   con fin ed</w>': 14, '   re f it</w>': 3, ' four</w>': 156, '   in ci den ta lly</w>': 24, '   ar ran ge men ts</w>': 43, '   s che du l ed</w>': 18, '   ar ri ve</w>': 45, '   we i gh t less</w>': 4, '   mar ve ll ous</w>': 6, '   f la w</w>': 17, '   ca u sing</w>': 33, '   tr ac king</w>': 31, '   a li g n ment</w>': 8, ' o</w>': 202, '   f p c</w>': 1, '   op en ing</w>': 102, '   p od</w>': 15, '   b ay</w>': 102, '   do ors</w>': 117, '   de com pre ssed</w>': 1, '   se cu re</w>': 64, '   a ir lo ck</w>': 18, ' v </w>': 66, '   ar ms</w>': 150, '   comp on ent</w>': 7, '   cont in u ation</w>': 1, '   pro gra m</w>': 167, '   fu r ther</w>': 145, '   gen er al</w>': 390, '   plan e ts</w>': 18, '   re co very</w>': 19, '   ve hi c le</w>': 53, '   r en de z ous</w>': 1, '   tr ans it</w>': 17, '   2 5 7 </w>': 1, '   pro fi le</w>': 49, '   ca lls</w>': 295, '   dis co very</w>': 28, '   sa turn</w>': 8, '   f al se</w>': 63, '   sta te men ts</w>': 17, '   ti r es</w>': 19, '   se ll</w>': 335, '   g as</w>': 185, '   char g in</w>': 3, '   ba tt er y</w>': 35, '   ha m mon d</w>': 16, '   o be y in</w>': 1, '   ha ss le</w>': 14, '   p al s</w>': 39, '   tal kin</w>': 218, '   pa l</w>': 201, '   tur b an</w>': 7, '   s qu a w</w>': 5, '   ki d d in</w>': 44, '   th ro at</w>': 124, '   du m my</w>': 31, '   pro s</w>': 14, '   wh ad d ya</w>': 31, '   ain</w>': 1876, '   sto le</w>': 180, '   tru ck</w>': 250, '   ton to</w>': 2, '   din ner</w>': 583, '   s an</w>': 173, '   fran ci sc o</w>': 82, '   dan c in</w>': 13, '   tr y in</w>': 115, '   pr o</w>': 66, '   st ru ck</w>': 60, '   hello</w>': 1095, '   ho p ing</w>': 136, '   ac ro ss</w>': 276, '   be in</w>': 77, '   pri est</w>': 63, '   wai ting</w>': 644, '   hur ry</w>': 336, '   thr ow</w>': 390, '   st y list s</w>': 1, '   inter est</w>': 237, '   go ver n ment</w>': 245, '   de fini tely</w>': 189, '   se ll in</w>': 6, '   sch oo l t ea ch er</w>': 6, '   bu st</w>': 106, '   can dy</w>': 72, '   cor ri d or</w>': 30, '   g an z</w>': 33, '   ven ti la ted</w>': 3, '   bi lly</w>': 228, '   la dy</w>': 551, '   ri p</w>': 66, '   l un gs</w>': 34, '   vi si t ors</w>': 25, '   ow es</w>': 50, '   dro p</w>': 386, '   god damn</w>': 610, '   be g</w>': 156, '   l ying</w>': 315, '   c ro ok</w>': 35, '   le g it</w>': 20, '   in ve st men ts</w>': 9, '   ven ti late</w>': 2, '   su it</w>': 260, '   a we some ly</w>': 1, '   we ir d</w>': 292, '   run n in</w>': 47, '   st y le</w>': 96, '   re g gi e</w>': 35, '   ca tes</w>': 7, '   mer it</w>': 15, '   sy st em</w>': 337, '   gon n na</w>': 1, '   m ac h o</w>': 16, '   bu st in</w>': 13, '   ch op s</w>': 16, '   part n er ship</w>': 15, '   en ded</w>': 93, '   ma k er</w>': 32, '   r ough</w>': 140, '   the ory</w>': 131, '   tur n s</w>': 179, '   ri o</w>': 34, '   j an e ir o</w>': 7, '   sc re w</w>': 153, '   bed</w>': 677, '   har d ly</w>': 252, '   r an</w>': 271, '   sha cked</w>': 2, '   d y ke</w>': 11, '   clo sing</w>': 65, '   imp oun ded</w>': 3, '   ha u l in</w>': 3, '   s la m</w>': 11, '   st op s</w>': 65, '   no ti ce</w>': 217, '   bu s</w>': 217, '   lu ther</w>': 48, '   ca ll in</w>': 34, '   wa ter me ll on</w>': 1, '   ni g ger</w>': 88, '   k in ds</w>': 83, '   le an in</w>': 3, '   m ee ting</w>': 375, '   8 </w>': 73, ' don</w>': 150, '   di stu r b</w>': 37, '   tra ding</w>': 19, '   ta x i</w>': 65, '   poli te</w>': 50, '   v ro man</w>': 2, '   fi ll more</w>': 3, ' course</w>': 72, '   sta ying</w>': 254, '   ar med</w>': 92, '   ro b ber y</w>': 54, '   pi mp ing</w>': 1, '   co ps</w>': 362, '   du mb </w>': 228, '   lan ded</w>': 39, '   clo se</w>': 771, '   pi lls</w>': 105, '   p un k</w>': 51, '   l ow life</w>': 5, '   e mb ar ra ssed</w>': 62, '   wh ee ls</w>': 40, '   a g b</w>': 1, '   ra tt l ed</w>': 3, '   bra in</w>': 302, '   mo ves</w>': 111, '   du mber</w>': 18, '   a ma z in</w>': 4, '   de pre ss ing</w>': 29, '   vi e w</w>': 154, '   s mi le</w>': 111, '   hi tting</w>': 60, '   ti r ing</w>': 9, '   p hi lo so ph y</w>': 40, '   com m on</w>': 156, '   que st</w>': 33, '   pu ss y</w>': 105, '   se par ate</w>': 67, '   sp en ding</w>': 77, '   si ze</w>': 155, '   s mar t</w>': 360, '   ta il ed</w>': 4, '   s k y b lu e</w>': 1, '   ca di ll ac </w>': 33, '   dar k er</w>': 7, '   for ei g n</w>': 75, '   jo b s</w>': 96, '   d ust</w>': 73, ' why</w>': 142, '   dis gr ace</w>': 22, '   pu lls</w>': 49, '   2 4 </w>': 48, '   cour ts</w>': 18, '   re vo l ving</w>': 3, '   pri m o</w>': 5, '   b on d s man</w>': 8, '   chi ck en shit</w>': 12, '   dro ve</w>': 128, '   t ow</w>': 20, ' away</w>': 16, '   z one</w>': 68, '   par ked</w>': 38, '   part n ers</w>': 82, '   tr un k</w>': 105, '   ma tt re ss</w>': 22, '   bu i lt</w>': 193, '   po in ts</w>': 83, '   5 0</w>': 57, ' 5 0</w>': 14, ' okay</w>': 68, '   bu n ch</w>': 261, '   dea l er</w>': 54, '   sa le</w>': 74, '   sto l en</w>': 95, '   si tt in</w>': 35, '   co tt on</w>': 59, '   fin ger ed</w>': 12, '   p sy ch o</w>': 80, '   ca pp ing</w>': 2, '   sa ved</w>': 241, '   ba il</w>': 64, '   wee k en d pa ss</w>': 1, '   spe ar</w>': 16, '   ch u ck er</w>': 1, '   st en ci l ed</w>': 1, '   fa ti gu es</w>': 2, '   4 8 </w>': 17, '   clo ck</w>': 89, '   bu st ing</w>': 39, '   d y kes</w>': 4, '   be d ded</w>': 1, '   la u gh</w>': 193, '   di cking</w>': 4, '   tur d</w>': 12, ' h un t</w>': 5, '   ga mes</w>': 175, '   su cks</w>': 60, '   man i ac </w>': 27, '   st re e ts</w>': 114, '   ou gh ta</w>': 98, '   ro a m ing</w>': 7, '   o ver dre ssed</w>': 4, '   char co al</w>': 5, ' co l or ed</w>': 6, '   la di es</w>': 148, '   la ps</w>': 7, '   s wi tch b la de</w>': 5, '   ci ti z en s</w>': 44, '   offi c er</w>': 291, '   bu ddy</w>': 439, '   en jo y</w>': 223, '   lo an</w>': 84, '   ba dge</w>': 65, '   na il ed</w>': 34, '   bu sted</w>': 130, '   tal ks</w>': 101, '   jo int</w>': 112, '   h our</w>': 564, '   la id</w>': 143, '   pla y in</w>': 38, '   b ars</w>': 68, '   r ou sted</w>': 2, '   cl ow n s</w>': 20, '   at ti tu de</w>': 111, '   ca ts</w>': 69, '   car ve</w>': 7, ' bi lly</w>': 4, '   t end</w>': 43, '   b ar</w>': 249, ' tell</w>': 47, '   ni gh ts</w>': 130, '   de ta i ls</w>': 97, '   di st ri ct</w>': 56, '   com pla int</w>': 20, '   po pu la tion</w>': 25, '   pi ss es</w>': 15, '   on o e</w>': 1, '   bo o king</w>': 9, '   man do lin s</w>': 1, '   h un g ry</w>': 256, '   mo ving</w>': 333, '   ti ll</w>': 470, '   t rea ting</w>': 50, '   pen i ten ti ary</w>': 10, '   sp ea kin</w>': 6, '   mo ans</w>': 5, '   st om ac h</w>': 115, '   star t in</w>': 19, '   gr ow l</w>': 3, '   b le e ds</w>': 10, ' tri m</w>': 1, '   h un t</w>': 135, '   mo an ing</w>': 9, '   tri m</w>': 12, '   tu e s day</w>': 81, '   war n</w>': 78, '   ice</w>': 265, '   po in t in</w>': 2, '   god da m</w>': 79, '   un do</w>': 11, '   cu ff</w>': 17, '   nu ts</w>': 205, '   bo ther</w>': 263, '   k no ck in</w>': 7, '   com pla in in</w>': 3, '   shi th ead</w>': 25, '   fi le</w>': 182, '   g an g</w>': 97, '   a we some</w>': 23, '   na med</w>': 268, '   pu tt in</w>': 34, '   kee p in</w>': 24, '   br ac e le ts</w>': 3, '   gen er o si t y of</w>': 1, '   c ea ses</w>': 5, '   a ma ze</w>': 8, '   j i ve</w>': 7, '   loo k in g real</w>': 1, '   sh ar p</w>': 74, '   st r ing</w>': 42, '   ho ok ers</w>': 22, ' 9 0 0</w>': 5, ' 4 0 0</w>': 5, '   su i ts</w>': 77, '   pro te ct</w>': 233, '   de li ver</w>': 89, '   ca pped</w>': 5, '   ro a d</w>': 354, '   con vi ction</w>': 23, '   p ink</w>': 88, '   s li p</w>': 93, '   bra ins</w>': 134, '   gu ts</w>': 97, '   cor ner</w>': 160, '   wa tch in</w>': 26, '   wri t in</w>': 8, '   ti ck e ts</w>': 97, '   bi ke</w>': 36, '   bu ll et</w>': 101, '   b ack up</w>': 31, '   g ran d stand</w>': 1, '   me ssed</w>': 49, '   e spe ci ally</w>': 216, '   t ar ge ts</w>': 15, '   c r ow ds</w>': 13, '   3 0</w>': 73, '   y ar ds</w>': 58, '   tra in</w>': 231, '   be ll</w>': 70, '   pla y er</w>': 110, '   less</w>': 370, '   bo th ers</w>': 40, '   an thing</w>': 1, '   bo ther in</w>': 8, '   lo s in</w>': 10, '   poli cy</w>': 99, '   ki ll ers</w>': 59, '   re pre s ent</w>': 51, '   pri ori ty</w>': 34, '   g rea ter</w>': 43, '   th re at</w>': 86, '   un ar med</w>': 14, '   re ven ge</w>': 72, '   al gr en</w>': 1, '   st ea ls</w>': 7, '   t ine</w>': 6, '   s qu a d</w>': 76, '   hea ding</w>': 97, '   red</w>': 403, '   ad di tion</w>': 17, '   lo sing</w>': 165, '   ha d en</w>': 1, '   pa t ro l man</w>': 3, '   j i g</w>': 7, ' been</w>': 14, '   a w ful</w>': 228, '   em</w>': 59, '   che cks</w>': 62, '   di ck</w>': 302, '   tr ac y</w>': 36, '   u h h</w>': 34, '   w ong</w>': 10, '   h en ry</w>': 168, '   be ar</w>': 150, ' 4 4 </w>': 10, '   d un no</w>': 165, '   s ki p</w>': 69, '   sta sh ed</w>': 18, '   ro sa lie</w>': 6, '   fa i ry</w>': 47, '   god mother</w>': 5, ' where</w>': 93, '   pro mi sed</w>': 214, ' ass</w>': 72, '   u su al</w>': 130, '   sh oo t in</w>': 28, '   hea d qu ar t ers</w>': 41, '   bar ten der</w>': 25, '   sh r ink</w>': 65, '   b le e ding</w>': 69, '   han ds</w>': 609, '   ir re sti ble</w>': 1, '   re ex a m ine</w>': 1, '   pri mi tive</w>': 36, '   v u l n er able</w>': 32, '   s lu mber</w>': 4, '   le s bi ans</w>': 1, '   qu a in tly</w>': 1, '   ob vi ous</w>': 121, '   sc hi z o</w>': 4, '   wi r es</w>': 33, '   c ro ssed</w>': 51, '   pa tt er n</w>': 75, '   a ir por t</w>': 140, '   do cks</w>': 16, ' dead</w>': 25, '   ch in</w>': 40, '   par don</w>': 200, '   hi ya</w>': 37, '   e la ine</w>': 115, '   sur pri sed</w>': 200, '   be have</w>': 44, '   s one</w>': 1, ' hea ded</w>': 27, '   o c ca si on al</w>': 19, '   ro om ma te</w>': 32, '   hu man ly</w>': 10, '   won der ing</w>': 212, '   s ac red</w>': 54, '   per son ne l</w>': 39, '   pr ac ti cal</w>': 45, '   fun ction</w>': 39, '   w ea p on</w>': 119, '   fa v ori tes</w>': 15, '   wa ll et</w>': 82, '   ho pe less</w>': 36, '   fa ir ly</w>': 34, '   c ru m my</w>': 20, '   shi r ts</w>': 22, '   sa d</w>': 181, '   out f it</w>': 63, '   bi lls</w>': 85, '   mi x</w>': 52, '   p in a</w>': 1, '   co la da s</w>': 2, '   bi g ger</w>': 152, '   d y in</w>': 26, '   sy m pa the ti c</w>': 15, '   b ru tal</w>': 26, '   in differ ence</w>': 4, '   p sy cho ther a p y</w>': 5, '   po si tive</w>': 115, ' i ma ge</w>': 3, '   as sho les</w>': 74, '   li st</w>': 255, '   ac a de mi c</w>': 12, '   cre den ti al s</w>': 13, '   bo o ze</w>': 36, ' h ound</w>': 2, '   ir ri ga ted</w>': 1, '   j </w>': 154, '   b</w>': 218, '   p ou red</w>': 8, '   de ck</w>': 76, '   su ck er</w>': 55, '   w et</w>': 104, '   lo ad ed</w>': 81, '   ma g nu m</w>': 13, '   p ow er ful</w>': 115, '   han d gu n</w>': 4, '   fee l in</w>': 49, '   s oun ds</w>': 512, '   pa the ti c</w>': 80, '   m om ma</w>': 44, '   ace</w>': 46, '   de te c ti ves</w>': 41, '   lo y al</w>': 47, '   bu ll</w>': 69, '   we ar in</w>': 12, ' you</w>': 1335, ' hold</w>': 17, '   sta ll in</w>': 2, '   com pla in</w>': 48, '   te m per a ture</w>': 36, '   me ss</w>': 223, '   ho les</w>': 71, '   hon est</w>': 304, '   op en s</w>': 51, '   mon day</w>': 103, '   wor r y in</w>': 12, '   hi d</w>': 56, '   s we at</w>': 79, '   nu mb ers</w>': 155, '   sh e lls</w>': 24, '   a mm o</w>': 30, '   hu s b and</w>': 605, '   da vi d</w>': 349, '   sha pe</w>': 117, '   a k ta</w>': 3, '   de der o</w>': 1, '   an si l a</w>': 1, '   me k t et</w>': 1, '   pr ou d</w>': 192, '   v an o</w>': 1, '   me ch te b a</w>': 1, '   s ou n</w>': 1, '   d om o</w>': 1, '   k al a</w>': 1, '   ch on</w>': 1, '   ha m ma s</w>': 1, '   i k set</w>': 1, ' ki b a</w>': 1, '   i man e ta b a</w>': 1, '   ou m</w>': 1, '   da l at</w>': 1, '   aga m at</w>': 1, '   cha y</w>': 1, '   en vo let</w>': 1, ' with</w>': 77, '   st on es</w>': 62, '   na po le on</w>': 36, '   la u gh ing</w>': 101, '   cor n</w>': 40, ' li us</w>': 1, '   app i pu la i</w>': 1, '   le e lo o</w>': 11, '   min a i</w>': 2, '   mon ster</w>': 168, '   z or g</w>': 5, '   rea s sure</w>': 10, '   re li gi on</w>': 41, '   2 0 0 0</w>': 14, '   de st ro ying</w>': 26, '   gla ss</w>': 212, ' ki lling</w>': 5, '   pro du ce</w>': 41, '   c rea ting</w>': 22, '   de st ru ction</w>': 46, '   en cou ra ging</w>': 8, '   rea li ty</w>': 100, '   ro bo ts</w>': 5, ' look</w>': 69, '   ba ll et</w>': 26, '   for m</w>': 174, ' life</w>': 43, '   no b ly</w>': 1, '   ser ve</w>': 120, ' would</w>': 25, '   de st ro y</w>': 167, ' but</w>': 358, '   cu st om er</w>': 45, '   j ean</w>': 105, ' ba p ti st e</w>': 1, '   e m man u el</w>': 1, ' we</w>': 365, '   bea ms</w>': 14, '   the or e ti ca lly</w>': 12, '   bea m</w>': 68, '   fif th</w>': 69, '   e le ment</w>': 30, '   re f er ence</w>': 22, '   di v a</w>': 4, '   su i te</w>': 45, '   no ble</w>': 54, '   ar my</w>': 368, '   of f er</w>': 286, '   v ac ation</w>': 137, '   pri e sts</w>': 19, '   re sor t</w>': 18, '   me th o ds</w>': 27, '   ra di o</w>': 306, '   f h lo st on</w>': 5, ' never</w>': 45, '   ga m at</w>': 2, '   ar ri ved</w>': 91, ' she</w>': 146, '   da ll as</w>': 72, '   k or b en</w>': 2, '   ex ce p tion</w>': 30, ' di st ant</w>': 1, '   gen t le</w>': 44, '   man kind</w>': 27, '   per fe ct</w>': 419, '   dro pped</w>': 164, '   bri de</w>': 48, '   f are</w>': 30, '   vi to</w>': 17, '   cor ne li us</w>': 21, '   gu i de</w>': 48, '   we d din gs</w>': 8, ' thing</w>': 20, ' s wa llow</w>': 1, '   ba tt le ship</w>': 8, '   gu m</w>': 48, '   op tions</w>': 48, '   en ter</w>': 59, '   ter ri t ori es</w>': 13, '   war me st</w>': 4, '   re gar ds</w>': 15, '   war ri or</w>': 47, '   stan ds</w>': 72, ' thank</w>': 29, '   go al</w>': 34, '   wi pe</w>': 40, '   for ms</w>': 43, '   u p se ts</w>': 6, ' e ight</w>': 101, '   a da pt</w>': 10, '   i t self</w>': 162, '   li ving</w>': 570, '   con di tions</w>': 49, '   2 0 0</w>': 25, '   bi lli on</w>': 57, '   fe llow</w>': 201, '   imp ort an ce</w>': 30, '   e h</w>': 350, '   i den ti fi ed</w>': 25, '   pre f ers</w>': 9, '   an ti the s is</w>': 3, '   plan e</w>': 351, '   par a di se</w>': 36, '   de ta il ed</w>': 15, '   b lu e pr int</w>': 3, '   li mp</w>': 15, ' said</w>': 12, ' at</w>': 98, '   na mes</w>': 207, '   le ar ning</w>': 86, '   5 0 0 0</w>': 8, '   ci r cu la tion</w>': 20, '   tri p le</w>': 27, '   g lo ves</w>': 37, '   su pre me</w>': 27, '   par i sh</w>': 5, '   mi r ac le</w>': 91, '   ad mi tt ed</w>': 25, '   lo ved</w>': 381, ' and</w>': 865, ' per fe ct</w>': 8, ' 7 </w>': 48, '   s kin</w>': 152, ' listen</w>': 29, '   la p</w>': 42, '   y</w>': 457, ' know</w>': 291, '   f ar es</w>': 4, '   re si st</w>': 38, '   bu d</w>': 139, '   d rea ms</w>': 237, '   li cen se</w>': 127, '   ca b</w>': 148, '   fi gh ter</w>': 50, '   for getting</w>': 40, '   s at</w>': 100, '   mi ssi ons</w>': 11, '   ex i st</w>': 112, '   p in ing</w>': 4, '   ti m ing</w>': 49, '   c at</w>': 209, '   ba si c</w>': 66, ' le e lo o</w>': 1, '   ni ck name</w>': 20, '   sh or ter</w>': 17, '   le k ar ar i b a</w>': 1, ' la min a i</w>': 1, ' t cha i</w>': 1, '   e k b at</w>': 1, '   se b at</w>': 1, '   gen tly</w>': 30, '   e to</w>': 1, '   c or</w>': 4, '   n i</w>': 32, ' li ous</w>': 1, ' pri est</w>': 1, '   da ya</w>': 1, '   de o</w>': 1, '   d on o</w>': 1, '   da to</w>': 1, '   da lu t an</w>': 1, ' cu se</w>': 10, '   a k in a</w>': 1, '   de lu t an</w>': 1, '   n ou </w>': 1, ' sha n</w>': 2, '   l or d</w>': 336, '   cen tu ri es</w>': 33, '   i ts</w>': 538, '   sen ses</w>': 27, '   e le men ts</w>': 29, ' ma</w>': 21, '   s lea ze</w>': 6, ' not</w>': 161, '   bl ar ing</w>': 2, '   b lo ck head</w>': 4, '   no ti fi ed</w>': 16, '   do lt</w>': 4, '   s ma sh ed</w>': 31, '   mu g ged</w>': 11, '   p ea ch y</w>': 6, '   ca l m ly</w>': 7, '   war ning</w>': 76, '   in gra te</w>': 1, '   shi tty</w>': 45, '   c ro que tt es</w>': 1, '   na sti est</w>': 2, '   di r t ba g</w>': 5, '   st in king</w>': 26, ' hello</w>': 46, '   6 </w>': 79, '   cra te</w>': 13, '   secon ds</w>': 241, '   re min ding</w>': 10, '   n ine</w>': 361, '   ni gh t m are</w>': 97, '   fu el</w>': 65, ' 0 3 </w>': 2, ' pro pu l sion</w>': 1, ' x</w>': 11, '   ok</w>': 605, '   bo ard</w>': 243, '   si gh ed</w>': 2, '   con cen tra te</w>': 52, '   list en ers</w>': 2, '   d j </w>': 11, '   t un e</w>': 52, '   ad ven ture</w>': 55, '   so li d</w>': 88, '   ti ght</w>': 171, '   f ly</w>': 254, '   lea der</w>': 74, '   man g al or es</w>': 1, '   gi ant</w>': 66, '   si st ers</w>': 88, '   ph y si ca lly</w>': 40, '   t re at</w>': 146, '   ro o ts</w>': 20, '   chi l d ho od</w>': 34, '   ma in</w>': 140, '   ad or ing</w>': 4, '   f ans</w>': 33, '   te ar</w>': 94, '   c r y st al</w>': 49, '   du mb o</w>': 2, '   cle ar</w>': 442, ' qui ver</w>': 1, '   de si r es</w>': 15, '   in ti ma te</w>': 21, '   in ti ma tes</w>': 2, '   stu d</w>': 27, '   mu ff in</w>': 20, '   n er v ous</w>': 279, ' hi</w>': 27, ' sorry</w>': 17, '   fu ture</w>': 324, '   ex </w>': 135, '   di sc re et</w>': 19, '   an nu al</w>': 20, '   ge min i</w>': 5, '   con te st</w>': 46, '   me s sa g es</w>': 47, '   hi gh ly</w>': 73, '   de cor a ted</w>': 11, '   e li te</w>': 12, '   fe der a ted</w>': 1, '   ex per t</w>': 86, '   sp ac e cra ft</w>': 23, '   nee ded</w>': 293, '   re tri eve</w>': 10, '   pla v a la g un a</w>': 1, '   u t most</w>': 10, '   s ong</w>': 181, '   5 7 </w>': 8, '   ow ed</w>': 21, '   fe der al</w>': 121, '   en list ment</w>': 2, '   me mor i es</w>': 81, '   n one</w>': 419, '   im mor tal</w>': 18, '   m ac h ine</w>': 273, '   pro gra mm ed</w>': 32, ' like</w>': 105, '   lo ts</w>': 238, '   sa ving</w>': 87, '   an ge l</w>': 97, '   ra in</w>': 127, '   w ind</w>': 207, '   bl ows</w>': 51, '   bur n s</w>': 69, '   w ars</w>': 39, '   di c tion ary</w>': 9, '   stu di ed</w>': 45, '   di f fi cu lt</w>': 191, '   o pp o si te</w>': 51, '   sing</w>': 192, '   differ en ces</w>': 18, '   sc re en</w>': 84, '   nor ma lly</w>': 42, ' ho pp i</w>': 1, ' ho pp a</w>': 1, ' make</w>': 27, '   a pi p ou ss an</w>': 2, ' good</w>': 84, '   9 0 0</w>': 8, '   lan gu a g es</w>': 21, ' some times</w>': 13, ' love</w>': 40, '   op er a tive</w>': 13, ' just</w>': 136, '   a pi p ou la i</w>': 2, '   hon e y mo on</w>': 53, '   din o ine</w>': 1, '   cha g an ta k at</w>': 1, '   v al o</w>': 1, '   ma ss a</w>': 1, '   cha ch a</w>': 1, '   ha ma s</w>': 1, ' see</w>': 48, '   pre p are</w>': 81, '   de ss er t</w>': 22, '   ma the ma ti ca lly</w>': 1, '   gu ar an te e</w>': 79, '   pu sh</w>': 153, '   ye llow</w>': 97, '   bu tt on</w>': 66, '   z</w>': 37, ' 1 4 0</w>': 1, '   a ll ev i a ted</w>': 1, '   ti t ani u m</w>': 7, '   ne u r o</w>': 7, '   as sa u lt</w>': 36, '   re ce i ve</w>': 47, '   in t ro du ce</w>': 90, '   pro fe ss or</w>': 166, '   m ac ti l bur gh</w>': 1, '   cen ter</w>': 120, '   e le ph ant</w>': 30, '   c ru ci al</w>': 16, '   p ha se</w>': 23, '   re con st ru ction</w>': 2, '   pi g ment</w>': 1, '   ce lls</w>': 56, '   b om bar ded</w>': 2, '   g rea sy</w>': 9, '   so l ar</w>': 20, '   at om s</w>': 3, '   rea ct</w>': 15, '   cl ever</w>': 112, '   ce ll u l ar</w>': 19, '   h y gi en e</w>': 9, '   de te ctor</w>': 7, '   c ell</w>': 122, '   vi ru s</w>': 104, '   com po si tion al</w>': 1, '   d na</w>': 48, '   cha in</w>': 71, '   ti gh tly</w>': 3, '   li mi t less</w>': 3, '   li br ary</w>': 65, '   gen e ti c</w>': 34, '   st or ed</w>': 16, ' en g in e er ed</w>': 2, '   do c</w>': 200, '   en coun ter ed</w>': 5, '   be in gs</w>': 41, '   4 0</w>': 36, '   me m o</w>': 20, '   gr ou ms</w>': 1, ' which</w>': 28, '   spe ci es</w>': 55, '   per pe tu ate</w>': 3, '   sur vi ved</w>': 43, '   de ser t</w>': 144, ' st on es</w>': 1, '   w re ck age</w>': 5, '   man a ged</w>': 52, '   mon do sha wa n</w>': 1, '   in ci dent</w>': 57, '   a po lo gi es</w>': 21, '   sta e der t</w>': 2, '   gen t le men</w>': 218, '   un in vi ted</w>': 5, '   gu e sts</w>': 84, '   re com men da tions</w>': 10, '   che mi cal</w>': 35, '   mo le cu l ar</w>': 23, '   an al y s is</w>': 32, '   ca li b ers</w>': 4, '   o ver sho t</w>': 3, '   ther m o</w>': 1, '   nu c lea ti c</w>': 1, '   i ma ging</w>': 3, '   1 0</w>': 105, '   sa lu d</w>': 1, '   to a st</w>': 57, '   a ma z ing</w>': 190, '   par ch ed</w>': 3, '   ex tra or din ary</w>': 41, ' can</w>': 94, '   imp li ca tions</w>': 13, '   a men</w>': 28, '   uni ver se</w>': 96, '   re si d es</w>': 5, ' all</w>': 185, ' pro te ct</w>': 2, '   di sp a tch</w>': 11, '   z f x</w>': 1, ' 2 0 0</w>': 9, '   ba tt er i es</w>': 31, '   per fe c tly</w>': 149, '   con cer t</w>': 34, '   4 </w>': 76, '   co sts</w>': 56, '   tri pl ed</w>': 2, '   di stu r b ing</w>': 19, '   pa per</w>': 340, '   to m</w>': 369, '   chri sti an</w>': 44, '   a my</w>': 107, '   ow e</w>': 265, '   p ack</w>': 144, '   c in dy</w>': 31, '   pre ca u tion</w>': 12, '   per s ons</w>': 28, '   mor t ga ge</w>': 27, '   co ll e ge</w>': 257, '   u sing</w>': 219, '   wor th less</w>': 49, '   bri dge</w>': 192, '   hea ter</w>': 12, '   wee k end</w>': 171, '   be ha vi or</w>': 86, '   ac cu sing</w>': 18, '   ac cu se</w>': 23, '   clo thing</w>': 37, '   re e ks</w>': 7, '   s nu ff</w>': 27, '   mi ster</w>': 419, '   har der</w>': 80, '   mi x ed</w>': 92, '   b on da ge</w>': 13, '   ra pe</w>': 51, '   fi l ms</w>': 51, '   din o</w>': 12, '   con ne ct</w>': 28, '   ri b</w>': 18, '   sp rea der</w>': 1, '   loo p</w>': 17, '   co op er a tive</w>': 5, '   lon g da le</w>': 6, '   de po sit</w>': 37, '   bo x</w>': 243, '   ar row</w>': 16, '   on to</w>': 173, '   we ll es</w>': 23, '   re move</w>': 56, '   fi re ar ms</w>': 11, '   h m m</w>': 118, '   k ni f es</w>': 2, '   pro ps</w>': 9, '   ex ce ll ent</w>': 137, '   bro ok l y n</w>': 55, '   fif te en</w>': 305, '   sh y</w>': 42, '   hi ts</w>': 76, ' ca p</w>': 3, '   per for m er</w>': 5, '   ma s k</w>': 107, '   de li very</w>': 62, '   l ar ge</w>': 154, '   brea sts</w>': 36, '   ar ti sti c</w>': 13, '   inter pre ta tion</w>': 10, '   sti pu la tions</w>': 1, '   spe ci fi c</w>': 44, '   in ve st or</w>': 4, '   f la tt er ing</w>': 17, '   co l or ful</w>': 19, '   ch u m</w>': 19, '   com mi ssion</w>': 86, '   ad mi r er</w>': 14, '   questi on able</w>': 7, '   fe y</w>': 5, '   pu ts</w>': 80, '   han d cu ff s</w>': 14, '   han d cu ff</w>': 2, '   h or ri ble</w>': 144, '   cha sing</w>': 78, '   fi les</w>': 118, '   di sa pp ear an ce</w>': 20, '   chi l dr en</w>': 547, '   a du l ts</w>': 27, '   ha ll</w>': 171, '   mo le sta tion</w>': 3, '   le ga l</w>': 115, '   re ce p tive</w>': 4, ' m</w>': 295, '   cont ac ted</w>': 27, '   p hi la de l p hi a</w>': 43, '   pi cked</w>': 276, '   hi tch hi king</w>': 5, '   8 1 </w>': 4, '   h ome less</w>': 24, '   con v in c ed</w>': 65, '   m ea l</w>': 71, '   ma ture</w>': 24, '   run away</w>': 15, '   te le ph one</w>': 101, '   sur pri sin g ly</w>': 3, '   ate</w>': 124, '   ex cu sed</w>': 9, '   s ke tch</w>': 12, '   ab i de</w>': 8, '   ac c ess</w>': 102, '   ar chi ve</w>': 8, '   sto od</w>': 82, '   wa tch ed</w>': 134, ' fe lt</w>': 4, '   mi ser y</w>': 77, '   wh in ing</w>': 14, '   bur ying</w>': 12, '   t rea sure</w>': 62, '   p ace</w>': 24, '   du g</w>': 33, '   ab so lu te</w>': 36, '   z er o</w>': 127, '   loo se</w>': 182, '   a ven ge</w>': 13, '   w oo ds</w>': 95, ' ho le</w>': 21, '   b ac ked</w>': 22, '   g ro cer i es</w>': 10, '   pla sti c</w>': 61, '   hu mp</w>': 14, '   fe lt</w>': 391, '   ma p</w>': 118, '   st ar</w>': 235, '   so da</w>': 41, '   mi ck ey</w>': 126, '   k no ck</w>': 206, '   mo de ls</w>': 18, '   co ck su ck er</w>': 30, '   cont ac ts</w>': 33, '   cla ssi fi e ds</w>': 2, '   h ow ever</w>': 118, '   en t re pr en e u r</w>': 4, '   ho ok</w>': 115, '   lea ves</w>': 136, '   hur ting</w>': 66, '   re co g ni ze</w>': 148, '   b ore</w>': 24, '   mu r der ed</w>': 165, '   u p s ca le</w>': 2, '   pro du ct</w>': 62, '   so f t c ore</w>': 1, '   pe d d ling</w>': 5, '   s cu m ba g</w>': 29, '   cu st om ers</w>': 64, '   j er k of f s</w>': 2, '   ta p es</w>': 112, '   f li ck</w>': 10, '   j ack ass</w>': 11, '   re ce i pt</w>': 25, '   pla y ed</w>': 180, '   win d shi e ld</w>': 16, '   li m it</w>': 43, '   en li gh ten</w>': 6, '   bu tch er ed</w>': 11, '   att a in ed</w>': 1, '   att ain</w>': 1, '   se cre ts</w>': 81, '   de gen er ate</w>': 10, '   per ver t</w>': 22, '   j er k</w>': 111, '   han d job</w>': 1, '   mar y</w>': 425, '   an n e</w>': 79, '   ma the w s</w>': 6, '   s mu t</w>': 7, '   dea l ers</w>': 28, '   di g ging</w>': 48, '   com m it</w>': 65, '   fo ll ow ed</w>': 125, '   f rea ked</w>': 28, '   bu d di es</w>': 46, '   hi red</w>': 150, '   comp en sa ted</w>': 8, '   tru sted</w>': 69, '   imp li ci tly</w>': 3, '   er r and</w>': 13, ' boy</w>': 77, '   re fu sed</w>': 39, '   bo ss</w>': 349, '   beau ty</w>': 153, ' c li ent</w>': 5, '   pri vi le ge</w>': 39, '   sta ture</w>': 9, '   mi d d le man</w>': 1, '   sho pp ing</w>': 74, '   en ga ging</w>': 6, '   ser vi ces</w>': 57, '   in ve sti ga tor</w>': 37, '   fee ls</w>': 206, '   re cor d</w>': 323, '   ex pre ssed</w>': 6, '   a dam ant</w>': 5, '   di sa pp ro v al</w>': 6, '   ex e cu t ors</w>': 2, '   e sta te</w>': 61, '   con cer n s</w>': 38, '   att end</w>': 39, '   in si sted</w>': 25, '   r ace</w>': 86, '   dri ver</w>': 129, '   gen er ous</w>': 45, '   pu r cha se</w>': 18, '   go o ds</w>': 29, ' d ev il</w>': 4, '   fri gh ten ed</w>': 112, '   ex ci ted</w>': 139, '   pre ssed</w>': 20, ' cho ke</w>': 1, '   j uni or</w>': 145, '   p</w>': 127, '   we ars</w>': 51, ' m ac h ine</w>': 3, '   v el v et</w>': 14, '   sto ck</w>': 109, '   pla y ers</w>': 43, '   wh a</w>': 45, '   f le a</w>': 23, '   pre si den ti al</w>': 14, '   lu x u ri ous</w>': 2, '   di g</w>': 146, '   re ce i p ts</w>': 17, '   e ye</w>': 329, '   pl us</w>': 180, '   ex pen ses</w>': 44, '   sti ff</w>': 46, '   po ps</w>': 47, '   f l ying</w>': 125, '   cu ts</w>': 27, '   p ho to gra ph s</w>': 42, '   ne w s re el</w>': 3, '   su b li min al</w>': 1, '   i ma g es</w>': 19, '   i ll e ga l</w>': 78, '   bor der l ine</w>': 6, '   tr ans ve sti te</w>': 9, '   ru b b er</w>': 50, '   im mer sion</w>': 1, '   en e ma</w>': 2, '   ma ga z in es</w>': 35, '   com mi ssi ons</w>': 3, '   spe ci al ty</w>': 22, '   fe ti sh</w>': 11, '   vi de o s</w>': 18, '   go th i c</w>': 4, '   har d c ore</w>': 13, '   s qu ea mi sh</w>': 7, '   we ir do</w>': 13, '   joh n</w>': 881, '   lu c</w>': 18, '   god ard</w>': 2, '   f li cks</w>': 4, '   pen n sy l v ani a</w>': 16, '   par a de</w>': 32, '   lo s ers</w>': 32, '   nu mb </w>': 12, '   har v ard</w>': 55, '   gen i us</w>': 90, '   bea ts</w>': 60, '   pu mp ing</w>': 26, '   ha m bur g ers</w>': 8, '   ha y se ed</w>': 3, '   ba se ment</w>': 75, '   sa les</w>': 65, '   ri s k y</w>': 21, '   inter n et</w>': 26, '   pa ti ence</w>': 62, ' ha w ks</w>': 1, '   s wa pp ing</w>': 4, '   for th</w>': 58, '   mo de ms</w>': 4, '   z a p</w>': 6, '   coun ter</w>': 54, '   lo cal</w>': 132, '   under gr ound</w>': 67, '   sp rea ds</w>': 8, '   z om bi es</w>': 11, '   j un ki es</w>': 12, '   pr un ed</w>': 1, '   b list er</w>': 7, '   cla ssi fi ed</w>': 29, '   a ds</w>': 12, '   hi d d en</w>': 88, '   co d es</w>': 30, '   cou ri ers</w>': 4, '   or d ers</w>': 215, '   cor por a tions</w>': 3, '   inter sta te</w>': 7, '   wi re</w>': 113, '   tr ans f ers</w>': 2, '   bo x es</w>': 42, '   se ll er</w>': 9, '   sta ys</w>': 52, '   bu y er</w>': 14, '   ver s a</w>': 7, '   tr ace</w>': 73, '   y an king</w>': 2, '   wa ter sp or ts</w>': 1, '   sp an king</w>': 7, '   fi st ing</w>': 1, '   ma les</w>': 12, '   he ma ph ro di tes</w>': 1, '   stra d dle</w>': 2, '   ti ed</w>': 85, '   b all</w>': 242, '   ga g</w>': 30, '   con sen ting</w>': 2, '   st ep</w>': 273, '   ki d die</w>': 6, '   por no gra ph y</w>': 14, '   den ti st</w>': 39, '   pen thou se</w>': 24, '   pla y boy</w>': 13, '   hu st l er</w>': 20, '   e t c</w>': 18, '   con si d ers</w>': 8, '   ma in st rea m</w>': 6, '   x</w>': 92, '   pen e tra tion</w>': 14, '   in du st ry</w>': 33, '   v a ll ey</w>': 81, '   wri t ers</w>': 25, '   di re c t ors</w>': 22, '   st ars</w>': 135, '   ce le bri ti es</w>': 8, '   pu mp</w>': 44, '   1 5 0</w>': 15, '   por no</w>': 23, '   ac a de my</w>': 44, '   a war ds</w>': 10, '   lo ves</w>': 221, '   bu ying</w>': 108, '   per ver se</w>': 7, '   ev o lu tion</w>': 24, '   de sen si ti z ation</w>': 1, '   el v is</w>': 51, '   pre sle y</w>': 4, '   wi g g ling</w>': 2, '   hi ps</w>': 15, '   of f en si ve</w>': 25, '   n ow a days</w>': 25, '   m t v </w>': 11, '   sh ow ing</w>': 104, '   dan c ing</w>': 111, '   th ong</w>': 3, '   bi k in is</w>': 1, '   as ses</w>': 58, '   han ging</w>': 168, ' ad di ct</w>': 1, '   brea st</w>': 24, '   im plan ts</w>': 5, '   li ter ally</w>': 41, '   me di cal</w>': 167, '   j er king</w>': 8, '   la ying</w>': 41, '   w oun ds</w>': 36, '   e du ca tion</w>': 48, '   be g ins</w>': 66, ' po ps</w>': 1, '   f an ta sy</w>': 53, '   du mp</w>': 125, '   p a</w>': 44, '   sha ft</w>': 21, '   dea ling</w>': 113, '   con ne ction</w>': 98, '   th o ma s</w>': 140, ' op er a ted</w>': 2, '   v a g in a</w>': 7, '   d ru mm ed</w>': 4, '   por no gra p her</w>': 1, '   uni on</w>': 77, '   per ver ts</w>': 3, '   pa st e</w>': 13, '   tru man</w>': 57, '   ca po te</w>': 3, '   ca tch y</w>': 6, '   wor th</w>': 481, '   hi gh li gh ting</w>': 1, '   every day</w>': 60, '   si tu a tions</w>': 22, '   te mp ting</w>': 11, '   su g ge sti ve</w>': 2, '   op er a ted</w>': 6, ' v a g in a</w>': 1, '   pen ci l</w>': 32, '   man ha tt an</w>': 43, '   u p da te</w>': 8, '   a il ment</w>': 3, '   po st</w>': 155, '   ar ran ged</w>': 71, '   sta tu te</w>': 7, '   li mi ta tions</w>': 8, '   o ver night</w>': 32, '   jo king</w>': 64, '   to ta lled</w>': 1, '   ac coun ts</w>': 43, '   e qu al</w>': 33, '   thir te en</w>': 71, '   cen ts</w>': 91, '   a m oun ts</w>': 11, '   dea lt</w>': 36, '   comp en sa tion</w>': 14, '   app ro pri ate</w>': 45, '   pro vi de</w>': 62, '   re mo tely</w>': 17, '   di v or ce</w>': 125, '   ca ses</w>': 80, '   cor por ate</w>': 44, '   in ve sti ga tions</w>': 12, '   co ll e ct</w>': 72, '   an on y m ou s ly</w>': 3, '   ser ved</w>': 69, '   ru in</w>': 111, '   p uni sh ed</w>': 28, '   con fe ssed</w>': 15, '   cou ra ge</w>': 79, '   jo ke</w>': 244, '   cu tting</w>': 78, '   sta g</w>': 14, '   si mu la ted</w>': 9, '   rea li sti c</w>': 21, '   fa ke</w>': 107, '   e f fe c ts</w>': 50, ' s nu ff</w>': 1, '   u r b an</w>': 13, '   my th</w>': 19, '   fo l k l ore</w>': 5, '   be lon ged</w>': 31, '   at ro ci ty</w>': 2, '   ra p es</w>': 7, '   cer ti fi ca tes</w>': 16, '   pre ven ted</w>': 10, '   con ten ts</w>': 7, '   cu ri ous</w>': 151, '   c li ent</w>': 125, '   ex pe c ts</w>': 17, '   tr ou b les</w>': 40, '   de e p ly</w>': 38, '   su c c ee ded</w>': 14, '   e m pi re</w>': 75, '   no t ori ous</w>': 4, '   e c cen tri c</w>': 12, '   cu l ti v a ted</w>': 3, '   le gen d ary</w>': 10, '   pi tt s bur gh</w>': 25, '   pa ss ing</w>': 87, '   di le mm a</w>': 16, '   ter ri ble</w>': 342, '   con do l en ces</w>': 8, '   pa ssed</w>': 147, '   p ra i sed</w>': 6, '   st ri ct</w>': 19, '   ad h er ence</w>': 2, '   con fi den ti a li ty</w>': 15, '   pri vi le ged</w>': 9, '   sp ok en</w>': 90, '   har ri s bur g</w>': 2, '   lan ca ster</w>': 3, '   h er sh ey</w>': 2, '   in f lu en ti al</w>': 2, '   ev en ing</w>': 317, '   te a</w>': 147, '   pro mp t</w>': 2, '   ra p ed</w>': 42, '   d ru g ged</w>': 13, '   mo te l</w>': 83, '   ca li for ni a</w>': 169, '   lo s</w>': 77, '   an ge les</w>': 64, '   ac t re ss</w>': 49, '   at ti c</w>': 26, '   si l ver w are</w>': 8, '   hea ls</w>': 3, '   r in gs</w>': 44, '   sin g le</w>': 234, '   pro fe ssion</w>': 34, '   be lon g in gs</w>': 8, '   na tu ra l</w>': 152, '   hi de</w>': 227, '   no te</w>': 193, '   a side</w>': 84, '   gu i lt</w>': 49, '   p ra ying</w>': 33, '   ch u r ch</w>': 255, '   fini sh</w>': 340, '   a po lo gi ze</w>': 168, '   bo b</w>': 347, '   f b i</w>': 192, '   ci r cu m stan ces</w>': 80, '   in di ca tions</w>': 4, '   se p te mber</w>': 24, '   1 9 9 3 </w>': 4, '   un com for ta ble</w>': 45, '   com mi tt ed</w>': 58, '   su i ci de</w>': 141, '   de pre ssed</w>': 50, '   in di ca tion</w>': 10, '   tr en ch</w>': 4, '   h at</w>': 137, '   bea ten</w>': 34, '   no ses</w>': 14, '   star ing</w>': 73, '   pi ss ing</w>': 28, '   ch ea ting</w>': 45, '   gla mor ous</w>': 7, ' li p</w>': 1, '   dan d ru ff</w>': 1, '   c ro ok ed</w>': 24, '   b la me</w>': 224, '   ex pe c ta tions</w>': 27, '   on going</w>': 8, '   he ight</w>': 23, '   p oun ds</w>': 142, '   bl on de</w>': 82, '   a pri l</w>': 47, '   1 9 7 6 </w>': 4, '   1 1 </w>': 34, '   1 9 9 2 </w>': 5, '   list ed</w>': 22, '   co le</w>': 108, '   ne il</w>': 48, '   vo l un te er</w>': 20, '   or g ani z a tions</w>': 8, '   inter con ne c ted</w>': 1, '   fun c tion ing</w>': 14, '   en for ce ment</w>': 31, '   r</w>': 90, '   re vi e w</w>': 46, ' che ck</w>': 15, '   re cor ds</w>': 220, '   vi r g in i a</w>': 57, '   ju sti ce</w>': 134, '   sp o ke</w>': 135, ' b</w>': 214, '   supp or t</w>': 125, '   or g ani z ation</w>': 53, '   un like</w>': 35, '   ex p lo i ted</w>': 7, '   wa sh in g ton</w>': 178, '   in de pen dent</w>': 38, '   con tr ac tor</w>': 14, '   re s our ce</w>': 5, '   inter na l</w>': 49, '   au d it</w>': 5, '   li cen sed</w>': 9, '   k in de st</w>': 5, '   s wee te st</w>': 13, '   ad or ed</w>': 4, '   ho p ed</w>': 48, '   p ra y</w>': 104, '   mo ved</w>': 244, '   su i t case</w>': 49, '   po s se ssed</w>': 36, '   sh ow ed</w>': 167, '   sen a tor</w>': 160, '   as si stan ce</w>': 35, '   in vo ice</w>': 4, '   en ve lo pe</w>': 40, '   im be ci le</w>': 20, ' la w</w>': 79, '   cle an ing</w>': 75, '   fran chi se</w>': 8, '   spe ci fi c s</w>': 7, '   un plea s ant</w>': 23, '   spi el</w>': 3, '   re qui red</w>': 21, '   in for m</w>': 37, '   i te ms</w>': 13, '   co de</w>': 195, '   ma st er c ard</w>': 2, '   ex pre ss</w>': 52, '   k ri st en</w>': 34, ' was</w>': 43, '   z app er</w>': 2, '   e di son</w>': 4, '   sh e il a</w>': 67, '   ma ster</w>': 224, '   sc rea m ing</w>': 93, '   k in ca id</w>': 17, '   a li ce</w>': 86, '   ri ck</w>': 129, '   mi r r or</w>': 81, '   de f ea ts</w>': 6, '   ni gh t mar es</w>': 51, '   da y d rea m</w>': 4, '   f able</w>': 2, ' gu ar di an</w>': 2, '   te ddy</w>': 60, '   ho st</w>': 36, '   any where</w>': 328, '   cra mm ing</w>': 2, '   ph y si c s</w>': 37, '   ma tch ing</w>': 9, '   lu g ga ge</w>': 30, '   o h h h h baby</w>': 1, '   fe et</w>': 371, '   stu ck</w>': 235, '   ta b les</w>': 31, '   clo ses</w>': 21, '   de b bi e</w>': 61, ' co ll e c ted</w>': 1, '   we ir de st</w>': 11, '   f re ddy</w>': 57, '   au th ori ti es</w>': 31, '   ni gh t st al k er</w>': 1, '   b en ch</w>': 32, '   pre ss es</w>': 8, '   dea th s</w>': 23, '   shi f ts</w>': 18, '   ac ci dent</w>': 377, '   thir ds</w>': 7, '   f our th s</w>': 1, ' a li ce</w>': 2, '   d an</w>': 57, '   as th ma</w>': 7, '   att ack</w>': 258, '   1 7 </w>': 44, ' year</w>': 62, '   fa tal</w>': 24, '   dis se c ting</w>': 1, '   ad mi r ing</w>': 8, '   sa l v ation</w>': 17, '   te en a ger</w>': 20, '   b la m ing</w>': 31, '   ki lls</w>': 84, '   ne i gh bor ho od</w>': 128, '   h un ted</w>': 16, '   ro a sted</w>': 6, '   le g end</w>': 40, '   f re ed</w>': 11, '   te ch ni ca li ty</w>': 6, ' t</w>': 526, ' thanks</w>': 22, '   ten n is</w>': 34, '   t or ch ed</w>': 4, '   kee p ing</w>': 173, '   ban qu et</w>': 12, '   di stra u ght</w>': 7, '   app e ti te</w>': 30, '   do om ed</w>': 15, '   po pp ing</w>': 11, '   a sp ir ing</w>': 3, '   po p cor n</w>': 31, '   a vo id</w>': 74, ' cont act</w>': 2, ' day</w>': 52, '   sta ir s</w>': 73, '   na st y</w>': 92, '   bu mp</w>': 30, '   fee lin gs</w>': 188, '   sta ke</w>': 70, ' con di tion</w>': 3, '   ab i li ty</w>': 64, '   g ran d chi ld</w>': 7, '   of f er ing</w>': 53, '   won der ed</w>': 87, '   in t end</w>': 86, '   clo sed</w>': 195, ' off</w>': 70, '   hi d es</w>': 13, '   d ow n sta ir s</w>': 109, ' wait</w>': 35, '   lon e ly</w>': 153, '   j ac o b</w>': 29, ' wa ke</w>': 5, '   re gu l ar</w>': 118, '   cer e mon y</w>': 39, '   co ac h</w>': 41, '   s ea ts</w>': 39, '   lan ds</w>': 39, '   par is</w>': 207, '   he ll u v a</w>': 59, '   h on</w>': 73, '   be came</w>': 145, '   sho pp er</w>': 2, '   so ber ing</w>': 1, ' k ru e ger</w>': 2, '   par k</w>': 224, '   m ee t in gs</w>': 39, '   as y lu m</w>': 29, '   mar k</w>': 237, '   s ou l</w>': 243, '   a man da</w>': 21, '   bo ther ed</w>': 67, '   y v on n e</w>': 8, '   fin ds</w>': 109, '   sp r in g w o od</w>': 6, '   k ru e ger</w>': 41, '   ca l m</w>': 253, ' no</w>': 326, '   gre ta</w>': 10, '   wan der ed</w>': 15, '   w ard</w>': 42, '   vi si ted</w>': 20, ' d an</w>': 1, '   re vo ke</w>': 3, '   di p lo ma</w>': 8, '   gre y</w>': 39, '   lo ck</w>': 207, '   t or ment</w>': 8, '   p lot</w>': 34, '   me mor i al</w>': 17, '   st one</w>': 162, '   v ac ant</w>': 15, '   gra ve</w>': 88, '   n un s</w>': 34, '   bu mp ing</w>': 2, '   ne w sp a p ers</w>': 60, '   f li pped</w>': 19, '   h un g</w>': 78, '   l ink</w>': 27, '   f red</w>': 96, '   who a</w>': 149, '   plan et</w>': 229, '   gra d</w>': 11, '   po lly</w>': 7, '   i di o t</w>': 213, '   fa ll en</w>': 55, '   ac es</w>': 11, '   he ar t brea k</w>': 5, '   p sor i as is</w>': 2, '   g ore</w>': 9, '   pa int</w>': 97, '   com i c s</w>': 10, '   an ci ent</w>': 67, '   me li cer tes</w>': 1, '   pi m pl es</w>': 2, '   he ar t bur n</w>': 4, '   ce ll u li te</w>': 3, '   mo de ling</w>': 4, '   mi l k sha kes</w>': 1, '   ch er ry</w>': 45, '   pi e</w>': 80, '   ban an a</w>': 25, '   sp li ts</w>': 5, '   und ying</w>': 3, '   bo tt om</w>': 138, '   su per na tu ra l</w>': 20, '   pu sh y</w>': 16, '   pu sh ing</w>': 56, '   bi tch ing</w>': 15, '   pre s sure</w>': 128, '   a gen cy</w>': 89, '   na ah</w>': 6, '   shi ft</w>': 70, '   k ey</w>': 275, '   s wi mm ing</w>': 73, '   pr ac ti ce</w>': 142, '   nu n</w>': 57, '   in vi ted</w>': 103, '   fa int</w>': 26, '   di ve</w>': 37, ' e g gh ead</w>': 1, ' u</w>': 7131, ' r ough</w>': 5, ' re ce p tion</w>': 2, '   bo a ts</w>': 54, '   ti ger</w>': 44, '   sh ar k</w>': 115, '   s ki pp er</w>': 63, '   k ent</w>': 52, '   i an</w>': 51, ' a y be</w>': 7, '   re mo te</w>': 34, '   s wi m</w>': 95, '   c ow ard</w>': 45, '   l un g fi sh</w>': 1, '   ne il s en</w>': 7, '   ho ll ow ay</w>': 2, '   bo ar ds</w>': 20, '   par a ding</w>': 5, '   au to</w>': 41, '   ma ti ca lly</w>': 1, '   en d ow</w>': 2, '   b one</w>': 70, ' f l ying</w>': 2, '   sa u c er</w>': 7, '   uni den ti fi ed</w>': 13, '   ob je c ts</w>': 12, ' w in</w>': 4, ' e gra de</w>': 1, '   pre ser ve</w>': 20, ' long</w>': 23, '   de st in y</w>': 77, '   sp ir it</w>': 121, ' those</w>': 15, ' nothing</w>': 53, '   wi ll in g ly</w>': 5, ' school</w>': 21, '   pa le o z o i c</w>': 1, '   pa sti me</w>': 3, '   th und er</w>': 13, ' li z ar ds</w>': 1, '   re f er r ing</w>': 26, ' that</w>': 397, '   st ri kes</w>': 36, ' wor st</w>': 3, '   c ow ar di ce</w>': 8, '   spi ri tually</w>': 4, '   o d d b all</w>': 2, '   ri s king</w>': 18, '   be st ow</w>': 4, '   au to ma ti c</w>': 30, '   mon o po ly</w>': 8, ' com man der</w>': 2, '   ph y si cal</w>': 102, ' ra de</w>': 1, '   a like</w>': 32, ' bra i ded</w>': 1, '   pu pp et</w>': 22, '   i p so</w>': 1, '   f ac to</w>': 1, '   no me</w>': 6, '   a la s k a</w>': 21, ' co p ter</w>': 1, ' wanted</w>': 9, '   pro ving</w>': 19, '   de p th</w>': 21, ' ex pl or er</w>': 1, '   mo der ate</w>': 5, '   su spe c t c d</w>': 1, ' o ctor</w>': 3, '   la d</w>': 84, '   p ow ell</w>': 30, '   car ne y</w>': 2, ' will</w>': 38, '   om it</w>': 1, '   2 7 4 </w>': 1, ' f</w>': 60, '   b ow</w>': 40, '   ra m</w>': 35, '   sa w t ee th</w>': 1, '   s ea l ed</w>': 45, ' alone</w>': 9, '   re e f</w>': 11, '   ex pl or er</w>': 14, '   cla mp</w>': 2, ' us</w>': 27, '   ga u ge</w>': 11, '   di st re ss</w>': 34, '   f re i gh ter</w>': 8, '   e ll e s m ere</w>': 1, '   gre en land</w>': 3, '   may day</w>': 3, '   c y clo ps</w>': 8, ' his</w>': 57, '   car l</w>': 168, ' mon ger ing</w>': 2, '   re sig ned</w>': 11, '   na v y</w>': 68, '   pro je c ts</w>': 23, ' l un g fi sh</w>': 1, '   dro ps</w>': 33, '   g ins</w>': 1, '   no i ses</w>': 22, '   p ac i fi st</w>': 2, '   e g gh ead</w>': 10, ' go o der</w>': 4, '   c r ack po t</w>': 8, ' b an</w>': 1, '   a to m</w>': 15, '   te sts</w>': 78, '   j un k</w>': 71, '   nu cle ar</w>': 89, '   su b s</w>': 12, '   mi l i</w>': 1, '   t ary</w>': 2, '   bu d ge t for</w>': 1, '   en g in e er ing</w>': 40, '   de sig n</w>': 64, '   de m on</w>': 32, '   d ev e lo p</w>': 41, '   en ro lled</w>': 6, '   b ac hel or</w>': 44, '   di sa pp o in ting</w>': 13, ' big</w>': 24, '   ing</w>': 23, '   j ani e</w>': 5, '   hel en</w>': 77, '   app ea l</w>': 31, '   ju lie</w>': 62, '   de ser ves</w>': 32, '   ex a g ger a ting</w>': 11, '   lo y al ty</w>': 56, '   ex i sts</w>': 53, '   ex e c</w>': 6, '   na vi ga tion</w>': 10, '   fir ing</w>': 52, '   li e u ten ant</w>': 226, '   mi l bur n</w>': 2, '   si de ar ms</w>': 2, '   under wa ter</w>': 21, '   in v al u able</w>': 5, '   f l are</w>': 16, '   pi sto ls</w>': 16, '   fro g men</w>': 3, ' e y ed</w>': 24, '   ma ss</w>': 68, '   je lly</w>': 15, '   t or pe do</w>': 29, '   spe ed</w>': 117, ' about</w>': 40, '   k no ts</w>': 19, ' du e</w>': 1, '   sho ve</w>': 33, '   g ri ff</w>': 2, '   in di ca tes</w>': 15, '   in er ti al</w>': 4, '   k no cked</w>': 86, '   cra sh</w>': 138, '   re pa ir s</w>': 18, '   comp le ted</w>': 23, '   ex ter i or</w>': 9, '   gu i dan ce</w>': 23, '   sy st e ms</w>': 83, '   i c b m</w>': 2, ' home</w>': 17, '   ri ses</w>': 22, '   po le</w>': 47, '   ra di ation</w>': 56, ' con stan tly</w>': 1, '   ri sing</w>': 31, ' down</w>': 41, '   our self</w>': 3, '   brea th able</w>': 2, ' c y clo ps</w>': 2, '   di re ction</w>': 77, '   in di ca te</w>': 20, '   de p le ted</w>': 5, ' char ging</w>': 1, '   ma g n et</w>': 6, '   har ne ss</w>': 4, '   s ca le</w>': 50, '   ma g ne ti c</w>': 19, ' su per</w>': 3, '   s our ce</w>': 92, ' ma g ne ti c</w>': 1, '   an ti ci pa te</w>': 12, '   the ori z ing</w>': 2, ' could</w>': 20, ' ab o ve</w>': 2, '   mu r man s k</w>': 1, '   fin land</w>': 6, ' in ten si ty</w>': 1, '   ar c s</w>': 1, '   su b mer ged</w>': 1, '   mi lli ons</w>': 92, '   vo l ts</w>': 6, ' dis char ged</w>': 1, '   di re c tions</w>': 38, ' wa ter</w>': 9, '   e le c tri cal</w>': 31, '   st or m</w>': 119, '   min us</w>': 21, '   cor re c ted</w>': 9, '   be ar ing</w>': 29, '   t or ch es</w>': 6, '   at mo sp here</w>': 41, '   ma x i mu m</w>': 20, '   en g in es</w>': 44, '   sc re w s</w>': 11, '   de cl in ation</w>': 3, '   re ver se</w>': 39, '   ei gh ty</w>': 114, '   fa th om s</w>': 2, '   sin king</w>': 18, ' ra m</w>': 2, '   de ter m ine</w>': 24, '   ex t ent</w>': 18, '   jo in ed</w>': 58, '   pa ss en g ers</w>': 20, '   c li f for d</w>': 9, '   com man der</w>': 150, '   ri ch ard</w>': 155, ' little</w>': 22, '   na vi ga te</w>': 5, '   g in</w>': 28, '   lea ked</w>': 10, '   ra mm ed</w>': 2, '   da ma ged</w>': 33, ' w oun ded</w>': 2, ' hea ls</w>': 1, '   se le c ted</w>': 11, '   se ver al</w>': 106, '   spe ci men s</w>': 13, '   ex a m ine</w>': 26, '   dis co ver</w>': 39, '   de si r able</w>': 6, '   f ea tur es</w>': 10, '   in cor por ate</w>': 4, ' ear th</w>': 7, ' co lon i z ers</w>': 1, '   un har med</w>': 10, '   re ma in</w>': 74, '   s well</w>': 109, '   h or r ors</w>': 5, '   v ar i ous</w>': 32, '   se le ct</w>': 10, '   su i ta ble</w>': 10, '   co lon i z ation</w>': 5, ' your</w>': 116, '   app e ar</w>': 79, ' face</w>': 29, '   lin ger</w>': 4, '   re char ge</w>': 1, '   ban ks</w>': 92, ' ans w ers</w>': 2, '   our se l ves</w>': 218, '   p lo tt ed</w>': 6, ' our</w>': 37, '   wh er ever</w>': 89, ' until</w>': 13, '   thr ows</w>': 26, '   th under bo lt</w>': 3, '   mu sing</w>': 1, '   ad ver s ary</w>': 8, '   h om er</w>': 61, ' c y clo p es</w>': 1, '   s ons</w>': 47, '   for ged</w>': 14, '   bo l ts</w>': 11, '   z e us</w>': 5, '   o c cu r red</w>': 55, '   o c cu r r ence</w>': 8, '   ci r c le</w>': 41, '   ye om an</w>': 1, '   figu r es</w>': 64, '   g or don</w>': 116, '   l ar i vi ere</w>': 10, '   se le c t man</w>': 11, '   tw om b le y</w>': 40, '   con cor d</w>': 5, '   me l</w>': 20, ' pre si dent</w>': 25, '   m oun tain</w>': 92, '   wa de</w>': 107, ' 3 6 4 </w>': 1, '   lea gu e</w>': 76, '   nor th coun try</w>': 1, '   d ev e lo p ment</w>': 13, '   as so ci ation</w>': 25, '   ma ter</w>': 1, '   of f er ed</w>': 94, '   ta x</w>': 71, '   bi ll</w>': 516, '   f ar m</w>': 125, '   la tely</w>': 161, '   si m pl er</w>': 13, '   he wi t t</w>': 9, '   ev an</w>': 52, '   al ma</w>': 18, '   di r ty</w>': 188, '   p ry</w>': 18, '   dam ned</w>': 139, '   to o th</w>': 60, '   bu g ging</w>': 20, '   su g ar</w>': 134, '   c b</w>': 9, '   pro per ty</w>': 137, '   ch u b</w>': 4, '   fi red</w>': 213, '   fi x in</w>': 16, '   c lu tch</w>': 9, '   fi x ed</w>': 112, '   le ga lly</w>': 26, '   fin an ci ally</w>': 9, '   de po si tions</w>': 4, '   p sy chi a tri c</w>': 28, '   ev al u a tions</w>': 2, '   vi si ta tion</w>': 6, '   re d ra w n</w>': 2, '   as su m ing</w>': 36, '   und u ly</w>': 2, '   re st ri c tive</w>': 1, ' 5 0 0</w>': 15, '   re ta in er</w>': 6, '   de c ree</w>': 2, '   j i ll</w>': 59, '   al co ho l</w>': 36, ' wife</w>': 33, '   se x u al</w>': 109, '   u p se tting</w>': 23, '   sp r ing</w>': 100, '   a gre ed</w>': 104, '   ma ss ac hu se tt s</w>': 15, '   com ment</w>': 32, '   any how</w>': 77, '   sp o tt ed</w>': 32, '   bu ck</w>': 87, '   c li ff</w>': 45, '   fuck er</w>': 75, '   dea der</w>': 2, ' n</w>': 300, '   f re sh</w>': 125, '   tr ac ks</w>': 49, '   de er</w>': 44, '   de ers</w>': 1, '   ru in ed</w>': 103, '   ar m</w>': 215, '   rea li z ed</w>': 99, '   cle men s</w>': 4, '   pi tch er</w>': 14, ' best</w>': 5, '   ba ll pla y er</w>': 10, '   ha mp shi re</w>': 28, '   car l ton</w>': 21, '   fi s k</w>': 6, '   bri tain</w>': 16, '   so x</w>': 23, '   d ra f ted</w>': 8, '   m ea d ow</w>': 12, '   c ri pp le</w>': 23, '   what the fuck</w>': 1, '   mm m</w>': 98, '   gu t</w>': 38, ' sh oo t</w>': 7, '   ta g</w>': 46, '   sh oo ting</w>': 150, '   ri f le</w>': 51, '   ye p</w>': 130, '   ho ling</w>': 1, '   b ru sh</w>': 30, '   pi les</w>': 5, '   3 5 </w>': 21, '   s ea son</w>': 73, '   gu ar an te ed</w>': 19, '   su ck ers</w>': 23, '   s now</w>': 93, '   ba star ds</w>': 83, '   ad v an ta ge</w>': 69, '   sur f</w>': 20, '   we lls</w>': 28, '   win ter</w>': 89, '   la w for d</w>': 10, ' ga u ge</w>': 4, '   br ow ning</w>': 6, '   br and</w>': 69, '   f an cy</w>': 89, '   to o ling</w>': 2, '   de ser ve</w>': 144, '   s li pped</w>': 61, '   fi sh</w>': 241, '   sle eve</w>': 15, '   a m bu lan ce</w>': 38, '   lu g ged</w>': 3, '   st ee p</w>': 6, '   mi le</w>': 83, '   lu mber</w>': 6, '   wh er ea b ou ts</w>': 17, '   che st</w>': 103, '   ar gu ment</w>': 65, '   son of ab i tch</w>': 83, ' bl own</w>': 5, ' 1 0 0</w>': 14, '   mu ck y</w>': 2, ' mu ck</w>': 2, '   er</w>': 149, '   har da ss</w>': 3, '   car e ful</w>': 335, '   wa ck y</w>': 1, '   ta b ack y</w>': 1, '   to ke</w>': 4, '   4 5 0</w>': 6, '   5 0 0</w>': 30, '   bra g</w>': 8, '   bra g ging</w>': 7, '   cer e mon i es</w>': 6, '   poli ce man</w>': 60, '   s n ea k</w>': 68, '   m ac </w>': 105, '   wi ck ha m</w>': 3, '   ha m bur ger</w>': 35, '   char ge</w>': 256, '   pi z z a</w>': 62, '   tur no ver</w>': 7, '   g ran d p a</w>': 40, '   po p</w>': 206, '   be lon gs</w>': 93, '   sho p</w>': 159, '   dan dy</w>': 8, ' yes</w>': 152, '   plan ned</w>': 99, '   sa ke</w>': 321, '   t ea ch ers</w>': 26, '   god dam ned</w>': 73, '   cla m</w>': 10, '   com pla in ts</w>': 23, '   st ea ling</w>': 78, '   pu mp k ins</w>': 2, '   so a p ing</w>': 2, '   win d ows</w>': 70, '   por ch</w>': 33, '   li gh ts</w>': 143, '   ho y t</w>': 2, '   sha ving</w>': 15, '   c rea m</w>': 113, '   ma il bo x</w>': 17, '   h er b</w>': 19, '   c ran e</w>': 48, '   bu sh es</w>': 21, '   h</w>': 95, '   ho y ts</w>': 1, '   tri ck</w>': 162, ' or</w>': 123, ' t rea ting</w>': 4, '   co stu me</w>': 36, '   s la ves</w>': 22, '   ke ys</w>': 169, '   sc re w ing</w>': 38, '   a ll er gi c</w>': 23, '   wor r ying</w>': 68, '   fa v ors</w>': 32, '   su m mon s</w>': 5, '   app re ci ation</w>': 20, '   mer i t t</w>': 1, '   ro l fe</w>': 6, '   bu g ged</w>': 14, '   shi t lo a d</w>': 10, '   pl ow ing</w>': 4, '   to p</w>': 416, '   a vo i ded</w>': 17, '   s ling</w>': 18, '   li tt le ton</w>': 1, '   h un ter</w>': 36, '   gra der</w>': 6, '   h un ting</w>': 89, '   f re e z ing</w>': 49, '   2 9 </w>': 11, '   to by</w>': 67, '   li lli an</w>': 20, '   be long</w>': 138, '   gu ard</w>': 164, '   c ro ss ing</w>': 32, '   b m w</w>': 11, '   j im my</w>': 322, '   pri ze</w>': 54, '   fun ni est</w>': 11, '   sc ar i est</w>': 3, '   i ma g in a tive</w>': 6, '   su n k</w>': 20, '   bo o ts</w>': 62, '   gr ound</w>': 213, '   su ff er</w>': 60, '   tur ned</w>': 377, '   t ou ch y</w>': 22, '   tra gi c</w>': 28, ' r oun ds</w>': 1, '   r ound</w>': 183, '   gr own</w>': 103, '   be ers</w>': 25, '   he t ti e</w>': 2, '   ro d g ers</w>': 7, '   what z i z name</w>': 1, '   sp at</w>': 5, '   bo st on</w>': 72, '   hea ded</w>': 68, '   la ke</w>': 142, '   aga way</w>': 1, '   ru m ma ge</w>': 1, '   cle an ers</w>': 13, '   mar gi e</w>': 24, '   te mp or ary</w>': 38, '   mer ri t t</w>': 3, '   wa y la id</w>': 1, '   fe ll a</w>': 155, '   some time</w>': 134, '   wan ting</w>': 78, '   ma fi a</w>': 34, '   hi re</w>': 107, '   in ve sti ga ting</w>': 31, '   lin ks</w>': 12, ' j ack</w>': 14, '   sen si tive</w>': 72, '   re ar ran ged</w>': 4, '   ow n s</w>': 71, '   nee ding</w>': 31, '   fro ze</w>': 12, '   per ked</w>': 1, '   pi p es</w>': 25, '   e le c tri c</w>': 34, '   be d room</w>': 112, '   fu r n ace</w>': 8, '   hea ting</w>': 10, '   sto ve</w>': 17, '   ti ck et</w>': 160, '   de ci ding</w>': 10, '   tra f fi c</w>': 79, '   d rea m ing</w>': 73, '   f la sh ing</w>': 8, '   sp ee ding</w>': 10, '   whi te house</w>': 4, '   i s su ing</w>': 6, ' next</w>': 9, '   g ri lled</w>': 4, '   ch ee se</w>': 70, '   du b</w>': 6, '   j i lli e</w>': 1, '   t ro op ers</w>': 11, '   k er</w>': 2, ' ban g</w>': 4, '   ac ci den tal</w>': 13, '   mar g</w>': 2, '   car t</w>': 23, '   co o king</w>': 40, '   mo der n</w>': 50, '   pro sp er ing</w>': 2, '   im pro ve</w>': 21, '   spe lled</w>': 8, '   p an ts</w>': 140, '   cra w l</w>': 45, '   ban g</w>': 49, ' na h</w>': 4, ' na w</w>': 4, '   fo g g</w>': 2, '   dre ssed</w>': 130, '   p up</w>': 13, '   a tta</w>': 6, ' go</w>': 80, '   le s son</w>': 103, '   re war ds</w>': 11, '   sa lly</w>': 105, '   di st r ac ting</w>': 12, '   lo ca ls</w>': 18, '   son so f bi tch es</w>': 2, '   g or d ons</w>': 1, '   ac ci den ta lly</w>': 17, '   cle ar ly</w>': 102, '   a du lt</w>': 37, '   af f li c ted</w>': 3, '   el b our n e</w>': 2, '   ha y</w>': 21, '   lo ft</w>': 9, '   vi e t na m</w>': 48, '   cho pp ing</w>': 12, '   fi re w o od</w>': 6, '   stu dent</w>': 83, '   god dam n it</w>': 79, '   fin ed</w>': 6, '   li f ted</w>': 33, '   fe ds</w>': 40, '   de e per</w>': 43, '   un a w are</w>': 10, '   lo ans</w>': 20, '   no sing</w>': 3, '   te sti f y</w>': 56, '   mo b</w>': 76, '   spe ci a list s</w>': 5, '   re sp on se</w>': 53, '   br in ging</w>': 135, '   po in t less</w>': 16, '   gr ow l ed</w>': 1, '   whi pped</w>': 12, '   bi te</w>': 129, '   ex ha u sted</w>': 31, '   ha ll ow e en</w>': 46, '   com mi tt e e</w>': 87, '   or g ani z ed</w>': 31, '   en g land</w>': 134, '   con st ru ction</w>': 50, '   ha bi ts</w>': 29, '   j i m</w>': 301, '   god sa ke</w>': 3, '   ear li er</w>': 111, '   ne w s wee k</w>': 8, '   t ea sing</w>': 17, '   lo t s a</w>': 14, '   ver sus</w>': 13, '   t ea ses</w>': 1, '   pla gu e</w>': 37, '   brea ks</w>': 55, '   ban gla de sh</w>': 2, '   me mo ir s</w>': 6, '   d ev o te</w>': 7, '   cha p ter</w>': 48, '   co co a</w>': 11, '   di stu r bed</w>': 44, '   bo l sho i</w>': 1, '   li fe time</w>': 59, '   fi sh b ow l</w>': 1, '   j et</w>': 59, ' la g ged</w>': 2, '   du h</w>': 18, '   re si li ent</w>': 4, '   s qu an der</w>': 2, '   per son a li ty</w>': 64, ' qu o t</w>': 85, ' as sho le</w>': 6, ' pro pa g an da</w>': 1, '   tur ki en i st an</w>': 2, '   mu s li ms</w>': 8, '   s la u gh ter ed</w>': 22, '   stra v an vi tch</w>': 1, '   cle an sin gs</w>': 1, '   h q w</w>': 1, '   fun er al s</w>': 11, '   stra v an a vi tch</w>': 4, '   g un fi re</w>': 7, '   s l ow ing</w>': 15, '   ny</w>': 5, '   s wee ti e</w>': 78, '   hu g</w>': 30, '   oo oo oh</w>': 5, '   re sp on d</w>': 49, '   in ti mi da ted</w>': 6, '   pi lot</w>': 94, '   mar k in gs</w>': 2, '   k in d ly</w>': 27, '   a ir lin er</w>': 3, '   a ir sp ace</w>': 4, '   pre li min ary</w>': 12, '   a ir st ri ke</w>': 4, '   t ro o p</w>': 5, '   mo ve ment</w>': 60, '   al ter</w>': 13, '   7 4 7 </w>': 4, '   ha i ls</w>': 3, '   e sc or ting</w>': 2, '   pi lo ts</w>': 28, '   con fir m</w>': 27, '   sur r oun ding</w>': 7, '   sta ti c</w>': 16, '   o ver wh el ms</w>': 2, '   a er a ted</w>': 1, '   ci r cu it</w>': 32, '   a vi on i c s</w>': 2, '   to g g le</w>': 2, '   ma in ten an ce</w>': 24, '   p an el</w>': 17, '   t ower</w>': 86, '   s lu g gi sh</w>': 5, '   ja mm ed</w>': 34, '   t or n</w>': 39, '   el ev a ter</w>': 1, '   ru d der</w>': 5, '   re sp on ding</w>': 14, '   w ing</w>': 63, '   mi gs</w>': 4, '   co lon el</w>': 191, '   ar c</w>': 9, '   f at</w>': 233, '   bo e ing</w>': 3, '   ma tch</w>': 117, '   pre f er ed</w>': 1, '   ter r ori sts</w>': 35, '   ch u tes</w>': 3, '   ans w er ing</w>': 36, '   wal ter</w>': 336, '   s li ps</w>': 9, '   fu ti li ty</w>': 2, '   ni mi t z</w>': 1, '   i ra q i</w>': 15, '   a m ba s sa d or</w>': 62, '   cla im ing</w>': 15, '   sh e et</w>': 42, '   u z is</w>': 2, '   si x th</w>': 34, '   gra d ers</w>': 3, '   re pre sen ta tive</w>': 17, '   ta y l or</w>': 117, '   com pro mi se</w>': 31, '   w ro te</w>': 266, ' li b</w>': 3, ' na i ls</w>': 1, '   sp ee ch</w>': 139, '   di p lo ma ti c</w>': 12, '   th re w</w>': 135, '   fi gh t ers</w>': 20, '   en ga ge</w>': 27, '   sp ying</w>': 26, '   cha ll en ged</w>': 11, '   re s cu e</w>': 72, '   po k er</w>': 53, '   che cking</w>': 89, '   car ds</w>': 134, '   di sc ard</w>': 1, '   dea d l ine</w>': 15, '   ba ses</w>': 14, '   tur k ey</w>': 65, '   ki tty</w>': 29, '   ha w k</w>': 44, '   re ta li at ory</w>': 1, '   p ra g ma ti c</w>': 3, '   pa w n s</w>': 3, '   k in gs</w>': 37, '   sen i or</w>': 55, '   st af f</w>': 104, '   ex e cu ting</w>': 4, '   ho sta g es</w>': 35, '   ad d</w>': 90, '   d ean</w>': 59, '   gre e ly</w>': 5, '   gi b b s</w>': 5, '   hel p ed</w>': 156, '   g a</w>': 5, '   af fir ma tive</w>': 27, '   re du ce</w>': 15, '   2 5 0</w>': 13, '   ac kn ow le d ged</w>': 10, '   t n t</w>': 5, '   par ac hu te</w>': 10, '   la un ch</w>': 54, '   ra mp</w>': 9, '   a f</w>': 2, ' 1 3 5 </w>': 1, ' r a</w>': 2, '   in st ru c ted</w>': 12, '   re fu el</w>': 5, '   en te be</w>': 1, '   f li es</w>': 57, '   mu sh room</w>': 8, '   cl ou d</w>': 56, '   ev ad es</w>': 1, '   mi ssi les</w>': 35, '   re fu el s</w>': 1, '   mi d</w>': 37, ' a ir</w>': 6, '   ge or gi a</w>': 19, '   sy ri a</w>': 5, '   i ra q </w>': 15, '   du mp ing</w>': 13, '   ser ge</w>': 2, '   b ac ks</w>': 34, '   com part ment</w>': 11, '   z e de ck</w>': 1, '   mar t y r</w>': 11, '   p sy cho lo g y</w>': 32, '   un n er ve</w>': 2, '   re mar k able</w>': 36, '   a ir cra ft</w>': 13, '   man i fe st</w>': 10, '   ac coun ted</w>': 12, '   in spe c ted</w>': 3, '   ro me o</w>': 79, '   t an go</w>': 9, '   z u l u</w>': 15, '   chan ging</w>': 76, '   sig n s</w>': 81, '   on bo ard</w>': 10, '   gra p hi c s</w>': 2, '   s ea l</w>': 53, '   bo ar ded</w>': 7, '   mi tch ell</w>': 112, '   po in ting</w>': 36, '   re la tions</w>': 31, '                                         </w>': 6, '   k it</w>': 55, '   a ir bor n e</w>': 12, '   s che du le</w>': 74, '   inter vi e w s</w>': 26, '   ro om s</w>': 97, '   fun c tions</w>': 18, '   d ou b les</w>': 8, '   du bi ous</w>': 10, '   di st in ction</w>': 9, '   shi el ded</w>': 4, '   u pp er</w>': 38, '   co ck p it</w>': 9, '   m c c</w>': 1, '   lin ked</w>': 21, '   ne t work</w>': 55, '   mi li t ary</w>': 112, '   sa te lli tes</w>': 18, '   sta tions</w>': 31, '   de li gh ted</w>': 32, '   ac com mo da te</w>': 8, '   cle ar ed</w>': 45, '   v al u able</w>': 63, '   na tion</w>': 63, '   mi sta ke</w>': 341, '   be g in</w>': 176, '   plan es</w>': 58, '   b om bed</w>': 12, '   o il</w>': 149, '   fi el ds</w>': 31, '   s k in ned</w>': 5, '   f ac es</w>': 63, '   wi ves</w>': 52, '   dis like</w>': 10, '   sen ds</w>': 40, '   so l di ers</w>': 89, ' way</w>': 37, '   st ea l</w>': 187, '   cou ra ge ous</w>': 8, '   di es</w>': 102, '   co op er ation</w>': 24, '   e s ca pe</w>': 140, '   bar ri ca de</w>': 6, '   com ra de</w>': 57, '   ex i ting</w>': 3, '   f oo li sh</w>': 70, '   ta st e</w>': 173, '   de fe at</w>': 25, '   bi tter</w>': 40, '   mor a li ty</w>': 14, '   poli ti c s</w>': 74, '   w ea k ne ss</w>': 45, '                                       </w>': 4, '   some day</w>': 104, '   pe t ro v </w>': 4, '   mo sc ow</w>': 52, '   ar ran ge ment</w>': 35, '   ma da me</w>': 132, '   chan d l er</w>': 10, '   wa ves</w>': 42, '   le e</w>': 175, '   rea ds</w>': 36, '   con gre ss</w>': 70, '   ca b in et</w>': 33, '   hu d dle</w>': 5, '   as su mes</w>': 3, '   al ter na ti ves</w>': 7, '   so lu tion</w>': 61, '   re fu el ing</w>': 3, '   gra vi ty</w>': 38, '   ca i d well</w>': 1, '   je ts</w>': 23, '   nor ther n</w>': 26, '   bor der</w>': 74, '   ha i ry</w>': 43, '   tom ca ts</w>': 3, '   i tch ing</w>': 4, '   k h</w>': 1, ' ll</w>': 78, '   0 1 0 0</w>': 1, '   mo bi li z ation</w>': 4, '   me chan i z ed</w>': 1, '   bri ga d es</w>': 2, '   dri ven</w>': 41, '   sh ep</w>': 20, '   ter m er</w>': 1, '   e f fi g y</w>': 2, '   su b je ct</w>': 153, '   con gre ssi on al</w>': 10, '   app ro v al</w>': 21, '   vo t ers</w>': 10, '   ro se</w>': 327, '   god sa kes</w>': 5, '   du tch</w>': 20, '   pl u g</w>': 43, '   lea ks</w>': 15, '   sp in</w>': 28, '   sen ding</w>': 106, '   di sa bl ed</w>': 3, '   com m uni ca tions</w>': 49, '   fa x</w>': 22, '   m ac h in es</w>': 72, '   f re e ing</w>': 5, '   ten s</w>': 10, '   ro y al ty</w>': 9, '   e le c ted</w>': 26, '   in fini tely</w>': 4, '   ho l ds</w>': 66, '   ne go ti ta te</w>': 1, '   under sto od</w>': 122, '   be com es</w>': 67, '   t ar get</w>': 88, '   ne go ti a ting</w>': 16, '   under way</w>': 12, '   lo y a list s</w>': 1, '   tr ac ed</w>': 22, '   s wi tch bo ard</w>': 5, '   ho ok ed</w>': 37, '   ho ve</w>': 1, '   t ee m</w>': 1, '   f le w</w>': 49, '   mi g</w>': 14, '   sh e ph er d</w>': 7, '   pu mp kin</w>': 22, '   de f en se</w>': 101, '   u l c ers</w>': 7, '   st e w ard</w>': 9, '   du ke</w>': 66, '   sp ea k er</w>': 6, '   att ac ked</w>': 82, '   af for d</w>': 154, '   ba it</w>': 38, '   du cks</w>': 12, '   bab i es</w>': 89, '   a men d ment</w>': 20, '   t y p es</w>': 34, '   chi ps</w>': 32, '   n r a</w>': 1, '   wor ding</w>': 3, '   han dy</w>': 32, '   c b s</w>': 23, '   com mi ssi on er</w>': 36, '   a qu ar i us</w>': 1, '   k ru ger</w>': 4, '   sa gi tt ar i us</w>': 3, '   m c c ro s k y</w>': 1, '   s ou ther n</w>': 29, '   plan ta tion</w>': 8, '   ow ner</w>': 50, '   shu ttle</w>': 43, '   cl ar ence</w>': 103, '   o ve u r</w>': 7, '   l un ar</w>': 8, '   j ac ob s</w>': 1, '   d ow n town</w>': 98, '   sen se le ss ly</w>': 2, '   si mp le</w>': 376, '   poli ti cal</w>': 81, '   rea li ti es</w>': 5, ' me</w>': 88, '   mer cu ry</w>': 15, ' la un ch</w>': 1, '   sc rea m</w>': 66, '   d un n</w>': 8, '   a st er o id</w>': 13, '   o ver lo a d</w>': 9, '   di sp o sa l</w>': 24, '   un ger</w>': 2, '   ma l fun ction</w>': 18, '   ac c el er at ors</w>': 2, '   k ra m er</w>': 51, '   wor p</w>': 1, '   l ever</w>': 11, '   com pu te</w>': 5, '   de gre es</w>': 61, '   ro a st</w>': 18, '   sp it</w>': 54, '   pre ma ture</w>': 13, '   e je ction</w>': 2, '   b om b</w>': 217, '   si m on</w>': 50, '   e je ct</w>': 14, '   e je c ted</w>': 4, '   k u r t z</w>': 34, '   h un k</w>': 15, '   t in</w>': 36, '   to i let</w>': 81, '   dan ger ou s ly</w>': 16, '   wi r ing</w>': 15, '   i ce ber g</w>': 3, '   b le w</w>': 131, '   g ri p</w>': 41, '   ab or ted</w>': 3, '   f s a</w>': 1, '   in spe ction</w>': 20, '   che w ing</w>': 19, '   go o d b yes</w>': 4, '   s ani ty</w>': 11, '   ad mi tting</w>': 16, '   e le c tr o</w>': 11, ' sho ck</w>': 5, '   sto ck man</w>': 1, '   sa m my</w>': 45, '   da v is</w>': 36, '   sp a gh e tt i</w>': 33, '   st able</w>': 32, '   pro vi der</w>': 1, '   sta ge</w>': 87, '   fa u l ts</w>': 4, '   spe ci fi ca tions</w>': 8, '   i ta li an</w>': 101, '   o ver wor ked</w>': 7, '   in vi ta tions</w>': 18, '   te</w>': 34, '   pi le</w>': 45, '   mu d</w>': 34, '   comp le x</w>': 50, '   or g an</w>': 19, '   a g gre ssi ons</w>': 1, '   gar d en</w>': 123, '   b al an c ed</w>': 10, '   m ea ls</w>': 22, '   ger i to l</w>': 1, '   ru m ack</w>': 2, '   b la d der</w>': 7, '   ro k</w>': 6, '   ga ssed</w>': 3, '   secon d ary</w>': 17, '   rea d out</w>': 9, '   stra i gh ten ed</w>': 26, '   d ru g st ore</w>': 14, '   i ll</w>': 107, '   pro mo te</w>': 15, '   di vi sion</w>': 41, '   ju pi ter</w>': 10, '   pro be</w>': 27, '   we d ding</w>': 214, '   comp li ca ted</w>': 95, '   en ga ged</w>': 79, '   th om p son</w>': 45, '   st e war de ss</w>': 22, '   den ver</w>': 29, ' chi ca go</w>': 1, '   mon i tor</w>': 36, '   re gu la t ory</w>': 5, '   fi e ld</w>': 225, '   inter f er ence</w>': 19, '   s can</w>': 37, '   sp o ts</w>': 22, ' an al y s is</w>': 2, '   har d w are</w>': 29, '   so f tw are</w>': 14, '   re gar ding</w>': 29, '   be ha vi or al</w>': 10, '   chan g es</w>': 107, '   i g nor ing</w>': 15, '   re late</w>': 6, '   com pre h en si ve</w>': 3, '   out bur st</w>': 6, '   d ou b ting</w>': 5, '   ne ga tive</w>': 101, '   inter mi t ant</w>': 1, '   mo de</w>': 10, ' r</w>': 108, '   an al y ze</w>': 10, '   th i ev ing</w>': 4, '   ra pi st</w>': 6, '   ter ri fi c</w>': 146, '   sc ra ps</w>': 12, '   sle e ps</w>': 27, '   d es</w>': 17, '   mo in es</w>': 6, '   cl in i c</w>': 41, '   po ten cy</w>': 3, '   op er a tions</w>': 44, '   pa w ned</w>': 2, '   jo e</w>': 392, '   c in na m on</w>': 11, '   po ac h ed</w>': 9, '   e g gs</w>': 115, '   ju ice</w>': 71, '   sa lu c c i</w>': 2, '   di sc re e tly</w>': 4, '   si x te en</w>': 102, '   pa ss en ger</w>': 32, '   na vi ga tor</w>': 11, ' everything</w>': 25, '   di ge sti ve</w>': 4, '   a mp hi bi ans</w>': 2, '   pre su me</w>': 37, ' y</w>': 65, ' ging</w>': 1, '   in sti tu te</w>': 33, '   app re ci a ted</w>': 16, '   bar r in g ton</w>': 10, '   r en ow ned</w>': 4, '   a gr on om i st</w>': 2, '   th u r s days</w>': 11, '   th u r s day</w>': 100, '   mar ch</w>': 56, '   dam m it</w>': 74, '   st ri k er</w>': 10, '   su b tly</w>': 1, '   sp rea ding</w>': 19, '   brea k down</w>': 27, '   m ee ts</w>': 34, '   sh or ting</w>': 3, '   te m per a tur es</w>': 5, '   bu gs</w>': 50, '   la u gh ed</w>': 38, '   l</w>': 198, ' g</w>': 48, ' h</w>': 69, '   re ma ins</w>': 54, '   un chan ged</w>': 1, '   re la tion</w>': 15, '   op en ed</w>': 126, '   v ac u u m</w>': 22, '   cle an er</w>': 23, '   nu r se</w>': 172, '   comp li ment</w>': 60, ' well</w>': 152, '   mi l ton</w>': 15, ' mi lt</w>': 1, '   e tt en h en i m</w>': 1, ' bu b b les</w>': 1, '   e le an or</w>': 16, '   sc hi ff</w>': 1, '   dis cu ss ing</w>': 30, '   e con om y</w>': 18, ' fu ll</w>': 8, '   beli ev ing</w>': 28, '   ex pe c ted</w>': 119, '   pa ir</w>': 113, ' when</w>': 94, '   e the l</w>': 16, '   so ci ally</w>': 9, '   lo b b y i st</w>': 1, '   busin e ss men</w>': 11, '   as so ca tion</w>': 1, '   att en ding</w>': 17, '   se min ar</w>': 12, '   vi su al</w>': 25, '   a i ds</w>': 46, '   v ac a tion ing</w>': 2, '   cor fe e</w>': 1, ' a while</w>': 1, ' get</w>': 77, '   ex ci ting</w>': 88, ' sit</w>': 11, '   pa u l</w>': 349, '   re x</w>': 49, '   car ey</w>': 1, '   a ir l ine</w>': 37, ' do c</w>': 3, ' some time</w>': 4, '   bea ting</w>': 65, '   z i p p</w>': 4, '   shi r le y</w>': 19, '   de ce i ve</w>': 14, '   chi ca go</w>': 108, '   p ani c</w>': 84, ' sa fe</w>': 11, '   pu r su e</w>': 23, '   fu l fi ll ment</w>': 2, '   f la ps</w>': 2, '   thr ust</w>': 19, '   lan ding</w>': 40, ' en g ine</w>': 5, '   en ti re ly</w>': 79, ' together</w>': 12, ' am</w>': 303, '   a ir plan e</w>': 27, '   lin a</w>': 8, '   w er t mu ll er</w>': 1, '   fe sti v al</w>': 16, ' land</w>': 7, '   fo g</w>': 36, '   m oun ta ins</w>': 65, '   chan ces</w>': 110, '   me mber</w>': 94, '   s li ght</w>': 26, '   f ever</w>': 44, '   c o</w>': 54, ' pi lot</w>': 9, '   ha m men</w>': 1, '   ran dy</w>': 40, '   la sa g na</w>': 3, '   st ea k</w>': 48, '   pa ti en ts</w>': 65, ' huh</w>': 306, '   f l own</w>': 20, '   ra ts</w>': 44, '   sp on ge</w>': 6, '   pl ans</w>': 206, ' en li st</w>': 1, '   ba se b all</w>': 99, '   li e u ten n t</w>': 1, '   hur wi t z</w>': 1, '   se v ere</w>': 28, '   sho ck</w>': 96, '   mer man</w>': 3, '   pa ti ent</w>': 177, '   s an d l er</w>': 2, '   raid</w>': 24, '   gen er al s</w>': 11, '   te le gra m</w>': 32, ' qu ar t ers</w>': 3, '   wri g g le</w>': 3, ' only</w>': 27, '   s qu ad r on</w>': 8, '   lea ding</w>': 68, '   ci ti es</w>': 38, ' w ar</w>': 13, ' re cor d</w>': 1, '   l ou i e</w>': 7, '   ne t z</w>': 1, '   at lan ta</w>': 24, '   qui e tly</w>': 37, '   ac u te</w>': 8, '   a ir si ck</w>': 2, '   vi si tor</w>': 24, '   fin al</w>': 113, '   run way</w>': 6, '   n in er</w>': 9, '   al ti tu de</w>': 13, '   er ra ti c</w>': 7, '   w ea ther</w>': 143, '   th ro ttle</w>': 9, '   lon ge st</w>': 23, '   di re c tly</w>': 73, '   e ar</w>': 137, '   g under son</w>': 7, ' great</w>': 19, '   b li ffer t</w>': 1, '   b lu e</w>': 336, '   tur t l en e ck</w>': 2, '   bu n s</w>': 8, '   pla y of f s</w>': 3, ' n in er</w>': 2, '   cour t</w>': 297, ' z er o</w>': 12, '   inter se c ting</w>': 1, '   vi ctor</w>': 110, '   a ir way</w>': 3, '   ta il w ind</w>': 1, '   ad di tion al</w>': 15, '   ro c ki es</w>': 5, '   se at</w>': 158, ' are</w>': 112, '   k ar ee m</w>': 3, '   con fu sed</w>': 101, '   mu r do ck</w>': 3, '   ab du l</w>': 5, '   j ab b ar</w>': 2, '   ba s ke t b all</w>': 34, '   la k ers</w>': 4, '   li s a</w>': 87, '   gu i t ar</w>': 31, '   al ar med</w>': 9, '   c up</w>': 163, '   o t</w>': 6, '   ve ctor</w>': 7, '   cle ar an ce</w>': 11, '   la te st</w>': 44, '   so cked</w>': 5, '   sa lt</w>': 55, '   lin co l n</w>': 48, '   b orrow</w>': 84, '   che er</w>': 48, '   f our te en</w>': 91, '   ha l f way</w>': 38, '   h ori z ons</w>': 5, '   n ar row</w>': 32, '   cu ri o si ty</w>': 40, '   s wi r ling</w>': 1, '   bro ad en ing</w>': 1, '   vi r g in</w>': 67, '   ar ou sing</w>': 6, '   en ter tain</w>': 21, '   ma t t</w>': 52, '   to d d</w>': 26, '   men tion ed</w>': 120, '   stra der</w>': 9, '   as si st ant</w>': 89, '   wa t son</w>': 15, '   plea sur able</w>': 8, '   oo oh</w>': 28, '   har d b all</w>': 2, '   ti e</w>': 165, '   ro p es</w>': 20, '   ss</w>': 12, ' a i</w>': 12, '   k</w>': 66, '   ser ge ant</w>': 130, '   sy kes</w>': 7, '   ca ss an dr a</w>': 9, '   ou ter</w>': 36, '   je t son</w>': 4, '   wa d ers</w>': 1, '   h y dro ch l ori c</w>': 1, '   ac id</w>': 44, '   fe d or ch u k</w>': 4, '   sur f in</w>': 2, '   di l do</w>': 5, '   s and</w>': 76, '   stan ding</w>': 256, '   s la g town</w>': 2, '   ne w com er</w>': 7, '   in for man ts</w>': 7, '   me x i can</w>': 43, '   h er o es</w>': 35, '   s qu at</w>': 10, '   al ter e z</w>': 2, '   tu g g le</w>': 2, '   har cour t</w>': 7, '   as si st</w>': 22, '   o ver du e</w>': 11, '   s la g</w>': 7, '   war r en</w>': 51, '   a w are</w>': 120, '   ne w com ers</w>': 6, '   ran k</w>': 42, '   hu b le y</w>': 12, '   brea ch</w>': 22, '   an der son</w>': 25, '   hu man s</w>': 42, '   o ver do sed</w>': 2, '   ma ssi ve</w>': 27, '   se i ze</w>': 9, '   in cu ba tion</w>': 3, '   e mer ge</w>': 5, '   ki pl ing</w>': 6, '   re que st ing</w>': 17, '   rea ssi g n ment</w>': 4, '   ki lo s</w>': 25, '   ma t the w</w>': 30, '   li gh ten</w>': 26, '   ha ir s</w>': 19, '   c ru sh</w>': 41, '   qu ar an t ine</w>': 17, '   por ter</w>': 26, '   che mi st ry</w>': 28, '   for mu l a</w>': 15, '   man u f ac tu red</w>': 5, '   re fin er y</w>': 6, '   ni gh t c lu b</w>': 14, '   e st ab li sh ed</w>': 15, '   di st ri bu tion</w>': 5, '   th rea ten</w>': 44, '   ex i st ence</w>': 64, '   pro du c ing</w>': 13, '   gu ar ded</w>': 13, '   co ca ine</w>': 66, ' hi gh</w>': 12, '   la sts</w>': 14, '   re w ard</w>': 72, '   l ab or</w>': 40, '   po t ent</w>': 6, ' j ab ro k a</w>': 1, '   n ar co ti c</w>': 3, '   ex p lo si ve</w>': 21, '   mi st</w>': 7, '   s our</w>': 20, '   in vo l ve</w>': 19, '   god dam m it</w>': 53, '   b en t ner</w>': 1, '   p each</w>': 21, '   pi ts</w>': 4, ' po p</w>': 7, '   min i</w>': 22, ' mar t</w>': 10, '   man a ge ment</w>': 27, '   jo shu a</w>': 9, '   su c ce ss ful</w>': 61, '   du de</w>': 300, '   bea ch</w>': 181, '   ro ll</w>': 149, '   pro gre ssi ve</w>': 6, '   ma te</w>': 54, '   i dea ls</w>': 9, '   pa in ful</w>': 32, '   con fu sing</w>': 26, '   your se l ves</w>': 54, '   in vi te</w>': 93, '   e qu a li ty</w>': 13, '   f re e d om s</w>': 2, '   o pp ort uni ti es</w>': 19, '   ow n er ship</w>': 14, '   su bor din ate</w>': 2, '   a spi re</w>': 4, '   ho l mes</w>': 9, '   chri s sa kes</w>': 28, '   pe o pl es</w>': 12, '   ta</w>': 89, ' ra k y on a</w>': 1, '   hea l th</w>': 102, '   sti cking</w>': 40, '   fi lls</w>': 9, '   p en</w>': 73, '   ther m ome ter</w>': 6, '   na da</w>': 10, '   re c tu m</w>': 2, '   su n day</w>': 127, '   hon e st y</w>': 19, '   di sor der ed</w>': 4, '   bur g l ar i z ed</w>': 2, '   el ev en th</w>': 10, '   re mo ved</w>': 45, '   con ce pt</w>': 45, '   wor ri es</w>': 42, '   lea sh</w>': 14, '   h er p es</w>': 5, ' of</w>': 175, ' bi tch</w>': 61, '   man a ger</w>': 127, ' ro k ya</w>': 1, ' la to</w>': 1, ' na</w>': 7, ' lo k a</w>': 1, ' s ma ll</w>': 5, '   in te lli g ent</w>': 63, '   c rea ture</w>': 102, '   lo ses</w>': 25, '   tr ans la tion</w>': 9, '   k a</w>': 8, '   as si mi late</w>': 4, '   nu tri en ts</w>': 1, '   co ok ed</w>': 24, '   to ssed</w>': 23, '   mo le</w>': 22, '   ho sti le</w>': 48, '   en vi r on men ts</w>': 3, '   su g ge sti on</w>': 47, '   w ea k</w>': 112, '   n er ve</w>': 109, '   p le x us</w>': 1, '   ca mp s</w>': 15, '   lo d ged</w>': 4, '   se le ction</w>': 12, '   fami li es</w>': 92, ' ar row</w>': 1, '   hou sed</w>': 1, '   se ssion</w>': 33, '   ma tes</w>': 12, '   ra th o le</w>': 2, '   con te mp t</w>': 15, '   out ca st</w>': 5, '   as so ci ate</w>': 35, '   ru d y ard</w>': 2, ' right</w>': 55, '   p ho to</w>': 56, '   wi d ow</w>': 48, '   u p sta ir s</w>': 193, ' possi ble</w>': 4, '   re fin er i es</w>': 2, '   me than e</w>': 6, '   fu mes</w>': 9, '   har m ful</w>': 6, '   sp ou se</w>': 4, '   ga ss es</w>': 5, '   sp ar ks</w>': 7, '   fi ts</w>': 50, '   st re tch es</w>': 2, '   con do m</w>': 13, '   con ey</w>': 7, '   whi te fi sh</w>': 1, '   pen i ses</w>': 2, '   se tt l ed</w>': 68, '   un rea son able</w>': 11, '   possi b ly</w>': 164, '   ex a min ing</w>': 12, '   sh ed</w>': 36, '   ra y</w>': 326, '   sh oo t in gs</w>': 5, '   coun ts</w>': 52, '   shi t ba g</w>': 2, ' as sed</w>': 17, '   ex e cu tions</w>': 3, '   pi ll ar</w>': 7, '   com m uni ty</w>': 83, '   ri pped</w>': 53, '   sig na l</w>': 125, '   di ck wa d</w>': 3, '   ru p tur ing</w>': 1, '   pri mar y</w>': 37, '   he ar ts</w>': 55, '   br i</w>': 5, '   sa bo t</w>': 1, '   s lu gs</w>': 11, '   a ll ey</w>': 45, '   a ven u e</w>': 62, ' ss</w>': 3, ' ex cre ment</w>': 1, ' c ran i u m</w>': 1, '   wh a tta</w>': 89, '   ge su n d he it</w>': 3, ' an g ya</w>': 1, ' sor en t s a</w>': 1, '   hu mp h re y</w>': 4, '   bo g art</w>': 7, '   har le y</w>': 8, '   da vi d son</w>': 12, '   p un ch y</w>': 6, '   qu ar ter</w>': 108, '   mo on li ght</w>': 14, '   wal ks</w>': 96, '   pa per work</w>': 28, '   c y lin der</w>': 10, '   ca su ll</w>': 1, ' 4 5 4 </w>': 2, '   imp act</w>': 28, '   lo a ds</w>': 21, '   af ter no on</w>': 274, ' n ea l</w>': 4, '   ha d da</w>': 11, '   fe ll as</w>': 72, '   ja mes</w>': 292, '   sc ra mb le</w>': 6, '   re lea sed</w>': 30, '   de ce mber</w>': 24, '   mo de s to</w>': 1, '   co al in g a</w>': 1, '   ex per ti se</w>': 23, '   or g ani c</w>': 25, '   dar l in</w>': 69, '   no ve mber</w>': 28, '   thir ti e th</w>': 7, '   re lo ca ted</w>': 5, '   ri ver side</w>': 5, '   fe b ru ary</w>': 12, '   man u f ac tur ing</w>': 6, '   ro ll er</w>': 11, '   en coun t ers</w>': 4, ' k ya</w>': 1, '   comp ound</w>': 36, '   a li en</w>': 51, '   b lo b</w>': 1, '   fini sh ing</w>': 19, '   po st mor te m</w>': 3, '   hea d sho t</w>': 2, '   fa sc in a ting</w>': 44, '   pu mp s</w>': 20, '   loo se ly</w>': 7, '   re f er</w>': 24, '   po ck e ts</w>': 32, '   fin ger pr in ts</w>': 33, '   f ac i al</w>': 8, '   mar t in</w>': 144, '   hel der</w>': 1, ' se ven</w>': 120, '   w ra p</w>': 42, '   cont ro lled</w>': 25, '   su b stan ce</w>': 17, '   wi red</w>': 33, '   co ke</w>': 86, '   cu sh y</w>': 3, '   coun ty</w>': 105, '   g ran ger</w>': 4, '   pi tt s</w>': 6, '   ca se lo a d</w>': 3, '   pla in clo th es</w>': 1, '   de part men tal</w>': 4, '   p in ned</w>': 22, '   com for ting</w>': 13, '   ve st</w>': 10, '   k ri st in</w>': 3, '   re ce p tion</w>': 36, '   bur g ers</w>': 13, '   bu mm ed</w>': 7, '   tu g</w>': 4, '   con ce i ved</w>': 8, '   ca b in</w>': 74, '   e die</w>': 53, '   ban ged</w>': 15, '   car o l</w>': 22, '   pla ster</w>': 10, ' we ar</w>': 2, '   c li p</w>': 21, '   lan d la dy</w>': 12, '   pl u mb ing</w>': 21, '   hu d son</w>': 52, '   ju i cy</w>': 15, '   co lon i sts</w>': 7, '   vi r g in i ty</w>': 14, ' o p</w>': 2, '   o p</w>': 14, ' let</w>': 76, '   c y c le</w>': 18, '   wh oo o ah</w>': 1, '   bu n ch a</w>': 14, '   fe tch</w>': 21, '   s li pp ers</w>': 15, '   in c in er at ors</w>': 2, ' a p one</w>': 1, '   supp re ss ing</w>': 3, '   s qu a ds</w>': 4, '   a p c</w>': 2, '   f la me</w>': 27, '   ri f les</w>': 11, '   s l un g</w>': 3, '   v as que z</w>': 6, '   hi cks</w>': 21, '   cor don</w>': 4, '   si x ty</w>': 171, '   me t ers</w>': 43, '   te le me try</w>': 4, '   ma st</w>': 13, ' cle ar</w>': 3, '   a w right</w>': 27, '   di sp er sa l</w>': 1, ' t ea m</w>': 5, '   bi sho p</w>': 33, '   se ven te en</w>': 90, '   v a p or</w>': 5, '   ne bra s k a</w>': 9, '   ri p le y</w>': 59, ' six</w>': 105, '   clo s er</w>': 143, '   qu a li fi ed</w>': 27, '   sy n the ti c</w>': 8, '   in ev i ta ble</w>': 16, '   pro je c ting</w>': 7, '   b last</w>': 53, '   ra di us</w>': 16, '   ki lo me t ers</w>': 20, '   me ga t ons</w>': 4, '   ven ting</w>': 2, '   in st in ct</w>': 36, ' att r ac tion</w>': 1, '   in cu ba te</w>': 2, '   sp o t</w>': 180, '   l ar ger</w>': 37, '   ter mi te</w>': 3, '   ab d om en</w>': 2, '   b lo a ted</w>': 9, '   ten ded</w>': 2, '   dr one</w>': 3, '   wor k ers</w>': 40, '   de f en ded</w>': 7, '   war ri ors</w>': 23, '   so ci e ty</w>': 108, '   la ys</w>': 15, '   par a ll el</w>': 12, '   in se ct</w>': 18, '   hi ve like</w>': 1, '   ant</w>': 24, '   co lon y</w>': 31, '   ex a mp le</w>': 96, '   ru l ed</w>': 16, '   fe ma le</w>': 93, '   fo ll ows</w>': 31, '   par a si tes</w>': 20, ' over</w>': 33, '   a li en s</w>': 20, '   par al y z ed</w>': 9, '   co co on ed</w>': 2, '   ho sts</w>': 4, '   i so la ted</w>': 21, ' mu s cu l ar</w>': 1, '   to x in</w>': 6, '   par al y s is</w>': 9, '   me ta bo li z ing</w>': 1, '   st e pped</w>': 32, '   ex c lu si ve</w>': 29, '   bur ke</w>': 39, '   di re c tive</w>': 8, ' tw el ve</w>': 12, ' se v enty</w>': 7, ' n ine</w>': 72, '   car ter</w>': 92, '   h ome work</w>': 58, '   imp ound</w>': 7, '   or g ani s m</w>': 18, '   i c c</w>': 1, '   2 2 3 5 0</w>': 1, '   com mer ce</w>': 16, '   bi o</w>': 21, ' w ea p ons</w>': 1, '   ju ri s di ction</w>': 31, '   cor por al</w>': 16, '   ar bi tr ar i ly</w>': 2, '   ex ter min ate</w>': 3, '   e mo tion al</w>': 79, '   ju d g men ts</w>': 1, '   ca u ti ou s ly</w>': 5, '   in sta ll ation</w>': 5, '   su b stan ti al</w>': 13, '   v al u e</w>': 85, '   att ac h ed</w>': 37, '   au th ori z ing</w>': 3, '   bi o che mi st ry</w>': 1, '   nu ke</w>': 27, '   si te</w>': 45, '   or b it</w>': 30, '   g or man</w>': 12, '   ru p ture</w>': 5, '   co o ling</w>': 13, '   or g ani s ms</w>': 5, '   ex chan g ers</w>': 1, ' h un h</w>': 3, '   au to ma ted</w>': 10, '   man u f ac ture</w>': 9, '   pro ce ss or</w>': 9, '   dis ks</w>': 33, '   o c cu r</w>': 33, '   an dro id</w>': 13, ' cl ean</w>': 3, ' for ever</w>': 3, '   ye llo</w>': 1, '   sh ee ts</w>': 52, '   so a king</w>': 6, '   p sy ch</w>': 14, '   ev al u ation</w>': 9, '   ki d do</w>': 20, '   re in sta ted</w>': 5, '   con tr act</w>': 136, '   lo ad ers</w>': 1, '   for k li f ts</w>': 1, '   cor por ation</w>': 31, ' fin an c ed</w>': 1, '   co lon i al</w>': 6, '   min er al</w>': 10, '   ter ra for m ing</w>': 2, ' bu il ding</w>': 3, '   rea med</w>': 2, '   st ea med</w>': 7, '   d on u t</w>': 26, '   min ds</w>': 79, '   un e mo tion al</w>': 1, '   hea v y we i gh ts</w>': 1, '   inter st e ll ar</w>': 5, '   ac cu ra te</w>': 27, '   li cked</w>': 23, '   cre ma ted</w>': 8, '   inter red</w>': 1, '   par k side</w>': 1, '   re po si t ory</w>': 4, '   ch u te</w>': 4, '   wi s con s in</w>': 16, ' m c cl aren</w>': 1, '   in que st</w>': 2, '   lo ca ted</w>': 20, '   dri f ted</w>': 8, '   c ore</w>': 55, ' sa l v age</w>': 1, ' oh</w>': 171, '   a p one</w>': 2, '   ther mon u cle ar</w>': 3, '   ex p lo sion</w>': 40, '   p d t</w>': 1, ' da ta</w>': 1, '   tr ans mi tt ers</w>': 4, '   co lon i st</w>': 1, '   sur gi ca lly</w>': 5, '   im plan ted</w>': 8, '   sc an ning</w>': 8, '   in su red</w>': 14, '   chan ne ls</w>': 24, '   mar in es</w>': 22, '   h om br es</w>': 3, '   p ac king</w>': 41, ' art</w>': 13, '   fi re p ower</w>': 9, '   t ro op s</w>': 46, '   tr ans mi tter</w>': 13, '   ad vi s or</w>': 18, '   han go ver</w>': 10, ' then</w>': 75, '   fu sion</w>': 4, '   con ta in ment</w>': 21, '   shu ts</w>': 14, '   pro ce ed</w>': 55, ' b lo ck</w>': 2, '   in t act</w>': 29, ' si mu la ted</w>': 1, '   pro ce ss ing</w>': 10, ' su bl ev el</w>': 1, '   ha h</w>': 37, '   gr in n in</w>': 1, '   lin en</w>': 15, '   c p u</w>': 6, ' l ine</w>': 27, '   sa f er</w>': 49, '   se cu red</w>': 22, ' l ink</w>': 3, '   x en om or p h</w>': 1, '   bu g</w>': 78, '   ea se</w>': 61, '   bri e f</w>': 36, '   ga te way</w>': 14, '   fu </w>': 17, '   per i me ter</w>': 26, '   st run g</w>': 21, '   fro st y</w>': 3, '   en tri es</w>': 3, '   de mor a li z ed</w>': 2, '   a ye</w>': 163, ' fir ma tive</w>': 1, '   out stan ding</w>': 20, '   t un ne l</w>': 73, '   e ll en</w>': 76, '   per cen ta ge</w>': 18, '   sa bo ta ged</w>': 6, '   f re e z ers</w>': 9, '   je t ti son</w>': 1, '   gr en a de</w>': 18, '   la un ch er</w>': 5, '   ne w t</w>': 12, '   app ro ac h</w>': 69, '   w el ded</w>': 5, '   bar ri ca d es</w>': 2, '   inter se c tions</w>': 1, '   du c ts</w>': 9, '   cor ri d ors</w>': 8, '   sen try</w>': 9, '   de cl ar ed</w>': 19, '   e mb r y o</w>': 2, '   im plan ta tion</w>': 1, '   mar ac hu k</w>': 1, '   cu r r ent</w>': 49, '   c ani st ers</w>': 5, '   c n</w>': 1, ' 2 0</w>': 19, '   ne st</w>': 38, '   any time</w>': 57, '   mi sta k en</w>': 49, '   rea d in</w>': 11, '   du ct</w>': 11, '   f lo ors</w>': 21, '   ran ge</w>': 82, ' must</w>': 12, '   ani ma ls</w>': 138, '   por ta ble</w>': 14, '   ter min al</w>': 24, '   man u ally</w>': 7, '   wa sted</w>': 46, ' ship</w>': 4, '   su l ac o</w>': 1, '   su bl ev el</w>': 2, ' hey</w>': 77, '   c r ow e</w>': 4, '   di e tri ch</w>': 2, '   st un g</w>': 7, '   ge sta ting</w>': 2, ' e mer g es</w>': 2, '   m ou l ts</w>': 1, '   gr ows</w>': 32, '   ra pi d ly</w>': 11, '   hon ch</w>': 1, '   g ri d</w>': 23, '   h or n</w>': 37, '   h om ing</w>': 4, '   cla i m</w>': 76, '   hon or ed</w>': 33, '   wi l d ca tt ers</w>': 1, '   pla t ea u</w>': 4, '   i li u m</w>': 2, ' m om my</w>': 4, '   fa ster</w>': 133, '   ss sh</w>': 12, '   wa h</w>': 5, '   gr ow</w>': 230, '   mon st ers</w>': 42, '   na p</w>': 39, '   wa lls</w>': 79, ' a ze</w>': 1, '   ma ze</w>': 23, ' ne w t</w>': 2, '   ca se y</w>': 20, ' name</w>': 26, '   re be c c a</w>': 31, '   d or k</w>': 25, ' ho sti le</w>': 1, '   ac h er on</w>': 1, '   l v </w>': 1, ' 4 2 6 </w>': 1, ' because</w>': 45, '   ha tch ing</w>': 3, '   plan e to id</w>': 2, '   re cor der</w>': 13, '   da ta</w>': 165, '   t ea ms</w>': 22, '   cen ti me ter</w>': 3, '   con tain</w>': 15, '   con cer ning</w>': 19, '   a ll e ge d ly</w>': 6, '   pe ck</w>': 16, '   c ru el ty</w>': 9, '   w re ck</w>': 52, '   bea k</w>': 6, '   pa pa gen o</w>': 3, '   bi lling</w>': 10, '   lo v ey</w>': 19, '   do v ey</w>': 3, '   ta st y</w>': 20, '   di sh</w>': 43, '   n et</w>': 53, '   bi r die</w>': 30, '   a id</w>': 38, '   gh o st</w>': 88, '   fa de</w>': 18, '   pr in ce</w>': 81, '   wi s do m</w>': 36, '   w ine</w>': 154, '   gr ace</w>': 182, '   vi en ne se</w>': 4, '   re mar k ably</w>': 8, '   ar ch bi sho p</w>': 6, '   su b li me</w>': 8, '   hon e st ly</w>': 92, '   plea ses</w>': 16, '   ma e st r o</w>': 9, '   k a ther in a</w>': 2, '   en han ce</w>': 5, '   ce le bra ted</w>': 9, '   thr ou gh out</w>': 21, '   vi en na</w>': 38, '   sin ging</w>': 88, '   tur k</w>': 18, '   tur ki sh</w>': 14, '   bro the l</w>': 3, '   har em</w>': 4, '   com mi ssi on ed</w>': 7, '   op er a</w>': 97, '   tra v el s</w>': 26, '   h er r</w>': 64, '   mo z art</w>': 75, '   go ssi p</w>': 30, '   ha ir dre ss er</w>': 20, '   f ra u le in</w>': 12, '   sig n ore</w>': 3, '   w o l f g an g</w>': 6, '   vi r tu o so</w>': 3, '   re li ev ed</w>': 29, '   ar i a</w>': 4, '   di so be y ed</w>': 5, '   in du l g ent</w>': 5, '   im pl ore</w>': 7, '   fa i th fu lly</w>': 6, '   un pr in ci pl ed</w>': 1, '   sp o il ed</w>': 32, '   con ce i ted</w>': 8, '   br at</w>': 21, '   fri en d ship</w>': 46, '   s lu t</w>': 46, '   tra p</w>': 106, '   sa l z bur g</w>': 8, '   pro te ction</w>': 74, '   in ten tion</w>': 47, '   dis mi ss ing</w>': 4, '   hu mi li ty</w>': 12, '   dis mi ssed</w>': 13, '   sa ti s f y</w>': 29, '   pa ti en tly</w>': 1, '   sa ti s fi ed</w>': 76, '   dis miss</w>': 12, '   pro vo ca tion</w>': 3, '   en du re</w>': 13, '   a llow</w>': 121, '   ser v an ts</w>': 19, '   po t</w>': 60, '   st e w</w>': 44, '   sin ger</w>': 60, '   me at</w>': 117, '   o tt a vi o</w>': 1, '   no b le man</w>': 11, '   d ine</w>': 16, '   din n n n n ner</w>': 1, '   gi o v an n n n n n n n i</w>': 1, '   stan z i</w>': 11, '   w o l f i</w>': 13, '   vo i ces</w>': 57, ' man z i</w>': 2, '   mon k ey</w>': 74, ' f l un k i</w>': 1, ' p un k i</w>': 1, '   f ar t s bi sho p</w>': 3, '   sc hi k an e der</w>': 1, '   c ri ti ci ze</w>': 8, '   co ok</w>': 113, '   f ea st</w>': 12, '   pa p a</w>': 49, '   par ti es</w>': 76, '   d rea d ful</w>': 24, '   con fe ss</w>': 61, '   pre t ti er</w>': 21, '   pri de</w>': 66, '   sh e er</w>': 12, '   stu pi di ty</w>': 21, '   bor r ow ed</w>': 36, '   bor r ow ing</w>': 11, '   ro y al</w>': 86, '   pu pi l</w>': 12, '   f lo cking</w>': 1, '   ma ma</w>': 321, '   ti sh</w>': 2, ' te e</w>': 2, '   te e</w>': 24, '   tu b</w>': 39, ' tu b</w>': 4, '   vo l</w>': 1, '   u i</w>': 2, ' vo l</w>': 1, '   i ra m</w>': 5, '   fi end</w>': 20, '   b ack war ds</w>': 45, '   s r a</w>': 5, ' si ck</w>': 13, '   ar se</w>': 17, ' w it</w>': 6, '   bri lli ant</w>': 111, '   f art</w>': 22, '   re si d ence</w>': 21, '   sto pp ing</w>': 59, '   s l ow ly</w>': 80, '   sa li er i</w>': 11, '   as si st ing</w>': 6, '   han d wri ting</w>': 14, '   can d le sti cks</w>': 2, '   be ha ved</w>': 7, '   v a gu e</w>': 20, '   see ks</w>': 13, '   e m per or</w>': 92, '   mi r ac u l ous</w>': 8, '   co pi es</w>': 32, '   ori g in al s</w>': 8, '   fran ti c</w>': 9, '   com po s er</w>': 20, '   sp ends</w>': 22, '   ear n</w>': 39, '   la zy</w>': 35, '   fin g ers</w>': 95, '   mu si ci ans</w>': 16, '   in de b ted</w>': 5, '   fu ri ous</w>': 15, '   le i sure</w>': 11, '   hon our</w>': 14, '   be half</w>': 36, '   h us</w>': 1, '   sa m pl es</w>': 30, '   f ra u</w>': 7, '   re qui em</w>': 4, '   li b re t to</w>': 3, '   cl own</w>': 49, '   inter ru p ting</w>': 21, '   pi g st y</w>': 3, '   hou se kee per</w>': 10, '   un heard</w>': 4, '   ad mi r ers</w>': 4, '   gi f ts</w>': 37, '   be hold</w>': 17, '   st ran ger</w>': 64, '   th ea tr es</w>': 5, '   in so l ence</w>': 6, '   in supp or ta ble</w>': 2, '   we b er</w>': 2, '   fa v ou red</w>': 1, '   st in ks</w>': 38, '   ger tru de</w>': 6, '   a g es</w>': 28, '   out gre w</w>': 2, '   re f re sh ment</w>': 4, '   cho co late</w>': 65, '   li king</w>': 18, '   de cre es</w>': 1, '   mo le sted</w>': 8, '   pu pi ls</w>': 13, '   mu si ci an</w>': 22, '   si re</w>': 52, '   in qui ri es</w>': 11, '   al ar m ing</w>': 3, '   a gon i z ing</w>': 2, '   re com m end</w>': 53, '   wi sh es</w>': 52, ' h a</w>': 52, '   fa v ou ri ti s m</w>': 2, '   su spi ci on</w>': 27, '   re su lt</w>': 46, '   sa ti s fi es</w>': 2, '   t re men d ous</w>': 28, '   hi gh ne ss</w>': 40, '   ni e ce</w>': 30, '   pr in c ess</w>': 118, '   e li z a be th</w>': 75, '   ca t ti v o</w>': 2, '   ger man</w>': 81, '   for ei g ner</w>': 9, '   im men se</w>': 8, '   di le t to</w>': 1, '   stra or din ar i o</w>': 1, '   i ll u st ri ous</w>': 4, ' f l at</w>': 4, '   per m it</w>': 47, '   tri f le</w>': 11, '   cha mb er la in</w>': 9, '   no tion</w>': 18, '   in fu ri ate</w>': 2, '   m ea sure</w>': 35, '   pa ssi on ate</w>': 23, '   per su a de</w>': 21, '   v u l g ar</w>': 13, '   cont in u ous</w>': 4, '   re ci ta ti ves</w>': 1, '   du et</w>': 4, '   qu ar re ling</w>': 6, '   s che m ing</w>': 2, '   ma id</w>': 69, '   un ex pe c te d ly</w>': 6, '   tri o</w>': 5, '   e qu ally</w>': 12, '   v al et</w>': 20, '   qu ar t et</w>': 12, '   gar den er</w>': 22, '   qu in t et</w>': 5, '   se x t et</w>': 1, '   se p t et</w>': 1, '   o c t et</w>': 1, '   su sta in</w>': 16, '   fro li c</w>': 2, '   pro vo ke</w>': 7, '   th ea t re</w>': 63, '   to l er ant</w>': 6, '   cen s or</w>': 3, '   li gh tly</w>': 8, '   fi gar o</w>': 5, '   sti rs</w>': 2, '   cla ss es</w>': 40, '   fran ce</w>': 100, '   ca used</w>': 70, '   bi tt er ne ss</w>': 8, '   an to in e tt e</w>': 6, '   wri tes</w>': 34, '   fe ars</w>': 25, '   c ome dy</w>': 35, '   un su i ta ble</w>': 1, '   me mor able</w>': 7, '   fa v our</w>': 23, '   bra v o</w>': 28, '   con s ent</w>': 16, '   al together</w>': 22, '   en chan ted</w>': 6, '   in gen i ous</w>': 9, '   qu a li ty</w>': 49, '   no tes</w>': 71, '   o c ca si on ally</w>': 25, '   el ab or ate</w>': 16, '   un questi on ably</w>': 1, '   pa sh a</w>': 1, '   ser a g li o</w>': 2, '   a mu sing</w>': 28, '   vo te</w>': 78, '   gra z i e</w>': 3, '   son o</w>': 1, '   com mo s so</w>': 1, '   u n</w>': 59, '   on ore</w>': 1, '   per</w>': 99, '   m o</w>': 14, '   e c ce z i on a le</w>': 1, '   com po si t ore</w>': 1, '   bri lli an te</w>': 1, '   fa mo ssi ssi m o</w>': 1, '   com po sed</w>': 7, '   or sin i</w>': 3, ' ro sen ber g</w>': 2, '   re li c</w>': 4, '   st ool</w>': 18, '   ju mp ed</w>': 69, '   pla in</w>': 86, '   te mp ted</w>': 19, '   e f for t</w>': 44, '   ac qui re</w>': 11, '   hu sh</w>': 29, '   gra ti tu de</w>': 29, '   com man d ment</w>': 3, ' hon our</w>': 1, '   bur d en</w>': 24, '   pen al ty</w>': 17, '   ki s sa ble</w>': 2, '   mi s sa ble</w>': 1, '   vi si ble</w>': 14, '   mar v el ous</w>': 48, '   com po si tion</w>': 7, '   ma li ci ous</w>': 6, '   de b t</w>': 37, '   fin an ci al</w>': 58, '   fe ed</w>': 146, '   fee ds</w>': 15, '   stu ff s</w>': 1, '   go ose</w>': 41, '   u r g ent</w>': 45, '   st e ps</w>': 54, '   gu lli ble</w>': 6, '   b id</w>': 25, '   a wait</w>': 11, '   si ts</w>': 50, '   sp ea ks</w>': 45, '   pa ins</w>': 21, '   fri gh ten s</w>': 10, '   dr in ks</w>': 100, '   b en e</w>': 9, '   ca v a li er i</w>': 1, '   l or l</w>': 3, '   si en a</w>': 5, '   m ac ar o ons</w>': 1, '   fa v ou ri tes</w>': 1, '   ba k er</w>': 25, '   an on</w>': 5, '   an on y m ous</w>': 34, '   ma i d ser v ant</w>': 2, '   re he ar sa l</w>': 26, '   v au d ev i ll e</w>': 3, '   son gs</w>': 52, '   tri u mp h</w>': 16, ' lu x e</w>': 2, '   s na ke</w>': 79, '   dre s d en</w>': 5, '   fo l ds</w>': 6, '   in ch es</w>': 28, '   s no b by</w>': 1, '   pro du ction</w>': 39, '   ma gi c</w>': 140, '   f loo d</w>': 16, '   bu n dle</w>': 22, '   gi o v an n i</w>': 5, '   f oo ls</w>': 43, '   p uni sha ble</w>': 2, '   u tt er ly</w>': 21, '   ru le</w>': 152, '   pen al ti es</w>': 4, '   per for med</w>': 28, ' ban z i</w>': 1, ' wan z i</w>': 1, '   d ou b t</w>': 264, '   en d ow ed</w>': 2, '   app e ars</w>': 75, '   ea ger</w>': 21, '   in st ru ction</w>': 3, '   man n he i m</w>': 6, '   an x i ous</w>': 47, '   in st ru ct</w>': 11, '   do gs</w>': 150, '   st ru de l</w>': 8, '   au di ence</w>': 89, '   sch lu mb er g</w>': 1, '   t ea ch ing</w>': 58, '   han na h</w>': 62, '   wi ll ful</w>': 4, '   har m less</w>': 38, '   l ac ri mo s a</w>': 1, ' i c t is</w>': 1, '   so p ran o s</w>': 4, ' vo c a</w>': 1, '   al to s</w>': 2, '   v o</w>': 6, ' c a</w>': 4, '   cu m</w>': 14, ' n e</w>': 1, '   di c</w>': 2, ' t is</w>': 11, '   vo c a</w>': 2, '   so t to</w>': 2, '   vo ce</w>': 2, '   pi ani ssi m o</w>': 1, '   b en e di c t is</w>': 1, '   b le ssed</w>': 25, '   st r in gs</w>': 28, '   uni son</w>': 2, '   o st in a to</w>': 1, '   tru m pe ts</w>': 4, '   ti mp an i</w>': 1, '   d om in ant</w>': 6, '   i den ti cal</w>': 19, '   ten ors</w>': 2, '   ba s so on</w>': 3, '   ten or</w>': 9, '   t ro mb one</w>': 5, '   or che st r a</w>': 23, '   ba ss</w>': 19, '   ba ss es</w>': 2, '   r h y th m</w>': 12, '   f our th</w>': 73, '   f la m</w>': 4, ' m is</w>': 3, ' c r i</w>': 4, ' bu s</w>': 8, ' di c</w>': 6, '   m is</w>': 9, ' fu </w>': 4, ' ta</w>': 5, ' le</w>': 8, ' sh ar p</w>': 1, '   con fu ta t is</w>': 3, '   ma le di c t is</w>': 2, '   con sig ned</w>': 2, '   f la mes</w>': 21, '   w o e</w>': 4, '   wi cked</w>': 47, '   con f oun ded</w>': 4, '   f la mm is</w>': 1, '   ac ri bu s</w>': 1, '   ad di c t is</w>': 1, '   tr ans late</w>': 12, '   re cor d are</w>': 1, '   sta tu en s</w>': 1, '   par te</w>': 1, '   de x tr a</w>': 1, '   re g ard</w>': 27, '   di z zy</w>': 26, '   fo c us</w>': 70, '   f an ci es</w>': 10, '   g ran de st</w>': 5, '   op er one</w>': 1, '   wri tt en</w>': 135, '   co ll ea gu e</w>': 19, '   sp a</w>': 8, '   ne g le c ting</w>': 4, '   me ss en ger</w>': 25, '   de ser ved</w>': 34, '   f la tter</w>': 20, '   le i</w>': 2, '   sig n or</w>': 28, '   an ton i o</w>': 10, '   mi o</w>': 7, '   car o</w>': 2, '   im po sing</w>': 8, '   o ver e sti ma te</w>': 4, '   cla p</w>': 19, '   ou tra ge ous</w>': 18, '   gr ac e fu lly</w>': 5, '   per for man ces</w>': 8, '   with d ra w n</w>': 7, '   for t un ate</w>': 26, ' wri te</w>': 6, '   re wri te</w>': 8, '   un beli ev able</w>': 56, '   pa g es</w>': 55, '   st a</w>': 2, '   ca l m o</w>': 1, '   fa v ore</w>': 1, '   de b tor</w>': 2, '   di st in gu i sh ed</w>': 9, '   f l or ins</w>': 3, '   pa tri o ti c</w>': 7, '   g ran u la ted</w>': 1, '   su f fu sed</w>': 1, '   ru m</w>': 38, '   cre ma</w>': 1, '   al</w>': 161, '   ma sc ar p one</w>': 1, '   de li ci ous</w>': 32, '   l end</w>': 21, '   pa d lo ck</w>': 4, '   ter ms</w>': 91, '   ex p lo de</w>': 25, '   stu pen d ou s ly</w>': 1, '   con cer ts</w>': 10, '   f oo t man</w>': 3, '   in cl in ed</w>': 14, '   po ver ty</w>': 12, '   imp la u si ble</w>': 1, '   mu si cal</w>': 31, '   p al ace</w>': 68, '   s om m er</w>': 4, '   ac hi eve</w>': 14, '   me di o c ri ty</w>': 7, '   su b mi tting</w>': 5, '   hu mb ly</w>': 4, '   f la tt er ed</w>': 29, '   pre s sur es</w>': 9, '   br in gs</w>': 93, '   i ta ly</w>': 50, '   y i el ded</w>': 1, '   ad one</w>': 1, '   v ar i a tions</w>': 6, '   me lo dy</w>': 3, '   inter pre t</w>': 10, '   e di c ts</w>': 1, '   pro hi b it</w>': 1, '   ex pre ss ly</w>': 7, '   for bi d d en</w>': 33, '   op er as</w>': 6, '   sc ar ce ly</w>': 6, '   su b m it</w>': 26, '   mo du la tion</w>': 3, '   o hi me</w>': 1, '   mor bi de z z a</w>': 2, '   i ta li ans</w>': 15, '   i di o ts</w>': 24, '   ter ri fi es</w>': 2, '   ban a li ty</w>': 3, '   re sur re ction</w>': 15, '   b a</w>': 30, ' b a</w>': 4, '   mor b id</w>': 11, '   k a pe ll me i ster</w>': 1, '   b on no</w>': 1, '   ro sen ber g</w>': 2, '   mo de st y</w>': 6, '   app o int</w>': 8, ' old</w>': 92, '   e ter na l</w>': 26, '   e p he mer al</w>': 2, '   en no ble</w>': 1, '   el ev a ted</w>': 12, '   el ev ate</w>': 5, '   le gen ds</w>': 6, '   go ds</w>': 58, '   d ou b ts</w>': 41, '   li ter a ture</w>': 23, '   f ar ce</w>': 11, '   ru b bi sh</w>': 10, '   the mes</w>': 2, '   no i se</w>': 107, '   in di vi du al s</w>': 7, '   har mon y</w>': 16, '   bar on</w>': 113, '   con cu b in es</w>': 2, '   ex po sing</w>': 3, '   in de c ent</w>': 2, '   mor al</w>': 55, '   vi r tu es</w>': 8, '   s na tch ed</w>': 6, '   de st ro y ed</w>': 112, '   be lo ved</w>': 34, '   s ma ll est</w>': 9, '   den i es</w>': 9, '   u ses</w>': 47, '   wor n</w>': 29, '   f lu te</w>': 3, '   lo a thing</w>': 3, '   im mor ta li ty</w>': 8, '   in fa my</w>': 2, '   ob li vi on</w>': 7, '   mer ci ful</w>': 12, '   pen an ce</w>': 9, '   po i son ed</w>': 27, '   con fe ss ing</w>': 5, '   ac tu al</w>': 49, '   ca the d ra l</w>': 15, '   co ff in</w>': 36, '   si l ence</w>': 59, '   di v ine</w>': 33, '   bur sts</w>': 3, '   su b li mi ty</w>': 1, '   p ow er less</w>': 9, '   sta le</w>': 8, '   ans w ers</w>': 139, '   un ra ve l</w>': 3, '   my st er i es</w>': 18, '   lon ging</w>': 8, '   mu te</w>': 12, '   im pl ant</w>': 15, '   de si re</w>': 92, '   l ust</w>': 20, '   pen i t ence</w>': 1, '   brea d</w>': 74, '   v ow</w>': 20, ' wor king</w>': 10, '   cha st e</w>': 7, '   go o d ne ss</w>': 67, '   in com pre h en si ble</w>': 6, '   gi ft</w>': 171, '   in du l ging</w>': 5, '   re bu ke</w>': 2, '   te sted</w>': 38, '   for gi ven ess</w>': 18, '   fi lling</w>': 27, '   be g an</w>': 68, '   vi o l ent</w>': 45, '   c ru e lly</w>': 2, '   a ma de us</w>': 1, '   re call</w>': 105, '   you th</w>': 40, '   un bur d en</w>': 2, '   see k</w>': 58, '   con fe ssion</w>': 68, '   gra ve ly</w>': 2, '   vo g l er</w>': 1, '   cha pla in</w>': 4, '   spe ci fi ca lly</w>': 20, '   an ger</w>': 57, '   in cre di ble</w>': 120, '   re he ar sing</w>': 6, '   man u sc ri pt</w>': 9, '   out wi ts</w>': 1, '   ex po ses</w>': 2, '   le ch er</w>': 3, '   g ran der</w>': 3, '   ex po se</w>': 28, '   ban ned</w>': 15, '   re f re sh</w>': 8, '   in cl ine</w>': 3, '   ten der</w>': 23, '   de le c ta ble</w>': 1, '   bi r d</w>': 166, '   ten der ly</w>': 2, '   a li en ate</w>': 5, '   con stan tly</w>': 29, '   de b ts</w>': 23, '   b ac h</w>': 7, '   fe e</w>': 57, '   e mb ar ra ss ing</w>': 66, '   in t ro du c ed</w>': 46, '   be g ging</w>': 41, '   go o d night</w>': 154, '   per fe ction</w>': 20, '   su per b</w>': 10, '   won der fu lly</w>': 7, ' de</w>': 18, '   sh ou ted</w>': 7, '   or der ly</w>': 11, '   tra y</w>': 18, '   ho b b s</w>': 15, '   di stu r ban ce</w>': 28, '   le i ce ster</w>': 2, '   in vo l ving</w>': 16, '   sh or tly</w>': 34, '   lu ci d</w>': 5, '   no ti f y</w>': 12, '   im per a tive</w>': 9, '   st ran ge ly</w>': 14, '   z o o</w>': 60, '   ra tion al</w>': 22, '   wan der ing</w>': 40, '   f ou rs</w>': 6, '   h ow ling</w>': 7, '   der an ged</w>': 10, '   hi r s ch</w>': 12, '   su ffer ed</w>': 32, '   tra u ma</w>': 17, '   wi t ne ssed</w>': 16, '   ne u ro s is</w>': 6, '   pro ctor</w>': 5, '   vi ll a g ers</w>': 7, '   go o d man</w>': 11, '   w er e w o lf</w>': 42, '   su g ge st ing</w>': 34, '   lon don</w>': 138, '   vi ll age</w>': 87, '   wi t ne ss es</w>': 59, '   e s cap ed</w>': 62, '   l un a ti c</w>': 53, '   in ve sti ga te</w>': 25, ' i sp l ac ed</w>': 1, '   l ac er a tions</w>': 5, '   supp o se d ly</w>': 23, '   ex a min ed</w>': 27, '   pu b</w>': 15, '   app ear ed</w>': 19, '   a le x</w>': 189, '   per si sted</w>': 1, '   f an ta si es</w>': 26, '   mi d night</w>': 99, '   ex tr ac u r ri cu l ar</w>': 5, '   ac ti vi ti es</w>': 27, '   con se qu ence</w>': 12, '   su s an</w>': 123, '   w o lf</w>': 70, '   du ti es</w>': 23, '   ga ll a gh er</w>': 49, '   per for m</w>': 30, '   ke ss l er</w>': 12, '   c ried</w>': 36, '   mu ti la ted</w>': 10, '   c li mb ed</w>': 17, '   wa king</w>': 30, '   fri dge</w>': 20, '   do per</w>': 1, '   w er e w o l ves</w>': 7, '   app re h en si ve</w>': 3, '   en or m ous</w>': 40, '   ha ll u c in a ting</w>': 8, '   brea k fa st</w>': 186, '   te lly</w>': 25, '   possi bi li ty</w>': 80, '   vi si ts</w>': 13, '   dis char ged</w>': 10, '   af fa ir</w>': 83, '   a he m</w>': 11, '   war wi ck</w>': 2, '   ca st le</w>': 90, '   att r ac ted</w>': 47, '   can did</w>': 7, '   si mp li ci ty</w>': 7, '   fami li ar i ty</w>': 6, '   ar m or</w>': 29, '   re st fu l ne ss</w>': 1, '   mo de st</w>': 26, '   ta il</w>': 69, '   h er d</w>': 14, '   con ne c ti cu t</w>': 19, '   y an ke e</w>': 29, '   ar th u r</w>': 57, '   sa mu el</w>': 44, '   pre face</w>': 1, '   t wa in</w>': 3, '   ea ten</w>': 60, '   a w k w ard</w>': 32, '   b en ja m in</w>': 108, '   be an o s</w>': 1, '   char t</w>': 26, '   e m ba ss y</w>': 33, '   fe ll ows</w>': 38, '   d ra ma ti c</w>': 31, '   su ff ers</w>': 13, '   re fu sing</w>': 12, '   co cks</w>': 7, '   char les</w>': 200, '   fa g got</w>': 33, '   win st on</w>': 29, '   ch u r chi ll</w>': 15, '   und ead</w>': 11, '   fa ther less</w>': 4, '   b loo d l ine</w>': 3, '   se ver ed</w>': 18, '   cu r se</w>': 66, '   br in g s ly</w>': 2, '   wh er ea s</w>': 18, '   car ni v or ous</w>': 2, '   at lan ti c</w>': 29, '   na tions</w>': 25, '   sha n</w>': 15, '   sh h h</w>': 89, '   gla d ys</w>': 10, '   cra sh ing</w>': 11, '   ba mb i</w>': 10, ' e e</w>': 17, '   we d ne s day</w>': 59, '   ba d ly</w>': 82, '   att en ded</w>': 13, '   h ound</w>': 23, '   ba s k er vi ll es</w>': 2, '   in ven tion</w>': 25, '   con an</w>': 9, '   do y le</w>': 79, '   f ra u d</w>': 27, '   dis believe</w>': 2, '   la mb </w>': 37, '   tra u ma ti z ed</w>': 9, '   b are</w>': 33, '   ani ma l</w>': 211, '   r ab id</w>': 5, '   bi z ar re</w>': 24, '   se da tive</w>': 8, '   b ru i ses</w>': 12, '   du el ing</w>': 4, '   sc ars</w>': 25, '   bo a st</w>': 12, '   fi er ce</w>': 15, '   un con s ci ous</w>': 38, '   re c ent</w>': 36, '   tra u ma ti c</w>': 7, '   coun tr y man</w>': 1, '   co ll ins</w>': 29, '   cho ke</w>': 25, '   ger a ld</w>': 14, '   su b way</w>': 37, '   me ss y</w>': 28, '   w o l f man</w>': 1, '   cor p se</w>': 23, '   can in es</w>': 1, '   dam it</w>': 3, '   b en ea th</w>': 38, '   sp r out</w>': 3, '   f an gs</w>': 10, '   sur r ound</w>': 9, '   p ow ers</w>': 82, '   dar k ne ss</w>': 68, '   b loo d sh ed</w>': 14, '   bor is</w>': 11, '   k ar lo ff</w>': 3, '   di a lo gu e</w>': 15, '   st al k</w>': 5, '   be w are</w>': 22, ' be w are</w>': 4, '   cont in u e</w>': 120, '   re ma in ing</w>': 16, '   un na tu ra l</w>': 10, '   li mb o</w>': 2, '   z om bi e</w>': 14, '   l y can th ro pe</w>': 1, '   dis con cer ting</w>': 4, '   g ri s ly</w>': 4, '   lo a f</w>': 17, '   mo cks</w>': 4, '   ru dy</w>': 30, '   l ev ine</w>': 6, '   sh mu ck</w>': 4, '   st ri ck en</w>': 5, '   so l ace</w>': 2, '   k le in</w>': 10, ' li ked</w>': 4, '   some place</w>': 123, '   un se tt ling</w>': 1, '   who a a</w>': 1, '   do g gi e</w>': 7, '   sh ee p</w>': 34, '   ci r c ling</w>': 6, '   hea th c li f fe</w>': 2, '   h ow l</w>': 10, '   pe co s</w>': 2, '   co y o tes</w>': 6, '   co y o te</w>': 2, '   ro me</w>': 87, '   pen t an g le</w>': 2, ' thir st y</w>': 2, '   in n</w>': 49, '   can d les</w>': 22, '   po in ted</w>': 28, '   wi tch cra ft</w>': 10, '   l on</w>': 2, '   chan ey</w>': 1, '   j r</w>': 7, '   uni ver sa l</w>': 19, '   stu di o s</w>': 12, '   ow n ers</w>': 19, '   te x as</w>': 94, ' po in ted</w>': 1, '   pi ke</w>': 43, '   sy m bo l</w>': 34, '   fo x</w>': 58, ' li kes</w>': 3, '   ei gh th</w>': 24, '   for e play</w>': 8, '   du ll</w>': 43, '   fa sc in a tes</w>': 3, '   r en de z v ous</w>': 22, '   star r ing</w>': 10, '   sho cked</w>': 36, '   t or ri d</w>': 1, '   lo ve ma king</w>': 5, '   ex p li c it</w>': 5, '   ha lls</w>': 8, '   v a ti can</w>': 14, '   dar ed</w>': 9, '   me di o c re</w>': 9, '   rea ch</w>': 246, '   re gar d less</w>': 18, '   for en si c</w>': 7, '   la ds</w>': 12, '   s co t land</w>': 24, '   y ard</w>': 90, '   w ound</w>': 85, '   ma th i son</w>': 1, '   lt</w>': 25, '   vi lli ers</w>': 1, '   s g t</w>': 6, '   m c man us</w>': 10, '   tra di tion al</w>': 26, '   gu in ne ss</w>': 6, '   su f fi ce</w>': 6, '   ca m par i</w>': 1, '   ni ce ly</w>': 35, '   a la m o</w>': 13, '   spi ri ts</w>': 47, '   s ou p</w>': 61, '   ri gh tly</w>': 5, '   offi ci al do m</w>': 1, '   con spi red</w>': 4, '   au to p sy</w>': 27, '   te sti mon y</w>': 63, '   for e clo se</w>': 4, '   mor t ga g es</w>': 5, '   wi d ows</w>': 6, '   in ve st ment</w>': 54, '   c ent</w>': 61, '   sa mp son</w>': 17, ' c r ack</w>': 3, '   p in</w>': 75, '   wa tch man</w>': 9, '   o sc ar</w>': 19, '   an no y</w>': 12, '   s qu a w ks</w>': 1, '   char lie</w>': 361, '   st ro ll</w>': 18, '   n er ts</w>': 1, '   de pre ssion</w>': 23, '   man vi ll e</w>': 1, '   pa y ro lls</w>': 4, '   de la y</w>': 39, '   cl ar k</w>': 107, '   di ck son</w>': 48, '   de po si t ors</w>': 3, '   i di o ti c</w>': 17, '   d ou b t ful</w>': 10, '   mer ger</w>': 27, '   sch u l t z</w>': 3, '   un se cu red</w>': 2, ' st ri ck en</w>': 5, '   de mon stra tion</w>': 21, '   co ll a ter al</w>': 12, '   pa w n bro k ers</w>': 2, '   ex a min er</w>': 5, '   li qui da te</w>': 3, '   gar age</w>': 117, '   re sig n</w>': 25, '   j am</w>': 69, '   plea ded</w>': 9, '   li qu id</w>': 32, '   p rea ch ed</w>': 3, '   w re cked</w>': 18, '   op tion</w>': 50, '   p oun ce</w>': 2, '   for t un es</w>': 13, '   re fu ses</w>': 21, '   mer ge</w>': 3, '   f li m sy</w>': 2, '   cre di t ors</w>': 5, '   a le x an der</w>': 68, '   ha mi l ton</w>': 20, ' 5 </w>': 37, '   ban king</w>': 18, '   ex act</w>': 91, '   char ac ter</w>': 138, '   do l d ru ms</w>': 1, '   h m mp f</w>': 2, '   lo ss es</w>': 14, '   cla i ms</w>': 35, '   da ze</w>': 2, '   fin la y</w>': 4, '   in spe ctor</w>': 77, '   c lu e t t</w>': 14, '   sin gs</w>': 19, ' mother</w>': 11, '   m ac h ree</w>': 2, '   hu m or</w>': 83, '   hu mm ing</w>': 5, '   e ss en ti al</w>': 13, '   be g ged</w>': 23, '   ga mb ling</w>': 36, '   th i e f</w>': 90, '   s qu a w king</w>': 4, '   e st ab li sh</w>': 16, '   a li b i</w>': 23, '   hea ven s</w>': 33, '   com in</w>': 167, '   ga mb le</w>': 40, '   d ough</w>': 53, '   w el ch ers</w>': 1, '   m ou st ac he</w>': 13, '   mi su n der sto od</w>': 26, '   com for t</w>': 51, '   ru sh ing</w>': 15, '   stu ff y</w>': 11, '   sp ee ch es</w>': 23, '   ban k ers</w>': 8, '   dea ls</w>': 45, '   sp an k</w>': 6, '   c y ri l</w>': 9, '   sp or t</w>': 79, '   re spe c ta ble</w>': 27, '   ph y ll is</w>': 18, '   clo se st</w>': 40, ' fa shi on ed</w>': 23, '   ca ps</w>': 14, '   be lls</w>': 22, '   whi st les</w>': 10, '   ex per im en ting</w>': 10, '   p hi lan der er</w>': 1, '   fa m ou s ly</w>': 6, '   bl ar ne y</w>': 2, ' cla ss</w>': 26, '   ri o t</w>': 36, '   s li gh te st</w>': 35, ' run ning</w>': 6, '   re for m ing</w>': 3, '   af fe ct</w>': 21, '   li ar</w>': 141, '   su ffer ing</w>': 71, '   b oun der</w>': 1, '   ad mi ssion</w>': 24, '   t re mb ling</w>': 12, '   wa st ing</w>': 94, '   brea th</w>': 146, '   ge e</w>': 177, '   bu t l er</w>': 22, '   un an im ous</w>': 3, '   han d k er chi e f</w>': 7, '   po st p one</w>': 12, '   att r ac tive</w>': 82, '   h o</w>': 106, '   mi l d red</w>': 5, '   g w y n n</w>': 1, '   ban k er</w>': 13, '   ear mar ks</w>': 1, '   g or ge ous</w>': 52, ' le tter</w>': 8, '   tra mp le</w>': 2, '   ca shi er</w>': 9, '   par k er</w>': 60, ' le e ds</w>': 1, '   ex chan ge</w>': 65, '   win s l ow</w>': 5, '   har r is</w>': 24, '   u mm m</w>': 16, '   sta lling</w>': 16, '   sc ha ff er</w>': 2, '   pe mb ro ke</w>': 4, '   gu ar an ty</w>': 1, '   le tter</w>': 279, '   re ser v a tions</w>': 17, '   bri da l</w>': 4, '   ber en gar i a</w>': 1, '   li ck</w>': 33, '   sa ti s f ac tion</w>': 29, '   de s k</w>': 153, '   t ou gh est</w>': 12, '   g an g st ers</w>': 7, '   p in ning</w>': 3, '   gu il ty</w>': 138, '   l ou d</w>': 121, '   v au lt</w>': 29, '   1 2 </w>': 61, ' 0 9 </w>': 2, '   c in ch</w>': 9, '   la you t</w>': 10, '   v au l ts</w>': 2, '   fi ves</w>': 4, '   di st ri bu te</w>': 5, '   te ll ers</w>': 6, '   re coun t</w>': 2, '   ver i f y</w>': 13, '   sig na ture</w>': 52, '   se cu ri ti es</w>': 2, '   ni ck el</w>': 32, '   f lu r ry</w>': 2, '   a v a il able</w>': 78, '   ar gu ing</w>': 27, '   de po si tor</w>': 1, '   re ser ve</w>': 20, '   st ea dy</w>': 80, '   lo b by</w>': 46, '   cl y de</w>': 34, '   sti cks</w>': 52, '   d y na mi te</w>': 30, '   u se less</w>': 62, '   com pe ting</w>': 7, '   con way</w>': 28, '   wi lli a ms</w>': 106, '   sh h</w>': 43, '   bu t t</w>': 119, '   ne cked</w>': 1, ' 6 </w>': 41, '   r at</w>': 101, '   th ri lled</w>': 23, '   th ru </w>': 8, '   gr and</w>': 283, '   tra i ling</w>': 4, '   sc re w y</w>': 14, '   mi ke</w>': 238, '   ha lli g an</w>': 3, '   do o le y</w>': 7, '   r he u ma ti s m</w>': 1, '   wa ll pa per</w>': 15, '   el even</w>': 134, '   bur g l ar</w>': 13, '   e mp ti ed</w>': 11, '   s ore</w>': 65, '   po ck e t bo ok</w>': 13, '   re min ds</w>': 55, '   han k</w>': 47, '   hea ling</w>': 19, '   youn ger</w>': 92, '   re ser v ation</w>': 48, '   com an c he</w>': 11, '   t ou gh er</w>': 21, '   je s se</w>': 92, '   out la w s</w>': 5, '   ri ding</w>': 99, '   go at</w>': 23, '   y ell</w>': 38, ' ja mes</w>': 10, '   ra i sing</w>': 28, '   de e ds</w>': 65, '   f ar ms</w>': 17, '   ra il ro a d</w>': 54, '   p in k er t ons</w>': 2, '   sh er i ff</w>': 205, '   k an sa s</w>': 54, '   cl ell</w>': 2, '   joh n st on</w>': 2, '   cre e d ers</w>': 1, '   hi te</w>': 1, '   z e e</w>': 10, '   pi ss ant</w>': 6, '   s mi ling</w>': 39, '   ga t ling</w>': 5, '   inter i or</w>': 21, '   te mp er</w>': 31, '   cha se</w>': 75, '   mi m ms</w>': 9, '   per ry</w>': 20, '   hi g g ins</w>': 5, '   la ll er</w>': 1, '   y an ke es</w>': 13, '   di st r ac tion</w>': 14, '   mi gh ty</w>': 62, '   v in e g ar</w>': 8, '   ro b ber i es</w>': 8, '   po st ers</w>': 18, ' ban k</w>': 3, '   offi ci al s</w>': 12, '   e sti ma te</w>': 22, ' youn ger</w>': 5, '   w oun ding</w>': 1, '   sh er ri ff</w>': 1, '   t ow n s fo l k</w>': 1, '   ne i gh bor s</w>': 59, '   supp li es</w>': 57, '   h om es</w>': 38, '   sa y in</w>': 89, '   bar re l</w>': 28, '   por k</w>': 20, '   l ard</w>': 2, '   pi g g y</w>': 6, '   z er el da</w>': 4, '   out la w</w>': 7, '   ri d ers</w>': 8, '   p ff</w>': 3, '   spi lled</w>': 13, '   whi s k y</w>': 20, '   ban da g es</w>': 10, '   fi sh y</w>': 11, '   so l di er</w>': 99, '   han ged</w>': 18, '   p ea ce time</w>': 2, '   ho g ging</w>': 2, '   pu b li ci ty</w>': 47, '   fi de li ty</w>': 6, '   de po t</w>': 10, '   st r on g bo x</w>': 1, '   ex p lo si ves</w>': 26, '   t ow n s</w>': 15, '   har ass</w>': 7, '   supp ly</w>': 56, '   fe u d</w>': 7, '   ter ri f ying</w>': 15, '   sa die</w>': 10, '   son of ab i tch es</w>': 2, '   fi gh ts</w>': 39, '   plan ting</w>': 9, '   har ve st ing</w>': 2, '   h or ses</w>': 96, '   ran g ers</w>': 31, '   di r t</w>': 91, '   sha ck</w>': 20, '   mi ss our i</w>': 18, '   g al s</w>': 12, '   b lu e co a ts</w>': 1, '   tr ack er</w>': 5, '   do z en</w>': 108, '   stra te gi ca lly</w>': 3, '   ha u l</w>': 42, '   f ar m er</w>': 51, '   s n ea kin</w>': 9, '   bu ck e ts</w>': 5, '   bu n ty</w>': 1, '   gar ri son</w>': 43, '   o c cu pi ed</w>': 19, '   ter ri t ory</w>': 45, '   can n on</w>': 24, '   ye lled</w>': 20, '   fo l k</w>': 36, '   hea p</w>': 14, '   fi re wa ter</w>': 2, '   sa lo on</w>': 21, ' y ah</w>': 12, '   ri ver</w>': 192, '   ea g le</w>': 31, '   la w man</w>': 5, '   car vi ll e</w>': 1, '   ba d man</w>': 1, '   l ou is</w>': 206, '   in ju n</w>': 4, ' u m</w>': 24, '   ro b b ing</w>': 15, '   du r ned</w>': 4, '   tu ck er</w>': 21, '   ye s ss</w>': 5, '   hea l ed</w>': 21, '   we b</w>': 19, '   ro de</w>': 35, '   ban sh e e</w>': 2, '   f oo l ed</w>': 33, '   co in ci d ence</w>': 62, '   se lls</w>': 33, '   t rea son</w>': 15, ' po or</w>': 6, '   s mar te st</w>': 20, '   si d es</w>': 47, '   co st ing</w>': 9, '   po s se</w>': 14, '   f ar m ing</w>': 8, '   s nee z ing</w>': 2, '   gen ts</w>': 8, '   s ci en ti fi c</w>': 41, '   me th od</w>': 22, '   ra ge</w>': 32, '   coun ter fe it</w>': 15, '   comp are</w>': 22, '   p in k er ton</w>': 9, '   han g man</w>': 4, '   re bu i ld</w>': 7, '   sp ry</w>': 1, '   car ri age</w>': 16, '   mi ser able</w>': 70, '   lo af ing</w>': 1, '   cour thou se</w>': 21, '   ra il</w>': 13, ' from</w>': 25, '   do c tr ine</w>': 3, '   der i ve</w>': 4, '   sp ar k le</w>': 4, '   pro me th ean</w>': 1, '   ar ts</w>': 25, '   ac a de mes</w>': 1, '   n ou ri sh</w>': 5, ' f at</w>': 9, '   ha g g ard</w>': 2, '   char m er</w>': 5, '   wh y y y y</w>': 1, '   ca su al ty</w>': 9, '   sa d dle</w>': 14, '   pr in ci p le</w>': 23, '   mi ss our a</w>': 1, '   re ck on ed</w>': 4, '   figu r ing</w>': 22, '   f lo pp y</w>': 4, '   ea ster</w>': 22, '   ha ts</w>': 25, '   m m</w>': 119, ' h m m</w>': 31, '   att r act</w>': 24, '   inter e st in</w>': 6, '   ver sion</w>': 46, '   so f ten</w>': 2, '   e d g es</w>': 10, ' every</w>': 22, '   shu sh</w>': 14, '   p ra y ers</w>': 33, '   ven ge ful</w>': 6, '   s mi ting</w>': 1, '   mo od</w>': 105, '   or ch ard</w>': 7, '   ea st er n ers</w>': 1, '   ten ne s see</w>': 15, '   ar re sted</w>': 98, '   shu cks</w>': 9, '   ac c ent</w>': 26, '   di me</w>': 68, '   no v el s</w>': 15, '   sa un ter ed</w>': 1, '   sp u rs</w>': 2, '   j an g ling</w>': 1, '   f lo cked</w>': 1, '   can di ed</w>': 6, '   app le</w>': 59, ' b la z ing</w>': 1, '   han d some</w>': 74, '   char i s ma ti c</w>': 5, '   ex ci te ment</w>': 34, '   i ma g in ed</w>': 33, '   b oun ty</w>': 24, '   h un t ers</w>': 22, '   la w men</w>': 3, '   a sh es</w>': 16, '   la w s</w>': 67, '   th ad de us</w>': 7, '   ra ins</w>': 23, '   ki ss in</w>': 3, '   qu o ting</w>': 14, '   po e ms</w>': 13, '   sp ar k ling</w>': 7, '   ro cks</w>': 68, '   sh in y</w>': 13, '   bo ss y</w>': 3, '   gre w</w>': 94, '   ar r</w>': 1, '   je w el s</w>': 38, '   mi ss y</w>': 17, '   un fa ir</w>': 37, '   re s cu ed</w>': 10, '   t ea se</w>': 17, '   n uh</w>': 4, '   mm m m</w>': 42, '   a h h</w>': 70, '   co ck y</w>': 15, '   o ver see</w>': 5, '   bi ll in gs</w>': 5, '   ca ttle</w>': 43, '   a m bu sh</w>': 14, '   t or re ll</w>': 1, '   hou ses</w>': 65, '   si mi l ar</w>': 45, '   sy m pa th i z ers</w>': 2, '   com m uni on</w>': 4, '   re pa ir</w>': 28, '   lea k y</w>': 6, '   ro of</w>': 103, '   clo th</w>': 15, '   de pen ding</w>': 26, '   char i ta ble</w>': 9, ' je s se</w>': 1, '   w oo d son</w>': 1, '   un u su al</w>': 94, '   con gre ga tion</w>': 6, '   re st ing</w>': 33, '   l y l a</w>': 5, '   d ever e u x</w>': 2, '   da ted</w>': 24, '   gen t le man ly</w>': 2, '   sh er ry</w>': 23, '   si p</w>': 20, '   d on ate</w>': 5, '   f ar m ers</w>': 27, '   ch u r ch es</w>': 3, '   sh ar e c ro pp ers</w>': 1, '   ma d do x</w>': 5, '   f ac ing</w>': 29, '   sh ar p sh oo ter</w>': 3, '   al an</w>': 112, '   mo ti v ate</w>': 2, '   po pu l ace</w>': 3, '   re lo ca te</w>': 2, '   con de m ned</w>': 16, '   men ace</w>': 15, '   j en k ins</w>': 14, '   pre ci se</w>': 18, '   bl ow ing</w>': 34, '   ra il way</w>': 7, '   co ins</w>': 11, '   cu r r en cy</w>': 14, '   ex chan ged</w>': 7, '   ra i ded</w>': 7, '   po d un ks</w>': 1, '   cha ll en ge</w>': 46, '   ri gh te ou s ne ss</w>': 4, '   pa t ro ls</w>': 6, '   to o l</w>': 32, '   vo id</w>': 12, '   imp uni ty</w>': 4, '   im pe di ment</w>': 7, '   re co ver s</w>': 3, '   th u gs</w>': 11, '   ga ll ows</w>': 7, '   in spi red</w>': 17, '   re si stan ce</w>': 29, '   re d ou b ta ble</w>': 2, '   wh or es</w>': 40, '   ci g ars</w>': 29, '   ne ce ss ar i ly</w>': 42, '   god for sa k en</w>': 5, '   ev i ct</w>': 1, '   mu d ho les</w>': 1, '   ven ge an ce</w>': 17, '   tra ve l</w>': 135, '   ci r c les</w>': 25, '   vi c in i ty</w>': 9, '   de m and</w>': 45, '   op en ly</w>': 15, '   ri ver be ds</w>': 1, '   ca ves</w>': 11, '   op er a tes</w>': 7, '   c un ning</w>': 16, '   u ps</w>': 11, '   no o ses</w>': 1, '   sa bo ta ge</w>': 23, '   t ac ti c s</w>': 13, '   dis ci pl in ed</w>': 2, ' tra in ed</w>': 6, '   ch ess</w>': 31, '   pa tt er n s</w>': 35, '   th a x ton</w>': 1, '   fe ar ing</w>': 4, '   p ac i fi c</w>': 30, '   dar ing</w>': 10, '   ad ver s ar i es</w>': 2, '   cour t ne y</w>': 12, '   lu is</w>': 14, ' pa tri ck</w>': 1, '   s mo ked</w>': 27, '   ta d</w>': 9, '   ma u r a</w>': 10, ' lu is</w>': 1, '   ca fe</w>': 32, '   a u</w>': 19, '   than k s gi ving</w>': 19, '   pro mo tion</w>': 49, '   in ch</w>': 56, '   e j ac u late</w>': 4, '   ba te man</w>': 26, '   li th i u m</w>': 4, '   re ce p t ac le</w>': 6, '   de spi ca ble</w>': 8, '   tw it</w>': 9, '   pe an u t</w>': 38, '   bu tter</w>': 58, '   ma sh ed</w>': 5, '   s qu as h</w>': 11, ' pla y ful</w>': 1, '   my st er i ous</w>': 45, '   s na pp er</w>': 1, '   vi o le ts</w>': 2, '   d or si a</w>': 9, '   sin k</w>': 41, '   le ans</w>': 9, ' chi l dr en</w>': 1, '   di et</w>': 47, '   ca f fe ine</w>': 14, '   s lu mp</w>': 4, '   d on a ld</w>': 39, '   tru mp</w>': 5, '   ar d en</w>': 1, '   re la x ing</w>': 14, '   po tt er y</w>': 2, '   ba m</w>': 14, '   si l ver</w>': 80, '   fa bu l ous</w>': 51, '   tu mb ling</w>': 6, '   di ck we ed</w>': 3, '   ar i z on a</w>': 30, ' bu sy</w>': 6, '   ni c er</w>': 32, '   la w y ers</w>': 83, '   sur ger y</w>': 57, '   chri sti e</w>': 12, '   hu t</w>': 13, ' pa u l</w>': 6, '   vi de o ta p es</w>': 8, '   as se ssed</w>': 1, ' er ra ti c</w>': 2, '   hu man i ty</w>': 30, '   ev el y n</w>': 49, '   in hu man</w>': 4, '   li p</w>': 29, '   h om i ci da l</w>': 16, '   fu l fi ll</w>': 9, ' lo st</w>': 8, '   com mi t ment</w>': 30, '   ha ir l ine</w>': 1, '   re ce ding</w>': 2, '   min o x i di l</w>': 1, ' looking</w>': 64, '   v an d en</w>': 1, '   st as h</w>': 13, ' want</w>': 25, ' f it</w>': 2, '   ev i an</w>': 2, '   spi ked</w>': 3, '   mar z i p an</w>': 4, '   ten ts</w>': 10, '   hund re ds</w>': 75, '   ro ses</w>': 23, '   p ho to gra ph ers</w>': 4, '   an ni e</w>': 130, '   le i bo vi t z</w>': 2, '   con tri bu te</w>': 9, '   v an</w>': 190, '   pa tt en</w>': 6, '   ed</w>': 204, '   ge in</w>': 2, '   ma i t re</w>': 5, '   can al</w>': 10, '   l ou sy</w>': 88, '   c q </w>': 4, '   la u ri e</w>': 44, '   k en ne dy</w>': 96, '   har d body</w>': 1, '   e g g sh ell</w>': 1, '   ro ma li an</w>': 1, '   le tt er ing</w>': 3, '   si li an</w>': 1, '   co l or ing</w>': 4, '   pr in t ers</w>': 3, '   s w ea ter</w>': 34, '   fa shi on</w>': 44, '   su b t le</w>': 27, '   m c der mo t t</w>': 10, '   t ans</w>': 1, '   ow en</w>': 33, ' place</w>': 6, ' didn</w>': 13, '   car n es</w>': 4, '   cle ar er</w>': 10, '   com pre h en ding</w>': 2, '   t or tu red</w>': 16, '   do z en s</w>': 14, ' m ou th</w>': 11, '   sp in e less</w>': 6, '   li gh tw e ight</w>': 7, ' o ther wi se</w>': 4, '   hi l ar i ous</w>': 14, '   har old</w>': 159, '   c y n th i a</w>': 43, '   ja p an e se</w>': 57, ' 9 0</w>': 8, '   app ro ac h es</w>': 15, '   car ru th ers</w>': 9, ' re turn</w>': 4, '   com mes</w>': 1, '   gar c on</w>': 1, '   g w en do l y n</w>': 2, '   i chi b an</w>': 1, '   n ell</w>': 32, '   su f fi ci ent</w>': 17, '   j ack et</w>': 127, '   v al en t in o</w>': 5, '   cou ture</w>': 2, '   im pre ssi ve</w>': 53, '   ca mp er</w>': 17, '   ro ck in</w>': 8, ' ro ll in</w>': 1, '   re ha b</w>': 19, '   won d ers</w>': 16, '   uni ce f</w>': 1, '   z any</w>': 3, '   mor g an</w>': 42, '   stan le y</w>': 49, '   di sa pp ear ing</w>': 19, ' mer ch ant</w>': 2, ' lea ving</w>': 6, '   gra m</w>': 7, '   car on</w>': 1, '   g or b ac h ev </w>': 1, '   pa le</w>': 35, '   ni m bu s</w>': 4, '   a par the id</w>': 2, '   ter r ori s m</w>': 16, '   h un ger</w>': 24, '   a bu sing</w>': 4, '   sh el ter</w>': 47, '   o pp ose</w>': 7, '   r ac i al</w>': 12, '   di sc ri min ation</w>': 4, '   pro mo ting</w>': 6, '   ab or tion</w>': 28, '   f re e do m</w>': 99, '   s r i</w>': 1, '   lan k a</w>': 1, '   sp r in k le</w>': 2, ' br an</w>': 1, '   mi lli gra m</w>': 2, '   j ee z</w>': 70, '   d y sle x i a</w>': 3, '   in fe c ted</w>': 25, ' al z he im er</w>': 1, '   mu s cu l ar</w>': 3, '   d y st ro ph y</w>': 1, '   he mo p hi li a</w>': 1, '   le u ke mi a</w>': 3, '   di a be tes</w>': 8, '   pu ff y</w>': 13, '   bo ok ed</w>': 24, '   s ea ted</w>': 9, '   ha l ber st am</w>': 6, '   mar c us</w>': 58, '   u m m</w>': 42, '   s oun ding</w>': 20, ' say</w>': 24, '   cra i g</w>': 76, '   ti m</w>': 81, '   f re der i ck</w>': 39, '   b en n et</w>': 2, '   r ust</w>': 16, ' 2 1 </w>': 14, '   b ru i sed</w>': 9, '   ten den cy</w>': 9, '   un a v a il able</w>': 2, ' cont ro l</w>': 12, '   me an in g ful</w>': 9, ' have</w>': 78, '   car ton</w>': 7, ' need</w>': 19, '   ta p ing</w>': 14, '   cu p bo ard</w>': 9, '   co lli e</w>': 3, '   la ssi e</w>': 3, '   be gu n</w>': 43, '   d ev e lo p ing</w>': 18, ' gr ow ing</w>': 1, ' fu l fi lled</w>': 1, '   n ea tly</w>': 7, '   a x</w>': 10, '   tw ine</w>': 3, '   l o</w>': 23, '   possi bi li ti es</w>': 39, ' un sure</w>': 2, '   cont ro lling</w>': 16, '   wi ll p ower</w>': 2, '   thin ner</w>': 3, ' better</w>': 8, '   sor b et</w>': 2, ' fine</w>': 20, '   so oo o</w>': 1, ' d or si a</w>': 1, '   ac comp any</w>': 19, '   a sh tra y</w>': 8, '   ki mb all</w>': 9, '   whi s ks</w>': 2, '   ye</w>': 81, ' es</w>': 8, '   j e</w>': 15, ' an</w>': 49, '   s ki r t</w>': 29, '   ran so m</w>': 34, '   do ll</w>': 88, '   per ri er</w>': 6, '   ro man ti c</w>': 98, '   ca mo ls</w>': 1, '   cra y ons</w>': 4, '   can ce l</w>': 50, ' sp en c er</w>': 2, '   f lu ti es</w>': 2, '   pi er</w>': 18, '   ri ck y</w>': 77, '   h en dri cks</w>': 11, '   can c el ing</w>': 7, '   a er o bi c s</w>': 3, '   da i sy</w>': 17, '   vi b es</w>': 10, '   mer g ers</w>': 4, '   ac qui si tions</w>': 5, '   w el t</w>': 2, '   ma t in</w>': 3, '   les</w>': 15, '   mi ser ab les</w>': 1, ' chri st</w>': 16, '   vi c t ori a</w>': 43, '   hu ber t</w>': 10, '   u p town</w>': 26, ' bo sc o</w>': 1, '   do ve</w>': 13, ' h er sh ey</w>': 1, '   sy r up</w>': 7, '   sta ins</w>': 8, '   whi t ne y</w>': 15, '   c d</w>': 17, '   de but</w>': 7, '   l p</w>': 2, '   sin g les</w>': 9, '   lea ps</w>': 2, '   b oun dar i es</w>': 15, '   ver sa ti le</w>': 1, ' thou gh</w>': 5, '   ma in ly</w>': 20, '   ja z z</w>': 53, '   al bu m</w>': 28, '   le w d</w>': 5, '   dis ea se</w>': 113, ' f ree</w>': 17, '   ha l c y on</w>': 3, '   pi er ce</w>': 45, '   nor man</w>': 149, '   si c</w>': 3, '   ta st es</w>': 49, '   p ea k</w>': 11, ' fran ce</w>': 1, ' cou s in</w>': 2, '   bl ow job</w>': 4, '   t an ning</w>': 2, '   nu t so</w>': 3, '   si gh t see ing</w>': 4, '   at lan t is</w>': 6, '   di b ble</w>': 1, '   ne w man</w>': 6, '   bu t ner</w>': 1, '   den i ed</w>': 29, '   ver i fi ed</w>': 7, '   se cre t ary</w>': 125, ' mar c us</w>': 1, '   ha sh</w>': 13, '   br ow n s</w>': 6, '   or der ing</w>': 22, ' l un ch</w>': 2, ' hu ey</w>': 1, '   sin g ers</w>': 3, '   sor ted</w>': 11, '   in for ma tive</w>': 2, '   s ca tt er ed</w>': 16, ' gone</w>': 6, '   af ri c a</w>': 61, '   or so</w>': 2, '   pe tal u ma</w>': 1, ' last</w>': 16, '   te ll er</w>': 20, '   tw en ti e th</w>': 17, '   ti mo th y</w>': 6, '   en v y</w>': 38, ' comp le x</w>': 1, ' people</w>': 24, '   re la x ed</w>': 21, '   fin lan di a</w>': 2, '   in for ma l</w>': 6, '   con ver sa tions</w>': 14, '   sto l i</w>': 1, ' for ma l</w>': 3, ' wh en ever</w>': 4, '   k en d all</w>': 31, ' uh</w>': 154, '   ca mp be ll</w>': 20, '   1 0 0</w>': 55, '   o c cu rs</w>': 10, '   s ea s ons</w>': 17, '   hu x ta ble</w>': 3, '   e er i e</w>': 6, '   s wa ll ows</w>': 13, ' di sa pp e ar</w>': 2, '   t y pi cal</w>': 34, '   ba si ca lly</w>': 67, '   to i le tri es</w>': 1, '   con su l ted</w>': 4, '   p sy chi c</w>': 55, '   y a le</w>': 11, '   y i kes</w>': 6, '   la me</w>': 36, '   j er se y</w>': 52, '   sto ck bro k er</w>': 6, '   mu r der ing</w>': 12, '   chi can o</w>': 3, '   per for m ing</w>': 21, '   v oo do o</w>': 19, '   ri tu al s</w>': 11, '   o c cu l ti s m</w>': 1, '   sa t an</w>': 47, '   wor ship</w>': 27, '   ver i fi ca tion</w>': 3, '   st e ph en</w>': 44, '   hu gh es</w>': 11, '   re sta u ran t</w>': 101, '   mi st oo k</w>': 8, '   a in s wor th</w>': 1, '   h mm m</w>': 86, ' has</w>': 21, '   1 9 6 9 </w>': 7, '   sa int</w>': 56, '   y ac h t</w>': 40, '   ne w por t</w>': 7, '   en do ch ine</w>': 1, '   com ell</w>': 1, ' han g</w>': 3, ' ex a min ed</w>': 1, '   l ed</w>': 72, ' there</w>': 134, ' y a le</w>': 3, ' n er v ous</w>': 3, ' first</w>': 20, '   gar den s</w>': 7, '   o c to b er</w>': 24, '   li me</w>': 31, '   under stan d able</w>': 11, ' pa ge</w>': 13, '   pe ll e</w>': 1, '   pe ll e gr in o</w>': 2, '   mer e di th</w>': 16, '   bar ge</w>': 11, '   mu lling</w>': 1, ' ex chan ging</w>': 1, '   ru mor s</w>': 35, '   bo one</w>': 36, '   pi ck en s</w>': 1, '   re e k</w>': 5, '   st en ch</w>': 7, '   sh h h h</w>': 33, '   in si der</w>': 4, ' were</w>': 20, '   ra in co at</w>': 7, '   ch ow</w>': 22, '   wa ve</w>': 62, '   1 9 8 3 </w>': 3, '   com mer ci ally</w>': 3, '   ar ti sti ca lly</w>': 2, '   hu ey</w>': 21, '   le w is</w>': 40, '   pre vi e w</w>': 2, '   bar ne ys</w>': 1, '   ca ta lo gu e</w>': 8, '   ab so lu t</w>': 1, '   rea son able</w>': 75, '   h r</w>': 1, '   ni gh t ca p</w>': 9, '   a me x</w>': 2, '   s la ps</w>': 5, '   mar t in i</w>': 50, '   ce ce li a</w>': 2, '   ro th sc hi ld</w>': 2, '   ori g in ally</w>': 23, '   fi sh er</w>': 24, '   om i tt ed</w>': 2, '   lo in</w>': 1, '   je llo</w>': 6, '   ar u gu l a</w>': 1, '   ce ci li a</w>': 1, ' nor ma l</w>': 9, ' und an ger ous</w>': 1, '   con s ci ou s ly</w>': 5, '   cre di ble</w>': 4, '   en cou ra ge</w>': 20, ' gonna</w>': 3, '   f an a ti c</w>': 8, '   re f re sh men ts</w>': 2, '   l ac ed</w>': 3, ' al ter ing</w>': 1, ' wanna</w>': 5, '   sa v or</w>': 2, '   ch op sti cks</w>': 5, '   mi l o</w>': 34, ' ma il</w>': 15, '   c lu e less</w>': 6, '   s k y wi re</w>': 16, '   g ary</w>': 61, '   com i x</w>': 2, '   mu se u m</w>': 85, '   pa in ter</w>': 13, '   e mp lo ye es</w>': 26, '   na u gh ty</w>': 23, '   s ea ttle</w>': 43, '   stan for d</w>': 7, ' guess</w>': 14, ' differ ent</w>': 5, '   2 1 </w>': 27, '   l y le</w>': 8, '   bar ton</w>': 45, '   un fini sh ed</w>': 14, '   bro ad ca st</w>': 21, '   stu di o</w>': 81, ' wh ad do</w>': 1, '   par king</w>': 71, '   g ee ks</w>': 7, '   do j </w>': 4, '   a gen ts</w>': 63, '   sur ve i ll an ce</w>': 38, '   s wi pe</w>': 3, '   di sh es</w>': 37, '   po st p on ing</w>': 2, ' gotta</w>': 20, ' so l ving</w>': 1, '   sc ar ing</w>': 31, '   p c</w>': 3, '   ma in f ra me</w>': 22, ' con ta in ed</w>': 2, '   ha ck</w>': 35, '   pro gra ms</w>': 29, '   out po st</w>': 21, '   ne l son</w>': 18, '   a ir ti ght</w>': 5, '   ex t re me</w>': 40, '   pu mp ed</w>': 10, '   se same</w>': 4, '   se ed</w>': 32, '   ba s is</w>': 46, '   coun se l or</w>': 53, '   st ow ed</w>': 3, ' fi x es</w>': 1, ' did</w>': 59, '   im pl ying</w>': 22, '   pro b</w>': 30, ' ly</w>': 26, '   bri an</w>': 75, '   a lot</w>': 12, '   ca mp us</w>': 19, '   l ev el ed</w>': 3, '   1 </w>': 103, ' 0</w>': 15, '   bo y d</w>': 34, '   bu lly</w>': 17, '   l ar ry</w>': 121, '   cl one</w>': 6, '   en g in e er</w>': 36, '   in f er i or</w>': 9, ' i dn</w>': 3, '   car ed</w>': 54, '   c li ch</w>': 5, '   pa ss es</w>': 37, '   ne o</w>': 41, ' na z is</w>': 1, '   gen er ally</w>': 39, '   r ac i ally</w>': 2, '   mo ti v a ted</w>': 8, '   ch in e se</w>': 117, '   si li c on</w>': 11, '   l ac y</w>': 8, '   be g g ar</w>': 2, '   di sa d v an ta ge</w>': 5, '   ex per ts</w>': 25, '   te ch no lo gi es</w>': 1, '   po se</w>': 28, '   3 2 </w>': 10, '   bu i ck</w>': 10, '   supp li c ant</w>': 1, '   pro spe c ts</w>': 17, '   co or din a tes</w>': 36, '   inter f ac ed</w>': 1, ' e mp t</w>': 1, '   y o g a</w>': 6, '   be ta</w>': 7, '   de m o</w>': 10, ' for got</w>': 3, ' la ter</w>': 6, ' hou ght</w>': 2, '   wh ack</w>': 27, ' job</w>': 12, '   gu est</w>': 122, '   di gi tal</w>': 22, '   ro b in</w>': 36, '   ho od</w>': 36, '   sti g ma</w>': 3, '   6 0</w>': 29, '   bu ys</w>': 35, '   s l ack</w>': 16, '   ban k ro lling</w>': 1, '   imp li ca te</w>': 6, '   v ani sh ed</w>': 25, '   in di c t ment</w>': 7, '   con g lo mer a tes</w>': 3, '   lin ed</w>': 25, '   fin an ce</w>': 29, '   pre mi u m</w>': 11, ' hou se man</w>': 1, ' gu ard</w>': 3, '   se cre cy</w>': 12, '   com pe ti tion</w>': 36, ' couldn</w>': 8, '   dan ny</w>': 128, '   s wi tch es</w>': 10, '   wi p ed</w>': 30, '   fi l er</w>': 3, ' c lu e less</w>': 1, '   han d wri tt en</w>': 1, ' de ar</w>': 23, '   en jo y ed</w>': 60, ' or ry</w>': 2, '   con ni ve</w>': 1, '   mo cked</w>': 7, '   cu l ti v ate</w>': 5, '   con ni ving</w>': 6, ' wor se</w>': 3, '   pre f er red</w>': 5, '   g ee k</w>': 22, '   com pu l si ve</w>': 16, '   ga mb l er</w>': 20, '   app li ed</w>': 14, ' sy st em</w>': 3, ' lo s ers</w>': 1, ' read</w>': 7, '   sh ar ks</w>': 39, '   me chan i s m</w>': 17, '   han d l er</w>': 1, '   3 4 </w>': 9, '   hel ps</w>': 67, ' li mi ts</w>': 6, '   di sc lo sure</w>': 4, '   f la gs</w>': 10, '   s ca les</w>': 15, '   com pe ti t ors</w>': 6, '   b in ary</w>': 9, '   ob se ssi ve</w>': 11, '   tra it</w>': 3, '   sc ru t in y</w>': 2, '   stra te gi c</w>': 10, '   sor did</w>': 14, '   comp ani es</w>': 30, '   wor k able</w>': 2, '   con ver g ence</w>': 5, '   se ar ch ing</w>': 41, '   ve tt ed</w>': 2, '   s kill</w>': 33, ' hi d d en</w>': 4, '   lea f</w>': 13, '   p hi list ine</w>': 3, '   re pro du c tions</w>': 2, ' h ink</w>': 5, '   4 2 </w>': 11, '   c y b er</w>': 8, ' years</w>': 13, '   le g ac y</w>': 12, '   mar ke ting</w>': 5, '   se min ars</w>': 6, '   ca ble</w>': 62, ' ori en ted</w>': 1, ' be</w>': 52, '   bi ts</w>': 36, '   sa te lli te</w>': 41, '   mu l ti pla t for ms</w>': 1, '   om ni pla t for ms</w>': 1, '   in c lu ding</w>': 83, '   e mer g es</w>': 3, ' hi gh way</w>': 2, '   ph on es</w>': 40, '   cra m</w>': 5, '   s k y</w>': 99, '   re la y</w>': 12, '   sh own</w>': 58, '   la un ch ed</w>': 16, '   s d i</w>': 1, '   rea g an</w>': 12, '   te ch no lo g y</w>': 48, '   4 2 6 </w>': 1, '   st ri d es</w>': 4, ' knows</w>': 5, '   al ter ing</w>': 7, ' 1 2 </w>': 9, '   lon gi tu de</w>': 3, '   1 0 9 </w>': 1, '   0 6 </w>': 1, ' 2 </w>': 92, '   se tt in gs</w>': 6, '   co or din ate</w>': 7, '   af fe c ted</w>': 18, ' t art</w>': 1, '   a p p</w>': 2, '   e le g ant</w>': 18, '   na i ve</w>': 36, '   e mp lo y ment</w>': 20, '   en d ow men ts</w>': 2, '   pre ten se</w>': 4, '   sch oo ls</w>': 29, '   f ac u l ti es</w>': 3, '   con ne c ted</w>': 50, '   p hi l</w>': 79, '   b ack s la sh</w>': 1, '   de l ber t</w>': 7, '   ir re gu l ar i ty</w>': 1, '   en try</w>': 28, '   au th ori z ed</w>': 19, '   fri g ging</w>': 5, '   men s a</w>': 1, '   pr in t out</w>': 3, '   1 4 </w>': 34, '   fro z en</w>': 64, '   g li tch</w>': 12, '   la ti tu de</w>': 2, '   4 7 </w>': 7, '   lo g ged</w>': 8, '   lo ca tion</w>': 64, '   la p to p</w>': 8, '   or din ary</w>': 74, ' ni gh ter</w>': 2, '   ex pla in ing</w>': 18, '   ex cer p ts</w>': 1, '   ch oo sing</w>': 9, '   de sc ri bed</w>': 26, '   sh ro t</w>': 3, '   no ti ces</w>': 17, '   en ter ed</w>': 35, '   ac ce ssed</w>': 11, '   bl under ing</w>': 5, '   re vi e w ing</w>': 5, '   rea d ou ts</w>': 1, '   ju i c ed</w>': 4, '   a g let</w>': 2, '   pa s sa ble</w>': 3, ' ow ed</w>': 1, '   con su med</w>': 10, '   lu cra tive</w>': 7, '   a part men ts</w>': 12, '   s no op ing</w>': 10, '   ter min ation</w>': 8, '   car e less</w>': 20, '   stan d ard</w>': 73, '   gu i</w>': 2, '   gra p hi c</w>': 8, '   inter face</w>': 12, '   au di o</w>': 4, '   lo go s</w>': 1, '   me g a</w>': 5, ' ne t work</w>': 2, ' war ner</w>': 1, ' to p</w>': 5, '   c n n</w>': 7, ' ge</w>': 1, '   jo ins</w>': 7, '   ven ture</w>': 26, '   n b c</w>': 6, ' dis ne y</w>': 1, '   ab c</w>': 7, ' out po st</w>': 1, '   ne w sc or p</w>': 1, '   su b si di ary</w>': 2, ' mo le</w>': 3, ' c b s</w>': 1, '   part n er ed</w>': 1, '   2 0 0 1 </w>': 1, ' 6 0</w>': 31, '   li br ar i es</w>': 5, '   ar chi ves</w>': 7, ' re f er ence</w>': 2, '   ne w s ma ga z ine</w>': 1, '   di sin for ma tion</w>': 4, '   si tes</w>': 6, '   h or n s</w>': 18, '   c ro p</w>': 27, '   sa d da m</w>': 13, '   hu s se in</w>': 4, '   lo gi cal</w>': 41, '   un cl ean</w>': 2, ' mi l o</w>': 1, '   mi mi c s</w>': 2, '   al co ho li c</w>': 20, '   bl ack out</w>': 6, '   reme mb er ing</w>': 31, ' act</w>': 6, '   f ra me</w>': 54, '   se cre tive</w>': 5, '   un de te c ta ble</w>': 4, ' ta il or ed</w>': 1, ' sen se</w>': 6, '   sc en ar i o s</w>': 8, '   u p lo a d</w>': 1, '   ph ar m ac y</w>': 8, '   f ru i tion</w>': 2, '   s nu ck</w>': 15, '   wi sh ing</w>': 11, ' more</w>': 29, ' w ro te</w>': 3, '   ce o</w>': 7, '   in spi ra tion</w>': 25, '   pu sh ed</w>': 53, '   app s</w>': 2, '   pla t for ms</w>': 1, '   ran ging</w>': 1, '   n ar r ow est</w>': 1, '   ban d wi d th</w>': 1, '   gra p hi cal</w>': 2, '   inter f ac es</w>': 2, ' po sed</w>': 24, '   e s cap ing</w>': 7, ' something</w>': 36, '   pe ts</w>': 12, '   p et</w>': 51, '   re p</w>': 22, '   gir l friends</w>': 29, '   mo on i e</w>': 1, '   vi e t na me se</w>': 18, '   g ro c ers</w>': 1, '   su pre m ac i sts</w>': 1, '   r en ted</w>': 27, '   su n n y v a le</w>': 2, '   ca pi ta list s</w>': 2, '   star t up</w>': 1, '   sur vi ve</w>': 119, '   be ts</w>': 22, '   fi b er</w>': 7, '   op ti c s</w>': 4, '   be tting</w>': 24, '   r are</w>': 75, ' mar ried</w>': 7, ' po sti es</w>': 1, ' inter face</w>': 1, '   pro gra mm er</w>': 4, '   rea ction</w>': 61, '   mi c r o</w>': 4, ' man a ged</w>': 1, ' man age</w>': 2, '   s na pp les</w>': 1, '   al p ha be ti cal</w>': 1, '   d ow n lo a d</w>': 9, '   cor ey</w>': 12, '   jo se</w>': 10, '   sp ea king</w>': 151, '   f rea k a z o id</w>': 1, '   cu lt</w>': 22, '   o bi sp o</w>': 2, '   i p o</w>': 6, '   de pt</w>': 4, '   v a por w are</w>': 1, ' sa le</w>': 2, '   gra te ful</w>': 93, '   mu cking</w>': 2, '   di ver sion</w>': 11, '   ta il ga ting</w>': 2, '   g l ori fi ed</w>': 6, '   ir on y</w>': 16, '   wor d s wor th</w>': 2, '   pen ned</w>': 2, '   ge m</w>': 5, ' pre ss</w>': 2, '   cho o</w>': 4, ' cho o</w>': 2, '   po em</w>': 44, '   vi r tu c on</w>': 7, ' il k</w>': 1, '   le mon a de</w>': 11, '   fu dge</w>': 6, '   to o t si e</w>': 3, ' par don</w>': 3, '   ru de</w>': 59, '   po pped</w>': 17, '   re la x ation</w>': 5, '   c un n in g ha m</w>': 7, '   sa k</w>': 1, '   sa k i</w>': 1, '   ja p an</w>': 35, '   g ro o v y</w>': 16, '   min ing</w>': 17, '   su b ter ran ean</w>': 4, '   dri lls</w>': 2, '   ri t chi e</w>': 3, '   fa g in a</w>': 2, '   a lo tta</w>': 7, '   ear n in gs</w>': 5, '   s ma sh ing</w>': 20, ' fif te en</w>': 12, '   to ma to</w>': 12, '   au st in</w>': 54, '   an dy</w>': 194, '   war ho l</w>': 4, '   ba si l</w>': 17, '   n in e ti es</w>': 9, '   ra te</w>': 93, '   k en sin g ton</w>': 10, '   v an e ss a</w>': 44, '   su ms</w>': 6, '   ex po si tion</w>': 11, '   un s ca th ed</w>': 1, '   la ir</w>': 5, '   man ni sh</w>': 1, '   mi ssi le</w>': 24, '   fe ar ed</w>': 18, '   pu z z le</w>': 20, '   un co ver ed</w>': 4, '   v u l can</w>': 48, '   war head</w>': 11, '   ci r c us</w>': 36, '   no ma ds</w>': 2, '   ca b ba ge</w>': 7, '   car ni es</w>': 1, '   j ac ked</w>': 8, '   k re pl ac hi st an</w>': 2, '   th rea ten ing</w>': 47, '   un f re e z ing</w>': 4, '   f la tu l ence</w>': 2, '   con fir m ing</w>': 6, '   ve g as</w>': 146, '   ex e cu ti ves</w>': 10, ' ca sin o</w>': 1, ' a g ent</w>': 5, '   de di ca ted</w>': 21, '   sha gs</w>': 1, '   min x</w>': 2, '   in ner</w>': 36, '   mon o lo gu e</w>': 1, '   re ti red</w>': 52, '   ye a</w>': 14, '   ca pi ta li s m</w>': 3, '   ca pi ta li st</w>': 4, '   com ra d es</w>': 25, '   gi l m our</w>': 1, '   bor sch ev s k y</w>': 1, '   vo lu me</w>': 21, '   sh ou ting</w>': 19, '   min i st ry</w>': 30, '   1 9 9 7 </w>': 5, '   c r y o gen i ca lly</w>': 1, '   z e de l</w>': 1, '   e de l</w>': 1, '   ba ad en</w>': 2, '   p sy che de li c</w>': 4, '   pu ss y c at</w>': 4, '   s win ger</w>': 3, '   pi ca di lly</w>': 1, '   s win ging</w>': 22, '   inter na tion al</w>': 56, '   1 9 6 7 </w>': 8, '   se x u ally</w>': 17, '   mi ck</w>': 27, '   ja g ger</w>': 2, '   de st ru c t ac on</w>': 2, '   fu ti le</w>': 8, '   o d ds</w>': 86, '   sur vi v al</w>': 36, '   2 3 </w>': 28, ' 7 6 3 </w>': 1, ' 2 7 3 </w>': 1, '   chi lly</w>': 8, '   a ging</w>': 15, '   hi p ster</w>': 2, '   s win g ers</w>': 2, '   re be lling</w>': 1, '   u p ti ght</w>': 21, '   s qu ar es</w>': 9, '   d om in ation</w>': 8, '   con se qu en ces</w>': 41, '   li ber ation</w>': 3, '   re ma in ed</w>': 9, '   be l gi an</w>': 7, '   th in</w>': 78, '   o ver we ight</w>': 2, '   ir on i c</w>': 19, '   di st r ust</w>': 6, '   me lt</w>': 21, '   ma g ma</w>': 1, '   un ne ce ss ar i ly</w>': 2, ' mo ving</w>': 2, '   di pp ing</w>': 4, '   cu r tain</w>': 21, '   g ro cking</w>': 1, '   r en der ed</w>': 5, '   f ab </w>': 4, '   ti ck</w>': 25, '   to gs</w>': 1, '   li ke wi se</w>': 9, '   any ways</w>': 27, ' woman</w>': 14, '   sha g ging</w>': 5, '   me chan i cal</w>': 14, '   pi st on</w>': 1, '   an ni hi la tion</w>': 7, '   hea d st rong</w>': 4, '   f lo ss</w>': 6, '   le dge</w>': 11, '   si x ti es</w>': 19, '   sa d ly</w>': 15, '   so il</w>': 33, '   re gr ou p</w>': 3, '   h or ny</w>': 28, '   te ch ni qu e</w>': 19, '   in di a</w>': 45, '   gu ru </w>': 2, '   sha st r i</w>': 1, '   my st er i ou s ly</w>': 5, '   ha ll mar ks</w>': 1, '   sy p hi l is</w>': 1, '   ba d g es</w>': 9, '   wi lli e</w>': 64, '   ho o t chi e</w>': 1, ' k oo ch</w>': 1, ' u ps</w>': 23, '   tri pp y</w>': 4, '   un fro z en</w>': 2, '   ran ci d</w>': 2, '   so ds</w>': 2, '   li ber ace</w>': 5, '   u ph ea v al</w>': 3, '   ber l in</w>': 56, '   pri me</w>': 40, '   min i ster</w>': 56, '   ab o li sh ment</w>': 1, '   ta pe st ry</w>': 2, '   st ru m</w>': 1, '   un d</w>': 8, '   d ran g</w>': 4, '   lu v </w>': 8, '   sha g</w>': 10, '   fi l th y</w>': 59, '   be g g ars</w>': 11, '   por t</w>': 61, '   sa il ors</w>': 13, '   con d om s</w>': 10, ' mi lli me ter</w>': 1, '   sha g ged</w>': 3, '   ro tt en</w>': 59, '   ne u ro ti c</w>': 12, '   sha ke</w>': 82, '   re la tion shi ps</w>': 27, '   j ea l ou sy</w>': 16, '   i s su es</w>': 46, '   in se cu ri ti es</w>': 2, '   ir ra tion al</w>': 10, '   un fo cu sed</w>': 1, '   mu m</w>': 28, '   t rea ted</w>': 69, '   sa il ed</w>': 13, '   sen si ble</w>': 19, '   de sig na ted</w>': 9, '   nu t sh ell</w>': 9, '   ni b ble</w>': 4, '   t in k le</w>': 2, '   bu lls</w>': 10, '   hi ll</w>': 87, '   fo x y</w>': 5, '   c ow</w>': 58, '   re p li es</w>': 4, ' pro te c ted</w>': 2, '   li m ou s ine</w>': 13, '   under we ar</w>': 59, ' ra y</w>': 28, '   e ye pa tch</w>': 1, '   pre sc ri p tion</w>': 19, '   gla ss es</w>': 96, '   a sti g ma ti s m</w>': 1, '   gen er i c</w>': 1, '   d ra w</w>': 111, '   ding</w>': 12, '   ling</w>': 3, '   p ran k</w>': 13, '   ad v an ces</w>': 12, '   den ti st ry</w>': 2, '   gar o tt e</w>': 1, '   to o th pa st e</w>': 10, '   con ta ins</w>': 12, '   to o th b ru sh</w>': 14, '   de ton ation</w>': 19, '   re t ro fi tt ed</w>': 2, '   g lo ba l</w>': 23, '   ge o sy n ch r on ous</w>': 1, '   po si tion ing</w>': 4, '   pre ser ved</w>': 10, '   ja g</w>': 14, '   in su l ting</w>': 17, '   de pen ded</w>': 9, '   pro c rea tion</w>': 3, '   er o s</w>': 2, '   br r r r</w>': 1, '   fri gi d</w>': 5, '   so c c er</w>': 9, '   s we di sh</w>': 21, '   st e war de ss es</w>': 3, ' ma d</w>': 8, '   st e w s</w>': 2, ' ne gr o</w>': 1, '   chi gr o</w>': 4, '   ne gr o</w>': 29, '   li bi do</w>': 2, '   ve g</w>': 2, '   t ack le</w>': 11, '   o x for d</w>': 8, '   ex ce lled</w>': 1, '   su b je c ts</w>': 21, '   spe ci a li z ing</w>': 2, '   ac ce p ted</w>': 41, '   cu l tu ra l</w>': 13, '   se ctor</w>': 48, '   pr in ted</w>': 27, '   an n oun ce men ts</w>': 2, '   f ar s i</w>': 1, '   pro mo ted</w>': 23, '   ju mb o</w>': 16, '   hea thr ow</w>': 3, '   wi tty</w>': 16, '   pro mi s cu ous</w>': 2, ' ex p an ding</w>': 1, '   en vi r on ment</w>': 35, '   p ound</w>': 43, '   ac c li ma te</w>': 2, ' made</w>': 17, '   pen is</w>': 37, '   en l ar ger</w>': 4, '   war ran ty</w>': 2, '   c ru sh ed</w>': 27, ' v el v et</w>': 1, '   fri lly</w>': 1, '   l ace</w>': 10, '   cra v at</w>': 3, '   me da lli on</w>': 4, ' d y ed</w>': 1, '   so cks</w>': 43, '   pu r p le</w>': 43, '   v in y l</w>': 6, '   no oo oo oo oo oo oo o</w>': 1, '   sh er ber t</w>': 3, '   re st ri c ted</w>': 14, '   t ac h q </w>': 1, '   inter ce pt</w>': 9, '   com pe ti tive</w>': 16, '   pri ces</w>': 19, '   co m</w>': 21, '   me tal u r g</w>': 1, '   re c on</w>': 12, '   a ll o y</w>': 11, '   st ea l th y</w>': 2, '   car b on</w>': 39, '   com po si te</w>': 7, '   s la ter</w>': 1, '   s ow est</w>': 1, '   ve c t or ing</w>': 1, '   un or th o do x</w>': 2, '   ow</w>': 52, '   ga ther ed</w>': 13, '   re min der</w>': 8, '   a h h h h h h h h h</w>': 1, '   fe l ine</w>': 3, '   comp li ca tions</w>': 18, '   re ani ma tion</w>': 1, ' e m da sh</w>': 1, '   bi g g le s wor th</w>': 1, ' work</w>': 15, '   s qu ee ze</w>': 49, '   au ton om i c</w>': 3, '   re t ar ded</w>': 19, '   de mon stra te</w>': 13, ' te m per ed</w>': 4, '   mu ta ted</w>': 8, '   fri g g in</w>': 26, '   ex per im en ted</w>': 6, '   la s ers</w>': 9, '   ou tw ei gh ed</w>': 3, '   pi ran h a</w>': 3, '   s an k</w>': 23, '   t an k</w>': 55, '   pi ran has</w>': 2, '   pl ac ed</w>': 27, '   en dan ger ed</w>': 6, '   ca li b er</w>': 17, '   re p li c ant</w>': 1, '   le th al</w>': 16, '   e f fi ci ent</w>': 10, '   char ms</w>': 13, '   brea th ta king</w>': 6, '   st ru m pe ts</w>': 1, '   w o h l</w>': 2, '   fe m bo ts</w>': 1, '   ne u tra li z ed</w>': 10, '   su g ge sti ons</w>': 18, '   s co t t</w>': 131, '   im pa ti ent</w>': 21, '   se men</w>': 14, '   in con se qu en ti al</w>': 1, '   a stu te</w>': 6, '   as so ci a tes</w>': 27, '   ca u tion ed</w>': 1, '   ke on</w>': 1, '   e h vi ll e</w>': 1, '   ar ti fi ci ally</w>': 5, '   l ab </w>': 114, '   li qui da ted</w>': 2, '   in so l ent</w>': 6, '   brea k through</w>': 5, '   cha mber</w>': 52, '   p ee p</w>': 17, '   gr oun ded</w>': 26, ' e s cap able</w>': 1, '   o ver ly</w>': 7, '   ex o ti c</w>': 15, '   e s cap es</w>': 11, '   fee ding</w>': 40, '   ne me s is</w>': 4, '   ti t ti e</w>': 2, '   s k in e ma x</w>': 1, '   ar ca de</w>': 31, '   ha ss ling</w>': 3, ' a ke</w>': 8, '   pe tting</w>': 7, '   v et</w>': 10, '   hi j ack</w>': 5, '   brea k away</w>': 4, '   tr ans f er r ing</w>': 3, '   e ss ence</w>': 14, '   so p hi sti ca ted</w>': 25, ' la s er</w>': 3, '   p un ch</w>': 84, '   pro te c tive</w>': 21, '   la y er</w>': 22, '   s ci en ti sts</w>': 51, ' o z one</w>': 1, '   u l tra vi o let</w>': 3, '   ra ys</w>': 7, '   p our</w>': 48, '   in c rea sing</w>': 5, '   he f ty</w>': 6, '   bl ack ma il</w>': 28, '   w ea l th i est</w>': 3, '   lan d ow n ers</w>': 1, '   ex or bi t ant</w>': 1, '   a m oun t</w>': 92, '   he ir</w>': 14, '   thr one</w>': 24, '   fran k l in</w>': 36, '   min t</w>': 16, '   de cor a tive</w>': 3, ' pa in ted</w>': 2, '   the me</w>': 24, '   pla tes</w>': 44, '   co ll e c t ors</w>': 12, '   ser i es</w>': 77, '   car ing</w>': 20, '   de b on a ir</w>': 1, '   win es</w>': 8, ' r en ow ned</w>': 1, '   p ho to gra p her</w>': 26, '   lo ver</w>': 72, '   stan dar ds</w>': 27, '   mu m my</w>': 27, '   ob se ssed</w>': 38, ' kill</w>': 12, '   la be l</w>': 28, '   par ti ally</w>': 11, '   li mi ts</w>': 40, '   hu mi d</w>': 1, '   de li ver ed</w>': 57, '   ph on el ine</w>': 1, '   li mi ted</w>': 26, '   per i o ds</w>': 8, '   ob li ge</w>': 14, '   au g ust</w>': 41, '   mer r y w ea ther</w>': 2, '   tu s can</w>': 2, '   hi ll side</w>': 9, '   clo se ly</w>': 40, '   pe ter</w>': 308, '   f l ower</w>': 53, '   k night</w>': 56, '   p ee l</w>': 67, '   bro lly</w>': 3, ' for m ing</w>': 1, '   da sh</w>': 20, '   mon so on</w>': 3, '   ja i pu r</w>': 1, '   sti ff en s</w>': 1, '   re so l ve</w>': 12, '   pu ck ers</w>': 2, '   que lls</w>': 1, '   na mb y</w>': 2, ' pa mb y</w>': 2, '   me mb er ship</w>': 19, '   1 9 2 2 </w>': 5, '   bo o d les</w>': 3, '   st e ed</w>': 20, '   ma da m</w>': 65, '   con fr on t</w>': 13, '   w ins</w>': 37, '   wor th y</w>': 45, '   o pp on ent</w>': 7, '   tr ac ked</w>': 21, '   win ning</w>': 62, '   co ll e ctor</w>': 14, '   co ll e c ts</w>': 5, '   bu tt er f li es</w>': 13, '   f ar things</w>': 1, '   co ll e ction</w>': 59, '   m ac ar o on</w>': 1, '   as sig n ment</w>': 61, '   mon i t or ed</w>': 9, '   e st ab li sh ment</w>': 20, '   f und ing</w>': 19, '   re se ar ch</w>': 118, '   app ro ved</w>': 20, '   t rea t ment</w>': 50, '   min i st ers</w>': 13, '   de f ence</w>': 10, '   b oun c ed</w>': 11, '   pu r po ses</w>': 27, '   mo bi le</w>': 20, ' q </w>': 19, '   s mi ther e en s</w>': 5, '   ban ff shi re</w>': 1, '   cont ro ll er</w>': 2, ' father</w>': 12, '   as sig n men ts</w>': 11, ' hu sh</w>': 8, '   co con u t</w>': 16, '   wal nu t</w>': 4, '   bab a</w>': 3, '   wa g n er i an</w>': 1, '   chi p</w>': 67, '   a mu sed</w>': 11, '   j i g saw</w>': 3, '   di a mon d</w>': 57, ' c y cl one</w>': 1, '   fr ac tion</w>': 6, '   po s se ss</w>': 19, '   un li mi ted</w>': 10, ' bro ther</w>': 15, '   su spe c ted</w>': 34, ' rs</w>': 2, '   tr an sp or t</w>': 49, '   e con om i c</w>': 16, '   di sa ster</w>': 47, '   fro st bi te</w>': 3, '   su n bur n</w>': 1, '   ch er no b y ls</w>': 1, '   imp en e tr able</w>': 4, '   bri de s ma id</w>': 5, '   v al en t ine</w>': 37, '   e mm a</w>': 28, '   re v el s</w>': 5, '   c lu es</w>': 25, '   sp o il</w>': 44, '   ma j ori ty</w>': 19, '   sh ar e ho l d ers</w>': 1, '   won der land</w>': 9, '   re c ru i ting</w>': 15, '   co o ks</w>': 4, '   star t ers</w>': 18, '   be ars</w>': 34, '   un ho ly</w>': 8, '   tr in i ty</w>': 18, '   mi s ca l cu la tion</w>': 1, '   re co g ni tion</w>': 18, '   pla sti c s</w>': 8, '   pro sp er o</w>': 2, '   tru b sha w</w>': 9, '   t an</w>': 48, '   lo tion</w>': 10, '   sh op s</w>': 16, '   ne ar by</w>': 18, '   cha per on</w>': 3, '   si ber i a</w>': 12, '   pe a</w>': 13, '   bo at</w>': 317, '   ow l</w>': 32, '   sh ou l der</w>': 68, '   nee dn</w>': 45, '   u mb re ll a</w>': 13, '   mer ry</w>': 44, '   li mp et</w>': 1, '   comp act</w>': 10, '   in to x i ca ting</w>': 4, '   lea ther</w>': 35, '   ad vi sa ble</w>': 4, ' in vi si ble</w>': 2, '   lon ers</w>': 2, '   en e mi es</w>': 74, '   cu ck o o</w>': 7, '   clo cks</w>': 8, '   to ys</w>': 32, '   dr at</w>': 1, '   e qui vo ca ted</w>': 1, '   o be y</w>': 15, '   le pi do p t ori sts</w>': 1, '   ga te cra sh</w>': 1, '   ca me l</w>': 17, '   stra w</w>': 24, '   dis da in</w>': 4, '   t re sp as sed</w>': 1, '   pr er o ga tive</w>': 7, '   p sy ch op a th i c</w>': 4, '   sc hi z op h r en i c</w>': 8, '   de lu si ons</w>': 18, '   re cu r r ing</w>': 5, '   a m ne si a</w>': 22, '   re pre ssion</w>': 1, '   out bur sts</w>': 4, '   an t i</w>': 80, ' so ci al</w>': 6, '   q </w>': 23, '   un dis co ver ed</w>': 5, '   tw in</w>': 19, '   mi lli on a i re</w>': 22, '   e c cen tri c s</w>': 2, '   hi j ac ked</w>': 6, '   min i a tu ri z ed</w>': 4, '   att e mp ts</w>': 11, '   di o x i de</w>': 8, '   pro p an e</w>': 1, '   qu an ti ti es</w>': 2, '   cap tur es</w>': 3, '   ch l or ine</w>': 1, '   pro te c ts</w>': 7, '   o z one</w>': 11, '   pro ved</w>': 32, '   imp r ac ti cal</w>': 3, '   bu l k y</w>': 2, '   att e mp t</w>': 64, '   war m ing</w>': 11, '   c li ma te</w>': 13, '   f ea si ble</w>': 2, '   in je c ting</w>': 4, '   co ck ta il</w>': 46, ' qui ck</w>': 6, '   le tting</w>': 116, '   z oo lo gi cal</w>': 3, '   e le men t ary</w>': 8, '   sho e ma k er</w>': 2, '   c r ab t ree</w>': 14, '   for mer ly</w>': 8, '   m our ning</w>': 15, '   in vi ta tion</w>': 39, '   pi c ni c</w>': 27, '   ra in ed</w>': 7, ' f on d</w>': 1, '   ob ser v ation</w>': 17, ' pro of</w>': 7, '   wai st co at</w>': 2, '   br en da</w>': 23, '   di st r act</w>': 12, '   s no o p</w>': 22, '   f an a ti cal</w>': 2, '   w ea ther man</w>': 1, '   cha ir man</w>': 44, '   or g ani sa tion</w>': 2, '   la st ing</w>': 8, '   ta m per ed</w>': 6, '   vi ces</w>': 3, '   me te or o lo gi ca lly</w>': 1, '   ju ly</w>': 36, '   bar king</w>': 14, '   re cl use</w>': 9, '   hi gh lan ds</w>': 7, '   for m er</w>': 51, '   f en c ing</w>': 5, '   rea ch ed</w>': 60, '   f re el an ce</w>': 15, '   stu ff ed</w>': 28, '   pre di c ta ble</w>': 7, '   un pre di c ta ble</w>': 16, '   sh in ing</w>': 21, '   fin ger pr int</w>': 8, '   co cked</w>': 6, '   sh od</w>': 2, '   t ou c he</w>': 4, '   ki pp er</w>': 1, '   h er r ing</w>': 13, '   sh ow ers</w>': 12, '   su n ny</w>': 32, '   hon ori fi c s</w>': 1, '   wi sh ed</w>': 30, '   d y ed</w>': 8, '   w ool</w>': 8, '   pen e tra ted</w>': 9, '   gu e ssed</w>': 41, '   sti ck l er</w>': 4, '   ki pp ers</w>': 1, '   t ow el</w>': 28, '   war dro be</w>': 18, '   de con ta min ation</w>': 2, '   a gi ta ted</w>': 12, '   h q </w>': 5, '   fun k</w>': 5, '   sp y</w>': 62, '   pri v a tely</w>': 14, '   ow ned</w>': 53, '   mar ks</w>': 66, '   h y de</w>': 8, '   sur r oun ded</w>': 37, '   ser pen t ine</w>': 1, '   con ven e</w>': 7, '   in i ti a tive</w>': 16, '   ex per i ment</w>': 59, '   ha l f bro ther</w>': 1, '   e ton</w>': 2, '   ca mb ri dge</w>': 8, '   ro bo ti c s</w>': 1, '   o ver ta k en</w>': 1, '   h un ch</w>': 34, '   le ar n t</w>': 5, '   ca m ou f la ge</w>': 4, '   p ran g</w>': 1, '   pi pe</w>': 44, '   in vi si ble</w>': 47, '   s ni ff</w>': 10, '   sc ent</w>': 30, '   o ver p ow er ing</w>': 5, ' find</w>': 14, '   j un g le</w>': 63, '   plan ts</w>': 39, '   ar c ti c</w>': 18, '   lu sh</w>': 3, '   tr ans for med</w>': 7, '   af ri can</w>': 22, '   sc ru bl and</w>': 1, '   b li z z ar ds</w>': 3, '   f on d</w>': 35, '   e s ca pa de</w>': 4, '   man made</w>': 2, '   ti pped</w>': 16, '   sta kes</w>': 17, '   ex per im en ts</w>': 23, '   re vo lu tion i z ed</w>': 1, '   bur g l ars</w>': 12, '   l t d</w>': 2, '   nee dle</w>': 35, '   ba ddy</w>': 1, '   u gh</w>': 22, '   e mer gen ci es</w>': 10, '   har row</w>': 3, '   ori g in al</w>': 92, '   de b</w>': 18, '   or al</w>': 14, ' really</w>': 26, '   mar r ying</w>': 45, '   pre ten ding</w>': 52, '   por no s</w>': 6, ' ne i ll</w>': 9, '   to le</w>': 7, '   so o o</w>': 6, '   bo b bi e</w>': 11, '   ch u l o</w>': 3, '   bro a ds</w>': 19, '   o c ca sion</w>': 48, '   a ma te u r</w>': 28, '   ra u l</w>': 1, '   den mar k</w>': 3, '   na u ti l us</w>': 2, '   a pi e ce</w>': 17, '   din er o</w>': 1, '   th ea ter</w>': 99, '   mu l t i</w>': 10, ' p le x</w>': 1, '   c in e ma</w>': 13, '   th ea t ers</w>': 7, '   pro min ent</w>': 16, ' 2 4 </w>': 6, '   3 </w>': 98, '   d ra gs</w>': 5, '   lo bo tom y</w>': 5, '   mi lea ge</w>': 10, '   han d les</w>': 21, '   por sc he</w>': 10, '   s wa p</w>': 17, '   pri z ed</w>': 3, '   o oh</w>': 39, '   mi ch el in</w>': 2, '   me tri c</w>': 3, '   to o ls</w>': 32, '   to a ster</w>': 9, '   o ven</w>': 16, '   li tt on</w>': 1, '   cu i sin art</w>': 4, '   g room</w>': 21, '   b on d</w>': 95, '   th om er son</w>': 2, '   1 0 0 2 </w>': 3, '   s lo b</w>': 16, '   bo o th</w>': 51, '   in for med</w>': 38, '   ad vi se</w>': 53, '   per su a si ve</w>': 4, '   dis ne y land</w>': 7, '   un happy</w>': 72, '   wan ta</w>': 56, '   du mp ed</w>': 48, '   wi mp</w>': 11, '   li ons</w>': 29, '   k en ya</w>': 6, '   chri st ma s</w>': 260, '   s mo ke house</w>': 2, '   wor k out</w>': 7, ' go ers</w>': 2, '   ho ok er</w>': 43, '   so a p</w>': 55, '   cha pe l</w>': 11, '   jo ys</w>': 4, '   sin g l en ess</w>': 1, '   dri ft</w>': 24, '   s lu mb er land</w>': 1, '   brea k f own</w>': 1, '   so vi et</w>': 29, '   ca ter ers</w>': 3, '   g own</w>': 16, '   tu x e do</w>': 10, '   de ce i ving</w>': 5, '   thr ow ing</w>': 75, '   p ho e be</w>': 6, '   m ea t lo a f</w>': 12, '   s wi ss</w>': 22, '   char red</w>': 1, '   f le sh</w>': 86, '   il en e</w>': 5, '   pi gs</w>': 52, '   sc r ab ble</w>': 4, '   hou se wife</w>': 10, '   wan g</w>': 4, '   p al m</w>': 43, '   en jo ys</w>': 8, '   ca ter er</w>': 7, '   ba thing</w>': 14, '   p in ch</w>': 17, '   pa ge ant</w>': 20, '   el se where</w>': 21, '   li ps</w>': 98, '   ce le bra tion</w>': 26, '   st al li on</w>': 11, '   pa in ts</w>': 4, '   ne u tra l</w>': 24, '   b ow ling</w>': 69, '   tu e s days</w>': 9, '   fri days</w>': 9, ' co at</w>': 2, '   bu sh</w>': 32, '   1 5 0 0</w>': 3, '   st ing</w>': 37, '   mi lt</w>': 6, ' ra te</w>': 13, '   pi mp</w>': 34, '   p ea ce ful</w>': 37, '   bu ll win k le</w>': 2, '   wri sts</w>': 14, '   s mo o th</w>': 59, '   ra z or</w>': 38, '   b la d es</w>': 19, '   s la sh</w>': 15, '   l ar</w>': 4, '   co o ki es</w>': 27, '   re fri ger a tor</w>': 21, ' f al c on</w>': 1, '   cre st</w>': 8, '   gra ins</w>': 1, '   qu a al u de</w>': 1, '   mo o se head</w>': 1, '   mo ose</w>': 12, '   el ev a tor</w>': 75, '   ten th</w>': 24, '   ex pla ins</w>': 35, '   vi ta m in</w>': 15, '   e b bi e</w>': 1, '   ger</w>': 2, '   u mp h</w>': 1, '   l able</w>': 1, '   so b</w>': 5, '   out look</w>': 6, '   im ma ture</w>': 11, '   g ran d chi l dr en</w>': 13, '   ru st y</w>': 24, '   un re l en ting</w>': 1, '   con st ant</w>': 29, '   ma i m</w>': 2, '   ser ves</w>': 24, '   o op s</w>': 12, '   b re t t</w>': 13, '   k in k y</w>': 16, '   ir re sp on si ble</w>': 22, '   l ou der</w>': 21, '   he ck</w>': 61, '   g l are</w>': 7, '   bu tt ons</w>': 29, '   se par a ted</w>': 30, '   k lu p ner</w>': 2, '   s nor t</w>': 12, '   ga l</w>': 51, '   happ in ess</w>': 63, '   ends</w>': 111, '   pri ma tes</w>': 4, '   de si ree</w>': 1, '   t ro j an</w>': 1, '   d on k ey</w>': 13, '   co chi se</w>': 2, '   bl an k</w>': 51, '   sp a tu l a</w>': 3, '   par a me di c s</w>': 6, '   oo oo o o</w>': 2, '   tr ac ey</w>': 2, '   e g g</w>': 72, '   sa la d</w>': 51, '   star ved</w>': 12, '   o li vi er</w>': 2, ' n y mp ho s</w>': 1, ' 3 0</w>': 72, '   an n oun ce</w>': 19, '   st an</w>': 55, '   f oo t b all</w>': 70, ' er</w>': 52, '   ga ll ons</w>': 12, '   han d s om er</w>': 2, '   s n ea k y</w>': 9, '   ve sp ers</w>': 2, '   dr ow ning</w>': 30, '   in f ant</w>': 18, '   t ar din ess</w>': 1, '   1 3 8 </w>': 2, '   op er a ting</w>': 33, '   st ev i e</w>': 11, '   s wa y z a k</w>': 19, '   ad co x</w>': 9, '   sp li tt in</w>': 2, '   pen ge lly</w>': 1, '   ba tt a li ons</w>': 4, '   kn ow l ton</w>': 1, '   du mb shit</w>': 7, '   po les</w>': 8, '   la d d ers</w>': 7, '   ho se</w>': 43, '   fran ny</w>': 3, '   st ea m</w>': 38, '   f la sh</w>': 41, '   ce i ling</w>': 23, '   pu ke</w>': 37, '   pro bi e</w>': 9, '   m c ca ff re y</w>': 14, '   cl in ch er</w>': 2, '   star es</w>': 17, '   ri m ga le</w>': 6, ' jesus</w>': 26, '   can di da te</w>': 25, '   in si ght</w>': 19, '   sor ta</w>': 39, '   se an ce</w>': 3, '   dri lling</w>': 9, '   sp o t li ght</w>': 5, ' ro id</w>': 3, '   stan d pi pe</w>': 1, '   be side</w>': 61, ' pi pe</w>': 3, '   cou gh</w>': 21, '   fa ked</w>': 8, '   ca b ins</w>': 10, '   s co tch</w>': 54, '   cra w ling</w>': 50, '   mo an</w>': 4, '   bi z</w>': 8, '   ban k ru pt</w>': 10, '   g i</w>': 3, '   la u r a</w>': 75, ' tr ack</w>': 8, '   ri pp ing</w>': 20, '   fi t z ger a ld</w>': 3, '   bro ther ly</w>': 3, '   as sig ned</w>': 44, '   gra du ation</w>': 22, '   r ent</w>': 143, '   j en ni f er</w>': 59, '   ba k er y</w>': 10, '   mar ty</w>': 158, '   ar son</w>': 11, '   fi re fi gh ting</w>': 3, '   s ki lls</w>': 31, ' doing</w>': 16, '   in ve sti ga t ors</w>': 10, '   spe ci men</w>': 25, '   supp re ssion</w>': 2, '   cont ro ls</w>': 24, '   in ser tion</w>': 3, ' d ra w n</w>': 1, '   h ome y</w>': 9, '   o l de st</w>': 33, '   lo tta</w>': 68, '   din o sa u r</w>': 23, '   pro of</w>': 149, '   pl u gs</w>': 6, '   t or ch</w>': 24, '   b ack d ra f ts</w>': 3, ' i ll</w>': 32, '   al ber t</w>': 74, '   e in st e in</w>': 21, '   ac ci den ts</w>': 21, '   ju g g ling</w>': 4, '   s na il</w>': 1, '   3 0 0</w>': 17, '   s wa mp ed</w>': 2, '   fun d</w>': 45, ' ra i sing</w>': 10, '   stra i gh te st</w>': 1, '   po ster</w>': 23, '   ea t ers</w>': 2, '   gar ba ge</w>': 98, ' side</w>': 6, '   j p</w>': 7, '   i i</w>': 40, '   i ri sh</w>': 45, '   li t wal ks</w>': 1, '   bar f</w>': 11, '   plan t ers</w>': 1, '   b ou l ev ard</w>': 17, '   e f fi ci en cy</w>': 12, '   may or</w>': 79, '   no te bo ok</w>': 11, '   gr en ad ine</w>': 1, '   d ried</w>': 19, '   gra du a ted</w>': 28, '   tr y ch ti cho l or ate</w>': 4, '   out le ts</w>': 3, '   ci vi li ans</w>': 20, '   ho ll er ing</w>': 5, '   me di u m</w>': 21, '   ba ll sy</w>': 3, '   en g ine</w>': 51, '   sp ar k</w>': 18, '   mar g in</w>': 7, '   pro ba tion</w>': 29, '   sc or es</w>': 17, '   brea the</w>': 101, '   ho spi tal s</w>': 15, '   ac coun t ant</w>': 24, '   con su l ting</w>': 6, '   man p ower</w>': 10, '   p o</w>': 6, '   d ra f ts</w>': 4, '   man ning</w>': 7, '   co s g ro ve</w>': 24, '   co pp ers</w>': 6, '   la under ed</w>': 3, '   s ea gra ve</w>': 7, '   ho l com b</w>': 4, '   ac coun t an ts</w>': 6, '   de k om</w>': 1, '   i ll in o is</w>': 9, '   la ke side</w>': 2, '   d y na mi c s</w>': 8, '   win dy</w>': 3, '   ven tur es</w>': 1, '   je ff re y</w>': 172, '   s ea l ant</w>': 4, '   fri ed</w>': 42, '   fi re brea ks</w>': 2, '   b ack d ra ft</w>': 4, '   r on a ld</w>': 26, '   f ry</w>': 49, '   pu tty</w>': 6, '   out let</w>': 6, '   te ars</w>': 65, '   o x y g en</w>': 66, '   ma g ne si u m</w>': 8, '   p ow der</w>': 49, '   re si du e</w>': 6, '   mo le cu les</w>': 16, '   ba m m</w>': 1, ' du r ing</w>': 1, '   e pi so d es</w>': 10, '   k el v in</w>': 9, '   tr y ch</w>': 1, '   di ssi pa tes</w>': 2, '   con su me</w>': 7, '   pro per ti es</w>': 7, ' tr y ch ti ch l or ate</w>': 1, '   st ru c tu red</w>': 3, '   co pp er</w>': 27, '   me l ted</w>': 12, '   re qu ir es</w>': 42, '   7 0 0 0</w>': 1, '   i g ni tion</w>': 9, '   in di ca ting</w>': 7, '   dis co l or ation</w>': 3, '   s nu ff ed</w>': 2, '   gu l p</w>': 5, '   bar ri er</w>': 15, '   so o t</w>': 4, '   un bur ned</w>': 1, '   ther ma l</w>': 10, '   b al an ce</w>': 54, '   pe st</w>': 8, '   ha mm er</w>': 44, '   den n is</w>': 51, '   st ran g le</w>': 14, '   s ean</w>': 89, '   f ab ri ca tion</w>': 4, '   tom b st one</w>': 6, '   tra di tions</w>': 3, '   re pre sen ting</w>': 15, '   au th en ti c</w>': 10, '   cl an</w>': 4, '   man ne qu in</w>': 4, '   re pl ac e ment</w>': 17, '   in ju ry</w>': 23, '   sur vi ving</w>': 16, ' pro ba tion ary</w>': 1, '   bra very</w>': 4, '   ve ter an</w>': 13, '   fi re fi gh ter</w>': 1, '   ri s ked</w>': 9, '   e mer ging</w>': 5, '   vi c t ori ou s ly</w>': 1, '   an na</w>': 109, '   ro dri gu e z</w>': 6, '   s ea m st re ss</w>': 4, '   sh ore</w>': 57, '   ga in ed</w>': 17, '   pro min ence</w>': 1, '   1 9 7 2 </w>': 7, '   pu li t z er</w>': 10, '   p ho to gra p h</w>': 31, '   u s in</w>': 11, '   pu ll in</w>': 8, '   dr own</w>': 30, '   c ri mp</w>': 2, '   bri be</w>': 27, '   ba tt a li on</w>': 9, '   ti m my</w>': 15, '   1 1 5 </w>': 7, ' bri an</w>': 4, '   spe c tru m</w>': 9, ' la g un a</w>': 1, '   ja mm ing</w>': 5, '   cu sto m</w>': 16, '   sur f bo ar ds</w>': 1, ' pi on e er</w>': 1, '   gr ow th</w>': 24, '   co ll age</w>': 4, ' as si st ant</w>': 1, '   a sp en</w>': 9, '   s n ow mo bi le</w>': 3, '   t ou rs</w>': 11, '   wi lly</w>': 12, '   li tt le st</w>': 2, '   fi re house</w>': 5, '   bar n</w>': 37, '   fa st en</w>': 6, '   con fu se</w>': 20, '   pi an o</w>': 90, '   i f s</w>': 2, '   t rea ts</w>': 14, '   de ser ving</w>': 4, '   com m uni ca tor</w>': 6, '   ab sor p tion</w>': 1, '   ca tal y st</w>': 7, '   to x i c</w>': 25, '   pa tch</w>': 41, '   spe c tr o</w>': 1, '   tr ac es</w>': 11, '   sc ra p ed</w>': 7, '   c ri sp ers</w>': 1, '   ra y op h en e</w>': 1, '   sha d ow</w>': 62, '   de pri ving</w>': 1, '   te le ph on es</w>': 10, '   w oo d en</w>': 21, '   ma tch es</w>': 35, '   bor ed</w>': 78, '   cen t ers</w>': 10, '   re ti re ment</w>': 36, '   st ro ke</w>': 60, '   al der man</w>': 4, ' ri se</w>': 3, '   ca th o li c</w>': 40, '   me ts</w>': 17, '   cha l k</w>': 16, '   1 8 0</w>': 14, '   ar i an e</w>': 2, '   2 8 0</w>': 4, '   stra w ber ri es</w>': 9, '   ra pi sts</w>': 6, '   sp o on</w>': 15, '   sp ee d b all</w>': 1, '   sin s</w>': 35, '   r ack et</w>': 35, '   op er ate</w>': 41, '   spi lt</w>': 2, '   a un t</w>': 137, '   l u</w>': 7, '   pen gu in</w>': 22, '   con ne c ts</w>': 15, '   b ow ta y</w>': 1, '   hea d sho ts</w>': 2, '   d ev e lo p ers</w>': 4, '   di d ya</w>': 5, '   ch ea per</w>': 16, '   ju li o</w>': 1, '   pa o l o</w>': 3, '   cha li ce</w>': 5, '   busin e ss man</w>': 42, '   ton gu e</w>': 97, '   na i ls</w>': 30, '   p al ms</w>': 7, '   c r own</w>': 24, '   th or n s</w>': 4, '   bi g time</w>': 3, '   3 3 </w>': 10, '   n in th</w>': 24, '   in ning</w>': 10, '   wh om ever</w>': 5, '   p un ks</w>': 13, ' 1 2 0</w>': 3, '   p sy h o</w>': 1, '   w n at</w>': 1, '   bo ok ma k er</w>': 1, '   stra w ber ry</w>': 13, '   st ri ke out</w>': 1, '   sta di u m</w>': 15, '   do d ging</w>': 3, '   bu ll e ts</w>': 82, '   mo ther fucking</w>': 17, '   oo o o</w>': 2, '   ow ing</w>': 5, '   go o d en</w>': 2, '   pi tch ing</w>': 12, '   o a k land</w>': 13, ' 8 0 0</w>': 6, '   nee dy</w>': 12, '   sch oo l y ard</w>': 5, '   p ra y ed</w>': 16, '   de se cra tion</w>': 4, '   fa lling</w>': 109, '   p ra y er</w>': 50, '   su per i or</w>': 38, '   dam p</w>': 10, '   f re sh ly</w>': 2, '   mo i st</w>': 4, '   kn ea ding</w>': 1, '   ba ke</w>': 13, '   mu sc les</w>': 20, '   tri vi al</w>': 6, '   pre sc ri be</w>': 6, '   con tri tion</w>': 1, '   ab so l ve</w>': 2, '   con fe ssi on al</w>': 3, '   imp li es</w>': 3, '   mi sp l ac ed</w>': 16, '   con de m n</w>': 5, '   om i ssion</w>': 2, '   gra ver</w>': 2, '   sti l</w>': 1, '   y o</w>': 151, '   go ts</w>': 6, ' 1 6 </w>': 9, '   tru </w>': 4, '   s li de</w>': 32, '   in ju sti ce</w>': 12, '   de m</w>': 10, '   co ons</w>': 4, '   i z</w>': 4, ' kn ow what i</w>': 6, ' sa y in</w>': 7, '   si mp le ton</w>': 4, '   bu ff o ons</w>': 5, '   man t an</w>': 34, ' b en e di ct</w>': 1, '   ar no ld</w>': 46, ' l</w>': 80, ' k</w>': 134, '   b la k</w>': 3, '   ni g g ers</w>': 32, ' ce pt</w>': 21, '   ch ant</w>': 5, '   hon e y cu t t</w>': 4, ' what i</w>': 5, ' tal kin</w>': 6, '   mi ll en ni u m</w>': 16, '   min st re l</w>': 8, '   thr ow in</w>': 8, '   see ya</w>': 11, '   ni g g a</w>': 35, '   s la very</w>': 15, '   bl ow in</w>': 14, '   de l ac ro i x</w>': 9, '   you kn ow what i</w>': 1, '   de mo gra p hi c s</w>': 3, '   mon ke es</w>': 2, '   ma u</w>': 12, ' ma u</w>': 6, '   re vo l ting</w>': 4, '   en s la ving</w>': 2, '   ra pp ing</w>': 4, '   o ver thr ow</w>': 5, '   gu c c i</w>': 3, '   ti mb er land</w>': 1, '   ro le x</w>': 9, '   b en z</w>': 3, '   c ri st al</w>': 3, '   p se u do</w>': 6, '   f la g</w>': 38, ' wa ving</w>': 1, '   re vo lu tion a i ri es</w>': 1, '   ex po sure</w>': 22, '   s is</w>': 19, '   d is</w>': 13, '   ju li us</w>': 9, '   l i</w>': 37, '   ho p k ins</w>': 19, '   s la ve</w>': 52, '   hu g g in</w>': 1, '   cor on ary</w>': 6, '   p ee ps</w>': 2, '   s ci ence</w>': 158, '   ri gh te ous</w>': 15, '   th rea ts</w>': 22, '   re vo lu tion ary</w>': 20, '   ta v is</w>': 4, '   lo v in</w>': 11, '   per pe tra ting</w>': 1, '   se ll out</w>': 4, '   ca ll er</w>': 11, ' head</w>': 50, '   le e ch</w>': 9, '   k ling</w>': 1, '   s ki pp y</w>': 9, '   k ool</w>': 6, ' a id</w>': 10, '   per i od</w>': 88, '   s lo an</w>': 35, '   de l a</w>': 16, '   jo kin</w>': 6, '   di b s</w>': 4, '   tr i</w>': 10, ' se x u al</w>': 6, '   mo ori sh</w>': 3, '   d at</w>': 35, '   bo o ty</w>': 7, '   cer ti fi ed</w>': 22, '   ch no z</w>': 1, '   ho t ti e</w>': 3, '   sa ti ri cal</w>': 1, '   man ra y</w>': 14, '   ad v an ce</w>': 50, ' l a</w>': 34, ' c ro i x</w>': 1, '   de la po t</w>': 1, '   ch ee b a</w>': 4, '   ev i c ted</w>': 2, '   ri k ers</w>': 3, '   ho o se g ow</w>': 3, '   sa v in gs</w>': 31, '   pre s to</w>': 4, '   chan go</w>': 1, '   sen sa tion</w>': 11, '   chi ll</w>': 43, '   b en ja min s</w>': 4, '   c ri b</w>': 8, '   lu x u ri es</w>': 3, '   h b o</w>': 4, '   ar ro z</w>': 1, '   po llo</w>': 2, '   star v in</w>': 9, '   mar v in</w>': 33, '   bl ack face</w>': 4, '   co att a il</w>': 1, '   happen in</w>': 18, '   ki cks</w>': 32, '   ho of</w>': 5, '   pi tch</w>': 34, '   ta p</w>': 47, '   e u re k a</w>': 4, '   tal en ted</w>': 26, '   ho lo ca ust</w>': 10, ' st</w>': 24, '   cen tu ry</w>': 113, '   d ra w n</w>': 36, '   se x i st</w>': 7, '   r ac i st</w>': 11, '   ma ter i al</w>': 69, '   di sa gre e ing</w>': 1, '   in h er ent</w>': 4, '   cen sor ed</w>': 1, '   pi er re</w>': 16, '   c rea tor</w>': 38, '   cont ro ver si al</w>': 6, '   gen er ate</w>': 12, '   v ani ll a</w>': 11, '   m om s</w>': 14, '   gi t go</w>': 1, '   sc en es</w>': 19, '   c n s</w>': 2, '   bra ss</w>': 22, '   c li ps</w>': 9, '   mi d s ea son</w>': 1, '   fun ni er</w>': 5, '   ex pen se</w>': 34, '   d un wi tty</w>': 10, '   ne g ro es</w>': 24, '   a mo k</w>': 2, '   bo y co tt s</w>': 1, '   de mo stra tions</w>': 1, '   com m ence</w>': 4, '   pi ck et</w>': 17, '   la w n</w>': 28, '   gre en wi ch</w>': 6, '   p un ch ed</w>': 14, '   nor way</w>': 2, '   s we d en</w>': 10, '   bl on d</w>': 22, '   re vi si ons</w>': 7, '   ou tra ge</w>': 12, '   sha m</w>': 8, '   c rea tive</w>': 40, '   gen i u ses</w>': 7, '   der ri ck</w>': 4, '   co le man</w>': 6, '   a la s</w>': 9, '   wh oo op de e dam n do o</w>': 1, '   ac t ors</w>': 36, '   spi el bur g</w>': 1, ' a mi sta d</w>': 1, '   whi z</w>': 17, '   qu a li fi ca tions</w>': 9, '   di re c ting</w>': 12, '   ma d on na</w>': 15, '   nu th in</w>': 33, '   out si der</w>': 8, '   p he en om</w>': 1, '   so le ly</w>': 6, '   e th ni ci ty</w>': 1, '   gen der</w>': 9, ' a mer i can</w>': 26, '   st r on g ly</w>': 17, '   sa ti re</w>': 5, '   ga le</w>': 73, '   mi s st ep</w>': 1, ' a mo s</w>': 3, '   sen si ti vi ty</w>': 10, '   a war en ess</w>': 8, '   ter ra in</w>': 14, '   hi z</w>': 1, ' ho ok</w>': 2, '   wi d</w>': 8, '   mo v in</w>': 23, '   al ab a ma</w>': 25, '   mu r ph y</w>': 28, '   p j </w>': 2, '   di ck ey</w>': 2, '   char ac t ers</w>': 22, '   s n ow f la ke</w>': 1, '   ra st us</w>': 1, '   sa mb o</w>': 1, '   bu n ny</w>': 45, '   je mi ma</w>': 2, '   gi f ted</w>': 14, '   ho of er</w>': 2, '   e du ca ted</w>': 16, '   na ac p</w>': 1, '   d on ation</w>': 10, '   u pro ar</w>': 4, '   pro te st</w>': 17, '   com i cal</w>': 9, '   si de ki ck</w>': 5, '   un e du ca ted</w>': 1, '   ha y wi re</w>': 7, '   i g nor ant</w>': 34, '   du ll wi tt ed</w>': 1, '   un lu ck y</w>': 10, '   tra i ts</w>': 9, '   tri al s</w>': 9, '   tri bu la tions</w>': 2, '   du s k y</w>': 3, '   du o</w>': 2, '   e dge</w>': 82, '   who op i</w>': 2, '   ca st</w>': 77, '   b ack b one</w>': 4, '   ri s qu e</w>': 2, '   b on er</w>': 8, '   joh n son</w>': 126, '   dis re spe ct</w>': 18, '   en ter ta in ment</w>': 38, '   1 9 </w>': 23, '   v ar i e ty</w>': 17, '   jo kes</w>': 68, '   s ki ts</w>': 1, '   bur ne t t</w>': 5, '   he e ha w</w>': 1, '   pre vi ous</w>': 31, '   su per fi ci al</w>': 7, '   hea d lin es</w>': 18, '   hou se ho l ds</w>': 1, '   t un ed</w>': 3, '   g lu ed</w>': 8, '   te l ev i si ons</w>': 1, '   ad ver ti s ers</w>': 1, '   fo lli es</w>': 2, '   si t co m</w>': 7, '   wa ter me l on</w>': 9, '   gen er a tions</w>': 13, '   c r ack hea ds</w>': 2, '   an te</w>': 5, '   be ll u m</w>': 1, ' s ea son</w>': 3, '   pr on to</w>': 7, '   dea f</w>': 45, ' a mer i c ans</w>': 5, ' ni g g ers</w>': 4, '   tr end</w>': 3, '   st y les</w>': 12, '   or e o</w>': 1, '   h ome bo ys</w>': 2, '   de s mon d</w>': 34, '   p fe i ff er</w>': 1, ' ni g ger</w>': 4, '   whi ter</w>': 2, '   bl ack er</w>': 2, '   pre ten ti ous</w>': 7, '   fr on t in</w>': 1, '   c r ack ers</w>': 10, '   sc ri p ts</w>': 13, '   en cla ve</w>': 1, '   k k k</w>': 1, '   ra i ses</w>': 14, '   an ti se p ti c</w>': 2, '   co s by</w>': 3, '   pre s ence</w>': 65, '   cu l ture</w>': 48, '   mon si e u r</w>': 98, '   of f en ded</w>': 31, ' un qu o te</w>': 1, '   b i</w>': 13, ' r ac i al</w>': 2, '   spi ke</w>': 17, '   t ar an t in o</w>': 3, '   o le</w>': 20, '   cor ny</w>': 18, '   bl ow up</w>': 2, '   whi pp ing</w>': 3, '   co l or ed</w>': 52, '   st er e o t y pi cal</w>': 1, '   beli e f</w>': 36, '   mar i e</w>': 38, '   di sp en sed</w>': 1, ' exactly</w>': 8, '   pe er less</w>': 9, '   to l</w>': 16, '   en ter ta in ers</w>': 5, '   char le st on</w>': 11, '   car o lin a</w>': 12, '   ca te g ory</w>': 15, '   con sp ir ac i es</w>': 3, '   com i c</w>': 40, '   f ac t ors</w>': 12, '   as kin</w>': 54, '   chi tt l in</w>': 1, '   vi r g in ny</w>': 1, '   pu re ly</w>': 29, '   me di c in al</w>': 3, '   vi a gr a</w>': 3, '   do t</w>': 21, '   j un e bu g</w>': 9, '   stu b bor n</w>': 21, '   ho st ess</w>': 12, '   du lls</w>': 1, '   d ran k</w>': 43, '   stu per</w>': 1, '   n y u</w>': 7, '   mu mb o</w>': 12, '   ro man ti ca lly</w>': 5, '   cra z i er</w>': 13, '   bu ck et</w>': 25, '   for th right</w>': 3, '   j i g g y</w>': 2, '   ha mb one</w>': 3, '   han g in</w>': 14, ' hu man</w>': 8, '   co ll e c ti b les</w>': 2, '   app ro pi ate</w>': 1, '   re pr o</w>': 1, '   ci r c a</w>': 3, '   jo lly</w>': 10, '   vi c t ory</w>': 53, '   fo g gi est</w>': 3, '   im po sed</w>': 2, '   le g</w>': 166, '   jo se p h</w>': 73, '   sh oo ter</w>': 18, '   t wi sted</w>': 55, '   di st or ted</w>': 3, '   pr in ci pa l</w>': 54, '   to l er a tes</w>': 3, '   ne g ro i da l</w>': 2, '   pro ver bi al</w>': 2, '   en li gh ten ed</w>': 8, '   mi ll en i u m</w>': 1, '   u p side</w>': 23, '   sle d ge ha mm er</w>': 7, '   b le ed</w>': 37, '   ne t wor ks</w>': 10, '   co on</w>': 9, '   h ence</w>': 27, '   ton</w>': 25, '   bri cks</w>': 10, '   for m at</w>': 3, '   r ow an</w>': 43, ' la u gh</w>': 1, '   com men ts</w>': 7, '   ri g ged</w>': 18, '   st ac ked</w>': 2, '   min ori ty</w>': 14, '   st ru g g le</w>': 36, '   whi ten ess</w>': 1, '   mon a</w>': 8, '   k oo k</w>': 3, '   dre ss ing</w>': 46, '   bl ack en</w>': 3, '   sp ell</w>': 69, '   bur st</w>': 33, '   bu b ble</w>': 15, '   har de st</w>': 20, '   gu tt er most</w>': 1, '   u pp er most</w>': 1, '   l y n ch</w>': 10, '   m ou se</w>': 31, '   p un king</w>': 1, '   sc an da l</w>': 25, '   gi u li an i</w>': 1, '   b le ss ing</w>': 23, '   sta mp s</w>': 28, '   car d bo ard</w>': 9, '   ear n s</w>': 3, '   some kind</w>': 1, '   sh ee e t t</w>': 1, '   li gh t stuff</w>': 1, '   sc ra mb l in</w>': 1, '   j an u ary</w>': 22, '   bu ff o on er y</w>': 1, '   spe lling</w>': 10, '   be e</w>': 36, '   o ver jo y ed</w>': 5, '   ri ch mon d</w>': 4, '   bu ff o on</w>': 5, '   de fini tion</w>': 18, '   cer ti fi ca te</w>': 21, '   ho ll y w o od</w>': 118, '   do than</w>': 1, '   un gra te ful</w>': 15, '   ju k k a</w>': 2, '   tra ve st y</w>': 7, '   de b ac le</w>': 1, '   er o ti c</w>': 13, '   cor re c ting</w>': 3, '   a h h h</w>': 30, '   fi as c o</w>': 6, '   bo on do g g le</w>': 1, '   ab om in ation</w>': 5, '   hel sin k i</w>': 1, '   ca pi tal</w>': 61, ' th rea ten ing</w>': 1, '   vo il a</w>': 8, '   my r na</w>': 4, '   man i fe s to</w>': 7, '   ga in fu lly</w>': 2, '   e mp lo y</w>': 17, '   c ri ti c s</w>': 11, ' called</w>': 25, '   de ter min es</w>': 3, '   stra te g y</w>': 26, '   mar ch ed</w>': 8, '   se l ma</w>': 2, '   ra t in gs</w>': 17, '   go l d f ar b</w>': 1, '   con su l t ant</w>': 13, '   happ i est</w>': 20, '   ho of ing</w>': 1, '   com par es</w>': 1, '   k no cking</w>': 39, '   inter n ship</w>': 7, '   sp on sor s</w>': 3, '   1 2 5 </w>': 11, '   ma lt</w>': 8, '   li qu or</w>': 58, '   ti mm i</w>': 1, '   hi ll ni g ger</w>': 1, '   g it</w>': 56, ' to e</w>': 3, ' to ed</w>': 6, '   re t or t</w>': 3, '   comp ton</w>': 8, '   har le m</w>': 10, '   an ce st ors</w>': 17, '   si mp li sti c</w>': 2, '   f l ow</w>': 36, '   pro te st ing</w>': 3, '   what i</w>': 1, '   ma se</w>': 2, '   sp ar ked</w>': 1, '   cont ro ver sy</w>': 3, '   pro v ok ed</w>': 6, '   wa ll ace</w>': 40, '   bar bar a</w>': 60, '   wal t ers</w>': 9, '   j an e</w>': 99, '   pa u le y</w>': 5, '   what not</w>': 12, '   sh o</w>': 3, '   who pp er</w>': 4, '   st e p da u gh ter</w>': 3, '   th u s ly</w>': 1, '   ac cor din g ly</w>': 5, '   b en</w>': 317, '   p an ca ke</w>': 3, '   a ll e g ori cal</w>': 1, '   h y po the s is</w>': 5, '   h y ph en a ted</w>': 1, '   sen ten ces</w>': 10, '   in te ll e ct</w>': 19, '   con ta in ed</w>': 12, '   di minu tive</w>': 3, '   do se</w>': 18, '   un ex pe c ted</w>': 25, '   pe cu li ar i ty</w>': 1, '   g ri ts</w>': 5, '   m ac ar on i</w>': 7, ' too</w>': 26, ' ac on i te</w>': 1, ' d at</w>': 2, ' for d</w>': 1, '   vi ck i</w>': 24, '   w oo lly</w>': 4, '   pi ck in</w>': 21, ' be tw e en</w>': 4, '   s lin k y</w>': 6, '   lu c in dy</w>': 1, '   mu ster</w>': 6, '   as sa u l ted</w>': 6, '   ba tt er ed</w>': 2, '   we d lo ck</w>': 1, '   c r ack head</w>': 5, '   in fe sted</w>': 3, '   in f la ted</w>': 2, '   ro lls</w>': 25, '   pa y che ck</w>': 18, '   bi ble</w>': 76, '   th u mp ing</w>': 3, '   mon g ling</w>': 1, '   a th le tes</w>': 3, ' d un king</w>': 2, '   ho p</w>': 33, ' hi p</w>': 1, '   e b on i c</w>': 1, ' sp ea king</w>': 2, '   of f en d ers</w>': 7, '   fee ts</w>': 1, '   mi stu h</w>': 3, ' ci ty</w>': 5, '   s li ck ers</w>': 2, '   ar om a</w>': 2, '   ri pe</w>': 15, '   wa ter me l ons</w>': 3, '   al ab a my</w>': 1, '   coun tri fi ed</w>': 1, '   ba ma</w>': 4, '   hu st le</w>': 22, '   bu st le</w>': 3, '   w u z</w>': 3, ' i ho p</w>': 1, '   ear ned</w>': 41, '   ex pi red</w>': 5, '   inter n</w>': 4, '   f li pp in</w>': 5, '   ber t</w>': 44, ' th s</w>': 2, '   con sti tu tion</w>': 23, '   be g in n in</w>': 4, '   ra p</w>': 21, '   as th ma ti c</w>': 3, '   in ha l er</w>': 1, '   h ome boy</w>': 6, '   hi gh wa y man</w>': 3, '   b less</w>': 49, '   g ro g an</w>': 4, '   p ac es</w>': 5, '   hi gh wa y men</w>': 1, '   in su l ted</w>': 31, '   du g an</w>': 8, '   g ro ss ly</w>': 2, '   en g li sh man</w>': 9, '   ear ne st</w>': 7, '   po oh</w>': 15, '   lo ver s</w>': 35, '   su m</w>': 21, ' bo ys</w>': 4, '   a do pt</w>': 16, '   ho i ty</w>': 1, ' to i ty</w>': 1, '   no on</w>': 47, '   ch ev a li er</w>': 12, '   ar ri v al</w>': 14, '   c lu m sy</w>': 10, '   ba g ga ge</w>': 11, '   dri ves</w>': 41, '   gen dar mes</w>': 1, '   m oun t</w>': 27, '   co ac h man</w>': 1, '   in ten tions</w>': 30, '   ch ea ts</w>': 13, '   au st ri an</w>': 28, '   en vo ys</w>': 1, '   cont in u ally</w>': 7, '   su p</w>': 13, '   re p ea te d ly</w>': 10, '   par o le</w>': 48, '   chan ne l</w>': 51, ' bo x</w>': 6, '   f re der i c s</w>': 3, '   h un gar i an</w>': 9, '   lo ins</w>': 6, '   ga mb les</w>': 2, '   e m pre ss</w>': 15, '   be ll e</w>': 17, '   ri b and</w>': 1, '   po pe</w>': 37, '   sp u r</w>': 6, '   poli sh ed</w>': 4, '   ob li ging</w>': 2, '   li ber t ine</w>': 1, '   pre ju di ces</w>': 2, '   p ru dent</w>': 6, '   re gi ment</w>': 11, '   su c ce ed</w>': 22, '   de p end</w>': 42, '   dis char ge</w>': 14, '   gu in ea s</w>': 8, '   ne p he w</w>': 36, '   g al gen st e in</w>': 3, '   a i ded</w>': 2, '   de ser ter</w>': 5, '   im po st or</w>': 5, '   fo lly</w>': 7, '   con fir med</w>': 43, '   di sp a tch es</w>': 8, '   ac ted</w>': 46, '   co o lly</w>': 4, '   f ar ther</w>': 20, ' kee p ers</w>': 2, ' y ard</w>': 1, '   s ly</w>': 13, '   ro gu e</w>': 14, '   in f lu ence</w>': 58, '   lo ve li est</w>': 4, '   ger many</w>': 26, '   i re land</w>': 11, '   cor b ac h</w>': 2, ' gra dy</w>': 2, '   comp ani on</w>': 18, '   ci vi li ty</w>': 4, '   ans w er ed</w>': 50, '   in ven ted</w>': 54, '   ac qu a in t an ce</w>': 22, '   re com men da tion</w>': 12, '   d or o th y</w>': 134, '   de ser ting</w>': 5, '   du bl in</w>': 10, '   re co ver ed</w>': 23, '   du g ans</w>': 2, '   du el</w>': 5, '   c ow ar d ly</w>': 10, '   ro der i ck</w>': 20, '   th i ck</w>': 51, '   pl u g get</w>': 1, '   ob li ged</w>': 24, '   con so le</w>': 13, '   du g an st own</w>': 1, ' br own</w>': 4, '   be ss</w>': 2, '   ad dre ssed</w>': 15, ' do z en</w>': 7, '   w ink</w>': 4, '   to k en</w>': 9, '   sha king</w>': 50, '   cor di ally</w>': 1, '   a po lo gi z es</w>': 2, '   th i ther</w>': 1, '   a po lo g y</w>': 37, '   ga te</w>': 75, '   h in g es</w>': 4, '   s wor ds</w>': 13, '   k ne e</w>': 42, '   sc ar ce</w>': 5, '   s word</w>': 126, '   fa te</w>': 81, '   hu d de l st one</w>': 1, '   fu d de l st one</w>': 2, '   bar on et</w>': 1, '   fa ta lly</w>': 3, '   br en t for d</w>': 1, '   sc or ned</w>': 2, '   m are</w>': 14, '   sa d d l ed</w>': 3, '   b loo d thir st y</w>': 3, '   re sen ting</w>': 2, '   pre ten ds</w>': 11, '   k in d ne ss</w>': 30, '   har b or</w>': 44, '   ja me s vi ll e</w>': 1, '   y on der</w>': 19, '   di st re ssed</w>': 2, '   f lin ging</w>': 3, '   mar ri es</w>': 13, '   tu t</w>': 8, '   vi per</w>': 4, '   bo so m</w>': 8, ' hi g g ins</w>': 12, '   gen try</w>': 1, '   me th ro po l is</w>': 1, '   tra de s men</w>': 1, '   s che m er</w>': 2, '   de ce i ver</w>': 2, '   si r ra h</w>': 3, '   wa ter town</w>': 1, '   re d mon d st own</w>': 1, '   g ran by</w>': 1, '   s om er set</w>': 9, '   as su red</w>': 20, '   un able</w>': 29, '   vi ll ain</w>': 14, '   la d y ship</w>': 13, '   pe ar ls</w>': 20, '   ti me ly</w>': 3, '   hon or able</w>': 21, '   hon or ably</w>': 1, '   ac cu sa tions</w>': 11, '   con tr ab and</w>': 4, '   sc oun dre l</w>': 9, '   con ta in ing</w>': 8, '   sha me ful</w>': 4, '   in fa m ous</w>': 10, '   fr on ti er</w>': 11, '   gr ac i ous</w>': 25, '   ra s cal</w>': 7, '   la z l o</w>': 3, '   t re mb le</w>': 5, '   im po si tion</w>': 4, '   di sta st e ful</w>': 5, '   cour land</w>': 1, '   l ac ke ys</w>': 1, '   wh ence</w>': 3, '   bar r ac ks</w>': 17, '   f ar e well</w>': 28, '   s ou ls</w>': 34, '   wee p</w>': 11, '   f ac ed</w>': 17, '   mor n in gs</w>': 18, '   li ve li ho od</w>': 11, '   tr in ke ts</w>': 4, '   di a mon ds</w>': 42, '   gre en fi el ds</w>': 1, '   th us</w>': 24, '   k in d red</w>': 7, '   ex i le</w>': 9, '   fri en d ly</w>': 87, '   bo y ho od</w>': 2, '   see b ac h</w>': 3, '   e mp lo y er</w>': 14, '   z i la g y i</w>': 1, '   y i el ds</w>': 3, '   in st ant</w>': 37, '   y i e ld</w>': 25, '   ri v al s</w>': 3, '   mon se i g ne u r</w>': 1, '   be st ow ing</w>': 2, '   re p in ed</w>': 1, '   re bu ked</w>': 1, '   f on d ly</w>': 2, '   qu a li ti es</w>': 20, '   t re mb l ed</w>': 1, '   lo ving</w>': 61, '   s ea ling</w>': 4, '   spi te</w>': 22, '   sin gu l ar</w>': 6, ' bra in ed</w>': 5, '   al ter na tion</w>': 1, '   qu ar re l</w>': 12, '   for t night</w>': 4, '   e lo qu ent</w>': 7, '   ki ss es</w>': 21, '   hea ven ly</w>': 7, '   de ar est</w>': 21, '   in ex pre ssi ble</w>': 1, '   s wee ter</w>': 15, '   s wee t ne ss</w>': 6, '   de fin ed</w>': 7, '   par a do x es</w>': 1, '   un ac comp ani ed</w>': 2, '   i ma g in in gs</w>': 1, '   st ead</w>': 5, '   se l do m</w>': 12, '   s ea son ed</w>': 3, '   ta st e less</w>': 3, '   de man ded</w>': 8, '   chri st en do m</w>': 2, '   ve il</w>': 8, '   en jo ying</w>': 49, '   an e w</w>': 4, '   ab st in ence</w>': 1, '   sa ti s f ying</w>': 8, '   stu b bor n ly</w>': 1, '   with he ld</w>': 3, '   with ers</w>': 3, '   p ac i fi ed</w>': 3, '   tri f les</w>': 4, '   un de man ding</w>': 1, '   o be di ent</w>': 5, '   de ce i ved</w>': 7, '   dis cu ssed</w>': 35, '   re spe c t fu lly</w>': 5, '   l ow est</w>': 16, '   n on e the less</w>': 6, '   su b sc ri be</w>': 7, '   com man ds</w>': 15, '   po s se ss ing</w>': 2, '   ha st en</w>': 2, '   un happ i est</w>': 1, '   mu tu al</w>': 33, '   mi s for t un e</w>': 8, ' e mp ty</w>': 4, '   con gra tu late</w>': 19, '   v al u ing</w>': 2, '   ta il or</w>': 12, '   ro ddy</w>': 1, '   b es</w>': 1, '   pi lli on</w>': 1, '   v a li ant</w>': 29, '   pi sto l</w>': 58, '   gu in e a</w>': 23, '   fi g</w>': 7, '   dan ces</w>': 18, '   pre t ti ly</w>': 1, '   ra ttle</w>': 19, '   re gi men tal s</w>': 1, '   ch ose</w>': 52, '   cl an cy</w>': 1, '   dan c ed</w>': 27, '   ri b b on</w>': 39, '   win ner</w>': 46, '   wa ger</w>': 15, '   su ff o l k</w>': 1, '   sp l en di d ly</w>': 5, '   ex c ee din g ly</w>': 5, '   a ver se</w>': 1, '   pe ers</w>': 3, '   re p ly</w>': 12, '   im pu dent</w>': 2, '   d om in i ons</w>': 1, '   mer i ted</w>': 1, '   ha l ter</w>': 2, '   cor on et</w>': 1, '   with d ra w ing</w>': 2, '   whi ther so ever</w>': 1, '   re ti re</w>': 38, '   re ver ed</w>': 3, '   mon ar ch</w>': 4, '   con si der able</w>': 14, '   in ti m ac y</w>': 11, '   re li an ce</w>': 3, '   ad v an ce ment</w>': 7, '   vi sc oun ty</w>': 1, '   gu sta v us</w>': 1, '   a do l ph us</w>': 1, '   thir te en th</w>': 7, '   ear l</w>': 109, '   c r ab s</w>': 15, '   so li ci tor</w>': 1, '   ga m ing</w>': 12, '   tur b in g en</w>': 1, ' mm m m</w>': 3, '   ca u ti ous</w>': 11, '   ro gu es</w>': 2, '   ad ven tur ers</w>': 2, '   coun tri es</w>': 30, '   ab ound</w>': 1, '   ac ce p ting</w>': 13, '   lo d g in gs</w>': 4, ' i lly</w>': 5, '   de li ght</w>': 11, '   ga ll ant</w>': 11, '   pre ser ver</w>': 2, '   li ve ly</w>': 11, '   w en ch es</w>': 1, '   ki l wan g an</w>': 1, '   as se mb li es</w>': 2, '   sen t ence</w>': 61, '   par en ta ge</w>': 1, '   re p li ed</w>': 6, '   an nu m</w>': 3, '   in cu r</w>': 3, '   mo ha w k</w>': 4, '   mi sha p</w>': 4, ' p ound</w>': 8, '   po ck et</w>': 124, ' bo ok</w>': 7, '   u p war ds</w>': 4, ' bo x es</w>': 1, '   ra s ca ls</w>': 3, '   ha st en ing</w>': 1, '   o a f</w>': 2, '   ru f fi an</w>': 2, '   as su ran ces</w>': 7, '   con fir ms</w>': 10, '   coun ting</w>': 68, ' house</w>': 17, '   bi r ch in</w>': 1, '   lan e</w>': 43, '   in con ven i ence</w>': 15, '   de cl in ed</w>': 4, '   ha ck ton</w>': 1, '   me ss rs</w>': 1, '   sa lo m on</w>': 1, '   br ac e gir dle</w>': 3, '   me di ta ted</w>': 1, '   se par ation</w>': 7, '   re pu di ate</w>': 1, '   du ran ce</w>': 1, '   li ti ga tion</w>': 7, '   in si st ent</w>': 3, '   as su ran ce</w>': 13, '   f re e will</w>': 3, '   tr ans ac tion</w>': 11, '   shi lling</w>': 4, '   pro vi ded</w>': 19, '   a gre e ment</w>': 52, '   ex e cu ted</w>': 25, '   cha t wi ck</w>': 2, '   ne w com be</w>': 2, '   re je ct</w>': 11, '   on er ous</w>': 2, '   1 8 </w>': 51, '   p le d ged</w>': 4, '   e dri c</w>': 1, '   min es</w>': 23, '   re de em</w>': 11, '   en cu mb ran ces</w>': 1, '   lea se</w>': 18, '   in come</w>': 27, '   bro ok sy</w>': 1, '   re be ls</w>': 22, '   l or d ship</w>': 3, '   f lo g ging</w>': 3, '   do o l an</w>': 2, '   po ttle</w>': 1, '   ne lly</w>': 1, '   brea king</w>': 120, '   g ri ev ed</w>': 1, '   f la v or</w>': 10, '   ome le tt e</w>': 13, '   ar sen i c</w>': 2, '   su c ce ss or</w>': 6, '   de par ture</w>': 12, '   w ea l th</w>': 31, '   wor th i ly</w>': 1, '   lo f ty</w>': 3, '   s wor n</w>': 31, '   cor b le u</w>': 1, '   de ter</w>': 1, '   pr in ci pl es</w>': 20, '   cha pl in</w>': 7, '   a mi able</w>': 1, '   tr ou bl ed</w>': 18, '   sc ru pl es</w>': 11, '   re fu ge</w>': 9, '   do c t ors</w>': 137, '   dis ci p le</w>': 9, '   cu ck old</w>': 1, '   comp li ca tion</w>': 7, '   i lls</w>': 2, '   t in k er ed</w>': 2, '   pu r sing</w>': 1, '   gr oun d less</w>': 1, '   mi l k ma id</w>': 3, '   re st ri ct</w>': 4, '   a mu se ment</w>': 8, '   se le c ts</w>': 2, '   pro per ly</w>': 50, '   an no y an ce</w>': 3, '   in stan ce</w>': 64, '   g out</w>': 2, '   ten ds</w>': 12, '   ro b s</w>': 3, '   vi r tu ous</w>': 3, '   d ru dge</w>': 1, '   ad ded</w>': 23, '   pen si ons</w>': 4, '   tru mp s</w>': 1, '   man n ers</w>': 49, '   bar b er</w>': 22, '   ori g in a li ty</w>': 3, '   pl u ck</w>': 4, '   ga ll an try</w>': 3, '   pu r su ed</w>': 10, '   my ri a d</w>': 4, '   dis ea ses</w>': 15, '   wh ee l ed</w>': 1, '   p an gs</w>': 2, '   a gon y</w>': 11, '   he gen he i m</w>': 1, '   v al de z</w>': 2, '   sc z ort ar s k a</w>': 1, '   sch u v a lo ff</w>': 1, '   9 1 1 </w>': 17, '   hur ri can e</w>': 22, '   qu ad ru p le</w>': 1, '   ma tt ers</w>': 144, '   f ra med</w>': 9, ' him</w>': 25, '   mu e ll er</w>': 26, '   un ac coun ted</w>': 3, '   ba you</w>': 2, '   chi l ds</w>': 27, '   ca p</w>': 55, '   wa sh out</w>': 4, '   re je c ts</w>': 4, ' du mb fuck s</w>': 1, '   n un e z</w>': 18, '   ca d et</w>': 17, '   har dy</w>': 22, '   ra y mon d</w>': 41, '   lo ca te</w>': 38, '   cor ro bor ate</w>': 6, '   p x</w>': 4, '   char g es</w>': 74, '   inter ro ga tion</w>': 25, '   ca de ts</w>': 10, ' 4 5 </w>': 24, '   i v ory</w>': 6, '   an ti ci pa tion</w>': 4, '   ha bea s</w>': 4, '   cor p us</w>': 7, '   o h h h h</w>': 9, '   d un b ar</w>': 61, '   im per son a ting</w>': 6, ' mar ti al</w>': 6, '   p ho sp h or ous</w>': 3, '   cre e k</w>': 39, '   in fir mar y</w>': 13, '   en list ed</w>': 10, '   e pi le p sy</w>': 3, '   si ck ly</w>': 3, '   what d</w>': 2, ' y ac all</w>': 1, '   e po x y</w>': 2, '   wee ded</w>': 1, '   0 3 0 0</w>': 3, '   fa ll out</w>': 8, '   ph y si ca ls</w>': 2, '   gr en ad es</w>': 11, '   mar ti al</w>': 13, '   un ti e</w>': 12, '   cho pp er</w>': 28, '   mor ph ine</w>': 30, '   ad di ct</w>': 13, '   com b at</w>': 42, '   s ca m</w>': 31, '   s wee p ing</w>': 8, '   c li ck</w>': 14, '   ro ber to</w>': 10, '   j ay</w>': 26, '   k in d ling</w>': 2, '   a fi re</w>': 4, '   l ev i</w>': 15, '   car e ers</w>': 11, '   st ee l wor k er</w>': 1, '   pa in ki ll ers</w>': 5, ' d un b ar</w>': 1, '   ta p ed</w>': 18, '   o h h h h h h</w>': 2, '   si l ent</w>': 46, '   mi s de e ds</w>': 1, '   wee ps</w>': 3, '   e pi le p ti c</w>': 4, '   att ac ks</w>': 33, '   or g ans</w>': 14, '   mi x er</w>': 16, '   po e ti c</w>': 13, ' de gre es</w>': 2, '   in ve sti ga tive</w>': 5, ' mo tive</w>': 1, '   ex pe lled</w>': 13, '   inter ce ded</w>': 2, '   se c tion ed</w>': 2, '   nu mer ous</w>': 9, '   o c ca si ons</w>': 15, '   pro c li vi ti es</w>': 1, '   un plea s an tri es</w>': 1, '   h om o se x u al</w>': 27, '   gu i s se pe</w>': 3, '   t or r es</w>': 6, '   r ac ke te er ing</w>': 3, '   in di c t men ts</w>': 3, '   d ow ner</w>': 6, '   de ta il</w>': 64, '   bea ch es</w>': 20, '   f l ori da</w>': 72, '   to x i co lo g y</w>': 3, '   bri b es</w>': 5, '   mo b ster</w>': 4, '   le c ture</w>': 30, '   con du ct</w>': 38, '   di g ni f y</w>': 5, '   an e p ha dr ine</w>': 1, '   as th ma ti c s</w>': 1, '   nu r ses</w>': 34, '   pre po st er ous</w>': 13, '   di sa pp e ar</w>': 96, '   f ra m ing</w>': 3, '   do in gs</w>': 2, '   h or se shit</w>': 17, ' 4 2 </w>': 3, '   re f le ct</w>': 20, '   un a vo i d able</w>': 3, '   cu l p ab i li ty</w>': 2, '   o s bor n e</w>': 10, '   n ever the less</w>': 15, '   p sy ch ed</w>': 7, '   wi d en</w>': 3, '   en d z one</w>': 1, '   gi an ts</w>': 2, '   lon g sho t</w>': 4, ' pri v ate</w>': 3, '   se ssi ons</w>': 20, '   2 1 0 0</w>': 1, '   0 6 3 0</w>': 1, ' co m</w>': 5, '   t ro t</w>': 5, '   k in der</w>': 7, '   gen t l er</w>': 3, '   gre e ting</w>': 10, '   in coming</w>': 8, ' c lu st er fuck</w>': 1, '   be th</w>': 94, '   can c el ed</w>': 13, '   ex er ci ses</w>': 6, '   ju les</w>': 9, '   ju li a</w>': 58, '   con fe ssi ons</w>': 7, '   da w ned</w>': 2, '   f oun ded</w>': 11, '   f al se ly</w>': 9, '   ac cu sed</w>': 38, '   s k u lls</w>': 9, '   ne w sp a per</w>': 100, '   z oo m</w>': 4, '   bu tt fuck s</w>': 2, '   f li ps</w>': 5, '   sh oo ts</w>': 20, '   in ve sti ga ted</w>': 7, '   ph en om en al</w>': 2, '   c r ac ked</w>': 23, '   st ri pp ing</w>': 3, '   of f ed</w>': 8, '   f ra g ged</w>': 4, '   rea ssi g n</w>': 2, '   re stra in ts</w>': 13, '   wi lli es</w>': 4, '   no ted</w>': 15, '   ba y on e ts</w>': 4, '   n on co m</w>': 1, '   mi c r or e cor der</w>': 1, '   j ab </w>': 12, '   ban al</w>': 5, '   ch it</w>': 4, ' ch at</w>': 6, '   tru st wor th y</w>': 11, '   sh ou ts</w>': 6, '   ru b</w>': 47, '   b re e ding</w>': 13, '   s l ou ch</w>': 1, '   un pro fe ssi on al</w>': 7, '   su spi ci ou s ly</w>': 1, '   j on a than</w>': 20, '   o hi o</w>': 39, '   do g ta gs</w>': 3, '   ban ter</w>': 3, '   su b t le ty</w>': 5, '   fin er</w>': 18, '   bri ber y</w>': 6, '   or le ans</w>': 50, '   p d</w>': 6, '   un co op er a tive</w>': 2, '   cou pl ed</w>': 3, '   de ed</w>': 28, '   s ar ge</w>': 26, '   c ru ci f y</w>': 10, ' ever</w>': 22, '   fa i ri es</w>': 3, ' ex cu se</w>': 14, '   cl ev el and</w>': 24, '   i den ti ty</w>': 38, '   mi ran da</w>': 4, '   w o l ves</w>': 15, '   in con si st en ci es</w>': 2, '   de sig ned</w>': 52, '   b oun ce</w>': 28, '   ar ri ves</w>': 22, '   ac ce s sor y</w>': 16, '   tal es</w>': 26, '   be ds</w>': 26, '   ser ge an ts</w>': 2, '   we i gh</w>': 21, '   to ck</w>': 1, ' to ck</w>': 7, '   co o p</w>': 16, '   whi sp er ed</w>': 6, '   comp ly</w>': 8, '   mar ti al able</w>': 1, '   b al li sti c s</w>': 7, '   st ri pped</w>': 10, '   h er e by</w>': 14, '   na than</w>': 68, '   spe ck</w>': 21, '   dar ke st</w>': 11, '   t ou gh en s</w>': 1, '   s ma ll er</w>': 39, '   hu l ab a lo o</w>': 1, '   cor ps</w>': 15, '   1 7 0 0</w>': 2, '   f an ning</w>': 1, '   pi ck up</w>': 19, '   u no f fi ci al</w>': 4, '   t ack</w>': 8, '   un ne ce ss ary</w>': 22, '   l z</w>': 2, '   c li cks</w>': 10, '   b la st ing</w>': 9, '   su n sh in ey</w>': 1, '   le ar n s</w>': 10, '   har sh</w>': 19, '   en tr an ce</w>': 50, '   con done</w>': 2, '   w ea k en</w>': 7, '   f ab ri c</w>': 23, '   a ver age</w>': 59, '   in ci den ts</w>': 6, '   he si ta te</w>': 13, '   de em</w>': 5, '   un wor th y</w>': 5, '   ca mp</w>': 163, '   mo ther fuck ers</w>': 23, '   co o ze</w>': 2, '   s an dr a</w>': 25, '   te mp le ton</w>': 2, '   au bur n</w>': 4, '   se me ster</w>': 13, '   in di ge sti on</w>': 5, '   r ab bi ts</w>': 25, '   char i ty</w>': 38, '   de ter min ation</w>': 7, '   p on d</w>': 18, '   pu d d l ev i ll e</w>': 1, '   b ack p ack</w>': 12, '   hi ck vi ll e</w>': 1, '   k ar l</w>': 102, '   po et</w>': 42, '   spe c t re</w>': 5, '   a sh ton</w>': 5, '   nor ther</w>': 5, '   b loo m</w>': 33, '   ne i gh b or</w>': 84, '   wi chi ta</w>': 29, '   de li ver i es</w>': 12, '   un ca t cha ble</w>': 3, ' ei gh ty</w>': 9, '   re gu la ted</w>': 4, '   h y per ten sion</w>': 3, '   he ar t fe lt</w>': 5, '   an no y an ces</w>': 2, '   b en ne t t</w>': 18, '   jo se ph ine</w>': 34, '   j en ny</w>': 60, '   2 8 </w>': 21, '   bea men</w>': 1, '   o ver loo ked</w>': 9, '   reme dy</w>': 4, '   en ti re ty</w>': 3, '   3 8 </w>': 8, '   ar i th me ti c</w>': 8, '   ca tch es</w>': 29, '   ne k kid</w>': 2, '   t an g ent</w>': 2, '   ta le</w>': 38, '   sp o on ing</w>': 4, '   sh er b et</w>': 1, '   s qu ee z ing</w>': 5, '   cl an king</w>': 1, '   bo tt les</w>': 23, '   li cking</w>': 4, '   po p si c le</w>': 4, '   bu tt er ing</w>': 1, '   ch u r n</w>': 2, '   sp la sh ing</w>': 6, '   s li pp ing</w>': 18, '   ban ging</w>': 20, '   mi l k man</w>': 4, '   c row</w>': 38, '   gra mp a</w>': 16, '   gra mp s</w>': 2, '   t re pi da tion</w>': 1, '   st ac y</w>': 36, '   par ro ts</w>': 2, '   tru th fu lly</w>': 8, ' e at</w>': 8, '   bu f fe ts</w>': 2, '   to ps</w>': 28, '   be d time</w>': 17, '   my th o lo gi es</w>': 1, '   s an ta</w>': 94, '   cla us</w>': 24, '   com b in ed</w>': 16, '   i ce ber gs</w>': 6, '   9 0</w>': 14, '   me ta ph or</w>': 9, '   ha u ling</w>': 12, '   ma m mo th</w>': 4, '   ma p le</w>': 8, '   con go</w>': 4, '   of f end</w>': 17, '   ver si ons</w>': 5, '   p an han d l er</w>': 2, '   sen i le</w>': 6, '   wi tch</w>': 100, '   s wa mp</w>': 46, '   cor ru pt</w>': 20, '   mi s l ead</w>': 2, '   n on sen se</w>': 112, '   di a p ers</w>': 6, '   bur p ing</w>': 1, '   fee din gs</w>': 1, '   p un ch lin es</w>': 1, '   re vo l ve</w>': 3, '   re vo l ves</w>': 4, '   b ind</w>': 11, '   f oo t no te</w>': 3, '   con te xt</w>': 17, '   no v el ty</w>': 10, '   pro du c ts</w>': 20, '   pre di ca ment</w>': 7, '   pl ant</w>': 100, '   mi st ers</w>': 1, '   sp ra y</w>': 30, '   f er n</w>': 3, '   dr ying</w>': 4, '   ex pla in ed</w>': 42, '   da ff o di ls</w>': 1, '   lu c ki est</w>': 8, ' f raid</w>': 15, '   tra ve lled</w>': 5, '   mo t to</w>': 17, '   a st on i sh ed</w>': 2, '   ex pe c ta tion</w>': 7, '   so f ter</w>': 4, '   re con ci le</w>': 4, '   f li r ts</w>': 2, '   pre su med</w>': 9, '   pe cu li ar</w>': 25, '   pre g n ant</w>': 117, '   clo ck work</w>': 6, '   mor o c c o</w>': 1, ' true</w>': 8, '   li k ea ble</w>': 3, '   ro man ce</w>': 38, '   che m o</w>': 3, '   wa kes</w>': 23, '   di sc oun t</w>': 20, '   fun dam en tal</w>': 6, '   p ho to s</w>': 43, '   w ea k er</w>': 17, '   m chi b b on</w>': 1, '   s wa m</w>': 6, '   se ts</w>': 32, '   c ans</w>': 18, '   w ra pp ers</w>': 1, '   du st b ins</w>': 1, '   pre sen ts</w>': 36, ' ca ts</w>': 3, '   tr ou s ers</w>': 22, '   s mi th</w>': 150, ' 1 5 0</w>': 5, '   cre ep</w>': 57, '   app ro ac h ing</w>': 24, '   ch</w>': 8, '   ke ttle</w>': 7, '   na di a</w>': 9, '   y u r i</w>': 7, '   in do ors</w>': 9, '   mar ch es</w>': 5, '   he ar t ac he</w>': 5, '   min i st ri es</w>': 1, '   lon el in ess</w>': 11, '   di sa pp o in t ment</w>': 32, '   ru ssi a</w>': 82, '   ju d g men tal</w>': 4, '   b in o cu l ars</w>': 12, '   bi r ds</w>': 130, '   cla ssi c</w>': 43, '   vi s a</w>': 28, '   ar chi te c ts</w>': 6, '   poli ti ci ans</w>': 14, '   no v g or od</w>': 1, '   no o dle</w>': 7, '   re t ard</w>': 21, '   a le x e i</w>': 2, '   so p hi a</w>': 15, '   bu ck in g ha m</w>': 5, '   qui tting</w>': 27, '   bl an k et</w>': 38, '   al b ans</w>': 1, '   b lo ke</w>': 15, '   re co ver</w>': 34, '   sa fe st</w>': 13, '   er e ction</w>': 8, '   ro o ting</w>': 4, ' w et</w>': 3, ' ti ed</w>': 4, '   te ther ed</w>': 1, '   k ar en in a</w>': 2, '   de t ac h</w>': 2, '   or ga s m</w>': 13, '   mor al s</w>': 14, '   de cen cy</w>': 25, '   ab an don</w>': 30, '   pa s sp or t</w>': 53, '   bro o dy</w>': 1, '   mi ff ed</w>': 1, '   co ck</w>': 42, '   re st room</w>': 8, '   ba sh</w>': 9, '   v in di c tive</w>': 4, '   p uni sh</w>': 20, '   pi ll o ck</w>': 1, '   e mp lo ye e</w>': 30, '   hu mi li ation</w>': 13, '   gu tter</w>': 14, '   co ver s</w>': 42, '   gra sp ed</w>': 4, '   ob ser ver</w>': 11, '   cap ac i ty</w>': 41, '   sy m pa th y</w>': 22, '   so lu tions</w>': 4, '   ro b b er</w>': 18, ' fr en zy</w>': 2, '   sy ev o d n ya</w>': 2, '   gi ra f fe</w>': 4, '   s l ower</w>': 14, ' an ts</w>': 3, '   an ts</w>': 18, '   ro o t</w>': 27, '   ro man</w>': 41, '   war m er</w>': 15, '   la z</w>': 18, '   kn ow in</w>': 13, '   cu z</w>': 43, '   ear ful</w>': 2, '   car r y in</w>': 21, ' ch a</w>': 24, '   no se y</w>': 10, ' h m</w>': 41, '   mon days</w>': 3, '   we d ne s days</w>': 5, '   cho ir</w>': 29, '   ea t in</w>': 25, '   ga z e b o</w>': 2, '   br un g</w>': 10, '   cu red</w>': 28, '   ba s k et</w>': 27, '   go o di es</w>': 10, '   to ma to es</w>': 11, '   ok r a</w>': 6, '   be ans</w>': 39, '   mi gra ine</w>': 3, '   p ne u mon i a</w>': 19, '   co pi ed</w>': 13, '   f o</w>': 10, '   an ge l a</w>': 13, '   su n days</w>': 7, '   mor n in</w>': 55, '   la z ar us</w>': 11, '   fin ger ti ps</w>': 10, '   o l</w>': 78, '   sha kin</w>': 4, '   d ye</w>': 12, ' s kin</w>': 4, '   ha v in</w>': 40, '   ban ds</w>': 10, '   ki ck er</w>': 8, '   j ar</w>': 17, '   per fu me</w>': 33, '   sc en ted</w>': 1, '   c rea ms</w>': 5, '   mo i stu ri ze</w>': 2, '   g in ger</w>': 38, '   s ou ff le</w>': 4, '   app l ying</w>': 7, ' ch er</w>': 1, '   ac h ing</w>': 1, '   gi ll</w>': 10, '   cho king</w>': 8, '   fa i th ful</w>': 25, '   de gre e</w>': 59, '   a g ri cu l ture</w>': 2, '   r on ni e</w>': 35, '   spe lls</w>': 19, '   hi tch in</w>': 1, '   ra e</w>': 18, '   fo l der</w>': 2, '   ti c</w>': 3, '   sha k y</w>': 11, '   an x i e ty</w>': 23, '   mar v </w>': 7, '   tr ans mi ssion</w>': 50, '   li ft</w>': 107, '   te h r on n e</w>': 5, '   la u gh in</w>': 15, '   dis g ust</w>': 14, '   hu st les</w>': 2, '   do pe</w>': 81, '   hu b ca ps</w>': 2, '   j es</w>': 13, '   won der in</w>': 12, '   b out</w>': 31, '   g our d</w>': 2, '   ar ab s</w>': 7, '   k no x vi ll e</w>': 1, '   bl ed</w>': 13, '   ra di a tor</w>': 19, '   d ra f ty</w>': 2, '   m end</w>': 10, '   mu th a fuck a</w>': 2, '   de ke</w>': 5, '   pa ll be ar er</w>': 2, '   c ow bo ys</w>': 32, '   ch a</w>': 6, '   gi v in</w>': 45, '   te st in</w>': 2, '   st om p</w>': 8, ' fuck er</w>': 7, '   e ll a</w>': 10, '   ma e</w>': 9, '   f oo li sh ne ss</w>': 4, '   dr in kin</w>': 22, '   hea d ac he</w>': 56, ' ch u</w>': 8, '   for gi ves</w>': 2, '   d ev i lled</w>': 2, '   fi l th</w>': 25, '   fi gh t in</w>': 16, '   bo il</w>': 17, '   han d l in</w>': 1, '   re stra int</w>': 6, ' cu r s in</w>': 1, '   st ea ks</w>': 9, '   po ta to es</w>': 20, '   bi s cu i ts</w>': 12, '   g ri ll in</w>': 1, '   tra mp</w>': 24, ' bu ck et</w>': 2, '   ju kes</w>': 1, '   bi le</w>': 2, '   ru tt in</w>': 2, '   al mi gh ty</w>': 30, '   s an c ti ty</w>': 1, '   de gra de</w>': 2, '   hea l in</w>': 1, '   f ar m in</w>': 2, '   1 3 </w>': 34, '   par able</w>': 1, '   s ower</w>': 1, '   to ss</w>': 44, '   wa y side</w>': 1, '   i tch es</w>': 5, '   ni tra te</w>': 8, '   f er ti li z er</w>': 6, '   so y</w>': 7, '   gr ow in</w>': 9, '   men s</w>': 1, '   or g ani ze</w>': 20, '   con ser v ation</w>': 1, '   ro ta te</w>': 3, '   so f a</w>': 22, '   mu le</w>': 18, '   h m</w>': 57, ' m m</w>': 22, ' h at</w>': 14, '   ho ll a</w>': 3, ' e lf</w>': 6, '   ho ar se</w>': 5, '   fa i th less</w>': 2, '   gen er ation</w>': 17, '   si ck ne ss</w>': 26, '   pa th</w>': 55, '   a i m</w>': 53, '   cu re</w>': 87, '   wi ck e d ne ss</w>': 8, '   tru cks</w>': 48, '   cha in in</w>': 1, '   cha s in</w>': 3, '   cha in ed</w>': 14, '   wi ts</w>': 12, ' fore</w>': 12, ' ir</w>': 8, '   he ar in</w>': 13, '   n n</w>': 2, ' n n</w>': 3, ' r on ni e</w>': 1, '   te h</w>': 1, '   gen er o si ty</w>': 10, '   p in ks</w>': 1, '   ri d in</w>': 17, '   me an in</w>': 4, '   ta pped</w>': 19, '   1 6 </w>': 29, '   re y no l ds</w>': 35, '   ad di ction</w>': 11, '   h ers</w>': 58, '   bea t in</w>': 18, '   s na tch</w>': 10, '   i tch</w>': 10, '   do o le</w>': 1, '   we ed</w>': 35, '   p rea ch er</w>': 52, '   bo j o</w>': 2, '   dea l in</w>': 14, '   che f</w>': 27, '   supp er</w>': 61, '   to e</w>': 33, '   wi ck</w>': 13, '   sin n ers</w>': 4, '   st d</w>': 1, '   p rea ch in</w>': 1, '   f oo l in</w>': 13, '   h un t in</w>': 18, '   bri tch es</w>': 4, '   p rea ch</w>': 16, '   tur n in</w>': 8, '   ch ee k</w>': 27, '   s la p</w>': 34, '   s ki pped</w>': 22, '   bra g g in</w>': 5, ' b ought</w>': 1, '   mu l ch</w>': 2, '   bu ff et</w>': 12, '   cl ou ds</w>': 30, '   pe ar ly</w>': 3, '   ga tes</w>': 30, '   sh on ey</w>': 1, '   wa sh ed</w>': 57, '   en d in</w>': 1, '   b le e d in</w>': 7, '   sti tch</w>': 9, ' ya</w>': 60, '   sh out in</w>': 1, '   cer ea l</w>': 13, '   mo o ds</w>': 13, '   list en in</w>': 14, '   wa g g in</w>': 1, '   sh out</w>': 25, '   a i s le</w>': 23, '   di f</w>': 4, ' n t</w>': 5, '   dre ss in</w>': 2, '   ve sts</w>': 2, '   shi pped</w>': 13, '   bo o ti e</w>': 1, '   sp o on ful</w>': 2, '   ni g ga z</w>': 1, '   re d ne cks</w>': 3, '   s ki tt les</w>': 2, ' ways</w>': 1, '   ba ll ers</w>': 1, '   ni ck</w>': 303, '   jo y ce</w>': 17, '   ri ce</w>': 30, '   c ro tch</w>': 6, '   ro ck et</w>': 30, '   mo t or c y c le</w>': 13, '   i ch</w>': 19, ' ir o</w>': 1, '   ma t su </w>': 1, ' mo to</w>': 1, '   mo to</w>': 1, '   pa d</w>': 21, '   f ea th ers</w>': 13, '   pro se cu tor</w>': 13, '   que en s</w>': 21, '   god z i ll a</w>': 6, '   d rea ded</w>': 3, ' h our</w>': 24, '   a si an</w>': 30, '   shi ts</w>': 8, '   st re tch</w>': 54, '   ge i sh a</w>': 1, '   sa d der</w>': 4, '   e th i op i a</w>': 6, '   po con o s</w>': 1, '   ab o lo fi a</w>': 3, '   ca ve llo</w>': 5, '   su bar u</w>': 1, '   ja p</w>': 7, '   sc un gi ll i</w>': 1, '   an ti ci pa ted</w>': 7, '   ni ck la us</w>': 4, '   da w n</w>': 72, '   go om ba h</w>': 1, ' bea ter</w>': 1, '   re gu la tion</w>': 11, '   ca sin o s</w>': 9, '   we i s mu ll er</w>': 2, ' pa pp y</w>': 2, '   pa pp y</w>': 11, '   i chi r o</w>': 5, '   k a m pa i</w>': 3, '   s qu id</w>': 27, '   con ni e</w>': 28, '   pla te</w>': 67, '   o ha sh i</w>': 3, '   com pa d re</w>': 5, '   su ga i</w>': 7, '   ca vi ar</w>': 26, '   ha m</w>': 15, '   ob li ga ted</w>': 6, '   ga i i j in</w>': 5, '   be pp u</w>': 1, ' s an</w>': 3, '   y an k</w>': 11, '   k ob o</w>': 7, '   god father</w>': 25, '   pr in ting</w>': 9, '   ma t su mo to</w>': 1, '   y a k u z a</w>': 4, '   ba y on n e</w>': 1, '   ti de</w>': 32, '   bro il ed</w>': 1, '   k el p</w>': 1, '   so y b ean</w>': 1, '   cu r d</w>': 1, '   d ev i ls</w>': 13, '   chi k u w a</w>': 1, '   ha mp en</w>': 1, '   k o bu </w>': 1, '   k on na y a k u</w>': 1, '   g an mo do k i</w>': 1, '   la und ry</w>': 39, '   to l er</w>': 1, '   to k y o</w>': 20, '   s ni tch</w>': 15, '   pla t for m</w>': 15, '   po ts</w>': 10, '   j er ry</w>': 207, '   din k y</w>': 5, '   do o</w>': 11, '   mo town</w>': 2, ' go o d ne ss</w>': 5, '   lo mb ar d i</w>': 2, '   re d for d</w>': 4, '   br on son</w>': 7, ' na m</w>': 9, '   o y o</w>': 1, '   ran ged</w>': 2, '   k y o</w>': 3, '   in cen se</w>': 1, '   t rea sur es</w>': 10, '   sha d es</w>': 19, '   qui ck er</w>': 22, '   le ar ner</w>': 7, '   bar bar i an</w>': 5, '   coun ter fe i ting</w>': 6, '   ca tch ing</w>': 48, '   con k l in</w>': 23, '   a ll e ys</w>': 9, '   man u f ac tur er</w>': 6, '   r en tal</w>': 12, '   de por ted</w>': 4, '   for ei g n ers</w>': 11, '   wal t z</w>': 11, '   uni for ms</w>': 26, '   con vi c ts</w>': 8, '   un pro te c ted</w>': 5, '   de ce it</w>': 5, '   bu lli on</w>': 1, '   cu r r en tly</w>': 22, '   sy n di ca tes</w>': 3, ' hon or</w>': 3, ' u ty</w>': 1, '   o y a bu n</w>': 1, '   sy n di ca te</w>': 12, '   en gra ver</w>': 1, '   pro du c ed</w>': 12, '   se qu en ti ally</w>': 2, '   nu mb er ed</w>': 7, '   sp or ting</w>': 11, '   under e sti ma te</w>': 19, '   pr is</w>': 5, '   se ba sti an</w>': 35, '   sto cked</w>': 5, '   bi ome chan i c s</w>': 2, '   ro y ces</w>': 1, '   f er r ar i</w>': 19, '   r ac ing</w>': 16, '   ob st ru ct</w>': 2, '   re p li ca tion</w>': 3, '   er r or</w>': 32, '   ne w ly</w>': 9, '   for med</w>': 15, '   st r and</w>': 6, '   mu ta tion</w>': 6, '   re pre ss or</w>': 1, '   pro te in</w>': 15, '   b lo cks</w>': 48, '   e th y l</w>': 2, '   su l f on ate</w>': 1, '   al k y la ting</w>': 1, '   mu ta g en</w>': 1, '   c rea tes</w>': 17, '   re com b in ation</w>': 1, '   under gone</w>': 2, '   re ver sion</w>': 1, '   re ver t ant</w>': 1, '   sin ks</w>': 4, '   bl un t</w>': 14, '   al ter ation</w>': 6, '   ev o l ve ment</w>': 3, '   ma k ers</w>': 5, '   co ding</w>': 3, '   se qu ence</w>': 17, '   re vi sed</w>': 2, '   ra di cal</w>': 17, '   mo di fi ed</w>': 6, '   ba tty</w>': 8, '   ro y</w>': 92, '   t y re ll</w>': 10, '   s mi li ar</w>': 1, '   ac c el er a ted</w>': 7, '   de cre pi tu de</w>': 1, '   ne x us</w>': 19, '   an dro i ds</w>': 3, '   de e tch u m</w>': 1, '   ad mi r able</w>': 7, '   ma st er pi e ce</w>': 17, '   de ck ard</w>': 8, '   el don</w>': 2, '   ma ss ac red</w>': 2, '   ter min a ted</w>': 12, '   r ac ha el</w>': 1, '   can a pt</w>': 1, '   vi ll a</w>': 11, '   vi ta</w>': 4, '   o l y m pi a</w>': 2, '   br y ant</w>': 9, '   a sp ir in</w>': 13, '   sp in ner</w>': 1, '   r und own</w>': 6, '   app li can ts</w>': 4, '   e sp er</w>': 5, '   con c lu ded</w>': 6, '   ir re gu l ar</w>': 6, '   in du st ri al</w>': 17, '   ho l d en</w>': 9, '   vo ight</w>': 4, ' k a mp ff</w>': 4, '   de te ct</w>': 19, '   2 3 1 </w>': 2, '   mor p ho lo g y</w>': 1, '   in fi l tra te</w>': 3, '   z h or a</w>': 1, '   in ce pt</w>': 3, '   2 0 1 7 </w>': 3, '   a th le ti c</w>': 15, '   con di tion ing</w>': 15, '   ab i li ti es</w>': 13, '   sp ac ers</w>': 1, '   d ome sti c</w>': 18, '   fe ma les</w>': 12, '   lon g ev i ty</w>': 2, '   e sti ma tion</w>': 5, '   su m mar y</w>': 3, '   e mu late</w>': 2, ' e chan i s m</w>': 1, '   re sp on ses</w>': 4, '   s k u ll</w>': 52, '   o c ci pi tal</w>': 2, '   tra u ma s</w>': 2, '   de bi li ta ting</w>': 1, '   se ver</w>': 6, '   fa st est</w>': 15, '   si x es</w>': 4, '   con st ru c ted</w>': 7, ' f le sh</w>': 2, '   en o gen i c</w>': 1, '   tr ans f er</w>': 57, '   con ver sion</w>': 4, ' per pe tu a ting</w>': 1, '   par a</w>': 6, ' ph y si cal</w>': 7, '   d ev e lo p ed</w>': 42, '   e mi gra tion</w>': 2, '   e qui pped</w>': 26, '   for ger</w>': 3, '   st ee l</w>': 58, '   mi ll</w>': 57, '   bu m</w>': 79, ' st e er</w>': 1, '   per son a li ti es</w>': 9, '   2 0 1 5 </w>': 1, '   pro sit</w>': 4, '   co ck ro ac h es</w>': 20, '   e mo tions</w>': 38, '   e m pa th i ze</w>': 3, ' si x es</w>': 2, '   pe e</w>': 69, '   w r in k l ed</w>': 4, '   fuck en</w>': 9, '   un hea l th y</w>': 7, '   mo ti vi ty</w>': 1, '   n er ves</w>': 50, '   si li c one</w>': 3, '   te ch s</w>': 2, '   z on ers</w>': 1, '   c lu b s</w>': 42, '   au di tion</w>': 33, '   a lls</w>': 3, '   ta f fe y</w>': 1, '   und re ss</w>': 8, '   g li mp se</w>': 20, '   j ee z us</w>': 1, '   un sa v ory</w>': 2, '   re pu l si ve</w>': 7, ' ex p lo i ted</w>': 1, '   ar ti sts</w>': 19, '   fe der ation</w>': 75, '   an di es</w>': 1, '   si mu late</w>': 3, '   mo tions</w>': 9, '   app ear an ces</w>': 9, '   f re er</w>': 1, '   na ps</w>': 4, '   e f fe c tive</w>': 36, '   in se cu re</w>': 10, '   comp li ca tes</w>': 3, '   o y st ers</w>': 2, '   han gs</w>': 13, '   be ar s kin</w>': 1, '   nu de</w>': 12, '   wa s p</w>': 2, '   wri st</w>': 24, '   bu tt er f ly</w>': 15, '   bu rea u c r at</w>': 2, '   ar ti fi ci al</w>': 13, '   de te c ting</w>': 4, '   min i s cu le</w>': 2, '   hu man o id</w>': 4, '   ro bo t</w>': 9, '   b en e f it</w>': 45, '   ha z ard</w>': 14, '   per t in ent</w>': 3, ' la y</w>': 5, '   p ack et</w>': 10, '   tr an sc en ded</w>': 2, '   re li es</w>': 3, '   in f al li ble</w>': 2, '   b as</w>': 2, '   f oo l pro of</w>': 6, '   ca pi ll ary</w>': 2, '   di la tion</w>': 2, '   b lu sh</w>': 9, '   f lu c tu ation</w>': 1, '   in vo l un t ary</w>': 1, '   ir is</w>': 34, '   e m pa th y</w>': 5, '   sti mu l us</w>': 1, '   wh a tch a</w>': 36, '   t or to i se</w>': 3, '   tur t le</w>': 11, '   t ow ar ds</w>': 42, '   h y po the ti cal</w>': 6, '   f ac tor</w>': 22, '   pro vi si ons</w>': 7, '   g lan ds</w>': 10, '   me th u se la h</w>': 2, '   sy n dro me</w>': 30, '   dan c er</w>': 30, '   ro de o</w>': 4, '   dan c ers</w>': 7, '   a da m</w>': 111, '   we b b er</w>': 9, '   eve</w>': 224, '   n in a</w>': 5, '   ar on</w>': 3, '   sha llow</w>': 12, '   inter ru p tion</w>': 8, ' du mb </w>': 3, ' se ver al</w>': 3, '   poli tely</w>': 6, '   com o</w>': 16, '   so p hi e</w>': 31, '   ho spi ta ble</w>': 5, '   li mp ing</w>': 11, '   vi cen te</w>': 1, '   un sa fe</w>': 3, '   t ro y</w>': 15, '   1 9 6 2 </w>': 3, '   sa p</w>': 17, '   ju li et</w>': 53, '   sur pri sing</w>': 17, '   hea l th y</w>': 77, ' mu t ant</w>': 2, '   sig n al s</w>': 20, '   je w el ry</w>': 51, '   fin ger na i ls</w>': 25, '   un im pre ssed</w>': 1, ' yeah</w>': 93, '   s ci en ti st</w>': 45, '   s k ate</w>': 9, '   wa z o o</w>': 3, '   under p an ts</w>': 12, '   wh a ta ya</w>': 3, '   sur f er</w>': 14, '   gr un ge</w>': 1, '   e u r o</w>': 4, '   b ack y ard</w>': 21, ' a la s k a</w>': 1, ' s now</w>': 6, ' hu m</w>': 10, '   pro mi sing</w>': 22, '   pa sa den a</w>': 14, '   bi tes</w>': 29, '   gen i e</w>': 5, '   me an er</w>': 8, '   lo ck er</w>': 47, '   re fri ger a ted</w>': 7, '   p ou l try</w>': 7, '   to b ac c o</w>': 36, '   pe pp er</w>': 20, '   bu tt s</w>': 18, ' pa ying</w>': 4, '   de me an ing</w>': 4, ' men</w>': 9, '   b ru sh ed</w>': 3, ' won der ful</w>': 5, '   c ran ked</w>': 2, ' z oo i e</w>': 1, ' cu t</w>': 11, ' gu n</w>': 11, ' mi ster</w>': 10, '   an dre tt i</w>': 2, '   f re e way</w>': 28, '   di ma g gi o s</w>': 1, '   ro b in s ons</w>': 4, '   ro g ers</w>': 24, '   h or n s by</w>': 3, '   st ran g ers</w>': 53, '   pre di ct</w>': 23, '   l a</w>': 156, '   ju d ging</w>': 21, '   hea th c li ff</w>': 2, '   me mor ab i li a</w>': 4, '   man t le</w>': 2, '   b on</w>': 15, '   so ir</w>': 1, '   ma de mo i se ll e</w>': 23, ' ne e</w>': 1, '   me l k er</w>': 1, '   de ca y</w>': 8, '   oo oo o h h h h</w>': 1, '   ca ge y</w>': 4, '   dis se mb l ers</w>': 1, '   com mi es</w>': 7, '   b om b s</w>': 38, '   co ll a p sed</w>': 13, '   poli t bur o</w>': 1, '   beli ev es</w>': 67, ' dad</w>': 15, '   cra sh ed</w>': 29, '   u k ra in i an</w>': 1, '   g ran d par en ts</w>': 14, '   im mi gra ted</w>': 1, '   ru s</w>': 1, ' k o v </w>': 1, '   po i son</w>': 87, '   bo ok st ore</w>': 20, ' stay</w>': 16, ' a du lt</w>': 2, '   mer c i</w>': 6, ' est</w>': 13, ' th o ma s</w>': 1, ' we b b er</w>': 1, ' s k a tes</w>': 1, '   re de sig ned</w>': 2, '   run ner</w>': 19, '   no c tur na l</w>': 7, '   ma m ma l</w>': 4, '   b at</w>': 91, '   pa in ting</w>': 70, '   ac or n</w>': 5, '   ha b en</w>': 2, '   si e</w>': 22, '   e tu as</w>': 1, '   ne tt es</w>': 1, '   le der</w>': 1, '   sor g en</w>': 1, '   bi tt e</w>': 1, '   da fu r</w>': 1, '   da s</w>': 6, '   ge p ack</w>': 1, '   sor g f al ti c</w>': 1, '   be han de l d t</w>': 1, '   ger a de</w>': 1, '   a us</w>': 3, '   dan n</w>': 1, '   en</w>': 32, '   ar te</w>': 1, '   vo lu p tu s</w>': 1, '   qu e</w>': 17, '   b ons</w>': 1, '   te mp s</w>': 2, '   r ou l</w>': 1, '   te mp us</w>': 1, '   fu g it</w>': 1, '   la t in</w>': 31, '   ex am</w>': 17, ' in ve st ment</w>': 1, ' cer ti fi ca tes</w>': 1, '   b ye</w>': 204, ' see ms</w>': 5, '   to y</w>': 56, ' be e f</w>': 1, '   pa t ti es</w>': 3, '   v ain</w>': 18, '   ru st ok o v </w>': 1, '   re bu i lt</w>': 4, '   st ink</w>': 34, '   m t</w>': 8, '   m ck in le y</w>': 7, '   hi gh est</w>': 46, '   win t ers</w>': 6, '   j un ea u</w>': 2, '   ha ck er</w>': 18, '   l ar ge st</w>': 14, '   an ch or age</w>': 4, '   go tch a</w>': 24, '   ca pi to l</w>': 19, '   1 8 6 7 </w>': 1, '   se w ard</w>': 1, '   ti dy</w>': 11, '   1 9 5 9 </w>': 3, ' bu mp s</w>': 2, '   di on n e</w>': 1, '   nu t</w>': 67, '   lu cy</w>': 62, '   u r ch in</w>': 1, '   s ea we ed</w>': 7, ' nor i</w>': 1, '   su sh i</w>': 13, '   ce il in gs</w>': 3, '   fif te en th</w>': 7, '   be d bu gs</w>': 8, '   di al</w>': 21, ' ex ce pt</w>': 10, '   di e se l</w>': 7, ' e lo p ed</w>': 1, '   lo cks</w>': 32, '   cha sed</w>': 28, '   vo l un te er ed</w>': 21, '   bu z z</w>': 152, ' always</w>': 14, ' try</w>': 6, '   ni gh t f all</w>': 11, '   h or ri f ying</w>': 6, '   c ars</w>': 159, '   re si sts</w>': 5, '   ob s er</w>': 1, '   s an d wi ch es</w>': 31, '   re la ti ves</w>': 39, '   po st c ard</w>': 15, '   nu t case</w>': 6, '   ca l te ch</w>': 1, '   b on a fi de</w>': 2, '   fu r ni ture</w>': 38, ' coun try</w>': 3, ' hell</w>': 24, ' sur vi ved</w>': 1, '   en ter ta in ed</w>': 9, ' con si der ably</w>': 1, '   ca l v in</w>': 30, '   hou se hold</w>': 27, '   mu t an ts</w>': 14, '   au tom o bi les</w>': 3, ' mu l t i</w>': 1, '   ma s cu l ine</w>': 3, ' si mu l t an e ou s ly</w>': 1, ' co ver</w>': 1, '   hu m</w>': 13, '   sur vi v ors</w>': 28, '   su b spe ci es</w>': 1, '   ar cha i c</w>': 2, '   co ll o qui a li s m</w>': 1, '   r ou gh ly</w>': 17, '       </w>': 127, '   gen er a tor</w>': 18, '   ho pe ful</w>': 6, '   g l ow</w>': 17, '   b al on ey</w>': 12, '   gu e ss ing</w>': 30, '   pro mi ses</w>': 35, ' youn g</w>': 9, '   ca ved</w>': 4, '   ha tch way</w>': 1, '   tr an qui li z ers</w>': 7, '   tr an qui li z er</w>': 7, '   cre e ps</w>': 35, '   c ri s p</w>': 5, ' done</w>': 17, '   at om i c</w>': 24, '   ju mp ing</w>': 51, '   b ean</w>': 27, '   k ra ft</w>': 3, '   tr ou b le some</w>': 3, '   ra di o ac tive</w>': 16, '   au to ma ti ca lly</w>': 17, '   cu b an</w>': 25, '   me an time</w>': 56, '   p ra y er ful</w>': 1, '   stan ce</w>': 3, '   dar n</w>': 30, '   r he o st at</w>': 1, '   pe te</w>': 129, '   sh an a</w>': 1, '   gi ll ro y</w>': 1, '   ra l p h</w>': 31, '   la u r en s</w>': 1, '   or char ds</w>': 4, ' nice</w>': 16, '   i b m</w>': 5, '   po l ar o id</w>': 5, ' god damn</w>': 13, ' cer ta in ly</w>': 4, '   la mb s</w>': 10, '   s we pt</w>': 10, '   whi sp er ing</w>': 18, '   o d de st</w>': 1, '   app ear ing</w>': 6, '   we ir do s</w>': 7, '   s lu ts</w>': 13, ' s we et</w>': 6, '   e du ca te</w>': 8, '   c ru mb </w>': 5, '   tr ou ss ea u</w>': 1, ' wor ks</w>': 3, '   na p k ins</w>': 12, '   sen sing</w>': 6, '   la u r en</w>': 25, ' e le g ant</w>': 1, '   stra p less</w>': 2, '   app ar el</w>': 1, ' en ded</w>': 1, '   sp r in gs</w>': 23, ' town</w>': 13, ' fi le</w>': 5, '   ne go ti able</w>': 6, ' su sta in ing</w>': 1, ' li ev </w>': 1, ' able</w>': 5, ' p ower</w>': 4, '   mar king</w>': 12, '   ra di al</w>': 2, '   e je c ting</w>': 2, '   s ar</w>': 1, '   hel o</w>': 2, '   secon dar i es</w>': 1, '   de cl ar ing</w>': 4, '   ter mi tes</w>': 6, '   s an dy</w>': 66, '   g y n de</w>': 4, '   i ll ne ss</w>': 31, '   p or</w>': 10, '   che ers</w>': 34, '   ti d</w>': 2, '   su a ve st</w>': 2, '   ho l der</w>': 7, '   su a ve</w>': 6, '   al er ted</w>': 8, '   af fa ir s</w>': 51, ' dre ssed</w>': 5, '   v a ll en s</w>': 14, '   de fi an ce</w>': 3, '   se w ed</w>': 6, '   vi st a</w>': 14, '   beau mon t</w>': 47, '   d on ny</w>': 33, '   h mm mm mm m m</w>': 2, '   su cking</w>': 33, '   cra z in ess</w>': 8, '   sh h h h h h</w>': 10, '   wh a ti ya</w>': 14, '   c r y in</w>': 13, '   be ee e e</w>': 3, '   ni pp les</w>': 9, '   s n ea ked</w>': 8, '   sp ra y ed</w>': 7, '   re for med</w>': 6, '   whi st le</w>': 39, '   li p sti ck</w>': 29, '   fuck head</w>': 7, '   b our b on</w>': 26, '   ki d na pped</w>': 50, '   ki d na p</w>': 21, '   whi tes</w>': 19, '   p ab st</w>': 2, '   he in e k en</w>': 5, '   wan ch a</w>': 2, '   fri en da</w>': 3, ' go o d night</w>': 11, '   d un can</w>': 19, '   h in es</w>': 2, '   i v y</w>': 33, '   vi ta min s</w>': 12, '   je ff</w>': 158, '   stra pped</w>': 12, '   ro b ins</w>': 10, '   ge l for d</w>': 2, '   fa ther ly</w>': 5, '   al li ga tor</w>': 9, '   bri e f case</w>': 27, '   sta ir case</w>': 12, '   bl in ding</w>': 5, ' ye llow</w>': 3, '   supp li ed</w>': 10, '   da i ry</w>': 12, '   hon k</w>': 3, ' s l ow</w>': 7, '   go of</w>': 5, '   go of ed</w>': 2, '   ni f ty</w>': 7, '   sta ir well</w>': 10, '   go o d lu ck</w>': 1, '   se ven th</w>': 36, '   je ho v ah</w>': 3, ' a wa ke</w>': 2, '   o ver a lls</w>': 3, '   sp ra ying</w>': 3, '   st ran ge st</w>': 14, '   ga in ing</w>': 15, '   ob ser ve</w>': 23, '   oo oo o</w>': 4, '   cre e p y</w>': 23, '   cor on er</w>': 25, '   stu d ying</w>': 47, '   m ee t in</w>': 10, '   hi tt ers</w>': 4, '   de ci d es</w>': 27, '   dis man t le</w>': 3, '   pro di g y</w>': 6, '   il</w>': 24, '   du ce</w>': 2, '   gu ss y</w>': 1, '   s ni ff ing</w>': 14, '   re uni ons</w>': 4, '   sa kes</w>': 22, '   s not</w>': 9, ' no sed</w>': 4, '   au gu st us</w>': 1, '   pi ss an ts</w>': 1, '   gir li sh</w>': 6, '   s w oo p</w>': 3, '   sa g g in</w>': 1, '   an k les</w>': 15, '   fi gh ten</w>': 1, '   sc ra pp in</w>': 1, '   con n or</w>': 11, '   s me ck er</w>': 1, '   pro to co l</w>': 13, '   par af f in</w>': 4, '   ca sin gs</w>': 3, '   me tal</w>': 78, '   fa u c et</w>': 4, '   d ra in</w>': 20, '   ro c c o</w>': 16, '   de ll a</w>': 8, '   t it</w>': 17, '   m ee e</w>': 3, '   ra ff le</w>': 3, '   po sted</w>': 15, '   f l y ers</w>': 3, '   la k ev i e w</w>': 3, '   de l i</w>': 6, '   ro lling</w>': 35, '   ro lled</w>': 31, '   hi tter</w>': 11, '   na i ling</w>': 5, ' de p th</w>': 2, '   cor le one</w>': 42, '   re mor se ful</w>': 1, '   d on na</w>': 24, '   mu r der ers</w>': 30, '   mo le st ers</w>': 3, '   i ll u stra tes</w>': 1, '   di ver si ty</w>': 1, '   fuck s</w>': 46, '   po pp a</w>': 6, '   ma s ks</w>': 20, ' sh oo ter</w>': 2, '   bi ck er in</w>': 1, '   ru ck us</w>': 7, '   tr oun c ed</w>': 1, '   pa tty</w>': 11, '   si bea l</w>': 1, '   re vo l ver</w>': 3, '   ci ti z en</w>': 77, '   gra y</w>': 68, '   be ard</w>': 26, '   f lu ent</w>': 4, '   rea p</w>': 3, '   v ar ying</w>': 2, '   u r ge</w>': 23, '   le ss er</w>': 6, '   b oun ds</w>': 3, '   cor ru p tion</w>': 14, '   do ma in</w>': 9, '   e mb r ace</w>': 15, '   spi ll</w>': 33, '   s ki es</w>': 11, '   s ought</w>': 15, ' che ck o v </w>': 1, ' fuck in</w>': 14, '   he si ta tion</w>': 10, '   re mor se</w>': 13, '   t ea med</w>': 2, '   of f en der</w>': 3, '   ca tch in</w>': 5, '   stra y</w>': 12, '   ho p s co tch</w>': 3, '   ma fi o so s</w>': 1, '   ju mp s</w>': 15, '   sh</w>': 36, '   ro pe</w>': 77, ' to t in</w>': 1, '   s w ea t in</w>': 6, '   d ra g g in</w>': 7, '   ra mb o</w>': 13, '   f l ou ri sh</w>': 1, '   sp ani sh</w>': 50, '   i v an</w>': 11, ' c ro ss</w>': 12, ' pen ny</w>': 3, '   pre op er a tive</w>': 1, '   fa ir er</w>': 2, '   a a a w w</w>': 1, '   fe min i sts</w>': 3, '   bu l b</w>': 10, '   mon sig n or</w>': 1, '   ro z en gu r t le</w>': 3, '   ba u m gar t ner</w>': 1, '   pri cks</w>': 8, '   d om in an ce</w>': 2, '   1 9 0 0</w>': 2, '   wi der</w>': 8, '   th u mb </w>': 40, '   c ro s so ver</w>': 4, ' wal king</w>': 2, '   s cu m</w>': 33, '   in f ra st ru c ture</w>': 5, '   co p le y</w>': 1, '   pla z a</w>': 19, '   f ence</w>': 56, '   re ta li ation</w>': 1, '   e th i cal</w>': 14, ' ta sti c</w>': 2, '   sh oo t ers</w>': 7, '   do lly</w>': 8, '   o lo g y</w>': 1, '   p ee per man</w>': 1, ' per</w>': 5, '   clo sure</w>': 10, '   d ru g gi e</w>': 1, '   bo o th s</w>': 3, '   du ff y</w>': 28, '   in t ent</w>': 24, '   co in ci den ces</w>': 7, '   mo b st ers</w>': 2, '   com mon w ea l th</w>': 8, '   bro a d</w>': 58, '   da y li ght</w>': 35, '   s lot</w>': 12, '   an g les</w>': 12, '   a im ed</w>': 7, '   ir on</w>': 46, '   gre en ly</w>': 7, '   d ow n w ard</w>': 3, '   c ri ss</w>': 6, ' c ro ssed</w>': 5, '   ex i ted</w>': 2, '   e ye ba lls</w>': 13, '   di st in ct</w>': 10, '   si ci li ans</w>': 5, '   gre e ks</w>': 8, '   cu l tur es</w>': 8, '   pen ni es</w>': 14, '   per king</w>': 1, '   ab nor ma lly</w>': 2, '   si z ed</w>': 4, '   hi t man</w>': 9, '   f re u d</w>': 16, '   fa g</w>': 30, '   s cu m ba gs</w>': 9, '   dro pp ing</w>': 44, '   un re la ted</w>': 11, '   pe ons</w>': 2, '   s qu ab b ling</w>': 2, ' l ow</w>': 5, '   le m on</w>': 26, '   c ru sh er</w>': 15, '   no tch</w>': 7, '   c ru sh in</w>': 1, '   ni p</w>': 8, '   wai t re ss</w>': 42, ' 0 0</w>': 37, '   an vi l</w>': 1, '   ro c</w>': 6, '   li a</w>': 3, ' bi li ty</w>': 2, '   a im ing</w>': 15, '   mu sh</w>': 15, '   v in cen z o</w>': 6, '   y a k a ve tta</w>': 1, '   j er ks</w>': 17, '   ti tty</w>': 11, '   mi ss es</w>': 21, '   li ber a ting</w>': 4, '   co ck su ck ers</w>': 9, '   mu r p h</w>': 2, '   w o p</w>': 10, '   mi cks</w>': 2, '   per man ent</w>': 29, '   pi mp s</w>': 8, '   ha ll el u ja h</w>': 11, '   j af f ar</w>': 1, '   l one</w>': 11, '   ran ger</w>': 29, ' hea v y</w>': 1, '   beli e f s</w>': 10, '   in de ci sion</w>': 2, '   s ar ca sti c</w>': 9, '   h in ts</w>': 12, '   con du it</w>': 3, '   e th i c s</w>': 23, '   loo p ho les</w>': 1, '   fi st</w>': 13, '   te ch ni ca lly</w>': 33, '   d om in us</w>': 1, '   om in us</w>': 1, '   god dam it</w>': 13, '   ha f ta</w>': 22, '   g ran t</w>': 145, '   ru b s</w>': 8, '   vi v a</w>': 2, '   he e</w>': 19, '   he e e</w>': 2, '   be v o</w>': 1, '   di g n an</w>': 57, '   che ck point</w>': 4, '   tr ac tor</w>': 16, '   f lo at</w>': 29, '   sa il bo at</w>': 5, '   app le j ack</w>': 15, '   c rs</w>': 3, '   k u m ar</w>': 4, '   an th on y</w>': 48, '   tri pped</w>': 10, '   pu l se</w>': 25, '   wal ki e</w>': 8, '   tal ki e</w>': 5, '   de ser ted</w>': 17, '   sc ar e c row</w>': 9, '   in e z</w>': 7, '   s wi f ty</w>': 5, '   li ab i li ty</w>': 11, '   s mo other</w>': 5, '   p i</w>': 9, '   sa fe c r ack er</w>': 2, '   je ans</w>': 14, '   s ou th we st</w>': 10, '   imp or ted</w>': 8, '   f oo ds</w>': 9, '   st eve</w>': 111, '   h in ck le y</w>': 3, '   st or age</w>': 50, '   j ani tor</w>': 15, '   bra ts</w>': 4, '   sc ru b b ing</w>': 4, ' 3 8 3 </w>': 2, '   di a gra m</w>': 2, '   e le c t ro cu ted</w>': 9, '   ho t wi re</w>': 5, '   me x i c ans</w>': 16, '   fri es</w>': 26, '   co l der</w>': 7, '   w ea ther for d</w>': 1, '   fran ti ca lly</w>': 2, '   lan g st on</w>': 5, '   car di op u l mon ary</w>': 1, '   re ci ta tion</w>': 2, '   c p r</w>': 2, '   ve ins</w>': 24, '   car men</w>': 15, '   ani ta</w>': 8, '   re co g ni z ing</w>': 8, '   i den ti ti es</w>': 8, '   man ho le</w>': 3, '   ar mor ed</w>': 16, '   wor e</w>': 61, '   c ow boy</w>': 67, '   t rea ds</w>': 2, ' 1 1 </w>': 68, '   un lo ck</w>': 19, '   a i s les</w>': 3, '   sto ck er</w>': 3, '   h y po c ri ti cal</w>': 2, '   ta i ling</w>': 3, '   loo k out</w>': 16, '   ri di cu ling</w>': 2, '   sor ting</w>': 6, '   bur g l ary</w>': 26, '   ear r in gs</w>': 13, '   i te m</w>': 33, ' things</w>': 21, '   app ra i sed</w>': 1, '   v </w>': 84, ' 8 </w>': 23, '   fi l ed</w>': 33, '   star s k y</w>': 3, '   lo gi c</w>': 39, '   e pi so de</w>': 40, '   hu g gi e</w>': 2, '   hu tch</w>': 5, '   gla s er</w>': 2, '   r en e ga d es</w>': 3, '   e so ter i c</w>': 3, '   a be</w>': 9, '   pe tri fi ed</w>': 7, '   p al in dro me</w>': 1, '   p al in dro mes</w>': 1, '   con s ci ou s ne ss</w>': 16, '   w o ah</w>': 10, '   p hi lo so p hi cal</w>': 16, '   w ea k ne ss es</w>': 10, '   gra ss</w>': 48, '   b ani ja ma l i</w>': 1, '   con ver t</w>': 8, '   lo f ts</w>': 1, '   te ch ni qu es</w>': 4, '   gra mm ar</w>': 6, '   in st in c ts</w>': 27, '   r ac qu et</w>': 5, '   gh e t to</w>': 15, '   pa d dle</w>': 9, '   r ow bo at</w>': 2, '   dri ve way</w>': 10, '   ro t</w>': 27, '   a do p ted</w>': 23, '   a g gra v ation</w>': 7, '   ta to o s</w>': 2, '   man s la u gh ter</w>': 2, '   par tly</w>': 13, '   s wi tch ed</w>': 30, '   ac </w>': 5, '   d c</w>': 8, '   d ou bl ed</w>': 6, '   vo l ta ge</w>': 9, '   sh or ted</w>': 3, '   e le c t ro cu te</w>': 3, '   im pro ving</w>': 7, '   sha k en</w>': 12, '   bar k</w>': 17, '   he ctor</w>': 30, '   w ou dn</w>': 1, '   gar den ing</w>': 4, '   h er b s</w>': 5, '   ja gu ar</w>': 11, '   tr ans</w>': 19, '   si l k y</w>': 6, '   si l k</w>': 31, '   me x i c o</w>': 133, '   nu eve</w>': 2, '   n in e te en</w>': 79, ' sha ba z z</w>': 2, '   co op er</w>': 46, '   ce le bra te</w>': 44, '   se</w>': 18, ' 7 2 </w>': 9, ' 6 3 </w>': 10, '   p on ti ac </w>': 2, '   1 9 6 5 </w>': 4, '   bu ck e th ead</w>': 2, '   se tt ling</w>': 13, '   per cu ssion</w>': 2, '   ce ll ma te</w>': 3, ' come</w>': 61, '   un con sti tu tion al</w>': 2, '   mar i ju an a</w>': 20, '   prob ab i li ty</w>': 12, '   vi ci ous</w>': 31, '   ni ce st</w>': 12, '   p ran ks</w>': 7, '   ex ter na l</w>': 9, '   ge ta way</w>': 14, '   war ned</w>': 60, '   rea li sti ca lly</w>': 6, '   co a ster</w>': 4, '   war f are</w>': 16, '   cla y</w>': 58, '   bu ll shi tting</w>': 11, '   7 0 0</w>': 4, '   tr ou b le ma k er</w>': 5, '   pre ven tion</w>': 3, '   2 6 </w>': 15, '   c rea tion</w>': 21, '   bo x man</w>': 1, '   gra m ma ti cal</w>': 2, '   a da ms</w>': 17, '   re gi ster</w>': 39, '   ho to</w>': 3, '   te ca te</w>': 3, '   s i</w>': 17, '   chi qui ta</w>': 2, '   chi can o s</w>': 1, '   d rea m ers</w>': 5, '   re li eve</w>': 19, '   ma pp le th or pe</w>': 2, '   b our n e</w>': 31, '   ne s k i</w>': 3, '   ro ad b lo ck</w>': 6, '   u r i</w>': 2, '   la u g ha ble</w>': 5, '   bo gu s</w>': 8, '   lan dy</w>': 8, '   sp o t less</w>': 3, '   b re ck er</w>': 1, '   hu br is</w>': 4, '   z or n</w>': 3, '   po ck e ted</w>': 3, '   oun try</w>': 1, '   a gen ci es</w>': 9, '   con fu sion</w>': 24, '   ca l cu la ted</w>': 15, '   j as on</w>': 152, '   la tter</w>': 3, '   pa me l a</w>': 12, '   inter pre ted</w>': 5, '   re pa tri ate</w>': 1, '   s ni p ers</w>': 4, '   do d</w>': 1, '   t rea d st one</w>': 22, '   el u ded</w>': 6, '   na pl es</w>': 7, '   de fini tive</w>': 7, '   ca l en d ar</w>': 16, '   bu d get</w>': 38, ' i v an</w>': 1, '   m ev e d ev </w>': 1, '   pe t ro le u m</w>': 3, '   cla im ed</w>': 23, '   di tch</w>': 29, '   mo ga di sh u</w>': 1, '   sho v el ed</w>': 1, '   cont in en ts</w>': 4, '   pen sion</w>': 24, '   dan g le</w>': 3, '   ad just</w>': 20, '   si de ways</w>': 8, '   pa m</w>': 13, ' gra de</w>': 5, '   d un k</w>': 6, '   sta ged</w>': 5, '   si mu l t an e ou s ly</w>': 9, '   brea k er</w>': 22, ' ga m</w>': 1, '   k el</w>': 2, ' . ] </w>': 127, '   gre e ce</w>': 12, '   c ro ss es</w>': 12, '   h y un da i</w>': 2, '   s n ea k ers</w>': 6, '   t ent</w>': 58, '   ca mp gr ound</w>': 2, '   te le gra p h</w>': 8, '   sc ri b b ling</w>': 1, '   gen ev a</w>': 19, '   ab bo t t</w>': 14, '   pr int</w>': 79, '   de bri e f ed</w>': 5, '   tra m</w>': 5, '   ra m st e in</w>': 1, '   w ok en</w>': 2, '   a le x an der pla t z</w>': 1, '   di al ed</w>': 7, '   u pl ink</w>': 5, '   ki m</w>': 37, '   k u r t</w>': 23, '   re op en ing</w>': 2, '   w y f i</w>': 1, '   con su late</w>': 6, '   de ta in ed</w>': 4, '   un pro ce ssed</w>': 1, '   k g b</w>': 7, '   com par i son</w>': 16, '   d on ni e</w>': 40, '   we ll er</w>': 3, '   a ll o ca tion</w>': 1, '   as sa s sin a ted</w>': 12, ' ex per i en c ed</w>': 1, '   n ev ins</w>': 4, '   rea ssi g ned</w>': 5, '   do ber man s</w>': 3, '   f lo a ted</w>': 5, '   i den ti fi ca tion</w>': 21, '   di a g no s is</w>': 10, '   p sy cho lo gi cal</w>': 35, '   be ha vi ors</w>': 1, '   sy mp tom s</w>': 30, '   hea d ac h es</w>': 19, '   out b ound</w>': 1, '   6 4 5 </w>': 1, '   v la di mi r</w>': 3, '   poli ti ci an</w>': 20, '   s l ac ks</w>': 6, ' shi r t</w>': 12, '   cor ra l</w>': 5, '   m uni ch</w>': 4, ' t rea d st one</w>': 1, '   y up</w>': 70, '   tr ans la tes</w>': 2, '   s an c ti fi ed</w>': 2, '   di stu r b s</w>': 2, '   dam na tion</w>': 8, '   p ee k</w>': 10, '   per i sh ed</w>': 5, '   for fe i ted</w>': 1, '   a m bu sh ed</w>': 7, '   s co ts</w>': 7, '   d ra g ged</w>': 32, '   out la w ed</w>': 10, '   t ar t ans</w>': 1, '   t un es</w>': 5, '   lo qu is</w>': 1, '   la t in u m</w>': 2, '   b en e di ction</w>': 1, '   the e</w>': 76, '   b en e f ac tu m</w>': 1, '   ma l co l m</w>': 43, '   b ru ce</w>': 166, '   s we ars</w>': 6, '   mor n ay</w>': 2, '   wa g es</w>': 15, '   e din bur gh</w>': 9, '   su n set</w>': 30, '   p le dge</w>': 10, '   uni te</w>': 11, '   pu b li c ly</w>': 11, '   pa tri o t</w>': 15, '   lon g sh an ks</w>': 8, '   ca v al ry</w>': 16, '   h or se men</w>': 2, '   out man e u ver</w>': 1, '   b ow men</w>': 1, '   gu ar di an</w>': 16, '   in v a de</w>': 16, '   b al li o l</w>': 1, '   de sig na te</w>': 2, '   ho ma ge</w>': 4, '   en ca sed</w>': 4, '   c r ow n s</w>': 11, '   di vi ded</w>': 12, '   com mon ers</w>': 2, '   un b al an ce</w>': 1, '   b al li o ls</w>': 1, '   stra in ed</w>': 6, '   gre e t in gs</w>': 24, '   b ru ces</w>': 1, '   supp ort ers</w>': 2, '   pri ma</w>': 3, '   no c tes</w>': 1, '   star ve</w>': 20, '   ar mi es</w>': 11, '   ar mor ers</w>': 1, '   s ac ked</w>': 4, '   con sc ri p tions</w>': 1, '   ex p and</w>': 18, '   sti r ling</w>': 2, '   ban d it</w>': 36, '   s co t ti sh</w>': 14, '   r ou ted</w>': 1, '   bo tt om s</w>': 6, '   un fami li ar</w>': 3, '   thou </w>': 144, '   cor on ation</w>': 5, '   cap tu red</w>': 31, '   t rea ch er y</w>': 2, '   tru ce</w>': 12, '   lo char m bi e</w>': 1, '   e mi ss ary</w>': 1, '   tru sts</w>': 23, '   as sa s sin s</w>': 15, '   ra lli es</w>': 7, '   vo l un te ers</w>': 8, '   ha un t in gs</w>': 1, '   fo ll ow ers</w>': 8, '   che er ful</w>': 14, '   so o the</w>': 4, '   mi ser i es</w>': 1, '   shu d d ers</w>': 3, '   r en e w ed</w>': 2, '   re be lli on</w>': 8, '   mo ck</w>': 21, ' l or ds</w>': 1, '   lo a th some</w>': 6, ' l or d</w>': 9, '   ke en</w>': 21, '   mar i on</w>': 26, '   wi lls</w>': 13, '   spe ars</w>': 7, '   ha mi sh</w>': 2, '   ro ac h</w>': 22, '   ca ta pu lt</w>': 3, '   so l di er y</w>': 1, '   man ho od</w>': 11, '   h oun ds</w>': 9, '   e mer a ld</w>': 14, '   i s le</w>': 4, '   i ri sh man</w>': 7, '   fo ok in</w>': 1, '   con ver se</w>': 2, '   en g li sh men</w>': 4, '   be tra y</w>': 33, '   al li an ce</w>': 11, '   pre v a il</w>': 5, '   ac hi ev ed</w>': 12, '   in c rea sed</w>': 13, '   g ri eve</w>': 8, '   u se le ss ly</w>': 1, '   ro tting</w>': 14, '   g rea t ne ss</w>': 16, '   de mon stra ted</w>': 2, '   star ves</w>': 1, '   with d ra w</w>': 20, '   e mb ro i der y</w>': 2, '   min d less</w>': 15, '   wai ts</w>': 29, '   s cou ts</w>': 16, '   ad v an c ed</w>': 37, '   un killed</w>': 1, '   hea then</w>': 3, '   mor row</w>': 7, '   sha lt</w>': 6, '   pu ri fi ca tion</w>': 1, '   a x e</w>': 18, '   pu ri fi ed</w>': 1, '   a ll e gi an ce</w>': 9, '   vi le</w>': 17, '   ha st</w>': 14, ' tonight</w>': 8, '   in vi ting</w>': 16, '   dis li kes</w>': 1, '   wa ll ac es</w>': 3, '   de se cra ting</w>': 3, '   gra ves</w>': 19, '   ou i</w>': 21, '   ma gi stra te</w>': 6, '   cap ture</w>': 30, '   te mp t</w>': 9, '   re be l</w>': 17, '   se cre tly</w>': 17, '   sen se less</w>': 13, '   wa il</w>': 3, ' la dy</w>': 11, '   v ows</w>': 22, '   v ow ed</w>': 5, '   fa i th fu l ne ss</w>': 5, '   ju da s</w>': 12, '   pla in ly</w>': 11, '   con que st</w>': 8, '   pro po ses</w>': 3, '   g ran ts</w>': 8, '   e sta tes</w>': 11, '   sta ging</w>': 3, '   in v a sion</w>': 25, '   s ack er</w>': 1, '   ex e cu tion er</w>': 6, '   wa les</w>': 8, '   con su lt</w>': 9, '   ca l cu la tes</w>': 1, '   we i gh s</w>': 14, '   cl ans</w>': 2, '   ba tt le fi e ld</w>': 16, '   ca st les</w>': 13, '   spe ar men</w>': 1, '   da g ger</w>': 6, '   tri cked</w>': 28, '   t re bor n</w>': 4, '   d ra win gs</w>': 12, '   he mor r ha ging</w>': 8, '   ne u ra l</w>': 19, '   ir re par able</w>': 4, '   mo tor</w>': 44, '   se da ti ves</w>': 5, '   mon i t or ing</w>': 11, '   co p ing</w>': 2, '   bl ack ou ts</w>': 5, '   le si ons</w>': 12, '   tu mor s</w>': 1, '   jo g</w>': 6, '   j our na l</w>': 37, '   an dre a</w>': 2, '   di sor d ers</w>': 3, '   con c lu si ons</w>': 18, '   in h er i ted</w>': 15, '   b lo ck bu ster</w>': 9, '   gi st</w>': 2, '   k a y le i gh</w>': 12, ' de f en se</w>': 14, '   p han tom s</w>': 2, '   bl ac ked</w>': 7, '   t ou ch ing</w>': 56, '   gra d es</w>': 23, '   sle e p y</w>': 32, '   c ro ck e t t</w>': 9, '   re d fi e ld</w>': 4, '   da ds</w>': 6, '   l en ny</w>': 86, '   ri gh ty</w>': 5, ' ti gh ty</w>': 1, '   le f ty</w>': 1, ' lu cy</w>': 2, '   lo c o</w>': 6, '   we ir der</w>': 8, '   car lo s</w>': 40, '   bro ther ho od</w>': 10, '   sc rea med</w>': 16, '   al ter na te</w>': 14, '   uni ver ses</w>': 2, '   co ll e g es</w>': 6, '   pri s ons</w>': 8, '   par a p le gi a</w>': 1, '   j our n al s</w>': 5, '   lin ing</w>': 12, '   cer e bra l</w>': 8, '   cor te x</w>': 7, '   ar gh</w>': 3, '   v ani sh</w>': 6, ' sp li c ed</w>': 1, '   v ani sh es</w>': 1, '   pa use</w>': 15, '   re w ind</w>': 6, '   wh a sa matter</w>': 1, '   re pre ssed</w>': 14, '   ti gh ty</w>': 2, '   whi ti es</w>': 1, '   un b lo ck</w>': 1, '   han ni ba l</w>': 8, '   le c ter</w>': 41, '   ab sor b ing</w>': 4, '   wor m</w>': 58, '   o z zy</w>': 6, '   f la t wor ms</w>': 2, '   f la t wor m</w>': 2, '   me mor i z ed</w>': 6, '   th u mp er</w>': 5, '   thin k er</w>': 12, '   ev </w>': 7, '   an al y ti cal</w>': 4, '   ma tch ed</w>': 10, '   re la ti ve ly</w>': 10, '   p lo ts</w>': 5, '   ha l per n</w>': 7, '   spi ri tu al</w>': 26, '   k a ti e</w>': 6, '   he mor r ha g es</w>': 1, '   s li ck</w>': 25, '   man sion</w>': 18, '   ti ju an a</w>': 14, '   sor ori ty</w>': 11, '   ci lan tr o</w>': 2, ' si ster</w>': 8, '   mo les</w>': 3, '   th i gh</w>': 13, '   s w o on</w>': 4, '   g na w ing</w>': 1, '   si ck en ing</w>': 8, '   bl ink</w>': 13, '   o c cu pa tion al</w>': 7, '   s late</w>': 12, '   ta bu l a</w>': 2, '   ra s a</w>': 2, '   a ton e ment</w>': 4, '   ac t in</w>': 15, '   ac cen ts</w>': 6, '   op ra h</w>': 3, '   ma th</w>': 46, '   mu l ti p le</w>': 17, '   or ga s ms</w>': 2, '   f rea king</w>': 20, '   sti r</w>': 22, '   e man ci pa ted</w>': 3, '   ju v y</w>': 2, '   da le</w>': 10, '   au to body</w>': 1, '   com me</w>': 3, '   c a</w>': 8, '   k a g an</w>': 1, '   ou ch</w>': 45, '   en dea v ors</w>': 2, '   fuck ba g</w>': 2, '   ca stra te</w>': 1, '   pe do p hi le</w>': 1, '   u l ti ma tely</w>': 11, '   re ck on ing</w>': 4, '   tra u ma ti ze</w>': 2, '   in f o</w>': 14, '   p le d g es</w>': 1, '   h y st er i cal</w>': 29, '   e s say</w>': 6, '   ex ten sion</w>': 18, '   wor ms</w>': 28, '   hu mb le</w>': 22, '   as si mi la tion</w>': 2, '   pa v lo v </w>': 2, '   ju an</w>': 14, ' wor m</w>': 2, ' mr</w>': 33, ' ma ster</w>': 1, ' gen er al</w>': 6, '   ta pe wor ms</w>': 1, '   tu ck</w>': 16, '   com p</w>': 6, '   comp le x i ti es</w>': 3, '   du ly</w>': 7, '   b la de</w>': 59, '   sor ts</w>': 39, '   go o dy</w>': 9, '   sp in ning</w>': 12, '   te ch no</w>': 5, '   ha v a</w>': 1, '   na gi l a</w>': 1, '   po o ling</w>': 1, '   fun ds</w>': 24, '   hi ll el</w>': 1, '   shi t less</w>': 15, '   st int</w>': 7, '   di d j a</w>': 6, '   mar i an</w>': 6, '   no tt in g ha m</w>': 1, '   fu se</w>': 20, '   pu ss</w>': 13, '   n ar c</w>': 5, '   bo s well</w>': 1, '   g lu e</w>': 16, '   ca sa bl an c a</w>': 44, '   j an</w>': 9, '   bu l gar i a</w>': 3, '   ex it</w>': 62, '   tra v el ing</w>': 41, '   r en au lt</w>': 15, '   r ou le tt e</w>': 8, '   bro ad min ded</w>': 1, '   ca ver n e</w>': 1, '   d u</w>': 16, '   bo is</w>': 6, '   ber ger</w>': 7, '   u gar te</w>': 13, '   con cen tra tion</w>': 34, '   a pt</w>': 8, '   la s z l o</w>': 38, '   nor we gi an</w>': 9, '   l un d</w>': 4, '   s ac h a</w>': 6, '   lo cking</w>': 11, '   fran c s</w>': 34, '   be see ch</w>': 6, '   car t ons</w>': 4, '   han d sha ke</w>': 8, '   ran s ack</w>': 1, '   stra ss er</w>': 12, '   in ci den tal</w>': 6, '   h y po c ri te</w>': 13, '   sor ri er</w>': 4, '   shi p ment</w>': 19, '   i so la tion i s m</w>': 1, '   com mo di ty</w>': 9, '   re fu ge es</w>': 9, '   par ro t</w>': 11, '   sha d ow ed</w>': 4, '   vi sa s</w>': 7, '   mi r ac les</w>': 28, '   re spe c ted</w>': 22, '   ob ser ved</w>': 13, '   gu ar an te es</w>': 8, '   de st in ation</w>': 15, '   imp li ed</w>': 9, '   il s a</w>': 20, '   ad ds</w>': 14, '   f re ight</w>': 8, '   out s ki r ts</w>': 3, '   ge sta p o</w>': 19, ' vi ctor</w>': 2, '   app re h en ded</w>': 3, '   sa le s man ship</w>': 1, ' ri ch ard</w>': 2, '   en ti t l ed</w>': 42, '   so b er</w>': 18, '   t in ny</w>': 1, '   par l or</w>': 33, '   o s l o</w>': 3, '   wor shi pped</w>': 8, '   in si d es</w>': 13, '   mar se i ll es</w>': 10, '   7 7 </w>': 2, '   p oun ding</w>': 16, '   br ace</w>': 15, '   c ru mb ling</w>': 3, '   bl ack li st</w>': 4, '   o ver char ged</w>': 2, '   fran c</w>': 7, '   au r ore</w>': 1, ' ask</w>': 11, '   su g ge sted</w>': 29, '   sen ti ment</w>': 12, '   h er o i c s</w>': 5, '   li ll e</w>': 1, '   ha st y</w>': 8, '   b la ine</w>': 12, '   li s b on</w>': 9, '   ho b by</w>': 29, '   under do g</w>': 8, '   fa s ci sts</w>': 2, '   kee per</w>': 16, '   lea d ers</w>': 25, '   happ i er</w>': 34, '   g ent</w>': 5, '   o ver stay</w>': 2, '   ev in c ed</w>': 1, '   in so f ar</w>': 2, '   re i ch</w>': 13, '   re gre ts</w>': 15, '   pre fe ct</w>': 3, '   cu r few</w>': 14, '   pre ce dent</w>': 6, '   co ck ta i ls</w>': 15, '   con ven i ent</w>': 30, '   g ro ss</w>': 50, '   under sta te ment</w>': 9, '   cor di al</w>': 3, '   inter f ere</w>': 42, '   u no c cu pi ed</w>': 2, '   ne u tra li ty</w>': 1, '   e lo qu ence</w>': 2, '   na z is</w>': 35, ' per su a si ve</w>': 1, '   fu r ni sh</w>': 3, '   b ru s se ls</w>': 3, '   a m st er da m</w>': 5, '   be l gra de</w>': 1, '   a th en s</w>': 4, '   in de fini tely</w>': 12, '   proble ma ti cal</w>': 1, '   min ce</w>': 4, '   el u ding</w>': 2, '   ar i sing</w>': 1, '   c z e cho s lo v a ki an</w>': 3, '   in spi re</w>': 12, '   pro v in ce</w>': 6, '   hon e y com bed</w>': 1, '   tra i t ors</w>': 9, '   pre v a i ling</w>': 3, '   vi ch y</w>': 2, '   re gu late</w>': 3, ' bl under ed</w>': 1, '   1 9 1 8 </w>': 2, '   cl ever ne ss</w>': 4, '   th or ou gh ly</w>': 17, '   do ssi er</w>': 10, '   di p lo ma ti st</w>': 1, '   u no f fi ci ally</w>': 3, ' thir d</w>': 12, '   he in ri ch</w>': 5, '   ve u ve</w>': 1, '   c li qu o t</w>': 1, ' 2 6 </w>': 1, '   rea li z ing</w>': 11, '   r oun ding</w>': 6, '   c li ma tes</w>': 1, '   sa har a</w>': 3, '   a i de</w>': 9, '   ca s se ll e</w>': 1, '   w el com es</w>': 2, '   bra z z a vi ll e</w>': 1, '   in du c ed</w>': 5, '   sen ti men ta li st</w>': 4, ' do gs</w>': 2, '   se ar ch ed</w>': 21, '   wa tch do gs</w>': 1, '   gr oun ds</w>': 28, '   pe tty</w>': 29, '   ch u ck</w>': 48, '   f ea ther</w>': 17, '   ge stu re</w>': 34, '   ro man ces</w>': 3, '   con sti tu te</w>': 2, ' vi ch y</w>': 1, '   de st ru c tive</w>': 11, '   im pre ss es</w>': 2, '   fr en ch man</w>': 11, '   ga so l ine</w>': 17, '   ra tion ing</w>': 1, '   he in ze</w>': 1, '   1 9 3 5 </w>': 6, '   1 9 3 6 </w>': 3, '   lo y a li st</w>': 1, '   c y ni cal</w>': 20, '   or an</w>': 1, '   man a g es</w>': 10, '   im pre ss</w>': 36, '   im pre ss ing</w>': 2, '   b ran dy</w>': 23, '   ob ser v ant</w>': 5, '   a mu se</w>': 18, '   mi sin for med</w>': 5, '   wa t ers</w>': 34, '   c li pp er</w>': 1, '   de mo c r at</w>': 8, '   re b ound</w>': 6, '   ex tra v a g ant</w>': 4, '   pu b li sh ed</w>': 35, '   f ou le st</w>': 2, '   ce ll ar</w>': 29, '   a h a</w>': 30, '   se c tions</w>': 9, '   dr un k ard</w>': 3, '   na tion a li ty</w>': 3, '   ad di o</w>': 1, '   de spi se</w>': 18, '   par a si te</w>': 16, '   par a si ti c</w>': 3, '   cl er ks</w>': 7, ' hon or ed</w>': 1, '   de u t sch es</w>': 1, '   1 9 4 1 </w>': 1, '   fi sh ing</w>': 79, '   fa i lu r es</w>': 6, '   su c ce ss es</w>': 4, '   war saw</w>': 11, '   sha p ed</w>': 9, '   pe ter s bur g</w>': 11, '   war e house</w>': 28, '   fe de x</w>': 7, '   wi l son</w>': 42, '   mi ra ge</w>': 3, '   au to pi lot</w>': 3, '   tr ough</w>': 3, '   p ea body</w>': 2, '   ke lly</w>': 89, '   cap si ze</w>': 2, '   qui tter</w>': 7, '   ra ft</w>': 17, '   di ves</w>': 5, '   shi pp ing</w>': 18, '   lan es</w>': 14, '   po l ar is</w>': 1, '   und one</w>': 6, '   i den ti f y</w>': 45, '   f ra ying</w>': 1, '   p ack a g es</w>': 12, '   ca ve</w>': 60, '   pri ma l</w>': 6, '   re ci pi ent</w>': 2, '   win gs</w>': 64, '   bi c y c le</w>': 15, '   c ri pp l ed</w>': 22, '   me mp his</w>': 35, '   bab y si tter</w>': 8, '   fin al s</w>': 8, '   pl u g ged</w>': 6, '   c li ch es</w>': 3, '   hea d l ine</w>': 8, '   who o</w>': 17, '   so o ey</w>': 1, '   ke tch</w>': 4, '   hea d st one</w>': 4, '   e pi ta p h</w>': 2, '   cap si z ing</w>': 1, '   j an go</w>': 5, '   fro gs</w>': 11, '   ma la y si a</w>': 3, '   o ver kill</w>': 2, ' rea c ting</w>': 3, '   o ver f l ow</w>': 5, '   hu b</w>': 8, '   x er o x</w>': 6, '   for ma l de h y de</w>': 3, '   v a li u m</w>': 12, '   sy d ne y</w>': 5, '   p d rs</w>': 1, '   i v </w>': 10, '   dis ru p ti ven ess</w>': 1, '   he ge l</w>': 1, '   p ow</w>': 10, '   ir ri ta bi li ty</w>': 2, ' re pro ac h</w>': 1, '   sc hi z op h r en i a</w>': 3, '   in e f fi ci ent</w>': 3, '   lu c ki ly</w>': 10, '   p an c rea s</w>': 6, '   di ge sti on</w>': 2, '   co st ly</w>': 6, '   u sa ge</w>': 2, '   de h y d ra tion</w>': 3, '   de fi ci en cy</w>': 8, '   jo in ts</w>': 15, '   ac he</w>': 5, '   o ver all</w>': 7, '   bo i ls</w>': 4, '   u l cer a ted</w>': 1, '   no land</w>': 2, '   pa s ca g ou l a</w>': 2, '   ha tch</w>': 31, '   chi ck en s</w>': 34, '   ca pi ta</w>': 1, '   1 9 6 0</w>': 7, '   ta bo o s</w>': 1, '   h in d us</w>': 9, '   bo y co tting</w>': 1, '   pro di ga l</w>': 4, '   i i i</w>': 18, '   au th ori z ation</w>': 8, '   du ran go</w>': 1, '   co l or a do</w>': 10, '   a sh ev i ll e</w>': 1, ' can c el ed</w>': 1, '   be tt in a</w>': 1, '   pe ter son</w>': 16, '   mar f a</w>': 1, '   sen der</w>': 5, '   p t r</w>': 1, '   re ver ts</w>': 1, '   c p c</w>': 1, '   ba k u</w>': 1, '   de l hi</w>': 9, '   k u al a</w>': 6, '   la m pu r</w>': 1, '   for war ding</w>': 4, '   ad dre ss es</w>': 5, '   k a ma l</w>': 4, '   ch in n</w>': 1, '   de fu sed</w>': 1, '   i bri m</w>': 1, '   sor ter</w>': 3, '   rea d ers</w>': 25, '   sp in al</w>': 14, '   f lu id</w>': 14, '   ne on</w>': 6, '   mar b les</w>': 3, ' three</w>': 92, '   bra i ded</w>': 3, '   s me lled</w>': 30, '   a mber</w>': 28, ' ch u ck</w>': 2, '   in d on e si a</w>': 2, '   au stra li a</w>': 21, '   t an king</w>': 2, '   n in e ty</w>': 88, '   ru b les</w>': 2, '   9 5 </w>': 7, '   pl un ge</w>': 7, '   jo ck e ys</w>': 3, '   bo ld</w>': 29, ' 4 0</w>': 12, '   5 1 </w>': 4, ' 4 9 </w>': 1, '   di ce</w>': 10, '   ar k an sa s</w>': 11, '   m cle ll and</w>': 1, '   char ter ed</w>': 2, '   sa il bo a ts</w>': 1, '   st rea m</w>': 39, '   su n d ried</w>': 1, '   st ri ps</w>': 4, '   gi lls</w>': 8, '   m un ch</w>': 3, '   gra v y</w>': 17, '   c ran ber ri es</w>': 3, '   ba ked</w>': 22, '   tri m min gs</w>': 2, '   fri sc o</w>': 5, ' gr ace</w>': 5, ' meet</w>': 5, '   app les</w>': 37, '   wi l bu r</w>': 9, ' f ar</w>': 3, '   di sa pp o in t in g ly</w>': 1, '   m ea ly</w>': 2, '   pi es</w>': 13, ' cre den ti al s</w>': 1, ' re place</w>': 2, ' new</w>': 20, ' in ven ted</w>': 2, '   b ran ch</w>': 31, '   ac comp li sh ed</w>': 27, '   g y ne co lo gi cal</w>': 1, '   ob st e tri cal</w>': 7, '   sur ge on</w>': 57, '   pe di a tri c</w>': 6, ' car e ful</w>': 3, '   go o d ha ll</w>': 2, '   z ea l</w>': 2, '   e ther</w>': 9, ' help</w>': 22, ' com m it</w>': 1, ' see ing</w>': 3, '   tru st e es</w>': 1, '   a do p ting</w>': 1, '   re st ore</w>': 23, '   mon th ly</w>': 12, '   re gu l ar i ty</w>': 2, '   er got</w>': 1, '   pi tu i t ary</w>': 3, '   ex tr act</w>': 8, '   ru e</w>': 8, '   a g n es</w>': 74, '   bu ster</w>': 23, '   l ar ch</w>': 3, '   e d na</w>': 16, ' them</w>': 18, ' do c t ors</w>': 2, ' s me lls</w>': 3, ' s ni ff s</w>': 1, ' anyone</w>': 5, '   fu z zy</w>': 16, ' se cre cy</w>': 2, ' i g nor an ce</w>': 1, ' please</w>': 69, '   nee d in</w>': 4, '   com pa ss</w>': 14, '   wa lly</w>': 51, '   men tions</w>': 4, ' cap tain</w>': 8, '   wor thin g ton</w>': 8, '   o live</w>': 22, ' lo ved</w>': 5, ' hel p ed</w>': 2, ' a live</w>': 6, ' par al y z ed</w>': 1, ' de ci sion</w>': 2, ' t ea sing</w>': 1, '   cu d dle</w>': 2, ' care</w>': 11, ' sp ea k er</w>': 1, '   pri v ac y</w>': 37, ' as ked</w>': 3, ' re sp on si bi li ty</w>': 1, '   or p han age</w>': 8, '   din ing</w>': 26, '   k ong</w>': 50, '   w ea k en ed</w>': 3, ' king</w>': 9, '   me di ca lly</w>': 2, ' me di cal</w>': 1, ' out side</w>': 3, '   lo b ster</w>': 13, ' fee ling</w>': 6, '   ta per</w>': 1, '   cra mp s</w>': 12, '   bur ma</w>': 4, '   man da la y</w>': 2, '   lo b st er man</w>': 1, '   mu ddy</w>': 19, ' mu ddy</w>': 1, '   b om b ing</w>': 18, ' h om er</w>': 2, '   b ran ch es</w>': 7, ' pi cking</w>': 2, ' app les</w>': 1, ' keep</w>': 13, '   or p han s</w>': 7, '   cu r ly</w>': 18, '   di ss er ta tion</w>': 6, ' di p lo ma s</w>': 1, '   ab or tions</w>': 2, ' gra du a ted</w>': 1, '   b ow do in</w>': 1, '   1 9 3 9 </w>': 6, ' here</w>': 70, '   por t land</w>': 19, '   ma ine</w>': 19, '   1 9 1 5 </w>': 1, ' b et</w>': 3, '   se da tion</w>': 7, '   fe tu s</w>': 2, '   un ex pe lled</w>': 1, '   u ter us</w>': 4, '   p un c tu red</w>': 2, '   per i ton i t is</w>': 1, '   c ro ch et</w>': 1, '   im mi g ran t</w>': 8, '   n an ny</w>': 9, '   ma tt er ed</w>': 16, ' al co ho l</w>': 1, ' li ver</w>': 1, '   ci r r ho s is</w>': 3, '   po s th u m ous</w>': 2, ' d rea m ing</w>': 3, '   f la k</w>': 3, ' an ti a ir cra ft</w>': 1, '   mi g ran ts</w>': 2, '   hi ma la y as</w>': 5, ' bur ma</w>': 1, '   af fe c ts</w>': 11, '   o ver qu a li fi ed</w>': 1, '   ca pe</w>': 26, '   k en ne th</w>': 9, '   stra in</w>': 37, '   de fe ct</w>': 8, '   li ber a tor</w>': 3, '   run ned</w>': 2, '   sti cked</w>': 2, ' own</w>': 11, ' after</w>': 29, '   po ked</w>': 7, '   l ac er ation</w>': 1, '   mi su n der stand</w>': 13, '   rea ch in</w>': 2, ' hi tch</w>': 3, ' hi kin</w>': 1, '   mer th i o late</w>': 1, '   v u l v al</w>': 1, '   pa ds</w>': 6, '   ga u ze</w>': 3, ' lo ts</w>': 4, ' al most</w>': 15, ' ain</w>': 15, '   cer vi cal</w>': 1, '   st ab i li z er</w>': 1, '   di la t ors</w>': 1, ' d ou gla s</w>': 1, '   cu re tt e</w>': 1, '   spe cu lu m</w>': 1, '   v u l se ll u m</w>': 1, '   for ce ps</w>': 3, ' doctor</w>': 17, ' bo th</w>': 12, ' da u gh ter</w>': 5, '   hur r ying</w>': 5, ' pro te in</w>': 1, '   ci der</w>': 10, '   wa ter y</w>': 6, '   m ac s</w>': 4, '   gra ven st e ins</w>': 1, '   ban an as</w>': 16, '   b al d w ins</w>': 1, '   ru ss er ts</w>': 1, ' stu p id</w>': 3, '   gra ti f ying</w>': 1, ' any body</w>': 7, '   in ten ding</w>': 4, '   d ou b t less</w>': 4, '   im men se ly</w>': 6, '   wor th while</w>': 14, ' u se ful</w>': 2, '   in c in er a tor</w>': 5, '   mar ve l</w>': 5, ' her</w>': 42, '   sp li ce</w>': 2, ' do z en s</w>': 1, '   h en ce for th</w>': 2, '   d or r it</w>': 1, ' doesn</w>': 7, '   br on chi t is</w>': 3, '   br on chi al</w>': 1, '   in fe c tions</w>': 3, ' under d ev e lo p ed</w>': 1, '   under d ev e lo p ed</w>': 2, '   mor ons</w>': 11, '   un com m on</w>': 11, '   su sc e p ti ble</w>': 5, '   ab sor b</w>': 9, '   t an sy</w>': 1, '   ab or ti ci de</w>': 1, '   sa il or</w>': 98, '   ve ge ta b les</w>': 13, ' sa il or</w>': 3, '   vi r tually</w>': 15, ' di sin te gra ting</w>': 1, '   sti tch es</w>': 7, '   ti s su e</w>': 22, ' else</w>': 8, ' ob li ga ted</w>': 1, '   pre g n an cy</w>': 19, '   b li z z ard</w>': 7, '   in de st ru c ti ble</w>': 1, '   ver n on</w>': 31, ' even</w>': 30, '   de spi ses</w>': 4, ' war m</w>': 1, ' won der in</w>': 1, ' no where</w>': 1, ' happy</w>': 18, '   ven tri c le</w>': 2, '   en l ar ged</w>': 5, ' s wi mm in</w>': 2, '   v at</w>': 10, '   fi sh in</w>': 9, '   ba tch</w>': 10, '   wor th in</w>': 1, ' ons</w>': 6, ' r ound</w>': 40, '   chri sti ani ty</w>': 4, ' hur t</w>': 5, ' ser ved</w>': 2, '   da un ted</w>': 1, ' dis ea se</w>': 1, '   b om b ay</w>': 2, ' co ff in</w>': 1, '   de di ca tion</w>': 8, ' mi ssi on ary</w>': 3, ' in di a</w>': 1, ' wee ks</w>': 4, '   en ce p ha li t is</w>': 1, '   re co ver ing</w>': 5, '   la v in i a</w>': 1, '   ce y l on</w>': 2, '   wai st</w>': 8, '   ma l ar i a</w>': 7, '   ho spi ta li z ed</w>': 4, '   ra di om an</w>': 1, '   co pi lot</w>': 1, ' heard</w>': 7, ' live</w>': 9, '   p ea ch es</w>': 19, '   gr ind</w>': 7, '   brea th in</w>': 7, ' ou tra ge ous</w>': 1, '   b ru i se</w>': 10, '   st em</w>': 3, ' st ep</w>': 4, '   har ve st</w>': 16, '   brea kin</w>': 10, '   cu tt in</w>': 13, ' clo th es</w>': 2, ' ni pp les</w>': 1, '   te ch</w>': 13, '   t y pi ca lly</w>': 10, '   bra in i ac s</w>': 2, '   in no v a tive</w>': 2, '   o ver be ar ing</w>': 2, '   li ly</w>': 41, '   fin n</w>': 21, '   a sh by</w>': 3, '   ja ke</w>': 281, '   re cou p ing</w>': 2, '   con v in c ing</w>': 20, '   con v in c er</w>': 1, '   un no ti c ed</w>': 3, '   fin a li ze</w>': 2, ' 2 2 </w>': 13, '   bi d ding</w>': 11, '   e ti que tt e</w>': 7, '   mi c ro so ft</w>': 1, '   or ac le</w>': 12, '   in i ti al</w>': 10, '   ex c lu ded</w>': 3, '   fir ms</w>': 5, ' end</w>': 11, '   v p</w>': 4, '   ca l en d ars</w>': 1, '   bo ss es</w>': 31, '   al f on se</w>': 5, '   du cking</w>': 7, '   e g g ro lls</w>': 2, '   a w ard</w>': 28, '   shi ll</w>': 4, '   j in x</w>': 6, '   g or do</w>': 17, '   al r l ght</w>': 2, '   e d en</w>': 4, '   lo t to</w>': 4, '   su per sti ti ous</w>': 12, '   w w i</w>': 1, '   w wi i</w>': 2, '   w w</w>': 4, '   na z i</w>': 45, '   gi ll e t t</w>': 1, '   cu e</w>': 23, '   vi o lin s</w>': 2, '   pa y er</w>': 1, '   li on el</w>': 9, '   do l by</w>': 4, '   g ri f t ers</w>': 6, '   re d head</w>': 12, '   fin der</w>': 16, '   cu st om s</w>': 33, '   ad r en al ine</w>': 18, '   f ra i s er</w>': 1, '   mu mp s</w>': 2, '   p in ch ed</w>': 11, '   du m best</w>': 15, '   di cks</w>': 26, ' e g g</w>': 2, '   fo o</w>': 4, '   se m i</w>': 18, ' fucking</w>': 41, '   bo o ki e</w>': 12, '   ca sh ed</w>': 13, '   mi t z v ah</w>': 3, '   f le e c ed</w>': 4, ' t y pe</w>': 20, '   ba ld</w>': 21, ' bo o</w>': 3, '   mo on an</w>': 9, '   de mi se</w>': 8, '   ca y men s</w>': 1, '   e u c</w>': 1, '   l ean</w>': 25, '   gi ll e tt e</w>': 8, '   s ani ta tion</w>': 8, '   a gen da</w>': 10, '   do or k no b</w>': 6, '   he pa ti t is</w>': 3, '   in f lu en z a</w>': 3, ' what ever</w>': 21, '   in qu ir ing</w>': 4, '   fu d g es</w>': 1, ' s end</w>': 6, '   mo e</w>': 11, '   ho o p</w>': 16, '   con ver ti ble</w>': 14, ' b on ds</w>': 1, ' bro k er</w>': 1, ' di ck head</w>': 1, '   sp la tter</w>': 6, '   ru b b ing</w>': 14, '   s k in ny</w>': 42, '   s co o ter</w>': 21, ' ch u mp</w>': 1, '   ta tt oo ed</w>': 12, '   ven tu red</w>': 2, '   f le e ce</w>': 2, '   ll</w>': 1, '   g ri f ter</w>': 7, '   co o l er</w>': 19, '   e s ki m o</w>': 6, '   g ri ft</w>': 8, '   re ph ra se</w>': 15, '   for gi ven</w>': 19, '   che at</w>': 41, '   l ac ks</w>': 7, '   gra f ting</w>': 3, '   under world</w>': 9, '   mi a m i</w>': 84, '   con s ci ous</w>': 17, '   so b o</w>': 2, '   g un ther</w>': 3, '   ta x es</w>': 39, '   e con om i c s</w>': 7, '   s ack</w>': 43, '   st ea di est</w>': 1, '   br ac es</w>': 18, '   car o l y n</w>': 19, '   an ni ver s ary</w>': 42, '   pe ar son</w>': 4, '   la w school</w>': 1, '   w el ch</w>': 5, ' ta le</w>': 3, '   e u c li d</w>': 1, '   ra ving</w>': 13, '   g l en n</w>': 60, '   v a li u ms</w>': 1, '   s ca l pe l</w>': 3, '   gr ab bed</w>': 37, '   lin o le u m</w>': 2, '   ti l ed</w>': 2, ' fuck</w>': 64, '   per co ce ts</w>': 1, '   mor ri se y</w>': 1, '   be d p ans</w>': 2, '   te ch ni ci an</w>': 12, '   st ri per</w>': 1, '   in tu i tion</w>': 19, '   y o da</w>': 11, '   man i pu la ting</w>': 3, '   bu ll shi tt ed</w>': 3, '   under ac hi ever</w>': 1, '   dro o ling</w>': 10, '   po o</w>': 8, ' ba h</w>': 1, '   j in x es</w>': 1, '   cla ss y</w>': 22, '   de co der</w>': 3, '   ge tch a</w>': 7, '   li f ting</w>': 13, '   wa ll e ts</w>': 4, '   da ff y</w>': 6, '   k nee ca ps</w>': 4, '   bi tch y</w>': 2, '   ma x ed</w>': 2, '   e sc or ts</w>': 4, '   ar man i</w>': 8, '   cha ses</w>': 5, '   lu p us</w>': 3, '   mi che ll e</w>': 16, '   st ri go</w>': 1, '   ex p lo it</w>': 10, '   fu ga s i</w>': 1, '   al b any</w>': 13, ' twenty</w>': 34, '   pu t z</w>': 10, '   f lu sh</w>': 24, '   ra g</w>': 32, '   mo pe</w>': 5, '   ac comp li ce</w>': 12, '   dri pp ing</w>': 12, '   mo ti v ation</w>': 9, '   wa sh ing</w>': 26, '   a k r on</w>': 9, ' fo o</w>': 1, '   i ho p</w>': 1, '   ru tt i</w>': 1, '   tu tt i</w>': 1, ' f re sh</w>': 4, '   f ru i ty</w>': 3, '   wa ving</w>': 17, '   v ou ch</w>': 11, '   ee e</w>': 1, '   oo t t</w>': 1, '   au u ght</w>': 1, '   in fe ction</w>': 21, '   wh ee ling</w>': 4, '   w ea se l</w>': 18, '   dri lled</w>': 4, '   ho b</w>': 2, '   k no b</w>': 5, '   ho l ed</w>': 7, '   lea k</w>': 37, '   har l in</w>': 14, '   c ro o ks</w>': 13, '   sti ff ed</w>': 3, '   mu l ti p ly</w>': 8, '   bl ow off</w>': 1, '   r ac ke ts</w>': 11, '   lu g</w>': 6, '   k ne ck er</w>': 1, '   bar n ard</w>': 6, '   tra ps</w>': 22, '   m ace</w>': 23, '   ju g</w>': 6, '   h er o in</w>': 38, '   o ver bi te</w>': 1, '   mon ks</w>': 6, '   me di ev al</w>': 16, '   vi g</w>': 16, '   ba th ro om s</w>': 6, '   con fr on ta tion al</w>': 2, '   for en si c s</w>': 10, '   whi ff</w>': 9, '   ro tt o vi ch</w>': 1, '   so bo z in s k i</w>': 1, '   sh e il ds</w>': 1, '   ca pi ce</w>': 1, '   i a d</w>': 1, '   gu i se</w>': 2, '   shi el ds</w>': 55, '   be la ted</w>': 1, ' fi x</w>': 5, '   sp oo k</w>': 22, '   mo or e ly</w>': 1, '   l ever age</w>': 9, '   co s me ti c</w>': 6, '   ca vi ty</w>': 14, '   den tal</w>': 21, '   or th o d on ti st</w>': 2, '   ti gh ten</w>': 9, '   be ll a</w>': 14, '   sho pped</w>': 3, ' tri ck</w>': 3, '   ca sin o</w>': 77, '   ce s sp it</w>': 2, '   p un t ers</w>': 8, '   ten sion</w>': 23, '   man f red</w>': 1, '   ma the ma ti ci an</w>': 3, '   di ff</w>': 6, '   gi les</w>': 3, '   cor ning</w>': 1, '   sp o il sp or t</w>': 1, '   qu id</w>': 12, '   sp or ty</w>': 6, '   d ab </w>': 4, '   au th or</w>': 23, '   ha bi b</w>': 2, '   un so li ci ted</w>': 2, '   man u sc ri p ts</w>': 3, '   rea der</w>': 12, '   ma th s</w>': 1, ' 9 3 </w>': 3, ' use</w>': 4, '   c r ou pi er</w>': 9, '   re la tive</w>': 30, '   or g ani sed</w>': 2, '   g an gs</w>': 14, '   la m pp o st</w>': 1, '   sh at</w>': 2, '   wri ter</w>': 126, '   ga mb l ers</w>': 4, '   pu b li sh</w>': 17, '   f re e z er</w>': 15, '   fin an c ing</w>': 14, '   j ack o</w>': 6, '   li on</w>': 46, '   cha tting</w>': 10, '   ba y s wa ter</w>': 2, '   j an i</w>': 6, '   ban da ge</w>': 7, ' friend</w>': 18, '   com mi tting</w>': 11, '   ge min is</w>': 1, '   a st ro lo g y</w>': 2, '   li ars</w>': 8, '   tr ans ke i</w>': 1, '   cour te sy</w>': 23, '   ca l cu late</w>': 11, '   gra tu i ti es</w>': 1, '   u k</w>': 3, '   re co g ni sed</w>': 1, '   ro ss</w>': 33, '   a vo i ding</w>': 24, '   er ne st</w>': 6, '   he min g way</w>': 16, '   loo s en</w>': 15, '   ro cking</w>': 7, '   comp li ci ty</w>': 2, '   the ft</w>': 21, '   su per vi s or</w>': 15, '   gre e k</w>': 29, '   2 5 </w>': 45, '   un w ind</w>': 3, '   wa ter ing</w>': 6, '   bar red</w>': 4, '   re co g ni se</w>': 4, '   as set</w>': 13, '   st ac ks</w>': 5, '   t cha i</w>': 1, '   of f ence</w>': 5, '   p un ter</w>': 3, '   ga mb l ed</w>': 6, '   ca su ally</w>': 5, '   fri en d shi ps</w>': 3, '   c r ou pi ers</w>': 1, '   dis cou ra ged</w>': 14, '   da ta ba se</w>': 10, '   s nee ze</w>': 7, '   sti tch ed</w>': 1, '   al ter na ting</w>': 3, '   3 6 5 </w>': 3, '   p ence</w>': 1, '   ex ce p tions</w>': 7, '   r ows</w>': 4, '   b ow l</w>': 55, '   cor re sp on d</w>': 7, '   pri ve</w>': 1, '   f ac tion</w>': 6, '   mo t r</w>': 1, '   bea d les</w>': 1, '   y o o</w>': 4, '   ho o</w>': 9, '   tal by</w>': 21, '   dri f ting</w>': 17, '   je t p ack</w>': 2, '   do o little</w>': 45, '   dis ar m</w>': 8, '   bo il er</w>': 11, '   ev en tu al</w>': 1, '   su per no v a</w>': 7, '   de fini te</w>': 25, '   9 9 </w>': 20, ' pl us</w>': 10, '   d ev i ate</w>': 4, '   ro ta tions</w>': 1, '   spi ra l</w>': 6, '   g mr</w>': 1, '   f li ck er ing</w>': 2, '   p an el s</w>': 1, '   qu an tu m</w>': 18, '   in c rea se</w>': 26, '   c ri ti cal</w>': 22, '   re du c ed</w>': 17, '   la tch es</w>': 1, '   re chan ne l</w>': 1, '   re la ys</w>': 6, '   do me</w>': 12, '   p in back</w>': 9, '   ne bu l a</w>': 11, '   un st able</w>': 21, '   8 5 </w>': 10, '   sen ti ent</w>': 5, '   ba ll o on</w>': 31, '   s qu a w k</w>': 6, '   ma ge ll ani c</w>': 1, '   h or se head</w>': 1, ' 9 </w>': 26, ' dro p</w>': 8, '   sy n ch r on i ze</w>': 3, '   shi el ding</w>': 10, '   pla t in u m</w>': 8, '   e u ri di u m</w>': 2, '   de ton ate</w>': 11, '   sy n ch r on i z es</w>': 2, '   con ce p ts</w>': 9, '   v a li d</w>': 14, '   ori g in ate</w>': 4, '   im pu l ses</w>': 13, '   di st in c tly</w>': 17, '   in tri gu ing</w>': 14, '   re la y ed</w>': 4, '   con ne c tions</w>': 33, '   sen sor y</w>': 8, '   sti mu late</w>': 5, '   com pu ting</w>': 2, '   app ar a tu s</w>': 7, '   re v ea ls</w>': 5, '   con cre te</w>': 22, '   in tu i ti ve ly</w>': 2, '   hel m et</w>': 23, '   ven tra l</w>': 4, '   re mar k</w>': 15, '   re pe ll ant</w>': 2, '   brea k ers</w>': 1, '   char ts</w>': 26, '   d war f</w>': 11, '   te mp</w>': 10, '   gh f</w>': 2, '   cor re ction</w>': 8, '   inter co m</w>': 5, ' minute</w>': 18, '   di a me ter</w>': 2, '   app ro x i ma tion</w>': 2, '   ma f h kin</w>': 1, '   o ble</w>': 1, '   g ro o p</w>': 1, '   de br is</w>': 15, '   p ho en i x</w>': 27, '   co l ors</w>': 40, '   ra in b ow</w>': 13, '   a st er o i ds</w>': 9, '   sc an ner</w>': 10, '   p in point</w>': 8, '   star su it</w>': 1, ' some where</w>': 4, '   ci r cu i ts</w>': 14, '   p ea king</w>': 2, '   gla ss y</w>': 1, '   ma li bu </w>': 9, '   z u ma</w>': 1, ' ci r cu it</w>': 2, '   tri lli on</w>': 6, '   si gh ts</w>': 15, '   see thing</w>': 2, '   pe b b les</w>': 4, '   do o ll ttle</w>': 1, '   a ged</w>': 11, '   st are</w>': 30, '   me te ors</w>': 4, '   c lu st ers</w>': 2, ' should</w>': 13, '   en clo sed</w>': 8, '   ph en om en o lo g y</w>': 2, '   fa ded</w>': 7, '   ma l fun c tions</w>': 3, '   ac ti v ate</w>': 20, '   a z i mu th</w>': 2, '   do d g ers</w>': 5, '   dis ban ded</w>': 1, '   b ru no</w>': 9, '   in su l in</w>': 6, '   com a</w>': 40, '   cla ssi er</w>': 3, '   ca mi ll e</w>': 6, '   star in</w>': 7, '   p le a</w>': 9, '   son ri s a</w>': 7, '   he ar da</w>': 1, '   g rea sing</w>': 2, '   wa g ging</w>': 2, '   s k an k</w>': 8, '   re de cor a ting</w>': 2, ' wh att ya</w>': 1, '   ex e cu te</w>': 12, '   min i a ture</w>': 16, '   z in ged</w>': 1, '   p li ers</w>': 7, '   ti a</w>': 4, '   dar r en</w>': 4, '   ju st ine</w>': 5, '   un wi lling</w>': 6, '   par ti ci p ant</w>': 4, '   k en dr a</w>': 1, '   ro om ma tes</w>': 6, '   l en g th</w>': 21, '   mi s con st ru ed</w>': 2, '   the o</w>': 101, '   kin</w>': 39, '   l y in</w>': 25, '   ja h</w>': 5, '   in st ru ment</w>': 18, '   pro p he ts</w>': 2, '   un ri gh te ous</w>': 1, '   m on</w>': 39, '   b ru d da h</w>': 2, '   o ver head</w>': 7, '   de u ce</w>': 18, '   sc ri ll a</w>': 1, '   sa li v a</w>': 8, '   wa g on</w>': 35, '   ca tt in</w>': 1, '   ye ch</w>': 1, '   r en e e</w>': 2, '   j ad a</w>': 1, '   bro o ke</w>': 18, '   co pe</w>': 9, '   con f er en c ing</w>': 1, '   so ph y</w>': 3, '   m om en tu m</w>': 4, '   lo g an</w>': 30, '   cha ir s</w>': 31, '   ki tt en s</w>': 5, '   mi tt en s</w>': 2, '   z ack</w>': 25, '   han over</w>': 6, '   sen ten c ed</w>': 8, '   pen n</w>': 1, '   ra w lin s</w>': 1, '   w y om ing</w>': 11, '   cor re c tions</w>': 2, '   di st in gu i sh ing</w>': 3, '   ta tt oo s</w>': 7, '   ci vi li z ation</w>': 41, '   un ra v el ing</w>': 2, '   si pp ing</w>': 3, '   la tt es</w>': 2, '   pre da t ors</w>': 4, '   o ver s ea s</w>': 14, '   bi d der</w>': 9, '   youn ge st</w>': 13, '   dr ow ned</w>': 30, '   v an cou ver</w>': 22, '   s mu g g l ers</w>': 9, '   o ver bo ard</w>': 16, '   g un point</w>': 5, '   ac coun ta</w>': 3, '   son u v ab i tch</w>': 10, '   cor d</w>': 18, '   f ac i li ty</w>': 37, '   att r ac ting</w>': 3, '   l en g th s</w>': 12, '   can a di an</w>': 29, '   ha cks</w>': 9, '   e d g ar</w>': 40, '   man a g ers</w>': 9, '   su per vi sed</w>': 2, '   re mo ving</w>': 6, '   cor to di a z a p ine</w>': 1, '   ge l</w>': 2, '   re pl ac ing</w>': 10, '   p ow der ed</w>': 4, '   pl ac e bo s</w>': 2, '   di st ri bu ted</w>': 4, '   v a</w>': 19, '   ve ter ans</w>': 7, '   cl in i c s</w>': 7, '   man ti c ore</w>': 2, '   sur ro ga te</w>': 4, ' vi tr o</w>': 1, '   se qu en ces</w>': 3, '   ho dge</w>': 1, ' po dge</w>': 1, '   my th o lo g y</w>': 10, '   te ch ni cal</w>': 39, ' chi mer a</w>': 7, '   re com b in ant</w>': 5, '   in f an try</w>': 10, ' tri p to p han e</w>': 1, '   ne u ro tr ans mi tter</w>': 1, '   h ome op a th y</w>': 1, '   se i z u r es</w>': 4, '   li gh t bu l b</w>': 3, '   hea d first</w>': 2, '   ro ck y</w>': 37, '   s qu ir re l</w>': 7, ' night</w>': 46, '   f la tt er y</w>': 7, '   n ou v ea u</w>': 4, '   tra i p se</w>': 3, '   cre e p ed</w>': 2, '   te x</w>': 19, '   per v o</w>': 1, '   pa w ed</w>': 2, '   pri ce less</w>': 10, '   s n ea king</w>': 26, ' b ent</w>': 2, '   pe st er ing</w>': 5, '   ac tive</w>': 33, '   c li mb ing</w>': 31, '   ob sc en e</w>': 9, '   bl in ked</w>': 1, ' j our na li st</w>': 1, '   pa y day</w>': 13, '   ca le</w>': 1, '   god de ss</w>': 17, '   com pre h ends</w>': 1, '   god de ss es</w>': 2, '   r a</w>': 3, '   pro te ctor</w>': 12, '   a ven ger</w>': 3, '   de st ro y er</w>': 7, '   gi ver</w>': 2, '   e g y p ti an</w>': 9, '   ba st</w>': 1, '   1 9 2 0</w>': 5, '   att ri bu ted</w>': 4, '   chi t ar us</w>': 1, '   ten se</w>': 33, '   s ans om me</w>': 1, '   l y di a</w>': 62, '   ac r y li c</w>': 1, '   pri ss y</w>': 3, '   s ke tch y</w>': 7, '   fin ger na il</w>': 4, '   fi an ce</w>': 11, '   na tom a</w>': 1, '   ke y bo ard</w>': 8, '   fi x ture</w>': 1, '   ar re sts</w>': 11, '   ad mi ssi ons</w>': 12, '   cor on ers</w>': 2, '   sp a de work</w>': 1, '   da ta ba ses</w>': 2, '   i den ti f ying</w>': 1, '   ne cks</w>': 14, '   ne ar est</w>': 31, ' cl in i c</w>': 2, '   re gi st ry</w>': 8, '   te ch ni ci ans</w>': 2, '   a ir bur st</w>': 1, '   wi p es</w>': 4, ' anything</w>': 35, '   de sc ri p tion</w>': 36, '   g love</w>': 17, '   d m v </w>': 7, '   ta ho e</w>': 15, '   ta gs</w>': 9, '   a g t</w>': 1, ' 3 4 9 </w>': 1, '   op u l ent</w>': 1, '   li fe st y le</w>': 14, '   ro ar in</w>': 1, '   ser ving</w>': 33, '   he el</w>': 26, ' girl</w>': 11, ' li ci ous</w>': 2, '   sp ar ing</w>': 4, '   na ta lie</w>': 28, '   pri son ers</w>': 43, '   gen es</w>': 16, '   k in du v </w>': 1, ' my st er i ous</w>': 1, '   gra in</w>': 20, ' no ther</w>': 6, '   cra ps</w>': 9, '   wa st in</w>': 9, '   thou gh t ful</w>': 18, '   c ran i u m</w>': 4, '   w re cks</w>': 4, '   sp ok es</w>': 1, ' idea</w>': 3, '   e sc row</w>': 9, '   a gre es</w>': 15, '   gu ar an tor</w>': 1, '   bra z il</w>': 20, '   re in car na tion</w>': 10, '   st rea k</w>': 30, '   cont in u es</w>': 25, '   dea li o</w>': 1, '   wa ck</w>': 3, '   sc ar in</w>': 7, '   po o p</w>': 10, '   ve ts</w>': 4, '   ho ok y</w>': 2, '   ma x i e</w>': 31, '   ex er ci sed</w>': 4, '   s mi d g en</w>': 2, '   ju d ge ment</w>': 26, ' re stra int</w>': 2, ' mon ster</w>': 4, '   do or be ll</w>': 8, '   ma te o</w>': 1, '   dis so l ve</w>': 9, '   com pa ssi on ate</w>': 8, '   p hi lan der ing</w>': 2, '   s ou l ma tes</w>': 1, '   e go</w>': 25, '   de man ding</w>': 14, '   n at</w>': 2, '   p on y</w>': 44, '   con de sc en ding</w>': 8, ' ser ving</w>': 1, '   ar ro g ant</w>': 20, ' gu il ty</w>': 2, '   ex cu ses</w>': 33, ' offi ci ally</w>': 1, '   in vo l ves</w>': 14, '   8 4 2 </w>': 2, '   be u la h</w>': 1, '   ha ight</w>': 2, ' thir te en</w>': 4, '   de p lo y ed</w>': 4, '   ch ea p en</w>': 2, ' ex </w>': 12, ' hu s b and</w>': 19, '   cha m pi on</w>': 57, '   fa king</w>': 18, '   pro je c ti le</w>': 2, '   di ar r he a</w>': 3, '   w oo dy</w>': 33, '   ha w</w>': 3, ' ha w</w>': 5, '   man ly</w>': 7, '   b re w s k i</w>': 1, '   g lu m</w>': 10, '   comp en sa t ory</w>': 3, '   ex hi bi tion i sts</w>': 1, '   nu t ca ses</w>': 2, '   g ro o ve</w>': 3, '   bo o gi e</w>': 8, '   oo gi e</w>': 3, '   di ck y</w>': 1, '   fin ch</w>': 22, '   de t ro it</w>': 58, '   pa y di r t</w>': 3, '   dan i el s</w>': 43, '   sc ra w ny</w>': 7, '   an n</w>': 145, '   ar b or</w>': 7, '   re lo ca ting</w>': 1, '   do o d les</w>': 1, '   so l o</w>': 19, '   d ru mm er</w>': 6, '   tri vi a</w>': 4, '   si m mon s</w>': 4, '   gra f ted</w>': 1, '   j er e mi ah</w>': 19, '   st e ll a</w>': 25, '   ca l cu la ting</w>': 4, '   mon go lo i ds</w>': 1, '   dis gu st ing</w>': 56, '   no b s</w>': 1, '   hi tch ed</w>': 7, '   po th ea ds</w>': 1, '   ho o king</w>': 6, '   fun k y</w>': 10, '   di sc o</w>': 24, '   in f er no</w>': 4, '   dro pp in</w>': 3, '   bri lli an ce</w>': 5, '   no be l</w>': 7, '   wai ta min it</w>': 2, '   du d es</w>': 28, '   wai ls</w>': 1, '   d ru ms</w>': 18, '   fe m my</w>': 3, '   si mi an</w>': 4, '   sa m son</w>': 2, '   sa mo an</w>': 6, ' j </w>': 97, '   tra vo l ta</w>': 2, ' den ny</w>': 1, '   ter i o</w>': 1, '   ber n ard</w>': 18, '   s an du s k y</w>': 3, ' ho o</w>': 20, '   l un ch time</w>': 5, '   le x</w>': 64, '   bu ds</w>': 1, '   cu r l</w>': 9, ' my st er y</w>': 3, '   na me ly</w>': 4, '   cu r li er</w>': 1, '   t un age</w>': 1, '   b lu es</w>': 14, '   vo l v o</w>': 6, '   bar ter</w>': 5, '   s ca l per</w>': 2, '   re con ven e</w>': 3, '   inter se ction</w>': 4, '   s n ea ks</w>': 1, '   stu b s</w>': 3, ' re</w>': 207, ' en ter</w>': 5, '   con cer to</w>': 7, '   f er</w>': 31, ' ho ly</w>': 4, '   br o</w>': 36, '   s li c ed</w>': 8, '   di c ed</w>': 3, '   ju li en ned</w>': 1, '   chri st ine</w>': 29, '   cho p</w>': 27, '   g our m et</w>': 3, '   a bu ses</w>': 1, '   mo ther ho od</w>': 9, '   j ee z is</w>': 1, '   du ds</w>': 4, '   di tch ing</w>': 3, '   h are</w>': 13, '   s che me</w>': 40, '   d ru m</w>': 26, '   char ac ter i z ed</w>': 3, '   v a gu e ly</w>': 8, '   in di vi du al</w>': 48, '   che z</w>': 3, '   g ro in e co lo gi st</w>': 1, '   re ar vi e w</w>': 3, '   c in c in na t i</w>': 11, '   ra d ar</w>': 41, '   cra z i est</w>': 9, '   e le c tr on i c s</w>': 11, '   f ar ted</w>': 3, '   v an ha f ton</w>': 1, '   do o fu s</w>': 6, '   st e ll as</w>': 2, '   ha wai i an</w>': 5, '   de e f</w>': 2, '   re ca p</w>': 4, '   s la pped</w>': 21, '   s lu g</w>': 16, '   a do</w>': 3, '   un a du l ter a ted</w>': 2, '   un di lu ted</w>': 1, '   a sta i re</w>': 3, '   tri ba l</w>': 8, '   ri tu a li sti c</w>': 1, ' sp on t an e ous</w>': 2, '   e mb ar ra ss kin</w>': 1, '   ta k ers</w>': 3, '   bra in st or m</w>': 3, '   bo o ted</w>': 7, '   v ent</w>': 10, '   ho sti li ty</w>': 14, '   bra in less</w>': 5, '   in f er na l</w>': 2, '   mor e like</w>': 1, '   of f ers</w>': 28, '   s li ce</w>': 28, '   ti x</w>': 2, '   fe m bo t</w>': 1, '   ni m ro d</w>': 3, '   bu ll wor k er</w>': 1, '   cra pp er</w>': 7, '   bu tt e</w>': 4, '   mon t an a</w>': 35, '   qu an d ary</w>': 4, '   hea th en ous</w>': 1, '   wi lled</w>': 4, '   de mon i c</w>': 3, '   h or ri fi c</w>': 3, ' bo ar ding</w>': 1, '   p hi lli p</w>': 39, '   m c nu l ty</w>': 1, '   part ying</w>': 9, '   sa t ani c</w>': 4, '   s k a ting</w>': 14, '   ra gs</w>': 17, '   sp l en did</w>': 35, '   ni gh ti e</w>': 2, '   bu sti er</w>': 1, '   sin ful</w>': 5, '   p ee l ed</w>': 5, '   under n ea th</w>': 36, '   si r en</w>': 9, '   te mp ta tion</w>': 13, '   w rea king</w>': 2, '   bri de s ma i ds</w>': 4, '   th rea ds</w>': 10, '   ju i ci er</w>': 1, '   lu ci f er</w>': 4, '   car na l</w>': 1, '   im pu re</w>': 2, '   do o zy</w>': 3, '   o li vi a</w>': 1, '   ne w ton</w>': 4, ' joh n</w>': 9, '   j in x ing</w>': 2, '   b ong</w>': 4, '   hu ck</w>': 1, '   u p ho l st er y</w>': 5, '   e y ow ch</w>': 1, '   can ine</w>': 3, ' i ci ty</w>': 1, ' ca tch</w>': 2, '   j i z z</w>': 5, '   pro ver b</w>': 8, '   s mu g</w>': 6, '   s nu g</w>': 2, '   h un k y</w>': 2, ' d ory</w>': 2, '   j in x ed</w>': 3, '   s qu ee z ed</w>': 7, '   re o</w>': 1, '   sp ee d wa g on</w>': 1, '   ro ll ers</w>': 5, '   com b</w>': 21, '   sh ro om s</w>': 1, '   l s d</w>': 8, '   s me lling</w>': 15, '   g y ne co lo gi st</w>': 7, '   car ly</w>': 1, '   na d</w>': 1, '   ch on go</w>': 2, '   s n ac ks</w>': 3, '   w ea s ly</w>': 1, '   th rea ten s</w>': 7, '   ma g got</w>': 2, ' sta te</w>': 6, ' ta x i</w>': 1, '   y ee ee ha a a w w w</w>': 1, '   b ack sta ge</w>': 4, '   co b o</w>': 1, '   ei s en</w>': 1, '   f re h le y</w>': 1, ' c ri s cu l a</w>': 1, '   par al y sed</w>': 1, ' d ou b t</w>': 2, '   te mp ta tions</w>': 6, '   pre mar i tal</w>': 1, ' sp rea ding</w>': 1, '   be ha vi our</w>': 10, '   stu den ts</w>': 52, '   gu st a</w>': 1, '   mon gre ls</w>': 2, '   dar k o</w>': 5, '   s mo o sh ed</w>': 1, '   ne i gh b our</w>': 4, '   con tra di c ting</w>': 2, '   de st in es</w>': 1, '   vi su ally</w>': 3, '   de st in i es</w>': 3, ' for med</w>': 1, ' de ci ded</w>': 3, '   wor m ho les</w>': 4, '   ve s se l</w>': 61, '   por tal</w>': 14, '   wor m ho le</w>': 8, '   un for e seen</w>': 3, ' time</w>': 79, '   cen t re</w>': 9, '   in f an ts</w>': 5, '   in f an cy</w>': 1, '   cra ft</w>': 22, '   de l or ean</w>': 1, ' ro s en</w>': 1, '   sp ac e ship</w>': 8, '   ha w king</w>': 1, '   di st ant</w>': 23, '   re gi ons</w>': 3, '   mon ni to ff</w>': 2, '   uni cor n</w>': 20, '   sa man th a</w>': 5, '   ar i el</w>': 2, '   sa vi our</w>': 2, '   dea</w>': 17, '   m ac h in a</w>': 1, '   f loo ded</w>': 12, '   sle e p wal king</w>': 2, '   j i lli on</w>': 1, '   gre tch en</w>': 7, '   st e p dad</w>': 4, '   ri dge</w>': 23, '   gen er at ors</w>': 3, '   im gs</w>': 1, '   cha p t ers</w>': 9, '   c any on</w>': 38, '   per pe tu al</w>': 9, '   st er o i ds</w>': 5, '   car pa th i an</w>': 4, '   sle e p wal k</w>': 2, '   sle e p wal k er</w>': 3, '   af ter w ard</w>': 10, '   list er</w>': 1, '   1 8 9 5 </w>': 2, '   an ti se p ti c s</w>': 2, '   su per h er o</w>': 17, '   ab an d on ed</w>': 46, '   i ll u stra tions</w>': 1, '   re stra in ing</w>': 11, '   can ce lled</w>': 14, '   lu mp ing</w>': 1, '   ca te g ori es</w>': 6, '   de e pe st</w>': 24, '   li fe l ine</w>': 7, '   lu mp ed</w>': 2, '   wa ck o</w>': 14, '   th u r man</w>': 6, '   y ar n</w>': 12, '   ob st ac les</w>': 5, '   ro ber ta</w>': 2, '   sp ar row</w>': 17, '   co ll e c tions</w>': 3, '   sh in g le</w>': 1, ' bu tter</w>': 1, '   s n ack time</w>': 1, '   wal ru s</w>': 3, '   lin gu i st</w>': 2, '   ph ra ses</w>': 6, '   com b in a tions</w>': 9, ' ce ll ar</w>': 2, '   p om er o y</w>': 1, '   r ab b it</w>': 72, '   sor row</w>': 18, '   bu n ni es</w>': 3, '   m our n</w>': 6, '   fi ver</w>': 2, '   er a</w>': 21, ' f oo t</w>': 14, ' ta ll</w>': 3, '   sh re ds</w>': 13, '   a g no sti c</w>': 2, '   car ved</w>': 10, '   su b con s ci ous</w>': 16, '   hi pp o s</w>': 2, '   st om ac h s</w>': 2, '   de ba ting</w>': 4, '   we i gh ing</w>': 3, '   con s</w>': 6, '   de ba te</w>': 29, '   cra w l ed</w>': 15, '   ca lli e</w>': 2, '   re min ded</w>': 16, '   cre e pi est</w>': 2, '   al y ss a</w>': 13, '   mi lan o</w>': 1, '   p sy cho lo gi ca lly</w>': 5, '   s ki pp ing</w>': 8, '   c y c les</w>': 6, '   car ni v al</w>': 14, '   m ou thing</w>': 5, '   ex per im en tal</w>': 18, '   h y p no ther a p y</w>': 1, '   l en g th y</w>': 2, '   man i fe sta tion</w>': 5, '   i ma g in ary</w>': 9, '   ex per i en c ing</w>': 16, '   com mon ly</w>': 6, '   ha ll u c in ation</w>': 9, '   a g gre ssion</w>': 11, '   fran ki e</w>': 82, '   fee d l er</w>': 1, '   di men sion</w>': 22, '   fa a</w>': 4, '   fi x es</w>': 9, '   ta u ru s</w>': 2, '   f lu sh ing</w>': 5, '   con ta in er</w>': 10, '   rea li se</w>': 12, '   du k a k is</w>': 1, '   pri or</w>': 44, '   cla ss ro om s</w>': 2, '   mo tion</w>': 46, '   ar ra i g n ment</w>': 5, '   cha per one</w>': 8, '   d ev a sta ted</w>': 12, '   con sp ir ac y</w>': 44, '   spe ar head</w>': 1, '   ca m pa i g n</w>': 55, '   ci vi c</w>': 5, '   con f li ct</w>': 24, ' por n</w>': 1, '   d un ge on</w>': 5, '   c ri s is</w>': 52, '   a ll e ga tions</w>': 6, '   wi t ne ss ing</w>': 6, '   sig ni fi c ant</w>': 20, '   pa th s</w>': 15, '   su c cu mb </w>': 4, ' b on an z a</w>': 1, '   gra ha m</w>': 67, '   gre en e</w>': 13, '   p ta</w>': 15, '   ac kn ow le dge</w>': 17, '   cu r ri cu lu m</w>': 8, '   b an</w>': 7, '   v an da li s m</w>': 7, '   mi d d le se x</w>': 1, '   tr an sc ends</w>': 1, ' t ea ch er</w>': 5, '   ba ll o ons</w>': 8, '   v ani ty</w>': 17, '   h om o</w>': 14, '   s mu r fe tt e</w>': 2, '   s mu r f s</w>': 3, '   s mu r f</w>': 1, '   ra sp ber ry</w>': 3, ' 5 5 </w>': 5, '   mar th a</w>': 35, '   mo o</w>': 2, '   ch u t</w>': 1, '   di an e</w>': 66, '   sa w y er</w>': 8, '   na tion al s</w>': 5, '   min ne so ta</w>': 27, '   te en</w>': 12, '   l or e tta</w>': 46, ' j o</w>': 4, ' a ir por t</w>': 1, '   in do or</w>': 5, '   a h h h h</w>': 6, '   h ow ard</w>': 82, '   joh n s ons</w>': 1, ' per son al</w>': 7, '   con su l ta tion</w>': 7, '   ee e h</w>': 1, '   ch or e o gra p her</w>': 1, '   s wa n</w>': 21, '   bu n ge</w>': 1, '   cor ds</w>': 3, ' be tch a</w>': 2, ' once</w>': 9, '   car ni e</w>': 2, '   qui tt in</w>': 10, '   be tch a</w>': 20, ' mo m</w>': 9, '   s ac ri fi c ed</w>': 14, '   tu m my</w>': 5, '   th i gh s</w>': 11, '   chan g in</w>': 3, '   cl un g</w>': 4, '   f l y in</w>': 13, '   d art</w>': 10, '   an ne tt e</w>': 32, '   m om mm m</w>': 1, '   he ar se</w>': 8, ' be c ch a</w>': 1, '   y ah</w>': 160, '   le e man</w>': 5, ' s k in ny</w>': 2, '   n an cy</w>': 49, ' be lt</w>': 2, '   l is</w>': 4, '   ki d ne y</w>': 12, '   you s</w>': 4, ' li z a</w>': 2, '   wh </w>': 34, ' wh </w>': 10, '   com pe</w>': 1, ' e te</w>': 1, '   be ar er</w>': 6, '   ne ck l ine</w>': 1, ' spe ci al</w>': 11, '   c r ow ned</w>': 4, '   com pe t in</w>': 1, ' bring</w>': 10, '   lu ck ys</w>': 1, '   tra il er</w>': 39, ' ch ea p</w>': 3, ' lo v in</w>': 2, '   s n ow mo bi l in</w>': 1, ' sc ary</w>': 1, ' li f er</w>': 1, '   de sc ent</w>': 4, ' ra i s in</w>': 1, '   ran ch</w>': 71, '   ta m my</w>': 17, '   j i ff y</w>': 5, '   le tt in</w>': 10, '   k til</w>': 1, '   ni co t ine</w>': 11, '   ons</w>': 1, '   b le w ey</w>': 1, '   1 9 6 3 </w>': 6, '   l ou i s vi ll e</w>': 8, '   h ome town</w>': 6, '   at k ins</w>': 4, '   com pe te</w>': 17, '   o ver ru le</w>': 2, '   co stu mes</w>': 15, '   pre ten d in</w>': 2, '   op en in</w>': 1, '   be ck y</w>': 17, '   fi t ne ss</w>': 9, '   du r in</w>': 8, '   s no tty</w>': 7, '   wi ll in</w>': 4, '   le s lie</w>': 26, '   hi c ki es</w>': 1, '   j an e ll e</w>': 1, ' sure</w>': 36, '   le u te fi s k</w>': 1, ' mar y</w>': 5, ' jo se p h</w>': 4, '   tra il ers</w>': 2, '   ha ir do</w>': 2, ' most</w>': 8, '   sen d in</w>': 6, '   ta pp er</w>': 1, ' co ps</w>': 6, ' sa d dle</w>': 4, ' gr ac i ous</w>': 1, '   ne i gh bor ing</w>': 2, '   pa ge an ts</w>': 2, '   i ow a</w>': 13, ' f oun d ers</w>': 1, '   app li ca tions</w>': 15, '   ti k i</w>': 3, '   app li ca tion</w>': 19, '   sp un k y</w>': 5, '   ter ry</w>': 88, '   m ac ey</w>': 4, '   a m er</w>': 1, ' bu y</w>': 8, '   le g ged</w>': 1, '   li b b ers</w>': 1, ' pa ge an ts</w>': 1, ' de me an ing</w>': 1, '   s mo kin</w>': 16, '   i c ed</w>': 19, '   cu r ry</w>': 8, '   ha ha ha ha ha ha ha h a</w>': 1, '   t ac o s</w>': 1, '   le ster</w>': 63, '   sh ow room</w>': 3, '   g lo be</w>': 16, '   e qu a tor</w>': 3, '   wi tch es</w>': 21, '   en id</w>': 25, '   g list en ing</w>': 1, '   st re e t wal k ers</w>': 1, '   ex p lo ding</w>': 4, '   th re sh er</w>': 4, ' pr ou d</w>': 2, '   out fi ts</w>': 10, '   ma ll</w>': 47, '   lu ther ans</w>': 1, '   gra pe</w>': 8, '   k oo la id</w>': 2, '   com m un al</w>': 3, '   te mp t in</w>': 1, '   d on i g an</w>': 1, '   si de wal ks</w>': 4, '   sh ow y</w>': 1, ' today</w>': 8, '   in c lu d es</w>': 12, '   min n ea po l is</w>': 8, ' s in</w>': 6, '   d on u ts</w>': 7, '   han k ey</w>': 1, '   shi thou se</w>': 6, '   e e</w>': 3, ' a a a y ee e e</w>': 1, ' a a a a you i a a a ee ee ee e e</w>': 1, '   pi e ho le</w>': 1, '   ac ce p t an ce</w>': 6, '   uni ver si ty</w>': 73, '   t in a</w>': 32, ' t in a</w>': 1, '   se i k o</w>': 2, '   so lo m on</w>': 13, '   da me</w>': 40, '   ga ther ing</w>': 25, '   g l ori a</w>': 28, '   a mm uni tion</w>': 17, '   mi c ro b es</w>': 4, '   ven d or</w>': 1, '   mi li ti a</w>': 23, '   pe an u ts</w>': 10, '   en clo se</w>': 3, '   a ir ma il</w>': 1, ' ro s is</w>': 2, '   so li ta i re</w>': 4, '   sp ys</w>': 1, ' gr en ad es</w>': 1, '   ke gs</w>': 4, '   man e u ver s</w>': 8, ' f ac ed</w>': 23, '   s na tch ers</w>': 1, ' wa sh in g ton</w>': 2, '   de la w are</w>': 7, '   ou gh tta</w>': 14, '   t ea s da le</w>': 12, '   tr en t in o</w>': 3, '   in su lt</w>': 47, '   sc ra p</w>': 14, '   ex a min ation</w>': 23, ' offi ce</w>': 6, '   mi ce</w>': 18, '   ra pi ds</w>': 9, '   ni a gar a</w>': 11, '   nu i s an ce</w>': 19, ' pe an u ts</w>': 2, '   re pre sen ta ti ves</w>': 9, '   oun ces</w>': 9, '   po o dle</w>': 9, '   ye h</w>': 43, '   b loo d h ound</w>': 4, '   an e mi c</w>': 4, '   who le sa le</w>': 3, '   fi re f ly</w>': 12, '   chi co lin i</w>': 1, '   fe ar less</w>': 5, '   ra pi di ty</w>': 1, '   re lin qui sh es</w>': 1, '   gra ft</w>': 7, '   ex hi bi ted</w>': 1, '   pro hi bi ted</w>': 4, '   t rea su ry</w>': 17, '   che w</w>': 26, '   che w ed</w>': 10, '   che w er</w>': 1, '   ha il</w>': 23, '   f re e d on i a</w>': 5, '   c r ab </w>': 11, '   u p start</w>': 6, '   run t</w>': 6, '   s w ine</w>': 27, '   for ge ts</w>': 14, '   te mp ers</w>': 1, '   ba h</w>': 6, '   hu mi li a ting</w>': 19, '   chi ro po di st</w>': 1, '   ver a</w>': 11, '   r ac es</w>': 19, '   j ack k ni fe</w>': 2, ' 2 5 </w>': 9, '   go ver n</w>': 5, '   thr ong</w>': 2, '   whi ms</w>': 8, '   to tt er ed</w>': 1, '   d y na sti es</w>': 1, '   ro cked</w>': 2, '   pl un ged</w>': 6, '   hea d long</w>': 2, '   cha s m</w>': 1, '   hel p less</w>': 32, '   in si di ous</w>': 4, '   ir re si sti ble</w>': 6, '   be tta</w>': 2, '   re con si der</w>': 13, '   b re e ch es</w>': 2, '   cle men cy</w>': 5, '   ru fu s</w>': 16, '   ge tt y s bur g</w>': 6, '   vi e w s</w>': 13, '   li ber al</w>': 10, '   whi st ling</w>': 7, '   i ll u stra tion</w>': 1, '   no ta b les</w>': 1, '   g al a</w>': 5, '   re sts</w>': 16, '   f oo t st e ps</w>': 14, '   im pro ve ment</w>': 13, '   ma tri cu late</w>': 2, '   he ar ti ly</w>': 3, '   en d or se</w>': 10, '   in di g ni ty</w>': 2, '   re gre tt able</w>': 4, '   ce ment</w>': 10, '   di c ta tor</w>': 5, '   hu s ban ds</w>': 16, '   t re ves</w>': 37, '   mer ri ck</w>': 43, '   ma ter na l</w>': 4, '   fu l fi lled</w>': 8, '   f re d die</w>': 60, '   v a m pi re</w>': 49, '   b y tes</w>': 5, ' f re d die</w>': 1, '   car r</w>': 8, '   go m m</w>': 6, '   de li ver ing</w>': 21, '   ser m on</w>': 5, '   s way</w>': 11, '   poli sh ing</w>': 3, '   en d ow ing</w>': 1, '   ob li ga tion</w>': 17, '   un tr ou bl ed</w>': 1, '   par ro ting</w>': 1, '   sen sed</w>': 11, '   un sure</w>': 6, '   s w o ll en</w>': 7, '   ch u ms</w>': 3, '   gu v </w>': 3, '   wi ll fu lly</w>': 4, '   de pri v in</w>': 1, '   li v li ho od</w>': 1, '   mu ck</w>': 8, '   ra le</w>': 1, ' f all</w>': 5, '   b ac king</w>': 17, '   car ted</w>': 1, '   wi sh ers</w>': 2, '   und ou b te d ly</w>': 14, '   com mi tt e es</w>': 6, '   par ti cu l ars</w>': 6, '   h or ri b ly</w>': 6, '   bro ad ne ck</w>': 1, '   qu ea sy</w>': 4, '   ex ce p tion al</w>': 12, '   sh ort si gh te d ne ss</w>': 1, '   p sa l m</w>': 1, '   in sp ir ing</w>': 10, '   con si der ation</w>': 22, '   ne g le ct</w>': 3, '   in cu r ab les</w>': 3, '   dis com for t</w>': 3, '   com pre h end</w>': 13, '   con st ri c tive</w>': 1, '   de for mi ty</w>': 2, '   ea ger ne ss</w>': 4, '   sin gu l ar ly</w>': 3, '   cha p</w>': 12, '   di a g no se</w>': 5, '   e qui ta ble</w>': 1, '   mer i ts</w>': 3, '   sy m pa th i ze</w>': 11, '   de for med</w>': 6, '   in cu r able</w>': 1, '   sho cking</w>': 21, '   spi ri ting</w>': 1, '   ho o ded</w>': 2, '   li able</w>': 27, '   i so la tion</w>': 14, '   con ta gi ous</w>': 14, '   nu tri ti ous</w>': 3, '   ac qui red</w>': 14, '   pre par est</w>': 1, '   an o in te st</w>': 1, '   gi b ber i sh</w>': 8, '   pre sen tly</w>': 13, '   ab om in able</w>': 2, '   ho d g es</w>': 1, '   p ra ter</w>': 1, '   r h y me</w>': 13, '   ba ll a d</w>': 3, '   pa te</w>': 9, '   wi ther</w>': 2, '   wa x</w>': 19, '   ho llow</w>': 31, '   k ate</w>': 61, '   sh in es</w>': 8, '   ton gu es</w>': 8, '   man s</w>': 1, '   de ce i ts</w>': 2, '   di e u</w>': 6, '   lan gu es</w>': 1, '   h om mes</w>': 1, '   son t</w>': 1, '   p le in es</w>': 1, '   tra m per i es</w>': 1, '   par d on ne z</w>': 1, ' mo i</w>': 3, '   k a th ar ine</w>': 20, '   s oun d ly</w>': 2, '   bro k en ly</w>': 1, '   v ou ch sa fe</w>': 1, '   pl ead</w>': 17, ' su it</w>': 5, '   le st</w>': 10, '   ay</w>': 19, '   pi l g ri m</w>': 34, '   p al m ers</w>': 4, '   sh r ine</w>': 4, '   i ma g in ation</w>': 51, '   su sp end</w>': 11, '   dis beli e f</w>': 4, '   u no b ser ved</w>': 1, '   t y ran ts</w>': 2, '   on sta ge</w>': 3, '   happ i ly</w>': 42, '   bri gh te st</w>': 10, '   ga i e ty</w>': 2, '   f ru stra ting</w>': 12, '   ph y si ci an</w>': 24, '   de ter i or ate</w>': 5, '   k en da l</w>': 4, '   or deal</w>': 9, '   g ow n s</w>': 3, '   a le x an dr a</w>': 1, '   att r ac tion</w>': 22, '   spi re</w>': 4, '   ha ll way</w>': 18, '   wa st e can</w>': 1, '   p hi lli ps</w>': 14, '   ar ch es</w>': 1, '   go ver ning</w>': 1, '   ga u dy</w>': 3, '   co lu m n s</w>': 9, '   f l ows</w>': 4, '   a we ary</w>': 4, '   f l ow ing</w>': 22, '   f le e ting</w>': 4, '   man t le pi e ce</w>': 1, '   spe c t ac le</w>': 10, '   o h h h</w>': 32, '   can op y</w>': 3, '   wor k house</w>': 3, '   plea sing</w>': 4, '   com m end</w>': 6, ' may</w>': 13, '   na me sa ke</w>': 1, '   ma ke th</w>': 1, '   pa stu r es</w>': 4, '   lea de th</w>': 1, '   re st or e th</w>': 1, '   gu i de th</w>': 1, '   mo ther sh ead</w>': 14, '   mm mer ri ck</w>': 2, ' mm mer ri ck</w>': 2, ' mer ri ck</w>': 1, '   y ye ss</w>': 1, '   y y ye ss</w>': 2, ' y yes</w>': 1, '   y y y y</w>': 1, '   y y ye</w>': 1, '   li gh thou se</w>': 16, '   da sh ing</w>': 3, '   ba th ed</w>': 6, '   re li sh es</w>': 1, '   list en s</w>': 10, ' s m ac ks</w>': 1, '   ob st in ate</w>': 1, '   app ear an ce</w>': 47, '   han d some ly</w>': 4, '   pro pri e tor</w>': 4, '   h y</w>': 7, ' bra si l</w>': 4, '   au d</w>': 4, '   as ga ard</w>': 7, '   er i k</w>': 25, '   ke i te l</w>': 13, '   ab y ss</w>': 3, '   clo a k</w>': 25, '   ha l f d an</w>': 11, '   par ting</w>': 7, '   go al s</w>': 11, ' got</w>': 55, '   bra si l</w>': 2, ' friends</w>': 12, '   re s oun ding</w>': 7, '   dis gu i sed</w>': 5, '   lo k i</w>': 3, '   f re ya</w>': 2, ' vi king</w>': 1, '   pl und er</w>': 4, '   re gu la tions</w>': 18, '   ar nu lf</w>': 1, '   gen u in e ly</w>': 13, '   su mm on</w>': 9, ' tu m</w>': 5, '   tu m</w>': 3, ' t i</w>': 5, '   de cre ed</w>': 6, '   spi lls</w>': 10, '   sh or es</w>': 5, '   o d in</w>': 2, '   a a a gh</w>': 2, '   c ru de</w>': 21, '   loo ting</w>': 10, '   ci r cu l ar</w>': 1, '   ex pe di tion</w>': 31, '   pi ll a ging</w>': 2, '   th or fin n</w>': 2, '   har d shi ps</w>': 2, '   s nor r i</w>': 5, '   o ar</w>': 2, '   pi ff le</w>': 1, '   to il</w>': 2, '   loo t</w>': 9, '   pi ll age</w>': 3, ' ma gi c</w>': 2, '   o h h</w>': 25, '   le if</w>': 4, '   ber ser k</w>': 17, '   vi king</w>': 7, '   ra g n ar ok</w>': 4, '   f en ri r</w>': 2, '   ar r gh</w>': 1, '   o ars</w>': 5, '   di sh clo th</w>': 1, '   d ra g on</w>': 43, '   be ar ds</w>': 4, '   m ou st ac h es</w>': 1, '   or nu lf</w>': 1, '   s ven</w>': 8, '   mi ssi on ary</w>': 8, '   har a ld</w>': 3, '   bl ack s mi th</w>': 10, '   b j or n</w>': 4, '   ba g ged</w>': 5, ' fro st</w>': 3, '   im mer se</w>': 1, '   th or b j or n</w>': 2, '   vi fi l s son</w>': 2, '   bu d d hi st</w>': 4, '   con ver ted</w>': 8, '   la v at ory</w>': 3, '   o la f</w>': 6, '   tr y g g v as on</w>': 1, ' stop</w>': 30, '   b o</w>': 25, '   u r r gh</w>': 1, '   bl ack s mi th s</w>': 3, '   wa k en s</w>': 2, ' hea ds</w>': 5, ' bl ack s mi th</w>': 1, '   k ni ves</w>': 25, '   da g g ers</w>': 2, '   d ra gon s</w>': 9, '   fa ir ha ir</w>': 1, '   v al ha ll a</w>': 3, '   g rea sed</w>': 6, '   sh ar p en</w>': 4, '   a x es</w>': 2, '   cle men t ine</w>': 16, '   jo el</w>': 40, '   b ack l it</w>': 1, '   ha l o</w>': 13, '   fri z z</w>': 1, '   na om i</w>': 12, '   ho a x</w>': 13, '   any ho o</w>': 2, '   k ru c z y n s k i</w>': 3, '   mi er z wi a k</w>': 2, '   y ay</w>': 7, '   t an ger ine</w>': 3, '   sen sa tion al</w>': 14, '   he he h</w>': 1, '   he h</w>': 157, '   re qui re</w>': 30, '   un rea li sti c</w>': 9, '   ir re ver ence</w>': 1, ' gra tu i t ous</w>': 1, '   g ri p ing</w>': 4, ' fami li ar</w>': 1, '   tra i ls</w>': 3, '   con que sts</w>': 4, '   pa ssi ve</w>': 12, '   ti mi d</w>': 8, '   mo i</w>': 12, '   com m uni ca tive</w>': 1, '   st on ed</w>': 26, '   b on fi re</w>': 5, '   ou tru n</w>': 8, '   la s kin</w>': 1, '   f le x i ble</w>': 9, '   n er v ou s ne ss</w>': 4, '   c ri min ally</w>': 8, '   t re sp ass</w>': 7, '   he si ta ted</w>': 5, '   e le c tri ci ty</w>': 26, ' per ha ps</w>': 6, ' s ea side</w>': 1, '   gu sts</w>': 2, '   co in ci den tal</w>': 3, '   a k h ma to v a</w>': 1, '   x an a x</w>': 5, '   mo du la ted</w>': 1, '   hu ck le ber ry</w>': 10, '   h or ri d</w>': 5, '   y know</w>': 1, '   clo d</w>': 2, '   inter act</w>': 5, '   as sig n</w>': 14, '   ti p to e</w>': 5, '   st al k er</w>': 12, '   in fa tu ation</w>': 8, '   si l en ces</w>': 1, '   e ga d</w>': 1, '   h or ri fi ed</w>': 4, ' da te</w>': 6, '   cle m</w>': 5, '   s cou red</w>': 1, '   jo e ly</w>': 4, '   cu te y</w>': 1, '   an to ine</w>': 2, '   er as ed</w>': 14, '   er as er</w>': 3, '   er as ing</w>': 6, '   ra in y</w>': 13, '   s w ea ty</w>': 12, '   im possi b ly</w>': 1, '   fa ding</w>': 10, '   sha b by</w>': 6, '   v el ve te en</w>': 1, '   per ce p tive</w>': 9, '   s lo th</w>': 3, '   win o</w>': 4, '   wor my</w>': 1, '   dent</w>': 16, '   ti p sy</w>': 6, '   pi c ni c s</w>': 2, '   s ou l ma te</w>': 3, '   di x on</w>': 5, '   go of b all</w>': 2, '   w r in k les</w>': 4, '   re pu l si ve ly</w>': 1, '   co s mi c</w>': 16, '   con si st ent</w>': 6, '   ma tu ri ty</w>': 6, '   fu ll est</w>': 5, '   m ou th ed</w>': 3, '   pre mon i tions</w>': 3, '   se du ction</w>': 3, '   re pu g n ant</w>': 7, '   re fi ll</w>': 14, '   for ti es</w>': 6, '   fro st</w>': 14, '   in sc ri p tion</w>': 8, ' sh oo k</w>': 2, '   he m lo ck</w>': 2, '   ru ed</w>': 1, '   c r ows</w>': 12, '   g ee z</w>': 46, '   sta ir way</w>': 2, '   h en ne p in</w>': 2, ' ti ll</w>': 8, '   bi tt ers</w>': 1, '   bo ok st or es</w>': 1, '   wi l mon t</w>': 3, '   st al ked</w>': 4, '   st al k able</w>': 1, ' co l or</w>': 4, ' na m ing</w>': 1, '   ad je c ti ves</w>': 3, '   ar gu men ta tive</w>': 4, '   mu m pi sh</w>': 1, ' mer ci ful</w>': 1, '   or an ge</w>': 66, '   ha ze</w>': 7, '   s na pp y</w>': 9, '   re vo lu tion</w>': 34, '   do le</w>': 3, '   re co g ni z ed</w>': 36, '   u c ch</w>': 1, '   din er</w>': 27, '   bor d ers</w>': 8, '   ro ck vi ll e</w>': 2, '   lea p</w>': 23, '   in e pt</w>': 4, '   ma g da</w>': 12, '   o d d ly</w>': 8, '   z o lo ft</w>': 1, '   s w ea t shi r t</w>': 3, '   m om en t ar i ly</w>': 4, '   su ck er ed</w>': 6, '   ad ver ti sing</w>': 13, '   1 7 1 8 </w>': 2, '   di gi ts</w>': 5, '   secon d ly</w>': 7, '   wa ved</w>': 6, '   ca su al</w>': 22, '   s n ow ing</w>': 6, '   i ma g in ing</w>': 18, '   p ar</w>': 9, '   er a di ca te</w>': 3, '   de gra da tion</w>': 3, '   t ar ge ted</w>': 5, '   with er ed</w>': 2, '   ca l v in i s m</w>': 1, '   mi so g y ny</w>': 1, '   co lu m bi a</w>': 17, '   car ri e</w>': 14, '   th ri ll</w>': 34, '   bar t le t t</w>': 6, '   qu o ta tions</w>': 2, '   en gra ved</w>': 7, '   qu o tes</w>': 8, '   qu o ta tion</w>': 1, ' er as ed</w>': 1, '   vo l un t ar i ly</w>': 5, '   ex i sted</w>': 26, '   le an ing</w>': 15, '   go of y</w>': 24, '   gi g g l ed</w>': 2, '   be ha ving</w>': 11, '   s or</w>': 5, ' ry</w>': 15, '   t an g le</w>': 5, '   p ho bi as</w>': 1, '   sa d ne ss</w>': 13, '   ho pe le ss ne ss</w>': 1, '   ni e t z sc he</w>': 7, '   bar t le tt s</w>': 1, ' k ay</w>': 36, '   bo o</w>': 14, ' na ked</w>': 2, '   wor k place</w>': 4, ' b le ssed</w>': 1, '   for ge t ful</w>': 3, '   bl und ers</w>': 2, '   in spi ra tion al</w>': 2, '   pi ani st</w>': 8, '   s la w</w>': 7, '   pa stra m i</w>': 3, '   1 0 6 2 </w>': 1, '   sh er man</w>': 21, '   ju sti fi ably</w>': 1, ' ga te</w>': 1, '   2 0 0 4 </w>': 4, '   sp on t an e ous</w>': 9, '   sta g n ant</w>': 1, '   ho li days</w>': 21, '   vo lu p tu ous</w>': 4, '   un in spi red</w>': 1, '   pro gen i tor</w>': 1, '   ri d ge to p</w>': 1, '   de l or es</w>': 4, '   char ac ter i sti c s</w>': 11, '   si wa sh</w>': 9, '   po st ca ta st ro p he</w>': 1, '   of f sp r ing</w>': 9, '   ne ce ssi ty</w>': 8, '   inter mar ry</w>': 1, '   for m ing</w>': 12, '   th u mb s</w>': 28, '   re ly</w>': 17, '   vi r tu al</w>': 8, '   e qu al s</w>': 17, '   p ea ce fu lly</w>': 7, '   d ra sti ca lly</w>': 4, '   ca la mi t ous</w>': 1, '   ear th qu a kes</w>': 10, '   ca ta st ro ph es</w>': 2, '   v al u es</w>': 21, '   con su mp tion</w>': 7, '   con di tion ed</w>': 6, '   ye ar n in gs</w>': 1, ' e con om i c</w>': 1, '   p hi lo so p hi es</w>': 3, '   di sa st ers</w>': 5, '   a po ca l y p ti c</w>': 2, '   hon e ys</w>': 6, '   wor l d wi de</w>': 10, '   ca la mi ty</w>': 6, '   fa m ine</w>': 5, '   re du ction</w>': 5, ' ti ck et</w>': 6, '   re d s k ins</w>': 2, '   coun te ss</w>': 12, '   de e ded</w>': 2, '   c ow gir ls</w>': 14, ' 3 0 0</w>': 6, '   la be l ed</w>': 3, ' in di ans</w>': 3, '   or an g es</w>': 16, '   fi tting</w>': 14, '   mi s na med</w>': 1, '   1 9 0 6 </w>': 1, '   ch ink</w>': 11, '   tu le</w>': 1, '   si er r a</w>': 11, '   n ev ad a</w>': 19, '   clo ck wor ks</w>': 5, '   ti cks</w>': 8, '   ac cu ra tely</w>': 3, '   e ch o</w>': 29, '   de gen er a ted</w>': 2, '   war lo ck</w>': 2, '   u r ine</w>': 9, '   ex ten ded</w>': 9, '   o c cu p an cy</w>': 1, '   hi lls</w>': 64, '   con st ru ct</w>': 10, '   in i ti a ted</w>': 3, '   sha man</w>': 11, '   con fe der ate</w>': 4, '   po l k a</w>': 3, '   b on an z a</w>': 3, '   je ll y b ean</w>': 7, '   ru per t</w>': 4, '   car l a</w>': 34, '   tra v el er</w>': 8, '   re side</w>': 5, '   han k sha w</w>': 3, '   f lin ch</w>': 4, '   pe y o te</w>': 2, '   af fe c ting</w>': 8, '   mi gra t ory</w>': 3, ' ru g ged</w>': 1, '   ne st ing</w>': 8, '   c ran es</w>': 12, '   who op ing</w>': 9, '   c ow girl</w>': 6, '   com pro mi sed</w>': 13, '   d ra in ed</w>': 5, '   mar sh es</w>': 3, '   in v ad ed</w>': 10, '   po ll u ted</w>': 3, '   f ou l ed</w>': 5, '   bu ck sho t</w>': 1, '   ti ck les</w>': 5, '   sc ra pp er</w>': 1, '   sc ar</w>': 59, '   fe lled</w>': 1, '   a g gre ssi ve</w>': 29, '   ex a m pl es</w>': 4, ' de l or es</w>': 1, '   z on ks</w>': 1, '   ni w et</w>': 1, ' k a me</w>': 1, '   ri v al ing</w>': 1, '   c ro s st own</w>': 2, '   pro found</w>': 11, '   i de o lo g y</w>': 2, '   ex p ound</w>': 1, ' poli ti c s</w>': 2, '   pre v a i ls</w>': 4, '   ok la h om a</w>': 30, ' ri d ers</w>': 3, ' ri ding</w>': 2, '   ro de o s</w>': 2, '   un con ven tion al</w>': 2, '   pro sti tu tes</w>': 7, ' ri cking</w>': 1, '   pa in ted</w>': 32, '   ra di u m</w>': 2, '   pe ar l</w>': 71, '   bea t ni cks</w>': 1, '   hi pp i es</w>': 5, '   a we</w>': 9, '   hu g ging</w>': 6, '   si ss y</w>': 25, '   o a k le y</w>': 1, '   stu mp</w>': 5, '   fi el d st one</w>': 3, '   tru ck st op s</w>': 1, '   k er ou ac </w>': 4, '   po d ner</w>': 3, '   ho l y man</w>': 1, '   to es</w>': 31, '   lea gu ers</w>': 1, '   ear th y</w>': 5, '   hi ll bi lli es</w>': 2, ' ch ink</w>': 1, ' chi e f</w>': 10, '   stra p</w>': 13, '   can ni ba l</w>': 4, ' in du st ri al</w>': 3, '   pr in ce ton</w>': 15, '   se me st ers</w>': 3, '   wi mp s</w>': 3, '   ju li an</w>': 24, '   a ta vi sti c</w>': 1, '   sen ti men tal</w>': 28, '   di ther</w>': 2, '   ra d c li f fe</w>': 4, '   ar ti cu late</w>': 4, '   ne g li g ent</w>': 10, '   hi tch hi ke</w>': 6, '   su b t le ti es</w>': 2, '   nu an ces</w>': 2, '   a kin</w>': 1, '   ge o lo gi cal</w>': 3, '   v a st</w>': 12, '   de li ca tely</w>': 5, '   man i ac s</w>': 14, '   e m body</w>': 2, '   r h y th ms</w>': 3, '   wh oo e e</w>': 1, '   hi tch hi ked</w>': 1, '   co o l ed</w>': 4, '   o ce ans</w>': 10, '   ri d es</w>': 19, '   un li gh ted</w>': 1, '   hi gh ways</w>': 5, '   bu mm in</w>': 1, '   y on i</w>': 5, '   y u m</w>': 8, ' h y gi en e</w>': 1, '   de w</w>': 9, '   1 9 7 0</w>': 6, '   ra g ge dy</w>': 6, '   s cu z zy</w>': 3, '   c un ts</w>': 8, '   con ce i ve</w>': 11, '   di ab o li cal</w>': 3, '   ta mp er</w>': 1, '   f lo ck</w>': 12, '   ex t in ct</w>': 15, '   wa ter co l ori st</w>': 2, '   p on der</w>': 2, '   vi r g in al</w>': 6, '   ex a sp er a ted</w>': 1, '   ru di men t ary</w>': 5, '   in vo l ve men ts</w>': 1, '   he ter o se x u al</w>': 2, '   lo gi sti c s</w>': 3, '   com for ts</w>': 4, '   di ssi pa te</w>': 3, '   ba tt les</w>': 17, '   t wi ts</w>': 1, '   o d or</w>': 7, '   di v or ce es</w>': 2, '   d ou c he</w>': 8, '   au tu m n</w>': 8, '   ma g ni fi c ent</w>': 43, '   ph ra se</w>': 33, '   ma ting</w>': 4, '   wi l de st</w>': 7, '   bi r d wa tch ing</w>': 1, '   ma i ds</w>': 11, '   dea con s</w>': 1, '   gi g an ti c</w>': 8, '   lea p ing</w>': 5, '   ar ch ing</w>': 1, '   f la pp ing</w>': 4, '   st ru tting</w>': 3, '   de ars</w>': 1, '   o ver wh el m ing</w>': 12, '   for e gr ound</w>': 4, '   f ea ther y</w>': 1, '   sle ev es</w>': 5, '   tri mm ed</w>': 4, '   su b du ed</w>': 7, '   i mi ta tion</w>': 12, ' wal ks</w>': 1, ' ca mer a</w>': 3, '   de bu ss y</w>': 1, '   sen su ous</w>': 2, '   cour t ship</w>': 3, '   app la u d</w>': 4, '   g ran di ose</w>': 2, '   l y ri cal</w>': 2, '   sc out</w>': 41, '   ori en ted</w>': 4, '   wal t</w>': 37, '   dis ne y</w>': 11, '   wi l d life</w>': 8, '   c in e ma to gra ph ers</w>': 1, '   p ho to gra ph y</w>': 8, '   ci r cu i t ous</w>': 1, '   de o d or ant</w>': 5, '   sp ra ys</w>': 2, '   rea c tions</w>': 9, '   b list ers</w>': 2, '   ra sh es</w>': 2, '   f da</w>': 5, '   ju sti f y</w>': 10, '   re mo v al</w>': 11, '   man da t ory</w>': 7, '   war n in gs</w>': 1, ' de w</w>': 1, '   b lu sh ing</w>': 7, '   de si st</w>': 5, '   tri bu te</w>': 7, '   co op er a ted</w>': 5, '   e li min a ting</w>': 4, '   v a g in al</w>': 5, '   t ac o</w>': 9, '   ver ve</w>': 3, '   char min g ly</w>': 2, ' he e</w>': 7, '   lo a th</w>': 2, '   mu sh ro om s</w>': 4, '   ex ce ssi ve ly</w>': 1, '   ch l or in a ted</w>': 2, '   t un a</w>': 22, '   sin g l ed</w>': 2, '   sch m ee k</w>': 1, '   f rea ks</w>': 25, '   ba p ti st</w>': 8, '   mi ssi ssi pp i</w>': 23, '   u p hi ll</w>': 3, '   e ons</w>': 3, '   con ner</w>': 13, '   ju st in</w>': 26, '   co o l ant</w>': 1, '   sp ac e men</w>': 2, ' se cre t</w>': 8, '   li fe sa ver</w>': 5, '   he ar t brea k er</w>': 2, '   ba ll a st</w>': 9, '   we ir</w>': 46, '   pre p</w>': 20, '   t an ks</w>': 35, '   cou ch es</w>': 1, '   re pre s su ri ze</w>': 1, '   ev a</w>': 4, ' can ce l</w>': 1, ' be ar</w>': 8, '   star ck</w>': 19, '   sp am</w>': 3, '   ac cu mu la tor</w>': 1, ' r ou te</w>': 5, '   a p u</w>': 1, '   ni t ro g en</w>': 6, '   ba be l</w>': 2, '   bi b li cal</w>': 11, ' op ti cal</w>': 1, '   op ti cal</w>': 5, '   gra vi ta tion al</w>': 12, '   di st or tion</w>': 6, '   de sc ri b ing</w>': 10, '   h ori z on</w>': 34, '   bea c on</w>': 10, '   sc re en s</w>': 9, ' sa ve</w>': 10, '   in f er a</w>': 1, ' in f er a</w>': 1, '   ab la tive</w>': 1, ' in f er i</w>': 1, ' li ber at is</w>': 1, ' tu te m et</w>': 1, ' gra vi ty</w>': 3, '   s li d es</w>': 10, '   go li a th</w>': 3, '   cor ri ck</w>': 1, ' another</w>': 9, '   con ser ve</w>': 3, '   po i son ing</w>': 12, '   pro du ces</w>': 13, '   im pa i red</w>': 4, '   vi tal s</w>': 2, '   un re sp on si ve</w>': 3, '   sti mu l i</w>': 4, '   con ta min a ted</w>': 9, '   spe ci a li st</w>': 15, '   pe t ers</w>': 14, '   dri p</w>': 4, ' c c</w>': 3, '   fi br in o g en</w>': 1, '   per i ton e u m</w>': 1, '   ru p tu red</w>': 8, '   in tu ba te</w>': 1, '   bri e f ing</w>': 5, '   v ar i ab les</w>': 11, '   ti t ani c</w>': 16, '   pla u si ble</w>': 8, '   a er o sp ace</w>': 4, '   sa l v age</w>': 19, '   pa t ro l</w>': 44, '   be lt</w>': 59, '   do cked</w>': 3, '   b la mes</w>': 10, '   s ca pe go at</w>': 5, '   in qui ry</w>': 17, '   o pp en he im er</w>': 3, '   ea ts</w>': 45, '   mi lli ra ds</w>': 1, '   de cks</w>': 13, '   sin gu l ar i ty</w>': 5, '   co ex i st</w>': 1, '   de p lo y</w>': 11, '   u m bi li c us</w>': 1, '   b on n et</w>': 4, '   coun t down</w>': 10, '   ex pl or ing</w>': 11, '   co lon i z ing</w>': 2, '   pro pu l sion</w>': 4, '   ev al u a ting</w>': 3, '   per for man ce</w>': 78, '   me chan i c</w>': 15, '   pro ce du r es</w>': 25, '   ho ll is</w>': 26, '   fi l t ers</w>': 8, ' s can</w>': 8, '   c r y st al li z ed</w>': 3, '   wor k sta tion</w>': 1, '   con ser ving</w>': 1, '   hu ll</w>': 28, '   rea din gs</w>': 29, '   c r y st al s</w>': 8, '   den ny</w>': 6, '   lo a ding</w>': 18, '   sc ru b b ers</w>': 2, '   no o o</w>': 14, '   fa th om</w>': 5, '   t ac </w>': 3, '   le e way</w>': 4, '   ab or t</w>': 18, '   un ac ce p ta ble</w>': 7, ' other</w>': 18, '   re ver ber a tions</w>': 1, ' ex per t</w>': 3, '   st ab i li ze</w>': 14, '   con st ru c tive</w>': 5, '   2 0 4 1 </w>': 1, '   2 0 3 4 </w>': 1, ' in s an e</w>': 1, '   a st r on au ts</w>': 6, '   di st or ting</w>': 1, ' gra vi ta tion al</w>': 2, '   for e de cks</w>': 4, '   se par a ting</w>': 2, '   ar ea s</w>': 16, '   li fe bo at</w>': 15, ' do g ging</w>': 1, '   com m</w>': 7, '   ac ce p ta ble</w>': 11, '   lo a d</w>': 112, '   an ten na e</w>': 3, '   c lu ster</w>': 7, '   n s a</w>': 17, '   en c r y p tion</w>': 4, '   de ci ph er ed</w>': 1, '   ma i d en</w>': 25, '   i on</w>': 6, '   th ru st ers</w>': 25, ' a head</w>': 3, '   s co p es</w>': 1, '   en han c ed</w>': 9, '   ne p t un e</w>': 8, '   me ss ing</w>': 25, '   fr ac ture</w>': 5, ' pre s su ri ze</w>': 1, '   li t ers</w>': 3, '   we ld</w>': 3, '   t ac s</w>': 2, '   f l y by</w>': 2, '   8 0 0</w>': 18, '   i on o sp here</w>': 2, ' re pe at</w>': 1, '   ac ti v a ted</w>': 15, '   c r ou ch ed</w>': 2, '   con c lu sion</w>': 26, '   supp or ts</w>': 8, '   rea c ting</w>': 4, '   sur ge</w>': 20, '   cl ar ke</w>': 1, '   de f en si ve</w>': 35, '   im m un e</w>': 15, ' bi o</w>': 3, ' rea din gs</w>': 3, '   in de ter min ate</w>': 5, ' r na</w>': 1, '   fi l ter</w>': 12, '   re con figu re</w>': 2, '   a my la se</w>': 1, '   pro te ins</w>': 5, '   sen sor s</w>': 24, '   l ev el s</w>': 33, '   ar ra y</w>': 9, '   hi gh ga in</w>': 1, '   op ti mu m</w>': 5, ' min us</w>': 2, '   mar k er</w>': 18, '   ar ter i al</w>': 4, '   cla i re</w>': 89, '   ti t an</w>': 4, '   in c in er a ted</w>': 3, '   cap ta ins</w>': 9, '   gra v </w>': 1, '   l ab s</w>': 9, '   h y dro p on i c s</w>': 1, '   re pre sen ts</w>': 27, ' point</w>': 15, '   sh or te st</w>': 5, '   de com pre ssion</w>': 4, '   tur bu l ence</w>': 5, '   1 0 0 0</w>': 11, '   3 0 0 0</w>': 6, '   sc re en ing</w>': 13, '   ha i ling</w>': 7, '   re la ti vi ty</w>': 8, '   pro hi bi ts</w>': 1, ' li ght</w>': 8, '   fo cu ses</w>': 1, '   st as is</w>': 3, '   ani ma tion</w>': 5, '   rea ctor</w>': 22, '   la y man</w>': 4, '   ro ta ting</w>': 4, '   gra vi t ons</w>': 1, '   f old</w>': 11, '   we y l</w>': 1, '   ten s or</w>': 1, '   cu r v a ture</w>': 1, '   di men si on al</w>': 4, '   in stan t an e ou s ly</w>': 3, ' ju mp</w>': 2, '   du st y</w>': 16, '   re ed</w>': 101, ' sen su al</w>': 2, ' hard</w>': 6, '   per son al s</w>': 4, '   an al y z ing</w>': 5, '   joh n ny</w>': 293, '   ho th ead</w>': 5, '   ei gh ti es</w>': 9, '   we i gh ts</w>': 7, '   z app ed</w>': 5, '   vi c</w>': 25, '   re s our ces</w>': 21, '   o ver think</w>': 2, '   c run ch ing</w>': 6, '   re work</w>': 2, '   ch em</w>': 4, '   1 0 1 </w>': 10, '   su per co o l</w>': 1, '   lon ge</w>': 3, '   c li p bo ard</w>': 4, '   st re tch ed</w>': 11, '   i so late</w>': 4, '   po si tion al</w>': 2, '   gen om es</w>': 2, '   chi se l</w>': 7, '   sh ri mp</w>': 20, ' ba th</w>': 2, '   ar ran ging</w>': 10, '   ma tu red</w>': 4, '   si mu la tor</w>': 9, '   you th ful</w>': 5, '   win g nu t</w>': 2, '   na s a</w>': 19, '   wan n ab es</w>': 5, '   s r b s</w>': 2, '   shu tt les</w>': 7, '   fin an c ed</w>': 8, '   brea k thr ou gh s</w>': 3, ' fo od</w>': 7, '   st ri p</w>': 45, ' ma ll</w>': 3, '   c r ac ks</w>': 12, '   v on</w>': 50, '   do om</w>': 15, '   pro to t y pe</w>': 17, '   sur ge ons</w>': 10, '   wan der</w>': 27, '   t in y</w>': 79, '   po sta ge</w>': 3, '   sta mp</w>': 29, '   da w g</w>': 5, '   re k in dle</w>': 2, '   s mes</w>': 2, '   h y dro gen ba se</w>': 2, '   pro pe ll ant</w>': 2, '   s mb s</w>': 2, '   de gen er a tive</w>': 2, '   c d c</w>': 4, ' comp le te</w>': 2, ' me tal li c</w>': 2, '   st r on ger</w>': 68, '   bi op h y si cal</w>': 2, ' 0 1 </w>': 2, '   f re sh en</w>': 6, '   f ever i sh</w>': 4, '   si ck est</w>': 3, '   al ps</w>': 5, '   mu sc le</w>': 38, '   ten don</w>': 2, '   man i pu late</w>': 7, '   ma ll ea bi li ty</w>': 2, '   re di st ri bu te</w>': 2, '   den si ty</w>': 9, '   f la m ing</w>': 6, '   ho tter</w>': 18, '   de lu x e</w>': 4, '   ta ll er</w>': 20, '   ma s co t</w>': 7, '   g ri m m</w>': 2, '   gen u ine</w>': 33, '   or de</w>': 2, '   fun dam en ta lly</w>': 7, '   al ter ed</w>': 28, '   re sp on si ve</w>': 4, '   fa d</w>': 6, '   4 0 0 0</w>': 4, '   cra mp ing</w>': 3, ' f an ta sti c</w>': 3, '   sy n the ti c s</w>': 2, '   a da p ting</w>': 3, '   ca e s ar</w>': 97, ' hu man i z es</w>': 2, '   tw en ti es</w>': 12, '   le on ard</w>': 47, '   o ver su b sc ri bed</w>': 2, '   sh ow time</w>': 8, '   ga s k et</w>': 12, '   f ru stra tion</w>': 8, '   de f ea ted</w>': 14, '   bri bed</w>': 10, '   pro je c tion i st</w>': 6, '   bi o e th i c s</w>': 2, '   ba x ter</w>': 79, '   ra tt ling</w>': 7, '   v ar i able</w>': 9, '   cu r ing</w>': 5, '   coun t less</w>': 11, '   mu ta tions</w>': 2, ' c rea te</w>': 4, '   po l ar i ty</w>': 4, '   par ti c les</w>': 20, '   i ons</w>': 4, '   e le men tal</w>': 2, '   si mp le st</w>': 6, '   er go</w>': 17, '   la t ent</w>': 6, '   du p li ca te</w>': 13, '   mi c ro s co pe</w>': 7, ' in vi si bi li ty</w>': 2, '   b en ding</w>': 11, '   ma ll ea ble</w>': 2, '   pro je c ted</w>': 5, ' re gu la ting</w>': 2, '   bi o gen e ti c s</w>': 2, '   u h m</w>': 15, '   bi o te ch</w>': 2, '   mi c ro s co p es</w>': 2, '   e le c t ro ph o</w>': 2, '   win ds</w>': 22, '   f l ar ing</w>': 4, '   f ac t or ed</w>': 2, ' pro du c ed</w>': 2, '   pa ten ts</w>': 4, '   bi lli ons</w>': 15, '   ca ta st ro p he</w>': 9, '   sc ra p es</w>': 5, '   re el</w>': 19, '   dea d ly</w>': 26, '   op ti mi st</w>': 5, '   d ra g ging</w>': 31, '   an al y z ed</w>': 12, '   con do s</w>': 5, '   un afraid</w>': 2, '   ac comp li sh men ts</w>': 8, '   s ke p ti cal</w>': 10, '   bra d</w>': 130, '   m c d on a ld</w>': 10, '   d ow n hi ll</w>': 7, '   spi co l i</w>': 8, '   ki d d</w>': 3, ' 7 5 </w>': 5, ' gu ar an te e</w>': 2, '   lin da</w>': 93, '   s w en son</w>': 5, '   r on</w>': 14, '   c ru i sing</w>': 8, ' y o</w>': 17, '   pro f ani ty</w>': 8, '   sur f ers</w>': 6, '   cu r t is</w>': 19, '   bu t th o le</w>': 2, '   ki ss er</w>': 6, '   tr un ks</w>': 8, '   an nu al s</w>': 1, '   ra t ner</w>': 2, '   wi s k</w>': 1, '   j ac u z z i</w>': 7, '   dam one</w>': 22, '   vi be</w>': 18, '   v in c ent</w>': 64, '   g ough</w>': 2, ' a w</w>': 4, '   l ou d m ou th</w>': 4, ' a m one</w>': 1, '   s ea fo od</w>': 10, '   w u ss</w>': 6, '   ob tain</w>': 14, '   a er o s mi th</w>': 1, '   di c ta tes</w>': 6, '   p ra ys</w>': 2, '   ta pp in</w>': 1, '   c ru e le st</w>': 4, '   co o le st</w>': 4, '   fo tom at</w>': 1, '   di a ph ra g m</w>': 4, '   re gu l ar ly</w>': 9, '   nor ne l</w>': 1, '   f ra z i er</w>': 2, '   ju dy</w>': 48, '   h in ton</w>': 1, '   mer v </w>': 7, '   sur f ing</w>': 9, '   c li ff s</w>': 7, '   re pa ir man</w>': 5, '   mu stan gs</w>': 1, '   st e er ing</w>': 22, '   tur no ff</w>': 3, '   jo cks</w>': 4, '   su z an n e</w>': 6, '   s om ers</w>': 1, '   d ou g</w>': 71, '   sta ll wor th</w>': 1, '   dar t m ou th</w>': 10, '   ri d ge mon t</w>': 3, '   st er e o</w>': 18, ' bra in</w>': 6, '   ri car do</w>': 1, '   mon tal b an</w>': 1, ' ti ps</w>': 2, '   vi r g ins</w>': 20, ' o st</w>': 3, '   c li ma x</w>': 10, '   sa ti s f ac t ory</w>': 6, '   c li ma x es</w>': 1, '   er o gen ous</w>': 4, '   z on es</w>': 6, '   qui z z es</w>': 1, '   sa le s man</w>': 50, '   chri st ma sy</w>': 1, '   b en at ar</w>': 5, '   ex a g ger ate</w>': 9, '   fi an c</w>': 20, '   w u ssi es</w>': 1, '   st ace</w>': 1, ' r on</w>': 1, '   a lo h a</w>': 6, '   s qu ea k</w>': 2, '   f l un k</w>': 3, '   s qu ar ing</w>': 2, '   as so ci a ted</w>': 15, '   t rea ti es</w>': 7, '   cont in u ou s ly</w>': 3, '   sha me le ss ly</w>': 1, '   tru an cy</w>': 1, '   att en dan ce</w>': 9, '   cu er</w>': 1, ' v o</w>': 1, '   f i</w>': 11, ' ine</w>': 1, '   co lu m bi an</w>': 1, '   bl end</w>': 17, '   b on ha m</w>': 4, '   z e pp el in</w>': 3, '   re cor ded</w>': 22, '   de e g an</w>': 2, '   sc ra p bo o ks</w>': 2, '   co kes</w>': 10, '   ink</w>': 27, ' ph one</w>': 8, ' nu mber</w>': 8, '   j ac ke ts</w>': 10, '   pro te c ted</w>': 34, '   la u dan u m</w>': 4, ' call</w>': 14, '   a mer i can a</w>': 1, '   1 6 0 0</w>': 3, '   bl in king</w>': 2, '   fif th s</w>': 1, '   b ac car d i</w>': 1, ' se ction</w>': 3, ' crazy</w>': 13, '   ri gh to</w>': 4, '   sc ra mb l ed</w>': 14, '   we st er n</w>': 58, '   re ver sed</w>': 7, '   he em</w>': 2, '   gon z o</w>': 2, '   ra ou l</w>': 34, '   fi ction</w>': 28, '   fi ends</w>': 5, '   m at</w>': 9, '   ro am</w>': 7, '   p in ea l</w>': 3, '   g land</w>': 5, '   b ow i e</w>': 3, '   j un ki e</w>': 23, '   te ar ing</w>': 28, '   i g no to</w>': 1, '   g ran d da u gh ter</w>': 6, '   sc or pi o</w>': 2, '   po go</w>': 4, '   be ver ly</w>': 53, '   din g b at</w>': 2, '   pre ju di c ed</w>': 7, '   gu ff</w>': 3, '   con ver ti b les</w>': 3, '   t ou ri sts</w>': 19, '   tur gi d</w>': 1, '   mer in gu e</w>': 4, '   f m</w>': 1, '   po sing</w>': 7, '   s we lling</w>': 5, ' gr ow</w>': 3, '   cla w s</w>': 27, '   war ts</w>': 3, '   ga in</w>': 48, '   en c y clo pe di a</w>': 3, '   han d ful</w>': 16, '   me s ca l ine</w>': 6, '   me the dr ine</w>': 1, '   sa t ani s m</w>': 1, '   oun ce</w>': 13, '   ad r en o ch ro me</w>': 3, '   s lu mp ing</w>': 1, '   jo lt</w>': 5, '   s na pped</w>': 19, '   st om p ed</w>': 2, ' be er</w>': 2, '   sh ow down</w>': 3, '   le p ers</w>': 2, '   b ack stra ps</w>': 1, '   ru ms</w>': 1, '   li z ar ds</w>': 4, '   lu g ers</w>': 1, '   wor shi pp ers</w>': 5, '   ad di c ts</w>': 8, ' dea ling</w>': 3, '   ri der</w>': 12, '   sa l m on</w>': 11, '   ca b bi e</w>': 12, '   bar br a</w>': 1, '   con sen tu al</w>': 1, '   so d om y</w>': 2, '   t ow er ing</w>': 2, ' ba sed</w>': 6, '   ha zy</w>': 4, '   re co ll e ction</w>': 7, '   se du c ed</w>': 8, '   c ru el</w>': 57, '   sa v a ge ly</w>': 2, '   ori fi ce</w>': 4, '   un ci r cu m ci sed</w>': 1, '   su b mi ssion</w>': 5, '   mo te ls</w>': 8, '   pe d dle</w>': 10, '   f la min go</w>': 1, '   te le gra ms</w>': 10, '   te sti f ying</w>': 12, '   dis bar red</w>': 14, '   s li c ing</w>': 2, '   hon k y</w>': 5, '   hea ves</w>': 1, '   ra tes</w>': 10, '   cu shi on</w>': 6, '   mor tal</w>': 32, '   sa ps</w>': 7, '   je ffer son</w>': 23, ' whi te</w>': 12, '   th or a z ine</w>': 3, '   li mes</w>': 2, '   hon ki es</w>': 1, '   gr ou pi e</w>': 4, '   so d om i z ed</w>': 2, '   po l ar</w>': 10, '   v or te x</w>': 8, '   z ee p</w>': 1, '   br in ks</w>': 1, '   ro t ar i an</w>': 2, '   sho ves</w>': 7, '   mu mb ling</w>': 4, '   cap su le</w>': 4, '   af r o</w>': 3, '   wi g</w>': 17, '   cre e p ing</w>': 6, '   mu s ca te l</w>': 1, ' ni ck el</w>': 1, '   s lo ts</w>': 12, '   ho t do gs</w>': 1, '   mar l in</w>': 5, '   re p ti les</w>': 7, '   l ac er da</w>': 3, '   sa v age</w>': 17, '   el ev at ors</w>': 9, '   re p ti le</w>': 4, '   li br es</w>': 1, '   me s cal</w>': 3, '   tw el f th</w>': 18, '   ph on y</w>': 59, '   v u l tur es</w>': 6, '   n ar co ti c s</w>': 20, '   s an d ba g ging</w>': 1, '   j our na li s m</w>': 19, '   c ro a k</w>': 4, '   s ca g</w>': 3, '   sh y ster</w>': 4, '   cl ar i ty</w>': 6, '   de spi te</w>': 32, '   han di ca p</w>': 9, '   cho ck</w>': 4, '   sa mo ans</w>': 1, '   he li ow att s</w>': 1, ' ac ti v a ted</w>': 1, '   on coming</w>': 4, '   h or se p ower</w>': 4, '   ac a pu l c o</w>': 2, '   out back</w>': 1, '   le e ch es</w>': 10, '   ba ts</w>': 19, '   pa ss word</w>': 8, '   sta ted</w>': 11, '   g ran ge</w>': 14, '   k ar ma</w>': 8, '   w r on ged</w>': 4, '   bi ting</w>': 12, '   ch ea t in</w>': 2, ' 3 8 </w>': 15, '   re fi lls</w>': 1, '   s ca tt er er</w>': 1, '   ma im ed</w>': 4, '   ke g</w>': 3, '   di ck hea ds</w>': 2, '   s ca tter</w>': 3, '   brea th s</w>': 11, ' b loo ded</w>': 14, '   ch u ck t ow</w>': 1, '   fa ze</w>': 2, '   co ll e c ted</w>': 16, '   s me ar ed</w>': 9, '   bo tt l en e ck</w>': 1, '   spi ts</w>': 3, '   b ack side</w>': 3, '   di tch ed</w>': 10, '   g ran d pa p a</w>': 1, '   ob ser v a tions</w>': 7, '   un wi se</w>': 3, '   we tter</w>': 3, '   ger on im o</w>': 9, '   h er o i c</w>': 17, '   sp u ds</w>': 1, '   ma k en z i e</w>': 1, '   ser a</w>': 17, ' ser a</w>': 1, '   wi g g le</w>': 5, '   app ra i sing</w>': 2, '   s wee ty</w>': 5, ' h mm m</w>': 8, '   co t</w>': 9, '   ca l ms</w>': 5, '   po tion</w>': 4, '   d ab s</w>': 1, '   d ab ble</w>': 7, '   f re qu en cy</w>': 23, '   o a k</w>': 26, '   plan ks</w>': 2, '   re in for c ed</w>': 4, '   bea sts</w>': 13, '   hu b by</w>': 4, '   c rea tur es</w>': 55, '   vo la ti le</w>': 7, '   pa tr ons</w>': 4, '   k le en ex </w>': 3, '   co co on</w>': 6, '   ch un k y</w>': 1, ' un s</w>': 2, '   stu n</w>': 10, '   du ran t</w>': 1, '   i g a</w>': 1, '   he ll ll ll ll p</w>': 1, '   so s</w>': 6, '   ta v </w>': 1, '   hi ck ey</w>': 1, ' e h</w>': 18, '   he ll l pp p</w>': 1, '   pa po ose</w>': 1, '   wh ee l cha ir</w>': 27, '   b on sa i</w>': 1, '   en sure</w>': 9, '   ex p lo d es</w>': 8, '   g ro cer y</w>': 24, '   ro le</w>': 44, '   mi gra te</w>': 4, ' dam m it</w>': 4, '   st r on ge st</w>': 16, '   co dy</w>': 5, '   a ll er gi st</w>': 2, '   ra g we ed</w>': 1, '   sp or es</w>': 4, ' for give</w>': 4, '   co lu m ni st</w>': 3, '   l ou e ll a</w>': 8, '   fran ces</w>': 65, '   par s ons</w>': 7, '   pe pp y</w>': 1, '   re por t ers</w>': 37, '   be be</w>': 8, '   bu ck le</w>': 10, '   for d</w>': 58, '   in sta lls</w>': 2, ' li gh ts</w>': 1, ' ran ge</w>': 8, '   go l d w y n</w>': 3, '   cla ssi cal</w>': 14, '   gar b o</w>': 2, '   co ok bo ok</w>': 2, ' ac ts</w>': 1, '   di re c ts</w>': 2, '   d on a ting</w>': 1, '   per spi re</w>': 1, '   ja lo p y</w>': 4, '   ca d</w>': 1, ' sha p en</w>': 1, '   pi ti ful</w>': 18, '   gre ed</w>': 25, '   l or na</w>': 3, '   re dre ss ing</w>': 1, '   w r on gs</w>': 5, '   i d ly</w>': 1, '   fa s ci st</w>': 13, '   ob li ter ate</w>': 3, '   de mo c r ac y</w>': 13, '   fa s ci s m</w>': 4, '   je op ar di z ing</w>': 4, '   ci vi li z ed</w>': 23, '   vi tal</w>': 23, '   in du l g ence</w>': 5, '   t ar dy</w>': 2, '   o de ts</w>': 2, '   ro les</w>': 5, '   c lu r man</w>': 1, '   e m bo di ment</w>': 3, '   i deal</w>': 20, '   en for ce</w>': 4, '   ja ps</w>': 5, '   hea d li gh ts</w>': 6, ' di m out</w>': 1, '   pre mi ere</w>': 11, '   d wa y n e</w>': 38, '   re ser p ine</w>': 1, '   sa il right</w>': 1, '   d well</w>': 6, '   re sen t men ts</w>': 1, '   run around</w>': 1, '   in di ca t ors</w>': 1, '   pro fi ted</w>': 1, '   re cu per a ted</w>': 1, '   co ll e c ting</w>': 19, '   su b sc ri p tions</w>': 4, '   su b sc ri p tion</w>': 11, ' vo ice</w>': 3, '   stan i s la v s k i</w>': 3, '   ho st ing</w>': 5, '   ro o se v el t</w>': 11, '   fran ci e</w>': 9, '   so ci a li st</w>': 7, '   re pre sen ta tion</w>': 19, '   k a min s k i</w>': 8, '   com m uni st</w>': 43, '   li l</w>': 15, '   pu get</w>': 5, '   shi f t less</w>': 2, '   in k h or n</w>': 1, '   pa tri o ti s m</w>': 8, '   an ar chi sts</w>': 3, '   com m uni sts</w>': 18, '   in di gen ts</w>': 1, ' ear ned</w>': 3, '   a ma z es</w>': 9, '   he ar ty</w>': 4, '   ar gu men ts</w>': 15, '   tra pp ers</w>': 7, '   a v g</w>': 1, '   v a g ran t</w>': 1, '   v a g ab on d</w>': 2, '   a m bi tions</w>': 6, '   f ac e less</w>': 4, '   sin ner</w>': 6, ' en n h</w>': 1, '   gr ab b ing</w>': 9, '   st ee le</w>': 2, ' t ou ch</w>': 5, '   sti ck around</w>': 1, '   th or ou gh b re ds</w>': 2, ' h or ses</w>': 2, ' h or se</w>': 9, '   me an ne ss</w>': 4, '   fu ll a</w>': 12, '   w o od</w>': 86, '   mi sa pp re h en sion</w>': 1, '   wh att a ya</w>': 15, ' take</w>': 46, '   s che mes</w>': 6, '   i da h o</w>': 6, '   s wa ying</w>': 2, ' dam ned</w>': 5, '   b re e ze</w>': 33, '   bu z z ing</w>': 25, '   r oun ds</w>': 34, '   b ou ts</w>': 1, '   pla tter</w>': 7, '   ma g ne ti s m</w>': 3, '   cha u f fe u r</w>': 12, '   g li tt er ing</w>': 2, '   ra im en ts</w>': 1, '   dis so l ving</w>': 4, '   c in der e ll a</w>': 8, '   a wai ts</w>': 7, '   car p et</w>': 35, '   ch ev y</w>': 12, '   hu ff ed</w>': 3, '   mu sc le man</w>': 2, '   li fe gu ard</w>': 5, ' wi g</w>': 1, ' gen t le man</w>': 1, '   si de wal k</w>': 15, '   ven ding</w>': 4, '   in ven tor</w>': 6, '   tr ans cont in en tal</w>': 1, '   ra il ro a ds</w>': 7, '   wor k sho p</w>': 17, '   en h</w>': 1, '   s mo o th i e</w>': 1, '   ad d in</w>': 1, '   p in k o s</w>': 1, '   mar ton i</w>': 1, '   ne w sh ound</w>': 2, '   ne w sh oun ds</w>': 2, '   ca u s in</w>': 1, '   mi g ran t</w>': 1, '   f la g ran t</w>': 1, '   dis re g ard</w>': 11, '   un war ran ted</w>': 3, '   pla in ti ff</w>': 11, ' ab i ding</w>': 3, '   ci ta tion</w>': 3, '   pre par in</w>': 1, '   ten ant</w>': 8, '   cha mp</w>': 22, '   under se a</w>': 1, '   in cen di ary</w>': 3, '   ar son i st</w>': 7, ' go l d en</w>': 1, '   se du c</w>': 1, '   re sen ted</w>': 5, '   su rest</w>': 5, '   z ei ss</w>': 1, '   un h and</w>': 2, '   a me li a</w>': 1, '   ear har t</w>': 1, '   hi lli er</w>': 1, '   au di t ori u m</w>': 7, '   ad mi ra tion</w>': 6, '   se da te</w>': 5, '   sy min g ton</w>': 3, '   bu mb le</w>': 5, '   fo l d ers</w>': 4, '   pen ci ls</w>': 7, ' damn</w>': 19, '   el u si ve</w>': 4, '   con se qu en tly</w>': 3, '   su ll en</w>': 2, '   un com m uni ca tive</w>': 1, '   ta m per ing</w>': 13, ' ma ma</w>': 5, '   pro spe ct</w>': 21, ' y min g ton</w>': 1, '   bea ds</w>': 9, '   wee p ing</w>': 11, '   re course</w>': 3, '   ve ter n ar i an</w>': 1, '   ne u ro ses</w>': 2, '   di sa bi li ti es</w>': 4, ' w er en</w>': 1, ' han k</w>': 3, '   sha p ing</w>': 6, ' in cu r able</w>': 1, '   t rea t men ts</w>': 7, '   p y re</w>': 1, '   lon ged</w>': 3, '   see king</w>': 22, '   s cu r ry</w>': 3, '   sha d ows</w>': 22, '   su i tor</w>': 6, '   ex t re mi ti es</w>': 2, '   fu r the st</w>': 4, '   rea ch es</w>': 21, '   de mon i ca lly</w>': 1, '   in du l ge</w>': 7, '   fran k en st e in</w>': 32, '   ta un t</w>': 2, '   be ck on ing</w>': 1, '   ma ter i al s</w>': 19, '   com pri sed</w>': 1, '   m ould</w>': 1, '   so li c it</w>': 1, '   in go l sta d t</w>': 2, '   cho l er a</w>': 5, '   ha m bur g</w>': 2, '   pro fe s sor s</w>': 8, ' p om p ous</w>': 2, '   beau ti fu lly</w>': 19, '   t wi r l</w>': 2, '   sc an da l ous</w>': 3, '   in di g n ant</w>': 2, '   cla u de</w>': 63, '   lo ck et</w>': 5, '   dis se ction</w>': 7, '   op in i on a ted</w>': 2, '   ba ll room</w>': 13, '   mi s gu i ded</w>': 4, '   re so l ved</w>': 5, '   so li tu de</w>': 8, '   car ca ss</w>': 4, '   g lo a ting</w>': 2, '   cha ll en ging</w>': 9, '   in ce stu ous</w>': 4, '   pe sti l ence</w>': 8, '   pu r ge</w>': 5, '   pr ou der</w>': 2, '   han d some st</w>': 2, '   dri ve l</w>': 2, '   com for ted</w>': 7, '   di z z ying</w>': 3, '   ch er i sh</w>': 8, '   ga ther</w>': 33, '   for e see</w>': 2, '   li fe times</w>': 6, '   di ving</w>': 12, '   j ea l ou s ly</w>': 3, ' gu ar ded</w>': 1, '   er a di ca ted</w>': 1, '   v ac c ine</w>': 9, '   p it</w>': 34, '   char ne l</w>': 1, ' ou ts</w>': 6, '   f end</w>': 5, '   mo der ation</w>': 10, '   cl er v al</w>': 3, '   be d side</w>': 7, '   k re mp e</w>': 4, '   nu r sing</w>': 14, '   ex ha u sti on</w>': 4, '   ar ri ving</w>': 20, '   out brea k</w>': 6, '   wal d man</w>': 6, '   pre o c cu pi ed</w>': 8, '   b la sp he my</w>': 7, '   re s ent</w>': 23, '   und rea m t</w>': 1, '   s ma ll po x</w>': 1, '   e li min a ted</w>': 4, '   j en ner</w>': 3, '   pi on e er ed</w>': 2, '   ma g n us</w>': 2, '   a g ri pp a</w>': 2, '   to ad st oo ls</w>': 1, '   cle ar ing</w>': 16, '   co ve ted</w>': 1, '   mi s di re c ted</w>': 1, '   mor i t z</w>': 1, '   my sti ci s m</w>': 2, '   al ber tu s</w>': 1, '   par ac el sus</w>': 2, '   pu r su it</w>': 18, '   do g ma ti c</w>': 2, '   pre ce pt</w>': 3, '   dis ci pl es</w>': 8, '   ha ll ow ed</w>': 4, '   se ct</w>': 2, '   sc ri p ture</w>': 8, '   ver se</w>': 18, '   ev i dent</w>': 3, ' con figu re</w>': 2, '   al ou d</w>': 5, '   as se mb l ed</w>': 6, '   f ac u l ty</w>': 6, '   d ra ft</w>': 33, '   th ri lling</w>': 9, '   rea l m</w>': 10, '   re ck less</w>': 22, '   as se ss</w>': 8, '   com b in ing</w>': 2, '   gh ou li sh</w>': 2, '   cla ss room</w>': 12, '   can ton</w>': 1, '   mi ssi on ar i es</w>': 2, '   fa sc in a ted</w>': 18, '   di sc ar ded</w>': 6, '   st ran g l ed</w>': 14, '   c ling</w>': 6, '   pro vo king</w>': 4, '   pu r su i ts</w>': 2, '   ca pri ci ous</w>': 2, '   ma d ne ss</w>': 23, '   ac hi ev e men ts</w>': 5, '   per si st</w>': 8, '   be d ev il ed</w>': 1, '   pla gu ed</w>': 4, '   di f fi cu l ti es</w>': 11, '   star t ling</w>': 1, '   ar chan ge l</w>': 4, '   g an gr en e</w>': 2, '   bro th</w>': 4, '   chri st y</w>': 22, '   mar ci e</w>': 13, '   ned</w>': 62, '   me l v in</w>': 18, '   be ll i</w>': 1, '   g ru b by</w>': 3, '   bo a thou se</w>': 1, '   can o es</w>': 3, '   c ro ss ro a ds</w>': 10, '   j ee p</w>': 33, '   so f t b all</w>': 5, '   dr en ch ed</w>': 5, '   g lo om y</w>': 3, '   fi l m ma k er</w>': 5, '   sp oo ks</w>': 10, '   ca mp ers</w>': 3, '   coun se l ors</w>': 6, '   cla u de tt e</w>': 4, '   bar ry</w>': 57, '   wi ly</w>': 2, '   ori en tal</w>': 7, '   ha th</w>': 14, '   w r ought</w>': 4, '   b ows</w>': 7, '   ca mp fir es</w>': 1, '   su b stan ces</w>': 2, '   ca mp gr oun ds</w>': 2, '   dis mi s sa l</w>': 5, '   par a gra p h</w>': 4, '   wh as sa ma tta</w>': 1, '   u m hu mm m mp h</w>': 1, '   mm mm mm mp h</w>': 2, '   bri ck</w>': 40, '   st or ms</w>': 9, '   mo ther ly</w>': 5, '   le c tur es</w>': 8, '   ne ddy</w>': 3, '   shi f ted</w>': 10, ' min ded</w>': 19, '   bro ch u re</w>': 5, ' ca mp</w>': 3, '   sc ar let</w>': 5, '   coun se ll or</w>': 4, '   bab es</w>': 16, ' less</w>': 8, '   dr ow n ded</w>': 1, '   sta ti sti c s</w>': 9, '   h om i ci d es</w>': 5, '   bu ll fi ght</w>': 2, '   v a man o s</w>': 1, '   el</w>': 88, '   se th</w>': 13, '   p sy cho s</w>': 8, '   v a m pi r es</w>': 55, '   su n li ght</w>': 21, '   r ow dy</w>': 6, '   du s k</w>': 3, '   for sa k en</w>': 5, '   what sa matter</w>': 9, '   wi s ea c r es</w>': 1, '   pa so</w>': 24, '   no ma d</w>': 3, '   wai f s</w>': 1, '   tru ck ers</w>': 5, '   lon g time</w>': 1, '   shi p men ts</w>': 2, '   god less</w>': 3, '   ta i wa n</w>': 6, '   ho lin ess</w>': 21, '   c ru ci fi x</w>': 8, '   win ne ba go</w>': 9, '   me l on</w>': 4, '   s mi l in</w>': 2, '   st e w ing</w>': 2, '   a pe</w>': 42, '   s co tty</w>': 60, '   te mp le</w>': 31, '   t ou ch es</w>': 28, '   ri chi e</w>': 16, '   s co o t ers</w>': 1, '   re ce i ving</w>': 20, '   pr ying</w>': 11, '   m c co y</w>': 40, '   ir s</w>': 19, '   wh e w w w</w>': 1, '   in stan tly</w>': 16, '   bra kes</w>': 18, '   s li d</w>': 8, '   fu ll er</w>': 7, '   fa gs</w>': 9, '   f lo p</w>': 16, '   ch oo ses</w>': 9, '   r ab b i</w>': 4, '   re f le ction</w>': 14, '   la p se</w>': 8, '   a wa k en ing</w>': 4, '   pa st or</w>': 11, '   lea der ship</w>': 15, '   be the l</w>': 1, '   bi tt er est</w>': 1, '   co cking</w>': 1, '   no oo o</w>': 14, '   o y ster</w>': 8, '   cu ti e</w>': 5, '   pe ck er</w>': 8, ' di ga y o</w>': 1, '   di ga y o</w>': 1, '   t wi ster</w>': 4, '   da mo c les</w>': 1, '   so l ved</w>': 12, '   m ere</w>': 26, '   un so l v able</w>': 1, '   p an el ing</w>': 1, '   as su mp tion</w>': 9, '   bu rea u</w>': 82, '   man h un t</w>': 2, '   e ye wi t ne ss</w>': 3, '   op ti mi sti c</w>': 13, '   ab il en e</w>': 3, '   fu gi ti ves</w>': 4, '   con fi dent</w>': 31, '   app re h end</w>': 4, '   d ra g n et</w>': 1, '   s n are</w>': 2, '   ge ck o</w>': 2, '   pa y back</w>': 4, '   com mo de</w>': 5, '   g rea se</w>': 15, '   na d ine</w>': 3, '   po ta to</w>': 24, '   mon go lo id</w>': 5, '   re t ar ds</w>': 13, '   a w w w</w>': 8, '   shi ta ss</w>': 1, '   k ri spi es</w>': 2, '   pu kin</w>': 3, '   bur ri to s</w>': 2, '   hi pp i e</w>': 8, '   t an ked</w>': 3, '   b la st in</w>': 2, '   sc ra tch ing</w>': 9, ' b en ny</w>': 1, '   gr ab b in</w>': 2, '   d ya</w>': 2, '   sc ar red</w>': 8, '   sen ori ta</w>': 3, '   see in</w>': 35, '   si pp in</w>': 2, '   mar gar i ta s</w>': 2, ' b ar</w>': 4, '   li ke li ho od</w>': 3, ' s ou th</w>': 3, '   s m ac ked</w>': 3, '   nu tty</w>': 6, '   nu t ti est</w>': 1, '   spi c</w>': 8, '   m ou th ful</w>': 9, '   te x ans</w>': 2, '   fi re c r ack er</w>': 3, ' bar ter</w>': 2, '   s an c tu ary</w>': 15, '   ea ve s dro pp ing</w>': 6, '   wa ter bed</w>': 2, ' ra ted</w>': 6, '   sig n al ed</w>': 3, '   f ling</w>': 9, '   k a th y</w>': 22, '   ad den du m</w>': 1, ' sp on sor ing</w>': 2, ' wi ck wi re</w>': 3, '   dan e</w>': 4, '   li e u ten</w>': 1, '   en sig n</w>': 10, '   bl on d ell</w>': 1, '   k la tch es</w>': 1, '   u pri sing</w>': 5, '   d en</w>': 16, ' ge e</w>': 10, '   cle men te</w>': 3, '   star bo ard</w>': 14, ' se cu ri ty</w>': 3, '   j or d an</w>': 29, '   co or din a ted</w>': 3, '   vi a</w>': 8, '   ha yes</w>': 18, '   sha de</w>': 17, ' mi d night</w>': 3, '   ma ho g any</w>': 4, ' cu z</w>': 12, '   m ou th in</w>': 1, '   sen ate</w>': 61, '   s mar ts</w>': 9, '   in fr in ge</w>': 1, '   li ber ti es</w>': 4, '   to ad s qu at</w>': 1, '   hi gh way</w>': 48, '   te le p ho to</w>': 1, '   l en ses</w>': 4, '   s na pp in</w>': 1, '   un r ing</w>': 1, '   b ow el s</w>': 4, ' ne il</w>': 26, '   de li ca te</w>': 26, '   f ra ter ni z ation</w>': 1, '   man ner</w>': 58, '   un be coming</w>': 4, '   ev a por ate</w>': 2, '   so ci o lo gi st</w>': 2, '   ar ti c le</w>': 61, '   cont end</w>': 1, '   bi tch in</w>': 5, ' che sted</w>': 2, ' lea st</w>': 5, '   o b</w>': 3, ' g y n</w>': 2, '   pa p</w>': 5, '   s me ars</w>': 5, ' doll ar</w>': 16, '   p ha lli c</w>': 2, '   f ra gi le</w>': 19, '   sen si bi li ti es</w>': 2, ' f art</w>': 1, '   pe t ti co at</w>': 2, '   com pla in ing</w>': 41, ' stan d ard</w>': 4, '   de f er en ti al</w>': 1, '   mi st rea ted</w>': 2, '   comp le ting</w>': 7, '   cla ss ma te</w>': 2, '   har as sing</w>': 7, '   in te gra ting</w>': 2, '   spe c</w>': 8, ' re c on</w>': 5, '   pa in less</w>': 11, '   be ver age</w>': 12, '   ne w ber ry</w>': 4, '   mo ther ac h ri st</w>': 1, '   pa d d les</w>': 3, '   ban di to s</w>': 1, '   bl an ks</w>': 13, '   an n oun c ed</w>': 12, '   sp on sor ing</w>': 1, '   le gi s la tion</w>': 5, '   1 9 4 8 </w>': 5, '   ex c lu sion</w>': 3, '   pen ta g on</w>': 32, '   o ver pla y ed</w>': 1, ' sp ir it</w>': 2, '   de gra ding</w>': 5, '   re mar ks</w>': 16, '   a vi at ors</w>': 1, '   in nu en do</w>': 4, '   d ow ned</w>': 3, '   a vi a tor</w>': 6, '   bo d es</w>': 2, '   con fir ma tion</w>': 12, '   car ri er</w>': 14, ' 1 4 </w>': 14, '   l ar k</w>': 11, '   dis cont in u e</w>': 1, '   see min g ly</w>': 6, '   in cont ro ver ti ble</w>': 4, '   mar i time</w>': 7, '   spe ci al ti es</w>': 3, ' e u p he mi ze</w>': 1, '   na v al</w>': 8, '   in sti tu ted</w>': 1, '   cour ses</w>': 16, '   se c na v </w>': 1, '   ver i fi able</w>': 2, '   wi ck wi re</w>': 7, ' char ger</w>': 1, ' pi ss ing</w>': 1, ' world</w>': 10, '   r he t ori c</w>': 1, ' ra il</w>': 1, '   ro per</w>': 14, '   ga ll o p</w>': 2, '   ban d wa g on</w>': 2, '   no min a ted</w>': 10, '   ge ar ing</w>': 3, ' con ner</w>': 2, '   pro men a de</w>': 3, '   li tter</w>': 10, '   ex pe di te</w>': 2, '   d w y er</w>': 3, ' poli cy</w>': 1, '   be g g in</w>': 3, ' cor on a do</w>': 1, '   pen ta th le te</w>': 1, '   ro t c</w>': 2, '   sch o l ar ship</w>': 16, '   con sti tu ent</w>': 1, '   d ra w er</w>': 30, '   sto ck in gs</w>': 14, '   app ar ent</w>': 12, '   de haven</w>': 4, '   com man do</w>': 3, '   re con na i ss an ce</w>': 4, '   s ea ls</w>': 7, '   pr i</w>': 1, '   cor te z</w>': 6, '   min is</w>': 2, '   ba sh er</w>': 1, ' ba sh er</w>': 1, '   ex tr ac tion</w>': 8, '   pr c</w>': 3, '   mon t go mer y</w>': 9, '   whi tt l ed</w>': 1, ' f le a</w>': 2, '   rea li z es</w>': 13, ' ri gh ts</w>': 1, '   bar e back</w>': 3, '   tra in e e</w>': 6, ' si tter</w>': 6, '   co z a d</w>': 1, '   in ta g li at a</w>': 2, '   a y ers</w>': 1, '   sur pri ses</w>': 33, '   p y r o</w>': 3, '   u ran i u m</w>': 13, '   spe cu late</w>': 9, '   sa u d i</w>': 10, '   ar ab i a</w>': 5, ' secon d</w>': 12, '   de du ction</w>': 7, ' gen der</w>': 1, ' nor m ing</w>': 1, ' sp ea k</w>': 5, '   sta tion ed</w>': 12, '   nor fo l k</w>': 11, '   cor on a do</w>': 4, '   di sp en sa tion</w>': 3, ' app ro pri ation</w>': 1, '   li a i son</w>': 10, ' do cked</w>': 1, '   st in ts</w>': 1, '   l ever a ged</w>': 3, '   in st ru c t ors</w>': 4, '   u r ga y le</w>': 1, '   hea d work</w>': 3, '   un ans w er ed</w>': 4, '   ro y ce</w>': 9, '   tur r en t ine</w>': 1, '   sh or ta ge</w>': 14, '   st ri king</w>': 12, '   sy m bo ls</w>': 10, '   g un ning</w>': 8, '   si z ing</w>': 1, '   wa i</w>': 4, ' brea k er</w>': 3, '   to pped</w>': 3, '   u pp er de cks</w>': 1, '   bu ll p en</w>': 1, '   si de lin es</w>': 5, ' back</w>': 35, ' h it</w>': 8, '   cap one</w>': 11, '   d pr k</w>': 1, ' 5 7 </w>': 4, '   do p ed</w>': 7, '   cont in gen cy</w>': 7, '   in fi l tra tes</w>': 1, '   ha tch es</w>': 1, '   per i s co pe</w>': 2, '   k or ean</w>': 11, '   pro pa g an di z ed</w>': 1, '   de l ta</w>': 15, '   con ven tion al</w>': 5, ' sh ore</w>': 3, '   po l k</w>': 2, '   an al y st</w>': 16, '   sp ar in g ly</w>': 1, '   con c lu de</w>': 4, '   de co y</w>': 9, '   wa tch a</w>': 8, '   vi o la tions</w>': 4, '   pa st a</w>': 18, '   m c p</w>': 12, ' gir l friend</w>': 5, '   mar l bor o</w>': 3, '   re k ki e</w>': 1, '   li b y an</w>': 7, '   co a st l ine</w>': 2, '   k ha da ff i</w>': 1, '   ni gh t time</w>': 4, '   in fi l</w>': 1, '   ma ps</w>': 23, '   ar ti ll er y</w>': 13, '   pl ac e men ts</w>': 1, '   har as s ment</w>': 16, ' t an ks</w>': 1, '   mi gh ta</w>': 12, '   ho sti les</w>': 1, '   el b</w>': 1, '   win ch</w>': 15, '   b lo cked</w>': 27, '   g lu tt on ous</w>': 1, '   ba m bo o</w>': 4, '   ca ge</w>': 64, '   wom b</w>': 7, '   cre w s</w>': 7, '   m c co o l</w>': 1, '   co he sion</w>': 2, '   hi st ori cal</w>': 16, '   bar r ack</w>': 16, '   vi sc er a</w>': 2, '   li mb s</w>': 12, '   i s ra el i</w>': 3, '   w oun ded</w>': 50, '   de tri ment</w>': 1, '   en dan ger ing</w>': 6, '   s lu t ni k</w>': 1, '   w r en ch</w>': 10, ' ev a de</w>': 1, '   loo k it</w>': 19, '   br in g in</w>': 13, '   ta m pa x</w>': 1, '   dar th</w>': 5, '   v a der</w>': 29, '   ci a</w>': 71, '   out come</w>': 6, '   cont ro ll able</w>': 2, '   han d l ed</w>': 47, '   s lo g g in</w>': 1, '   ta ps</w>': 3, '   con sti tu en cy</w>': 1, '   sp ad es</w>': 10, '   f le et</w>': 52, '   tri dent</w>': 2, ' ed</w>': 16, ' r in g ers</w>': 1, '   in te gra tion</w>': 5, ' clo sed</w>': 6, '   au dre y</w>': 26, '   s li me</w>': 17, '   ch un ks</w>': 5, '   j f k</w>': 6, '   re pa i red</w>': 9, '   cont ac ting</w>': 5, '   in tr an et</w>': 1, '   o ver lo ad ed</w>': 4, '   re pro du ces</w>': 2, '   a se x u ally</w>': 2, '   di stan ces</w>': 4, '   re pro du ction</w>': 2, '   in su f fi ci ent</w>': 5, '   pro gen i t ors</w>': 1, '   h or mon al</w>': 6, '   sa mp le</w>': 25, '   in ad ver ten tly</w>': 3, '   i dea li s m</w>': 7, ' nu ke</w>': 3, '   ac ti vi sts</w>': 1, '   ear th wor ms</w>': 3, ' bor ing</w>': 2, '   de pen d able</w>': 14, '   lu ce</w>': 4, '   ca i man</w>': 7, '   for ger y</w>': 10, '   p an a ma</w>': 10, '   ta to p ou lo s</w>': 5, '   wee k ends</w>': 25, '   hu mp h ri es</w>': 2, '   ev ac u ate</w>': 11, '   ab er ra tion</w>': 4, '   h y bri d</w>': 2, '   re gi on</w>': 21, '   ch er no b y l</w>': 6, '   el si e</w>': 4, '   cha p man</w>': 7, '   pa le on to lo gi st</w>': 8, '   a ll e z</w>': 2, '   mer de</w>': 7, '   t un ne ls</w>': 18, '   bur row</w>': 2, '   qu ar an t in ed</w>': 4, '   char ge u r es</w>': 1, '   re v ea l ed</w>': 14, '   ev a de</w>': 4, '   si gh ted</w>': 6, '   po l y ne si an</w>': 1, '   an o ma ly</w>': 6, '   f oo t pr int</w>': 7, '   re ck le ss ne ss</w>': 3, '   bi o lo gi st</w>': 4, '   ear th wor m</w>': 2, '   inter ru p ted</w>': 9, '   ni k o</w>': 1, '   to po po lo s is</w>': 1, '   di ver s</w>': 8, '   r in ging</w>': 31, '   lu re</w>': 12, '   por tion</w>': 14, '   se que st er ed</w>': 2, '   re vo ir</w>': 11, '   5 5 5 </w>': 5, ' 7 6 0 0</w>': 1, '   d ra gon f ly</w>': 1, '   b om b er</w>': 9, '   h ors</w>': 13, ' o e u v r es</w>': 8, '   ven ti la tion</w>': 5, '   le gi on</w>': 9, '   s de ce</w>': 1, '   ex ter i e u re</w>': 1, '   cont re</w>': 1, ' e spi on na ge</w>': 1, '   que ll e k</w>': 5, ' su d den ly</w>': 3, '   nor ma li z ing</w>': 1, '   ma k</w>': 3, ' ar</w>': 4, '   st ea l th</w>': 5, '   i p th ar</w>': 1, '   gr ab th ar</w>': 3, '   war v an</w>': 1, '   ma pp er</w>': 1, '   pro gra med</w>': 1, '   ber y lli u m</w>': 11, '   sp here</w>': 37, '   f ac i li ti es</w>': 15, '   ab s ent</w>': 12, '   ex tra po late</w>': 3, '   an at om y</w>': 15, '   spi kes</w>': 3, '   di st r ac tion less</w>': 1, '   brea ch ing</w>': 1, '   hu mb l ed</w>': 3, '   ex ten si ve ly</w>': 2, '   ther mi an</w>': 1, '   s l ow ed</w>': 15, '   g w en</w>': 16, '   c et</w>': 2, '   t n e</w>': 2, '   s ar r is</w>': 12, '   shi d</w>': 1, ' he x</w>': 1, '   ok a v </w>': 1, ' e go man i ac al</w>': 1, ' pu r p le</w>': 1, '   mon st ro si ty</w>': 5, '   mon st r on si ty</w>': 1, '   e go man i ac al</w>': 2, '   car af t</w>': 1, '   tr an qui l</w>': 1, '   vi bra tions</w>': 11, '   s qu ea l</w>': 5, '   ne s mi th</w>': 2, '   th ru ster</w>': 3, '   bo o sts</w>': 2, '   b lo cking</w>': 16, '   ga la x y</w>': 23, '   su n down</w>': 12, '   m oun ting</w>': 1, '   3 1 </w>': 14, '   lu di c r ous</w>': 8, ' sle ev es</w>': 1, '   bi ce ps</w>': 2, ' l ar e do</w>': 2, '   ma ke up</w>': 22, '   1 5 9 </w>': 1, ' \t </w>': 166, ' 1 5 9 </w>': 1, '   can ni b al s</w>': 6, '   for ti fi ed</w>': 3, '   min ers</w>': 8, '   re p ea ting</w>': 15, '   ro om ful</w>': 6, '   han g ers</w>': 4, '   ter min ally</w>': 1, '   on l ine</w>': 8, '   k y le</w>': 41, '   d ow n lo a ding</w>': 3, '   se c t ors</w>': 2, ' 2 8 </w>': 3, '   u ti li ty</w>': 6, '   wal k through</w>': 1, '   plea s an tri es</w>': 4, '   o ver lo a ds</w>': 2, '   b ran don</w>': 37, '   di ssed</w>': 2, ' o thin o</w>': 1, ' ust</w>': 2, '   on cu on t</w>': 1, '   w ou </w>': 5, '   o c</w>': 3, '   o om oo oo a te o</w>': 1, '   on an</w>': 1, '   on at</w>': 1, '   o an e h</w>': 1, '   ber i th i u m</w>': 1, '   la v a</w>': 6, '   han d ho l ds</w>': 1, '   c ru sh ers</w>': 2, '   inter v al s</w>': 7, '   ch om p ers</w>': 1, '   sc re en wri ter</w>': 2, '   ome ga man</w>': 1, '   re ar ran ger</w>': 1, '   con ver ting</w>': 3, '   ac ti v ation</w>': 4, '   e f fe c ting</w>': 1, '   hea ted</w>': 3, '   co ll a p s er</w>': 1, '   ome g a</w>': 8, '   mar ked</w>': 35, ' c ore</w>': 4, '   do or way</w>': 17, '   an ti matter</w>': 1, '   bra in case</w>': 1, '   di a gi tal</w>': 1, '   con ve y or</w>': 4, '   con st ru c ti ve ly</w>': 2, '   sh ar p ly</w>': 3, '   ac ci den tly</w>': 8, '   tra ded</w>': 13, '   vo x</w>': 8, '   bu mp ed</w>': 19, '   f lu x</w>': 8, '   au x i li ary</w>': 12, '   qu as ar</w>': 3, '   ga mm a</w>': 10, '   b lu e pr in ts</w>': 10, '   ma tri x</w>': 33, ' p ink</w>': 2, '   fun c tion al</w>': 9, '   fr ac tu red</w>': 4, '   ve lo ci ty</w>': 11, '   su sta in ing</w>': 2, '   st ru c tu ra l</w>': 7, '   tu ar an</w>': 1, ' hi st ori cal</w>': 2, '   6 8 </w>': 2, '   b on ding</w>': 7, '   su b stra te</w>': 1, '   mo le cu le</w>': 4, '   sh ar ing</w>': 24, '   e le c tr on</w>': 5, '   b om b ard</w>': 2, '   re f le c tive</w>': 1, '   i so to p es</w>': 1, '   co v al ent</w>': 2, '   v al ence</w>': 2, '   b on ds</w>': 27, ' la ter ally</w>': 1, '   ba th wa ter</w>': 2, '   ho ff man</w>': 5, '   con si st en cy</w>': 5, '   k wa n</w>': 5, '   g ori g na k</w>': 3, '   con ve y er</w>': 2, '   cu b es</w>': 2, '   ho lo gra m</w>': 6, '   en do ther mi c</w>': 1, '   pl u ck y</w>': 4, '   v al ve</w>': 11, ' cre w man</w>': 3, '   f le e g man</w>': 3, '   e pi so</w>': 1, '   o e tting</w>': 1, '   ha m mer ed</w>': 5, '   la the</w>': 3, '   con tru ct</w>': 1, '   tur b o</w>': 4, ' ac ti v ate</w>': 1, '   bo o st ers</w>': 2, '   o v </w>': 1, '   gr ab t n ar</w>': 1, ' mm er</w>': 1, '   o ve</w>': 1, '   on o</w>': 2, ' a le</w>': 1, ' as sa u lt</w>': 1, '   vo l t ar e ck</w>': 1, ' 8 2 </w>': 4, '   in tr o</w>': 6, '   2 7 0</w>': 4, '   d ou b ted</w>': 11, '   ac c el er a ting</w>': 4, '   cho pp y</w>': 1, '   c ru sh y</w>': 1, '   ca t wal k</w>': 2, '   ne u tr on</w>': 10, '   an ne x</w>': 9, '   li z ard</w>': 16, '   chan ting</w>': 5, '   cl en ch ed</w>': 3, '   ja w</w>': 19, '   supp or tive</w>': 19, '   ter ra ki an</w>': 1, '   a gre e ing</w>': 6, '   no d ded</w>': 4, '   ma the s ar</w>': 5, '   n in ty</w>': 1, '   ad l er</w>': 4, '   man ned</w>': 3, '   re son an ce</w>': 4, '   com mi t m n t</w>': 1, '   s mi l ed</w>': 11, '   r d</w>': 1, ' pa sa den a</w>': 1, '   the v </w>': 1, '   do in o</w>': 1, '   int</w>': 75, ' 3 7 </w>': 3, ' di gi ti ze</w>': 1, '   ch en</w>': 2, '   te b</w>': 1, '   re set</w>': 14, '   ac c el er ate</w>': 5, '   di m wi tt ed</w>': 2, '   bo lt</w>': 20, '   hea l</w>': 29, '   ok ey</w>': 9, '   do k ey</w>': 1, '   par ti c le</w>': 9, '   can n ons</w>': 11, '   g an n et</w>': 1, '   ma g ne ts</w>': 4, '   ca ta pu l ts</w>': 1, '   de ce p tion</w>': 10, '   o an</w>': 1, '   o on tal n</w>': 1, '   la u ch ter</w>': 1, ' tom my</w>': 1, '   lo m man der</w>': 1, '   pl y w o od</w>': 2, '   de cor a tions</w>': 3, '   ex pl or ation</w>': 6, '   s ni p</w>': 6, '   dea lin gs</w>': 8, ' de ce p tion</w>': 1, ' li es</w>': 2, '   vi c t ori ous</w>': 4, '   to th i an</w>': 1, '   1 2 1 8 5 </w>': 1, '   po ds</w>': 9, '   1 3 4 </w>': 4, '   sur vi ves</w>': 4, '   ther a min i</w>': 1, '   ther a m in</w>': 1, '   nu ys</w>': 3, '   ther mi ans</w>': 2, '   k la tu </w>': 2, '   sy st e ma ti ca lly</w>': 5, '   ro th</w>': 26, '   fa tu </w>': 1, ' k re y</w>': 1, '   di sa st r ous</w>': 5, '   to y o ta</w>': 4, '   b lu r</w>': 8, '   l ar e do</w>': 3, '   ar ea o</w>': 1, '   pe da l</w>': 9, ' li z ard</w>': 1, ' ca w</w>': 1, '   ca w</w>': 1, ' wouldn</w>': 8, '   n t</w>': 1, '   o ack</w>': 1, '   en er a o</w>': 1, '   ma an e t l s m</w>': 1, '   dis ru p ting</w>': 2, '   on st ru </w>': 1, ' a it</w>': 1, '   imp lo sion</w>': 2, '   re ver si ble</w>': 1, '   har d wi red</w>': 4, '   ba u ble</w>': 2, '   wa gon s</w>': 11, '   bu mp k ins</w>': 2, '   he ssi ans</w>': 3, '   in v in ci ble</w>': 5, '   b en e di ct</w>': 16, '   re co g ni z es</w>': 8, '   in qui si tion</w>': 3, '   app ea l ed</w>': 3, '   bu d ding</w>': 1, '   c ro m well</w>': 2, '   con de m n s</w>': 1, '   t ori es</w>': 2, ' app ro pri a tes</w>': 1, '   t ory</w>': 5, '   sh el t ers</w>': 7, '   s ar a to g a</w>': 5, '   po tt s</w>': 2, '   be fi tting</w>': 1, '   con gre ss men</w>': 6, '   pa mp h let</w>': 1, '   ci r cu la ted</w>': 3, '   en cou ra ged</w>': 11, '   for ge</w>': 13, '   co a ts</w>': 16, '   de sp er a tely</w>': 22, '   ph ar m ac i st</w>': 2, '   na than a el</w>': 6, '   sho er</w>': 1, '   di sp ro por tion ate</w>': 1, '   att ac king</w>': 28, '   ci cer o</w>': 2, '   ra sh</w>': 18, '   in su bor din ate</w>': 2, '   g l ori ous</w>': 22, '   s ki r ts</w>': 10, '   b al ti more</w>': 42, '   vo ted</w>': 13, '   for e sts</w>': 5, '   de f y</w>': 9, '   mo b s</w>': 2, '   i ro qu o is</w>': 1, '   wi l der ne ss</w>': 19, '   que be c</w>': 3, '   for din gs</w>': 1, '   cha mp la in</w>': 3, '   ti con der o g a</w>': 2, '   ter ri f y</w>': 6, '   wi l</w>': 5, '   mon m ou th</w>': 2, '   lo y ally</w>': 2, '   a po lo gi z ing</w>': 12, '   f l int</w>': 4, '   com pe lled</w>': 6, '   in qu ir es</w>': 1, '   ri b b ons</w>': 5, '   pi tch for k</w>': 2, '   mar qu is</w>': 41, '   st e u b en</w>': 1, '   wa y n e</w>': 83, '   supp or ting</w>': 12, '   re con so li da tion</w>': 1, '   c ow ar ds</w>': 13, '   la fa ye tt e</w>': 9, '   de le ga tion</w>': 6, '   con sti tu tion al</w>': 7, '   spe cu la t ors</w>': 1, '   cou p</w>': 22, '   de cl are</w>': 25, '   de cl ar ation</w>': 10, '   in sur re ction</w>': 5, ' mar ti al ed</w>': 1, '   de f en ses</w>': 7, '   dis re pa ir</w>': 1, '   ne g le c ted</w>': 8, '   ye ssi r</w>': 40, '   di vi si ons</w>': 8, '   v an gu ard</w>': 2, '   re ar gu ard</w>': 1, '   g ee se</w>': 14, '   mi gra ting</w>': 2, '   app r en ti ce</w>': 9, '   c ro i x</w>': 2, '   con sc ri p ts</w>': 1, '   re c ru i ts</w>': 3, '   ha le</w>': 7, '   op en ne ss</w>': 3, ' ea l</w>': 3, '   re t re at</w>': 23, '   ne go ti a ted</w>': 5, '   de mo cra ti c</w>': 17, '   dr un k en</w>': 14, '   h or a ti o</w>': 4, '   mu s ke ts</w>': 6, '   ac qu it</w>': 3, '   go o d will</w>': 7, '   l ar ge ss</w>': 1, '   r ho de</w>': 12, '   a st on i sh ing</w>': 6, ' co st</w>': 2, '   a le</w>': 11, '   ar i sto c r at</w>': 9, '   tr en ton</w>': 3, '   de pre ca ting</w>': 1, '   han co ck</w>': 6, ' e a</w>': 1, '   s mu g g l er</w>': 5, '   in e qu a li ty</w>': 1, '   s ou ther ner</w>': 4, '   st ru g g ling</w>': 23, '   cla ssi st</w>': 1, '   e li ti st</w>': 5, '   p rea ch ing</w>': 7, '   bi tt er ly</w>': 2, '   a ll ow ing</w>': 16, '   de ta ins</w>': 1, '   fe int</w>': 5, '   man si ons</w>': 3, '   gre e dy</w>': 33, '   in di sp en sa ble</w>': 4, '   pu r su ing</w>': 8, '   re sig na tion</w>': 17, '   ca d wa la der</w>': 2, '   cor n wa ll is</w>': 18, '   de i st</w>': 3, '   chri sti ans</w>': 9, '   f oun ding</w>': 2, '   co lon i z ed</w>': 1, ' chri sti an</w>': 3, '   re so lu tion</w>': 9, '   cen sur ing</w>': 1, '   s ac ri fi c ing</w>': 3, '   bar g es</w>': 2, '   su per i ori ty</w>': 7, '   spi es</w>': 16, '   al li es</w>': 18, '   pe ti te</w>': 5, '   h en ri e tt e</w>': 2, '   me l ts</w>': 5, '   b lo ss om ing</w>': 2, '   ar mor y</w>': 2, '   char l ev i ll e</w>': 1, ' char m ing</w>': 2, '   vi c t ori es</w>': 3, '   so f ten ing</w>': 1, ' p hi la de l p hi a</w>': 1, ' cont in en tal s</w>': 1, '   fe i t ers</w>': 1, '   app o in ted</w>': 18, '   y ves</w>': 1, '   ro ch</w>': 1, '   gi l ber t</w>': 9, '   mo ti er</w>': 1, '   coun tr y men</w>': 3, '   por tra it</w>': 18, '   ad ri en n e</w>': 3, '   ra p ac i ous</w>': 2, '   vi ve</w>': 3, ' a mer i qu e</w>': 1, '   d ower</w>': 1, '   inter b re e ding</w>': 1, '   di st in gu i sh</w>': 4, ' kind</w>': 11, '   ter ri fi ed</w>': 24, ' u pp er</w>': 2, '   con de sc end</w>': 3, '   pro sp er ed</w>': 1, '   to ss ing</w>': 6, '   ar m ing</w>': 4, '   o ver tur n s</w>': 1, '   ou tra ged</w>': 5, '   be tra y al</w>': 13, '   do g ma</w>': 2, '   in qui re</w>': 8, '   fa ir fa x</w>': 4, '   tu t or ing</w>': 3, '   re ci te</w>': 6, '   ca to</w>': 4, '   d ev o tion</w>': 12, '   pu ri ty</w>': 12, '   fa ir fa x es</w>': 1, ' li ber ty</w>': 1, '   ro man s</w>': 7, '   con ce ssi ons</w>': 3, '   bo on</w>': 3, ' pri ze</w>': 1, '   f re es</w>': 4, '   wi lli a m s bur g</w>': 1, ' i stu r ban ce</w>': 1, '   fo x es</w>': 2, '   di sen fran chi sed</w>': 1, '   cla ss less</w>': 1, '   prob able</w>': 10, '   ra ti fi ed</w>': 1, '   e le ct</w>': 10, '   bo d y gu ard</w>': 35, '   re lu c t ant</w>': 7, '   w ea l th y</w>': 21, '   bl ac ks</w>': 9, ' fee ding</w>': 2, ' sho er</w>': 1, '   ir on ma ster</w>': 2, '   ba s er</w>': 1, '   i s su ed</w>': 19, '   dis ci pl ine</w>': 23, '   la sh es</w>': 6, '   vo l un te er ing</w>': 3, '   who le he ar ted</w>': 1, '   h or se sho er</w>': 1, '   com man ded</w>': 7, ' man kind</w>': 1, '   cha ins</w>': 23, '   h om i ly</w>': 2, '   p hi lo so p her</w>': 14, '   den oun ce men ts</w>': 1, '   p hi lo so ph ers</w>': 5, ' tu d ying</w>': 1, '   g l ow ing</w>': 7, '   en ga ge ment</w>': 38, '   ac r es</w>': 21, '   y ok el s</w>': 1, ' gr own</w>': 2, '   gra ti fi ed</w>': 1, '   c ro ps</w>': 13, '   re fin e ment</w>': 2, '   bar bar ous</w>': 1, '   be l vo ir</w>': 1, '   ci vi li ze</w>': 1, '   un f la tt er ing</w>': 1, ' fami ly</w>': 7, '   vo l ta i re</w>': 6, '   do st</w>': 7, '   e c st as y</w>': 6, '   un gu ar ded</w>': 2, '   s mo ther ed</w>': 2, '   lu st re</w>': 1, '   con c ea l</w>': 12, '   oo o</w>': 12, '   i ma g in a tions</w>': 3, '   ju b a</w>': 3, '   sha mb les</w>': 1, '   la u gh s</w>': 47, '   f re est</w>': 1, '   ho v el s</w>': 1, '   d rea m t</w>': 15, '   spe c t ac u l ar</w>': 12, '   de cor u m</w>': 4, '   minu et</w>': 2, '   cour tly</w>': 2, ' z</w>': 14, '   mo ve men ts</w>': 8, '   st ru t</w>': 5, ' ci vi li z ed</w>': 2, ' in u et</w>': 1, '   min i tu s</w>': 1, ' or der ly</w>': 1, '   s la ver s</w>': 1, '   sc ar es</w>': 39, '   in den tu red</w>': 4, '   ca sta ways</w>': 1, '   un e mp lo y ed</w>': 15, '   dr un k ar ds</w>': 2, '   v a g ab on ds</w>': 1, '   dre gs</w>': 3, '   re vo lu tions</w>': 2, '   pa mp h le ts</w>': 4, '   ab st r ac tion</w>': 4, ' o th ers</w>': 2, '   lo c ke</w>': 2, '   v att el</w>': 1, '   di der o t</w>': 1, '   sur ve ying</w>': 1, '   na ti ves</w>': 19, '   w ea ther ed</w>': 1, '   nu mi di an</w>': 1, '   pre ser ving</w>': 1, '   ti er r a</w>': 2, ' car</w>': 10, '   min i v ans</w>': 2, '   re ser ved</w>': 13, '   o l d s mo bi le</w>': 4, '   si l hou e tt e</w>': 2, '   hi tt in</w>': 11, '   f ar ra h</w>': 6, '   ki d na pp ing</w>': 30, '   p al m er</w>': 32, '   k aren</w>': 75, '   f l or es</w>': 8, '   un die</w>': 1, ' deal</w>': 6, '   st un t</w>': 32, '   co lo m bi ans</w>': 3, '   y o y o</w>': 2, '   e s co b ar</w>': 11, '   of f a</w>': 7, '   co st a</w>': 11, '   me s a</w>': 3, '   chi l i</w>': 23, '   lo ve jo y</w>': 14, '   la me st</w>': 2, '   qu ad ri ce ps</w>': 2, '   st un t man</w>': 1, '   ca t le t t</w>': 1, '   se t up</w>': 21, '   hi bi sc us</w>': 1, '   u c</w>': 4, '   ra m on</w>': 3, '   c ea s ar</w>': 1, '   sta p le</w>': 3, '   ju mp y</w>': 12, '   p ani cked</w>': 21, '   z i lli on</w>': 3, '   fo cking</w>': 1, '   y a y o</w>': 1, '   mi tch u m</w>': 1, '   d or a do</w>': 3, '   8 1 5 0</w>': 1, '   la u re l</w>': 9, '   le o</w>': 116, '   com ma s</w>': 2, '   tri ck y</w>': 19, ' fa de</w>': 3, '   ber ni e</w>': 38, '   sc ri pt</w>': 92, '   il on a</w>': 3, '   mu r ra y</w>': 38, '   sa ff r in</w>': 3, ' lo ve jo y</w>': 1, '   f re e man</w>': 6, '   har v ey</w>': 26, ' fin g ers</w>': 4, '   ro x y</w>': 6, ' i g n</w>': 1, '   any place</w>': 16, '   jo ck</w>': 6, '   le tt er man</w>': 8, '   z im m</w>': 9, '   con ven i ence</w>': 17, ' f rea ks</w>': 2, '   lo ver boy</w>': 4, '   d ev o e</w>': 3, '   fin d ers</w>': 4, '   di p shit</w>': 11, '   s ca mm ed</w>': 2, '   bar b on i</w>': 4, '   so ver ei g n</w>': 6, '   fi at</w>': 1, '   ja mm in</w>': 2, ' br o</w>': 2, '   fuck b all</w>': 2, '   att en d ant</w>': 8, '   lo ck ers</w>': 4, '   pro du c er</w>': 36, '   chi l</w>': 2, '   tu sh</w>': 2, '   lon e some</w>': 24, '   sp an ked</w>': 4, '   k en o</w>': 1, '   in ve st</w>': 21, '   c r ow ded</w>': 30, '   m om o</w>': 9, '   se l z ni ck</w>': 6, '   tra in er</w>': 10, '   gr ab b er</w>': 1, '   be f all</w>': 2, '   d or is</w>': 20, '   sc re en wri t ers</w>': 1, '   ra f ting</w>': 1, '   k er n</w>': 1, '   ome let</w>': 8, '   p an ca kes</w>': 12, '   bi ll bo ard</w>': 2, ' ri de</w>': 2, '   c y cl one</w>': 3, '   no min e e</w>': 1, '   g ro te s qu e</w>': 13, ' dri ving</w>': 3, '   in ve st ors</w>': 11, '   se du c ing</w>': 5, '   sh ar king</w>': 3, '   a ll en</w>': 22, '   sh y lo ck</w>': 7, '   beli ev able</w>': 7, '   se tt le ment</w>': 33, '   d ev e lo ps</w>': 6, '   s ca ms</w>': 9, '   f ea ture</w>': 22, '   star red</w>': 2, '   re lea ses</w>': 10, '   me sa s</w>': 1, '   u tt er ing</w>': 1, '   an no ys</w>': 1, '   op er a tor</w>': 36, '   mar k ers</w>': 9, ' er go</w>': 1, '   af for ding</w>': 1, '   h er a ld</w>': 9, '   a ir lin es</w>': 13, '   su es</w>': 3, '   gu t sy</w>': 1, '   s qu in ting</w>': 1, '   lo an sh ar k</w>': 1, '   pa to is</w>': 1, '   b en son hur st</w>': 3, '   t ar z an a</w>': 3, '   fin k</w>': 32, '   fin ks</w>': 1, '   f ay</w>': 2, ' ten</w>': 28, '   cap p</w>': 1, '   fin ger ti p</w>': 3, '   p ac in o</w>': 3, '   ser pi c o</w>': 1, '   h or r or</w>': 60, '   men u</w>': 14, '   cont in en tal</w>': 10, '   ni r o</w>': 1, '   whi ch ever</w>': 9, '   sh or ty</w>': 19, '   ni ck i</w>': 3, '   z i p</w>': 23, '   char l ton</w>': 2, '   he st on</w>': 3, '   f li pp ing</w>': 7, ' mi lli on</w>': 17, ' s li me</w>': 2, '   be tt e</w>': 2, '   f li r ty</w>': 1, ' co tt on</w>': 1, ' ca b in</w>': 2, '   jo an</w>': 25, '   cra w for d</w>': 46, ' mi l d red</w>': 1, '   in ten si ty</w>': 7, ' bri de</w>': 11, '   n y l ons</w>': 6, '   co stu m er</w>': 1, '   pa ti o</w>': 14, '   ex p lo ded</w>': 11, '   ki d ne ys</w>': 11, ' ter m</w>': 14, '   ar m cha ir</w>': 3, '   ca g ne y</w>': 4, '   g ran vi e w</w>': 1, '   bi s ca y n e</w>': 2, '   s ea si ck</w>': 4, '   st re i s and</w>': 2, '   fi l m ma king</w>': 3, ' e mo tion</w>': 2, '   ex e cu tive</w>': 53, '   par a m oun t</w>': 7, '   re con si der ed</w>': 4, '   car j ack</w>': 7, '   ca mar o</w>': 2, '   ar g y le</w>': 6, '   lo an ing</w>': 4, '   s ki m</w>': 9, '   mu ff</w>': 5, '   mi s gi v in gs</w>': 2, '   ra tion a li ze</w>': 3, '   he i gh ten s</w>': 2, '   wi st ful</w>': 1, '   s wee t face</w>': 1, '   mm mm m</w>': 24, '   in tri gu ed</w>': 4, '   c y l one</w>': 1, '   ma in ta in ed</w>': 6, '   lu f kin</w>': 1, '   ac qui si tion</w>': 7, '   in f li ct</w>': 5, '   f la sh es</w>': 10, '   f la sh b ac ks</w>': 4, '   al ph a</w>': 27, '   co a st gu ard</w>': 7, '   pen ding</w>': 9, '   chi mer a</w>': 25, '   af lo at</w>': 3, '   do b b ins</w>': 1, '   ki r k</w>': 74, '   ha li fa x</w>': 3, '   no v a</w>': 7, '   s co ti a</w>': 1, '   s ea s</w>': 9, '   lin er</w>': 10, '   sa l v or</w>': 1, ' po se ssion</w>': 1, '   tur b ine</w>': 7, '   s cu tt l ed</w>': 3, '   com pre ss or</w>': 3, '   la g</w>': 4, '   pre f er ably</w>': 5, '   si x te en th</w>': 6, '   al u min i u m</w>': 1, '   al on g side</w>': 12, '   d rea m in</w>': 8, '   gen ny</w>': 1, '   min ding</w>': 10, '   w el der</w>': 4, '   do dge</w>': 29, '   h y po the ti ca lly</w>': 5, '   si t k a</w>': 4, '   co x s wa in</w>': 2, '   e pp s</w>': 44, '   al ba t ro ss</w>': 7, '   lo ta</w>': 2, '   pa s sa g es</w>': 8, '   t ow ing</w>': 2, '   j ack po t</w>': 8, '   s cu tt les</w>': 1, '   supp o sing</w>': 15, '   ca b les</w>': 13, '   sp ok an e</w>': 1, '   mo or in gs</w>': 6, '   t an g l ed</w>': 7, '   ma u re en</w>': 24, '   a mi d shi ps</w>': 1, '   wa ter l ine</w>': 2, '   con ven i en tly</w>': 6, '   wi ll ya</w>': 15, '   stan d point</w>': 3, '   1 9 5 3 </w>': 5, '   4 0 0</w>': 22, '   bar bar i c</w>': 9, ' con si der</w>': 2, '   y a ho o</w>': 8, '   ou tri g ger</w>': 1, '   ru sted</w>': 5, '   ad ri ft</w>': 6, '   tur b in es</w>': 2, ' se at</w>': 1, '   ro tor</w>': 2, '   2 5 0 0</w>': 3, '   ou ta</w>': 68, '   under p ow er ed</w>': 1, '   per mi tting</w>': 2, '   f lo a ting</w>': 41, '   fi tt in gs</w>': 1, '   ber ing</w>': 2, '   a la s k an</w>': 2, '   gu lf</w>': 23, '   ca sh ing</w>': 4, '   na u se a</w>': 5, '   di z z in ess</w>': 7, '   lan k y</w>': 3, '   ki lo gra ms</w>': 2, '   s cu tt ling</w>': 2, '   gre er</w>': 14, '   a le u ti ans</w>': 1, '   stu mb le</w>': 4, '   re su l ting</w>': 5, '   co lli sion</w>': 6, ' their</w>': 9, '   di li g ence</w>': 3, '   li fe bo a ts</w>': 3, '   mu t in i ed</w>': 1, '   im a</w>': 2, '   f l ar es</w>': 10, '   sc r oun ge</w>': 5, ' ma sted</w>': 1, '   bri g</w>': 7, '   1 8 7 2 </w>': 1, '   mer ch ant</w>': 11, '   de i</w>': 2, '   gra ti a</w>': 2, '   hel m</w>': 10, '   ce le st e</w>': 4, '   bo ar ding</w>': 16, '   ev ac u a ted</w>': 10, '   f lo a ts</w>': 10, '   fe es</w>': 12, '   mo or ing</w>': 2, '   no a a</w>': 1, '   bu o y</w>': 2, '   s ca ven ging</w>': 2, '   stu mb l ed</w>': 12, '   wi s er</w>': 13, ' lo a d</w>': 9, '   s ni ff in</w>': 3, '   un lo a d</w>': 11, ' pre ser ved</w>': 1, '   st e er</w>': 13, ' we st</w>': 6, ' ma king</w>': 6, '   pa w s</w>': 5, '   t ar z an</w>': 3, '   wh y n</w>': 10, ' s ea m</w>': 1, '   bu z z ed</w>': 2, '   se y m our</w>': 34, '   ma x ine</w>': 51, '   re f er en ces</w>': 16, '   ba ge l</w>': 3, '   en ro ll</w>': 1, '   dan a</w>': 51, '   du st of f v ar n ya</w>': 1, ' po op er</w>': 2, '   bo s ni a</w>': 5, '   e lo qu en tly</w>': 1, ' fun k</w>': 1, '   mo ch a</w>': 3, '   li ck e ty</w>': 1, ' r en ding</w>': 1, '   no min ate</w>': 3, '   for c ing</w>': 15, ' pa ss ing</w>': 1, '   ex hi bi tion</w>': 16, '   j af fe e</w>': 1, ' w ow</w>': 8, '   hou sing</w>': 8, '   gra du a ting</w>': 2, '   r ac i s m</w>': 6, ' found</w>': 12, '   whi te wa sh ed</w>': 3, '   k no tt s</w>': 3, '   op tion al</w>': 1, '   s lea zy</w>': 8, '   si z es</w>': 4, '   u p si z ing</w>': 1, '   sh ti ck</w>': 2, '   c ri ti c</w>': 16, '   ev al u ate</w>': 6, '   jo sh</w>': 58, '   gu i t ars</w>': 4, '   wom an ly</w>': 5, '   coun se ling</w>': 4, '   ha bi t at</w>': 10, '   poli ci es</w>': 9, '   bo b b ing</w>': 1, ' we ir d</w>': 7, '   ta j </w>': 5, '   ma ha l</w>': 4, '   din ers</w>': 2, '   f la v ors</w>': 7, ' fro z en</w>': 3, '   y o gu r t</w>': 11, '   si de win der</w>': 2, '   b lu r ry</w>': 4, '   sc or ed</w>': 20, '   t ow el s</w>': 16, '   go a te e</w>': 2, '   gu s</w>': 61, '   dan i el</w>': 109, '   du sen tri e b</w>': 1, '   r en ting</w>': 9, ' maybe</w>': 82, '   le s bo s</w>': 3, '   ma stu r ba tion</w>': 8, '   ob no x i ous</w>': 11, '   ex t ro ver ted</w>': 1, '   bo he mi an</w>': 4, '   bi o lo gi cal</w>': 14, '   na g ged</w>': 2, '   bi tch ed</w>': 2, '   en ter ta in ing</w>': 19, '   e w</w>': 5, '   1 9 7 7 </w>': 4, '   fuck face</w>': 4, ' ha ss le</w>': 1, ' ex pen si ve</w>': 4, '   y u pp i es</w>': 4, ' fun k y</w>': 2, ' st ri king</w>': 1, ' se y m our</w>': 2, '   b in go</w>': 35, '   w</w>': 66, '   ne w sle tter</w>': 4, '   f oun da tion</w>': 21, '   un be ar able</w>': 6, '   car di g an</w>': 5, '   m mp h</w>': 3, '   in di st in ct</w>': 1, '   d w f</w>': 1, ' d war f</w>': 1, '   jo ani e</w>': 2, '   ho se ba g</w>': 1, '   l ent</w>': 16, '   e ll is</w>': 10, ' for w ard</w>': 2, '   o om i e</w>': 1, '   p sy cho ti ca lly</w>': 1, '   ob se ss ing</w>': 4, '   ne ck l ace</w>': 14, '   bo o ki sh</w>': 1, '   e li gi ble</w>': 6, '   b ac hel ors</w>': 2, '   bo d</w>': 2, '   s nu g g le li ci ous</w>': 1, ' win d sur f ing</w>': 1, '   men s an</w>': 1, '   i q </w>': 2, '   ma ver i ck</w>': 19, ' de ss er t</w>': 1, ' min d b en d ers</w>': 1, '   war p</w>': 50, ' au th en ti c</w>': 1, ' ma lls</w>': 2, '   1 9 5 0</w>': 2, '   bra in wa sh</w>': 2, '   sa t ani st</w>': 1, '   sa t ani sts</w>': 2, '   ea si est</w>': 17, ' ra p es</w>': 2, '   hi mm l er</w>': 1, '   har ba u gh</w>': 1, ' ac t re ss</w>': 1, '   chi p m un k</w>': 2, '   me l or r a</w>': 1, '   au di tions</w>': 11, '   reme di al</w>': 2, ' j am</w>': 2, ' a tor</w>': 1, '   app ea ls</w>': 6, '   sh read</w>': 1, '   mon go ose</w>': 5, '   s na kes</w>': 23, '   ta x i der my</w>': 4, ' p un k</w>': 1, '   bu ses</w>': 8, ' du de</w>': 4, '   re d ne ck</w>': 14, '   hi ck</w>': 14, '   t s k</w>': 7, '   je w s</w>': 77, '   je wi sh</w>': 75, '   ar y an</w>': 6, '   b la ze</w>': 11, '   i g nor ed</w>': 15, '   a mu sin g ly</w>': 1, '   c ran k y</w>': 6, '   pe ssi mi st</w>': 2, '   an ti qu e</w>': 9, '   thin ga ma j i gs</w>': 1, ' mon th</w>': 6, ' somebody</w>': 16, '   fa shi on able</w>': 4, '   y a a a</w>': 1, '   gir dle</w>': 8, '   e la sti c</w>': 3, '   lu mb ar</w>': 3, '   ar r r gh h h</w>': 1, '   ph en om en on</w>': 8, ' co ok</w>': 2, '   lo go</w>': 3, ' pi e ce</w>': 7, ' ge ous</w>': 1, '   k l ans man</w>': 3, ' yet</w>': 8, '   ho or ay</w>': 4, '   ni kes</w>': 9, '   sig ning</w>': 19, '   h y p no ti z ed</w>': 7, '   a ma z ed</w>': 22, '   hea d lin er</w>': 1, '   sh ri ll</w>': 2, '   pi er c ing</w>': 2, '   j ab b ing</w>': 2, '   k f to</w>': 1, '   a tch ya</w>': 1, ' lo lli po p</w>': 1, '   lo li ta s</w>': 1, ' mu si c</w>': 4, '   ra g time</w>': 2, '   sh ar es</w>': 25, '   ir re l ev ant</w>': 18, '   ob se ssi ve ly</w>': 3, '   7 8 </w>': 11, '   p are</w>': 3, '   min ni e</w>': 4, '   son g wri ter</w>': 2, '   re i s su e</w>': 1, '   l ps</w>': 1, ' i s su e</w>': 1, '   h in d u</w>': 13, ' for ei g n</w>': 3, '   che er lea d ers</w>': 5, '   che er ing</w>': 4, '   g ee shi e</w>': 1, '   wi le y</w>': 1, '   t or tur ing</w>': 8, '   mar gar et</w>': 26, '   fe min in i ty</w>': 1, '   t ea c up</w>': 2, '   wom an ho od</w>': 3, '   o l d en</w>': 1, '   s cu l p ture</w>': 11, '   ta mp on</w>': 4, ' st r on g ly</w>': 2, ' many</w>': 8, '   car to ons</w>': 7, '   pa th o lo gi cal</w>': 9, '   st e p mother</w>': 11, '   ar chi v al</w>': 1, '   ca ta lo gu ing</w>': 2, '   re i s su es</w>': 1, '   ab du ll ah</w>': 3, '   t sa v o</w>': 10, '   f le e</w>': 12, ' beau mon t</w>': 1, '   per su ad ed</w>': 6, ' ma l ar i a</w>': 1, '   con tr ac ted</w>': 7, '   re d be ard</w>': 2, '   k ni gh th o od</w>': 5, '   fini sh es</w>': 3, '   ra ted</w>': 9, '   hi r ing</w>': 20, '   tri pp ing</w>': 8, '   tra pp ing</w>': 6, '   s li ding</w>': 6, ' con tra p tion</w>': 1, '   b om a</w>': 1, '   i di o cy</w>': 3, '   star ling</w>': 69, '   bi b les</w>': 2, '   hi c c up</w>': 2, '   wi se ly</w>': 7, '   en cou ra ge ment</w>': 5, '   pa tt er son</w>': 7, '   con fi ded</w>': 1, '   u p coming</w>': 4, '   fir st bor n</w>': 2, ' bu i ld</w>': 3, '   t or men ting</w>': 8, '   hea ps</w>': 2, '   k ar i m</w>': 1, '   w el coming</w>': 4, '   ri ch er</w>': 15, ' sor ting</w>': 1, '   f ea sted</w>': 2, '   sp ran g</w>': 6, '   br un t</w>': 3, ' ea ter</w>': 10, '   ni ge l</w>': 19, '   ha w th or n e</w>': 7, '   kn ack</w>': 9, '   pa w</w>': 6, '   con te mp la ting</w>': 2, '   f ea st ing</w>': 3, '   si lli est</w>': 2, ' son</w>': 15, '   ge ting</w>': 1, '   d ow n t ro d d en</w>': 1, '   sh ow er ed</w>': 1, '   comp li men ts</w>': 19, '   hel en a</w>': 16, '   sin gh</w>': 7, '   ro ad be ds</w>': 1, '   pi ll ars</w>': 3, '   e mb an k ment</w>': 4, '   cra mp</w>': 9, '   ad ven tur ous</w>': 5, ' ea t ers</w>': 7, '   mor a le</w>': 7, '   pr ow l</w>': 3, ' sor t</w>': 6, ' h o</w>': 18, '   tr an si t ory</w>': 3, '   v an qui sh</w>': 1, '   ser en i ty</w>': 1, ' pro du c ts</w>': 1, '   s mi les</w>': 21, '   en ri ch</w>': 4, '   m ou th s</w>': 24, '   de f en d ers</w>': 2, '   b ru te</w>': 8, '   ter r ori z ed</w>': 3, '   to g ther</w>': 1, '   b ru tes</w>': 2, ' supp ose</w>': 4, '   gr ac e ful</w>': 5, '   pl ac e ment</w>': 9, ' mi s fi re</w>': 1, '   an th i lls</w>': 1, '   tr ack ers</w>': 6, '   con si der ate</w>': 14, '   th or n</w>': 13, '   f en ces</w>': 15, ' s la u gh ter</w>': 1, '   de te st</w>': 9, '   af ri c ans</w>': 2, '   g un p ow der</w>': 4, '   pr in ces</w>': 3, '   en ti c ing</w>': 2, '   lin g ers</w>': 1, '   se cu re ly</w>': 2, '   vi go</w>': 19, '   per ks</w>': 8, '   ga in s bor ough</w>': 2, '   ru l er</w>': 11, '   ven k man</w>': 30, '   j an o s z</w>': 5, '   po h a</w>': 1, '   ra in che ck</w>': 6, '   re jo in</w>': 4, '   re st or a tions</w>': 1, '   e g on</w>': 16, '   in de x</w>': 6, '   m ea sur able</w>': 1, '   p sy cho ma g ne ther i c</w>': 2, '   gh o st bu st ers</w>': 5, '   s lu g ger</w>': 4, '   do or man</w>': 4, '   su per in ten dent</w>': 10, '   li b</w>': 11, '   1 6 2 0</w>': 1, '   j an ine</w>': 12, '   j in g le</w>': 2, '   da i ly</w>': 44, '   re so lu tions</w>': 2, '   ha mp er</w>': 5, '   me l ni t z</w>': 2, '   v a se</w>': 10, '   ba th tu b</w>': 17, '   oo ze</w>': 11, '   s qu ea k y</w>': 4, ' bi ter</w>': 1, '   na ma th</w>': 3, '   ni gh t g own</w>': 5, '   pa ja ma</w>': 3, '   re c</w>': 5, '   f lu ff y</w>': 2, '   ki tt en</w>': 17, '   gen o ci da l</w>': 1, '   ma d man</w>': 31, '   ver</w>': 3, '   me er</w>': 1, '   ga to</w>': 1, '   ha ir ba lls</w>': 1, '   con ju ga l</w>': 2, '   bu g g y</w>': 8, '   su per vi se</w>': 6, '   s me lly</w>': 7, '   st in ki est</w>': 1, '   in t ro du c ing</w>': 5, '   de si red</w>': 3, '   dr ow sy</w>': 1, ' di tch</w>': 1, '   hon e y po t</w>': 1, '   b re w ing</w>': 4, '   star man</w>': 1, '   ha ir less</w>': 3, '   par a m us</w>': 3, '   bu mm er</w>': 8, '   2 0 1 6 </w>': 1, '   dr ac u l a</w>': 12, '   tr an sp lan ts</w>': 5, '   bar r ing</w>': 5, '   go ver n or</w>': 68, ' b it</w>': 12, '   f ra u ds</w>': 2, '   su ed</w>': 10, '   de s k wor ms</w>': 1, '   gh o sts</w>': 65, '   har de me y er</w>': 1, '   bar ter ed</w>': 2, '   hou se kee p ing</w>': 8, '   with hold</w>': 2, '   d war f s</w>': 6, '   bab y sit</w>': 4, '   gh o st bu ster</w>': 3, '   mi god</w>': 2, ' s ni ff ing</w>': 2, '   vi gi e</w>': 1, '   mo l da vi ans</w>': 1, '   re st or ing</w>': 7, '   b y z an t ine</w>': 2, ' por tra it</w>': 2, '   car pa th i a</w>': 3, '   mo l da vi a</w>': 3, '   s cour ge</w>': 6, '   s co l er i</w>': 1, '   o s sin ing</w>': 4, ' 4 8 </w>': 7, '   su sta in ed</w>': 17, '   tu lly</w>': 3, '   under pri vi le ged</w>': 3, '   sh er m</w>': 5, '   der ma bra sion</w>': 1, '   t ro x l er</w>': 1, '   pa per back</w>': 4, ' wee k</w>': 5, ' ser i es</w>': 1, '   pre di c ting</w>': 3, ' 9 4 </w>': 3, '   le c tur er</w>': 1, '   an g l un d</w>': 1, ' se ven th</w>': 5, ' de f en dan ts</w>': 1, '   ob je ction</w>': 59, '   im ma ter i al</w>': 4, '   cha mp s</w>': 4, '   o st ro v </w>': 2, '   pa d ded</w>': 8, '   g ev </w>': 1, '   stra i gh ts</w>': 1, '   se w ers</w>': 9, ' b re e ding</w>': 2, '   ma g ne ther i c</w>': 1, '   co ck ro ac h</w>': 11, '   mo l da vi an</w>': 1, ' vi go</w>': 1, '   1 5 0 5 </w>': 1, '   1 6 1 0</w>': 1, '   ab nor ma l</w>': 8, '   bra in i ac </w>': 2, '   e pi di d y m is</w>': 1, '   no i sy</w>': 12, '   bur g</w>': 7, '   mo bi li ze</w>': 5, ' ne cked</w>': 1, '   d ab bl ed</w>': 2, '   pro p he cy</w>': 6, ' dea th</w>': 13, ' al so</w>': 6, '   t or tur er</w>': 2, '   de spi sed</w>': 4, '   e b b ing</w>': 1, '   ti da l</w>': 3, '   p ne u ma ti c</w>': 1, ' for c ed</w>': 1, ' tra ins</w>': 1, '   1 8 7 0</w>': 2, '   gi g a</w>': 5, ' me ter</w>': 3, '   a ver a ging</w>': 2, '   p sy ch ok in e s is</w>': 1, '   ki lo me ter</w>': 6, '   e ye wi t ne ss es</w>': 5, '   c ran k</w>': 29, '   ho o ves</w>': 4, '   r en a i ss an ce</w>': 7, '   car a v a g gi o</w>': 12, '   br un e ll e sch i</w>': 1, '   ri ver s</w>': 14, '   p ke</w>': 2, ' g ran d par en ts</w>': 1, '   sta tu e</w>': 57, '   vi g i</w>': 3, '   5 9 </w>': 6, '   7 2 </w>': 5, ' n d</w>': 14, '   a sh ore</w>': 12, '   7 9 </w>': 8, '   nu r tur ing</w>': 3, '   mo ld</w>': 11, '   se w er</w>': 20, ' rea c tive</w>': 1, '   bar be cu e</w>': 11, '   si z z l er</w>': 2, '   z und in ger</w>': 1, '   ma gi ci ans</w>': 4, '   mar t y rs</w>': 3, '   ma d men</w>': 7, ' o od</w>': 1, '   p sy ch or ea c tive</w>': 1, '   re sp on ds</w>': 5, '   sp en g l er</w>': 8, '   o l y m pi c</w>': 10, ' pri ce</w>': 2, ' w el come</w>': 6, '   st re s sh ound</w>': 1, '   gh o st bu st ing</w>': 1, '   g ev s</w>': 1, '   a a ah</w>': 7, '   a er o so l</w>': 1, '   un con di tion ally</w>': 1, '   con cen tra ted</w>': 12, '   z e d de more</w>': 2, '   wh e w</w>': 25, '   me ta</w>': 2, '   plan ar</w>': 1, '   ri ft</w>': 7, ' st ab i li ze</w>': 1, '   di se mb ow el ed</w>': 2, '   qu ar ter ed</w>': 2, '   sa pp y</w>': 3, ' cu m ba ya</w>': 1, '   j ac ki e</w>': 95, '   bu b bl ed</w>': 1, '   mar sh ma llow</w>': 6, '   hi gh ri se</w>': 1, '   y u pp i e</w>': 11, '   l ar v a e</w>': 2, ' pri vi le ged</w>': 1, ' o l ds</w>': 5, '   li fe sa ving</w>': 2, '   to g a</w>': 2, '   pre di c tions</w>': 1, '   1 9 9 0</w>': 2, '   po th o l der</w>': 1, '   u u u u u u u gh</w>': 1, '   te le pa th</w>': 1, '   z u u l</w>': 6, '   ga te kee per</w>': 5, '   n ac ho s</w>': 2, '   ph on ed</w>': 26, '   ex er ci sing</w>': 9, '   ri di cu l ou s ly</w>': 3, '   de st ru ctor</w>': 3, '   go z er</w>': 8, '   ro y lan ce</w>': 2, '   ev en in gs</w>': 9, ' go z er</w>': 1, '   su mer i an</w>': 1, '   hi t ti tes</w>': 1, '   me so po ta mi ans</w>': 1, '   su mer i ans</w>': 1, ' z u u l</w>': 3, '   min i on</w>': 1, '   re f ers</w>': 5, '   de m i</w>': 2, '   6 0 0 0</w>': 1, '   so ci e ti es</w>': 3, '   se c ts</w>': 1, ' sti ff</w>': 1, '   p sy cho lo gi st</w>': 25, '   ce llo</w>': 9, '   vi o l in</w>': 32, '   pro k o fi ev </w>': 1, '   ac kn ow le d ging</w>': 4, ' do dge</w>': 2, ' hu st le</w>': 1, '   tri pe</w>': 4, '   s lo pp y</w>': 10, '   re gen ts</w>': 3, '   ter min ate</w>': 13, '   v ac ate</w>': 4, '   pre mi ses</w>': 5, '   ma llow</w>': 1, '   sp ani el</w>': 2, '   mu t t</w>': 9, '   stan t z</w>': 2, '   de l ac or te</w>': 1, '   li br ar i an</w>': 8, '   s ou ven ir</w>': 8, '   1 9 6 4 </w>': 4, '   b ac ter i a</w>': 6, '   mo l ds</w>': 1, '   fun gu s</w>': 11, '   ho b bi es</w>': 7, '   in te ll e c tu al</w>': 14, '   r ac ke t b all</w>': 1, '   e p a</w>': 4, '   di stu r ban ces</w>': 5, '   pro por tions</w>': 7, ' pe ck</w>': 1, '   mi x ture</w>': 11, '   ga ses</w>': 2, '   p sy ch op a th</w>': 14, '   w r on g ful</w>': 3, '   pro se cu tion</w>': 13, '   en vi r on men tal</w>': 12, '   no x i ous</w>': 1, '   ha z ar d ous</w>': 8, '   p h</w>': 7, '   par a p sy cho lo g y</w>': 2, '   re in for ce ment</w>': 4, '   e s p</w>': 2, '   sho cks</w>': 6, '   wa v y</w>': 3, '   7 5 </w>': 14, '   k r y p ton</w>': 15, '   may er</w>': 3, '   bo lo g na</w>': 1, '   su per man</w>': 79, '   thr ow ers</w>': 3, '   un u su ally</w>': 3, '   sh er lo ck</w>': 6, '   re si du al</w>': 3, '   mo i stu re</w>': 6, '   re d ne ss</w>': 1, ' st rea m</w>': 4, '   st ro g on</w>': 1, '   re f er red</w>': 18, '   sh an d or</w>': 4, '   con du c ted</w>': 16, '   i v o</w>': 1, '   to b in</w>': 1, '   ke y ma ster</w>': 6, '   gra p h</w>': 3, '   app li an ce</w>': 3, '   th re sh old</w>': 9, '   e st ab li sh ing</w>': 5, '   de ca de</w>': 14, '   par an or ma l</w>': 4, '   e li min a tions</w>': 1, '   pa y men ts</w>': 17, ' ri ve ted</w>': 1, '   gir d ers</w>': 4, '   se l en i u m</w>': 3, '   cor es</w>': 2, '   pu l s ars</w>': 1, '   ga la x i es</w>': 5, '   v al ves</w>': 1, '   ne u tr on i ze</w>': 1, '   st rea ms</w>': 5, '   co ll e c tive</w>': 11, '   cla ir vo y an ce</w>': 1, '   te le pa th i c</w>': 2, '   in tru ding</w>': 11, '   mar sh ma ll ows</w>': 6, ' pu ft</w>': 2, '   wa con da</w>': 1, '   a gi le</w>': 2, ' con sti tu ted</w>': 1, '   in ha bi t an ts</w>': 1, '   c ea se</w>': 20, '   ar chi te ct</w>': 31, '   wh ack o</w>': 7, '   p sy ch ok in e ti c</w>': 1, '   pu er to</w>': 11, '   ri can</w>': 10, '   be ll boy</w>': 2, '   ven ts</w>': 2, '   un li cen sed</w>': 2, '   ac c el er a tor</w>': 3, '   br ink</w>': 11, ' f l ow</w>': 2, ' pri c ey</w>': 1, '   fi x er</w>': 3, '   ad ver ti sed</w>': 1, '   de st in ed</w>': 10, '   pa t ent</w>': 5, '   pro d</w>': 1, '   ga li le o</w>': 1, '   f lo or ed</w>': 4, '   de sig ning</w>': 6, '   mi c ro chi p</w>': 3, '   mo le st</w>': 3, '   o ver ex ci ted</w>': 1, '   ro ck f all</w>': 2, '   un ex pla in ed</w>': 3, ' al ti tu de</w>': 1, ' seen</w>': 5, '   bor ou gh s</w>': 2, '   char ted</w>': 4, ' ro a m ing</w>': 1, '   v a por ous</w>': 1, ' t or so</w>': 1, '   app ar i tion</w>': 2, '   sh el ves</w>': 6, ' ch oo se</w>': 2, '   f ab ri ca ted</w>': 1, ' t un g st en</w>': 1, ' 3 2 5 </w>': 1, '   mm m h mm m</w>': 1, '   ir on work</w>': 1, '   re su me</w>': 22, '   e le c tr on i c</w>': 19, '   coun ter m ea sur es</w>': 1, '   k ar ate</w>': 6, '   e m per ors</w>': 3, '   re vo lt</w>': 5, '   l y k as</w>': 2, '   j er ses</w>': 5, '   re ti ar i us</w>': 4, '   sa m ni te</w>': 1, '   de ba tes</w>': 1, '   ar en a</w>': 13, '   de mi god</w>': 2, '   n ar ci s sus</w>': 21, '   un ju st ly</w>': 1, '   ce le bra ting</w>': 14, '   vi gi l</w>': 1, '   pro x im o</w>': 7, '   wai ling</w>': 3, '   sur ly</w>': 1, '   tri bu us</w>': 6, '   co lo s se u m</w>': 8, '   for u m</w>': 9, '   com mo d us</w>': 18, '   hea ving</w>': 4, '   hea ve</w>': 4, '   s cu l p tur es</w>': 4, '   st ab i li ty</w>': 11, '   ch or us</w>': 10, '   p ra i ses</w>': 4, '   lu ci ll a</w>': 1, '   de po se</w>': 1, '   dan u be</w>': 6, '   pre par ation</w>': 12, '   with d ra wa l</w>': 7, '   cl ou ded</w>': 4, '   under ta king</w>': 3, '   re pu b li can</w>': 15, '   di sa pp o in ts</w>': 2, '   le gi ons</w>': 8, '   g al en</w>': 3, '   un well</w>': 4, '   qu in tu s</w>': 5, '   mer i da s</w>': 3, '   p ra i se</w>': 12, '   hu mor s</w>': 3, '   sp on s or</w>': 20, '   s ac ks</w>': 5, '   li cen ses</w>': 8, '   di ver t</w>': 8, '   ga i us</w>': 4, '   j an us</w>': 6, '   e m per or ship</w>': 2, ' f re e do m</w>': 2, '   re pa id</w>': 4, '   in su bor din ation</w>': 6, '   gla di a tor</w>': 10, '   for t re ss</w>': 31, '   a war ding</w>': 2, '   te m per a ment</w>': 6, '   who lly</w>': 6, '   f al c o</w>': 13, '   sen at ors</w>': 12, '   au re li us</w>': 3, '   sta tu es</w>': 16, '   cla u di a</w>': 66, '   ri o ts</w>': 9, '   un rest</w>': 5, '   a gh</w>': 6, '   cra f t s men</w>': 1, '   co s</w>': 13, '   ha d es</w>': 7, ' ca e s ar</w>': 1, '   dis co ver ing</w>': 4, '   lin ea ge</w>': 1, '   con ver g es</w>': 1, '   h er cu les</w>': 2, '   ma g ni fi c ence</w>': 4, '   e di tion</w>': 18, '   a th le te</w>': 11, '   le ga te</w>': 5, '   pre co ci ous</w>': 2, '   sc ri be</w>': 1, '   a ll o ca ted</w>': 2, '   de f en ding</w>': 18, '   vi r go</w>': 2, '   fa tes</w>': 5, ' fif th</w>': 9, '   el de st</w>': 2, '   de fin ing</w>': 4, '   st ri ves</w>': 2, '   im per i al</w>': 16, '   lo y al ti es</w>': 2, '   th read</w>': 18, '   t uni c</w>': 2, '   e pi c te tu s</w>': 1, '   re ci ted</w>': 2, '   ca tu ll us</w>': 1, '   lu cre ti us</w>': 1, '   vi r gi l</w>': 17, ' believe</w>': 21, '   fi li al</w>': 2, '   in ex per i en c ed</w>': 7, '   o ver heard</w>': 13, '   can tu s</w>': 1, '   ver us</w>': 1, ' te lling</w>': 3, '   o ver se es</w>': 2, '   re bi r th</w>': 3, '   u p right</w>': 12, '   tr ans f er red</w>': 40, '   ta x ation</w>': 4, '   under ta k en</w>': 5, '   bu ll hea de d ne ss</w>': 1, '   ser v is</w>': 2, '   tri u mp ha l</w>': 4, '   den ted</w>': 2, '   pl u me</w>': 4, '   qu a il</w>': 6, '   s la u gh ter</w>': 20, ' h and</w>': 24, '   su c ce ssion</w>': 3, '   p ra e t ori an</w>': 1, '   di so be y</w>': 6, '   su m mon ed</w>': 4, '   v ac i ll a ting</w>': 1, '   co h or t</w>': 1, '   re sto cking</w>': 1, '   a ll o t ment</w>': 1, '   out sp ok en</w>': 2, '   cen tu ri ons</w>': 2, '   un pro ved</w>': 1, '   g l ar ing</w>': 4, '   com pe t ence</w>': 5, '   bab y lon i ans</w>': 1, '   he b re w s</w>': 2, '   nu mi di ans</w>': 1, '   e g y p ti ans</w>': 3, '   v a por s</w>': 1, '   the m is</w>': 1, '   man to</w>': 1, '   pa e stu m</w>': 2, '   o sti a</w>': 2, '   e g y pt</w>': 24, '   nu mi di a</w>': 1, '   i dea li sti c</w>': 6, '   ten ding</w>': 2, '   g ro ves</w>': 5, '   o ver see ing</w>': 2, '   prob ing</w>': 2, '   c y ni c s</w>': 3, '   a th en i ans</w>': 1, '   gla di at ors</w>': 4, '   char i o t</w>': 5, '   ti b er</w>': 1, '   i g no ble</w>': 1, '   bra ve st</w>': 8, '   ear th ly</w>': 6, '   be hea ding</w>': 1, '   ar en as</w>': 1, '   ga ll op ing</w>': 8, '   st ab s</w>': 4, '   gu sh es</w>': 2, '   an ti ci pa ting</w>': 2, '   ci m on</w>': 1, '   s my r na</w>': 1, '   as sa ss in</w>': 13, ' n ar ci s sus</w>': 1, '   p om pe i i</w>': 2, '   pro fi ta ble</w>': 9, '   fe li x</w>': 3, '   ac comp li sh es</w>': 1, '   cla mp ed</w>': 2, '   di ck er</w>': 1, '   me ta ph ors</w>': 8, '   la u gh ter</w>': 15, '   p al in dro mo s</w>': 1, '   gla di at ori al</w>': 1, '   bar bar i ans</w>': 4, '   cre t an</w>': 1, '   mar ble</w>': 11, '   con gen i al</w>': 3, '   gla d st one</w>': 5, '   ver an da</w>': 2, '   ta ft</w>': 3, '   com pu l sion</w>': 5, '   pre s b y ter i an</w>': 4, '   a ll an</w>': 8, '   re ver end</w>': 59, '   6 5 7 </w>': 1, ' 2 0 3 6 </w>': 1, '   m c qui re</w>': 2, '   m c cle er y</w>': 4, '   a gi ta tion</w>': 1, '   bra d do ck</w>': 6, '   a gi ta t ors</w>': 3, '   ro b in son</w>': 63, '   ra i ling</w>': 3, '   sin g le man</w>': 2, '   ru sh ed</w>': 14, '   g l en vi e w</w>': 1, '   bur g l er</w>': 1, '   per ver ted</w>': 11, '   bor e do m</w>': 7, '   s li my</w>': 14, '   ta bo o</w>': 3, '   un ho ok</w>': 3, '   bl ou se</w>': 5, '   no ve l</w>': 49, '   li ven</w>': 2, '   in a de qu ate</w>': 6, '   un de si r able</w>': 2, '   han ger</w>': 5, '   5 1 2 </w>': 1, '   wai ter</w>': 42, '   se du ce</w>': 25, '   un z i p</w>': 4, '   b ran i ff</w>': 2, '   de wi tt e</w>': 1, '   di sh on est</w>': 6, ' ba ked</w>': 2, '   pro se cu te</w>': 13, '   in sure</w>': 8, '   un cl en ch</w>': 1, '   fi sts</w>': 12, '   g ru dge</w>': 14, '   re sen t ment</w>': 4, '   f lin gs</w>': 3, '   ha l p in g ha m</w>': 1, '   sch o l ar</w>': 5, '   ac comp li sh ing</w>': 4, '   car l s ons</w>': 1, '   re jo ice</w>': 1, '   la mp</w>': 24, '   ou gh t n</w>': 5, '   ta sted</w>': 19, '   k r in ge le in</w>': 37, '   ch l or of or m</w>': 3, '   s ke le ton</w>': 12, '   f la e mm ch en</w>': 5, '   st en o gra p her</w>': 11, '   fro ck</w>': 12, '   st en o gra ph ers</w>': 1, '   re du c ing</w>': 5, '   di c ta tion</w>': 6, '   su z e tt e</w>': 17, '   ma d ly</w>': 7, '   g ru sin s k a ya</w>': 18, '   ad mi red</w>': 19, '   den oun ce</w>': 1, '   d ev our ing</w>': 3, '   mon a st er y</w>': 10, '   f li x</w>': 2, '   b en ven u to</w>': 1, '   ga i ger n</w>': 5, '   s li m</w>': 18, '   ver on al</w>': 2, '   b al con y</w>': 25, '   f ra ys</w>': 1, '   exac ting</w>': 5, '   we ar in ess</w>': 2, '   s la ved</w>': 3, '   i cy</w>': 8, ' co ld</w>': 4, '   app ea ling</w>': 11, '   stra i ts</w>': 4, '   l ou i si an a</w>': 12, '   a er o plan e</w>': 5, '   pre y sing</w>': 43, '   o tt er n sch la g</w>': 2, '   f re der s d or f</w>': 3, '   pi men o v </w>': 7, '   hu m m</w>': 4, '   im po st ers</w>': 2, '   a gre ea ble</w>': 5, '   im po ster</w>': 5, '   te le ph on ed</w>': 13, '   di ph ther i a</w>': 2, '   b ac i ll i</w>': 1, '   bar man</w>': 7, '   f la e m m</w>': 10, '   sti mu la ting</w>': 5, '   ab sin the</w>': 3, '   di st in c tive</w>': 4, '   te x ti le</w>': 3, '   d ra p es</w>': 21, '   bar on ess</w>': 3, '   man che ster</w>': 24, '   cu b</w>': 11, '   cor re sp on d ence</w>': 5, '   co g n ac </w>': 5, ' we ight</w>': 1, '   mor e over</w>': 17, '   f l or ence</w>': 8, '   gr ow n up</w>': 1, '   ad v an ta g es</w>': 13, '   s wi t z er land</w>': 16, '   s ki ing</w>': 17, '   z in n ow i t z</w>': 6, '   su c ce ss fu lly</w>': 11, '   mo p</w>': 10, '   ex por ts</w>': 3, '   b al k ans</w>': 2, '   sc ra tch ed</w>': 11, '   ten ta tive</w>': 3, '   sa x on i a</w>': 2, '   m ac h in er y</w>': 11, '   re spe c tive</w>': 2, '   sc ru pu l ous</w>': 2, '   cer ta in ti es</w>': 2, '   de si r ing</w>': 1, '   ther</w>': 2, '   fa ll on</w>': 5, '   fo g g y</w>': 4, '   fri gh t fu lly</w>': 3, '   me i er he i m</w>': 1, '   la u re ls</w>': 3, ' ro ses</w>': 2, '   t reme z z o</w>': 3, '   s sh h</w>': 6, '   or chi ds</w>': 5, '   mi lli on a ir es</w>': 12, '   ra t z vi ll e</w>': 1, '   can ce lling</w>': 4, '   whi m</w>': 10, '   ob li ga tions</w>': 13, '   g ru </w>': 3, '   po si ti ve ly</w>': 22, '   ra di ant</w>': 5, '   app la use</w>': 8, '   cla qu es</w>': 1, '   f right</w>': 11, '   pr ye sing</w>': 1, '   en ti c ed</w>': 2, '   e m be z z l er</w>': 5, '   di r ti er</w>': 1, '   ma g na te</w>': 2, '   e mp lo y ed</w>': 23, '   di ver si ons</w>': 2, '   be f it</w>': 1, '   bo ok kee per</w>': 9, '   dis sa ti s fi ed</w>': 6, '   in ex pen si ve</w>': 1, '   li sa ve ta</w>': 1, '   tr ou pe</w>': 4, '   sc en er y</w>': 14, '   ba ll e ts</w>': 2, '   b lu ff</w>': 35, '   di c ta ting</w>': 6, ' pre y sing</w>': 1, ' pu r po ses</w>': 1, '   di c ta te</w>': 10, '   ger st en k or n</w>': 1, '   bro ac h</w>': 1, '   ac h</w>': 8, '   bo b o</w>': 30, '   li lly</w>': 29, '   st ee ling</w>': 1, '   s ki mm ing</w>': 4, '   an g</w>': 8, ' g le e z</w>': 2, '   l ou se</w>': 10, '   pu h</w>': 4, '   f ra mm is</w>': 1, '   g le e z</w>': 2, '   na g</w>': 15, '   f lu tter</w>': 2, '   to te</w>': 6, '   ti ck le</w>': 9, '   du ran do</w>': 1, '   sle e per</w>': 15, ' su i te</w>': 1, ' f ra me</w>': 1, '   he b b ing</w>': 6, '   v ou ch es</w>': 1, '   wi d ow ed</w>': 1, '   re ver ses</w>': 2, '   wi de ly</w>': 6, '   fe ll ow es</w>': 2, '   di ll on</w>': 16, '   he mor r ha ge</w>': 2, '   pla y back</w>': 5, '   de l m ar</w>': 21, '   lan g try</w>': 5, '   pu r cha sed</w>': 7, '   fi li gre ed</w>': 1, '   v al u ed</w>': 9, '   ba d ger</w>': 1, ' r oun ded</w>': 1, '   per k</w>': 12, '   k a g gs</w>': 3, '   per cy</w>': 2, ' fa ir</w>': 3, '   re cu per a ting</w>': 2, '   of f h and</w>': 4, '   sa le s men</w>': 9, '   dea d w o od</w>': 4, '   s ar b er</w>': 1, '   we b b</w>': 21, ' f li ght</w>': 1, '   in cen tive</w>': 9, '   he el s</w>': 27, '   b oun c ing</w>': 11, '   s lo b s</w>': 2, '   b ack han ded</w>': 1, '   wh a da ya</w>': 9, '   ju st us</w>': 2, '   st in k er</w>': 4, '   le gi ti ma te</w>': 31, '   sur vi v or</w>': 8, '   my r a</w>': 14, ' co ok ed</w>': 3, '   re sta u ran ts</w>': 16, '   la jo ll a</w>': 1, '   win n in gs</w>': 4, '   b lu e be ll</w>': 2, '   cor n b all</w>': 2, '   sc ra m</w>': 5, '   ga h</w>': 1, '   li gh t ning</w>': 27, ' t ough</w>': 3, '   tu l s a</w>': 4, '   bor ok er</w>': 1, '   sc ra pe</w>': 14, '   tur n down</w>': 1, '   bro k er</w>': 18, '   kn ow le d g ea ble</w>': 2, '   co s me ti c s</w>': 4, '   hi ke</w>': 18, '   a ta s ca der o</w>': 1, '   u p sta te</w>': 10, '   sto cks</w>': 11, '   tra il</w>': 62, '   lan g le y</w>': 12, '   lin go</w>': 6, ' c on</w>': 6, '   af fi da v it</w>': 3, '   sp en der</w>': 3, '   ba d m ou th</w>': 1, '   pa y of f s</w>': 5, '   a but</w>': 1, '   g ran i te</w>': 8, '   ca u tion</w>': 17, '   g un sho t</w>': 12, '   si l en c er</w>': 4, '   pi er son</w>': 1, '   in con ven i ent</w>': 12, '   si m ms</w>': 9, '   ton i</w>': 6, '   b la ke</w>': 29, '   ha d d on fi e ld</w>': 11, '   star t le</w>': 5, '   ta te</w>': 5, '   sti ff s</w>': 5, '   ce me ter y</w>': 44, '   cu r ta ins</w>': 15, '   ve s se ls</w>': 12, '   for ear m</w>': 2, '   k no cks</w>': 6, ' ir on</w>': 3, '   st ru g g les</w>': 4, '   whi tt in g ton</w>': 2, '   lo om is</w>': 21, '   ran s ac ked</w>': 3, '   me y ers</w>': 12, '   con ta min ate</w>': 3, '   mi c ro po ll u t an ts</w>': 1, '   ca ter o</w>': 1, '   in ac cu r ac y</w>': 1, '   hi st ori ca lly</w>': 6, '   c lea ver</w>': 2, '   in ac cu ra te</w>': 3, '   clo se ts</w>': 10, '   k er i</w>': 5, '   mo lly</w>': 37, '   fu gi tive</w>': 13, '   d or ms</w>': 4, '   ro om ing</w>': 5, '   hea d ma ster</w>': 5, '   can cu n</w>': 2, '   bi mb o</w>': 11, '   s wi p ed</w>': 7, '   sh el ve</w>': 2, '   re si dent</w>': 15, '   di ge st</w>': 11, '   st ro de</w>': 6, '   er ran ds</w>': 9, '   co o ki e</w>': 35, '   ti gh ter</w>': 9, '   in si gh t ful</w>': 4, '   g y m na si u m</w>': 3, '   ca fe ter i a</w>': 13, '   sh an e</w>': 4, '   b la a a a a g g gh h h h h</w>': 1, '   ca sing</w>': 7, '   b en ad r y l</w>': 1, '   la te x</w>': 1, '   oo o oh</w>': 11, '   hur l</w>': 5, '   ac ed</w>': 5, '   k ar a</w>': 6, '   we ir d ne ss</w>': 6, ' ad dy</w>': 2, '   car ving</w>': 6, ' lan ter n s</w>': 2, '   di ll we ed</w>': 1, '   1 9 9 5 </w>': 9, '   or g an ni ze</w>': 1, '   ha un ted</w>': 21, '   de br a</w>': 3, '   ja mi e</w>': 65, '   ll o y d</w>': 148, '   shi th ea ds</w>': 3, '   de f ac ing</w>': 1, '   my ers</w>': 35, '   cre pt</w>': 3, '   ro se mar y</w>': 10, '   bl an k en ship</w>': 1, '   tr ans la ted</w>': 10, '   s ac ri fi ces</w>': 10, '   c el ti c</w>': 2, '   sa m ha in</w>': 2, '   run es</w>': 3, '   al p ha b et</w>': 19, '   ori g in a ted</w>': 3, '   cu l ts</w>': 1, '   por t end</w>': 2, '   in vo ke</w>': 7, '   run e</w>': 2, '   de li ver er</w>': 1, '   a ge less</w>': 2, '   in cu ba ted</w>': 1, '   pro gen y</w>': 1, '   bl in ded</w>': 9, '   w y n n</w>': 2, '   st ro d es</w>': 1, '   ea st b ound</w>': 1, '   re ser vo ir</w>': 9, '   lo ck down</w>': 6, '   ter ence</w>': 2, '   re ti r ing</w>': 12, '   per ce p tion</w>': 14, '   a st ound</w>': 1, ' ome times</w>': 1, '   th ri ve</w>': 4, '   se c lu sion</w>': 2, ' si tting</w>': 5, '   loo se lea f</w>': 10, '   na ga sa k i</w>': 4, ' sc re w</w>': 4, '   de ci si ve</w>': 5, '   w oo d ly</w>': 16, '   pen e lo pe</w>': 22, '   t at</w>': 1, ' de po sit</w>': 1, '   n in ny</w>': 2, '   he li co p ter</w>': 46, ' de ep</w>': 8, ' de e p ly</w>': 1, '   t ee m ing</w>': 1, '   ba y on e ted</w>': 1, '   de fe ca ted</w>': 1, '   pi an o s</w>': 5, '   un cont ro ll able</w>': 6, '   de cor ation</w>': 4, '   jo ck stra p</w>': 2, '   com pa ssion</w>': 22, '   un se l fi sh ne ss</w>': 1, '   p ea ce fu l ne ss</w>': 2, '   ma u d l in</w>': 2, '   e du ca tion al</w>': 6, '   mo t or c y c les</w>': 7, '   h or ning</w>': 2, '   h or r en d ou s ly</w>': 1, '   cra ve</w>': 5, '   re uni on</w>': 28, '   sp o ok y</w>': 24, '   wa d</w>': 9, '   bu b b le gu m</w>': 1, '   ca di ll ac s</w>': 1, '   tr an sp or ta tion</w>': 21, '   wh ee l er</w>': 30, '   f ru m pi sh</w>': 1, '   hou se wi ves</w>': 10, '   r y an</w>': 122, '   en g in e er ed</w>': 10, '   ca t su p</w>': 4, ' pa ss</w>': 5, '   vi et</w>': 6, '   na m</w>': 26, '   e le p han ts</w>': 16, '   sa pped</w>': 1, '   un en ter pri sing</w>': 1, '   ca stra tion</w>': 2, '   star v ation</w>': 4, '   lu p i</w>': 1, ' loo p o</w>': 1, '   har per</w>': 16, '   po si es</w>': 1, '   hon e y bu n ch</w>': 1, '   ev a por a tes</w>': 1, '   ar gu ed</w>': 7, '   sp ar r ow f art</w>': 1, '   car tri dge</w>': 3, '   fu l min ate</w>': 1, '   sc at</w>': 1, ' de ck er</w>': 1, '   bu n ks</w>': 7, '   vi o let</w>': 41, '   s ani t ary</w>': 3, '   pl ow ed</w>': 2, '   plan ted</w>': 32, '   tur ni ps</w>': 2, '   ca b ba g es</w>': 2, '   f er ti le</w>': 7, '   s la ying</w>': 3, '   y u go s la vi a</w>': 7, '   m h ra vi tch</w>': 6, '   sle w</w>': 9, '   f ar e we lls</w>': 2, '   te sta men ts</w>': 2, ' e di t ori a li z ing</w>': 1, '   ho spi ta li ty</w>': 16, ' ma j or</w>': 3, '   c y ani de</w>': 6, '   cap su les</w>': 4, '   k ra u ts</w>': 17, '   k ra u t</w>': 10, '   ch ok ed</w>': 5, '   a vo ca do</w>': 4, '   car ou sing</w>': 1, '   ha i ri er</w>': 2, '   u ck</w>': 1, '   u tter</w>': 11, ' f ought</w>': 2, '   pe er ing</w>': 3, '   mer ce d es</w>': 16, '   e sc or ted</w>': 4, '   mo t or c y c list s</w>': 2, '   ha te ful</w>': 5, '   si e g fri ed</w>': 5, '   k on i g s wa ld</w>': 1, '   par ac hu ted</w>': 2, '   gu er ri ll a</w>': 5, '   ra i ds</w>': 3, '   f le e ing</w>': 3, '   ab ra ha m</w>': 12, '   bri ga de</w>': 9, '   s ni per</w>': 10, '   pi ca du r a</w>': 1, '   der r ing</w>': 1, '   te en y</w>': 5, ' we en y</w>': 1, '   st re tch er</w>': 6, ' be ar er</w>': 1, ' har old</w>': 1, '   pa tr on</w>': 5, '   spi cy</w>': 4, '   in su l ts</w>': 9, '   e du ca ting</w>': 2, '   wor shi ps</w>': 13, '   war es</w>': 3, '   de sha bi ll e</w>': 1, '   wor shi per</w>': 1, ' h er b</w>': 1, '   wan da</w>': 7, '   nor ber t</w>': 14, '   mar sha l</w>': 27, '   su per b ow l</w>': 4, '   app a lled</w>': 4, '   ba y on et</w>': 8, '   gla m our</w>': 11, '   h un ts</w>': 5, '   v ani sh er</w>': 1, '   in ar ti cu late</w>': 1, '   gr ab s</w>': 19, '   vo ca bu l ary</w>': 7, '   re ar ing</w>': 4, '   no thin g ne ss</w>': 10, '   ba lled</w>': 6, '   inter ac tion</w>': 4, '   si ll in ess</w>': 1, '   la ven der</w>': 4, '   s an g</w>': 26, '   s no o z ed</w>': 1, '   h y d ra u li c</w>': 5, ' 3 6 </w>': 3, '   e di ble</w>': 1, '   pen u mb r a</w>': 1, '   ad ore</w>': 18, ' vi si ting</w>': 2, '   f ea ther ed</w>': 2, '   te sti mon i al s</w>': 2, '   b re w er y</w>': 9, '   bro om sti ck</w>': 5, '   att en u a ted</w>': 1, '   sha tt er ed</w>': 9, '   ser vi le</w>': 1, '   s ma sh</w>': 31, '   ch er i sh ed</w>': 5, '   sc ro g</w>': 1, ' part</w>': 8, '   me ss es</w>': 5, '   h er o i s m</w>': 4, '   in ju ri es</w>': 12, '   ex c ru ci a ting</w>': 3, '   car ho p</w>': 2, '   sen si ti z ed</w>': 1, ' al ong</w>': 4, '   sc ro g ged</w>': 4, '   co y</w>': 5, '   ten der ne ss</w>': 5, ' brea k fa st</w>': 6, '   c ro co di le</w>': 11, '   su n n y side</w>': 1, '   fr ying</w>': 8, '   pre ju di ce</w>': 11, '   la plan der</w>': 1, '   de l</w>': 64, '   fu e gi an</w>': 1, '   pen gu ins</w>': 12, '   nu tri ment</w>': 1, '   b lo a ts</w>': 1, '   ne w t ons</w>': 1, '   ba kes</w>': 1, '   ra di at ors</w>': 1, '   he ar t less</w>': 10, ' o ce ans</w>': 1, '   ki pp er ed</w>': 1, '   on i on</w>': 10, '   i dea lly</w>': 7, '   d ev on shi re</w>': 1, '   cra mm ed</w>': 5, '   ro ds</w>': 3, '   un s n ar l</w>': 1, '   ma s sa ge</w>': 17, '   app ro ves</w>': 2, '   star k</w>': 8, '   t wi tting</w>': 1, '   w ea k lin gs</w>': 1, '   g ha st ly</w>': 11, '   un a m bi gu ous</w>': 1, ' by</w>': 55, ' fir es</w>': 2, ' su ffer ing</w>': 2, '   in gra ti tu de</w>': 2, '   chan c re</w>': 1, '   bea u</w>': 9, '   b lu h</w>': 1, ' en jo y</w>': 2, '   un re he ar sed</w>': 2, ' sur gi cal</w>': 1, ' na tu ra l</w>': 2, '   ad ju sted</w>': 9, '   o ver he ar ing</w>': 4, '   pe lt</w>': 2, '   au tom o bi le</w>': 12, '   bu ff al o</w>': 27, '   bu mp ers</w>': 1, '   sp r in g bo k</w>': 1, '   or y x</w>': 1, '   ge m s bo k</w>': 1, '   ga z e ll e</w>': 5, ' ha m bur ger</w>': 1, '   cl ow ni sh</w>': 1, '   hea l er</w>': 4, '   bu mb le be e</w>': 1, '   sa b er</w>': 2, ' to o th ed</w>': 4, ' re spe ct</w>': 9, '   cl ow ning</w>': 3, '   ob so le te</w>': 12, '   car ni v or es</w>': 4, '   char i s ma</w>': 3, '   tra gi ca lly</w>': 4, '   pre po st er ou s ly</w>': 1, '   a ven ged</w>': 3, '   qui x o te</w>': 8, '   bo t ani st</w>': 1, '   1 8 9 3 </w>': 1, '   ex pl or ed</w>': 5, '   po ck e t k ni fe</w>': 1, '   cl ever ly</w>': 1, '   see p ing</w>': 2, '   lo be</w>': 8, ' ob so le te</w>': 1, ' cl own</w>': 1, '   lu cre ti a</w>': 6, '   bor gi a</w>': 2, '   po ta ssi u m</w>': 3, '   hu mi d or</w>': 2, '   t rea ch er ous</w>': 5, '   spi ttle</w>': 1, '   in war d ly</w>': 3, '   b al der da sh</w>': 2, '   spe w</w>': 5, '   pri mor di al</w>': 4, '   bu tch ers</w>': 6, '   ki te</w>': 8, '   ter r or</w>': 23, '   l ou d ly</w>': 4, '   fo ssi l</w>': 10, '   h or se sho e</w>': 4, '   ear d ru ms</w>': 1, '   de ar i e</w>': 1, '   f la v or ed</w>': 3, '   to o th pi cks</w>': 4, '   si e ge</w>': 5, '   h ow i t z ers</w>': 1, '   de mo li sh</w>': 1, '   cor p ses</w>': 15, '   c ri pp les</w>': 3, '   hu mor less</w>': 1, '   hi ther</w>': 4, '   y on</w>': 9, '   qu ack</w>': 3, '   e di t ori al</w>': 9, '   we e</w>': 35, ' wee ing</w>': 1, '   p ac qu al in in ch ee w a</w>': 3, '   a ma z on</w>': 4, '   ph ar m ac o po ei as</w>': 1, '   cu r are</w>': 2, '   e p he dr ine</w>': 2, '   cu r es</w>': 4, ' cou g ar</w>': 2, '   f an g</w>': 2, '   chi lls</w>': 6, '   f ev ers</w>': 3, '   s w ea ts</w>': 4, '   c ru d</w>': 1, '   t re e to p</w>': 1, '   bi tt en</w>': 14, '   a m bu la t ory</w>': 4, '   app u r ten an ce</w>': 1, '   pro mp tly</w>': 4, '   ta x i ca b</w>': 4, '   w s</w>': 1, '   un lo cked</w>': 9, '   star t l ed</w>': 15, '   oo op s</w>': 9, ' c ri p es</w>': 1, '   m ack er el</w>': 5, '   k ni tting</w>': 2, '   con ked</w>': 4, '   pu l mo t ors</w>': 1, ' ke st en ba u m</w>': 2, '   ke st en ba u m</w>': 3, '   un an n oun c ed</w>': 5, ' al co ho li c s</w>': 1, '   sa f ar i</w>': 4, '   j a</w>': 17, '   so oo oo oo oo o</w>': 1, '   e j ac u la tion</w>': 2, '   tru m pe ting</w>': 1, '   ro ar ing</w>': 4, '   cor n ers</w>': 19, '   fi re place</w>': 5, '   hel me ts</w>': 5, '   sta m pe ding</w>': 1, ' bu ff al o</w>': 1, '   p an sy</w>': 5, '   f ru i t ca ke</w>': 12, '   bu ll ll ll ll ll ll l ll</w>': 1, ' di ck ey</w>': 1, '   gr ow nu ps</w>': 5, '   pr ou de st</w>': 3, ' kid</w>': 5, '   in k ling</w>': 5, '   ja gu ars</w>': 3, '   bu g ger</w>': 14, '   shi ver s</w>': 2, '   g ro ans</w>': 2, '   cha tter</w>': 2, '   f lu sh ed</w>': 7, '   a a a a a a a a a a a a a ah</w>': 1, '   ca t fo od</w>': 1, '   que er</w>': 34, '   o ma h a</w>': 9, '   th u mb na il</w>': 1, '   mon u ment</w>': 10, '   r h in o cer o s</w>': 3, '   la sted</w>': 18, '   re spe c t ful</w>': 3, '   re spe c ts</w>': 13, '   s cou ting</w>': 7, '   ch u mp</w>': 18, '   du cked</w>': 2, '   s k ins</w>': 13, '   fu r</w>': 21, '   w re st le</w>': 12, '   w re st l ed</w>': 6, '   v an der bi lt</w>': 7, '   le hi gh</w>': 2, '   w re st ling</w>': 19, ' won</w>': 12, ' y a m mer ing</w>': 1, '   ta un ting</w>': 3, '   g l ori ou s ly</w>': 2, '   c ru ci fi ed</w>': 3, '   pa ged</w>': 2, ' since</w>': 6, '   cer ta in ty</w>': 11, ' pro mi se</w>': 9, '   x ke</w>': 1, '   wi l der</w>': 19, '   ca l m er</w>': 2, '   min k</w>': 7, '   li gh the ar te d ly</w>': 1, '   ad ver ti se</w>': 5, '   cra w ls</w>': 5, '   t ro pi cal</w>': 12, '   an ti hi sta min es</w>': 2, '   gen t l en ess</w>': 2, ' wan da</w>': 2, '   ven i son</w>': 6, '   st ri p ed</w>': 6, '   car i b ou </w>': 1, '   r ou gh hou ses</w>': 1, ' sho ve</w>': 1, '   s la u gh ter hou ses</w>': 1, '   du bu qu e</w>': 3, '   b en der</w>': 6, '   h or se sho es</w>': 4, '   mu s cu la ture</w>': 2, '   ca lf</w>': 15, '   re bor n</w>': 6, '   pi the can th ro p us</w>': 1, '   er e c tu s</w>': 1, '   au stra lo pi the c us</w>': 1, '   sin an th ro p us</w>': 1, '   pe k en s is</w>': 1, '   ne an der th al ers</w>': 1, '   p ee p sh ows</w>': 1, ' li f ting</w>': 1, '   n ou ri sh ing</w>': 1, ' gre en</w>': 7, '   do om s day</w>': 6, '   me te or</w>': 13, '   ther e on</w>': 1, ' dro pp ing</w>': 1, '   p ho o ey</w>': 6, '   s w ea t su it</w>': 1, '   i sc ar i o t</w>': 1, '   c ro ss b on es</w>': 1, '   h un ch ed</w>': 2, '   shu ff le bo ard</w>': 12, ' go ld</w>': 4, ' p on ti us</w>': 1, '   pi late</w>': 2, '   p on ti us</w>': 1, '   ri pp er</w>': 37, '   car ro ll</w>': 5, '   hi t l er</w>': 58, ' body</w>': 24, '   a mer i c ard</w>': 3, '   bea tri ce</w>': 4, '   re e ms</w>': 1, '   di sa pp e ars</w>': 17, '   un re por ted</w>': 3, '   inter vi e w ed</w>': 15, '   rea m</w>': 3, ' co cked</w>': 3, '   re sp on ded</w>': 7, '   a p b</w>': 5, '   fe li ce</w>': 2, '   en tra p ment</w>': 9, '   d ong</w>': 7, '   ha ss l ed</w>': 3, '   af fi li ation</w>': 2, '   ti pp ing</w>': 13, ' fi l m</w>': 3, '   7 7 7 </w>': 1, '   v ine</w>': 9, '   4 6 3 </w>': 1, ' 5 6 7 1 </w>': 1, ' per son al s</w>': 1, '   jo an n e</w>': 16, '   to d</w>': 21, '   dis ci pl in es</w>': 3, '   d or n</w>': 7, '   pa v on ine</w>': 2, '   st ri pe</w>': 8, '   t ea ch es</w>': 18, '   t int</w>': 1, '   de sig ner</w>': 5, '   v ri es</w>': 4, '   ni k i</w>': 12, ' pi mp</w>': 1, '   g ran vi ll e</w>': 5, ' ma ga z in es</w>': 1, '   sc an ty</w>': 1, '   e ddy</w>': 24, ' f ar re ll</w>': 2, ' pe ci al ty</w>': 1, '   d ac h sh un d</w>': 1, '   pre de st in ation</w>': 1, '   om ni s ci ent</w>': 2, '   r on a</w>': 4, '   re si sted</w>': 3, '   per se ver an ce</w>': 3, ' u li p</w>': 2, '   un con di tion al</w>': 5, '   e le ction</w>': 44, '   a ton ed</w>': 1, '   de p ra vi ty</w>': 3, '   an a gra m</w>': 1, '   can ons</w>': 1, '   d or t</w>': 1, '   ca l v in i st</w>': 1, '   den om in ation</w>': 2, '   what ja m ac i ll it</w>': 1, '   ra t an</w>': 11, '   car son</w>': 21, '   uni mp ort ant</w>': 5, '   su c</w>': 1, '   su per mar k et</w>': 8, ' u cking</w>': 1, ' 7 0 0</w>': 2, '   ra ma da</w>': 6, '   su lli v an</w>': 20, '   mi k ey</w>': 53, '   ac t re ss es</w>': 12, '   ar ran g es</w>': 4, '   po king</w>': 15, ' ex tr a</w>': 1, ' la ve</w>': 3, '   out right</w>': 6, ' nobody</w>': 20, ' har d c ore</w>': 1, '   por no gra p hi c</w>': 3, '   e m be z z le ment</w>': 6, '   p an t l ind</w>': 1, '   min i mu m</w>': 31, ' 7 5 0</w>': 2, '   re be lli ous</w>': 2, '   do g shit</w>': 4, ' du c tive</w>': 1, ' la tive</w>': 1, '   rea son ing</w>': 4, '   w es</w>': 51, '   re f er ra l</w>': 1, '   com pla ins</w>': 3, '   mar sh a</w>': 4, '   re c rea tion</w>': 8, '   k no t t</w>': 4, '   ber ry</w>': 8, '   sh or ts</w>': 17, '   j i s m</w>': 1, '   ca st ing</w>': 12, '   ru ck er</w>': 1, '   app li ca ble</w>': 3, '   po on</w>': 3, '   ri ve ts</w>': 3, '   out do ors</w>': 12, ' ri gh te ous</w>': 5, '   vi sh un d</w>': 1, '   sta ked</w>': 9, '   re hi re</w>': 1, '   under age</w>': 8, '   pre pa id</w>': 1, '   sta king</w>': 4, '   s sh</w>': 14, '   sta ke out</w>': 2, '   pa t ti c a</w>': 1, '   at ti c a</w>': 4, '   im pro ved</w>': 14, '   u pp i ty</w>': 5, '   shi the el</w>': 1, '   l ou i se</w>': 73, '   r h y mes</w>': 9, '   ne w s ca st</w>': 1, '   c r on ki te</w>': 5, '   e c</w>': 2, '   po l i</w>': 2, '   sc i</w>': 4, '   cha s en</w>': 3, '   offi ci ous</w>': 2, '   re si st ing</w>': 4, '   5 4 5 </w>': 1, '   sho ve l</w>': 25, '   re se mb lan ce</w>': 11, '   tr an sp lan ting</w>': 1, '   re gi stra tion</w>': 14, '   7 0</w>': 16, ' mi le</w>': 10, '   bar le y</w>': 5, '   har ri son</w>': 14, '   in vo i ces</w>': 2, '   tru cking</w>': 5, '   h en der son</w>': 7, '   e di th</w>': 7, '   t un ing</w>': 3, '   ver i ly</w>': 1, '   i li a d</w>': 1, '   a ph ro di te</w>': 4, '   con su m ma te</w>': 2, '   ma u de</w>': 25, ' d ying</w>': 3, '   char d in</w>': 2, '   mar j ori e</w>': 5, '   gi ddy</w>': 4, '   ei gh ti e th</w>': 2, '   e c sta ti ca lly</w>': 1, '   lo ve li er</w>': 4, '   re pre sen ta tions</w>': 2, '   de ar ly</w>': 7, '   gra du ally</w>': 5, '   fa d es</w>': 8, '   k o da k</w>': 1, '   f ra mes</w>': 12, '   a st r on au t</w>': 7, '   ma ge ll an</w>': 3, '   ga z ers</w>': 1, '   p ea s ant</w>': 16, ' ga z ers</w>': 1, '   ra king</w>': 1, '   pu ri fi es</w>': 1, '   sc ru pu l ou s ly</w>': 1, '   l un che on</w>': 7, '   can dle</w>': 17, '   y o de ling</w>': 2, '   car t wh ee ls</w>': 2, '   s om er sa u l ts</w>': 1, '   com m uni ca te</w>': 31, '   sp re ck l ed</w>': 1, '   un coun ta ble</w>': 1, '   bl ack ne ss</w>': 4, '   co s mo s</w>': 2, '   ma im ing</w>': 2, '   po p gu n</w>': 1, '   pe ll et</w>': 2, '   per si a</w>': 3, '   ba z a ar</w>': 6, '   ma x i m</w>': 1, '   wi se st</w>': 4, '   tru est</w>': 2, '   in st ru c tive</w>': 4, '   pe er ed</w>': 1, '   ma g ni f ying</w>': 3, '   con fu ci us</w>': 2, '   pu ff</w>': 33, '   gla u c us</w>': 5, '   me l ting</w>': 6, ' think</w>': 32, '   re pl en i sh</w>': 1, '   re vo l ts</w>': 2, '   k in g d om s</w>': 5, '   par a so ls</w>': 1, '   chi de</w>': 2, '   o pp o si tion</w>': 6, '   li cor ice</w>': 4, '   nu tri tion al</w>': 1, ' din ner</w>': 6, '   li que u r</w>': 1, '   ven us</w>': 7, '   comp le tion</w>': 2, '   un fu l fi lled</w>': 3, '   tr an sp l ant</w>': 8, '   su ff o ca ting</w>': 12, '   s mo g</w>': 6, '   su n f l ower</w>': 2, '   st er ling</w>': 8, '   t ea po t</w>': 2, '   gre et</w>': 12, '   s wee ten</w>': 1, '   o at</w>': 3, '   su b ways</w>': 3, '   co lo g n e</w>': 5, '   che st nu ts</w>': 3, '   in fa tu a ted</w>': 5, ' o d ori fi c s</w>': 1, '   ti t l ed</w>': 1, ' ra in b ow</w>': 1, ' ra pe</w>': 2, '   de pi ction</w>': 3, '   le da</w>': 6, '   re f re sh ed</w>': 1, '   cont ou rs</w>': 1, '   di sa pp ro ve</w>': 5, '   y are</w>': 1, '   fa shi on ed</w>': 18, '   in te gra l</w>': 3, ' ow n er ship</w>': 1, '   w r on ging</w>': 1, '   ad di tions</w>': 2, '   v ar i ation</w>': 7, '   ye ar se</w>': 1, '   cu r ves</w>': 8, '   bu ri al s</w>': 1, '   bi r th s</w>': 3, '   e mp ha s is</w>': 8, '   ac cen tu ate</w>': 3, ' miss</w>': 9, '   sh r ou d</w>': 3, '   1 8 9 0</w>': 2, '   we ds</w>': 1, '   co l ds</w>': 8, '   pr one</w>': 3, '   be e ts</w>': 5, '   se ine</w>': 3, '   cu r r en ts</w>': 5, ' en f lu ence</w>': 1, ' ar g ent</w>': 1, '   har i</w>': 3, ' k ar i</w>': 2, '   vi ri le</w>': 3, '   char ma ine</w>': 1, '   ca ll u ses</w>': 3, '   t ar ni sh ed</w>': 1, '   ban j o</w>': 5, ' su n sh ine</w>': 1, '   d ore</w>': 2, '   sta in</w>': 12, '   re la ting</w>': 7, '   re lu c t an ce</w>': 3, '   de tri men tal</w>': 3, ' an al y ti cal</w>': 1, '   h in der</w>': 1, '   cla ss ma tes</w>': 9, '   i ll u min a tive</w>': 1, '   f on d ne ss</w>': 5, '   de mo li tions</w>': 4, '   in di ca tive</w>': 1, ' de st ru c tive</w>': 1, '   u r g es</w>': 8, '   a li en ation</w>': 2, '   co p ed</w>': 1, ' fu n</w>': 9, '   en jo y ment</w>': 7, '   fu l fi lling</w>': 8, '   j un k y ar ds</w>': 1, '   fa sc in ation</w>': 4, ' b en e f it</w>': 1, '   sta g es</w>': 9, '   su i ci d es</w>': 5, '   com mi e</w>': 8, ' dis gu st ing</w>': 1, '   s ca l p</w>': 15, '   pri v a tes</w>': 3, '   s ou ven ir s</w>': 9, '   s qu ir t</w>': 10, '   p ac ho i e</w>': 1, '   z at</w>': 8, '   go lly</w>': 17, ' t at</w>': 8, ' thr ow</w>': 3, ' m ac </w>': 3, '   whi z z ing</w>': 2, '   z o t</w>': 1, '   me di c</w>': 5, '   un con s ci ou s ne ss</w>': 2, '   cha l ked</w>': 1, '   j er ri es</w>': 2, '   war time</w>': 4, '   con de m na tion</w>': 1, '   ba ll point</w>': 2, '   pe ti tion er</w>': 2, ' sir</w>': 18, '   sa ves</w>': 16, '   a e s the ti c</w>': 2, '   da sh bo ard</w>': 5, '   de la ys</w>': 5, '   ki r st y</w>': 26, '   tri ps</w>': 23, '   sur real</w>': 5, '   ni gh t mar i sh</w>': 1, '   ma la hi de</w>': 3, ' sa w s</w>': 1, '   ti ff any</w>': 2, '   con fu si ons</w>': 1, '   cen o bi tes</w>': 4, '   de mon s</w>': 33, '   con v ey</w>': 9, ' v al u e</w>': 1, '   ni x on</w>': 61, '   hou din i</w>': 3, '   my sti c</w>': 1, '   in si gh ts</w>': 8, '   la tch ed</w>': 2, '   sa m ma el</w>': 1, '   de so late</w>': 1, '   n er ga l</w>': 1, '   be l ts</w>': 5, '   en ti ty</w>': 6, '   sha v in gs</w>': 2, '   li z</w>': 39, '   h int</w>': 39, '   har b in ger</w>': 1, '   ru pre ch t</w>': 1, '   k ro en en</w>': 1, '   th u le</w>': 1, '   o c cu lt</w>': 5, '   whi t man</w>': 10, ' par an or ma l</w>': 1, '   par ab nor ma l</w>': 1, '   ru ins</w>': 17, '   tr on d ha m</w>': 1, '   ab be y</w>': 1, '   le y</w>': 2, ' pro fe ss or</w>': 1, '   ab h or</w>': 2, '   a mon g st</w>': 17, '   to p side</w>': 10, '   ro man o v s</w>': 1, '   1 9 1 6 </w>': 2, '   c lu b bed</w>': 2, '   ca stra ted</w>': 3, '   ra sp u t in</w>': 10, '   g ri g ory</w>': 3, '   ye fi mo vi ch</w>': 1, '   s lu mb ers</w>': 1, '   in fe c ts</w>': 3, '   t re v or</w>': 35, ' gu est</w>': 6, '   f und ed</w>': 6, '   a vi g n on</w>': 1, '   re li qu ary</w>': 1, '   s mu g g l ed</w>': 4, '   o ver z ea l ous</w>': 1, '   cu ra tor</w>': 6, '   war ds</w>': 5, '   di on y si us</w>': 1, '   a er op a gi te</w>': 1, ' pu re</w>': 1, '   1 9 4 5 </w>': 7, '   1 9 5 8 </w>': 2, '   a do lf</w>': 15, '   ab sen ti a</w>': 1, '   lu c i</w>': 1, '   ten e bra e</w>': 1, '   v in ci un t</w>': 1, '   po s se ss es</w>': 1, '   fr on tal</w>': 5, ' uni qu e</w>': 3, ' a be</w>': 1, '   sa pi en</w>': 2, '   i c th y o</w>': 1, '   sa pi en s</w>': 2, '   1 8 6 5 </w>': 1, '   w oo zy</w>': 3, '   fa kes</w>': 3, ' my ers</w>': 1, '   te en s</w>': 2, '   he ll boy</w>': 1, '   un su per vi sed</w>': 2, '   bro om</w>': 24, '   an un g</w>': 2, ' u n</w>': 7, ' ra ma</w>': 2, '   z e pp o</w>': 1, '   lu ll ab y</w>': 4, '   u sh er ed</w>': 1, '   in sta lled</w>': 4, '   p ins</w>': 10, ' bo om</w>': 8, ' ju mb o</w>': 3, ' 6 5 </w>': 6, '   ar i se</w>': 7, ' nee dy</w>': 1, ' pa m ca kes</w>': 2, '   p un c tu a li ty</w>': 2, '   mm m h</w>': 1, ' po l o</w>': 1, '   lo ca t ors</w>': 1, '   de fe c ts</w>': 3, '   un sto pp able</w>': 5, '   pre ca u tions</w>': 14, ' li z</w>': 2, ' ho les</w>': 4, '   so d</w>': 17, '   c has</w>': 2, '   sig ni fi can ce</w>': 11, '   plea sur es</w>': 10, '   ex pl or ers</w>': 7, '   an ge ls</w>': 35, '   la ment</w>': 2, '   con figu ra tion</w>': 5, '   g ri m</w>': 7, '   en du red</w>': 8, '   in di vi si ble</w>': 2, '   t ore</w>': 42, '   ho o ks</w>': 16, '   we d ded</w>': 3, '   b li ss</w>': 9, '   so a ked</w>': 10, '   hi dea way</w>': 3, ' ti ght</w>': 5, '   co in t rea u</w>': 2, '   k ev in</w>': 68, '   ban ni ster</w>': 3, '   qui b ble</w>': 3, '   a li son</w>': 9, '   b our ge o is</w>': 12, '   son i c</w>': 4, '   bar r y town</w>': 5, ' bar r y town</w>': 1, '   s d m</w>': 1, '   tur ner</w>': 21, '   al bu ms</w>': 2, '   1 1 0</w>': 3, '   b ran son</w>': 3, '   spe ctor</w>': 2, ' fuck ers</w>': 4, '   k as d an</w>': 1, ' bi ll</w>': 13, '   bu ck town</w>': 1, '   br ow s er</w>': 2, '   r ac ks</w>': 4, '   st ee ly</w>': 2, '   com mi t men ts</w>': 3, '   ar ro g an ce</w>': 10, '   b on ers</w>': 1, ' beli ev able</w>': 3, '   b on ing</w>': 4, '   la sa ll e</w>': 3, '   s oun d tr ack</w>': 3, '   c in e ma ti c</w>': 2, ' sc re en</w>': 5, '   f our te en th</w>': 7, '   st or ing</w>': 4, '   l int</w>': 8, ' sta tu s</w>': 2, '   cla ssi c s</w>': 7, '   ga ye</w>': 11, ' a ir ba g</w>': 1, '   ra di o head</w>': 1, ' th und er</w>': 3, '   sp r in g st e en</w>': 4, '   ni r v an a</w>': 3, '   n ever mind</w>': 12, '   e h h</w>': 7, ' j ani e</w>': 1, '   cla sh</w>': 7, '   s ni ck er ing</w>': 2, ' bu mp ing</w>': 1, '   bu mb le fuck</w>': 1, '   pu la s k i</w>': 3, '   of f en ding</w>': 5, '   go of ing</w>': 3, '   per pe tra ted</w>': 3, ' 8 0</w>': 7, '   su b questi on</w>': 1, '   cra pp y</w>': 9, '   sti mu la tor</w>': 1, '   re gi me</w>': 4, '   pre f er ence</w>': 9, '   lu pe</w>': 3, '   du m ba ss es</w>': 1, '   tur n ta b les</w>': 2, '   bu n n y men</w>': 1, '   t ack y</w>': 5, ' lea der</w>': 2, '   cu r ve</w>': 16, '   mo ss</w>': 35, '   mo ss y</w>': 2, '   fu r ry</w>': 10, '   g ab les</w>': 1, '   con da</w>': 1, '   t or on to</w>': 8, '   mi tch</w>': 76, '   r y der</w>': 2, '   shi i te</w>': 1, '   c ri pp ling</w>': 2, '   sh op li f t ers</w>': 1, '   vi si on ary</w>': 5, '   m c cl aren</w>': 1, '   b li t z</w>': 3, ' be ll a</w>': 1, '   lu go s i</w>': 19, '   ba u ha us</w>': 1, '   ha h a</w>': 3, '   de e j ay</w>': 5, '   wi z ar ds</w>': 2, '   a m bi ent</w>': 2, '   sta x</w>': 1, ' dan ce</w>': 5, '   u p se tter</w>': 1, ' sc ra tch</w>': 1, '   fun k a de li c</w>': 2, ' shi p bu il ding</w>': 1, '   co st e llo</w>': 22, '   imp or t</w>': 11, ' sp ac ed</w>': 2, '   do d ger</w>': 4, '   bur ri to</w>': 7, '   b ack dro p</w>': 2, '   car o l ine</w>': 5, '   for t is</w>': 1, ' does</w>': 14, ' mean</w>': 7, '   den se</w>': 12, '   mi sts</w>': 2, '   su n ni er</w>': 1, '   sp ar ki er</w>': 1, '   ju d ge men ts</w>': 1, '   con so la tion</w>': 5, '   un n er ving</w>': 2, '   re ha sh</w>': 4, '   v in ce</w>': 58, ' ab ra ha m</w>': 1, '   tal en ts</w>': 10, '   tru x</w>': 1, '   sh er y l</w>': 3, '   c r ow i sh</w>': 1, '   ab sor bed</w>': 4, '   ch er y l</w>': 2, '   la d d</w>': 3, '   com fi ts</w>': 1, '   te sta ment</w>': 7, ' or ning</w>': 3, '   pa tch ou l i</w>': 2, '   jo k ey</w>': 1, '   ac tion able</w>': 1, '   fun ni ly</w>': 1, '   pro po sa ls</w>': 4, '   per su ad able</w>': 1, '   su l k</w>': 2, '   e mb ar ra ss ment</w>': 17, '   sti ck er</w>': 5, '   an no y ed</w>': 6, ' tri u mp h ant</w>': 2, '   lo ca lly</w>': 3, '   k or e t z k y</w>': 3, '   un tur ned</w>': 2, '   ha ir st y les</w>': 1, '   ar t ful</w>': 4, '   de e ja y ed</w>': 1, ' shi r ts</w>': 3, '   lo tt er y</w>': 12, '   ja ma i c a</w>': 4, '   lo b</w>': 3, '   stu b</w>': 3, '   re ci pro ca te</w>': 2, '   mm n n</w>': 1, '   star bu ck</w>': 1, '   ob tu se</w>': 4, '   l ow e</w>': 7, '   bor e d om s</w>': 1, '   c ds</w>': 10, '   in f re qu en tly</w>': 2, ' ma v is</w>': 2, ' si ss y boy</w>': 1, '   si ss y boy</w>': 1, '   mor ti fi ed</w>': 1, ' sho e</w>': 4, ' we ar ing</w>': 5, ' wa tch ing</w>': 4, '   un pre par ed</w>': 7, ' mo te l</w>': 1, '   p ha ir</w>': 1, '   ad vo ca te</w>': 5, '   d y l an</w>': 37, '   hea d ph on es</w>': 4, '   cl ar i f y</w>': 14, '   chi l di sh</w>': 22, '   t ack l ed</w>': 2, '   re or g ani z ation</w>': 2, '   i s ra el is</w>': 6, '   pa le st in i ans</w>': 3, '   gar fun k el</w>': 4, ' cen ter ed</w>': 3, '   j o</w>': 11, '   ha ting</w>': 16, '   cre ma t ori u m</w>': 3, '   pro c ee ded</w>': 4, '   sy m pa th i z ed</w>': 1, '   sch mu ck</w>': 27, '   de li c ac y</w>': 3, '   whi s ke ys</w>': 2, ' ac tly</w>': 1, '   con for mi st</w>': 1, '   c ru mb l ers</w>': 6, '   b la st ers</w>': 1, '   un att ac h ed</w>': 2, ' b one</w>': 3, '   di vi ding</w>': 2, ' pa t sy</w>': 1, '   cl ine</w>': 3, '   ba th ro be</w>': 1, '   o t is</w>': 69, '   re d ding</w>': 2, '   po i son ous</w>': 11, '   ac hi ev e ment</w>': 12, '   du b bed</w>': 1, '   su b ma st ers</w>': 1, ' mo th ers</w>': 1, '   b in</w>': 13, '   fri s k</w>': 7, '   s la mm ing</w>': 5, '   ni c o</w>': 1, '   en o</w>': 3, '   si gu e</w>': 2, '   sp u t ni k</w>': 1, '   ga in s b our g</w>': 1, '   r y u ch i</w>': 1, '   sa k a mo to</w>': 1, '   sy d</w>': 2, '   con gen i a li ty</w>': 1, '   ta u p in</w>': 14, '   mor an</w>': 5, '   4 1 </w>': 11, '   ki d ell</w>': 1, '   dre w</w>': 72, '   cla y more</w>': 8, '   e pe e</w>': 1, ' to ys</w>': 1, '   un as se mb l ed</w>': 1, '   m ac ab re</w>': 2, '   du s an</w>': 1, '   p ea s an ts</w>': 8, '   cha t ea u</w>': 10, '   be lli es</w>': 5, '   re use</w>': 1, '   sy m me try</w>': 4, ' di ed</w>': 3, '   t y ke</w>': 2, '   ho pe le ss ly</w>': 5, '   mu se u ms</w>': 6, '   un su al</w>': 1, '   mar y land</w>': 8, '   tom or r ows</w>': 2, '   b oun ded</w>': 1, '   in h er i t an ce</w>': 11, '   fin a li ty</w>': 1, '   br en na</w>': 5, '   e mp t y ne ss</w>': 1, '   bor der ed</w>': 1, '   ex chan ging</w>': 3, '   i a m bi c</w>': 1, '   ta me ter</w>': 1, '   in no cen ts</w>': 6, ' 3 1 </w>': 3, '   di st ru ction</w>': 1, '   m ac le od</w>': 14, '   au st ri a</w>': 13, '   f er ence</w>': 1, '   1 6 7 2 </w>': 1, '   ne ss es</w>': 1, '   w oo d car ver</w>': 1, '   re ci eve</w>': 1, '   con si der ably</w>': 7, '   a qui red</w>': 1, '   imp ort an tly</w>': 10, '   cu mb er some</w>': 2, '   fe l ton</w>': 5, '   hi st ori an</w>': 9, ' m ac le od</w>': 1, '   car tw right</w>': 10, '   du p on t</w>': 3, '   ce me t ar i es</w>': 2, '   ro b b ers</w>': 19, '   t our ing</w>': 5, '   re f le x es</w>': 8, '   d on ke ys</w>': 1, '   ser bi ans</w>': 1, '   for g ers</w>': 1, '   f l un ked</w>': 5, '   ne ice</w>': 2, '   ga ll on</w>': 10, '   with dre w</w>': 5, '   hu man i t ar i an</w>': 7, ' o le</w>': 3, '   bea ver</w>': 5, '   a m bu l ence</w>': 2, '   cha s er</w>': 12, '   ja v el in</w>': 2, '   ce me t ary</w>': 1, '   fa shi on ably</w>': 2, '   s mi th son i an</w>': 5, '   wor sti ck</w>': 1, '   b al ti c</w>': 1, '   fi ling</w>': 13, '   du st ing</w>': 5, '   y a k</w>': 3, ' ow e</w>': 3, '   pi l ed</w>': 7, '   s an c tu s</w>': 1, '   e sp er i tu </w>': 1, ' au di t ori u m</w>': 1, '   no st ru m</w>': 1, ' lu ce at</w>': 1, '   e i</w>': 1, ' et</w>': 5, '   lu x</w>': 1, '   per pe tu a</w>': 1, '   ac er</w>': 1, '   d on a e i</w>': 1, '   con or</w>': 15, '   re sp on si bi li ti es</w>': 28, '   wh e at</w>': 16, '   i tu de</w>': 1, ' kn ow le dge</w>': 2, '   du es</w>': 6, '   fa sc in a ti o on</w>': 1, '   con c ee ded</w>': 1, '   s co t</w>': 2, '   ro mi re z</w>': 7, '   plan ation</w>': 1, '   s me lt</w>': 5, '   mu y</w>': 3, '   bi en</w>': 16, '   im po t ent</w>': 11, '   im per fe ction</w>': 4, '   p uni sh ment</w>': 27, '   a qu i</w>': 5, '   mon st r ous</w>': 13, '   cont in u i ty</w>': 1, '   tra d</w>': 1, '   i tion</w>': 1, '   in ju re</w>': 3, '   in ver ne ss</w>': 2, '   su l t ant</w>': 1, ' d ome sti c</w>': 1, '   ph ea s ant</w>': 4, '   ri c o</w>': 41, '   in v ent</w>': 19, '   ci d</w>': 1, '   sur ve y or</w>': 1, '   al che mi st</w>': 1, '   mar a</w>': 8, '   ba g g ins</w>': 1, '   p ran c ing</w>': 1, '   su ther lan ds</w>': 1, '   h er i ta ge</w>': 2, '   mu let</w>': 2, '   ri v al</w>': 10, '   a mb it</w>': 1, '   f re ck le</w>': 4, ' th rea ten</w>': 1, '   ad dre ss ing</w>': 4, '   b re men</w>': 1, '   mu s ke te er</w>': 9, '   s ar ge ant</w>': 1, '   stan d by</w>': 6, '   t ow n house</w>': 3, '   sig na t ory</w>': 1, '   p ant</w>': 3, '   li ver po o l</w>': 11, ' ans w er</w>': 4, '   bu l gar i an</w>': 2, '   ta st ing</w>': 12, '   k a h n</w>': 3, '   ran go on</w>': 1, ' g ran d father</w>': 3, '   z oo s</w>': 2, '   pi us</w>': 1, '   b ru sh es</w>': 5, '   pa pa l</w>': 5, '   c li mb s</w>': 5, '   d ra w s</w>': 12, '   ne u vi ch</w>': 4, '   c ru sa d es</w>': 6, ' h er e ti c</w>': 1, '   st r on gh old</w>': 4, '   ni t z hi c</w>': 1, '   dro o l</w>': 9, '   la ger</w>': 4, '   qu ar ry</w>': 7, '   la d die</w>': 11, '   w right</w>': 7, '   fa si l</w>': 3, '   ne i gh</w>': 3, '   bor ho od</w>': 2, '   ci ti z en ship</w>': 5, '   sy ri an</w>': 1, '   i man</w>': 2, '   jo gs</w>': 2, '   st even</w>': 49, '   ca t ro ch</w>': 1, '   ti re</w>': 33, '   7 4 </w>': 5, '   mi s spe ll in gs</w>': 1, '   m uni ci pa l</w>': 6, '   ter min al s</w>': 2, '   ra di ca ls</w>': 3, '   s lo p</w>': 6, '   b en sin ger</w>': 7, '   tri bu n e</w>': 16, '   ba tting</w>': 9, '   s we en ey</w>': 15, '   hi l dy</w>': 104, '   j our</w>': 4, '   ca pi ta ine</w>': 1, '   r h y m ing</w>': 1, '   c ow er ing</w>': 2, '   mer v y n</w>': 1, '   he ar t brea king</w>': 2, '   hur ri e d ly</w>': 2, '   su per sti tion</w>': 16, '   hi l da</w>': 6, '   hi c</w>': 3, '   par d ner</w>': 8, '   sp o i ling</w>': 6, '   ts</w>': 4, '   ear th qu a ke</w>': 9, '   a li mon y</w>': 11, '   b en e fi ci ary</w>': 4, ' e le c ted</w>': 3, '   l ow down</w>': 1, '   re li ev es</w>': 1, '   att ab o y</w>': 2, '   ru b b ers</w>': 4, '   cl ou dy</w>': 3, '   b al d w in</w>': 15, '   bro om s</w>': 3, ' ar ra st y</w>': 1, '   har t well</w>': 3, '   s co op ed</w>': 4, '   har t man</w>': 11, '   imp oun ding</w>': 1, '   ob st ru c ting</w>': 4, '   pi mp le</w>': 3, '   e di tor</w>': 48, '   p in k us</w>': 5, '   re pri ev ed</w>': 2, '   ti ck l ed</w>': 7, '   li e b ow i t z</w>': 1, '   co al</w>': 9, ' hi l dy</w>': 2, '   w ed</w>': 3, '   wor l d ly</w>': 5, '   ther e to</w>': 1, '   p li ght</w>': 7, '   t ro th</w>': 1, ' c ro ss ing</w>': 5, '   mi gh t n</w>': 3, '   b ou do ir</w>': 1, '   fi an ce e</w>': 18, '   sh or el and</w>': 1, '   ha g ger ty</w>': 1, '   he i re ss</w>': 1, '   sa u er k ra u t</w>': 1, '   s co o p</w>': 28, '   fif i</w>': 1, '   ran d ell</w>': 1, '   ja ms</w>': 5, '   imp ea ch ment</w>': 4, '   c ri ck et</w>': 9, '   bu tch</w>': 22, ' c ent</w>': 10, '   m ou sing</w>': 1, ' op er ation</w>': 3, '   mor n in g side</w>': 1, '   s ma sh up</w>': 1, ' chi ld</w>': 10, '   pre y</w>': 24, '   li be l ous</w>': 3, '   t y pe wri ter</w>': 18, '   pu ll e ys</w>': 4, '   att a girl</w>': 1, '   na m ing</w>': 8, '   par ks</w>': 9, '   bi ll bo ar ds</w>': 3, '   mu g g</w>': 3, ' car t</w>': 3, ' ac h ing</w>': 1, ' ri d d en</w>': 3, '   gu ar di a</w>': 2, ' he el ers</w>': 1, '   li v in g st one</w>': 2, '   tra pe ze</w>': 3, '   s me ar</w>': 19, '   com for ta b ly</w>': 6, '   me ter</w>': 13, '   c li cking</w>': 8, '   e ge l ho ff er</w>': 4, '   en act</w>': 1, ' en act</w>': 2, '   co o le y</w>': 5, '   mi l d ly</w>': 7, '   tw ins</w>': 35, '   mi r r ors</w>': 27, '   bri de g room</w>': 5, ' he art</w>': 8, '   par a g on</w>': 4, '   an ter oo m</w>': 1, '   bu f fa lo es</w>': 1, '   ber mu da</w>': 10, '   j our na li st</w>': 23, '   p ee king</w>': 7, '   ke y ho les</w>': 2, '   a pe men</w>': 1, '   bu tt in s ki es</w>': 1, '   mo t or men</w>': 1, '   re min i sc ing</w>': 2, '   bl ack j ack</w>': 13, '   bab o on</w>': 4, '   s ven g al i</w>': 3, '   co ver ing</w>': 37, '   je ho so ph at</w>': 1, '   go o</w>': 4, ' go o</w>': 1, '   shu d der ing</w>': 1, '   lo a the some</w>': 1, '   ph on ing</w>': 6, '   k ru p t z k y</w>': 1, '   di mp le</w>': 5, '   di v or ces</w>': 9, ' ea th</w>': 3, '   mu mb l ed</w>': 2, '   r en o</w>': 15, '   s lo t m ac h ine</w>': 1, '   po l ack</w>': 5, ' f our th</w>': 6, '   re pri eve</w>': 11, '   war d en</w>': 16, '   a spe c ts</w>': 8, ' ear l</w>': 1, '   mo lli e</w>': 26, '   ma ll o y</w>': 14, '   ex t in gu i sh</w>': 3, '   h y st er i c s</w>': 5, '   sa l ts</w>': 6, '   fa in ted</w>': 9, '   p in k y</w>': 7, '   en di co t t</w>': 1, '   s mu g g le</w>': 7, '   li on ess</w>': 2, '   ru sh es</w>': 6, ' w o lf</w>': 4, '   a li en i st</w>': 2, '   p sy cho an al y z ing</w>': 1, '   y on son</w>': 1, ' l o</w>': 7, '   ad mi ts</w>': 7, '   sh en k en</w>': 1, '   mer w y n</w>': 1, ' w ea ther</w>': 1, '   sh er i ff s</w>': 18, '   ho o do o</w>': 1, '   a i ding</w>': 9, '   o l s en</w>': 15, '   d on t</w>': 3, '   mo ses</w>': 16, '   af ter thought</w>': 2, '   de men ti a</w>': 6, '   p ra e co x</w>': 1, '   vo tes</w>': 18, '   com m uni sti c</w>': 1, '   s lo g an</w>': 4, ' for m</w>': 2, '   re ds</w>': 7, '   be ssi e</w>': 2, '   de st ro ys</w>': 7, '   c z er ne ck i</w>': 1, '   r he u ma ti c</w>': 1, '   un c les</w>': 7, '   de pu ti es</w>': 16, '   per f</w>': 1, '   fa ti gu ed</w>': 3, '   4 3 </w>': 9, '   ro ll in</w>': 10, '   pe lu so</w>': 1, '   e di t ori al s</w>': 2, '   sc ru b</w>': 13, '   li tt l er</w>': 2, '   f oo ey</w>': 1, '   qui ver</w>': 2, '   ho o ey</w>': 5, '   h y en a</w>': 1, '   fri s ked</w>': 1, '   ha ll ow e</w>': 1, ' en</w>': 6, ' k in gs</w>': 1, '   af fi x es</w>': 1, '   tra ins</w>': 11, '   d ra in pi pe</w>': 1, '   wa kin</w>': 4, '   bo l sh ev i k</w>': 9, '   an ar chi st</w>': 3, '   ne ar er</w>': 15, ' pro du ction</w>': 2, '   in s ani ty</w>': 19, '   s ea l er</w>': 3, '   du ck sh oo ting</w>': 2, '   st r en u ous</w>': 2, '   p in o ch le</w>': 4, '   o ver p ow er ed</w>': 1, '   m c clo s k y</w>': 2, '   a li en i sts</w>': 1, '   go ver n men ts</w>': 8, '   b al my</w>': 2, '   ne g le c ts</w>': 1, '   sch war t z</w>': 33, '   ev ans</w>': 7, '   ro se hi ll</w>': 2, '   pe t ro l</w>': 3, '   con ce de</w>': 4, '   go o g li es</w>': 2, '   go o g ly</w>': 4, '   z e m se l ves</w>': 1, '   a v ay</w>': 1, '   vi z</w>': 1, '   ze</w>': 14, '   v ay</w>': 1, '   z ay</w>': 1, '   t or k</w>': 1, '   sh ra p ne l</w>': 8, '   pa u l ine</w>': 19, '   th ru sts</w>': 1, '   re gi men tal</w>': 3, '   fi b b er</w>': 2, '   stu tt ers</w>': 1, '   f al ter</w>': 2, '   ba tter</w>': 10, '   ba t s man</w>': 1, '   b ow l er</w>': 19, '   fi b s</w>': 2, '   so d ding</w>': 5, '   c ro co di les</w>': 9, '   a ah</w>': 15, '   ber th as</w>': 1, '   sh e lling</w>': 2, ' raid</w>': 3, '   ch ee k y</w>': 5, '   su sp en d ers</w>': 1, '   bri de g ro om s</w>': 1, '   bu ff s</w>': 3, '   c live</w>': 11, '   win k le</w>': 4, '   ser vi ce men</w>': 1, '   fi d dle</w>': 13, '   pro pa g an da</w>': 12, '   g y p sed</w>': 1, '   cu mb er land</w>': 1, '   tur ks</w>': 8, '   fi fe</w>': 3, '   mar ch ing</w>': 13, '   wee ds</w>': 8, '   gar ter</w>': 7, ' bu ried</w>': 1, '   la un ch ing</w>': 4, '   ca u l k</w>': 1, '   bu n g al ow</w>': 3, '   t ow pa th</w>': 1, '   t y p ing</w>': 12, '   ra tion</w>': 5, '   m ac he tes</w>': 1, '   b lu ff ing</w>': 10, '   ph on ey</w>': 18, ' such</w>': 7, '   bo at ers</w>': 1, '   b la z ers</w>': 1, '   p un ts</w>': 3, '   lan ter n s</w>': 4, '   fi re f li es</w>': 3, '   gra mo ph one</w>': 2, '   un said</w>': 3, '   wi re less</w>': 7, '   su mm ers</w>': 10, '   pro ms</w>': 3, ' h en ry</w>': 4, '   the l ma</w>': 65, '   cor n f l ower</w>': 2, '   ri char d son</w>': 7, '   h in ds</w>': 1, '   s an der son</w>': 14, '   wh a ts</w>': 3, '   to p sy</w>': 1, '   tur v y</w>': 1, '   t ou gh ed</w>': 1, '   sc ri mp ing</w>': 1, '   s qu al or</w>': 2, '   b loo di er</w>': 1, '   bu n d l ed</w>': 2, '   c one</w>': 11, '   c ru mp le</w>': 2, '   i ck y</w>': 9, '   t wi d dle</w>': 2, ' na u gh ty</w>': 1, '   to a d</w>': 12, '   st ran g ling</w>': 4, '   who op i e</w>': 1, '   ever body</w>': 2, '   f li r ting</w>': 7, '   ma mm a</w>': 18, '   sle e p in</w>': 9, '   ri m</w>': 7, '   ch er r y pi ck er</w>': 1, '   pi c tu red</w>': 5, '   pi ss y</w>': 5, '   que ers</w>': 5, '   u mm m m</w>': 4, '   go ob er</w>': 2, '   ob er</w>': 3, '   ger ry</w>': 16, ' go ob er</w>': 1, ' ze</w>': 1, '   sch oo ling</w>': 7, '   of f be at</w>': 2, '   ro ad side</w>': 2, '   att r ac tions</w>': 4, '   n ab bed</w>': 4, '   f la sh li ght</w>': 13, '   fi l med</w>': 4, '   hi tch hi k er</w>': 2, '   sp au l ding</w>': 2, '   j un ction</w>': 13, '   u f o</w>': 3, '   sa m son i te</w>': 1, '   mo or e house</w>': 1, '   hou ton</w>': 1, '   bo di ly</w>': 7, '   pu k er</w>': 1, ' pe e</w>': 4, '   ch u m s k i</w>': 1, '   ba w ling</w>': 6, '   s q ea k y</w>': 1, '   l y ne tt e</w>': 22, '   fro m me</w>': 1, '   s no o ty</w>': 2, '   b on k ers</w>': 5, '   comp en sa te</w>': 14, '   vi e w ing</w>': 7, '   ma gi cal</w>': 8, '   he ar t land</w>': 1, '   sa mar i t an</w>': 2, '   gr it</w>': 2, '   b ack as s war ds</w>': 1, '   s mar ta ss</w>': 6, '   h mm mm mm mm m</w>': 1, '   din ging</w>': 1, '   al ac ard</w>': 1, '   sha cking</w>': 1, '   z a i us</w>': 10, '   a p es</w>': 21, '   sp ins</w>': 3, '   e ye b all</w>': 12, '   wa cking</w>': 1, '   t wi st ing</w>': 8, '   sh ar pen ed</w>': 2, '   nu die</w>': 7, '   wor k bo ok</w>': 1, '   co b b</w>': 38, '   h mm mm m</w>': 2, '   st ack</w>': 10, '   to p less</w>': 4, '   wi l k in son</w>': 1, '   per son a li z ed</w>': 2, '   stu ck y</w>': 1, '   t rea t ers</w>': 2, '   li d</w>': 17, '   i t t</w>': 1, '   lo on ey</w>': 7, '   a m pu ta ted</w>': 4, '   b la h</w>': 69, '   1 9 3 7 </w>': 4, '   rea pp e ar</w>': 1, '   el lie</w>': 96, '   bo g d an</w>': 1, ' c ri ss</w>': 1, '   car ni v al s</w>': 2, '   1 9 4 6 </w>': 4, '   nu thou se</w>': 7, '   1 9 3 0</w>': 3, ' re cor ds</w>': 2, '   1 9 1 4 </w>': 1, '   be ar in gs</w>': 4, '   wi ll ows</w>': 3, '   mu ti late</w>': 5, '   bor ne o</w>': 3, '   may he m</w>': 12, '   de cked</w>': 5, '   gra ve y ard</w>': 21, '   c r ab by</w>': 3, '   ca pt</w>': 4, '   ru g g s vi ll e</w>': 1, '   den i se</w>': 6, '   shi th o le</w>': 9, '   li ck ers</w>': 1, '   c y bor g</w>': 3, '   wa ff le</w>': 3, '   st ri pp er</w>': 7, '   ra tty</w>': 3, '   re gen er ate</w>': 4, ' u gh</w>': 2, '   qu a int</w>': 14, '   shu tt er bu g</w>': 1, '   ge or gi e</w>': 19, '   hu d le y</w>': 1, '   ho e down</w>': 1, '   la ssi es</w>': 2, '   g ri pp ing</w>': 4, '   pro por tion</w>': 6, '   hi ll bi lly</w>': 6, '   con vo y</w>': 6, '   sp r in k ling</w>': 1, '   cu ss ing</w>': 2, '   fee der</w>': 3, '   im mor ta li z ed</w>': 1, '   st ro kes</w>': 9, ' to tal</w>': 1, '   bu mp er</w>': 8, '   wi ll is</w>': 6, '   w y d ell</w>': 4, '   p ac ks</w>': 21, '   p ack r at</w>': 1, '   f ar m hou ses</w>': 1, '   na i sh</w>': 1, '   bo o gi e man</w>': 1, ' co p</w>': 14, '   pro ba lly</w>': 2, '   hu l k</w>': 5, '   fa v or ed</w>': 3, '   t in g ling</w>': 3, '   spi der</w>': 38, '   h y per</w>': 4, '   spi de y</w>': 3, ' bu ts</w>': 2, '   for k</w>': 39, '   p y g my</w>': 1, '   chi mp</w>': 4, '   ch ro mo some</w>': 1, '   se par a tes</w>': 6, '   fu r ther more</w>': 10, '   li l a</w>': 38, '   e le c t ro l y s is</w>': 3, '   chri s sa ke</w>': 45, '   n on beau ti ful</w>': 1, '   app re ci a tive</w>': 3, '   er u di te</w>': 1, '   w o of</w>': 2, '   mon gre l</w>': 4, '   g ab ri e ll e</w>': 14, '   for ks</w>': 4, '   ch er i</w>': 1, '   bo ok in gs</w>': 5, '   en ga ge men ts</w>': 3, '   h un h</w>': 18, '   ne an der th al</w>': 3, ' su n</w>': 3, '   jo han n s en</w>': 2, '   u r gh</w>': 1, '   ta m ing</w>': 1, '   un n h h</w>': 2, '   chi pp y</w>': 1, '   par le z</w>': 2, ' v ous</w>': 2, '   g ab by</w>': 2, '   h m mp h</w>': 4, '   h mm mp h</w>': 1, '   spe c t ac u l ar ly</w>': 3, '   br on f man</w>': 6, '   wa ll f l ower</w>': 1, '   pa ja ma s</w>': 12, '   di st r ac ted</w>': 11, '   ac comp any ing</w>': 5, '   pu pp i e</w>': 1, '   sha g g y</w>': 1, '   tr es</w>': 5, '   wa y war d ne ss</w>': 1, '   hu man kind</w>': 1, '   con v ent</w>': 47, '   en chan ting</w>': 3, '   a pe y</w>': 1, '   re tra in</w>': 1, '   af for ded</w>': 2, '   ro si e</w>': 10, '   re v ok ed</w>': 4, '   st e pla d der</w>': 1, '   mi d get</w>': 13, '   mi d ge ts</w>': 3, '   comp ani on ship</w>': 3, '   co ver t</w>': 9, '   cu s p</w>': 2, '   mi d ge th o od</w>': 1, '   ab or ma lly</w>': 1, ' pa ssi on ate</w>': 2, '   e ye si ght</w>': 10, ' mor ti f y in g ly</w>': 1, '   thir ti es</w>': 6, '   o ok a</w>': 1, '   re p ay</w>': 14, '   oo ok</w>': 2, '   u g n h</w>': 2, '   s ea bo ard</w>': 2, '   th u mb t ack</w>': 2, '   hi r su ti s m</w>': 1, '   f lo p house</w>': 3, '   ea st side</w>': 2, '   st ri pp ers</w>': 2, '   di gs</w>': 8, '   li ter ate</w>': 1, '   in ex cu sa ble</w>': 4, '   din n ers</w>': 7, '   qui l ting</w>': 2, '   un ci vi li z ed</w>': 2, '   c ru ddy</w>': 4, '   el c t ro l y s is</w>': 1, '   dis gu sted</w>': 8, '   s have</w>': 33, '   j an et</w>': 76, '   re ve l</w>': 2, '   pi tter</w>': 1, ' pa tter</w>': 1, '   mo by</w>': 3, '   mon et</w>': 5, '   re war ding</w>': 5, '   e li z a</w>': 4, '   do little</w>': 1, '   be ha vi ori st</w>': 3, '   chi mp s</w>': 3, '   f er al</w>': 4, '   un con ta min a ted</w>': 2, '   dis ea sed</w>': 9, '   r ab i es</w>': 4, '   sur vi v a li st</w>': 2, '   be ho o ve</w>': 1, ' qu it</w>': 5, '   per fe c tom un do</w>': 1, '   re pe ll ent</w>': 6, '   sp f</w>': 1, '   st om p ing</w>': 2, '   na tu ra l ne ss</w>': 1, '   si mi l ar ly</w>': 1, '   pe cu li ar i ti es</w>': 3, '   b ack w ard</w>': 14, '   bi g gi e</w>': 9, '   pro gre ss es</w>': 2, ' mm m</w>': 5, '   de li sh</w>': 1, '   inter course</w>': 16, '   ru den ess</w>': 3, '   v u l gar i ty</w>': 1, '   nor m</w>': 22, '   f oun da tions</w>': 9, '   c ru mb le</w>': 7, '   the s is</w>': 13, '   da ft</w>': 11, '   so ci o lo gi cal</w>': 4, '   fe der ally</w>': 1, '   ther a pi st</w>': 26, '   na tur es</w>': 2, '   s lu g do m</w>': 1, '   s lu g gi sh ne ss</w>': 1, '   kee l</w>': 3, ' tr ac ked</w>': 2, ' e w w w</w>': 1, '   me di ta tions</w>': 1, '   du p li ci t ous</w>': 1, '   an al</w>': 14, '   sur r oun din gs</w>': 6, '   re in t ro du c ed</w>': 1, '   ab st r act</w>': 11, '   un k a</w>': 5, '   oun po o</w>': 1, '   un gh</w>': 1, '   un n</w>': 1, '   wi tt gen st e in</w>': 1, '   na te</w>': 3, '   fe ces</w>': 3, '   fran z</w>': 5, '   k l ine</w>': 1, '   be u ys</w>': 2, '   mar ce l</w>': 14, '   du cha mp</w>': 1, '   as se mb la ge</w>': 1, '   in sti lled</w>': 2, '   in de pen den tly</w>': 6, '   a ver sion</w>': 3, '   g lin ting</w>': 2, '   s mar tly</w>': 2, '   fro cks</w>': 2, '   ven d ors</w>': 3, ' eve</w>': 1, ' ing</w>': 15, ' de es</w>': 1, ' g ent</w>': 1, '   el men</w>': 1, '   bro ac h ed</w>': 2, '   der e k</w>': 26, ' sp an</w>': 4, '   ar i sto cra ti c</w>': 2, '   la d y bi r d</w>': 2, '   p ack ard</w>': 10, '   der by</w>': 2, '   mo de l ed</w>': 3, '   fe l son</w>': 18, '   bi lli ar ds</w>': 8, '   win n in</w>': 1, '   a a a a h h h h</w>': 3, '   out play</w>': 1, '   hu st l in</w>': 3, '   fin d le y</w>': 6, '   fa ts</w>': 32, '   com bed</w>': 4, '   f la tt en ed</w>': 2, '   s wi mm in</w>': 6, '   ta b s</w>': 9, '   hu st l ers</w>': 1, '   cor k</w>': 4, ' ti pped</w>': 1, '   k en tu ck y</w>': 26, '   s cu ff le</w>': 1, ' si ze</w>': 2, '   y ar da ge</w>': 3, '   mo t ors</w>': 7, '   fi re pl u g</w>': 1, ' money</w>': 16, '   a mes</w>': 9, '   po o l room</w>': 7, ' he el ed</w>': 2, '   po o l ro om s</w>': 1, '   sa tch el s</w>': 1, '   han d bo ok</w>': 11, '   hu st ling</w>': 6, '   s cu ff ling</w>': 1, '   da mes</w>': 9, '   car na tion</w>': 2, '   s cu ff l in</w>': 1, '   s l ab s</w>': 2, '   f la g po le</w>': 4, '   be st est</w>': 3, '   d ru g gi st</w>': 2, ' ber t</w>': 2, '   qui ts</w>': 10, '   jo ck ey</w>': 15, '   poli o</w>': 3, '   hu st l ed</w>': 2, '   bri ck la ying</w>': 1, '   bl in ds</w>': 4, '   ga ll er i es</w>': 2, '   m ack</w>': 43, '   cra mp ed</w>': 4, '   ban k ro ll</w>': 3, '   ta ver n</w>': 5, '   ch u b by</w>': 6, ' beau ti ful</w>': 14, '   v in es</w>': 3, '   pu tt er ing</w>': 3, '   sa un ter ing</w>': 2, '   sa un ter</w>': 1, ' loo ked</w>': 2, '   g l ow ed</w>': 1, '   ri ge l</w>': 2, '   ca ssi us</w>': 11, '   hea vi est</w>': 5, '   me te ori te</w>': 3, '   s wi tch ing</w>': 13, '   so l en o id</w>': 1, '   g lea son</w>': 3, '   gar d ner</w>': 18, ' ms</w>': 3, '   ma g nu son</w>': 3, '   ad ju st ing</w>': 10, '   t un ne ling</w>': 1, '   m ck el tch</w>': 2, '   ro a ds</w>': 27, '   f oo t pa th s</w>': 1, '   fro g</w>': 17, '   s li pp er</w>': 2, '   pen i ci ll in</w>': 6, '   ex pi re</w>': 8, '   e m be d ding</w>': 1, '   e m be d ded</w>': 5, '   com m uni ca ting</w>': 12, '   l en n on</w>': 8, '   sp an k y</w>': 13, '   mo i she</w>': 2, '   mar t in s bur g</w>': 1, '   che ck ma te</w>': 2, '   re c y c ling</w>': 4, '   ex t in ction</w>': 8, '   re du ces</w>': 3, '   re c y c les</w>': 2, '   an a lo g</w>': 1, '   se qu en ti al</w>': 2, '   spe c tr a</w>': 1, '   an al y z er</w>': 2, '   o ver la y</w>': 2, '   re t ro f it</w>': 1, '   tr an sp on der</w>': 7, '   di st or tions</w>': 2, '   be e per</w>': 9, '   po ll u te</w>': 2, '   ro s well</w>': 3, '   bu n k er</w>': 7, '   tri an gu late</w>': 3, '   ha l bro ok</w>': 3, '   con stan ce</w>': 2, '   o l ds</w>': 3, '   de com po se</w>': 3, '   o ver ri ding</w>': 1, '   ex ter min a ted</w>': 2, '   si gh t in gs</w>': 6, '   ft</w>': 6, '   e ta</w>': 4, '   ne ll is</w>': 1, '   he li co p t ers</w>': 8, ' or din ate</w>': 2, '   co ck a ma mi e</w>': 5, '   de pen dent</w>': 8, '   in for m ing</w>': 4, '   chi e f s</w>': 20, '   f re qu en ci es</w>': 8, '   loo sing</w>': 3, '   cap ab i li ti es</w>': 5, '   ni m z i k i</w>': 1, '   u p gra de</w>': 4, '   de f c on</w>': 4, ' m un ch</w>': 3, ' we ed</w>': 2, '   t or o</w>': 3, ' i de</w>': 1, '   m ac h</w>': 5, ' si de win d ers</w>': 1, '   mm mm mm m</w>': 7, '   j as m ine</w>': 6, '   do l ph ins</w>': 13, '   a p ac he</w>': 8, '   har ri er</w>': 2, '   se g a</w>': 1, '   6 4 </w>': 3, '   a li ci a</w>': 2, '   lu ca s</w>': 19, '   b ac on</w>': 22, '   ear ning</w>': 6, '   b om b ers</w>': 5, '   den i ab i li ty</w>': 1, '   nor a d</w>': 2, '   ad vi sing</w>': 3, '   h y st er i a</w>': 10, '   f ra il</w>': 5, '   di ssi mi l ar</w>': 2, '   brea th es</w>': 6, '   com par able</w>': 2, ' ex ci ting</w>': 1, '   gi z mo s</w>': 2, '   ta ma le</w>': 2, ' ex i stan ce</w>': 1, '   i s k en der u n</w>': 3, '   pre ce d es</w>': 2, '   de t our</w>': 9, '   bro dy</w>': 24, '   in dy</w>': 28, '   sa ll ah</w>': 5, '   a le x an dre tta</w>': 4, '   o as is</w>': 7, '   gra il</w>': 61, '   ta b let</w>': 6, ' ac ro ss</w>': 4, '   cre sc ent</w>': 2, ' a le x an dre tta</w>': 1, '   nu mer al s</w>': 1, '   sch ne i der</w>': 10, '   ven ice</w>': 43, '   d on o v an</w>': 21, '   ha a a</w>': 3, '   mi gh ti er</w>': 3, '   ar gh h h</w>': 1, '   me d d ling</w>': 2, '   ar ti f act</w>': 6, '   ta pe st ri es</w>': 6, '   bu ttle</w>': 18, '   br un wa ld</w>': 2, '   m ac d on a ld</w>': 10, '   ever la st ing</w>': 6, '   do do</w>': 2, '   st oo ge</w>': 10, '   s na g</w>': 2, '   fri ar</w>': 4, '   ch r on i cl ed</w>': 1, ' mar k ers</w>': 2, '   in comp le te</w>': 9, '   en tom bed</w>': 3, ' supp o se d ly</w>': 1, '   im par ted</w>': 1, '   fran ci s can</w>': 1, ' be d time</w>': 1, '   en tru sted</w>': 8, '   ar i ma th a e a</w>': 2, '   k ni gh ts</w>': 35, '   de ser ts</w>': 5, '   c any ons</w>': 2, '   c ru ci fi x i on</w>': 1, '   en g in e ers</w>': 8, '   un ear th ed</w>': 5, '   an k ar a</w>': 2, '   ex ca v a ting</w>': 1, '   as se ss ment</w>': 4, '   s an d st one</w>': 1, '   te xt</w>': 3, ' tw el f th</w>': 1, '   an ti qui ti es</w>': 3, '   con tri bu tions</w>': 7, '   el s a</w>': 11, '   s wa sti k a</w>': 3, '   br un wal ds</w>': 1, '   ar ti st ry</w>': 4, '   car v in gs</w>': 2, '   sc ro ll work</w>': 1, '   ar k</w>': 11, '   co ven ant</w>': 2, '   cha mb ers</w>': 13, '   pa g an</w>': 5, '   at ti l a</w>': 3, '   sch oo l boy</w>': 8, '   in di an a</w>': 16, '   i ll u min ation</w>': 4, ' je ho v ah</w>': 1, '   pen i t ent</w>': 13, '   ar cha e o lo g y</w>': 4, '   in to l er able</w>': 5, '   cha ll en g es</w>': 8, ' re li an ce</w>': 1, '   dis gr ac e ful</w>': 5, '   f a</w>': 13, '   ad ven tur es</w>': 3, '   ob se ssion</w>': 31, '   se l f less</w>': 4, '   ch r on i c les</w>': 1, '   an se l m</w>': 1, '   d ev i ces</w>': 23, '   u m h</w>': 1, '   ma il ed</w>': 11, '   mar x</w>': 20, '   ca t ac om b s</w>': 7, '   hu mp f</w>': 1, '   v an qui sh ed</w>': 2, '   ca me ls</w>': 10, ' ger man</w>': 1, '   c ru ci for m</w>': 1, '   k a z i m</w>': 1, '   ma har a ja h</w>': 14, '   p an k o t</w>': 10, '   th u g ge es</w>': 4, '   ob sc en i ty</w>': 1, '   k al i</w>': 6, '   1 8 5 7 </w>': 2, '   th u g ge e</w>': 3, '   b lu m bur t t</w>': 1, ' na ti ves</w>': 1, '   in se c ts</w>': 10, '   d ev out</w>': 2, '   sh re w d</w>': 7, '   ad mi tt e d ly</w>': 3, '   do lls</w>': 17, '   k r y ta</w>': 2, '   cu ri o s</w>': 1, '   op i u m</w>': 12, '   sh an g ha i</w>': 12, '   la l</w>': 2, '   u h med</w>': 1, '   cha tt ar</w>': 1, '   wor shi pp ing</w>': 5, '   mo l a</w>': 3, '   su l t an</w>': 4, '   ma da ga sc ar</w>': 2, '   ex a g ger a ted</w>': 8, '   hon du ra s</w>': 6, '   s an k ar a</w>': 8, '   ar ti f ac ts</w>': 11, '   e min ent</w>': 5, '   ar cha e o lo gi st</w>': 3, '   an ce st or</w>': 2, '   an ti do te</w>': 19, '   la o</w>': 5, '   nu r ha ch i</w>': 2, '   ne e</w>': 3, '   bra ke</w>': 8, '   na in su k h</w>': 1, ' fo llow</w>': 6, '   shi v a</w>': 8, '   f la tt en</w>': 1, '   w u</w>': 9, '   ha n</w>': 42, ' do ke</w>': 3, '   vi vi d</w>': 8, '   ge ms</w>': 5, '   di sp er sed</w>': 1, '   si v la lin g a</w>': 1, '   k ri sh na</w>': 5, '   si v al in g a</w>': 2, '   ou tr un ning</w>': 2, '   fu l c ru m</w>': 3, '   c ri tter</w>': 3, '   pr ac ti ces</w>': 7, ' no c tur na l</w>': 2, '   le f to ver s</w>': 6, '   s ans k r it</w>': 3, '   k a li s a</w>': 1, '   la tch</w>': 11, '   p in head</w>': 6, '   ori ent</w>': 4, '   st ro king</w>': 5, '   ma x i ll ary</w>': 1, '   pre ca u da l</w>': 1, '   ver te bra e</w>': 4, '   cu r ling</w>': 1, '   wh ey</w>': 1, '   ta m er</w>': 1, '   si am</w>': 6, '   c ri s sa ke</w>': 5, '   a w w w w</w>': 5, ' j ea l ous</w>': 1, '   r en e</w>': 16, '   di e ter</w>': 20, '   gr un er</w>': 16, '   chi an g</w>': 5, '   ch o</w>': 2, '   pe m</w>': 7, '   ar ea dy</w>': 1, '   wal k man</w>': 5, '   h ong</w>': 31, '   ha k a</w>': 2, '   x u k i</w>': 2, '   y en</w>': 3, '   son y</w>': 1, '   hi ro shi ma</w>': 3, ' 1 0 0 0</w>': 10, '   sh e lf</w>': 16, ' i ma g ine</w>': 1, ' dr ink</w>': 5, '   z en</w>': 7, ' jo e</w>': 17, '   sa pp or o</w>': 1, '   ca kes</w>': 5, ' co o ki es</w>': 2, '   i t se</w>': 1, '   f oun ta in head</w>': 3, ' te ch</w>': 15, '   o sa k a</w>': 4, '   r y u j i</w>': 9, ' r y u j i</w>': 1, '   ba ll ga me</w>': 8, '   5 6 </w>': 6, '   bu tting</w>': 4, '   s ke w ed</w>': 1, '   su tt on</w>': 12, '   re co g ni ses</w>': 1, '   l ends</w>': 1, ' gr un er</w>': 2, '   bu s boy</w>': 7, '   so p hi sti ca tion</w>': 3, '   op ti c</w>': 5, ' sa vo ir</w>': 1, '   fa i re</w>': 5, '   id</w>': 48, '   stu mb le bu m</w>': 1, ' r en e</w>': 2, ' pretty</w>': 7, '   cle an est</w>': 2, '   li ma s</w>': 1, '   per son a</w>': 3, '   in c on</w>': 1, '   pi cu ous</w>': 1, ' tr ace</w>': 3, '   ust</w>': 2, '   co de lo ck</w>': 1, '   2 6 9 9 3 </w>': 1, ' tal ent</w>': 3, '   mo bi li ty</w>': 1, '   tr ys</w>': 1, '   th i</w>': 2, '   fi e st a</w>': 2, '   c y c li st</w>': 2, ' go oo od</w>': 1, ' bu z z ing</w>': 1, '   ni cho l son</w>': 4, '   no de</w>': 1, '   da t su n</w>': 2, ' 1 1 4 </w>': 14, '   bu n d les</w>': 3, '   o ver si ght</w>': 8, '   do pe y</w>': 6, '   con do</w>': 11, '   ar r r gh</w>': 1, ' re por ting</w>': 1, '   ven tri lo qui s m</w>': 1, '   re por ting</w>': 23, ' tr ans it</w>': 1, ' in side</w>': 7, ' al</w>': 5, '   ban da i o</w>': 1, '             </w>': 5, ' sc ar ed</w>': 5, '   h y u t n</w>': 1, '   s lu l p t s a</w>': 1, ' en l ar ge ment</w>': 1, ' put</w>': 10, ' in je ct</w>': 1, '   di ver ted</w>': 4, '   ne w er</w>': 1, ' 1 1 5 </w>': 1, '   de la y ed</w>': 14, '   sur ve i lli an ce</w>': 1, ' z x f l b b g t</w>': 1, '   vi o l a</w>': 10, '   in tru de</w>': 4, ' so on</w>': 6, ' com pla in ing</w>': 1, '   com plan ing</w>': 1, '   in ten tion al</w>': 6, '   fin ne g an</w>': 9, ' lo ve ly</w>': 2, ' very</w>': 25, '   cho le st er o l</w>': 2, '   h en e kin</w>': 1, '   ki r in</w>': 3, '   tru ck l hou s er</w>': 1, '   sa vo ir</w>': 1, '   a po lo gi se</w>': 2, '   ri u j i</w>': 1, ' und er</w>': 13, '   f ea ts</w>': 3, '   con ned</w>': 5, ' de sp er ate</w>': 1, '   to o ts</w>': 5, '   in land</w>': 7, '   hi r es</w>': 6, '   k o sh er</w>': 8, '   ki d ded</w>': 1, ' u mm m</w>': 2, '   di sin ter est</w>': 1, ' di sin ter est</w>': 1, '   cor ni est</w>': 1, ' b la ke</w>': 1, '   the f ts</w>': 1, '   f oun tain</w>': 20, '   a th o l</w>': 1, '   do a kes</w>': 1, '   j an e an e</w>': 1, '   brea th ed</w>': 5, '   a ir ways</w>': 3, '   la u f er</w>': 1, '   l ow ell</w>': 28, ' ge o lo gi sts</w>': 1, '   ge o lo gi sts</w>': 6, ' su per vi s or</w>': 1, '   wi lli a m son</w>': 31, '   un ab om b er</w>': 1, '   la x</w>': 11, '   su e in</w>': 1, '   h wan g</w>': 1, '   ge y el in</w>': 1, '   dea d lin es</w>': 3, '   au th ori ta tive</w>': 2, '   o ver wh el min g ly</w>': 2, '   do cu men ted</w>': 7, '   se le c tive</w>': 5, '   se le c ti ve ly</w>': 1, ' hea ding</w>': 1, '   sc ru g gs</w>': 4, ' bl ower</w>': 3, '   wi g and</w>': 27, '   sh ar on</w>': 9, '   re stra in</w>': 7, '   re ta li ate</w>': 2, '   a gre e men ts</w>': 8, '   con stra int</w>': 1, '   pri ma ver a</w>': 1, '   in ver t</w>': 1, ' ca pi ta li st</w>': 1, '   re fu tes</w>': 1, '   as sa s sin ation</w>': 26, '   di sa gre ed</w>': 3, '   k lu ster</w>': 2, '   se g ment</w>': 7, '   e di ting</w>': 8, ' pro fi le</w>': 5, '   la w su it</w>': 31, '   ne w s man</w>': 4, '   vi o la tes</w>': 4, '   re for m</w>': 16, ' ma l f ea s an ce</w>': 1, '   ne w s wor th y</w>': 2, ' rea son able</w>': 1, ' t or ti ous</w>': 2, ' po ten ti al</w>': 1, '   we st in gh ou se</w>': 3, ' shut</w>': 11, '   j our na li sti c</w>': 3, '   in f lu en c ed</w>': 7, '   er i c</w>': 72, '   ti s ch</w>': 3, ' 8 1 </w>': 3, '   mu l ti bi lli on</w>': 1, ' k oo ls</w>': 1, ' s ea l ed</w>': 3, '   ve sted</w>': 6, ' per s ons</w>': 1, '   ca per e ll i</w>': 2, '   e d it</w>': 3, '   la under ing</w>': 4, '   n ar c o</w>': 4, ' mi ke</w>': 6, '   a i red</w>': 2, '   di sc lo sing</w>': 3, '   tru er</w>': 3, '   di sc lo se</w>': 8, '   da ma g es</w>': 8, '   t or ti ous</w>': 1, ' ver ac i ty</w>': 1, ' st ar</w>': 9, '   ver ac i ty</w>': 2, ' ri fe</w>': 1, '   bri d g es</w>': 10, ' ta p ing</w>': 1, '   se ver an ce</w>': 9, '   pa you ts</w>': 1, '   cont in u ing</w>': 14, '   s an de fu r</w>': 10, '   sa u ce</w>': 31, '   ber g man</w>': 12, '   ber man</w>': 2, ' m ea sur es</w>': 2, '   in fo ta in ment</w>': 1, '   for e gone</w>': 2, '   p hi li p</w>': 25, '   mor r is</w>': 14, ' i g ni tion</w>': 2, '   pro pen si ty</w>': 3, '   stu art</w>': 6, '   po ll</w>': 7, ' bl ow ers</w>': 1, '   2 1 2 </w>': 5, ' 5 5 5 </w>': 3, ' 0 1 9 9 </w>': 1, ' fu ture</w>': 3, '   re gar ded</w>': 6, '   con si der a tions</w>': 5, '   tu te la ge</w>': 4, '   r he t ori cal</w>': 3, '   b ani sh ed</w>': 12, '   li e u</w>': 6, '   con stra in ed</w>': 2, ' com pe lled</w>': 1, ' ci gar e tt es</w>': 2, '   ce o s</w>': 4, '   mar ine</w>': 23, '   sh ei k h</w>': 2, '   mu s sa w i</w>': 1, '   ar ab i c</w>': 6, '   ob je c ti vi ty</w>': 4, ' re spe c ted</w>': 1, ' ma ga z ine</w>': 1, '   he z bo ll ah</w>': 2, '   bro ad en</w>': 3, ' z i on i st</w>': 1, '   man i pu la ted</w>': 7, '   ra i ls</w>': 3, '   re fu te</w>': 1, '   ac cu sa tion</w>': 9, '   pro v able</w>': 1, '   di st or t</w>': 1, '   op en er</w>': 6, ' 3 9 </w>': 4, ' 9 5 </w>': 3, '   f la w s</w>': 4, '   te le mar ke ting</w>': 2, '   sp ar red</w>': 1, '   ju do</w>': 4, '   o l y m pi c s</w>': 6, '   a ir ing</w>': 3, ' ni co t ine</w>': 1, '   ad di c tive</w>': 8, '   per ju red</w>': 3, ' vo l un te er ed</w>': 1, '   li an e</w>': 4, '   di ck in</w>': 1, '   t our na men ts</w>': 3, '   s ke p ti ci s m</w>': 3, '   sta tu s</w>': 39, '   vo ye u ri s m</w>': 2, '   com mo di ti es</w>': 2, '   com mer ci al s</w>': 19, '   ci ted</w>': 2, '   sh op li f ting</w>': 2, '   st re ssed</w>': 9, '   cen sor ing</w>': 1, '   mar cu se</w>': 1, '   men tor</w>': 9, '   jo ll a</w>': 2, '   te m pu r a</w>': 1, '   b en e fi ci al</w>': 2, ' to b ac c o</w>': 1, '   k ni cks</w>': 10, '   car bi de</w>': 1, '   p fi z er</w>': 2, ' re la ted</w>': 6, ' z one</w>': 1, '   ex p an ded</w>': 6, '   j our na list s</w>': 5, '   ra m par ts</w>': 3, '   9 3 0</w>': 1, '   t y l en o l</w>': 4, '   per il ous</w>': 2, '   mo ore</w>': 19, '   man u f ac tur ers</w>': 5, '   char ging</w>': 10, '   su b sti tu te</w>': 15, ' cou mar in</w>': 1, ' w</w>': 45, '   tr an si tion</w>': 9, '   cou mar in</w>': 2, '   un su c ce ss ful</w>': 2, '   cou ma d in</w>': 1, ' spe ci fi c</w>': 3, '   car c in o g en</w>': 1, ' imp act</w>': 2, '   bo o st ing</w>': 1, '   spi king</w>': 1, '   ex ten si ve</w>': 9, ' a m mon i a</w>': 1, '   man i pu la tes</w>': 2, '   ad ju sts</w>': 2, '   ad ding</w>': 9, '   en han c ing</w>': 2, '   a m mon i a</w>': 2, '   mi s sta ted</w>': 2, '   s we ar ing</w>': 5, '   te sti mon i es</w>': 1, '   le f ts</w>': 2, '   d ow n s</w>': 8, '   mi d d les</w>': 1, '   pro c ee ding</w>': 8, '   s mi r k</w>': 4, '   in st ru c ting</w>': 2, '   con tr ac tu al</w>': 1, '   mo t le y</w>': 4, '   t y p ed</w>': 5, '   tu i tion</w>': 8, ' coun se l</w>': 1, '   re im bur sed</w>': 2, '   me di ca id</w>': 1, '   vi o la ting</w>': 9, ' re se ar ch</w>': 1, '   go l f er</w>': 2, '   inter ven tion</w>': 5, '   t an tru m</w>': 3, '   sch mo e</w>': 3, '   in di sc re tion</w>': 1, '   b ran di sh ing</w>': 1, '   fi re ar m</w>': 4, '   for e si ght</w>': 3, '   b on ni e</w>': 6, ' wri g g ling</w>': 1, '   ma s se y</w>': 23, ' pen e tra te</w>': 1, ' n up</w>': 17, '   nu p ti al</w>': 2, '   un ju sti fi ed</w>': 1, '   s lan der ed</w>': 1, '   man dy</w>': 6, '   re i s er</w>': 2, '   pro vi d es</w>': 7, '   dis so lu tion</w>': 4, '   w ea l th i er</w>': 1, '   ma st e c tom y</w>': 2, '   bo ob s</w>': 13, '   mar y l in</w>': 52, '   so f ti e</w>': 4, '   mu h</w>': 6, '   f l ou ri sh ing</w>': 1, '   in pa ti ent</w>': 1, '   y om</w>': 3, '   ki pp u rs</w>': 1, ' nu r se</w>': 4, '   me ds</w>': 9, '   sh el ter ed</w>': 2, '   du mb ar ton</w>': 5, '   ne u tra li ze</w>': 10, '   re x ro th</w>': 21, '   du mb art</w>': 1, '   pi ge on</w>': 19, '   ex ter min a tor</w>': 4, '   un kind</w>': 5, '   ho o t</w>': 9, '   r ab in ow</w>': 4, '   re d ra f ted</w>': 1, '   un ter me y er</w>': 1, '   wri g le y</w>': 7, '   in ab i li ty</w>': 4, '   my er son</w>': 2, '   co lo st om y</w>': 1, '   den i al</w>': 12, '   car di ac </w>': 13, '   bo to x</w>': 3, '   a spi ra tions</w>': 4, '   be l</w>': 6, '   le cou l t re</w>': 1, '   re ver s</w>': 1, '   fr own</w>': 3, '   bo tu li s m</w>': 1, '   in je ct</w>': 13, '   par al y z es</w>': 2, '   e ye br ows</w>': 4, '   h m o</w>': 1, '   man u f ac tur es</w>': 2, '   sta pl es</w>': 3, ' t ac ks</w>': 1, '   k or e a</w>': 17, '   gu tt man</w>': 1, '   te sti fi ed</w>': 13, '   m ing</w>': 6, '   t ong</w>': 4, '   cour t room</w>': 20, '   pen t an ge l i</w>': 15, '   se du c tive</w>': 4, '   ga ll er y</w>': 22, '   si ci ly</w>': 19, '   so li di f y</w>': 2, '   r ab in o</w>': 1, '   di c ta t or ship</w>': 5, ' ca st r o</w>': 1, '   ev a sion</w>': 7, '   go on</w>': 7, '   n on on o</w>': 2, '   ba si c s</w>': 6, '   bu ts</w>': 6, '   re ven u e</w>': 10, '   under pa id</w>': 1, '   s ou th we st er n</w>': 1, '   ten ac i ous</w>': 4, '   un sc ru pu l ous</w>': 1, '   inter v al</w>': 4, '   in j un ction</w>': 8, '   pri ma te</w>': 7, '   t or tu ous</w>': 1, '   w ra th</w>': 7, '   ju li e ta</w>': 1, '   be ll a gi o</w>': 1, '   ar ty</w>': 3, '   f ar ty</w>': 2, '   li th o gra ph s</w>': 1, '   ca st r o</w>': 28, '   cu b ans</w>': 29, ' stuff</w>': 9, '   stu d mu ff in</w>': 1, '   re fun d</w>': 11, '   com f y</w>': 5, '   chi c</w>': 8, '   ti me less</w>': 5, '   e cle c ti c</w>': 2, '   nu r ser y</w>': 16, '   re fri ger ation</w>': 1, '   fr in ge</w>': 6, ' gir ly</w>': 1, '   for t un y</w>': 1, '   pa s se men ter i e</w>': 1, '   bi lli ard</w>': 3, '   per son a li ze</w>': 1, '   mar y lin i ze</w>': 1, '   be dro om s</w>': 11, '   s qui sh ing</w>': 1, '   mu son</w>': 1, '   sp ee ch less</w>': 5, '   ro bu st</w>': 3, ' de fi ci en ci es</w>': 1, '   pa tri ci a</w>': 4, '   co li ck y</w>': 1, '   re cou p</w>': 1, '   ex pre ssi ve</w>': 3, '   hu mi d ors</w>': 1, '   li th o gra p h</w>': 1, '   ho ck ne y</w>': 4, '   star bu cks</w>': 7, '   bl en ded</w>': 3, '   fu c t ard</w>': 1, '   as s wi pe</w>': 3, '   be tra ying</w>': 4, '   ju ri es</w>': 1, '   pen e</w>': 1, '   re con ci li ation</w>': 3, '   m oun ted</w>': 4, '   dar ts</w>': 4, '   ha wai i</w>': 27, '   lo v </w>': 5, '   t our ne do s</w>': 1, '   car ni v ore</w>': 1, '   com pl ac ent</w>': 1, ' dis miss</w>': 1, '   fe i g ned</w>': 1, '   po si t an o</w>': 5, '   pi e tr o</w>': 1, '   en z y mes</w>': 2, '   ta ga m et</w>': 1, '   me ta bo li s m</w>': 6, '   at che son</w>': 1, '   to pe k a</w>': 1, '   fe</w>': 4, '   win ding</w>': 9, '   sp ree</w>': 8, '   as se ts</w>': 18, '   c y ni ci s m</w>': 8, '   ru gs</w>': 7, '   man s our</w>': 1, '   sor kin</w>': 1, '   ra mon a</w>': 12, '   bar c el on a</w>': 3, '   an th ro po lo gi st</w>': 2, '   per p le x ed</w>': 1, '   in tr an sig ence</w>': 1, '   ru in ation</w>': 1, '   m c d on al ds</w>': 3, '   ru m se y</w>': 2, '   har ri man</w>': 5, '   k ra v is</w>': 1, '   pe ar l man</w>': 1, '   ne en er</w>': 3, '   con d om in i u m</w>': 2, '   in tru der</w>': 10, '   pro fe ssi on ally</w>': 7, '   out lin ed</w>': 3, '   b ru tu s</w>': 4, '   di e ti ci an</w>': 3, '   re mar ried</w>': 13, ' ar ty</w>': 3, '   ki r sh ner</w>': 5, '   con ser v a tive</w>': 16, '   mar i tal</w>': 5, '   i ce brea k ers</w>': 1, '   in i ti ate</w>': 9, '   pro c ee din gs</w>': 6, '   su f fi ci en tly</w>': 5, '   di sp as si on ate</w>': 3, '   de lu si on al</w>': 5, '   a mi ca ble</w>': 2, '   ma m mo gra m</w>': 1, '   y er t</w>': 3, '   bo c a</w>': 9, '   spe ci al s</w>': 5, '   de mon st r able</w>': 1, '   in fi de li ty</w>': 4, '   u no ff en ding</w>': 1, '   mor t ga ged</w>': 3, '   par a me t ers</w>': 7, '   for en si ca lly</w>': 2, '   re ta in ed</w>': 4, '   ex p an ding</w>': 6, ' k no ck</w>': 7, '   an at om i ca lly</w>': 3, '   ar d or</w>': 1, '   co o ls</w>': 5, '   e mo tion ally</w>': 13, '   du ra tion</w>': 3, '   m un son</w>': 2, '   be ck</w>': 12, '   c ro e sus</w>': 3, '   sp o ons</w>': 2, '   st e w art</w>': 28, '   men t or ing</w>': 2, '   cl er ked</w>': 1, '   s ca li a</w>': 1, '   men te e</w>': 1, '   a spi ra tion</w>': 2, '   ba ll er in a</w>': 1, '   an ch or woman</w>': 2, ' some</w>': 39, '   mon ro e</w>': 13, ' pa tty</w>': 1, '   f re qu en tly</w>': 9, '   ban der as</w>': 1, '   th or st en son</w>': 1, '   gi e se l en s en</w>': 1, '   mu ri el</w>': 2, '   f ar row</w>': 2, '   mi a</w>': 5, '   lo v in g ton</w>': 2, '   tra il ed</w>': 3, '   we st le y</w>': 38, '   an dre w s</w>': 25, '   thou gh t fu lly</w>': 3, '   dis so l ves</w>': 5, '   en t ers</w>': 11, '   g in ger ly</w>': 1, '   du mp s</w>': 11, ' wh e ther</w>': 2, '   j i tt ers</w>': 5, '   i te mi z ed</w>': 1, '   ju sti fi ed</w>': 2, '   war n e</w>': 11, '   ja mb or e e</w>': 1, '   stu pi d ly</w>': 3, '   pa m per ed</w>': 1, '   in sin c ere</w>': 1, '   au to g y r o</w>': 1, '   fu r or</w>': 1, '   go o ey</w>': 3, '   ac cu mu late</w>': 4, '   a vi ation</w>': 10, ' par ti cu l ar ly</w>': 1, '   f li ers</w>': 1, '   gi go l o</w>': 6, '   han ding</w>': 16, '   ke tch up</w>': 14, '   sta g ger ing</w>': 6, '   fi let</w>': 6, '   mi g n on</w>': 3, '   an nu lled</w>': 2, '   com par ing</w>': 4, '   stra te gi st</w>': 1, '   g an d hi</w>': 11, '   ob je c tions</w>': 13, ' lo v in g ton</w>': 1, '   tr an sp or ts</w>': 4, '   youn g ster</w>': 6, '   wi l d c at</w>': 3, '   cl ar k son</w>': 1, '   ta med</w>': 2, '   br ow bea ten</w>': 1, '   fin an ci ers</w>': 1, '   sta te s men</w>': 1, '   an nu l ment</w>': 6, '   po ke</w>': 15, '   j er i ch o</w>': 6, '   to pp ling</w>': 1, '   to pp le</w>': 4, '   g l en</w>': 15, '   mi chi g an</w>': 18, '   oo oo o oh</w>': 3, ' che cking</w>': 3, '   nor th b ound</w>': 1, '   op er a ti ves</w>': 5, '   ye sti d day</w>': 1, '   ton si l</w>': 3, '   ton si ls</w>': 10, '   wh ad da</w>': 23, '   fe ll er</w>': 27, '   co tt age</w>': 10, '   car ro ts</w>': 8, '   car ro t</w>': 12, ' 1 3 </w>': 5, '   a le ck</w>': 4, '   pi p</w>': 6, '   j er k y</w>': 6, '   in de pen d ence</w>': 17, ' hi k ers</w>': 1, '   p an han d ling</w>': 2, ' car ro ts</w>': 1, '   pen k ni fe</w>': 1, '   to o th pi ck</w>': 11, '   hi king</w>': 6, '   hi tch</w>': 14, ' hi king</w>': 2, '   ta hi t i</w>': 10, '   ca t fi sh</w>': 3, ' pi g g y</w>': 11, ' b ack er</w>': 2, ' had</w>': 8, '   ri d d en</w>': 8, '   pla y ful</w>': 4, ' he ar ted</w>': 16, '   s ou </w>': 3, '   fa in ting</w>': 4, ' star v ation</w>': 1, '   cha tt er bo x</w>': 1, '   st ro lls</w>': 2, '   sha pe le y</w>': 5, '   mu sh y</w>': 6, '   mu s ke te ers</w>': 8, ' ar ta g n an</w>': 36, ' ea st</w>': 4, '   l y n n e</w>': 4, ' per son</w>': 5, '   au di t ori u ms</w>': 1, '   el k</w>': 8, '   s we de</w>': 14, '   wi l kes</w>': 9, ' bar re</w>': 2, '   d un king</w>': 2, '   so a k</w>': 9, '   p lo p</w>': 2, '   ou tw it</w>': 2, '   im mor al</w>': 8, '   go ver ne ss es</w>': 1, '   cha per on es</w>': 1, ' gu ar ds</w>': 1, '   ac cu st om ed</w>': 8, '   d ou gh nu t</w>': 3, '   a po lo gi z ed</w>': 10, '   dis gu st in g ly</w>': 2, '   p int</w>': 11, ' d ou gh nu ts</w>': 1, ' bl ack</w>': 13, '   dri ver s</w>': 15, ' c r ac king</w>': 1, '   ro be</w>': 10, '   whi pp o or will</w>': 1, '   c ri es</w>': 15, '   car e ss es</w>': 1, '   i di o sy n cra sy</w>': 1, '   ho y le</w>': 2, '   i s ra e li tes</w>': 2, '   und re ss es</w>': 2, '   tru mp et</w>': 12, ' qui te</w>': 4, '   ma the ma ti c s</w>': 15, '   wi fe y</w>': 1, '   e st ab li sh men ts</w>': 2, '   co lo s sa l</w>': 9, ' lea p</w>': 1, '   ar men i ans</w>': 1, '   fa i ls</w>': 11, ' hu mi li ty</w>': 2, ' gr ab </w>': 2, '   me ani es</w>': 1, '   bu ss es</w>': 4, '   than ked</w>': 6, '   j ack son vi ll e</w>': 2, '   ca da ver ous</w>': 1, '   ye g g</w>': 1, '   ho b o</w>': 2, '   te mp o</w>': 1, '   la g ging</w>': 2, '   cap ti vi ty</w>': 4, '   pl y in</w>': 1, '   z e ke</w>': 9, ' do v ey</w>': 2, ' much</w>': 3, '   fi lli es</w>': 2, '   ex ce ll en tly</w>': 1, '   mu g gs</w>': 1, '   g ab b in</w>': 2, ' loo kin</w>': 8, '   y an ked</w>': 3, '   mi s sus</w>': 23, '   un ki ssed</w>': 1, ' around</w>': 12, '   da w son</w>': 27, ' un less</w>': 5, ' me ss es</w>': 1, '   e le c tri fi ed</w>': 5, ' any time</w>': 1, '   di ck en s</w>': 7, '   go b</w>': 2, '   ga sh ou se</w>': 1, '   pa loo k a</w>': 1, '   re ck on</w>': 78, '   ma g ne si a</w>': 1, '   me b be</w>': 10, ' tru st in</w>': 1, '   tal k a tive</w>': 6, ' sh</w>': 8, ' an te</w>': 3, '   s m ack ers</w>': 2, '   g at</w>': 4, '   fi re wor ks</w>': 12, '   ho g g in</w>': 1, ' 1 0</w>': 13, '   tra v el in</w>': 4, '   co p ying</w>': 4, '   l ou </w>': 74, '   o s wa ld</w>': 99, ' th re at</w>': 1, '   as sa s sin ate</w>': 5, ' 2 3 </w>': 2, '   mi li t ant</w>': 7, '   mo t or ca de</w>': 2, '   sha w</w>': 45, '   bra ding</w>': 1, '   or vi ll e</w>': 2, '   t ow n s end</w>': 13, '   ar c ac h a</w>': 1, '   mo ff et</w>': 1, '   gr un t</w>': 9, ' de cl in es</w>': 1, '   su b po en a</w>': 17, '   du ll es</w>': 3, '   ca be ll</w>': 4, '   hel ms</w>': 24, '   gi ra ff es</w>': 1, '   nu ma</w>': 7, '   f er ri e</w>': 22, '   de ter i or a ting</w>': 2, '   1 3 0</w>': 3, '   ex hi bi ts</w>': 5, '   pu r por ting</w>': 1, '   co on s kin</w>': 1, '   im per son at ors</w>': 1, '   re pped</w>': 1, '   fee be es</w>': 1, '   ber tr and</w>': 20, '   m un i</w>': 1, '   mar i o</w>': 16, '   bu t si e</w>': 1, '   cu ban o</w>': 2, '   s wi sh es</w>': 3, '   bi m be tt es</w>': 1, '   s wi sh</w>': 3, '   ro ck e fe ll er</w>': 5, '   cor n mu f fin s</w>': 1, '   j i ving</w>': 2, ' 5 6 </w>': 1, '   ch ow der</w>': 3, '   de an o</w>': 1, '   ca be z a</w>': 1, ' cla y</w>': 1, '   con ning</w>': 3, '   th i ck er</w>': 7, '   mo la ss es</w>': 4, '   ch er ri es</w>': 6, '   ca me lot</w>': 14, '   h un gar i ans</w>': 7, '   ca th o li c s</w>': 4, '   b lu b ber ing</w>': 6, '   g ri ev ing</w>': 6, '   in sti tu tions</w>': 6, '   ru by</w>': 44, '   b ani ster</w>': 12, '   h om o se x u a li ty</w>': 4, '   mar t</w>': 7, ' 5 9 </w>': 3, '   di sen chan ted</w>': 3, '   at su g i</w>': 1, '   un t ou cha ble</w>': 3, '   h in ted</w>': 5, '   su mm it</w>': 7, '   k h ru sh ch ev </w>': 3, '   ei sen h ower</w>': 5, '   pu bi c</w>': 2, '   dro p out</w>': 2, '   o ver tly</w>': 1, '   mar x i st</w>': 1, '   f li gh ts</w>': 11, ' 1 5 0 0</w>': 2, ' 2 0 3 </w>': 1, '   ho g</w>': 14, ' ru m or</w>': 3, '   di sc re d it</w>': 6, '   co tt on ba lls</w>': 1, '   re gi st er ing</w>': 3, '   s ea wa ll</w>': 1, '   p on tch ar tra in</w>': 2, '   cl in ton</w>': 8, '   vo ter</w>': 3, ' kee fe</w>': 8, '   e di t ors</w>': 11, '   f l ow er y</w>': 2, ' bea t ni k</w>': 1, '   ca g es</w>': 12, '   de fro cked</w>': 2, ' 6 2 </w>': 6, '   ma s qu er a de</w>': 8, '   si gh s</w>': 6, '   sa d den ed</w>': 1, ' b ru tu s</w>': 1, '   ex tra di tions</w>': 1, '   su b po en as</w>': 1, '   l y n don</w>': 10, '   dre dge</w>': 4, '   ca m</w>': 5, '   ran h</w>': 1, '   op en ers</w>': 6, '   in vo l ve ment</w>': 16, '   e li min ate</w>': 17, '   ma g ni tu de</w>': 9, '   s che du les</w>': 7, ' st y le</w>': 2, ' e t at</w>': 4, '   mar ce llo</w>': 4, '   s an to s</w>': 35, '   tra f fi can te</w>': 3, '   ho ff a</w>': 5, ' wor ms</w>': 1, '   to e hold</w>': 1, '   te le x</w>': 2, '   st on er</w>': 3, '   as su mp tions</w>': 5, '   de po si t ory</w>': 4, '   en fi e ld</w>': 1, '   p ho to gra ph ed</w>': 5, '   ma u s er</w>': 1, '   we i t z man</w>': 1, '   pa t sy</w>': 15, '   a li as</w>': 9, '   hi d ell</w>': 1, '   po st bo x</w>': 1, '   de fe ctor</w>': 2, '   cor ro bor ation</w>': 1, '   5 4 4 </w>': 1, '   un sig ned</w>': 3, '   i o d ine</w>': 2, '   as cer tain</w>': 2, '   pro lo id</w>': 2, '   an e u r y s m</w>': 4, '   ca u ses</w>': 34, '   h y p no s is</w>': 8, '   go l d ber g</w>': 1, ' o s wa ld</w>': 4, ' al an</w>': 1, '   youn g b loo d</w>': 2, '   ro se ll i</w>': 7, '   ba g man</w>': 3, ' mon go ose</w>': 1, '   s ou th ea st</w>': 7, '   w ea ve</w>': 5, '   com pro mi sing</w>': 1, '   co ck su ck in</w>': 1, '   su b cu t an e ous</w>': 1, '   c ro ss brea ding</w>': 1, ' give</w>': 21, '   f on ta in b lea u</w>': 2, '   ma g go ts</w>': 5, '   ri d dle</w>': 25, '   en i g ma</w>': 4, ' un t ou cha ble</w>': 1, '   im per son a tor</w>': 2, '   bl ack ma i ling</w>': 8, '   e la di o</w>': 11, '   v a ll e</w>': 4, '   pa y ma ster</w>': 3, ' le on</w>': 4, '   in con ven i en tly</w>': 1, '   sho t g un s</w>': 13, '   g al ve st on</w>': 2, '   th under st or ms</w>': 1, '   stra i gh ten ing</w>': 9, '   gra s sho pp ers</w>': 8, '   car on do let</w>': 1, '   con gra tu la ted</w>': 2, '   ad mi ra ls</w>': 3, '   k en ne y</w>': 1, '   pa th o lo gi st</w>': 5, '   hu mes</w>': 1, '   sta ting</w>': 2, '   fin ck</w>': 2, '   dis se ct</w>': 5, '   ha bi gh or st</w>': 1, '   ru ling</w>': 13, '   in ad mi ssi ble</w>': 4, '   im me mor i al</w>': 1, '   re qui re ment</w>': 4, '   ja sp er</w>': 18, '   fe l on</w>': 8, '   a ll e ged</w>': 12, '   h ind</w>': 6, '   mar gu er i te</w>': 9, '   an g ri er</w>': 5, '   e cho es</w>': 4, '   s na p sho ts</w>': 4, '   b ack gr oun ds</w>': 2, '   su b po en a ed</w>': 6, '   par k land</w>': 1, ' e cho es</w>': 1, '   ho st y</w>': 1, '   mar in a</w>': 5, '   ba ha ma s</w>': 14, ' mi c ro do ts</w>': 1, '   min o x</w>': 1, '   di z</w>': 40, ' 2 0 1 </w>': 1, '   o s wal ds</w>': 3, '   in c ri min a ting</w>': 6, '   1 9 6 1 </w>': 3, '   bo l ton</w>': 2, '   dea l er ship</w>': 4, '   in cor por ation</w>': 1, '   br en n an</w>': 1, '   ta pp ing</w>': 4, '   de sc en d ant</w>': 1, '   ca bo ts</w>': 1, '   d or n ber ger</w>': 1, '   ro ck e ts</w>': 11, '   pre su ma b ly</w>': 5, '   st ac king</w>': 2, '   te x ts</w>': 3, '   mo h r en sc hi l d t</w>': 1, '   el m</w>': 11, '   o il man</w>': 1, '   ti es</w>': 29, '   in con g ru ous</w>': 1, '   ch u m my</w>': 3, ' ft</w>': 1, '   ja g g ars</w>': 1, ' chi les</w>': 1, '   sto v all</w>': 1, '   p ho to gra p hi c</w>': 8, '   con tr ac ts</w>': 34, '   bl ack list ed</w>': 1, '   le f ti st</w>': 2, '   af fi li a tions</w>': 1, '   di sp a tch ing</w>': 2, '   min ks</w>': 6, '   s wee the ar ts</w>': 2, '   de fe c ted</w>': 1, '   sp as</w>': 1, '   ra i kin</w>': 1, ' com m uni st</w>': 2, '   mi sc re ant</w>': 1, '   pro se cu ted</w>': 3, '   re v ea ling</w>': 6, '   si de tr ac ked</w>': 7, '   sp ok e s man</w>': 4, '   ro om y</w>': 2, '   1 2 0 0</w>': 3, '   u ss r</w>': 2, '   2 0 1 </w>': 3, '   su n ri se</w>': 21, '   for e most</w>': 6, '   ra in dro ps</w>': 2, '   pi pe l ine</w>': 15, '   sch lu mber</w>': 1, '   hou ma</w>': 1, '   ga u ll e</w>': 2, '   su ing</w>': 9, '   cen tr o</w>': 1, '   mon do</w>': 4, '   com mer ci a le</w>': 1, ' e spi on age</w>': 1, '   p an ac he</w>': 1, '   im pe c ca ble</w>': 6, '   con sor ting</w>': 2, '   att or ne ys</w>': 9, '   par ti ed</w>': 2, '   p ff ft</w>': 1, '   g ri my</w>': 2, '   ho o d lu ms</w>': 2, '   e le g an ce</w>': 2, '   s me d le y</w>': 1, '   un cou th</w>': 1, ' fran ki e</w>': 1, '   uni for med</w>': 3, '   chi pp en da le</w>': 1, '   sp l en d or</w>': 1, '   da u ph ine</w>': 1, '   ru s so</w>': 3, '   ac qu a in ted</w>': 17, '   con du c ting</w>': 9, '   mer c er</w>': 4, ' mer c er</w>': 1, '   s l ou ch ed</w>': 1, '   no t ar i z ed</w>': 4, '   no t ary</w>': 4, '   under pa ss</w>': 1, '   dea le y</w>': 4, '   o st ri ch</w>': 15, ' tri al</w>': 2, '   ch ore</w>': 4, '   sti r r ing</w>': 5, '   ki d na pp er</w>': 4, '   ar can e</w>': 3, '   din o sa u rs</w>': 27, '   sy m pa th i es</w>': 3, '   inter vi e w ing</w>': 8, '   re st or ation</w>': 1, '   vo lu mes</w>': 7, '   t ou ch down</w>': 7, '   te sti fi es</w>': 3, ' hear</w>': 8, '   inter ro ga ted</w>': 5, '   pre si den ts</w>': 10, '   su m bi tch</w>': 8, '   f ac i s m</w>': 1, '   so di u m</w>': 8, '   pen to th al</w>': 4, '   su per vi sion</w>': 2, '   li mp ed</w>': 2, '   su n n in</w>': 3, '   u l ti ma tu ms</w>': 1, '   u l ti ma tu m</w>': 4, '   in fi gh ting</w>': 2, ' 5 3 </w>': 1, ' 6 1 </w>': 6, ' tra i tor</w>': 1, '   ear le</w>': 3, '   sp o tter</w>': 4, '   ri f le men</w>': 3, ' ab or t</w>': 2, '   z a p ru der</w>': 3, '   sh e ds</w>': 1, '   8 8 </w>': 3, '   fo li age</w>': 2, '   sh ar p sh oo t ers</w>': 1, '   de fe c tive</w>': 8, '   re c y c le</w>': 3, '   th y ro id</w>': 1, '   1 9 6 6 </w>': 4, '   vi ra l</w>': 5, '   can c ers</w>': 2, '   de tri ck</w>': 1, '   un thin k able</w>': 3, '   mo b bed</w>': 4, '   clo cked</w>': 5, '   tr ac ea ble</w>': 2, '   ch er a mi e</w>': 1, '   mi d lo th i an</w>': 1, '   f ra y ed</w>': 1, '   ha ir cu ts</w>': 3, '   sha ves</w>': 1, '   hea d set</w>': 3, '   ho bo es</w>': 5, '   fin es</w>': 3, '   o s er</w>': 1, '   hi st ori es</w>': 4, '   on i</w>': 4, ' f b i</w>': 2, '   5 3 1 </w>': 1, '   can ned</w>': 13, ' 6 8 </w>': 6, '   v ar min t</w>': 2, '   of f ra mp s</w>': 1, '   ru s se ll</w>': 11, '   k o ok y</w>': 5, '   p ra i ri e</w>': 7, '   in ve sti ga t in</w>': 3, '   pre ci sion</w>': 9, '   ac cor d in</w>': 1, '   ma g gi e</w>': 40, '   z i g z a g ging</w>': 1, '   con na lly</w>': 1, ' pri st ine</w>': 1, ' fi red</w>': 6, '   g n at</w>': 2, '   po le c at</w>': 4, '   con c ea ling</w>': 2, '   in no c ence</w>': 22, '   re pu ta ble</w>': 4, '   u r ged</w>': 3, '   per ju ry</w>': 7, '   me mor an du m</w>': 2, '   lo dge</w>': 16, '   as sa s sin a tions</w>': 3, ' pro te ction</w>': 2, '   cre di bi li ty</w>': 15, '   lu mu mb a</w>': 2, '   tru j i llo</w>': 2, '   d om in i can</w>': 4, '   an n oun ces</w>': 3, '   ro of to p</w>': 7, '   con tr ac t ors</w>': 5, '   bo ar d room</w>': 2, '   l un ch room</w>': 1, '   b on n</w>': 2, '   spe cu la ting</w>': 2, '   bu b b a</w>': 9, ' cra z ed</w>': 2, '   car i ca ture</w>': 1, '   shi t st or m</w>': 6, ' bu b b a</w>': 1, '   ga g ged</w>': 1, '   hi st ori c</w>': 5, '   ex i les</w>': 3, '   sto ck pi ling</w>': 2, ' e ye</w>': 8, ' whi pped</w>': 5, ' 3 5 7 </w>': 3, '   fr on ts</w>': 3, '   fi sh er man</w>': 14, '   con tra di ction</w>': 8, '   con ce p tion</w>': 10, '   ever est</w>': 7, '   a st oun ding</w>': 4, '   en l ar ge ment</w>': 2, '   pro fu sion</w>': 1, ' b on ding</w>': 1, '   in di ca ted</w>': 6, '   ser u m</w>': 8, '   pr on oun ce</w>': 6, '   re tri al</w>': 1, '   ba da ss</w>': 5, '   k or e ans</w>': 5, '   ro s co e</w>': 4, '   wa ff les</w>': 4, '   pe an u th ead</w>': 1, '   ni g g as</w>': 7, '   c li mb in</w>': 3, '   gi tt in</w>': 4, '   k or ea town</w>': 2, '   a si ans</w>': 3, '   a a a a a w w w</w>': 1, '   plan n in</w>': 5, '   ho p in</w>': 8, '   fa tty</w>': 3, '   ki ll in</w>': 32, '   or d ell</w>': 43, '   man il a</w>': 5, '   li v in g st on</w>': 14, '   sy bi ll</w>': 1, '   de f en der</w>': 7, '   ro b bi e</w>': 6, '   ni co let</w>': 16, '   1 9 8 5 </w>': 6, '   tw a</w>': 2, '   shi t ti est</w>': 1, '   who pp ing</w>': 2, '   hu ll u v a</w>': 1, '   ce dri c</w>': 1, '   wal k er</w>': 36, '   ca b o</w>': 6, '   dar gu s</w>': 4, '   ma dri d</w>': 6, '   con fi s ca ted</w>': 8, ' w o e fu lly</w>': 1, '   wan t only</w>': 1, '   chi ck en ed</w>': 8, '   li ter al</w>': 4, '   ra tion a li z ing</w>': 1, '   mi l de w</w>': 1, '   mon te</w>': 5, '   bu ll shi tt in</w>': 5, '   p lo tting</w>': 11, ' spe ed</w>': 4, '   bu ll o ck</w>': 3, '   c ru sh es</w>': 5, '   b en ing</w>': 1, ' 7 4 </w>': 1, ' 7 6 </w>': 2, '   de l f on i c s</w>': 1, '   wi ll in g ne ss</w>': 4, '   so l v in</w>': 1, '   bo o t le g ger</w>': 6, '   s mu g g l in</w>': 1, '   star k y</w>': 1, '   lu cked</w>': 4, '   da vi do ff s</w>': 1, '   hi l ton</w>': 8, '   han g out</w>': 1, '   ri ver bo tt om</w>': 1, '   sh er on da</w>': 9, '   s na z zy</w>': 1, '   me l ani e</w>': 39, '   gar a</w>': 4, '   po l y gra p h</w>': 4, '   tu ss le</w>': 5, '   sa le s woman</w>': 1, '   comp r en de</w>': 4, ' fi l ed</w>': 1, '   bu tt on ed</w>': 2, '   ra l st on</w>': 1, ' comp ton</w>': 1, '   si m one</w>': 93, '   ha w k ins</w>': 17, '   su s an vi ll e</w>': 4, '   h er mo s a</w>': 2, '   im m uni ty</w>': 12, '   tra f fi cking</w>': 6, '   po se ssion</w>': 1, '   ja e ger</w>': 19, '   ki lli an</w>': 1, ' bu ddy</w>': 2, '   pa ging</w>': 2, '   p ee ks</w>': 1, '   s ca ven ger</w>': 3, '   pa ss in</w>': 7, '   f ac in</w>': 1, '   man a ging</w>': 12, '   a m o</w>': 5, '   lu r king</w>': 4, '   co ck a too</w>': 3, '   ro o ster</w>': 10, ' a g ine</w>': 2, '   sc re w dri ver</w>': 6, '   mi ch i</w>': 1, '   hi ro sh</w>': 1, ' f li pp ers</w>': 1, '   dis co s</w>': 4, '   bu il d ers</w>': 3, '   me tri x</w>': 2, '   cou gh ing</w>': 6, '   ca pi ll ar i es</w>': 1, '   cou gh s</w>': 2, '   g un n</w>': 1, '   b on ded</w>': 3, '   dri v in</w>': 32, '   dea l er shi ps</w>': 1, '   mo ther fuck in</w>': 9, ' we en</w>': 2, '   bar e f oo t</w>': 4, '   mer chan di se</w>': 9, '   pen e ten ti ary</w>': 1, '   lo ck bo x es</w>': 1, '   l en o</w>': 3, ' b le ep</w>': 1, '   mi lli me ter</w>': 8, '   we s son</w>': 4, '   5 9 4 6 </w>': 1, '   under co ver</w>': 33, '   so on est</w>': 2, '   k nee ca p</w>': 1, '   go damn</w>': 3, '   man din go</w>': 1, '   sp o ok ed</w>': 13, '   du m ba ss</w>': 13, '   ra mm ing</w>': 2, '   loo k y</w>': 3, '   in de m ni ty</w>': 3, ' s mo ke</w>': 1, '   r ou st</w>': 7, '   sh er on a</w>': 1, '   cont in g ent</w>': 2, '   pro mi s sor y</w>': 2, '   s ki ps</w>': 4, '   1 4 3 6 </w>': 1, '   9 0 2 2 2 </w>': 1, '   di sp o si tion</w>': 10, '   con c ea l ed</w>': 9, '   un re gi st er ed</w>': 1, ' po s se ssion</w>': 1, '   ch un k</w>': 13, '   as h</w>': 15, '   sho pp in</w>': 2, ' o l a</w>': 2, '   je z z i e</w>': 9, '   sh r in ks</w>': 5, '   h y p no ti st</w>': 2, '   pen e tra tes</w>': 3, '   ge ary</w>': 4, '   ga be</w>': 26, '   th a il and</w>': 12, '   n an g</w>': 1, '   bi z ar r o</w>': 3, '   ma il man</w>': 6, '   ma son</w>': 61, '   in ju sti ces</w>': 1, '   w oo d work</w>': 4, '   bu tt in</w>': 1, '   car l son</w>': 7, '   me i ster</w>': 1, '   e ck art</w>': 2, '   do c t or ate</w>': 4, '   dis c</w>': 33, '   sa ver</w>': 7, '   er ro l</w>': 2, '   f l y n n</w>': 27, '   o ver gr own</w>': 3, '   ch er u b</w>': 1, '   ad ju st ment</w>': 12, '   ph d</w>': 3, '   cla ms</w>': 6, '   p hi lo sp h ers</w>': 1, '   in te ll e c tu </w>': 1, '   ally</w>': 17, '   ence</w>': 2, '   t le</w>': 1, '   pi l g ri ma ge</w>': 3, '   gu i d es</w>': 10, '   su d</w>': 2, '   den ly</w>': 1, '   be ll v u e</w>': 1, '   e m pi ri cal</w>': 3, '   li eve</w>': 1, '   en ter na l</w>': 1, '   pr of</w>': 10, '   ter</w>': 2, '   r en e ga de</w>': 2, '   ex i st en ti a li st</w>': 2, '   su f</w>': 1, '   f er ing</w>': 1, '   p sy cho s is</w>': 5, '   p sy</w>': 2, '   chi a tri st</w>': 1, '   in v a ding</w>': 5, '   co b</w>': 1, '   f le sh y</w>': 1, '   li ter ary</w>': 14, '   dan te</w>': 34, '   au gu s</w>': 1, '   ki er ke ga ard</w>': 1, '   fi er y</w>': 3, '   m in</w>': 6, '   u te</w>': 1, '   po s i</w>': 2, '   tion</w>': 4, '   tu sc on</w>': 1, '   do c t or a tes</w>': 2, '   mi d town</w>': 1, '   sle et</w>': 3, '   po st al</w>': 11, '   wor k er</w>': 56, '   be e f y</w>': 7, '   je z e be l</w>': 5, '   bl an ke ts</w>': 10, '   or th o pe di c</w>': 4, '   chi ro pr ac tor</w>': 5, '   che mi st</w>': 4, ' de lu sion</w>': 4, '   je z</w>': 9, '   lin o is</w>': 1, '   ter e st ing</w>': 1, '   cle ans</w>': 7, '   hel li sh</w>': 4, '   do d ged</w>': 3, '   sa i g on</w>': 8, '   bro od</w>': 3, '   man han d l er</w>': 1, '   ex or ci sed</w>': 2, '   ni gh t in ga le</w>': 11, '   pi p kin</w>': 1, '   re cu per ate</w>': 1, '   s an de l man</w>': 1, ' s ar ah</w>': 1, '   1 0 7 </w>': 2, '   win o s</w>': 3, '   bar ri ca ded</w>': 2, '   c ru sh ing</w>': 6, '   el i</w>': 9, '   j ed</w>': 4, '   hea d bo ard</w>': 2, '   ro ber ts</w>': 13, ' ye a</w>': 2, '   shi tt in</w>': 4, '   che mi ca lly</w>': 3, '   con g</w>': 8, '   in fin te ssi ma l</w>': 1, '   e f fe c ti ven ess</w>': 1, '   ra ti o</w>': 6, '   t n ought</w>': 1, '   h un t le y</w>': 2, ' br in k le y</w>': 1, '   fu ry</w>': 10, '   sy n the si z ing</w>': 1, '   i so la ting</w>': 3, '   co lon el s</w>': 3, '   me da ls</w>': 8, '   cl in i cal</w>': 10, '   no st r and</w>': 3, '   ga sh</w>': 3, '   spi lling</w>': 10, '   de li ri u m</w>': 2, '   l en no x</w>': 1, '   s ea ms</w>': 4, '   fo ll ow in</w>': 9, '   vi s or</w>': 2, '   br ow n s vi ll e</w>': 1, '   gr un e ger</w>': 2, '   ho ck ey</w>': 31, '   de lon g p re</w>': 4, '   de pre s su ri z ing</w>': 1, '   o ver hea ting</w>': 1, '   min i ma l</w>': 13, '   wal k ways</w>': 2, '   ri z z o</w>': 18, '   ja so</w>': 1, '   de pre s su ri ze</w>': 1, '   c ru tch</w>': 4, '   un con ci ous</w>': 2, '   wal k way</w>': 6, '   dam n it</w>': 30, '   th or g an</w>': 6, '   k k in s a</w>': 4, '   k ay</w>': 71, '   j an e ss a</w>': 1, '   con ver t ers</w>': 1, '   imp lo ding</w>': 1, '   bri g gs</w>': 1, '   ge k o</w>': 1, '   bo x ed</w>': 3, '   fu se la ge</w>': 1, '   bo e man</w>': 2, '   fi ssion</w>': 3, '   tr an si st ors</w>': 1, '   bro d s k i</w>': 3, '   y llo</w>': 8, '   de c r y on i z ation</w>': 1, '   br in ing</w>': 1, '   re ju ven ate</w>': 1, '   cu r sed</w>': 21, '   th a w ing</w>': 1, '   vi su al s</w>': 3, '   no sed</w>': 2, '   st ee l b al de</w>': 1, '   gr en de l</w>': 20, '   happ end</w>': 6, '   imp or ts</w>': 5, '   rea c t ors</w>': 5, '   au di ome t ers</w>': 1, ' sh oo t ers</w>': 1, '   med</w>': 9, ' k it</w>': 2, '   sh ow t un es</w>': 1, '   t i</w>': 14, '   gr un ts</w>': 1, '   ba d de st</w>': 3, '   m ac he te</w>': 7, '   in tri ca te</w>': 3, '   f ar ts</w>': 9, '   k ab oo m</w>': 2, '   y u ck</w>': 2, '   cha m pa i g n</w>': 1, '   hea d ge ar</w>': 1, '   t ar a</w>': 1, '   so l ar as</w>': 1, '   st on ey</w>': 3, '   h re</w>': 1, '   di min i sh</w>': 2, '   shu tt t le</w>': 1, '   be ow u lf</w>': 4, '   ga d ge ts</w>': 4, '   te le por ta tion</w>': 2, '   en tra i ls</w>': 3, ' c p o</w>': 1, '   c y ber ne ti c s</w>': 1, '   dro id</w>': 7, '   re pro gra mm ed</w>': 4, '   di sen ga ged</w>': 1, '   pro gra mm ing</w>': 16, '   sor y</w>': 1, '   dro i ds</w>': 2, '   k no ff la p od</w>': 1, '   c r y o</w>': 7, '   r en der</w>': 10, '   vi ru ses</w>': 10, '   ex cu r sion</w>': 3, '   cre di ts</w>': 6, '   b y pa ss ing</w>': 2, '   f ar r ar i</w>': 1, '   in te ll e g ent</w>': 1, '   o ver rea c ting</w>': 8, '   a z ra el</w>': 1, ' sta t is</w>': 1, '   p r</w>': 9, '   n an o</w>': 8, '   n an o te ch no lo g y</w>': 3, '   o out</w>': 1, '   th s i</w>': 1, '   th se</w>': 1, '   p on ds</w>': 8, '   s an ded</w>': 2, '   a st er n</w>': 3, '   z i g</w>': 3, ' z a g</w>': 2, '   bi l ge</w>': 2, '   ta tt o o</w>': 25, ' i m</w>': 45, '   app en di x</w>': 4, '   i w o</w>': 1, '   j im a</w>': 1, '   un bu ck le</w>': 1, '   char ter</w>': 9, '   o ce an o gra p hi c</w>': 4, '   ho op er</w>': 10, '   a pri co t</w>': 3, '   z on ing</w>': 2, '   t ow n ship</w>': 1, '   a mi ty</w>': 11, '   su n t an</w>': 4, '   b list ex </w>': 1, '   si lli er</w>': 1, '   mu g g ers</w>': 4, '   th ri ft</w>': 4, '   mar ci a</w>': 17, '   v au gh n</w>': 8, ' y a w k</w>': 1, '   y a h d</w>': 1, '   ca h</w>': 1, ' y a h d</w>': 1, '   au th ori ze</w>': 7, '   qu int</w>': 3, '   v ou ch er</w>': 5, ' op en</w>': 4, '   ge o gra p hi c</w>': 8, ' cor ro si ve</w>': 2, '   me sh</w>': 3, '   ne tting</w>': 2, '   m ea d ows</w>': 7, '   out si d ers</w>': 5, '   s ea son al</w>': 2, '   sp o tt ers</w>': 3, '   fi sh er men</w>': 5, '   pu ri t ans</w>': 1, '   s mor ga s bor d</w>': 2, '   co d</w>': 3, '   wh a le</w>': 37, '   in di an a po l is</w>': 3, '   s wi ms</w>': 3, '   t ow ed</w>': 4, '   sti mu la ted</w>': 2, '   s wi mm ers</w>': 3, '   man ea ter</w>': 2, '   din gh y</w>': 1, '   li fe j ac ke ts</w>': 1, '   c lu mp ed</w>': 1, '   ea ter</w>': 7, ' fin der</w>': 1, '   fa th ome ter</w>': 1, '   son ar</w>': 8, '   fore</w>': 4, '   af t</w>': 13, '   r d f</w>': 1, '   fee d ers</w>': 2, '   f l oun der</w>': 3, '   bur la p</w>': 1, '   h ome coming</w>': 14, '   shi p bu il d ers</w>': 1, '   bea t ni k</w>': 6, '   re sor ts</w>': 4, '   ter ri t ori a li ty</w>': 2, ' i ss ing</w>': 1, '   ma k o s</w>': 1, '   ha m mer hea ds</w>': 1, '   bo z o s</w>': 4, '   be g in ner</w>': 3, ' ac ci dent</w>': 7, '   pro pe ll er</w>': 3, '   cor al</w>': 7, '   st er nu m</w>': 2, '   por tions</w>': 3, '   ri b ca ge</w>': 3, '   mor te m</w>': 2, '   ab ra si ons</w>': 1, '   s qu al i</w>': 2, '   car chan in us</w>': 1, '   lon i man us</w>': 1, '   i su ru s</w>': 1, '   gla u ca s</w>': 1, ' mor te m</w>': 1, '   er o sion</w>': 1, '   sur f ac es</w>': 1, '   ja w s</w>': 7, ' he ight</w>': 1, '   e sti ma ted</w>': 7, '   t or so</w>': 8, ' th or a x</w>': 1, '   ev i sc er a ted</w>': 1, '   den u ded</w>': 1, '   f an ta il</w>': 1, '   brea k wa ter</w>': 1, '   f er ry</w>': 11, '   wor thin g s ly</w>': 1, '   en qu ir er</w>': 9, '   k in t ner</w>': 1, '   den h er der</w>': 1, ' bea ch</w>': 3, '   a v ri l</w>': 2, '   may he w</w>': 4, '   si r lo in</w>': 3, '   ni b b les</w>': 3, '   o tt ers</w>': 1, '   cou st ea u</w>': 2, '   com pa g no</w>': 1, ' f und ed</w>': 3, ' au r or a</w>': 1, '   or c a</w>': 1, ' pu sh</w>': 2, '   ther ea b ou ts</w>': 3, '   ba tt en</w>': 3, '   c c s</w>': 1, '   st r y ch n ine</w>': 2, '   1 9 4 4 </w>': 3, '   di sh on or</w>': 6, ' e mp er</w>': 1, '   u h h h</w>': 10, '   t read</w>': 10, ' ma k o</w>': 1, '   a mu se men ts</w>': 2, '   fr en chi e</w>': 1, '   s wee t p ea s</w>': 1, '   mor ay</w>': 4, '   ee l</w>': 8, '   b ack stay</w>': 1, '   st in gra y</w>': 2, '   su per car go</w>': 3, '   b ack st ro ke</w>': 1, '   re pe l</w>': 2, '   bo ar d ers</w>': 2, '   ma in sa il</w>': 3, '   whi te y</w>': 10, '   com pre ssed</w>': 4, '   app e ti z er</w>': 4, '   por k er</w>': 1, '   y a p</w>': 1, ' sh ar k</w>': 2, '   har po on er</w>': 1, '   st ow</w>': 4, ' 5 0 0 0</w>': 1, ' 2 0 0 0</w>': 2, '   ki ddy</w>': 1, '   s ci s sor s</w>': 5, '   s ea man ship</w>': 1, '   por k ers</w>': 1, '   sh ee p sh an k</w>': 1, '   do g fi sh</w>': 1, '   bo a ting</w>': 11, '   cre w ed</w>': 4, ' p ac s</w>': 1, '   de li ber ate</w>': 7, '   mu ti la tion</w>': 3, '   car car a don</w>': 1, '   car char i as</w>': 2, '   ju no</w>': 37, '   m ac gu ff</w>': 6, '   co f fee house</w>': 2, '   lin ki es</w>': 1, '   star r ye y ed</w>': 1, '   st ee p le</w>': 2, '   fu ll time</w>': 1, '   s mi tt en</w>': 4, '   wi z ard</w>': 43, '   8 0</w>': 21, ' j un e bu g</w>': 1, '   bl ack bo ard</w>': 5, '   fa ve</w>': 1, ' ca l ori e</w>': 1, '   b le e k</w>': 6, '   k a tr in a</w>': 18, '   p ack er</w>': 2, '   b la ir</w>': 17, '   star z</w>': 1, '   ro y ally</w>': 4, '   ti cked</w>': 1, '   ch ee sed</w>': 1, '   st in ke ye</w>': 2, '   b en i han a</w>': 2, '   vi j ay</w>': 3, '   l eah</w>': 5, '   vo or t</w>': 4, '   t in o</w>': 2, '   d ru m head</w>': 1, '   br en</w>': 6, '   a do p tion</w>': 5, '   u l tra s ound</w>': 5, '   c in e p le x</w>': 3, '   dea d be at</w>': 14, '   in tru si ve</w>': 1, '   car o le</w>': 3, '   b lea ch</w>': 5, '   ran ci ck</w>': 1, '   ev en ly</w>': 2, ' ti t le</w>': 1, '   shi t ti er</w>': 1, '   b le e k er</w>': 8, '   we in er</w>': 3, '   sa di sts</w>': 3, '   di la ted</w>': 4, '   fu c ki ty</w>': 1, ' ow</w>': 4, '   we i mar an ers</w>': 1, ' du h h h</w>': 1, '   o ver st e pp ing</w>': 1, '   v a g</w>': 2, '   o ver st e pped</w>': 2, '   b oun d ary</w>': 8, '   u p da ted</w>': 4, ' de li ver</w>': 3, '   pr en a tal</w>': 1, '   ex pu l sion</w>': 2, '   u r n</w>': 6, '   sti ll wa ter</w>': 2, '   g un k</w>': 1, '   st e p mo m</w>': 6, '   s mi r no ff</w>': 1, '   i ces</w>': 1, '   man k a to</w>': 2, '   bu g let</w>': 1, '   cu sto di ans</w>': 1, '   n er ds</w>': 2, '   vo ge l</w>': 1, '   f ru i t ful</w>': 3, '   y u k i</w>': 2, '   i g g y</w>': 5, '   st oo g es</w>': 3, '   i po ds</w>': 1, '   s wa g</w>': 3, '   pe l v is</w>': 2, '   ver ba lly</w>': 3, '   di f fu sing</w>': 1, '   con tri bu ting</w>': 3, '   we i gh ed</w>': 4, '   di an a</w>': 6, '   do ve ta i ling</w>': 1, '   su spi ri a</w>': 1, '   s la sh er</w>': 1, '   ar gen to</w>': 2, '   y in</w>': 4, ' y an g</w>': 1, '   dar i o</w>': 1, '   h er s che l</w>': 1, '   pa tt i</w>': 1, '   run a ways</w>': 2, '   car pen t ers</w>': 4, '   al t</w>': 3, '   st ri p es</w>': 5, ' su per st ar</w>': 1, '   ne u ter</w>': 2, ' bab i es</w>': 1, '   a mor ph ous</w>': 1, '   me l v ins</w>': 3, '   jo han n es</w>': 1, '   bra h ms</w>': 2, ' p ac ks</w>': 1, ' in fu sed</w>': 1, '   ju i ces</w>': 3, '   g in sen g</w>': 1, ' su ck</w>': 3, '   ac c ru e</w>': 1, '   ki mber</w>': 2, ' 9 6 </w>': 2, '   s no b</w>': 1, '   ro ck er</w>': 6, '   gi b son</w>': 29, '   a sa p</w>': 6, '   po op ing</w>': 2, '   cl in i qu e</w>': 1, '   k le p to</w>': 2, '   who op s</w>': 9, '   ki ck in</w>': 8, '   who ah</w>': 3, '   re e ds</w>': 3, '   g ee e er ta</w>': 1, '   ra u u u ss</w>': 1, '   ger ta</w>': 3, '   ra u ss</w>': 3, '   f rea k out</w>': 1, '   ri d ge da le</w>': 1, ' b la a a a ah</w>': 1, '   k ra k en</w>': 1, '   ad der all</w>': 1, '   su </w>': 13, ' ch in</w>': 2, '   fu r ni sh in gs</w>': 2, '   ca u tion ary</w>': 3, '   s no o ze</w>': 2, '   f la ke</w>': 7, '   p ha lan g es</w>': 1, '   cu ter</w>': 7, '   1 0 4 </w>': 2, ' ti l ed</w>': 1, '   s ea bi s cu it</w>': 1, '   tri me ster</w>': 1, '   pi c</w>': 2, '   con gra ts</w>': 1, '   bo y sen ber ry</w>': 1, '   th under ca ts</w>': 1, '   pi ck le</w>': 9, '   s k an k y</w>': 4, '   s ke ev y</w>': 1, '   cou pl es</w>': 14, '   mor ose</w>': 6, '   to t</w>': 4, '   to ts</w>': 2, ' e qui pped</w>': 1, '   ca vi ti es</w>': 2, '   f lu ori da ted</w>': 1, '   ra di a ting</w>': 2, ' o d d</w>': 4, '   pre vi a</w>': 1, '   s lu shi e</w>': 1, '   h v ac </w>': 1, '   ha v a su </w>': 1, '   o c to p us</w>': 10, '   ha z m at</w>': 1, '   pro g no s is</w>': 1, '   my r t le</w>': 3, '   un plan ned</w>': 1, '   pre g n an ci es</w>': 2, '   de li gh ts</w>': 1, '   si l en ci o</w>': 1, '   sp er ms</w>': 1, '   w en i ses</w>': 1, '   li p ton</w>': 1, '   no o ki e</w>': 4, '   p f ft</w>': 1, '   e w w</w>': 3, '   be ar dy</w>': 1, '   ke i th</w>': 23, '   con y ers</w>': 1, ' ke i th</w>': 1, '   he i gh ten ed</w>': 3, '   gra ding</w>': 2, '   ge ome try</w>': 5, '   con v ex </w>': 1, '   fun ba gs</w>': 1, '   sp er my</w>': 1, '   ke bo b</w>': 1, '   l or in gs</w>': 1, '   l or ing</w>': 1, '   h ly</w>': 1, '   sh h t</w>': 1, '   pre t z el</w>': 7, '   a do p tive</w>': 2, '   y ee sh</w>': 1, ' hea l th y</w>': 2, ' thir ti es</w>': 1, '   dre ss es</w>': 23, '   g un play</w>': 2, '   in ce st</w>': 3, ' who le some</w>': 2, '   e d gi er</w>': 1, '   wan na be</w>': 3, ' de sp er a tely</w>': 2, '   sp a w n</w>': 2, '   i gu an as</w>': 1, '   ter ri ers</w>': 1, '   can on i ze</w>': 1, '   mi l k ta te</w>': 1, '   o v ary</w>': 4, '   le</w>': 30, '   g ru e some</w>': 7, '   sp ri t z</w>': 1, '   re ci p es</w>': 3, '   re ce p tion i st</w>': 8, '   k u ah</w>': 1, '   hu mp ing</w>': 5, '   b on y</w>': 3, '   pre me di ta ted</w>': 7, '   ha ven bro o ke</w>': 2, '   p hu k et</w>': 1, '   sho ck in g ly</w>': 3, '   ca v a li er</w>': 6, '   sp out</w>': 3, ' y i g gi ty</w>': 1, '   pi la tes</w>': 1, ' co ll ab or a tive</w>': 1, '   ti cking</w>': 17, '   f la g ged</w>': 6, ' da ddy</w>': 3, '   t wi gs</w>': 4, '   rea d ying</w>': 2, ' ne st ing</w>': 1, '   cu st ard</w>': 4, ' ne u tra l</w>': 1, '   pa le tt e</w>': 3, '   ch ee se ca ke</w>': 3, '   th a ts</w>': 4, '   a de le</w>': 42, '   re si st in</w>': 2, '   beau ti ci an</w>': 1, '   por t fo li o</w>': 3, '   whi ps</w>': 14, '   gra y ce</w>': 1, ' j us</w>': 3, ' sa ving</w>': 3, '   than king</w>': 10, '   any th</w>': 3, '   lu mp y</w>': 3, '   s qu ar ed</w>': 7, '   di e bo ld</w>': 3, '   to pp a</w>': 2, '   o z</w>': 44, ' mind</w>': 5, '   li ve sy</w>': 1, '   hi tch hi k ers</w>': 3, '   pro spe c t ors</w>': 1, ' the ory</w>': 1, '   bi se c ting</w>': 1, '   da h li a</w>': 3, '   do l ya</w>': 1, '   cu ff s</w>': 8, '   wi el ding</w>': 2, '   bar row</w>': 4, '   g ee z us</w>': 1, '   me lo d ra ma ti c</w>': 5, ' de ser ves</w>': 1, '   who op in</w>': 3, '   d ran kin</w>': 1, '   than g</w>': 5, ' de li ver an ce</w>': 1, ' still</w>': 17, '   ch oo s ers</w>': 1, '   ten u re</w>': 5, '   vi sc er al</w>': 1, '   ve g g in</w>': 2, '   shu r r</w>': 1, ' fini sh ing</w>': 1, ' app re h en si ve</w>': 1, '   sp en d in</w>': 3, '   si e st a</w>': 4, ' j ani tor</w>': 1, ' di e bo ld</w>': 1, '   tr ans mi tt ed</w>': 8, '   bu ff</w>': 30, '   dar cy</w>': 10, '   ti t ti es</w>': 8, '   lu ke</w>': 133, '   han di ca pped</w>': 5, '   st om ac ha c he</w>': 1, '   pe p to</w>': 3, '   ma ssi ve ly</w>': 1, '   bu ms</w>': 9, '   a m pu te es</w>': 1, '   e li ti s m</w>': 3, '   z oo ted</w>': 1, '   mo th a</w>': 2, '   fuck a</w>': 1, '   pro mo tes</w>': 2, '   j ac king</w>': 3, '   s li ces</w>': 7, '   b en ny</w>': 29, '   ca sp er</w>': 18, '   b on ed</w>': 1, ' won der</w>': 1, '   cl it</w>': 1, '   car to on</w>': 16, '   bo o st</w>': 20, '   de f l ower</w>': 3, '   7 6 </w>': 7, ' l ev el</w>': 14, '   ad di c ted</w>': 9, '   pu ber ty</w>': 5, '   p ow d ers</w>': 1, '   bu tt er s co tch</w>': 2, '   pi llow</w>': 21, '   mo th a fuck in</w>': 1, '   j en ni e</w>': 8, '   mo th a fuck a</w>': 3, '   na</w>': 15, '   no then</w>': 1, '   nu ff in</w>': 1, '   du kes</w>': 2, '   ra ve</w>': 3, '   bo y friends</w>': 8, '   bu mp s</w>': 5, '   fi d get</w>': 2, '   e u ph ori c</w>': 1, '   cor n ba lls</w>': 1, '   e ff ex </w>': 1, '   hi v </w>': 5, '   g ran d son</w>': 17, '   stu tter</w>': 8, '   bar ks</w>': 16, '   ru ff</w>': 2, '   dar l en e</w>': 14, '   sh in ning</w>': 1, '   l un ch es</w>': 8, '   e so p ha gu s</w>': 2, '   some t in</w>': 1, '   wh ah</w>': 3, '   di ss de e</w>': 2, '   di ss</w>': 6, '   di g g</w>': 6, '   s wa ll ow ed</w>': 17, '   le t down</w>': 2, '   wa cked</w>': 3, '   b la sted</w>': 13, '   h or ne y</w>': 2, '   f li pp er</w>': 6, '   wi p</w>': 1, '   sa l ty</w>': 3, '   fin ger ing</w>': 2, '   mon o ton ous</w>': 2, '   l en til</w>': 2, '   b ack se at</w>': 7, '   fi r</w>': 1, '   h ome se ar ch ers</w>': 4, '   p ho to st at</w>': 1, '   ex ce p tion ally</w>': 6, '   r ac co on</w>': 10, '   r ac co ons</w>': 1, '   cra f ty</w>': 3, '   fe lon y</w>': 13, '   mor p he us</w>': 37, '   mo du la tor</w>': 2, '   j er e my</w>': 22, '   y u ri li vi ch</w>': 3, '   no l an</w>': 23, '   de b i</w>': 15, '   ru t le ge</w>': 4, '   ca u ca si an</w>': 4, '   o a ks</w>': 9, '   un bi as ed</w>': 1, '   ju x ta po sing</w>': 1, '   g ro p ing</w>': 2, '   hu x le y</w>': 5, '   p un c tu ation</w>': 3, '   an ton</w>': 5, '   t ar a k o ss</w>': 2, '   re he ar se</w>': 5, '   im pro mp tu </w>': 2, '   f ra me work</w>': 3, '   de tr act</w>': 1, '   ir ri ta ting</w>': 6, ' we e</w>': 5, '   win ki e</w>': 2, '   a ma te u ri sh</w>': 1, '   t ac ti c</w>': 5, '   st ro ked</w>': 1, '   1 9 8 6 </w>': 3, '   vi k tor</w>': 63, '   se ven ti es</w>': 5, '   ho ll and</w>': 31, '   han s</w>': 28, '   k or sha u d</w>': 1, '   ba in es</w>': 7, '   t our na ment</w>': 7, '   ra ting</w>': 9, '   cor re late</w>': 3, '   mo de m</w>': 4, '   po ten ti ally</w>': 7, '   su i ted</w>': 5, '   er i c a</w>': 26, '   par ch ee s i</w>': 2, '   lu t z</w>': 5, '   g ran d ma st ers</w>': 1, '   fi b ers</w>': 12, '   win d ow si ll</w>': 2, '   ad he si ve</w>': 2, '   comp on en ts</w>': 4, '   b ru i sing</w>': 1, '   pe l vi c</w>': 3, '   wh ad da ya</w>': 19, '   sh red</w>': 12, '   g ri ds</w>': 3, '   bl an ke ted</w>': 1, '   o l y mp us</w>': 1, '   ea st man</w>': 2, '   dis gu i se</w>': 21, '   f an ta si ze</w>': 6, '   e u ph ori a</w>': 2, '   pr er e cor ded</w>': 1, ' e d mon ds</w>': 1, '   k ar po v </w>': 1, '   op en in gs</w>': 4, '   fu l ton</w>': 4, ' pl u ra l</w>': 1, '   po sed</w>': 8, '   che ss bo ard</w>': 2, '   sh e pp ard</w>': 4, '   di re c t ory</w>': 7, '   imp ly</w>': 11, ' 0 4 </w>': 2, '   p m</w>': 9, '   gre g ory</w>': 26, '   pr ac ti se</w>': 1, '   fri t z</w>': 8, '   tr an sp ose</w>': 1, ' car e fu lly</w>': 4, '   s ar a</w>': 23, '   ba in bri dge</w>': 4, '   6 3 9 </w>': 1, ' 7 3 9 3 </w>': 1, ' pr in ci p al s</w>': 1, '   gra m ma ti ca lly</w>': 1, '   re pla ying</w>': 2, ' ad m it</w>': 1, '   pe pp er on i</w>': 3, '   che ss bo ar ds</w>': 1, '   di st r ac tions</w>': 5, '   con fr on ted</w>': 7, '   cl in i ca lly</w>': 3, '   ob je c ti ves</w>': 1, '   se d man</w>': 1, '   k ri k ori an</w>': 1, '   ci r cu m stan ce</w>': 7, '   ob st ac le</w>': 5, '   v al s ne y</w>': 1, '   co l w y n</w>': 9, '   l y ss a</w>': 22, '   ou tw ard</w>': 3, '   har med</w>': 7, '   par a p et</w>': 1, ' might</w>': 8, '   g ri me</w>': 1, ' re li ev ed</w>': 1, '   gi mp y</w>': 1, '   s la y ers</w>': 10, '   y n y r</w>': 6, '   pla ton i c</w>': 2, '   gla i ve</w>': 2, '   o pp o ses</w>': 1, '   under lin gs</w>': 2, '   f er o ci ty</w>': 2, '   on s la u ght</w>': 2, '   cu r ved</w>': 2, ' bor n</w>': 7, '   bar ons</w>': 7, '   o c cu p y</w>': 11, '   k ru ll</w>': 4, '   con qu er</w>': 14, '   coun ci l or</w>': 1, '   gu ar ding</w>': 7, '   tu ro ld</w>': 3, '   sa d d les</w>': 2, '   se er</w>': 3, '   k in g ship</w>': 2, '   mi gh t in ess</w>': 1, '   l out</w>': 2, '   de part</w>': 8, '   nee d les</w>': 18, '   qui ck s and</w>': 4, '   ab di ca te</w>': 1, '   war l or d</w>': 1, '   re s our ce ful</w>': 8, ' mar es</w>': 1, '   un happ in ess</w>': 3, '   f li ck er</w>': 1, '   st un ts</w>': 4, '   sh ri v el s</w>': 1, '   vi ll a g es</w>': 8, '   wa lled</w>': 3, '   com pe l</w>': 2, '   g ran d s ons</w>': 3, '   t or qui l</w>': 1, '   con ten t ment</w>': 4, '   ye ar n</w>': 5, '   shu n</w>': 1, '   qu ell</w>': 7, '   e t ce ter a</w>': 3, ' bo ar</w>': 1, '   l ou ts</w>': 2, '   pi g let</w>': 3, '   bo ar</w>': 4, '   bo ors</w>': 2, ' u g ly</w>': 6, '   lo af ed</w>': 1, '   spi ces</w>': 1, '   ti tch</w>': 1, '   su gar ba lls</w>': 1, '   gu m dro ps</w>': 1, '   ke g an</w>': 2, '   sor cer er</w>': 3, '   har es</w>': 1, '   for e fa th ers</w>': 1, '   bar ga in ed</w>': 7, '   p al try</w>': 4, '   bu mp kin</w>': 1, '   or da in ed</w>': 4, '   mo d red</w>': 3, '   god fa th ers</w>': 1, '   in v ad ers</w>': 4, '   di vi de</w>': 13, '   un c ea sing</w>': 1, '   f re y la g</w>': 2, '   bar a k</w>': 1, '   ten do</w>': 1, '   for ts</w>': 1, '   le op ard</w>': 39, '   ban ner</w>': 17, '   app ose</w>': 1, '   w ea k en ing</w>': 4, '   ex pen ded</w>': 1, '   fe w er</w>': 8, '   en s la ved</w>': 1, '   so li t ary</w>': 17, '   re f le c tions</w>': 5, '   tri ck er y</w>': 7, '   be t ro th ed</w>': 4, '   2 7 0 2 </w>': 1, '   st ans fi e ld</w>': 2, '   ra p ha e ll a</w>': 1, '   re stu ran t</w>': 1, '   ex tra s</w>': 4, '   tur f</w>': 19, '   o pp o sing</w>': 5, '   ma th il da</w>': 12, '   un lea sh</w>': 3, '   te le s co pe</w>': 14, '   f li r ted</w>': 4, ' ma th il da</w>': 1, ' o ver s</w>': 2, '   dar k en ing</w>': 1, '   ho te ls</w>': 32, ' m om ent</w>': 1, '   de par ted</w>': 2, '   de par ting</w>': 4, '   pi d dle</w>': 1, '   with ho l ds</w>': 1, ' 1 7 </w>': 4, ' ok</w>': 12, ' chan ge</w>': 7, '   thous an d th</w>': 1, '   re f ra in</w>': 5, '   mu si ca ls</w>': 4, ' ready</w>': 5, '   car d bo ar ds</w>': 1, '   1 2 0</w>': 2, ' 1 3 0</w>': 1, '   1 4 0</w>': 5, '   ru b en s</w>': 1, '   dan i e ll e</w>': 3, ' com po s er</w>': 1, '   le ss en</w>': 3, '   sta ir ways</w>': 1, '   sa lu te</w>': 16, '   me s sin a</w>': 5, ' pe s ce</w>': 1, '   sp ad a</w>': 1, '   al f re do</w>': 12, '   gi an car l o</w>': 1, '   r in al d i</w>': 4, '   e mi li o</w>': 7, '   gra pp e</w>': 1, '   sur name</w>': 1, '   hi t men</w>': 1, '   br y n</w>': 6, '   ma w r</w>': 5, ' j our na l</w>': 1, ' p sy cho lo g y</w>': 1, ' s ci ence</w>': 3, '   a de ll e</w>': 5, '   ma u ri ce</w>': 31, '   den n ys</w>': 1, '   re do</w>': 2, '   re sted</w>': 5, '   sy m bo li c</w>': 6, '   f re qu ent</w>': 11, '   f l y er</w>': 16, '   ni ke</w>': 5, ' tra in ers</w>': 1, ' cu shi on ed</w>': 1, '   so les</w>': 5, '   un t rea ted</w>': 2, '   man i a</w>': 3, ' con fi dent</w>': 1, ' ri s k</w>': 2, '   ar ter y</w>': 14, '   ar ter i es</w>': 3, '   a g gra v a ted</w>': 2, '   ex er tion</w>': 2, ' cla u di ca tion</w>': 1, '   sp as ms</w>': 3, '   re com men ding</w>': 4, '   en coun ter ing</w>': 3, '   f ar the st</w>': 2, '   re cap ture</w>': 1, '   ga ge</w>': 6, '   vo l un t ary</w>': 3, '   p ac i fi c a</w>': 7, '   car e f ree</w>': 1, '   pla st er ed</w>': 4, ' 1 5 </w>': 10, '   pre sti gi ous</w>': 2, ' pi ck</w>': 4, '   t an dy</w>': 1, '   p hi lly</w>': 10, '   han s en</w>': 2, '   b al li sti c</w>': 7, ' as ses</w>': 3, '   st ea d man</w>': 1, '   ev a si ve</w>': 8, '   i sa ac </w>': 13, '   win ked</w>': 1, '   pi tch ed</w>': 9, '   so ar ed</w>': 1, '   che er ed</w>': 2, '   star ed</w>': 9, '   hi gh li gh ted</w>': 1, '   co al vi ll e</w>': 1, '   u ta h</w>': 38, '   ga z e tt e</w>': 5, '   sa lu ting</w>': 1, '   p un c tu al</w>': 1, '   c ru sa der</w>': 3, '   k r is</w>': 9, '   gra p es</w>': 6, '   jo a d</w>': 7, '   co a st al</w>': 5, '   p hi lea s</w>': 1, '   inter i m</w>': 5, '   ver ses</w>': 1, '   en li gh ten ing</w>': 2, ' li zy</w>': 1, '   li zy</w>': 1, '   st e p han i e</w>': 28, '   st e p h</w>': 8, '   re war ded</w>': 12, '   mon e t ar i ly</w>': 1, '   ca l d well</w>': 5, ' no oo o</w>': 1, ' al co ho li c</w>': 1, '   c li pped</w>': 6, '   may f l ower</w>': 11, '   pa tr on i z ed</w>': 3, '   e pi lo gu e</w>': 2, '   ea sed</w>': 2, '   hu g ged</w>': 5, '   dr un k er</w>': 6, '   dro o p</w>': 1, '   wh am</w>': 8, '   s w un g</w>': 6, '   mi s ma tch ed</w>': 2, '   cl in king</w>': 1, '   comp li men t ary</w>': 8, '   in differ ent</w>': 3, '   on war ds</w>': 1, ' k ey</w>': 3, '   mi x es</w>': 2, '   di sp en se</w>': 4, '   for ma li ti es</w>': 2, '   ch ee sy</w>': 14, '   ra mb ling</w>': 10, '   ca b s</w>': 7, '   sta g ger</w>': 3, '   ge ars</w>': 7, '   the ta</w>': 2, '   ch i</w>': 11, '   l sa ts</w>': 1, '   b ack u ps</w>': 2, '   mer chan di sing</w>': 6, ' hi story</w>': 9, '   l y c r a</w>': 1, '   ch u t ne y</w>': 5, '   en ri qu e</w>': 16, '   ch er</w>': 4, '   e ll e</w>': 17, '   win d ha m</w>': 15, '   car din al</w>': 6, ' ac ti v a ting</w>': 1, '   a m mon i u m</w>': 2, '   th i g l y co late</w>': 1, '   mar c in k o</w>': 2, '   cu r ls</w>': 2, '   ho sed</w>': 2, '   pa u le tt e</w>': 1, '   b on af an te</w>': 2, '   en for c ing</w>': 1, '   de w ey</w>': 47, '   ne w com b</w>': 1, ' car e er</w>': 4, '   he y wor th</w>': 4, '   si st er ho od</w>': 4, '   sa l v at ore</w>': 14, '   ju di ci al</w>': 8, '   inter n s</w>': 1, '   war ner</w>': 18, ' ki ll ers</w>': 1, '   sp a sti c</w>': 1, '   ma sc ar a</w>': 8, '   gre en er</w>': 2, '   j ack cra p</w>': 1, '   car at</w>': 6, '   un poli sh ed</w>': 2, '   pa w ning</w>': 2, ' list ed</w>': 2, '   h un t in g ton</w>': 6, '   de f en d ant</w>': 24, '   th or ough</w>': 12, '   e mi ssion</w>': 3, '   r s v p</w>': 3, '   k no tt in g ha m</w>': 1, '   sti ck y</w>': 10, '   mar got</w>': 2, '   n er dy</w>': 3, '   bra in wa sh ed</w>': 1, '   st e f an o</w>': 1, ' days</w>': 3, '   sc run chi e</w>': 2, '   g p a</w>': 1, '   u s c</w>': 7, '   de mu re</w>': 4, '   sig ma</w>': 2, ' to y</w>': 1, '   t ro ph y</w>': 11, '   p an ty</w>': 5, '   de sig n ers</w>': 2, '   g ori ll a</w>': 12, '   fe ll a ted</w>': 2, '   br un e tt e</w>': 13, '   bl on d es</w>': 8, '   br un e tt es</w>': 3, '   hi l ary</w>': 1, ' ser vi ce</w>': 2, '   pro vi ding</w>': 10, '   out rea ch</w>': 2, '   di sc ri min a ted</w>': 3, '   m ou sy</w>': 2, '   ser en a</w>': 2, '   con di ment</w>': 1, '   ly</w>': 4, '   au st en</w>': 2, '   pla t t</w>': 2, '   j ar et</w>': 1, '   v an der mar k</w>': 1, '   bu t th ead</w>': 4, '   ha w ks</w>': 8, '   un tru st wor th y</w>': 3, '   f oo t no tes</w>': 1, '   l ev in son</w>': 2, '   ro y al ton</w>': 2, '   sh ri ve l</w>': 3, '   what ev ers</w>': 1, '   st ro m well</w>': 1, '   so cra ti c</w>': 1, '   la k er</w>': 4, '   ban gs</w>': 10, '   at ro ci ous</w>': 3, '   he ll o o</w>': 3, '   mi x ers</w>': 1, '   for ma ls</w>': 1, '   li do</w>': 1, '   l s at</w>': 1, '   qu a li f y</w>': 11, '   h y po s</w>': 1, '   ha ll u c in a ted</w>': 1, '   ci v </w>': 1, '   b en ch es</w>': 2, '   a ar on</w>': 22, '   a men able</w>': 1, '   s wi z z le</w>': 2, '   ho ver ing</w>': 3, '   ca ban a</w>': 6, '   in f om er ci al</w>': 2, '   hu mp ed</w>': 6, '   g ori ll as</w>': 5, '   cu n</w>': 1, '   t art</w>': 13, '   mon to ya</w>': 12, '   ce ci l</w>': 19, '   c un ni lin gu s</w>': 1, '   p he w</w>': 3, '   di d d ling</w>': 3, '   k app a</w>': 3, '   mo at</w>': 7, '   dis man t ling</w>': 3, '   af la me</w>': 1, '   a wai ting</w>': 8, '   p y g mi e</w>': 1, '   la sh</w>': 3, '   car ess</w>': 2, '   fa er i es</w>': 3, '   bi r di es</w>': 1, '   p ee we e</w>': 1, '   de lu sion</w>': 11, '   ma gi ci an</w>': 23, '   tur bu l ent</w>': 1, '   sur f ac ed</w>': 3, '   in si p id</w>': 2, '   mor tal s</w>': 8, '   ra p ture</w>': 4, '   par a pe ts</w>': 1, '   con tri ving</w>': 1, '   cour t y ard</w>': 10, '   r ou ting</w>': 2, '   con te mp late</w>': 2, '   n ou ri sh ment</w>': 6, '   dam se l</w>': 1, '   f la y</w>': 1, '   o be di ence</w>': 10, '   pl u mp</w>': 3, '   cap on</w>': 1, '   sa g ac i ous</w>': 1, '   to ad st ool</w>': 1, '   ab h or r ent</w>': 1, '   con c ea l ment</w>': 2, '   pu ling</w>': 1, '   p al li d</w>': 1, '   je st ing</w>': 1, '   in sig ni fi c ant</w>': 18, '   wh el p</w>': 2, '   st ea le e</w>': 2, '   t lea so o</w>': 2, '   bl ack he art</w>': 5, '   ac hi ll es</w>': 8, '   cu ss</w>': 6, '   f lo z en</w>': 1, '   fo l ev a</w>': 1, '   o g re</w>': 6, '   a li cor n</w>': 3, '   d la g on</w>': 2, '   sp il it</w>': 2, '   st l en g th</w>': 1, '   ra v age</w>': 2, '   coun tr y side</w>': 5, '   ma i den s</w>': 2, '   i de e</w>': 2, '   loo ke e</w>': 1, '   di f fe l ent</w>': 2, '   sp ea ke e</w>': 2, '   coun t le e</w>': 1, '   b ling</w>': 1, '   ma ke e</w>': 1, '   la in</w>': 1, '   mer ci a</w>': 1, '   li sing</w>': 1, '   ca th ay</w>': 2, '   f lu m</w>': 2, '   ve lly</w>': 2, '   so lly</w>': 1, '   in te ll u pt</w>': 1, '   co il</w>': 14, '   na u ght</w>': 3, '   o g g</w>': 6, '   cou er</w>': 13, '   no ir</w>': 21, '   b li ght</w>': 3, ' ga in st</w>': 1, '   d war ves</w>': 5, '   han di work</w>': 3, '   sc re w b all</w>': 11, '   t was</w>': 5, '   un fa ir ly</w>': 3, '   s qu ar e f oo t</w>': 6, '   b y gon es</w>': 4, '   hon e y th or n</w>': 7, '   gu mp</w>': 15, '   o c ca si on ed</w>': 1, '   ri d d les</w>': 6, '   comp le tes</w>': 2, '   t is</w>': 21, '   fa er i e</w>': 11, '   li l i</w>': 8, '   jo you s</w>': 4, '   sp r in g times</w>': 1, '   sa pp hi re</w>': 3, '   gre en t ee th</w>': 8, '   v au l ted</w>': 1, '   vi c tu al s</w>': 2, '   v al or</w>': 3, '   be si e ged</w>': 4, '   a to p</w>': 2, '   por t cu l is</w>': 2, '   ere</w>': 5, '   god spe ed</w>': 2, '   pe sti es</w>': 1, '   tra je c t ory</w>': 15, '   un brea k able</w>': 2, '   ti mb ers</w>': 1, '   d ra w bri dge</w>': 2, '   ar r ows</w>': 8, '   as se mb le</w>': 4, '   ar ch ers</w>': 3, '   be sted</w>': 7, '   ni pp ing</w>': 2, '   whi l st</w>': 4, '   s ca ling</w>': 2, '   ar ight</w>': 1, '   imp s</w>': 1, '   ra ven</w>': 10, '   o on a</w>': 12, '   en ven om ed</w>': 2, '   th i st le down</w>': 1, '   ra g wor t</w>': 1, '   st e ms</w>': 2, '   ha st e</w>': 7, '   s win gs</w>': 12, '   te m per ed</w>': 4, '   si gu r d</w>': 4, '   vo l su n g</w>': 2, '   fa f ni r</w>': 2, '   lin d f ar n e</w>': 5, ' ne ck</w>': 2, '   st ou tly</w>': 1, '   gr ou ch es</w>': 1, '   he ed</w>': 3, '   spi te ful</w>': 4, '   g ru mb ling</w>': 1, '   may ha p</w>': 4, '   f oo t pr in ts</w>': 16, '   do or stop</w>': 1, '   g le an in gs</w>': 1, '   car ri on</w>': 3, '   fri gh ten</w>': 25, '   cha ps</w>': 5, '   ha m mer ing</w>': 6, '   won dr ous</w>': 3, '   t would</w>': 2, '   tra v el ers</w>': 4, '   pre ce de</w>': 2, '   lan d mar k</w>': 3, '   ro o sted</w>': 1, '   of t times</w>': 1, '   sta un ch</w>': 2, '   co b we b s</w>': 2, '   f ever ed</w>': 1, '   br ow</w>': 8, '   tr ou b ling</w>': 6, '   k na ve</w>': 2, '   dro pp in gs</w>': 5, '   mi sc hi e f</w>': 8, '   co lt</w>': 7, '   ar chan ge ls</w>': 1, '   wi e ld</w>': 3, '   ser p ent</w>': 9, '   stu mp s</w>': 4, '   k na ves</w>': 1, ' la d</w>': 2, '   s ke w er</w>': 1, '   ho ar ds</w>': 1, '   m ound</w>': 6, ' s la y er</w>': 3, ' ro b b er</w>': 2, '   t will</w>': 1, '   bra ve ly</w>': 9, '   ac or n s</w>': 2, '   ne sts</w>': 3, '   la men ta ble</w>': 2, '   fi e</w>': 5, '   lu ll</w>': 5, '   cra dle</w>': 7, '   wa llow</w>': 5, '   d we lls</w>': 3, '   ha g</w>': 5, '   bo g</w>': 6, '   d ev ou rs</w>': 2, '   en jo y able</w>': 6, '   en chan t ment</w>': 5, '   p ra ttle</w>': 3, '   b ou qu et</w>': 8, '   ar o ma ti c</w>': 1, '   el der ber ry</w>': 1, '   af fe c tions</w>': 1, '   n y mp h</w>': 2, '   he ar th</w>': 4, '   co d fi sh</w>': 2, '   co ck les</w>': 2, '   ga mm on</w>': 2, '   t ro tt ers</w>': 1, '   b lu e be lls</w>': 2, '   st ru mm ing</w>': 1, ' dan c ing</w>': 3, '   ser en a de</w>': 1, '   win d la ss</w>': 1, '   li e ge</w>': 6, '   f lo k i</w>': 1, '   o i s in</w>': 1, '   th ri f ty</w>': 2, ' st e w</w>': 1, ' n ea th</w>': 1, '   th u r g is</w>': 2, '   w oo d pe ck er</w>': 1, ' sti ck er</w>': 1, '   st out</w>': 2, ' sti cking</w>': 2, '   p in ch ing</w>': 4, '   fe tch ed</w>': 3, '   ma s ked</w>': 5, '   dam n able</w>': 4, '   be wi tch ed</w>': 3, '   per chan ce</w>': 3, '   b ower</w>': 1, '   w ary</w>': 6, '   me thin ks</w>': 4, '   na p kin</w>': 7, '   a mb ro si a</w>': 2, '   e lf</w>': 2, '   ser pen ts</w>': 2, '   bo o t</w>': 28, '   a p tly</w>': 2, '   fri gh t ful</w>': 3, ' h un ting</w>': 1, '   hu m min g bi r d</w>': 2, '   fa i rest</w>': 4, '   b lo s so m</w>': 5, '   ex c ee ding</w>': 5, '   god w in</w>': 1, '   sh ar pe st</w>': 2, '   be fa ll en</w>': 2, '   s la in</w>': 3, '   be sp ea k</w>': 2, '   en t re at</w>': 2, '   con ce it</w>': 4, '   dro ss</w>': 1, '   af ar</w>': 4, '   di sc er ning</w>': 1, '   lo v el in ess</w>': 2, '   re f le c ted</w>': 6, '   por ri dge</w>': 3, ' po t</w>': 2, '   bar r en</w>': 7, '   spi ri t less</w>': 1, '   v ex </w>': 1, '   po ll en</w>': 3, '   je st</w>': 6, '   won dr ou s ly</w>': 1, '   wi tt ed</w>': 2, '   char med</w>': 10, '   may n</w>': 1, '   n ea th</w>': 1, '   b ou gh s</w>': 1, '   w ea p on ry</w>': 2, '   me thought</w>': 1, '   t wi xt</w>': 1, '   chan c ed</w>': 2, '   uni cor n s</w>': 1, '   bri gh ten</w>': 2, '   en ch ant</w>': 3, '   gar men ts</w>': 6, '   se w ing</w>': 12, '   o gr es</w>': 1, '   f er re ts</w>': 2, '   fo al</w>': 2, '   mar sh</w>': 81, ' fo d der</w>': 1, '   re g in</w>': 1, '   a v at ar</w>': 1, '   th or</w>': 2, '   m jo l ni r</w>': 1, '   ex ca li bu r</w>': 13, '   cra f t s man ship</w>': 3, '   mor t ar</w>': 4, '   sp o i ls</w>': 2, '   a pl enty</w>': 3, '   lo is</w>': 70, '   gu e st room</w>': 1, ' u sing</w>': 3, '   ca tal in a</w>': 3, '   han d le b ars</w>': 1, '   ar mo i re</w>': 1, '   f rea ki sh</w>': 1, '   sha m po o</w>': 7, '   l ow er ed</w>': 5, '   do k o s</w>': 3, '   bu il ds</w>': 11, ' a im ed</w>': 1, '   ca mp ing</w>': 10, '   ma stu r ba te</w>': 6, ' mrs</w>': 7, '   cor li ss</w>': 1, '   en vi r ons</w>': 2, ' st ru c ture</w>': 1, '   wh in ers</w>': 1, ' f ac ing</w>': 2, '   a men ded</w>': 2, '   st ev en s</w>': 29, '   ex c ee ded</w>': 7, '   ex ha ust</w>': 5, '   l ar son</w>': 3, '   un en clo sed</w>': 1, '   d rea ding</w>': 2, '   ber k le y</w>': 1, '   r en der in gs</w>': 1, ' ex po sure</w>': 1, ' pen is</w>': 1, '   d wee b</w>': 2, '   p act</w>': 14, '   su n gla ss es</w>': 9, '   vi co d in</w>': 1, '   sha ved</w>': 8, '   un beli ev ably</w>': 7, '   ca stra t i</w>': 1, ' a w ful</w>': 5, '   s n ore</w>': 6, '   hu ff ing</w>': 1, '   cor y</w>': 7, '   pa ts</w>': 2, '   p an c rea ti c</w>': 3, '   mor ti se</w>': 1, '   ten on</w>': 1, '   per mi ts</w>': 8, '   gu e s thou se</w>': 1, '   r en ta ble</w>': 2, '   g ran d fa ther ed</w>': 2, '   la g un a</w>': 1, '   in cor por a ted</w>': 1, '   s qu a tting</w>': 4, '   di ver</w>': 9, ' told</w>': 16, '   god dam mi t t</w>': 1, '   fr o</w>': 6, '   de mer o l</w>': 5, '   h in d si ght</w>': 1, '   an ti bi o ti c s</w>': 2, '   pl u mb ed</w>': 1, '   sa l v a ged</w>': 1, '   f lo or bo ar ds</w>': 3, '   sp ool</w>': 1, '   in con si der ate</w>': 3, '   d ev o id</w>': 6, '   pa y off</w>': 9, '   re co g ni z able</w>': 5, '   p g</w>': 33, '   na g ger</w>': 1, '   bi s cu it</w>': 12, '   do or st ep</w>': 8, '   j an g le</w>': 5, '   par d ons</w>': 5, '   b lo ck er</w>': 4, '   st e pp ing</w>': 20, '   sy l vi a</w>': 17, '   ton k</w>': 1, '   y ve tt e</w>': 2, '   wi l k ins</w>': 9, '   com m uni ti es</w>': 2, '   den tur es</w>': 1, ' lin ed</w>': 1, '   cra d do ck</w>': 1, '   h en sha w</w>': 1, '   ba p ti sts</w>': 1, '   no min al</w>': 2, '   may n ard</w>': 10, '   b our b ons</w>': 2, '   mu st ard</w>': 17, '   u r in a ting</w>': 2, '   b on e y ard</w>': 4, '   go l d m ou th</w>': 6, '   pro sta te</w>': 4, '   b ends</w>': 6, '   gu mb o</w>': 2, '   pl ac ing</w>': 8, '   p ee ing</w>': 7, '   gre en vi ll e</w>': 5, ' li cking</w>': 1, '   tru st y</w>': 3, '   b li ss ful</w>': 1, '   w ra pp ing</w>': 2, '   du ster</w>': 3, '   con gen i tal</w>': 2, '   cra w for ds</w>': 2, '   cra w da ds</w>': 1, '   nu r tu red</w>': 3, '   han d l ers</w>': 3, '   cha m pi on ship</w>': 18, '   lin e up</w>': 3, '   ba ll par k</w>': 4, '   al lie</w>': 8, '   c lea ts</w>': 2, ' hea der</w>': 2, '   br on x</w>': 18, '   g ri p es</w>': 2, '   ni b b ling</w>': 2, '   vi bra tion</w>': 5, ' st ink</w>': 1, ' ban ks</w>': 2, '   e en y</w>': 1, '   me en y</w>': 1, '   min ey</w>': 1, '   e s qui re</w>': 3, '   s wa mp s</w>': 2, ' ea ten</w>': 2, '   ir on ing</w>': 3, '   c li pp ing</w>': 4, '   to en a i ls</w>': 7, '   st ab b ing</w>': 5, '   he in ous</w>': 4, '   ra il ro ad ed</w>': 1, '   b loo d ba th</w>': 6, ' le g ged</w>': 8, '   3 6 </w>': 11, '   th re es</w>': 4, '   con tri bu tion</w>': 14, ' fe ar ing</w>': 4, '   fi st ful</w>': 3, '   k l an</w>': 3, '   co l or e ds</w>': 5, ' den ying</w>': 2, '   h y pe</w>': 9, '   ne war k</w>': 2, '   ea g les</w>': 5, '   s la y</w>': 5, ' sho t</w>': 11, '   ri le y</w>': 7, '   cu sh</w>': 5, '   supp le</w>': 3, ' pre ci ate</w>': 3, '   ho ll er</w>': 11, '   do d i</w>': 1, '   tru sti es</w>': 1, '   ci gs</w>': 2, '   g ri lling</w>': 4, '   c rea med</w>': 3, '   im pro vi se</w>': 11, '   sh or ten</w>': 1, '   j i g g ab o o</w>': 2, '   me t ro poli t an</w>': 9, '   me te or o lo gi cal</w>': 2, '   as se ss men ts</w>': 2, '   ar m st rong</w>': 7, '   si z z l ed</w>': 1, ' ti m er</w>': 3, '   ho l d in</w>': 16, '   com mo tion</w>': 8, '   bo o t le g g ers</w>': 4, '   re ci pe</w>': 10, '   ho o ch</w>': 1, '   1 9 3 2 </w>': 2, '   ra y for d</w>': 4, '   na t che z</w>': 3, '   to ll</w>': 6, '   bo o t le g ging</w>': 2, '   sa tch m o</w>': 2, '   mar le en</w>': 1, '   di ll ard</w>': 1, '   s win g in est</w>': 1, ' hi ll</w>': 1, '   cho ir boy</w>': 1, '   b re w</w>': 5, '   f la s k</w>': 8, '   ad ri an</w>': 35, '   sch na pp s</w>': 4, '   bi r th right</w>': 8, '   sin ning</w>': 3, '   u r r</w>': 1, '   u g g g</w>': 1, '   er r r</w>': 2, '   l ow ers</w>': 3, ' da</w>': 10, '   par son</w>': 2, '   a stra y</w>': 3, '   da i qu ir is</w>': 1, '   j en na</w>': 1, '   chri st a</w>': 2, '   v al er i e</w>': 11, '   per v </w>': 4, '   un h</w>': 13, '   ch r on i c</w>': 14, '   ha li to s is</w>': 2, '   ru pa u l</w>': 1, '   su per im po sed</w>': 2, ' sc ar face</w>': 1, ' sha p ed</w>': 6, '   s nor es</w>': 3, '   co l a</w>': 7, '   so da s</w>': 3, '   n ar r ow ing</w>': 2, '   ying</w>': 1, ' ying</w>': 1, '   c r ac king</w>': 13, '   po pe ye</w>': 6, ' who o</w>': 1, '   mar in o</w>': 2, '   re gre t fu lly</w>': 2, '   de cl ine</w>': 8, '   sho ving</w>': 3, '   p in ea pp le</w>': 4, '   c ru e</w>': 2, '   ho lly</w>': 56, '   mu lli g an</w>': 2, '   me tal li c a</w>': 2, ' li ck</w>': 1, '   kee ster</w>': 2, '   com pi la tion</w>': 1, ' ar ran ging</w>': 2, '   bra ssi ere</w>': 7, '   or gi es</w>': 5, '   bo o ger land</w>': 1, '   f oo ts</w>': 1, '   pe pp er min t</w>': 4, '   m ea d ow lan ds</w>': 1, '   el ton</w>': 9, '   sle e p y head</w>': 1, '   den ir o</w>': 2, '   ba sh ing</w>': 3, '   un t ou cha b les</w>': 6, '   al u minu m</w>': 5, '   re e ge</w>': 1, '   bo z o</w>': 10, '   gen i tal</w>': 2, '   tu cking</w>': 4, '   d or an</w>': 1, '   co con u ts</w>': 13, ' comp to ir</w>': 1, '   hu l a</w>': 4, '   c r ow le y</w>': 2, '   ru ther for d</w>': 4, ' po s se ssed</w>': 1, '   d ra w st r ing</w>': 2, '   hea vi er</w>': 8, ' o c to p us</w>': 1, '   ge la t i</w>': 2, '   d ru m sti ck</w>': 2, '   op t ome tri st</w>': 2, '   sp ar k ly</w>': 1, '   t ou ri st</w>': 18, '   bo i ling</w>': 8, '   ad ver ti se ment</w>': 6, ' f la s k</w>': 1, '   cu b s</w>': 6, ' ri p</w>': 1, '   s an d man</w>': 14, '   plan k ton</w>': 3, '   gre en s</w>': 7, ' came</w>': 4, '   s cu l pt</w>': 3, '   e ter na lly</w>': 5, '   s cu l p tor</w>': 4, '   car ou se l</w>': 4, '   s k u lling</w>': 1, '   4 8 3 </w>': 2, '   ye ll ows</w>': 2, '   sc ra mb ling</w>': 2, '   s l ows</w>': 3, '   sle e p t ea ch er</w>': 1, '   f la me out</w>': 1, '   h y dro g al v ani c</w>': 1, '   ti d es</w>': 2, '   je ssi c a</w>': 78, ' be lo ved</w>': 7, '   li fe clo cks</w>': 2, '   s an d men</w>': 5, '   bri gh ter</w>': 5, '   y c ch</w>': 1, '   pen s</w>': 9, '   re live</w>': 4, '   la st day</w>': 1, '   nu r ser i es</w>': 1, '   mo ther ing</w>': 1, '   me c can o</w>': 2, ' b re e d ers</w>': 1, '   r en e wa l</w>': 3, '   mi s fi ts</w>': 8, ' killed</w>': 9, '   run n ers</w>': 10, ' m ac a vi ty</w>': 1, '   m ac a vi ty</w>': 3, '   de ce i t fu l ne ss</w>': 1, '   su a vi ty</w>': 1, '   je we l</w>': 6, '   ha tter</w>': 1, '   j el li c le</w>': 3, ' j el li c le</w>': 1, '   a ir s</w>': 3, '   gr ac es</w>': 2, '   in tru sion</w>': 9, '   sc an n ers</w>': 10, '   te x ac o</w>': 3, ' to ps</w>': 3, '   l ani e</w>': 1, '   al ar ms</w>': 13, '   da y ton</w>': 7, '   la u r ent</w>': 6, '   a wa k en ed</w>': 6, '   g li ded</w>': 2, '   pa s sp or ts</w>': 11, '   2 2 2 4 </w>': 1, '   d ell</w>': 25, '   stu c c o</w>': 1, '   mo ke</w>': 2, '   star li ght</w>': 2, '   sy ca more</w>': 3, '   ar ni e</w>': 19, '   tru e wor th y</w>': 1, '   s la mm er</w>': 3, ' f ever</w>': 3, '   in ma te</w>': 4, '   fa x ed</w>': 7, '   na tion wi de</w>': 3, '   fin ger pr in ted</w>': 2, '   ro go ff</w>': 1, '   ta il ga te</w>': 2, '   sa x op h one</w>': 12, '   s oun d pro of ed</w>': 2, '   ca ddy</w>': 13, ' ca ddy</w>': 1, '   pr on oun c ed</w>': 8, '   un cer ta in ty</w>': 5, '   ex qui si te</w>': 10, ' ab o ard</w>': 1, '   man ch u ri a</w>': 3, '   sh an gr i</w>': 25, '   ti b et</w>': 19, '   sin ga p ore</w>': 6, '   o ver take</w>': 4, '   fe ar ful</w>': 7, '   ga in s for d</w>': 5, '   ro ar ed</w>': 2, '   ob li ga te</w>': 1, '   bar ne y</w>': 26, '   e ff r on ter y</w>': 2, '   cha l m ers</w>': 3, '   s win d l er</w>': 3, ' f oo ted</w>': 2, '   e mp lo ys</w>': 4, '   ba s k u l</w>': 3, '   con ven i en ces</w>': 1, '   ri t z</w>': 7, '   fa tt en</w>': 2, ' han ded</w>': 23, '   con ver sa tion al</w>': 3, ' i ster</w>': 1, ' lo v ey</w>': 2, '   lo ve t t</w>': 7, '   der went</w>': 1, '   k ni gh ted</w>': 5, '   me so z o i c</w>': 2, '   me ga ther i u m</w>': 2, '   s ou p b one</w>': 1, '   re con st ru ct</w>': 8, '   hea th en s</w>': 3, '   fo ssi ls</w>': 13, '   chan g</w>': 24, '   ni ck el s</w>': 4, '   la ma s</w>': 3, '   por t ers</w>': 29, '   u ti li ti es</w>': 2, '   un bo so m</w>': 1, ' po kes</w>': 1, '   we ary</w>': 13, '   f en ner</w>': 4, '   pla y ma tes</w>': 3, '   co a x ing</w>': 1, '   ch in a man</w>': 10, '   im po se</w>': 8, '   we st er n ers</w>': 1, '   re ver t</w>': 2, '   1 8 8 8 </w>': 2, '   car ri ers</w>': 13, '   la ma</w>': 22, '   a ma z e ment</w>': 3, '   comp ani ons</w>': 4, '   in di re ct</w>': 3, '   b ow l ed</w>': 3, '   1 0 8 </w>': 2, '   a st on i sh in g ly</w>': 1, '   per ra u lt</w>': 11, '   a m pu ta tion</w>': 2, '   1 7 1 3 </w>': 1, '   gu i ding</w>': 7, '   un cer tain</w>': 8, '   di sp u tes</w>': 2, '   a v ar i ci ou s ne ss</w>': 1, '   su f fi ci en cy</w>': 1, '   in cor ri gi b les</w>': 1, '   st ri c t ne ss</w>': 1, '   mo der a tely</w>': 4, '   ex ce ss es</w>': 2, '   un en li gh ten ing</w>': 1, '   ge o lo g y</w>': 2, ' un gra te ful</w>': 1, '   in ven t ory</w>': 7, '   u to pi a</w>': 6, '   w oo d wor k ers</w>': 1, ' w ea ver s</w>': 1, '   b la z es</w>': 3, ' por t ers</w>': 1, '   fa v or able</w>': 5, '   s n ow st or m</w>': 4, '   sh e d ding</w>': 1, '   st r en g th en ing</w>': 1, '   pa ssi ons</w>': 4, '   w ea p on ed</w>': 1, '   for e saw</w>': 3, '   ex u l ting</w>': 1, '   ho tly</w>': 2, '   un in te lli g ent</w>': 1, '   s cu r r ying</w>': 2, '   be wi l der ed</w>': 3, '   pro pe lled</w>': 3, '   d ev ou red</w>': 1, '   e th i c</w>': 4, '   m ee k</w>': 2, '   in h er it</w>': 11, '   pro lon ged</w>': 2, '   ex ci te</w>': 4, '   youn gi sh</w>': 1, '   di min i sh ing</w>': 3, '   g rea tly</w>': 10, '   th ri ving</w>': 1, '   son dr a</w>': 2, '   bi z et</w>': 1, '   ar du ous</w>': 1, '   be qu ea th</w>': 4, '   si l k en</w>': 1, '   pre side</w>': 1, '   ra g es</w>': 2, '   bab b ling</w>': 15, '   ser en e ly</w>': 1, '   cor ro ding</w>': 1, '   bu n k</w>': 13, '   de cre p it</w>': 2, ' fe et</w>': 2, ' gr ound</w>': 2, '   f an a ti c s</w>': 3, '   f an ta sti cal</w>': 2, '   be wi l der ment</w>': 1, '   ma ter i a li ze</w>': 2, '   un ex pl or ed</w>': 1, '   di a le ct</w>': 3, '   mon go li an</w>': 1, '   f re shi e</w>': 6, '   se cre t ar i es</w>': 9, ' talk</w>': 18, '   dis b and</w>': 1, '   ba tt le shi ps</w>': 1, '   war cra ft</w>': 1, '   du p ed</w>': 4, '   hur ra y</w>': 4, '   an ni hi la ted</w>': 3, '   ba s k</w>': 2, '   c ru i s er</w>': 15, '   con su l</w>': 3, '   han g ar</w>': 6, '   v a ll e ys</w>': 4, '   out st re tch ed</w>': 2, '   w r ing</w>': 7, '   mu sn</w>': 4, '   p ha ses</w>': 4, '   la ma ser y</w>': 1, '   ro o ted</w>': 6, '   ex pl or es</w>': 2, '   a mi ably</w>': 1, '   a mi ab i li ty</w>': 1, '   pi ge ons</w>': 24, '   f lu tes</w>': 1, '   fun ne ls</w>': 1, '   tu g ging</w>': 1, '   sh ri e king</w>': 4, '   p oun c ing</w>': 1, '   stu b ing</w>': 1, '   c ow an</w>': 8, '   dam ed</w>': 1, '   oo om p h</w>': 2, ' j ay</w>': 2, '   at ti re</w>': 5, '   spe ci ally</w>': 9, '   s an c tion ed</w>': 1, '   ins</w>': 8, '   e de l son</w>': 1, '   hu mor ous</w>': 6, '   su per vi sor s</w>': 4, '   lon ey</w>': 1, '   de e</w>': 15, '   z ed</w>': 3, '   pi ther</w>': 1, ' e yes</w>': 5, '   ro a st ing</w>': 3, '   s mor es</w>': 5, '   c m on</w>': 2, '   ve in</w>': 14, '   ne u re l y s er</w>': 1, '   i so la tes</w>': 1, '   m ea sur es</w>': 16, '   pa s en</w>': 1, ' fu r g on</w>': 1, '   l ar gu en se</w>': 1, '   v a y an se</w>': 1, '   di ge sted</w>': 3, ' z ea l ous</w>': 1, '   t wi r p</w>': 1, '   ha tch ed</w>': 3, '   su d bu ry</w>': 3, '   sh ou ll d</w>': 1, '   lo ch</w>': 2, '   ne ss</w>': 7, '   4 4 4 </w>': 1, '   er ni e</w>': 27, ' happen ing</w>': 2, '   na z c a</w>': 1, '   j ee b s</w>': 2, '   s li ther ing</w>': 3, '   s cu m wa d</w>': 1, '   sig na z o id</w>': 2, '   ni t ro gen i z er</w>': 2, '   di ge sti ble</w>': 1, '   re t in a</w>': 1, '   si x e y ed</w>': 1, '   in ver te bra tes</w>': 1, '   win ces</w>': 2, '   k in der gar ten</w>': 8, '   att ac h men ts</w>': 4, ' in t ro du ce</w>': 1, '   bl and</w>': 1, '   te st er ro s a</w>': 1, '   pri v y</w>': 10, '   ha si ds</w>': 1, '   t ar ry</w>': 3, '   n on co l or</w>': 1, '   in fini ty</w>': 10, '   sc ar i er</w>': 5, '   s war m ing</w>': 2, '   in di vi du ally</w>': 5, '   en to po lo gi st</w>': 1, '   an d ean</w>': 1, '   mo ll a to o s a</w>': 1, '   en tom o lo gi st</w>': 3, '   en to po l gi st</w>': 1, '   en to po lo gi sts</w>': 1, '   ex c lu si on ary</w>': 1, '   f re c to</w>': 1, ' in hi bi t ors</w>': 1, '   bi r th mar k</w>': 3, '   chi se l ed</w>': 2, '   qu ir k y</w>': 3, '   di m pl es</w>': 1, '   w r ack</w>': 1, '   e min en tly</w>': 3, '   de por ta tion</w>': 1, '   tr ans mo gra p hi c</w>': 1, '   de x a h y dro ch l or op ha ll om i x a loo sa l y s er</w>': 1, '   li c ence</w>': 6, ' f lo a ting</w>': 2, '   pla s ma</w>': 13, '   z o i ds</w>': 1, '   o ver run</w>': 6, '   in fe sta tion</w>': 1, '   bri li i ant</w>': 1, '   mo se b ac ke</w>': 1, '         </w>': 46, '   4 4 </w>': 5, '   hu ts</w>': 4, '   in ten tion ed</w>': 2, '   un au th ori z ed</w>': 10, '   un con c ea l ed</w>': 1, '   lan din gs</w>': 3, '   star ter</w>': 3, '   tra de off</w>': 1, '   en e mi e</w>': 1, '   3 5 0</w>': 10, '   h m n n</w>': 1, '   na tu ra li z ation</w>': 1, '   inter g al ac ti c</w>': 3, ' a li en</w>': 2, '   en ve lo p ing</w>': 2, '   ca mo f la u ge</w>': 1, '   no v </w>': 1, '   ev o l ved</w>': 19, '   go o se ls</w>': 1, '   vi si b ly</w>': 1, '   tra il b la z ers</w>': 1, '   qu ad ra se c tion al s</w>': 1, '   1 1 9 </w>': 2, '   wh att ya</w>': 5, '   mo ons</w>': 5, '   ho ok up</w>': 2, '   in st all</w>': 7, '   o ver po pu la tion</w>': 1, '   aga th a</w>': 11, '   an der ton</w>': 11, '   mi c ro bo ts</w>': 1, '   f lea s</w>': 11, ' re con st ru ction</w>': 1, '   an e s the si a</w>': 6, '   i ri ses</w>': 1, '   se w</w>': 11, '   an ti bi o s</w>': 1, '   vi si ons</w>': 10, '   l ar a</w>': 9, '   ne u ro in</w>': 1, '   do e</w>': 66, '   bea ton</w>': 2, ' g li tch</w>': 1, '   pre vi si ons</w>': 4, '   s wa pped</w>': 6, '   e ye s can</w>': 1, ' sa ved</w>': 1, '   pre co gs</w>': 17, '   pre co ps</w>': 1, '   gi de on</w>': 18, '   re la x es</w>': 5, '   la m ar</w>': 24, '   wi tw er</w>': 11, '   ja d</w>': 5, '   e ye den ts</w>': 1, '   out put</w>': 5, '   o ver ri d d en</w>': 2, '   lo ca tor</w>': 2, '   pre c ri me</w>': 13, '   sp ra w l</w>': 2, '   pre vi sion</w>': 5, '   en ter pri se</w>': 110, '   f le tch er</w>': 20, '   h or se back</w>': 2, '   re po in ted</w>': 1, '   gla z ing</w>': 1, '   m ou l din gs</w>': 1, '   den ti ls</w>': 1, '   r en o v ation</w>': 1, ' sc ru b b ing</w>': 1, '   tw ink</w>': 1, '   for war ded</w>': 2, ' sti cks</w>': 2, '   tri g</w>': 5, '   n ci c</w>': 4, '   cl er g y</w>': 3, '   no d ding</w>': 5, '   ge or ge town</w>': 1, '   ma g l ev </w>': 1, '   d c p d</w>': 2, '   fo x ha ll</w>': 1, '   4 4 2 1 </w>': 1, ' 1 1 0 8 </w>': 2, '   pre vi su a li z ed</w>': 1, '   ho lo sh p ere</w>': 1, ' st ac ks</w>': 1, '   k a ther ine</w>': 13, '   po ll ard</w>': 1, '   do p ing</w>': 4, '   be the s da</w>': 2, '   se min ary</w>': 2, '   lu ther an</w>': 2, ' te mp le</w>': 1, '   de i f y</w>': 1, '   h in e man</w>': 2, '   inter f ac ing</w>': 1, '   as sa u l ts</w>': 3, '   po si ti ves</w>': 1, '   par a do x</w>': 9, '   com pre ssion</w>': 3, '   do pa m ine</w>': 2, '   en d or ph ins</w>': 1, '   ser o ton in</w>': 3, '   pre co g</w>': 5, '   la m</w>': 4, ' pi e</w>': 5, '   hou se ca lls</w>': 2, '   e ye sc an ned</w>': 1, '   pre di c ted</w>': 9, '   co ll a p se</w>': 13, '   fu tur es</w>': 5, '   bur ge ss</w>': 5, '   f al li bi li ty</w>': 1, '   in sti lls</w>': 1, '   pre co g ni ti ves</w>': 1, '   in ex or ably</w>': 2, '   pa ter na l</w>': 1, '   to d d l er</w>': 4, '   un in ten ded</w>': 1, '   pre d ac i ous</w>': 1, '   ban e ber ry</w>': 1, '   ta g ged</w>': 8, '   f ra gi li ty</w>': 1, '   u s a</w>': 6, '   ho ver cra ft</w>': 1, '   c li c ki ty</w>': 1, '   sp y d ers</w>': 1, '   de j a</w>': 9, '   v u</w>': 9, '   cont in u u m</w>': 5, '   ri pp les</w>': 1, '   da shi el</w>': 1, '   c y ber par l or</w>': 1, '   d ow n lo ad ed</w>': 6, '   su per vi sing</w>': 4, '   ro ster</w>': 8, '   sy mp hon y</w>': 17, '   cra sh es</w>': 21, ' or g y</w>': 2, '   con su m ers</w>': 3, '   ad j ac ent</w>': 6, '   ar chi te c ture</w>': 8, ' con for mi st</w>': 1, '   me ta ph y si cal</w>': 4, '   b in ds</w>': 3, '   un ti me ly</w>': 4, '   le ga li sti c</w>': 1, '   d ra w back</w>': 3, '   me th o do lo g y</w>': 5, '   pre me di ta tion</w>': 3, '   sc ra mb les</w>': 3, '   ev an na</w>': 1, '   sp y der</w>': 1, '   sor ti e</w>': 1, '   re ss l er</w>': 1, '   re s che du le</w>': 4, '   s an da l</w>': 1, '   hi ve</w>': 2, '   pre di c tive</w>': 1, '   re u b en</w>': 4, '   mar te l</w>': 2, '   s ni f ter</w>': 4, '   s ni f t ers</w>': 3, '   cu be</w>': 5, ' wom en</w>': 4, '   ni gh t life</w>': 1, '   su i tes</w>': 8, '   s la de</w>': 4, '   3 1 5 </w>': 2, ' con ci er ge</w>': 1, ' in ci den tal s</w>': 1, '   car di ff</w>': 5, '   in ci den tal s</w>': 3, '   re f le c ting</w>': 5, '   s la ms</w>': 5, '   cl in ging</w>': 6, ' fa ster</w>': 1, '   ba th s</w>': 9, '   h or r ace</w>': 7, '   ch lo e</w>': 10, '   sp a gh e tt is</w>': 1, '   en cou ra g es</w>': 4, '   w en dy</w>': 70, '   la p dan ce</w>': 1, '   fi l a</w>': 1, '   bl ow jo b s</w>': 3, '   pl u mb </w>': 9, '   man n er ed</w>': 4, '   wh u</w>': 2, '   w el sh man</w>': 6, ' r ou gh ed</w>': 2, '   ri c ans</w>': 3, '   im mi g ran ts</w>': 1, '   ru i z</w>': 35, '   tr ou gh ed</w>': 2, '   cap o</w>': 2, '   w el sh</w>': 12, ' ok en</w>': 1, '   go ons</w>': 8, '   t c b</w>': 2, '   s n ow ba lling</w>': 2, '   j ess</w>': 11, '   sh ee t ro ck</w>': 1, '   sp ort s man</w>': 1, '   pa ger</w>': 8, '   a bu n dan tly</w>': 1, '   de ter r ent</w>': 5, '   se er ed</w>': 1, ' as sig ning</w>': 1, '   ter ri er</w>': 1, '   h er ding</w>': 4, '   sch u l</w>': 1, '   pi sh er</w>': 2, '   si l ver back</w>': 1, '   ye shi v a</w>': 2, '   bu c has</w>': 1, '   st op wa tch</w>': 1, '   p sh h h</w>': 1, ' 6 6 </w>': 4, ' i x ti es</w>': 1, '   ha i</w>': 2, '   a la i</w>': 4, ' ea sy</w>': 7, '   th on</w>': 1, '   mi mo s a</w>': 1, '   mi mo sa s</w>': 2, '   li me y</w>': 2, '   no ki a</w>': 1, ' ru st af ar i ans</w>': 1, '   so h o</w>': 7, '   u h u</w>': 1, '   f ru i t pi e</w>': 2, '   c li en te le</w>': 6, ' ro p</w>': 1, '   er e c tions</w>': 1, '   v ou ch ed</w>': 8, '   m ou th es</w>': 1, '   sh an k</w>': 1, '   j on e sing</w>': 1, '   ki b b les</w>': 1, '   ro ck can dy</w>': 1, ' in su ran ce</w>': 2, '   do m</w>': 11, '   si d down</w>': 18, '   pa y che cks</w>': 3, '   men ta li ty</w>': 11, '   ow ning</w>': 6, '   he in e k en s</w>': 1, ' car di ff</w>': 2, '   car pe ts</w>': 2, '   bor d ner</w>': 1, '   at m</w>': 8, '   d b</w>': 1, '   bo x ing</w>': 8, '   y u c c a</w>': 1, '   per si an</w>': 5, ' hu gs</w>': 1, '   sp la sh</w>': 3, '   ga ve l</w>': 1, ' k in da</w>': 4, '   sc re e ch</w>': 3, '   le ban e se</w>': 1, '   tu p ac </w>': 1, '   g an g ster</w>': 15, ' h are</w>': 6, '   he mi sp h er es</w>': 1, ' bo at</w>': 3, '   we sti es</w>': 1, '   shu cking</w>': 1, '   bar re tta</w>': 1, '   per ms</w>': 1, '   bab a lo o</w>': 1, ' tra pped</w>': 3, '   qu en t in</w>': 10, '   bri lli an tly</w>': 4, '   l un a</w>': 1, '   sle ds</w>': 1, '   vi p</w>': 3, '   wi g g ers</w>': 1, '   tru st af ar i ans</w>': 1, '   per si ans</w>': 4, '   ar ti cho ke</w>': 2, '   f er n et</w>': 4, '   c ru mb s</w>': 8, '   pa g ers</w>': 2, '   di em</w>': 10, ' no tes</w>': 1, '   nu mer i c</w>': 1, ' tri p</w>': 2, '   j en n in gs</w>': 4, '   of f sh ore</w>': 3, '   pa ss bo ok</w>': 1, '   an g in a</w>': 2, '   go tt i</w>': 1, '   di ge st if</w>': 1, '   a per ti f</w>': 1, '   t ar</w>': 15, '   st re g a</w>': 1, ' ra g on</w>': 1, '   a ll ah</w>': 11, '   bi lli e</w>': 4, '   6 0 0</w>': 9, ' took</w>': 3, '   s la ts</w>': 1, '   com b in ate</w>': 1, '   d in</w>': 2, '   wh e w w</w>': 1, '   n n n n n n n</w>': 1, '   out fi tt ed</w>': 1, '   gr en ad a</w>': 1, '   s om bi tch</w>': 2, '   be an town</w>': 1, '   ch ere</w>': 1, '   hur t in</w>': 11, '   pro sp er ous</w>': 7, '   be mb ry</w>': 11, '   l ou d sp ea k er</w>': 4, '   g ro ve l</w>': 4, '   i s la m</w>': 1, '   me c c a</w>': 6, '   ho li es</w>': 2, '   mu s li m</w>': 6, '   st ri k in g ly</w>': 1, '   e li ja h</w>': 22, '   mu ha m ma d</w>': 9, '   ma j ors</w>': 14, '   e y ed</w>': 3, '   a ar d v ar k</w>': 3, '   n ou n</w>': 1, '   so il ed</w>': 4, '   for bi d ding</w>': 1, '   f ou lly</w>': 1, '   ou tra ge ou s ly</w>': 2, ' wh a tch a</w>': 1, '   of ay</w>': 1, '   le g go</w>': 3, '   na pp y</w>': 1, '   con k</w>': 1, '   con ks</w>': 1, '   re e f er</w>': 4, '   nu t me g</w>': 4, '   a mp le</w>': 8, ' mu ha m ma d</w>': 1, '   in ti m ac i es</w>': 1, '   1 9 5 7 </w>': 3, '   s lan d ers</w>': 1, ' lo s</w>': 1, '   u p i</w>': 2, '   6 7 </w>': 4, '   pa ter ni ty</w>': 6, '   fa ther ed</w>': 3, '   po di u m</w>': 4, '   do g ged</w>': 5, '   po s se ssi ve</w>': 10, '   i dle</w>': 8, '   ma j or ed</w>': 1, '   tu s ke ge e</w>': 1, '   s ci en ti fi ca lly</w>': 2, '   m ea ts</w>': 3, '   re st ri ction</w>': 3, '   h en</w>': 12, '   c ack le</w>': 1, '   chi ck en w ing</w>': 1, '   ti gr is</w>': 1, ' e u ph ra tes</w>': 1, '   me di ter ran ean</w>': 1, '   ne w s re el s</w>': 1, '   na tive</w>': 20, '   he b re w</w>': 10, '   ca st off</w>': 1, '   k nee gre w</w>': 1, '   o ver si mp li fi es</w>': 1, '   d y na mi c</w>': 3, '   inter sti ces</w>': 1, '   su b cu l ture</w>': 1, ' po l ar i ze</w>': 1, ' er r on e ou s ly</w>': 1, '   app ra i se</w>': 1, '   su pre m ac y</w>': 1, '   de ma go gu e</w>': 1, '   ex a g ger a tes</w>': 2, '   di ss er vi ce</w>': 2, '   a k b ar</w>': 1, '   re le ar n</w>': 1, '   men i al</w>': 2, '   de ce i t ful</w>': 5, '   sor ri est</w>': 1, '   1 1 4 </w>': 1, '   con cu r r en tly</w>': 3, '   char le st own</w>': 1, '   ba lling</w>': 2, '   re for ma t ory</w>': 1, '   f ra min g ha m</w>': 1, '   pe g</w>': 8, '   re gre tting</w>': 1, '   de lin qu ent</w>': 5, '   can d or</w>': 2, '   si d ne y</w>': 116, '   un tru th s</w>': 1, '   sh ee et</w>': 1, '   ri b s</w>': 10, '   p al med</w>': 1, '   b al dy</w>': 3, '   bu ll e t pro of</w>': 5, '   re et</w>': 1, '   tal cu m</w>': 2, '   jo lli es</w>': 2, '   b lo tter</w>': 2, '   pe di gre ed</w>': 1, '   par c el ed</w>': 1, '   c rea sy</w>': 44, '   ro s an na</w>': 3, '   re lo a d</w>': 2, '   poli ce men</w>': 19, '   ga lls</w>': 2, '   gra s p</w>': 15, '   f oun d ers</w>': 2, '   h er man i dad</w>': 6, '   car n e</w>': 1, '   a sa da</w>': 1, '   h om b re</w>': 4, '   fu e go</w>': 2, '   mu g sho ts</w>': 1, '   che ck er ed</w>': 2, '   inter po l</w>': 15, ' sen or</w>': 2, ' ? ] </w>': 53, '   ju ar e z</w>': 15, '   k al fu s</w>': 4, '   de po si ts</w>': 11, '   with d ra wa ls</w>': 3, '   ra mo s</w>': 9, '   pa tr one</w>': 1, '   ta z in ar i</w>': 4, '   cor o ll a</w>': 1, ' 3 1 7 0 4 </w>': 1, '   di g it</w>': 4, '   mu er te</w>': 2, ' au ri lli o</w>': 1, '   ro sa s</w>': 3, ' dan i el</w>': 4, ' s i</w>': 1, ' re in a</w>': 1, ' ki d na pp ing</w>': 1, ' ! ] </w>': 9, '   ce ll ph one</w>': 1, ' ro man s</w>': 1, '   o ver come</w>': 14, ' " ] </w>': 3, '   r ou tes</w>': 6, '   ar c o</w>': 1, ' thir ds</w>': 4, '   han d b all</w>': 2, '   cha pu l te pe c</w>': 1, '   p in ta</w>': 16, '   gu ar di ans</w>': 6, '   ne go ti at ors</w>': 1, ' or der ed</w>': 2, '   j or ge</w>': 13, '   ra mi re z</w>': 3, ' dri ve</w>': 5, ' - ] </w>': 6, ' ne go ti able</w>': 3, ' lu ck y</w>': 5, '   com man dan te</w>': 2, ' cor re ct</w>': 1, '   co or din ation</w>': 2, '   si g</w>': 8, '   sa u er</w>': 3, '   2 2 6 </w>': 1, '   la un ch ers</w>': 1, '   we b le y</w>': 1, ' 3 2 </w>': 4, '   pri m er</w>': 7, '   mi s fi re</w>': 1, '   ju di ci al s</w>': 1, '   c rea se</w>': 4, '   im pu l se</w>': 44, '   ra y bur n</w>': 1, '   in e s cap able</w>': 1, '   ju de</w>': 19, ' who ever</w>': 4, '   d y sle x i c</w>': 4, '   s wi mm er</w>': 6, '   un tra in ed</w>': 7, ' show</w>': 14, '   mu l ti p li ed</w>': 6, '   con cu b ine</w>': 2, '   inter sch oo ls</w>': 1, '   f re e st y le</w>': 1, '   an k le</w>': 18, '   m ac a w</w>': 1, '   ki d na pp in gs</w>': 2, '   thin s</w>': 2, ' bi r d</w>': 9, '   fran k fu r t</w>': 7, '   e in</w>': 6, '   k lin es</w>': 1, '   bi ss ch en</w>': 1, '   f oo t bri dge</w>': 1, '   re for ma</w>': 2, '   au to se que st r a</w>': 1, '   sa fe house</w>': 4, '   so ci able</w>': 3, '   w ea ke st</w>': 4, '   ca mi l a</w>': 2, '   v al en ci as</w>': 1, '   ar co s</w>': 1, '   gu er r er o</w>': 1, '   in com pe t ence</w>': 4, '   bo d y gu ar ds</w>': 11, '   a i g</w>': 1, '   co lu m bu s</w>': 6, '   can v as</w>': 10, ' c rea sy</w>': 2, '   s ki mp</w>': 1, '   ex tra v a g an ce</w>': 3, '   ban k ru p t cy</w>': 5, '   stra ta</w>': 1, '   pre te xt</w>': 3, '   bar ri o</w>': 1, '   out c ry</w>': 1, '   tri bu n al s</w>': 1, '   le ban on</w>': 2, '   d ru ze</w>': 1, '   com man d ers</w>': 6, '   pi bi l</w>': 1, '   ch ori z o</w>': 1, '   h mm m h</w>': 1, '   mar in ate</w>': 1, '   a de pt</w>': 2, '   in na</w>': 39, '   h er bi e</w>': 4, '   mi lli e</w>': 17, '   wa d da ya</w>': 15, '   an gi e</w>': 17, '   d ow n na</w>': 1, '   im poli te</w>': 7, '   cl ar a</w>': 12, '   c ans a</w>': 1, '   si d</w>': 194, '   fe en ey</w>': 4, '   a ll a</w>': 15, '   r k o</w>': 5, '   che ster</w>': 2, '   wa d da</w>': 8, '   on na</w>': 10, '   ca ther ine</w>': 74, '   g loo m</w>': 1, '   par ta</w>': 2, '   pi ll e tt i</w>': 6, '   p on d ers</w>': 1, '   af ra i da</w>': 1, '   sha w l</w>': 2, '   fr on ta</w>': 1, '   ther e s a</w>': 20, '   wan t s a</w>': 6, '   u f a</w>': 1, '   p le u ri sy</w>': 2, '   di gi or gi o</w>': 1, '   ab ru z z i</w>': 2, '   ni c ki e</w>': 9, '   ac h es</w>': 4, '   th ro b s</w>': 2, '   sp ou ting</w>': 5, '   wa tta</w>': 1, '   h un k a</w>': 1, ' chi ck en</w>': 7, '   i ce bo x</w>': 5, '   por t che ster</w>': 2, '   ad min i stra tive</w>': 6, '   we b ster</w>': 60, '   ga z z ar a</w>': 4, '   mar k up</w>': 1, '   su per mar ke ts</w>': 4, '   mo st a</w>': 1, '   tw en ny</w>': 6, '   bi ll er</w>': 4, '   un mar ried</w>': 6, '   thin k a</w>': 1, '   the o d ore</w>': 3, '   for d ha m</w>': 1, '   ar oun na</w>': 5, '   je su it</w>': 7, '   go o d he ar ted</w>': 2, '   bur st ing</w>': 3, '   c r y er</w>': 1, '   har sh ly</w>': 3, '   ha t che ck</w>': 1, '   re ev es</w>': 3, '   thou gh ta</w>': 1, '   spi ll an e</w>': 5, '   ga sp ing</w>': 1, '   ce ll op han e</w>': 3, '   can na</w>': 4, '   re ta il</w>': 12, '   mer chan ts</w>': 3, '   al do</w>': 1, '   ca pe ll i</w>': 1, '   da ir y man</w>': 1, '   g ro c er</w>': 5, '   d ow na</w>': 2, '   pl en ny</w>': 2, '   s n y der</w>': 11, '   star d ust</w>': 6, '   ri d da</w>': 1, '   ne x ta</w>': 1, '   fu s ar i</w>': 3, '   wa t s a</w>': 2, '   can du so</w>': 1, '   k in g s bri dge</w>': 1, '   g in ni e</w>': 2, '   k a pl ans</w>': 1, '   in st ea da</w>': 1, '   bo tt l a</w>': 1, '   any wh er es</w>': 1, ' ca ther ine</w>': 1, '   p ans</w>': 8, '   sha d d up</w>': 5, '   pre ven ting</w>': 5, '   l ou sed</w>': 2, '   s ki ds</w>': 2, '   par ted</w>': 4, '   wal d ow s k i</w>': 3, '   bl in d f old</w>': 4, '   bo o z ing</w>': 1, '   com man ding</w>': 7, '   cu tt ers</w>': 5, '   fi g ger</w>': 11, '   hou li ha n</w>': 3, '   han k er ing</w>': 2, '   e pi du ra l</w>': 4, '   he ma tom a</w>': 4, '   o li ves</w>': 10, '   ba h st on</w>': 1, ' j on</w>': 19, '   de mo cra sh</w>': 1, '   thin e</w>': 5, '   in to x i can ts</w>': 1, '   hou se boy</w>': 2, '   per fe c tion i st</w>': 2, '   s li ver</w>': 1, '   spe ar ch u ck er</w>': 3, '   con ce ssion</w>': 3, '   se ou l</w>': 4, ' ro lling</w>': 3, ' ar m</w>': 8, '   re ha bi li ta ting</w>': 1, '   cho pp ers</w>': 8, ' of a</w>': 1, '   ha w ke ye</w>': 11, '   ev ac </w>': 2, '   ta e gu </w>': 1, '   ni gr a</w>': 1, ' pe ar ch u ck er</w>': 1, '   an dro s co g g in</w>': 6, '   be ar ers</w>': 2, '   c ro a ks</w>': 1, '   tra pp er</w>': 11, '   da go</w>': 5, '   b ead</w>': 2, ' j i g g l er</w>': 1, '   en a me l</w>': 2, '   pa in le es</w>': 1, '   pa ss er</w>': 1, '   inter ce p ting</w>': 1, '   pu l mon ary</w>': 4, ' cu tter</w>': 4, '   cu r b</w>': 13, '   in i ti al s</w>': 17, '   bra y more</w>': 1, '   c r ab app le</w>': 1, '   co ve</w>': 3, '   a y uh</w>': 9, '   for rest</w>': 6, '   ra ven ous</w>': 2, ' t re sp as ses</w>': 1, '   an e</w>': 1, '   gi m mi cks</w>': 1, '   mi su n der stan din gs</w>': 5, '   3 2 5 </w>': 1, '   co ac h ed</w>': 1, '   m ea t b all</w>': 4, '   re t ro per i ton ea l</w>': 1, '   sig mo id</w>': 1, '   b ow el</w>': 6, '   s an d ba g</w>': 3, '   be tch er</w>': 2, '   qu ar ter back</w>': 7, '   d on er</w>': 1, ' ne ga tive</w>': 4, ' ma tch ed</w>': 1, '   mo an er</w>': 1, '   dro op y</w>': 2, '   li m bur ger</w>': 1, '   ca me mb er t</w>': 3, '   sur gi cal</w>': 18, ' pr o</w>': 2, '   r in ger</w>': 12, '   ne u ro sur ge on</w>': 2, '   o li ver</w>': 29, '   har m on</w>': 1, '   er o de</w>': 2, '   ca v a</w>': 3, '   j i g g l ed</w>': 1, '   a spi ra te</w>': 3, '   lo b st ers</w>': 5, '   au stra li an</w>': 4, '   gi bra l t ar</w>': 2, '   ca d die</w>': 2, '   ba il ed</w>': 7, '   con gre ss man</w>': 4, '   so ci a li z ing</w>': 3, '   mar st on</w>': 1, '   pre par a tions</w>': 9, '   m c in t y re</w>': 2, '   ou ts</w>': 6, '   ha m tr ack</w>': 2, '   con du ctor</w>': 23, '   k no ck o</w>': 1, '   wi l ma</w>': 5, '   in con si st ent</w>': 3, ' ha w ke ye</w>': 1, '   ob ser ving</w>': 4, '   o ver si z ed</w>': 2, ' a th le tes</w>': 1, '   ha l f back</w>': 2, '   ra ms</w>': 6, '   out con ned</w>': 1, '   ha l ves</w>': 5, ' pre s sure</w>': 6, '   im per son al</w>': 8, '   ther a pe u ti c</w>': 10, '   re li ev ing</w>': 1, '   ten si ons</w>': 4, '   t ro op ship</w>': 1, '   mo hi c ans</w>': 1, '   so l ving</w>': 8, '   un f it</w>': 4, '   cu te st</w>': 13, ' ja p an e se</w>': 1, '   an e s the ti st</w>': 1, '   in spe ct</w>': 10, '   b oun c er</w>': 3, '   y a m ac hi</w>': 1, '   wh or e house</w>': 10, '   sc run ch</w>': 4, '   cla m di g g ers</w>': 1, '   do ver</w>': 1, '   s mo k ey</w>': 25, '   ex t in gu i sh ing</w>': 2, '   fi an ce es</w>': 1, '   re pre ss ing</w>': 1, '   in hi bi tion</w>': 1, '   ju ani s m</w>': 1, '   no ti c ing</w>': 8, '   man ea ting</w>': 1, '   den ti sts</w>': 5, '   for re sts</w>': 1, '   de lu ge</w>': 3, '   dis ci pl in ary</w>': 2, '   j ab ber ing</w>': 1, '   h eah</w>': 1, '   re ma tch</w>': 3, '   cla u st ro p ho bi a</w>': 1, ' ri e lly</w>': 1, ' co ac h ed</w>': 1, '   t ack les</w>': 1, '   ha l fa ssed</w>': 1, '   li pi o do l</w>': 1, '   car ts</w>': 2, '   mer ri ll</w>': 8, ' gi ven</w>': 6, '   f oo th old</w>': 3, '   ri ck sha w s</w>': 1, '   co lu m n</w>': 43, '   n ort on</w>': 16, '   pla ti tu d es</w>': 2, ' cl ou ds</w>': 1, '   lin in gs</w>': 2, '   wi ll ou gh by</w>': 5, '   sh el don</w>': 23, '   u ban g i</w>': 1, ' ear ed</w>': 3, '   lo p</w>': 3, '   be lli ed</w>': 1, '   y uh</w>': 45, '   u m pi re</w>': 2, '   do o hi c ki es</w>': 1, '   har mon i c a</w>': 2, '   pre si den cy</w>': 11, '   re com men ds</w>': 4, '   for ma tion</w>': 5, '   j ell</w>': 6, '   we b st ers</w>': 2, '   win d f all</w>': 2, '   c lu ck</w>': 2, ' lea gu e</w>': 2, '   f l y ca st ing</w>': 1, '   li be l</w>': 3, '   a w w w w w</w>': 1, ' si x th</w>': 4, '   bu ll e t in</w>': 10, '   du e ts</w>': 2, '   con n ell</w>': 31, '   can di da tes</w>': 2, '   ch r on i c le</w>': 5, '   ra in ing</w>': 14, ' be any</w>': 1, '   l un k head</w>': 1, '   dis cou ra ge ment</w>': 2, '   cha o ti c</w>': 2, '   sle dge</w>': 4, '   ha mm ers</w>': 3, ' start</w>': 3, ' w o od</w>': 1, '   tu b by</w>': 3, '   be any</w>': 5, '   he e lot</w>': 2, ' ri ke</w>': 1, ' hund re ds</w>': 4, ' sho es</w>': 3, '   lo v able</w>': 4, '   he e lo ts</w>': 2, ' r ab b it</w>': 4, '   what s a</w>': 9, '   a hold</w>': 11, '   g ru b be l</w>': 5, ' ac coun t</w>': 1, '   h er m it</w>': 4, '   po st man</w>': 3, '   s our pu ss</w>': 5, '   do g gon ed</w>': 1, '   sch wa b ack er</w>': 1, '   de lan ey</w>': 9, '   la w n s</w>': 4, '   de lan e ys</w>': 4, '   mi s er</w>': 5, '   g ri mes</w>': 12, '   p ee ked</w>': 5, '   th</w>': 34, '   s ni ck ers</w>': 8, '   s mi th ers</w>': 1, '   d ou gh nu ts</w>': 6, '   j i tt er bu gs</w>': 1, '   fe ll ers</w>': 4, '   ha m me t t</w>': 1, '   g an der</w>': 4, '   un men tion able</w>': 1, '   li gh thou ses</w>': 3, '   hi y ah</w>': 1, ' se tter</w>': 1, '   se tter</w>': 2, '   mon ta ge</w>': 2, '   sha kes</w>': 21, '   ne ga ti ve ly</w>': 1, ' bu ll e t in</w>': 1, '   di mes</w>': 6, '   le c tur ing</w>': 7, '   i do li ze</w>': 1, ' f lo pped</w>': 1, '   o car in a</w>': 2, '   do g gone</w>': 2, '   chi pped</w>': 10, '   pi tch in</w>': 1, ' in ning</w>': 1, '   b re w ster</w>': 5, ' gu ll</w>': 4, '   j u</w>': 9, '   om en</w>': 4, '   gu ll</w>': 4, '   sp en c er</w>': 25, ' e le ction</w>': 3, '   3 0 6 </w>': 3, '   gre g</w>': 25, '   bar ne t t</w>': 21, '   so l is</w>': 3, '   s w at</w>': 18, ' ph on ey</w>': 1, '   g ran ny</w>': 6, '   an ton u c c i</w>': 4, '   he i st</w>': 7, '   t ea l</w>': 3, '   en t an g l ed</w>': 2, '   k or da</w>': 5, '   e ll in g ton</w>': 1, '   ba ffer t</w>': 2, '   la mar r a</w>': 1, '   l ow b all</w>': 1, '   sp r in g fi e ld</w>': 2, '   har d to p</w>': 1, '   f re qu en ts</w>': 2, '   ve hi c les</w>': 22, '   s ma ll time</w>': 1, '   4 5 9 </w>': 1, '   de me an or</w>': 2, '   bo tch ed</w>': 7, '   ki mu r a</w>': 2, '   wi per</w>': 2, '   m c call</w>': 11, '   shi p y ard</w>': 3, '   ha un t</w>': 11, '   vi si bi li ty</w>': 3, ' mo bi le</w>': 2, '   exac ta</w>': 1, ' ac </w>': 1, '   wa sh y</w>': 1, ' jo ck ey</w>': 1, '   com b o</w>': 4, '   c ro ss fi re</w>': 3, '   sa w ed</w>': 1, '   ber re tta</w>': 1, ' pa per</w>': 5, '   com man d men ts</w>': 3, '   ne go ti a tor</w>': 10, '   fro g shit</w>': 1, '   ho bo k en</w>': 1, '   li v in g t st on</w>': 1, '   gra du a tes</w>': 2, ' su i ci de</w>': 3, ' fa st</w>': 5, '   ex ci ta ble</w>': 1, '   mar k s man</w>': 2, '   re c ru i ted</w>': 12, '   la ter al</w>': 6, '   f la g ran te</w>': 1, '   me tr o</w>': 15, '   ter e s a</w>': 40, '   un e mp lo y ment</w>': 14, '   for ga ve</w>': 5, '   ton gs</w>': 1, '   th on gs</w>': 1, '   p ru de</w>': 3, ' ne k ked</w>': 1, ' ro ad te st</w>': 1, '   s co t ti e</w>': 32, '   p ac o</w>': 4, '   cra ving</w>': 2, '   gar li c</w>': 12, '   p le e ea se</w>': 2, '   b ack tr ack</w>': 6, '   su per i ors</w>': 13, '   je w el ers</w>': 1, '   sa le spe ople</w>': 2, '   el der ly</w>': 5, '   re po ed</w>': 1, '   he ar tw ar m ing</w>': 5, '   ha l d en</w>': 3, '   gen a</w>': 1, '   re d w o od</w>': 2, '   di v in i ty</w>': 5, '   i ll u si ons</w>': 12, '   re con figu ra tion</w>': 1, '   pro fu se ly</w>': 2, '   v ou ch ers</w>': 3, '   dis k</w>': 16, '   t an ner</w>': 10, '   di gi ti z ed</w>': 3, '   di gi ti ze</w>': 3, '   di gi ti z ation</w>': 1, '   mo du le</w>': 11, '   st af f ing</w>': 1, '   chri st op h</w>': 6, '   mo du les</w>': 1, '   ter r ace</w>': 7, '   re gs</w>': 2, ' di gi ti z ation</w>': 1, '   re je c ting</w>': 1, ' into</w>': 5, '   de b it</w>': 2, '   un i</w>': 4, ' n et</w>': 17, '   m ou th pi e ce</w>': 5, ' ei gh te en</w>': 2, '   c ome ts</w>': 2, '   0 0 9 8 4 3 </w>': 2, '   z or don</w>': 3, '   p ha e do s</w>': 2, '   y i</w>': 5, '   st y l in</w>': 1, '   a i sh a</w>': 1, '   st al w art</w>': 1, '   na th a di ans</w>': 1, ' n in</w>': 1, ' je tt i</w>': 1, ' ani ma l</w>': 1, '   n in je tt i</w>': 3, '   s qu ir b s</w>': 1, ' con du c t ors</w>': 1, '   v a st ly</w>': 3, '   ac e ll er a ted</w>': 1, '   ob ser v at ory</w>': 8, '   du l ce a</w>': 2, '   ac ti v a ting</w>': 3, '   ver m in</w>': 6, '   u l tr a</w>': 9, '   pu l s ar</w>': 2, '   com pre h en sion</w>': 8, '   re tr o</w>': 2, ' fi tt ed</w>': 1, '   op t i</w>': 1, '   de p th s</w>': 7, '   g ro ve</w>': 12, '   tri ac </w>': 1, '   tri ac s</w>': 2, '   s wi ft</w>': 8, ' na th a di ans</w>': 1, '   mon o li th</w>': 2, '   te en a g ers</w>': 18, '   n in j a</w>': 10, '   z or d</w>': 4, '   tri cer a to ps</w>': 3, '   p ter o d ac t y l</w>': 1, '   com et</w>': 5, '   mor ph ers</w>': 1, '   me ga z or d</w>': 1, '   bu l k</w>': 5, '   st e ll ar</w>': 4, '   hur t ling</w>': 1, '   di t to</w>': 5, '   s w o op ing</w>': 2, '   f al c on</w>': 13, '   win ged</w>': 3, '   gen e s is</w>': 45, '   ga p</w>': 7, '   mor d ant</w>': 3, '   on i ons</w>': 14, '   ch ee se d on gs</w>': 1, ' im on</w>': 1, ' u ck</w>': 1, '   do d der ing</w>': 1, '   p un y</w>': 6, '   re bu il ding</w>': 4, '   en ti ce</w>': 1, '   ex o s ke le ton</w>': 2, '   h or ni tor</w>': 1, '   sc or pi tr on</w>': 1, '   e c to</w>': 1, ' mor p hi c on</w>': 1, '   an ni hi late</w>': 4, ' ba si ca lly</w>': 2, '   ra q ing</w>': 1, '   ga st r on om i c</w>': 1, '   tra ve</w>': 1, '   ri ta</w>': 34, '   r r gh</w>': 1, '   mm ff pp r r</w>': 1, '   br gh uh</w>': 1, '   bi s mar k</w>': 3, '   an g l o</w>': 15, ' sa x ons</w>': 1, '   ki ke</w>': 6, '   a w wh </w>': 1, '   t or ri o</w>': 2, '   si ci li an</w>': 12, '   co lo si m o</w>': 1, ' si ci ly</w>': 1, '   lu ci an o</w>': 7, '   cl ow es</w>': 1, '   me y er</w>': 10, '   en t re pr en e u rs</w>': 2, '   pro hi bi tion</w>': 3, '   o ver ri pe</w>': 2, '   ro th st e in</w>': 27, '   mar an z an o</w>': 36, '   ma ss er i a</w>': 17, '   re in a</w>': 6, '   con so les</w>': 1, '   ber ea ved</w>': 4, '   per pe tra tor</w>': 6, '   bar one</w>': 1, '   fin d in</w>': 3, '   co ll</w>': 1, '   ha un t in</w>': 2, '   bu g sy</w>': 5, '   pa in t in</w>': 2, '   pro f ac i</w>': 5, '   sp rea d in</w>': 1, '   hi d in</w>': 4, '   hon or ary</w>': 6, '   k re pl ac h</w>': 1, '   ra vi o l i</w>': 1, '   s w ea t bo x</w>': 1, '   sc rea m in</w>': 10, '   mo li ar i</w>': 1, ' char lie</w>': 6, '   in c lu d in</w>': 1, '   au di ted</w>': 2, '   c c</w>': 7, '   lea d in</w>': 1, '   ch oo s in</w>': 1, '   nu ck y</w>': 2, '   l ans k y</w>': 4, '   si e ge l</w>': 1, '   d ons</w>': 2, '   ven de tt as</w>': 3, ' ja m in</w>': 1, '   shi t face</w>': 1, '   k lo b</w>': 1, '   ger ar do</w>': 1, '   fr a</w>': 2, '   di a vo l o</w>': 1, '   an ti pa s to</w>': 1, '   pa st ry</w>': 8, '   car l o</w>': 12, '   ga mb in o</w>': 3, '   v in ni e</w>': 5, '   man g an o</w>': 1, '   wa ss a</w>': 1, '   bo ar d wal k</w>': 9, '   son ny</w>': 116, '   ba mb in o</w>': 1, '   t rea t in</w>': 4, '   ca l ab ri an</w>': 2, '   sa l</w>': 139, ' v a</w>': 1, '   ri gh t fu lly</w>': 4, '   re i g n s</w>': 2, '   gi u se pp e</w>': 2, '   com b ine</w>': 3, '   le d g ers</w>': 3, '   ven de tta</w>': 5, '   mu s so lin i</w>': 2, '   sa lli e</w>': 1, '   ho gs</w>': 5, '   me tt er ni ch</w>': 3, '   ra lli ed</w>': 1, '   pu pp i es</w>': 13, '   di sti ll ers</w>': 1, '   s win g in</w>': 3, '   shu d d up</w>': 2, '   ne w s boy</w>': 2, '   men s ch</w>': 3, '   joh n s</w>': 31, '   u ten si ls</w>': 2, '   car ves</w>': 1, '   st u</w>': 55, ' au to gra ph ing</w>': 1, ' fun ny</w>': 8, ' sto pped</w>': 1, '   mon ke y b one</w>': 11, '   bi ca mer al</w>': 1, '   dis j un ction</w>': 1, '   ri gh ti e</w>': 1, '   pl u per fe ct</w>': 1, '   jo ked</w>': 1, ' cu r ing</w>': 1, '   mar t in is</w>': 11, ' ni gh t mar es</w>': 1, ' ad d</w>': 1, ' in i</w>': 1, ' ven tri lo qui st</w>': 1, '   com men t ary</w>': 5, '   mi le y</w>': 3, '   pi ca s so</w>': 8, ' gu er ni c a</w>': 1, ' b end</w>': 1, '   ch u ck le</w>': 2, '   dar ned</w>': 16, '   u r in al</w>': 1, ' ir re sp on si ble</w>': 1, '   h y p no s</w>': 5, '   d ra ma ti ca lly</w>': 4, '   e de l st e in</w>': 1, ' h or r or</w>': 1, '   co ma s</w>': 1, ' m ac h in es</w>': 1, '   st ab i li z ed</w>': 5, ' at a</w>': 4, '   ba z oo m</w>': 1, ' en ter ta in ment</w>': 1, '   gi v ea way</w>': 6, '   per pe tu a ted</w>': 1, ' st ri ke</w>': 2, '   la mb or gh in is</w>': 1, '   m ou th wa sh</w>': 3, '   e mp tive</w>': 1, '   en d or se ment</w>': 3, '   po t lo a d</w>': 1, '   bea k er</w>': 2, ' sc are</w>': 1, '   on ei ri x</w>': 1, ' ni gh t m are</w>': 3, '   po e</w>': 13, '   a ti ll a</w>': 2, '   gen gh is</w>': 2, '   k ha n</w>': 14, '   s na tch ing</w>': 4, ' ti ck e ts</w>': 1, '   h y p</w>': 1, '   sch ti ck</w>': 1, ' gra te ful</w>': 1, ' pu t z</w>': 1, '   ba i li wi ck</w>': 1, ' f ar e well</w>': 1, ' die</w>': 5, ' ca vi ar</w>': 2, ' shi ts</w>': 2, '   he ir loo m</w>': 4, ' used</w>': 9, '   ab b a</w>': 5, '   d ab b a</w>': 1, '   s mar ty</w>': 4, ' to p co at</w>': 1, ' po t lo a d</w>': 1, '   per t</w>': 1, '   sa u cy</w>': 2, '   m c el ro y</w>': 2, ' mon ke y b one</w>': 1, '   a wa k en s</w>': 1, ' word</w>': 11, '   st in kin</w>': 8, ' ki lls</w>': 3, ' me di ca lly</w>': 1, ' rest</w>': 1, '   s wan ki er</w>': 1, '   ra ke</w>': 4, ' ta il</w>': 3, '   loo k er</w>': 2, ' e spe ci ally</w>': 6, '   ki m my</w>': 10, '   g ran d ma ma</w>': 10, ' en ga ge ment</w>': 1, ' war n</w>': 1, '   fi g ment</w>': 7, '   lu l u</w>': 5, ' st u</w>': 2, '   sch war z en e g ger</w>': 2, ' si de ki ck</w>': 1, '   wi se c r ac ks</w>': 2, ' ab so lu te</w>': 2, '   re su m</w>': 1, ' de ser ve</w>': 4, ' app ly</w>': 1, ' de pre ssed</w>': 1, ' par ty</w>': 4, ' do o dle</w>': 1, '   rea per</w>': 3, ' mon th s</w>': 4, ' ki ck</w>': 5, '   ca s k et</w>': 9, '   un di es</w>': 8, '   si de ki cks</w>': 1, ' wa tch</w>': 11, '   re por t in</w>': 1, '   bo l d ly</w>': 2, '   wi sh in</w>': 2, ' spe ci es</w>': 3, '   mo on ing</w>': 1, ' coun t</w>': 6, '   ma so chi sti c</w>': 3, '   b y r n e</w>': 1, '   pre sc ri b ing</w>': 2, '   c lu b house</w>': 3, '   rea s sur ing</w>': 8, '   si ou x</w>': 11, '   ca pp u c c in o</w>': 11, '   mar i k a</w>': 1, '   pa u l a</w>': 21, '   to lls</w>': 2, '   he ll ho le</w>': 5, ' vi ce</w>': 1, ' pre si den ts</w>': 4, '   pre si d es</w>': 2, '   bu sts</w>': 6, '   si gh t se ers</w>': 2, ' re gu l ar</w>': 1, '   g al er y</w>': 1, '                     </w>': 4, '   cl in ch es</w>': 1, '   or a tor</w>': 2, ' isn</w>': 10, ' pro mi sed</w>': 2, '   app ro pri a tions</w>': 1, ' tru mp ed</w>': 1, '   bu g le</w>': 1, '   pa ine</w>': 41, '   sa und ers</w>': 43, '   bl in d fo l ding</w>': 1, ' a fi re</w>': 1, '   bar ging</w>': 9, ' re sig n</w>': 1, '   bo lon ey</w>': 1, ' fi ght</w>': 7, ' r ack</w>': 1, ' cl ar i ss a</w>': 2, ' car es</w>': 3, ' wa tch in</w>': 1, '   to d d ling</w>': 1, '   bi b</w>': 1, ' ho p ing</w>': 1, '   hu mp ty</w>': 5, ' du mp ty</w>': 2, ' fo ll ow in</w>': 1, '   fa ss</w>': 1, '   p oun d in</w>': 1, '   dr un ks</w>': 6, '   r ar est</w>': 3, '   gen t ee l</w>': 1, '   wi sed</w>': 3, '   do p es</w>': 2, '   re ga l</w>': 2, '   cl ou ts</w>': 1, '   si cking</w>': 1, ' di z</w>': 2, '   de co y ed</w>': 1, ' wi ll et</w>': 3, '   ti gh t ro pe</w>': 5, ' left</w>': 6, ' ro pe</w>': 1, ' pu ss</w>': 2, '   m c g an n</w>': 13, '   com men ces</w>': 1, ' wa tch do g</w>': 1, '   te le ph on ing</w>': 2, ' whole</w>': 3, ' si mm er</w>': 1, ' pla ying</w>': 4, '   bi b s</w>': 1, ' cha mb er ma id</w>': 1, '   pi ed</w>': 2, '   pi per</w>': 4, ' hon or ary</w>': 1, ' a m bi tion</w>': 1, '   com b ing</w>': 3, '   pla y ma te</w>': 4, ' po et</w>': 2, '   cor re sp on den ts</w>': 3, '   fo le y</w>': 43, '   s qu ir ts</w>': 3, ' m c g an n</w>': 1, ' pa ine</w>': 1, '   b la z er</w>': 2, '   b la z ing</w>': 5, ' h or ace</w>': 2, ' without</w>': 19, '   se tt les</w>': 11, '   beli tt l ed</w>': 2, '   cor n er st on es</w>': 1, '   sta te s man</w>': 3, '   ho pp er</w>': 11, ' fi re</w>': 7, '   w inter bo ok</w>': 1, ' s we en ey</w>': 1, '   f ar re ll</w>': 3, ' wai ting</w>': 6, ' lot</w>': 10, '   sta tion er y</w>': 4, ' sto od</w>': 1, '   h or ace</w>': 12, ' im me di a tely</w>': 1, '   man na</w>': 4, ' j i m</w>': 6, ' man na</w>': 1, ' mi ll er</w>': 1, '   ca ll ou s ly</w>': 1, ' feel</w>': 2, ' har mon y</w>': 1, '   g ab </w>': 1, '   con f er</w>': 4, ' en or m ous</w>': 1, ' t ar dy</w>': 1, ' everybody</w>': 14, '   s we ll est</w>': 2, '   sp lin t ers</w>': 5, '   mi me o gra p h</w>': 2, ' je ffer son</w>': 3, ' thought</w>': 8, ' ta y l or</w>': 2, '   tw er p</w>': 4, '   pu b li sh er</w>': 18, ' sen a tor</w>': 5, '   mon u men ts</w>': 7, '   li ve sto ck</w>': 3, '   re c ess</w>': 5, '   bab bl in gs</w>': 1, ' de la y ed</w>': 1, ' mi lli ons</w>': 3, '   stan d still</w>': 1, ' go ver n ment</w>': 3, '   po st p on e ment</w>': 1, '   f an ci fu lly</w>': 1, '   dro ves</w>': 1, ' saw</w>': 7, '   im pu ted</w>': 2, '   da m</w>': 36, ' f ra u d</w>': 1, ' star ted</w>': 1, '   sc ru ff</w>': 1, ' a spe ct</w>': 1, '   wi ll et</w>': 12, ' questi on</w>': 3, ' y i e ld</w>': 1, ' gen t le men</w>': 4, '   ri s en</w>': 3, ' b en e fi ts</w>': 1, '   le gi s la ture</w>': 3, ' wrong</w>': 4, ' looks</w>': 6, ' pre s ent</w>': 1, ' bi lls</w>': 4, '   s lu mp ed</w>': 1, '   min er</w>': 3, '   in ti mi da tion</w>': 3, '   cha m pi ons</w>': 3, '   cla y ton</w>': 26, '   pr in ter</w>': 2, '   a mo s</w>': 4, '   cl ar i ss a</w>': 3, '   bl ac ki e</w>': 4, '   que en i e</w>': 1, '   har d en</w>': 1, ' na tu ra lly</w>': 2, ' ow ning</w>': 1, ' sti ck</w>': 6, ' remember</w>': 11, '   un com pro mi sing</w>': 1, ' ri gh t ne ss</w>': 1, '   ta y l ors</w>': 1, '   co ck e y ed</w>': 4, ' tru th</w>': 11, ' sa und ers</w>': 4, '   m ea s ly</w>': 10, '   j ack al s</w>': 3, ' haven</w>': 9, '   pre ser ves</w>': 2, ' h or ser a di sh</w>': 1, ' pr in c ess</w>': 3, ' e sc or t</w>': 3, ' as king</w>': 2, ' je ff</w>': 2, ' you rs</w>': 10, '   pla gu es</w>': 7, ' see k ers</w>': 1, '   c ran ks</w>': 3, ' ou tta</w>': 2, '   h y m n</w>': 2, '   sp an g l ed</w>': 1, ' rea son</w>': 6, '   si tu a ted</w>': 3, '   l en a</w>': 15, '   le ti ti a</w>': 3, '   ab i ga il</w>': 2, ' vi o let</w>': 1, ' d we ll ers</w>': 1, ' d we ll er</w>': 1, '   p ra i ri es</w>': 1, ' getting</w>': 4, ' i dea ls</w>': 1, ' go sh</w>': 5, ' par ti cu l ars</w>': 1, '   li gh ted</w>': 5, ' t wi ce</w>': 2, ' sha ll</w>': 3, '   a men d men ts</w>': 1, ' wai ts</w>': 1, '   st y mi e</w>': 1, '   lo b b y i sts</w>': 1, '   de part men ts</w>': 4, ' ca b in et</w>': 1, ' bu d get</w>': 1, '   bu rea us</w>': 1, ' e m ba ssi es</w>': 1, '   vi vi se ction</w>': 1, ' su b</w>': 1, '   he ar in gs</w>': 3, ' com mi tt e es</w>': 1, '   si ft</w>': 2, ' stu dy</w>': 1, '   ro st ru m</w>': 1, '   for e fin ger</w>': 1, '   tra ys</w>': 3, ' brea king</w>': 3, ' tri p le ts</w>': 1, '   fa in te st</w>': 9, ' p un ch</w>': 2, ' no on</w>': 2, '   con ven es</w>': 2, '   par li a men t ary</w>': 1, ' sp ar k ling</w>': 1, '   cu st om ary</w>': 6, ' le ar n</w>': 2, '   cour ting</w>': 2, ' cla y ton</w>': 1, '   pe s k y</w>': 3, ' sp l en did</w>': 1, '   tru th s</w>': 9, '   un a li en able</w>': 1, '   qu or u m</w>': 3, ' ar ms</w>': 5, ' com pe l</w>': 1, ' f ra med</w>': 1, ' im pre ssion</w>': 1, '   l un ged</w>': 1, '   so le m n ly</w>': 3, ' aga in st</w>': 5, '   ad min i st er ed</w>': 4, '   cor n er st one</w>': 4, '   a pr on</w>': 6, '   e mb ro i der ed</w>': 2, ' pen sion</w>': 2, '   pa h w er fu lly</w>': 1, ' fi gh ts</w>': 1, '   pi ll ori ed</w>': 1, '   di st ri c ts</w>': 2, ' while</w>': 7, ' tomorrow</w>': 5, '   le er y</w>': 2, ' a ma z ing</w>': 1, '   nor th ea st</w>': 13, '   g ri f fi th</w>': 3, ' s co o p</w>': 1, ' le m me</w>': 2, ' story</w>': 6, '   ex pe l</w>': 6, '   com pro mi ses</w>': 6, ' co ver ing</w>': 1, ' im possi ble</w>': 2, '   re sig n s</w>': 1, ' b ran ded</w>': 1, ' ex pe lled</w>': 1, ' go ver n or</w>': 1, '   fi tt ed</w>': 13, ' sa m</w>': 4, ' comp li ment</w>': 1, '   y ok el</w>': 5, '   c ru ci f ying</w>': 4, '   ra d ner</w>': 1, '   di g gs</w>': 1, '   st ea m ro ll er</w>': 5, ' spe cked</w>': 1, '   app ro pri ation</w>': 1, '   be ha ves</w>': 1, ' s mi th</w>': 2, '   ba ll y ho o</w>': 1, '   re ci tes</w>': 1, ' tur ned</w>': 4, ' han dle</w>': 2, '   bl in d ers</w>': 2, '   pr ac ti sing</w>': 2, ' pe ci ally</w>': 3, '   ho op s</w>': 8, ' d on na</w>': 2, '   lo ch in v ar</w>': 1, ' whi ff</w>': 2, ' vi tal</w>': 1, ' ra s ca lly</w>': 1, '   con te mp ti ble</w>': 4, '   den i al s</w>': 4, '   da ma ging</w>': 4, '   im pre ssi ons</w>': 6, '   th t</w>': 2, '   pre sen ted</w>': 15, '   u r ging</w>': 2, '   f al si fi ed</w>': 1, '   in de e dy</w>': 3, '   lon g fe llow</w>': 8, '   pi x i la ted</w>': 8, '   man d ra ke</w>': 15, '   ta g g art</w>': 1, '   mi d wife</w>': 2, '   do d s wor th</w>': 1, '   su b se qu en tly</w>': 4, '   f ra me up</w>': 1, ' hu mor e s qu e</w>': 3, ' s wan e e</w>': 1, '   tu b a</w>': 8, '   vi tu s</w>': 1, '   th or ea u</w>': 2, '   no b le men</w>': 1, '   pu z z les</w>': 7, '   in au gu ra ted</w>': 1, '   a qu ar i u m</w>': 7, '   bro ok fi e ld</w>': 2, '   har t for d</w>': 3, ' a le cks</w>': 1, '   ma be l</w>': 8, '   g y p si es</w>': 6, '   stu pi de st</w>': 5, '   im be ci li c</w>': 1, '   un fa i ling</w>': 1, '   hea d wai t ers</w>': 1, ' c in der e ll a</w>': 7, '   w oo s</w>': 1, '   s no op y</w>': 21, '   ba g ful</w>': 3, '   in gen u e</w>': 5, '   se mp le</w>': 11, '   bu din g ton</w>': 8, '   ce d ar</w>': 24, '   fo is</w>': 1, '   gra s</w>': 3, '   ha ll or</w>': 2, '   in si st ing</w>': 5, '   war p ed</w>': 14, '   f al kn er</w>': 2, '   mu gs</w>': 4, '   pe sts</w>': 1, '   do or m at</w>': 3, '   wee k ly</w>': 8, '   sti p end</w>': 3, '   ne w sp a per man</w>': 3, '   ver mon t</w>': 12, '   d on a tes</w>': 1, '   tra v el ed</w>': 19, '   mo o ch ers</w>': 1, '   en ta i ls</w>': 2, '   bu ff er</w>': 2, ' ne w sp a per man</w>': 1, '   ta llow</w>': 5, '   ma l</w>': 7, '   kn ow ed</w>': 9, '   mon i ck er</w>': 1, '   sle i gh</w>': 1, '   s li ck est</w>': 1, '   so cking</w>': 1, '   sta tely</w>': 3, ' lu ci a</w>': 1, ' ar th u r</w>': 1, '   su i t ca ses</w>': 9, ' ack ar o o</w>': 1, ' 1 8 0</w>': 3, '   ni ch t</w>': 4, '   wa h r</w>': 1, '   fo s di ck</w>': 1, '   pl ows</w>': 1, '   mo o ch er</w>': 5, '   pa il</w>': 4, '   te m per a men tal</w>': 2, '   po ck e t ful</w>': 5, '   da sh es</w>': 3, '   po e ts</w>': 15, '   p om p on i</w>': 5, '   for ci b ly</w>': 4, ' fi ll er</w>': 1, '   sp ac es</w>': 5, '   hea d pi e ce</w>': 1, '   n it</w>': 2, '   om ar</w>': 6, '   s ou sed</w>': 2, '   b in ge</w>': 5, '   pi s a</w>': 3, '   p y ra mi ds</w>': 7, '   p y ra mi d d es</w>': 1, '   sp h in x</w>': 6, '   de f la tion</w>': 1, '   s mu g ne ss</w>': 1, '   un to</w>': 12, '   po st car ds</w>': 1, '   fu ss y</w>': 4, '   s wi p ing</w>': 3, '   s ven son</w>': 1, ' wor ked</w>': 3, '   ac qu ir er</w>': 1, '   de pre s so id</w>': 1, ' 3 3 </w>': 2, ' 9 9 </w>': 14, '   mm mm ff f st tt u b ll</w>': 1, '   ab bi tt m m</w>': 1, '   g an ging</w>': 1, '   so fi e</w>': 7, '   mu m for d</w>': 13, '   fa ti gu e</w>': 2, '   e p st e in</w>': 1, ' bar r</w>': 1, '   sy mp to m</w>': 8, '   de l ban c o</w>': 3, '   di sp osed</w>': 6, '   cer ti fi ca tion</w>': 5, '   ex a ms</w>': 4, '   beli ev ers</w>': 2, '   er n</w>': 1, '   sh ee l er</w>': 2, '   ca tal y z ing</w>': 1, '   ro i ling</w>': 1, '   under t ow</w>': 1, '   ther a pi es</w>': 1, '   char ac ter i ze</w>': 3, '   f ow l er</w>': 2, '   a m tra k</w>': 2, '   ro th ber g</w>': 1, '   b en ton</w>': 4, '   man d le ba u m</w>': 1, '   ha un ting</w>': 13, '   i ma ger y</w>': 5, '   g y ne co lo g y</w>': 1, '   vi x en</w>': 1, '   fo ll e t t</w>': 4, '   gi l ro y</w>': 2, '   1 2 9 4 </w>': 1, ' 6 7 </w>': 1, '   re vi e w ed</w>': 4, '   may ber ry</w>': 1, '   al the a</w>': 3, ' be ha vi or</w>': 1, ' 8 5 </w>': 3, '   z en s</w>': 1, '   bro ck e t t</w>': 2, '   cor re c tion al</w>': 7, '   ad j our ned</w>': 4, '   e m pa th i z ing</w>': 1, '   dis re spe c ting</w>': 1, '   f ru stra ted</w>': 16, '   un su spe c ting</w>': 2, '   ni co i se</w>': 3, '   que sa di ll as</w>': 1, ' or der</w>': 7, '   en t re es</w>': 1, '   un sy m pa the ti c</w>': 2, '   sa i to</w>': 1, '   wa st e ful</w>': 2, '   su per mo de l</w>': 2, ' ha d da</w>': 1, '   bro ck e tt s</w>': 1, '   a in ge</w>': 5, '   ev en in</w>': 6, '   p an da</w>': 5, '   s ki pp er ton</w>': 2, '   in qui si tive</w>': 3, ' no sy</w>': 1, '   hu b ble</w>': 1, '   hi mb all</w>': 1, '   re tain</w>': 16, '   hea d sh r in k er</w>': 3, '   mi ti ga ting</w>': 1, '   i ra te</w>': 2, ' vi ll e</w>': 3, '   tr ans f er ence</w>': 4, '   tur ni p</w>': 3, '   att ri bu te</w>': 2, '   mi sin ter pre t</w>': 3, '   some th</w>': 3, '   in tu i tive</w>': 2, ' pa ti ent</w>': 4, '   vi ll a ins</w>': 5, '   bra dy</w>': 35, '   ro to</w>': 1, ' ro o ter</w>': 1, '   pu r po se ly</w>': 3, '   sa di sti c</w>': 9, '   mi lli on th</w>': 3, '   con f li c ting</w>': 2, ' a king</w>': 1, ' u m for d</w>': 1, '   l y mp h</w>': 1, '   fu l ne ss</w>': 1, ' mar ry</w>': 4, '   ex ha u st ing</w>': 5, '   sta min a</w>': 3, '   app en da ge</w>': 1, '   a a</w>': 8, '   lo on y</w>': 8, ' going</w>': 19, '   ab so</w>': 3, '   lu tely</w>': 1, '   de lu ded</w>': 3, '   hu mm er</w>': 4, '   app o in t men ts</w>': 7, '   ad di c tions</w>': 2, '   ne ss a</w>': 2, '   b on o</w>': 7, ' na om i</w>': 1, '   chan dr a</w>': 1, '   im pro per</w>': 7, '   un e th i cal</w>': 7, '   st ok ed</w>': 4, ' s ki p</w>': 1, ' mu m for d</w>': 1, '   k in k o</w>': 1, '   s ani t ar i u m</w>': 1, '   un con ne c ted</w>': 3, '   ce ter a</w>': 8, '   st ab i li z ing</w>': 2, '   re pu l sed</w>': 3, ' fi ll</w>': 1, '   ja ys</w>': 4, '   n b a</w>': 17, '   bi gs</w>': 2, '   p an da s</w>': 1, ' ro o ted</w>': 1, ' g li ding</w>': 1, '   gre en s bur g</w>': 1, '   qu ac ks</w>': 3, '   cor n ell</w>': 4, '   har r ow ing</w>': 1, '   im ho te p</w>': 4, '   ha m un a p tr a</w>': 7, '   un lea sh ed</w>': 3, '   an ck</w>': 3, ' na mu n</w>': 3, '   gir ly</w>': 3, '   b en i</w>': 4, '   o si r is</w>': 1, '   st in k we ed</w>': 1, '   s l ink</w>': 6, ' con n ell</w>': 10, '   ca ir o</w>': 15, '   ex t or tion</w>': 15, '   sy na go gu es</w>': 4, '   sy na go gu e</w>': 5, '   te m pl es</w>': 2, '   mo s qu es</w>': 1, '   ban do li er</w>': 1, '   ph ar a oh</w>': 5, ' su </w>': 2, '   ne c ro po l is</w>': 2, '   ju sti fi es</w>': 3, '   car na v on</w>': 1, '   car t ou c he</w>': 1, '   se t i</w>': 5, '   de ci p her</w>': 8, '   hi er o g l y ph s</w>': 1, '   hi er a ti c</w>': 1, '   ra me ss es</w>': 1, '   bu mb ling</w>': 2, '   an u b is</w>': 3, '   in can ta tions</w>': 1, '   an ci en ts</w>': 3, '   re gen er a ted</w>': 2, '   spi tt o on</w>': 1, '   in fe c ting</w>': 3, ' talking</w>': 4, '   ho m</w>': 2, ' da i</w>': 2, '   cu r ses</w>': 3, '   b la sp he m ers</w>': 1, '   ad ven tur er</w>': 2, '   g un fi gh ter</w>': 2, '   pre s su ri z ed</w>': 1, ' tra p</w>': 2, '   mi sa d ven ture</w>': 3, '   di g g ers</w>': 1, '   qu ar ried</w>': 1, '   co b al t</w>': 4, '   s ar co p ha gu s</w>': 1, '   sch oo l bo ys</w>': 2, '   af ter life</w>': 9, '   be mb ri dge</w>': 1, '   sch o l ars</w>': 5, '   ho k u m</w>': 1, '   a h m ar</w>': 1, '   o ssi ri on</w>': 1, ' pa s sa ge way</w>': 1, '   tu ar e gs</w>': 1, '   be d ou in</w>': 3, '   f ar thing</w>': 2, '   gla a a a</w>': 1, '   per ta ins</w>': 4, '   a h men op h us</w>': 1, '   ho o ta sh</w>': 2, '   i m</w>': 5, '   ev y</w>': 4, '   de com po sing</w>': 2, '   mu m mi fi ca tion</w>': 1, '   y an ks</w>': 8, '   ca s ba h</w>': 10, '   the b es</w>': 1, ' 1 3 4 </w>': 1, '   d un es</w>': 4, '   ph ar a o h s</w>': 2, '   tr in k et</w>': 3, ' ba st ard</w>': 2, '   po in ty</w>': 5, '   h or us</w>': 1, '   gra s sho pp er</w>': 5, '   fri s k y</w>': 3, '   gi z z ar ds</w>': 1, '   ow ch</w>': 1, '   we b s</w>': 1, '   f lea ba gs</w>': 1, '   lan ce</w>': 23, '   h en ch men</w>': 1, '   ca s an o v a</w>': 14, '   s k y li ght</w>': 6, '   po o tch ki e</w>': 2, '   pla y thing</w>': 1, '   tw ea k</w>': 2, '   f an ta si z ing</w>': 1, '   wom ani z ing</w>': 1, '   i me l da</w>': 4, '   ni co le</w>': 29, '   s li pp er y</w>': 15, '   con ta in ers</w>': 4, '   we ir does</w>': 2, '   hea d in</w>': 8, '   in sc ru ta ble</w>': 1, '   h or st</w>': 2, '   bu ck ho l t z</w>': 1, '   su per h er o es</w>': 10, '   s qui sh ed</w>': 2, '   he ll er</w>': 24, '   su per vi ll ain</w>': 2, '   what cha ma thing</w>': 1, '   un se l fi sh</w>': 2, '   see ds</w>': 7, ' what cha ma bo b</w>': 1, '   d ev i an ts</w>': 1, '   p an ch o</w>': 4, '   a di o s</w>': 8, '   mu cha cho s</w>': 1, '   co i ls</w>': 3, '   mar sh me ll ows</w>': 1, '   pe ev ed</w>': 1, '   spi der man</w>': 4, '   pe z</w>': 1, '   di sp en s er</w>': 2, '   un su n g</w>': 2, '   c ru m pl ed</w>': 1, '   ar ch en e mi es</w>': 1, '   spe w ing</w>': 1, '   si mi l ar i ty</w>': 4, '   o bi e</w>': 1, ' wa n</w>': 10, '   er a di ca t ors</w>': 1, '   ob li ter at ors</w>': 1, '   ki wan is</w>': 1, '   su per p ower</w>': 2, '   en d ow ment</w>': 4, '   sho v el er</w>': 1, '   ra j a</w>': 1, '   ra mp ant</w>': 2, '   ba z o ok as</w>': 1, '   su per w re st l er</w>': 1, '   mo hi can</w>': 3, '   c run ch</w>': 18, '   ba t man</w>': 41, '   li cks</w>': 7, '   hu mp s</w>': 2, '   a ll u ding</w>': 1, '   re miss</w>': 2, '   re fu ta tion</w>': 1, '   t rea ty</w>': 35, '   ti l sit</w>': 3, '   dan g ers</w>': 4, '   en ta il</w>': 2, '   ca u la in cour t</w>': 2, '   au st er li t z</w>': 2, '   fri e d land</w>': 2, '   un pro fi ta ble</w>': 1, '   s mu g g ling</w>': 15, '   gu er ri ll as</w>': 7, '   b lo ck a de</w>': 7, '   bo y co t t</w>': 7, '   di p lo m ac y</w>': 7, ' comp li an ce</w>': 1, '   k u tu so v </w>': 5, '   with d ra w s</w>': 3, '   pro long</w>': 3, '   ob ta in ing</w>': 3, '   e mp t in ess</w>': 8, '   con fe der ation</w>': 1, '   s l en der</w>': 3, '   p ru ssi a</w>': 4, '   out lin es</w>': 1, '   pre li min ar i es</w>': 1, ' supp li ed</w>': 2, '   dar e say</w>': 1, '   re fi ts</w>': 1, '   ro st op ch in</w>': 2, '   ar ou sed</w>': 5, '   di ss en ting</w>': 1, '   j en a</w>': 2, '   mu r at</w>': 2, '   ber th i er</w>': 4, ' ber th i er</w>': 1, '   t ac ti cal</w>': 17, '   re t rea ting</w>': 3, '   pu r sed</w>': 1, '   tru dge</w>': 1, '   st ri c te st</w>': 1, '   ra f ts</w>': 3, '   p ru den tly</w>': 1, '   da y brea k</w>': 2, '   sa bl ons</w>': 1, '   d ra go ons</w>': 4, '   v a gu est</w>': 3, '   bar ra s</w>': 3, '   be lli ard</w>': 2, '   mar sha ls</w>': 4, '   mor ti er</w>': 5, '   mar mon t</w>': 4, '   f on ta in e b lea u</w>': 2, '   re en ter</w>': 1, '   ra mb ou i ll et</w>': 1, '   un happ i ly</w>': 2, '   sur r en der ed</w>': 3, '   d ow n f all</w>': 3, '   a ll u r ing</w>': 2, '   sp en d th ri ft</w>': 1, '   l ar re y</w>': 1, '   cor vi s art</w>': 3, '   t s ar</w>': 1, '   ab ou ki r</w>': 1, '   b on a par te</w>': 23, '   re qu ir ing</w>': 6, '   j ea l ou si es</w>': 3, '   cl er c</w>': 2, '   as sig ning</w>': 2, '   mi l an</w>': 4, '   ac comp ani ed</w>': 5, '   a i d es</w>': 4, '   ju not</w>': 6, ' h h</w>': 9, '   a j ac ci o</w>': 2, '   ba sti a</w>': 1, '   cor si can</w>': 2, '   m out on</w>': 1, '   co ss ack</w>': 2, '   di di er</w>': 1, '   pi car t</w>': 1, '   com bu sti b les</w>': 2, '   be for e h and</w>': 8, ' en g in es</w>': 2, '   in cen di ar i es</w>': 2, '   ar ch du ch ess</w>': 2, ' l ou is</w>': 11, '   du ro c</w>': 4, '   du ch ess</w>': 20, ' l ou i se</w>': 4, '   re sc ind</w>': 5, '   beau har na is</w>': 8, '   kee p sa ke</w>': 2, ' sle ev ed</w>': 1, '   con ju re</w>': 4, '   bri lli an ci es</w>': 1, '   pre de st in ed</w>': 1, '   in ha bi ted</w>': 2, '   li se tt e</w>': 3, '   b r</w>': 5, ' r r r</w>': 1, '   per r in</w>': 1, '   l y on</w>': 1, '   chi lled</w>': 1, '   coun ci ls</w>': 2, '   mi st re ss</w>': 14, '   im pe tu ou s ly</w>': 1, '   mo li ere</w>': 3, '   pro tr act</w>': 1, '   di sh on or ed</w>': 2, '   hi pp o l y te</w>': 3, '   un da ted</w>': 1, '   mon te s qui ou </w>': 2, '   p lo m bi ers</w>': 1, '   f lo ch</w>': 1, '   ro qui er</w>': 1, '   tri ll au d</w>': 1, '   a du l ter y</w>': 3, '   mi st re ss es</w>': 2, '   e u gen e</w>': 30, '   he ar t bro k en</w>': 6, '   h or ten se</w>': 2, '   bri e f ly</w>': 16, '   di j on</w>': 2, '   ad jo in ing</w>': 2, '   ser vi c ea ble</w>': 2, '   fi don</w>': 1, '   cou ri er</w>': 12, '   bor de llo</w>': 2, '   bi j ou </w>': 21, '   qu ad ri ll e</w>': 1, '   bo t ani cal</w>': 3, '   har p</w>': 10, '   re se mb le</w>': 4, '   por tra i ts</w>': 5, '   sch on br un n</w>': 1, '   im pre ssi on able</w>': 7, '   e mb ar ra ss men ts</w>': 1, '   ta ll e y r and</w>': 1, '   s mar ting</w>': 1, '   me di a tor</w>': 2, '   cla u ses</w>': 5, '   por ts</w>': 4, '   4 6 </w>': 4, '   dis me mb er ment</w>': 1, '   v ar l ac </w>': 6, '   mi s l ed</w>': 3, '   in f la med</w>': 5, '   cha l on</w>': 1, '   f re li co t</w>': 1, '   b our ri en n e</w>': 1, '   f ou c he</w>': 1, '   la v a ll e tt e</w>': 1, '   1 8 1 4 </w>': 1, '   ti l ted</w>': 3, '   ir re tri ev ably</w>': 1, '   bi v ou ac s</w>': 1, '   bor der ing</w>': 1, '   vi stu l a</w>': 1, '   lan c er</w>': 4, '   au er sta d t</w>': 1, '   war like</w>': 2, '   l ou i s a</w>': 10, '   f re der i ch</w>': 1, '   wi l hel m</w>': 20, '   p ru ssi ans</w>': 1, '   p ru ssi an</w>': 3, '   ti ro t</w>': 1, ' cha ir s</w>': 1, '   1 8 0 4 </w>': 1, '   ar ou se</w>': 4, '   con qu er or</w>': 2, '   en l ar ging</w>': 1, '   po pu l ar i ty</w>': 7, '   con si sted</w>': 2, '   co lu m bi er</w>': 1, '   f li r ta tion</w>': 5, '   v ar i ed</w>': 3, '   cor si c a</w>': 4, '   1 7 6 9 </w>': 1, '   le ti z i a</w>': 1, '   la vi sh ed</w>': 2, '   el ro y</w>': 10, '   su g a</w>': 7, '   un c</w>': 3, '   su ck as</w>': 1, '   su bur b s</w>': 7, '   2 3 0</w>': 3, '   t y son</w>': 5, '   de b o</w>': 4, '   whi pp in</w>': 6, '   ch ee c o</w>': 4, '   c r on</w>': 1, ' br ow ni es</w>': 1, ' 2 7 </w>': 6, '   g lo ck</w>': 3, '   fu u u u u u ck</w>': 1, ' wan a</w>': 2, ' 2 4 7 </w>': 1, '   ran ch o</w>': 9, '   cu ca mon g a</w>': 7, '   wa tt s</w>': 9, '   pu dge</w>': 3, ' sp ra y ed</w>': 1, '   jo k er</w>': 21, '   ju ven i le</w>': 13, '   si r en s</w>': 7, '   cu ss in</w>': 2, '   cu ssed</w>': 1, '   fa ye</w>': 4, '   c r ac kin</w>': 4, '   k ar l a</w>': 13, '   c r ack er</w>': 13, '   en t ou ra ge</w>': 6, '   1 8 7 </w>': 1, '   cho ca m un g a</w>': 1, '   b om b sh ell</w>': 4, '   j ack s ons</w>': 1, '   l ou de st</w>': 1, '   bu ll do gs</w>': 1, '   fr i</w>': 1, '   ha t ers</w>': 1, '   k y m</w>': 2, ' ro ac h</w>': 1, '   y oo o o</w>': 1, '   bea m er</w>': 1, '   in ten se ly</w>': 7, '   v a mp ed</w>': 1, ' ga mes</w>': 2, '   mi y a te a</w>': 1, '   e z al</w>': 2, '   so s a</w>': 1, '   ti c s</w>': 1, '   cho com un g a</w>': 1, '   au ction</w>': 19, '   k ri st a</w>': 1, '   pa ten ted</w>': 3, '   tri bu tes</w>': 1, '   s ac ra men to</w>': 10, '   po lls</w>': 8, '   in te st in al</w>': 2, '   ha vo c</w>': 5, '   pri me time</w>': 2, '   ho ps</w>': 5, '   a m tr ack</w>': 2, '   b on a ven ture</w>': 3, '   l y n n</w>': 20, '   w at</w>': 2, '   pi sto l er o</w>': 1, ' a h h</w>': 2, ' ah</w>': 29, ' wa t son</w>': 1, '   d on or</w>': 9, ' in vi ta tion</w>': 1, '   i t in er ary</w>': 3, '   br en d an</w>': 1, '   sa mar a</w>': 1, ' ki ss es</w>': 1, '   no ds</w>': 13, '   si gh</w>': 4, '   ir en e</w>': 14, '   wor k a ho li c s</w>': 1, '   ti pp er</w>': 2, '   den i gra te</w>': 2, '   s w ea ted</w>': 5, '   win d sh ei ld</w>': 1, '   win d shi el ds</w>': 1, '   k u wait</w>': 7, ' bo ar d ers</w>': 1, '   ro ll er b la d es</w>': 1, '   be t wi xt</w>': 2, '   pro fe ss</w>': 2, '   sti ck k ni fe</w>': 1, ' bl an ke ts</w>': 1, '   b loo di ed</w>': 3, '   ta ber n ac le</w>': 2, '   ro a m in</w>': 2, '   w oo d lan ds</w>': 1, ' hea ps</w>': 1, '   youn g ins</w>': 2, '   qu o t in</w>': 2, ' com in</w>': 3, ' p a</w>': 1, ' ha ir</w>': 4, '   lu mp in</w>': 1, '   ha in</w>': 3, '   n ary</w>': 2, '   g ar</w>': 1, '   ma w</w>': 2, '   s ki ff</w>': 10, '   cre sa p</w>': 2, '   st e p to e</w>': 1, '   y ore</w>': 5, '   wh ar f</w>': 2, '   p sha w</w>': 1, '   min d in</w>': 1, '   youn g in</w>': 2, '   ri i v </w>': 1, ' h oun d in</w>': 1, ' fu mes</w>': 1, '   a a a</w>': 3, ' men n</w>': 1, '   bo il in</w>': 1, '   mi z</w>': 22, '   r out</w>': 2, '   or n er i er</w>': 1, '   b lu e be ard</w>': 2, '   tra m pl ed</w>': 4, '   k ni f ed</w>': 3, '   figu r in</w>': 3, '   i c ey</w>': 8, '   th o t</w>': 1, '   tru b ble</w>': 1, '   wi ll a</w>': 10, '   s sh h h</w>': 5, '   f re tt in</w>': 1, '   mu li sh</w>': 1, '   pa ow</w>': 1, '   do tes</w>': 2, '   ac h in</w>': 1, '   fi d d le sti cks</w>': 4, '   youn g st ers</w>': 7, '   dan de li on</w>': 3, '   some wh er es</w>': 5, '   t ar ried</w>': 1, '   b re w in</w>': 1, '   stan k</w>': 4, '   he ll fi re</w>': 5, '   li e th</w>': 1, '   in c rea se th</w>': 1, '   tr ans gre s sor s</w>': 2, ' ni ck</w>': 4, '   y u m my</w>': 5, '   w end</w>': 1, '   spi der y</w>': 1, '   m oun d s vi ll e</w>': 1, '   h ing</w>': 1, '   s an d b ar</w>': 1, '   r ow ed</w>': 3, '   par k er s bur g</w>': 2, '   bu tch er in</w>': 1, '   ha ms</w>': 5, '   can n in</w>': 1, '   ta tt l ed</w>': 1, '   si st er s vi ll e</w>': 1, '   s wee ts</w>': 6, '   cor n sti cks</w>': 1, '   co b bl er</w>': 1, '   t st</w>': 1, ' t st</w>': 1, '   pu r ti est</w>': 1, '   cho cl it</w>': 1, '   so dy</w>': 2, '   pro mi s in</w>': 1, '   sc ra w l</w>': 3, '   no te pa per</w>': 1, '   ta int</w>': 2, '   any th in</w>': 17, '   thr ow ed</w>': 3, '   be get</w>': 1, '   bl en ding</w>': 2, '   pa w ing</w>': 3, '   im pu d ence</w>': 1, '   of f en</w>': 1, '   fi g ger in</w>': 1, '   den s</w>': 2, '   per di tion</w>': 1, '   so d om s</w>': 1, ' m ee t in</w>': 1, '   ca in</w>': 11, '   se w in</w>': 1, '   ra p id</w>': 7, '   ma pped</w>': 1, '   ho ok us</w>': 1, '   po k us</w>': 1, '   fi li p in o</w>': 1, '   pa th o lo g y</w>': 1, '   ee g</w>': 5, '   n an</w>': 4, '   i ll ne ss es</w>': 2, '   con cu ssi ons</w>': 1, ' co cking</w>': 1, '   gu in ess</w>': 1, '   z en da</w>': 2, ' o ing</w>': 1, '   ven tur a</w>': 1, ' bo o by</w>': 1, '   im pro vi sed</w>': 1, '   an ti per son ne l</w>': 1, ' rea m</w>': 1, '   b al in e se</w>': 2, '   bo o ge y</w>': 3, '   ob je c tive</w>': 10, '   t re ll is</w>': 1, '   whi s k ers</w>': 5, '   bra w ling</w>': 3, '   sc rea ms</w>': 7, '   bo tt om ed</w>': 2, '   de cla w ed</w>': 1, '   ca ked</w>': 1, ' cu ri ty</w>': 1, '   pe te sa kes</w>': 1, '   ra z ors</w>': 3, '   ob sc en e ly</w>': 1, '   bi g g y</w>': 1, '   n in o tch k a</w>': 46, '   ne g li ge e</w>': 3, ' re vo lu tion</w>': 1, '   hu d d l ed</w>': 1, '   clo the s l ine</w>': 2, '   under min es</w>': 2, '   gu r g an o v </w>': 1, '   wa sh room</w>': 5, '   bo il ed</w>': 4, '   k re m l in</w>': 10, '   be e th o ven</w>': 22, '   son at a</w>': 3, '   min ar et</w>': 1, '   whi sp er</w>': 24, ' d es</w>': 2, '   ra i sin s</w>': 2, '   f ra m bo i ses</w>': 1, '   con fi tur es</w>': 1, '   pr un es</w>': 4, '   ca f</w>': 6, '   f la kes</w>': 5, '   sh oo k</w>': 13, '   t act</w>': 3, '   com mi ss ar</w>': 5, '   ra z in in</w>': 10, '   ei f fe l</w>': 10, '   n ev s k y</w>': 1, '   t ro i k a</w>': 1, '   c z ar</w>': 13, '   bu l j an off</w>': 17, '   la ven ha m</w>': 1, '   u p hold</w>': 9, '   li fe long</w>': 3, '   le on i tch k a</w>': 4, '   mi sh a</w>': 2, '   ca pi ta li sti c</w>': 3, '   ter min us</w>': 2, '   i ran off</w>': 12, '   st er n</w>': 15, '   win try</w>': 2, '   y a k u sho v a</w>': 11, '   of f set</w>': 2, ' al g out</w>': 10, '   ga st on</w>': 21, '   e mp ha ti ca lly</w>': 1, '   rea c tion ary</w>': 3, '   un fa ir ne ss</w>': 1, '   f oo ting</w>': 3, '   so ci a li sti c</w>': 1, '   pro ce ss es</w>': 1, '   coun ter p an e</w>': 2, '   con tri bu ted</w>': 1, '   de ci de d ly</w>': 4, '   sa vi t z k y</w>': 3, '   s wan a</w>': 28, '   con stan t in ople</w>': 10, '   pi ro sh k i</w>': 1, '   de ser tion</w>': 3, '   bor sch t</w>': 3, '   st ro g an off</w>': 2, '   bl in is</w>': 1, ' al i</w>': 1, '   a la d d in</w>': 1, '   pe ti tes</w>': 1, '   f ra i ses</w>': 1, '   c r</w>': 2, '   b re ta g n e</w>': 1, '   ha lt</w>': 8, '   en vo y</w>': 7, '   h er e with</w>': 2, '   mer ci er</w>': 5, ' bu l j an off</w>': 2, '   bo l sh ev i ks</w>': 4, '   l en in</w>': 9, ' su cking</w>': 3, '   bl ack ma il er</w>': 5, '   fa shi ons</w>': 1, '   li f ts</w>': 10, ' d ine</w>': 1, '   bu l j an of</w>': 1, '   k op al s k i</w>': 11, '   i v an o v na</w>': 1, '   si mon o vi tch</w>': 2, '   a p fe l</w>': 2, '   co ok er</w>': 1, '   sc oun dre ls</w>': 10, '   tr ac t ors</w>': 1, '   dis fa v or</w>': 1, '   gre e ted</w>': 2, '   in ci ting</w>': 1, '   att en dan ts</w>': 6, '   su c c ee ds</w>': 2, '   su spi ci ons</w>': 18, '   c z ar i sti c</w>': 1, '   wa ver</w>': 2, '   bu l j an of f s</w>': 1, '   i ran of f s</w>': 1, '   k op al s k is</w>': 1, '   de po pu late</w>': 1, ' n in o tch k a</w>': 1, '   ma ss es</w>': 12, ' dar ling</w>': 4, '   ser f s</w>': 1, '   in sta ll ment</w>': 4, '   cl en ch</w>': 3, '   n in o tch u a</w>': 1, '   du che ss es</w>': 2, '   go a ts</w>': 5, '   o o</w>': 3, '   s co tch men</w>': 3, '   m c in to sh</w>': 8, '   m c gi lli cu ddy</w>': 8, ' wai ter</w>': 2, ' wh e w</w>': 1, '   fr en ch men</w>': 3, '   p om p ous</w>': 5, '   ci r cu late</w>': 1, '   wor k in g men</w>': 1, '   h y ah</w>': 2, '   wor k men</w>': 4, '   re li sh</w>': 3, '   gu s to</w>': 2, '   si ck le</w>': 3, '   pre d om in a ting</w>': 1, ' for ty</w>': 19, '   st ri k ers</w>': 1, '   un ro man ti c</w>': 1, '   sta ti sti cal</w>': 3, '   re st ful</w>': 3, '   do ves</w>': 1, '   co o</w>': 2, '   s na i ls</w>': 4, '   co l de st</w>': 3, '   inter min ably</w>': 1, '   mo th s</w>': 3, '   pe tal s</w>': 12, '   war m th</w>': 7, '   hea v in ess</w>': 2, '   thir st</w>': 17, '   t an ta li z ing</w>': 1, '   ex al ting</w>': 2, '   sen ti men ta li ty</w>': 4, '   ex hi l ar ation</w>': 2, '   co g</w>': 4, '   co gi t s k a</w>': 1, '   b our ge o i si e</w>': 1, '   im prob able</w>': 5, '   de sig na tion</w>': 4, '   cor ne a</w>': 2, ' no tch k a</w>': 1, '   in cor re c tly</w>': 2, '   no tch k a</w>': 1, '   con fu ses</w>': 5, '   wom an kind</w>': 1, '   b lea k</w>': 6, '   app ea sed</w>': 1, '   w oo ed</w>': 2, '   ba th sh e b a</w>': 2, '   ca l ori es</w>': 1, ' bu t l er</w>': 1, '   ki tch en e tt e</w>': 1, '   g ran ds</w>': 2, '   b ou l ev ar ds</w>': 1, '   tri om p he</w>': 1, '   mon t mar t re</w>': 2, '   mon t par na s se</w>': 1, '   bo h</w>': 1, '   sp ar k les</w>': 3, '   g li tt ers</w>': 1, '   fri vo li ty</w>': 2, '   pi ers</w>': 5, '   ma son ry</w>': 1, '   inter l ac ed</w>': 1, '   supp re ss</w>': 4, '   f li r t</w>': 15, '   par i si an</w>': 2, ' t ea u</w>': 1, '   t ou ra ine</w>': 1, ' dre ss er</w>': 2, '   re gen er ation</w>': 1, '   rea d ju st ment</w>': 2, '   sta m mer ing</w>': 1, '   fran k ne ss</w>': 1, '   vo l g a</w>': 7, '   bo at man</w>': 10, '   b ou l ev ar di er</w>': 1, '   b en son</w>': 5, '   ta il ors</w>': 1, '   sc hi a par e ll i</w>': 1, '   re spe c ting</w>': 5, '   di r ti ed</w>': 1, '   mu z hi k</w>': 1, '   pro le t ar i an</w>': 1, '   f lo g ged</w>': 1, ' bo il ed</w>': 2, '   h en s</w>': 3, '   un fee ling</w>': 2, '   nu g get</w>': 1, '   li v el ong</w>': 2, '   te p id</w>': 2, '   cor ni ll on</w>': 3, '   ra k on in</w>': 2, '   b loo d h oun ds</w>': 2, '   gu i z o t</w>': 2, '   sa u sa g es</w>': 4, '   mi d st</w>': 8, '   ro man of f s</w>': 2, '   ta b lo id</w>': 4, '   pu b li sh ing</w>': 19, '   par i si en n e</w>': 1, '   gu i z o ts</w>': 2, '   me llow</w>': 9, '   bri ttle</w>': 3, '   com po se</w>': 3, '   di m</w>': 27, '   ex pe di ti ous</w>': 1, '   im par ti al</w>': 2, '   con den se</w>': 1, '   e mo tion a li s m</w>': 1, '   no ti f ying</w>': 2, '   wa k en</w>': 3, '   ear li est</w>': 6, '   re ga li a</w>': 1, '   di a de m</w>': 2, '   b ow ing</w>': 2, '   pe t ro gra d</w>': 2, '   re d ouble</w>': 1, '   co ss ac ks</w>': 3, '   un par d on able</w>': 1, '   kn ou ts</w>': 1, '   fin e s se</w>': 3, '   o ver st ep</w>': 1, '   en tr ust</w>': 1, '   mo sle ms</w>': 1, '   pro v in ces</w>': 2, '   ma th i e u</w>': 9, '   po i v re l</w>': 1, '   pr uni ers</w>': 1, '   je w el er</w>': 3, '   a le x is</w>': 1, '   so i ls</w>': 1, '   imp each</w>': 9, '   ca m bo di a</w>': 7, '   ob st ru ction</w>': 5, ' s an g</w>': 1, '   fro id</w>': 2, '   ha l de man</w>': 26, '   e h r li ch man</w>': 7, '   whi tt a k er</w>': 1, '   in for m er</w>': 4, '   go l d wa ter</w>': 5, ' o e u v re</w>': 5, '   so f th ea ds</w>': 1, '   in di ct</w>': 4, ' 3 5 0</w>': 3, '   e ll s ber g</w>': 14, '   par d on ed</w>': 2, '   w oo d w ind</w>': 1, '   f er n st e in</w>': 1, '   si ri c a</w>': 2, '   wa ter ga te</w>': 23, ' ci a</w>': 4, '   m c cor d</w>': 18, ' e le ct</w>': 3, '   lan d s ca pe</w>': 7, '   te mer i ty</w>': 1, '   under han ded</w>': 1, '   d om in o es</w>': 2, '   shi tting</w>': 14, ' ins</w>': 4, '   fi re b om b ing</w>': 1, '   bro ok in gs</w>': 2, '   m c go ver n</w>': 10, '   l b j </w>': 1, ' 6 9 </w>': 8, '   lea king</w>': 10, '   be ll ev u e</w>': 9, ' gla ss</w>': 3, '   st on e wa ll</w>': 3, '   ear v in</w>': 1, '   f d r</w>': 3, '   i ke</w>': 33, '   par d on ing</w>': 1, '   e h r en ber g</w>': 1, '   k en ne d ys</w>': 3, '   c happ a qui d di ck</w>': 2, '   har p ing</w>': 2, '   dr ow n s</w>': 5, ' un st able</w>': 2, ' ni x on</w>': 2, '   mi s qu o ted</w>': 2, '   sp ir o</w>': 5, '   a g new</w>': 3, '   han o i</w>': 5, '   la o s</w>': 3, '   f ra ter ni ty</w>': 2, '   an ar ch y</w>': 11, '   un mar ked</w>': 5, '   li ddy</w>': 15, '   hi ss</w>': 11, '   p ho to gen i c</w>': 1, '   imp li ca ted</w>': 3, '   ki s sin ger</w>': 16, ' cre ep</w>': 1, '   ma g ru der</w>': 13, '   qu a k er</w>': 1, '   co l son</w>': 15, '   app ea se</w>': 5, '   me tal li c</w>': 4, ' b ay</w>': 4, ' s mo king</w>': 7, '   under lin ed</w>': 3, ' vi c t ory</w>': 1, '   co x</w>': 13, '   ga s p</w>': 3, ' lo ving</w>': 5, '   e le c tri f y</w>': 1, '   tri ci a</w>': 4, '   ga ss ing</w>': 2, '   p an th ers</w>': 22, '   li on i ze</w>': 2, '   rea li st</w>': 2, '   whi te wa sh</w>': 1, '   ir an</w>': 6, '   sha h</w>': 3, '   be l gi u m</w>': 4, '   a m ba s sa d or ship</w>': 2, '   whi mp er</w>': 1, '   fri vo l ous</w>': 6, '   bra d le e</w>': 2, '   e go s</w>': 6, '   su l z ber ger</w>': 1, '   tra d ers</w>': 2, '   je w boy</w>': 4, ' w ing</w>': 9, ' sch war t z es</w>': 1, '   ma i</w>': 9, ' ta is</w>': 1, '   ven e z u el a</w>': 9, '   o ver tur ned</w>': 4, '   c ri ses</w>': 5, '   mu d mu tt s</w>': 1, '   t v s</w>': 1, '   f l un king</w>': 2, '   whi t ti er</w>': 6, '   de fi c it</w>': 2, '   d ow</w>': 5, '   pu ss y f oo ting</w>': 2, '   de mo cra ts</w>': 16, ' bea st</w>': 1, '   stu r g is</w>': 1, '   ve to</w>': 5, '   chi le</w>': 8, '   gu a te ma l a</w>': 2, '   cap ab i li ty</w>': 8, '   t rea ding</w>': 2, '   5 9 0</w>': 1, '   z i e g l er</w>': 2, ' mor al</w>': 2, '   tu b es</w>': 13, '   er r ors</w>': 4, '   han k y</w>': 4, ' p an k y</w>': 3, '   tr an sc ri b ing</w>': 1, '   tr in i</w>': 6, '   cho t in er</w>': 2, '   ba p ti ze</w>': 2, ' na tion al</w>': 8, '   com int</w>': 1, '   cla ssi fi ca tion</w>': 4, '   hu st on</w>': 1, '   s ca b</w>': 5, '   un co ver</w>': 7, '   p us</w>': 5, '   b re m er</w>': 2, '   h or ri b les</w>': 1, '   pl u mb ers</w>': 5, '   wi re ta pp ing</w>': 1, ' pl u mb ers</w>': 2, '   cap er</w>': 6, '   fi re b om b</w>': 2, '   d n c</w>': 2, ' bri en</w>': 17, '   ev i den tly</w>': 11, '   cor n si l k</w>': 1, '   wor l d lin ess</w>': 1, ' tur ning</w>': 2, '   wi den ing</w>': 2, '   g y re</w>': 1, '   f al con er</w>': 2, '   loo sed</w>': 1, '   om in ous</w>': 4, '   s l ou ch es</w>': 2, '   be th le he m</w>': 1, '   j un c ture</w>': 5, '   y ea ts</w>': 1, '   cont in u al</w>': 2, '   re min d ers</w>': 2, '   mor ta li ty</w>': 5, '   com m uni s m</w>': 7, '   in n er most</w>': 1, '   for ma li ty</w>': 12, '   su b mar ine</w>': 8, '   ci en fu e go s</w>': 1, '   cu sh man</w>': 4, '   d c i</w>': 1, '   cha ir ing</w>': 2, '   bu ri es</w>': 4, '   imp le men ted</w>': 2, '   pa d do ck</w>': 2, '   jo a qu in</w>': 2, '   or e g on</w>': 14, '   ran ting</w>': 2, '   lea k er</w>': 1, '   1 9 1 7 </w>': 2, '   no min ation</w>': 5, '   po or est</w>': 3, '   bor man</w>': 1, '   in je c tions</w>': 6, '   ra di o s</w>': 11, '   2 1 4 </w>': 1, '   gre y h ound</w>': 7, '   me in</w>': 6, '   k a mp f</w>': 3, '   ra gh ea ds</w>': 1, '   de s i</w>': 1, ' 4 6 </w>': 4, ' de ten te</w>': 2, ' 6 4 </w>': 6, '   shi t po t</w>': 1, '   sen ti men ts</w>': 5, '   pa tri o ts</w>': 5, '   con te mp or ar i es</w>': 1, '   ma o</w>': 8, '   b re z h n ev </w>': 2, '   wi re ta ps</w>': 2, '   imp li ca tion</w>': 5, '   hou se cle an ing</w>': 1, '   ne g ro s</w>': 2, '   e le c tions</w>': 6, '   cre t in</w>': 3, '   e mp ha si ze</w>': 4, '   ge o poli ti c s</w>': 1, '   con ten tion</w>': 3, '   tri an gu l ar</w>': 1, ' w r in ging</w>': 1, ' rea l poli ti k</w>': 1, ' ma d man</w>': 1, ' in cu r sion</w>': 1, '   mi li t ar i ly</w>': 1, '   c ri ti ci s m</w>': 10, '   d ome sti ca lly</w>': 1, '   de ga u ll e</w>': 1, '   dis ra el i</w>': 4, '   i do l</w>': 3, '   al li an ces</w>': 2, '   go b s</w>': 5, '   li ber al s</w>': 4, ' st ab b ing</w>': 3, '   in te ll e c tu al s</w>': 4, '   w ea ther men</w>': 1, '   si eve</w>': 1, '   fun g</w>': 3, '   be i j ing</w>': 3, '   wri t in gs</w>': 5, '   man o l o</w>': 4, '   rea c tion ar i es</w>': 1, '   ev i ls</w>': 5, '   a ll en de</w>': 2, '   na tion a li z ing</w>': 1, '   busin e ss es</w>': 9, '   stu de ba k er</w>': 2, '   d om in a ted</w>': 1, '   tu ber cu lo s is</w>': 5, '   g an g bu st ers</w>': 3, '   mo o la h</w>': 1, '   s co tch es</w>': 2, '   re pu b li c ans</w>': 10, '   al ger</w>': 2, '   lea k ers</w>': 1, '   y or b a</w>': 1, ' vo l un te er</w>': 1, '   ab o li tion i st</w>': 1, '   qu a k ers</w>': 1, '   ab o li sh</w>': 1, '   ho bo s</w>': 1, '   t ack ling</w>': 3, '   or an ge men</w>': 1, '   ta il back</w>': 2, '   sy r ac use</w>': 1, '   lo mb ar do</w>': 27, '   sa t in</w>': 2, ' god dam n s</w>': 1, '   chri sts</w>': 1, ' ex p le tive</w>': 1, '   de le ted</w>': 4, '   de le te</w>': 9, '   bu ll r ing</w>': 1, ' me a</w>': 1, '   cu l p a</w>': 3, '   me a</w>': 3, '   hi ss ing</w>': 1, '   bo o ing</w>': 1, '   p ows</w>': 8, '   re vo king</w>': 1, '   po sts</w>': 9, '   ber n st e in</w>': 43, '   wi d th</w>': 2, '   un pre di c ta bi li ty</w>': 4, '   con fi de</w>': 6, '   in f la tion</w>': 6, ' 5 2 </w>': 3, '   fa g go ts</w>': 6, ' l y n don</w>': 2, ' b om b</w>': 7, '   pri mar i es</w>': 1, '   m ac mi ll an</w>': 4, '   ad en au er</w>': 1, '   ju bi la tion</w>': 2, '   pla y house</w>': 4, '   fi de l</w>': 1, ' ca lling</w>': 1, ' c ri s is</w>': 1, '   ex t re mi sts</w>': 3, '   m c car th y</w>': 2, ' che ck ers</w>': 1, ' c ri ses</w>': 1, '   re par te e</w>': 1, '   h ow ya</w>': 4, '   di g g in</w>': 6, '   ca li c he</w>': 1, '   hu d spe th</w>': 1, '   wan t in</w>': 10, '   o ver ma tch ed</w>': 1, '   ci r cl ed</w>': 3, '   g ran d dad</w>': 3, '   t our ni qu et</w>': 2, '   an go l a</w>': 5, ' pre da te</w>': 1, '   wa l</w>': 7, ' wi ld</w>': 3, '   de du c ed</w>': 1, '   dis ma l</w>': 4, '   ll e w el y n</w>': 15, '   wal s er</w>': 1, '   wan d ers</w>': 7, '   com ers</w>': 1, '   hi s self</w>': 9, '   w en d ell</w>': 20, '   be t sy</w>': 24, '   ac cu mu la ted</w>': 2, '   nor e en</w>': 1, '   a g gra v a ting</w>': 2, '   shu ck</w>': 2, '   s w o le</w>': 2, '   un lo ad ed</w>': 5, '   ac p</w>': 1, '   f la tt en s</w>': 1, '   lin e ar</w>': 2, ' 7 7 </w>': 1, '   wi en ers</w>': 2, '   de di ca t in</w>': 1, '   de di ca te</w>': 9, '   t or ber t</w>': 1, '   o de ss a</w>': 5, '   pre d ni z one</w>': 1, '   se tt in</w>': 8, '   for e seen</w>': 5, '   lu r ch</w>': 1, '   ter re ll</w>': 1, '   ever thing</w>': 2, '   fa ll in</w>': 4, '   ho ll er in</w>': 6, '   sa t che l</w>': 7, '   da y tra der</w>': 1, '   re si den ts</w>': 3, '   li gh t in</w>': 2, '   a ir st ri p</w>': 6, '   b le ep</w>': 4, '   sc re w gi e</w>': 1, '   ac coun ta ble</w>': 3, '   bi r sho t</w>': 1, '   ac o st a</w>': 2, '   re ce i ver</w>': 7, '   clo s in</w>': 3, '   fri en do</w>': 1, ' go l a</w>': 1, '   wa ll o p</w>': 7, '   v a li da ted</w>': 1, '   go at fuck</w>': 1, '   chi gu r h</w>': 5, '   tr an sc end</w>': 2, '   bra ze</w>': 2, '   ac e t y l en e</w>': 1, '   ti g</w>': 1, '   im mo bi le</w>': 1, '   ba tal li on</w>': 1, '   j ac kin</w>': 2, '   o ver co at</w>': 3, '   lon ni e</w>': 9, ' tra u ma ti c</w>': 1, '   dis so ci a tive</w>': 1, '   go dam m it</w>': 4, '   el d en</w>': 8, '   sh oo t out</w>': 3, '   o st re y</w>': 2, ' wi lli a ms</w>': 1, '   ow w</w>': 7, ' fi les</w>': 3, ' min stra l</w>': 1, '   ra m pa ge</w>': 2, ' ch lo e</w>': 1, ' lon ni e</w>': 1, '   si z e more</w>': 8, ' o l</w>': 6, '   te pe es</w>': 1, '   te pe e</w>': 4, '   pre sen ta tion</w>': 4, '   ma u l ed</w>': 4, '   la y ed</w>': 2, ' pl enty</w>': 1, '   lon el y he ar ts</w>': 1, ' ch u r ch</w>': 4, '   li ve li er</w>': 1, ' ban n ers</w>': 1, '   pri c ing</w>': 2, '   ban n ers</w>': 3, '   we sle y</w>': 22, '   gar ba ge man</w>': 2, '   fo x ho le</w>': 4, '   su n se ts</w>': 3, '   pa ssi on a tely</w>': 5, '   app re ci ably</w>': 1, '   st y li sh</w>': 8, '   di ver si f y</w>': 3, ' po se</w>': 10, '   ba ll ard</w>': 7, '   lo ma</w>': 4, '   ra v ell</w>': 13, '   mar o on</w>': 23, '   le sa b re</w>': 2, '   a a h h h</w>': 2, '   si tter</w>': 15, '   star li te</w>': 1, ' ge or ge</w>': 3, '   re pe ti tion</w>': 4, ' te ll a</w>': 1, '   pa t in a</w>': 1, '   go o se bu mp s</w>': 3, '   im pro v </w>': 1, '   a ll ow an ce</w>': 12, ' s ki es</w>': 1, '   ou ting</w>': 4, '   car le ton</w>': 1, '   re l en t less</w>': 5, '   ten p ins</w>': 1, '   9 7 </w>': 7, '   le sa br es</w>': 1, '   pu ri fi ers</w>': 2, '   bu i cks</w>': 2, '   s ca l p ed</w>': 7, '   mer le</w>': 45, '   ce d ars</w>': 4, '   sin a i</w>': 3, ' fi an c</w>': 1, '   si st in a</w>': 1, '   pi l g ri ms</w>': 5, '   g ro p ed</w>': 1, '   t uni si an</w>': 1, '   he p bur n</w>': 9, ' ro man</w>': 1, '   re si den cy</w>': 3, '   ver ba ti m</w>': 4, '   ro s a</w>': 19, ' lo ma</w>': 2, '   po o ls</w>': 8, '   di re c t ori es</w>': 1, '   di p lo m at</w>': 13, '   de ca pi ta ted</w>': 3, ' ra v ell</w>': 1, '   tu st in</w>': 2, ' u st in</w>': 1, '   k o i</w>': 1, ' a vi d</w>': 1, '   ma s sa ging</w>': 1, '   he ar t be at</w>': 23, '   h er n an de z</w>': 3, '   h er r er a</w>': 2, '   si l ver la ke</w>': 1, '   an der son vi ll e</w>': 1, '   in hu man e</w>': 3, '   je b</w>': 6, '   gh en g is</w>': 1, '   k un t</w>': 1, '   de mi li t ar i z ed</w>': 1, '   di x i e</w>': 21, '   du an e</w>': 13, '   ki ow a</w>': 2, '   ki ck a po o</w>': 1, '   o sa ge</w>': 2, '   in j un s</w>': 4, ' in j un s</w>': 2, '   p ou rs</w>': 10, '   st re tch ing</w>': 6, ' nu tty</w>': 1, '   al mon d</w>': 3, '   ro c a</w>': 1, '   tru ck er</w>': 7, '   pe es</w>': 1, '   ve ter in ar i an</w>': 4, '   sh ow girl</w>': 2, '   ta ss les</w>': 1, '   cra ss</w>': 3, '   cor n brea d</w>': 1, '   t wi li ght</w>': 7, '   pu r ga t ory</w>': 7, '   an te lo pe</w>': 3, '   po i se</w>': 3, '   co ar se</w>': 3, '   th as</w>': 3, '   mu d pa tch</w>': 1, '   he ar t lan d ers</w>': 1, ' fin n</w>': 1, '   mi d we st er n</w>': 2, '   sto i c</w>': 1, ' a m ong</w>': 1, ' re la x</w>': 3, '   na u se ous</w>': 9, ' 9 7 </w>': 3, '   sa b re</w>': 5, '   a x le</w>': 2, '   bu m per sti ck er</w>': 1, '   f la sh y</w>': 4, '   ty</w>': 5, '   wh ine</w>': 6, '   af fi li a ted</w>': 1, '   ri gh t ful</w>': 3, '   bu y ers</w>': 8, ' i cy</w>': 1, '   to ying</w>': 7, '   e m my</w>': 2, '   re ju ven a ting</w>': 1, ' be tty</w>': 2, '   pe p</w>': 9, '   ma l pr ac ti ce</w>': 3, '   ro om i e</w>': 1, '   d wi ght</w>': 25, '   ne i gh bor ly</w>': 5, '   tu t ori al</w>': 2, '   hi a tu s</w>': 2, '   gu sta tion</w>': 1, '   than ke e</w>': 1, '   fri ca see</w>': 1, '   mi te</w>': 4, '   pe c ki sh</w>': 1, '   m c gi ll</w>': 2, '   gen e see</w>': 1, '   re ve la tions</w>': 6, '   t ea gu e</w>': 1, '   go ver ner</w>': 1, '   wai ta minute</w>': 2, '   t wi sts</w>': 3, '   ho g wa ll o p</w>': 2, '   w oo l wor th</w>': 1, ' pe te</w>': 4, '   ever e t t</w>': 21, '   sto l ed</w>': 1, '   for e clo s in</w>': 1, '   son o fa g un s</w>': 1, '   in di an o l a</w>': 1, '   bar re l head</w>': 1, ' cu mp</w>': 2, ' n ust</w>': 1, '   go p her</w>': 5, ' on n ell</w>': 3, '   shu t up</w>': 11, '   wh ar v ey</w>': 2, '   wal dri p</w>': 5, '   pa tr on age</w>': 1, '   di tch es</w>': 1, '   s cu se</w>': 3, '   pa ter fami li as</w>': 2, '   fi en di sh</w>': 2, '   d ev i sed</w>': 3, '   be d ev il</w>': 1, '   su b je c tive</w>': 4, '   pu r su in</w>': 1, '   for ni ca ting</w>': 1, '   bab y l on</w>': 8, '   ca in tch a</w>': 1, '   re en s</w>': 1, '   loo k a</w>': 5, '   wh u h h</w>': 1, ' see kin</w>': 1, '   me g ri ms</w>': 1, '   w u d d ya</w>': 10, '   un rea son ing</w>': 2, '   op ti mi s m</w>': 6, '   po made</w>': 3, '   s ke da d dle</w>': 4, '   i z z at</w>': 3, '   h un n er t</w>': 9, '   ba p ti s m</w>': 6, '   har d no sed</w>': 1, '   re de e med</w>': 4, '   war sh ed</w>': 3, '   pi g g ly</w>': 1, '   wi g g ly</w>': 1, '   y a z o o</w>': 1, '   tr ans gre ssi ons</w>': 1, ' n ar row</w>': 2, ' a in tch a</w>': 1, '   re pu ted</w>': 1, '   sen si ti vi ti es</w>': 1, '   comp en sa t in</w>': 1, ' ta g ger y</w>': 1, '   ou tch a</w>': 1, '   as s a</w>': 3, '   h ar</w>': 1, ' e p tion</w>': 1, ' i g nor ant</w>': 1, '   s lo pe</w>': 6, ' sh ou l der ed</w>': 1, '   joh n ni e</w>': 62, ' la te li es</w>': 1, '   stu mp y</w>': 2, '   st ok es</w>': 6, '   co or ant</w>': 1, '   par ful</w>': 1, ' ti m in</w>': 1, '   com m uni ca t in</w>': 1, '   re g</w>': 1, ' ani el</w>': 5, '   p ow as</w>': 1, ' u a sion</w>': 1, '   bl an di sh men ts</w>': 2, '   en ti ce men ts</w>': 1, '   se tt er a</w>': 1, ' t ead</w>': 11, '   wi p in</w>': 2, '   ma w v a</w>': 1, '   si cha tion</w>': 1, '   sc ri p tion</w>': 1, ' h ind</w>': 1, '   a w ga z ation</w>': 1, ' run</w>': 11, '   su ck in</w>': 6, '   so pp in</w>': 1, ' i pp i</w>': 1, ' he c t ac re</w>': 1, '   chi ff on i er</w>': 2, '   ro ll to p</w>': 5, '   hur l en e</w>': 1, '   we d d in</w>': 4, '   for e or da in ed</w>': 1, ' s mi l in</w>': 1, '   re po se</w>': 2, '   ad ven tur ing</w>': 1, '   pe w ter</w>': 1, '   en c ru sted</w>': 1, '   re ti e</w>': 1, '   k not</w>': 13, '   mi x a ph ori ca lly</w>': 1, '   un con st ant</w>': 1, '   su c cu bu s</w>': 1, ' por t</w>': 2, '   cl ar in et</w>': 9, '   b on a</w>': 4, '   fi de</w>': 4, '   al v in e ll e</w>': 1, '   f loo ding</w>': 6, '   h y dr o</w>': 4, ' e le c tri c</w>': 1, '   sin g in</w>': 6, '   ha y see ds</w>': 2, '   sh ow in</w>': 5, '   in na le ct</w>': 1, '   pi ti ed</w>': 1, ' con su med</w>': 1, '   su n st ro ked</w>': 1, '   so g gi ed</w>': 1, '   gu mm in</w>': 1, '   p ab </w>': 1, ' lu m</w>': 1, '   1 9 8 7 </w>': 1, '   tom or r a</w>': 5, '   sp o il in</w>': 1, '   y er</w>': 30, '   lu red</w>': 8, '   ba the</w>': 9, '   d un ked</w>': 1, '   tru ssed</w>': 1, '   qui tch a</w>': 1, '   bab bl in</w>': 1, '   i tta</w>': 1, '   b en a</w>': 1, '   ran c or</w>': 1, '   ne ga ti vi s m</w>': 2, '   li li es</w>': 3, '   par a di g m</w>': 3, '   un sha ved</w>': 1, '   ab so l ved</w>': 3, '   bar th o lo me w</w>': 17, '   cor a</w>': 11, '   de ll is</w>': 1, '   as so ci a tions</w>': 4, '   ye ll a</w>': 4, ' be lli ed</w>': 1, '   s k un ks</w>': 2, '   vo t in</w>': 2, '   con sen sus</w>': 4, '   ge o gra p hi cal</w>': 5, '   o d di ty</w>': 1, '   ha ir ne ts</w>': 1, '   da pp er</w>': 4, '   fo p</w>': 3, '   bri sto l</w>': 3, '   mer t</w>': 1, '   a lo y si us</w>': 1, '   x es</w>': 1, '   so g g y</w>': 4, '   st ee p ed</w>': 2, ' ti me y</w>': 3, '   a in tch a</w>': 2, '   bro ad ca st in</w>': 1, '   f l our</w>': 9, '   sto pp in</w>': 3, ' ac com pl uh</w>': 1, ' t ar</w>': 3, '   co tt on e li a</w>': 1, '   sa l ve</w>': 1, '   hu r</w>': 2, '   hon ch o</w>': 3, '   l un n</w>': 1, '   su n g</w>': 10, '   s ke da d d l ed</w>': 1, '   ac cu mp</w>': 1, ' ac cu mp</w>': 1, '   re co ll e ct</w>': 9, '   in te gra ted</w>': 5, '   in ter</w>': 10, ' gra ted</w>': 1, ' mo ly</w>': 1, '   con sti ch en cy</w>': 3, '   a g g i</w>': 1, ' cu l ture</w>': 2, '   fa h mu h</w>': 1, '   di d d l in</w>': 1, '   lan gu i sh ing</w>': 2, '   b loo ey</w>': 2, '   qui ff</w>': 1, '   par d</w>': 11, '   z ac ki e</w>': 4, '   de b s</w>': 3, '   pen s ac o l a</w>': 6, '   ran i er</w>': 1, '   pa le qu er o</w>': 2, '   doll a</w>': 1, '   ho ch i</w>': 1, '   ma g na</w>': 3, '   la u de</w>': 3, '   b y r on</w>': 7, '   see ger</w>': 4, '   bo on i es</w>': 4, '   may o</w>': 35, '   s an ds</w>': 2, '   k an tr ow i t z</w>': 1, '   d or ed</w>': 4, '   fi tt est</w>': 3, '   e s ther</w>': 15, ' bi r th</w>': 1, '   re que sts</w>': 19, '   wor le y</w>': 13, '   e ye ba lling</w>': 5, '   char a de</w>': 9, '   tra der</w>': 2, '   j on</w>': 48, ' na i se</w>': 1, '   su bi c</w>': 1, '   p hi li pp in es</w>': 2, '   po op i e</w>': 1, '   may on na i se</w>': 11, '   st e ers</w>': 3, ' e we</w>': 2, '   la u der</w>': 1, '   e we</w>': 7, '   na p al m</w>': 6, '   per i ph er al</w>': 2, ' understand</w>': 10, '   ru ffer well</w>': 1, '   cle an up</w>': 5, '   j c</w>': 1, '   pen ne y</w>': 2, '   di l ber t</w>': 1, '   d un k er</w>': 1, '   ho t sho t</w>': 6, '   r ou gh est</w>': 3, ' id</w>': 2, '   po k ri f</w>': 1, '   po op i es</w>': 1, '   o ver hear</w>': 2, '   o ki e</w>': 1, '   mu s co ge e</w>': 1, '   be ev i ll e</w>': 1, '   sh ri v el ed</w>': 1, '   co s m o</w>': 19, '   k at man d u</w>': 5, '   na i ro b i</w>': 1, '   po k ri f k i</w>': 1, '   d or</w>': 2, '   per r y man</w>': 1, '   god d damn</w>': 1, '   so ci a li te</w>': 1, '   o a ki e</w>': 4, ' ne lli e</w>': 1, '   n y mp ho s</w>': 1, ' cu z zy</w>': 1, ' pu get</w>': 1, '   pe sc i</w>': 1, '   pu ff s</w>': 9, '   gre en haven</w>': 1, '   ad min i stra te</w>': 2, '   bur n ha m</w>': 4, '   un p ac ked</w>': 5, '   sc ru b b er</w>': 3, '   mon i t ors</w>': 12, '   under things</w>': 1, '   vo gu e</w>': 2, '   ger man e</w>': 2, '   pe ar l st ine</w>': 3, '   in app ro pri ate</w>': 16, ' lo ad ed</w>': 3, '   s an c tu m</w>': 3, '   br ow n st on es</w>': 1, ' h om es</w>': 1, '   f la t bu sh</w>': 1, '   el more</w>': 1, '   g lu co g en</w>': 2, '   sp a z z</w>': 1, ' ti t ani c</w>': 1, '   mor se</w>': 8, '   0 1 5 0</w>': 1, '   4 7 8 </w>': 2, ' 0 1 5 0</w>': 1, '   sa bu </w>': 6, '   t y r one</w>': 8, '   s ea le</w>': 6, '   cy</w>': 11, '   f le d ged</w>': 1, '   p an ther</w>': 20, '   su ga h</w>': 1, '   dan g ling</w>': 2, '   de cl ar es</w>': 2, '   b ru ta li ze</w>': 1, ' offi c er</w>': 3, ' ra ge</w>': 2, '   el dri dge</w>': 3, '   re v </w>': 4, '   st op li ght</w>': 1, '   den z il</w>': 2, '   d ow ell</w>': 1, '   ar m pi ts</w>': 2, '   bo o j i e</w>': 1, '   jo k ers</w>': 7, '   fr an</w>': 48, '   bri mm er</w>': 8, ' under co ver</w>': 4, '   under min ing</w>': 1, '   pa t ro lling</w>': 3, '   pro fe s sor i al</w>': 1, '   shu ff ling</w>': 4, ' fri en d ly</w>': 3, '   a a a ah</w>': 6, '   mo th a fuck</w>': 1, '   co op ed</w>': 11, ' gla d</w>': 4, '   pe</w>': 4, ' poli ti cal</w>': 3, ' cl ou d</w>': 1, '   d or se t t</w>': 2, '   uni fi ed</w>': 5, '   sp on sor ed</w>': 5, '   under m ine</w>': 5, '   me mo s</w>': 5, '   su b ver si ve</w>': 6, ' ad vi sor y</w>': 1, '   di sc re di ted</w>': 2, '   au spi ces</w>': 1, '   in fi l tra ting</w>': 5, '   re par a tions</w>': 3, ' sen si tive</w>': 1, '   a gi ta te</w>': 1, '   i dea list s</w>': 2, '   st re e t li ght</w>': 2, '   k oo ks</w>': 2, '   h or n et</w>': 3, '   re i ter ate</w>': 3, '   ad vi sor y</w>': 2, '   f le dge</w>': 1, '   ra lly</w>': 16, '   s ar t re</w>': 1, '   a li g ning</w>': 1, '   in c ri min ate</w>': 3, ' under e sti ma ted</w>': 1, '   ra ll ying</w>': 2, '   r ou st ing</w>': 1, '   poli c ing</w>': 2, ' p an ther</w>': 1, '   sha ba z z</w>': 4, '   ph on i es</w>': 2, '   sa v a g es</w>': 10, '   pa in fu lly</w>': 3, ' p an th ers</w>': 1, ' vi o l ent</w>': 3, '   su b si di z ing</w>': 1, ' in for ma tion</w>': 2, '   in fi l tra tion</w>': 4, '   chi ck en shi ts</w>': 3, '   j un g les</w>': 3, '   mo th a fuck er</w>': 1, ' al right</w>': 10, '   mo ther fuck</w>': 4, '   mo ther fuck a</w>': 1, '   go o ks</w>': 5, '   bu ll sh</w>': 1, '   a ll right</w>': 7, '   g ab ri el</w>': 28, '   ca m d en</w>': 3, '   t ac ti ca lly</w>': 3, '   lo gi sti ca lly</w>': 1, '   mor n</w>': 5, '   mu s k et</w>': 2, '   wa d ding</w>': 1, '   da l ton</w>': 11, '   o ver land</w>': 1, '   ba king</w>': 5, '   re d co at</w>': 2, '   char lo tt e</w>': 12, '   re gu l ars</w>': 7, '   ran ks</w>': 11, '   li e u ten an ts</w>': 5, '   out set</w>': 2, '   mar k s men</w>': 1, ' ex u ber an ce</w>': 1, '   t ar ge ting</w>': 7, ' l ed</w>': 1, '   ac cor ded</w>': 3, ' mi li t ary</w>': 2, '   a g g ri ev ed</w>': 2, '   in i ti a ting</w>': 5, '   ir re gu l ars</w>': 1, '   su per se ded</w>': 1, '   par le y</w>': 2, '   t ar le ton</w>': 5, '   pa ll et</w>': 4, '   a mb er c on</w>': 1, '   a sh e u lot</w>': 1, '   ch er ok e e</w>': 5, '   re ga in</w>': 3, '   ch er ok e es</w>': 1, '   re took</w>': 1, '   ch er a w</w>': 1, '   fu r l ough</w>': 2, '   f lan king</w>': 3, '   b en n in g ton</w>': 1, '   har ri s vi ll e</w>': 1, '   ac wor th</w>': 1, '   h or se man</w>': 23, '   s an te e</w>': 1, '   hi ll s bor o</w>': 1, '   re d co a ts</w>': 3, '   en li st</w>': 3, '   de le ga tes</w>': 4, '   l ev y</w>': 3, '   con ven ed</w>': 1, '   ri o ting</w>': 2, '   che st er town</w>': 1, ' f ea ther ed</w>': 1, '   wi l min g ton</w>': 1, '   dis so l ved</w>': 2, '   ja me st own</w>': 1, '   re ar ed</w>': 5, '   s own</w>': 1, '   har ve sted</w>': 3, '   t y ran t</w>': 5, '   y or k town</w>': 2, '   re in for ce</w>': 2, '   che sa p ea ke</w>': 1, '   ver sa i ll es</w>': 1, '   re el ing</w>': 2, '   mu z z le</w>': 1, ' mu z z le</w>': 1, '   imp ru dent</w>': 1, '   chi l d less</w>': 1, '   re gi ci de</w>': 1, '   pe ti tion</w>': 6, '   re dre ss</w>': 2, '   a ven u es</w>': 1, '   s ki r mi sh ers</w>': 2, '   po or er</w>': 6, '   re ce i ves</w>': 3, '   pri g gi sh</w>': 1, ' p oun d ers</w>': 1, '   ga d get</w>': 5, '   bl ow d ow n s</w>': 1, '   c ran d all</w>': 5, '   lu d l ow</w>': 2, '   cre ed</w>': 30, '   6 4 2 </w>': 2, '   al d en</w>': 2, '   de ar bor n</w>': 2, '   pa x c ow</w>': 5, '   bo o ge y man</w>': 2, ' el lie</w>': 2, '   se ma t ary</w>': 5, '   y a ay</w>': 2, '   tru ant</w>': 1, '   bo o g ers</w>': 1, ' bur g l ar</w>': 2, '   m c d ow ell</w>': 3, '   s qu ea ks</w>': 2, '   se w er si d es</w>': 1, '   dan dri dge</w>': 5, '   di a per</w>': 7, '   po op ed</w>': 3, '   chi ca go land</w>': 1, ' su cks</w>': 3, ' wi sh</w>': 3, '   y a y y y</w>': 1, '   in ten si ve</w>': 4, '   hur r ts</w>': 1, '   hur r r r ts</w>': 1, '   pa sc ow</w>': 1, '   ju d</w>': 11, '   d eah</w>': 1, '   he ar ted</w>': 2, '   r ac he l</w>': 104, '   w en di go</w>': 3, '   s ou red</w>': 3, '   mi c m ac s</w>': 2, ' s qu a tting</w>': 1, '   me l t down</w>': 17, '   stan ny</w>': 2, '   w en di go s</w>': 1, '   tri ck st ers</w>': 1, ' stan le y</w>': 3, '   b ou ch ard</w>': 1, '   mi c m ac </w>': 3, ' e d g ar</w>': 1, '   ca ir n</w>': 1, ' move</w>': 3, '   ci pro ca ted</w>': 1, '   ce me ter i es</w>': 4, '   la v as se u r</w>': 1, '   sto pp ard</w>': 1, '   c ow le y</w>': 1, '   or in c o</w>': 2, '   a ll man</w>': 2, ' in ner</w>': 2, '   under done</w>': 1, '   b lot</w>': 1, '   ba ter man</w>': 1, ' baby</w>': 17, '   re st less</w>': 20, '   pa tch ing</w>': 3, '   z el da</w>': 1, '   shi lly</w>': 1, ' sha lly</w>': 1, '   ca t bur g ers</w>': 1, ' ex ci ted</w>': 6, '   ex tra or din ar i ly</w>': 4, '   ar th ri t is</w>': 3, '   u r in ary</w>': 2, '   mo ver s</w>': 2, '   bu ck ar o o</w>': 34, '   ban z a i</w>': 10, '   sp an</w>': 8, ' sha pe</w>': 1, '   an a mar i a</w>': 2, '   an ch or ed</w>': 4, '   d ory</w>': 1, '   im mor tal s</w>': 2, '   e pi c</w>': 5, '   co in ci de</w>': 5, '   da un t less</w>': 6, '   nor r in g ton</w>': 8, '   plan k</w>': 18, '   bo su n</w>': 1, '   f re t</w>': 12, '   ho ar ded</w>': 1, '   i s l a</w>': 10, '   mu er ta</w>': 4, '   in v ok ed</w>': 1, '   par la y</w>': 8, '   n e</w>': 9, '   di sin cl in ed</w>': 2, '   ac qui e s ce</w>': 2, '   pi ra tes</w>': 27, '   bar bo ss a</w>': 5, '   ce s sa tion</w>': 3, '   ho sti li ti es</w>': 3, '   ru de ly</w>': 4, '   bo s om s</w>': 1, '   re ma in der</w>': 2, '   se c lu de</w>': 1, '   com mo d ore</w>': 7, '   pi r ac y</w>': 5, '   ir ons</w>': 2, '   ca sts</w>': 7, '   s wan n</w>': 35, '   dis so lu te</w>': 1, '   sa i ls</w>': 12, '   do te</w>': 1, '   comp or t</w>': 2, '   be fi ts</w>': 1, '   b re ther n</w>': 1, ' e li z a be th</w>': 1, ' si ded</w>': 2, '   fe ar some</w>': 3, '   he ar ti es</w>': 1, '   g na sh ing</w>': 1, '   cont re te mp s</w>': 1, '   car i b b ean</w>': 11, '   ru m run n ers</w>': 1, '   c ac he</w>': 5, '   re s cu ing</w>': 4, '   un told</w>': 2, '   ja w ed</w>': 2, '   mar o on ed</w>': 4, '   car to gra p her</w>': 1, '   n in es</w>': 5, '   m om en t ary</w>': 5, '   ru mb l ed</w>': 2, '   com man de er</w>': 3, '   t or tu g a</w>': 2, '   pi ra ting</w>': 2, '   inter ce p tor</w>': 2, '   su per f l ous</w>': 1, '   s om one</w>': 2, '   re e f s</w>': 1, '   st e er s man</w>': 1, '   st ran ded</w>': 12, '   ri d d l ed</w>': 5, '   ma g ni f y</w>': 1, '   bo o t stra p</w>': 2, '   da v y</w>': 6, '   wa ter f all</w>': 1, '   la go on</w>': 8, '   mu t in ous</w>': 1, '   th ri st</w>': 1, '   qu ar ter ma ster</w>': 3, '   gla s g ow</w>': 2, '   a ma ss</w>': 1, '   bur den s</w>': 3, ' er ch ant</w>': 1, '   o be y ed</w>': 2, '   pu z z ling</w>': 1, ' bo o t stra p</w>': 2, '   a v a st</w>': 2, ' ro ar ing</w>': 2, '   na u ti cal</w>': 4, '   bro ther ing</w>': 1, ' r ing</w>': 3, '   mu re ta</w>': 1, '   ber th</w>': 1, '   cap ta in ed</w>': 2, '   ta x ing</w>': 1, '   f er v or</w>': 2, '   ar g on</w>': 31, '   as cen sion</w>': 4, '   si m</w>': 9, '   ge la t in</w>': 8, '   n an o bo t</w>': 28, '   ac ro po l is</w>': 1, '   ru sh more</w>': 2, '   po ll u tion</w>': 7, '   ex per im en ta tion</w>': 4, '   po pp y</w>': 11, '   b ru ta lly</w>': 4, '   ne b b le man</w>': 6, '   de lu ding</w>': 2, '   ac ce ssi ble</w>': 5, '   o bi tu ary</w>': 5, '   re mor se less</w>': 2, '   a pro po s</w>': 2, '   e go man i ac s</w>': 2, '   i car us</w>': 7, '   o tt om ans</w>': 1, '   de sig n s</w>': 5, '   o t t</w>': 1, '   jo pl in</w>': 2, '   ne b bi sh man</w>': 1, '   po l y mer i z ation</w>': 3, '   l ab or at ory</w>': 27, '   d on a ted</w>': 4, '   out done</w>': 7, '   ma ke o</w>': 1, '   as se mb l er</w>': 4, '   re p li ca t ors</w>': 8, '   c r y o gen i c s</w>': 1, ' do k ey</w>': 4, '   tri x</w>': 5, '   st y ro fo am</w>': 2, '   ex x on</w>': 1, '   li tt er bu g</w>': 2, '   se cre ted</w>': 2, '   li tt er bu gs</w>': 1, '   li tt er ing</w>': 3, '   di a g no sti c</w>': 2, '   ca lu m et</w>': 4, '   ex p on en ti ally</w>': 3, '   v a li da te</w>': 2, '   pi le up</w>': 1, '   tur n pi ke</w>': 8, '   as se mb l ers</w>': 2, '   pu r n ell</w>': 1, '   a a about</w>': 1, '   w el lie</w>': 1, '   me th y l c y an o ac r y late</w>': 1, ' h er o</w>': 4, ' in f an ti le</w>': 1, '   po l y mer i z ed</w>': 4, '   in je c ted</w>': 9, '   sh ort ne ss</w>': 1, '   ad r en al s</w>': 1, '   se cre ting</w>': 1, '   par as y m pa the ti c</w>': 2, '   qui ver ing</w>': 3, '   e stra di o l</w>': 1, '   cour sing</w>': 3, '   sh r in k er</w>': 1, '   v u l c ani z ation</w>': 1, '   ch l or op r en e</w>': 1, '   e la st om ers</w>': 1, '   f lu i ds</w>': 5, '   su n ba thing</w>': 1, '   ph y si ci sts</w>': 2, '   y u k</w>': 6, ' y u k</w>': 1, '   sy n the si z ed</w>': 1, '   me th y l</w>': 1, '   c y an o ac r y late</w>': 1, ' ex i st ing</w>': 1, '   a ll er gi es</w>': 2, '   di a g no sti c s</w>': 4, ' pro du ct</w>': 3, ' imp le</w>': 1, '   mi c ro s co pi c</w>': 11, '   en co ded</w>': 5, '   sy n the si ze</w>': 3, '   po l y i so pr en es</w>': 1, '   tr act</w>': 5, '   di u re ti c s</w>': 1, '   rea s se mb le</w>': 3, ' ce lled</w>': 2, ' po l y mer i z ation</w>': 1, '   ph y si ci st</w>': 4, ' che mi cal</w>': 1, '   de pen den cy</w>': 3, '   su ze</w>': 4, '   sy ll able</w>': 6, '   che mi sts</w>': 4, '   de fo li ant</w>': 2, '   re vo lu tion i z es</w>': 1, '   uni fi es</w>': 1, '   mi d we st</w>': 8, '   fin din gs</w>': 3, '   ce lled</w>': 1, '   ho li est</w>': 2, '   r y k er</w>': 1, ' bea ting</w>': 1, '   ac com mo da ting</w>': 2, '   pi ss boy</w>': 1, '   z o ok ee per</w>': 1, '   l ab or at ori es</w>': 4, '   e qu ation</w>': 11, '   in st ab i li ty</w>': 6, '   f ru stra tions</w>': 2, '   in v a li da te</w>': 2, ' par an o i a</w>': 2, '   ke y c ard</w>': 1, '   de x ter</w>': 6, '   sch u y l er</w>': 27, '   s na ps</w>': 8, '   vi ci ou s ly</w>': 2, ' m ou th ed</w>': 6, '   mi st ru sted</w>': 1, ' go es</w>': 1, '   bar room</w>': 4, '   fu ri ou s ly</w>': 5, ' s me lling</w>': 3, '   pu b li ca tion</w>': 5, '   cour te ous</w>': 8, '   su b b ing</w>': 1, '   gar t ers</w>': 8, '   sp in ac h</w>': 4, '   ra z z ing</w>': 2, '   gi l ded</w>': 11, '   co li se u m</w>': 1, ' d i</w>': 1, '   ea mes</w>': 2, '   ra d c li ff</w>': 2, '   si ber i an</w>': 7, '   bab y k ins</w>': 3, '   s my the</w>': 20, '   bo o ful</w>': 1, '   s wee tu ms</w>': 1, '   3 1 0 0</w>': 1, ' ho l der</w>': 1, ' b less</w>': 2, '   b in g y</w>': 10, '   th ra sh ed</w>': 3, '   b wan a</w>': 3, '   o i ls</w>': 4, '   tw e et</w>': 2, '   ar ab y</w>': 3, '   re wri tt en</w>': 3, '   bo id</w>': 1, '   con ro y</w>': 2, ' la w s</w>': 4, '   man i cu red</w>': 1, '   de s ks</w>': 4, ' st or ms</w>': 1, '   b lu en o ses</w>': 1, '   car f are</w>': 2, '   a a gh</w>': 2, '   mon o ton y</w>': 2, '   bu tt on ing</w>': 1, '   ra m se y</w>': 5, '   gra y son</w>': 9, '   sp a ts</w>': 20, '   t ea s</w>': 1, '   t re men d ou s ly</w>': 4, ' st r en g th</w>': 6, '   h er o ine</w>': 5, '   con ra d</w>': 18, '   t ying</w>': 14, '   p an si es</w>': 5, ' hi tting</w>': 1, '   s lu ms</w>': 3, '   ma g no li a</w>': 3, '   sp ea k ea sy</w>': 2, '   car lo a d</w>': 1, '   que en ly</w>': 3, '   s no z z le</w>': 2, '   ber ri es</w>': 9, '   si z z ling</w>': 1, '   p s st</w>': 4, '   so o thing</w>': 3, '   lea gu er</w>': 4, '   sh e k el s</w>': 1, '   c ru de ly</w>': 1, '   1 0 6 </w>': 1, ' w are</w>': 2, '   s wor d fi sh</w>': 1, '   con ra ds</w>': 1, '   g lan c ing</w>': 4, '   bi car b on ate</w>': 4, '   ne w sp a per men</w>': 3, '   bra w l</w>': 7, '   b le s sin gs</w>': 7, '   pu tt er er</w>': 8, '   pu tter</w>': 10, '   ru g ged</w>': 3, '   m oun ta in to p</w>': 3, '   pu tt er ers</w>': 2, '   s ke le t ons</w>': 6, '   vi bra te</w>': 2, '   li very</w>': 3, '   je ev es</w>': 4, ' ki ds</w>': 5, '   b lu e ber ri es</w>': 2, ' su e</w>': 5, '   han d ba g</w>': 12, '   bi ff</w>': 12, '   bu </w>': 10, ' u d</w>': 4, '   plea s an t vi ll e</w>': 8, '   sp r in k l ers</w>': 2, ' a ll ey</w>': 2, ' ma k er</w>': 4, '   con ne l</w>': 1, ' p in</w>': 2, '   ci vi c s</w>': 1, '   ke en est</w>': 4, '   whi z z</w>': 4, '   ch ee se bur ger</w>': 8, ' hon ey</w>': 9, '   k ab ob s</w>': 2, '   bo x boy</w>': 1, ' chan g es</w>': 5, ' cer tain</w>': 5, ' plea s ant</w>': 2, '   de f en dan ts</w>': 4, '   sur re p ti ti ou s ly</w>': 2, '   ver mi lli on</w>': 2, '   pu ce</w>': 1, '   char t re use</w>': 1, '   u mber</w>': 2, '   a qu a</w>': 2, '   o x</w>': 12, '   c ri m son</w>': 4, '   ma gen ta</w>': 1, '   te ch ni co l or</w>': 3, '   d or k y</w>': 4, ' wasn</w>': 8, '   fa tt en ing</w>': 2, ' att r ac tive</w>': 3, ' o c cu r</w>': 1, ' om i god</w>': 4, ' bu d</w>': 1, '   ch ee se bur g ers</w>': 8, ' llo</w>': 1, '   under wi re</w>': 2, '   j en</w>': 21, '   pe g g y</w>': 42, '   por es</w>': 2, '   no oo oo o</w>': 1, '   u ch h</w>': 1, '   pa st y</w>': 2, '   st re ss ing</w>': 3, ' sta i i ir s</w>': 1, '   we il</w>': 2, '   om i god</w>': 4, ' li ee ee ve</w>': 1, '   mar a th on</w>': 10, '   pro p</w>': 6, '   cu r ran ts</w>': 1, '   mm mm gh</w>': 1, '   o at m ea l</w>': 1, '   mar ma la de</w>': 1, '   m c in ti re</w>': 5, ' re f re sh</w>': 1, ' happ en</w>': 5, ' happened</w>': 3, ' run s</w>': 3, '   gu tt ers</w>': 3, ' m c in ti re</w>': 1, ' oo om p h</w>': 1, '   m c g in ty</w>': 1, '   chri st ma sti me</w>': 1, '   bu n</w>': 5, '   le tt u ce</w>': 15, '   wi p ing</w>': 7, '   m ea s les</w>': 9, '   list ing</w>': 2, '   2 1 5 </w>': 3, '   mon e t ary</w>': 1, '   pu r cha ses</w>': 9, '   cou p on</w>': 9, ' f ly</w>': 5, '   pu d ding</w>': 31, ' mo ther fuck er</w>': 1, ' i ll e ga l</w>': 2, ' 3 4 </w>': 5, '   s co op er</w>': 1, '   i i i i i i i i i i i</w>': 1, '   so f te st</w>': 1, '   cu ps</w>': 18, '   ter i y a k i</w>': 3, ' 7 9 </w>': 3, '   8 9 </w>': 4, '   brea ds</w>': 1, '   s ou ps</w>': 1, '   pro mo tion al</w>': 2, '   gi v ea ways</w>': 3, '   de mo li sh ed</w>': 4, ' 8 1 8 </w>': 1, '   3 3 7 </w>': 2, ' 9 8 7 6 </w>': 1, '   e g an</w>': 4, ' 1 2 7 4 </w>': 1, '   mo or par k</w>': 1, '   9 1 4 0 3 </w>': 1, ' 3 4 0 7 </w>': 1, '   2 6 2 7 </w>': 1, '   3 4 4 4 </w>': 1, '   8 0 9 5 </w>': 1, '   ex pi ra tion</w>': 4, '   0 5 </w>': 1, ' con fi den ti al</w>': 1, '   j ani ce</w>': 4, '   ex t or ted</w>': 1, '   bo s sin ess</w>': 1, '   pl un g ers</w>': 2, ' brea k able</w>': 1, '   m h m</w>': 1, '   ex i ts</w>': 12, '   to le do</w>': 5, '   cou p ons</w>': 8, ' bar ry</w>': 1, ' lan ce</w>': 6, ' who a</w>': 3, '   sc an ned</w>': 6, '   k a th le en</w>': 1, '   co br a</w>': 7, '   la ti sh a</w>': 1, ' sha ke</w>': 3, '   u ch</w>': 1, ' ke ys</w>': 1, ' do e</w>': 1, '   4 8 4 </w>': 1, '   re star a un t</w>': 1, '   per ver sion</w>': 3, '   e mi li on</w>': 1, '   st ee l ed</w>': 1, '   g ri s li est</w>': 1, '   of f er in gs</w>': 1, '   har mon i ous</w>': 3, '   char en ton</w>': 14, '   sa de</w>': 5, '   do ci le</w>': 4, '   list less</w>': 1, '   b in ding</w>': 4, '   sh ri ve lled</w>': 1, '   pi tt an ce</w>': 1, '   ab be</w>': 16, '   ma u p as</w>': 1, '   s lan der</w>': 4, '   wh e t st one</w>': 1, '   co lo m be</w>': 1, '   p an ta le tt es</w>': 1, '   ne st l ed</w>': 2, '   tu li p</w>': 1, '   f an ch on</w>': 5, '   ex tra v a g an tly</w>': 2, '   in spe c ting</w>': 4, '   fo lli c les</w>': 2, '   fo lli c le</w>': 3, '   a w o ke</w>': 2, '   d ru m sti cks</w>': 2, '   mar row</w>': 18, '   wi tch ing</w>': 2, '   in ci te</w>': 2, '   ma de le ine</w>': 25, '   h er o in es</w>': 2, '   st ru mp et</w>': 2, '   mu r der ess</w>': 7, '   s la ving</w>': 2, '   qui lls</w>': 2, '   v al c our</w>': 2, '   cou l mi er</w>': 2, '   im po t ence</w>': 1, '   co o ed</w>': 1, '   cour ted</w>': 2, '   la p do g</w>': 2, '   au th ors</w>': 5, '   gra ti fi ca tion</w>': 4, '   b ou ch on</w>': 2, '   a ph ro di si ac </w>': 2, '   f la y ed</w>': 3, '   pro ce ssion</w>': 4, '   gu i ll o t ine</w>': 3, '   un che cked</w>': 3, ' b on ed</w>': 2, '   or vo ll e</w>': 1, '   o x y mor on</w>': 1, '   nu mb s k u ll</w>': 1, '   cha i se</w>': 4, '   p lo pp ing</w>': 1, '   be so tt ed</w>': 1, '   pe der a sts</w>': 1, '   s ac ri le ge</w>': 3, '   pro vo ca te u r</w>': 1, '   v ea l</w>': 5, '   shu d der</w>': 1, '   che er i er</w>': 1, '   ar t fu lly</w>': 1, '   re ha bi li ta tion</w>': 5, '   du pe</w>': 3, '   qui ll</w>': 5, ' wi ts</w>': 2, '   p in hea ds</w>': 1, '   clo the</w>': 1, '   u sur p</w>': 1, '   t rea ti se</w>': 1, '   sy mp hon i es</w>': 3, '   ta x ed</w>': 1, '   per ver si ons</w>': 3, '   in an e</w>': 1, ' ni pp le</w>': 1, ' pi ke st af f</w>': 1, '   di sc er n</w>': 2, '   stra i gh ta way</w>': 2, '   s an ction</w>': 3, '   im pl or ed</w>': 1, '   cu ra tive</w>': 2, '   sta ve</w>': 1, '   f ea ther bed</w>': 1, '   l ac o st e</w>': 1, '   s mo o th ly</w>': 4, '   lu bri ca ted</w>': 2, '   ran ts</w>': 1, '   ra ves</w>': 4, '   po or ly</w>': 5, '   f ar ing</w>': 2, '   cor ru p ted</w>': 4, '   me ttle</w>': 1, '   t in c ture</w>': 1, '   in ma tes</w>': 7, '   ver i ta ble</w>': 2, '   see k ers</w>': 3, '   s an at ori u m</w>': 1, '   a v ow ed</w>': 1, '   pla y w right</w>': 10, '   e mer i tu s</w>': 1, '   ma d house</w>': 6, '   ex p lo i ting</w>': 8, '   cre t ins</w>': 3, '   b ou gi v al</w>': 1, '   cla ir wi l</w>': 1, '   w re sted</w>': 1, '   th ra sh ing</w>': 3, '   ca l m ing</w>': 5, '   f lo g</w>': 2, '   bar ome ter</w>': 2, '   a g ha st</w>': 1, '   un pr in ta ble</w>': 1, '   pu r ga tive</w>': 1, '   to x ins</w>': 3, '   ro ars</w>': 2, '   ta un ts</w>': 1, '   mo le sts</w>': 1, '   a men i ti es</w>': 1, '   p sy c he</w>': 5, '   wa ter co l or</w>': 1, '   no t ori e ty</w>': 2, '   in di sc re tions</w>': 3, '   v in cen n es</w>': 16, '   ca u ter i z ing</w>': 1, '   p rea ch es</w>': 2, '   ro y er</w>': 1, ' co ll ard</w>': 1, '   ph y si ci ans</w>': 6, '   ro be spi er re</w>': 1, '   dan ton</w>': 1, '   mar at</w>': 1, '   de sp o t</w>': 1, '   loo sen ed</w>': 3, '   ro b es</w>': 5, '   mu tt er ed</w>': 2, '   de l b en</w>': 3, '   sta ir</w>': 1, ' i dea li s m</w>': 1, '   i dea li st</w>': 2, '   poli ti c</w>': 3, '   r en own</w>': 3, '   ma s se</w>': 2, '   p ore</w>': 2, '   ri s q u</w>': 1, '   e u gen i e</w>': 2, '   da in ty</w>': 1, '   mor se l</w>': 1, '   d or sa l</w>': 2, '   r ou </w>': 1, '   su ck ling</w>': 1, '   sc e p ter</w>': 2, '   r ar i ty</w>': 2, '   sa te</w>': 8, '   ma ddy</w>': 6, '   la und re ss</w>': 1, '   vi g or</w>': 3, '   dis lo dge</w>': 1, '   ha bi tu </w>': 1, '   de ca d es</w>': 3, '   de c ea sed</w>': 27, '   su c cu l ent</w>': 1, '   cle an te</w>': 1, '   da u ph in</w>': 2, '   fran v al</w>': 1, '   vo ll ey</w>': 1, '   re sen ts</w>': 1, '   doll op ed</w>': 1, '   c ru el er</w>': 1, '   sto ke</w>': 2, '   ne we st</w>': 4, '   t or tur es</w>': 5, '   co que tt e</w>': 1, '   ad min i ster</w>': 5, '   fi en di sh ly</w>': 1, '   as su re d ly</w>': 1, '   la ss</w>': 5, '   en tom b</w>': 1, '   per il</w>': 9, '   com po sing</w>': 2, '   pl u m ca ke</w>': 1, '   for sa ke</w>': 1, '   un co ver s</w>': 1, '   chan de li ers</w>': 1, '   si ph on ed</w>': 2, '   sc or n</w>': 3, '   op i a tes</w>': 2, '   war ran ts</w>': 9, '   mar qui se</w>': 3, '   im part</w>': 2, '   b en ev o l ent</w>': 6, '   p hi lan th ro pi st</w>': 1, '   per ver si ty</w>': 3, ' go tt en</w>': 1, '   bor n e</w>': 4, '   de gen er ac y</w>': 1, '   ta in ted</w>': 6, '   bu tt re ss</w>': 1, '   en t rea ti es</w>': 1, '   cha st en</w>': 1, '   mi s be ha ves</w>': 1, '   du ti fu lly</w>': 1, '   per i sh es</w>': 1, '   dan k</w>': 4, '   ro den ts</w>': 7, '   pr ou i x</w>': 1, '   su c c in ct</w>': 1, '   go ver ned</w>': 2, '   l un a ti c s</w>': 6, '   de sp er ation</w>': 6, '   ho i st</w>': 5, '   ja il ers</w>': 1, '   bl an ch es</w>': 1, '   pe ti tion ed</w>': 2, '   an on y mi ty</w>': 6, '   f la un t</w>': 6, '   d ev i an ce</w>': 1, '   d on a ti en</w>': 1, '   b on b ons</w>': 1, '   g or ging</w>': 1, '   s wee t m ea ts</w>': 1, ' sha me</w>': 2, '   p ac i f y</w>': 1, '   ev a si ons</w>': 1, '   or son</w>': 30, '   win ch ell</w>': 2, '   k an e</w>': 115, '   5 5 </w>': 5, '   ha m my</w>': 1, '   ra p un z el</w>': 2, '   li ce</w>': 3, '   ju mp in</w>': 4, ' tra ps</w>': 2, '   in ti ma tely</w>': 8, '   fi er ce st</w>': 2, '   co h n</w>': 2, '   ex p lo i ts</w>': 3, ' hu b</w>': 1, '   ran do l p h</w>': 7, '   s lo ven ly</w>': 1, '   un att r ac tive</w>': 7, ' to l er ate</w>': 1, '   dea u vi ll e</w>': 2, '   th ea tri cal</w>': 2, ' pi ck le</w>': 1, '   e u p he mi sti ca lly</w>': 2, ' sta tu es</w>': 1, ' pa id</w>': 5, '   mi lli c ent</w>': 1, ' b al in e se</w>': 1, '   ca m pa i g n s</w>': 4, '   je de di ah</w>': 1, '   ho pped</w>': 6, '   b en z e dr ine</w>': 4, '   ge tt ys</w>': 1, '   he ar st</w>': 15, '   v a un ted</w>': 2, '   minu te men</w>': 1, '   a ma l ga ma tion</w>': 2, '   bo ff o</w>': 1, '   comp le x i ty</w>': 2, '   to me</w>': 2, '   i ll u sion</w>': 28, '   er ran t</w>': 1, '   f al si ty</w>': 2, '   ni x</w>': 27, '   ma l ar k ey</w>': 3, '   sc or ing</w>': 6, '   sc ha e f er</w>': 10, '   clo se u ps</w>': 3, '   man k</w>': 7, '   ro se bu d</w>': 21, '   s l ed</w>': 5, '   go sp el s</w>': 2, ' b lu e</w>': 5, '   go l d fi sh</w>': 4, ' je w</w>': 4, '   e ster</w>': 9, '   he dy</w>': 1, '   la mar r</w>': 1, '   l un k</w>': 1, ' be d ra g g l ed</w>': 1, '   s war th y</w>': 3, '   be d ra g g l ed</w>': 1, ' p un y</w>': 1, ' sp o il ed</w>': 1, '   di le tt an te</w>': 1, '   gre g g</w>': 1, '   fo ster</w>': 22, ' k an e</w>': 2, '   mar g ins</w>': 2, '   ad or es</w>': 6, '   au d ac i ty</w>': 4, '   f ac et</w>': 2, '   i ll u min a ted</w>': 2, '   sc or ch</w>': 2, '   kee p ers</w>': 7, '   sy co ph ant</w>': 2, '   l or ds</w>': 11, '   je ster</w>': 4, '   le ga li ty</w>': 2, '   per mu ta tions</w>': 2, '   fe u da l</w>': 1, '   bi o gra ph y</w>': 4, '   x an ad u</w>': 3, '   k u bl a</w>': 1, '   fa u x</w>': 2, '   cap es</w>': 1, ' b loo d</w>': 11, '   l r ving</w>': 1, '   an da lu si an</w>': 1, '   pla ins</w>': 2, '   bu ll fi gh ter</w>': 2, '   man o le te</w>': 2, '   su per e go</w>': 1, '   sig m un d</w>': 2, '   lo ck he ed</w>': 1, '   har l ow</w>': 3, ' ki ck er</w>': 1, '   la ten ess</w>': 2, ' ri o t</w>': 1, '   pe p s i</w>': 9, ' co l a</w>': 8, '   tri st an</w>': 1, '   i so l de</w>': 1, ' con ra d</w>': 1, '   mu ck ey</w>': 1, ' mu cks</w>': 1, '   do lled</w>': 2, '   a z te c</w>': 1, '   h er man</w>': 14, '   mon st r o</w>': 1, '   h er od</w>': 1, '   s wan be ck</w>': 3, '   sto ck ho l d ers</w>': 6, '   da vi es</w>': 4, '   ti vo l i</w>': 1, '   re vi v al</w>': 2, ' di sc re tion</w>': 1, '   be tt in</w>': 3, '   ex i st ing</w>': 9, ' ro se bu d</w>': 8, '   gra u man</w>': 3, '   ca pi t an</w>': 5, '   lo ca tions</w>': 5, ' jo ke</w>': 2, '   s ou ll ess</w>': 4, '   s la sh ing</w>': 2, '   di st ri bu t ors</w>': 2, '   k a la ma z o o</w>': 1, '   he d da</w>': 3, '   sti pu la tes</w>': 1, '   par ed</w>': 1, '   pro je c tions</w>': 7, '   u p tur ned</w>': 2, '   cap ra e s qu e</w>': 1, '   vi b ran t</w>': 4, '   ti lt</w>': 4, '   so ar ing</w>': 2, '   j ani ro s</w>': 1, '   ex pe c t in</w>': 6, '   sa l v y</w>': 17, '   la mo tta</w>': 5, '   ga mb l in</w>': 2, '   sa tt er fi e ld</w>': 2, '   vi c ki e</w>': 14, '   l en ore</w>': 8, '   win d up</w>': 1, '   co p a</w>': 4, '   su sp en sion</w>': 7, '   ru m my</w>': 7, '   j an ir o</w>': 10, ' coming</w>': 10, '   k no ck out</w>': 3, '   1 5 5 </w>': 4, '   1 6 0</w>': 1, '   for fe it</w>': 5, '   1 6 1 </w>': 1, '   p un ch es</w>': 2, '   hea v y we ight</w>': 6, '   mi d d le we ight</w>': 2, '   s wi m su it</w>': 2, ' la mo tta</w>': 1, '   vi ck</w>': 5, ' jo ey</w>': 1, '   fuck l ed</w>': 1, '   chri st sa ke</w>': 6, ' m ou l an</w>': 1, '   y ans</w>': 1, '   con ten der</w>': 8, '   je ffer i es</w>': 11, ' c lu b b ing</w>': 1, '   br y ce</w>': 1, '   in do</w>': 4, ' ch in a</w>': 3, '   th or wa ld</w>': 30, '   pre sen ta ble</w>': 3, '   di sp o sing</w>': 4, '   da lly</w>': 1, '   mer ri t s vi ll e</w>': 2, '   stu ff ing</w>': 11, '   pi lo ting</w>': 2, '   he ck ling</w>': 2, '   ta x pa y er</w>': 6, '   sle u th</w>': 2, '   pen al</w>': 2, ' se ar ch</w>': 1, '   un supp or ted</w>': 3, ' gr and</w>': 6, '   ten an ts</w>': 6, '   j i bed</w>': 2, '   th or wal ds</w>': 3, '   a ye m</w>': 3, '   dr un k en ne ss</w>': 2, '   ja y wal king</w>': 1, '   in v a li d</w>': 4, '   sh re w d ly</w>': 1, '   sle ight</w>': 5, '   sa w s</w>': 7, '   g un ni son</w>': 3, '   di sh wa sh er</w>': 4, '   na g ging</w>': 6, '   d ra sti c</w>': 10, ' room</w>': 16, '   la mb er t</w>': 1, '   b re vo or t</w>': 2, '   f re mon t</w>': 7, '   to tal s</w>': 3, ' n in e ty</w>': 2, '   s na p sho t</w>': 5, '   tri e ck on al</w>': 1, '   a ll e y way</w>': 4, '   or n er y</w>': 2, '   d ev e lo p men ts</w>': 8, '   f la sh li gh ts</w>': 5, '   lu min ous</w>': 3, '   di al s</w>': 3, '   sp li tting</w>': 13, '   b loo d sho t</w>': 1, '   my les</w>': 1, '   ma la d ju sted</w>': 1, '   p sy cho an al y ze</w>': 2, '   ta x i es</w>': 1, '   tra mp ing</w>': 1, '   r ar e fi ed</w>': 2, '   su l try</w>': 1, ' wor shi p ers</w>': 1, '   h or m one</w>': 9, '   te ss</w>': 2, '   ch el se a</w>': 8, ' 7 0 9 9 </w>': 1, '   b ac hel or ho od</w>': 2, '   re vi ving</w>': 1, '   gh ou ls</w>': 3, '   f re ck les</w>': 4, '   c lu tch es</w>': 3, '   re bu tt al</w>': 12, '   be d po st</w>': 2, '   l ars</w>': 2, '   n on cha l ant</w>': 2, '   bi ck er ing</w>': 1, '   under ta k ers</w>': 2, '   qu o</w>': 6, '   de f la ting</w>': 1, ' oun ce</w>': 4, '   lin ger i e</w>': 22, '   s an d ba g ged</w>': 2, '   un fa v or able</w>': 2, ' 5 4 </w>': 3, '   si mm er</w>': 1, '   an o in ted</w>': 2, '   re se mb l ed</w>': 2, ' man less</w>': 1, '   me lan cho li a</w>': 2, '   f lan ne l</w>': 5, '   pa ki st an</w>': 5, '   ha y w ard</w>': 3, '   wal d or f</w>': 3, '   du f re s n e</w>': 1, '   sh ow in gs</w>': 1, '   le land</w>': 36, '   ti re some</w>': 3, '   cor k sc re w</w>': 2, '   or na te</w>': 1, ' mi ll</w>': 5, '   mar k u ps</w>': 1, ' 5 5 9 8 </w>': 1, '   as sa u l ting</w>': 2, '   bu z z i e</w>': 1, '   mar l on</w>': 17, '   g under s en</w>': 1, '   mi ll er town</w>': 1, '   m ac hi s m o</w>': 3, '   co ok ab o o</w>': 1, '   j i mb o</w>': 5, '   s a</w>': 3, '   bra w ls</w>': 1, '   pla to</w>': 18, ' sh r in k er</w>': 2, '   sc hi z o id</w>': 4, '   w ow e e</w>': 1, '   chi c ki e</w>': 5, '   pu sh es</w>': 10, ' aren</w>': 10, '   ca l med</w>': 6, '   e p so m</w>': 1, '   i mi ta te</w>': 15, '   nu rs</w>': 1, '   ne w l y we ds</w>': 5, '   plan e t ar i u m</w>': 3, '   lon e li est</w>': 2, '   un fri en d ly</w>': 1, '   sc ri mp</w>': 1, ' ja mi e</w>': 3, ' sc rea m ing</w>': 1, '   u g li est</w>': 5, ' vi e w</w>': 2, '   min i b ar</w>': 1, '   t in se l</w>': 2, '   r ac e tr ack</w>': 2, '   lea ven wor th</w>': 10, '   na z ar e th</w>': 3, '   c ran ber ry</w>': 5, '   li s be th</w>': 1, '   mu st ac he</w>': 12, '   ex p an sion</w>': 3, '   a sh le y</w>': 14, '   war e hou ses</w>': 5, '   coun t ers</w>': 3, ' hel p ing</w>': 1, '   re mo de l ed</w>': 1, '   pa lls</w>': 1, '   for ti e th</w>': 2, '   fif ti e th</w>': 4, '   j an ey</w>': 10, ' e mber</w>': 5, '   can e</w>': 12, '   f ra g ran ces</w>': 2, '   q et</w>': 1, '   s an ta s</w>': 1, '   p ow w ow</w>': 11, '   s k i</w>': 16, '   s wee per</w>': 1, '   ke y pa d</w>': 1, ' ru m pu m</w>': 1, '   pu m</w>': 2, ' pu m</w>': 3, '   ri gs</w>': 4, '   n n n</w>': 1, ' tt t</w>': 1, ' t t</w>': 2, '   lo ck up</w>': 4, '   r er un s</w>': 1, ' we st er n</w>': 3, '   na sh vi ll e</w>': 14, '   to ma ha w k</w>': 5, '   si dle</w>': 1, '   bu tt er ed</w>': 4, '   he y y y</w>': 1, '   chi m ne y</w>': 12, '   ro b b in</w>': 5, '   ca sed</w>': 2, '   gu i de bo o ks</w>': 1, ' times</w>': 8, '   ven u es</w>': 1, '   mer y l</w>': 3, '   st re ep</w>': 1, '   ju g g le</w>': 4, '   ca pa d es</w>': 3, '   cha let</w>': 2, '   i g lo o</w>': 1, '   d on a tions</w>': 4, '   be e f s</w>': 1, ' li ps</w>': 2, '   m ac kin</w>': 1, '   cha w</w>': 1, '   ro tt ed</w>': 4, '   jo li et</w>': 2, ' che w er</w>': 1, ' su ck er</w>': 4, '   mer l in</w>': 23, '   re mo de ling</w>': 1, '   ma ke over</w>': 1, '   pu g</w>': 2, ' ja il</w>': 2, '   ca shi ers</w>': 1, ' cho co late</w>': 2, '   z o ok er man</w>': 1, '   shi v </w>': 5, '   li f er</w>': 1, '   g un run ning</w>': 1, '   tal k ers</w>': 2, '   z oo k</w>': 3, '   ho t wi r ing</w>': 1, '   p in sch er</w>': 1, '   ra tt ed</w>': 6, '   c ree</w>': 1, '   si d na w</w>': 2, '   no g</w>': 2, '   dr un ke st</w>': 1, '   sto cking</w>': 5, '   pe can</w>': 5, '   a mu ses</w>': 3, '   ma t the w s</w>': 7, '   for e man</w>': 7, '   ti me c ard</w>': 1, '   s ca g ne tt i</w>': 10, '   cra tes</w>': 14, '   man g y</w>': 3, ' vi c</w>': 1, '   la d or a</w>': 3, '   e lo is</w>': 6, '   ro ta tion</w>': 7, '   ha cked</w>': 11, '   we t back</w>': 2, '   g ri er</w>': 6, '   pa lo s</w>': 1, '   ver d es</w>': 1, '   bl on die</w>': 3, '   ca bo t</w>': 8, '   bl ower</w>': 2, '   sta lls</w>': 2, '   me mor i ze</w>': 11, '   an e c do te</w>': 2, '   b ran do</w>': 1, '   na tu ra li sti c</w>': 2, '   b re w ers</w>': 3, '   di d dle</w>': 2, '   mi l wa u ke e</w>': 10, '   gar den a</w>': 2, '   s li ts</w>': 2, ' bu st in</w>': 1, '   re my</w>': 3, ' v a le</w>': 1, ' y u m</w>': 1, '   di ll in ger</w>': 15, '   than k ful</w>': 9, '   re d und ant</w>': 5, '   un cu t</w>': 12, '   who le sa l er</w>': 6, '   i s ra el</w>': 48, '   mar se ll us</w>': 1, '   spi v ey</w>': 5, '   m c g ar</w>': 1, '   dr on ing</w>': 3, ' to by</w>': 1, '   ch un g</w>': 3, '   cha n</w>': 1, '   ra mb l ers</w>': 2, '   s la sh ed</w>': 5, '   po si tion ed</w>': 4, '   f er che tt i</w>': 1, '   ne w en d y ke</w>': 1, ' h ar</w>': 1, '   u huh</w>': 4, '   p ani cking</w>': 4, '   o in t ment</w>': 5, ' wor ds</w>': 3, '   wa ge</w>': 11, '   ti p wor th y</w>': 1, ' tri lli on</w>': 1, ' he ar t be at</w>': 1, '   lo ve be at</w>': 1, '   de fran c o</w>': 2, ' s oun ding</w>': 1, '   tru i s m</w>': 1, '   so li dar i ty</w>': 1, '   sti ck in</w>': 10, '   pla y gr ound</w>': 8, '   nu ll</w>': 18, '   p ani c s</w>': 4, '   brea ther</w>': 4, '   wai t re ss ing</w>': 2, ' co ll e ge</w>': 1, '   ser v in</w>': 2, '   wai t re ss es</w>': 9, '   le d no v </w>': 14, '   son or a</w>': 13, '   w y at t</w>': 66, '   pe de st al s</w>': 1, '   g in g ha m</w>': 6, '   cu i te</w>': 1, '   wa ked</w>': 1, '   a ways</w>': 1, '   po po ver s</w>': 1, '   t ou lo mo e</w>': 1, '   stra g g l in</w>': 2, '   le ar n in</w>': 6, '   stra g g ling</w>': 1, '   g ru b</w>': 7, '   t ar p</w>': 3, '   sur re y</w>': 3, '   g un sh y</w>': 1, '   an d me</w>': 1, '   s l mi gh ty</w>': 1, '   o a ts</w>': 1, '   mu les</w>': 3, '   no g al es</w>': 2, '   ra w son</w>': 1, '   ok e h</w>': 1, '   for ge tt in</w>': 4, '   t ou lo m n e</w>': 1, '   lea ded</w>': 1, '   ro ans</w>': 1, '   si ck er</w>': 4, '   lo de</w>': 6, '   s no op in</w>': 2, '   out gr own</w>': 2, '   ga s li gh ts</w>': 2, '   re gi st ers</w>': 2, '   ar mb an ds</w>': 2, '   pi on e er</w>': 5, '   min d en</w>': 2, '   pl ac er vi ll e</w>': 1, '   co ok in</w>': 5, '   men d in</w>': 2, '   wa sh in</w>': 1, '   pr ow ling</w>': 2, '   t ea s in</w>': 2, '   w ran g l ers</w>': 1, '   sa d d ling</w>': 1, '   ea t m ea l</w>': 1, '   wi d ower</w>': 4, '   c li pp in gs</w>': 5, '   hu gh</w>': 13, '   jo tt ed</w>': 1, ' 3 0 2 </w>': 2, '   han son</w>': 5, '   f lin t st one</w>': 3, '   who op ed</w>': 2, '   u ri ah</w>': 2, '   b loo d wor th</w>': 1, '   k ea ton</w>': 34, '   1 8 9 8 </w>': 1, '   ar chi ba ld</w>': 6, '   sp ac ed</w>': 6, '   s qu in ty</w>': 1, '   un can ny</w>': 7, '   c in</w>': 2, '   se que l</w>': 10, '   car e ta k er</w>': 7, '   o l d man</w>': 1, '   a w w</w>': 10, '   u h n</w>': 7, '   s lan g</w>': 8, '   f la v a</w>': 1, '   in ti mi da ting</w>': 3, '   y i pp i e</w>': 1, '   su b stan ti a ted</w>': 1, '   po l ter ge i st</w>': 2, '   in c rea sin g ly</w>': 7, '   shi f ting</w>': 5, '   jo g ging</w>': 5, '   chi ck a de es</w>': 2, '   i mb al an ce</w>': 8, '   t in i est</w>': 2, '   un re fu ted</w>': 1, '   par ti ci pa ting</w>': 3, '   a wa k en</w>': 3, '   st ev e st on</w>': 2, '   ma ss ac re</w>': 8, '   st ab b in gs</w>': 1, '   di sor der</w>': 11, '   s qu ir t in</w>': 1, '   sta ti sti ca lly</w>': 4, '   su g ge sti bi li ty</w>': 1, ' e ds</w>': 1, '   ki er se y</w>': 2, '   bo ok sh e lf</w>': 3, '   ro ga ine</w>': 1, '   tur ke ys</w>': 4, '   han d y man</w>': 5, '   s k a tes</w>': 2, '   war med</w>': 8, '   th u mp in</w>': 1, '   pu mp in</w>': 3, '   m c fee ly</w>': 1, '   re mo ves</w>': 4, '   cor n p one</w>': 1, '   f rea kin</w>': 9, ' fuck ed</w>': 8, ' j en ner</w>': 1, '   j en son</w>': 3, '   e my</w>': 1, '   3 5 1 1 </w>': 1, '   su t ph in</w>': 17, '   stu b b ins</w>': 10, ' ci ti z en</w>': 4, '   ti mon i ou m</w>': 1, '   ack er man</w>': 6, '   mi st y</w>': 10, '   j ee e z z z</w>': 1, ' ser i al</w>': 2, '   c u</w>': 3, ' u u u te</w>': 1, '   re win ding</w>': 3, '   j en s en</w>': 4, ' gh o st</w>': 4, ' an ni e</w>': 3, '   ri ps</w>': 4, '   inter f er ing</w>': 13, '   fa w ce t t</w>': 8, '   pri e st ly</w>': 1, '   ca l ver ton</w>': 1, '   sch oo l work</w>': 1, '   men op a use</w>': 2, '   st er n ers</w>': 1, '   tra ge di es</w>': 2, '   de de</w>': 2, '   chi ck a de e</w>': 1, '   oo h h h h h</w>': 1, '   oo h h h h</w>': 2, '   oo oo h h h</w>': 1, '   oo o h h h h</w>': 1, '   oo oo h h h h</w>': 1, ' co o</w>': 3, '   bro c co l i</w>': 2, ' hi ll side</w>': 1, '   st ran g l er</w>': 6, '   t re sp as sing</w>': 8, ' chi cks</w>': 1, '   con tu si ons</w>': 2, '   fr ac tur es</w>': 2, '   do t ti e</w>': 10, '   h in k le</w>': 6, '   d et</w>': 2, '   bra d for d</w>': 3, '   in f le ction</w>': 1, ' dr in king</w>': 1, '   ph on e ca lls</w>': 1, ' wi ll ows</w>': 2, ' c up</w>': 7, ' co ck su ck er</w>': 4, '   tr ac ing</w>': 3, '   4 2 1 5 </w>': 1, '   pi ck les</w>': 3, ' s qu a d</w>': 2, '   oo o h h h h h h</w>': 2, '   oo h h h</w>': 1, '   e d mon son</w>': 1, '   men s room</w>': 1, ' j ac ke ts</w>': 1, '   st oo d up</w>': 1, '   ch om p ing</w>': 1, '   ch ee sing</w>': 1, '   su gar less</w>': 1, '   st er ner</w>': 1, '   pa d ge t t</w>': 2, '   f ab er ge</w>': 2, ' br ow sing</w>': 1, ' mar k et</w>': 4, '   e w w w w</w>': 1, '   o ver pri c ed</w>': 7, '   con s ci en ti ous</w>': 3, '   par ti ci pa tes</w>': 1, '   ac ti ve ly</w>': 3, '   app e t it</w>': 2, '   oo oo h h</w>': 1, '   u s er</w>': 24, '   m ow ed</w>': 1, '   ra dea u</w>': 1, '   j ac qu el ine</w>': 67, '   je un e s se</w>': 1, '   re d i</w>': 21, '   gir t</w>': 1, '   di p s om ani ac </w>': 1, '   ju d d</w>': 10, '   en dan ger</w>': 4, '   im po ses</w>': 1, '   pe ary</w>': 1, ' f ool</w>': 2, '   tra de mar k</w>': 2, '   ber k shi r es</w>': 2, '   ho a g</w>': 4, '   ir ving</w>': 6, '   ro mar is</w>': 1, '   fu r ni sh ed</w>': 5, ' fri gh ten ing</w>': 1, '   un for ge tt able</w>': 3, '   i ll u min a ting</w>': 2, ' e d na</w>': 1, '   mi ll ay</w>': 1, '   ro mar i</w>': 6, ' clo s er</w>': 1, '   mi m i</w>': 20, '   pa ll a di sts</w>': 3, '   c y ran o</w>': 2, '   wh et</w>': 2, '   ad v ent</w>': 3, '   hi gh c li f fe</w>': 1, '   se ar ch li ght</w>': 2, '   v ary</w>': 3, '   go tt sc ha l k</w>': 2, '   ex to ls</w>': 1, ' l ac es</w>': 1, '   re ca l ci tr ant</w>': 2, '   a du la tion</w>': 4, '   re vi e w s</w>': 11, '   pe t uni as</w>': 1, '   pu ll e ts</w>': 1, '   min i ons</w>': 2, '   clo ven</w>': 3, '   di sh ear ten ing</w>': 1, '   di si ll u si on ing</w>': 1, ' qu ar re l ed</w>': 1, '   s co ff ed</w>': 1, '   ho o ted</w>': 1, '   ri gh t ne ss</w>': 2, '   f li pp ant</w>': 2, '   br un s</w>': 2, '   p uni tive</w>': 5, ' an al y st</w>': 1, '   h in ting</w>': 4, '   cha t s wor th</w>': 1, '   fu rs</w>': 6, '   r en ts</w>': 9, '   be lli ss lin a</w>': 1, '   l ow o od</w>': 2, '   v ou ch sa f es</w>': 1, '   de si der are</w>': 1, '   se mp re</w>': 1, '   d i</w>': 27, '   ve d ere</w>': 1, '   c he</w>': 7, '   co s a</w>': 4, ' er a</w>': 1, '   que ll a</w>': 1, '   stan z a</w>': 1, '   du r k</w>': 1, '   doe th</w>': 1, '   jo han n</w>': 1, '   ro z en qu ar t z</w>': 1, '   con si st</w>': 1, '   an a lo g y</w>': 7, '   cra f t s man</w>': 5, '   in hi bi ted</w>': 1, '   al co ho li s m</w>': 3, ' re co ver ing</w>': 1, '   f an ta si z ed</w>': 3, '   mi ll an ey</w>': 8, '   cu l ti v a ting</w>': 2, '   min i ma li st</w>': 1, '   in e f fe c tu al</w>': 3, ' con s ci ous</w>': 7, '   o ver ra ted</w>': 12, '   per ce i ves</w>': 1, '   de fin es</w>': 5, '   re ci ting</w>': 2, '   l y ri c s</w>': 6, '   mon o t one</w>': 2, '   pu r po se fu lly</w>': 1, '   an g lo p hi le</w>': 1, '   un con te sted</w>': 1, '   ab du c ted</w>': 9, '   con co ct</w>': 2, '   ki r k land</w>': 3, '   con je c ture</w>': 3, '   di pped</w>': 6, '   ex t ro ver t</w>': 2, '   uni c y c le</w>': 1, '   ex er t</w>': 4, '   p hi lan th ro p y</w>': 4, '   dis li king</w>': 1, '   ther a pi sts</w>': 4, '   a gen da s</w>': 1, '   han d sha kes</w>': 1, '   cu mu la tive</w>': 1, '   wa ter ed</w>': 9, '   ra don</w>': 2, '   lea k age</w>': 2, '   gre en house</w>': 8, ' con di tion er</w>': 1, '   ma stu r ba ted</w>': 1, '   he si t an ce</w>': 1, '   s l ac ked</w>': 1, '   no t with stan ding</w>': 1, '   hou se work</w>': 1, '   in v ar i ably</w>': 4, '   fa ta li ti es</w>': 5, '   o ver f l ow ing</w>': 4, '   tri g ger ed</w>': 6, '   ta b le clo th</w>': 1, ' st ran ge</w>': 5, '   ear r ing</w>': 6, '   und en i ably</w>': 1, '   de sc ri p tive</w>': 1, '   ir ri ta te</w>': 6, '   v ar i co se</w>': 3, '   ri d g es</w>': 1, '   chi ck en ing</w>': 1, ' la di es</w>': 5, '   u r in a ted</w>': 1, '   pa tri ce</w>': 1, '   pre f er en ces</w>': 3, '   ber ate</w>': 1, '   d ev i ant</w>': 10, ' fa i th ful</w>': 1, '   b la t an tly</w>': 2, '   bar ten ding</w>': 5, ' a g gre ssi ve</w>': 2, '   he si ta ting</w>': 1, '   cha un c ey</w>': 83, '   for man</w>': 1, '   be tt s</w>': 1, '   co sy</w>': 2, '   lu mp s</w>': 5, ' probably</w>': 9, '   c y sts</w>': 1, '   ju gs</w>': 3, '   du mb head</w>': 2, '   ho b b es</w>': 18, ' e et</w>': 1, '   1 2 0 8 </w>': 2, '   fu r ther ed</w>': 1, '   u ro lo g y</w>': 1, '   p sy ch op har m ac o lo g y</w>': 1, '   star lin er</w>': 4, ' ser ge ant</w>': 2, '   an na be ll e</w>': 30, '   for sy the</w>': 4, '   ro llo</w>': 10, '   tu d or</w>': 2, '   do tty</w>': 3, '   h or se fi e ld</w>': 2, '   s win bur n e</w>': 1, '   ve la k of s k y</w>': 1, '   lin s k y</w>': 2, '   le fe b v re</w>': 1, '   com pi le</w>': 3, '   e men th al</w>': 1, '   plea se plea se please</w>': 1, '   j an in e j an in e j an ine</w>': 1, '   wri g g l ed</w>': 1, '   gar a g es</w>': 2, '   sp er ga z z i</w>': 2, '   7 0 3 </w>': 1, ' de gre e</w>': 2, '   ni c co l o</w>': 1, '   ca bi ri a</w>': 1, ' ra ys</w>': 7, '   ra di o lo g y</w>': 1, '   par k ins</w>': 1, '   ro g</w>': 9, '   1 0 0 9 </w>': 1, '   pa th o gen i c</w>': 1, '   ab d om in al</w>': 3, '   gr ow th s</w>': 1, '   sha f ting</w>': 1, '   g p</w>': 2, '   as si mi la ted</w>': 4, '   ci r cu la t ory</w>': 2, '   ki sh k as</w>': 1, ' loo k it</w>': 1, '   par a si to lo gi st</w>': 1, '   v d</w>': 2, '   he m</w>': 4, ' b yes</w>': 2, ' me mor i es</w>': 1, '   v el come</w>': 2, '   fa v ou rs</w>': 3, '   in f an ti le</w>': 3, '   re gre ssion</w>': 2, '   e mi ly</w>': 41, '   we i ss</w>': 11, '   den ton</w>': 7, ' ful</w>': 3, '   shi ver</w>': 3, ' p ee p</w>': 2, '   un ve il</w>': 2, '   f ar le y</w>': 8, ' fa i th</w>': 2, '   so ar</w>': 5, '   de si r ab i li ty</w>': 1, ' spe ci a list s</w>': 2, '   f ea tur ing</w>': 3, '   per en ni al</w>': 2, '   ex ce ll ence</w>': 5, ' en ton v a le</w>': 2, '   en d or ses</w>': 1, '   se l f le ss ne ss</w>': 1, '   den ton vi le</w>': 1, '   e mb r ac ed</w>': 5, '   den ton v a le</w>': 3, '   pu z z l ed</w>': 5, '   pa ve ment</w>': 4, '   en s la ve ment</w>': 1, '   sch ni ck</w>': 1, '   co l er i dge</w>': 1, ' ta tely</w>': 1, '   o ver rea ct</w>': 4, '   con sp ir ac </w>': 1, '   m ac y</w>': 6, '   st ru th ers</w>': 4, '   or p he us</w>': 1, '   f ar fe tch ed</w>': 4, '   r er u n</w>': 1, ' m ck in le y</w>': 1, '   re ti c ence</w>': 2, '   e mo tive</w>': 1, '   man i pu la tive</w>': 3, '   t rea d ment</w>': 1, '   wee per</w>': 1, '   wa il er</w>': 1, '   u r g in</w>': 1, '   bro gu es</w>': 1, '   sle p st r in i</w>': 1, ' lo cal</w>': 2, ' plea s er</w>': 1, '   or p ha n</w>': 19, '   fa v ou ri te</w>': 15, ' win d ow</w>': 2, ' v an ce</w>': 1, '   v an ce</w>': 3, '   ven a</w>': 1, '   ra ver</w>': 1, '   har m ing</w>': 4, '   b ack less</w>': 1, '   rea wa k en</w>': 1, '   f la w less</w>': 4, '   te x t bo ok</w>': 4, '   fe l</w>': 1, '   stu b ble</w>': 2, '   spe ci a li ty</w>': 3, '   pl u m</w>': 5, '   ther a pe u ti c s</w>': 1, ' ho st</w>': 1, '   s om mer s by</w>': 2, ' ca mm i</w>': 1, '   ger t l er</w>': 1, '   fe l d man s</w>': 2, '   s ea ting</w>': 5, '   l ev in</w>': 2, '   gu t less</w>': 8, '   con und ru m</w>': 5, '   k u r t z man</w>': 1, '   may a</w>': 42, '   hi gh lin er</w>': 3, '   bu s lo a d</w>': 1, '   q o od</w>': 1, '   ab so lu te ment</w>': 1, '   n ac i do</w>': 1, '   jo g ger</w>': 3, '   f an ci er</w>': 2, '   tu m or</w>': 12, '   hi tch ing</w>': 6, '   bu e ll ton</w>': 2, '   win d mi ll</w>': 4, '   y ne z</w>': 1, '   sy ra h</w>': 1, '   der e li ct</w>': 5, '   h r n r n r n</w>': 1, '   f lo pp ing</w>': 1, '   tr out</w>': 9, '   so l v an g</w>': 1, '   beli ev ea ble</w>': 1, '   as so ci a ting</w>': 2, '   sa bo ta ging</w>': 3, '   bu k ow s k i</w>': 1, '   th u mb pr int</w>': 2, '   s k y sc ra per</w>': 5, '   s mu dge</w>': 7, '   ex cre ment</w>': 1, '   sur ging</w>': 2, '   se wa ge</w>': 4, '   con fe der ac y</w>': 1, '   d un ces</w>': 1, '   se x ton</w>': 2, '   w oo lf</w>': 2, '   pla th</w>': 4, '   de l more</w>': 3, '   e s says</w>': 4, '   p in o t</w>': 12, '   sch oo l bu s</w>': 1, ' st em</w>': 2, '   se mb lan ce</w>': 2, '   tur pen t ine</w>': 2, '   f ra ss</w>': 1, '   bur g un dy</w>': 7, '   v in ta g es</w>': 1, '   re tr ac ted</w>': 3, '   ab att o ir</w>': 3, '   s la u gh ter house</w>': 6, '   e ss en ti ally</w>': 8, '   pa stu re</w>': 6, '   per si st ence</w>': 2, '   sti ll ne ss</w>': 2, '   ne ga ti vi ty</w>': 1, '   cl in g y</w>': 1, '   ri c he</w>': 1, '   v in e y ard</w>': 4, ' p our er</w>': 1, '   sho ck er</w>': 2, '   r ac h et</w>': 1, '   a gu a</w>': 2, '   v ou v ra ys</w>': 1, '   mer lot</w>': 5, ' a ye</w>': 2, '   ta il sp in</w>': 1, '   p our ers</w>': 2, '   par i ah</w>': 1, ' pi ty</w>': 3, '   pa late</w>': 7, '   differ en ti ate</w>': 1, '   op us</w>': 3, '   ar ti ch ok es</w>': 1, '   ne g</w>': 2, ' ran ch er</w>': 1, '   vi bra tor</w>': 2, '   s ans</w>': 4, '   p in b all</w>': 2, '   oo oo oo o h h</w>': 1, '   ro k u</w>': 1, '   ar men i an</w>': 1, '   pi tt ed</w>': 1, '   ch r is</w>': 57, '   f er men ted</w>': 3, '   so li d ly</w>': 1, '   v ar i e tal</w>': 2, '   sy ru p y</w>': 1, '   in k y</w>': 2, '   p in o ts</w>': 2, '   at y pi ca lly</w>': 1, ' li me</w>': 1, '   gla ze</w>': 1, ' di ll</w>': 1, '   v ar i e tal s</w>': 1, '   char d on n ay</w>': 3, '   ma lo l ac ti c</w>': 1, '   f er men ta tion</w>': 2, ' no tch</w>': 2, '   pro du c ers</w>': 5, '   s an for d</w>': 4, '   s mo o ch ed</w>': 1, '   le x a pr o</w>': 1, ' gra d ers</w>': 1, ' 2 5 0 0</w>': 2, '   im pro ve men ts</w>': 3, '   con g ea l ed</w>': 2, '   h un go ver</w>': 4, '   fir s th and</w>': 3, '   ex tri ca te</w>': 1, '   com mu te</w>': 5, '   bor dea u x</w>': 2, '   pri c ey</w>': 2, '   ca ber ne ts</w>': 1, '   pro sa i c</w>': 2, '   t y po s</w>': 2, '   pro of ed</w>': 2, '   win er y</w>': 1, '   h or ti cu l ture</w>': 2, '   chi pp ing</w>': 3, '   m r n m m</w>': 1, '   n ar ra tive</w>': 4, '   ev o l ves</w>': 1, '   d ev o l ves</w>': 1, '   ro b be</w>': 1, ' g ri ll et</w>': 1, '   su m mar i ze</w>': 2, '   un po ll u ted</w>': 1, '   o ver did</w>': 1, '   sa ssi ca i a</w>': 1, '   ch ev al</w>': 3, '   bl an c</w>': 5, '   sy ra h s</w>': 1, '   c love</w>': 1, '   win e ma k er</w>': 1, '   sa u vi g n on</w>': 2, '   fi d d le head</w>': 1, '   c y c li cal</w>': 1, '   uni ver sa lly</w>': 3, '   cor t land</w>': 1, '   ab ack</w>': 2, '   o li vo s</w>': 1, '   a m bi an ce</w>': 1, '   co tes</w>': 1, '   bea un e</w>': 1, '   t an n ins</w>': 1, '   car y l</w>': 1, '   go l f ing</w>': 2, '   ri che b our g</w>': 2, '   under e sti ma ted</w>': 9, '   ja y er</w>': 1, '   ch ee ses</w>': 2, '   exac te ment</w>': 2, '   f l ab by</w>': 4, '   ca ber n et</w>': 2, '   win er i es</w>': 1, '   ro b les</w>': 4, '   qu af f able</w>': 1, '   tr an sc en dent</w>': 2, '   br un ch</w>': 5, '   su m mer s by</w>': 1, '   a mo e b as</w>': 1, '   i sa be ll e</w>': 4, '   re ci tal</w>': 4, '   co ll e en</w>': 21, '   han d fu ls</w>': 1, '   out l ying</w>': 2, '   coun ti es</w>': 1, '   ev ac u ation</w>': 5, '   e sti ma tes</w>': 4, '   bro ad ca st ing</w>': 8, '   dis lo ca ted</w>': 5, '   re ddy</w>': 1, '   ab er na th y</w>': 1, '   pa s k i</w>': 1, '   bi m bo o</w>': 2, '   per se cu ted</w>': 4, '   ve ge t ar i ans</w>': 3, ' tal ki e</w>': 3, '   s w er ved</w>': 3, '   e d g y</w>': 4, '   pre da tor</w>': 12, '   ex hi bi ting</w>': 2, '   un char ac ter i sti c</w>': 2, '   so l ver s</w>': 1, '   sa d d le ba gs</w>': 1, '   tur le y</w>': 8, '   hi de out</w>': 5, '   l ar u e</w>': 2, '   ry</w>': 2, '   si ms</w>': 2, '   mu r do</w>': 2, '   e m me t t</w>': 14, '   win ging</w>': 1, '   h ow dy</w>': 30, '   au gi e</w>': 4, '   m ck en dri ck</w>': 5, '   sa v an na h</w>': 4, '   ho e</w>': 1, '   si l ver a do</w>': 4, '   pa d en</w>': 14, '   par k ers</w>': 3, '   ha w le y</w>': 2, '   ho b art</w>': 1, ' s wa tting</w>': 1, '   r ar ing</w>': 2, '   f ar m land</w>': 2, '   bri dle</w>': 1, ' wi se</w>': 22, '   chi may o</w>': 2, '   p into</w>': 2, '   ma l ac hi</w>': 1, '   par ce l</w>': 7, '   m ck en dri cks</w>': 1, '   g un s lin g ers</w>': 1, '   en for c ed</w>': 3, '   ad ju st men ts</w>': 7, '   tra ve ll er</w>': 5, '   e z r a</w>': 2, '   sa lo ons</w>': 4, '   al ter a tions</w>': 4, '   t ar ans k y</w>': 30, '   al en o</w>': 2, '   z er o s</w>': 5, '   o sc ars</w>': 4, '   la in ey</w>': 16, ' si m one</w>': 2, ' nu mb ing</w>': 1, '   re fine</w>': 2, '   app re ci a tes</w>': 9, '   pi x el s</w>': 1, '   mo l ded</w>': 1, '   ma the ma ti cal</w>': 6, '   cla w ed</w>': 1, '   fo cu ssed</w>': 1, ' f act</w>': 3, ' op p</w>': 1, ' b y te</w>': 1, '   per man ence</w>': 2, '   o ver bl own</w>': 2, '   te le pro mp ter</w>': 4, ' st un tw om an</w>': 1, '   nu di ty</w>': 6, '   fi l m ma k ers</w>': 1, ' in st ru ment</w>': 1, ' e ter ni ty</w>': 5, '   f on da</w>': 3, '   l or en</w>': 2, '   ke ll ey</w>': 1, '   un kn ow n s</w>': 2, '   re sh oo t</w>': 2, '   ni co l a</w>': 5, '   as si stan ts</w>': 8, '   ban k able</w>': 1, '   r en e w ing</w>': 1, '   i ll u min ate</w>': 6, '   pro je ctor</w>': 5, '   ca s se ve tes</w>': 1, '   dar k en ed</w>': 3, '   no st al gi c</w>': 2, ' fr on t</w>': 3, '   per cen ta g es</w>': 5, '   mo cking</w>': 7, '   su per st ars</w>': 1, '   an d ers</w>': 5, '   sa g</w>': 7, '   in sa ti able</w>': 2, '   sh ying</w>': 1, ' ex po sed</w>': 5, '   li me li ght</w>': 1, '   sin cla ir</w>': 8, '   in de sc ri b able</w>': 3, '   und en i able</w>': 2, '   in op er able</w>': 1, ' stra w</w>': 1, '   t an tru ms</w>': 2, ' v ac t ors</w>': 1, '   sy n the spi an</w>': 1, '   mi c r ow a ves</w>': 1, '   gra pe f ru it</w>': 2, '   bo o ed</w>': 1, '   ke y no te</w>': 1, '   co att a i ls</w>': 3, '   do sto y ev s k y</w>': 2, ' a ged</w>': 8, '   b lu d ge on ed</w>': 2, '   i c on</w>': 5, '   u tt er ed</w>': 2, '   sa y er</w>': 17, '   ab du ction</w>': 8, ' ob ta in ed</w>': 1, ' op er a tive</w>': 1, '   wa tch do g</w>': 3, ' cap tive</w>': 1, ' i sa pp ear ing</w>': 1, '   de mi ll e</w>': 17, '   i do ls</w>': 3, '   w oo d w ard</w>': 13, '   ti p ster</w>': 2, '   che ck bo ok</w>': 10, '   sti lls</w>': 2, '   du sted</w>': 7, '   shu tter</w>': 3, '   ca mp ed</w>': 3, '   con ci er ge</w>': 2, '   co s me ti ca lly</w>': 1, '   w r in k le</w>': 5, '   in cor por a ting</w>': 1, ' su n ri se</w>': 1, '   ci ting</w>': 1, ' c rea tive</w>': 1, '   a ir st rea m</w>': 1, '   con tr ac tually</w>': 1, '   un pro du c ed</w>': 1, '   ex c ee ds</w>': 3, '   se u ss</w>': 2, '   ho ck</w>': 9, '   man u re</w>': 10, '   a mi sh</w>': 9, '   ta m mi si m o</w>': 3, '   mar sc ha l</w>': 3, '   qui er o</w>': 1, '   mor ir</w>': 1, '   wi g ging</w>': 1, '   p sy cho lo gi sts</w>': 4, '   co f fin s</w>': 13, '   k in ne y</w>': 4, '   ani s m</w>': 1, '   ra in b ows</w>': 4, '   imp ri son</w>': 3, '   pro f und is</w>': 1, '   cla m o</w>': 1, '   d om ine</w>': 1, '   pen d ant</w>': 5, ' sh h h h h hu t</w>': 1, '   u pp p p</w>': 1, '   ff ff ff f rea k</w>': 1, '   stu tt er ing</w>': 7, ' ss ss stop</w>': 1, ' ss stop</w>': 1, '   la w ma k ers</w>': 1, '   je thr o</w>': 6, '   m ac d on al ds</w>': 1, ' du el ing</w>': 1, ' ha il</w>': 1, '   or well</w>': 3, '   al ter na tor</w>': 4, '   su z i e</w>': 43, '   un fu fi lled</w>': 1, '   gr ou pi es</w>': 1, '   a a a a a a a a a a a a</w>': 2, ' a x e</w>': 2, ' j ean</w>': 1, ' pe g g y</w>': 3, '   mor ri son</w>': 3, '   f ra tri ci de</w>': 2, ' f ra tri ci de</w>': 1, '   pa tri ci de</w>': 3, '   g un ned</w>': 4, ' pa tri ci de</w>': 1, '   s oun d man</w>': 1, '   car r r r l</w>': 1, '   mo on sh ine</w>': 4, '   mo on sh in ing</w>': 1, ' s la sh</w>': 1, ' ha ck</w>': 1, '   har ve sts</w>': 1, '   th re sh es</w>': 1, '   sp r in k l ed</w>': 4, '   c ow shit</w>': 1, '   a z te c s</w>': 2, '   cu r t</w>': 3, '   co ba in</w>': 1, '   a men ds</w>': 3, '   f ar m han ds</w>': 1, '   tr an si en ts</w>': 1, '   g ran da ddy</w>': 2, '   ir ri ga tion</w>': 3, ' har ve st</w>': 2, '   g ran d da ddy</w>': 1, '   la u re en</w>': 1, '   sh e en a</w>': 1, ' ci vi li z ation</w>': 2, '   l y n ch ed</w>': 4, ' bo b by</w>': 3, '   ma st a</w>': 1, '   k l u</w>': 1, '   k lu x</w>': 1, '   be d sh e et</w>': 1, '   in sen si tive</w>': 8, '   b z z z z z z z z z t</w>': 1, '   bu z z z z z z z z z z</w>': 1, '   a sp h y x i ation</w>': 2, '   ro o st ers</w>': 4, '   por k y</w>': 1, '   sh ow case</w>': 2, '   ir ri ga te</w>': 2, '   sc ra tch es</w>': 9, '   har ve ster</w>': 6, ' sc ar e c row</w>': 1, '   sh e ars</w>': 2, '   th re e some</w>': 2, ' je thr o</w>': 1, ' fee d back</w>': 1, '   he c ti c</w>': 6, '   mo on li gh ting</w>': 3, '   t ar o t</w>': 3, ' war med</w>': 2, '   l y me</w>': 3, '   con te sts</w>': 3, ' qu id</w>': 1, ' s qu id</w>': 3, '   di re</w>': 7, '   f la g g</w>': 1, ' or n in</w>': 1, '   star l a</w>': 10, '   bi sh op vi ll e</w>': 1, '   i bu pro f en</w>': 1, '   re gre tt ed</w>': 6, ' m ou thing</w>': 2, ' \x85 </w>': 62, '   lin k s k i</w>': 1, '   b lu ff s</w>': 2, '   k a le i do s co pe</w>': 1, '   lu r k er</w>': 2, '   sh el by</w>': 13, '   mar ti an</w>': 5, '   ca sta ve ts</w>': 2, ' st or m</w>': 2, '   ran ch es</w>': 1, '   t re v </w>': 2, '   d ou che ba g</w>': 1, '   r or sc ha ch</w>': 16, '   m c ca mm on</w>': 1, '   a fi ci on a do</w>': 1, '   a po lo g</w>': 1, '   m un chi es</w>': 1, ' me x i can</w>': 3, '   co v in g ton</w>': 1, '   me g an</w>': 13, '   ha le sy</w>': 1, '   o s so</w>': 1, '   bu c o</w>': 1, '   su gar pl u m</w>': 2, '   y ea h h h h</w>': 1, '   dis re spe c t ful</w>': 4, '   pro c rea te</w>': 1, '   a mo e b a</w>': 1, '   r h in o</w>': 5, ' p ra i se</w>': 1, '   pro lly</w>': 1, ' di g ger</w>': 3, '   sh an ti es</w>': 1, '   every time</w>': 11, '   bu ell</w>': 1, '   k en wor th</w>': 6, '   be an ers</w>': 1, '   te x ar k an a</w>': 2, '   ro a de o</w>': 2, '   hi gh li ght</w>': 2, '   cer ti fi able</w>': 5, '   l ev is</w>': 1, '   co ors</w>': 4, '   o be ying</w>': 5, '   ra mb l in</w>': 2, '   ho lt</w>': 2, ' po in ts</w>': 3, '   bu sing</w>': 1, '   cle d us</w>': 6, '   tru ck in</w>': 1, ' ar ling</w>': 1, '   re je ction</w>': 8, '   ci a o</w>': 5, '   a st ro do me</w>': 3, ' bri ga do on</w>': 1, '   ma tt o on</w>': 1, ' wh ee l ers</w>': 1, '   fe at</w>': 5, '   m c con n ell</w>': 6, '   wa ter ho le</w>': 4, '   im pu l si ve</w>': 7, ' b id</w>': 1, '   ma tri mon y</w>': 2, '   su n ki st</w>': 1, '   te x an</w>': 2, '   con di men ts</w>': 2, '   ge ar ja mm er</w>': 1, ' wa ll</w>': 6, '   he ar ti est</w>': 1, '   cu cu mber</w>': 2, '   ea tu m</w>': 1, '   pre ci pi ta tion</w>': 1, ' ba ma</w>': 1, '   st rea king</w>': 1, '   ch u r ning</w>': 2, '   ti me wi se</w>': 1, '   vo cal</w>': 5, ' pu ke</w>': 2, '   ti me x</w>': 6, '   bl ack to p</w>': 1, '   wa y ne tt e</w>': 1, '   g or d</w>': 1, '   ge ar ja mm ing</w>': 1, '   ton gu ed</w>': 2, ' ers</w>': 2, ' who pp er</w>': 1, '   jo es</w>': 4, '   s oun d in</w>': 2, '   bo d ac i ous</w>': 1, '   pa s a</w>': 3, '   mo ther fu </w>': 2, '   le ar</w>': 3, '   su m bi tch es</w>': 1, '   do ke</w>': 1, '   r en o v ate</w>': 2, '   wh al es</w>': 40, ' con st ru ction</w>': 1, '   u p keep</w>': 1, '   mo bi li z ed</w>': 1, '   sta ir we lls</w>': 1, '   in di c ta ble</w>': 2, '   comp li an ce</w>': 2, '   me ck l en</w>': 4, '   sp ar a z z a</w>': 22, '   l ore</w>': 2, '   per ce i ved</w>': 3, '   bl ack ba lled</w>': 1, '   sy n on y m ous</w>': 2, '   un real</w>': 9, '   of f en ses</w>': 3, '   l ar cen y</w>': 4, '   e le c tive</w>': 3, '   fi sti cu ff s</w>': 1, ' some th in</w>': 3, '   whi c</w>': 1, '   car ru t</w>': 1, '   t re m or</w>': 3, '   th ass</w>': 3, ' bro th ers</w>': 2, '   h y po ther mi a</w>': 1, '   a m pu te e</w>': 1, '   a mi go</w>': 14, '   dar w in</w>': 8, '   hu go</w>': 18, '   con st ri c ted</w>': 2, '   g an gr en ous</w>': 1, '   in tra ven ous</w>': 5, '   coun ter act</w>': 2, '   se gu e</w>': 2, '   ven tri cu l ar</w>': 1, '   fi bri ll ation</w>': 2, '   ho ss</w>': 6, ' v in ed</w>': 1, '   hi pped</w>': 1, '   k no x</w>': 45, '   s lin g sho t</w>': 3, '   s ni tch es</w>': 2, '   p sy cho ti c s</w>': 2, '   pre lu de</w>': 4, '   t ea m st ers</w>': 4, '   ba ll o t</w>': 3, '   po lling</w>': 1, ' ti e</w>': 2, '   pe er</w>': 6, '   in c in er ate</w>': 4, '   de ter g ent</w>': 3, '   qu a li fi es</w>': 6, ' fin ger na i ls</w>': 1, ' mo ther fucking</w>': 1, '   s kee ze</w>': 1, '   g ou ge</w>': 1, '   mor re y</w>': 1, '   gi f tw ra pped</w>': 1, '   be ani e</w>': 3, '   co op er a ting</w>': 10, '   re gi men ted</w>': 2, '   fa it</w>': 4, '   ac com pl i</w>': 3, ' ha bi ta ting</w>': 1, '   pre mi er</w>': 9, '   o ver look</w>': 13, '   se man ti c s</w>': 6, '   de fa u l ted</w>': 2, '   pre da tes</w>': 1, '   t ro ve</w>': 1, '   1 9 4 0</w>': 2, '   a ma ss ing</w>': 2, ' app al ac hi a</w>': 1, '   a ll e gi an ces</w>': 1, '   re con st ru c tive</w>': 1, '   sp ar ra z a</w>': 1, '   in g st ro m</w>': 1, ' si te</w>': 3, '   la ver n e</w>': 1, '   de bri e f</w>': 4, '   gu l f st rea m</w>': 1, '   je t way</w>': 1, '   loo m</w>': 4, '   dam n de st</w>': 4, '   cu l pe pp er</w>': 1, '   di sp a tch ed</w>': 4, '   du pre e</w>': 1, '   v al ac ch i</w>': 1, '   f ra ti an o</w>': 1, '   gra v an o</w>': 1, '   no st r a</w>': 3, '   ex tra di tion</w>': 7, '   sa l v ad or</w>': 7, '   de t ac h ment</w>': 11, '   cor ro bor a tes</w>': 2, '   ser na</w>': 1, '   pa di c he</w>': 3, '   v ani sh ing</w>': 1, '   bu z zy</w>': 4, '   si t down</w>': 1, ' u st a</w>': 2, '   p un ch l ine</w>': 6, ' a dy</w>': 2, '   n ab </w>': 3, ' spe ci a li st</w>': 1, '   chi ts</w>': 1, '   ea ve s dro pp in</w>': 1, '   ma fuck a</w>': 2, '   s la y in</w>': 1, '   ma fuck as</w>': 2, '   j us</w>': 58, ' a ma ge</w>': 1, '   be e f in</w>': 1, ' sh h h h h h it</w>': 1, '   i f b</w>': 1, '   ear pi e ce</w>': 2, '   pu sh in</w>': 5, '   tri pp in</w>': 6, ' ell</w>': 2, ' al k</w>': 1, '   cha la m ar</w>': 1, '   bab y do ll</w>': 1, ' lan d s li de</w>': 1, ' li l</w>': 1, '   k ev l ar</w>': 3, '   la y in</w>': 17, ' r y in</w>': 1, '   re s cu in</w>': 1, '   ra m pa g in</w>': 1, '   han d lo a ds</w>': 1, ' n in a</w>': 1, ' brea k</w>': 5, '   gu lly</w>': 3, '   sp in n in</w>': 1, '   fee d back</w>': 3, ' li b b ing</w>': 1, '   ma fuck</w>': 1, '   ho tt est</w>': 6, ' f la v or</w>': 1, '   tri a d</w>': 3, '   tr you t</w>': 3, '   bu ll ll l shit</w>': 1, '   pu pp e ts</w>': 10, '   gi bar i an</w>': 12, '   r he ya</w>': 13, '   di sin te gra tes</w>': 1, '   s ar t ori us</w>': 6, '   su ba tom i c</w>': 3, '   ne u tr in o s</w>': 1, '   ne u tr in o</w>': 2, '   in car na tion</w>': 1, '   imp lo de</w>': 3, '   so li di f ying</w>': 1, ' sta ke</w>': 1, '   de st ab i li z er</w>': 1, '   rea pp ear ed</w>': 1, '   mo di fi ca tion</w>': 2, '   so l ar is</w>': 8, '   tr ans for ma tion</w>': 5, '   di sin te gra te</w>': 2, '   tri lli ons</w>': 1, '   der an ge ment</w>': 1, '   a th en a</w>': 2, '   e qui vo ca ting</w>': 2, '   e qui vo ca te</w>': 4, '   un n er ved</w>': 1, '   rea pp ear an ces</w>': 1, '   ma ter i a li z ed</w>': 1, '   ex ce ed</w>': 4, '   re de mp tion</w>': 5, '   fu l fi lls</w>': 1, '   un sp ok en</w>': 2, '   un bl in king</w>': 1, '   gen e ti c s</w>': 3, '   un c ro ss ing</w>': 1, ' ob se ssed</w>': 3, '   pe de st al</w>': 2, '   as c ri b ing</w>': 1, ' in tu i tive</w>': 1, '   un le ar n</w>': 3, '   bea t les</w>': 2, '   d om in i on</w>': 4, '   di st in c tions</w>': 1, '   re su l ted</w>': 4, '   man i fe sta tions</w>': 1, '   cor re sp on ds</w>': 2, '   re co il</w>': 4, '   comp en sa ting</w>': 2, '   au ton om ous</w>': 2, '   ma ter i a li z ing</w>': 1, ' so li d</w>': 1, '   con sti pa ted</w>': 2, '   st e in har t</w>': 1, '   s ki ddy</w>': 1, '   ven z a</w>': 11, '   gar b er</w>': 6, ' mar ch ing</w>': 1, '   mu g ger</w>': 2, '   my op i c</w>': 1, '   r en o ir</w>': 5, '   u r d u</w>': 1, '   h in d i</w>': 1, '   sp ri t z er</w>': 1, '   a per i ti f</w>': 1, '   ber g d or f</w>': 3, '   sen i ori ty</w>': 5, '   pa i sle y</w>': 6, '   i den ti fi es</w>': 6, '   sh i</w>': 11, '   per p</w>': 4, '   bab y si tting</w>': 6, '   d t</w>': 1, '   s k a te bo ard</w>': 2, '   dis cou ra ge</w>': 5, '   kee g an</w>': 2, ' se ven te en</w>': 3, '   k o on t z</w>': 4, '   nu r se ma id</w>': 5, ' cho ice</w>': 1, '   er e ct</w>': 5, ' b on er</w>': 1, ' ba g</w>': 7, ' sin ging</w>': 3, ' pl ans</w>': 2, '   s k a te bo ar ds</w>': 1, '   en se mb le</w>': 3, '   mi l k y</w>': 2, '   ca ho o ts</w>': 2, '   fee ble</w>': 4, '   min ded</w>': 9, '   lu mp</w>': 5, '   du d</w>': 4, '   li z zy</w>': 6, '   ro s om off</w>': 3, ' ber men s ch</w>': 1, '   be ck ons</w>': 3, '   sch o l ar shi ps</w>': 3, '   uni ver si ti es</w>': 1, '   fe ti d</w>': 1, ' sch o l ar ship</w>': 1, ' har ry</w>': 5, '   di min i sh es</w>': 1, '   ho g an</w>': 4, '   bu g face</w>': 1, '   e d ge wi se</w>': 1, '   sa u c ers</w>': 1, ' b om b s</w>': 2, '   ja me son</w>': 9, '   app li an ces</w>': 5, '   bi g f oo t</w>': 3, '   ger bi l</w>': 2, ' ex po sur es</w>': 2, '   pa th o s</w>': 2, '   da l i</w>': 3, '   ar t sy</w>': 2, ' f ar t sy</w>': 1, '   a m bu l an</w>': 1, '   h m f</w>': 1, ' h r r</w>': 1, '   un stu ck</w>': 1, '   fr m pp h</w>': 1, ' y r r</w>': 1, '   ni cen ess</w>': 2, '   co ck ar o ac h</w>': 1, '   s k in ti ght</w>': 1, '   s n ack</w>': 8, '   hou se f ly</w>': 2, '   sho el ac es</w>': 6, '   f la ge ll ate</w>': 1, ' pe ter</w>': 2, '   mi s qu o te</w>': 2, '   o the llo</w>': 4, '   br r r</w>': 1, ' f la sh</w>': 5, '   ri pp le</w>': 2, '   t re ed</w>': 2, '   k af k a</w>': 13, '   al li ter a tive</w>': 1, ' lo a the</w>': 1, '   wal do s</w>': 3, '   ro z</w>': 4, '   c y clo tr on</w>': 4, '   k a put</w>': 3, '   o c ta vi us</w>': 9, '   s l</w>': 2, '   tri an gu la ted</w>': 1, '   ri pp ling</w>': 1, '   inter re la tion</w>': 1, '   e le c t ro ma g ne ti s m</w>': 1, '   ch ar</w>': 1, ' bro il ed</w>': 1, '   o ck</w>': 3, '   th or k el</w>': 4, '   na pp ing</w>': 1, '   du ll ard</w>': 1, '   o s bor n</w>': 1, '   to fu </w>': 2, '   i ck</w>': 2, '   ro ac h es</w>': 2, '   z it</w>': 3, '   me d dle</w>': 1, '   5 6 1 </w>': 1, ' 5 1 5 1 </w>': 1, '   pro por tion al</w>': 1, '   a gi li ty</w>': 2, '   ha z ar ds</w>': 1, '   al u m n i</w>': 2, '   so ph om ori c</w>': 2, '   e qu a tions</w>': 8, '   o t to</w>': 20, '   har mon i es</w>': 1, '   im per fe ct</w>': 4, '   ex per i en ti al</w>': 1, '   k al u z a</w>': 1, ' k le in</w>': 1, '   in t ro du c t ory</w>': 1, '   a do le sc en ts</w>': 2, ' spi der</w>': 1, '   for t ean</w>': 1, '   ph en om en a</w>': 4, '   an o ma li es</w>': 7, '   a st r on om y</w>': 2, '   re i ss</w>': 4, '   s ar d i</w>': 1, '   do ck y</w>': 1, '   o ck y</w>': 1, '   lo on y t un es</w>': 1, '   b ha ga v a d</w>': 1, '   gi ta</w>': 2, '   me ssi ani c</w>': 1, '   e go ti sti cal</w>': 5, '   h er e di ty</w>': 1, '   im per t in ent</w>': 4, '   4 9 </w>': 3, '   sto ck pi le</w>': 4, '   ten t ac les</w>': 7, '   z al u te</w>': 1, '   fe l d we be l</w>': 1, '   z u</w>': 2, '   sch u l z</w>': 23, '   ja w o h l</w>': 6, '   a do l f s</w>': 1, '   ho ff y</w>': 8, '   v oo m</w>': 2, '   k om man d ant</w>': 7, '   bar bed</w>': 5, '   se f ton</w>': 21, '   mi t t</w>': 3, '   sta la g</w>': 1, '   s qu ea ling</w>': 5, '   st oo lie</w>': 8, '   k an gar o o</w>': 4, '   h or sing</w>': 1, '   di sti ll er y</w>': 3, '   h or ser ac es</w>': 1, '   pa tt on</w>': 1, '   stu pe</w>': 2, '   sh ar per</w>': 8, '   f oo t lo ck ers</w>': 2, '   co ll ab or a tor</w>': 3, '   man f re d i</w>': 5, '   be l ch</w>': 5, '   bu b</w>': 3, '   vi gi lan te</w>': 3, '   ke i ster</w>': 2, '   a st ori a</w>': 1, '   ci r cu la ting</w>': 2, '   li fe be lt</w>': 1, '   gla m or</w>': 2, '   n an tu ck et</w>': 2, '   sch er b ac h</w>': 3, '   sch er</w>': 1, ' b ac h</w>': 1, '   p fe f fin ger</w>': 1, '   car lo a ds</w>': 1, '   m uni tions</w>': 2, '   ar re st ing</w>': 8, '   ab fu e h r en</w>': 1, '   in s om ni a</w>': 15, '   ac com mo da tions</w>': 1, '   or din ar i ly</w>': 11, '   1 0 5 </w>': 2, ' 3 5 3 </w>': 1, '   mo th</w>': 11, '   sa bo te u r</w>': 2, '   ge mu e t li ch</w>': 1, '   ra us</w>': 3, '   dro pp en</w>': 5, '   m it</w>': 10, '   of en</w>': 1, '   ba v ar i a</w>': 1, '   wi se c r ack ers</w>': 2, '   sp re ch en</w>': 4, '   de u t s ch</w>': 4, '   au f st e h en</w>': 1, '   sha pi r o</w>': 8, '   gr able</w>': 10, '   du ran te</w>': 1, '   g able</w>': 4, '   su b mi tt ed</w>': 3, '   ev a der</w>': 1, '   pl y m ou th</w>': 5, '   gi m mi ck</w>': 4, '   sch ni ck el fri t z</w>': 2, '   e qui po i se</w>': 3, '   gr r r r rea te st</w>': 1, '   w under b ar</w>': 4, ' bur ner</w>': 1, '   j er k o</w>': 3, ' con gra tu la tions</w>': 4, '   ga lo sh es</w>': 2, '   o tch i</w>': 2, '   tch or ni ya</w>': 2, '   ru ss k i</w>': 3, '   bu b li ch k is</w>': 1, '   k o</w>': 1, '   ca m ou f la ging</w>': 1, '   sh ort ca ke</w>': 1, '   ge fi ll te</w>': 1, '   g ri d dle</w>': 1, '   bar ack en</w>': 1, ' fu e h r er</w>': 1, '   la tr ine</w>': 5, '   tr en ch es</w>': 1, '   h or se play</w>': 1, '   b b c</w>': 5, '   sp la tt er ed</w>': 6, '   k ran k</w>': 1, '   du mm k op f</w>': 1, '   pre i s ma i er</w>': 2, '   pre i s sin ger</w>': 1, '   fa ther land</w>': 2, '   bu n di st</w>': 1, '   st oo li es</w>': 2, '   k u z a w a</w>': 2, '   pi re ll i</w>': 1, '   ce s sp ool</w>': 3, '   ru do l p h</w>': 1, '   cu sh in g ha m</w>': 1, '   den o tta</w>': 1, '   co h en</w>': 5, '   sc re en ed</w>': 2, '   p ing</w>': 6, ' p ong</w>': 2, '   ch el ve st on</w>': 1, '   bo o b</w>': 4, '   e in f ac h</w>': 1, '   st re i ch ho el z er</w>': 1, '   e ine</w>': 1, '   z i gar e tt e</w>': 1, '   wi e</w>': 2, '   ge m ac h t</w>': 1, '   a ll es</w>': 4, '   i st</w>': 8, '   h er au s ge f und en</w>': 1, '   ge wi ss</w>': 1, '   ma tt re ss es</w>': 2, '   nu mm er</w>': 2, '   e in und si e b z i g</w>': 1, '   dre i und si e b z i g</w>': 1, '   bar ac ke</w>': 1, '   vi er</w>': 1, '   li e b er</w>': 1, '   go t t</w>': 1, '   i ll e gi ti ma te</w>': 4, '   ga u le i ter</w>': 1, '   z in z in na t i</w>': 1, '   sch u l z es</w>': 1, ' mu ff s</w>': 1, '   g y m na sti c</w>': 1, '   sho v el s</w>': 4, '   un di g</w>': 1, '   di g ged</w>': 1, '   g lo ck en spi el s</w>': 1, '   stan i s la us</w>': 1, '   g un ner</w>': 4, '   dis ru p t ors</w>': 5, '   star f le et</w>': 84, '   ho s</w>': 1, '   q or d u</w>': 1, '   to h</w>': 1, ' pa k</w>': 1, '   su l u</w>': 57, '   chri st en ing</w>': 3, ' el li p ti cal</w>': 1, '   or bi ts</w>': 2, ' en try</w>': 4, '   sor an</w>': 11, ' li fe for ms</w>': 1, '   li fe for ms</w>': 5, '   ver i di an</w>': 6, '   un in ha bi ted</w>': 3, '   a mar go s a</w>': 6, '   bo z e man</w>': 3, '   e mi ssi ons</w>': 2, ' 0 5 </w>': 2, '   star ship</w>': 27, '   g ori k</w>': 1, '   ha l ted</w>': 1, '   de c rea sed</w>': 1, '   ge or d i</w>': 11, '   o ver wh el med</w>': 10, '   a st r o</w>': 4, '   gu in an</w>': 9, '   con f lu x</w>': 1, '   te mp or al</w>': 10, '   3 9 </w>': 4, '   po si tr on i c</w>': 6, '   ma g ne ti ca lly</w>': 2, '   f ar point</w>': 2, '   dea c ti v ate</w>': 5, '   li fe for m</w>': 5, '   im pa s se</w>': 4, '   en dea v or ed</w>': 1, '   si ck b ay</w>': 2, '   r er ou ted</w>': 2, '   i on i c</w>': 1, '   clo a king</w>': 7, ' pre o c cu pi ed</w>': 2, '   comp le ment</w>': 4, '   bu ck ling</w>': 3, '   n ac e ll e</w>': 3, '   for ce fi el ds</w>': 3, ' al ph a</w>': 1, '   bea med</w>': 10, '   la k u l</w>': 2, '   tr an sp ort ers</w>': 8, ' e ch o</w>': 1, ' lu c</w>': 10, ' ne x us</w>': 1, '   gra vi me tri c</w>': 2, '   gen er a ting</w>': 4, '   su b sp ace</w>': 7, '   sp ac e do ck</w>': 8, '   sp o ck</w>': 130, '   i ll o gi cal</w>': 6, '   an ton i a</w>': 5, '   le c tu red</w>': 6, ' cen tu ry</w>': 3, '   de f le ctor</w>': 10, '   ba ys</w>': 4, '   p ho ton</w>': 12, ' matter</w>': 8, '   dis ru pt</w>': 8, '   p ha sing</w>': 3, '   pi c ard</w>': 28, '   on set</w>': 1, ' pro be</w>': 2, '   b loo d st rea m</w>': 7, '   na vi ga ting</w>': 2, '   car di o v as cu l ar</w>': 3, '   bor g</w>': 31, ' inter ro ga tor</w>': 1, '   tri li th i u m</w>': 7, '   in e le g ant</w>': 1, '   pro s the s is</w>': 1, '   k lin g on</w>': 31, '   f la g ship</w>': 1, '   nu cle ar fu sion</w>': 1, '   ro mu l an</w>': 21, '   de an na</w>': 14, '   sh ou l der ed</w>': 1, '   pi car ds</w>': 2, '   tra f al g ar</w>': 1, '   un re so l ved</w>': 2, '   con f li c ts</w>': 2, '   in put</w>': 12, ' ad mi ra l</w>': 5, '   lu s by</w>': 1, '   ir ri ta ble</w>': 5, '   star ba se</w>': 1, '   be ta z o id</w>': 2, '   e m pa th</w>': 2, '   gi ga wa t t</w>': 1, '   for ce fi e ld</w>': 2, '   ou </w>': 3, '   le an dr a</w>': 1, '   dis su a de</w>': 3, '   wor f</w>': 30, '   t or pe does</w>': 20, ' lo ck</w>': 10, '   p ha s ers</w>': 19, '   pen e tra te</w>': 11, '   lu r s a</w>': 1, ' e tor</w>': 1, '   ki d na ps</w>': 3, '   be lon ging</w>': 13, '   du ra s</w>': 1, '   sen s or</w>': 10, '   ro mu l ans</w>': 9, '   tri cor der</w>': 5, '   lo gs</w>': 6, '   n ar r ows</w>': 3, '   b re en</w>': 1, '   y ar dar m</w>': 2, ' stu ds</w>': 2, '   tr ans war p</w>': 2, '   ma in ta in ing</w>': 3, '   hel m s man</w>': 3, ' qu ar ter</w>': 4, '   au to ma tes</w>': 1, '   sp ee ds</w>': 2, '   che k o v </w>': 40, '   g ri s so m</w>': 16, '   e st e b an</w>': 3, '   ex c el si or</w>': 9, '   k o ba y a sh i</w>': 9, '   mar u</w>': 6, '   g al ac ti c</w>': 4, '   su b se qu ent</w>': 3, '   sa a vi k</w>': 46, '   p ha s er</w>': 12, '   p on</w>': 1, '   f ar r</w>': 1, '   pro to matter</w>': 3, '   sur g es</w>': 2, '   im pa ti ence</w>': 4, '   den oun c ed</w>': 1, '                   </w>': 2, '   re f er en c ed</w>': 3, '   uni den ti fi able</w>': 1, ' sh ouldn</w>': 3, '   in fr ar ed</w>': 12, '   en han ce ment</w>': 4, '   ver i f ying</w>': 2, '   tri min i u m</w>': 1, '   com men c ing</w>': 3, '   c y lin dri cal</w>': 1, '   v ar i e ti es</w>': 4, ' t ro pi cal</w>': 1, '   ve ge ta tion</w>': 3, '   de c rea sing</w>': 2, '   re cor d ers</w>': 1, '   c el si us</w>': 1, '   ex t ro ad in ary</w>': 1, '   con cu r</w>': 3, '   bea m ing</w>': 5, '   au to ma tion</w>': 3, '   din na</w>': 2, '   pre ca u tion ary</w>': 2, ' do ors</w>': 4, '   chi mp an z e e</w>': 2, '   tra in e es</w>': 2, '   fa l</w>': 1, '   tor</w>': 14, '   re fu sion</w>': 1, '   el d ers</w>': 15, '   k a tr a</w>': 3, '   re t ro th ru st ers</w>': 1, '   s ar e k</w>': 5, '   shi m mer ing</w>': 2, '   mu t ar a</w>': 2, '   e sti ma ting</w>': 8, '   sp ac e do ors</w>': 1, '   de com mi ssi on ed</w>': 1, ' app ro ac h</w>': 1, '   u hur a</w>': 27, '   com men da tions</w>': 2, '   in di sp osed</w>': 1, '   ex hi l ar a ting</w>': 5, '   ma l t z</w>': 1, '   se la ya</w>': 1, '   me l ded</w>': 1, ' me l ded</w>': 1, '   pro sp er</w>': 3, '   ou tw ei gh</w>': 4, ' m et</w>': 2, '   k lin gon s</w>': 12, '   out gu n</w>': 1, '   clo a ked</w>': 10, '   k el li ca ms</w>': 4, '   inter ru p tions</w>': 2, '   t or g</w>': 1, ' plan et</w>': 2, '   e mi ss ar i es</w>': 2, ' p ea ce</w>': 2, '   pre ser v ation</w>': 5, '   d om in ate</w>': 4, '   bo l der</w>': 2, ' clo set</w>': 2, '   ri g or</w>': 2, '   ga ll ey</w>': 8, ' ar ran ge</w>': 1, '   ho pp ing</w>': 5, '   bi r th days</w>': 3, '   re t la x</w>': 2, '   f le x i bi li ty</w>': 2, '   an ti qu es</w>': 10, '   f er ment</w>': 1, '   pr ing</w>': 1, '   th y self</w>': 5, '   re li ant</w>': 20, '   re gu l a</w>': 16, '   ce t i</w>': 9, '   v i</w>': 10, '   u ss</w>': 6, '   wi l d ly</w>': 4, '   l ab or a</w>': 1, '   li fe less</w>': 8, '   si mu la tion</w>': 15, ' leave</w>': 8, ' ver s a</w>': 1, '   t ea m ing</w>': 1, '   w rea th</w>': 2, '   su ra k</w>': 1, ' bo t any</w>': 3, '   ss s sh</w>': 4, ' pi c</w>': 1, '   pre ani ma te</w>': 1, '   d y no</w>': 1, '   b en i g n</w>': 5, '   shi p ma te</w>': 1, '   m c gi ver</w>': 1, '   re qui ted</w>': 1, '   gen e ti ca lly</w>': 8, '   en ab l ed</w>': 2, '   pro me the us</w>': 1, '   pa tt ed</w>': 1, '   in gen u i ty</w>': 3, '   tr an sp or ter</w>': 14, '   in op er a tive</w>': 10, '   man u al s</w>': 1, ' gen e s is</w>': 1, '   no on i an</w>': 1, '   de pri ve</w>': 4, ' en ter pri se</w>': 1, '   1 9 9 6 </w>': 1, '   c r y o gen i c</w>': 1, '   ar i s en</w>': 1, '   1 8 0 0</w>': 3, '   di sc re te</w>': 2, '   au g ment</w>': 1, '   ma ins</w>': 6, '   en er gi z er</w>': 2, '   b y pa ssed</w>': 1, '   mi d shi p man</w>': 3, '   pre st on</w>': 3, '   ri b b ing</w>': 1, '   shi p sha pe</w>': 2, ' k o ba y a sh i</w>': 4, '   sp or a di c</w>': 2, '   1 5 3 </w>': 1, '   con si sts</w>': 3, '   or es</w>': 1, '   ti ber i an</w>': 2, '   plan to id</w>': 2, '   pre fi x</w>': 3, '   1 6 3 0 9 </w>': 1, '   e le c tr on i ca lly</w>': 2, '   en er gi ze</w>': 4, '   gar bl ed</w>': 2, '   pre su mp tion</w>': 4, '   bo at lo a d</w>': 2, '   h ru mm m</w>': 1, '   ad mi x ture</w>': 1, '   pro t</w>': 1, ' k o ba y sh i</w>': 1, '   w rea ks</w>': 1, '   under stan d ably</w>': 2, '   lo i ter ing</w>': 5, '   com m pi c</w>': 1, '   de sc end</w>': 3, '   un co ded</w>': 1, '   tr ans mi ssi ons</w>': 8, '   im mo bi li z ed</w>': 1, '   com men da tion</w>': 2, '   a ir less</w>': 1, ' ex pre ssion</w>': 1, '   can di d ly</w>': 1, '   no st al gi a</w>': 7, '   h up</w>': 1, '   par ab o li c</w>': 2, '   est</w>': 7, '   pi lo ted</w>': 2, '   to l er an ce</w>': 9, '   f la w ed</w>': 5, '   a x i om</w>': 1, '   re tra in ing</w>': 1, '   gi lli an</w>': 6, '   ra vi sh</w>': 1, '   p le x i c o</w>': 1, '   ni cho ls</w>': 2, '   pa ve l</w>': 7, '   men in g ea l</w>': 1, '   hu mp back</w>': 9, '   com pu ta tions</w>': 2, '   mar sha l ed</w>': 2, '   b ori te</w>': 1, ' gu e ss es</w>': 1, ' gu e ss ing</w>': 1, '   ac c el er ation</w>': 2, '   re f er ent</w>': 1, '   co e f fi ci ent</w>': 1, '   e la p sed</w>': 4, '   ha m let</w>': 10, '   hu mp b ac ks</w>': 9, '   fun do s co pi c</w>': 2, '   un re v ea ling</w>': 1, '   re spi ra t ory</w>': 4, '   ob ser ver s</w>': 2, '   imp ac ting</w>': 1, '   a mp li fi ca tion</w>': 1, '   w er y</w>': 1, '   v ar n</w>': 1, '   vi ll</w>': 3, '   6 5 6 </w>': 1, ' 5 8 2 7 </w>': 1, '   au x</w>': 1, '   we s se ls</w>': 1, '   we s se l</w>': 2, '   mo d es</w>': 2, ' ne u tra li z ed</w>': 1, '   qu ad ran ts</w>': 1, '   star shi ps</w>': 4, '   sh e par d</w>': 3, '   4 0 1 </w>': 3, '   me ga h er t z</w>': 2, ' f re qu en cy</w>': 1, '   he ll o v a</w>': 1, '   gr ac i e</w>': 10, '   per si st ent</w>': 7, '   dis gu i ses</w>': 6, '   re po pu late</w>': 1, '   di t zy</w>': 1, '   sa u sa li to</w>': 4, '   ce t ac ean</w>': 2, '   mi che lo b</w>': 1, '   lan d lu b b er</w>': 1, '   l ds</w>': 2, '   ki d di es</w>': 6, '   ca l ves</w>': 3, '   sy n the ti ca lly</w>': 1, ' thr own</w>': 2, '   har po ons</w>': 1, ' sin k</w>': 2, ' fun c tion al</w>': 2, '   bra king</w>': 3, ' mi ssion</w>': 1, '   prob ab i li ti es</w>': 3, '   me ld</w>': 2, '   fe ll ah</w>': 3, '   ro b b ins</w>': 4, ' d ouble</w>': 4, '   p ho t ons</w>': 3, '   di li th i u m</w>': 6, '   c r y sta ll ine</w>': 1, '   re st ru c ture</w>': 1, '   a v a il ab i li ty</w>': 3, '   ba s in</w>': 4, '   in di gen ous</w>': 7, '   mo di f y</w>': 2, '   sa lin i ty</w>': 1, '   ho ver</w>': 2, ' clo a king</w>': 1, '   de sc en ding</w>': 1, '   m s l</w>': 1, '   0 1 0</w>': 1, '   3 2 8 </w>': 1, '   3 2 7 </w>': 2, '   2 8 3 </w>': 1, '   ori en ta ted</w>': 1, '   ter r a</w>': 1, '   in co g ni ta</w>': 1, '   for gone</w>': 1, ' ter re st ri al</w>': 3, '   o ver la pp ing</w>': 1, '   pre ju dge</w>': 1, '   bea sti es</w>': 1, ' en er gi ze</w>': 1, '   s n it</w>': 1, ' c r y st al li z ed</w>': 1, ' c r y st al li z ing</w>': 1, '   hu mp b ac ked</w>': 1, '   se qu en c er</w>': 2, '   v u l c ans</w>': 7, '   de ton a ted</w>': 5, '   tr an sp ar ent</w>': 7, '   p le x i gla ss</w>': 1, '   with stand</w>': 7, '   cu bi c</w>': 2, '   si ll</w>': 2, '   po l y m ers</w>': 1, '   2 0 5 </w>': 1, '   in cor re ct</w>': 3, '   ar i ga to</w>': 1, '   sa y on ar a</w>': 1, '   sor en ar a</w>': 1, '   ma z u</w>': 2, '   na ga i k i</w>': 1, '   s ar er u</w>': 1, '   mi r a</w>': 1, '   hi k ar u</w>': 1, '   cho t to</w>': 1, '   om ac hi</w>': 1, '   na s ar e i</w>': 2, '   na ma e</w>': 1, '   w a</w>': 7, '   n an to</w>': 1, '   mo o s ar er u k a</w>': 1, '   chi ga a u</w>': 1, '   hi to</w>': 2, '   han a sh i</w>': 1, '   k at a</w>': 1, '   ok as hi i</w>': 1, '   go men</w>': 1, '   chi ga i</w>': 1, '   go z ar an u k a</w>': 1, '   o j i cha n</w>': 1, '   a ki r a</w>': 1, '   o j i cha an</w>': 1, '   de w a</w>': 1, '   na in o</w>': 1, '   k ok o</w>': 6, '   n an i</w>': 1, '   shi ter u</w>': 1, ' mi gh ty</w>': 1, '   ru </w>': 5, ' af o</w>': 4, '   e s ca la ted</w>': 3, '   par ri ci de</w>': 1, '   r o</w>': 9, '   ca ver n</w>': 8, '   ja k</w>': 2, ' a h l a</w>': 2, '   in hi bi t ors</w>': 2, '   con cen tra tions</w>': 2, '   of f lan der</w>': 2, '   ar ti s ans</w>': 1, '   bl ac ke st</w>': 2, '   re lo ca tion</w>': 3, '   of f lan d ers</w>': 2, '   of f land</w>': 1, '   me ta p ha si c</w>': 7, '   re gen er a tes</w>': 1, '   ab du ct</w>': 3, '   ho lo de ck</w>': 1, '   tr an sp or ted</w>': 2, '   re tr ace</w>': 1, ' ar ti fi ci al</w>': 1, ' inter f er ence</w>': 1, '   t our ne l</w>': 1, '   ani j </w>': 1, '   r out in es</w>': 3, '   be d times</w>': 1, '   cen ti me t ers</w>': 5, ' lo ca ted</w>': 1, ' pre ssed</w>': 1, '   en d or ph in</w>': 1, '   me d s can</w>': 1, ' ho sta g es</w>': 1, '   che st nu t</w>': 1, ' ye w</w>': 2, ' che en</w>': 4, ' fa w</w>': 4, '   ca l ci te</w>': 1, ' k u</w>': 10, '   k el b on i te</w>': 2, '   per son i fi ca tion</w>': 2, '   ph y si ome tri c</w>': 1, '   po pu la ted</w>': 2, '   ri k er</w>': 10, '   car da ssi ans</w>': 1, '   de f en se less</w>': 7, '   u p gra ded</w>': 2, '   mar ti al ed</w>': 2, '   bri ar</w>': 4, '   ther mo l y ti c</w>': 2, '   un li v able</w>': 1, '   me ta p ha si c s</w>': 1, '   li fe sp ans</w>': 1, '   ev o l ve</w>': 9, '   re gen er a tive</w>': 1, '   ch ro mo d y na mi c</w>': 1, ' bri ar</w>': 1, '   d ou gh er ty</w>': 5, '   re in for ce men ts</w>': 11, '   s che ma ti c s</w>': 6, '   pre ci pi ta ted</w>': 1, '   li fe sig n s</w>': 1, '   di sa ble</w>': 7, '   in je ctor</w>': 5, '   be gs</w>': 1, '   un sp ea k able</w>': 2, '   i so lin e ar</w>': 2, '   per for ms</w>': 2, '   r oun ded</w>': 5, '   ho l o</w>': 2, '   ga ll at in</w>': 1, '   o cu l ar</w>': 1, '   su br out in es</w>': 2, '   ma l fun c tion ed</w>': 1, '   en gra ms</w>': 4, ' mi c r on</w>': 1, '   t or qu e</w>': 4, '   mi c r ons</w>': 1, '   ds</w>': 2, '   g or en</w>': 1, '   de ton a ting</w>': 1, '   ca s ca de</w>': 1, '   man i fo l ds</w>': 1, '   man i f old</w>': 2, '   ra m s co o p</w>': 2, '   n ar a</w>': 1, '   i so l y ti c</w>': 1, '   p ow er ing</w>': 1, '   k hi tom er</w>': 3, '   ac cor d</w>': 2, '   op s</w>': 4, '   c ome t ary</w>': 1, '   me t re on</w>': 1, ' en vi r on men tal</w>': 2, '   a st ro me tri c</w>': 2, '   g or gon z o ll a</w>': 1, '   ar cha e o lo gi cal</w>': 1, '   han or an</w>': 1, '   me di ate</w>': 1, '   t ro i</w>': 4, '   so je f</w>': 1, '   h y po sp ra y</w>': 1, '   le c tra z ine</w>': 1, '   de clo a k</w>': 1, ' sy st e ms</w>': 2, '   ra tions</w>': 2, '   ge o</w>': 1, '   g or ch</w>': 2, '   de bri e fin gs</w>': 1, '   da mp ing</w>': 1, '   cou pl ing</w>': 4, '   com po s ers</w>': 2, '   n in e te en th</w>': 8, '   p in af ore</w>': 1, '   en er ge ti c</w>': 2, '   di c ta t ori al</w>': 1, '   i on o sp h er i c</w>': 1, '   in hi bi tor</w>': 2, '   re se tting</w>': 3, '   har mon i c s</w>': 2, '   t ac h y on</w>': 2, ' v ar i ant</w>': 1, '   ac tu ation</w>': 1, '   ser vo s</w>': 1, '   b al sa mi c</w>': 1, '   v in a i gre tt e</w>': 1, '   ch r y s an the mu ms</w>': 1, '   ye w</w>': 3, '   au gh</w>': 2, '   po stu r es</w>': 2, '   pro c rea ting</w>': 1, '   no ma di c</w>': 1, '   me tal s</w>': 5, '   so ci o lo g y</w>': 1, '   ma l fun c tion ing</w>': 3, '   si mu la tions</w>': 3, '   list s</w>': 8, '   al m ack</w>': 1, '   f ra g men ts</w>': 11, '   s ca l pe ls</w>': 1, '   si l o</w>': 10, '   co ch ran e</w>': 10, '   com ba d g es</w>': 1, '   ra di ome tri c</w>': 1, '   z e ph ra m</w>': 5, '   f lu c tu a ting</w>': 2, '   re spi ra tor</w>': 3, '   con figu re</w>': 1, '   con figu red</w>': 1, '   dea c ti v a ted</w>': 2, '   in pu ts</w>': 1, '   en do</w>': 1, ' s ke le tal</w>': 1, '   sen sa tions</w>': 4, '   2 0 6 3 </w>': 2, '   co ll a p sing</w>': 4, '   ter r an</w>': 2, '   di sp er si ve</w>': 1, '   cre w men</w>': 2, '   t in k er ing</w>': 6, ' a p r</w>': 1, '   re la ti vi sti c</w>': 1, '   da mp ers</w>': 1, ' en ga ge</w>': 1, '   n ac e ll es</w>': 2, '   i g ni te</w>': 3, '   in take</w>': 8, '   i on o sp ere</w>': 1, '   lan ge</w>': 2, '   la un ch es</w>': 3, '   fu sed</w>': 5, '   tri t ani u m</w>': 1, '   v a p ori z ed</w>': 1, '   al p han u mer i c</w>': 1, '   at r</w>': 1, '   in je c t ors</w>': 2, ' v u l c ans</w>': 1, '   ex p lo si ons</w>': 4, '   sc ri m m</w>': 2, ' b ound</w>': 4, '   su ch a</w>': 1, '   under gra du ate</w>': 1, '   ha li de</w>': 1, '   he mi sp here</w>': 4, '   en dea v or</w>': 7, '   on sc re en</w>': 2, ' fir ing</w>': 1, '   re c rea te</w>': 1, '   ex o</w>': 1, '   rea li g n</w>': 1, '   re pro gra m</w>': 2, '   lo ca li z ed</w>': 2, '   con ne c ting</w>': 7, '   con du i ts</w>': 5, ' r ou ting</w>': 1, '   mo di f ying</w>': 2, '   cre w me mb ers</w>': 1, ' e st ab li sh</w>': 3, '   in t re p id</w>': 2, '   ca su al ti es</w>': 11, '   shu tt le b ay</w>': 3, '   al er ts</w>': 1, '   da mp en ing</w>': 1, '   tri cor d ers</w>': 1, '   pro to co ls</w>': 5, '   plan e t ary</w>': 3, '   ex tra ter re st ri al s</w>': 2, '   mon o x i de</w>': 1, '   f lu or ine</w>': 1, '   st ri de</w>': 5, '   vi ri di u m</w>': 1, '   g or k on</w>': 13, '   chan ce ll or</w>': 26, '   po s er</w>': 1, ' ex pl or ation</w>': 2, '   di p lo ma ts</w>': 3, ' bar e ly</w>': 2, '   2 7 </w>': 12, '   k r on o s</w>': 16, '   di so be ying</w>': 2, '   de mo ted</w>': 1, '   con spi ra t ors</w>': 3, ' b loo de d ly</w>': 1, '   ti ber i us</w>': 1, '   1 8 4 </w>': 3, '   as sa s sin a ting</w>': 2, '   di rest</w>': 1, '   fe i g n</w>': 1, '   pre o c cu pi es</w>': 1, ' stu n</w>': 1, '   lin gu i sti c</w>': 1, '   le ger de ma in</w>': 1, '   in t re pi di ty</w>': 1, '   ru r a</w>': 3, '   pen the</w>': 3, '   att ac h</w>': 5, ' or din a tes</w>': 3, '   v al t an e</w>': 1, '   e di f ying</w>': 1, '   char ting</w>': 2, '   ca ta lo ging</w>': 3, '   at mo sp h er es</w>': 1, '   an al y ti c</w>': 3, '   cha me lo i ds</w>': 1, ' sha pe shi f t ers</w>': 1, '   my th i cal</w>': 2, '   li ke li est</w>': 1, ' for sa k en</w>': 2, '   gen i tal s</w>': 3, '   k ran do g</w>': 1, '   ar an ty</w>': 1, '   f raid</w>': 2, '   dis man t l ed</w>': 3, '   en th u si a sti c</w>': 10, '   spe ci f y</w>': 2, '   co ded</w>': 5, '   qu o ted</w>': 4, '   re ver sing</w>': 3, '   4 2 3 6 </w>': 1, '   out li ved</w>': 5, '   u se fu l ne ss</w>': 3, '   in f le x i ble</w>': 3, '   pre ju du c ed</w>': 1, '   di c ta ted</w>': 5, '   a z e t bu r</w>': 1, '   un di sc lo sed</w>': 3, '   t or pe do ed</w>': 1, '   in sti ga t ors</w>': 1, ' s ca le</w>': 4, '   cu ri ou s ly</w>': 2, '   e man ate</w>': 3, '   an n al s</w>': 1, '   ri g ger</w>': 1, '   con n</w>': 2, '   sur r en d ers</w>': 1, '   sur mi se</w>': 1, '   un f old</w>': 1, '   spe cu la tions</w>': 3, '   har bor s</w>': 1, '   f l un g</w>': 2, '   sa bo ts</w>': 1, '   u r gen tly</w>': 6, '   p ra x is</w>': 4, '   sho ck wa ve</w>': 2, '   2 4 0</w>': 3, '   ga se ous</w>': 2, '   at mo sp h er i c</w>': 6, '   di sp ose</w>': 10, '   f oo tw e ar</w>': 1, '   con c lu si ve</w>': 9, '   ex p ends</w>': 1, ' pla s ma</w>': 1, '   i on i z ed</w>': 2, '   en vi si on ed</w>': 1, '   ou tw ar ds</w>': 1, '   0 1 3 0</w>': 1, '   in gra in ed</w>': 1, '   ab i</w>': 1, ' i ti es</w>': 1, '   sh in z on</w>': 17, '   co g ni tive</w>': 3, ' ta lo si ans</w>': 1, ' pa k le ds</w>': 1, '   h ome world</w>': 2, ' pa k je ds</w>': 1, ' bo li ans</w>': 2, '   ra g ged</w>': 8, '   ba j or an</w>': 1, '   x en o bi o lo g y</w>': 1, '   ci vi li z a tions</w>': 6, '   z e ph y r</w>': 1, '   co ch r an</w>': 1, '   bea g le</w>': 1, '   3 2 7 4 </w>': 2, '   di ver ting</w>': 3, '   se qu en c ing</w>': 3, '   r na</w>': 2, '   ad r en al in</w>': 2, '   th al ar on</w>': 6, '   s ea son ing</w>': 1, '   ha il ed</w>': 5, '   a x is</w>': 1, '   car to gra ph y</w>': 3, '   ba ss en</w>': 1, '   pa th ways</w>': 4, '   re man</w>': 9, '   dar k ly</w>': 1, '   in sti ga ted</w>': 1, '   o ver ri d es</w>': 2, '   er e c ted</w>': 1, '   por tal s</w>': 2, '   pi c to gra ph s</w>': 1, '   ver b</w>': 2, '   al ac ri ty</w>': 1, '   9 4 8 </w>': 1, '   f l or id</w>': 1, '   in f er</w>': 2, '   cap ab le com man der</w>': 1, '   app ro x</w>': 1, '   i ma tely</w>': 1, ' sa d</w>': 3, '   ev o ke</w>': 1, '   den o te</w>': 1, '   con f lu ence</w>': 2, '   in ver se</w>': 1, '   en din gs</w>': 9, '   app ro x i ma te</w>': 1, '   en gra m</w>': 1, '   so ong</w>': 4, '   pro vi si on al</w>': 2, ' ac tu a li z ation</w>': 1, '   me chan i c s</w>': 6, '   ment</w>': 3, '   tr an sp or ting</w>': 3, '   hu man o i ds</w>': 1, ' war p</w>': 2, '   sig na tur es</w>': 10, '   the ori ze</w>': 2, '   k o l ar us</w>': 1, '   mo du l ar</w>': 1, '   u ri ous</w>': 1, '   te le pa th y</w>': 2, ' ve</w>': 79, '   j our ne ys</w>': 3, '   in ve st ing</w>': 5, '   be ta z ed</w>': 3, '   con stra in</w>': 1, '   im z ad i</w>': 1, '   e m pa th i c</w>': 1, '   d on a tr a</w>': 2, '   v al d ore</w>': 3, '   war bi r d</w>': 2, '   ar ri v al s</w>': 1, '   s ci mi t ar</w>': 6, '   2 1 1 </w>': 4, '   ca s ca ding</w>': 1, '   gen i c</w>': 1, '   en com pa ss</w>': 1, '   bi o q en i c</w>': 1, '   sc ans</w>': 4, '   the or e ti cal</w>': 6, '   r er ou ting</w>': 1, '   u pl in ks</w>': 2, '   su b sta tions</w>': 1, '   p ra e tor</w>': 17, '   un char ted</w>': 3, '   k o l ar in</w>': 3, '   j an e way</w>': 2, '   sha ke up</w>': 1, '   v a li an tly</w>': 1, '   so o th</w>': 1, '   so o th es</w>': 3, '   imp ac ts</w>': 2, '   el ev ation</w>': 1, '   dis ru p tor</w>': 1, '   p ha sed</w>': 2, '   e le c t ro ma g ne ti c</w>': 4, '   stra pp ing</w>': 2, '   cha tty</w>': 3, '   un ea sy</w>': 9, '   co ll ab or at ors</w>': 1, '   for e se ea ble</w>': 2, '   ar go</w>': 11, '   re m us</w>': 4, '   ho lo gra p hi c</w>': 2, '   e mi tt ers</w>': 1, '   cap ta in n</w>': 1, '   me an in g less</w>': 17, '   re cap tu red</w>': 1, '   mo di fi ca tions</w>': 3, '   re man s</w>': 3, '   re sp ons</w>': 1, '   i bi li ty</w>': 1, '   li ber ate</w>': 2, '   ro mu l us</w>': 3, ' in f lu ence</w>': 1, '   di a g no sed</w>': 5, '   sha la ft</w>': 1, '   li gh ting</w>': 12, '   a ll you</w>': 1, '   p ea ce ma k er</w>': 2, '   mi st r ust</w>': 5, '   uni ty</w>': 5, '   de clo a king</w>': 1, '   e p si l on</w>': 4, '   in vi g or a ting</w>': 2, '   sur an</w>': 1, '   supp or ted</w>': 6, '   en li ght</w>': 1, '   war bi r ds</w>': 1, '   el m o</w>': 3, '   ex tra ter re st ri al</w>': 9, '   prob es</w>': 3, '   sh er m in</w>': 16, '   la th ro p</w>': 8, '   un pa tri o ti c</w>': 1, '   mu stan g</w>': 23, ' con tain</w>': 1, '   in v a</w>': 1, '   ha y dn</w>': 9, '   mar c</w>': 8, ' go o d b ye</w>': 6, '   j en n y ha y dn</w>': 5, ' la th ro p</w>': 1, ' la s</w>': 3, ' al t</w>': 2, ' co ke</w>': 2, '   m i</w>': 19, ' ch i</w>': 6, ' g an</w>': 1, ' en s</w>': 1, '   l y man</w>': 3, '   cont in gen ci es</w>': 2, '   re he ar sed</w>': 5, '   ca li b re</w>': 2, '   han d g un s</w>': 1, '   a gh h</w>': 1, '   ma tt ed</w>': 1, '   ea u</w>': 2, ' c li cking</w>': 1, '   ter re st ri al</w>': 3, '   ac ce p ts</w>': 8, '   f ac e pla te</w>': 2, '   ex ce ssi ve</w>': 9, '   v ous</w>': 11, '   fran ca is</w>': 7, '   ha bl a</w>': 3, '   in g les</w>': 6, '   z i e</w>': 1, '   d ra p ed</w>': 1, '   e du ar do</w>': 7, '   re ce p tor</w>': 1, ' being</w>': 8, '   su per con du c ting</w>': 2, '   re ce p t ors</w>': 1, ' stand</w>': 3, '   tr an</w>': 24, '   n er o</w>': 8, ' in ce</w>': 1, '   od</w>': 7, '   ra t fuck</w>': 2, '   po ac h</w>': 2, '   t ro d es</w>': 2, '   j er i k o</w>': 5, '   pe l ch er</w>': 2, '   v c r</w>': 7, ' tra de</w>': 3, '   do able</w>': 2, '   f ab r i</w>': 1, '   din k</w>': 3, '   shi t bi r d</w>': 2, ' can n on</w>': 1, '   na a w w w</w>': 1, '   fro g g y</w>': 1, '   re gre tt ably</w>': 4, ' bo y friend</w>': 4, '   co l or bl ind</w>': 1, '   sh in di g</w>': 1, '   2 2 0 3 </w>': 1, '   la u gh in g ly</w>': 1, '   st ri ck land</w>': 5, '   har d l ine</w>': 2, '   su sp en ding</w>': 2, '   lo b es</w>': 3, '   run ny</w>': 2, '   a mp li fi er</w>': 1, ' fri ed</w>': 5, '   mer ging</w>': 2, '   he i gh ten ing</w>': 1, '   st al ks</w>': 2, '   ca m pa d re</w>': 1, ' sh or t</w>': 3, '   bra in p an</w>': 3, '   lu c ki er</w>': 1, '   bo b b y y y y</w>': 1, '   gen er ac i on es</w>': 1, '   pre pp y</w>': 1, '   to p si d ers</w>': 1, '   ra ti o s</w>': 2, '   le sion</w>': 9, '   we ar er</w>': 2, '   e ye fuck</w>': 1, '   m und an e</w>': 2, ' pa tt er n</w>': 1, '   s nor in</w>': 1, '   so a ps</w>': 2, ' di ve</w>': 1, '   b la mm o</w>': 1, '   s cu ttle</w>': 1, '   di sa pp o in t men ts</w>': 2, '   fu mi t su </w>': 3, '   shi t can ned</w>': 1, ' ho ok er</w>': 1, '   what ya</w>': 3, '   ce ci le</w>': 29, '   s an c ti mon i ous</w>': 5, '   s qui d head</w>': 1, '   la p d</w>': 17, '   s wa ll ow ing</w>': 2, '   re si st ant</w>': 3, '   co l tr an e</w>': 1, '   what up</w>': 1, '   z an der</w>': 10, '   si ck o</w>': 3, ' ki ll er</w>': 2, '   gh o st ing</w>': 1, ' f rea k</w>': 4, ' j ac ks</w>': 1, '   b ack st ro king</w>': 1, ' c li mber</w>': 1, '   be e m er</w>': 2, '   g ro v el ing</w>': 2, '   re t in al</w>': 2, '   wi re head</w>': 1, '   l or ne tt e</w>': 1, '   ba i ling</w>': 5, '   s cu m su cking</w>': 2, '   in fe ct</w>': 4, ' per v </w>': 1, '   wi re hea ds</w>': 1, ' sa ver</w>': 1, '   s lea z o id</w>': 2, '   j er ked</w>': 6, '   che er i o s</w>': 5, '   d ra ins</w>': 4, '   g an g ban g ers</w>': 1, ' bur n</w>': 4, '   ss ss sh</w>': 3, '   tr an qui li ze</w>': 1, '   be ar ded</w>': 1, '   te st a</w>': 8, '   k o e ss l er</w>': 28, '   in ac tive</w>': 2, '   nor th we st</w>': 6, '   ho t bed</w>': 1, '   ser i al s</w>': 6, '   fri ction</w>': 3, '   bra in chi ld</w>': 1, '   s ci en ces</w>': 4, '   qu an ti c o</w>': 9, '   c ri min o lo g y</w>': 3, '   pro fi ling</w>': 4, '   z or r o</w>': 6, '   n g</w>': 1, '   mo pp ing</w>': 2, '   m ack el way</w>': 19, '   ca si o</w>': 1, '   5 4 </w>': 9, '   win on a</w>': 3, '   su mp ter</w>': 6, ' to o th</w>': 4, '   un su b</w>': 5, '   so cor r o</w>': 2, '   hi sp ani c</w>': 2, '   nu ev o</w>': 1, '   a la mo g or do</w>': 1, '   vi ca p</w>': 3, '   e mb al med</w>': 3, '   ex hu me</w>': 2, '   g ou ged</w>': 2, '   po l ar o i ds</w>': 2, '   per i mor tal</w>': 1, '   k u lo k</w>': 10, '   li ga ture</w>': 2, '   st ran gu la tion</w>': 4, '   n y l on</w>': 3, '   in den ta tion</w>': 1, '   cu st om i z ed</w>': 1, '   ma m</w>': 18, '   di sc re p an cy</w>': 4, '   ja v a</w>': 3, '   h er ea b ou ts</w>': 3, '   lu g ging</w>': 3, '   what j a</w>': 4, '   bo l o</w>': 1, '   j un k er</w>': 1, ' z or r o</w>': 3, '   e ye li ds</w>': 3, ' com bed</w>': 2, '   out stan din gs</w>': 1, ' e mo tion al</w>': 1, '   st al king</w>': 12, '   dar r y l</w>': 25, '   a mar i llo</w>': 6, '   sti pu la tion</w>': 2, '   mu r man</w>': 12, '   f oun der</w>': 2, ' i me l da</w>': 1, '   da i t z</w>': 7, '   d un l ev y</w>': 1, '   ga s ba g</w>': 3, ' mu r man</w>': 1, '   he ir en s</w>': 1, ' mu r der</w>': 6, '   he si t ant</w>': 3, '   ra pp or t</w>': 5, '   ci i ac </w>': 3, '   my di ck</w>': 7, '   shi ver ing</w>': 3, '   plea ds</w>': 1, '   cre di ted</w>': 3, '   p li ant</w>': 1, '   so ci op a th</w>': 7, '   b ack lo g</w>': 5, '   ye ar ly</w>': 3, '   ab ru p tly</w>': 2, '   ab du c ts</w>': 1, '   b on e y ar ds</w>': 1, '   ja i me</w>': 8, '   de te ction</w>': 2, '   t an gi ble</w>': 4, '   to y ed</w>': 5, '   p ee p ho le</w>': 1, '   in con so l able</w>': 1, '   de l vi e w</w>': 1, '   ta mp a</w>': 2, ' sa w ed</w>': 1, ' in tr a</w>': 1, ' a gen cy</w>': 1, '   f ra ter ni z ing</w>': 3, '   n er d</w>': 8, '   pro fi l ers</w>': 1, ' pro mo tion</w>': 4, '   an g or a</w>': 8, '   su n da e</w>': 2, '   o c cu pa tions</w>': 5, '   st r y k er</w>': 2, '   j ar g on</w>': 2, '   f an ta si sts</w>': 1, '   di sa gre e ment</w>': 2, '   we b si te</w>': 4, '   cha u c er</w>': 1, '   per en ni al s</w>': 1, '   d ev i a tes</w>': 1, '   y a k king</w>': 1, '   cor re sp on dent</w>': 11, ' gh o sted</w>': 1, '   un cu ff</w>': 1, '   dri f ts</w>': 2, '   li on he art</w>': 4, '   pre di ca ble</w>': 1, '   mor gu es</w>': 1, '   questi on na i re</w>': 11, '   c r ack er j ack</w>': 2, '   te chi e</w>': 2, ' as sig ned</w>': 2, '   rea d ju sted</w>': 1, '   d ow n lo a ds</w>': 2, '   possi b les</w>': 1, '   n pe</w>': 1, '   di sa pp ear an ces</w>': 1, '   un de te c ted</w>': 4, '   bu rea u cra ti c</w>': 7, '   pi g gi e</w>': 1, '   nee d le point</w>': 3, '   1 4 7 </w>': 1, '   ran e</w>': 1, ' li on he art</w>': 1, ' rea li ty</w>': 6, '   c ro ss b ar</w>': 1, ' f an ta sy</w>': 2, '   au to p si es</w>': 1, ' i te m</w>': 2, '   ch ok es</w>': 1, ' my di ck</w>': 1, '   sa un a</w>': 5, '   f la m er</w>': 1, '   c ri ti ci z ed</w>': 1, '   un di p lo ma ti c</w>': 1, ' at ti tu de</w>': 1, ' com pu ter</w>': 5, '   u p da ting</w>': 2, '   sa lin as</w>': 2, ' ti ck</w>': 2, '   har d boy</w>': 1, '   ni k k i</w>': 14, '   e sp re s so</w>': 6, '   c ome di an</w>': 21, ' mon day</w>': 4, '   k el b o</w>': 1, '   pi c o</w>': 4, '   pre ce ded</w>': 4, '   qui c he</w>': 4, '   l or ra ine</w>': 7, '   g l en li v et</w>': 3, ' go of y</w>': 3, ' bro ad way</w>': 2, ' de e p sp ace</w>': 1, '   au di tion ing</w>': 7, ' d or o th y</w>': 2, '   m g m</w>': 3, ' min i mu m</w>': 1, '   re b oun ding</w>': 1, '   cle an ly</w>': 2, '   tr ent</w>': 8, '   sin a tr a</w>': 12, ' com s</w>': 1, '   bi tt er s we et</w>': 3, '   pro lo gu e</w>': 1, '   spe ci a li z es</w>': 3, '   u u u u u gh</w>': 1, '   u z i</w>': 4, ' s mi les</w>': 1, '   win ger</w>': 1, '   dar win i s m</w>': 1, '   in b re e ding</w>': 3, '   de w ars</w>': 2, '   un e qui vo ca lly</w>': 1, '   ju da i s m</w>': 4, '   ri v al ry</w>': 3, '   pl u to</w>': 3, '   wee k night</w>': 1, '   re t ro ac tive</w>': 2, '   wh oo pe e</w>': 4, '   car j ac ked</w>': 1, '   mu g ging</w>': 4, '   do g g</w>': 1, '   der i v a tive</w>': 4, '   sc or se se</w>': 1, ' of f s</w>': 7, '   gre t s k y</w>': 1, '   2 1 3 </w>': 3, ' 4 6 7 9 </w>': 2, ' p an ca kes</w>': 1, '   en li gh ten ment</w>': 7, '   co f fe es</w>': 2, '   vi b ing</w>': 1, '   t wi r ly</w>': 1, '   whi r ly</w>': 1, ' through</w>': 10, ' r oun der</w>': 1, ' s la p</w>': 1, '   e st ee m</w>': 6, '   gar land</w>': 12, '   cla pp ing</w>': 2, '   ba y wa tch</w>': 1, ' d ou b le down</w>': 1, ' r en a i ss an ce</w>': 1, '   ex ca li b er</w>': 1, ' age</w>': 5, '   ob s cu re</w>': 5, ' je d i</w>': 1, '   cl ar i f ying</w>': 1, ' bo ss</w>': 3, '   s k an ks</w>': 4, ' le g</w>': 1, ' pu be sc ent</w>': 1, '   h un dy</w>': 1, '   ra in man</w>': 1, '   dar ting</w>': 1, ' de fini tely</w>': 1, '   wh in ey</w>': 1, ' c y pre ss</w>': 1, ' an a he i m</w>': 1, '   mm money</w>': 1, ' hon e st ly</w>': 2, ' shi ver ing</w>': 1, '   con v u l se</w>': 1, '   ro en i ck</w>': 2, '   che li o s</w>': 1, '   bl ack be ard</w>': 1, ' chri st y</w>': 1, '   sti r red</w>': 10, '   ha g en</w>': 9, '   b on as er a</w>': 3, '   bar z in i</w>': 16, '   ta tt a g li a</w>': 8, '   st r ac hi</w>': 1, '   c un e o</w>': 1, '   s an t in o</w>': 7, '   g ou le</w>': 1, '   g ou ll e</w>': 1, '   so ll o z z o</w>': 18, '   te ssi o</w>': 6, '   cle men z a</w>': 18, '   lu c a</w>': 19, '   con sig li ere</w>': 8, '   re gi mes</w>': 2, ' ge ts</w>': 4, ' says</w>': 4, '   ter ri f</w>': 1, '   ta tt a g li as</w>': 5, '   n ar se i ll es</w>': 1, '   imp or ting</w>': 2, '   pe z z on o v an ta</w>': 3, '   out f ought</w>': 1, '   ca por e gi mes</w>': 2, '   w o l t z</w>': 2, '   con sig l ere</w>': 3, '   gen c o</w>': 2, '   god son</w>': 4, '   con sig l er o</w>': 1, '   pro sti tu tion</w>': 4, ' a po lo gi z ed</w>': 1, '   bra s i</w>': 6, '   ma ss ac r es</w>': 3, '   en z o</w>': 27, '   re pa tri a ted</w>': 1, '   na z or ine</w>': 1, '   si r ac use</w>': 1, '   p al er m o</w>': 9, '   tom ma s sin o</w>': 2, '   ca l o</w>': 4, '   f ab ri z z i o</w>': 1, '   sh h h h h</w>': 9, '   lu par a</w>': 1, '   se ee e</w>': 1, '   vi te ll i</w>': 11, ' re ti red</w>': 1, '   f re do</w>': 19, '   sta le ma te</w>': 3, '   m c c lu s k ey</w>': 9, '   in v u l n er able</w>': 7, '   out ca sts</w>': 3, '   dis cu ssi ons</w>': 4, '   can c el s</w>': 1, ' mi ck</w>': 1, '   p in k o</w>': 1, '   pro te g es</w>': 1, ' i ri sh</w>': 3, '   f on t an e</w>': 2, '   g rea se b all</w>': 2, '   go om ba h s</w>': 1, '   cor le on es</w>': 1, '   bo ok kee p ers</w>': 2, '   la mp one</w>': 2, '   n er i</w>': 3, '   ca por e gi me</w>': 1, '   in g ri d</w>': 2, ' to m</w>': 6, '   su ff o ca ted</w>': 2, '   inter f er ed</w>': 2, '   in for m ers</w>': 2, ' son ny</w>': 1, '   pa y ph one</w>': 5, '   ba p ti z ed</w>': 11, '   r en oun ce</w>': 5, '   pi ck in gs</w>': 1, '   ban k ro lled</w>': 1, '   ev en ed</w>': 2, '   da go s</w>': 1, '   ci vi li sed</w>': 2, '   fa u s to</w>': 2, ' k in g sle y</w>': 3, '   cre w ing</w>': 2, '   di c ki e</w>': 75, '   sh er w o od</w>': 9, '   mar ge</w>': 63, '   h er ber t</w>': 20, '   gre en lea f</w>': 21, '   no se b le ed</w>': 1, '   mon g i</w>': 7, '   si l v an a</w>': 1, '   sa x</w>': 10, '   o g ling</w>': 1, '   e sa t to</w>': 1, '   mon gi be llo</w>': 2, '   re m o</w>': 16, '   or g ani z ing</w>': 4, '   cor t in a</w>': 5, '   car o z z a</w>': 2, '   ba ti st on i</w>': 1, '   pr un e</w>': 2, ' di c ki e</w>': 1, '   er me lin da</w>': 2, '   c in que cen to</w>': 1, '   for ging</w>': 2, '   under co at</w>': 1, '   hu llo</w>': 9, ' ma in ten an ce</w>': 1, '   ca pr i</w>': 4, '   f re der i c o</w>': 2, '   be llo</w>': 1, ' de ll e</w>': 1, '   c ro ce</w>': 3, '   cor so</w>': 2, '   o te llo</w>': 2, '   de ll e</w>': 3, ' p m</w>': 2, '   di c ki es</w>': 1, '   m ac car r on</w>': 4, '   s ow</w>': 5, '   con fr on ts</w>': 3, '   al v in</w>': 4, '   e mp lo ying</w>': 1, '   pa la z z o</w>': 2, '   gi o i a</w>': 1, '   app a lling</w>': 3, '   c r y p ti c</w>': 3, '   thin g y</w>': 3, '   te x ti les</w>': 1, '   k in g sle y</w>': 2, '   din e ll i</w>': 2, '   cha tt ed</w>': 3, '   un re li able</w>': 4, '   ran d all</w>': 16, '   lo gu es</w>': 1, '   sh ru g</w>': 3, '   lo gu e</w>': 2, '   st rea m lin ed</w>': 1, '   que st a</w>': 2, '   le tt er a</w>': 1, '   sta ta</w>': 2, '   t ro v at a</w>': 1, ' ab i ta z i one</w>': 1, '   ro ma</w>': 12, '   ra gi one</w>': 2, '   que s to</w>': 1, '   lu o go</w>': 1, '   vo st re</w>': 1, '   con ver sa z i on i</w>': 1, ' se qui tu r</w>': 1, '   s vi lu pp ate</w>': 1, '   ten den ze</w>': 1, '   om o se s su al i</w>': 1, ' a per to</w>': 2, '   co l</w>': 17, '   f re d do</w>': 1, '   fa t to</w>': 1, '   sta to</w>': 1, '   a ll or a</w>': 1, '   qu an do</w>': 1, ' u l ti ma</w>': 1, '   vo l ta</w>': 1, '   vi s to</w>': 1, '   as su n to</w>': 1, '   i o</w>': 1, '   gu i da</w>': 1, '   in da g in i</w>': 1, '   se gu i to</w>': 1, '   ne ga ti v a</w>': 1, '   v al u ta z i one</w>': 1, '   dis di c ev o l i</w>': 1, '   ci r co stan ze</w>': 1, '   ver i fi ca te s i</w>': 1, '   pre de ce ss ore</w>': 1, '   ro ver in i</w>': 2, '   no to</w>': 3, '   ri u s ci to</w>': 1, '   im pe di re</w>': 1, '   ver i fi car s i</w>': 1, '   s com par s a</w>': 1, '   qu a le</w>': 1, ' uni c a</w>': 1, '   m om en to</w>': 1, '   pa ssi bi le</w>': 1, '   in c ri min a z i one</w>': 1, '   rea to</w>': 1, '   om i ci di o</w>': 1, '   t or men ted</w>': 4, '   hur t ful</w>': 3, '   un a</w>': 4, '   fi dan z at a</w>': 2, '   prob ab il men te</w>': 1, '   mo l te</w>': 1, '   fi dan z ate</w>': 1, '   h om o se x u al s</w>': 4, '   le on ar do</w>': 7, '   mi ch el an ge l o</w>': 2, '   di f fi ci le</w>': 1, '   d or mi v a</w>': 1, '   b ack p ac king</w>': 3, '   t re</w>': 1, '   se t ti man e</w>': 1, '   bu rea u c r ac y</w>': 1, '   pi a z z a</w>': 4, '   sp a g na</w>': 2, '   r ow ing</w>': 1, '   con fr on to</w>': 1, '   go l d on i</w>': 3, '   al to</w>': 4, '   h oun ded</w>': 1, '   sen ta</w>': 1, ' ea t in</w>': 4, '   au to ma ti c s</w>': 3, '   ho l ster</w>': 6, ' 1 2 5 </w>': 1, '   tra v is</w>': 29, '   sti ff ne ss</w>': 1, '   p al an t ine</w>': 6, '   co c a</w>': 11, '   k ri st of f er son</w>': 2, '   pu sh er</w>': 5, '   tr an si st or</w>': 3, '   d m z</w>': 3, '   plan n ers</w>': 2, '   bu sh ed</w>': 2, '   sa ks</w>': 6, '   ta x is</w>': 2, '   or g an e z i z i ed</w>': 1, '   or g ani z a tion al</w>': 1, '   can v ass</w>': 4, '   ne w s stand</w>': 8, '   p uni sh men ts</w>': 2, '   as be sto s</w>': 3, '   chri st sa kes</w>': 5, ' se x y</w>': 2, '   a p</w>': 2, '   ok a ys</w>': 1, ' ju kes</w>': 2, '   ma l ted</w>': 1, '   a wh </w>': 1, ' b all</w>': 11, '   po ly</w>': 4, '   di sp a tch er</w>': 5, '   ha c ki es</w>': 1, '   che ck in</w>': 3, '   h ow sit</w>': 2, ' bor o</w>': 1, '   p an t y ho se</w>': 7, '   r ou ge</w>': 5, ' sha d ow</w>': 2, '   1 2 2 </w>': 2, '   com m un e</w>': 10, '   sc or pi on</w>': 4, '   li bra s</w>': 1, '   com m un es</w>': 1, '   li br a</w>': 2, ' sp or t</w>': 1, '   g rea s ers</w>': 1, '   sti ck ers</w>': 2, '   mo on li gh t ers</w>': 1, '   mo on li gh t in</w>': 1, '   1 9 7 1 </w>': 5, '   ch oo sy</w>': 3, '   figu r</w>': 1, '   k r in k le</w>': 5, '   0 7 4 1 0</w>': 1, '   s qu ir m</w>': 2, '   wi z</w>': 1, '   go o d spe ed</w>': 7, '   pe sta lo z z i</w>': 1, '   e th ni c</w>': 3, '   fi el d work</w>': 1, '   may ta g</w>': 2, '   wi mp ing</w>': 1, '   can ter bu ry</w>': 1, '   imp ri son ed</w>': 7, '   al chi ma d us</w>': 1, '   wal ton</w>': 3, '   pe w</w>': 4, '   j ou ey</w>': 1, '   wom ack</w>': 1, '   dro ll</w>': 1, '   v ou </w>': 2, '   tru d g es</w>': 1, '   tu </w>': 12, ' de l</w>': 1, '   s lin ging</w>': 4, '   co tt on m ou th</w>': 1, '   de lly</w>': 11, '   app le ton</w>': 18, ' in c ri min ation</w>': 1, '   in c ri min ation</w>': 1, '   l ev ers</w>': 3, '   ter wi lli ger</w>': 2, '   k u be l s k y</w>': 1, '   un true</w>': 6, '   ro sen ber gs</w>': 2, ' su spe c ted</w>': 1, ' s and</w>': 1, '   la w son</w>': 61, '   hi c cu ps</w>': 2, '   sp ence</w>': 3, '   di vi d ends</w>': 1, '   nee d less</w>': 4, '   un lo cks</w>': 1, '   ba se man</w>': 2, '   me th o di sts</w>': 1, '   me th o di st</w>': 2, '   pre s b y ter i ans</w>': 1, ' al ber t</w>': 2, '   tru mb o</w>': 7, '   re fu ting</w>': 2, ' re sp on si ve</w>': 2, '   b ran ded</w>': 3, '   an g st ro m</w>': 5, ' lu ci ll e</w>': 2, ' brea d</w>': 3, '   p ho to sta ti c</w>': 1, ' b lo tt ed</w>': 1, '   sta te side</w>': 1, '   di x</w>': 8, '   ma s qu er a ding</w>': 5, '   in cu r r ing</w>': 1, '   in cu r red</w>': 3, '   con sti tu ted</w>': 2, '   for th coming</w>': 2, ' w o</w>': 4, '   fi tt s</w>': 2, '   st or e fr on t</w>': 1, ' stan ding</w>': 2, '   pro c ee ds</w>': 3, '   re fun ds</w>': 1, ' cu e b all</w>': 1, '   dr ought</w>': 6, '   fr y er</w>': 2, ' 8 4 </w>': 3, '   stan w y ck</w>': 3, '   y ow s a</w>': 1, ' fe b ru ary</w>': 2, '   1 9 4 2 </w>': 4, '   bo ok kee p ing</w>': 3, ' op en ed</w>': 1, ' te l ev i sion</w>': 1, ' lu ke</w>': 3, ' i d es</w>': 6, '   j er</w>': 4, '   i s k ow i t z</w>': 1, '   gi m me e</w>': 2, '   z ack ly</w>': 1, ' mo by</w>': 1, '   i sh ma el</w>': 4, '   e mp ti es</w>': 3, '   shu tting</w>': 13, '   l ar d ner</w>': 2, ' sc ra y</w>': 1, '   el v in</w>': 1, '   pu r ging</w>': 2, '   hi r sch fe ld</w>': 1, ' stu di o</w>': 1, ' a sh es</w>': 2, '   c ri s sa kes</w>': 3, '   ch op in</w>': 3, ' hea ven</w>': 16, ' t each</w>': 2, ' r ent</w>': 2, '   u sh er</w>': 2, ' uni for m</w>': 1, ' ho spi tal</w>': 3, ' ga ve</w>': 2, '   k n</w>': 1, ' k n</w>': 1, '   ru s kin</w>': 1, '   ok in a w a</w>': 3, '   k en ny</w>': 43, '   an z i o</w>': 3, '   nor man dy</w>': 4, '   g ran d pa p</w>': 1, ' an ce st ors</w>': 1, '   ca th y</w>': 31, '   be ani es</w>': 1, '   bo de g a</w>': 9, '   o e di p us</w>': 1, '   di sp lea se</w>': 1, '   sha cks</w>': 2, '   qui lt</w>': 4, '   u ti li t ar i an</w>': 1, '   lo ve bi r ds</w>': 18, ' i lling</w>': 1, '   br en ner</w>': 20, '   ha y wor th</w>': 10, '   ho o ds</w>': 6, '   o ver doing</w>': 1, '   br in k me y er</w>': 3, '   out bo ard</w>': 3, '   mm mm m m</w>': 6, '   br en n ers</w>': 3, ' 6 5 0</w>': 1, '   ma ssed</w>': 1, '   bl ack bi r ds</w>': 4, '   gu lls</w>': 11, '   e ye la sh</w>': 10, '   mar ys</w>': 3, '   fu r n i</w>': 1, '   ma ss ing</w>': 4, '   ca vi at</w>': 1, '   e mp tor</w>': 1, ' mi tch</w>': 2, '   si ps</w>': 1, '   re f le c ti ve ly</w>': 1, '   ma l one</w>': 5, ' li gh ted</w>': 1, '   or din an ce</w>': 3, '   f lu tt er ing</w>': 2, '   chi m ne ys</w>': 1, '   la mp s</w>': 6, '   s wi f ts</w>': 1, '   sp ar r ows</w>': 2, ' bi r ds</w>': 4, '   ma l cont ent</w>': 1, '   te ss a</w>': 1, '   ve ddy</w>': 3, '   pri m</w>': 1, '   stra it</w>': 5, ' l ac ed</w>': 1, '   my na</w>': 3, '   ni tr o</w>': 7, ' g l y cer in</w>': 1, '   f oun ta ins</w>': 1, '   pro pri e ty</w>': 1, '   fami sh ed</w>': 3, '   s ea gu lls</w>': 5, '   lo a the</w>': 7, '   vi o la t ors</w>': 2, '   per o x i de</w>': 2, '   lo ve bi r d</w>': 1, '   can ar i es</w>': 2, '   han g do g</w>': 1, '   m ou l ting</w>': 2, '   or ni th o lo gi cal</w>': 2, '   ca ged</w>': 4, '   fin ch es</w>': 3, '   re d bi r ds</w>': 1, '   a lo of</w>': 2, '   de mon stra tive</w>': 2, '   d k q </w>': 1, '   m ac g ru der</w>': 1, '   uni ma g in able</w>': 3, '   ga me bi r ds</w>': 2, '   o ven s</w>': 2, '   or ni th o lo g y</w>': 1, '   a vo ca tion</w>': 1, '   ar cha e op ter y x</w>': 1, '   sho les</w>': 3, ' in si st</w>': 1, '   per ch ing</w>': 1, '   br ac h y r h y n cho s</w>': 1, '   bl ack bi r d</w>': 2, '   c y an o ce p ha l us</w>': 1, '   c ru z</w>': 14, '   al ar mi st</w>': 4, '   fe tt es</w>': 31, '   m ac f ar lan e</w>': 30, '   ca b man</w>': 2, '   ge or g in a</w>': 12, '   inter ce de</w>': 1, '   pen n y cu i k</w>': 2, '   tr y st</w>': 2, '   y a w n s</w>': 1, '   be fu d d l ed</w>': 2, ' he el s</w>': 1, '   l ow lan der</w>': 1, '   hi gh lan d ers</w>': 2, '   th ru ms</w>': 2, '   ro i st er ed</w>': 1, '   re p ti li an</w>': 6, '   ma tt o ck</w>': 1, '   to sh</w>': 1, '   g l en cor se</w>': 1, '   ch u r ch y ard</w>': 1, '   con found</w>': 2, '   chi l di sh ne ss</w>': 1, '   in ci sion</w>': 2, '   in tri c ac i es</w>': 1, ' su b je c ts</w>': 2, '   ir k</w>': 1, '   pr ac ti tion er</w>': 1, '   pa u p ers</w>': 2, '   g li b</w>': 4, '   vi car</w>': 2, '   to ddy</w>': 24, '   le i th</w>': 1, '   su st en an ce</w>': 2, '   ma li g n ant</w>': 8, '   c r on y</w>': 2, '   me g</w>': 9, '   he mp</w>': 1, '   hou se to ps</w>': 2, '   ce ll ars</w>': 2, '   gra ve y ar ds</w>': 3, '   an at om i st</w>': 1, '   pa tch ed</w>': 1, '   au ld</w>': 2, '   lan g</w>': 5, '   sy n e</w>': 2, ' bur ked</w>': 1, ' n or</w>': 5, '   to s sp o t</w>': 1, '   qu ar ts</w>': 1, '   co z en ed</w>': 2, '   be l da m</w>': 1, '   m ea ger</w>': 2, '   pe d d l ers</w>': 2, '   ch u r ch y ar ds</w>': 1, ' bur ke</w>': 1, '   under take</w>': 2, '   d om in i e</w>': 1, '   in ci se</w>': 1, '   co lu m na</w>': 1, '   d or s i</w>': 1, '   wi l d ne ss</w>': 2, '   ro sy</w>': 6, '   cor v is</w>': 12, '   oo z ing</w>': 5, '   tr ans fu sion</w>': 9, '   con no i s se u r</w>': 1, '   de lt</w>': 1, '   du tt on</w>': 4, '   er li ch</w>': 3, '   ex hi b it</w>': 19, '   ss s sh h h</w>': 2, '   a a a an k</w>': 1, '   er in</w>': 48, '   di r t b all</w>': 1, '   ma j or ly</w>': 4, '   gi mp ing</w>': 1, '   j an ni e</w>': 1, '   ta is</w>': 3, '   bar n har d t</w>': 14, '   d ra ma ti ze</w>': 1, '   ser i ou s ne ss</w>': 1, ' e li min a ted</w>': 2, '   k la a tu </w>': 8, ' plan e t ary</w>': 1, '   e du ca t ors</w>': 1, '   l ev el ing</w>': 1, '   po ten ti a li ty</w>': 1, '   un con cer ned</w>': 2, '   3 0 9 </w>': 1, '   ne g li gi ble</w>': 1, '   re pro du ce</w>': 5, ' sp ace</w>': 7, '   den om in a tor</w>': 2, '   su b tr act</w>': 2, '   con es</w>': 1, '   ar lin g ton</w>': 8, '   ce le sti al</w>': 1, '   ex pe c t an cy</w>': 2, '   s ke le tal</w>': 1, '   at ti tu d es</w>': 3, '   ca bl ed</w>': 1, '   s qu ab b les</w>': 1, '   bro ad ca sts</w>': 4, '   app ra i sa l</w>': 2, '   de pen den ts</w>': 2, ' sti mu late</w>': 1, ' k la a tu </w>': 2, '   bar ad a</w>': 2, '   ni k to</w>': 2, '   g or t</w>': 3, '   bo ar din gh ou se</w>': 1, '   j i tt er y</w>': 4, '   un ru ff l ed</w>': 1, '   sc ri b b les</w>': 1, ' ba sh ers</w>': 1, ' st al kin</w>': 1, ' sc ra mb le</w>': 1, '   un ra v el ed</w>': 1, '   v a m pi r o</w>': 1, '   ni gh t cra w l er</w>': 1, '   b loo d su ck er</w>': 1, '   no s f er a tu </w>': 1, '   mar k o</w>': 2, '   hon da</w>': 2, '   du n</w>': 1, '   n an oo k</w>': 5, '   v a mp</w>': 1, ' v a m pi r es</w>': 3, ' every where</w>': 1, ' fi er ce</w>': 1, ' pu r po se</w>': 4, ' da ting</w>': 1, '   ne w sp r int</w>': 2, '   k r y ton</w>': 1, '   ar chi es</w>': 1, ' again</w>': 9, '   sy n ch</w>': 1, ' gu i de</w>': 1, ' me mor y</w>': 2, '   o h my god</w>': 7, '   hi ked</w>': 2, '   fa im ly</w>': 1, ' gar li c</w>': 1, '   ar e ound</w>': 1, '   o h m</w>': 1, ' ei gh ti es</w>': 1, '   r en d ers</w>': 2, '   par me s an</w>': 2, ' hi pp i es</w>': 2, '   pi er c ed</w>': 7, '   n on oo k</w>': 1, '   ma lls</w>': 2, '   c in e p le x es</w>': 1, '   ther mi te</w>': 1, '   ni cked</w>': 5, '   k en ne l</w>': 2, '   re dre ssed</w>': 1, '   nor we gi ans</w>': 5, ' h y st er i cal</w>': 1, '   cle ars</w>': 5, '   m ac mu r do</w>': 1, '   gar ry</w>': 3, '   fu ch s</w>': 5, '   s k u a</w>': 1, '   ma g ne to</w>': 10, '   char i o ts</w>': 1, '   in ca s</w>': 2, '   na u ls</w>': 3, '   mu ther fuck er</w>': 3, '   m ac ready</w>': 13, '   fro st b it</w>': 1, '   gu i de l ine</w>': 1, '   nor r is</w>': 7, '   th a w ed</w>': 2, '   fi bri ll a tor</w>': 1, '   m ac re</w>': 1, '   i mi ta ting</w>': 3, '   s n ow ba lled</w>': 3, '   b en n in gs</w>': 1, '   che ck up</w>': 2, '   ar gen t ine</w>': 1, ' c y lin der</w>': 1, '   car bu re t ors</w>': 1, '   s n ow mo bi les</w>': 3, '   ex t in gu i sh ers</w>': 1, ' su f fi ci ent</w>': 4, '   t ow l ine</w>': 2, '   we i gh ted</w>': 4, '   k al en</w>': 16, '   mor lo cks</w>': 11, '   c li mber</w>': 11, '   plan ner</w>': 3, '   s wa sh bu ck ling</w>': 1, '   b y r on i c</w>': 2, '   p an d or a</w>': 3, '   lu mb er ing</w>': 2, '   re gen er a ting</w>': 1, '   c ran i al</w>': 3, '   e lo i</w>': 4, ' au to ma ton</w>': 1, '   bi ome chan i cal</w>': 1, ' these</w>': 18, '   sh or ed</w>': 1, '   e li o t</w>': 1, '   d our</w>': 1, '   au to ma ton</w>': 2, '   1 8 9 9 </w>': 1, '   car c in o gen s</w>': 3, '   po ll u t an ts</w>': 1, '   su sp en se</w>': 5, '   d ome sti ca ted</w>': 2, '   bro ca de</w>': 1, '   cla w</w>': 16, '   im mu ta ble</w>': 2, '   in di g na tion</w>': 1, '   at ro p hi ed</w>': 2, '   a da p ta tion</w>': 1, '   mi ll en ni a</w>': 1, '   sc ra p ing</w>': 5, '   li ch en</w>': 1, '   en d le ss ly</w>': 4, '   mor lo ck</w>': 1, '   per a m bu late</w>': 1, '   per a m bu la tion</w>': 1, '   can ti l ever ed</w>': 1, '   com bu sti on</w>': 1, '   inter n al s</w>': 1, '   ro c he</w>': 2, '   stra tu m</w>': 1, '   re t ro gra de</w>': 3, '   2 0 0 5 </w>': 2, '   ex ca v a tions</w>': 1, '   f lu c tu a tions</w>': 1, '   se i s mi c</w>': 4, '   in for ma tion al</w>': 1, '   ki o s k</w>': 1, '   ori on</w>': 6, '   mo on st one</w>': 1, '   lin ers</w>': 1, '   con c ea ls</w>': 1, '   wa tch it</w>': 7, '   s ni ff ed</w>': 1, '   s nor ted</w>': 2, '   p hi l by</w>': 6, '   con ser v a ti ve ly</w>': 1, '   ar che o lo gi sts</w>': 2, ' x y</w>': 1, '   pro te sta tions</w>': 1, '   sc or ch ing</w>': 1, '   un da ma ged</w>': 1, '   w el com ed</w>': 2, '   bo t any</w>': 2, '   c li mb ers</w>': 2, '   s cu l p t ors</w>': 2, '   coun ter we i gh ts</w>': 1, '   thin k ers</w>': 3, '   s w o on ing</w>': 1, '   hu z z ah</w>': 1, '   di men si ons</w>': 5, '   ge ome tri cal</w>': 1, ' du ra tions</w>': 1, '   spe cu la tion</w>': 8, ' tri cks</w>': 3, '   ev o lu tion ary</w>': 4, '   cha l k bo ard</w>': 1, '   hu s ban d ry</w>': 1, '   har t de g en</w>': 4, '   f ow l</w>': 7, '   qu an ti fi able</w>': 1, ' in sp ir ing</w>': 1, ' cha ll en ge</w>': 1, ' f ed</w>': 1, ' d ev i a ted</w>': 1, '   sy ll a bu s</w>': 2, ' app li ed</w>': 1, '   e mp lo y ers</w>': 5, '   el der</w>': 3, '   pa tri ar ch</w>': 1, '   y or k ers</w>': 5, '   mor en</w>': 1, ' e lo i</w>': 1, '   s ca m per ing</w>': 1, ' be at</w>': 91, '   star ks</w>': 19, '   al p ine</w>': 7, '   be ck er</w>': 27, '   ca da ver</w>': 1, '   re p ea ted</w>': 9, ' pr ac ti ces</w>': 1, '   w o of ing</w>': 1, '   pa u sing</w>': 2, ' sh r ou ded</w>': 1, ' po in te d ly</w>': 1, '   re stra ins</w>': 1, ' un cer ta in ly</w>': 1, '   ga ze</w>': 8, '   g ri es</w>': 3, '   l or en son</w>': 5, '   lo el</w>': 1, '   ex a min ers</w>': 1, '   1 1 2 </w>': 2, ' rea li z ing</w>': 1, ' j ack et</w>': 1, '   me d le y</w>': 2, '   sh el b our n e</w>': 1, '   ab st r ac ts</w>': 2, ' so f tly</w>': 2, '   re pla y ed</w>': 1, '   mi st rea t ment</w>': 1, '   d ru g ging</w>': 1, '   f al t ers</w>': 1, '   b lu r red</w>': 2, '   n er v ou s ly</w>': 2, ' 1 9 9 3 </w>': 1, '   plea ding</w>': 8, ' re s our ces</w>': 2, '   ad vo c ac y</w>': 2, ' pr on oun c ed</w>': 1, ' ab s an ce</w>': 1, '   m ack en z i e</w>': 4, ' f ru stra ted</w>': 1, '   vi g or ou s ly</w>': 1, '   li ab i li ti es</w>': 1, ' sin cer e ly</w>': 1, ' sh ru g ging</w>': 1, '   fi c tions</w>': 3, ' gra ve ly</w>': 1, ' cu tting</w>': 1, ' makes</w>': 6, '   fi sh ed</w>': 4, ' re so lu tely</w>': 1, '   har l an</w>': 6, '   di ck in son</w>': 3, ' clo se</w>': 4, '   par kin</w>': 3, '   pu ck e t t</w>': 1, '   in car cer ation</w>': 6, '   imp ri son ment</w>': 3, '   e le c t ro cu tion</w>': 2, '   s n ow b all</w>': 4, ' qu e</w>': 1, '   te le t y pe</w>': 3, '   har den ed</w>': 3, '   ha in ey</w>': 1, '   st ea l in</w>': 1, '   in st in c ti ve ly</w>': 5, '   pri d es</w>': 1, '   ma j or ing</w>': 1, '   1 9 2 1 </w>': 2, '   ha ci en da</w>': 1, '   ma ma si ta</w>': 1, '   ee u w w</w>': 1, '   ger al do</w>': 4, '   sa v in</w>': 2, '   pi on e ers</w>': 4, ' po o s</w>': 1, '   chi hu a hu a</w>': 2, '   su g ge st in</w>': 1, '   sa tur days</w>': 3, '   wh ad do</w>': 1, '   bo i se</w>': 1, '   th i ev in</w>': 1, '   oo oo ow ee e</w>': 1, '   inter sta tes</w>': 1, '   con spi cu ous</w>': 2, '   t ou ch in</w>': 2, '   in con spi cu ous</w>': 2, '   j ad ed</w>': 3, '   mar gar i ta</w>': 4, '   cu er v o</w>': 25, '   lan ter n</w>': 15, '   bea ver s</w>': 1, '   f re s no</w>': 3, '   gr r r re at</w>': 1, ' a tter</w>': 3, ' bra ins</w>': 3, '   hi tch er</w>': 2, '   w oo gi e</w>': 9, '   w oo g an ow s k i</w>': 1, '   co pped</w>': 5, '   al to i ds</w>': 1, '   min ts</w>': 1, '   hea ly</w>': 15, '   ad ver se ly</w>': 2, '   di a be ti c</w>': 6, '   ja i</w>': 3, '   ru gra ts</w>': 7, '   li ber a ted</w>': 6, '   pro cra st in ate</w>': 1, '   pro vi d ence</w>': 3, '   pa d ding</w>': 5, ' j er k er</w>': 1, '   st al k ers</w>': 2, '   ri ck les</w>': 1, '   y ok o</w>': 1, '   v a le di c t ori an</w>': 3, '   bor r in g ton</w>': 1, '   ro se y</w>': 2, '   bu n d t</w>': 2, ' di p</w>': 2, '   ne pa l</w>': 4, ' kin</w>': 1, ' t an</w>': 3, '   lu bed</w>': 1, '   sch m o</w>': 2, '   cor p</w>': 4, '   sa w b on es</w>': 1, '   su lly</w>': 8, '   sp o tting</w>': 4, '   su ll</w>': 4, '   al k y</w>': 1, '   co ke head</w>': 2, '   hea l th i er</w>': 7, '   pi ss er</w>': 4, '   be e p ed</w>': 2, '   u r r g g gh h</w>': 1, '   mor on i c</w>': 2, '   to ta li t ar i ani s m</w>': 1, '   ve sti bu le</w>': 3, '   por ti c o</w>': 1, '   de c o</w>': 3, '   pu ffer b all</w>': 1, '   b lo ss om ed</w>': 1, '   en clo sure</w>': 1, '   mon go</w>': 1, '   poli ti ca lly</w>': 6, '   p g a</w>': 1, '   c ri p es</w>': 1, '   ne pa le se</w>': 1, '   b ack s w ing</w>': 1, '   out lin ing</w>': 2, '   shi t b all</w>': 1, '   su m o</w>': 2, '   bri d es</w>': 2, '   bu ni on</w>': 1, '   bu tt pl u g</w>': 1, '   sp ar k pl u g</w>': 2, '   ro ll er pi g</w>': 1, '   fr on ton</w>': 1, '   h y per ac tive</w>': 1, '   un list ed</w>': 4, '   l ab ra d or</w>': 3, '   po o ch</w>': 19, '   st al kin</w>': 1, '   da de</w>': 10, '   pi z z as</w>': 12, '   s an ti ago</w>': 5, '   e sta di o</w>': 2, '   o li m pi c o</w>': 1, '   do o b</w>': 1, '   o l y m pi c o</w>': 1, '   s mo ke sc re en</w>': 1, '   ab s</w>': 7, '   st ever in o</w>': 1, '   ba ton</w>': 11, '   ca l ori e</w>': 1, ' coun t in</w>': 1, '   ki el ba s a</w>': 2, '   pe c s</w>': 3, '   ha m mo ck</w>': 8, '   pe lo qu in</w>': 2, '   po ll y an na</w>': 2, '   shu ck er</w>': 1, '   y app ing</w>': 2, '   sch le p</w>': 2, '   mo ok alone</w>': 1, '   mo ok</w>': 5, '   p om p an o</w>': 1, '   pe tt y g ro ve</w>': 1, '   in car cer a ted</w>': 2, '   de com po sed</w>': 2, '   me d d le some</w>': 1, '   al u m n us</w>': 1, '   pu ssi es</w>': 2, '   f ar ter</w>': 1, '   cor n do gs</w>': 1, '   se x i er</w>': 3, '   com po sure</w>': 2, '   sp ort sc en ter</w>': 1, '   pi ck y</w>': 4, '   pe pp ers</w>': 3, '   to pp in gs</w>': 1, '   cor ned</w>': 9, '   po p si c les</w>': 1, '   fu d ge si c les</w>': 1, '   lo lli po ps</w>': 2, ' si c les</w>': 2, '   bro o ks</w>': 9, '   cl y de s da le</w>': 1, '   un ne ces</w>': 1, '   h y gi en i st</w>': 1, '   shi r tt a il</w>': 1, '   fran ks</w>': 4, '   pi g g y back</w>': 1, '   so vi e ts</w>': 12, '   di st en tion</w>': 1, '   b lo ck ad es</w>': 2, '   star sh e lls</w>': 1, '   g ro z n y y</w>': 2, '   in b ound</w>': 2, '   ga gar in</w>': 1, '   ki mo v s k</w>': 1, '   ca ve at</w>': 1, '   k h ru sch ev </w>': 14, '   m c na mar a</w>': 3, '   m c c one</w>': 2, '   z or in</w>': 5, '   ad la i</w>': 4, '   app ea s er</w>': 1, '   st ev en son</w>': 2, '   ac he son</w>': 5, '   do tt ed</w>': 1, '   e m ba ssi es</w>': 2, '   c r y p to gra p her</w>': 1, '   s n ow job</w>': 1, '   mi d ter ms</w>': 2, '   ge or g i</w>': 1, '   o ver e sti ma ted</w>': 1, '   li pp man</w>': 9, '   ju pi t ers</w>': 1, '   f om in</w>': 2, '   p lo y</w>': 3, '   ex co m</w>': 2, '   m c clo y</w>': 6, '   ni t ze</w>': 1, '   gi l pa tri c</w>': 1, '   under se cre t ar i es</w>': 1, '   pr in ci p al s</w>': 3, '   poli ti ci z ed</w>': 1, '   th ro a ts</w>': 10, '   t ou gh ne ss</w>': 1, ' ar ran ged</w>': 2, '   re de p lo y ment</w>': 1, '   re in t ro du ce</w>': 1, '   fa v or ably</w>': 2, '   sa ms</w>': 1, '   cen tra li z ed</w>': 2, '   e s ca la tion</w>': 2, '   o ver f li gh ts</w>': 1, '   ther ea f ter</w>': 1, '   e s ca late</w>': 1, '   o pl an</w>': 1, '   3 1 6 </w>': 2, '   dis ru p tions</w>': 1, '   un fo l ds</w>': 1, '   ri bi co ff</w>': 1, '   s k y bo lt</w>': 1, '   na s sa u</w>': 3, '   mi d ter m</w>': 2, '   ru s k</w>': 1, '   di sa v ow</w>': 1, '   lin k age</w>': 2, '   ci r cu m stan ti al</w>': 6, '   fe k li so v </w>': 1, '   a k a</w>': 3, '   god d man</w>': 2, '   na to</w>': 4, '   clo b ber ed</w>': 6, '   h or se tra de</w>': 1, ' h row</w>': 1, '   de la ying</w>': 1, '   da ly</w>': 3, '   o ver thr own</w>': 1, '   pu r vi e w</w>': 1, '   bi s se l</w>': 3, '   be es</w>': 15, ' bar re l</w>': 2, '   den i able</w>': 1, '   be la y</w>': 1, '   f re i gh t ers</w>': 3, ' di men si on al</w>': 4, '   b ow en</w>': 1, ' ha mi l ton</w>': 1, '   ti gh ten ing</w>': 1, '   a da ge</w>': 1, '   di st ru st ful</w>': 1, '   ba d lan ds</w>': 5, ' ei gh th</w>': 4, '   l ev o i</w>': 4, '   co o ch</w>': 13, '   in sti ga tion</w>': 1, '   wa sh in ton</w>': 1, '   qui ll work</w>': 3, '   o gla l a</w>': 3, '   cer e mon i al</w>': 3, '   per ta in ing</w>': 2, '   gen o ci de</w>': 4, '   ba i t sho p</w>': 1, '   ever gla d es</w>': 4, '   w oh</w>': 2, '   u l c er</w>': 9, '   do er</w>': 4, '   ma i sy</w>': 4, '   l ar a mi e</w>': 4, '   di sc or d</w>': 3, '   in for m ant</w>': 14, '   u pri ver</w>': 8, ' bor ed</w>': 2, '   3 5 7 </w>': 1, '   car ne gi e</w>': 5, ' me l on</w>': 1, '   mo ha w ks</w>': 2, '   un sa ti s f ac t ory</w>': 2, '   k o l a</w>': 1, '   ho k a</w>': 1, '   hon or in</w>': 1, '   1 8 6 8 </w>': 1, '   fi t ful</w>': 2, '   son u v a bu ck</w>': 2, '   tur t le sh ell</w>': 1, '   me ss in</w>': 7, '   so bri e ty</w>': 2, '   bur a</w>': 1, '   min e o</w>': 1, '   my th s</w>': 6, '   3 0 2 </w>': 1, '   ta k u</w>': 1, '   wa k an</w>': 1, '   wan a g i</w>': 1, '   ind</w>': 1, '   wa s i</w>': 3, ' c u</w>': 2, ' te</w>': 1, '   ou thou se</w>': 1, '   hi gh ba lling</w>': 1, '   shi tter</w>': 4, '   wa mb l i</w>': 1, '   r es</w>': 4, '   ta g g in</w>': 1, ' pr in t in</w>': 1, '   s ac ra ment</w>': 2, '   tri ck ster</w>': 1, '   e le c t ro sta ti c</w>': 1, '   p sy cho lin gu si ti c s</w>': 1, '   fin ger pr in ting</w>': 2, '   sc or pi ons</w>': 2, '   mu ff l er</w>': 3, '   b al ing</w>': 1, ' sh e e</w>': 1, '   du m pl ing</w>': 3, '   wa sh e e</w>': 1, '   man i pu la tion</w>': 3, '   de se cra ted</w>': 2, '   ma g de lan a</w>': 2, '   h er e di t ary</w>': 2, ' shi ft</w>': 4, '   por cu p ine</w>': 3, '   g un g</w>': 4, '   d or m ant</w>': 2, '   co in te l pr o</w>': 1, '   te ton</w>': 1, '   ta sh k a</w>': 3, '   sh a</w>': 16, '   b ack do or</w>': 1, '   con fin e ment</w>': 2, ' c us</w>': 1, '   ro ar</w>': 5, ' ho te l</w>': 2, ' in ver se</w>': 1, '   su ch i</w>': 1, ' bra ver</w>': 1, '   si mp li fi es</w>': 1, '   su b ver ting</w>': 1, '   m ea t ba lls</w>': 7, ' play</w>': 7, '   fo e</w>': 4, '   di sen ga ge</w>': 4, '   win g man</w>': 4, '   ho t sho ts</w>': 2, '   pro du c tive</w>': 10, '   in su l ter</w>': 1, '   in ver ted</w>': 4, '   sle e k</w>': 1, ' ma tch es</w>': 1, '   ye ow</w>': 1, '   cou g ar</w>': 14, '   ra g ging</w>': 2, '   stra ps</w>': 14, '   l or dy</w>': 1, '   bur ps</w>': 2, '   st ro be</w>': 4, '   2 9 0</w>': 2, '   y a w ing</w>': 1, '   ma v </w>': 4, '   cha ff</w>': 1, ' so l o</w>': 2, '   tom c at</w>': 1, '   i ce man</w>': 2, '   k a z ans k y</w>': 1, '   s li der</w>': 2, ' self</w>': 5, '   im mo la tion</w>': 1, '   a ir spe ed</w>': 2, '   ke ll er</w>': 2, '   m c g own</w>': 1, '   z i pp ers</w>': 3, ' pla y ed</w>': 1, '   go on ed</w>': 1, '   k a z ans k i</w>': 1, '   bo ge ys</w>': 1, '   co o g an</w>': 4, '   win der</w>': 1, '   1 2 4 </w>': 4, '   bar ga mi an</w>': 1, '   gu e ss es</w>': 3, ' ma ver i ck</w>': 1, '   son o c o</w>': 1, '   co o g</w>': 1, '   ta x pa y ers</w>': 5, '   t ea m work</w>': 4, '   to p gu n</w>': 1, '   v f</w>': 1, '   ori s k any</w>': 1, ' c n c</w>': 1, '   ri ch to f en</w>': 1, ' mi ssed</w>': 1, '   p ee ling</w>': 3, ' t ea m work</w>': 1, '   ki ck back</w>': 3, '   d om es</w>': 2, '   p sy chi c s</w>': 2, '   mu ff le</w>': 1, '   b ru ba k er</w>': 2, '   6 1 0</w>': 1, '   en co de</w>': 1, '   qu a id</w>': 27, '   ha u s er</w>': 15, '   tur b in i u m</w>': 2, '   co ha a g en</w>': 15, '   me lin a</w>': 4, '   min d fuck</w>': 4, '   ri ch ter</w>': 11, '   ho d</w>': 1, ' fran k ly</w>': 1, '   re k all</w>': 16, '   k u a to</w>': 10, ' im plan ta tion</w>': 1, '   l or i</w>': 6, '   d ou gla s</w>': 8, '   r en at a</w>': 3, '   e m bo li s m</w>': 5, '   he ter o</w>': 1, '   spe ci fi ed</w>': 4, '   in ven ting</w>': 5, '   fo lled</w>': 1, '   p y ra mi d</w>': 9, '   bl ab bed</w>': 2, ' ne ar ly</w>': 1, '   lo bo tom i z ed</w>': 2, ' re k a ll re k a ll re k all</w>': 1, '   ga ll er i a</w>': 1, '   por kin</w>': 2, '   joh n n y ca b</w>': 2, '   a g gh h</w>': 1, ' spi es</w>': 2, '   t an ta li ze</w>': 1, ' e go</w>': 1, '   fami li ar i ze</w>': 2, '   can al s</w>': 1, '   ven u s vi ll e</w>': 1, '   c ru i ses</w>': 1, ' bo b</w>': 6, '   m c cl an e</w>': 5, ' rea ctor</w>': 1, '   se tt l ers</w>': 2, '   j ack as ses</w>': 2, '   ha u l ed</w>': 9, '   bur t</w>': 23, '   hea ther</w>': 70, '   r hon da</w>': 13, '   le be ck</w>': 2, '   4 5 8 </w>': 1, '   so li ds</w>': 2, '   v al</w>': 16, '   sp o or</w>': 2, '   mi gu el</w>': 9, '   p ha m</w>': 8, '   lo an ed</w>': 8, '   bi x by</w>': 10, '   pre da te</w>': 1, ' co ll ar</w>': 4, ' ge o gra p hi c</w>': 1, '   la s so</w>': 1, '   j er k off</w>': 2, '   f la t bed</w>': 2, '   mi tt s</w>': 3, '   se i s mo s</w>': 1, '   t wi tty</w>': 1, '   can n ers</w>': 1, '   sta lled</w>': 4, ' fi x ing</w>': 1, '   car wa sh</w>': 2, '   s om body</w>': 1, '   c ac tu s</w>': 12, '   ne st or</w>': 4, '   1 9 6 8 </w>': 5, '   se i s mo gra ph s</w>': 3, ' di g ging</w>': 1, '   han d y men</w>': 3, '   co tt on w o od</w>': 1, '   sta m pe de</w>': 3, '   v ar min ts</w>': 1, '   no wh er es</w>': 4, '   s cu m su ck ers</w>': 1, '   ra di al s</w>': 2, '   p le i sto cen e</w>': 1, '   a ll u vi al s</w>': 1, '   ba da la to</w>': 3, '   co lo m bi an</w>': 7, '   e sp ar z a</w>': 8, '   ch in at own</w>': 25, '   d ow d</w>': 19, '   sh u</w>': 30, '   v in</w>': 5, '   re y n ard</w>': 13, '   bo a sted</w>': 2, '   or te g a</w>': 6, ' in ten tion ed</w>': 1, ' ex a min ing</w>': 1, ' ex a m ine</w>': 4, '   mo d us</w>': 4, '   op er an d i</w>': 4, '   ch u c ki e</w>': 18, '   ro e der</w>': 8, '   r en o v a ting</w>': 2, '   b ran de is</w>': 1, ' di r ty</w>': 2, '   ma p p</w>': 2, '   stu b by</w>': 1, '   s nu b</w>': 3, '   c y lin d ers</w>': 1, '   k a i</w>': 8, ' de sc ri p tion</w>': 1, '   sti pe</w>': 8, '   s ea t be lt</w>': 5, '   ho li er</w>': 5, '   ro lo de x</w>': 4, '   1 5 3 0</w>': 2, '   ri v in g ton</w>': 1, '   be sts</w>': 1, '   bur n out</w>': 3, '   me th</w>': 9, '   u p stan ding</w>': 4, '   te ar dro ps</w>': 3, '   mu g sho t</w>': 2, ' ce ci l</w>': 2, '   d d</w>': 2, '   af fa da v it</w>': 2, ' no ve mber</w>': 1, '   1 9 8 0</w>': 7, '   re op en</w>': 3, '   lin de man</w>': 2, '   ex al ted</w>': 1, '   spe ci a li ze</w>': 4, '   ven er ate</w>': 1, '   ch in a men</w>': 2, '   can ton e se</w>': 4, '   man dar in</w>': 3, ' bo o ze</w>': 2, ' ro e der</w>': 1, '   ho t p an ts</w>': 1, '   re ta il er</w>': 2, ' d ow d</w>': 1, '   e m be lli sh ed</w>': 2, '   par ti ci pa te</w>': 2, '   p c p</w>': 2, '   s li me do gs</w>': 1, '   sen i ors</w>': 6, ' ou gh ta</w>': 3, '   r ab in</w>': 3, '   w r it</w>': 3, '   tru th ful</w>': 5, '   vi bra tes</w>': 2, ' ci l</w>': 2, '   sh ow bo a ting</w>': 2, '   ar tur o</w>': 3, '   s k l ar off</w>': 1, '   mon tell</w>': 1, '   as si sted</w>': 2, ' ca lls</w>': 2, '   cont in u an ce</w>': 3, '   m ac h in a tions</w>': 1, '   mar sha lls</w>': 1, '   su b stan ti ate</w>': 1, '   ba d ger ing</w>': 2, '   di a le c ts</w>': 3, '   pro se cu ting</w>': 3, '   stra y ed</w>': 2, '   o a h u</w>': 2, '   wai ving</w>': 1, '   re con vi ct</w>': 1, '   con se cu ti ve ly</w>': 1, '   con cu r r ent</w>': 1, '   re op en ed</w>': 1, '   du m pl in gs</w>': 4, '   o cho a</w>': 1, '   tra m pl es</w>': 1, '   pu sh ers</w>': 4, '   pro se cu t ors</w>': 3, ' poli ce</w>': 1, '   sti p es</w>': 1, '   ri ver head</w>': 1, '   su m ma tion</w>': 4, '   an th o lo g y</w>': 1, '   so l da do</w>': 5, '   comp ani a</w>': 3, ' ta kes</w>': 4, ' ma ter i el</w>': 1, '   ma ter i el</w>': 2, '   re gi men ts</w>': 2, ' la w y er</w>': 2, ' ta k en</w>': 1, '   a bu n dan ce</w>': 2, '   kn ow in g ne ss</w>': 1, ' ter e s a</w>': 1, '   la x a tive</w>': 6, '   di v v y</w>': 1, '   ss sh h h h</w>': 1, '   dri f ter</w>': 8, ' fe der al</w>': 2, '   cha l f on t</w>': 2, '   cha l f on ts</w>': 1, '   ro d d</w>': 1, '   ch et</w>': 35, '   ha p</w>': 11, '   y a ki ma</w>': 1, '   je f fri es</w>': 3, '   whi te man</w>': 3, '   e s the ti c s</w>': 1, '   fa u l ty</w>': 6, '   e s the ti c</w>': 1, ' dis cu ssion</w>': 1, '   re che ck</w>': 1, '   imp r in ted</w>': 1, '   con tu sion</w>': 2, '   an g l ed</w>': 1, '   ta il or ed</w>': 3, '   sh or th and</w>': 6, '   j ac qu es</w>': 29, '   mu f fin s</w>': 2, ' la st ing</w>': 1, ' someone</w>': 10, '   the or e ms</w>': 1, ' ha ving</w>': 3, '   h or n e</w>': 3, '   bo x er</w>': 8, '   s an ty</w>': 1, ' j ac qu es</w>': 1, '   go b ble</w>': 3, '   ph ys</w>': 1, '   ni co ll et</w>': 1, '   mar que tt e</w>': 1, '   exac t ment</w>': 1, '   u v u l a</w>': 1, '   he im li ch</w>': 1, '   tr ac he o tom y</w>': 1, '   sp l at</w>': 1, '   jo si e</w>': 45, '   a sp ar a gu s</w>': 3, '   dar ted</w>': 1, '   h y g ge li g</w>': 1, '   mo te</w>': 1, '   je g</w>': 1, '   he ter</w>': 1, '   ti le</w>': 3, '   sh e lly</w>': 73, '   b en ni es</w>': 1, ' m ea ls</w>': 2, '   nor ma</w>': 39, '   ab ra ms</w>': 5, '   ro on ey</w>': 15, '   con can n on</w>': 15, '   ar ch di o ce se</w>': 9, '   in di c ted</w>': 13, '   g al v in</w>': 32, '   mor ri s se y</w>': 6, ' ev enty</w>': 3, ' h ree</w>': 2, '   bro ph y</w>': 4, '   d on e gh y</w>': 12, '   de bor ah</w>': 41, '   k a ye</w>': 24, '   g ru b er</w>': 20, '   t ow l er</w>': 22, '   ad mi tt an ce</w>': 10, '   p ho to co p y</w>': 5, '   d rs</w>': 2, ' s wor n</w>': 2, ' cer ti fi ed</w>': 4, ' e th o do lo g y</w>': 3, '   an e s the si o lo g y</w>': 10, '   or th o pe di c s</w>': 2, '   ne u ro lo g y</w>': 2, ' u r pri se</w>': 1, '   re but</w>': 2, '   ev i den ti ary</w>': 2, '   di sa ll ow ed</w>': 10, '   un su b stan ti a ted</w>': 4, '   ci te</w>': 4, '   m c ge e</w>': 2, '   1 3 1 </w>': 2, '   2 1 6 </w>': 2, '   pre supp ose</w>': 3, '   r i</w>': 3, '   v in di ca tion</w>': 5, '   a spi ra ted</w>': 4, '   v om i tu s</w>': 2, '   tr ac he a</w>': 2, ' co de</w>': 9, ' an e s the ti st</w>': 2, '   de b by</w>': 2, '   af fir ma ti ve ly</w>': 2, '   an e s the si o lo gi st</w>': 4, ' ru th</w>': 1, '   ha g man</w>': 2, '   d rea d fu lly</w>': 3, '   wor k in g man</w>': 4, ' a part ment</w>': 2, '   ne g li g ence</w>': 10, '   con sti tu tes</w>': 4, '   ea s th a mp ton</w>': 2, '   l ab ou re</w>': 6, '   an e s the ti c s</w>': 2, '   an e s the ti c</w>': 16, '   in du ce ment</w>': 4, '   4 1 4 </w>': 5, ' pr ac ti ce</w>': 2, '   an a e s the si a</w>': 2, '   4 0 6 </w>': 2, ' con tra in di ca tions</w>': 2, '   in du ction</w>': 3, '   ar ch di o ce ses</w>': 2, ' e sta te</w>': 3, ' de bor ah</w>': 2, '   nu mer al</w>': 2, '   si m mon ds</w>': 2, '   so le</w>': 14, '   o ver ru l ed</w>': 4, '   mi st ri al</w>': 9, '   ba i li ff</w>': 4, '   dis qu a li f y</w>': 3, '   tr an sc ri pt</w>': 7, '   li lli bri dge</w>': 2, '   con fine</w>': 3, '   pa d re</w>': 2, '   pre s sur ing</w>': 7, '   re ver sa l</w>': 3, '   chi l d care</w>': 2, ' su b sc ri p tion</w>': 2, '   la p sed</w>': 5, '   qu ar ter ly</w>': 2, '   a li to</w>': 2, '   tu c son</w>': 4, '   la u gh l in</w>': 5, '   fran k y</w>': 17, ' p at</w>': 4, '   d or che ster</w>': 2, ' k a th y</w>': 2, '   ob st e tri c</w>': 4, ' r our ke</w>': 2, '   m ac an u do s</w>': 2, '   cu e st a</w>': 2, '   f l ack</w>': 5, '   g b h</w>': 2, '   bu sh mi lls</w>': 2, '   m cl ean</w>': 2, '   br in dis i</w>': 3, ' ary</w>': 2, '   ki l d are</w>': 3, ' g ru b er</w>': 2, '   sur f bo ard</w>': 4, '   h y an n is</w>': 2, '   le g work</w>': 3, '   ad min i st er ing</w>': 2, '   no ta tions</w>': 4, '   ro ving</w>': 3, '   har r in g ton</w>': 30, '   mo p es</w>': 2, '   ju r or</w>': 2, ' p in na k er</w>': 2, ' fran k y</w>': 1, '   st ear n s</w>': 8, '   d or se t shi re</w>': 2, '   dar ne de st</w>': 1, '   un sc re w ing</w>': 1, '   un sc re w</w>': 2, '   mar ti ans</w>': 4, '   vi ou s ly</w>': 1, '   fu tu ri sti c</w>': 1, '   ba w l</w>': 2, '   cor on a</w>': 1, '   f ar m house</w>': 2, '   bo g any</w>': 1, '   bur en</w>': 3, '   he ff ner</w>': 2, '   i g ni tions</w>': 1, '   in su la ted</w>': 3, ' i me</w>': 2, ' s li gh tly</w>': 3, '   for re ster</w>': 8, '   s ci en</w>': 1, '   ti sts</w>': 1, '   s ki d ded</w>': 1, '   cra ter</w>': 3, '   gi z m o</w>': 2, '   gra t z man</w>': 2, '   bi o ti c s</w>': 1, '   an a e mi c</w>': 1, '   for l or n</w>': 1, '   sp u r ts</w>': 4, '   per su a der</w>': 1, '   man n</w>': 2, '   p om on a</w>': 4, '   t an k er</w>': 12, '   ta ma les</w>': 1, '   en chi la da s</w>': 1, '   cla mor ing</w>': 1, '   re sur re ct</w>': 3, ' e pi so de</w>': 1, '   p al m da le</w>': 7, '   supp l ying</w>': 4, '   bu b b les</w>': 6, '   cra f ts</w>': 4, '   ani ma tr on i c s</w>': 3, '   ten d ons</w>': 2, '   do ber man</w>': 2, '   sle e p wal ks</w>': 2, '   se da ted</w>': 12, '   lan gen k a mp</w>': 5, '   st at</w>': 6, '   qu a ke</w>': 4, '   f an ta si z es</w>': 1, '   a i ls</w>': 1, ' p ra y</w>': 3, ' trying</w>': 5, '   brea d c ru mb s</w>': 1, '   gre te l</w>': 2, '   k in dle</w>': 1, '   kn ea ded</w>': 1, ' ev il</w>': 4, '   de ca pi ta tion</w>': 4, '   sa x on</w>': 1, '   in f lu en c ing</w>': 2, '   inter e st in g ly</w>': 1, '   st or y te ll ers</w>': 1, ' en ti ty</w>': 1, '   ri sh er</w>': 1, '   bu mp y</w>': 2, ' n one</w>': 4, '   ri g ging</w>': 2, ' tal es</w>': 1, '   j i be</w>': 3, '   j i b</w>': 3, '   for e sta y sa il</w>': 1, '   ma in sh e et</w>': 1, '   a lo ft</w>': 4, '   un stop</w>': 1, '   for e course</w>': 2, '   p in ra il</w>': 1, '   bu n t lin es</w>': 1, '   cle w lin es</w>': 1, '   u p w ard</w>': 5, '   gi e g</w>': 2, '   w ea th ers</w>': 25, '   la p chi ck</w>': 1, '   sch u car t</w>': 1, '   cor ry</w>': 1, '   st ri ck l in</w>': 1, '   bar n es</w>': 70, '   b ow sp r it</w>': 1, ' b ow sp r it</w>': 1, '   b re gi tta</w>': 2, '   do c t or ed</w>': 5, '   s nor ing</w>': 4, ' wi lli am</w>': 3, '   tri bu na l</w>': 13, '   pi ss in</w>': 6, '   sch o on er</w>': 4, '   sh el d ra ke</w>': 40, '   bri g an t ine</w>': 1, '   be e ch</w>': 1, '   k en n et</w>': 1, ' u sh room</w>': 1, '   ta b or</w>': 1, '   do l ph in</w>': 5, '   un fu r l</w>': 1, '   cu r ac a o</w>': 2, '   com pe lling</w>': 2, '   ro mp</w>': 2, '   pro mp t ne ss</w>': 1, '   ser vi tu de</w>': 2, ' o c to b er</w>': 1, '   b ou ti lli er</w>': 3, '   than k less</w>': 2, '   dis re spe c t fu lly</w>': 1, '   por k cho p</w>': 1, '   p ee l in</w>': 1, '   go o d all</w>': 1, '   a ll e g ory</w>': 1, '   rea l ms</w>': 2, '   go o d ly</w>': 3, '   bar ds</w>': 1, '   f ea l ty</w>': 1, '   a po llo</w>': 15, '   of t</w>': 3, '   ex p an se</w>': 2, ' br ow ed</w>': 1, '   de me s n e</w>': 1, '   han d k er chi e f s</w>': 1, '   mer is</w>': 1, '   cor k er</w>': 1, '   f an dan go</w>': 1, '   su i ci da l</w>': 15, '   run g</w>': 3, '   a a au u u h h h</w>': 1, '   for e to p</w>': 1, '   s qu all</w>': 2, '   i ma g in able</w>': 5, '   ac ro p ho bi c</w>': 1, '   sp ar ed</w>': 5, ' ea g le</w>': 1, ' bea m</w>': 1, '   ch u ck y</w>': 7, '   v a se l ine</w>': 1, ' for ge tta</w>': 1, '   ar ti e</w>': 23, '   en ter ta in in</w>': 1, '   in f ar ction</w>': 1, '   f l ab </w>': 1, '   un wa x ed</w>': 1, '   cont a</w>': 1, '   j or as</w>': 1, '   per u</w>': 10, '   every th in</w>': 6, '   ca o</w>': 2, '   sto ck in</w>': 1, '   i th ac a</w>': 1, '   el do</w>': 1, '   lu l a</w>': 49, '   ra i s in</w>': 6, '   fa m</w>': 1, '   stu d y in</w>': 3, '   lo b o</w>': 4, '   i gu an a</w>': 3, '   re st in</w>': 2, '   ro s ar i ta</w>': 1, '   chi c a</w>': 3, '   ju an a</w>': 1, '   ri l ed</w>': 2, '   la fi tt e</w>': 2, '   g al ve z</w>': 1, '   8 6 </w>': 3, '   z an z i b ar</w>': 2, '   mar i e tta</w>': 16, '   sle u th in</w>': 1, '   f ar ra gu t</w>': 11, '   pe dr o</w>': 15, '   su l a</w>': 8, '   lea ch</w>': 2, '   car y</w>': 9, '   u ti l a</w>': 1, '   mu c has</w>': 2, '   gr ac i as</w>': 2, '   mi lli me t ers</w>': 1, '   per mi so</w>': 1, '   mu ss</w>': 5, '   v f w</w>': 1, '   k o vi ch</w>': 1, '   man s la u gh ter er</w>': 3, '   par en tal</w>': 5, '   s na ke s kin</w>': 3, '   in di vi du a li ty</w>': 4, '   ha st a</w>': 3, '   si e mp re</w>': 2, '   hon du ran s</w>': 1, '   o s v al do</w>': 1, '   ta mar in do</w>': 1, '   te le f on o</w>': 2, '   6 6 6 </w>': 2, '   cap ac i ti es</w>': 1, ' si z ed</w>': 7, '   pu c in s k i</w>': 2, '   u p se tt in</w>': 2, '   ga la to i re</w>': 1, '   pi e d mon t</w>': 1, '   de f y in</w>': 1, '   f re ch man</w>': 1, '   un fi x</w>': 1, '   st re e t cor ner</w>': 1, '   in vo l v in</w>': 1, '   pro te c t in</w>': 1, '   than kin</w>': 2, '   for ce ful</w>': 2, '   vo li tion</w>': 4, '   lea kin</w>': 1, '   de f en d in</w>': 1, '   sin cer e ly</w>': 13, '   fi ll in</w>': 4, '   bar f ed</w>': 3, '   th ri ll in</w>': 1, '   com for t in</w>': 1, '   inter tw in ed</w>': 2, '   sho ck in</w>': 1, '   bur n in</w>': 3, '   bu il d in</w>': 6, '   tr ou bl in</w>': 1, '   pr ac ti c in</w>': 3, '   hi tch ers</w>': 1, '   dri pp in</w>': 1, ' s me ll in</w>': 2, '   be i g n et</w>': 1, '   mon de</w>': 1, '   he x</w>': 3, '   o d der</w>': 1, '   f re e z in</w>': 2, '   par ac hu tes</w>': 3, '   m oun ds</w>': 4, '   in ser t in</w>': 1, ' ta li an</w>': 1, '   bar fi ght</w>': 1, '   for ear ms</w>': 2, '   l ani er</w>': 1, '   bar b ers</w>': 1, '   bi lo x i</w>': 2, '   mor es</w>': 3, ' p p</w>': 1, '   fa d in</w>': 2, '   li gh t n in</w>': 1, '   be d sp rea ds</w>': 2, '   ra ma da s</w>': 1, '   dis gu st in</w>': 1, '   c ro ss in</w>': 3, '   v an ta g es</w>': 1, '   por ta ge e</w>': 1, '   p ee ls</w>': 4, ' pu sh ed</w>': 1, '   ex ci t in</w>': 1, '   ir ma</w>': 1, '   vi si t in</w>': 4, '   ho ok in</w>': 2, '   fi x ation</w>': 3, '   ro o ti e</w>': 4, '   mi ss in</w>': 6, '   an us</w>': 11, '   ob ser v in</w>': 1, '   be ha v in</w>': 1, '   cha tt an oo g a</w>': 4, '   coun t in</w>': 4, '   ab or tion i st</w>': 1, '   da t in</w>': 2, '   di sa pp ear in</w>': 1, '   ho li da y in</w>': 1, '   de ci be ls</w>': 4, '   gr in ding</w>': 4, ' gen t le</w>': 1, '   da y room</w>': 2, '   cu r l ers</w>': 1, '   de so to</w>': 3, '   sho ved</w>': 6, '   cor n er ed</w>': 7, '   cle an in</w>': 7, '   mar l bor o s</w>': 5, '   k oo ls</w>': 2, '   p ow er ma d</w>': 1, '   g an g es</w>': 2, '   v ar an as i</w>': 1, '   d ev our</w>': 3, '   f lo at in</w>': 3, '   cre ma tion</w>': 3, '   u tt ar</w>': 1, '   p ra de sh</w>': 1, '   tur t les</w>': 5, '   b re e d ers</w>': 3, '   bu n g l ed</w>': 2, '   an nu ally</w>': 2, '   s ca ven ge</w>': 1, ' he se</w>': 1, '   sta te l ine</w>': 1, '   ha v an a</w>': 14, '   ar gen t in a</w>': 8, ' joh n ni e</w>': 1, '   re in de er</w>': 4, ' l m</w>': 1, '   s n ort in</w>': 1, '   di still</w>': 1, '   ei gh ts</w>': 4, '   tri p p</w>': 35, '   h h huh</w>': 1, '   pi ll ows</w>': 6, '   gr an</w>': 3, '   can de l ab ra s</w>': 1, '   ne m o</w>': 1, '   sp r in ging</w>': 5, '   le er</w>': 8, '   n ar ra ting</w>': 1, '   ga s k ell</w>': 9, '   oo l a</w>': 1, '   de pre ci ate</w>': 1, '   ha d le y</w>': 3, '   m ac ca u la y</w>': 1, '   car l y le</w>': 2, '   ju mp er</w>': 6, '   har da pp le</w>': 5, '   pu p ci k</w>': 1, '   gen et</w>': 1, '   a il</w>': 4, '   s ki mm ed</w>': 1, '   gar g ling</w>': 2, '   me th a mp he ta min es</w>': 1, '   cla u d ell</w>': 1, '   f l y we ight</w>': 3, '   pri mar i ly</w>': 1, '   ph ar m ac o po ei a</w>': 1, '   que er ed</w>': 1, '   s lo vi a k</w>': 2, '   bri mm ing</w>': 1, '   fa u l kn er i an</w>': 1, '   ga la x i e</w>': 2, '   inter ni st</w>': 2, ' spe lls</w>': 1, '   mor m on</w>': 4, '   cu p ca ke</w>': 3, '   gra dy</w>': 54, ' re con ci le</w>': 1, '   k in ship</w>': 5, '   ca s ks</w>': 1, '   a mon ti ll a do</w>': 1, '   car ve l</w>': 4, '   sc ran ton</w>': 3, ' e pi so d es</w>': 1, '   si m pa ti c o</w>': 1, '   ar dent</w>': 1, '   af fini ty</w>': 3, '   ab sta in</w>': 1, '   co de ine</w>': 2, '   shi m my</w>': 1, '   out l ine</w>': 4, '   se i t z</w>': 1, '   gla u com a</w>': 1, '   hu m bo l d t</w>': 1, '   se wi ck le y</w>': 2, '   z o di ac </w>': 2, '   mm r r mm m</w>': 1, '   k na p</w>': 2, '   th a w</w>': 6, '   mm h mm m</w>': 1, '   t wi r ling</w>': 2, '   gar ment</w>': 2, '   tri g ger man</w>': 1, '   ti er ne y</w>': 1, '   gen ea lo gi es</w>': 1, '   c ri b bed</w>': 1, '   bor g es</w>': 1, '   se wi ck ly</w>': 1, '   h h</w>': 1, '   d ow ny</w>': 2, ' rea ding</w>': 3, '   s an d ers</w>': 4, '   im pro vi sing</w>': 2, '   pri ck ly</w>': 2, '   wor d fe st</w>': 1, ' saying</w>': 4, '   c ri sta i le</w>': 2, '   sp ou ses</w>': 3, '   f ra m er</w>': 1, '   be e t le</w>': 8, '   mer ri er</w>': 3, '   tra x l er</w>': 2, '   pa pri k a</w>': 1, '   we ll e sle y</w>': 2, '   ne u ro lo gi st</w>': 3, '   a a h by</w>': 1, '   pre ma tu re ly</w>': 2, '   an ge l i</w>': 1, '   bo y er</w>': 2, '   1 9 7 8 </w>': 4, '   bu tt er wor th</w>': 1, '   de k k er</w>': 1, '   lan d is</w>': 1, '   se ber g</w>': 1, '   1 9 7 9 </w>': 2, '   si o an e</w>': 1, '   su ll a v an</w>': 1, '   ve le z</w>': 1, '   bl an di ck</w>': 1, '   gi a</w>': 1, '   s ca i a</w>': 1, '   bra v a</w>': 1, '   la t our</w>': 3, '   ti g ers</w>': 13, '   cal</w>': 27, '   mar in ers</w>': 4, '   ye lls</w>': 5, '   con gre ga te</w>': 2, '   bo om ing</w>': 4, '   nor th land</w>': 4, '   in ten tly</w>': 1, '   wa tch er</w>': 2, ' wa tch er</w>': 1, '   vi re o</w>': 3, '   star lin gs</w>': 2, '   bi r ch</w>': 1, '   as sa il ant</w>': 3, '   w oo d s man</w>': 3, '   sy m bi o ti c</w>': 1, '   wan ger</w>': 1, '   con te mp or ary</w>': 3, '   sp ar r ow like</w>': 1, '   ro s en</w>': 8, ' plea sure</w>': 4, '   h y p no ti ze</w>': 3, ' problem</w>': 1, '   pre pu be sc ent</w>': 1, '   che er y</w>': 7, ' d ence</w>': 1, '   di ar i es</w>': 2, '   s ar ca s m</w>': 9, '   ch r on o lo gi cal</w>': 2, '   de p ra ved</w>': 5, '   pu rest</w>': 2, '   lu mb er y ard</w>': 3, '   s ea wa ter</w>': 1, '   lin d se y</w>': 18, ' pu sh er</w>': 2, ' ei gh th s</w>': 1, '   so ck et</w>': 4, '   u m bi li cal</w>': 4, '   un ho ok ed</w>': 2, '   lin s</w>': 7, '   b en th i c</w>': 3, '   de e p c ore</w>': 5, '   ja mm er</w>': 2, '   ki r k hi ll</w>': 4, '   hi pp y</w>': 8, '   bri g man</w>': 6, '   a a ar gh</w>': 1, '   to tal ed</w>': 2, ' s wi m</w>': 1, '   pla sti ci ze</w>': 1, '   po l y mer i ze</w>': 1, ' in du c ed</w>': 2, '   j ar head</w>': 2, '   mo ther f</w>': 1, '   co f fe y</w>': 6, '   cl un k y</w>': 1, '   t re mor s</w>': 4, '   s lu r red</w>': 2, '   z i g ging</w>': 1, '   ex t end</w>': 9, '   hea t ers</w>': 1, '   p ran ged</w>': 1, '   re gu la tor</w>': 2, '   su b mer si ble</w>': 3, '   lo v el or n</w>': 2, ' mi cha el</w>': 1, ' ro d</w>': 3, '   wi en er</w>': 3, '   wi l hi te</w>': 2, '   sch o en i ck</w>': 2, '   m c whi r ter</w>': 2, '   brea ch ed</w>': 7, '   war hea ds</w>': 3, '   mo on po o l</w>': 2, ' fi b</w>': 1, '   sy r in ge</w>': 3, '   nu kes</w>': 3, '   de ton a tor</w>': 4, '   s li ck er</w>': 1, '   fin l er</w>': 1, '   di e t z</w>': 2, '   di d d ly</w>': 3, ' fi sh</w>': 4, '   con v u l sing</w>': 2, '   mi l king</w>': 4, '   fi z z in</w>': 1, '   de com pre ss</w>': 2, '   wa ll er</w>': 1, '   e qu a li z ing</w>': 1, '   ro v </w>': 3, '   n t is</w>': 2, '   al bu qu er qu e</w>': 5, '   ra m j et</w>': 1, '   hu ev o s</w>': 1, '   ab y s sa l</w>': 1, '   in cu r sion</w>': 1, '   1 9 2 </w>': 1, '   mi r v s</w>': 1, '   ti me l ine</w>': 1, '   ma g ne t ome ter</w>': 1, '   h p n s</w>': 1, '   1 2 0 0 0</w>': 1, ' w re st ling</w>': 1, '   te ther</w>': 2, '   u fo s</w>': 1, '   0 6 5 </w>': 1, '   1 8 4 0</w>': 2, '   ki lo t ons</w>': 1, '   r in g side</w>': 4, '   t wi tch ing</w>': 1, '   8 5 0 0</w>': 1, '   4 8 0 0</w>': 2, '   bu g go</w>': 1, '   ex ci ta bi li ty</w>': 1, '   di sor i en ta tion</w>': 5, '   cor ra do</w>': 6, '   gi u li a</w>': 8, '   s an dr o</w>': 23, '   e tt ore</w>': 8, '   lu lls</w>': 1, '   im pro pri e ty</w>': 1, '   sha f for d</w>': 1, '   li sc a</w>': 4, '   in di sc re et</w>': 6, '   pa tri z i a</w>': 9, '   mer lu z z o</w>': 1, '   ba si lu z z o</w>': 2, '   man i fe sto es</w>': 1, '   li re</w>': 5, ' cla u di a</w>': 1, '   a w k war d ne ss</w>': 1, '   s qu ar e ly</w>': 2, '   on e self</w>': 4, '   mon tal do</w>': 2, '   ha u gh ty</w>': 2, '   d y na mo s</w>': 1, '   qui e ter</w>': 2, '   ra i mon do</w>': 5, '   nu d es</w>': 6, '   ger ani u ms</w>': 1, '   p an ar e a</w>': 2, '   li par i</w>': 1, '   con cu b in age</w>': 1, '   a e o li an</w>': 1, '   i s les</w>': 1, '   vo l can o es</w>': 2, '   com par i s ons</w>': 1, '   h in d ran ce</w>': 2, '   b ou l d ers</w>': 1, '   mi la z z o</w>': 1, '   ci r cu m sc ri bed</w>': 1, '   go ff re do</w>': 2, '   re mb ran d t</w>': 11, '   con tra di ct</w>': 2, '   gar den ers</w>': 1, '   ca t ani a</w>': 1, '   cre v as ses</w>': 1, '   tr in ac ri a</w>': 1, '   re g in a</w>': 4, '   v a li se</w>': 1, '   di ssi pa tions</w>': 1, '   de gra da tions</w>': 1, '   in fi de li ti es</w>': 1, '   de ba u ch er i es</w>': 1, '   la z in ess</w>': 4, '   un wi ll in g ne ss</w>': 3, '   st or e kee per</w>': 2, '   t ro in a</w>': 2, '   z u ri a</w>': 1, '   no vi ce</w>': 3, '   men st ru ation</w>': 3, '   sta in ed</w>': 9, '   e qui v al ent</w>': 7, '   wa f er</w>': 1, '   h in den bur g</w>': 3, '   b li mp</w>': 5, '   mi ri am</w>': 6, '   be ll t ower</w>': 2, '   than k you</w>': 3, '   wa st e pa per</w>': 2, '   b ru s se l</w>': 2, '   sp r ou ts</w>': 2, '   sur r oun ds</w>': 3, '   h y p no ti sed</w>': 1, '   whi sp ers</w>': 9, '   s qu ee z es</w>': 2, '   men st ru a ting</w>': 1, '   mar t in ea u</w>': 2, ' v ani ll a</w>': 1, '   i ce c rea m</w>': 1, '   ca ther in es</w>': 1, '   tu n</w>': 1, '   c r y pt</w>': 2, '   ex pla in able</w>': 1, '   p sy chi a try</w>': 5, '   ha ll u c in a tes</w>': 1, '   sp on t an e ou s ly</w>': 5, '   h y p no ti s m</w>': 1, '   sor e ly</w>': 1, '   ma g de l en</w>': 1, '   par take</w>': 4, '   i g na ti us</w>': 1, '   fi l ter ed</w>': 2, '   as ce ti c s</w>': 1, '   pr ac </w>': 1, '   c li tu s</w>': 1, '   po l k a do t</w>': 1, ' ca th o li c</w>': 1, '   h er e ti c s</w>': 1, '   in du l gen ces</w>': 1, '   d ou b ly</w>': 3, '   a qui tt al</w>': 1, '   o c ca s</w>': 1, '   si on ally</w>': 1, '   bu lli ed</w>': 2, '   pre s su red</w>': 2, '   ca th o li ci s m</w>': 4, '   h y st er i c</w>': 2, '   ri di cu le</w>': 1, '   sp ee di ly</w>': 1, '   ex a min a tions</w>': 2, '   un fi l ter ed</w>': 1, '   s mo k er</w>': 4, '   con na ta tions</w>': 1, '   que u es</w>': 1, '   sc ru ff y</w>': 5, '   pa ddy</w>': 3, '   r in go</w>': 22, '   hea ved</w>': 1, '   que lled</w>': 2, '   dis ar med</w>': 2, '   ma u l er</w>': 1, '   man a ger i al</w>': 2, ' co s</w>': 14, '   de pre ss es</w>': 1, '   shu r r up</w>': 6, '   ger r on</w>': 2, ' ho o p</w>': 1, '   st ro pp y</w>': 2, '   pu l p</w>': 4, '   bri de well</w>': 1, '   si x p ence</w>': 4, ' ph on es</w>': 1, '   ar bi tr ary</w>': 3, '   lea ther y</w>': 1, '   ma ge e</w>': 2, '   sha m us</w>': 3, '   sa w bu ck</w>': 3, ' s na tch er</w>': 1, '   in f er i ori ty</w>': 2, '   pro gra m me</w>': 4, '   a miss</w>': 2, ' kn own</w>': 6, '   gi g g le</w>': 3, '   po sh</w>': 4, '   ca m pe y</w>': 1, '   chi ck y</w>': 1, '   v al u e less</w>': 1, '   g ro tty</w>': 3, ' di g</w>': 4, ' f ab </w>': 1, '   pi mp ly</w>': 2, '   h y per bo les</w>': 1, '   har t</w>': 1, '   co ke ar a ma</w>': 1, '   du ck y</w>': 1, '   ev en tu a li ty</w>': 1, '   ad en o i da l</w>': 1, '   g lo tt al</w>': 1, '   cra w l er</w>': 1, '   su l king</w>': 4, '   fi ck le</w>': 3, '   cl out</w>': 4, '   a un ti e</w>': 7, '   any ro a d</w>': 6, '   c ri sp s</w>': 1, '   to f fe e</w>': 4, '   t rea c le</w>': 4, '   s cu ll er y</w>': 1, '   me self</w>': 3, '   th i ck hea ded</w>': 1, '   bu tty</w>': 1, '   under stu dy</w>': 20, '   men ch</w>': 2, '   sp o i lt</w>': 2, '   bri s k et</w>': 2, ' bl ow</w>': 4, '   m ea su red</w>': 5, '   dri pp in gs</w>': 1, '   de f en ce less</w>': 1, '   r ou gh ed</w>': 2, '   pen si on er</w>': 4, '   que u e</w>': 2, '   b lo om in</w>': 2, '   je er ing</w>': 1, '   s cu ff</w>': 5, ' e d ged</w>': 1, '   sh ee ps</w>': 1, '   ho o ter</w>': 2, '   in vi tes</w>': 4, '   cor ne ts</w>': 1, '   tr un che on</w>': 2, '   bo y o</w>': 2, ' cl out</w>': 1, ' ta p</w>': 1, '   sa dis m</w>': 2, '   sta mp ed</w>': 11, '   ho ses</w>': 2, '   ru f fi ans</w>': 1, '   pa u ly</w>': 5, '   g ran dad</w>': 7, '   hu m our</w>': 6, '   d ra ma ti se</w>': 1, '   m c car t ne y</w>': 1, '   sto pp er</w>': 1, '   stra di v ar i us</w>': 1, '   to o l ed</w>': 1, '   da gen ha m</w>': 1, '   ga w</w>': 1, '   pa la ti al</w>': 1, '   ca ll ous</w>': 1, '   gi e</w>': 1, '   s wee t brea ds</w>': 2, '   ma t in e e</w>': 4, '   w o l ver ha mp ton</w>': 1, '   sp u r t</w>': 1, '   o ink</w>': 2, ' imp ort ant</w>': 4, '   pen man ship</w>': 1, '   bro ok</w>': 3, '   po tty</w>': 4, '   ra tt l er</w>': 1, '   cle op a tr a</w>': 4, '   as p</w>': 1, '   in b al an ce</w>': 1, '   can te en</w>': 1, '   k no tt ed</w>': 2, '   t wan ging</w>': 1, '   star k ey</w>': 1, '   re fini sh ed</w>': 1, '   che m in</w>': 1, '   b ac car at</w>': 2, '   b li me y</w>': 2, '   e le c tri ci an</w>': 8, '   mi s la id</w>': 2, '   ab ori g ine</w>': 1, '   ab ro a d</w>': 9, '   s mo ther ing</w>': 1, '   bl und er</w>': 2, '   un e qui pped</w>': 1, '   ro by</w>': 2, ' cra mp s</w>': 1, '   cha z</w>': 12, '   clo g ged</w>': 5, '   ba sa lt</w>': 1, '   ver ti cal</w>': 4, '   t an gen ti al</w>': 1, '   or bi ting</w>': 5, '   ter min a tor</w>': 11, '   li f ter</w>': 1, '   qu a ds</w>': 1, '   ma g ni fi ca tion</w>': 2, '   z e ta</w>': 1, '   re ti cu l i</w>': 1, '   con st e ll ation</w>': 4, '   ir th</w>': 6, '   sa li ent</w>': 1, '   sy st e ma ti z ed</w>': 1, '   inter ce p ted</w>': 5, '   6 7 8 </w>': 1, '   te ch no lo gi cal</w>': 2, '   ta pe wor m</w>': 2, '   bu l b s</w>': 3, '   in ju r ing</w>': 2, '   pro ds</w>': 1, '   me tal li te</w>': 1, '   ac i ds</w>': 1, '   sh ort cu t</w>': 4, '   ta ke off</w>': 3, '   fa ust</w>': 4, '   ven ti la tor</w>': 1, '   bi ds</w>': 7, '   br ou ss ard</w>': 9, '   com bu sti ble</w>': 1, ' hu ge</w>': 1, '   me l k on is</w>': 2, '   ca g</w>': 2, '   in ta kes</w>': 2, '   o ver hea ted</w>': 7, '   h y per sle ep</w>': 4, '   o x y gen a ted</w>': 2, '   f la me thr ower</w>': 5, ' c rea ture</w>': 2, '   reme di es</w>': 1, '   co b ble</w>': 1, ' ba it</w>': 1, '   min er al s</w>': 2, '   re pro du c tive</w>': 3, '   o v al</w>': 4, '   sp ore</w>': 2, '   st y li z ed</w>': 2, '   j ars</w>': 3, '   fi z z l ed</w>': 2, '   in ser ted</w>': 5, '   c leave</w>': 1, '   b li ps</w>': 1, '   h y per sp ace</w>': 4, '   e in st e in i an</w>': 1, '   vi su a li ze</w>': 5, '   s n ar k</w>': 1, '   s l ab </w>': 3, '   pi c t ori al</w>': 4, '   f an ci ful</w>': 3, '   ro ta tes</w>': 1, '   tra ve ll ers</w>': 2, '   fa un a</w>': 1, '   e co lo g y</w>': 3, '   di sor i en ting</w>': 1, '   ven om</w>': 8, '   de po si ting</w>': 2, '   st or my</w>': 5, '   au to do c</w>': 1, '   k ab loo ey</w>': 2, '   o ver he at</w>': 2, '   star dri ve</w>': 2, '   f er ti li ty</w>': 5, ' c y c le</w>': 4, '   ju mb le</w>': 2, ' f lo p</w>': 1, '   a ir sha ft</w>': 1, '   f ou ling</w>': 1, ' st or age</w>': 3, '   e je c ts</w>': 1, '   da ta sti ck</w>': 1, '   mi c ro or g ani s ms</w>': 1, '   d y n es</w>': 1, '   n on to x i c</w>': 1, '   un brea th able</w>': 1, '   sig na lled</w>': 2, '   t y co ons</w>': 2, ' p ack</w>': 3, '   cla m my</w>': 1, '   ca lli s to</w>': 1, '   min h</w>': 4, '   r h in o s</w>': 4, '   mi gra tion</w>': 2, '   h ow i t z er</w>': 2, '   cu l ver t</w>': 1, ' sho p</w>': 8, '   ca ssi e</w>': 3, '   d c m gs</w>': 1, '   in com in gs</w>': 2, ' i x ty</w>': 1, '   mar sc o</w>': 1, ' ba se</w>': 9, '   re mo tes</w>': 2, '   s li mm ed</w>': 1, '   un con fir med</w>': 1, '   no oo o o</w>': 6, '   re v na</w>': 1, '   ack land</w>': 5, '   in tr a</w>': 2, ' or bi tal</w>': 1, '   che w in</w>': 2, '   co j on es</w>': 1, '   me tr es</w>': 3, '   e d ging</w>': 1, '   ma mm ac i tta</w>': 1, '   go l d s mi th</w>': 1, ' ten si le</w>': 1, '   gu t ti er e z</w>': 1, '   sp oo king</w>': 1, '   qui e te st</w>': 2, ' p ack et</w>': 1, '   di ll er</w>': 4, ' pu mp</w>': 1, '   re pl ac es</w>': 4, '   f lu i di c</w>': 1, '   sh un t</w>': 1, '   gra ting</w>': 2, '   no gu ch i</w>': 1, '   ad m in</w>': 3, '   sha f ts</w>': 2, '   pro gra m ma ble</w>': 2, '   au to lo a der</w>': 1, '   dri s co ll</w>': 14, '   en ab ling</w>': 2, '   in ser t</w>': 6, ' ve e</w>': 1, ' ban ds</w>': 1, '   re p ea ter</w>': 2, '   ll o y ds</w>': 3, '   al man ac </w>': 3, ' re ff</w>': 1, '   hea d se ts</w>': 1, '   or g ani se</w>': 2, '   cor n er ing</w>': 1, '   cou pl ers</w>': 1, '   al d ho ven</w>': 1, ' sig n</w>': 3, '   g un bo at</w>': 1, '   se l t z er</w>': 1, ' w enty</w>': 1, '   da tu s</w>': 1, '   in sta ll a tions</w>': 1, ' cla u st ro p ho bi c</w>': 1, '   co ll ars</w>': 5, '   ber th a</w>': 1, '   sh ea thing</w>': 1, '   al ga e</w>': 1, ' pro gra mm ed</w>': 2, '   na v i</w>': 2, ' bea c on</w>': 1, '   lin son</w>': 1, '   de wi t t</w>': 15, '   ad di son</w>': 31, ' cor a</w>': 11, '   mar go</w>': 98, '   sle sc y n s k i</w>': 1, '   fi s ke</w>': 1, '   bl un tly</w>': 3, '   me lo d ra ma</w>': 2, '   s k o l</w>': 2, '   di mm ers</w>': 1, '   g y p sy</w>': 9, '   ri char ds</w>': 16, '   al co t t</w>': 1, '   pa ved</w>': 4, '   f an ta sti ca lly</w>': 3, '   f ab i an</w>': 9, '   bea ch head</w>': 1, '   shu ber t</w>': 3, '   un t ou ch ed</w>': 3, '   i do la try</w>': 2, '   chan ning</w>': 28, '   o ver c r ow ded</w>': 2, '   ca s well</w>': 15, '   co p ac ab an a</w>': 2, '   da i le y</w>': 1, '   as sur es</w>': 3, '   ab nor ma li ty</w>': 1, '   di sp l ac ed</w>': 2, '   chi lli co the</w>': 1, '   ga ther in gs</w>': 2, '   ne u ro ti c s</w>': 1, '   l un ch ing</w>': 3, '   s mar t ne ss</w>': 2, '   k a z o o</w>': 1, '   re ve la tion</w>': 6, '   tra pp i st</w>': 1, '   je an n e</w>': 75, '   ea ge ls</w>': 1, '   we s se ly</w>': 1, '   un pre g n ant</w>': 1, '   vi o l en tly</w>': 7, '   pro te ge e</w>': 1, '   l en ding</w>': 5, '   ti mi di ty</w>': 1, '   t ro p</w>': 2, '   z an u ck</w>': 2, '   f our s qu are</w>': 2, '   d ow n right</w>': 3, '   out guess</w>': 1, '   bar re</w>': 1, '   g no me</w>': 1, '   no g g in</w>': 2, '   sa l ted</w>': 2, '   har p y</w>': 2, '   par an o i ac </w>': 3, '   to l er a ted</w>': 2, '   g ong</w>': 3, '   fo l ded</w>': 10, '   pa der e w s k i</w>': 1, '   bi car b</w>': 2, '   p an try</w>': 3, '   e mb al m ing</w>': 3, '   d rea my</w>': 4, '   me ga lo man i ac </w>': 1, '   ad or ation</w>': 2, '   par an o i c</w>': 1, '   in se cu ri ty</w>': 6, '   la u g ha b ly</w>': 1, ' st ru ck</w>': 3, '   fro th</w>': 2, '   fi tch</w>': 2, '   be e hi ve</w>': 5, '   na v a j o</w>': 2, '   pre vi e w ed</w>': 1, '   pre vi e w s</w>': 3, '   kn it</w>': 5, '   thou gh t less</w>': 3, '   re li gi ons</w>': 1, '   di re c t ne ss</w>': 1, '   gr ac i ou s ly</w>': 2, '   wi gs</w>': 4, '   bu sh wa h</w>': 1, '   th ea tu h</w>': 3, '   i b s en</w>': 1, '   ber n har d t</w>': 1, '   po o d les</w>': 1, '   han ne for d</w>': 1, '   l un t</w>': 1, '   f on t an n e</w>': 1, '   e le an or a</w>': 1, '   d use</w>': 1, '   ex c lu si ve ly</w>': 6, '   ha li but</w>': 2, '   sa b les</w>': 2, '   ven tri lo qui st</w>': 2, '   bu t l ers</w>': 3, '   t ou chi est</w>': 1, ' uni on</w>': 3, '   gir d les</w>': 3, '   v au d ev i lli an</w>': 1, '   co on an</w>': 2, '   su m ter</w>': 2, '   car per</w>': 1, '   er as m us</w>': 2, '   s ni de</w>': 2, '   per su a ding</w>': 3, '   w ob b ly</w>': 4, ' d ra ma ti c</w>': 4, '   an a e s the ti c</w>': 8, '   nee di est</w>': 1, '   comp en sa tes</w>': 1, '   under pla ying</w>': 1, '   o ver pla ying</w>': 1, '   sa ble</w>': 2, '   be d j ack et</w>': 1, '   dis cou ra ging</w>': 3, '   gu sh ing</w>': 3, ' li li om</w>': 1, ' me mb ran ce</w>': 2, ' f oo t st e ps</w>': 3, '   s n ow y</w>': 1, '   v ar ni sh</w>': 2, '   gu i ld</w>': 15, ' se ven te en i sh</w>': 1, '   de cor a ting</w>': 4, '   f er ra day</w>': 1, '   u p sta ge</w>': 1, '   ar ch</w>': 8, ' pa d ded</w>': 1, '   stan i s la v s k y</w>': 1, '   e qui ty</w>': 9, '   en tr en ch ed</w>': 1, '   fir m ly</w>': 5, '   w oo ll co t t</w>': 1, '   fu mb le</w>': 2, '   o ver sen si tive</w>': 1, '   pro v in ci al</w>': 2, '   dis gr ac e fu lly</w>': 1, '   c ea sed</w>': 6, '   c rea m ing</w>': 1, '   mi s be have</w>': 2, '   un wanted</w>': 6, '   un lo ved</w>': 1, '   sh run k en</w>': 2, '   go ver ne ss</w>': 1, '   un y i el ding</w>': 1, '   de lin qu en ts</w>': 2, '   chan n in</w>': 1, '   sta h ved</w>': 2, '   un n er stand</w>': 4, ' sta h ved</w>': 1, '   su th</w>': 1, '   n ev ah</w>': 1, '   p ho bi a</w>': 2, '   un con cer n</w>': 1, '   e lo pe ment</w>': 1, '   s lin ger</w>': 2, '   dis lo y al ty</w>': 2, '   pla y wri gh ts</w>': 2, '   ge stu r es</w>': 6, '   dri ps</w>': 2, '   ro y al ti es</w>': 3, '   thr ow able</w>': 1, '   su per b ly</w>': 1, '   pre su mp tu ous</w>': 6, '   au di en ces</w>': 6, '   re f und ed</w>': 1, '   re think</w>': 3, '   in ac cu ra tely</w>': 1, '   go spe l</w>': 5, '   ven om ous</w>': 1, '   fi sh wife</w>': 1, '   tw en t y i sh</w>': 2, '   thir t y i sh</w>': 1, '   m ac be th i sh</w>': 1, '   gr ac i ou s ne ss</w>': 1, '   8 9 9 7 0</w>': 1, '   h or ow i t z</w>': 2, ' li e be stra u m</w>': 4, '   bur p</w>': 2, '   o ver sle pt</w>': 2, ' s lo an</w>': 1, '   hu n</w>': 9, ' mi tch ell</w>': 1, '   st ans</w>': 6, '   k al mb ac h</w>': 3, '   cla w s en</w>': 2, '   in i ti ation</w>': 6, '   ri te</w>': 3, '   tra i p ses</w>': 1, '   ch or ds</w>': 2, '   se go vi a</w>': 1, '   an dr es</w>': 1, '   se gre tt i</w>': 14, ' de st ru c ted</w>': 2, '   mu s ki e</w>': 6, '   de st ru c ted</w>': 1, ' ha l de man</w>': 1, '   s lu sh</w>': 3, ' co l son</w>': 1, '   o ver se er</w>': 2, '   shi t ki ck</w>': 1, '   dis lo y al</w>': 1, ' ab so lu tely</w>': 3, '   al p ha be ti ca lly</w>': 2, '   e ye br ow</w>': 4, ' 7 1 </w>': 4, '   be ll ho ps</w>': 2, '   bar k er</w>': 16, ' con di tion ing</w>': 2, '   si mon s</w>': 2, ' m ine</w>': 5, ' bra d le e</w>': 1, '   me te ori c</w>': 2, '   re wri ting</w>': 1, '   w r in ger</w>': 2, ' j ee ee ee ee e sus</w>': 1, ' j ee ee ee e sus</w>': 3, ' ac cor ding</w>': 1, ' fun d</w>': 2, '   da h l ber g</w>': 6, '   dar d is</w>': 4, ' te ddy</w>': 2, '   s cu tt le bu t t</w>': 3, ' h ow ard</w>': 1, '   ra t fucking</w>': 3, '   ra t fuck er</w>': 1, '   can u ck</w>': 2, '   can a di ans</w>': 5, '   h er t z</w>': 1, '   li fe st y les</w>': 4, '   de ser t ers</w>': 4, ' ber n st e in</w>': 1, '   be i ru t</w>': 1, '   f ra z z l ed</w>': 1, ' god dam n it</w>': 1, '   so l v ent</w>': 1, '   sh re d ding</w>': 4, ' g or don</w>': 2, '   dis bur se</w>': 2, '   ir w in</w>': 8, ' den i al</w>': 4, ' che cked</w>': 4, '   fin an ces</w>': 5, '   de st ru c ting</w>': 2, '   un na med</w>': 2, '   re pr in ting</w>': 1, '   di al ing</w>': 5, '   ver i fi es</w>': 2, '   ra ffer ty</w>': 3, ' mar il y n</w>': 1, '   re sig ning</w>': 2, '   ra t fuck ers</w>': 2, '   su b ver t</w>': 1, ' se gre tt i</w>': 1, '   se ar ed</w>': 4, ' me lo d ra ma ti c</w>': 1, '   men ta li ti es</w>': 1, '   mu ll en</w>': 1, ' cen tra l</w>': 3, '   w oo d st e in</w>': 1, ' cor re ction</w>': 1, '   s ou c i</w>': 1, '   re per cu ssi ons</w>': 3, ' 7 0</w>': 4, '   cap tion</w>': 1, '   er li ch man</w>': 1, '   sh e ph er ds</w>': 1, '   li k en</w>': 1, '   qu er y</w>': 1, '   t ro t s k y</w>': 1, '   st al in</w>': 3, '   br on c o</w>': 1, '   na gu r s k i</w>': 1, '   e le c t or al</w>': 4, '   re e le ction</w>': 3, '   co or din a tor</w>': 1, '   b ac h in s k i</w>': 1, '   pla y te x</w>': 1, '   se qu en c ed</w>': 1, '   s lu r r ing</w>': 1, '   den un ci a tions</w>': 1, '   th u mb su cking</w>': 1, '   par ti ci pa tion</w>': 1, '   de pl or able</w>': 2, '   me li ss a</w>': 8, '   de po si ted</w>': 2, '   o st re i ch er</w>': 3, '   re ci pro cal</w>': 1, '   ch ee se b all</w>': 1, '   mo cha c c in o</w>': 1, '   b ac chan a li a</w>': 1, '   n y mp h s</w>': 1, '   u l tra do g</w>': 2, ' n ever mind</w>': 1, '   sti f l er</w>': 14, '   vi ck y</w>': 9, ' mo ch a</w>': 1, ' c c in o</w>': 1, ' oo h h h h</w>': 1, '   mo ca sh</w>': 1, ' ch in o</w>': 1, '   s l ac king</w>': 6, '   l ac ro s se</w>': 8, '   lan sing</w>': 1, '   nor th we st er n</w>': 4, ' ch r is</w>': 1, ' ki cked</w>': 2, '   pri ss</w>': 4, '   pi ge on ho le</w>': 1, '   ki ck ass</w>': 2, '   de ca pi ta te</w>': 1, '   m c f er r in</w>': 1, '   c li cked</w>': 5, ' je ssi c a</w>': 2, ' o h h h</w>': 1, ' to ting</w>': 1, '   y ee ee ee ea a a a w w w w w w</w>': 1, '   minu te man</w>': 1, '   pri med</w>': 4, '   k un g</w>': 10, '   p on ti fi ca ted</w>': 1, '   po stu red</w>': 1, '   pro cra st in a ted</w>': 1, ' p act</w>': 1, ' o l der</w>': 1, ' you th ful</w>': 2, '   bo ok wor m</w>': 2, ' st ro ke</w>': 4, '   sa la m i</w>': 7, '   ma stu r ba ting</w>': 3, '   mor t</w>': 1, '   ma stu r ba tes</w>': 1, '   c li t or is</w>': 12, '   de sen si ti ze</w>': 1, ' na st y</w>': 1, '   fun ner</w>': 1, '   k ev </w>': 6, '   h ome made</w>': 3, '   sh ort stop</w>': 2, '   bl i</w>': 1, ' h in ded</w>': 1, '   re made</w>': 2, '   s w f</w>': 2, '   out going</w>': 2, ' car to on</w>': 1, '   mer ma id</w>': 4, '   ar sen al</w>': 4, '   de p lo y ment</w>': 2, '   s co p in</w>': 1, ' re ser ve</w>': 1, '   re ser ves</w>': 10, '   ar ou ses</w>': 2, ' st ri p</w>': 2, '   shi t brea k</w>': 1, ' le e</w>': 4, '   st ea m ing</w>': 4, ' 4 5 0</w>': 1, ' ex pre ss</w>': 2, ' un g gh h h h h</w>': 1, ' shi th ead</w>': 3, '   wh oo p</w>': 6, '   9 7 6 </w>': 2, ' p ly</w>': 1, ' l in</w>': 1, '   re make</w>': 2, '   hur ly</w>': 1, ' bur ly</w>': 1, ' w u r ly</w>': 1, ' hi de</w>': 1, '   m su </w>': 1, '   h or n do g</w>': 1, '   sti f fi es</w>': 1, '   s w ea t ers</w>': 6, '   stu ds</w>': 9, ' go o dy</w>': 2, '   un ta pped</w>': 1, '   rea tt ac h ed</w>': 1, '   ha a a ah</w>': 1, '   he i s man</w>': 1, '   ab ou </w>': 1, ' k are</w>': 1, '   l ou v re</w>': 1, '   ca da ver s</w>': 1, '   tr ans for m</w>': 5, ' du h</w>': 7, '   a a a a a</w>': 2, '   a h h h h h h h</w>': 1, '   o ver f l ow ed</w>': 1, '   ad m</w>': 11, '   mar ga u x</w>': 2, '   wa st es</w>': 2, '   r ou s se l</w>': 2, '   de men to</w>': 1, '   ser af ine</w>': 11, '   ro d in</w>': 2, '   fu c</w>': 1, '   hu i ti e me</w>': 1, '   vo t re</w>': 3, '   s an te</w>': 1, '   p le e z</w>': 1, '   loo gi es</w>': 1, '   p ere</w>': 1, '   l ac ha i se</w>': 1, '   ha ll u c in o gen i c</w>': 1, '   he ll ri de</w>': 1, '   m un ch ed</w>': 1, '   ter r ence</w>': 10, '   r ou se l</w>': 1, '   me du s a</w>': 1, '   tr an sc ri p ts</w>': 3, '   re bo o t</w>': 1, '   pa ss wor ds</w>': 10, '   sa lo ts</w>': 1, '   shi t fuck er</w>': 1, '   f lo c qu et</w>': 3, '   fe m me</w>': 2, '   ni ki ta</w>': 3, '   ba z in</w>': 1, '   r ac ine</w>': 1, '   g ran de</w>': 6, '   se ver in</w>': 6, '   en cu l</w>': 1, ' a mer i ca in</w>': 1, '   b ou l ard</w>': 1, ' co z</w>': 1, '   d ra in o</w>': 1, '   l y can th ro p y</w>': 2, '   de co de</w>': 7, '   ad en ine</w>': 1, ' me th y lo x i de</w>': 1, '   an ti gen s</w>': 2, '   in fe ctor</w>': 1, '   in fe c te e</w>': 1, ' che mi sts</w>': 1, ' si m on</w>': 1, '   cle an sed</w>': 2, '   su b cu l tur es</w>': 1, '   z b h</w>': 1, ' da y d rea m</w>': 1, '   e c to co s mi c</w>': 1, '   le ary</w>': 5, '   sy st e mi c</w>': 2, '   ph y si o</w>': 1, '   t ro pi c</w>': 4, '   tri p ta m ine</w>': 1, '   ph en e th y la m ine</w>': 1, '   mi da ir</w>': 1, '   go gh</w>': 9, '   h er ve</w>': 1, '   vi ll ac ha i se</w>': 1, '   s k in hea ds</w>': 1, '   le gu me</w>': 2, '   sor b on n e</w>': 1, ' hi pp i e</w>': 1, '   bi d et</w>': 1, '   m c</w>': 2, ' da ir</w>': 1, ' m o</w>': 1, '   mu ti la tions</w>': 1, '   ex cu se z</w>': 1, ' in ten se</w>': 2, '   h mp h</w>': 3, '   a llo</w>': 4, '   den f er t</w>': 2, '   co lo mb o</w>': 7, '   an a sta si a</w>': 16, '   son ya</w>': 1, '   any a</w>': 12, '   v la d</w>': 5, '   m ee too</w>': 2, '   sh ant</w>': 1, '   ra vi sh ing</w>': 2, '   d mi tr i</w>': 21, '   i ll ea ga l</w>': 1, '   pro vi d ded</w>': 1, '   po i sed</w>': 2, '   ta ti an a</w>': 11, '   le d g able</w>': 1, '   na ta sh a</w>': 5, '   f ea sta vi ch</w>': 1, '   na shi e</w>': 1, '   f oo shi e</w>': 1, '   d ow a ger</w>': 3, '   e mber</w>': 1, '   par se</w>': 1, ' se c ts</w>': 1, '   sa u ces</w>': 3, '   bar to k</w>': 2, '   o y</w>': 4, ' cu r ses</w>': 1, '   ro dent</w>': 4, '   t re pl ev </w>': 1, ' gre w</w>': 1, '   sha k er</w>': 4, '   w en ch ing</w>': 1, '   c ru i s in</w>': 3, '   ra f t ers</w>': 4, '   o y y y y</w>': 1, ' win gs</w>': 3, ' o de ss a</w>': 1, ' b lea k</w>': 1, ' bl ind</w>': 6, '   le an ed</w>': 4, '   re mb ran t</w>': 1, ' di sh on est</w>': 1, '   ro mon o v s</w>': 2, '   u l o</w>': 1, '   m ee ee e e</w>': 1, '   ru s se</w>': 2, '   b ru d der</w>': 1, '   al v y</w>': 52, '   ma ll et</w>': 3, '   al li son</w>': 27, '   por tch ni k</w>': 2, '   gr ou ch o</w>': 2, '   t s ch</w>': 11, '   mo le ster</w>': 1, '   con c lu si ve ly</w>': 1, '   ear pl u gs</w>': 1, '   gra mm ys</w>': 2, '   m un ch kin</w>': 4, '   re m</w>': 10, ' ca tch er</w>': 6, '   r ye</w>': 13, '   an dro g y n ous</w>': 1, ' al v y</w>': 1, '   bur ban k</w>': 5, '   de f ea ting</w>': 3, '   as ser ted</w>': 1, '   ri p en</w>': 1, ' wh a tta</w>': 3, '   re ha bi li ta te</w>': 1, '   comp le x i on</w>': 2, '   spi d ers</w>': 5, '   s qui sh</w>': 1, '   ge</w>': 1, '   bu ck le y</w>': 11, '   hu m din ger</w>': 1, ' o ci ty</w>': 1, '   ex t in gu i sh er</w>': 2, ' pi tch ed</w>': 3, '   g l</w>': 1, '   su ff o ca te</w>': 3, '   re a</w>': 2, ' ne at</w>': 3, '   chi pp e w a</w>': 4, '   ex pre ssi ons</w>': 3, ' ex i st en ti al</w>': 1, '   mo ti f s</w>': 2, ' con te mp or ary</w>': 1, ' co ll</w>': 1, ' r n m</w>': 1, '   un be ar ably</w>': 1, '   s ou th a mp ton</w>': 4, ' mo der n</w>': 1, ' in t ro du ction</w>': 1, '   en tom o lo g y</w>': 1, ' ra pi d ly</w>': 1, ' bu </w>': 3, ' o ve</w>': 3, '   po l y mor p hou s ly</w>': 1, ' a like</w>': 6, ' ru m my</w>': 1, '   pe ssi mi sti c</w>': 4, '   ha ll u c in o gen i c s</w>': 1, '   b al z ac </w>': 1, ' vi ta min s</w>': 1, '   le tch a</w>': 1, ' au di tion ing</w>': 1, '   gra m my</w>': 11, '   per spi red</w>': 1, '   n ar co le p sy</w>': 3, ' sho cked</w>': 2, '   po e te ss</w>': 1, '   mi sin ter pre ted</w>': 3, '   l our d es</w>': 2, '   ro ck well</w>': 7, '   v w</w>': 1, '   nu t c r ack er</w>': 2, '   sh e ll fi sh</w>': 1, '   b lo om in g da le</w>': 4, '   com par a ti ve ly</w>': 1, '   y or k er</w>': 5, '   o ver sle ep</w>': 1, '   do cu men t ary</w>': 13, '   ch ee ch</w>': 1, '   sha tt er ing</w>': 3, '   c ri ck e ts</w>': 2, '   hon king</w>': 1, ' re du ce</w>': 1, '   p sy cho an al y ti c</w>': 1, '   di ss ent</w>': 5, '   mer ged</w>': 1, '   d y sen ter y</w>': 2, ' ra g</w>': 2, '   le o po ld</w>': 3, '   lo e b</w>': 1, '   k af k a e s qu e</w>': 1, ' pl en did</w>': 2, '   al ta m oun t</w>': 1, '   ad ver ti ses</w>': 2, '   ro si c ru ci an</w>': 1, '   pl en did</w>': 1, '   m c lu ha n</w>': 5, ' t v </w>': 2, '   v a li di ty</w>': 1, '   p on ti fi ca te</w>': 1, ' thir t y i sh</w>': 1, '   be ck e t t</w>': 34, ' in du l g ent</w>': 3, '   sa t y ri c on</w>': 1, '   sa un as</w>': 1, '   j ac u z z is</w>': 1, '   sha w n</w>': 6, '   pe tr on i a</w>': 1, '   pl u ton i u m</w>': 9, '   le o t ard</w>': 1, '   ge ll er</w>': 10, '   ta ster</w>': 1, '   gu ac a mo le</w>': 2, '   mu tt er ing</w>': 1, '   tw o s</w>': 2, '   wh e e</w>': 2, '   l ac ey</w>': 3, '   ger m</w>': 6, '   su n st ro ke</w>': 3, '   dri b ble</w>': 3, '   por no gra ph ers</w>': 2, '   di sa gre es</w>': 1, ' se mi ti s m</w>': 3, '   for e s kin</w>': 1, '   be e k man</w>': 1, ' cu tt ed</w>': 1, '   wa g ner</w>': 10, '   sig ni fi can tly</w>': 3, '   par an</w>': 1, '   di d cho o</w>': 2, '   per se cu ting</w>': 2, ' di a be tes</w>': 1, '   s qu ir re ls</w>': 7, '   m ee tch a</w>': 1, '   me tch a</w>': 1, '   an ge li c a</w>': 1, '   te ssi e</w>': 1, '   ther ri ans</w>': 1, '   an ou k</w>': 1, '   g all</w>': 4, ' i sh</w>': 5, '   cla ir</w>': 14, '   chi l d bi r th</w>': 7, '   st ru ts</w>': 2, '   f re ts</w>': 1, '   so b bed</w>': 1, '   sho sta k o vi ch</w>': 7, '   s k ye</w>': 17, '   b re e zy</w>': 1, '   ju d ge men tal</w>': 1, '   ser en e</w>': 3, '   x an ex </w>': 1, '   e ye la sh es</w>': 4, '   dan der</w>': 2, '   so p h</w>': 3, '   sh run k</w>': 4, '   j on ah</w>': 26, '   w ra i th</w>': 2, '   i c u</w>': 1, '   ther ri an</w>': 3, '   mi sp lan ted</w>': 1, '   che y en e</w>': 1, '   g in a</w>': 18, '   lan d s cap er</w>': 1, '   v ho l</w>': 1, '   rea l tor</w>': 5, '   che y en n e</w>': 6, '   ca l der</w>': 1, '   a ll ow an ces</w>': 1, '   sha ll ow est</w>': 1, '   gen na</w>': 3, '   coun se lling</w>': 3, '   e c st ac y</w>': 3, ' ma e</w>': 1, '   p an es</w>': 16, '   fi l mi c</w>': 2, '   un in vi te</w>': 1, ' sc ri pt</w>': 2, '   bo ok er</w>': 10, '   au to bi o gra p hi es</w>': 1, '   bi o gra p hi es</w>': 1, ' fi ction</w>': 5, '   st y r on</w>': 1, '   fa ther ho od</w>': 1, '   de p ra v ation</w>': 1, '   sta pe l ton</w>': 1, '   dar i us</w>': 2, '   k hon j i</w>': 1, '   d p</w>': 2, '   s no ok u ms</w>': 2, '   me da lli ons</w>': 1, '   lo lly</w>': 2, '   a bu si ve</w>': 3, '   in ce ss an tly</w>': 4, '   e s ca la ting</w>': 1, '   de cor a tor</w>': 7, '   p ch</w>': 1, '   ye t ve sh en k o</w>': 1, '   y ar</w>': 2, '   gi ttle</w>': 1, '   se ll ers</w>': 9, '   z h dan o v </w>': 4, '   pro k o fi eve</w>': 1, '   my as k o v s k y</w>': 1, '   an d re</w>': 3, '   st al in i st</w>': 1, '   im per i a li s m</w>': 1, '   char ad es</w>': 2, '   na sh</w>': 12, '   ga li an o</w>': 2, ' l ou sy</w>': 1, '   co de pen dent</w>': 2, '   lo ve si ck</w>': 1, '   bar ked</w>': 2, '   a m bi v al ent</w>': 2, '   no ve li st</w>': 9, '   por tra y al</w>': 2, '   n ar ci ssi sti c</w>': 3, '   par en ting</w>': 4, '   la ma se</w>': 1, '   so ci a li z ed</w>': 2, '   wa ter co l ors</w>': 1, '   in t ro spe c tive</w>': 1, '   ev i e</w>': 1, '   brea k time</w>': 1, '   a z te c a</w>': 2, '   oo p</w>': 1, ' in di vi du al</w>': 2, ' han de d ly</w>': 4, '   b ack talk</w>': 2, ' d ow n si z ed</w>': 1, '   re pe ti tions</w>': 2, '   p in c ers</w>': 1, '   man t is</w>': 2, '   v is</w>': 2, ' v is</w>': 1, ' few</w>': 2, '   b al a</w>': 9, '   for mi c a</w>': 6, ' f loo d</w>': 1, ' co lon y</w>': 1, ' nee ded</w>': 1, '   sc ar f</w>': 8, ' tr ouble</w>': 2, ' 9 8 5 </w>': 1, ' in se c to pi a</w>': 4, '   pr in ce ss es</w>': 3, '   h ll ll ll l p</w>': 1, ' happ i ly</w>': 1, ' con si der ing</w>': 1, '   in se c to pi a</w>': 1, ' i di o t</w>': 2, ' s lu mm ing</w>': 1, '   sen su al</w>': 6, ' ha te</w>': 10, '   mon ar chi cal</w>': 1, '   hi er ar ch y</w>': 2, '   under l ying</w>': 1, ' wor k er</w>': 4, '   we ir d ly</w>': 1, '   di v in e ly</w>': 1, '   v an qui sh ing</w>': 1, '   s la u gh ter ing</w>': 3, '   ne st ful</w>': 1, '   h mm n n</w>': 1, '   pe b ble</w>': 3, '   pro s the ti c</w>': 2, '   an ten n as</w>': 1, '   ro y al s</w>': 1, '   ca pi tu la ting</w>': 1, '   o pp re ssi ve</w>': 1, ' stu ff y</w>': 1, '   an th o od</w>': 1, '   bar ba tu s</w>': 7, '   gon er</w>': 1, '   pla to on</w>': 11, '   be ver a g es</w>': 1, '   un att a in able</w>': 1, '   bri s k</w>': 1, '   for a ging</w>': 3, ' be lli es</w>': 1, ' sin g le</w>': 3, '   ru se</w>': 3, '   pl un ging</w>': 5, ' li ving</w>': 5, '   ne ga t ory</w>': 2, '   beli tt ling</w>': 1, '   s ni ck er</w>': 1, ' mon ger</w>': 2, ' la u gh s</w>': 1, '   mar ve ll ou s ly</w>': 1, '   ad vi se ment</w>': 1, '   s n ac king</w>': 1, '   pu ss y f oo t</w>': 3, '   s wi f tly</w>': 3, '   de ci si ve ly</w>': 2, ' pa ins</w>': 1, '   wi d dle</w>': 2, ' i so la ted</w>': 1, '   ab an d on ment</w>': 2, '   l ar v a</w>': 1, '   si bl in gs</w>': 1, ' g ru b</w>': 1, ' in sig ni fi c ant</w>': 1, '   w ea ver</w>': 5, '   st ro lling</w>': 5, ' b in go</w>': 1, '   re k in d l ed</w>': 1, '   c ru m pe ts</w>': 1, '   g ru mp ing</w>': 1, '   g ro an ing</w>': 1, '   a p hi d</w>': 1, '   ou sp en s k a ya</w>': 2, '   do bi s ch</w>': 10, '   dre y fu ss</w>': 12, '   f ar ou k</w>': 1, '   na sh ing</w>': 1, ' whi pp ing</w>': 2, '   k u beli k</w>': 35, '   8 6 1 </w>': 1, '   p an el ed</w>': 1, '   din ah</w>': 1, '   c ru so e</w>': 1, '   shi p w re cked</w>': 2, '   b ack h and</w>': 1, '   pa w n sho p</w>': 1, '   ta k er</w>': 4, '   ha ir p ins</w>': 3, '   st or k</w>': 3, '   di sor g ani z ed</w>': 2, '   c ru mb les</w>': 2, '   se cre t ar i al</w>': 4, '   con so li da ted</w>': 2, '   ma j or e tt e</w>': 3, '   en ter ta in er</w>': 4, '   un ve i ling</w>': 2, '   un for gi v able</w>': 4, '   chi c o</w>': 2, '   ma je sti c</w>': 4, '   k a pu t t</w>': 1, '   ki r ke by</w>': 8, '   pa ss k ey</w>': 6, ' ton k y</w>': 1, '   li e ber man</w>': 4, '   k r in g le</w>': 1, '   me shu ga ss</w>': 1, '   can a ver al</w>': 2, '   pa y o l a</w>': 1, '   com mu ter</w>': 1, '   o ver do se</w>': 6, '   bu sy body</w>': 1, '   a sp ir ins</w>': 5, ' 2 5 9 </w>': 1, '   con du c ts</w>': 3, '   bi l t more</w>': 2, '   ban que ts</w>': 1, '   v an der h of</w>': 5, '   ei ch el ber ger</w>': 3, '   ac tu ar i al</w>': 4, '   ma tu sch k a</w>': 4, '   clo b b er</w>': 2, '   fa un t l er o y</w>': 3, '   tu mb le</w>': 6, '   ver m ou th</w>': 2, '   mu ra l</w>': 1, '   s k a ter</w>': 4, '   sch me ar</w>': 1, '   pen ci l ed</w>': 2, '   m ac in to sh</w>': 1, '   pro c to lo gi st</w>': 1, '   man s ch</w>': 1, '   poli ten ess</w>': 1, '   tw i</w>': 1, '   ne b bi sh</w>': 1, '   be l ting</w>': 1, '   sch ra f ft</w>': 1, '   m ac d ou g all</w>': 1, '   se x po t</w>': 1, '   s nu g s vi ll e</w>': 1, '   du ll s vi ll e</w>': 1, '   be l ted</w>': 2, '   gu z z ling</w>': 2, '   gu g gen he i m</w>': 1, '   to o t</w>': 3, '   di v or c ing</w>': 2, '   y ong</w>': 1, '   ro ss i</w>': 3, '   k o ch</w>': 2, '   di sa bi li ty</w>': 6, ' ding</w>': 5, '   so l ves</w>': 6, ' f our te en</w>': 5, '   man i cu ri st</w>': 3, '   da i qu ir i</w>': 1, '   ni e ces</w>': 1, ' hu r</w>': 1, '   mi x up</w>': 2, ' n in th</w>': 3, '   ber n he i m</w>': 1, '   sch no ok</w>': 1, '   pro pa ga te</w>': 3, '   na tion a li ti es</w>': 1, '   e sp r it</w>': 1, '   dis gu sts</w>': 2, '   me d ev ac </w>': 1, '   wi ll ard</w>': 16, '   ha u</w>': 4, '   th ta</w>': 2, '   ter r i</w>': 11, '   mo on by</w>': 4, ' cor por al</w>': 1, '   win st ons</w>': 2, '   min ed</w>': 1, '   4 0 5 </w>': 5, ' se c</w>': 4, '   n h a</w>': 3, '   tr an g</w>': 3, '   pa d di es</w>': 2, '   l r r p</w>': 1, '   mu ti la ting</w>': 1, '   cen ter f old</w>': 3, '   go ok</w>': 5, '   f li cked</w>': 2, '   ma g</w>': 7, '   prob ed</w>': 1, '   n v a</w>': 5, '   what do ya</w>': 1, '   mar bl ed</w>': 1, '   ca u l dr ons</w>': 2, '   sa u ci ere</w>': 4, '   e s co f fi er</w>': 1, '   man go es</w>': 4, '   ju d t</w>': 1, '   man go s</w>': 1, '   d ow n ri ver</w>': 3, '   tri bu t ar i es</w>': 1, '   hu e ys</w>': 3, '   m un g</w>': 6, '   n un g</w>': 4, '   co p ter</w>': 1, '   n u</w>': 6, '   ac qu in t an ce</w>': 1, '   ru m ou rs</w>': 8, '   t et</w>': 1, '   ber e ts</w>': 4, '   k on tu m</w>': 1, ' ev ac u ate</w>': 1, '   in do ch in a</w>': 3, '   a mer i ca in</w>': 1, '   ri d able</w>': 2, '   cu t back</w>': 3, '   sur f ed</w>': 1, '   ki l g ore</w>': 1, '   m c d on ne l</w>': 1, '   sp ar a ying</w>': 1, '   f ac </w>': 1, '   a k</w>': 4, ' 4 7 </w>': 8, '   s lo p es</w>': 2, '   or d n an ce</w>': 1, '   ber et</w>': 4, '   c ac h es</w>': 2, '   ha i ph ong</w>': 2, '   p ho sp h or us</w>': 1, '   a ir plan es</w>': 6, '   ans</w>': 3, '   ki l er</w>': 1, ' pri cks</w>': 1, '   b lo t to</w>': 1, '   li gh t show</w>': 1, '   h y po der mi c</w>': 3, '   par at ro op er</w>': 3, '   re con do</w>': 1, '   y a ter</w>': 3, '   st r in ger</w>': 2, '   tu bed</w>': 1, '   th ran g</w>': 1, '   ha shi sh</w>': 4, '   tri an g le</w>': 9, '   ow sle y</w>': 1, ' au stra li an</w>': 1, '   p hi li pp e</w>': 1, '   wh e</w>': 2, '   tr ans f er ed</w>': 1, '   a su b je ct</w>': 1, '   o c cu red</w>': 3, '   com se c</w>': 1, '   me y er ling</w>': 6, '   1 7 5 </w>': 2, '   1 3 9 </w>': 2, '   1 1 7 </w>': 3, '   1 0 2 </w>': 1, ' ri pp ing</w>': 1, '   1 6 8 </w>': 1, '   1 6 5 </w>': 2, '   cor be t t</w>': 16, '   1 5 7 </w>': 1, '   a v al an c he</w>': 6, '   1 1 3 </w>': 2, '   9 4 </w>': 2, '   ch ee cha k o</w>': 1, '   tra pl in es</w>': 1, '   co l d f oo t</w>': 1, '   ho th ea ded</w>': 1, '   re f le c ts</w>': 4, '   ca u l dr on</w>': 8, '   fa ir ban ks</w>': 8, '   din n er time</w>': 2, '   po ac h ers</w>': 1, '   ca m cor der</w>': 1, '   t und r a</w>': 2, '   bi g ne ss</w>': 1, '   bu y off</w>': 1, '   g un men</w>': 4, '   y u k on</w>': 2, '   el t</w>': 2, '   du f fe l</w>': 2, ' t one</w>': 1, '   g li mp sing</w>': 1, '   stu mb ling</w>': 3, '   out s mar t</w>': 2, '   thr ow back</w>': 1, '   mu s k r at</w>': 2, '   wh al er</w>': 1, '   h y po ther mi c</w>': 1, '   fr on ti ers</w>': 2, '   b al d hea ded</w>': 1, '   gra y ling</w>': 1, ' a lls</w>': 1, '   le ma ll e</w>': 3, '   e co lo gi cal</w>': 3, '   k en a i</w>': 3, '   po ac h ing</w>': 3, '   st ran ding</w>': 1, '   sp ort s men</w>': 1, '   ne ff</w>': 3, '   la d y friend</w>': 1, '   j i min y</w>': 2, '   s n ow pl ow</w>': 1, '   ce ss na</w>': 3, '   han gar ed</w>': 1, '   d we lling</w>': 5, '   sp ac e shi ps</w>': 1, '   wan g le</w>': 2, '   sp l int</w>': 2, '   re min g ton</w>': 9, ' lo ver s</w>': 2, ' s wi tch</w>': 2, ' b en</w>': 4, '   c ro ss wal k</w>': 4, '   ba t shit</w>': 3, '   sor es</w>': 1, '   f ru st r</w>': 1, '   be tt es</w>': 4, '   u d all</w>': 11, '   bi lled</w>': 4, '   car pe ting</w>': 2, '   o ver shi r t</w>': 1, '   under one</w>': 1, '   con ne lly</w>': 1, '   gr ou ch y</w>': 6, '   under be lly</w>': 3, '   ho pa ho pa ho p a</w>': 1, '   mon om in u te</w>': 1, '   la kes</w>': 3, '   y ar d sti ck</w>': 2, '   dre ss y</w>': 2, '   ob li ga tes</w>': 1, '   fi ll er</w>': 1, ' k no cking</w>': 1, '   br y an</w>': 2, '   ba i ls</w>': 1, '   w are</w>': 2, '   c li pp i ty</w>': 2, '   clo p</w>': 2, '   pu t z e tt e</w>': 2, '   por ce la in</w>': 2, '   ca ve men</w>': 1, '   g l ows</w>': 3, '   s ke tch ing</w>': 1, '   sh y er</w>': 1, '   mon u men tal</w>': 4, '   sh h h h h h h</w>': 4, '   re gre w</w>': 1, ' ob se ssi ve</w>': 1, '   ver d ell</w>': 7, '   un re co g ni z able</w>': 3, ' m un ch ing</w>': 1, '   z i pped</w>': 2, '   6 1 </w>': 1, ' hu mp</w>': 2, '   g ee z er</w>': 4, '   ja m mi es</w>': 1, ' ex tra or din ary</w>': 1, '   su b let</w>': 2, '   g ru d g es</w>': 6, '   ti re d ne ss</w>': 1, '   ne lli e</w>': 14, '   th u d</w>': 2, '   de ca ying</w>': 1, ' p ack er</w>': 1, '   dan ging</w>': 1, '   y ee ee ss</w>': 1, '   s ac h s</w>': 1, '   ac coun ta bi li ty</w>': 2, '   ba in</w>': 13, '   mi c ro c ell</w>': 1, '   a ir ba g</w>': 1, '   t ea m ster</w>': 2, '   sha ded</w>': 2, '   t ac h lin k o v </w>': 2, '   f an boy</w>': 1, '   ni cho la i</w>': 5, '   tal in k o v </w>': 1, '   ra th</w>': 4, '   ex pre ss way</w>': 3, '   v a ll i</w>': 3, '   le ev i o</w>': 1, '   b y stan d ers</w>': 4, '   tra tt ori a</w>': 1, '   f on te ll a</w>': 1, '   par a i so</w>': 6, '   so l</w>': 9, '   bl an c a</w>': 4, '   f re el an c er</w>': 1, ' pu ss y</w>': 1, '   e le c tr a</w>': 7, ' pre ser v ation</w>': 2, '   comp li ca te</w>': 6, '   ke y ho le</w>': 5, '   un na tu ra lly</w>': 2, '   d or m</w>': 7, '   fun c tion ed</w>': 1, '   bi sh op s</w>': 1, '   ro la i ds</w>': 1, '   mi kes</w>': 2, '   de te c t ors</w>': 6, '   a i i e e</w>': 1, '   tr ans mi ts</w>': 2, '   b loo ded</w>': 3, ' y or k</w>': 2, '   o bi tu ar i es</w>': 3, '   aga me m n on</w>': 1, '   e mp t ying</w>': 4, '   re cor din gs</w>': 3, ' he m</w>': 3, '   b ran ds</w>': 1, '   wa ds</w>': 1, '   fin l sh</w>': 1, '   bu si est</w>': 2, ' h en</w>': 5, '   mo on pi es</w>': 1, '   mo on pi e</w>': 1, ' a k en</w>': 1, ' ay</w>': 13, '   ve</w>': 16, '   en ro l</w>': 1, '   po st gra d</w>': 1, '   je we ll er y</w>': 2, '   na z</w>': 1, '   d ra v i</w>': 1, '   k at k a</w>': 10, '   che qu ers</w>': 1, '   re u t ers</w>': 1, '   ja h n</w>': 11, '   hon z a</w>': 4, '   ha ve l</w>': 3, '   bu g l er</w>': 6, '   ho o li g an</w>': 1, '   pu b s</w>': 3, '   min d set</w>': 2, '   ru b ble</w>': 5, '   di sp o sa ble</w>': 5, '   ru st le</w>': 2, ' bo ttle</w>': 2, '   p y sc he</w>': 1, '   re min i s ce</w>': 1, '   ex p at</w>': 1, '   s ca ff old</w>': 2, '   a be tting</w>': 6, '   k a v lo v a</w>': 1, '   no things</w>': 2, '   lo ck s mi th</w>': 4, '   sp ar ki e</w>': 1, '   n ea ten</w>': 1, '   z ep</w>': 1, '   i c ing</w>': 2, '   c z e ch s</w>': 2, '   ev i ction</w>': 1, '   ba i li ff s</w>': 1, '   lu bo sh</w>': 3, '   cu ster</w>': 2, '   ex pre ssi on i s m</w>': 1, '   s qu att ed</w>': 1, '   ven u e</w>': 3, '   in spi ra tions</w>': 1, '   ca b by</w>': 2, '   st er i le</w>': 2, '   wan k er</w>': 2, '   bu d jo vi ce</w>': 1, '   cl ar i fi ed</w>': 1, '   th or n ton</w>': 1, '   co ven try</w>': 1, ' si x ty</w>': 7, '   gre en well</w>': 4, '   ho o li g ans</w>': 2, '   w rea k</w>': 1, '   w and</w>': 10, '   ce li ba te</w>': 3, '   v a y v u do o</w>': 1, '   der o ga t ory</w>': 3, '   sp o t li gh ts</w>': 1, ' lin gu al</w>': 1, '   o beli s k</w>': 1, '   f lo or ing</w>': 1, '   l ab our er</w>': 1, ' do b cha y</w>': 2, '   do b ree</w>': 1, '   j ir i</w>': 1, ' ba gs</w>': 1, '   cra pp ing</w>': 1, '   j an o v s k a</w>': 1, '   vi to ve tch</w>': 1, ' ei ther</w>': 5, '   k ra k ow</w>': 1, ' p sy cho s is</w>': 1, '   f lo ps</w>': 1, '   be t le m s k a</w>': 1, '   ne t wor ked</w>': 1, ' go o d ers</w>': 2, ' pri ck</w>': 3, ' t in ted</w>': 1, '   af ter g l ow</w>': 1, '   di mm ing</w>': 1, '   k ar el</w>': 1, '   la st ly</w>': 2, '   su b ti t les</w>': 3, '   ac comp li ces</w>': 2, '   en qui ri es</w>': 6, '   lan don</w>': 20, '   be lin da</w>': 4, '   s ca mm ing</w>': 2, '   pu ri t an</w>': 6, '   a ve</w>': 2, '   h y po the ti ca ls</w>': 1, '   bo a st ful</w>': 2, '   part on</w>': 1, '   x x i</w>': 1, ' 5 6 3 9 </w>': 1, '   se du ci ble</w>': 1, '   ther mo s</w>': 2, ' a z i mu th</w>': 1, '   cha ll en ger</w>': 7, '   w r</w>': 5, '   stra d d ling</w>': 1, '   ac hi ev ing</w>': 2, '   nu cle us</w>': 2, '   ha le y</w>': 1, '   vo y a ger</w>': 1, '   so f ty</w>': 1, '   car t wh ee l</w>': 1, '   li z z i e</w>': 2, '   be friend</w>': 1, '   di sa gre ea ble</w>': 3, '   fo cu s er</w>': 1, '   ph on o gra p hi c</w>': 1, '   tur n ta ble</w>': 1, '   ma ter i ally</w>': 2, '   re he ar sa ls</w>': 2, '   di sa d v an ta ged</w>': 1, '   j ani t ori al</w>': 1, '   i so sc el es</w>': 1, '   s ca l en e</w>': 1, '   t an n en</w>': 2, '   s mi r ked</w>': 1, '   m c f ly</w>': 3, '   1 9 5 2 </w>': 5, '   ei le en</w>': 5, ' j ee z</w>': 4, '   po l ye ster</w>': 4, '   pa used</w>': 1, '   bi c</w>': 2, ' bi ke</w>': 3, ' pr in g time</w>': 1, '   su zy</w>': 5, '   ar k y</w>': 1, '   an n hi li ation</w>': 1, '   me ga ton</w>': 2, '   ge o ther ma l</w>': 2, '   3 4 8 9 </w>': 1, ' ra y ed</w>': 2, '   re e se</w>': 9, '   com mi tion</w>': 1, '   or p he u m</w>': 3, '   8 7 </w>': 12, '   b ans</w>': 2, '   fe l sti en</w>': 1, '   1 9 8 2 </w>': 4, '   so f tly</w>': 5, '   gr in ned</w>': 1, '   re fr i</w>': 1, '   con ver ter</w>': 5, '   no ting</w>': 3, '   ho ts</w>': 3, '   4 2 0 0</w>': 2, '   ra ds</w>': 2, '   in stan t an e ous</w>': 3, '   mo o t</w>': 2, '   gen er a ted</w>': 7, '   d ev a sta ting</w>': 4, '   in e bri ation</w>': 2, '   ss sh h h h h</w>': 2, '   sh e mp</w>': 10, ' 0 2 </w>': 1, '   h y dro g en</w>': 8, '   1 9 4 9 </w>': 3, '   e f fi ci en tly</w>': 2, '   bo o t le g</w>': 3, '   sh h h h h h h h</w>': 1, '   an t l ers</w>': 1, '   au ght</w>': 3, '   p all</w>': 2, '   loo ki e</w>': 2, '   k on</w>': 5, ' ti k i</w>': 3, '   su ction</w>': 5, ' u z</w>': 2, '   fu d ge si c le</w>': 2, '   gr ow ling</w>': 2, ' red</w>': 11, '   tr ab a j o</w>': 2, ' d ru th ers</w>': 1, '   sc ar bor ough</w>': 1, '   1 9 3 8 </w>': 5, '   g ro ver</w>': 4, '   y o y o d y n e</w>': 4, '   ra whi de</w>': 2, '   hi ki ta</w>': 3, '   v a p ori ze</w>': 4, '   ca v a li ers</w>': 1, '   bu tt er fin g ers</w>': 1, '   lin d le y</w>': 1, '   ar ac h to i ds</w>': 2, '   sc ri b ble</w>': 4, ' con du ctor</w>': 2, '   f la p j ac ks</w>': 1, '   gra di ent</w>': 1, '   sy n ch r on i z er</w>': 1, '   so f ten s</w>': 2, '   att en u a ting</w>': 1, '   e le c tr ow ea k</w>': 1, '   li z ar do</w>': 8, '   spe c t ro gra p h</w>': 1, '   ar ac h to id</w>': 2, ' gra z ed</w>': 1, '   gra z ed</w>': 2, '   o ver th ru ster</w>': 3, '   sh an g ha i ed</w>': 1, '   r er ou te</w>': 1, '   wh or f in</w>': 5, '   mi lls</w>': 10, '   po stu la ted</w>': 1, '   ar ac h to i da l</w>': 1, '   de du c ti ble</w>': 6, '   ri d ding</w>': 2, '   pri ddy</w>': 2, '   re pre ss</w>': 2, '   pi vo t</w>': 2, '   pri d di es</w>': 1, '   ne ther lan ds</w>': 2, '   sp on ged</w>': 1, '   6 9 </w>': 7, '   s mo l en s k</w>': 1, '   pre ci pi ta te</w>': 1, ' con n or</w>': 7, '   ther mo p od</w>': 2, '   go me z</w>': 6, ' so l ar</w>': 1, '   s ac </w>': 7, '   ni gh</w>': 3, '   mo se y</w>': 3, '   spi tt in</w>': 1, '   hu g ger</w>': 2, '   d ow n ran ge</w>': 1, '   un s che du l ed</w>': 1, '   wal t z ing</w>': 6, '   b ack w oo ds</w>': 1, '   wa if</w>': 1, '   man chi ld</w>': 1, '   spi ri ted</w>': 3, '   e st e ll e</w>': 2, '   fa ye tt e s vi ll e</w>': 1, '   ab ru pt</w>': 3, '   w r es</w>': 1, '   e mb ar as sing</w>': 6, '   he sh</w>': 1, '   qui e ts</w>': 1, '   ga d damn</w>': 1, '   ba se st</w>': 1, ' o l f ac t ory</w>': 1, '   wom ani sh</w>': 2, '   m und t</w>': 9, '   l ev e e</w>': 3, '   ra g in</w>': 1, '   la pp in</w>': 1, ' o or</w>': 2, '   th i sa here</w>': 1, '   shu tt in</w>': 1, '   ma i k n</w>': 1, ' ba lls</w>': 2, '   f ru i t pi ck ers</w>': 1, '   rea li z ation</w>': 4, '   lu bri c ant</w>': 1, '   und ome sti ca ted</w>': 1, '   a ll us</w>': 5, '   li p ni k</w>': 13, '   s ou se</w>': 4, '   gen re</w>': 9, '   ru g g les</w>': 1, '   ti gh ts</w>': 3, '   be er y</w>': 11, '   mo s qui to s</w>': 3, '   mo s qui to</w>': 5, '   ok u m</w>': 4, '   ge i s l er</w>': 2, ' bar ton</w>': 1, '   ca ven</w>': 2, '   per cen ter</w>': 1, '   rea li s m</w>': 6, '   sp r ou ting</w>': 1, '   cho ir s</w>': 1, '   st om p in</w>': 2, '   p ly</w>': 2, ' ac h in</w>': 1, '   a m oun ted</w>': 1, '   gr un ting</w>': 2, '   s qu ir m ing</w>': 1, '   ho l d ers</w>': 8, '   b art</w>': 11, '   la s so ed</w>': 1, '   re v el ry</w>': 1, '   re mar ked</w>': 2, '   in su late</w>': 1, '   re gre ss es</w>': 1, '   for ma li s m</w>': 1, '   ba st ro p</w>': 1, '   hi g g in bo tt om</w>': 1, '   gr in ch</w>': 1, ' gi b b ons</w>': 1, '   sh op wor n</w>': 1, '   ab st r ac tions</w>': 2, '   the ma ti ca lly</w>': 1, '   sc ri b bl er</w>': 1, ' do or</w>': 2, '   car r y in gs</w>': 1, '   mor gen th a u</w>': 1, '   ten e ment</w>': 1, '   sh ow man ship</w>': 1, '   wi l shi re</w>': 8, '   re z</w>': 3, '   pa y able</w>': 4, '   che ck out</w>': 1, '   tr an si ent</w>': 1, '   tr an z</w>': 1, '   6 0 5 </w>': 1, ' f red</w>': 1, '   fe li z</w>': 3, '   wa gs</w>': 1, '   cu r r an</w>': 1, ' mi l</w>': 1, '   z il ch</w>': 4, '   i ce pi ck</w>': 5, '   no ah</w>': 6, '   go l d st e in</w>': 4, '   st y l ed</w>': 2, '   se du ces</w>': 3, '   tra m ell</w>': 14, '   ni l s en</w>': 8, '   e le c t ro lu x</w>': 1, '   re se ar ch ing</w>': 4, '   a do le sc ent</w>': 4, '   o ber man</w>': 1, '   bo z</w>': 8, '   af fe c tion a tely</w>': 1, '   ha z el</w>': 6, '   do b k ins</w>': 3, '   an n oun c ing</w>': 11, '   cor ri g an</w>': 2, '   sa do</w>': 1, ' su sp en sion</w>': 1, '   some h t in</w>': 1, ' ma kin</w>': 1, '   p an ned</w>': 3, '   tw ee ty</w>': 1, '   f lu tt er in</w>': 1, '   nu t ti er</w>': 1, '   la w dy</w>': 1, '   f ar m girl</w>': 1, ' pla te</w>': 1, '   win ks</w>': 2, '   ro ck ers</w>': 2, '   be l ab or</w>': 1, '   de mon stra ting</w>': 2, '   1 9 5 6 </w>': 4, ' as so s</w>': 1, '   tal co t t</w>': 1, ' si x ti es</w>': 2, '   har ri g an</w>': 2, '   pri ors</w>': 5, '   con vi c tions</w>': 6, ' 1 1 0</w>': 1, '   to a sted</w>': 1, ' la p do g</w>': 1, '   k a bu k i</w>': 1, '   ca lli gra ph ers</w>': 1, '   na a</w>': 3, '   mi che l</w>': 5, '   ba s qui at</w>': 7, ' war ho l</w>': 1, '   re p li ca s</w>': 1, '   pi tt s bur g</w>': 1, '   gar con s</w>': 1, '   sa m o</w>': 2, '   tur n out</w>': 3, '   an n in a</w>': 1, '   no se i</w>': 1, '   nor a</w>': 4, '   b ow er y</w>': 6, '   mo th a fuck ah</w>': 1, '   thin gi es</w>': 1, '   na a a a</w>': 1, '   pi ck an in ny</w>': 1, '   pri mi ti ves</w>': 1, '   ha i t i</w>': 2, '   ha i ti an</w>': 4, ' pu er to</w>': 1, ' le on ar do</w>': 1, '   v in c i</w>': 25, ' par a si tes</w>': 1, '   ex pre ssi on i st</w>': 1, '   sh en ge</w>': 3, '   por to s</w>': 1, '   cer u l ean</w>': 1, ' u pp i ty</w>': 2, '   ma ys</w>': 5, ' oo o ves</w>': 1, '   loo oo oo o ves</w>': 1, '   ri c ard</w>': 2, '   ar t for u m</w>': 1, '   stra w s</w>': 3, ' med</w>': 3, '   sta ten</w>': 2, '   lan a i</w>': 2, '   ni i ha u</w>': 1, '   k a ho o la we e</w>': 1, '   ma u i</w>': 4, '   k au i</w>': 1, '   mo lo k a i</w>': 2, '   man z ani ta</w>': 1, '   in sta lling</w>': 2, '   4 7 7 </w>': 1, '   0 4 9 6 </w>': 1, '   mu d d</w>': 1, '   mo ver</w>': 2, ' sa d ne ss</w>': 1, '   ri ch ne ss</w>': 1, '   un just</w>': 3, '   run gs</w>': 1, '   hu h h</w>': 2, ' a m o</w>': 1, '   ni g ga h</w>': 1, ' p hi lli ps</w>': 1, '   to o l bo x</w>': 1, '   le ch</w>': 3, '   ir t</w>': 1, '   gra f fi t i</w>': 6, '   ra m me ll z e e</w>': 1, '   ra me ll z e e</w>': 1, '   se lin a</w>': 32, '   ho sted</w>': 1, '   o di ous</w>': 1, '   sh re ck</w>': 6, '   re s oun din g ly</w>': 1, ' af fa ir</w>': 1, '   f ac e ts</w>': 2, '   al f red</w>': 38, '   vi ch y s so i se</w>': 2, '   re sen t ful</w>': 2, '   dea d li er</w>': 4, '   mi st el to e</w>': 1, '   mi st le to e</w>': 4, ' bu n s</w>': 1, ' sh re ck</w>': 1, '   go th am</w>': 24, '   as se ss ing</w>': 2, '   d ev a sta tion</w>': 1, ' ma x</w>': 3, ' so l ve</w>': 2, ' ean</w>': 4, '   be d ding</w>': 2, '   si ck o s</w>': 1, '   com mi ted</w>': 3, '   ba tes</w>': 20, ' vi ck i</w>': 2, '   du a li ty</w>': 5, '   p ho to j our na li st</w>': 1, ' s k a ter</w>': 1, '   re con ci ling</w>': 2, '   pla ted</w>': 2, '   bra ver</w>': 3, '   com pl ac en cy</w>': 3, '   co b b le po t</w>': 6, '   re li gh ting</w>': 1, ' ca tw om an</w>': 2, ' sle ep</w>': 3, ' ba t man</w>': 3, ' vi ll a ins</w>': 2, '   e b en ee z er</w>': 1, ' k y le</w>': 2, '   y a w n</w>': 2, ' may or</w>': 1, '   si x ed</w>': 2, '   ro om i es</w>': 1, ' co b b le po t</w>': 1, '   may ors</w>': 2, '   he ir s</w>': 3, '   mu ha mm ed</w>': 1, '   ban di ts</w>': 4, '   car ny</w>': 2, ' p int</w>': 1, ' pen gu in</w>': 1, '   sp a y ed</w>': 2, '   na p al med</w>': 2, '   o z z i e</w>': 1, ' m ou se</w>': 1, '   di sa s se mb le</w>': 1, '   spi ff y</w>': 1, '   ba t mo bi le</w>': 2, '   ca pi c he</w>': 1, '   may or al</w>': 1, '   c z ars</w>': 1, '   ex ce p ting</w>': 1, '   ba d der</w>': 1, ' b lu r ry</w>': 1, '   im mer sed</w>': 1, '   i o ta</w>': 2, '   win n ers</w>': 10, '   t an g</w>': 1, '   re cla i m</w>': 5, '   car e le ss ly</w>': 1, ' que st</w>': 1, ' s oun ds</w>': 4, '   par a so l</w>': 1, '   re called</w>': 3, '   re i ch sta g</w>': 1, '   ton kin</w>': 1, ' fri gi d</w>': 1, ' fr en ch</w>': 2, '   o ver turn</w>': 1, '   in ha l ed</w>': 1, '   supp l ant</w>': 1, ' pro gre ss</w>': 2, '   lu c re</w>': 1, '   or che stra te</w>': 1, '   ad k ins</w>': 1, '   sh re d ded</w>': 4, '   fi re tra ps</w>': 1, ' h y p no ti ze</w>': 1, '   spe w ed</w>': 2, '   pro fi te er ing</w>': 2, '   gon i ff s</w>': 1, '   ma x i mi lli ons</w>': 1, ' se cre t ary</w>': 2, ' cap ac i tor</w>': 1, ' fin ster</w>': 1, '   p om er ani an</w>': 4, '   in du st ri ous</w>': 1, '   wi l f red</w>': 12, '   ye ar n s</w>': 1, '   mi ra j an p ore</w>': 2, '   dis may</w>': 1, '   jo an na</w>': 44, '   ba t sig na l</w>': 1, '   g lan du l ar</w>': 1, '   se cre tions</w>': 2, '   ph er om on es</w>': 1, '   sp li c ing</w>': 3, '   ph er om one</w>': 2, '   ex tr ac tions</w>': 1, '   i sle y</w>': 6, '   en ter pri ses</w>': 9, '   s wee ps</w>': 4, '   pi gh ea ded</w>': 2, '   app re h en sion</w>': 3, '   mon o poli z ed</w>': 1, '   pr er e qui si te</w>': 1, '   do z ed</w>': 1, '   sin cer est</w>': 1, ' ea ger</w>': 1, '   o x bri dge</w>': 2, '   su b ju ga ted</w>': 1, '   fa bu l ou s ly</w>': 1, ' a li g n</w>': 1, '   k a z ow</w>': 1, '   c ri me fi gh ter</w>': 3, '   ba t girl</w>': 2, '   ba at girl</w>': 2, '   you w s a</w>': 1, '   ban e</w>': 1, '   sh ow bi z</w>': 3, '   pl u mm et</w>': 1, '   win ded</w>': 2, '   s n ow man</w>': 6, '   a da p ted</w>': 2, '   m c gre g or</w>': 2, ' h mm m m</w>': 2, '   su d an</w>': 2, '   ba t com pu ter</w>': 1, '   bur r ow ing</w>': 1, '   ni gh tw ing</w>': 1, '   wi t less</w>': 2, '   ti ti ll a ted</w>': 1, '   att en tions</w>': 2, '   d ow dy</w>': 3, '   sp in ster</w>': 1, '   bi lli on a i re</w>': 5, '   plea s</w>': 1, '   co o lan ts</w>': 1, '   to x i f y</w>': 1, '   i de o lo gi es</w>': 1, '   w oo d ru e</w>': 1, '   ar bor ea l</w>': 1, ' en chan ting</w>': 1, '   ba t li ght</w>': 1, '   bi r d call</w>': 1, ' kn ow ing</w>': 1, '   proble m o</w>': 4, '   dis figu red</w>': 2, ' vi ll ain</w>': 1, '   c r y o s lu tion</w>': 1, ' sig na l</w>': 1, '   bri d</w>': 1, '   d un ce ly</w>': 1, '   im pe tu ous</w>': 1, '   ci ti z en ry</w>': 1, '   dea d li est</w>': 3, '   re play</w>': 3, '   sen ten c ing</w>': 1, '   un p uni sh ed</w>': 2, '   un s ea son ably</w>': 1, '   dis ar m ing</w>': 1, '   s li mm er</w>': 1, '   un ab om in able</w>': 1, '   pa ir ing</w>': 2, '   th a w s</w>': 1, ' o la tion</w>': 1, '   f re e zy</w>': 1, '   bi pe da l</w>': 1, '   he e ded</w>': 1, '   ear ed</w>': 1, '   man i ac al</w>': 4, '   en tw in ed</w>': 2, '   n y g ma</w>': 5, '   ba t su i ts</w>': 1, ' m re</w>': 1, ' afraid</w>': 1, '   mer i di an</w>': 4, '   na pi er</w>': 1, '   con tri bu tes</w>': 1, '   z om bi ed</w>': 1, '   de st ru c ts</w>': 1, ' ro lls</w>': 1, '   bo o sted</w>': 5, '   di st re ss ing</w>': 4, '   g un sho ts</w>': 5, '   ha sti ly</w>': 1, '   sch o l ar ly</w>': 2, '   hi tch es</w>': 2, '   bl ends</w>': 1, '   p sy cho ses</w>': 1, ' du al</w>': 1, '   ri d d l er</w>': 3, '   di ve st</w>': 2, ' ca pi ta li z ation</w>': 1, ' di vi ded</w>': 1, '   r in k a</w>': 1, '   s lan ta</w>': 1, ' chi em</w>': 1, '   no st ro vi a</w>': 1, '   out so ld</w>': 1, '   out cla ssed</w>': 1, '   bra in wa ves</w>': 5, '   te sto st er one</w>': 7, '   be the le m</w>': 1, ' sch o l ar ly</w>': 1, '   ja g ged</w>': 1, '   un p ack</w>': 7, ' ha u l</w>': 2, '   ob se ssi on al</w>': 1, '   ro ck e tt es</w>': 2, '   ma la y si an</w>': 4, '   dam n s</w>': 1, '   ba t boy</w>': 2, '   ser mon s</w>': 5, '   me t ro po l is</w>': 14, '   tru ck lo a d</w>': 3, '   ye ss ss s ss</w>': 1, '   bur ton</w>': 5, '   gi m mi e</w>': 4, ' f ry</w>': 1, '   ne u r ons</w>': 1, '   no ta tion</w>': 2, ' bo k</w>': 1, ' co g</w>': 1, ' na te</w>': 1, '   pen ch ant</w>': 1, '   an a gra ma ti c</w>': 1, '   ac ro sti c</w>': 1, '   c r y p to</w>': 4, ' gra p hi c</w>': 1, '   do th</w>': 12, '   bo de</w>': 2, '   mar cu ti o</w>': 1, '   sti ck le y</w>': 1, '   b ru ta li z ed</w>': 1, '   un car ing</w>': 3, ' wants</w>': 5, '   no th</w>': 2, '   har v </w>': 1, ' ro de</w>': 1, '   tu e</w>': 1, '   me s n on g es</w>': 1, '   g ac he</w>': 1, '   vi e</w>': 3, '   e c li p se</w>': 1, '   ar k ha m</w>': 1, '   bu n g le</w>': 1, '   out s mar ted</w>': 1, '   ac ned</w>': 1, '   ac ro b at</w>': 4, '   chi pp er</w>': 9, '   z on al</w>': 1, ' co a st al</w>': 2, '   un tr ac ea ble</w>': 3, '   ta m per er</w>': 1, '   in gre di en ts</w>': 3, '   sc en i c</w>': 3, '   cor to</w>': 2, '   ma l te se</w>': 2, '   v a le</w>': 5, '   w old</w>': 1, '   tr ac er</w>': 3, '   ex tr an e ous</w>': 1, ' b ru ce</w>': 3, '   mi mes</w>': 3, '   a m bi gu ous</w>': 1, '   ro of to ps</w>': 3, '   re ins</w>': 2, '   e ck har d t</w>': 2, '   si gh ting</w>': 3, ' su g ar</w>': 2, '   e spi on age</w>': 5, '   car m ine</w>': 10, '   ch ea p s k a tes</w>': 1, '   a m ne st y</w>': 3, '   ve he men tly</w>': 1, '   ba ll si est</w>': 2, ' st ard</w>': 1, '   re c lu si ve</w>': 2, '   ban k ro lls</w>': 2, '   char i ti es</w>': 6, '   a ver</w>': 1, '   f und ra i s er</w>': 3, '   ev il do ers</w>': 2, '   s w o op s</w>': 1, '   pre ys</w>': 1, '   ran d om ly</w>': 4, '   ar se ho le</w>': 4, '   g ri er son</w>': 4, '   al i</w>': 10, '   a h m</w>': 23, '   y ee e es</w>': 1, '   whi st l er</w>': 20, '   ch oo s er</w>': 1, '   ex pen di ture</w>': 1, '   ca ter ing</w>': 8, '   fi c tion al</w>': 2, '   9 3 </w>': 2, '   s ac king</w>': 2, ' e mer gen cy</w>': 2, '   h ow ls</w>': 6, '   inter je ction</w>': 1, '   pro fi ta bi li ty</w>': 2, '   s mar ten ing</w>': 1, '   el m er</w>': 5, '   ne g le c t ful</w>': 1, '   pro spe c tive</w>': 1, '   cl in ch</w>': 2, '   c ant</w>': 7, ' li ft</w>': 3, '   tri u mp h ant</w>': 3, '   s mo o thing</w>': 1, '   con n er y</w>': 2, '   li am</w>': 1, '   nee son</w>': 1, ' co o chi e</w>': 1, '   j ac ob son</w>': 1, '   ber n</w>': 4, '   mo s qui to es</w>': 3, ' go bl in</w>': 1, ' s na tch</w>': 2, '   di ll</w>': 5, '   bu n gh o le</w>': 4, ' wa d</w>': 1, ' du m pl ing</w>': 2, ' mon k ey</w>': 1, '   bea v is</w>': 24, '   a a a a ee e h h h h g</w>': 1, '   c ac tu ses</w>': 1, '   supp o st</w>': 1, '   por ta</w>': 2, ' po t ti es</w>': 1, '   m re</w>': 1, '   bu u u u u t</w>': 1, '   a g g gh g</w>': 1, '   n n n na ah</w>': 1, '   n n n n dam m it</w>': 1, '   n n n no o o</w>': 1, '   en ter t</w>': 1, ' an us</w>': 1, '   bo oo oo i i i ing</w>': 1, '   t ee ev ee ee e</w>': 1, '   l ac to se</w>': 2, '   in to l er ant</w>': 2, '   p al pi ta tions</w>': 1, '   thr r</w>': 1, ' ei ee e</w>': 1, ' oo oo ee e oo oo o</w>': 1, '   sch long</w>': 1, ' al mi gh ty</w>': 1, ' bu n gh o li oo o</w>': 1, '   j er kin</w>': 2, '   bu n gh o li o</w>': 1, '   o le o</w>': 1, '   bor k</w>': 10, '   hur le y</w>': 3, '   wh ac king</w>': 8, '   pre po si tion</w>': 2, '   re lea sing</w>': 2, '   ho o t ers</w>': 1, '   ma h</w>': 13, '   v ea gs</w>': 1, '   e ll o i se</w>': 1, '   d or e en</w>': 8, '   bu ss ing</w>': 1, '   hi gh land</w>': 7, '   ro a die</w>': 2, '   pa in t b ru sh</w>': 2, '   bo o y y</w>': 1, '   li en</w>': 8, ' a da m</w>': 3, '   ge use</w>': 1, ' ju ice</w>': 2, ' be e t les</w>': 1, ' be te l ge use</w>': 1, '   be te l ge use</w>': 6, '   gra ve st on es</w>': 1, '   be te l my er</w>': 1, ' p ho to s</w>': 1, '   sh r ou ds</w>': 2, ' mo an</w>': 1, '   tr an si tion al</w>': 1, '   bu tt er fi e ld</w>': 10, ' j an e</w>': 9, ' war ds</w>': 1, ' ge o gra p hi cal</w>': 1, '   per i me t ers</w>': 2, ' c ea sed</w>': 1, '   dis co ver i es</w>': 4, '   bo z man</w>': 1, ' t ac t ful</w>': 1, '   man ch u ri an</w>': 3, '   t un g</w>': 3, '   re fini sh</w>': 1, '   ga te le g</w>': 1, ' le e ze</w>': 1, '   de e t z es</w>': 5, '   de li a</w>': 5, ' han d bo ok</w>': 1, ' re cen tly</w>': 2, ' sh ee ts</w>': 2, '   ca se wor k er</w>': 1, '   inter ce ssi ons</w>': 1, ' bar bar a</w>': 3, '   ma i t lan ds</w>': 1, ' de st ro ying</w>': 1, ' und ead</w>': 1, '   m om mi e</w>': 3, '   lo li ta</w>': 1, '   ca sh ar o on i e</w>': 2, ' be e e</w>': 1, ' te l</w>': 3, ' jo oo se</w>': 1, ' can not</w>': 2, ' yourself</w>': 8, '   inter me di ate</w>': 1, ' li ver ed</w>': 1, '   de e t z</w>': 2, '   s an d wor ms</w>': 2, ' ye c ch h h</w>': 1, ' bu st ing</w>': 2, ' lo ves</w>': 2, '   pi g le ts</w>': 1, ' o th o</w>': 2, '   tr an ce</w>': 7, '   me di u ms</w>': 2, '   for mu la s</w>': 3, '   o th o</w>': 21, '   stu mb les</w>': 3, ' ex or ci s m</w>': 1, ' p ho to gra ph ed</w>': 1, ' ex or ci st</w>': 1, '   s lea z ing</w>': 1, '   be te l ge</w>': 1, '   a mi t y vi ll e</w>': 1, '   plan e lo a d</w>': 1, ' bed</w>': 6, '   ma i t land</w>': 2, '   bar b</w>': 2, '   be e y o o</w>': 1, ' t ee ful</w>': 1, '   q u</w>': 5, ' be e t le</w>': 1, '   ye c ch</w>': 1, ' de e e</w>': 2, ' li a</w>': 1, ' t z</w>': 1, '   vo l k s wa g en</w>': 3, '   gir r r l</w>': 1, ' be h ind</w>': 5, '   sy ll ab les</w>': 3, ' si mp le</w>': 3, '   g rea ts</w>': 2, '   de ca mp ed</w>': 1, '   dea d s k i</w>': 1, ' ho o ved</w>': 1, '   f ab l ed</w>': 2, '   re ar ran ge</w>': 4, '   fu mi ga ted</w>': 1, '   s z e ch u an</w>': 1, '   h un an</w>': 1, ' par king</w>': 1, '   p ra ti cal</w>': 1, '   c r ow b ar</w>': 5, '   tri ck le</w>': 1, ' fi lled</w>': 1, '   dar k room</w>': 3, '   d or mi t ory</w>': 1, '   in f er i ors</w>': 2, ' li ved</w>': 1, '   k a ma l i</w>': 1, ' d are</w>': 5, ' ar ti st</w>': 3, '   pu be sc ent</w>': 1, ' gh o sts</w>': 2, '   de mo li tion</w>': 7, ' ar chi te c tu ra l</w>': 1, ' v ani ty</w>': 1, ' dan ger ous</w>': 2, '   m s g</w>': 1, ' de ta il</w>': 2, ' nu r</w>': 1, ' ture</w>': 1, '   mar v el ou s ly</w>': 1, '   u r ban e</w>': 2, ' rea der</w>': 2, ' di ge st</w>': 1, '   cha bl is</w>': 2, '   in gr ound</w>': 1, '   li v able</w>': 1, '   on w ard</w>': 2, '   ti les</w>': 3, '   h y d ra ted</w>': 1, '   ch ro mi c</w>': 1, '   o x i de</w>': 1, '   sch oo l ed</w>': 4, '   vi ri di an</w>': 2, '   go oo od</w>': 1, ' ha un ted</w>': 2, '   he ll oo o</w>': 1, '   sh an n on</w>': 1, '   go bl in</w>': 2, '   vi z ard</w>': 1, '   pre sen ces</w>': 1, '   hu ge ly</w>': 2, '   re cu per a tive</w>': 1, '   h oun ding</w>': 3, '   sor en ess</w>': 1, '   gar din er</w>': 74, '   r and</w>': 41, '   in je ction</w>': 5, '   fo es</w>': 1, ' la w y ers</w>': 3, '   re st ri c ting</w>': 1, '   le gi s la ted</w>': 1, '   pre f er able</w>': 3, '   re f re sh ing</w>': 6, '   ba ff les</w>': 1, '   tr ans ac tions</w>': 6, '   in e f fi ci en cy</w>': 1, '   thir d ly</w>': 1, '   ra p ha el</w>': 4, '   mor ton</w>': 9, '   un co il</w>': 1, '   pu r ged</w>': 2, '   de t ac h ed</w>': 2, '   in fu se</w>': 1, '   au b re y</w>': 17, '   mu d d ling</w>': 1, '   a ll en by</w>': 2, '   ex a min es</w>': 1, '   per di ta</w>': 2, '   vi e w ers</w>': 6, '   v la di m ar</w>': 1, '   s k ra p in o v </w>': 6, '   k r y lo v </w>': 3, '   f ab les</w>': 1, '   k r y lo vi an</w>': 1, '   inter chan ge</w>': 3, '   en d or sed</w>': 1, '   r ow le y</w>': 2, '   hon or ing</w>': 1, '   mi su se</w>': 2, ' k in g ma k er</w>': 1, '   f lin ty</w>': 1, '   l ab or er</w>': 1, '   st er o id</w>': 1, '   tr ans fu si ons</w>': 2, '   g lan ces</w>': 5, '   gh o st wri ter</w>': 1, '   sti e g l er</w>': 1, '   st e w in</w>': 1, '   ye ll in</w>': 8, '   go b b le de go ok</w>': 1, '   s li pp in</w>': 3, '   re spi te</w>': 1, '   sh ru b s</w>': 2, '   he d g es</w>': 5, '   sp r in g time</w>': 3, '   re si ded</w>': 1, '   be dri d d en</w>': 1, '   s ar ac in i</w>': 1, '   bri ck work</w>': 1, '   in cen ti ves</w>': 1, '   l ev i ty</w>': 2, '   l ac on i c</w>': 1, '   jo han na</w>': 1, '   ta ss</w>': 1, ' cha un c ey</w>': 1, '   ab le st</w>': 1, '   k au f man</w>': 20, '   so p hi sti ca te</w>': 2, '   re kn own</w>': 1, '   no vo g ro d</w>': 1, '   as cer ta in ed</w>': 1, ' business</w>': 12, '   de pen d ence</w>': 1, ' pu pp et</w>': 1, '   ex tra v a g an z a</w>': 1, '   pu pp e te er</w>': 11, '   ma l k o vi ch</w>': 45, '   ma k el</w>': 1, '   o ver ea t ers</w>': 1, '   lo tt e</w>': 18, '   la mb cho p</w>': 2, '   pla ton i ca lly</w>': 1, '   p ea ked</w>': 4, '   e s ki mo s</w>': 3, '   fe l d man</w>': 5, '   tr ans se x u al</w>': 4, '   f er re t</w>': 2, '   p un c ture</w>': 6, '   in a de qu ac y</w>': 1, '   man t in i</w>': 10, '   si mu la tes</w>': 2, '   f l or is</w>': 8, '   re son a tes</w>': 1, '   con st ri ct</w>': 1, '   e pi g lo tt is</w>': 1, '   pi d d ling</w>': 2, '   y ex </w>': 1, '   le st er cor p</w>': 4, '   u ti li ze</w>': 1, '   ab lu tions</w>': 1, ' ex por t</w>': 4, '   e f fe c ti ve ly</w>': 4, '   be et</w>': 1, ' sp in ac h</w>': 1, '   ju i c er</w>': 2, '   nu bi le</w>': 1, '   in to x i ca te</w>': 1, '   sp un k</w>': 6, '   sp as m</w>': 4, '   in de ci ph er able</w>': 1, '   under st an</w>': 5, '   im pe di men to lo g y</w>': 1, '   g loo p h</w>': 2, '   mi s sp o ke</w>': 2, '   in c</w>': 4, '   un re qui ted</w>': 1, '   un bo lt</w>': 1, ' pro min ent</w>': 1, '   b al d ne ss</w>': 2, '   ob li qu e</w>': 2, '   mer t in</w>': 5, ' f le mm er</w>': 1, '   doll face</w>': 4, '   i ll u sor y</w>': 1, '   bu d d h a</w>': 8, '   bu d we i s er</w>': 4, '   ki s m et</w>': 3, '   bu u u h h pp a a h h h h n n n</w>': 1, '   mu h h ha h h h h h</w>': 1, '   a h h h n n na a a</w>': 1, '   no ll l tu u u k k k a a a ar a ll l ll</w>': 1, '   ta sha bar ar as ss ss su u u u u sa a a a a a a</w>': 1, '   n n n n n n na a a a a an n n n n n n n n c c c c c c ee ee ee e</w>': 1, '   m wa a a a a a</w>': 1, ' ma h h h h h k k k k k</w>': 1, '   ss s se ee ee en</w>': 1, '   dre ary</w>': 6, '   si t com s</w>': 3, '   an ch ors</w>': 5, '   wa tch ma k er</w>': 3, '   di e ts</w>': 1, '   vi v ant</w>': 1, '   sch op en ha u er</w>': 1, '   ex tra or din a i re</w>': 2, '   ph y si o lo gi cal</w>': 4, '   t re be k</w>': 1, '   pu pp e try</w>': 2, ' tru </w>': 3, ' du el</w>': 1, '   ar ose</w>': 1, '   p hi lo den dr on</w>': 1, '   lu x or</w>': 1, '   w r ac king</w>': 3, '   com in gs</w>': 2, '   go in gs</w>': 4, '   t ou gh i e</w>': 2, '   f l ori st</w>': 2, '   ge h g in n is</w>': 1, '   on da h</w>': 1, '   fo am</w>': 4, '   y ad da</w>': 3, '   be ck on</w>': 1, '   f le mm er</w>': 1, '   in ha bi ts</w>': 2, '   p sy chi ca lly</w>': 1, '   in do c tr in ation</w>': 1, '   su n k en</w>': 3, '   me mb ran ous</w>': 1, '   supp re ssed</w>': 3, '   ac tu a li z ation</w>': 1, '   t ou red</w>': 3, '   con for mi ty</w>': 2, ' e qu us</w>': 1, '   bur ly</w>': 1, '   p y ro te ch ni c s</w>': 2, '   ob s cu ri ty</w>': 1, '   ou i j a</w>': 5, ' cra i g</w>': 2, '   w und er</w>': 1, '   w ba i</w>': 1, '   cra i g g y</w>': 1, ' lo tt e</w>': 1, '   k no b s</w>': 1, '   un wi el dy</w>': 2, ' ce il in ged</w>': 2, '   mar ri e</w>': 1, '   di minu ti ven ess</w>': 1, '   al ms</w>': 1, '   f en se</w>': 1, ' f en se</w>': 1, '   ja il bi r d</w>': 1, '   j one</w>': 1, '   whi te fo l ks</w>': 3, '   ha ll e</w>': 10, ' ha ll e</w>': 1, '   whi te fo l k</w>': 1, ' n ough</w>': 3, '   bo d w in</w>': 4, '   o ber l in</w>': 1, '   bo d w ins</w>': 3, '   s lo ps</w>': 1, '   w oo d land</w>': 3, '   su g gs</w>': 7, '   gar ner</w>': 4, '   su g g</w>': 2, '   ch or es</w>': 5, '   se the</w>': 8, ' sh ow ed</w>': 4, '   han d saw</w>': 1, '   whi te woman</w>': 1, '   un ri le</w>': 1, '   ou th u r t</w>': 1, '   hur ter</w>': 1, ' sch oo l t ea ch er</w>': 1, ' sta mp</w>': 1, '   c in c in att i</w>': 1, '   re pro du c ed</w>': 2, '   b re e der</w>': 1, '   th a ta way</w>': 1, '   t re e h or n</w>': 7, '   le b ow s k i</w>': 41, ' pi ss ers</w>': 1, '   cu l pri ts</w>': 1, '   dis con fir m</w>': 1, '   le bar on</w>': 3, '   re fu ge e</w>': 2, '   k un st l er</w>': 1, ' com pe ers</w>': 1, '   g under s ons</w>': 2, '   fa w n</w>': 1, '   ff f</w>': 1, ' fa bu l ous</w>': 1, ' se mi te</w>': 3, '   nu ss ing</w>': 1, '   ni hi li st</w>': 2, '   ve e</w>': 10, '   ha f</w>': 3, '   ee f en</w>': 1, '   z ere</w>': 3, '   ro o l z</w>': 1, '   mo de st ly</w>': 2, '   pri c ed</w>': 2, '   mor tu ary</w>': 4, '   s ca tt er ing</w>': 4, '   sh ee sh</w>': 4, ' pe ers</w>': 1, '   pe ed</w>': 13, '   p y ja ma s</w>': 4, '   pu sh over</w>': 3, '   di g by</w>': 3, '   sha m ma s</w>': 1, '   din ged</w>': 1, '   sh om er</w>': 1, '   sha b b as</w>': 6, ' po st</w>': 1, '   bur k ha l ter</w>': 1, '   qu in t an a</w>': 2, '   il y i ch</w>': 1, '   u l y an o v </w>': 1, '   pe der a st</w>': 3, '   b ran d t</w>': 5, ' he h</w>': 5, '   o c cu p ying</w>': 3, '   ac hi ev ers</w>': 3, '   ac hi ever</w>': 1, '   cre e d ence</w>': 2, '   co l or ation</w>': 1, '   ab i d es</w>': 1, '   se m is</w>': 2, '   than ki e</w>': 1, '   ra d for d</w>': 2, '   si z able</w>': 4, '   inter ac tive</w>': 2, '   lo g ja mm in</w>': 1, '   un sp o il ed</w>': 1, '   d on ne lly</w>': 1, '   fin o</w>': 1, '   fuck e en</w>': 2, '   j or</w>': 10, '   in c rea ses</w>': 3, '   ad h er ing</w>': 1, '   re gi men</w>': 2, '   li mber</w>': 1, '   f la sh back</w>': 4, '   mm nu n</w>': 1, '   hur on</w>': 17, ' e ff</w>': 1, '   st ran ds</w>': 3, '   du der</w>': 2, '   ab du ctor</w>': 1, '   au to ba h n</w>': 1, ' te ch no</w>': 1, '   or bi son</w>': 3, '   fr in g es</w>': 1, ' au to ba h n</w>': 1, ' c o</w>': 3, '   ha u ff</w>': 1, '   re spe c fu lly</w>': 1, ' ab du ction</w>': 1, '   fa tu ous</w>': 1, '   n y mp h o</w>': 1, '   k al hu a</w>': 1, '   z e st y</w>': 1, '   sa t y ri as is</w>': 1, '   n y mp h om ani a</w>': 1, '   com pu l si ve ly</w>': 2, '   co i tu s</w>': 4, ' joh n son</w>': 2, ' di ck</w>': 3, '   el fran c o</w>': 1, '   com men ded</w>': 2, '   so b cha k</w>': 1, '   me di c s</w>': 5, '   ca ke wal k</w>': 1, '   c r y baby</w>': 1, '   ac h t un g</w>': 1, '   k ou fa x</w>': 1, '   t ev ye</w>': 1, '   sa b ba th</w>': 9, '   ca th ar ti c</w>': 1, '   ki d na pp ers</w>': 3, '   er ev </w>': 2, ' tt e</w>': 4, '   9 6 </w>': 4, '   d un ce</w>': 1, '   1 5 6 </w>': 1, '   ca m ro se</w>': 1, '   sy m pa th i z ing</w>': 1, '   mar mo t</w>': 1, '   a mp hi bi ous</w>': 1, '   ni hi list s</w>': 3, ' sta ins</w>': 1, '   vi c ti m less</w>': 2, '   se l ves</w>': 5, '   s che du ling</w>': 3, '   a i t z</w>': 1, '   cha i m</w>': 2, ' a i tch</w>': 1, ' cho p</w>': 3, '   th a a at</w>': 1, '   han do ff</w>': 1, '   ba k sh ee sh</w>': 1, '   und u de</w>': 1, '   th a a a at</w>': 1, '   ch in o</w>': 8, '   wa v in</w>': 3, ' p ac i fi s m</w>': 1, '   ca me l fuck er</w>': 1, '   p ac i fi s m</w>': 3, '   pu sho ver s</w>': 1, ' ro b in</w>': 3, ' be y on d</w>': 2, '   p ac i fi ci sts</w>': 1, '   ob je ctor</w>': 1, '   hon o lu l u</w>': 2, ' br ought</w>': 4, ' c y n th i a</w>': 1, '   h er z el</w>': 1, '   v oo d en</w>': 1, '   bri tch</w>': 1, '   s row</w>': 1, '   v in d ow</w>': 1, '   k ar</w>': 1, '   v a tch</w>': 3, '   di st en ded</w>': 1, '   par lan ce</w>': 1, '   b la ther ing</w>': 2, '   ni tw it</w>': 8, '   te sti c les</w>': 2, '   wee k day</w>': 2, '   du den ess</w>': 1, '   du der in o</w>': 1, '   b re vi ty</w>': 3, '   han d out</w>': 3, '   mi c tu ra ted</w>': 1, '   par l a</w>': 2, '   u sted</w>': 1, '   in g le se</w>': 1, '   u r in ate</w>': 3, '   bu lli es</w>': 2, '   sp in al s</w>': 1, '   go l d bri ck er</w>': 1, '   go l d bri cking</w>': 1, '   me x</w>': 2, '   stu mb le bu ms</w>': 1, '   han d pi cked</w>': 1, '   we ts</w>': 4, '   j ar hea ds</w>': 1, '   to ma s</w>': 2, ' b loo d bro ther</w>': 1, '   pu re b loo d</w>': 1, '   da ma s k in o s</w>': 2, ' each</w>': 3, '   rea p ers</w>': 5, ' dea l er</w>': 1, '   si mp li fi ed</w>': 2, ' wi r ing</w>': 1, '   no ma k</w>': 9, '   he mo g lo b in</w>': 5, '   me ta bo li s ms</w>': 1, '   b loo d ban k</w>': 1, '   pi er c in gs</w>': 1, '   sc ar r ing</w>': 1, '   re in har d t</w>': 3, '   ni h</w>': 1, '   do ts</w>': 7, '   su ck head</w>': 1, '   for war ds</w>': 1, '   mu ff et</w>': 1, ' s li de</w>': 1, '   he i f er</w>': 2, '   o j </w>': 1, '   sp li c ed</w>': 1, '   f le che tt e</w>': 1, '   re t ro vi ra l</w>': 1, ' ra p id</w>': 1, '   de to x</w>': 3, ' tur k ey</w>': 1, '   bu tt er c up</w>': 10, '   ra tch et</w>': 2, ' na ds</w>': 1, '   no tch es</w>': 1, '   f ar ting</w>': 2, '   war mb loo d</w>': 1, '   a the l fi k i</w>': 1, '   sin gen i a</w>': 1, '   a ma to</w>': 1, '   th i a vo lo s</w>': 1, '   sc ri p ted</w>': 2, '   un fo l ding</w>': 2, '   in ev i ta bi li ty</w>': 1, '   n y ss a</w>': 1, '   ph en o t y pe</w>': 1, '   pu ck er ed</w>': 2, '   co o t chi e</w>': 1, '   f re e b all</w>': 1, '   ri ff</w>': 3, '   s cu d ster</w>': 1, '   tw ea ked</w>': 3, '   p ho sp h or</w>': 1, '   co lli ma ted</w>': 1, '   u v </w>': 1, '   fi d d ling</w>': 3, ' wi lli es</w>': 1, '   ra mp ing</w>': 1, '   ni tr ous</w>': 1, ' o x i de</w>': 1, '   pi st ons</w>': 1, '   c ran k sha ft</w>': 1, '   char ger</w>': 1, '   s cu d</w>': 4, '   he ma to lo gi st</w>': 1, '   stra ys</w>': 2, '   ne w bor n</w>': 4, '   re f re sh es</w>': 1, '   p un g ent</w>': 1, '   ir re pre ssi ble</w>': 1, ' b re ed</w>': 4, '   st r en g th s</w>': 4, ' wal k er</w>': 1, '   ph ar m ac e u ti ca ls</w>': 3, ' b lo ck er</w>': 1, '   o c t y l</w>': 1, '   sa li c y late</w>': 1, '   a po ca l y p se</w>': 2, '   di sp ar ad or</w>': 1, '   la ma gr a</w>': 5, '   cle an sing</w>': 1, '   ha ll mar k</w>': 2, ' di r t</w>': 3, '   v a le ts</w>': 5, '   d om in o</w>': 27, '   pre ven tive</w>': 1, ' b es</w>': 3, '   g l y p h</w>': 2, '   dea c on</w>': 8, '   k am</w>': 1, '   per i car di al</w>': 1, '   bi con v ex </w>': 1, '   h y po ch ro mi c</w>': 1, '   p m n s</w>': 1, '   b in u c lea ted</w>': 1, '   mon on u c lea ted</w>': 1, '   er e bu s</w>': 2, ' ton gu e</w>': 1, '   d ra gon e tt i</w>': 1, '   sc ri b bl ed</w>': 2, ' pro p he si ed</w>': 1, '   re t ro vi ru s</w>': 1, '   mu ta te</w>': 2, ' c ell</w>': 1, '   an e mi a</w>': 2, '   v a m pi ri s m</w>': 1, '   he mo l y ti c</w>': 1, '   bo d y coun t</w>': 1, '   o ver com es</w>': 1, '   an a ph y l ac ti c</w>': 2, '   h om un cu l us</w>': 1, '   ra tt le s na ke</w>': 3, '   k u r th</w>': 4, '   e d m un ds</w>': 18, '   d om in i</w>': 17, ' an na</w>': 1, ' c ow ar d ly</w>': 1, '   ta o s</w>': 2, '   lea vi t t</w>': 3, '   ob st e tri ci an</w>': 1, ' d ou b t ful</w>': 1, '   pa st or al</w>': 1, '   gla de</w>': 3, '   ga m bo ling</w>': 2, '   d ru id</w>': 2, '   ti ck e ted</w>': 1, '   com po st</w>': 5, '   la y ers</w>': 6, '   b ack p ac ks</w>': 1, '   vo i i i i i ces</w>': 1, ' ma il ed</w>': 1, ' morning</w>': 6, ' s wee ter</w>': 1, '   v om i ted</w>': 2, '   ta ssi o</w>': 1, ' jo sh</w>': 1, '   pi e ti es</w>': 1, '   gi g g ling</w>': 2, ' co tter</w>': 4, ' bu ll shit</w>': 6, ' case</w>': 2, ' chan ce</w>': 3, '   s co o ch</w>': 1, '   we ir ded</w>': 2, ' bi g ger</w>': 1, '   par r</w>': 6, ' t ree</w>': 3, '   ru st in</w>': 2, '   1 8 5 8 </w>': 1, ' a gre e</w>': 1, ' s ke tch</w>': 1, '   sa pl ing</w>': 1, ' cha l ked</w>': 1, ' my th</w>': 1, '   t ow n spe ople</w>': 1, '   what z er name</w>': 2, '   te er</w>': 3, ' p sy chi c</w>': 2, ' hou rs</w>': 1, '   e b ay</w>': 1, '   bur ki tt s vi ll e</w>': 2, ' chi ll</w>': 2, '   bab o ons</w>': 1, ' h un ch</w>': 1, '   st on ers</w>': 1, '   a mp he ta min es</w>': 1, ' b la ir</w>': 3, ' sh h h h</w>': 1, ' chri s sa ke</w>': 1, '   do ze</w>': 1, '   t y p ho id</w>': 7, ' d om in i</w>': 1, ' hi lls</w>': 1, '   e lly</w>': 11, '   ke d w ard</w>': 5, ' f re e z ing</w>': 1, ' god da mi t t</w>': 1, '   co tter</w>': 4, ' any where</w>': 2, '   dea d bo lt</w>': 2, '   te le s co pi c</w>': 2, ' se lling</w>': 3, '   ca s a</w>': 6, ' p ac king</w>': 1, '   car ca ss es</w>': 2, '   sch le pped</w>': 1, '   re li ever</w>': 1, '   p an as on i c</w>': 1, '   d v d</w>': 1, '   nu t job</w>': 1, '   ar en d t</w>': 1, '   ar end</w>': 1, '   ex e mp ted</w>': 1, '   s nu ff ing</w>': 1, '   sha p es</w>': 4, '   en t re pr en e u ri al ship</w>': 1, '   4 0 3 </w>': 1, ' wi tch</w>': 2, ' ser i ous</w>': 2, '   wi c can</w>': 1, ' gra du ate</w>': 1, '   u ma ss</w>': 1, ' s ac red</w>': 1, ' da w n</w>': 2, ' si gh t in gs</w>': 1, '   ta pp y</w>': 3, '   du sts</w>': 1, '   cha f ing</w>': 1, '   mi sc ar ry</w>': 1, '   ex plan a tions</w>': 6, '   di se mb ow el ment</w>': 1, '   su m ac </w>': 1, '   de pri v ation</w>': 4, ' i f s</w>': 1, '   per se cu tion</w>': 3, '   in f la m ma t ory</w>': 1, '   to a st ing</w>': 2, '   ma ll ows</w>': 1, '   me u ri ce</w>': 12, '   ab by</w>': 8, '   win d brea k er</w>': 3, ' anyway</w>': 6, '   hea l th i est</w>': 1, '   com m uni ca ted</w>': 2, '   o il ers</w>': 1, '   r c</w>': 1, '   lu b bo ck</w>': 1, ' fri day</w>': 2, ' figu red</w>': 1, ' han d l ed</w>': 3, '   s lu i c ing</w>': 1, '   ir ri ta ted</w>': 4, '   al co ho li c s</w>': 1, '   ge o lo gi st</w>': 2, ' je ff re y</w>': 2, ' s an dy</w>': 1, '   ca ll ate</w>': 1, '   ki l o</w>': 8, '   p ab l o</w>': 10, '   j un g</w>': 9, '   di st ri bu ting</w>': 1, '   ca y</w>': 2, '   au gu s to</w>': 3, '   o li ver as</w>': 1, '   de j en</w>': 1, '   mar i ca da</w>': 1, '   pu es</w>': 2, '   jo d an</w>': 1, '   na die</w>': 2, '   ma t ar</w>': 2, '   de be mo s</w>': 1, '   ha bl ar le</w>': 1, '   es</w>': 14, '   uni c a</w>': 1, '   man er a</w>': 1, '   man o</w>': 1, '   vo s</w>': 1, '   pi ca hi el o</w>': 1, '   e so</w>': 1, '   ti e mp o</w>': 1, '   e sta mo s</w>': 1, '   ca s i</w>': 1, '   o ch en ta</w>': 1, '   ti r o</w>': 1, '   vo l ar</w>': 1, '   he ch ar</w>': 1, '   hi ju e pu ta</w>': 1, '   car r o</w>': 2, '   en ci ma</w>': 1, '   qu er es</w>': 1, '   de ci r</w>': 1, '   ha c er</w>': 4, '   mi r th a</w>': 9, '   no se b le e ds</w>': 1, '   sin se mi ll a</w>': 1, '   o ti s vi ll e</w>': 1, '   ven g an ce</w>': 1, '   a mor ci to</w>': 1, '   pa dr in o</w>': 3, '   di st ri bu tor</w>': 3, '   clo th s</w>': 2, '   mo ther lo de</w>': 1, '   ma ss ac hu se tt es</w>': 2, '   for ea l</w>': 5, '   bl an c o</w>': 1, '   j un to</w>': 1, '   e sto y</w>': 2, '   qui er en</w>': 1, '   hi ju e pu ta s</w>': 1, '   ca m pe sin o s</w>': 1, '   to do</w>': 1, '   e st a</w>': 4, '   n ori e g a</w>': 5, '   l en i ent</w>': 3, '   pe sa do s</w>': 1, '   la und er</w>': 3, '   co lo m bi a</w>': 5, '   dan bu ry</w>': 2, '   ha llo</w>': 4, '   de l ga do</w>': 8, '   me de ll in</w>': 2, '   ma gi c o</w>': 2, '   in tri gu es</w>': 2, ' l ead</w>': 1, '   gra ms</w>': 4, '   t ac h y car di a</w>': 1, '   ki ll jo y</w>': 1, '   la y away</w>': 5, '   cre den z a</w>': 1, '   ti gh t wa d</w>': 1, '   ch ea p s k ate</w>': 2, '   sh may away</w>': 1, '   fami li a</w>': 3, '   car ce l</w>': 1, '   ra ta</w>': 1, '   poli ci a</w>': 2, '   sa p o</w>': 1, '   er m ine</w>': 1, '   mi d d l ed</w>': 2, '   du ll i</w>': 4, '   li ds</w>': 3, '   ba g gi es</w>': 2, '   or e g an o</w>': 4, '   k ri st in a</w>': 6, '   so ci a li ze</w>': 2, '   a mer i can o</w>': 2, '   mo ta</w>': 1, '   ri d dan ce</w>': 2, '   a m h er st</w>': 2, '   ho l y o ke</w>': 1, '   pre supp o ses</w>': 1, '   par at ro op ers</w>': 2, '   n lf</w>': 9, ' hi d i</w>': 1, '   al ger i a</w>': 10, '   d ac ha u</w>': 1, '   bu ch en wa ld</w>': 1, ' t or ture</w>': 1, '   p ru d ence</w>': 2, '   a ll u si ve</w>': 2, '   di en</w>': 2, '   ph u</w>': 1, '   de ter min ing</w>': 2, ' gr ou p</w>': 2, '   al gi ers</w>': 3, ' di st ri ct</w>': 1, '   a h med</w>': 1, '   ha cen e</w>': 2, '   re or g ani ze</w>': 1, '   un de ci ded</w>': 1, '   k a der</w>': 5, '   sa ar i</w>': 1, '   under go</w>': 1, '   al ger i an</w>': 1, '   al ger i ans</w>': 2, ' con fe ssed</w>': 1, '   cor bi ere</w>': 2, '   con su l ar</w>': 1, '   la qui ere</w>': 1, '   bra s l ow</w>': 10, '   k o e h l er</w>': 4, '   pa le y</w>': 15, '   ro se bur g</w>': 2, '   na sa l</w>': 16, '   car di o lo gi st</w>': 5, '   sho v el ing</w>': 8, '   m c cu r dy</w>': 3, '   ni gh t stand</w>': 3, '   dri st an</w>': 2, '   in ge st ing</w>': 1, '   me mb ran e</w>': 3, '   de con ge st ant</w>': 1, '   ro st on</w>': 12, ' lo ver</w>': 1, '   du lan ey</w>': 8, '   bi se x u al</w>': 1, '   su b stan ti a ting</w>': 1, '   t ro x ell</w>': 2, '   re con st ru c ted</w>': 1, '   bl in d fo l ded</w>': 3, '   pre li m</w>': 2, '   me mb ran es</w>': 2, '   ar r h y th mi a</w>': 1, '   vi al</w>': 10, '   car den as</w>': 6, '   tra m me l</w>': 3, '   ex er ting</w>': 1, '   se x u a li ty</w>': 11, '   b y pa ss</w>': 7, '   per ju re</w>': 1, '   pe on y</w>': 3, '   men st ru al</w>': 1, '   di sp en s ary</w>': 4, '   con fi ding</w>': 1, '   sle e p less</w>': 9, '   pro mp ted</w>': 1, '   sha tter</w>': 9, '   ma st er fu lly</w>': 1, ' bu m</w>': 3, '   b re e z es</w>': 3, '   vo ye u r</w>': 3, '   tr en dy</w>': 6, '   s n ar l</w>': 1, '   di v or ce e</w>': 1, '   a ll e ga tion</w>': 1, '   in te lli gen tly</w>': 1, '   a sp er si ons</w>': 1, '   co a x ed</w>': 1, ' dis cour te ous</w>': 1, '   in sin u a tions</w>': 1, '   sa tt l er</w>': 8, '   do g gi es</w>': 1, ' wor th</w>': 3, '   chri st ma ses</w>': 1, '   ee e w w</w>': 2, '   an a is</w>': 3, '   n in</w>': 2, ' ex pl or ing</w>': 1, '   hi ll cre st</w>': 1, '   no te bo o ks</w>': 3, '   su b side</w>': 1, '   ro se man</w>': 3, '   cre ma ting</w>': 1, '   fi x in gs</w>': 2, '   ma dge</w>': 2, ' f lo y d</w>': 1, '   s an d for d</w>': 1, '   fran ce sc a</w>': 8, '   ar r ac c in o s</w>': 1, '   z e pp o l is</w>': 1, '   a w ning</w>': 1, '   bar i</w>': 4, '   c ri er</w>': 1, ' jo king</w>': 1, '   me an in gs</w>': 2, ' cle an ing</w>': 2, '   lon er</w>': 4, '   sho sh one</w>': 1, '   sa f ar is</w>': 2, ' ad ju sted</w>': 1, '   un cor ked</w>': 1, '   i ll in o s</w>': 1, ' mo tor</w>': 1, '   lo am</w>': 1, '   d y an</w>': 1, '   fran ni e</w>': 1, '   de li ri ous</w>': 5, '   c in n ab ar</w>': 1, '   h un g ers</w>': 1, '   bro th a</w>': 1, ' b on es</w>': 1, '   wh u pped</w>': 1, '   mu cho s</w>': 1, '   mo s co s</w>': 1, '   bu g g in</w>': 6, ' bu ry</w>': 1, '   le go land</w>': 1, '   vi al s</w>': 1, '   ba d da ss</w>': 1, ' ad j ac ent</w>': 1, '   car pa l</w>': 2, '   g an g st a</w>': 1, '   he ll out</w>': 1, '   bu z z es</w>': 2, '   le x us</w>': 1, '   ju sti f ying</w>': 1, ' wh oo o</w>': 1, '   m ac ki e</w>': 1, '   f ack</w>': 2, '   wa ck o s</w>': 2, ' un h</w>': 3, ' f oo ls</w>': 1, '   k en w o od</w>': 1, ' ey</w>': 1, '   go oo oo od</w>': 1, '   loo oo ong</w>': 1, '   g ran ma</w>': 7, '   ma ma s</w>': 1, '   sh in in</w>': 1, '   ad vi s er</w>': 1, ' nu ff</w>': 2, '   un fuck ed</w>': 1, '   gra s sho pp as</w>': 1, ' at ti e</w>': 1, '   mu th a fucking</w>': 1, '   ac re</w>': 5, '   offi s a</w>': 1, '   k or ru p t s k y</w>': 1, '   h un g ri er</w>': 4, '   ra ven ing</w>': 1, ' ho od</w>': 4, '   c in n</w>': 1, '   h or d es</w>': 2, '   sha mb ling</w>': 1, '   ro ss more</w>': 3, '   under v al u ed</w>': 1, ' om b</w>': 1, '   fa tter</w>': 5, '   ei gh t b all</w>': 2, '   s ca a a a a a a a a ar ed</w>': 1, '   bl ack st one</w>': 1, '   g on</w>': 11, '   g in su </w>': 1, '   fran c o</w>': 9, ' to a sted</w>': 1, '   cre e ds</w>': 1, '   in h er i ts</w>': 3, '   sta go le e</w>': 1, '   car pe</w>': 7, '   s q wee ep</w>': 1, '   in n er up</w>': 1, ' a li en s</w>': 1, '   re ti cu l a</w>': 1, '   pe et</w>': 1, '   mo ly</w>': 2, '   hea d ch ee se</w>': 1, ' s ou l</w>': 1, '   i lli b ent</w>': 1, '   o ver rea c ted</w>': 3, '   g in o</w>': 31, '   mar z z one</w>': 9, '   k ar po l i</w>': 1, '   hon ked</w>': 1, '   bi an ch in n i</w>': 5, '   cor k y</w>': 16, '   ra je ev </w>': 4, '   har d w o od</w>': 1, '   an ge l o</w>': 21, ' sh e lly</w>': 5, '   wor k a ho li c</w>': 2, '   re di st ri bu tion</w>': 2, '   la under er</w>': 1, '   t in k er</w>': 5, '   u l ter i or</w>': 3, '   wom bo s i</w>': 4, ' sle e p ing</w>': 4, ' fin ally</w>': 3, ' ends</w>': 1, '   bu d ge ts</w>': 2, '   le d ger</w>': 4, '   mar se i ll e</w>': 2, '   n y k wan a</w>': 2, ' pi lls</w>': 1, ' hea d ac h es</w>': 1, '   mp g</w>': 2, '   de b en ture</w>': 1, ' s wa ps</w>': 1, '   pla it</w>': 2, ' ac cor d</w>': 1, '   mo t or way</w>': 2, '   b on j our</w>': 7, '   cu t th ro at</w>': 1, '   x x x x x x</w>': 18, '   z u ri ch</w>': 9, ' qui et</w>': 4, '   si gh t lin es</w>': 1, '   ga z i lli on</w>': 2, '   k re u t z</w>': 3, '   f ra u du l ent</w>': 1, '   ci r cu m v ent</w>': 2, '   ev ac u a ting</w>': 1, '   vi e tr i</w>': 1, ' er gen cy</w>': 1, ' er vi ces</w>': 1, '   wa sh er</w>': 5, ' 4 1 </w>': 1, '   in du ctor</w>': 1, '   bo b b in</w>': 1, '   th rea ded</w>': 1, ' so l en o id</w>': 1, ' so d</w>': 1, '   l ow ry</w>': 9, ' ven</w>': 1, ' sa bo ta ge</w>': 2, ' y er</w>': 1, '   no st ri l</w>': 14, ' a ven</w>': 1, '   tu ttle</w>': 19, '   mu mb le</w>': 6, ' a m per ed</w>': 1, '   d ow s er</w>': 1, ' i x ed</w>': 1, ' di tion ing</w>': 1, ' e le ph on ed</w>': 1, '   st or er oo m</w>': 3, '   la y ton</w>': 13, ' j i ll</w>': 1, '   fi el ding</w>': 7, '   con si st en tly</w>': 3, ' pa y ers</w>': 2, ' e f fe c tive</w>': 1, '   hel p man n</w>': 11, '   sp ort s man ship</w>': 2, '   b om b in gs</w>': 3, '   pa vi li on</w>': 5, ' tu ttle</w>': 1, '   x ma s</w>': 2, '   de tain</w>': 3, '   un sa v ou ry</w>': 1, '   en qui ry</w>': 3, ' ba ff l ed</w>': 1, '   bu tt les</w>': 2, '   su b ver sion</w>': 1, '   inter ro ga te</w>': 4, '   er a di ca tion</w>': 2, '   ri gi d</w>': 5, ' 2 1 5 </w>': 3, ' 5 8 </w>': 6, ' 7 3 2 </w>': 4, '   4 1 2 </w>': 3, '   j af fe</w>': 3, '   vi vi d ly</w>': 2, ' win king</w>': 1, '   tri p le ts</w>': 1, '   ter ri fi ca lly</w>': 1, '   s lu mm ing</w>': 1, '   ar sed</w>': 1, '   ob st ru c tive</w>': 1, '   l or ry</w>': 2, '   v as</w>': 9, '   ou tr ace</w>': 1, '   d z</w>': 1, ' 0 1 5 </w>': 1, '   ir q </w>': 1, '   au th ori se</w>': 1, '   ti er</w>': 2, '   2 7 1 5 6 7 8 9 </w>': 1, ' 0 7 4 3 2 8 </w>': 1, '   cen sus</w>': 2, '   d or man ted</w>': 1, '   st or e house</w>': 1, '   ex ci sed</w>': 1, '   w r on g ly</w>': 1, '   e le c t ro me mor y ther a p y</w>': 1, '   ex pe di ting</w>': 3, '   in vo i c ed</w>': 1, '   mi s ma tch</w>': 1, ' 0 6 </w>': 2, '   de bi ted</w>': 1, ' 5 1 7 3 </w>': 1, '   5 0 0 1 </w>': 1, '   nu mer o</w>': 9, '   un e</w>': 5, '   cre ve tt es</w>': 1, '   may on a a i se</w>': 1, '   un c ti ous</w>': 1, '   to k en s</w>': 3, '   er a di ca ting</w>': 2, '   t ro is</w>': 3, '   me s da mes</w>': 1, '   i da</w>': 10, '   de u x</w>': 1, ' or an ge</w>': 2, '   hu it</w>': 1, '   bra i sed</w>': 2, '   c en</w>': 2, '   cen tra lly</w>': 2, '   2 3 0 0</w>': 1, '   0 9 0 0</w>': 1, '   5 7 9 </w>': 1, '   f l y over</w>': 1, '   f li r ta ti ous</w>': 1, '   mi r ac u l ou s ly</w>': 3, '   par o dy</w>': 2, '   que l</w>': 1, '   ther mo st at</w>': 2, '   un n c gh</w>': 1, '   x ra y</w>': 1, '   v ee b er</w>': 1, '   bor ough</w>': 2, '   no mb re</w>': 1, '   pu e d es</w>': 1, '   ca min ar</w>': 1, '   e st as</w>': 1, '   e mb ar a z ad a</w>': 1, '   par a do x i cal</w>': 1, '   co a tes</w>': 3, '   k ani ta</w>': 3, '   me di ta ting</w>': 3, '   me di ta te</w>': 5, ' j un g le</w>': 1, '   par a me di c</w>': 1, '   y a hear</w>': 1, '   i b</w>': 3, '   ban g in</w>': 6, '   s nor ting</w>': 1, '   bu ck y</w>': 6, '   ti e brea k er</w>': 2, '   no el</w>': 9, '   ter min a ting</w>': 1, '   di sc ri min ate</w>': 2, '   fa la fa l</w>': 1, '   le per</w>': 2, '   bu z z er</w>': 7, '   o ver do ses</w>': 1, '   n in o</w>': 2, '   fuck up</w>': 3, '   di sp a tch ers</w>': 1, '   n ar c on</w>': 1, '   c ru p p</w>': 1, '   e le c t ro d es</w>': 1, '   de fri bi la tor</w>': 1, '   b re e ch</w>': 1, '   ma ter ni ty</w>': 3, '   o de tt e</w>': 5, '   f la t l ine</w>': 2, '   re spi ra tion</w>': 3, '   th ro m bo y ti c s</w>': 1, '   ni t ro dri ps</w>': 1, '   he par in</w>': 1, ' car di ac </w>': 2, '   se i z ed</w>': 6, '   h y per ton i c</w>': 1, '   te x t bo o ks</w>': 1, '   pr on oun c ing</w>': 2, ' men ace</w>': 1, '   st ea k head</w>': 1, ' a i ds</w>': 1, '   or o</w>': 3, '   an da le</w>': 1, '   a mp</w>': 3, '   g lu co se</w>': 3, '   sa l ar i ed</w>': 1, '   ca u s a</w>': 1, '   what ja think</w>': 1, '   z e br a</w>': 9, '   f li er</w>': 1, '   ex c el s</w>': 3, '   r ori sh</w>': 1, '   po in t ers</w>': 4, '   shi f ty</w>': 2, '   pro mp ter</w>': 1, '   ca pi to ls</w>': 1, '   con ver sing</w>': 1, '   un see m ly</w>': 2, ' out la w</w>': 1, '   ga d da f i</w>': 4, ' pre si den ti al</w>': 1, '   pro mo tions</w>': 2, '   o bi ts</w>': 2, '   per son i f y</w>': 4, '   in f lu en ces</w>': 4, '   co a x</w>': 4, '   a a ac h</w>': 1, '   fu si ll a de</w>': 1, '   dr y ers</w>': 1, '   in t ro du c tions</w>': 1, ' wri ting</w>': 1, ' wor k ers</w>': 2, '   con cla ve</w>': 1, '   e llo</w>': 1, ' om ca ts</w>': 2, '   1 9 8 1 </w>': 3, '   ni ck na med</w>': 1, '   sti ll er</w>': 1, '   lo b bi es</w>': 1, ' si c</w>': 1, ' > ] </w>': 1, '   ta st e fu lly</w>': 1, '   bu n ting</w>': 2, ' e f fi ci ent</w>': 1, '   an ch or ing</w>': 1, '   bl un t ne ss</w>': 2, '   ph ra sed</w>': 2, '   fuck es</w>': 1, '   fuck e e</w>': 1, '   bu tt o cks</w>': 4, '   s mo o ch es</w>': 1, '   en du r ing</w>': 2, ' h ome coming</w>': 2, '   te ar ful</w>': 3, '   la y off</w>': 1, '   f re der i cks</w>': 1, '   z ir in s k y</w>': 1, '   c s</w>': 1, '   tri po l i</w>': 2, '   wa n</w>': 6, '   ni t pi cking</w>': 1, '   e di ted</w>': 4, '   cu ta way</w>': 2, '   ver bo ten</w>': 1, '   ou tt a kes</w>': 1, ' ac ted</w>': 1, '   fee ing</w>': 1, '   wh oo o</w>': 3, '   gr uni ck</w>': 2, ' in ging</w>': 1, '   un pre ce den ted</w>': 4, '   a ll er g y</w>': 3, ' ca ps</w>': 3, '   w el n</w>': 2, '   th u mp s</w>': 1, '   fir s</w>': 3, '   sti ll well</w>': 1, ' ally</w>': 4, ' con fi d ence</w>': 1, '   f lu ke</w>': 2, ' god s end</w>': 1, ' an ge l</w>': 3, '   god s end</w>': 1, ' li fe sa ver</w>': 1, '   pe u ge o t</w>': 1, '   se d an</w>': 2, '   no d</w>': 3, '   de sc ri p</w>': 1, '   wi t ti est</w>': 1, '   in com par able</w>': 2, '   sp on t an ei ty</w>': 3, '   se ga l</w>': 1, '   di cho tom y</w>': 1, ' da i ry</w>': 1, '   ma x well</w>': 23, '   dri f tw o od</w>': 1, '   su l fu ri c</w>': 1, '   pla gi ar i s m</w>': 1, '   b lo wh ard</w>': 1, ' wal ter</w>': 1, '   bu b b ly</w>': 1, ' a ven u e</w>': 1, '   par a ke et</w>': 3, '   s an t is</w>': 3, '   app la u ding</w>': 1, '   al co ve</w>': 1, '   a sh tra ys</w>': 2, '   in de fin able</w>': 1, '   be l d ere</w>': 2, ' bi d ded</w>': 1, '   may o li a</w>': 5, ' con ne ction</w>': 1, '   j ab ber ja w</w>': 1, '   gu a v a</w>': 1, '   ne c t ar</w>': 3, '   gar ban z o</w>': 1, '   y ea st</w>': 3, '   lo om s</w>': 2, ' ma x well</w>': 1, '   bro ck</w>': 9, ' c at</w>': 6, '   un c rea tive</w>': 1, '   to i ling</w>': 1, '   r t d</w>': 1, '   s wi ck er</w>': 11, '   sy bar i ti c</w>': 1, ' ab </w>': 1, ' h ing</w>': 8, '   bu ff y</w>': 28, '   na po le on i c</w>': 1, '   j im i</w>': 3, '   h en dri x</w>': 3, ' bu ff y</w>': 1, '   s cu b</w>': 1, '   bo o o</w>': 1, '   che er lea der</w>': 6, '   ee y u u</w>': 1, '   si ck ne ss es</w>': 1, '   u u m</w>': 1, '   mm m n k ay</w>': 1, ' qu a li ty</w>': 1, ' ti m ers</w>': 4, '   pu l sing</w>': 3, '   con ver sed</w>': 1, '   f ra i dy</w>': 1, ' an ce</w>': 1, '   w r en ch es</w>': 2, '   o x n ard</w>': 4, '   mm k ay</w>': 1, ' pri son</w>': 3, '   bo gu tu de</w>': 1, '   nor man s</w>': 1, '   sa x ons</w>': 2, '   ki mb er ly</w>': 15, '   si tch</w>': 4, ' br ack et</w>': 1, '   b la se</w>': 1, '   to a st y</w>': 3, '   u sh ers</w>': 1, '   ac n e</w>': 2, ' 9 1 </w>': 2, '   kn ow le d ge fu ll ne ss</w>': 1, '   che er lea ding</w>': 5, ' la y ers</w>': 2, ' t ar z an</w>': 1, '   inter f er es</w>': 4, '   st on e h en ge</w>': 1, '   lo on i es</w>': 3, '   wa tch ers</w>': 2, '   s la y er</w>': 2, '   po o ba h</w>': 1, '   un na tu ra l ne ss</w>': 1, '   con ver sa tion a li st</w>': 1, ' u h h</w>': 1, '   ca li gu l a</w>': 1, '   men ge le</w>': 1, '   p an g bor n</w>': 1, '   t ea ba g</w>': 1, '   p ow er fu lly</w>': 2, '   pr ow ess</w>': 3, '   sh ort cu ts</w>': 1, '   sh e en</w>': 1, '   s con e head</w>': 1, '   g y m na st</w>': 3, '   g y m na sti c s</w>': 4, '   lo th o s</w>': 3, '   gr oun d work</w>': 2, '   go a ds</w>': 1, '   chan ne ling</w>': 2, '   ma g y ar</w>': 1, '   p ra tt ling</w>': 2, '   te st y</w>': 1, '   bu ff ers</w>': 1, '   th x</w>': 12, '   li ven ed</w>': 1, '   a mi l y n</w>': 1, '   re st or a tive</w>': 2, '   co ven</w>': 2, '   gr in ning</w>': 3, '   g ru e ll er</w>': 1, '   be s se l</w>': 1, '   sp ra in</w>': 1, '   u p s w ing</w>': 1, '   un even</w>': 2, '   par a ll el s</w>': 1, '   om ni p le x</w>': 1, ' ta y</w>': 1, '   star ch</w>': 2, '   un wa sh ed</w>': 1, '   pri ori ti z ing</w>': 1, '   wi g g ans</w>': 1, '   mar y an n e</w>': 2, '   he in el</w>': 1, '   cla use</w>': 2, '   h ome le ss es</w>': 1, '   lo z en ge</w>': 1, '   bu n gu s</w>': 1, '   e b by</w>': 9, '   la loo sh</w>': 7, '   qui c ki e</w>': 5, '   un s na p</w>': 1, '   fun dam en tal s</w>': 4, ' chan ne ling</w>': 3, '   p in g al a</w>': 4, '   pi tch ers</w>': 4, '   ori g in a tes</w>': 1, '   te sti c le</w>': 2, '   ter min a tes</w>': 1, ' p in g al a</w>': 1, ' i da</w>': 1, '   s nu g ly</w>': 1, ' cra sh</w>': 5, '   om i ga w d</w>': 1, '   pi tch es</w>': 6, ' lu red</w>': 1, ' mi lli e</w>': 1, ' lin e ar</w>': 2, '   sp ac i ous</w>': 4, '   sa le m</w>': 4, '   din ger</w>': 2, '   a le x an dri a</w>': 7, '   c z ar e tt e</w>': 2, '   ba ll pla y ers</w>': 1, '   u mp</w>': 1, ' to k</w>': 1, '   to k</w>': 2, '   du r ha m</w>': 2, ' un comp li ca ted</w>': 1, '   p y n ch on</w>': 3, '   ten th s</w>': 1, '   wa ll ow ing</w>': 3, '   me ta ph y si c s</w>': 1, '   ju da o</w>': 1, '   mon o ga m ous</w>': 2, '   2 4 7 </w>': 1, '   h om ers</w>': 1, '   2 2 7 </w>': 1, '   un comp li ca ted</w>': 2, ' di m</w>': 1, ' wi s do m</w>': 1, ' wan ta</w>': 1, '   out la w ing</w>': 1, ' tur f</w>': 1, '   vo ting</w>': 6, '   sa vo y</w>': 1, '   du g out</w>': 3, '   g al in do</w>': 1, ' 3 1 4 </w>': 1, '   l y n ch bur g</w>': 2, '   di sc or ru pt</w>': 1, '   en gir th</w>': 2, ' st ars</w>': 3, ' chan ne l ed</w>': 1, ' ro e bu ck</w>': 1, '   co ck su cking</w>': 1, ' me at</w>': 3, ' ba se b all</w>': 1, ' 2 5 0</w>': 8, '   g or k</w>': 1, '   ca tch er</w>': 8, '   ba ll c lu b</w>': 2, '   fa st b all</w>': 6, '   cu r ve b all</w>': 3, '   s k at ers</w>': 1, '   ra in out</w>': 1, '   qu ad ra ph on i c</w>': 2, '   b la u p un k t</w>': 2, '   9 4 4 </w>': 1, '   w oo ly</w>': 2, '   te ed</w>': 1, '   ho ck e t t</w>': 1, '   sh re ve por t</w>': 1, '   go o d year</w>': 3, '   ro ok</w>': 1, '   cha k ra s</w>': 1, '   o ver thr ow ing</w>': 1, '   pa t kin</w>': 1, '   lo ll y ga g</w>': 3, '   in fi e ld</w>': 1, '   lo ll y ga g g ers</w>': 1, '   k en mor es</w>': 1, '   9 8 </w>': 3, '   ri g g ins</w>': 1, '   o h yeah</w>': 1, '   an n oun c er</w>': 3, '   sp ort s wri ter</w>': 2, '   ni g ger town</w>': 1, '   t an gi ers</w>': 7, ' a llow</w>': 1, '   gre en st e in</w>': 1, '   comp ed</w>': 2, '   au spi ci ous</w>': 1, '   an tw er p</w>': 1, '   1 8 6 2 </w>': 2, '   7 0 2 </w>': 1, '   4 7 2 </w>': 1, ' ro th st e in</w>': 1, ' c lu b</w>': 2, '   th a</w>': 5, ' g in ger</w>': 2, '   s k ell</w>': 1, ' s k ell</w>': 1, '   p ee k ab o o</w>': 1, '   ga g g i</w>': 2, '   bar r y more</w>': 1, '   e tch ed</w>': 1, ' de tri men tal</w>': 1, ' fe m me</w>': 1, '   fa ta le</w>': 1, ' sta kes</w>': 1, ' ni ck y</w>': 1, ' e ddy</w>': 1, '   bo o ki es</w>': 4, '   bo ok ma king</w>': 1, '   p in ch es</w>': 2, '   har e li ps</w>': 1, '   hi r in</w>': 2, '   je op ar di z es</w>': 2, '   ga m in</w>': 1, '   e ye ba ll in</w>': 3, '   s an t or o</w>': 1, '   fr ac as</w>': 1, ' mi chi g an</w>': 1, ' m i</w>': 3, '   ri vi er a</w>': 9, '   und ers</w>': 1, ' po si tion</w>': 1, ' cha i m</w>': 1, ' y i d di sh</w>': 1, " ' ] </w>": 5, '   bu dge</w>': 8, '   pen thou ses</w>': 1, ' be ep</w>': 5, '   sp ra in ed</w>': 5, '   sha ke down</w>': 7, '   ch in chi ll a</w>': 1, '   s ca m st ers</w>': 1, ' p it</w>': 3, ' st un ned</w>': 2, '   com mi ssi on ers</w>': 1, '   har an gu e</w>': 1, ' ace</w>': 3, '   pa i ge</w>': 1, '   no vo d or</w>': 1, '   li cen sing</w>': 1, '   be d la m</w>': 1, '   f la m bo y ant</w>': 2, '   fir in</w>': 4, ' re el</w>': 1, '   re el s</w>': 5, '   j ack po ts</w>': 3, '   se ven s</w>': 2, '   dan g</w>': 4, '   pro gre ssi ves</w>': 1, ' ac tu al</w>': 1, '   mu sc l ed</w>': 1, '   p l</w>': 2, '   sc ha ff</w>': 1, ' i ta li an</w>': 9, ' a p</w>': 1, '   cu mm a</w>': 3, '   fa t so</w>': 2, '   fa ir ne ss</w>': 4, '   fo il</w>': 2, ' man u re</w>': 1, '   in st in c tive</w>': 2, ' me tr o</w>': 1, '   co ac h es</w>': 3, '   d om in i ck</w>': 2, '   ti m ers</w>': 6, '   r ac com an do</w>': 1, ' a g gi a</w>': 1, '   vi en</w>': 1, '   ac c a</w>': 1, '   ca z z o</w>': 4, '   in vo l</w>': 1, '   any b</w>': 1, '   ye ssed</w>': 1, ' same</w>': 8, ' to le</w>': 1, '   y a k kin</w>': 2, ' be ll</w>': 2, ' u t</w>': 1, '   ni ger i a</w>': 2, '   clo gs</w>': 2, '   j i g gs</w>': 1, ' bo ss es</w>': 1, '   no bi l is</w>': 1, '   tri pp a</w>': 1, ' ri pe</w>': 2, '   su f fri t t</w>': 1, ' re m o</w>': 1, '   mer ch</w>': 2, '   ri pp in</w>': 3, '   s mar ten</w>': 2, '   vi se</w>': 1, '   un con fu sed</w>': 1, '   secon ded</w>': 1, ' fri ck in</w>': 1, '   n an ce</w>': 1, '   fri ck in</w>': 2, '   ad on is</w>': 9, '   o v ah</w>': 1, '   na xt</w>': 1, '   te h n</w>': 1, '   a y y</w>': 1, '   e p i</w>': 2, ' cen ter</w>': 2, '   la w su i ts</w>': 3, '   cer ea ls</w>': 1, '   ver w y</w>': 2, '   ca tw om en</w>': 3, ' ar med</w>': 4, '   o a si s bur g</w>': 8, '   ca tw om an</w>': 16, ' ter r ori st</w>': 3, '   con de sc en sion</w>': 2, '   app ea sing</w>': 2, ' war e house</w>': 3, ' se in fe ld</w>': 2, ' du a li ty</w>': 2, ' o oh</w>': 7, '   l ev i a than</w>': 4, ' v u</w>': 1, '   di g ni fi ed</w>': 5, '   a g gre ssi ve ly</w>': 3, '   bl en der</w>': 11, '   co l d ne ss</w>': 1, ' se lin a</w>': 2, '   u r in al s</w>': 1, ' c li mb ing</w>': 2, ' s k y li ght</w>': 1, '   fu r b all</w>': 1, '   sc ra p bo ok</w>': 3, '   de fu se</w>': 6, '   gar fi e ld</w>': 3, '   ki ck b ac ks</w>': 2, '   ch ould</w>': 1, '   mi gra in es</w>': 2, '   sh op kee per</w>': 2, '   dan i sh</w>': 1, ' sh oo ting</w>': 1, ' d ra g</w>': 2, ' su per h er o</w>': 1, '   t re e house</w>': 1, '   tr an qui li ty</w>': 1, '   dis co the qu e</w>': 1, '   fri da</w>': 40, '   k a h l o</w>': 1, '   z il ch o</w>': 1, '   vi gi lan tes</w>': 1, ' mi ssi ons</w>': 1, '   o a si bur gi ans</w>': 1, '   h er o i ca lly</w>': 2, ' ta il ed</w>': 1, '   ta b by</w>': 1, '   vo i ce bo x</w>': 1, '   thin gi e</w>': 3, ' c lu es</w>': 1, '   che st pla te</w>': 1, ' s is</w>': 2, ' con su el a</w>': 1, ' gir ls</w>': 3, ' stu dent</w>': 2, ' bi cen ten ni al</w>': 3, ' ma le</w>': 4, '   7 3 </w>': 4, ' po si tions</w>': 1, ' p re</w>': 5, '   a m ne si ac </w>': 1, '   ban da i ds</w>': 3, '   fini te</w>': 2, '   in ten tion ally</w>': 8, ' bro ck</w>': 2, '   ru g ge d ly</w>': 1, ' a m ne si a</w>': 2, ' de sc ri b ing</w>': 1, '   bu ll e th o les</w>': 2, ' fran k</w>': 7, '   da y d rea ms</w>': 3, ' ha g</w>': 1, ' le tting</w>': 1, '   for e going</w>': 1, '   he i sts</w>': 1, ' fa king</w>': 2, '   c ri me fi gh t ers</w>': 1, '   se i z es</w>': 5, '   k ru ger an ds</w>': 2, '   au th en ti ci ty</w>': 1, ' tra sh</w>': 3, '   s la pp ing</w>': 7, ' wi fe y</w>': 1, '   ch r y s l er</w>': 2, '   cu ff ed</w>': 5, '   cra pped</w>': 3, '   ne ls</w>': 16, '   br en tw o od</w>': 4, '   mi s de me an or</w>': 10, '   s la mm ed</w>': 8, '   wi s ea ss</w>': 2, '   ca i t l in</w>': 48, ' ho ck ne y</w>': 1, ' re p</w>': 1, '   un pro v ok ed</w>': 1, '   d ra in age</w>': 2, '   3 0 5 </w>': 1, ' 4 4 1 0</w>': 1, '   v al u ab les</w>': 3, '   o ber fe ld</w>': 5, '   sa li v a ting</w>': 2, '   c el ine</w>': 2, '   an ge li c</w>': 1, '   in f li c ting</w>': 1, '   d ev e lo per</w>': 2, '   2 5 9 </w>': 2, ' 7 8 8 1 </w>': 2, '   nu dge</w>': 3, '   re tri bu tion</w>': 1, '   c ro a ti an</w>': 1, '   to p an g a</w>': 2, '   po th o les</w>': 1, ' mar k</w>': 3, '   ti med</w>': 3, ' sto le</w>': 3, '   j i g g le</w>': 1, '   tu f ts</w>': 1, '   v in cen te</w>': 1, '   p on y ta il</w>': 3, ' c li ck</w>': 1, '   re char ger</w>': 1, '   re se mb les</w>': 2, ' the o</w>': 1, '   si ding</w>': 3, '   o ver pa ss</w>': 2, '   el li o t</w>': 56, '   shu tt ers</w>': 7, '   secon al</w>': 2, '   de du c ting</w>': 1, ' no ble</w>': 2, '   3 0 8 </w>': 2, ' 9 9 6 2 </w>': 2, '   te me s cal</w>': 1, '   un de cl ar ed</w>': 2, '   ac r y li c s</w>': 1, '   ge tty</w>': 4, ' pa tr on</w>': 1, '   ci sc o</w>': 2, ' 2 9 </w>': 2, '   vi g ori sh</w>': 1, '   st ro lled</w>': 1, '   ro dri e go</w>': 1, '   wh ee l man</w>': 1, '   ha ir p in</w>': 1, '   re se ar ch ed</w>': 1, ' qui c ki e</w>': 1, ' ne ls</w>': 1, '   f ru i t stand</w>': 1, '   di l</w>': 45, '   e s se x</w>': 1, '   sc ra g</w>': 1, '   d ever ou x</w>': 2, '   fran k nu m</w>': 2, '   f er gu s</w>': 31, '   supp le ment</w>': 1, '   mu l ti vi ta min s</w>': 1, '   ta b le ts</w>': 2, '   en de ar men ts</w>': 1, '   jo dy</w>': 35, '   be l fa st</w>': 1, '   car na tions</w>': 3, '   re pe ti ti ous</w>': 1, '   f er gi e</w>': 4, '   o ch</w>': 1, '   s ar ac en</w>': 1, '   ma gu i re</w>': 6, '   ga ff</w>': 14, '   ju die</w>': 1, '   h en ne ss y</w>': 3, '   hur ling</w>': 4, '   ha l f pen ny</w>': 1, '   spi tal fi el ds</w>': 1, '   p in ts</w>': 4, '   u l ster</w>': 1, '   to ff s</w>': 3, '   to tt en ha m</w>': 3, '   un de lu ded</w>': 1, '   p in up</w>': 1, '   ho o d win ked</w>': 2, '   ba tt ed</w>': 4, '   gre en h or n</w>': 2, ' wi tt ed</w>': 5, ' s k u lled</w>': 2, '   d y le</w>': 33, '   gr ou sing</w>': 2, '   j ar d in</w>': 2, ' l ys</w>': 1, '   g ran d pi er re</w>': 3, ' f la v or ed</w>': 1, '   st ea m ship</w>': 1, '   li li an</w>': 1, '   n ar r ow ed</w>': 3, '   can fi e ld</w>': 5, '   la m per t</w>': 55, '   pa la is</w>': 1, '   co lon na de</w>': 1, ' y le</w>': 1, '   s co bi e</w>': 12, '   gra t in</w>': 1, '   ch ou c r ou te</w>': 1, '   gar ni e</w>': 1, '   sa la de</w>': 1, '   p om mes</w>': 1, '   ba ll on</w>': 1, '   ha ll es</w>': 1, '   vo ss</w>': 6, ' a ll o ca ted</w>': 1, ' li ver w u r st</w>': 1, '   li ver w u r st</w>': 2, ' cle an in g wi se</w>': 1, '   p on th i e u</w>': 1, '   out f l ow</w>': 2, '   pen th o llow</w>': 1, '   cor du ro y</w>': 2, '   c ru i k sh an k</w>': 3, '   c ro ck</w>': 7, '   lo a th es</w>': 1, ' se ll er</w>': 4, '   se w n</w>': 3, '   vi en s</w>': 2, '   ob je c ted</w>': 2, '   sy l vi e</w>': 7, '   whi te f oo t</w>': 5, '   bl ack f oo t</w>': 2, '   mo c ca sin s</w>': 1, '   whi te fe et</w>': 1, '   bl ack fe et</w>': 2, '   tra de s man</w>': 1, '   com me mor a tive</w>': 2, ' li x</w>': 3, '   gu y an n e</w>': 1, '   1 8 5 2 </w>': 1, '   ma u ri ti us</w>': 1, '   1 8 5 6 </w>': 1, '   1 8 9 4 </w>': 1, '   gu l a</w>': 1, '   f y ra s ki ll in g en</w>': 1, '   1 8 5 4 </w>': 1, '   n ac hur ly</w>': 2, ' ran s</w>': 1, '   ti ck l in</w>': 1, '   me ge ve</w>': 1, '   un s ea l ed</w>': 2, '   el ys</w>': 1, '   ab sur de</w>': 2, ' ment</w>': 1, ' ar an gu a pe</w>': 1, '   mar ac a i b o</w>': 1, '   b our dea u x</w>': 1, '   e d ou ard</w>': 1, '   ju di ci a i re</w>': 1, '   vo il</w>': 2, ' d ry</w>': 3, '   bu ll ying</w>': 4, '   m ac du ff</w>': 1, '   gar de</w>': 1, '   a v an t i</w>': 4, '   f lo or show</w>': 1, '   un e sc o</w>': 2, '   si mu l t an e ous</w>': 3, '   tr ans la tor</w>': 4, '   er el ong</w>': 1, '   di sh on e st y</w>': 2, '   in fu ri a tes</w>': 2, '   s n ow ba lls</w>': 1, '   si re e bo b</w>': 1, '   under st a</w>': 3, '   si ree</w>': 3, ' ri e</w>': 1, '   tra d es</w>': 4, '   v ac he</w>': 1, '   al ors</w>': 4, '   j ou er</w>': 1, '   an ge</w>': 1, ' ne i gh bor ly</w>': 1, '   in te ll e c tually</w>': 6, '   bea st ing</w>': 1, '   s n ow bo ard</w>': 3, '   mar list on</w>': 3, '   ber r in ger</w>': 1, '   br ent</w>': 4, ' a un t</w>': 1, '   fuck fe st</w>': 1, '   mi cha el s</w>': 16, '   re e king</w>': 2, ' vi r g in</w>': 2, '   c ri ti ci z ing</w>': 4, '   si s l er</w>': 4, '   er u p tion</w>': 1, '   sh er m er</w>': 1, '   at ti c s</w>': 2, '   ba se men ts</w>': 3, '   cer e be ll u m</w>': 3, '   w oo d sto ck</w>': 4, '   do of y</w>': 1, '   pri st ine</w>': 1, '   fa ta li sti ca lly</w>': 1, '   h y men</w>': 1, '   sha le</w>': 2, '   a o l</w>': 1, ' i ck</w>': 1, '   che w s</w>': 2, '   mo l ars</w>': 2, '   ba u er</w>': 1, '   a mo e bi c</w>': 1, '   ea ve s dro p</w>': 1, '   d un lo p</w>': 2, '   li b ary</w>': 1, '   mi c ro fi c he</w>': 1, '   h y po c ri tes</w>': 3, ' k en ny</w>': 2, '   bo tt l ed</w>': 5, ' que ers</w>': 2, '   for ma lly</w>': 3, '   tw in ki es</w>': 6, '   z u c ch in i</w>': 1, '   br y n ner</w>': 20, '   ex t</w>': 29, ' an dy</w>': 3, ' sta ir well</w>': 1, '   un be</w>': 2, ' li ev able</w>': 2, ' ca b</w>': 4, '   i stan bu l</w>': 5, '   ar l o</w>': 13, ' ti at in</w>': 1, '   ne go</w>': 1, '   re sti tu tion</w>': 2, '   tru ck less</w>': 1, '   jo b less</w>': 1, ' tru ck</w>': 5, '   li mp in</w>': 1, '   pu b li ci ze</w>': 3, ' t un ne l</w>': 4, ' na ti c</w>': 1, ' s qu ir m</w>': 1, ' mi les</w>': 3, ' sp l it</w>': 3, '   mi ss ou l a</w>': 9, '   j er o me</w>': 21, '   na tion ally</w>': 2, '   ran ked</w>': 4, ' o ver d ra w n</w>': 1, ' wh ee l ed</w>': 1, ' tion</w>': 2, '   g l ac i er</w>': 1, '   li k el i</w>': 1, '   pe te y</w>': 2, '   wee d ki ll er</w>': 1, '   be e p ing</w>': 1, ' ch o</w>': 1, ' re con si der</w>': 1, '   v an lo a d</w>': 1, '   m c g ru der</w>': 7, '   m c g ru d</w>': 3, '   mu d ho le</w>': 3, ' t an a</w>': 1, ' mor on</w>': 1, ' dar l en e</w>': 1, ' ty</w>': 2, '   comp un ction</w>': 2, '   per i sha ble</w>': 1, '   te di ous</w>': 3, ' with in</w>': 4, ' ar l o</w>': 1, ' on ds</w>': 1, '   sa ge</w>': 4, ' le m</w>': 2, ' ma son</w>': 2, '   imp le men ts</w>': 1, ' went</w>': 5, ' p le</w>': 1, '   pe o</w>': 2, ' de st ro y er</w>': 1, ' si tt in</w>': 1, '   re sp on s i</w>': 1, ' l ar</w>': 1, ' tor</w>': 1, ' de st ry</w>': 1, '   gu st ing</w>': 1, ' s ea ttle</w>': 1, ' en wor th</w>': 1, '   s qu ea mi sh ne ss</w>': 1, '   lea v </w>': 2, ' i ans</w>': 1, '   d ev i a tions</w>': 1, '   du ti ful</w>': 1, '   bi kes</w>': 5, ' com es</w>': 2, ' f re e z er</w>': 1, ' ther</w>': 2, ' m and</w>': 1, '   bra g g</w>': 5, '   pa pp as</w>': 8, ' ge ar</w>': 1, '   c r ab gra ss</w>': 1, ' s ci en ti sts</w>': 1, '   ch r i</w>': 2, ' sc re w up</w>': 1, '   he m min gs</w>': 1, ' spi te</w>': 1, ' ta in er</w>': 1, '   re lin qui sh</w>': 1, ' de er</w>': 1, '   ge t up</w>': 2, ' l en e</w>': 1, '   d ar</w>': 2, '   bo sc o</w>': 1, ' some how</w>': 3, '   sh ee p s k ins</w>': 1, '   k a bu l a</w>': 1, '   ta i pe i</w>': 1, '   gr in d in</w>': 2, ' cor ri d or</w>': 1, '   out man e u ver ed</w>': 2, '   un lo a ding</w>': 5, ' j er k</w>': 4, ' co lon el</w>': 3, '   pu mp er</w>': 1, '   t y in</w>': 1, ' me ans</w>': 1, '   de g re</w>': 1, ' fi sh in</w>': 1, ' for c ing</w>': 1, '   o ver dri ve</w>': 1, ' sen si ti vi ty</w>': 1, '   pre di ction</w>': 2, '   fo l</w>': 1, '   s ea l in</w>': 1, ' han ds</w>': 2, ' pu b li c</w>': 2, '   v a g ran cy</w>': 1, ' ser i ou s ly</w>': 1, ' ba dge</w>': 5, ' wh y n</w>': 2, '   pa sted</w>': 1, '   pl at</w>': 2, '   gi tt es</w>': 54, '   a que du ct</w>': 1, '   s l ou gh s</w>': 1, ' po o ls</w>': 2, '   per co late</w>': 1, '   be dro ck</w>': 2, '   ev a por a ting</w>': 1, '   re ser vo ir s</w>': 4, '   ho lli e</w>': 1, '   mu l w ra y</w>': 60, '   chi pp i e</w>': 2, '   m ac an do</w>': 2, '   en sen ad a</w>': 3, '   au g gi e</w>': 46, '   i ll u stra te</w>': 2, '   s ki p j ack</w>': 2, '   ch u ba sc o</w>': 1, '   al b ac ore</w>': 4, '   un wri tt en</w>': 3, '   di m w it</w>': 2, '   ex t or t</w>': 3, '   lo ac h</w>': 1, '   qu ar re l ed</w>': 3, '   ru no ff</w>': 4, '   bi fo ca ls</w>': 1, '   1 4 1 2 </w>': 1, '   a de la i de</w>': 1, '   m ar</w>': 3, '   c r ab b</w>': 6, '   spe er</w>': 3, '   der</w>': 24, '   c rea m co l or ed</w>': 1, ' che at</w>': 1, '   ma tri mon i al</w>': 3, '   me ti ay</w>': 1, '   sh ort chan ged</w>': 3, '   me h r ho l t z</w>': 2, '   wal sh</w>': 5, '   1 7 1 2 </w>': 1, '   a la me da</w>': 1, '   b y r d</w>': 4, '   ri ch fi e ld</w>': 1, ' ho se</w>': 1, '   y el bur ton</w>': 3, '   mu l vi hi ll</w>': 5, '   ru ss</w>': 3, '   ri ver bed</w>': 2, '   t ea sp o on</w>': 1, '   mor ty</w>': 6, '   ho ll en be ck</w>': 1, '   ha br a</w>': 1, '   star ki st</w>': 1, '   can n er y</w>': 1, '   pro je ction</w>': 5, '   par a di so</w>': 5, '   to to</w>': 38, '   bl in der</w>': 1, '   ba st a</w>': 1, '   bu d ged</w>': 2, '   ga un t</w>': 2, '   st rea med</w>': 1, '   he ee y</w>': 1, '   to to oo o o</w>': 1, ' so on er</w>': 1, '   pe pp in oo o o</w>': 1, '   so o on</w>': 1, '   a de l fi o</w>': 2, '   c ran king</w>': 3, '   pro je c t ors</w>': 1, '   sig nor a</w>': 4, ' wh oo sh</w>': 1, ' mi r ac le</w>': 1, '   lo a ves</w>': 1, '   fi sh es</w>': 3, '   bo c ci a</w>': 1, '   bu r</w>': 4, '   sa l v at or es</w>': 2, '   bo l ting</w>': 1, '   ti t an us</w>': 1, '   ca ten e</w>': 1, '   ci c ci o</w>': 1, '   ven u ses</w>': 3, ' en qu ir er</w>': 13, '   re i lly</w>': 26, ' ch r on i c le</w>': 9, ' figu re</w>': 1, '   re vi e w er</w>': 1, '   th a is</w>': 1, '   sen ori ta s</w>': 1, '   su n ri ses</w>': 1, '   lan d s cap es</w>': 2, '   th a tch er</w>': 20, '   mer v in</w>': 1, '   1 8 9 6 </w>': 1, '   pa st ro l</w>': 1, ' mar m</w>': 2, '   gu ar an t ee ing</w>': 1, '   har as sed</w>': 3, '   si l ver st one</w>': 8, '   re ctor</w>': 3, ' co lu m n</w>': 1, '   cha m pe en</w>': 2, '   re ci ev ed</w>': 1, '   gra f ter</w>': 2, '   1 8 5 </w>': 1, '   bri b ing</w>': 2, '   br ac e let</w>': 26, '   cra ted</w>': 3, '   bu ll do g</w>': 5, '   to o th ac he</w>': 7, '   se li g man</w>': 1, '   poli ti can</w>': 1, '   ca b in ent</w>': 1, '   lea ses</w>': 2, '   go ver ne ment</w>': 1, '   in den ti cal</w>': 1, '   ma ti st i</w>': 3, '   la u gh in g sto ck</w>': 1, '   p hi lan th ro pi c</w>': 2, '   \t \t </w>': 1, '   ho l din gs</w>': 4, '   tr ac tion</w>': 3, ' en e my</w>': 2, '   ar ma da</w>': 3, '   por ch es</w>': 1, '   so o th ed</w>': 1, '   nu r sed</w>': 4, '   do s en</w>': 1, '   ne w bur g</w>': 1, ' sa ti s f ac tion</w>': 1, '   im per son ally</w>': 2, '   ca m pa i g ned</w>': 1, '   s ine</w>': 1, '   mar t y red</w>': 1, '   lea sed</w>': 1, '   with o l ding</w>': 1, ' ex i st ent</w>': 1, '   d ev o ting</w>': 2, '   reme di ed</w>': 3, ' hi ck vi ll e</w>': 1, '   pri z es</w>': 6, '   si x y</w>': 1, '   en ter ta in in g ly</w>': 1, '   mon ar ch y</w>': 4, '   whi te ha ll</w>': 6, '   di sta te ful</w>': 1, '   ra w l st on</w>': 5, '   g ro b le s k i</w>': 1, '   in f li c ted</w>': 2, '   wa s n t</w>': 1, '   un w ra p</w>': 1, '   p ho to gra ph ing</w>': 4, '   hi gh b all</w>': 2, '   che w li es</w>': 1, '   tr ac h</w>': 1, '   che w lie</w>': 1, '   mon g ers</w>': 1, '   ran da l</w>': 10, ' t or n</w>': 2, '   h er ma ph ro di te</w>': 1, '   b ree</w>': 23, ' s cour ge</w>': 1, '   r en ter</w>': 1, '   vi o late</w>': 3, '   di sen ga ge ment</w>': 1, '   pl ac ate</w>': 1, '   m om en t ous</w>': 1, '   o st r ac i z ed</w>': 1, '   su c cu mb ing</w>': 1, '   i d y lli c</w>': 1, ' ver on i c a</w>': 1, '   ca u sti c</w>': 1, '   im pe tu s</w>': 1, ' a de</w>': 1, '   der r is</w>': 2, '   pe p ti c</w>': 1, '   k o al a</w>': 1, '   ga t or ad es</w>': 1, '   ga t or a de</w>': 4, '   kn ead</w>': 1, '   o ver comp en sa te</w>': 1, '   en c ore</w>': 5, '   an ally</w>': 1, '   h er ma ph ro di tes</w>': 1, '   o st en si b ly</w>': 1, '   f ds</w>': 1, '   dis ru p ts</w>': 1, '   si mp li f y</w>': 3, '   mi c ro co s m</w>': 1, '   ca ta ton i c</w>': 3, ' po o</w>': 3, ' ba m</w>': 4, '   ne w bor n s</w>': 3, '   h er ma ph ro di ti c</w>': 1, '   star le ts</w>': 3, '   ra tion a le</w>': 1, '   gra tu i t ous</w>': 3, '   r st</w>': 1, '   re spe c ti ve ly</w>': 1, '   h un h h</w>': 4, '   pa u l s en</w>': 2, '   y m c a</w>': 1, ' b ack st ro ke</w>': 1, '   b la t ant</w>': 2, '   l ac es</w>': 1, '   bu ck l ed</w>': 3, '   le i t mo ti f</w>': 2, '   bur t less</w>': 1, '   mi l k ma i ds</w>': 2, '   gr ou p in gs</w>': 1, ' mo pp er</w>': 3, '   st rea ks</w>': 7, ' bo o th</w>': 3, '   un ru ly</w>': 5, '   ro of er</w>': 1, '   su bur bi a</w>': 3, '   mi li t an ts</w>': 1, '   im per i al s</w>': 2, '   si d ers</w>': 1, '   ro of ers</w>': 1, '   di g ni t ar i es</w>': 1, '   lan do</w>': 20, '   ca l ri ssi an</w>': 5, '   je d i</w>': 33, '   le i a</w>': 22, '   the o c r ac y</w>': 2, '   ba tt ling</w>': 3, '   al be it</w>': 3, '   bo b a</w>': 1, '   fe t t</w>': 1, '   mu pp e ts</w>': 2, ' l en g th</w>': 1, ' al ge br a</w>': 1, '   lu ri d</w>': 3, '   ro man ti ci ze</w>': 1, '   po st sc ri pt</w>': 1, '   mi cha el son</w>': 3, ' sen i or</w>': 1, '   b li t z ed</w>': 1, '   pe li can</w>': 3, '   gla d es</w>': 15, ' g or ge ous</w>': 1, ' we i gh ts</w>': 1, '   under u ti li z ed</w>': 1, '   in ha la tion</w>': 1, '   sig ni fi es</w>': 2, '   la sa g n e</w>': 5, ' ha l f s</w>': 1, '   no in ch</w>': 3, '   n y n n e</w>': 1, '   jo ck e ying</w>': 1, ' w y n ar s k i</w>': 1, '   ma gs</w>': 1, '   he mor r ho i ds</w>': 2, '   co tt on y</w>': 1, '   in cont in ent</w>': 1, '   f lu or e sc ent</w>': 1, '   c ru sts</w>': 1, '   sy l v an</w>': 4, '   sch o la sti c</w>': 1, '   en ro ll ment</w>': 3, '   par a p le gi c</w>': 1, '   fran son</w>': 1, '   st ans l y k</w>': 1, '   gen er a li z ation</w>': 2, ' bro a ds</w>': 1, '   tri vi a li ze</w>': 2, '   ther e in</w>': 2, '   pro fi ci en cy</w>': 1, '   ac co sted</w>': 2, '   in di g ni ti es</w>': 1, '   con te sted</w>': 1, '   re m it</w>': 1, '   com pla in ed</w>': 6, '   n j ac </w>': 1, ' ber ser k er</w>': 3, '   bi o gra p her</w>': 2, '   ra in st or m</w>': 2, '   bu l ki er</w>': 1, '   sh ort com in gs</w>': 1, '   ne h</w>': 1, '   wi ll am</w>': 2, '   ch ee se head</w>': 1, ' brea th</w>': 2, ' f le d ged</w>': 2, '   gu tru sh</w>': 1, '   cu r s in</w>': 1, '   un sp ort s man like</w>': 1, '   je ssi e</w>': 15, '   sig na lling</w>': 2, '   bi t k er</w>': 2, '   qu al en</w>': 4, '   tra ver s</w>': 11, '   re su s ci ta tion</w>': 4, ' ho l ds</w>': 2, '   1 3 5 </w>': 2, '   l b s</w>': 4, '   1 9 0</w>': 4, '   cre vi ce</w>': 2, ' ac tually</w>': 6, '   han d hold</w>': 1, '   p ea ks</w>': 2, '   h y per ven ti la tion</w>': 1, '   k ri st el</w>': 1, '   hi j ac king</w>': 1, '   ke y co de</w>': 1, '   s ke et</w>': 2, '   der a il ed</w>': 2, '   t ar m ac </w>': 1, '   ma the son</w>': 3, '   h or r or show</w>': 3, '   fi lli ed</w>': 1, '   ma l en k y</w>': 4, '   dro o gi es</w>': 3, '   wal ki es</w>': 1, '   ha ha ha ha h a</w>': 1, '   vi ddy</w>': 3, '   dro o g</w>': 2, '   min oo ta</w>': 1, '   dro o gi e</w>': 1, '   s ma sh es</w>': 2, '   h or sy</w>': 1, '   ga pe</w>': 1, '   gu lli ver</w>': 6, '   dis ci pl in ing</w>': 1, '   we lly</w>': 6, '   do o bi do o b</w>': 1, '   ma l chi cks</w>': 1, '   be d ways</w>': 1, '   ri g th ways</w>': 1, '   h ome ways</w>': 1, '   sp a tch k a</w>': 1, '   no z h</w>': 2, '   w ea k en s</w>': 1, '   y ar b les</w>': 1, '   bo l sh y</w>': 1, '   y ar b lo ck o s</w>': 2, '   bri t v a</w>': 1, '   to l cho cks</w>': 1, '   rea son less</w>': 1, '   do ok</w>': 1, '   pu b li c wi se</w>': 1, '   so p hi sto s</w>': 1, '   go v or ee ting</w>': 1, '   d ev o tch k a</w>': 1, '   s me cking</w>': 1, '   t wan ged</w>': 1, '   mi l k b ar</w>': 1, '   p lo t t</w>': 1, '   en d wi se</w>': 1, '   a the</w>': 1, ' tru mp</w>': 1, '   pr on ging</w>': 1, '   cl ow ny</w>': 1, '   gu f fa w</w>': 1, ' f l u</w>': 2, '   cont or ted</w>': 1, '   f re e z es</w>': 4, '   pro vi den ti al</w>': 1, '   fr en z i ed</w>': 1, '   p ha ll us</w>': 1, '   so o ma k a</w>': 1, '   fi b re gla ss</w>': 1, '   go ver e et</w>': 1, '   r ou g es</w>': 1, '   ma s ki es</w>': 1, '   ru b in st e in</w>': 1, '   ha ha h a</w>': 2, '   ee e e</w>': 1, '   to l cho ck</w>': 1, '   un mu d di ed</w>': 2, '   a z u re</w>': 2, '   de l to id</w>': 5, '   re tch es</w>': 1, '   pro bo sc is</w>': 2, '   ro ok ers</w>': 2, '   mi li cen ts</w>': 2, '   na st in ess</w>': 4, '   bi ll y boy</w>': 1, '   a mb lu en c ed</w>': 1, '   der i si ve ly</w>': 1, '   di st r ac te d ly</w>': 1, '   cl ink</w>': 2, ' cu b es</w>': 1, '   mi lli cen ts</w>': 1, '   st ri p y</w>': 2, '   cor re c tive</w>': 4, ' pla y fu lly</w>': 1, '   cha i</w>': 2, '   af ter l un ch</w>': 1, '   p ti t s a</w>': 1, '   che ll o ve ck</w>': 1, '   di d st</w>': 3, '   con fr on ta tion</w>': 3, '   no ch y</w>': 1, '   mo lo k o</w>': 3, '   man si ze</w>': 1, '   cra st</w>': 2, '   cra st ing</w>': 1, '   ro ok er ful</w>': 2, '   app y</w>': 1, '   lo g gi es</w>': 1, '   k ni f y</w>': 1, '   ab o de</w>': 2, '   ca mer a men</w>': 1, '   l ou d sp ea k ers</w>': 2, ' f i</w>': 3, '   t ro ll ey</w>': 1, '   in st ru men tal</w>': 3, '   un fa v our able</w>': 1, '   im pe lled</w>': 2, '   par a do x i ca lly</w>': 1, '   di a me tri ca lly</w>': 1, '   f la sh ed</w>': 7, '   re tch ing</w>': 5, '   so b s</w>': 3, '   wee p y</w>': 4, '   lo d ger</w>': 1, '   m un ch y</w>': 1, '   w un ch ing</w>': 1, '   lo m ti cks</w>': 1, '   n ar ra tor</w>': 2, '   y a h z i k</w>': 1, '   gra h z ny</w>': 1, '   v on ny</w>': 1, '   w oo sh ed</w>': 1, '   6 5 5 3 2 1 </w>': 4, '   par k mo or</w>': 2, '   lu do vi c o</w>': 5, '   sta j a</w>': 1, '   bri s k ly</w>': 1, '   mo th ba lls</w>': 3, '   wri st let</w>': 1, ' ti ma wri st</w>': 1, '   to ss es</w>': 3, '   bro d s k y</w>': 5, '   under n ou ri sh ed</w>': 1, '   p y ja ma</w>': 2, '   re lu c t an tly</w>': 3, '   h y p o</w>': 2, '   in ser ts</w>': 2, '   b ran om</w>': 1, '   e g gi we gs</w>': 1, '   e lot</w>': 1, '   ow w w</w>': 4, '   bo ll o cks</w>': 9, '   qu ar re lled</w>': 1, '   k ni ck ers</w>': 3, '   pl u ma ge</w>': 1, '   p ea co ck</w>': 1, '   to l cho cked</w>': 1, ' vi o l ence</w>': 4, '   hel p le ss ne ss</w>': 1, '   sti f ling</w>': 2, '   ca ta st ro p hi c</w>': 4, '   dar en</w>': 2, '   mu r der ous</w>': 3, '   tra mp s</w>': 6, '   s war m</w>': 4, '   th u mp ed</w>': 1, '   me th s</w>': 1, '   dr in k ers</w>': 2, '   ja me y</w>': 2, '   ho k ey</w>': 5, '   to d d les</w>': 1, '   cu tter</w>': 5, ' h h h</w>': 1, '   dr un ki e</w>': 1, '   bl er p</w>': 2, '   wh ys</w>': 2, '   wh er e for es</w>': 1, '   war d ers</w>': 2, '   wor k sh op s</w>': 1, '   in sti tu tion al</w>': 2, '   in fr ac tions</w>': 1, '   di min i sh ed</w>': 3, '   in spe c ts</w>': 2, '   pen li ght</w>': 1, '   wai st b and</w>': 1, '   p in st ri p ed</w>': 1, '   p in st ri pe</w>': 2, '   com mu ted</w>': 1, '   ga th ers</w>': 1, '   w be</w>': 1, '   me th o di ca lly</w>': 1, '   bl ack j ac ks</w>': 2, ' dr ow ned</w>': 3, '   me s to</w>': 1, '   ma l chi ck</w>': 1, '   be wi l der ing</w>': 1, '   sle e p ers</w>': 1, '   ho o d lu m</w>': 4, '   g go ver n ment</w>': 1, '   out mo ded</w>': 1, '   pen o lo gi cal</w>': 1, '   imp le men ta tion</w>': 2, '   c ri min a li ty</w>': 1, ' d mi tr i</w>': 1, '   com or o s</w>': 4, ' bo tt e g a</w>': 1, '   dar row</w>': 6, '   bo tt e g a</w>': 2, '   ho t l ine</w>': 1, '   ca b bi es</w>': 2, '   chi v al ry</w>': 1, '   bo g ged</w>': 1, '   lin en s</w>': 2, '   co l d wa ter</w>': 1, ' po ta to</w>': 1, ' sle e p y</w>': 1, ' cor n et</w>': 1, '   su ey</w>': 2, '   win ton</w>': 8, '   mar sa l is</w>': 1, '   bu ss in</w>': 1, ' ju ly</w>': 2, '   un sa ti s fi ed</w>': 5, '   bu ll do z er</w>': 2, ' win ded</w>': 1, '   bl in t z</w>': 1, '   ni cho la s</w>': 21, '   hel per</w>': 6, '   y i p</w>': 1, '   ca b dri ver</w>': 3, '   ri l ke</w>': 4, ' pe t ro v </w>': 1, '   ci cer no</w>': 2, '   mu l do on</w>': 2, '   n y p d</w>': 1, '   l ow li ves</w>': 1, '   wi se guys</w>': 1, '   han d pr in ts</w>': 1, ' gu su no v </w>': 1, '   di mi tr i</w>': 3, '   wi lt</w>': 12, ' st e er ing</w>': 2, '   wh as su p</w>': 1, '   ma a a a a a x</w>': 1, '   ma a a x</w>': 1, '   fa st en ed</w>': 3, ' some day</w>': 1, '   an e s the ti ze</w>': 2, '   bar ca l oun ger</w>': 1, ' be l ted</w>': 1, '   clo cking</w>': 1, ' pi ss</w>': 2, ' ant</w>': 1, '   4 0 9 </w>': 1, ' in differ ent</w>': 2, ' wi t ne ss es</w>': 1, ' beli ev ed</w>': 2, '   tri be c a</w>': 1, '   wa ter fr on t</w>': 11, ' as sa ss in</w>': 1, '   li mo s</w>': 3, '   wi sh ful</w>': 4, '   cap ti v a ted</w>': 1, '   pro v ok es</w>': 1, ' ta u ght</w>': 1, '   bi r th place</w>': 2, '   be bo p</w>': 1, '   the lon i ous</w>': 1, '   st ro b es</w>': 1, '   r wan d ans</w>': 1, '   an ge l en o</w>': 1, '   hi ss y</w>': 1, '   r wan da</w>': 3, ' bur und i</w>': 1, '   ch ing</w>': 8, '   sto d g y</w>': 1, '   5 8 </w>': 3, '   t wi tch ed</w>': 1, '   u no</w>': 2, '   do s</w>': 2, '   sp h in c t ers</w>': 1, '   do ers</w>': 2, '   da y shi ft</w>': 1, '   z ea land</w>': 4, '   ro ad work</w>': 2, ' ger on im o</w>': 2, '   p oun ded</w>': 3, '   ar i</w>': 1, '   on as is</w>': 2, ' j er ry</w>': 2, '   wi sh b one</w>': 2, '   me k ong</w>': 3, ' on ne l</w>': 1, '   ph en o bar b</w>': 1, '   li z a</w>': 10, '   e z e ki el</w>': 3, '   j on as</w>': 10, '   h in k le y</w>': 10, '   g un man</w>': 4, ' wi l kes</w>': 2, ' da vi d</w>': 1, ' har v ey</w>': 1, ' ba se ment</w>': 5, '   bu l ow</w>': 1, '   war r ing</w>': 1, '   f ac tions</w>': 1, '   su b sc ri b ers</w>': 2, '   ca u l fi e ld</w>': 3, ' r ye</w>': 1, '   tri la ter al</w>': 2, '   o c to gen ar i ans</w>': 1, '   un pro v able</w>': 1, '   f l un k y</w>': 2, ' con sp ir ac y</w>': 1, '   e qui ta tion</w>': 1, '   ca bo o dle</w>': 2, '   che ch en i a</w>': 1, '   re se ar ch ers</w>': 2, '   ad di tive</w>': 1, ' u r ning</w>': 1, '   w ra par ound</w>': 1, '   ar f</w>': 1, '   comp li c it</w>': 1, '   r ac e h or se</w>': 2, ' na s a</w>': 1, '   pa s se</w>': 1, '   l y ser gi c</w>': 1, '   di e th y la mi de</w>': 1, '   h er e ti c</w>': 2, ' hi gh s</w>': 1, ' su l ting</w>': 1, '   ha ll u c in o gen s</w>': 2, '   ve ge ta tive</w>': 1, '   1 9 7 3 </w>': 2, '   r en a med</w>': 1, ' man ch u ri an</w>': 1, ' can di da te</w>': 1, ' par an o id</w>': 1, ' e du ca ting</w>': 1, '   ver i ta s</w>': 3, '   st or y bo ok</w>': 1, '   s la ted</w>': 1, ' under sto od</w>': 2, '   o e u v re</w>': 1, '   ex p on en ti ate</w>': 1, '   ju ri s di c tion al</w>': 2, '   b en z el</w>': 2, '   ar r ow ay</w>': 16, '   or p han ed</w>': 4, '   in du ces</w>': 1, '   re uni ting</w>': 1, '   pri mes</w>': 2, '   ve g a</w>': 5, '   ki t z</w>': 3, '   st ri dent</w>': 1, '   can di d ac y</w>': 1, '   l un ac har s k y</w>': 2, '   e t is</w>': 1, ' ro o t</w>': 1, '   the o lo gi cal</w>': 3, '   ra mi fi ca tions</w>': 1, '   jo ss</w>': 1, '   coun ter clo ck</w>': 1, '   inter po la tion</w>': 1, '   po l ar i z ation</w>': 2, '   ne sted</w>': 1, '   de c r y p tion</w>': 4, '   al g ori th ms</w>': 3, '   ri g ors</w>': 1, '   con stan ts</w>': 1, '   tr an sc en dan tal s</w>': 1, '   je tting</w>': 1, '   re mo te st</w>': 1, '   pro f oun d ly</w>': 4, '   comp oun ds</w>': 3, '   d ru m l in</w>': 7, '   spe cu la tive</w>': 2, '   do c t or al</w>': 1, '   t an ta m oun t</w>': 1, '   g al ac ti c a</w>': 1, '   c y cl ed</w>': 1, '   par ch ment</w>': 2, '   pa li mp se st</w>': 1, '   s no op s</w>': 1, '   ar e ci b o</w>': 2, '   v b</w>': 1, '   sig na</w>': 1, '   dr ac on is</w>': 1, ' 1 8 6 </w>': 1, ' pa g es</w>': 3, ' 4 1 3 </w>': 1, '   star fi e ld</w>': 1, '   a st r on om ers</w>': 3, '   cen ta u ru s</w>': 1, '   1 9 1 9 </w>': 3, '   star qu a ke</w>': 1, '   cu ll ers</w>': 1, '   j ar ro d</w>': 1, ' si l ent</w>': 2, '   cla ssi f y</w>': 3, '   ca u sa li ty</w>': 1, '   de sc en dan ts</w>': 3, '   in te lli gen ces</w>': 1, '   un sp ea k ably</w>': 3, '   li br ar i ans</w>': 3, '   cu ra t ors</w>': 1, '   car e ta k ers</w>': 1, '   sh ort wa ve</w>': 1, '   ha d d en</w>': 5, '   in du st ri es</w>': 8, '   ho k k a i do</w>': 2, '   con sor ti u m</w>': 2, ' ow ned</w>': 2, '   su b si di ar i es</w>': 2, '   pi x i e</w>': 1, '   stu r m</w>': 1, '   lo b b ying</w>': 1, '   en or m ou s ly</w>': 6, '   ci r cu i try</w>': 2, '   ve g an</w>': 1, '   lo ca ting</w>': 4, ' pri m er</w>': 1, '   un fa shi on able</w>': 2, '   out li ving</w>': 1, '   y el t s in</w>': 1, '   da sch a</w>': 1, '   na i ve te</w>': 2, ' o om s day</w>': 1, ' inter ven tion i st</w>': 1, '   f la g ran tly</w>': 1, '   inter ven ing</w>': 2, '   v al er i an</w>': 3, '   ow en s</w>': 8, '   ra me y</w>': 1, '   gre en ban k</w>': 1, ' na tion al s</w>': 1, '   spe c tra l</w>': 1, '   shi t work</w>': 1, '   in di st in gu i sha ble</w>': 1, ' co per ni c ans</w>': 1, '   re vo l ved</w>': 1, '   vi c t ori ans</w>': 1, '   in sig ni fi can ce</w>': 2, '   pre con ce p tions</w>': 1, '   fe ti sh es</w>': 1, '   e t i</w>': 1, '   y a vo l</w>': 1, '   an ti so ci al</w>': 1, '   de so la tion</w>': 1, '   v a st ne ss</w>': 1, '   to l er able</w>': 3, '   be ar able</w>': 2, '   fr on d</w>': 1, '   in de li ca te</w>': 1, '   t ac t ful</w>': 2, '   an it</w>': 1, '   ob s cu red</w>': 3, ' wri tt en</w>': 3, '   a spi red</w>': 2, '   out cla ss es</w>': 1, '   ex qui si tely</w>': 3, '   se le c te es</w>': 1, '   min g ling</w>': 2, '   se le c t ors</w>': 1, '   f lin ch ed</w>': 1, '   pen du lu m</w>': 3, '   ther mo d y na mi c s</w>': 1, '   de sc ri b es</w>': 4, '   s ke p ti c s</w>': 1, '   mu tually</w>': 2, '   di lu te</w>': 2, '   x en op ho bi c</w>': 2, '   ab sur di ty</w>': 1, '   en case</w>': 1, '   son ar gra ms</w>': 1, '   b ack sa ss</w>': 2, '   ar le tta</w>': 6, '   mar r y in</w>': 3, '   bor in</w>': 1, '   ab o ve bo ard</w>': 1, '   bu n t in</w>': 1, '   a bu il d in</w>': 1, '   wh ar</w>': 1, '   wi sh t</w>': 4, '   pu ps</w>': 4, '   g ran d ki ds</w>': 4, '   supp o s in</w>': 5, '   b ack s li de</w>': 1, '   gi tch a</w>': 1, '   gi ts</w>': 3, '   ba st id</w>': 1, '   af ter a</w>': 1, '   cl in kin</w>': 1, '   br on ze</w>': 5, '   sto ck ad es</w>': 1, '   ma li ci ou s ly</w>': 1, '   de st ro y in</w>': 1, '   bor r ow in</w>': 2, '   nu thing</w>': 1, '   co o ts</w>': 1, '   f la p</w>': 5, '   ne w m ea ts</w>': 2, '   mu ll et</w>': 4, '   fri s kin</w>': 1, '   d ra g l ine</w>': 3, '   lu ci ll e</w>': 7, ' g an ging</w>': 2, '   u se ta</w>': 1, '   re d hea ds</w>': 1, '   wi ck er</w>': 1, '   fi st fu ll</w>': 1, '   ne w me at</w>': 3, '   a im in</w>': 4, '   har d case</w>': 1, '   te ar in</w>': 1, ' sha kin</w>': 1, '   who e e</w>': 1, '   gr ins</w>': 3, ' ga tor</w>': 1, ' at s a</w>': 1, '   af fe c tion ate</w>': 2, '   bar ra ge</w>': 2, '   dea d lo ck</w>': 1, '   i ff</w>': 2, '   han d fu ll</w>': 1, '   re d ho ts</w>': 2, '   sa ssed</w>': 1, '   com pla in er</w>': 1, '   gu mb all</w>': 1, '   g na w</w>': 2, ' gu t</w>': 1, '   cho c</w>': 1, ' l at</w>': 1, '   li k able</w>': 3, '   ex a g ger ation</w>': 4, ' e ars</w>': 3, '   mu ther</w>': 3, '   pa ir a</w>': 3, '   k ok on u t</w>': 1, '   n in as</w>': 4, '   an a</w>': 2, '   bab al u ga ts</w>': 1, '   ha l f a</w>': 1, '   fee d in</w>': 1, '   mi l k men</w>': 1, '   h ome op a th i c</w>': 1, '   a g or a p ho bi a</w>': 1, '   un read</w>': 2, ' s lu t</w>': 1, ' ir ty</w>': 3, ' s m ell</w>': 1, '   di sa pp ro ves</w>': 2, ' forget</w>': 9, ' ro ts</w>': 1, '   ta ll u la h</w>': 1, '   ga a h h d</w>': 1, ' li mi ting</w>': 1, '   h y per ven ti la tes</w>': 1, '   ha ll or an</w>': 15, '   go e t z</w>': 7, '   au ra l</w>': 3, '   dar y ll</w>': 12, ' l ying</w>': 1, '   c ri ter i a</w>': 4, ' f l un ked</w>': 1, '   sa di st</w>': 4, '   k u r ten</w>': 9, '   cu ll u m</w>': 6, '   fe lon i ous</w>': 1, '   k lu t z</w>': 5, ' 5 5 0</w>': 1, ' sy mp hon y</w>': 1, '   ba g gi e</w>': 3, '   cho cked</w>': 1, '   car na ge</w>': 4, ' ne ss</w>': 2, '   e pi ph any</w>': 1, ' so le m n</w>': 1, '   possi bi l</w>': 3, '   i ty</w>': 2, '   qu in n</w>': 6, ' ha ll or an</w>': 2, '   da h m er</w>': 5, '   de sa l v o</w>': 4, '   bi an ch i</w>': 2, '   bu on o</w>': 2, '   ber k ow i t z</w>': 4, '   ru b en</w>': 16, ' h er o es</w>': 2, '   co p y c at</w>': 6, '   st ran g</w>': 1, '   l er</w>': 2, '   win de x</w>': 5, ' sa ks</w>': 1, '   sc hi ff er</w>': 1, '   in no v at ors</w>': 1, '   ho b go bl in</w>': 1, '   h y per ven ti late</w>': 3, '   ro bo ti c</w>': 1, ' f ans</w>': 1, '   th ri lls</w>': 7, '   de st ru ct</w>': 2, '   al b er</w>': 1, ' k not</w>': 1, '   pi ca s so s</w>': 2, '   wa ding</w>': 2, '   a g or a p ho bi c</w>': 1, '   mar y j an e</w>': 3, '   mo on bi kes</w>': 1, '   pre sc ri b es</w>': 2, '   mer y he w</w>': 1, '   per for m ers</w>': 2, '   possi b</w>': 2, '   i li ty</w>': 1, '   gra p ho lo g y</w>': 1, '   cl on ing</w>': 2, '   se cre tor</w>': 2, '   un bea ta ble</w>': 2, '   ni k k o</w>': 10, '   ou tr an ked</w>': 1, ' ev i d ence</w>': 2, '   g ran d stan ding</w>': 1, '   a sp h y x i a ted</w>': 1, '   ni c co le tt i</w>': 2, '   ru be</w>': 2, '   ex ter m in</w>': 1, '   at ors</w>': 1, '   ma il men</w>': 1, '   c ri ti qu e</w>': 2, ' sp ok es</w>': 1, '   ho ok u ps</w>': 1, '   pr ow l er</w>': 2, '   i lli g al s</w>': 1, ' dis ci pl ine</w>': 1, '   ex pen d able</w>': 8, ' s mu g g ling</w>': 3, ' so l ver</w>': 1, '   du i</w>': 3, '   ma a a a an</w>': 1, '   f on d ling</w>': 1, '   k nu ck les</w>': 4, '   de f er ence</w>': 2, '   co in ci d es</w>': 2, '   f l ow er ing</w>': 2, '   cla pped</w>': 1, '   b li ss ing</w>': 1, '   tu li ps</w>': 1, '   se que st er ing</w>': 2, ' con fir m</w>': 1, '   lea sing</w>': 1, '   t wi tch</w>': 2, '   f ing</w>': 1, '   w u h ve</w>': 1, '   co st ner</w>': 1, '   su i t ors</w>': 2, '   v au g ha n</w>': 31, '   pe de st ri ans</w>': 2, '   pe de st ri an</w>': 1, '   ci r cu m ci sed</w>': 2, '   so d om i ze</w>': 1, '   sa l ti er</w>': 1, '   tu cked</w>': 10, '   en vi ous</w>': 8, '   po tt ed</w>': 4, '   f le cks</w>': 1, '   sp att er ed</w>': 1, '   in war ds</w>': 1, '   sp ee d ome ter</w>': 7, '   ha ll ways</w>': 1, '   w en de l</w>': 2, ' char ter</w>': 1, '   a er on au ti cal</w>': 1, '   gr oun d loo p</w>': 1, '   a sh for d</w>': 1, '   ra di o lo gi st</w>': 1, '   com pu ter i z ed</w>': 6, ' ra mp</w>': 1, '   ma ter i a li sti c</w>': 3, '   ki mon o</w>': 1, '   sc ra pped</w>': 1, ' te st</w>': 5, '   re en ac t men ts</w>': 1, '   ja y n e</w>': 8, '   man s fi e ld</w>': 1, '   ca m us</w>': 2, ' wa sh</w>': 1, '   k ne el s</w>': 3, '   re si li ence</w>': 2, '   p sy ch op a th o lo g y</w>': 2, '   re sha p ing</w>': 2, '   f er ti li z ing</w>': 1, '   me di a tes</w>': 1, '   f ac el</w>': 1, '   na than i el</w>': 3, '   ro ver</w>': 3, '   3 5 0 0</w>': 1, '   le tty</w>': 27, '   wor sen ed</w>': 1, '   sc hi z op h r en i c s</w>': 2, '   ha ll st ro m</w>': 3, '   man i c</w>': 7, '   de pre ssi ve</w>': 6, '   lo ck y er</w>': 1, '   g ran de u r</w>': 3, '   e m le e</w>': 4, '   do sa ge</w>': 1, '   l he</w>': 1, '   ru th i e</w>': 5, '   sc ro ll</w>': 5, '   ro man e s qu e</w>': 2, '   e c ru </w>': 2, '   pu g li a</w>': 1, '   br ine</w>': 2, '   o ver p ower</w>': 2, '   h er b es</w>': 1, '   dre ck</w>': 2, ' cho s en</w>': 1, '   mo i re</w>': 1, '   da ma s k</w>': 1, '   of f ing</w>': 2, '           </w>': 23, '   mo tt l ed</w>': 1, '   me di ca tions</w>': 2, '   cor t</w>': 5, ' go s</w>': 1, '   sch mi st</w>': 1, '   bor in g ly</w>': 1, '   l oun ging</w>': 1, '   cre vi ces</w>': 1, '   re ci di vi s m</w>': 1, '   d om in o s</w>': 1, '   ab r ac ad ab r a</w>': 1, '   f l or e ts</w>': 1, ' gr oun d ho g</w>': 1, ' fa il</w>': 1, ' t un e</w>': 2, '   mi s gu i de d ly</w>': 1, '   car a ts</w>': 2, '   e mer al ds</w>': 2, '   f ra ter ni ze</w>': 2, '   con f er en ces</w>': 4, '   sa b ba ti cal</w>': 1, '   z ac h</w>': 9, '   ga il</w>': 10, '   sa pp hi r es</w>': 1, '   z ir con i u m</w>': 1, '   sa m i</w>': 8, '   de li ver s</w>': 3, '   z a m mi to</w>': 14, '   ba st al d i</w>': 20, '   ma ti s se</w>': 2, ' v a se</w>': 3, '   su n f l ow ers</w>': 4, '   ga v one</w>': 1, '   la u ran t</w>': 4, '   e u ro s</w>': 4, '   su per nu mer ary</w>': 4, '   bab b o</w>': 1, '   b on an no</w>': 3, '   g ro in</w>': 3, '   for gi ving</w>': 2, ' su n f l ow ers</w>': 1, '   ri d ge way</w>': 4, '   ri d ger o a d</w>': 2, '   a ir por ts</w>': 6, ' ki ss er</w>': 1, '   ju li en</w>': 5, '   bar cla y</w>': 2, '   bro k er age</w>': 3, '               </w>': 4, '   v in ny</w>': 1, '   supp le men ts</w>': 1, '   1 4 5 </w>': 2, ' po ck e ting</w>': 1, '   au to gra ph ed</w>': 3, '   pe pe</w>': 1, '   g l en mor an gi e</w>': 4, ' a mp</w>': 5, '   dr in k er</w>': 3, '   beau jo la is</w>': 1, '   ca s an dr a</w>': 1, '   m u</w>': 37, '   ba i</w>': 25, '   ja de</w>': 23, '   sha an</w>': 1, '   x i</w>': 1, '   g en</w>': 3, '   in fi l tra ted</w>': 3, '   y us</w>': 3, '   y u</w>': 17, '   w u d an</w>': 8, '   sh ar p ne ss</w>': 1, ' x u an</w>': 1, '   pi u</w>': 1, '   gi an g</w>': 5, '   h u</w>': 7, '   sur pa ss</w>': 1, '   di a gra ms</w>': 1, '   pe king</w>': 12, '   si st er ly</w>': 3, '   men g</w>': 3, '   z ha o</w>': 1, '   for nu ate</w>': 1, '   g ous</w>': 1, '   ca lli gra ph y</w>': 2, ' in fe sted</w>': 1, '   s ca b b ard</w>': 1, '   li gh te st</w>': 1, '   k ne el</w>': 6, '   dar es</w>': 4, '   ca ll used</w>': 2, '   in tri gu e</w>': 4, '   comp un d</w>': 1, '   ju st ly</w>': 1, '   me di ta tion</w>': 5, '   z h en g</w>': 1, ' me an ing</w>': 3, '   cu sto di an</w>': 1, '   al b re ch t</w>': 6, '   fun boy</w>': 7, ' ow n s</w>': 1, ' er a se</w>': 1, '   dar l a</w>': 1, '   e u w w</w>': 1, '   reme mb ran ce</w>': 1, ' t in</w>': 2, ' t re at</w>': 2, '   h om i e</w>': 2, '   bu sh wa ck</w>': 1, '   ga ps</w>': 8, '   dea th g ri p</w>': 1, '   p al p able</w>': 1, ' s co p ed</w>': 1, ' si gh ted</w>': 1, '   ow w wa a a a</w>': 1, '   a a a a a a</w>': 1, '   god d d</w>': 1, '   what sh er name</w>': 1, '   b loo d sta in</w>': 1, ' ne ck l ace</w>': 1, '   bi g m ou th</w>': 2, '   a de qu a tely</w>': 1, '   re ci pi en ts</w>': 1, ' ro ll er</w>': 1, ' c row</w>': 1, ' man e u ver ing</w>': 1, '   k in k o s</w>': 1, '   men a g es</w>': 1, '   bu li mi a</w>': 2, '   k a thr y n</w>': 31, '   an ac h r on i s m</w>': 4, '   o a k w o od</w>': 5, '   v al mon t</w>': 2, '   p hi li pe</w>': 4, '   f l or en t in o</w>': 2, '   b ack ga mm on</w>': 3, '   ma u g ha m</w>': 2, '   un in ten tion ally</w>': 1, '   ba d m ou thing</w>': 1, '   di sh on or able</w>': 3, '   ger i a tri c</w>': 2, '   ro ar k</w>': 1, '   d om in i qu e</w>': 1, '   fran c on</w>': 1, '   cou id</w>': 1, '   le s b o</w>': 1, '   sch war z</w>': 3, '   ro se mon d</w>': 2, '   pu n</w>': 2, '   sp ar t ac us</w>': 1, '   st e in way</w>': 2, '   ju li ard</w>': 1, ' e du ca tion al</w>': 1, '   pr ac ti c ed</w>': 4, '   ha mp t ons</w>': 5, '   bu li mi c</w>': 2, '   hea d case</w>': 3, ' hea s</w>': 1, '   sa d de st</w>': 5, '   f la ir</w>': 3, '   ta un ted</w>': 1, '   mer ci le ss ly</w>': 1, '   chi r p ing</w>': 1, '   ci r qu e</w>': 1, '   so le il</w>': 1, '   ac ro ba ti c s</w>': 1, '   re c ti fi ed</w>': 1, '   wi ttle</w>': 1, '   y i pp y</w>': 2, ' la a a a dy</w>': 1, ' k a thr y n</w>': 3, '   p sy cho an al y s is</w>': 1, '   bo b sle d der</w>': 1, '   th i ck en s</w>': 2, '   en er gi es</w>': 3, '   cor ru p ting</w>': 2, '   po o ti e</w>': 1, '   de ca d ence</w>': 1, '   de ba u ch er y</w>': 1, '   b al i</w>': 3, '   z i pp ing</w>': 2, '   co l in</w>': 3, '   ma am</w>': 1, '   pe do p hi les</w>': 1, '   cra ves</w>': 4, ' f oo li sh</w>': 3, '   mu mm ers</w>': 3, '   cle ar est</w>': 1, ' bur ning</w>': 2, '   f ar r en</w>': 11, '   go ya</w>': 1, '   can on</w>': 1, '   ca ll a ha n</w>': 3, '   ir en a</w>': 13, '   o lli e</w>': 5, '   bo y ds</w>': 2, '   ir v in gs</w>': 2, ' 3 0 0 0</w>': 1, '   o i lie</w>': 1, '   bro o ding</w>': 3, ' ir en a</w>': 1, '   sh one</w>': 2, '   mo ss es</w>': 1, '   s wi f t ne ss</w>': 1, '   hu g g ins</w>': 1, '   lea pt</w>': 2, '   qui ck ne ss</w>': 1, '   be tt or</w>': 1, '   tw in k ling</w>': 1, '   h er n e</w>': 1, '   h un t s man</w>': 1, '   hea d less</w>': 5, ' ci tr on</w>': 1, '                                                               </w>': 3, '   h er so lf</w>': 1, '   t ar r y town</w>': 3, '   a fe ard</w>': 1, '   so b b ing</w>': 2, ' a my</w>': 1, '   a bu n d ant</w>': 2, '   p m sing</w>': 2, '   nor ma l cy</w>': 1, '   y u l</w>': 1, '   ba g ger</w>': 3, ' co y o te</w>': 3, '   con ven tion ally</w>': 1, '   w o l f ing</w>': 1, '   par k a</w>': 4, '   pro p on ent</w>': 1, '   cra ze</w>': 1, '   star ch es</w>': 1, '   b en i to</w>': 4, '   s con ce</w>': 3, '   cor re sp on ded</w>': 1, '   cra v in gs</w>': 1, '   che w b ac c a</w>': 5, '   bra s</w>': 2, '   bor es</w>': 1, '   ta mp ons</w>': 2, '   w us</w>': 1, '   bo x ers</w>': 5, '   dr y er</w>': 1, '   ve ge t ar i an</w>': 10, '   sa la ds</w>': 2, '   sc re en play</w>': 4, '   pa mp er</w>': 1, '   g ro ss es</w>': 2, '   mor ten son</w>': 1, '   wh att da ya</w>': 1, '   supp or ter</w>': 4, '   won der br a</w>': 1, '   t ou pe e</w>': 3, '   ni el son</w>': 2, '   a j ar</w>': 1, '   par a ph er na li a</w>': 2, '   k ha k is</w>': 1, '   be st se ll er</w>': 3, '   de ca f</w>': 3, '   pi ll ow case</w>': 4, '   se x u a li z ing</w>': 1, '   ber a ting</w>': 1, '   sig ni f y</w>': 4, '   f ra s er</w>': 2, '   sy n c</w>': 5, ' ba g ger</w>': 2, '   po se u r</w>': 3, '   g ro ss er</w>': 1, '   pi t bu ll</w>': 3, '   sen i li ty</w>': 1, '   er e ctor</w>': 4, '   m n un n</w>': 1, '   sch re b er</w>': 1, '   bu m st ead</w>': 5, '   ger ms</w>': 10, ' in spe ctor</w>': 3, '   re qui si tion</w>': 3, '   c r en sha w</w>': 4, '   s ea m less</w>': 1, ' mu r der er</w>': 1, '   di se mb ow e lled</w>': 1, ' sh e et</w>': 1, '   b li p</w>': 3, '   man n er i s ms</w>': 1, '   e m be lli sh</w>': 1, '   w re st les</w>': 1, '   hund re d th</w>': 4, ' car ing</w>': 1, ' pen ding</w>': 1, '   y u t z</w>': 2, '   com men ce ment</w>': 2, '   pr es</w>': 4, ' da ve</w>': 3, '   ve to ed</w>': 2, '   ve to es</w>': 1, ' a mer i c a</w>': 6, '   sho al</w>': 1, '   gu es</w>': 1, '   g q od</w>': 1, '   bo l ster</w>': 2, '   au tom o tive</w>': 2, '   per ce p tu al</w>': 3, ' bo o st</w>': 1, '   con su m er</w>': 6, '   re gi st r ar</w>': 1, '   coun ci l man</w>': 1, '   bur und i</w>': 1, ' b le ed</w>': 2, '   k o vi c</w>': 1, ' ta te</w>': 1, '   he si ta tes</w>': 2, '   ja un t</w>': 2, ' ca m pa i g n</w>': 1, '   tur n around</w>': 1, '   m cl n ti re</w>': 1, '   in au gu ra tion</w>': 2, ' hea d qu ar t ers</w>': 2, ' re be ls</w>': 1, ' no oo oo o o</w>': 1, '   sh oo oo oo oo o t</w>': 1, '   m ee ee ee ee e</w>': 1, ' wan ting</w>': 1, ' nee ding</w>': 1, ' de st ro y</w>': 3, ' el ev a tor</w>': 1, '   h mm mm m m</w>': 1, '   re c ru i t in</w>': 1, '   vi ct</w>': 1, '   re l ying</w>': 2, '   a w re ddy</w>': 1, '   da id</w>': 1, '   fo gi es</w>': 1, '   ga sp ar i ll a</w>': 4, '   ar ou n</w>': 7, '   t y ere</w>': 1, '   p ran c in</w>': 1, ' sp o ke</w>': 3, '   t y l er</w>': 56, '   con si der in</w>': 4, '   y er self</w>': 1, '   r ho d es</w>': 11, '   h en ried</w>': 5, '   pro c ee d in</w>': 1, '   da tur a</w>': 5, '   me te l</w>': 2, '   re li gi o so</w>': 1, '   bu mp in</w>': 1, '   re li gi o so s</w>': 2, '   in le ts</w>': 1, '   b ack wa t ers</w>': 2, ' tra f fi c</w>': 2, '   lea s in</w>': 1, '   di ck er son</w>': 4, ' ga sp ar i ll a</w>': 1, '   c r on i es</w>': 2, '   ta ll a ha s see</w>': 3, '   ca l y p so</w>': 1, '   un lo ad in</w>': 1, ' sp ac ing</w>': 1, '   ca te chi s m</w>': 1, '   sa vi ou rs</w>': 3, ' de ser ving</w>': 1, '   no oo oo oo o o</w>': 1, ' p uni sh</w>': 1, ' ton y</w>': 2, '   a a a a a a ah</w>': 1, ' of f en ded</w>': 1, '   di o s</w>': 1, ' au th ori ti es</w>': 2, '   un sa ti s f ying</w>': 1, '   be ha vi ou ra li st</w>': 1, ' fi l th</w>': 1, ' h un g ry</w>': 2, '   w ean</w>': 1, '   at ro ci ti es</w>': 2, '   wa st el and</w>': 2, '   gu er i ll a</w>': 1, '   in f la ta ble</w>': 1, ' fo ll ow ing</w>': 3, '   t ow ni es</w>': 1, '   h en le y</w>': 5, '   de mer i ts</w>': 1, '   dan bur ry</w>': 3, '   m ee ks</w>': 20, '   ra i s er</w>': 1, '   o ver st re et</w>': 5, '   y a a</w>': 1, '   f la tt ers</w>': 2, ' ton</w>': 2, '   nu wan da</w>': 11, '   k ea ting</w>': 7, ' fi x ed</w>': 2, '   te m pe sts</w>': 1, '   im pe di men ts</w>': 1, '   al t ers</w>': 4, '   re mo ver</w>': 3, '   ex ci tes</w>': 3, '   w el ton</w>': 2, '   sh ins</w>': 2, '   pro of ing</w>': 1, '   pro of ers</w>': 1, '   son or ous</w>': 1, '   re f re sh er</w>': 1, ' no sing</w>': 1, '   in fu ri a ting</w>': 1, '   brea stu m</w>': 1, '   g in ny</w>': 2, '   dan bur r ys</w>': 1, '   o ver st re ss</w>': 1, '   mi d su mm er</w>': 2, '   ra he sh</w>': 2, '   en li gh ten s</w>': 1, '   ho i</w>': 2, '   po ll o i</w>': 3, ' ho i</w>': 1, '   ro se bu ds</w>': 2, ' car pe</w>': 1, '   re pri m and</w>': 2, '   mu mb les</w>': 1, '   a h h h h h h</w>': 4, '   y a w p</w>': 4, '   jo ck o</w>': 6, '   app ro ving</w>': 1, '   tr you ts</w>': 1, '   di sp u ting</w>': 1, '   di sp u ted</w>': 1, '   h er mi a</w>': 1, '   pla in er</w>': 1, '   b al in cre st</w>': 2, '   o c to pu ss es</w>': 1, '   cor ked</w>': 1, '   mo or ed</w>': 1, '   ma mo o l i</w>': 2, '   vi v o</w>': 3, '   sp ee d bo at</w>': 3, '   o ver looking</w>': 2, '   in de sc re tion</w>': 1, '   pla y gr ou p</w>': 1, ' mi d dle</w>': 3, '   f ok</w>': 2, '   su ma tr a</w>': 4, '   mi ss le</w>': 1, '   go int</w>': 1, ' po pu la tion</w>': 2, ' ri d es</w>': 1, ' ri ch</w>': 6, '   hu lls</w>': 1, ' al thou gh</w>': 4, '   li ven s</w>': 1, '   ma mber</w>': 1, '   pa tter</w>': 2, '   sti ch</w>': 1, '   u g li er</w>': 2, '   den t in ation</w>': 1, '   mo jo s</w>': 1, ' ja g ger</w>': 1, '   al t mon t</w>': 1, '   mo kes</w>': 1, '   s q au d</w>': 2, '   po o l side</w>': 1, '   coun ti r es</w>': 1, '   ger al d ine</w>': 4, '   tw in ki e</w>': 5, '   s lan ts</w>': 2, ' u u</w>': 1, '   a mu s ant</w>': 1, '   na tu re ll e ment</w>': 2, '   g l ace</w>': 1, '   vi ll e</w>': 2, ' ver</w>': 1, '   bi e der man</w>': 7, '   to p most</w>': 1, '   th u mp</w>': 1, '   ar m and</w>': 3, '   cre me</w>': 3, '   ma is</w>': 3, '   cer ta in ment</w>': 1, '   tch</w>': 16, '   qu an</w>': 2, '   hu mp er</w>': 2, '   p han to m</w>': 20, '   can a st a</w>': 2, '   b ow ls</w>': 2, '   s qui re</w>': 12, ' ch om p</w>': 1, '   ch om ps</w>': 1, '   d ou c et</w>': 17, '   man ny</w>': 8, '   te ch no th u gs</w>': 3, '   du b b s</w>': 3, '   a v on</w>': 1, '   fri ck</w>': 2, '   fr ack</w>': 2, '   n p a</w>': 3, '   gre ssion</w>': 1, '   e di c on</w>': 2, '   de fro sted</w>': 2, ' ex act</w>': 1, ' ani ma ted</w>': 2, ' ani ma ting</w>': 2, '   j ack off</w>': 1, '   ju r ors</w>': 1, '   fr ac tions</w>': 3, ' part ner</w>': 6, ' com wa tch</w>': 1, '   0 8 0 0</w>': 2, '   e mp h y se ma</w>': 2, '   te ch no p ho bi a</w>': 1, '   w el by</w>': 1, '   ver m is</w>': 1, '   ab i ding</w>': 1, '   p v c</w>': 1, '   c ms</w>': 1, '   g c i</w>': 3, '   t m</w>': 3, '   me ta mor p ho s is</w>': 1, '   bi o lo gi ca lly</w>': 3, '   ve gi ro ll</w>': 1, '   c ome di es</w>': 3, ' pu r su it</w>': 1, ' mo de</w>': 1, '   2 0 0 6 </w>': 1, ' fami li ar i ze</w>': 1, '   de fi ant</w>': 2, '   f la t f oo t</w>': 3, '   me d che ck</w>': 1, '   ja be z</w>': 89, '   s cu ff ing</w>': 1, '   con s ar n</w>': 8, '   gi d d y a p</w>': 1, '   ci r ca ssi an</w>': 1, '   a ll e gh en y</w>': 1, '   un par a</w>': 1, '   un par a ll el ed</w>': 5, '   ci r</w>': 1, ' ca ss</w>': 1, ' i an</w>': 1, '   be an sh oo ter</w>': 3, '   l an</w>': 7, '   ro p ed</w>': 2, '   mu ll</w>': 1, '   pri cking</w>': 2, ' na tu red</w>': 4, '   t ac t less</w>': 1, '   1 8 4 7 </w>': 1, '   he ssi an</w>': 4, '   me d for d</w>': 4, '   t ee to tal er</w>': 1, '   in ch wor m</w>': 1, '   chri st en</w>': 1, '   ac cla i m</w>': 2, '   whi g</w>': 1, '   ca m om i le</w>': 3, '   sch oo l ma ster</w>': 2, '   mar sh fi e ld</w>': 3, '   sp ru ce</w>': 1, '   wa sh woman</w>': 1, '   su ds</w>': 2, ' star ting</w>': 2, ' sti r r ing</w>': 1, '   g ru d ging</w>': 1, '   re pro ac h</w>': 3, '   supp li ca te</w>': 1, '   de b t ors</w>': 1, '   po ll u ting</w>': 1, '   bar n y ard</w>': 2, '   c ri tt ers</w>': 1, '   fi d ge ty</w>': 2, ' ra ps</w>': 1, '   out lan di sh</w>': 1, '   a i ling</w>': 1, '   fi g ger ed</w>': 11, '   con s ar</w>': 1, '   u z</w>': 1, '   e s che w ed</w>': 1, '   ou gh ter</w>': 1, '   can di da ture</w>': 1, ' r in ging</w>': 2, '   s lo s su m</w>': 1, '   ma li g n</w>': 2, '   al dri ch</w>': 1, '   par s ni ps</w>': 1, '   ma de ir a</w>': 1, '   hu ll oo o</w>': 1, ' bar re l ed</w>': 1, ' sha ving</w>': 1, '   att e sted</w>': 1, '   s la ver</w>': 1, '   nor ther ner</w>': 1, '   d ev </w>': 1, '   ad du ce</w>': 1, '   la ssi tu de</w>': 1, '   so f t ne ss</w>': 2, '   re pen t an ce</w>': 1, '   e c cen tri ci ty</w>': 1, '   com ar ad er i e</w>': 1, '   sp ri ck en z i e</w>': 1, '   ta k a g i</w>': 4, '   de ton at ors</w>': 5, '   ta ke over</w>': 4, '   gre en ma il</w>': 1, '   he b es</w>': 1, ' a si an</w>': 2, ' \t \t \t \t \t </w>': 13, ' \t \t </w>': 15, '   sa fe gu ar ds</w>': 2, '   ac ce ss ing</w>': 1, ' ex p lo it</w>': 1, '   exac t ne ss</w>': 1, ' y i pp e</w>': 1, ' k i</w>': 2, '   ne in</w>': 1, '   na k at om i</w>': 1, ' o h god please</w>': 1, '   se qu in ed</w>': 1, ' ra mb o</w>': 1, '   di lli on</w>': 1, '   re p on se</w>': 2, ' \t \t \t </w>': 42, '   e le c t ro ma gen ti c</w>': 1, '   me chan i ca ls</w>': 1, '   po stu r ing</w>': 1, '   gen n er o</w>': 5, ' e an in g ful</w>': 1, '   je t wa sh</w>': 1, '   ca pp y</w>': 1, ' han s</w>': 1, '   pla y bo ok</w>': 1, ' u g ar</w>': 1, '   en ri ch ed</w>': 1, '   h y dro gen a ted</w>': 1, '   po l y sor ba te</w>': 1, '   y u c ck</w>': 1, '   pla sti qu e</w>': 3, ' ro y</w>': 2, '   pre ps</w>': 1, ' s qu at</w>': 1, '   de mo li sh ing</w>': 1, '   op ted</w>': 1, '   st re tch out</w>': 1, '   el ks</w>': 1, '   par l ors</w>': 2, '   he i d i</w>': 7, '   ca ter pi ll ar</w>': 3, '   mor e tt i</w>': 3, '   na tu ra le</w>': 1, '   may e</w>': 1, '   un crazy</w>': 1, '   sha k ey</w>': 1, '   oo o h h</w>': 2, '   d ow n ers</w>': 2, '   u pp ers</w>': 1, '   sc rea m ers</w>': 1, '   g ro g g y</w>': 3, '   un n er st an</w>': 1, '   con fe der a tes</w>': 1, '   w new</w>': 1, ' 0 7 </w>': 1, '   mu l v an ey</w>': 5, '   fri o</w>': 1, ' ban ged</w>': 2, ' 1 0 5 </w>': 1, '   pa s sa ge way</w>': 3, '   un na stand</w>': 1, '   sti ck up</w>': 2, ' re gar ds</w>': 1, '   wa st e ba s k et</w>': 1, '   co pp in</w>': 2, ' ro b</w>': 1, '   hou se bro ke</w>': 1, '   ma sh</w>': 5, '   stra to sp here</w>': 3, '   e sp in o z a</w>': 1, '   cl ar e mon t</w>': 11, '   d ra ke</w>': 6, '   cho c o</w>': 8, ' cho c o</w>': 1, '   dan an g</w>': 3, '   al f</w>': 6, '   ho sing</w>': 1, ' ch in g a</w>': 1, '   ta m bi en</w>': 1, '   par ta king</w>': 1, '   ca min o</w>': 4, '   ci g li u t i</w>': 3, ' re move</w>': 1, '   la t ee sh a</w>': 4, '   m ee k a</w>': 2, '   loo p ho le</w>': 3, '   for mu la ted</w>': 1, '   cu r sing</w>': 7, '   k in i son</w>': 1, '   bi r thing</w>': 2, '   un fu l fi lling</w>': 1, '   wan der er</w>': 5, '   t ar y n</w>': 4, '   cha gr in</w>': 1, ' me s ca l ine</w>': 2, '   per ps</w>': 2, ' f la sh back</w>': 2, '   lo c us</w>': 4, '   un sha ven</w>': 1, '   di sh ev el ed</w>': 2, '   sc ra w ling</w>': 1, '   f an ci ed</w>': 3, ' ch ok o</w>': 1, ' fa s ci st</w>': 2, '   c lea v age</w>': 4, ' th under bi r ds</w>': 1, '   la tch ing</w>': 1, '   e y el in er</w>': 1, ' mu m</w>': 1, '   t ro ma s</w>': 1, '   w o l f s</w>': 1, ' han d ling</w>': 1, '   s lea z e ba gs</w>': 1, '   re tri ev es</w>': 2, '   g ri ps</w>': 3, '   la u r ence</w>': 7, '   z i er ing</w>': 1, '   r v </w>': 2, '   ba t ons</w>': 1, ' ch u cks</w>': 1, '   g ri tty</w>': 4, '   ca s sa ve tes</w>': 1, ' re f le x i ve</w>': 1, '   f on t</w>': 1, '   sy l ve ster</w>': 1, '   sta ll one</w>': 1, '   y o ke</w>': 3, '   ra he em</w>': 3, '   gen tri fi ca tion</w>': 1, '   br ow n st one</w>': 2, '   j or d ans</w>': 1, '   pi z z er i a</w>': 8, '   s qu a sh ed</w>': 3, '   mo o ki e</w>': 27, '   bu n t</w>': 1, '   rea c ted</w>': 4, '   lo v ell</w>': 2, '   k un ta</w>': 1, '   k in te</w>': 1, '   tri f ling</w>': 1, ' ou ch</w>': 1, '   du r ac ell</w>': 1, '   du r ac e lls</w>': 2, '   h er o s</w>': 1, '   par mi gi an a</w>': 1, '   m l</w>': 4, ' ore</w>': 1, '   what da fuck</w>': 1, '   de fin a tely</w>': 2, '   ha a g en</w>': 3, ' da z s</w>': 1, '   me da</w>': 2, '   ra w ne ss</w>': 2, ' mo on</w>': 2, ' a g en</w>': 1, '   da z s</w>': 2, ' t ee th</w>': 1, ' cha in</w>': 3, ' bi s cu it</w>': 1, ' ba s ke t b all</w>': 1, '   sp a de</w>': 8, '   m ou l an</w>': 2, '   y an</w>': 2, ' s lin ging</w>': 1, ' b en ding</w>': 1, '   pa v ar o tt i</w>': 2, '   n on sin ging</w>': 1, ' v it</w>': 1, '   f ar ra k ha n</w>': 2, '   p in o</w>': 9, '   b ru u u c ce</w>': 1, '   co o l in</w>': 3, '   wh ad d up</w>': 2, '   m ou li es</w>': 1, '   se z</w>': 7, '   de se</w>': 1, '   pi z z er i as</w>': 1, '   an cho vi es</w>': 2, '   you se</w>': 8, ' sha d d up</w>': 1, '   ch ong</w>': 5, '   k an</w>': 1, '   ma a a a w</w>': 1, '   sle ee ee w</w>': 1, '   sh ee k</w>': 1, '   ba w</w>': 1, '   ting</w>': 4, '   ta o</w>': 1, '   k u u n</w>': 1, '   le e k a</w>': 2, '   p ow w w</w>': 1, ' m ea l</w>': 1, ' cra pped</w>': 1, ' p an ts</w>': 11, '   j i ff</w>': 1, '   e m be z z ling</w>': 7, '   ac cre di ted</w>': 1, '   de li ca tes</w>': 2, ' ee e</w>': 2, '   wa ld</w>': 1, '   tw ri p</w>': 1, '   th i ll</w>': 1, '   ar w ti c le</w>': 1, '   je th i c a</w>': 1, '   he d do</w>': 1, '   je th</w>': 1, ' i c a</w>': 1, '   d un n e</w>': 7, '   n ea to</w>': 1, '   mar gar ine</w>': 1, ' ti m ing</w>': 2, '   wi l c o</w>': 2, '   o c cu pi es</w>': 1, '   we d gi es</w>': 1, '   as s face</w>': 3, '   har sh est</w>': 2, '   fa g g y</w>': 1, '   de sp er a do</w>': 2, ' re ss</w>': 1, '   m ou p h</w>': 1, '   ow w w w</w>': 5, '   s lu sh e e</w>': 3, '   bu tt li ck</w>': 1, '   can d y land</w>': 1, '   ch in g y</w>': 2, ' pe ci al</w>': 1, '   t in g ly</w>': 1, '   co o ti es</w>': 2, '   wor dy</w>': 1, '   pi s ca h p o</w>': 1, '   wa x er</w>': 1, '   i th</w>': 1, '   ne w th</w>': 1, '   d ou </w>': 2, ' ar r r r rs</w>': 1, '   d ow n side</w>': 3, ' do ll</w>': 1, ' ha ir s</w>': 2, '   s lu sh e es</w>': 1, '   th un k</w>': 1, '   s no op ed</w>': 1, '   s no op cu ff ed</w>': 1, ' par a de</w>': 1, '   f loo zy</w>': 1, '   fa u ce ts</w>': 1, ' of f it</w>': 1, '   z im m er</w>': 3, '   wai ki k i</w>': 2, '   mo ff it</w>': 2, '   ex ten der</w>': 1, ' a lo h a</w>': 1, ' ca kes</w>': 1, '   bi l king</w>': 1, '   que sa di ll a</w>': 1, '   fe y d</w>': 2, '   du cal</w>': 1, '   sig n et</w>': 1, '   k an ly</w>': 2, '   pi ter</w>': 2, '   at re i d es</w>': 7, '   ca la d an</w>': 1, '   car r y all</w>': 1, '   ac c el er a tes</w>': 1, '   or ni th op ter</w>': 1, '   f re men</w>': 5, '   sti ll su i ts</w>': 1, '   plan e to lo gi st</w>': 1, '   k y n es</w>': 2, '   e co lo gi st</w>': 1, '   tru th sa y er</w>': 4, '   sa ph o</w>': 1, ' li pped</w>': 2, '   men ta ts</w>': 1, '   har k on n en s</w>': 5, '   lan d s ra a d</w>': 1, '   ar ra k is</w>': 8, '   har k on n en</w>': 4, '   s ar da u k ar</w>': 1, '   fe u ding</w>': 1, '   ri che s se</w>': 1, '   i x</w>': 2, '   wor m sig n</w>': 1, '   te ll ta le</w>': 1, ' e f fi ci en cy</w>': 1, '   per spi ra tion</w>': 3, '   re cla im ed</w>': 1, '   ci r cu la tes</w>': 1, '   ca tch po ck e ts</w>': 1, '   pro ce ssed</w>': 8, ' wor l d ers</w>': 1, ' si re</w>': 1, '   gu r ne y</w>': 10, '   th u fi r</w>': 2, '   ha w at</w>': 1, '   cap ti ves</w>': 3, '   ha ll e ck</w>': 1, '   wh it</w>': 2, ' mo od</w>': 1, '   par ry</w>': 20, '   d ow n war ds</w>': 1, '   un bor n</w>': 4, '   sa y y ad in a</w>': 2, '   ge ss er it</w>': 5, '   we ir ding</w>': 3, '   min g le</w>': 5, '   sti l g ar</w>': 2, '   s la sh es</w>': 1, '   u tt ers</w>': 2, '   whi r ls</w>': 3, '   ma p es</w>': 1, '   sha d out</w>': 1, '   cha k ob s a</w>': 3, '   c r y s k ni fe</w>': 1, '   mi ssi on ar i a</w>': 1, '   pro te c ti v a</w>': 1, '   mi se ces</w>': 1, '   pre j in</w>': 1, '   b ho t an i</w>': 1, '   inter mi x</w>': 1, ' car ved</w>': 1, '   k wi sa t z</w>': 3, '   ha der ac h</w>': 3, '   t ea ch in gs</w>': 3, '   y u e h</w>': 3, '   re ver ence</w>': 1, '   mo hi am</w>': 1, '   en jo in</w>': 1, '   li et</w>': 1, '   cle an se</w>': 2, ' re fin ed</w>': 1, '   sh ea th</w>': 1, '   sti ll su it</w>': 1, '   u su l</w>': 3, '   fe da y kin</w>': 1, '   mu a d</w>': 3, ' i b</w>': 3, '   k u ll</w>': 1, '   wa had</w>': 1, '   with sto od</w>': 1, '   c ri sp ing</w>': 1, '   ob li ter ation</w>': 1, '   g lea m ing</w>': 1, '   go m</w>': 1, '   po i s ons</w>': 3, '   for mi d able</w>': 8, '   cho am</w>': 1, '   i mi ta ted</w>': 2, ' be g</w>': 4, '   du p le x</w>': 3, ' sh ower</w>': 1, '   s lo mo pa vi t z</w>': 2, '   see k er</w>': 5, '   be e f ea ter</w>': 1, '   da g m ar</w>': 1, '   m ou sta ph a</w>': 1, '   be l a</w>': 40, ' lu go s i</w>': 3, '   c ri s well</w>': 2, '   en vi ed</w>': 3, '   v a m pi r a</w>': 7, '   d ow ne y</w>': 1, '   c ome back</w>': 9, ' dr ac u l a</w>': 9, ' fran k en st e in</w>': 7, '   war ms</w>': 1, '   gu la g</w>': 2, '   e mi g ran t</w>': 1, ' stra p</w>': 1, '   chan d u</w>': 1, ' in cor por a tes</w>': 1, '   in car p ra ta tes</w>': 1, '   in por por a tes</w>': 1, ' gre e t in gs</w>': 2, '   g ou la sh</w>': 2, '   ac u l a</w>': 6, ' bi tch es</w>': 1, ' g l en</w>': 4, '   g l en da</w>': 6, '   ja il ba it</w>': 1, '   pi c s</w>': 2, '   we st er n s</w>': 3, ' pre s ence</w>': 2, ' pu ll</w>': 1, '   pu pp e t ma ster</w>': 1, '   je k y ll</w>': 2, '   tr ans ve sti tes</w>': 2, ' jo in ted</w>': 1, '   inter ru p ts</w>': 1, ' sc ore</w>': 1, '   re pe ls</w>': 1, '   att r ac ts</w>': 1, ' my th i c</w>': 1, '   sp oo ki er</w>': 1, ' bo ge y man</w>': 1, '   ba ve</w>': 1, ' e mer ge</w>': 1, '   r en fi e ld</w>': 1, '   p ou gh kee p si e</w>': 7, '   e mb ar king</w>': 2, '   te s l a</w>': 1, '   bea k ers</w>': 2, ' cla u de</w>': 2, '   ac tu a li ty</w>': 3, ' la ke</w>': 1, '   po stu re</w>': 2, '   i tch y</w>': 2, '   c r is</w>': 3, '   ma g ni fi c o</w>': 1, '   ra z z le</w>': 1, ' da z z le</w>': 1, '   fo l der o l</w>': 1, '   1 9 7 5 </w>': 2, '   e li gi bi li ty</w>': 1, ' ad di c ted</w>': 3, '   me th ad one</w>': 1, '   do l or es</w>': 36, '   pro du c tions</w>': 1, '   re ca st</w>': 1, '   b ack er</w>': 2, '   b ack ers</w>': 7, '   gir lie</w>': 3, '   ra th b one</w>': 1, '   j or gen s en</w>': 10, ' supposed</w>': 3, '   la w ton</w>': 1, '   p ho to play</w>': 1, '   k ra vi t z</w>': 1, '   la e mm le</w>': 1, ' d r</w>': 3, '   gh ou l</w>': 3, '   re su mes</w>': 2, '   i had</w>': 1, '   un in te lli gi ble</w>': 2, ' be l a</w>': 1, '   gra ver ob b ers</w>': 1, ' gra ver ob b ers</w>': 1, '   a po st les</w>': 3, '   u p li f ting</w>': 2, '   h ru ph h</w>': 1, '   ki ll ink</w>': 1, '   loo k ink</w>': 1, '   t ink</w>': 3, '   g lu g</w>': 1, '   o op sy</w>': 1, '   be d der</w>': 1, '   ea t ink</w>': 1, '   g li de</w>': 1, '   s wa tch</w>': 1, '   e d v ard</w>': 5, '   vi fe</w>': 1, ' lo b o</w>': 1, '   w re st l ink</w>': 1, '   de y</w>': 3, ' loo k ink</w>': 1, '   mo o v f</w>': 1, '   th ri ll er</w>': 1, '   re cu t</w>': 3, '   win d ba g</w>': 1, '   pro m o</w>': 1, '   ma l ta</w>': 1, '   st in k b om b</w>': 1, ' tr ans ve sti te</w>': 1, ' e st ab li sh ing</w>': 1, '   ex hi bi t ors</w>': 1, ' pi c ture</w>': 3, '   s me l ting</w>': 3, '   st or m ing</w>': 3, ' im pre ssi ve</w>': 1, '   mon e y ma k er</w>': 1, '   con fi den ti ally</w>': 1, '   par at ro op ed</w>': 1, ' ter ri fi ed</w>': 1, '   under gar men ts</w>': 3, ' so ld</w>': 1, '   o ki es</w>': 1, '   fi c ti tion a li ze</w>': 1, ' v ar i e ty</w>': 1, '   stra in er</w>': 1, '   bra ssi er es</w>': 1, ' mer cu ry</w>': 1, '   u sh er ing</w>': 1, '   d ra ma s</w>': 2, '   g ro ver s</w>': 1, '   bo o ti es</w>': 1, ' gh ou l</w>': 3, '   p un chi er</w>': 1, '   ha if</w>': 1, '   man ch u</w>': 1, '   loo k a li kes</w>': 1, ' inter e sted</w>': 2, '   c lu tter</w>': 2, ' fran c is</w>': 2, ' ter ri ble</w>': 2, ' an e</w>': 1, '   pre mi se</w>': 2, '   con su m ma ted</w>': 1, '   ho t ca kes</w>': 2, ' ac tor</w>': 1, ' wri ter</w>': 3, ' pro du c er</w>': 2, ' b red</w>': 2, '   t ac ti le</w>': 1, '   sen su a li ty</w>': 1, '   je an e tt e</w>': 1, '   ru mp le y</w>': 1, '   fo l k sy</w>': 1, '   v om i ting</w>': 2, '   ban an a head</w>': 1, '   te ar j er k er</w>': 1, '   lo v ea ble</w>': 2, '   st e p father</w>': 4, '   sh ar i</w>': 21, ' che ers</w>': 1, ' ma sh</w>': 2, ' pu mp kin</w>': 2, ' ca ms</w>': 1, '   go o fi est</w>': 1, '   l ab or ing</w>': 2, '   mi s con ce p tion</w>': 1, ' in f la ted</w>': 1, '   po lled</w>': 1, '   fuck able</w>': 2, '   pe k u r ny</w>': 3, '   na me ta g</w>': 2, '   sha qui ll e</w>': 3, '   h y st er e c tom y</w>': 2, '   po x</w>': 6, ' en ter ta in er</w>': 1, '   fo i b les</w>': 6, '   bu n ted</w>': 1, '   s lea z i est</w>': 2, '   z a mb on i</w>': 1, '   r in se</w>': 1, '   lo g ging</w>': 7, '   con te st ant</w>': 1, '   s na pp le</w>': 1, '   bu ll shi tter</w>': 1, '   re ar ran ging</w>': 1, ' er ne st</w>': 1, '   sto ck room</w>': 1, '   th u mb su ck er</w>': 1, '   sti tch ing</w>': 2, '   be d we tter</w>': 1, '   proble ma</w>': 1, '   tu mb l ers</w>': 1, '   sh me ar</w>': 1, ' k o ok y</w>': 1, '   pi er de</w>': 1, ' e ll a</w>': 1, '   pi er d es</w>': 1, '   pi er do</w>': 1, '   sh e er r ry</w>': 1, '   in sp ir es</w>': 4, '   no vo t ny</w>': 2, '   mi ll ard</w>': 4, '   do it</w>': 2, '   ba ll o ts</w>': 3, '   8 0 3 </w>': 2, '   8 0 1 </w>': 1, '   e le c ting</w>': 1, '   m ca l list er</w>': 9, '   s g a</w>': 3, '   du al</w>': 3, ' co ll ea gu e</w>': 1, '   lea ch ing</w>': 1, '   s lan der ous</w>': 2, ' le ga l</w>': 1, '   re pu ta tions</w>': 1, '   th i e son</w>': 1, '   sch en k en</w>': 1, '   me t z i er</w>': 1, ' inter ro ga te</w>': 1, '   inter ro ga ting</w>': 2, '   pe char da</w>': 1, '   k ob z a</w>': 3, '   ca m pa i g ning</w>': 1, '   u no pp osed</w>': 1, '   s ke p ti ca lly</w>': 1, '   pe ars</w>': 1, ' ge tter</w>': 1, '   we st side</w>': 2, '   f ou ch</w>': 1, ' dis re gar ds</w>': 1, '   2 5 6 </w>': 1, '   2 5 5 </w>': 2, '   2 5 3 </w>': 1, '   2 5 4 </w>': 1, '   mer c</w>': 3, '   cou pe</w>': 2, '   ex le y</w>': 11, '   h te</w>': 1, '   fo l so m</w>': 3, '   ga vi l an</w>': 1, '   pa t che t t</w>': 30, '   br ack en</w>': 13, '   in ve sts</w>': 1, '   con tra di c tions</w>': 2, ' pi er ce</w>': 1, '   le ffer ts</w>': 15, '   ni te</w>': 15, '   hu mor ing</w>': 2, '   st en s land</w>': 15, '   cl ar i fi ca tion</w>': 1, '   du d le y</w>': 32, '   per pe tra te</w>': 1, '   per p le x ing</w>': 1, '   re tr act</w>': 1, '   re can ted</w>': 1, '   m ow ing</w>': 4, '   ad ju t ant</w>': 1, '   bea t ers</w>': 2, '   a m d</w>': 1, '   sta ms</w>': 1, '   st en s</w>': 8, '   m un s</w>': 1, '   l un ts</w>': 1, '   b re un ing</w>': 1, '   car li s le</w>': 2, '   lo e w</w>': 5, ' l y n n</w>': 1, '   wor k d</w>': 1, '   w re cking</w>': 5, '   b ru en ing</w>': 1, ' st en s land</w>': 1, '   tri g g ers</w>': 5, '   per k ins</w>': 10, '   st om p an a to</w>': 2, ' ju sti ce</w>': 3, '   mor e house</w>': 2, '   li qu ors</w>': 2, '   so li ci ting</w>': 3, '   ber do o</w>': 2, ' pa e s an o</w>': 2, ' ca pi s ce</w>': 1, ' e ar</w>': 13, '   f la i ling</w>': 1, '   con tra st</w>': 1, ' for th right</w>': 1, ' 4 3 </w>': 2, '   f on ta ine</w>': 1, '   w o of in</w>': 1, '   ca si ta s</w>': 2, '   si ssi es</w>': 1, '   f er st un k en er</w>': 1, '   ver on i ca a a</w>': 1, '   re ci pro ci ty</w>': 1, '   hu d ge ons</w>': 3, '   gre en b ac ks</w>': 2, '   per i o di c</w>': 2, '   s ni ff er</w>': 1, '   tra sh can</w>': 2, '   e d m un d</w>': 7, '   you th s</w>': 1, ' mo de l</w>': 4, '   cor ro bor a tive</w>': 1, '   app li c ant</w>': 5, '   lan a</w>': 9, '   f le u r</w>': 4, ' l is</w>': 5, ' f r</w>': 1, '   s na tch er</w>': 2, ' li e u ten ant</w>': 1, '   s ni tch ing</w>': 3, '   ho b no b</w>': 2, ' thr ow ing</w>': 1, '   i ci c les</w>': 1, ' cap ac i ty</w>': 1, '   re lo a ding</w>': 1, '   su bor n</w>': 1, '   i lli c it</w>': 2, ' ho ok ers</w>': 1, '   bi s be e</w>': 1, '   pen si on ers</w>': 1, '   you with</w>': 1, '   s me ar ing</w>': 2, '   s co o ts</w>': 1, '   bu p k is</w>': 2, '   sh v ar t ze</w>': 1, '   si d ster</w>': 1, ' wi li ght</w>': 1, '   sin u en do</w>': 1, '   in gen u es</w>': 2, '   bo y chi ck</w>': 2, '   po s ers</w>': 1, '   ho ph ea ds</w>': 2, ' sto pp er</w>': 1, '   b ack gr oun s</w>': 1, '   pa t ro l men</w>': 1, '   2 2 4 5 </w>': 1, '   mar a vi ll a</w>': 1, ' in gen u e</w>': 1, '   f lea u r</w>': 1, '   2 0 9 </w>': 1, '   z a vi t z</w>': 20, '   de p into</w>': 10, '   bri ll</w>': 16, '   u p sho t</w>': 2, '   d h</w>': 1, '   di gi te ch</w>': 1, '   su b je c ted</w>': 3, '   vo ll o tt i</w>': 2, '   a mer i te ch</w>': 1, '   sp ort sp h one</w>': 1, ' sto ck</w>': 2, ' se ar ch ed</w>': 1, '   ma i ling</w>': 1, ' di re c tly</w>': 1, ' ne ar</w>': 1, '   ha mer sle y</w>': 1, '   sh en an do ah</w>': 1, ' ra m par ts</w>': 1, '   m ea de</w>': 1, '   ph on e call</w>': 1, ' f la g</w>': 4, '   com par a tive</w>': 1, '   a ir wa ve</w>': 1, '   ce ll u l ars</w>': 1, '   sa ts</w>': 8, '   d ru g gi es</w>': 1, ' m ou th s</w>': 1, '   te le</w>': 3, ' com m uni ca tions</w>': 1, '   te m per an c ev i ll e</w>': 1, '   sa li s bu ry</w>': 5, '   wri st wa tch</w>': 1, ' h er t z</w>': 2, '   a qu ac a de</w>': 1, ' s at</w>': 1, '   l ow j ack</w>': 1, '   pu l ses</w>': 1, ' tr ack er</w>': 1, '   s r k</w>': 1, ' 1 3 3 9 </w>': 1, '   de l s an o</w>': 1, '   ba u d more</w>': 1, '   con su l t an ts</w>': 1, ' knew</w>': 6, '   ga me boy</w>': 2, '   di or</w>': 1, ' wi l d life</w>': 1, '   mor e lo s</w>': 1, '   r ac que t b all</w>': 2, ' pre sen ts</w>': 1, ' st ac y</w>': 1, ' e le c tr on i ca lly</w>': 1, ' ye st er day</w>': 4, ' coun se l or</w>': 1, ' s wor e</w>': 2, ' ar e a</w>': 2, ' s our ces</w>': 2, ' he ar ing</w>': 2, '   s na fu </w>': 1, '   sp o ok in</w>': 2, '   mor ad a</w>': 1, ' mo b</w>': 1, '   si l ver ber g</w>': 4, ' bri ll</w>': 1, '   v ar i es</w>': 1, '   be ll mo th</w>': 5, '   uni ons</w>': 6, '   mi x ing</w>': 2, ' tru th s</w>': 1, '   fa well</w>': 1, '   gu i do</w>': 2, '   lu kes</w>': 1, '   sp in ks</w>': 1, '   chi an t i</w>': 2, ' e sp re s so</w>': 1, '   la tt e</w>': 3, ' ma in</w>': 1, '   re fin an c ing</w>': 1, ' ter r ori s m</w>': 2, '   un con s ci on able</w>': 2, ' par ti s an</w>': 2, '   l ab our</w>': 3, '   can st</w>': 4, '   el su r</w>': 1, '   re si den ti al</w>': 2, '   sta tion ary</w>': 2, ' or g ani z ation</w>': 1, ' la p</w>': 1, '   w ea ther ship</w>': 1, ' ph on e bo ok</w>': 1, '   co in te l</w>': 1, '   inter se c ts</w>': 1, '   n r o</w>': 1, '   in di sc ri min a tes</w>': 1, ' z a vi t z</w>': 1, '   pu ff ed</w>': 2, '   co p se</w>': 1, '   ge of f re y</w>': 9, '   h y m n s</w>': 2, '   f en el on</w>': 3, ' bar n es</w>': 3, '   th i mb le</w>': 1, '   ver an da h s</w>': 1, '   cap si z ed</w>': 1, '   sa ff r on</w>': 1, '   ma do x</w>': 4, '   al ma sy</w>': 2, '   bo sp h or ous</w>': 1, '   s z er e le m</w>': 1, '   se w s</w>': 1, '   da i j k a</w>': 1, '   ha mp ton</w>': 1, '   im pa ti en tly</w>': 2, '   mar mi te</w>': 2, '   al m</w>': 7, ' sy</w>': 8, '   he d ge ho gs</w>': 2, '   f re sh wa ter</w>': 1, '   c li f ton</w>': 18, '   a er i al</w>': 1, '   h er o do tu s</w>': 6, '   si mo on</w>': 1, '   har ma tt an</w>': 1, '   un ti dy</w>': 1, '   au f</w>': 2, '   z er z u r a</w>': 5, '   wa d i</w>': 2, '   ar ab </w>': 1, '   sh ru gs</w>': 4, '   pre da t ory</w>': 1, '   g y g es</w>': 6, '   da s k y l us</w>': 2, '   can da u les</w>': 6, '   s an d st or ms</w>': 1, ' dri ven</w>': 2, '   co lon i al s</w>': 3, ' \x85   </w>': 5, ' h r er</w>': 2, '   bri ga di er</w>': 6, '   u x ori ou s ne ss</w>': 1, '   gi lf</w>': 1, '   ke bi r</w>': 1, '   k u fr a</w>': 1, '   bri ts</w>': 4, '   ber man n</w>': 3, '   inter ned</w>': 1, ' a g</w>': 1, '   wi st er i a</w>': 2, '   d or set</w>': 1, '   g ri pped</w>': 1, '   la sh in gs</w>': 1, '   re sur face</w>': 1, '   ex pe di tions</w>': 2, ' win ds</w>': 1, '   o ver f l own</w>': 1, '   fo ssi li z ed</w>': 1, '   in n it</w>': 4, '   bi s mar ck</w>': 1, '   li b ya</w>': 2, '   be ll in i</w>': 3, '   pa in t ers</w>': 3, '   k in g st on</w>': 1, '   bu on</w>': 2, '   gi or no</w>': 2, '   ki p</w>': 40, '   to b ru k</w>': 2, '   han a</w>': 21, '   p hi al s</w>': 1, '   st ab les</w>': 1, ' b la h</w>': 10, '   par ti s ans</w>': 1, '   mon t real</w>': 2, '   la u ri er</w>': 1, '   ro m me l</w>': 3, '   pu r da h</w>': 1, '   cu tt in gs</w>': 2, '   cour t ro om s</w>': 3, '   sle e pi ly</w>': 1, '   fr ow n s</w>': 3, '   shu ff les</w>': 2, '   pi ck po ck et</w>': 3, '   s w el ter ing</w>': 1, '   z in c</w>': 3, '   har e ms</w>': 3, '   ber e ft</w>': 1, '   d ow ning</w>': 1, '   shi ll in gs</w>': 2, '   shu d der ed</w>': 1, ' t wi st ing</w>': 2, '   c li f t ons</w>': 1, '   app r en ti ces</w>': 2, '   con sig n</w>': 1, '   sc ra ph ea p</w>': 1, '   bo sp h or us</w>': 2, '   con den sed</w>': 3, '   in ven tions</w>': 2, '   si k h</w>': 2, '   be tra y al s</w>': 2, '   chi l d like</w>': 1, '   co lli de</w>': 3, '   ac ac i as</w>': 1, '   ac ac i a</w>': 1, '   pl u ms</w>': 2, '   n ya</w>': 3, ' n ya</w>': 3, '   d un g</w>': 3, '   la h or</w>': 1, '   z a m z a m ma h</w>': 1, '   k a ma su tr a</w>': 1, '   gi g g les</w>': 1, '   fu si li ers</w>': 1, '   on t ar i o</w>': 1, '   pi c ton</w>': 1, '   vi a du ct</w>': 1, '   rea pp e ars</w>': 1, '   hur ri es</w>': 4, ' a go st in o</w>': 1, '   mu d dle</w>': 4, '   fi on a</w>': 2, '   chri st en ed</w>': 1, ' ma k ers</w>': 2, '   com ma</w>': 3, '   la h ore</w>': 1, '   a ja i b</w>': 1, ' gh er</w>': 1, '   ho t sp u r</w>': 1, '   m ac d ou ga l</w>': 2, '   man z in i</w>': 1, '   mon te z u ma</w>': 2, '   bi s ma l</w>': 1, '   ca la li li es</w>': 1, '   un sc re w ed</w>': 1, '   p an e</w>': 3, '   e f fe c tu a ted</w>': 1, '   th i ba dea u x</w>': 1, '   k ow lo on</w>': 2, '   dis co ver s</w>': 1, '   x j </w>': 3, '   re ck le ss ly</w>': 1, '   ro m</w>': 3, '   re t in as</w>': 1, '   o ver e sti ma ting</w>': 2, '   han do ver</w>': 2, '   stu mp ing</w>': 1, '   o s ci ll o s co pe</w>': 1, '   in st ru c ts</w>': 1, '   in no cu ous</w>': 2, '   ro ck e ted</w>': 1, '   con ver si ons</w>': 2, '   re ca l cu late</w>': 1, ' mi sta ke</w>': 2, '   pi rs</w>': 1, '   po p up</w>': 1, '   b la m</w>': 1, ' part n er ship</w>': 1, '   he d ger ow</w>': 1, '   sy n ch r on i z ed</w>': 3, ' nee ds</w>': 2, '   f ru i t loo ps</w>': 1, ' ba g ga ge</w>': 1, '   gi ver ne y</w>': 1, '   l ac ked</w>': 1, '   ca llow</w>': 1, '   ro k i</w>': 3, '   sh e k</w>': 1, '   re se ll</w>': 1, '   de ca ys</w>': 1, '   mi c ro chi ps</w>': 6, '   com mo s</w>': 1, '   th i bea u</w>': 1, '   b al en ci a g a</w>': 1, ' com m</w>': 1, '   ki ts</w>': 2, '   ir</w>': 1, ' ther m o</w>': 1, '   man or</w>': 3, '   f er r ar is</w>': 1, '   ma s ry</w>': 14, '   he x a v al ent</w>': 5, '   ch ro mi u m</w>': 17, ' dan c ers</w>': 1, '   bro ck o vi ch</w>': 5, '   men in gi t is</w>': 3, '   in f la m ma tion</w>': 2, '   ma x i</w>': 1, ' pa ds</w>': 1, '   to x i co lo gi st</w>': 2, '   cor re sp on ding</w>': 1, '   su b di vi ding</w>': 1, '   mo le y</w>': 1, '   re mi ssion</w>': 1, '   ho d g kin</w>': 1, '   vi ti to e</w>': 3, '   po tter</w>': 24, '   pla in ti ff s</w>': 8, '   ar bi tra tion</w>': 3, '   lin king</w>': 3, ' per ks</w>': 1, '   st en o</w>': 1, '   dea th bed</w>': 2, '   de mu r</w>': 2, '   ba u m</w>': 1, '   a qui f er</w>': 1, '   me l en de z</w>': 1, '   f l ow ed</w>': 1, '   gr oun d wa ter</w>': 1, '   se ep</w>': 1, '   sha m ro ck</w>': 1, '   ch ro m</w>': 4, '   ce l</w>': 1, '   t or t</w>': 1, '   st y</w>': 1, '   fran k el</w>': 1, '   wa sh clo th</w>': 2, ' su m</w>': 1, '   mi s ca l cu la ted</w>': 2, ' se ttle</w>': 2, ' af for d</w>': 2, '   sh re d der</w>': 2, '   pa tt e e</w>': 2, ' brea thing</w>': 2, '   car c in o gen i c</w>': 1, '   de ter i or ation</w>': 2, '   ro s</w>': 2, ' do g</w>': 9, '   ro sa l ind</w>': 2, '   7 1 4 </w>': 1, ' 9 3 4 6 </w>': 1, ' wh ose</w>': 2, '   cou ri er ed</w>': 1, '   ear lo b es</w>': 2, '   hou se ca ts</w>': 1, '   af ter no ons</w>': 3, '   4 5 4 </w>': 1, ' 3 9 4 3 </w>': 1, '   un cle ar</w>': 5, '   lin w o od</w>': 1, '   a i</w>': 2, '   bu en o</w>': 2, '   sp r in k l er</w>': 1, '   e pi de mi o lo gi cal</w>': 1, '   mu l ti vi ta m in</w>': 1, '   sh an na</w>': 1, ' ba x ter</w>': 1, '   e pi de mi o lo g y</w>': 1, '   li gh t spe ed</w>': 2, '   s k y wal k er</w>': 11, '   da go ba h</w>': 1, '   gr ow ls</w>': 2, '   do le ful</w>': 2, '   che wi e</w>': 26, '   con st er na tion</w>': 1, '   ar too</w>': 17, '   pu d dle</w>': 4, '   ta un ta un s</w>': 1, '   sp ee d ers</w>': 3, '   ob i</w>': 8, '   un doing</w>': 2, '   pre car i ous</w>': 2, '   j ab b a</w>': 6, '   hu t t</w>': 4, '   h y per dri ve</w>': 5, '   th re e pi o</w>': 11, '   ti ban na</w>': 1, '   sa b ac c</w>': 1, '   be sp in</w>': 2, '   rea cha ble</w>': 2, ' ha h</w>': 6, '   an o at</w>': 1, '   de st ro y ers</w>': 3, '   my no ck</w>': 1, '   pu l ver i z ed</w>': 1, '   ho pp in</w>': 1, '   n er f</w>': 1, ' h er der</w>': 1, '   ri ee k an</w>': 1, ' de st ru ct</w>': 2, '   ma k in gs</w>': 1, '   w oo ki e e</w>': 3, '   g an k</w>': 1, '   or d</w>': 1, '   man tell</w>': 1, '   g und ar k</w>': 1, '   in gs</w>': 2, '   re sp on</w>': 1, '   si ble</w>': 1, '   en ter pri s ers</w>': 1, '   un p ar</w>': 1, '   a ll el ed</w>': 1, '   ga ting</w>': 1, '   in di sp o si tion</w>': 1, '   b ac ta</w>': 1, '   for e kn ow le dge</w>': 1, '   ve ers</w>': 1, '   pi e t t</w>': 1, '   ho th</w>': 1, ' pre ten ding</w>': 1, '   chi mp an z e es</w>': 1, '   gu ar d house</w>': 1, '   z ir a</w>': 21, ' b ack w ard</w>': 1, '   cap t ors</w>': 1, '   e di fi ed</w>': 1, '   c in der</w>': 1, '   car pe t ba g</w>': 1, ' pri mi tive</w>': 2, ' pri mi ti ve ly</w>': 1, '   pl u to x in</w>': 2, '   p li ss k en</w>': 15, '   o ver lo a ding</w>': 1, '   on of re</w>': 1, '   shu t down</w>': 2, '   c y ber sp ace</w>': 4, '   tr ans mi tting</w>': 4, '   com st at</w>': 1, '   p sy cho se ar ch</w>': 1, '   so ci op a th i c</w>': 3, '   o ow w</w>': 1, '   gr in go</w>': 3, '   h er she</w>': 5, '   ho ver while</w>': 1, ' to x in</w>': 2, '   ca hu en g a</w>': 1, '   bra z i li ans</w>': 1, '   b lu e b ac ks</w>': 2, '   uni f y</w>': 1, '   ci t y wi de</w>': 1, '   mo j o</w>': 1, '   de ll as an dr o</w>': 1, '   an a he i m</w>': 1, '   mu ch o</w>': 1, '   tr an sp lan ted</w>': 1, '   ta s li ma</w>': 1, ' ha y</w>': 2, '   wi el ds</w>': 1, '   per c ev al</w>': 3, ' per c ev al</w>': 1, '   gu en ev ere</w>': 9, '   lan ce lot</w>': 14, '   men ded</w>': 4, '   re pen ted</w>': 3, '   chri sti an do m</w>': 2, '   a vo i dan ce</w>': 1, '   j ou st</w>': 1, '   sha d ow y</w>': 2, '   mor d red</w>': 2, ' ga wa in</w>': 1, '   bor s</w>': 1, '   bo h or t</w>': 1, '   car a do c</w>': 1, '   e ctor</w>': 2, '   fe ll ow ship</w>': 9, ' mor d red</w>': 1, '   mor g an a</w>': 3, '   sor cer y</w>': 3, ' st ori es</w>': 1, '   con ver ge</w>': 3, '   co i ling</w>': 1, '   un co i ling</w>': 1, '   ri di cu l ed</w>': 2, '   t ro tting</w>': 1, '   re i g ned</w>': 1, '   g al ys</w>': 1, '   c in d ers</w>': 1, '   le on de g ran ce</w>': 2, ' le on de g ran ce</w>': 1, '   u r y en s</w>': 2, '   u ther</w>': 7, '   i gra y n e</w>': 1, '   re be lled</w>': 1, '   co a sts</w>': 1, '   mi gh ti est</w>': 2, '   be see ch ed</w>': 1, '   ga ha lt</w>': 2, '   i d l en ess</w>': 2, '   dam se ls</w>': 1, '   sti tch er y</w>': 1, '   for bi ds</w>': 2, '   p le d ging</w>': 2, ' ke pt</w>': 1, '   inter ven ed</w>': 3, '   be ho l d en</w>': 3, '   ga wa in</w>': 3, '   cl ou d bur st</w>': 1, '   qu en ch es</w>': 1, '   wi z en ed</w>': 1, '   su m mon ing</w>': 1, '   pre ying</w>': 1, '   th i ck en</w>': 1, '   con ju r ing</w>': 2, '   lo ve st ru ck</w>': 1, '   mo on bea ms</w>': 2, '   ir ks</w>': 1, '   lu d</w>': 2, '   b al du r</w>': 3, '   mu r mu r ing</w>': 1, '   ex or ci s m</w>': 16, '   tom b s</w>': 3, '   mer r in</w>': 5, '   n in ev e h</w>': 1, '   lan k a ster</w>': 1, '   ex or ci st</w>': 5, '   r ab b is</w>': 2, '   in te ll e g ence</w>': 1, '   ph on o gra p h</w>': 1, '   re g an</w>': 25, '   m ac ne il</w>': 11, '   k in der man</w>': 2, '   i saw</w>': 1, '   den n in gs</w>': 10, '   ba ff ling</w>': 2, '   fr ac tur ing</w>': 1, '   af f raid</w>': 2, '   te m per al</w>': 3, '   mu su cl ar</w>': 1, '   con v u l sion</w>': 2, '   de stu r ban ce</w>': 1, '   che mi c o</w>': 1, '   un char ac ter</w>': 1, '   ob sc en i ti es</w>': 1, '   ri tal in</w>': 4, '   h y per k in e ti c</w>': 1, '   se ms</w>': 1, '   o ver rea ction</w>': 2, '   sti mu l ant</w>': 3, '   mi li gra ms</w>': 1, '   a do le sc ence</w>': 2, '   h y per ac ti vi ty</w>': 1, '   war ran ted</w>': 2, '   je su i ts</w>': 9, '   m ac ine</w>': 1, '   sp ri tu al</w>': 1, '   d y er</w>': 3, '   k ar ra s</w>': 11, '   gu il ding</w>': 2, '   ma ir</w>': 1, '   reme b er</w>': 2, '   da mi en</w>': 6, '   ar ter i o gra m</w>': 1, '   con ce i v ably</w>': 1, '   p ne u mo en ce p he lo gra m</w>': 1, '   au th en ti ca ted</w>': 1, '   so ma ti c</w>': 1, '   di ms</w>': 1, '   l ye</w>': 2, '   chi v as</w>': 1, '   ear n sha w</w>': 1, '   with er ing</w>': 1, '   u o y</w>': 3, '   o h w</w>': 2, '   t se ir p</w>': 2, '   a h h h h h h h h h h h</w>': 1, '   y do b</w>': 1, '   e h t</w>': 1, '   m ra w</w>': 1, '   e es</w>': 1, '   e m it</w>': 2, '   ev i g</w>': 1, '   ni r re m</w>': 2, '   ro ts</w>': 3, '   t an te</w>': 1, '   qu od</w>': 3, '   no men</w>': 2, '   mi hi</w>': 2, '   ab s lo v o</w>': 1, '   mi r ab i le</w>': 1, '   di c tu </w>': 1, '   di m my</w>': 1, '   di sp a ir</w>': 1, '   possi bi lli ty</w>': 1, '   pre y er</w>': 1, '   be go tt en</w>': 1, '   in i qui ty</w>': 1, '   su sta ins</w>': 1, '   t re sp as ses</w>': 1, '   man i fe sted</w>': 8, '   re s d ence</w>': 1, '   ca s so ck</w>': 1, '   sur p li ces</w>': 1, '   d om in i c ans</w>': 1, '   de s de mon a</w>': 1, '   de se cra tor</w>': 1, '   a po st le</w>': 1, '   under f oo t</w>': 1, '   wai t ers</w>': 4, '   dr y land</w>': 1, ' ca mp town</w>': 2, '   ki b ble</w>': 1, '   den ture</w>': 1, '   bea sle y</w>': 3, '   jo ck stra ps</w>': 1, '   pe da ls</w>': 2, '   sti ff er</w>': 3, '   stu r dy</w>': 3, '   re gen cy</w>': 1, '   cla ys</w>': 1, '   vo ca ls</w>': 1, '   ma ll ory</w>': 34, '   sh el ton</w>': 1, '   f on der</w>': 1, '   tra ve lo dge</w>': 1, '   ba s ke t ba lls</w>': 2, '   m ea d ow l ar k</w>': 1, '   te le th on</w>': 1, '   ti me x es</w>': 1, '   y o l ks</w>': 1, '   de di ca tions</w>': 1, '   o c ta ve</w>': 1, ' i e</w>': 4, ' f le x i ble</w>': 1, '   se ven te en th</w>': 1, '   bu n ked</w>': 2, '   dri es</w>': 2, '   pi ling</w>': 2, ' ch op sti cks</w>': 1, '   si x ti e th</w>': 1, '   ow ls</w>': 1, '   ei gh th s</w>': 1, '   g ust</w>': 1, '   kn able</w>': 1, '   y a ma h a</w>': 1, '   bo s en</w>': 1, '   har pi st</w>': 1, '   app en di ci t is</w>': 1, '   co z i er</w>': 2, '   t or me</w>': 1, '   s ni ff les</w>': 1, '   bl an c he</w>': 3, ' late</w>': 2, '   chan el</w>': 4, '   un ab ri d ged</w>': 1, '   che ck ers</w>': 2, '   me ss sa ge</w>': 1, '   a ve don</w>': 2, '   ne w stand</w>': 2, '   bro ok haven</w>': 1, '   op al s</w>': 1, '   h ow s a</w>': 1, '   s n ows</w>': 2, '   ser ver</w>': 3, ' ca ses</w>': 1, '   ca st or</w>': 27, '   sta te ho od</w>': 1, '   wa x y</w>': 1, '   ar ch er</w>': 17, '   par li a ment</w>': 1, ' spe c tru m</w>': 1, '   comp li men ting</w>': 1, '   ca z</w>': 2, ' re ten tive</w>': 1, '   rea c ti v a ted</w>': 3, '   po ll u x</w>': 11, '   ju mb l ed</w>': 2, '   por ked</w>': 1, '   do b b s</w>': 7, '   pro c ni as</w>': 1, '   a ver an o</w>': 1, '   la z ar r o</w>': 5, '   b oun ti es</w>': 1, '   n an d i</w>': 1, '   ma tty</w>': 2, '   a pa th y</w>': 1, '   mi ser ably</w>': 3, ' supp or t</w>': 2, '   g ri pe</w>': 8, '   pa u per</w>': 1, ' ti t ani ca lly</w>': 1, '   s f p d</w>': 2, '   er e wh on</w>': 1, '   re cu l ti v ate</w>': 1, '   st in g ers</w>': 2, '   car b in es</w>': 2, ' te ch no lo g y</w>': 2, '   qu ad ru pl ed</w>': 1, ' war ran ts</w>': 3, ' as sig n ment</w>': 2, '   hu tt on</w>': 15, '   o x x for d</w>': 1, '   chri st ma ss y</w>': 1, '   t z u</w>': 2, '   to te m</w>': 2, '   ch in t z</w>': 1, '   min t z</w>': 3, '   z er o es</w>': 1, '   j an es</w>': 2, ' sh ine</w>': 1, '   gi m mi ck y</w>': 1, '   b ow l ers</w>': 6, ' m on</w>': 9, '   ar n</w>': 1, '   cre di tor</w>': 1, '   jo i e</w>': 1, '   vi v re</w>': 1, '   re ver i e</w>': 1, '   bar c a</w>': 1, ' l oun ger</w>': 1, ' ex ten ded</w>': 2, '   supp li er</w>': 5, '   im per man ent</w>': 1, '   ca ll able</w>': 1, ' fa te</w>': 1, ' the i sti c</w>': 1, '   1 9 8 8 </w>': 4, '   de u t sc he</w>': 1, '   re f er en du m</w>': 1, '   la ssi ter</w>': 2, '   do gh ou se</w>': 2, '   z e en a</w>': 1, '   na u s ea ted</w>': 3, '   t re k</w>': 2, ' 2 4 0 0</w>': 1, '   car ting</w>': 1, ' v an</w>': 2, '   ra ff i</w>': 1, ' re ta il</w>': 1, '   fun ne l</w>': 1, '   sp ra w l ed</w>': 1, '   win ing</w>': 1, '   c n b c</w>': 2, '   th om p s ons</w>': 1, '   no b le s se</w>': 1, '   ho y o</w>': 1, '   mon ter re ys</w>': 1, '   we pt</w>': 7, '   b ou ti qu e</w>': 3, '   con l in</w>': 1, '   go o dri ch</w>': 1, ' for ce</w>': 3, '   mo ti v a tion al</w>': 1, '   tu sc any</w>': 1, '   un w ra pp ing</w>': 2, ' du st y</w>': 1, ' su per co o l</w>': 1, ' si mu la tor</w>': 1, ' enough</w>': 2, ' do om</w>': 1, ' pro to t y pe</w>': 1, ' ac ting</w>': 1, ' s mar ter</w>': 1, ' fa d</w>': 2, ' na mes</w>': 3, ' plan ned</w>': 1, ' ca ta st ro p he</w>': 1, ' rea ction</w>': 2, ' qui ck ly</w>': 1, ' minutes</w>': 3, ' dea d ly</w>': 1, ' ga s k et</w>': 1, ' re ed</w>': 1, ' ma ssi ve</w>': 1, '   pen ta th o l</w>': 1, '   qui ck est</w>': 4, ' ac tive</w>': 3, '   pro te us</w>': 6, '   b en es</w>': 14, '   p le u ra l</w>': 2, '   to p spe ed</w>': 1, '   min i a tu ri z ation</w>': 2, '   co ma to se</w>': 3, '   ar ter i o</w>': 1, ' ven ous</w>': 1, '   fi stu l a</w>': 2, '   ju gu l ar</w>': 5, '   cor pu sc les</w>': 5, '   an ti bo di es</w>': 4, '   uni que ly</w>': 2, '   fro g man</w>': 1, '   bl an ked</w>': 1, '   du v al</w>': 15, '   min i a tu ri z er</w>': 1, '   clo t</w>': 2, '   ar ter</w>': 2, ' com b in ed</w>': 3, ' c m d f</w>': 1, ' con so li da ted</w>': 1, '   s ki ll ful</w>': 5, '   j ar red</w>': 1, '   whi r l po o l</w>': 1, '   fa st en ing</w>': 1, ' s l ow ed</w>': 1, '   a great</w>': 1, ' m r c ro s co pe</w>': 1, '   cha ss is</w>': 1, '   c m d f</w>': 1, '   du v all</w>': 1, '   o cu lo mo tor</w>': 1, '   ar ac h no id</w>': 1, ' re ction</w>': 1, ' pro cla im ing</w>': 1, '   in can de sc ent</w>': 1, '   en do l y mp ha ti c</w>': 1, '   s nor k el</w>': 3, '   al vi o l i</w>': 1, '   o x y gen ation</w>': 1, ' min i a tu ri z ation</w>': 1, '   in v a der</w>': 2, '   re ti cu l ar</w>': 1, '   l y mp ha ti c</w>': 3, '   ti s su es</w>': 3, '   nu cle i</w>': 1, '   s na pp ing</w>': 3, '   p le u r a</w>': 1, '   in ha les</w>': 2, '   ex ha les</w>': 2, '   im pu ri ti es</w>': 2, '   im be d ded</w>': 2, '   spe cks</w>': 2, ' n g</w>': 1, '   in de li ble</w>': 3, ' pre p are</w>': 1, '   o c tu p us</w>': 1, '   pi s ca t ori al</w>': 1, '   sp a w ning</w>': 1, '   can o e</w>': 5, '   la w n m ower</w>': 6, '   f lo ta tion</w>': 1, '   car o ti d</w>': 2, '   ven ous</w>': 1, ' thous an d th</w>': 2, '   por ous</w>': 2, '   op h he m</w>': 1, '   fu ch em</w>': 1, '   ga p he</w>': 1, '   un gh h</w>': 1, '   bra in er d</w>': 10, '   m un ce</w>': 1, '   fe li ci an o</w>': 1, '   no ti sh</w>': 1, '   i sh</w>': 1, '   fa i sh</w>': 1, ' shi k sh</w>': 1, '   hou r z h</w>': 1, '   fuck er z h</w>': 1, ' a ve</w>': 2, '   shi er a</w>': 1, '   je z hu sh</w>': 1, ' un gu ent</w>': 1, '   g ri m sur d</w>': 1, '   un gu ent</w>': 1, '   ge y s er</w>': 1, '   i ds</w>': 2, '   ta ll est</w>': 2, '   ha use</w>': 2, '   ra dis son</w>': 2, '   ta s king</w>': 1, '   ci er a</w>': 9, '   in ta</w>': 4, '   g m ac </w>': 2, '   differ en ti al</w>': 1, ' p in i on</w>': 1, '   tru co at</w>': 6, '   l un de ga ard</w>': 11, '   o x i di z ation</w>': 1, '   mo h r a</w>': 3, '   e ck l un d</w>': 2, '   s we d l in</w>': 2, ' g ee z</w>': 5, '   ten d in</w>': 1, '   o l son</w>': 1, '   pr ou d f oo t</w>': 5, '   l ac s</w>': 1, '   fri ca s see</w>': 1, '   cha s k a</w>': 1, '   le se u re</w>': 1, '   di sp ar i ty</w>': 2, ' tra ding</w>': 1, '   fin an ci al s</w>': 3, '   b le e me e</w>': 1, '   bu ck e yes</w>': 1, '   go ph ers</w>': 2, '   g ro ss man</w>': 5, '   su mp n</w>': 2, '   to o t in</w>': 3, '   ar gu in</w>': 2, '   s ni pp y</w>': 2, '   per pe tra t ors</w>': 1, ' ba be</w>': 1, '   bu n y an</w>': 1, ' bra in er d</w>': 1, '   ma l f ea s an ce</w>': 1, '   cu t la ss</w>': 1, '   e mb ers</w>': 1, '   de i f en b ac h</w>': 1, '   ma je u re</w>': 1, '   f ar go</w>': 2, '   gr ow sh ri es</w>': 1, '   sh ow al ter</w>': 1, '   ga e ar</w>': 1, '   g ri m s ru d</w>': 3, '   whi st l in</w>': 1, '   won t</w>': 2, '   in n ar e sted</w>': 2, '   f di c</w>': 1, '   wa y z at a</w>': 1, '   nor st ars</w>': 1, ' 3 5 </w>': 2, '   4 6 8 5 </w>': 1, '   prob ly</w>': 2, '   d l r</w>': 3, '   poli ce work</w>': 1, '   cra w l ers</w>': 1, '   mon ke y ed</w>': 1, '   hon e y well</w>': 2, '   co ok se y</w>': 4, '   w y n ch a</w>': 1, '   he ck u v a</w>': 1, '   pr ar i e</w>': 1, ' g under son</w>': 1, '   e din a</w>': 1, '   g ee ze</w>': 1, '   do on</w>': 1, '   o l m st ead</w>': 2, '   d int</w>': 1, '   en t an g le men ts</w>': 1, '   re si ding</w>': 1, '   1 4 2 5 </w>': 1, '   men om in i e</w>': 1, '   per pi ch</w>': 1, '   gu st af son</w>': 4, '   su pri se</w>': 2, '   see m d</w>': 1, '   y an a gi ta</w>': 3, '   ha u t man</w>': 1, ' win ged</w>': 1, '   ma ll ard</w>': 1, '   mu s ki es</w>': 1, '   bi t in</w>': 1, '   ha u t man s</w>': 1, '   ar bi e</w>': 1, '   si ber t</w>': 1, '   vi e w point</w>': 1, '   nor st ar</w>': 1, '   di e h l</w>': 1, '   di sc lo sed</w>': 4, '   du r d en</w>': 9, '   la u di sh</w>': 1, ' re ma in ing</w>': 1, '   ru b ber y</w>': 1, '   di ab on o l</w>': 1, '   wi st er o l</w>': 1, '   r ac e h or ses</w>': 1, '   bo d y bu il der</w>': 1, '   pe c</w>': 1, '   e st ro g en</w>': 1, '   te sti cu l ar</w>': 6, '   mo o si e</w>': 1, '   s lo b ber ing</w>': 2, '   ab sen t ee i s m</w>': 2, '   un pre sen ta ble</w>': 1, ' a bu se</w>': 2, '   s na gs</w>': 1, ' pri ori ti ze</w>': 1, ' cor n f l ower</w>': 1, ' f la gs</w>': 1, '   o x a late</w>': 1, '   per ch l ori de</w>': 1, '   f re on</w>': 2, '   tu in al</w>': 1, '   secon al s</w>': 1, '   mar l a</w>': 22, '   ru ra l</w>': 3, '   brea d th</w>': 2, '   je ck le</w>': 1, '   chi ff on</w>': 1, '   sa w ing</w>': 2, '   han i ver</w>': 1, '   ra in es</w>': 5, '   as cen ding</w>': 1, '   l y mp h om a</w>': 1, ' st ri pe</w>': 1, '   fa k er</w>': 4, ' n in ed</w>': 1, '   s ea t b ac ks</w>': 1, '   in de fini te</w>': 1, '   ar i c le</w>': 1, '   vi bra ting</w>': 2, ' thr ow ers</w>': 1, '   me lan om a</w>': 3, '   ton gu ing</w>': 1, '   s qu int</w>': 3, ' par a si te</w>': 1, '   cha k r a</w>': 1, '   i ke a</w>': 2, '   fa il in gs</w>': 2, '   p sy cho gen i c</w>': 1, '   fu gu e</w>': 1, ' i in n</w>': 2, '   ff</w>': 2, ' n n y in</w>': 2, '   imp li c it</w>': 1, ' may he m</w>': 1, '   mar t y r do m</w>': 1, '   3 7 </w>': 4, '   ni t ro g l y cer in</w>': 2, ' tra in ing</w>': 1, ' app li c ant</w>': 1, ' pro je ct</w>': 1, '   he s se l</w>': 3, '   ne w ca st le</w>': 2, '   pen n s</w>': 1, '   gu i ded</w>': 13, '   g l y cer in</w>': 1, '   li po su ction</w>': 2, ' a li g ned</w>': 1, '   sti r ru ps</w>': 1, '   gh an d i</w>': 1, '   sha t ner</w>': 3, '   con cu ssi ve</w>': 1, ' can dle</w>': 1, '   b la sts</w>': 1, ' ve lo ci ty</w>': 1, '   dis bur se ment</w>': 1, '   o c cu p ant</w>': 1, '   fran chi ses</w>': 1, '   s nee z ed</w>': 1, '   en di ve</w>': 1, '   af for ds</w>': 1, ' chan ge over</w>': 1, ' ci gar e tt e</w>': 2, '   wi de sc re en</w>': 1, ' co b al t</w>': 1, ' e b on y</w>': 1, ' fu ch si a</w>': 1, '   ar m cha ir s</w>': 1, '   sh el ving</w>': 1, '   din e tt e</w>': 1, '   sp or k</w>': 1, ' li tter</w>': 1, '   ac comp li sed</w>': 1, '   s mo l der ing</w>': 1, '   pa u l son</w>': 4, '   p le ee ee ea se</w>': 1, '   ir v ine</w>': 3, '   co ll ar b one</w>': 1, ' ter ms</w>': 1, '   ye ss s ss</w>': 2, '   1 3 2 0</w>': 1, '   b en ning</w>': 1, '   k al ar j i an</w>': 3, '   pre mon i tion</w>': 6, '   i sa be ll a</w>': 15, '   r ory</w>': 3, ' h or ri ble</w>': 2, '   sha e ff er</w>': 3, '   con gra tu fuck in gla tions</w>': 1, '   bur r ou gh s</w>': 1, '   on ra mp</w>': 3, '   mor ti ci an</w>': 2, '   da y ton a</w>': 3, '   dan o</w>': 3, '   cl un k er</w>': 2, ' he b re w</w>': 1, '   v ans</w>': 2, ' pi le</w>': 1, '   sha in a</w>': 2, '   th a a a a</w>': 1, '   ch ev e tt e</w>': 1, '   su n b lo ck</w>': 2, '   mon go li a</w>': 2, '   bl an e</w>': 1, '   le w ton</w>': 4, '   om en s</w>': 2, '   f lan ne ls</w>': 1, '   pla i ds</w>': 1, '   li k en ess</w>': 1, ' inter ven tion</w>': 1, ' st e p mo m</w>': 1, '   re f le x i ve</w>': 1, '   o c cu r ing</w>': 1, '   co in ci den c ess</w>': 1, '   mi sha ps</w>': 1, '   cu ti c le</w>': 1, '   wh in in</w>': 1, '   fuck n</w>': 1, '   min i gu n</w>': 2, '   mi l</w>': 4, ' see king</w>': 1, '   b re w er</w>': 6, '   ga w d</w>': 4, '   r oun de ye</w>': 1, '   sc run ch ed</w>': 1, ' da u</w>': 1, '   da u</w>': 1, '   ban g k ok</w>': 8, '   cu pped</w>': 1, '   si on i c s</w>': 1, '   supp re ss or</w>': 1, '   tr ac or</w>': 1, '   l ac </w>': 1, '   ki a</w>': 2, '   p hu ong</w>': 2, '   ba o</w>': 3, '   k li cks</w>': 1, '   tr ans at</w>': 2, ' v u l n er ab o</w>': 1, '   n gu y en</w>': 4, ' d y na st y</w>': 1, ' v u l</w>': 1, ' n er ab o</w>': 1, '   k in h</w>': 1, '   cu ong</w>': 1, '   0 5 0 0</w>': 1, '   g un shi ps</w>': 2, '   inter n ment</w>': 1, '   lan d s at</w>': 1, '   din ks</w>': 3, '   tra u t man</w>': 1, '   te h r an</w>': 1, ' ter min ate</w>': 1, ' com mi tt e e</w>': 2, ' p on y</w>': 2, '   di re c ti ves</w>': 1, '   di r t wa ter</w>': 1, '   su b com mi tt e e</w>': 1, '   f ra vi o</w>': 2, '   do g pa tch</w>': 1, '   w ra pp er</w>': 3, ' j er r i</w>': 1, '   pla tt ers</w>': 1, ' di v or ce</w>': 1, ' co f fe e</w>': 2, '   te qui ll a</w>': 1, '   c r ust</w>': 5, ' pa in</w>': 4, ' who ah</w>': 1, '   be g in ing</w>': 1, ' par ry</w>': 2, ' l y di a</w>': 1, ' bo oo o or r ing</w>': 1, '   ta w k</w>': 1, '   par an o i ac s</w>': 1, ' wh ad da</w>': 1, ' ex per i ence</w>': 1, ' ex per i en c ing</w>': 1, '   ca ta ton i a</w>': 1, '   we in tra u b</w>': 2, ' ca ta ton i c</w>': 1, '   stu p or</w>': 3, ' ver ba l</w>': 1, '   b ru sh er o o</w>': 1, '   e d w in</w>': 5, ' p in no chi o</w>': 1, ' e d w in</w>': 1, '   fa ir y ta le</w>': 2, ' y u pp i e</w>': 2, '   bab bi t t</w>': 1, ' ni gh t c lu b</w>': 1, '   re v u es</w>': 1, '   gra du al</w>': 2, '   su m mer time</w>': 2, '   de bu t an te</w>': 2, ' le ts</w>': 1, '   ex cla ma tion</w>': 2, '   br an</w>': 2, ' ra ou l</w>': 1, '   sta in less</w>': 1, '   si de l ine</w>': 2, '   la min ate</w>': 2, '   na poli t an o</w>': 1, '   con j un ction</w>': 1, '   f l oun cy</w>': 1, ' gi ves</w>': 1, '   gra i ls</w>': 1, '   gu in ev ere</w>': 2, ' ear n</w>': 1, ' in no c ent</w>': 1, ' te ars</w>': 1, '   in v in ci bi li ty</w>': 1, ' ra di cal</w>': 1, '   ex i stan ce</w>': 1, '   a v a il</w>': 1, '   v ying</w>': 1, '   s killed</w>': 6, ' sing</w>': 2, '   be d stand</w>': 1, '   f la pp in</w>': 1, '   as sed</w>': 2, '   cl ou d bu st ing</w>': 1, '   de bu t an tes</w>': 2, '                                                     </w>': 17, '                                                 </w>': 21, '                                                         </w>': 8, '                                                           </w>': 14, '                                                                     </w>': 11, '                                                       </w>': 19, '                                               </w>': 13, '   j ac ck</w>': 1, '   lan g don</w>': 1, '   car mi cha el</w>': 1, ' so ver i e g n</w>': 1, '   cap tive</w>': 5, '   af f li ction</w>': 7, '   fe tt er ed</w>': 1, '   ri g or ous</w>': 3, '   de i g n</w>': 1, '   en du r es</w>': 1, '   en ra ge</w>': 1, ' tch</w>': 3, '   s sh h h h</w>': 1, '   d om i ci le</w>': 1, '   ha ha ha ha a a</w>': 1, '   n on on on o</w>': 1, '   ha ha ha h a</w>': 1, ' war r en</w>': 1, '   be se ts</w>': 1, ' fa g</w>': 2, ' cu r st</w>': 1, ' th y</w>': 1, ' fu ry</w>': 1, ' in w ard</w>': 1, ' th y self</w>': 1, ' con su me</w>': 1, ' the e</w>': 1, '   be gone</w>': 3, '   ru th s</w>': 1, '   un co or din a ted</w>': 1, '   e m pi r es</w>': 3, '   tra sh y</w>': 3, '   g y p</w>': 1, '   gar lan ds</w>': 1, '   z bi e g new</w>': 1, '   spe i z a k</w>': 1, '   por tra y ed</w>': 2, '   tw in k y</w>': 3, ' tw in k y</w>': 1, ' ta tion</w>': 1, '   ra ye tt e</w>': 2, '   ti ta</w>': 10, '   di pe s to</w>': 1, '   ser ve z a</w>': 1, '   cer ve z a</w>': 2, '   gen ti li ty</w>': 1, ' s ound</w>': 3, '   o of h</w>': 4, '   a po d ac a</w>': 1, '   su b sti tu tions</w>': 1, '   t an de m</w>': 1, '   y v </w>': 1, '   o ver sta y ed</w>': 2, '   app le sa u ce</w>': 1, '   g in ger brea d</w>': 1, '   co ac h ing</w>': 3, '   fi d d l er</w>': 2, '   oo st</w>': 2, '   wal d n it</w>': 1, '   sch ne ch ter</w>': 1, '   pre w ar</w>': 1, '   2 7 5 </w>': 1, '   er o i c a</w>': 1, '   c el li st</w>': 1, '   mu si ca lly</w>': 1, '   im per ce p ti ble</w>': 1, '   re v u e</w>': 2, '   h y dro ther a p y</w>': 1, '   in vi g or a ted</w>': 1, '   ki tt y c at</w>': 1, '   bor z o i</w>': 1, '   do g g y</w>': 4, '   in tru d in</w>': 1, '   de l y on</w>': 1, '   be ten th a ll er</w>': 1, '   da mp en ed</w>': 1, '   sa mi a</w>': 2, '   da mp en</w>': 1, ' ob je c tive</w>': 1, '   ex c lu de</w>': 2, ' m ack</w>': 6, '   wai t re ssed</w>': 1, '   co ll e c t in</w>': 1, ' ma ss</w>': 1, ' s qu a sh ed</w>': 1, '   ju x ta po sed</w>': 1, '   so p ori fi c</w>': 2, '   du pe a</w>': 1, '   ch u</w>': 1, '   a int</w>': 1, '   pro v in</w>': 2, '   wh ad j a</w>': 2, '   w u s su p</w>': 4, '   bar u ch</w>': 2, '   pi t t</w>': 3, '   ph at</w>': 1, '   ar</w>': 4, '   sa l s a</w>': 3, '   br ow ni sh</w>': 1, '   gir l fr l end</w>': 1, '   stan w y k</w>': 26, '   pro v o</w>': 8, '   bu cking</w>': 3, '   han ra ha n</w>': 1, '   s ca pe go a ts</w>': 1, '   im pro pri e ti es</w>': 2, '   ca v an au gh</w>': 6, '   f le tch</w>': 31, '   wh ad d y a mean</w>': 1, '   min ori ti es</w>': 1, '   la sor da</w>': 3, '   pen n y an te</w>': 1, '   cu m min gs</w>': 6, '   ma u ve</w>': 1, '   de cor ate</w>': 3, ' f oo l pro of</w>': 1, '   ba s king</w>': 1, '   gu m my</w>': 7, '   he br on</w>': 1, ' ki d ne ys</w>': 1, '   da h h h</w>': 1, '   bab ar</w>': 4, '   b s</w>': 1, '   bri gh ten s</w>': 1, '   bi op sy</w>': 1, '   car c in om a</w>': 2, '   no ma</w>': 1, '   pi ll ory</w>': 1, '   ro sen pen is</w>': 1, '   gar ni sh</w>': 1, '   re mar ri es</w>': 1, '   ma in ta ins</w>': 1, '   mon ty</w>': 5, '   ja il able</w>': 1, '   mar ri o t t</w>': 4, '   bi ga mi st</w>': 2, '   s war th out</w>': 5, '   nu g ent</w>': 7, '   see m ing</w>': 3, '   v el ma</w>': 4, '   in sur er</w>': 1, '   ca se well</w>': 1, '   under wri t ers</w>': 1, '   m our n ful</w>': 1, '   u l tra ma l en s k y</w>': 3, '   su f i</w>': 2, '   de par ts</w>': 2, '   dri f t ers</w>': 1, '   dis ar ra y</w>': 2, '   nu lli f y</w>': 1, '   ho p al ong</w>': 2, '   ca ssi dy</w>': 5, '   un sor ted</w>': 1, '   bur ner</w>': 3, ' stan w y k</w>': 1, '   b la h b la h b la h</w>': 1, '   do l en</w>': 1, ' t an w y k</w>': 1, ' a gi c</w>': 2, ' ir w in</w>': 2, '   under hi ll</w>': 6, '   de f ra u ding</w>': 1, '   z n h c ne el s k y</w>': 1, '   f l y boy</w>': 1, ' ru man i an</w>': 1, '   u l tr ar e la men s k y</w>': 1, '   per i g n on</w>': 3, '   ther mi d or</w>': 1, '   be lu g a</w>': 4, '   under hi lls</w>': 3, '   u r in al y s is</w>': 1, '   t ro pi can a</w>': 3, '   sp on sor ship</w>': 1, '   we d ged</w>': 2, ' pr ac ti ca lly</w>': 1, '   fri e da</w>': 3, ' ne y</w>': 1, '   f ac t ori es</w>': 5, '   ca pp u c in o</w>': 2, '   d ru mm ers</w>': 1, '   z u z u</w>': 22, '   fa ir lan e</w>': 7, '   mo on ey</w>': 7, '   as su age</w>': 1, '   qu al ms</w>': 2, '   ro se an n e</w>': 1, '   bar r</w>': 1, '   war ps</w>': 1, '   of f ends</w>': 2, '   sh e t land</w>': 1, '   s l o</w>': 1, ' ow ly</w>': 1, '   di sc s</w>': 5, ' u re</w>': 1, '   ti z z</w>': 1, '   de c r y p tor</w>': 1, '   comp r en do</w>': 1, '   sp ex </w>': 1, '   b lu e s man</w>': 1, '   sta h ting</w>': 1, '   f le es</w>': 1, '   mo on e ys</w>': 1, '   sy n ch r on i ci ty</w>': 1, '   p in z o l o</w>': 1, ' c run ch</w>': 1, '   hou se bo at</w>': 2, '   ear th men</w>': 1, '   a ir wa ves</w>': 1, '   ca v al ca de</w>': 1, '   bi m bo s</w>': 2, '   d ori to s</w>': 1, '   m ou la h</w>': 1, '   wa m pu m</w>': 4, ' c ro co di le</w>': 1, '   d un de e</w>': 1, '   au stra li ans</w>': 2, '   s lea z e ba g</w>': 4, '   s ca l p ing</w>': 1, '   ho h</w>': 1, ' star ts</w>': 1, ' v an cou ver</w>': 1, '   ar t ar t ar t ar t mo on e y mo on e y mo on ey</w>': 1, '   a v a</w>': 1, '   mo ga mb o</w>': 1, '   f lu cking</w>': 1, ' g l o</w>': 1, '   or ga s mi c</w>': 1, '   f lu ck</w>': 2, '   m ou s se</w>': 1, ' ro ll</w>': 3, ' f l</w>': 3, '   ha ir head</w>': 1, '   re se da</w>': 1, '   ri b b it</w>': 1, '   ce z an n e</w>': 1, '   ar che o lo gi st</w>': 2, ' den ti st</w>': 1, '   sc re w dri ver s</w>': 3, '   ba j a</w>': 1, '   stra to ca ster</w>': 1, '   hu m bu cking</w>': 1, '   k re s kin</w>': 1, ' cu ss</w>': 1, '   ra in dro p</w>': 1, ' in ch</w>': 10, ' bo o ty</w>': 1, '   pu b li ci st</w>': 5, '   un fuck in g beli ev able</w>': 1, '   cor sa g es</w>': 1, '   di an e ti c s</w>': 1, '   s na pp er head</w>': 1, '   s l ow est</w>': 1, '   pu ss b all</w>': 1, '   in gra tes</w>': 1, ' bi ll bo ard</w>': 1, '   ta st e ful</w>': 2, '   un sc ra mb les</w>': 1, ' wi ck</w>': 1, '   bo ing</w>': 2, '   com pu ter i sed</w>': 1, '   he be de e bu h</w>': 2, '   re d ho ok</w>': 1, '   go in na</w>': 1, '   h en r i</w>': 4, '   a la in</w>': 4, ' ci d</w>': 1, ' ac ce p ter</w>': 1, '   gen til</w>': 2, ' t re</w>': 2, '   ven u</w>': 1, ' sen te</w>': 1, '   as so c i</w>': 1, '   ni co l i</w>': 2, '   d ever ea u x</w>': 1, '   char ni er</w>': 5, '   o pp ort un e</w>': 1, '   gen dar me</w>': 1, '   s om mes</w>': 1, '   n ous</w>': 3, '   ja ma is</w>': 2, '   t ant</w>': 2, ' r en it</w>': 1, '   tes</w>': 1, '   n ou ve ll es</w>': 1, ' tions</w>': 1, '   he u re u x</w>': 1, ' e ll e</w>': 1, '   to i</w>': 3, ' ha bi ll er a is</w>': 1, '   do ck er</w>': 2, '   su is</w>': 1, '   vo ir</w>': 1, ' o i se</w>': 1, '   re gar de</w>': 2, '   par fa i te ment</w>': 1, '   a ve c</w>': 4, '   p our ra s</w>': 1, ' che u r</w>': 1, '   ba le ine</w>': 1, '   sa is</w>': 4, '   t r</w>': 2, '   hi ver</w>': 1, '   mer ve i ll e u x</w>': 1, ' tes</w>': 3, ' a i me</w>': 1, '   att ends</w>': 2, '   v a is</w>': 1, '   mon tr er</w>': 1, '   au ss i</w>': 1, '   ce</w>': 1, '   ac h et</w>': 1, '   ve u x</w>': 2, '   pe u x</w>': 1, ' ou v ri r</w>': 1, '   t out</w>': 1, '   lon gu e ment</w>': 1, '   ca dea u</w>': 1, '   cho i s i</w>': 1, '   ti en s</w>': 1, ' ca u ti ous</w>': 1, '   so is</w>': 1, '   lu i</w>': 3, '   sa it</w>': 1, '   pe u t</w>': 2, '   tra v a i ll er</w>': 1, ' vi sion</w>': 2, '   p as</w>': 5, '   con fi an ce</w>': 1, '   er re u r</w>': 2, ' ni al</w>': 1, '   ve de tt e</w>': 1, '   a ll er</w>': 1, '   part out</w>': 1, ' on n</w>': 1, '   be so in</w>': 2, '   fri c</w>': 1, '   c ro is</w>': 1, '   pr en d re</w>': 1, '   fa ll a it</w>': 1, '   b ou lot</w>': 1, '   b ow st r ing</w>': 1, '   4 0 8 </w>': 1, '   g an a po lo s</w>': 1, '   j i lly</w>': 1, '   mu tch</w>': 1, ' v a li se</w>': 1, '   we st bu ry</w>': 2, '   co ll ar ed</w>': 1, '   ti me z it</w>': 1, '   mu l der i g</w>': 1, '   we in sto ck</w>': 1, '   fr ac i s i</w>': 1, '   g rea s er</w>': 1, '   ha ck en s ack</w>': 1, '   sp en d ers</w>': 1, '   b on w it</w>': 1, '   chri st man s</w>': 1, '   he ll ga te</w>': 1, '   au c tion ed</w>': 1, '   cra w l in</w>': 2, '   fin e ll i</w>': 6, '   dar y l</w>': 10, '   x a vi er</w>': 8, '   de le on</w>': 3, '   bu l k head</w>': 3, '   gi b by</w>': 2, '   gi b</w>': 3, '   f lo or bo ard</w>': 1, '   bu tch y</w>': 1, '   bo lo g ne se</w>': 1, '   6 5 </w>': 4, '   c p w</w>': 1, '   sa tch</w>': 13, '   we is</w>': 2, '   s w o bo da</w>': 2, '   cl en den on</w>': 2, '   cle on</w>': 2, '   4 1 5 </w>': 1, ' sa tch</w>': 1, '   pe da ling</w>': 2, ' ju li a</w>': 1, ' 1 9 9 8 </w>': 1, '   g li mm er</w>': 1, ' chan ged</w>': 3, '   li gh ten ing</w>': 3, '   t y r in</w>': 1, '   1 9 9 8 </w>': 3, '   1 0 6 0</w>': 1, ' y x b</w>': 2, '   g or dy</w>': 2, '   bu for d</w>': 2, ' w b</w>': 1, '   s ea ver</w>': 1, ' b and</w>': 1, '   ba y side</w>': 1, '   w b</w>': 1, ' y k x b</w>': 1, '   lon g b ran ch</w>': 1, '   bu x ton</w>': 1, '   n in ten do</w>': 7, '   par o l ed</w>': 4, '   s mar ten ed</w>': 1, '   b c i</w>': 1, ' re por ted</w>': 1, '   sor r ys</w>': 1, '   re ti r es</w>': 4, '   at t</w>': 2, '   d y ck man</w>': 1, '   a be l</w>': 2, '   wh ac ks</w>': 1, '   fu se bo x</w>': 1, '   go l di lo cks</w>': 1, '   bu ll se ye</w>': 1, '   fuck mo bi le</w>': 1, ' re si d ence</w>': 2, '   pe ll e ted</w>': 1, '   qu o ta</w>': 10, '   j ar v is</w>': 4, '   pa d lo cks</w>': 2, '   d ra w in</w>': 3, '   ro ad b lo cks</w>': 5, ' me g an</w>': 1, '   de ba ta ble</w>': 1, ' vi sit</w>': 1, '   li z be th</w>': 1, '   ha w es</w>': 1, '   cre ma te</w>': 2, '   re pa int</w>': 2, '   ca se fi le</w>': 1, '   vo or he es</w>': 10, ' sa l ty</w>': 1, ' b oun ty</w>': 1, '   b in go s</w>': 1, '   mi sc ar ri age</w>': 7, ' f ell</w>': 2, '   ma gi ca lly</w>': 2, '   ta mar a</w>': 2, ' vo or he es</w>': 1, '   m c cu ll o ch</w>': 3, '   n ar c ing</w>': 1, '   te en sy</w>': 2, '   re c rea tion al</w>': 4, '   un de f ea ted</w>': 2, '   r en ni e</w>': 18, '   w o l fe</w>': 3, '   sta ter oo m</w>': 3, '   ro ber t son</w>': 3, '   de u s en</w>': 3, '   d on al d son</w>': 1, ' i ma g in ing</w>': 1, '   in be tw e en</w>': 1, '   in gen u ou s ne ss</w>': 1, '   l or an</w>': 2, '   bar ri ster</w>': 1, '   win d sc re en</w>': 1, '   in ha le</w>': 8, '   t in fo il</w>': 1, '   n on stop</w>': 1, '   pla y gr oun ds</w>': 1, '   ne u ro sur ger y</w>': 1, '   pu tt an e sc a</w>': 2, '   li t v a k</w>': 2, '   ro g an</w>': 4, '   e know</w>': 1, '   un man a g ea ble</w>': 1, '   si l en tly</w>': 3, '   e for</w>': 1, '   n ought</w>': 2, '   ru mb ling</w>': 1, '   sch wi mm er</w>': 4, '   thous an d f old</w>': 1, '   ex u de</w>': 3, '   st un ned</w>': 2, '   p be</w>': 1, '   ex ter min a ting</w>': 2, ' c ri ti c</w>': 1, '   re d mon d</w>': 3, '   lu ke war m</w>': 2, '   al b right</w>': 1, '   un for gi ving</w>': 1, '   se tt le men ts</w>': 1, '   i ran i ans</w>': 1, '   inter mi ssion</w>': 1, '   pa u ses</w>': 3, '   im pe c ca b ly</w>': 1, '   e la ted</w>': 1, '   we t z el</w>': 3, '   s ca ll op s</w>': 1, '   st ea m er</w>': 2, '   fa ta li s m</w>': 1, '   n have</w>': 1, '   man ni on</w>': 2, '   do d gi e</w>': 2, '   r or ty</w>': 1, '   t my</w>': 1, '   hu man i ze</w>': 1, '   n work</w>': 1, '   out li ves</w>': 1, '   la z z ar o</w>': 3, '   ro x bu ry</w>': 1, '   af gh ani st an</w>': 4, '   u p da tes</w>': 1, '   c ru m pl es</w>': 1, '   bo y le st on</w>': 1, '   f en way</w>': 2, '   s ca b by</w>': 1, '   wa sh ou ts</w>': 1, '   n com mi tt ed</w>': 1, '   a y be</w>': 1, ' pa l</w>': 1, ' m ac be th</w>': 1, '   ye ssi ree</w>': 2, '   j re pe at</w>': 1, '   si en na</w>': 2, '   cra y on</w>': 2, '   fe l der</w>': 1, '   f ab ri k ant</w>': 1, '   o d d ne ss</w>': 1, '   ex t re mes</w>': 1, '   ba tt l ed</w>': 1, '   en de ar ing</w>': 3, '   pu t an e sc a</w>': 1, '   thin ne st</w>': 2, '   wi sp y</w>': 1, '   ma i ms</w>': 1, '   sor a</w>': 1, '   j in na h</w>': 4, '   pre d om in an tly</w>': 1, ' ba p u</w>': 1, '   pu ri t ani s m</w>': 1, ' f ar m</w>': 2, '   di le m ma s</w>': 1, '   cl er g y man</w>': 3, '   por ban d ar</w>': 1, '   ba as</w>': 2, '   p an di t j i</w>': 1, '   un ci vi l</w>': 1, '   si k h s</w>': 1, '   p ran a m i</w>': 1, '   k or an</w>': 1, ' com m uni ty</w>': 1, ' vi ll age</w>': 1, ' a sh ra m</w>': 1, ' re sp on se</w>': 1, '   s mu ts</w>': 2, '   mo s qu e</w>': 1, '   ma st er y</w>': 2, '   fa st ing</w>': 2, '   k a ll en b ac h</w>': 1, '   ma st er ing</w>': 1, '   se di ti ous</w>': 1, '   ne h ru </w>': 4, '   s no o t</w>': 1, '   pa te l</w>': 7, '   de fi ed</w>': 1, ' in ve sti ga te</w>': 1, '   re p ea l ed</w>': 2, '   re p ea l</w>': 2, ' ex ce p tion</w>': 1, '   din ed</w>': 1, '   f la tly</w>': 1, '   d har as an a</w>': 1, ' de cl ar ation</w>': 1, '   ba p u</w>': 2, '   ca l cu tta</w>': 2, ' dis co ver ing</w>': 1, ' myself</w>': 2, ' e mb r y o</w>': 1, '   c ri ck</w>': 2, '   c rea ti vi ty</w>': 2, '   pro fi les</w>': 3, '   wor k for ce</w>': 1, ' gen e</w>': 2, ' er ate</w>': 1, '   ga tt ac a</w>': 2, ' bor r ow ed</w>': 1, '   ca ter</w>': 3, ' op er ate</w>': 4, ' v a li d</w>': 2, '   fe lon i es</w>': 1, ' li ke ly</w>': 2, ' de fi ci en ts</w>': 1, '   pro c li vi ty</w>': 1, '   b lu d ge on</w>': 1, '   un ac coun ta ble</w>': 2, ' uni ver sa lly</w>': 1, ' b ac ks</w>': 2, '   a li b is</w>': 4, '   me ti cu l ous</w>': 1, '   s w ea the art</w>': 1, ' e u gen e</w>': 1, '   g ro om ed</w>': 1, '   pl u cking</w>': 1, '   ho o ver s</w>': 2, '   na vi ga t ors</w>': 1, '   sh ow er ing</w>': 2, '   e d g ars</w>': 1, '   b en ev o l ence</w>': 1, '   no one</w>': 1, '   s ou th pa w s</w>': 2, ' han der</w>': 1, '   dis ow n s</w>': 1, ' dis ea ses</w>': 1, '   pre ju di ci al</w>': 1, '   my op i a</w>': 2, '   su sc e p ti bi li ty</w>': 1, '   o be si ty</w>': 2, '   cl on es</w>': 2, '   au stra la si a</w>': 1, ' pro du c tive</w>': 1, ' v in c ent</w>': 1, ' sc ar es</w>': 1, ' cou ra ge ous</w>': 2, ' no se</w>': 3, '   o v u m</w>': 1, ' ra ther</w>': 2, ' fif ti es</w>': 1, ' wee k ly</w>': 2, '   thin n ers</w>': 1, ' un for t un a tely</w>': 1, '   c r ack er bo x</w>': 1, '   ra pp in</w>': 2, '   u t</w>': 1, '   be y n on</w>': 9, '   a ir fi e ld</w>': 5, '   ja i ls</w>': 5, '   h un t s vi ll e</w>': 1, '   be y no t r</w>': 1, '   to days</w>': 1, '   go lli e</w>': 2, '   br ac ed</w>': 2, '   mor mon s</w>': 2, ' hi ve</w>': 3, '   or em</w>': 1, '   d y na mi ted</w>': 2, ' ought</w>': 2, '   in vi c ta</w>': 1, '   3 1 8 </w>': 1, '   du ff</w>': 36, '   b ru mb y</w>': 10, '   pa ke</w>': 1, '   n ow t</w>': 2, '   k in ne ar</w>': 8, '   ar ca d es</w>': 1, '   f lo gs</w>': 1, '   p an ti les</w>': 1, '   th or pe y</w>': 7, '   do d g y</w>': 2, '   every h ing</w>': 1, '   wi l ton</w>': 1, '   w oo l wor th s</w>': 1, '   we st se a</w>': 1, '   th or pe</w>': 1, '   att en tive</w>': 2, ' ru le</w>': 1, '   bri t an ni a</w>': 1, '   t wi g g y</w>': 1, '   no sy</w>': 5, '   ber ea ve ment</w>': 2, '   pa ice</w>': 1, '   su e de</w>': 3, '   sta mp ing</w>': 1, '   bo ll o ck</w>': 1, '   gra ff</w>': 18, ' ro b ber y</w>': 2, '   g l en gar ry</w>': 3, ' bu ck</w>': 4, '   en s la ve</w>': 2, '   th ra ll</w>': 1, ' sa les</w>': 3, ' se ll</w>': 4, ' c ower</w>': 1, '   ab </w>': 7, '   ab so l u</w>': 1, '   gra p ev ine</w>': 4, '   su per ci li ous</w>': 1, ' pa te l</w>': 1, '   go an</w>': 1, '   sh ou n</w>': 1, '   dea d bea ts</w>': 5, ' po l ac ks</w>': 1, '   po l ac ks</w>': 3, ' h t m l</w>': 1, '   in u red</w>': 1, '   re clo se</w>': 4, ' wh a</w>': 4, '   l ev en e</w>': 6, '   har ri e t t</w>': 3, '   n y bor g</w>': 3, '   sus</w>': 2, ' co op er ate</w>': 1, '   l ev </w>': 1, '   ch in ks</w>': 3, '   sh el</w>': 5, '   mu r r ra y</w>': 1, '   le m kin</w>': 1, '   lin g k</w>': 10, '   f r</w>': 2, '   t y pe wri t ers</w>': 2, '   a ar on ow</w>': 1, ' mo ss</w>': 3, ' wi lli a m son</w>': 2, '   har ri et</w>': 5, '   de lu de</w>': 1, '   tw en</w>': 3, '   c ust</w>': 1, '   te l</w>': 3, ' mu r</w>': 1, '   pla in es</w>': 2, '   ac qu ir es</w>': 2, '   de f ea ti st</w>': 2, ' a part</w>': 1, '   ge t z</w>': 2, '   h ome st ead</w>': 2, '   b en e fi ted</w>': 1, '   sen ny</w>': 2, '   an ta gon i ze</w>': 3, ' mar sha l</w>': 1, '   mar sha ling</w>': 1, '   or g</w>': 1, '   se vi ll e</w>': 3, ' lu ck</w>': 1, '   sta ts</w>': 3, '   in v a li da ted</w>': 2, '   ja go ff</w>': 2, '   gen i un e</w>': 1, '   re st a</w>': 1, '   k en il wor th</w>': 4, '   s co o t</w>': 2, '   i ss</w>': 1, '   lin g ks</w>': 1, '   j in ny</w>': 1, '   pla ts</w>': 1, '   k en il w</w>': 1, ' k en il wor th</w>': 1, ' mar sha ling</w>': 1, '   pu r lo in ed</w>': 2, '   wi l ted</w>': 1, ' con ver t</w>': 1, '   jo in tly</w>': 2, '   re go ti ate</w>': 1, '   ye st</w>': 1, '   gi m let</w>': 1, '   la it</w>': 2, '   ga ins</w>': 5, '   com part men ts</w>': 8, '   sh e ean</w>': 1, '   in ge sted</w>': 1, '   p ea ch fu z z</w>': 1, '   as sur ing</w>': 1, '   vi sh n u</w>': 1, '   ra vi da m</w>': 1, '   th ses</w>': 1, '   w o gs</w>': 2, ' no st al gi a</w>': 1, '   re sta u r a</w>': 1, ' lin g k</w>': 1, '   ci c c i</w>': 3, '   1 9 2 7 </w>': 1, ' gen c o</w>': 1, '   mar con i</w>': 1, '   gar i b al d i</w>': 1, '   in for ms</w>': 1, '   f an u c c i</w>': 8, '   mar an z a ll a</w>': 2, '   pi s an i</w>': 1, '   bu f al in o</w>': 1, '   vi t one</w>': 4, '   con stan z i a</w>': 1, ' ti es</w>': 1, ' what s a</w>': 2, '   c r in ge</w>': 2, '   fran ce sc o</w>': 1, '   de sti tu te</w>': 2, '   pa i s an</w>': 1, ' i t down</w>': 1, ' can a p es</w>': 1, ' an ge ls</w>': 5, '   que sta d t</w>': 2, '   o l a</w>': 8, '   h y man</w>': 8, '   sin cer o</w>': 1, '   be d sh ee ts</w>': 1, ' su per man</w>': 2, '   j ee ze</w>': 1, '   ro sa to</w>': 12, '   f ra te llo</w>': 1, ' r at</w>': 1, ' pa tr one</w>': 1, '   for s we ar</w>': 2, ' con ne c ted</w>': 3, '   n ea p on i t an</w>': 1, '   ju sti fi ca tion</w>': 3, ' ven ge</w>': 1, '   bar re tt s</w>': 1, '   ru bi c on</w>': 1, '   i do li z es</w>': 1, '   hou st an</w>': 3, '   re lu ca t an ce</w>': 1, ' pre si den cy</w>': 1, ' ter min al</w>': 1, '   re pri sa ls</w>': 1, '   bu s se tta</w>': 1, '   ta s ks</w>': 1, '   dis cu ss es</w>': 1, '   se t b ac ks</w>': 1, '   bu en o s</w>': 4, '   a ir es</w>': 4, '   p ou ch</w>': 1, '   p lo tt ers</w>': 2, ' gi mes</w>': 1, ' ca po s</w>': 1, ' o l di ers</w>': 1, ' pro vo king</w>': 1, '   om er ta</w>': 1, '   k lin g man</w>': 5, '   sig n po st</w>': 1, '   p an t an ge l i</w>': 1, '   ba ti st a</w>': 1, '   ro sa to s</w>': 2, '   la k ev i ll e</w>': 1, '   f al se ho od</w>': 1, '   con so li da te</w>': 2, '   ne f ar i ous</w>': 2, '   1 9 4 7 </w>': 1, '   e st ee med</w>': 2, '   pre ce ding</w>': 2, '   ac qui e sc ed</w>': 1, ' guys</w>': 2, '   spi cks</w>': 1, '   ne i gh bor ho o ds</w>': 1, '   son su v bi tch es</w>': 1, '   w el ch ed</w>': 1, '   app li es</w>': 2, '   tur n bu ll</w>': 2, '   fu ci llo</w>': 1, '   der o s a</w>': 1, '   par i</w>': 4, '   per su </w>': 3, '   br ac ci o le</w>': 1, '   sha med</w>': 3, '   han na</w>': 16, '   fi z z le</w>': 1, '   ye ar ned</w>': 1, ' ling</w>': 1, '   re en li st</w>': 1, ' ne ar si gh ted</w>': 1, '   s ke tch es</w>': 7, '   me men to</w>': 7, '   im mo de st</w>': 2, '   ki lt</w>': 4, '   ba g g y</w>': 3, '   cu k or</w>': 5, ' wa g</w>': 2, '   po of</w>': 5, '   be tt ers</w>': 2, '   bl in d ly</w>': 3, '   cha st e ly</w>': 1, '   c ri mean</w>': 1, '   fo x ho les</w>': 2, '   a the i sts</w>': 1, '   t ea m ma tes</w>': 4, '   ba sh ful</w>': 2, '   out la sted</w>': 1, '   s la p sti ck</w>': 1, '   p in n ac le</w>': 1, '   un bu tt on</w>': 2, '   ci en e g a</w>': 1, '   gi b es</w>': 1, '   c ro ck er y</w>': 1, '   ro a sts</w>': 3, '   sc ru t in i ze</w>': 1, '   sh y ne ss</w>': 2, '   pi sh</w>': 3, '   ti lly</w>': 1, '   under shi r t</w>': 1, '   ex pre ssi ve ly</w>': 1, '   ar chi te c tu ra l</w>': 1, '   s ke tch ed</w>': 2, '   f loo ey</w>': 2, '   sc or ch er</w>': 1, '   ne c ro p hi li a</w>': 1, '   lu min al</w>': 2, '   o l f ac t ory</w>': 1, ' ci r cu i ted</w>': 1, '   pa y n e</w>': 6, '   un im pa i red</w>': 1, '   r in</w>': 1, '   ha ir dre ss ers</w>': 1, '   vi sa ge</w>': 2, ' bri mm ed</w>': 1, '   fe d or a</w>': 1, '   y ar d man</w>': 3, '   du ll est</w>': 1, '   po of s</w>': 8, '   da y bed</w>': 1, '   mu ch ly</w>': 1, '   t in ned</w>': 1, '   b al d ly</w>': 2, '   gu i g no l</w>': 1, ' j our ne y</w>': 2, '   be ck on ed</w>': 4, '   en g</w>': 1, '   ho ll an da i se</w>': 3, '   fi ll e ts</w>': 1, '   sa u ce p an</w>': 2, '   f la v or ful</w>': 1, ' k ar a o ke</w>': 1, ' j ack er</w>': 1, '   fun go ld</w>': 1, '   at le y</w>': 5, '   fu mb l ed</w>': 1, '   f ar t in</w>': 1, '   de ee ep</w>': 1, '   be a</w>': 2, '   op i e</w>': 2, ' gen er a tion al</w>': 1, '   ca li tr i</w>': 2, ' h ough</w>': 1, '   1 6 7 </w>': 2, '   a st ri ck y</w>': 3, '   j ack er</w>': 2, '   ju mp su it</w>': 1, '   pe ar</w>': 6, ' li m it</w>': 1, '   mi s de e</w>': 1, ' the ft</w>': 2, '   r in g lea der</w>': 1, '   bri gh ten ing</w>': 1, '   ca st le be ck</w>': 2, '   h in ked</w>': 1, '   f re b</w>': 5, ' no th in</w>': 3, ' f re b</w>': 1, '   ma ser at i</w>': 3, '   ro ll ins</w>': 1, '   ra ve l</w>': 1, '   r ac h man in off</w>': 1, '   de fi es</w>': 3, '   qu a y le</w>': 1, '   pe u got</w>': 1, '   co lu mb o</w>': 1, '   tu mb l er</w>': 4, '   re den b ac her</w>': 1, '   ar right</w>': 2, ' uni cor n</w>': 1, '   ex o ti c s</w>': 1, '   ori en tal s</w>': 1, '   s ki pp in</w>': 1, '   ob s cu rest</w>': 1, '   m c o</w>': 1, '   ti me con su m ing</w>': 1, '   tr un ca ted</w>': 1, ' ta ble</w>': 3, '   h y p es</w>': 1, '   g ran vi a</w>': 2, '   lo tu s</w>': 1, ' bo ok ed</w>': 2, '   re si den ces</w>': 2, '   1 4 4 3 </w>': 1, '   lo ck l in</w>': 1, ' mar ked</w>': 1, ' car o l</w>': 1, '   1 9 8 </w>': 1, ' e le an ors</w>': 1, '   gir li es</w>': 1, ' de mi se</w>': 2, ' re qui si te</w>': 1, ' c ru sh er</w>': 1, '   lea f y</w>': 1, '   ro ck for d</w>': 2, '   bo li vi an</w>': 1, ' w o o</w>': 1, ' w oo oo o</w>': 1, ' un lo ck</w>': 1, ' ch ee se</w>': 1, '   be ll ying</w>': 1, '   f l or al</w>': 1, ' pr int</w>': 1, '   c ro ft</w>': 1, '   wa y land</w>': 1, ' s way</w>': 1, '   s n ar ed</w>': 2, '   po in t five</w>': 1, '   w r en ch ing</w>': 1, '   b ac chi o ch i</w>': 1, '   tri fe c ta</w>': 1, '   op al</w>': 3, '   spe c s</w>': 2, '   ru st ing</w>': 2, '   le gi ti m ac y</w>': 1, '   un po pu l ar</w>': 1, ' ci r c us</w>': 1, '   o d ome t ers</w>': 1, '   b ins</w>': 2, '   j uni e</w>': 2, '   ab rea st</w>': 2, '   i s dn</w>': 1, '   so ars</w>': 1, ' per for man ce</w>': 1, '   2 8 9 </w>': 1, ' e le an or</w>': 3, '   g t</w>': 1, '   vo l t me ter</w>': 1, '   i g ni tion co il</w>': 1, '   je an ni e</w>': 5, '   oo on n n</w>': 1, '   tom m ee e e</w>': 1, '   oo on n</w>': 1, '   g ri b b s</w>': 2, '   je z u z</w>': 1, '   b on d man</w>': 1, '   en ve lo p es</w>': 7, '   tu ddy</w>': 2, '   ba tt s</w>': 2, '   con d om in i u ms</w>': 1, '   ga ys</w>': 3, '   hi j ack ers</w>': 1, '   a pr ons</w>': 1, '   h one</w>': 2, '   j ee su z</w>': 1, '   pi g p en</w>': 1, '   so ber ed</w>': 2, '   sta ti e</w>': 5, '   re st ru c tur in</w>': 2, '   c ow lin s</w>': 1, '   coun se l in</w>': 1, '   l en i en cy</w>': 1, '   comp le men t ary</w>': 1, '   s an d which</w>': 4, ' ke lly</w>': 5, '   b oun c in</w>': 1, '   c r ow d in</w>': 2, '   b ru sh in</w>': 1, '   ro s lin da le</w>': 1, '   co s mon au t</w>': 1, '   ra tion a li z a tions</w>': 1, '   r ou ter</w>': 1, '   b on do</w>': 1, '   p an han d l in</w>': 1, '   g ri ll e</w>': 1, '   a po lo gi z in</w>': 1, '   ju v in i le</w>': 1, ' l ar ry</w>': 1, '   mi ti ga ted</w>': 2, '   mo da li ti es</w>': 1, '   a gr ar i an</w>': 1, ' se i ze</w>': 1, '   wor k lo a d</w>': 2, '   co de brea king</w>': 1, '   ho ver in</w>': 1, '   att ac h ing</w>': 1, '   ver te x</w>': 1, '   fa il sa fe</w>': 1, '   ra m ses</w>': 1, '   che if</w>': 2, '   m c lu ll en</w>': 1, '   p in n in</w>': 1, '   in te ger</w>': 4, '   re c t an g le</w>': 3, '   su b di vi ded</w>': 1, '   re c t an g les</w>': 1, '   li p kin</w>': 1, '   la mb ea u</w>': 8, '   the or em</w>': 2, '   sa l k</w>': 2, '   ma il er</w>': 1, '   be ca u e</w>': 1, ' under m ine</w>': 1, '   f al si f y</w>': 1, ' ger ry</w>': 1, ' w re cks</w>': 1, '   re i g n</w>': 10, '   v u l n er ab i li ty</w>': 4, '   s ou th i e</w>': 2, '   ra man u j an</w>': 3, '   com bu na t ori al</w>': 1, '   s k y l ar</w>': 3, ' s ki in</w>': 1, ' can ned</w>': 1, ' ra m ro d</w>': 1, '   s k u l k</w>': 1, '   c lu s ea u</w>': 1, ' dan ny</w>': 2, '   ter ri o</w>': 2, '   ca e s er</w>': 1, '   wi l de</w>': 1, '   st e in</w>': 2, '   la be ling</w>': 1, '   ru t</w>': 3, '   f oo l er y</w>': 1, '   sh en ani g an</w>': 1, ' c had</w>': 1, '   er n har t</w>': 2, '   r ac in</w>': 2, ' ma te</w>': 4, '   ab ou ta</w>': 1, '   cou se ling</w>': 1, '   h om er u n</w>': 1, '   ba se l ine</w>': 1, '   sin ge l</w>': 1, ' mi ch el an ge l o</w>': 1, '   c ri ti ci s ms</w>': 1, '   si st ine</w>': 2, '   be v y</w>': 1, ' fi c tion al</w>': 1, '   son n et</w>': 4, ' star ry</w>': 1, '   2 8 5 </w>': 1, '   ja z z er ci z ing</w>': 1, '   ba ff le</w>': 1, '   no am</w>': 1, '   ch om s k y</w>': 3, ' man u f ac tur ing</w>': 1, '   z in n</w>': 1, ' sh r ink</w>': 3, '   s lu mm in</w>': 3, ' ch em</w>': 1, '   o de</w>': 2, '   di vi d es</w>': 3, '   han d red</w>': 1, '   su b stan ti ally</w>': 2, '   ba tt in</w>': 1, '   car a me ls</w>': 5, '   mar k y</w>': 1, '   da v ey</w>': 1, '   ro b by</w>': 1, ' ate</w>': 1, '   to o th less</w>': 1, '   car a me l</w>': 1, '   t wi c et</w>': 1, '   thous an</w>': 5, '   a st</w>': 8, '   con tr ack</w>': 2, '   con tr ac ting</w>': 2, '   go at in</w>': 1, '   r ou st in</w>': 1, '   li ss en</w>': 5, '   fa mb ly</w>': 12, ' ble</w>': 1, '   co f fee po t</w>': 1, '   e d di es</w>': 1, '   wa ter fa lls</w>': 1, '   ch ev v y</w>': 1, '   wor l</w>': 1, '   fri en</w>': 2, '   ro sa sh ar n</w>': 5, ' go in</w>': 15, '   un bro ke</w>': 1, '   e f</w>': 3, ' ar n</w>': 1, '   wh up</w>': 5, ' ti ck l in</w>': 1, '   ac ro st</w>': 5, ' hi k er</w>': 1, '   cu pp a</w>': 1, ' ca mp in</w>': 1, '   e le c ts</w>': 1, ' le ar n in</w>': 1, '   f in</w>': 19, ' ge tt in</w>': 3, '   st ab b in</w>': 1, '   the ir self</w>': 1, '   ex pe ck</w>': 1, '   d ru v </w>': 1, '   ca sy</w>': 7, '   se tt l in</w>': 1, '   mu le y</w>': 5, ' gr ow ed</w>': 1, '   some p in</w>': 5, '   ba p ti z in</w>': 2, '   bi g g es</w>': 1, '   do ses</w>': 3, '   sp er it</w>': 5, '   h ow l in</w>': 1, '   pi an a</w>': 1, '   tra ve ll in</w>': 1, '   p ra y in</w>': 4, ' par o le</w>': 1, ' sh out in</w>': 1, '   s qu ir m in</w>': 1, '   ca mp in</w>': 2, ' lon gs</w>': 1, ' ny</w>': 4, '   f ust</w>': 5, '   sho v in</w>': 1, '   sh ar e c ro pp er</w>': 1, '   some p</w>': 1, ' sted</w>': 1, '   exac k ly</w>': 1, '   ne x</w>': 3, ' mi g ran t</w>': 1, ' af ter war ds</w>': 2, '   mi d day</w>': 2, '   b re sh</w>': 1, '   le tta</w>': 1, ' sta y in</w>': 5, ' a ys</w>': 1, '   cu ri ou s er</w>': 1, '   st om i ck ac he</w>': 1, '   gi g g ly</w>': 2, '   de p en</w>': 1, '   s ca ir t</w>': 3, '   su mp</w>': 8, '   ni m sy</w>': 1, ' mi m sy</w>': 1, '   ex try</w>': 2, '   sh et</w>': 2, ' en in</w>': 1, '   le f</w>': 1, ' wor l</w>': 1, ' un t</w>': 1, '   bl ow ed</w>': 1, ' wh er ever</w>': 1, '   fa llow</w>': 1, '   m w</w>': 1, ' han k er in</w>': 1, '   j i b bi t in</w>': 1, ' d ra g g in</w>': 1, ' win fi el</w>': 1, '   tru s</w>': 1, '   po ss es</w>': 2, '   l y n ch in</w>': 2, '   ba kin</w>': 3, ' de t our</w>': 1, ' tr y in</w>': 1, '   d ou se</w>': 2, '   so o th in</w>': 1, '   si r up</w>': 1, '   chi ll u n</w>': 3, ' bu st</w>': 1, '   lo af s</w>': 1, ' wi ch</w>': 1, ' wi d g es</w>': 2, '   s of</w>': 1, '   hon g ry</w>': 1, '   g ro c</w>': 1, '   be h in</w>': 1, ' t re sp as s in</w>': 1, '   su p r</w>': 1, ' en d ant</w>': 1, '   du st ers</w>': 2, ' bl ow in</w>': 1, '   our n</w>': 1, ' be in</w>': 2, '   sha w ne e</w>': 4, '   fr y in</w>': 1, '   gi z z ard</w>': 5, '   go v </w>': 2, ' ent</w>': 1, ' dr un k</w>': 1, '   h un n er d</w>': 2, '   shu cked</w>': 1, '   pi ck ers</w>': 3, '   po s</w>': 4, ' car ds</w>': 1, ' under st an</w>': 1, '   qu i</w>': 2, '   ca ta lo gu es</w>': 2, '   mo s</w>': 2, ' st an</w>': 2, '   j i g ger</w>': 1, '   pi x le y</w>': 1, '   h or se y</w>': 2, '   p ac ke ts</w>': 2, '   s ou th land</w>': 3, '   ar l en e</w>': 4, '   o s lo t t</w>': 1, '   g ran d stan ds</w>': 1, '   de st e p han o</w>': 2, '   h er ba li st</w>': 2, '   s ci en to lo gi st</w>': 2, '   nu tri tion i st</w>': 2, '   sa y in gs</w>': 1, ' me chan i cal</w>': 2, '   j un k y</w>': 1, ' ki t sch y</w>': 1, '   o tter</w>': 2, '   m ow</w>': 5, '   a e t na</w>': 1, '   sho ck a bu k u</w>': 1, '   ra tion a li z ation</w>': 1, ' lan er</w>': 1, '   g ro s se</w>': 1, '   po in te</w>': 1, ' op ti c</w>': 1, '   brea ting</w>': 1, '   an t sy</w>': 4, '   u l y ss es</w>': 1, '   d ome sti ci ty</w>': 1, '   s ar ac en s</w>': 1, '   o at man</w>': 1, ' with ho l ding</w>': 1, '   par a gu ay</w>': 1, '   su gar m ou th</w>': 1, '   bu ll e th o le</w>': 2, ' as so ci a tes</w>': 1, ' lon er</w>': 2, ' gra in</w>': 1, '   en able</w>': 1, ' sh e lf</w>': 1, '   bri ce</w>': 1, '   fro me y er</w>': 1, '   pl o</w>': 1, ' a der</w>': 1, ' me in h of</w>': 1, ' no o o</w>': 1, '   fr y sa l</w>': 1, '   po st ing</w>': 1, '   gre en p ea ce</w>': 6, '   e st on i a</w>': 1, '   in tr ac ta ble</w>': 2, '   mar ce ll a</w>': 1, '   p out in</w>': 1, ' bo om er</w>': 1, '   o pp re ssed</w>': 2, ' re f le c tive</w>': 1, '   vi su a li z ed</w>': 1, '   s l ack ster</w>': 1, ' mi s an th ro p es</w>': 1, '   ss sh h h</w>': 2, '   ran che st</w>': 1, '   re ci tal s</w>': 1, '   re d de st</w>': 1, '   b lu est</w>': 1, '   co l ts</w>': 1, '   ri m ro ck</w>': 1, '   g ri z z li es</w>': 2, '   jo g g ers</w>': 1, '   i t z ha k</w>': 2, '   per l man</w>': 1, '   r ac h man in o v </w>': 1, '   vo ca li ze</w>': 2, '   b ran ding</w>': 9, ' shu cks</w>': 1, '   whi sp er er</w>': 1, '   m ac l ean</w>': 3, '   sh er i d en</w>': 1, '   i g nor es</w>': 2, '   li be l ed</w>': 1, '   f ar l ow</w>': 1, '   tu bab </w>': 1, ' tu bab </w>': 3, '   fa x es</w>': 1, '   go tt cha l ks</w>': 1, '   de l c o</w>': 1, ' fa v ori te</w>': 3, '   re doing</w>': 1, '   ju di th</w>': 3, '   an gu s</w>': 2, '   han ks</w>': 1, ' sho pp ing</w>': 1, '   ran ch er</w>': 3, '   per ch er on</w>': 1, '   b ran ton</w>': 1, '   plan ter</w>': 3, '   br ow ni es</w>': 1, '   cho t ea u</w>': 2, '   u d d ers</w>': 1, ' h er e for ds</w>': 1, '   h er e for ds</w>': 2, '   imp r int</w>': 2, '   br on ty</w>': 1, '   di ved</w>': 1, '   ba u d</w>': 1, '   bi as</w>': 1, '   un cor ru p ted</w>': 1, '   ha ck ers</w>': 7, '   na me less</w>': 3, '   g lu tt ons</w>': 1, '   ton es</w>': 2, '   mi c ro ca s se tt e</w>': 1, '   gi b s ons</w>': 2, '   com pi l er</w>': 1, '   uni x</w>': 1, '   ni k on</w>': 1, '   c ri sp y</w>': 1, '   su per com pu t ers</w>': 1, '   op er at ors</w>': 4, '   fro o t</w>': 1, '   loo ps</w>': 6, '   1 9 8 4 </w>': 8, '   t y p o</w>': 2, '   a i a i a i a i a e e</w>': 1, '   ph rea k</w>': 6, ' in fe c ting</w>': 1, '   e ll in g son</w>': 8, '   t an k ers</w>': 4, '   n on on on on o</w>': 1, '   sp a ke</w>': 1, '   cor in th i ans</w>': 1, '   mor t is</w>': 1, '   in s an e ly</w>': 2, '   b ps</w>': 1, '   con sor t</w>': 3, '   ex il ed</w>': 4, '   n ar king</w>': 1, '   we tw are</w>': 1, ' 4 8 1 7 </w>': 1, '   z i ps</w>': 1, '   er as es</w>': 2, '   co p y ri gh ts</w>': 1, ' me ss</w>': 1, '   ri s c</w>': 2, '   p c i</w>': 1, '   pen ti u m</w>': 1, '   o t v </w>': 1, ' 4 2 4 0</w>': 1, '   bo x y</w>': 1, '   u h h h m m</w>': 1, '   a ha ha h a</w>': 1, '   bl t</w>': 1, '   a w o l</w>': 2, '   k a wa sa k i</w>': 1, '   k ar i</w>': 1, '   u h h mm m</w>': 1, '   ve d der</w>': 1, '   li b by</w>': 2, '   par de ll a</w>': 1, '   be l for d</w>': 2, '   re p li ca tes</w>': 2, '   re p li ca ting</w>': 1, '   di ck wee ds</w>': 2, ' vi ru s</w>': 4, '   lo g in</w>': 1, '   su b di re c t ory</w>': 1, '   u s ers</w>': 6, '   ha p less</w>': 1, ' we en i e</w>': 1, '   mi sc e ll an e ous</w>': 1, '   n on o</w>': 1, '   cra y o l a</w>': 1, '   what what what</w>': 1, '   ph rea k ph rea k ph rea k ph rea k ph rea k</w>': 1, '   du de du de du de du de du de du de</w>': 1, '   in ser ting</w>': 1, ' min or</w>': 1, '   un t rea c ea ble</w>': 1, '   re con ci l ed</w>': 5, '   u h m m</w>': 1, '   si e ze</w>': 1, '   su per t an k er</w>': 1, '   mi sta k en ly</w>': 1, '   me ti cu l ou s ly</w>': 2, '   su per u s er</w>': 1, '   ac oun t</w>': 1, ' 4 2 0 2 </w>': 1, '   ex p l</w>': 1, '   fa ll bro o ks</w>': 1, '   o in k er</w>': 1, '   r ink</w>': 2, '   c ru tch es</w>': 1, '   pre pped</w>': 1, '   cor ru th ers</w>': 1, '   4 1 0</w>': 1, '   m ee k er</w>': 4, '   ki do</w>': 2, '   s co op s</w>': 2, '   in s om ni ac s</w>': 1, '   r ac h</w>': 3, '   lin d say</w>': 1, '   cor ru ther</w>': 1, '   1 3 2 </w>': 3, '   1 3 3 </w>': 2, '   ar ma ge d don</w>': 4, '   du m on</w>': 1, '   a qu an au ts</w>': 1, '   do y les</w>': 3, '   l y n da</w>': 2, '   tra m er</w>': 3, '   god f re y</w>': 1, ' lan ter n</w>': 4, '   l y n d se y</w>': 2, '   in hu man ly</w>': 1, '   e mo tion less</w>': 2, '   br ac ke t t</w>': 1, '   bab y si ts</w>': 1, '   bo g y man</w>': 8, ' ne u tr on</w>': 1, ' ar an tu l a</w>': 1, '   e la m</w>': 3, '   m ack en si e</w>': 1, '   j i b ber i sh</w>': 1, '   th or a z in</w>': 1, '   i cha bo d</w>': 6, '   st ru c tur es</w>': 1, ' p es</w>': 1, '   la s a</w>': 1, '   ri go le t to</w>': 2, '   inter de pen dent</w>': 1, ' or g ani c</w>': 1, '   au di tion ed</w>': 2, '   sh e b a</w>': 1, '   lu lled</w>': 1, '   n on con te x tu ra l</w>': 1, '   er n an i</w>': 1, '   au di ome try</w>': 1, '   de ci be l</w>': 1, '   h y po ch on dri ac </w>': 2, '   sin us</w>': 1, '   n ev </w>': 2, '   af fi li a tes</w>': 1, '   w r ac ked</w>': 1, ' in ev i ta ble</w>': 1, '   s ki ll fu lly</w>': 2, '   di p lo ma ti ca lly</w>': 1, '   h y gi en i sts</w>': 1, '   br ow se</w>': 1, '   a li en a ted</w>': 2, ' pre con ce i ved</w>': 1, '   s li ther</w>': 1, '   au sch wi t z</w>': 2, '   my sti fi ca tion</w>': 1, '   sy st e ma ti c</w>': 4, '   lu sts</w>': 1, ' con te mp tu ous</w>': 1, '   ha m ster</w>': 1, '   n n n n</w>': 1, ' r on ny</w>': 1, ' in st ead</w>': 4, '   spe ll man</w>': 1, '   in se min ation</w>': 2, ' st ran ger</w>': 1, '   m h</w>': 1, '   under con fi dent</w>': 1, '   c ome di ans</w>': 3, '   ha ir do s</w>': 1, '   s our er</w>': 1, '   ma v is</w>': 1, '   under cu t</w>': 1, '   cl er i cal</w>': 1, '   pro du c ti ve ly</w>': 1, '   s we a</w>': 1, ' han na h</w>': 1, ' na me ly</w>': 1, '   e ma s cu la ting</w>': 1, '   nu re mb er g</w>': 3, '   ti gh ta ss</w>': 1, ' ho lly</w>': 2, '   an ti hi sta m ine</w>': 2, ' sa l ary</w>': 2, ' s we ar</w>': 2, ' v ac ation</w>': 1, '   ver ger</w>': 4, '   le ch ter</w>': 19, '   re fa shi on ed</w>': 1, '   le ch ter i an a</w>': 1, '   tr ans gre ssed</w>': 1, '   ga vo tt e</w>': 1, '   ja il ed</w>': 2, '   wa al</w>': 14, '   an no ta ted</w>': 1, '   du ma s</w>': 1, '   p an z</w>': 1, ' ma ter i al s</w>': 1, ' bo o ks</w>': 3, ' de lu si ve</w>': 1, '   com po st ing</w>': 1, ' v al u ed</w>': 1, '   ph ra se o lo g y</w>': 1, '   pro fe ssi ons</w>': 2, '   wor k man</w>': 1, ' h in der ed</w>': 1, ' s qu an der</w>': 1, ' ad v an ta ge</w>': 1, ' in for m</w>': 2, ' do ci l ed</w>': 1, '   bri g ha m</w>': 3, '   ev el da</w>': 5, '   fi re c r ack ers</w>': 4, '   d ru m go</w>': 5, '   po tom ac </w>': 2, ' t ar get</w>': 3, '   w u un t</w>': 2, ' ma ke work</w>': 1, ' pu b</w>': 1, ' la tions</w>': 2, ' hi sp ani c</w>': 1, ' li br ar i an</w>': 1, '   wh ad di z it</w>': 1, '   gar re t</w>': 2, ' w ea l th y</w>': 1, '   f an e ll i</w>': 6, ' en e mi es</w>': 1, '   do c t ore</w>': 1, '   pa ll a z o</w>': 1, '   ca pp on i</w>': 1, '   o h for god sa ke</w>': 1, '   son g bi r d</w>': 1, ' me d dle</w>': 1, '   o pp re ss or</w>': 2, '   e mp ow er ed</w>': 3, '   no t l on</w>': 1, ' t ou ch ed</w>': 2, ' d on ation</w>': 1, ' un mar ked</w>': 1, ' su i t case</w>': 4, ' lo ck er</w>': 1, ' sp in al</w>': 1, ' pu lling</w>': 1, ' e en</w>': 1, '   sho tta</w>': 1, '   ev ab o dy</w>': 1, ' fu ss</w>': 1, '   what say</w>': 2, ' con fir ms</w>': 1, ' bri be</w>': 1, ' re lea se</w>': 2, '   pro ff ers</w>': 1, '   re in sta te ment</w>': 4, '   k r en d l er</w>': 3, ' ho b b y i st</w>': 1, '   pu r por ts</w>': 2, '   ve sti gi al</w>': 2, '   tr an so m</w>': 2, '   fa ll back</w>': 1, '   co b bl ed</w>': 1, ' in st ru c ted</w>': 1, '   ba t f</w>': 1, '   gi v ver</w>': 1, '   o ver w r ought</w>': 2, ' b ack up</w>': 2, ' le ar ned</w>': 1, '   ev one</w>': 1, '   th i z z</w>': 2, ' ho tter</w>': 1, ' stu di o l o</w>': 1, ' h er self</w>': 1, '   a m end</w>': 1, ' co ll e ct</w>': 1, '   do tt ore</w>': 4, ' pro gra m</w>': 2, ' un ear th</w>': 1, '   for e be ars</w>': 1, '   stu di o l o</w>': 1, '   en chan te</w>': 1, '   pa z z i</w>': 3, ' ha ma me l is</w>': 1, ' ha z el</w>': 1, '   ad re s a</w>': 1, '   de pi c ted</w>': 1, '   ro bi a</w>': 1, '   r on de ls</w>': 1, '   com men da t ore</w>': 3, ' w ound</w>': 1, ' sc ar</w>': 3, ' fa ti gu ed</w>': 2, '   di sh ev e lled</w>': 1, ' ther e fore</w>': 1, ' we d ding</w>': 2, '   nu o v a</w>': 1, ' no te</w>': 2, '   pre de ce ss or</w>': 3, '   shi r ly</w>': 2, '   ki tt y k at</w>': 2, '   st r in g b ean</w>': 1, ' fran k l in</w>': 2, ' tra in</w>': 2, '   figu r ine</w>': 2, ' ev en tually</w>': 2, '   mi ck e y m ou se</w>': 1, ' ta s k for ce</w>': 1, '   i z s at</w>': 1, '   son of a</w>': 2, ' sp end</w>': 4, ' pa use</w>': 1, '   co a li tion</w>': 1, '   at f</w>': 4, ' s ou p</w>': 2, '   f re q s</w>': 2, ' ar rest</w>': 1, '   ob je c tion able</w>': 1, '   p sy cho an al y z ed</w>': 1, '   re ga in ing</w>': 1, '   pro te st ant</w>': 1, '   an gu i sh</w>': 1, '   uni ma g in a tive</w>': 2, ' mi r th</w>': 1, '   en qu ir ing</w>': 1, '   cor d ell</w>': 1, '   d on tch a</w>': 2, '   ar de li a</w>': 3, '   com men ta t ors</w>': 1, '   pu r por ted</w>': 1, '   g le e</w>': 1, ' ju ri s di ction</w>': 1, '   ha z ing</w>': 1, ' han d sha ke</w>': 1, ' can ni ba l</w>': 1, '   fri en d less</w>': 1, '   h ow about</w>': 1, ' fr en d l er</w>': 1, ' ex ce ll ent</w>': 2, ' en han c ing</w>': 1, ' gre e ce</w>': 1, ' p ay</w>': 5, '   su ffer an ce</w>': 1, ' fa sc in a ting</w>': 2, ' f ac ts</w>': 1, ' so l ved</w>': 1, ' c lu e</w>': 1, '   fa med</w>': 1, '   gra ve st one</w>': 3, ' fe tch</w>': 2, ' ca v a li er</w>': 1, ' re spe c ts</w>': 2, '   cl ar ice</w>': 37, ' sa d ly</w>': 1, ' d ou b ted</w>': 1, '   bra z i li an</w>': 6, ' li a i son</w>': 1, ' wa tch man</w>': 1, ' ti me clo ck</w>': 1, ' li ce</w>': 1, ' ac ce pt</w>': 2, ' d ru m go</w>': 1, ' ad vi ce</w>': 1, ' po si tive</w>': 5, ' na g ged</w>': 1, ' de sp a ir</w>': 1, ' fa i lu re</w>': 1, ' s wee p in gs</w>': 1, ' bar ga in</w>': 1, ' sy m pa th y</w>': 1, ' wh ore</w>': 5, ' ra ti fi ca tion</w>': 1, '   hi de ou s ly</w>': 1, '   b b b</w>': 1, ' or p ha n</w>': 1, ' hu mi li ation</w>': 2, ' dan ger</w>': 2, ' be tra y al</w>': 1, ' af f li c ted</w>': 2, ' e mb r ace</w>': 1, ' sh y</w>': 1, '   gh ou li sh ne ss</w>': 1, ' cap ture</w>': 1, '   p ee p ers</w>': 2, '   ree</w>': 6, '   ta li a</w>': 14, '   out did</w>': 1, '   p ou ty</w>': 1, '                                                   </w>': 17, '                                                                                       </w>': 9, '                                                                         </w>': 4, ' se tt ed</w>': 1, '                 </w>': 2, '   se ar ch ers</w>': 1, '   com man d ant</w>': 1, '                       </w>': 1, '                                                                                           </w>': 13, '                                                                   </w>': 4, '                                                                             </w>': 1, '                                                                                     </w>': 3, '                                                                                                     </w>': 3, '   y ow z a</w>': 1, '                                                                 </w>': 2, '                                                                                                 </w>': 3, '   ha y le y</w>': 6, '   o ver h and</w>': 1, ' o ber on</w>': 1, '                                           </w>': 21, '   ther e of</w>': 3, '                                                                                                       </w>': 2, '                                                                                         </w>': 3, '   ma l er</w>': 1, ' gu m</w>': 1, '                                                             </w>': 2, '   c it</w>': 1, '   sh op li ft</w>': 2, '                                                                                                         </w>': 1, '   pla y fu lly</w>': 2, '   bi ck er</w>': 1, '   ga z es</w>': 1, '   t in ge</w>': 1, '                             </w>': 2, '   b om b sh e ll e tt es</w>': 1, ' coun ter par ts</w>': 1, '                                                                               </w>': 3, '   pi x el</w>': 5, '                                                                                             </w>': 1, ' b er</w>': 2, ' w en dy</w>': 2, '   cha in saw</w>': 5, '   jo c el y n</w>': 1, '   na i</w>': 2, ' i ve</w>': 1, '   o ber on</w>': 2, '   m un ch k ins</w>': 6, '   s wh oo sh ed</w>': 1, ' wh ad d ya</w>': 1, ' bu n k</w>': 1, '   con fi s ca te</w>': 2, '                                                                                 </w>': 4, '                                             </w>': 3, '                                                                                               </w>': 2, '                                                                                                           </w>': 1, '   con qu ers</w>': 2, '   con qu er ed</w>': 4, ' sh are</w>': 4, '                                                                                                   </w>': 2, ' ve ter in ar i an</w>': 1, ' ba ll er in a</w>': 1, '   i un no</w>': 2, ' de li ber a tely</w>': 1, ' he at</w>': 1, ' dar k</w>': 1, '                                                                                   </w>': 2, ' thin gi e</w>': 1, ' rea ct</w>': 1, ' i un no</w>': 1, '   pla y sta tions</w>': 1, '   re char ged</w>': 2, ' un no</w>': 1, ' wi chi ta</w>': 2, '                                                                                                             </w>': 1, ' bi o ti c s</w>': 1, '   ca f fe in ing</w>': 1, '   re st ru c tur ing</w>': 1, '   o ver used</w>': 1, '   s no</w>': 1, ' con es</w>': 1, '   ma l co m</w>': 1, ' d ru g</w>': 3, '   f le m ing</w>': 11, '   with d ra w l</w>': 1, ' lo v n g</w>': 1, '   we st er bur g</w>': 7, ' k u m ba ya</w>': 1, '   que ll e</w>': 2, '   re gu r gi ta tion</w>': 1, '   bar bi es</w>': 1, '   br ow ni e</w>': 7, '   b lu e bi r d</w>': 1, '   al b in o</w>': 6, '   c li qu e</w>': 1, '   lo ser ne ss</w>': 3, '   qu as i</w>': 3, '   b ru sh ing</w>': 6, '   co o ti e</w>': 1, '   pi ra h na</w>': 1, '   no z z le</w>': 1, '   d un n sto ck</w>': 1, '   du mp tru ck</w>': 4, '   ke g g ers</w>': 2, ' wh am</w>': 1, ' ma am</w>': 1, '   ex p ell</w>': 1, ' 8 6 </w>': 1, '   k u</w>': 2, ' u r t</w>': 2, '   te en y bo pp er</w>': 1, '   ne ga ti ves</w>': 4, '   toge ther ne ss</w>': 1, ' ni ce st</w>': 1, '   can o e ing</w>': 1, '   d wee be tt e</w>': 1, '   me g ab i tch</w>': 1, ' con v in c ing</w>': 2, '   for b es</w>': 1, '   h y l ton</w>': 1, '   bi g fu n</w>': 1, '   t un e less</w>': 1, '   e u ro fa gs</w>': 1, '   prob s</w>': 1, '   ho t lin es</w>': 1, '   co l fa x</w>': 1, '   be ll y f lo pped</w>': 1, '   sta ti sti c</w>': 3, '   he a</w>': 1, '   di ge st ing</w>': 3, '   ta ter</w>': 1, '   ther ma ls</w>': 1, '   per fe c to</w>': 3, '   to may to</w>': 1, '   to ma h to</w>': 1, '   under lin ing</w>': 1, ' me an in g ful</w>': 2, '   s li tting</w>': 1, '   a ir head</w>': 1, '   p ea cen i k</w>': 1, '   c li qu es</w>': 2, '   at one</w>': 3, '   re de fine</w>': 2, '   b re w s k y</w>': 1, '   pla y girl</w>': 1, '   un under stan ding</w>': 2, ' jo ck</w>': 1, '   lu ge</w>': 1, ' cl y de</w>': 1, '   s lu r pe e</w>': 3, ' my ri a d</w>': 2, '   vo ca b</w>': 1, '   po ss</w>': 2, '   f ra u ght</w>': 1, '   d ran o</w>': 1, '   ph le g m</w>': 1, '   g lo b b er</w>': 1, ' be si d es</w>': 1, ' no o dle</w>': 1, ' b ac on</w>': 1, ' so l</w>': 1, ' ti med</w>': 1, '   hea th ers</w>': 1, '   g lo ss</w>': 4, '   c ro qu et</w>': 2, ' j as on</w>': 2, '   com mer i cal</w>': 1, ' br in ging</w>': 3, '   de con st ru ction</w>': 1, '   c ow ti pp ing</w>': 1, '   pu d wa pp er</w>': 1, '   di d d le y</w>': 1, '   g rea te</w>': 1, '   con te stan ts</w>': 1, '   inter offi ce</w>': 1, '   me mor an da</w>': 1, '   con te mp la ted</w>': 2, '   comp le x i ons</w>': 1, '   do o dy</w>': 1, '   ri e per</w>': 6, '   co tt on ed</w>': 1, '   pla sti c ine</w>': 1, '   hon or a</w>': 9, '   ac comp ani es</w>': 1, '   hu l me</w>': 2, '   bi r d house</w>': 2, '   p in ny</w>': 1, '   cu t l er y</w>': 1, '   ba y li ss</w>': 1, ' fri ger a tor</w>': 1, '   hu l mes</w>': 1, '   s na tch es</w>': 2, '   n an a</w>': 1, '   for e bo ding</w>': 2, '   in fe c ti ous</w>': 4, '   ma tr on</w>': 1, '   t re la w ne y</w>': 1, '   car me li ta</w>': 3, '   ni cks</w>': 1, '   bo ar der</w>': 4, '   di e llo</w>': 1, '   bu n k u m</w>': 1, '   un fa i th ful</w>': 4, '   k ri st y</w>': 1, '   star tr ed</w>': 1, ' bu tch er ed</w>': 1, '   stu mp ed</w>': 3, '   go o ding</w>': 4, '   li t ll e</w>': 1, '   gr in der</w>': 1, '   b re t</w>': 4, '   de ar don</w>': 1, '   d or m ere</w>': 1, '   re la p ses</w>': 1, '   a mb ro se</w>': 26, '   pre school</w>': 1, '   gra mm a</w>': 1, '   a a a g g g gh h h</w>': 1, '   con tr ac tion</w>': 2, '   wh ack job</w>': 1, '   lo do vi c o</w>': 1, '   inter na lly</w>': 2, '   g li b ne ss</w>': 1, '   b en e f ac tor</w>': 2, '   di r ti est</w>': 1, ' a m bi gu ous</w>': 1, '   min d ga mes</w>': 1, '   f er v ent</w>': 1, '   mar har a g i</w>': 1, '   cor ro de</w>': 1, '   ta w ny</w>': 3, '   f en ne l</w>': 1, '   a ph ro di si ac s</w>': 1, '   cap ers</w>': 3, '   al ri gh ty</w>': 1, ' ta w ny</w>': 1, '   in s om ni ac </w>': 2, ' w ou l da</w>': 1, '   la plan te</w>': 36, ' in spi re</w>': 1, '   bu b b er</w>': 14, '   ga y le y</w>': 5, '   mi c ro ph on es</w>': 1, '   man i pu la t ors</w>': 1, ' ber ni e</w>': 2, ' na i ve</w>': 2, ' in spi red</w>': 1, ' att or ne y</w>': 1, '   wh a a a a a at</w>': 1, ' j a</w>': 1, '   fami li al</w>': 1, ' au g ment</w>': 1, '   su mm a</w>': 2, ' an ti ci pa tion</w>': 1, '   w ou l d j a</w>': 2, '   to l d j a</w>': 2, '   god dam n life</w>': 1, '   comp le x es</w>': 2, '   ten dan cy</w>': 1, ' ex a g ger ate</w>': 1, ' t y pi sts</w>': 1, ' el li o t</w>': 1, '   mu d ba th</w>': 1, '   d on ch a</w>': 2, ' s cu m ba g</w>': 1, '   o pp ort uni s m</w>': 1, ' mi s be ha vi or</w>': 1, '   wh a a a a a a at</w>': 1, '   c li ma x ed</w>': 1, ' t in y</w>': 1, ' har as sed</w>': 1, ' g ru b b ing</w>': 2, '   g wa n</w>': 2, '   bo ok case</w>': 1, '   no m m</w>': 1, '   what si s name</w>': 1, '   bo d y bu il ding</w>': 2, ' ne w s</w>': 1, ' un wor th y</w>': 1, '   s wi p es</w>': 2, ' pu r se</w>': 1, ' si l ver</w>': 1, ' a w ard</w>': 1, ' fi end</w>': 1, '   pi e c ing</w>': 2, '   nee d j a</w>': 1, '   un en ding</w>': 1, '   har d bi tt en</w>': 2, '   ne w s woman</w>': 1, ' ne w s woman</w>': 1, '   dea k</w>': 3, '   pri e s th o od</w>': 1, ' beau ty</w>': 3, ' con ce pt</w>': 1, ' f re sh ne ss</w>': 1, '   si de b ar</w>': 2, '   po sh est</w>': 1, ' c y ni cal</w>': 1, ' be ha ves</w>': 1, ' un kn own</w>': 2, ' pa int</w>': 1, '   h or mon es</w>': 8, '   de pen d ab i li ty</w>': 1, '   re e bo ks</w>': 2, '   sy n the s is</w>': 1, '   mon u men ta lly</w>': 3, ' de cl are</w>': 1, '   un he si ta t in g ly</w>': 1, '   fo cal</w>': 2, ' hope</w>': 3, '   car pen try</w>': 5, '   a i ry</w>': 6, '   en su red</w>': 1, '   hou se si tting</w>': 1, '   re fri ger at ors</w>': 2, '   ru do lf</w>': 9, '   hu ff</w>': 1, ' st run g</w>': 3, '   wh in es</w>': 1, ' p sy ch o</w>': 1, '   l ow en th al</w>': 2, ' st ab b er</w>': 1, ' l ow en th al</w>': 1, ' p en</w>': 4, ' ar an son</w>': 1, '   men us</w>': 1, ' tra tt ori a</w>': 1, ' v al en t in o</w>': 1, ' z el man</w>': 1, '   z el man</w>': 1, ' car b ons</w>': 1, ' m ee t in gs</w>': 1, ' s an d wi ch es</w>': 1, ' p hi li p</w>': 1, '   j uni ors</w>': 1, '   re mo de l</w>': 1, '   br ack et</w>': 1, '   list in gs</w>': 1, '   dre y er</w>': 6, '   co si er</w>': 1, '   hi ll man</w>': 5, '   be d so e</w>': 6, '   o c cu r r en ces</w>': 1, '   re d der</w>': 1, '   gre le y</w>': 1, '   v a si l ni c</w>': 1, '   lu man</w>': 1, '   ca sta ge er</w>': 1, '   ro mer i z</w>': 4, '   hi gh lan der</w>': 5, '   pen d ra g on</w>': 2, '   ca ven au gh</w>': 1, '   an o th ers</w>': 1, '   some things</w>': 1, '   sy n di ca ted</w>': 1, '   s oun din gs</w>': 1, '   ca l ans</w>': 1, '   t ou re z</w>': 1, '   1 7 8 9 </w>': 1, '   ex ca v ation</w>': 2, '   ca l an</w>': 2, '   tra t in o</w>': 1, '   du p li ca tion</w>': 2, '   el u d es</w>': 1, '   k u r g an</w>': 2, '   li f es</w>': 1, '   te le ph on i c</w>': 2, '   mi k k el son</w>': 3, '   ta ll ey</w>': 12, '   fuck u ps</w>': 1, '   c lu st er fuck</w>': 1, '   b en z a</w>': 3, '   cor d w o od</w>': 1, '   ra gh ead</w>': 1, ' gi m me</w>': 4, ' de pre ss an ts</w>': 1, '   bri s to</w>': 2, '   un ti ed</w>': 2, '   k ru p che k</w>': 3, ' ars</w>': 1, '   i den ti fi er</w>': 1, '   ri ver a</w>': 1, '   cra w l sp ace</w>': 2, '   un e qu al</w>': 1, '   pu pi la tion</w>': 1, '   in tr ac ran i al</w>': 1, '   s mi th s</w>': 2, '   j or gen son</w>': 1, '   min i mar t</w>': 1, '   do a</w>': 1, '   f la sh ban gs</w>': 1, '   a mp ed</w>': 6, '   du l ce</w>': 1, '   f lan d ers</w>': 2, '   h ow ell</w>': 1, ' as su me</w>': 1, '   co ck su ck in g mo ther fuck er</w>': 1, '   bl ow t or ch</w>': 2, '   fuck wa d</w>': 1, '   hu tu </w>': 8, ' tu t s i</w>': 1, '   ac cor ds</w>': 2, '   bi k</w>': 1, '   tu t s is</w>': 1, '   gre go i re</w>': 10, '   n ar a m un j u</w>': 1, '   mi ll e</w>': 13, '   co ll in es</w>': 5, '   tu t s i</w>': 6, '   ru se sa ba g in a</w>': 5, '   bi z im un gu </w>': 2, ' en for ce men ts</w>': 1, ' in for ce men ts</w>': 1, '   man da te</w>': 3, '   re pri sa l</w>': 2, '   inter ven e</w>': 1, '   r wan d an</w>': 2, '   be l gi ans</w>': 2, '   st ab i li z es</w>': 1, '   e u ro pe ans</w>': 2, '   gi t ar a ma</w>': 3, '   i sh c a</w>': 1, '   ba h a</w>': 1, '   cla mb ers</w>': 1, '   a mi d st</w>': 1, '   sa b en a</w>': 2, ' na tch</w>': 1, '   e pa u le ts</w>': 1, ' sto cked</w>': 1, '   in e y s i</w>': 1, '   wi mb le don</w>': 1, '   ru ta g und a</w>': 3, '   hu mb le st</w>': 1, '   da g li sh</w>': 3, '   ar th u rs</w>': 1, '   b ou li er</w>': 1, '   gar an d i</w>': 1, '   ki g al i</w>': 2, '   ru h en ger i</w>': 1, '   ba g es</w>': 1, '   8 4 </w>': 4, '   ta t s i</w>': 2, '   pa ki stan is</w>': 1, '   gar in d i</w>': 1, '   z o z o</w>': 6, '   fe den s</w>': 2, '   ba p ti st e</w>': 3, '   s qu an der ed</w>': 2, '   char in g as</w>': 1, '   go de fro id</w>': 1, '   re ce p tion i sts</w>': 1, '   cho co la tes</w>': 3, '   gi ter a ma</w>': 1, '   ru ta g an da</w>': 1, '   ser ad un gu </w>': 2, '   mar r</w>': 7, '   j en z en</w>': 5, '   at ro p ine</w>': 1, ' as se ts</w>': 1, '   sp li t s vi ll e</w>': 2, '   pri t che t t</w>': 25, '   v an n ac u t t</w>': 5, ' in spi ra tion al</w>': 1, ' sa tu ra tion</w>': 2, ' fri gh ten ed</w>': 1, '   un s ea ls</w>': 1, '   s che c ter</w>': 1, '   a q ain</w>': 1, '   bl ack bur n</w>': 4, '   u h h h h</w>': 2, ' nor ma l cy</w>': 1, '   gre e ter</w>': 1, '   ne ck ti e</w>': 3, '   s li ther ed</w>': 1, ' com for ta ble</w>': 2, '   su c c ee ding</w>': 2, ' an n</w>': 1, '   sto ck ard</w>': 1, '   re wi re</w>': 4, '   att ac he</w>': 1, ' dr y er</w>': 1, ' un so l ved</w>': 2, ' le s bo s</w>': 1, ' ne ss es</w>': 1, '   ani ma tr on i c</w>': 1, '   mu m mi es</w>': 1, ' chi lls</w>': 1, '   s ca l ding</w>': 2, '   re tr ac ta ble</w>': 1, '   con gra tu </w>': 1, '   gen i ta li a</w>': 1, '   j on e st own</w>': 2, '   be hea ded</w>': 1, ' fa tal</w>': 1, ' un like</w>': 1, ' chri st en</w>': 1, ' cha m pa g n e</w>': 1, '   gra tes</w>': 1, '   un ex pla in able</w>': 1, ' j en z en</w>': 1, ' ex e cu tive</w>': 2, ' pi c tur es</w>': 3, '   di v vi ed</w>': 1, '   f la il</w>': 1, '   sc rea m er</w>': 6, ' du m my</w>': 1, '   per co la ting</w>': 1, '   fe st er ing</w>': 1, '   re fu r bi sh men ts</w>': 1, '   1 9 3 1 </w>': 1, ' re ver se</w>': 2, ' lo ck down</w>': 1, '   1 8 8 2 </w>': 1, '   1 8 8 0</w>': 1, '   sh e a</w>': 3, '   w ex for d</w>': 1, '   pre vi ou s ly</w>': 3, '   on y x</w>': 1, '   a mu let</w>': 3, '   le pre cha u n</w>': 2, '   le pre cha un s</w>': 1, '   s ea m us</w>': 13, '   pe d d l er</w>': 1, ' set</w>': 3, '   de bu n ked</w>': 1, '   di sor i en ted</w>': 4, '   lon dri g an</w>': 1, ' a mu let</w>': 1, '   ro od</w>': 1, '   r uni c</w>': 1, ' r uni c</w>': 1, '   cla ir vo y ant</w>': 1, '   in ha bi ting</w>': 1, '   do o z i e</w>': 1, '   f lan k</w>': 4, ' sho ts</w>': 1, '   ma li ce</w>': 3, '   gu ru s</w>': 1, '   co ve ts</w>': 2, '   un fe tt er ed</w>': 2, '   f le sh ed</w>': 1, '   un b ound</w>': 3, '   m our ned</w>': 2, '   sp en s er</w>': 3, '   el li o t t</w>': 2, '   ex pl or at ory</w>': 2, '   ba tt le fi el ds</w>': 1, '   su m mer s kill</w>': 1, '   un cla im ed</w>': 1, '   ta w d ry</w>': 1, '   your se</w>': 2, '   con di tion al</w>': 1, '   sa ted</w>': 1, '   in du l ged</w>': 1, '   5 3 </w>': 8, '   qu ar re ls</w>': 1, ' u t t</w>': 2, '   sh w o of</w>': 1, ' di em</w>': 1, ' att e mp ted</w>': 2, ' te mp t</w>': 1, ' ted</w>': 2, '   p b s</w>': 1, ' s for z a</w>': 2, '   e que st ri an</w>': 2, '   ob je ts</w>': 1, ' e que st ri an</w>': 1, '   a li sta ir</w>': 1, '   hu mp ti ed</w>': 1, '   du mp ti ed</w>': 1, '   li gh thou </w>': 1, ' s win ging</w>': 1, ' th i e f</w>': 3, ' wi tch cra ft</w>': 2, '   sor r</w>': 1, ' list en ing</w>': 2, '   k er o sen e</w>': 1, '   sp a tu la s</w>': 2, '   7 1 </w>': 2, '   hu mm m</w>': 1, '   mar in a ted</w>': 2, '   k ran e po o l</w>': 1, '   bro k ers</w>': 3, '   sto le y</w>': 1, '   sp ri t z ers</w>': 1, ' wa ter ing</w>': 1, ' gu in e a</w>': 1, ' w o p</w>': 3, '   un ma s cu l ine</w>': 1, '   la z i est</w>': 1, ' con s ci ou s ne ss</w>': 1, '   s mo o ch</w>': 1, '   bo ori sh</w>': 1, '   al fi e</w>': 3, ' lo o</w>': 2, '   u k ra ine</w>': 4, '   gra pp le</w>': 1, '   bi k er</w>': 2, '   ha ir sp ra y</w>': 1, '   co ll a p si ble</w>': 1, '   h er ba l</w>': 1, '   na tal ya</w>': 2, ' a par the id</w>': 1, '   k a pl an</w>': 4, '   di sp o sa ls</w>': 1, ' bu r</w>': 1, ' ger</w>': 35, '   fr en</w>': 1, ' ch</w>': 4, '   may f l ow ers</w>': 6, '   a i e</w>': 1, ' y i</w>': 1, ' me ssed</w>': 1, ' poli te</w>': 1, '   fo il ed</w>': 1, '   co de x</w>': 7, '   de co ded</w>': 3, '   clo g</w>': 1, ' co ver t</w>': 1, ' v a ti can</w>': 1, ' hu man i t ar i an</w>': 1, ' ch l ori de</w>': 1, ' li f ted</w>': 1, '   s for z a</w>': 2, '   re p li c a</w>': 2, ' g as</w>': 1, '   ho i st ing</w>': 1, '   min er v a</w>': 3, '   ba d min ton</w>': 1, '   pa s sa ge ways</w>': 1, '   re per to i re</w>': 1, '   an e c do tes</w>': 2, '   un my st er i ous</w>': 1, '   un ti r ing</w>': 1, '   the en k</w>': 1, '   mi la dy</w>': 1, '   ex ten ds</w>': 2, '   s lu r p</w>': 3, ' en de ar ing</w>': 1, ' bu cks</w>': 2, '   in gre di ent</w>': 1, '   al che my</w>': 2, '   li r a</w>': 4, '   sha z am</w>': 1, '   de ta i ling</w>': 1, '   au t re</w>': 1, '   ge la to</w>': 1, '   ni ven</w>': 2, ' bu n ny</w>': 1, '   s ke tch bo ok</w>': 1, '   ha w k me i ster</w>': 1, '   out b id</w>': 1, '   bu mm ere</w>': 1, '   fran k i</w>': 1, '   ha w k a sa u ru s</w>': 1, '   ta ter head</w>': 1, '   ban d stand</w>': 3, '   ha w k ster</w>': 1, ' ca bo t</w>': 1, ' p al ace</w>': 1, '   bu t l er head</w>': 1, ' ar i ta</w>': 1, ' so i t in ly</w>': 1, ' dan o</w>': 1, ' ho o t</w>': 1, ' po ll u te</w>': 1, ' di a be ti c</w>': 1, '   in car na te</w>': 2, '   sh mo e</w>': 2, '   sh mo es</w>': 1, '   e con om i sts</w>': 1, '   brea k d ow n s</w>': 1, '   mar ke ts</w>': 7, ' c ru mb le</w>': 1, '   ma s cu lin i ty</w>': 1, ' ea bi li ty</w>': 1, '   pro ton</w>': 1, '   br ow sing</w>': 3, '   bar keep</w>': 1, '   sh an king</w>': 1, ' a le x</w>': 1, '   con cor de</w>': 7, '   under e qui pped</w>': 1, '   s li mi est</w>': 2, '   p in at a</w>': 1, '   bu tt er fin ger</w>': 2, ' bo l sh ev i ck</w>': 1, '   qui ps</w>': 1, '   we en i e</w>': 3, '   we en i es</w>': 1, '   tri s cu it</w>': 1, '   bar e han ded</w>': 1, '   s cu m si c le</w>': 1, '   hu d su ck er</w>': 19, ' de e</w>': 2, ' do o</w>': 9, '   hu d s win ger</w>': 1, '   ho op su ck er</w>': 1, ' ga i e ty</w>': 1, '   sha z z a m me ter</w>': 1, '   din gu s</w>': 4, '   sti l son</w>': 1, '   de pre ss</w>': 1, '   nor vi ll e</w>': 29, '   war in</w>': 2, '   a x ed</w>': 4, '   ar ch uh</w>': 4, '   k nu ck le head</w>': 3, '   pi g g li es</w>': 1, '   o th uh</w>': 1, '   th a tch es</w>': 1, ' sp in n in</w>': 1, ' r ou n</w>': 1, ' ation</w>': 1, '   lea st ways</w>': 3, '   hu d su ck uh</w>': 1, '   m un ci e</w>': 10, '   mu ss bur ger</w>': 4, '   what sit</w>': 3, ' bu t t</w>': 5, '   s mi tty</w>': 3, '   ca ba l</w>': 1, '   i mi ta tions</w>': 1, '   lu m ba go</w>': 2, '   ad en o i ds</w>': 1, '   ch u mp s vi ll e</w>': 1, '   li fe b loo d</w>': 1, '   pr un ing</w>': 1, '   n in co m</w>': 1, '   n in com po op s</w>': 1, '   ma mi e</w>': 1, ' k ar ma</w>': 1, '   i be x</w>': 2, '   wi l de be est</w>': 1, '   war th o g</w>': 1, '   under b ru sh</w>': 2, '   g ru b s</w>': 1, '   bur rs</w>': 1, '   bea t ni ks</w>': 1, '   4 4 0</w>': 1, '   vi da li a</w>': 1, '   sh in er</w>': 1, '   i ce p ack</w>': 1, '   b y l ine</w>': 1, '   s wa ps</w>': 1, '   ho b no b s</w>': 1, '   ne w s room</w>': 2, '   s mo o ch er</w>': 1, '   bi as ed</w>': 1, '   m un ci an</w>': 1, '   ar gu s</w>': 2, '   ran t in gs</w>': 2, '   per i o di cal</w>': 1, '   k nu ck le</w>': 2, ' i ss</w>': 1, ' i ssed</w>': 1, ' gu ts</w>': 1, ' cou ra ge</w>': 5, ' com m on</w>': 1, '   po l y te ch ni c</w>': 1, '   v ac an cy</w>': 4, '   hu d</w>': 3, '   in d om i ta ble</w>': 1, '   a ll e me in i sch er</w>': 1, '   z ei t un g</w>': 1, '   w o l fi sh ly</w>': 1, '   ma il room</w>': 2, '   un a ll o y ed</w>': 1, '   war ing</w>': 4, '   st ri ve</w>': 5, '   en er ge ti ca lly</w>': 1, '   di li gen tly</w>': 2, ' u c c ess</w>': 1, '   bo il er pla te</w>': 1, '   w o es</w>': 3, ' e ta is</w>': 1, '   de st ro ye e</w>': 1, ' on si e u r</w>': 1, '   gon z a le z</w>': 1, '   a z al ea s</w>': 2, '   ter min e e</w>': 1, '   un di stu r bed</w>': 1, '   ri ver da le</w>': 4, '   re in ven ted</w>': 2, '   di sc re tion ary</w>': 1, '   bur ge on ing</w>': 1, ' h no ok</w>': 1, ' op e</w>': 3, ' i p sti ck</w>': 1, ' la me bra in</w>': 1, ' ch mo e</w>': 1, '   ex ce l</w>': 2, ' e du ca tion</w>': 2, '   b en ni e</w>': 4, ' ad en o i ds</w>': 1, '   ye ll ow st one</w>': 1, '   wa ter wor ks</w>': 1, '   a w du h s</w>': 1, '   fa h</w>': 1, '   h oun g an</w>': 7, '   ra da</w>': 2, '   sh an go</w>': 2, '   dam ba ll a</w>': 1, '   hou m for t</w>': 8, ' bri o c he</w>': 1, ' ago</w>': 2, '   figu re head</w>': 3, '   sor r ow ful</w>': 2, ' mi ser y</w>': 4, '   bro o ded</w>': 1, ' min d less</w>': 1, '   de can ter</w>': 4, '   t ro pi c s</w>': 1, '   di re c t re ss</w>': 1, '   o tt a w a</w>': 3, '   cle ment</w>': 1, '   ma d d en</w>': 1, '   un gen er ous</w>': 1, '   vi ci ou s ne ss</w>': 1, '   g lea m</w>': 2, '   g li tter</w>': 4, '   pu t re sc ence</w>': 1, '   for e st all</w>': 2, '   co l d ly</w>': 2, '   su per sti tions</w>': 4, '   d ru mm ing</w>': 2, '   b en j </w>': 1, '   p ack a ging</w>': 4, '   can a an</w>': 1, '   b en j i e</w>': 3, '   b la tty</w>': 1, '   sha ck le y</w>': 1, '   p sy cho an al y sts</w>': 1, '   un fa i th fu l ne ss</w>': 2, '   da lli an ce</w>': 1, '   ha l for d</w>': 4, '   mu s k</w>': 1, '   ha l for ds</w>': 1, '   w oo l en s</w>': 1, '   pre s ci ence</w>': 1, '   tr ends</w>': 1, '   li b be ts</w>': 14, '   ga mu t</w>': 1, '   h n n n</w>': 1, '   o ver came</w>': 1, '   v an ta ge</w>': 3, '   ex p li ci tly</w>': 2, '   tr ac ts</w>': 2, '   ex pen si ve ly</w>': 1, '   lan d s cap ed</w>': 1, '   to pi c s</w>': 1, '   gir l ho od</w>': 1, '   ac a de mi es</w>': 1, '   da h</w>': 1, '   op i a ted</w>': 1, '   se min al</w>': 2, '   e d war ds</w>': 9, '   c y ni c</w>': 1, '   si mi le</w>': 1, ' rea li z ation</w>': 2, '   min i st er ing</w>': 1, '   di sor g ani z ation</w>': 1, '   jo in er</w>': 1, '   tr ans for ma tive</w>': 1, '   uni t ar i an</w>': 1, '   bar ri ers</w>': 4, '   ph ar m ac e u ti cal</w>': 3, '   ab s con ded</w>': 1, '   tra v a i ls</w>': 1, '   ex i st en ti al</w>': 1, '   ad d l ed</w>': 2, '   re e du ca tion</w>': 1, '   a mar e t to</w>': 1, '   m ead</w>': 2, '   au r a</w>': 1, '   se ll able</w>': 1, '   hu mon go id</w>': 1, ' si st ers</w>': 1, '   re f ra in ed</w>': 1, '   pre c in c ts</w>': 2, '   man g l ed</w>': 4, ' cont ro lled</w>': 2, '   d or m er</w>': 2, '   du g g ar</w>': 2, '   re tr ac ed</w>': 1, '   e ck har t</w>': 2, '   si b ling</w>': 1, '   pu f fin s</w>': 1, '   k na p s ack</w>': 5, '   e di tions</w>': 1, '   o ver si gh ts</w>': 1, '   mi s de me an ors</w>': 1, '   ni gh t mu te</w>': 1, '   pa per b ac ks</w>': 2, '   st e t z</w>': 4, '   st or med</w>': 4, '   pl ed</w>': 1, '   tri sh</w>': 1, '   we st on</w>': 1, '   z er o ing</w>': 1, '   war fi e ld</w>': 3, ' cor ru p tion</w>': 2, ' o ver si gh ts</w>': 1, '   me lan cho ly</w>': 5, '   le st at</w>': 22, '   sp u n</w>': 2, '   de ca dent</w>': 2, '   dis course</w>': 3, '   th i mb le fu ll</w>': 1, '   ir ri ta tes</w>': 1, '   ta tt er ed</w>': 1, '   ch er i e</w>': 4, '   y ea ars</w>': 1, '   ve sti ge</w>': 1, '   bi d es</w>': 1, '   lo the</w>': 1, '   par i si ans</w>': 1, '   d ev i li sh ly</w>': 1, '   du ck lin gs</w>': 1, '   s wan s</w>': 2, '   du ck ling</w>': 1, '   sa v ou red</w>': 1, '   wi h out</w>': 1, '   si ck en ed</w>': 1, '   h er ea f ter</w>': 1, '   je sts</w>': 2, '   pr ac ti ca li ty</w>': 1, '   in di sc ri min a tely</w>': 1, '   a e s the te</w>': 1, '   sp l en d our</w>': 1, '   l er n t</w>': 1, '   i ma g in es</w>': 3, '   tr an sy l v ani a</w>': 3, '   b le ss es</w>': 1, '   c ru ci fi x es</w>': 1, '   1 7 9 1 </w>': 1, '   co pp er fi e ld</w>': 1, '   ga s li ght</w>': 1, '   t ke</w>': 2, '   con tra i re</w>': 3, '   mar g in ally</w>': 1, '   2 0 2 </w>': 3, ' k ar l a</w>': 1, ' ju dy</w>': 1, ' he m min g way</w>': 1, '   thin kn g</w>': 1, ' ti gh te ous</w>': 1, '   ti tu s</w>': 6, ' n an cy</w>': 1, '   do ck h and</w>': 1, '   k ar a o ke</w>': 2, '   s ou th por t</w>': 2, '   e st es</w>': 3, '   ga ff ing</w>': 1, '   wa y be</w>': 1, '   app o lo gi ze</w>': 1, ' r in gs</w>': 2, '   fin ger pa in ting</w>': 1, ' t our</w>': 3, ' di re ctor</w>': 3, '   dis cu ss in</w>': 1, '   wa a a a a h h h h h h h h h h</w>': 1, ' thinking</w>': 7, '   qui ck y</w>': 1, '   ba th ro b es</w>': 1, '   dea a ad d d d d</w>': 1, '   t y re ll ll</w>': 1, '   jo in n n n</w>': 1, '   u s ss</w>': 1, ' ta i</w>': 5, '   e qui li bri u m</w>': 2, '   w o ll st en</w>': 6, '   clo ver</w>': 1, '   the a</w>': 40, '   po se i don</w>': 1, '   ra ked</w>': 1, '   mar b le head</w>': 1, '   sta g ger ed</w>': 2, '   in co h er en tly</w>': 1, '   j ac ks</w>': 6, '   ni k o la s</w>': 8, '   ph er i d es</w>': 2, '   h er mes</w>': 2, '   vo tive</w>': 2, '   ban dy</w>': 1, '   v r y k o la k a</w>': 2, '   si ro c c o</w>': 1, '   dro s so s</w>': 5, '   si ck en s</w>': 1, '   re st le ss ne ss</w>': 2, ' bro ke</w>': 1, '   de sp o il ed</w>': 1, '   c r y p ts</w>': 1, '   pa pp a</w>': 2, '   au b y n</w>': 1, '   a le ther a</w>': 1, '   ca ta le p ti c</w>': 2, '   bi d d en</w>': 1, '   plan ing</w>': 1, '   v r y k o la k as</w>': 1, ' ca ta le p ti c</w>': 1, '   pre y ed</w>': 1, '   the o do si a</w>': 4, ' con su l</w>': 1, '   ad ri an ople</w>': 1, ' app o in ted</w>': 1, ' pa le</w>': 1, '   ran g es</w>': 2, ' lo gi sti c s</w>': 1, '   la d y s mi th</w>': 1, '   nu k d en</w>': 1, '   cor ph on</w>': 1, '   per se cu te</w>': 3, '   fe u ds</w>': 1, '   ba i le y</w>': 29, '   g ower</w>': 8, '   do g gon e de st</w>': 1, '   o d body</w>': 2, '   be d for d</w>': 8, '   bi ck</w>': 3, '   g in ger s na p</w>': 1, '   gr in d st one</w>': 1, '   wa in w right</w>': 8, '   la be ls</w>': 2, '   ba gh dad</w>': 2, '   sa mar k and</w>': 1, '   th i s a</w>': 1, '   ne cking</w>': 2, '   pi x i es</w>': 1, '   bu z z ard</w>': 2, '   fa tt ed</w>': 1, '   p in es</w>': 2, ' por gi e</w>': 1, '   si mo le ons</w>': 1, '   ro ck e fe ll ers</w>': 1, '   sha p in</w>': 1, '   ga lling</w>': 1, '   r ab ble</w>': 5, '   dis con ten ted</w>': 1, '   star ry</w>': 1, '   ro che ster</w>': 4, '   so y be ans</w>': 2, '   te en si e</w>': 1, '   la s so s</w>': 2, '   v ac a tions</w>': 4, '   in gi e</w>': 1, '   ha t ful</w>': 1, '   par th en on</w>': 1, '   s k y sc ra p ers</w>': 1, '   u mm mm m</w>': 3, '   ca bo ose</w>': 1, '   fi j i</w>': 13, '   ha vi land</w>': 1, '   wh ee e</w>': 1, '   pa d d ling</w>': 1, '   ye e</w>': 3, '   fu t z ing</w>': 1, '   bea ch ed</w>': 3, '   g lu b</w>': 1, ' g lu b</w>': 1, ' bu b ble</w>': 1, '   can es</w>': 1, '   s na g ged</w>': 3, '   g an g way</w>': 1, '   tur ki es</w>': 1, '   le e w ard</w>': 1, '   fo gar ty</w>': 2, '   9 0 8 </w>': 2, ' du ty</w>': 3, '   u p ton</w>': 1, '   di f fu sion</w>': 1, '   mon ta u k</w>': 1, '   ci d es</w>': 1, '   ni b bl er</w>': 1, '   r h y th mi c</w>': 1, '   car char a don</w>': 1, ' ca don</w>': 1, '   car ad an</w>': 1, '   f er r y bo at</w>': 1, ' ri di cu l ous</w>': 1, '   f an ny</w>': 2, '   se le c t men</w>': 2, '   l en</w>': 3, ' 1 2 0 0</w>': 2, '   ma the w</w>': 3, '   au r or a</w>': 1, '   an t ar c ti c</w>': 1, ' pa st</w>': 4, '   si l ver a</w>': 3, '   mm mm m morning</w>': 1, '   nu mb s k u lls</w>': 1, '   el k ins</w>': 1, '   s ki ers</w>': 2, '   r en tal s</w>': 2, '   ju ve</w>': 1, '   ni les</w>': 2, ' sa i ling</w>': 3, '   s k is</w>': 1, '   sta y sa il</w>': 1, '   may an</w>': 1, '   b re b ner</w>': 1, '   wi l co x</w>': 1, '   bu da pe st</w>': 1, '   bro o ki e</w>': 1, '   un t an g le</w>': 2, '   o x y gen ate</w>': 1, '   in di re c tly</w>': 1, '   ne ts</w>': 1, '   o ver man</w>': 6, '   su i ting</w>': 1, '   h y dro p ho bi a</w>': 1, '   sa s so on</w>': 1, '   bl ack ass</w>': 1, '   s cu b a</w>': 3, ' pre da tor</w>': 1, '   cap tur ing</w>': 3, '   por po i ses</w>': 1, '   di c ey</w>': 2, '   e mb r y o s</w>': 4, '   co d s w o pp le</w>': 1, ' bi t ers</w>': 1, '   k an gar oo s</w>': 1, '   s ki t ti sh</w>': 2, ' sen e ph r en ed</w>': 1, '   mu sc le head</w>': 1, '   gon do l a</w>': 2, ' s ki er</w>': 1, '   ma ss ac hu s se tt ans</w>': 2, '   o h h h h h</w>': 2, '   sh h h hi i i it</w>': 1, '   ba l</w>': 1, '   an t ar c ti c a</w>': 4, '   bu ll sho t</w>': 1, '   ca mer a man</w>': 4, '   cha pe ls</w>': 1, '   wa c o</w>': 1, '   ba s kin</w>': 1, '   p ac </w>': 2, '   in comp le tes</w>': 1, '   c ru mp et</w>': 1, '   ho a gi e</w>': 9, '   g ran d ma s</w>': 2, '   sh r in king</w>': 2, ' ou ri st</w>': 1, '   h en ny</w>': 1, '   youn g man</w>': 1, '   ba ha mi an</w>': 1, '   con co c ted</w>': 1, '   bur lin g ton</w>': 1, ' com pla in</w>': 1, '   char t ers</w>': 2, '   ta g ging</w>': 3, '   con ch</w>': 5, ' con fu se</w>': 1, '   t ro p o</w>': 2, ' nu ck</w>': 1, ' lo de</w>': 1, '   ro man ti c s</w>': 2, '   ee ls</w>': 1, '   m oun ta in side</w>': 2, '   ki tt en i sh</w>': 1, '   ja z zy</w>': 1, ' ch u tes</w>': 1, '   ja ki e</w>': 2, '   g in s ber gs</w>': 1, '   gu tt en ber gs</w>': 1, '   go l d ber gs</w>': 1, '   ber gs</w>': 2, '   fri e d man</w>': 38, '   wi p ers</w>': 2, '   er ent</w>': 1, ' bi ts</w>': 1, '   ha v ta</w>': 3, '   sp ose</w>': 1, '   e ssi on al</w>': 1, '   ci tr ine</w>': 5, '   i ev ed</w>': 1, '   p ee p ing</w>': 2, '   co z</w>': 10, '   bab ly</w>': 1, '   ory</w>': 1, ' bl an k</w>': 1, '   i eve</w>': 1, ' j en ni f er</w>': 3, '   sti tu te</w>': 2, '   li c</w>': 1, '   en se</w>': 1, '   im en tal</w>': 1, '   reme m</w>': 2, '   b er</w>': 3, ' ca sh</w>': 5, ' pu ff</w>': 1, '   z i pp o</w>': 3, '   st r in g ent</w>': 1, '   h y g</w>': 1, '   i en e</w>': 1, '   fu k k er</w>': 1, '   e tt e</w>': 1, '   e m se l ves</w>': 1, ' tra v is</w>': 1, '   bar be cu es</w>': 2, '   b la tt is</w>': 3, ' t ou ri st</w>': 1, '   do z er</w>': 3, '   el o</w>': 1, '   ser a to</w>': 2, '   en ts</w>': 2, '   go o dri dge</w>': 2, '   he in e man</w>': 1, ' mi ss ing</w>': 3, '   co er ce</w>': 1, '   i ll e g</w>': 1, ' ha sn</w>': 1, '   for e ca st</w>': 1, '   tri c y c le</w>': 1, '   e st ing</w>': 1, '   po sit</w>': 1, '   i ve</w>': 1, '   ni f er</w>': 1, '   st ri a tions</w>': 1, '   ra in f all</w>': 1, '   ser g</w>': 1, '   e ant</w>': 1, '   ner</w>': 1, '   no d dle</w>': 1, '   o sen e</w>': 1, ' f re ddy</w>': 2, '   of f er in</w>': 2, ' wi t ne ss</w>': 1, ' rea ds</w>': 1, '   le c ts</w>': 1, ' att ac ked</w>': 2, '   sh ar pen ing</w>': 2, '   app re c</w>': 1, '   i ate</w>': 1, '   a mb </w>': 1, '   w er</w>': 2, '   pla in ed</w>': 1, '   ac tly</w>': 1, '   co g ni z an ce</w>': 1, '   to p a</w>': 1, ' ha m let</w>': 2, ' thou gh ts</w>': 3, '   con ver</w>': 1, '   sa tion al</w>': 1, ' ho llow</w>': 2, ' mar ks</w>': 2, '   er no on</w>': 1, '   ople</w>': 1, '   fir st ly</w>': 1, '   u al</w>': 1, '   ce pt</w>': 2, ' e le ment</w>': 1, '   ca li for ni an</w>': 3, '   con fe ss es</w>': 2, ' pi z z a</w>': 1, '   a g</w>': 1, ' s er</w>': 1, '   i al</w>': 1, '   ki l</w>': 1, ' op er a</w>': 1, '   wh ass</w>': 1, '   tri mb le</w>': 2, '   in fi ll</w>': 1, ' b ru tal</w>': 1, ' tr ust</w>': 4, ' con fu sion</w>': 1, '   ja z z ed</w>': 1, ' lo s er</w>': 4, ' gu i re</w>': 1, '   hea t see k ers</w>': 1, '   su per st ar</w>': 4, '   sp ort boy</w>': 1, ' h on</w>': 1, '   ma st er ed</w>': 1, '   a i right</w>': 3, '   w r on g ne ss</w>': 2, '   1 1 8 </w>': 1, ' re sp on si ble</w>': 1, '   su g ars</w>': 2, '   ph h t</w>': 1, '   brea k up</w>': 1, '   a very</w>': 4, '   re v ved</w>': 1, '   c had</w>': 14, '   mu r k</w>': 2, '   d ev al u ed</w>': 1, '   wi l bur n</w>': 1, '   fa x ing</w>': 1, '   ti d well</w>': 3, '   har ra ss ing</w>': 1, '   g li mp e</w>': 1, '   a lo e</w>': 1, '   4 0 4 </w>': 1, ' 4 5 3 </w>': 1, ' 2 2 2 2 </w>': 1, ' sig ned</w>': 2, '   wa ter lo o</w>': 1, '   t sh t</w>': 1, '   an x i ou s ly</w>': 2, '   n n n n n</w>': 1, '   cu bi c le</w>': 2, '   mor ph ed</w>': 1, '   n ea l</w>': 1, '   j un k et</w>': 1, '   ac h in g ly</w>': 1, '   fro sted</w>': 1, ' sy n dro me</w>': 1, '   mi d life</w>': 1, ' bo tt om</w>': 1, ' run g</w>': 1, ' ne w ly</w>': 1, '   k a y de e</w>': 1, '   car din al s</w>': 8, ' con tr ac ts</w>': 1, '   s m i</w>': 3, ' de di ca tion</w>': 1, ' sh op li ft</w>': 1, '   po o ty</w>': 2, '   sh op li f ted</w>': 2, ' po o ty</w>': 1, '   1 0 3 </w>': 1, '   shi v ver</w>': 1, ' s wa ll ow ing</w>': 1, '   se i ge</w>': 1, '   ra s ci st</w>': 1, '   i con o gra ph y</w>': 1, '   ra s ci s m</w>': 1, '   por ty</w>': 1, '   mar que e</w>': 1, ' co in</w>': 2, '   k wa a a an</w>': 1, ' k wa n</w>': 1, '   8 2 </w>': 4, '   ri son</w>': 1, '   k ar o a ke</w>': 1, ' sh ow ing</w>': 2, '   e sp n</w>': 1, '   g ran d ly</w>': 1, '   p ou ting</w>': 1, ' hur ry</w>': 5, ' k a</w>': 1, '   n f l</w>': 1, '   vo ll e y b all</w>': 3, '   sin g le ho od</w>': 1, '   lo om ing</w>': 2, '   bo b b i</w>': 1, '   b p i</w>': 1, '   mar ce e</w>': 4, '   qu ar ter b ac ks</w>': 1, '   fo cu sing</w>': 4, '   en d or se men ts</w>': 1, '   han d ho l ding</w>': 1, '   bab y si tt ers</w>': 1, ' fe w er</w>': 1, '   s cu lly</w>': 41, '   c r on in</w>': 1, '   do o l er</w>': 1, ' i i i c</w>': 1, '   ki r by</w>': 8, '   su ff o ca t in g ly</w>': 1, '   u de s k y</w>': 1, '   tri c y cla to ps</w>': 1, '   ri c a</w>': 3, '   par a sa i ling</w>': 1, '   sor na</w>': 3, ' li fe time</w>': 1, '   ga la pa go s</w>': 1, '   ni le</w>': 1, '   en th u si a sts</w>': 1, '   hi l de br and</w>': 1, ' c li mb ers</w>': 1, '   nu bl ar</w>': 3, '   ac a de mi c s</w>': 2, '   in g en</w>': 8, '   sp in o sa u ru s</w>': 1, '   a e g y p ti c us</w>': 1, '   bar y on y x</w>': 1, '   su per pre da tor</w>': 1, '   su chi mi m us</w>': 1, '   s n out</w>': 1, ' dar win i s m</w>': 1, '   g li ding</w>': 1, '   u p d ra ft</w>': 1, '   pro to t y per</w>': 1, '   ra p tor</w>': 3, '   pa le on to lo g y</w>': 2, ' ac ce p ted</w>': 1, '   ju ra ssi c</w>': 4, '   ra p t ors</w>': 6, '   re son a ting</w>': 1, '   b on i ta s</w>': 1, '   a st r on om er</w>': 1, '   p rea ch y</w>': 1, '   we st ga te</w>': 2, '   fi x tur es</w>': 1, '   fi re pl ac es</w>': 2, '   ac ce s sor i es</w>': 2, ' ex por ts</w>': 1, '   re st ri c tions</w>': 1, '   u p w ind</w>': 1, '   a j ay</w>': 2, '   ro land</w>': 2, '   af ter s have</w>': 3, '   m om ba ss a</w>': 1, ' min i ma l</w>': 1, '   uni mp ea cha ble</w>': 1, '   har ding</w>': 19, ' s ca ven g ers</w>': 1, '   pu b li sh es</w>': 1, '   ha tch er y</w>': 2, '   o pp on en ts</w>': 4, '   sp li c ers</w>': 1, ' hu mi di ty</w>': 1, ' re x</w>': 10, '   h y en as</w>': 1, '   st e go sa u r</w>': 1, '   ne st b ound</w>': 1, ' in g en</w>': 1, '   ve lo ci ra p t ors</w>': 1, '   s ni pe</w>': 1, ' 6 0 0</w>': 3, '   1 9 0 4 </w>': 2, '   k ar i mo j o</w>': 1, '   8 7 0 0</w>': 1, '   t y ran no sa u rs</w>': 1, '   h er bi v or es</w>': 2, ' di sc lo sure</w>': 1, '   for ba de</w>': 1, '   in qu ir er</w>': 1, '   ca mp fi re</w>': 3, '   ju tt son</w>': 1, '   wh al ers</w>': 1, '   gu tty</w>': 1, ' te st ing</w>': 1, ' de st ru ction</w>': 2, '   char ac ter i sti c</w>': 1, '   bi r d shit</w>': 1, '   for ma tions</w>': 1, '   gu an o</w>': 3, '   mi sp l ace</w>': 1, '   v in di ca ted</w>': 2, ' pr in ci pa l</w>': 1, '   he i sen ber g</w>': 1, '   ton i c s</w>': 2, '   p li able</w>': 1, '   fi bu l a</w>': 1, '   in v a si ve</w>': 1, '   ma m mo th s</w>': 1, '   ni gh t l ine</w>': 1, '   a ir li ft</w>': 7, '   loo ter</w>': 1, '   ne d ry</w>': 8, ' sh ows</w>': 1, '   di ssi pa ted</w>': 1, '   un att en ded</w>': 1, '   l y s ine</w>': 5, '   en z y me</w>': 1, '   a min o</w>': 1, '   men th o l</w>': 2, '   vi able</w>': 3, '   do d g son</w>': 3, '   tri ke</w>': 1, '   i ll u stra ted</w>': 4, '   e co sy st em</w>': 2, '   pa le o</w>': 1, ' d na</w>': 1, '   ph ar m ac o lo gi cal</w>': 1, '   mi to ti c</w>': 1, '   l ab or ed</w>': 1, '   tu b ing</w>': 1, '   cor re c ta ble</w>': 1, '   see saw</w>': 1, '   mo t ori z ed</w>': 1, '   per fe c ting</w>': 2, '   bar f s</w>': 1, '   re gu r gi ta tes</w>': 1, '   un di ge sted</w>': 2, '   per i o di ci ty</w>': 2, '   di lo p ho sa u ru s</w>': 2, '   un f er ti li z ed</w>': 1, '   mi to s is</w>': 1, '   h er ds</w>': 2, ' b loo de d ne ss</w>': 2, '   pa le o bo t ani st</w>': 1, '   f ru stra tes</w>': 1, '   cre t ac e ous</w>': 2, '   po st er i or</w>': 1, '   li ga men ts</w>': 1, '   ve lo ci ra p tor</w>': 1, '   ta ph on om y</w>': 1, '   se x i s m</w>': 1, '   im per fe c tions</w>': 3, '   di st en ding</w>': 1, '   ori en ta tions</w>': 1, '   t y ran no sa u r</w>': 1, ' na ture</w>': 2, '   di g ger</w>': 5, '   ju ani to</w>': 5, '   in spe c tions</w>': 1, '   ho l a</w>': 6, '   bi en ven i do</w>': 1, '   an e u ri s ms</w>': 1, '   ro ad way</w>': 1, ' po ll u ting</w>': 1, '   b loo d su cking</w>': 1, '   mo a ts</w>': 1, '   a mp hi bi an</w>': 1, ' sa u ru s</w>': 2, '   br on to sa u rs</w>': 1, '   br ow s ers</w>': 1, '   gen n ar o</w>': 1, '   lo y</w>': 1, '   re c rea ted</w>': 2, ' cha mb er ed</w>': 1, '   br ac hi o sa u r</w>': 2, '   pa le o bo to ani st</w>': 1, '   te sti mon i al</w>': 1, ' s che du le</w>': 1, '   d or k a to ps</w>': 1, ' thin k te lli g ence</w>': 1, '   n ar r ow ly</w>': 1, '   pen e tra tive</w>': 1, '   lu d di te</w>': 1, '   ob li ter a ted</w>': 1, '   de for e sta tion</w>': 1, '   con d ors</w>': 2, '   pe sti ci d es</w>': 1, '   vi e w po in ts</w>': 1, '   ex p an ds</w>': 1, '   co d s w o ll o p</w>': 1, '   cha o ti ci an</w>': 2, '   un app re ci a ted</w>': 2, ' bu g</w>': 1, '   or lan do</w>': 3, '   ga z und he it</w>': 1, '   a h h h ch oo o</w>': 1, '   pen d le ton</w>': 4, '   sh or el ine</w>': 5, '   sa l v a g ea ble</w>': 1, '   lan d mar ks</w>': 2, '   in b red</w>': 4, '   p sy ch op a th s</w>': 3, '   t ou ri s m</w>': 2, '   j as</w>': 1, '   da ven por t</w>': 1, '   rea f fir m ing</w>': 1, '   for k li ft</w>': 1, '   im pa le ment</w>': 1, ' z om bi e</w>': 1, '   no i re</w>': 1, '   ha ll u c in ate</w>': 1, ' ne u ro l ar</w>': 1, '   supp re ss es</w>': 1, '   hi pp o ca mp us</w>': 1, ' sp r in g w o od</w>': 1, '   co e ds</w>': 1, ' as le ep</w>': 1, '   te en a ged</w>': 1, '   ne u ro lo gi cal</w>': 3, '   s om no le s ent</w>': 1, '   re ms</w>': 1, '   in s om no le s ence</w>': 2, '   h y po th a la m us</w>': 5, '   sle e p in ess</w>': 2, ' in s om no le s ence</w>': 1, '   i mb al an c ed</w>': 1, '   he ar ten ing</w>': 1, ' ra tion a li ze</w>': 1, '   o v ar i an</w>': 3, '   pre sc ri bed</w>': 1, ' di a g no sed</w>': 1, '   n y mp ho le p sy</w>': 1, ' we tter</w>': 2, '   doe s n it</w>': 1, '   e du ard</w>': 23, '   si l en c ed</w>': 1, '   en su ing</w>': 1, '   in ci tes</w>': 1, ' offi ci al</w>': 3, '   h er al ds</w>': 1, '   b lo ck ad ed</w>': 1, '   cen o ta p h</w>': 1, ' ca st le</w>': 1, ' to o ls</w>': 1, '   g ab ri el a</w>': 9, '   in sc ri b ing</w>': 1, '   ro ss man n</w>': 2, '   bur ge l</w>': 16, '   con c lu ding</w>': 1, '   er lan ger</w>': 2, '   de te st able</w>': 1, '   un p un c tu a li ty</w>': 1, '   com m uni qu es</w>': 1, '   o ver ra te</w>': 1, ' or l ac </w>': 1, '   r ab an</w>': 5, '   ob li ga t ory</w>': 1, '   wor k ma tes</w>': 1, '   per fe c ted</w>': 2, '   qu ar ri es</w>': 1, '   e li x ir</w>': 1, '   con tri ved</w>': 1, '   ja il er</w>': 4, '   re t rea ted</w>': 2, '   la men ted</w>': 1, '   br in ger</w>': 1, '   ni l</w>': 1, '   con com i t ant</w>': 1, '   u bi qui t ous</w>': 1, '   mu si l</w>': 2, '   la be lled</w>': 1, '   a gi ta tor</w>': 2, '   do ck e ting</w>': 1, '   li vi d</w>': 3, '   f en c ers</w>': 1, '   in com pa ti ble</w>': 1, '   sh ort chan ging</w>': 1, ' your se l ves</w>': 1, ' cl er k</w>': 1, '   de x ter ous</w>': 1, '   ev a ding</w>': 1, '   mu r na u</w>': 2, '   or l ac </w>': 4, '   ma l ev o l ent</w>': 1, '   ex ten ding</w>': 4, '   ci vi li z ing</w>': 1, '   bu n g ling</w>': 1, '   in ten ts</w>': 1, ' ra ff</w>': 1, '   ca st e</w>': 1, ' r ab an</w>': 3, '   a mer i k a</w>': 2, '   o s k ar</w>': 10, '   su n ba the</w>': 1, '   ro ss man</w>': 1, '   den ha m</w>': 13, '   re pre h en si ble</w>': 2, ' que en</w>': 2, '   en ge l h or n</w>': 1, '   pa in sta king</w>': 2, '   ac cu r ac y</w>': 1, '   ne o li th i c</w>': 1, '   re min i sc ent</w>': 1, '   ni as</w>': 1, '   de fi an tly</w>': 1, '   ar che o lo gi cal</w>': 6, '   ev i den ti al</w>': 1, '   ja k at a</w>': 1, '   an th ro po lo gi cal</w>': 3, '   for e st ry</w>': 1, '   li me ys</w>': 1, '   g ru b b ing</w>': 1, '   sto king</w>': 1, '   bo il ers</w>': 2, '   or din a tes</w>': 1, '   char ter ing</w>': 1, ' en ge l h or n</w>': 1, '   jo t</w>': 2, ' s k u ll</w>': 1, '   g in ting</w>': 1, ' in d on e si a</w>': 1, '   he m min g way</w>': 3, '   n ar ra tion</w>': 2, '   la e m ma le</w>': 1, '   god s sa ke</w>': 1, '   ar l y n</w>': 16, '   du mp er</w>': 22, '   tr in a</w>': 4, '   a h l ba ir</w>': 1, '   ca mo o</w>': 1, '   k lu te</w>': 13, '   can n es</w>': 2, '   che min de f er</w>': 1, '   sp an g l er</w>': 2, '   gr un e man n</w>': 26, '   m ck en na</w>': 30, ' ch ee ked</w>': 1, '   tra s k</w>': 4, '   z i pp i dy</w>': 1, '   sch men dri ck</w>': 1, '   f ab er</w>': 4, '   sp la sh ed</w>': 3, '   li g our in</w>': 5, '   tu sc ar or a</w>': 5, '   hi ra m</w>': 1, '   d ou bl er</w>': 1, '   i li a</w>': 9, '   fa ke out</w>': 1, '   may time</w>': 1, '   du mp ers</w>': 1, '   ca b ba g ev i ll e</w>': 1, '   bu b i</w>': 1, ' d y kes</w>': 1, '   ha y u v </w>': 1, ' den ti fi k y shu n</w>': 1, ' ri ver</w>': 1, '   hu mi di ty</w>': 1, '   a vo i ds</w>': 1, '   pro mi s cu i ty</w>': 2, '   he li por t</w>': 1, '   g ri ev an ce</w>': 2, '   dis cont ent</w>': 2, '   f re e k y</w>': 1, '   st re i ger</w>': 1, '   e mo tion a ll v </w>': 1, '   al v ar e z</w>': 4, '   k ri p t ar i u m</w>': 1, '   wi ll e w s k a</w>': 1, ' o h h h h</w>': 1, ' o h h h h h</w>': 1, ' om my</w>': 1, '   mm my</w>': 1, ' sus</w>': 2, '   af fi da vi ts</w>': 1, '   el in ore</w>': 1, '   f re e d man</w>': 1, ' un happy</w>': 1, '   j an t z en</w>': 1, '   sp ort s we ar</w>': 1, '   re la tes</w>': 1, '   re spe c ta bi li ty</w>': 3, ' ter</w>': 1, ' per man en tly</w>': 2, ' per man ent</w>': 1, '   ra tion ally</w>': 1, '   re v l on</w>': 5, '   r ou gh s</w>': 1, ' p ow</w>': 2, '   sch mi d t</w>': 15, '   ha s k ins</w>': 4, '   v s</w>': 3, '   app e ll ate</w>': 1, '   mm mm n p h</w>': 1, '   ad ju d ged</w>': 1, '   a war ded</w>': 2, '   re sp on dent</w>': 1, '   sha un e ss y</w>': 1, '   char le y</w>': 67, ' di v or c ed</w>': 1, '   ea s</w>': 1, '   the l</w>': 6, ' the l</w>': 1, '   l en o x</w>': 1, ' 0 8 0 0</w>': 1, '   k ha mb as</w>': 1, '   k ha m</w>': 2, '   l ha m o</w>': 3, '   k und u n</w>': 5, '   ti be t an</w>': 7, '   mo ther land</w>': 1, '   fin a li z ed</w>': 1, '   l ha s a</w>': 10, '   qu an ti ty</w>': 2, ' ev al u ate</w>': 2, '   lo d ging</w>': 1, '   g ri ev an ces</w>': 2, '   da la i</w>': 10, ' w u</w>': 5, '   dro m o</w>': 1, '   n g ab o</w>': 3, '   cha m do</w>': 3, '   j i g me</w>': 1, '   de le ga tions</w>': 1, '   ti be t ans</w>': 3, ' ch in e se</w>': 1, '   pro te sted</w>': 1, '   re bu tting</w>': 1, '   de ce p tive</w>': 2, '   po tal a</w>': 1, '   sh ee p s kin</w>': 1, '   un k in d ne ss</w>': 1, '   ar i ses</w>': 1, '   af f li c tions</w>': 1, '   sur f er in gs</w>': 1, '   sa k y a m un i</w>': 1, '   di sp er se</w>': 1, '   k ha mb a</w>': 2, '   li than g</w>': 1, '   a m do</w>': 1, '   re di st ri bu ted</w>': 1, '   lan d l or ds</w>': 1, '   re lin qui sh ed</w>': 1, '   de i ti es</w>': 1, '   min st ers</w>': 1, '   a mer i ca s</w>': 2, ' el</w>': 16, '   mor t ars</w>': 2, ' a ir cra ft</w>': 1, '   re de fin ing</w>': 1, '   al s ace</w>': 1, '   re g ent</w>': 7, ' ca st e</w>': 1, ' ex p lo i ta tion</w>': 1, '   ab o li sh ed</w>': 2, '   imp le ment</w>': 1, '   pre par at ory</w>': 1, '   nor bu </w>': 2, ' la vi sh</w>': 1, ' wi ll ful</w>': 1, '   b al der</w>': 1, '   re ting</w>': 2, '   r in po c he</w>': 3, '   ta k tr a</w>': 2, '   k u m bu m</w>': 1, '   re li qui sh</w>': 1, ' app ear ed</w>': 1, '   bi ck er man</w>': 3, '   fuck me at</w>': 1, ' li ves</w>': 2, '   en dan ger ment</w>': 2, '   pe ta</w>': 1, '   al z he im er</w>': 1, '   co h er ent</w>': 2, '   in co h er ent</w>': 1, ' co h er ent</w>': 1, '   s ki ll et</w>': 1, ' lan ding</w>': 1, ' he ctor</w>': 2, '   per pe tu i ty</w>': 2, ' d ra g on</w>': 1, '   so be k</w>': 1, '   c ro c</w>': 3, ' bi te</w>': 1, ' bi sho p</w>': 1, '   se t back</w>': 2, '   dis ru p tive</w>': 1, ' wal t</w>': 1, '   ha tch ling</w>': 1, '   ten ta ti ve ly</w>': 1, '   bra z en</w>': 4, '   ke y st on es</w>': 1, '   ke y st one</w>': 2, '   c ro c s</w>': 4, '   sa l t wa ter</w>': 2, '   dis cour te ous</w>': 1, '   he ck le</w>': 1, '   c y r</w>': 2, '   to po gra p hi c</w>': 1, '   tr an q </w>': 5, ' tru cks</w>': 1, '   th ra sh es</w>': 2, '   f la x e di l</w>': 1, '   pi t man s</w>': 1, '   au ton om y</w>': 1, '   sy co p han ts</w>': 1, '   s li cks</w>': 1, ' p ac i fi c</w>': 1, '   f la x</w>': 1, '   he e ds</w>': 1, '   mo o ing</w>': 1, '   god ly</w>': 2, '   b ack st ro kin</w>': 1, ' ca l m</w>': 1, ' mu sh ing</w>': 1, '   mu sh ing</w>': 2, '   ban da id</w>': 1, ' ma ine</w>': 3, '   we t lan ds</w>': 1, '   gu i sing</w>': 1, '   de i fi ed</w>': 1, '   ha tch lin gs</w>': 1, '   t re ks</w>': 1, ' ten ts</w>': 3, '   hu mor ed</w>': 2, '   pre hi st ori c</w>': 2, '   bi go ts</w>': 1, ' ki d ding</w>': 3, '   per ch</w>': 4, '   tu g ged</w>': 2, '   g ri me tt i</w>': 1, '   ni pped</w>': 1, '   pro ver b s</w>': 1, ' a po lo gi ze</w>': 1, '   pl ac id</w>': 1, '   gu r g l ed</w>': 1, ' din o sa u r</w>': 1, '   shi t bu t t</w>': 1, '   b ack st ab b er</w>': 1, ' ta pped</w>': 1, '   f re dri c a</w>': 4, '   bi m me l</w>': 4, '   be l ve d ere</w>': 6, '   un la w ful</w>': 2, '   f lo a ter</w>': 4, '   ge t me out</w>': 1, '   tur n k ey</w>': 1, '   chi l ton</w>': 13, '   ra sp a il</w>': 6, '   e k g</w>': 1, '   pa per c li ps</w>': 1, '   u v a</w>': 1, ' ta st e</w>': 1, '   ru b in</w>': 2, '   y ow</w>': 6, '   ca ter pi ll ars</w>': 1, '   sur in am</w>': 1, '   ad dre s see</w>': 1, '   gu mb </w>': 3, '   vo ca tion al</w>': 1, '   dre ss ma k er</w>': 1, '   re c y cl ed</w>': 1, '   rea d mi ssion</w>': 1, '   cle ar an ces</w>': 1, ' pa le tt e</w>': 1, '   ex hu ma tions</w>': 1, '   b re e ds</w>': 2, ' kn ack</w>': 1, '   ve c t ors</w>': 1, '   cor re la tion</w>': 2, '   mi g gs</w>': 8, '   mo f et</w>': 3, ' mo f et</w>': 1, '   fo ll ow up</w>': 1, '   p sy cho be ha vi or al</w>': 1, '   pi l ch er</w>': 1, '   man il ow</w>': 1, '   ma la ve s i</w>': 1, '   an th ra x</w>': 2, '   co ve ting</w>': 2, '   co v et</w>': 3, '   f ru s</w>': 1, '   wh ee dle</w>': 1, '   in ven t ori es</w>': 1, '   we ch s l er</w>': 1, '   tr ans se x u a li s m</w>': 1, '   tr ans se x u al s</w>': 2, '   ter n s</w>': 2, ' tur n s</w>': 1, ' pl u m</w>': 1, '   vi e w ed</w>': 1, ' la te st</w>': 1, '   t or men ts</w>': 2, '   dan g les</w>': 1, '   vi su a li z es</w>': 1, '   ex chan g es</w>': 1, '   ta b lea u x</w>': 1, '   ex hi l ar a ted</w>': 1, '   clo mp ing</w>': 1, '   dis cour te sy</w>': 1, '   fa v a</w>': 2, '   t ro p hi es</w>': 2, '   mi s lea ding</w>': 3, '   du om o</w>': 1, '   ev y an</w>': 1, ' mu l ti p le</w>': 1, '   hi ssed</w>': 1, '   ex pi r es</w>': 1, '   dan i el son</w>': 4, '   e mb er g</w>': 1, '   t ou gh en ed</w>': 2, ' fe ed</w>': 2, '   per for a ting</w>': 1, '   cha e ta x y</w>': 1, '   pi l ch</w>': 1, '   por t man</w>': 1, ' tu ck</w>': 2, ' than</w>': 6, '   af for d able</w>': 1, '   ne w ga te</w>': 1, '   con se qu en ti a li ti es</w>': 1, '   un ca s</w>': 1, '   wh ee lo ck</w>': 1, '   ch in g ac h go ok</w>': 4, '   se di tion</w>': 5, ' d un can</w>': 2, '   mi s re pre sen ted</w>': 1, '   m un r o</w>': 11, '   per i ph er y</w>': 1, '   comp li men ted</w>': 1, '   gra ve st</w>': 1, '   ma gu a</w>': 25, ' ha tr ed</w>': 1, ' lo a thing</w>': 1, '   in de ci si ve</w>': 1, '   for e go</w>': 2, ' sc out</w>': 1, ' pro ac h</w>': 1, '   ha i red</w>': 2, ' for t</w>': 1, ' tra pp ing</w>': 1, '   win th ro p</w>': 2, '   ca se ment</w>': 1, '   t y ran ny</w>': 3, '   car i ll on</w>': 1, '   tra d in</w>': 3, '   ca st le ton</w>': 1, '   h ori can e</w>': 1, '   al gon qu in</w>': 2, '   sen e c a</w>': 1, '   y en g ee se</w>': 9, ' ma gu a</w>': 3, '   hur ons</w>': 1, '   ab na kes</w>': 1, '   sa u k</w>': 1, '   mon t ca l m</w>': 4, '   he y w ard</w>': 1, '   s co t s man</w>': 2, '   po l t ro on</w>': 1, ' s co t s man</w>': 1, '   6 2 </w>': 1, ' rea s ons</w>': 1, ' ne go ti a ting</w>': 1, '   pro v in ci al s</w>': 1, '   sa pp ers</w>': 1, '   en qui re</w>': 1, '   att en</w>': 1, ' hu t</w>': 3, '   o x en</w>': 2, '   pa le face</w>': 1, '   ab so lu ti s m</w>': 1, '   ca pi tu la tion</w>': 1, '   can a da s</w>': 1, '   wi g wa ms</w>': 1, '   lo d g es</w>': 1, '   r en ard</w>': 16, '   su b til</w>': 1, '   ha tch et</w>': 16, '   s qu att ers</w>': 1, '   con sen ted</w>': 2, '   im min ence</w>': 1, '   ob st in ac y</w>': 1, '   re con no i ter</w>': 1, '   du p li ci ty</w>': 2, '   app ri se</w>': 1, '   ad mon i tions</w>': 2, '   an ta gon i st</w>': 1, '   ne ce s sa i re</w>': 1, ' e t re</w>': 2, '   vi gi l ant</w>': 2, '   en f ant</w>': 1, '   pro men e z</w>': 1, '   vi c to i re</w>': 1, '   mo t</w>': 1, ' or d re</w>': 1, '   a qu a man</w>': 1, ' dan a ki l</w>': 1, ' a da p ted</w>': 1, '   dan a ki l</w>': 2, '   a qu a ti c us</w>': 1, '   si x p ack</w>': 9, '   n ar ra g an se t t</w>': 1, ' ca tch ing</w>': 2, '   de jesus</w>': 11, ' imp lo sion</w>': 1, '   do ss</w>': 2, '   t y p ho on</w>': 3, '   b ow man</w>': 5, '   re ci r cu la ting</w>': 1, ' i ts</w>': 2, '   tr an so c ean</w>': 1, ' spe ci men</w>': 1, ' si x p ack</w>': 1, '   bl in k ers</w>': 1, '   ser b o</w>': 1, '   c ro at</w>': 1, '   s ma tt er ing</w>': 1, '   s wa hi l i</w>': 2, ' pre ca u tion</w>': 1, '   vi de o ta p ing</w>': 1, '   out c ro p</w>': 1, '   c rea k</w>': 1, '   c r ack ling</w>': 1, ' en chi la da s</w>': 1, ' re gen er ation</w>': 1, '   d or man cy</w>': 1, '   app en da g es</w>': 1, '   co b bi e</w>': 1, ' pro x i mi ty</w>': 1, ' b ow man</w>': 1, ' ja me son</w>': 2, ' ti p</w>': 2, '   j on e sy</w>': 2, ' wh ys</w>': 1, ' de b bi e</w>': 2, '   h in ge</w>': 1, '   dr ac h ma s</w>': 1, ' fa ted</w>': 1, '   sc ar face</w>': 3, '   cl ar et</w>': 1, '   s mi th y</w>': 4, '   ru m ou red</w>': 1, '   bu d gi e</w>': 2, '   ra in for est</w>': 1, '   or an g</w>': 1, ' u t an</w>': 1, '   lon s da le</w>': 1, '   ba z z a</w>': 3, '   sh ar pi sh</w>': 1, '   j d</w>': 3, '   shi tes</w>': 1, '   su n bed</w>': 1, '   1 3 7 </w>': 2, '   do zy</w>': 1, ' st ro si ty</w>': 1, '   sta gs</w>': 1, '   wan k ers</w>': 1, '   har ms</w>': 1, '   shi te</w>': 7, '   g ee z ers</w>': 2, '   ne i gh b ou rs</w>': 2, '   min c er</w>': 1, '   co l our ful</w>': 1, '   ni ck er</w>': 1, '   ho o d w ink</w>': 1, '   o i</w>': 2, '   ke bab </w>': 3, '   sa w n</w>': 2, '   bea ks</w>': 1, '   co ok y</w>': 1, ' d u</w>': 5, '   cu p id</w>': 4, '   ar m ou red</w>': 1, '   co pi ous</w>': 1, '   g an j a</w>': 1, '   h or t</w>': 1, ' cu l tu ra li st</w>': 1, '   bu g ger ed</w>': 4, '   or o z c o</w>': 1, '   gu er r a</w>': 1, '   p ac hu c o</w>': 1, '   wa sh bur n</w>': 1, '   a ma do</w>': 8, '   in fr ac tion</w>': 2, '   in n is</w>': 2, '   mo ja do</w>': 1, '   s w ea t sh op s</w>': 1, '   i ll e g al s</w>': 3, ' so l di er</w>': 2, '   mor di da</w>': 1, '   ci u dad</w>': 4, '   mu cha ch o</w>': 2, '   we t b ac ks</w>': 1, '   bi d ne ss</w>': 4, ' pre ss es</w>': 1, ' u h m</w>': 2, '   bea dy</w>': 1, ' ti gh tly</w>': 1, ' al ter ca tion</w>': 1, '   s qu ea k er</w>': 1, '   a g gi es</w>': 1, ' son of ab i tch</w>': 1, '   sta u b ac h</w>': 1, '   3 1 0</w>': 1, '   ho se a</w>': 1, '   lon gh or n s</w>': 2, '   in di o</w>': 1, ' al ph on se</w>': 1, '   le or a</w>': 1, ' o t is</w>': 1, ' in di an</w>': 1, '   pa y n es</w>': 1, '   pi e d ra s</w>': 1, '   ne gra s</w>': 1, ' cou p le</w>': 2, '   p da</w>': 3, '   se min o le</w>': 2, '   ru st l ers</w>': 3, ' ra i d ers</w>': 2, '   o sc e o l a</w>': 2, '   ok ee cho be e</w>': 2, '   s la ve ho l d ers</w>': 1, ' ar my</w>': 1, '   ca ba llo</w>': 1, ' a ver age</w>': 1, '   ca mi on</w>': 1, '   ja v el in a</w>': 1, ' ha l f way</w>': 1, '   re y</w>': 2, '   ll an ta s</w>': 1, '   pri s ci ll a</w>': 2, '   wal nu ts</w>': 3, '   fi st fi ght</w>': 1, '   po in se t ti a</w>': 1, '   no p al s</w>': 1, '   y u c ca s</w>': 1, '   j ack r ab bi ts</w>': 2, '   ma s ons</w>': 2, ' war ning</w>': 2, '   de mar ca tion</w>': 2, '   su ce ss fu ll</w>': 1, '   what cha ca ll it</w>': 1, '   re f er e e</w>': 1, '   men u do</w>': 1, '   sp r int</w>': 1, '   dar k town</w>': 2, ' poli sh</w>': 1, '   se min o les</w>': 1, ' ten go</w>': 1, '   e s co pe to</w>': 1, '   ban di do s</w>': 1, ' h out</w>': 1, '   ll a ma</w>': 3, '   je fe</w>': 1, '   ten go</w>': 1, '   o tr a</w>': 1, '   a m gi o</w>': 1, '   proble ma s</w>': 1, '   ll an ta</w>': 1, ' bi en ven i do</w>': 1, '   te j as</w>': 1, '   gon z al es</w>': 1, '   mo le st as</w>': 1, '   ten e mo s</w>': 1, '   ro s ar i a</w>': 2, '   ar a stra do</w>': 1, '   cor ri en te</w>': 1, '   ven g a</w>': 1, '   pu e do</w>': 1, '   ori ll a</w>': 1, '   d on de</w>': 2, '   per di do</w>': 2, '   sen or a</w>': 4, '   por fi ri o</w>': 3, '   z a y as</w>': 1, '   v a mo s</w>': 1, '   ra pi dam en te</w>': 1, '   ti pi c o</w>': 1, '   no vi a</w>': 1, '   ver dad</w>': 1, '   pu e de</w>': 1, '   e s to</w>': 1, '   an se l ma</w>': 1, '   je f a</w>': 1, '   qui en es</w>': 1, '   e sto s</w>': 1, '   pa sa do</w>': 1, '   ac ci den te</w>': 1, '   ti en e</w>': 1, '   mi e do</w>': 1, '   qui en</w>': 1, '   bu en as</w>': 1, '   no ch es</w>': 1, '   lin do</w>': 1, '   co c he</w>': 1, ' ma s</w>': 1, ' bu ll e ts</w>': 1, '   f en ton</w>': 1, '   on n ac oun a</w>': 1, '   fi sh ba it</w>': 2, ' fi sh ba it</w>': 1, '   po o ter</w>': 1, '   hu mm in</w>': 2, '   sc ra p in</w>': 1, '   i mon na</w>': 1, '   b le d so e</w>': 3, '   ro ad house</w>': 2, '   b le d so</w>': 1, '   ma son i c</w>': 1, '   ac com mo da tion</w>': 2, ' s an ti ago</w>': 1, '   tr ou b le ma k ers</w>': 2, '   pa di ll a</w>': 1, '   sen t in el</w>': 3, ' fr on ter a</w>': 1, '   cu i dad</w>': 1, ' he ar t th ro b</w>': 1, '   ha m st ers</w>': 2, ' ma m i</w>': 1, '   an g lo s</w>': 2, '   ma m i</w>': 2, '   pa lo ma</w>': 3, '   te j an o</w>': 1, '   e la l c o</w>': 1, '   a de lan te</w>': 1, '   c li en tes</w>': 1, '   e sp er an</w>': 1, '   stu pi der</w>': 2, '   me s kin</w>': 1, '   pi l ar</w>': 5, ' coun ty</w>': 2, '   ch u ch o</w>': 1, '   ca br on</w>': 1, '   n an do</w>': 2, ' ten th</w>': 2, '   pi ti es</w>': 1, ' s mar t</w>': 3, '   ra tt les</w>': 1, '   g is</w>': 2, '   brea k sho t</w>': 1, '   fr on ter a</w>': 1, '   ar ro y o</w>': 4, '   cl er o e</w>': 1, ' a m our</w>': 21, '   v in o vi ch</w>': 4, '   v al en t in</w>': 6, '   pi m m</w>': 5, '   de si der i o</w>': 4, '   l ev i ta te</w>': 2, '   d or o the a</w>': 3, '   i ll u si on i sts</w>': 4, '   i ll u si on i st</w>': 3, '   o v ation</w>': 1, ' v ani sh ed</w>': 1, '   f ra u d ster</w>': 1, '   ta per t</w>': 5, '   me ssi a h s</w>': 1, '   ha v an as</w>': 1, ' wal k</w>': 4, '   vo ca tion</w>': 1, '   vi vo vi ch</w>': 1, '   re sur re c ted</w>': 2, '   wa v </w>': 1, '   ven e er</w>': 1, '   l ev i ta tion</w>': 2, '   me w</w>': 1, '   wai t list ed</w>': 1, '   lu f than s a</w>': 1, '   la vi sh</w>': 1, '   da i k any a ma</w>': 1, '   can d le li ght</w>': 3, '   hi ro mi x</w>': 2, '   sc ow l</w>': 2, '   fran co is</w>': 3, '   por c he</w>': 1, '   k a z u z o</w>': 2, '   z o e</w>': 17, '   5 6 0 1 </w>': 1, '   p se u d on y m</w>': 2, '   wa u gh</w>': 4, '   i ke ban a</w>': 1, '   so o</w>': 4, '   pen t ac le</w>': 3, '   jo se f</w>': 3, '   an ago</w>': 1, '   wal t z es</w>': 1, '   qu ar ks</w>': 1, '   vi z ni ck</w>': 13, '   bi r d son</w>': 11, '   k el son</w>': 10, '   u r su l a</w>': 1, '   l ar kin</w>': 1, '   se cu l ar</w>': 5, ' lo gi cal</w>': 1, ' ye lling</w>': 1, '   cre d ence</w>': 1, '   offi ci ate</w>': 1, '   re sc in ding</w>': 1, '   be little</w>': 3, '   ra v in gs</w>': 2, '   mi e tt e</w>': 2, ' tri u mp h ed</w>': 1, '   r h</w>': 2, '   de u ter on om y</w>': 1, '   al t ars</w>': 1, '   gra ven</w>': 2, ' gr oun ding</w>': 1, '   l ar ea u x</w>': 4, '   st ru g g l ed</w>': 3, '   ten u ous</w>': 2, ' vi si ons</w>': 1, '   m ck en z i e</w>': 1, '   s an c ti f y</w>': 1, '   out wi tting</w>': 1, '   nu mer i cal</w>': 1, '   di sa st r ou s ly</w>': 1, '   sa l w en</w>': 1, ' k a y y y y y y</w>': 1, '   n ar ci ssi st</w>': 6, '   n ar ci ssi s m</w>': 4, '   whi sh</w>': 1, '   s z ab o</w>': 1, '   gra ye st</w>': 1, '   man i pu la tor</w>': 3, '   as ser ts</w>': 1, ' de mon i c</w>': 1, '   my sti f ying</w>': 1, '   g ran d kid</w>': 1, '   f ac a de</w>': 1, '   qu in cy</w>': 18, '   tu pp er</w>': 1, ' pri ss y</w>': 1, '   co ok out</w>': 1, '   re c ru i t ers</w>': 1, '   tom boy</w>': 1, '   si dr a</w>': 3, '   d or a</w>': 1, '   k y r a</w>': 3, '   re c ru i ter</w>': 1, '   re b oun ds</w>': 2, '   dri b bl ed</w>': 1, '   ac l</w>': 3, '   fu z z</w>': 1, '   dam ni t t</w>': 1, '   vi ta le</w>': 1, '   k er ry</w>': 3, ' c li pp ers</w>': 2, '   ton ya</w>': 1, '   mu l ho ll and</w>': 7, '   sp al ding</w>': 3, '   co o chi e</w>': 1, '   ea st on</w>': 1, '   n on a</w>': 2, '   as si sts</w>': 1, '   ho o chi e</w>': 1, '   bo l de st</w>': 1, ' c li ps</w>': 1, '   bu s sin ess</w>': 3, '   a v i</w>': 2, '   sur er y</w>': 1, '   bar e han ds</w>': 1, '   par ma</w>': 2, '   as si t ant</w>': 1, '   t j </w>': 4, '   m ack y</w>': 1, '   par ti dge</w>': 1, '   me ta sta si z ed</w>': 1, '   list en er</w>': 3, '   thin k ni g</w>': 1, ' cu r se</w>': 1, ' lie</w>': 2, '   bi ll in g sle y</w>': 2, '   o ver looks</w>': 2, '   4 2 2 </w>': 3, '   te m pu ra l</w>': 1, ' man di bu l ar</w>': 1, '   t m j </w>': 2, '   da me sti c</w>': 1, '   ar li ght</w>': 1, '   in co ven i en c ed</w>': 1, ' or din a tor</w>': 1, '   we ther</w>': 2, '   me te or lo gi cal</w>': 1, '   en d or s men ts</w>': 1, '   pro f fe s or</w>': 1, '   co o li o</w>': 1, '   ca u ze</w>': 1, '   con fe ss or</w>': 2, '   bu st a</w>': 1, ' r ouble</w>': 1, '   we en in</w>': 1, '   c r ack as</w>': 1, '   bo d y st ack as</w>': 1, '   di ck to o t in</w>': 1, '   ma stu r ba t in</w>': 1, '   tri g g a</w>': 1, '   bu tch a</w>': 1, '   ju ven i ll e</w>': 1, '   ra pp er</w>': 2, '   under pri ve la ged</w>': 1, '   wa s su p</w>': 1, '   inter u pt</w>': 1, ' car b on</w>': 2, '   gra p hi te</w>': 1, ' a h h h</w>': 2, '   chi d l r en</w>': 1, '   bu s sin es</w>': 1, ' i i</w>': 12, '   tom mor row</w>': 3, '   su b tr ac ting</w>': 1, '   cla u di us</w>': 1, '   ex o d us</w>': 2, '   wi llow</w>': 35, ' ru in ed</w>': 1, '   du ll ne ss</w>': 2, ' ru b</w>': 2, ' a du b</w>': 1, '   a vi d ly</w>': 2, '   in ta gi ble</w>': 1, '   ac ci den ta ly</w>': 1, '   ho spi ce</w>': 3, '   dro pp er</w>': 2, '   t oun ge</w>': 2, ' pr ac ti cal</w>': 1, ' ge t that</w>': 1, '   na me wa s li l y see</w>': 1, ' por ce la in</w>': 1, ' li ly</w>': 2, '   ci gar e tt e e</w>': 1, ' no t my</w>': 1, '   gi mm me</w>': 1, '   m ack ey</w>': 8, ' p hi l</w>': 2, ' shi t ba lls</w>': 1, '   5 0 9 </w>': 1, ' 9 0 2 7 </w>': 1, '   cl ar y f ying</w>': 1, ' att ack</w>': 1, '   mu ff y</w>': 3, ' se du ce</w>': 3, '   de story</w>': 1, '   lin ger ing</w>': 2, ' in si ght</w>': 1, ' under stan ding</w>': 1, ' list en er</w>': 1, '   con ver sa tion ally</w>': 1, '   se du c er</w>': 1, '   se du ce e</w>': 1, '   sa me th in</w>': 1, '   su c ce ding</w>': 1, ' y ad d da</w>': 1, ' y ad da</w>': 2, '   b ing</w>': 1, '   re li gi ou s ly</w>': 1, '   u no f fi cal</w>': 1, '   lan g t ree</w>': 1, '   cont ro ling</w>': 1, '   offi ci a ly</w>': 1, '   ber ke ly</w>': 2, '   u cl a</w>': 4, ' 8 9 </w>': 2, '   te l ev i son</w>': 1, '   pro f fe sion</w>': 1, '   ad re ss</w>': 1, '   di ss con ne c ted</w>': 1, '   po kin</w>': 2, '   pre ff er</w>': 1, '   mar ice</w>': 1, '   f lu z z i es</w>': 1, '   po qu el in</w>': 1, ' tra ge dy</w>': 1, ' pla y w right</w>': 1, '   ca ther</w>': 1, ' j im my</w>': 1, '   cla u i da</w>': 1, '   be cu a se</w>': 1, '   ga tor</w>': 3, ' sa u cy</w>': 1, '   th i c ke</w>': 1, '   ha i m</w>': 1, '   om mi tt ed</w>': 1, '   in te st ac y</w>': 1, '   la y w er</w>': 1, '   con r act</w>': 1, '   pri ve la ge</w>': 1, '   att or n er y</w>': 1, ' in de cen cy</w>': 1, '   de x ad r ine</w>': 1, '   mm m h m m</w>': 1, '   s wan k</w>': 1, ' pe an u t</w>': 1, '   8 1 8 </w>': 1, ' 7 5 3 </w>': 1, ' 0 0 8 8 </w>': 1, ' c m on</w>': 2, '   ar a m is</w>': 16, '   por th o s</w>': 8, '   a th o s</w>': 19, '   p hi lli pp e</w>': 22, '   an g ling</w>': 2, '   ba sti ll e</w>': 5, '   go b let</w>': 3, '   tur mo il</w>': 3, '   as sor t ment</w>': 1, '   su r</w>': 3, '   uni ting</w>': 1, '   pa p ac y</w>': 1, '   we an ed</w>': 1, '   re v ere</w>': 1, '   e mb le m</w>': 3, '   ma g da l en e</w>': 1, '   mo p ing</w>': 2, '   re v el ing</w>': 1, '   we i gh ty</w>': 1, '   or da in</w>': 1, '   pro f an e</w>': 4, '   ni pp le</w>': 5, '   lu r es</w>': 1, ' p hi lli pp e</w>': 1, '   re pl ac e men ts</w>': 1, '   un ready</w>': 1, '   fro mb er ge</w>': 5, '   re jo in ed</w>': 1, '   ex e cu tes</w>': 1, '   beau for t</w>': 4, '   v ac a ted</w>': 1, '   h un ch b ac ks</w>': 2, '   tu t ors</w>': 2, '   be d cha mber</w>': 2, '   comp li ant</w>': 1, '   un hur t</w>': 1, '   ja pe th</w>': 1, '   o be di ah</w>': 1, '   z e bu l on</w>': 1, '   he z e ki ah</w>': 1, '   nu lli fi es</w>': 1, '   wh ar ves</w>': 1, '   dis re gar ded</w>': 1, '   tru d ell</w>': 6, '   z y d ow s k i</w>': 4, '   mar vo s a</w>': 11, '   tu b b s</w>': 8, '   stan ton</w>': 6, ' to es</w>': 1, '   f an ned</w>': 1, ' stan ton</w>': 1, ' se ed</w>': 1, ' 8 3 </w>': 3, '   b al an ces</w>': 2, '   sa l ar i es</w>': 3, ' mar vo s a</w>': 1, '   l ev en wor th</w>': 1, '   1 4 9 0</w>': 2, '   b re ck in ri dge</w>': 2, '   cl on ed</w>': 3, '   lin k up</w>': 1, '   sc ra mb l er</w>': 1, ' b re ck in ri dge</w>': 1, ' 4 6 7 </w>': 1, ' 0 9 7 2 </w>': 1, ' what sa matter</w>': 1, ' un u su al</w>': 2, '   na h h h</w>': 1, '   l ar y n x</w>': 1, '   pro d ded</w>': 1, '   ei de te k er</w>': 1, '   si ck en</w>': 1, ' ar ran g es</w>': 1, ' be coming</w>': 1, ' fa i ry</w>': 3, '   b loo d sta ins</w>': 1, '   le e ds</w>': 11, '   le ck tor</w>': 33, '   a my tal</w>': 2, '   wi ll in g ha m</w>': 1, '   ga la ti ans</w>': 2, '   pa i red</w>': 1, '   sc ri p tu ra l</w>': 1, '   ja il house</w>': 2, '   in den ts</w>': 1, '   bi te mar k</w>': 1, ' i x</w>': 2, ' con tra st</w>': 1, '   ta tt l er</w>': 4, '   an il ine</w>': 1, '   d yes</w>': 1, '   in ks</w>': 1, ' ta tt l er</w>': 1, '   te le x ed</w>': 1, ' han di ca pped</w>': 1, '   fo ge l</w>': 1, '   me i gs</w>': 2, '   j ac ob i</w>': 8, '   bo l t cu tter</w>': 2, '   z e ll er</w>': 1, '   j im mi e</w>': 2, '   v t r</w>': 2, '   pe ori a</w>': 2, ' att l er</w>': 1, '   j on g g</w>': 1, ' gra ha m</w>': 1, '   3 8 6 0</w>': 1, '   le ctor</w>': 3, ' bro k en</w>': 2, '   l oun ds</w>': 12, '   e z i o</w>': 1, '   p in z a</w>': 1, '   vi de o ga mes</w>': 1, '   ha y st ack</w>': 2, '   bi r min g ha m</w>': 11, ' s nu ck</w>': 1, '   cor n ea s</w>': 1, ' i a m i</w>': 1, ' i mes</w>': 1, '   c r y p to gra ph y</w>': 1, '   re b a</w>': 3, ' ou ting</w>': 1, '   fri ca ti ves</w>': 1, '   si bi lan ts</w>': 1, '   m c cla in</w>': 1, '   doll ar h y de</w>': 1, '   sh er man s</w>': 1, '   har e li p</w>': 1, '   d read</w>': 9, '   har le qu in ed</w>': 1, '   clo th ed</w>': 2, '   ti ti ll ate</w>': 1, '   a ton ing</w>': 1, '   under going</w>': 1, '   ra di an ce</w>': 1, '   pu r po se ful</w>': 1, '   a vi d</w>': 2, '   t ee th mar ks</w>': 1, ' t y le</w>': 1, '   3 6 8 0</w>': 1, '   po in ter</w>': 1, '   j ac ob is</w>': 1, '   a w l</w>': 1, ' in ti ma te</w>': 2, '   a un ts</w>': 4, '   f li ck er ed</w>': 1, '   f en c ed</w>': 1, '   mm mm h</w>': 1, '   g ran d boy</w>': 1, '   bo o ger</w>': 6, '   sc an din a vi an</w>': 1, '   v h s</w>': 1, '   th rea der</w>': 1, '   fi tt ers</w>': 1, '   gar re t t</w>': 2, '   be g ru dge</w>': 2, ' inter e st ing</w>': 1, '   out do or</w>': 3, '   b ack y ar ds</w>': 1, '   di sa d v an ta g es</w>': 1, '   fu mb les</w>': 3, '   the ma ti c</w>': 1, '   app er ce p tion</w>': 1, '   m f</w>': 1, ' la y man</w>': 1, '   ca ll ers</w>': 4, '   cor n fi e ld</w>': 1, ' ra t ers</w>': 1, ' com mer ci ally</w>': 1, '   te f l on</w>': 1, '   e h h h</w>': 1, ' fu l fi lling</w>': 1, '   vi su a li z ation</w>': 2, '   in w ard</w>': 1, ' in tri gu ing</w>': 1, ' min d bo g g ling</w>': 1, ' hea d ac he</w>': 1, ' in du c ing</w>': 1, '   po st mo der n</w>': 1, '   la t k a</w>': 2, '   ne wh a</w>': 1, ' la t k a</w>': 1, '   f on z i e</w>': 3, '   brea k out</w>': 1, '   im per son ate</w>': 2, '   l un ch bo x es</w>': 1, '   ne wh art</w>': 1, '   ca spi an</w>': 4, '   ca spi ar</w>': 2, '   li th u ani a</w>': 1, '   sin g al on gs</w>': 1, '   be ss er man</w>': 1, '   sh oo m</w>': 2, '   da z</w>': 1, '   sh to p</w>': 1, '   ri le</w>': 1, ' ma u i</w>': 1, ' w ow i e</w>': 1, ' s an ta</w>': 1, ' can c er</w>': 1, ' ha ll el u ja h</w>': 1, '   c rea ti ve ly</w>': 1, '   ho a x es</w>': 2, '   z mu da</w>': 1, ' p hi lo so p hi ca lly</w>': 1, '   p hi lo so p hi ca lly</w>': 1, '   har ra h</w>': 1, ' bur le s qu e</w>': 1, '   c rea king</w>': 1, '   g or s k y</w>': 2, '   me shu g a</w>': 1, ' g or s k y</w>': 1, '   sen t in el s</w>': 1, '   ju r is</w>': 1, '   cho i</w>': 1, '   c y p her</w>': 6, '   re in ser t</w>': 1, '   go o p</w>': 1, ' z us</w>': 2, '   tr ans la t ors</w>': 4, '   tr ans la ting</w>': 2, '   be j ee z us</w>': 1, '   er i e</w>': 2, '   a po c</w>': 1, '   no ts</w>': 1, '   te th ers</w>': 1, '   un ki ll able</w>': 1, '   sp a w ned</w>': 2, '   mar v el ed</w>': 1, '   2 1 9 7 </w>': 1, ' out put</w>': 2, '   sp lin ter</w>': 4, '   ir on i ca lly</w>': 2, '   b al b o</w>': 1, ' beli ev ers</w>': 1, '   ran king</w>': 2, '   de j u</w>': 1, '   di sa b les</w>': 1, '   z i on</w>': 7, '   ser u ms</w>': 1, ' 1 0 9 </w>': 1, '   g ro pp i</w>': 6, '   sp ar k l ers</w>': 2, '   ch in x</w>': 1, ' as h</w>': 1, '   pro p he sy</w>': 1, ' hou </w>': 2, '   sa ye st</w>': 2, '   thr ower</w>': 3, '   2 3 5 </w>': 5, '   par l our</w>': 1, '   gi o v an in o</w>': 1, '   pa y ed</w>': 1, ' a men</w>': 3, '   ev il do er</w>': 1, ' joh n ny</w>': 2, '   n ev a mind</w>': 1, '   under sig ned</w>': 1, '   gu ar ran te e</w>': 1, '   wa ver ly</w>': 2, ' u p town</w>': 1, ' cu m par i</w>': 1, ' wor ri es</w>': 1, ' vi to</w>': 1, '   gen o ve se</w>': 1, '   he m in</w>': 1, ' com me</w>': 1, '   chi a ma</w>': 1, ' h en ning</w>': 1, '   he m ing</w>': 1, '   de f er red</w>': 1, ' gre g ory</w>': 1, '   m ac om b er</w>': 2, '   gu ev ar a</w>': 2, '   u mm mm mm mm mm mm m</w>': 1, '   ear har d t</w>': 1, '   pro pri e t or ship</w>': 1, '   he i sted</w>': 3, '   a st ar</w>': 1, '   wh er</w>': 1, '   qu in ce</w>': 19, '   bar i t one</w>': 1, '   ba la la i k a</w>': 1, '   pe t ro ssi an</w>': 1, ' par ri sh</w>': 1, ' god da m</w>': 3, '   ro se tt e</w>': 1, ' fe u i ll e</w>': 1, '   mar o ons</w>': 1, '   st ru c ting</w>': 1, '   p al t z</w>': 1, '   mor bi di ty</w>': 1, '   lu i s a</w>': 1, '   pa i ll ar de</w>': 1, ' ga me</w>': 1, '   f c c</w>': 1, '   g ri gi o</w>': 1, '   ri e s ling</w>': 1, '   par ri sh</w>': 18, '   out fi e ld</w>': 1, '   pl u sh</w>': 1, '   j if</w>': 1, ' la u r a</w>': 1, '   s cu d der</w>': 1, '   b on te cou </w>': 20, '   sp ee di er</w>': 1, '   pro vi sor y</w>': 1, '   di sc re</w>': 1, '   i c ers</w>': 1, '   gen t le man lin ess</w>': 1, '   g ro ton</w>': 1, '   u tes</w>': 2, '   ci ous</w>': 1, '   who pp ers</w>': 1, '   bur r ow ed</w>': 1, '   cre d en</w>': 1, '   ti al s</w>': 1, '   sp on se</w>': 1, '   er sted</w>': 1, '   con ven</w>': 1, '   a bu n</w>': 1, '   dan tly</w>': 1, '   ad d i</w>': 1, '   ad j our n</w>': 1, '   ture</w>': 3, ' mi th</w>': 1, ' j on es</w>': 1, ' hou gh ts</w>': 1, '   ki ck off</w>': 1, '   men ts</w>': 2, '   s m ac ks</w>': 1, '   un sc ra mb le</w>': 1, '   w ob b ling</w>': 1, '   s pp li ed</w>': 1, '   qu in</w>': 1, ' ce e</w>': 1, '   ke y cha ins</w>': 1, '   o ver thinking</w>': 1, '   ra h ti d</w>': 1, '   du tty</w>': 1, '   f ac e ty</w>': 3, '   t in gs</w>': 1, '   ba d ne ss</w>': 4, '   brea d f ru it</w>': 1, '   in n ers</w>': 1, '   ok a y in</w>': 1, '   ra ss</w>': 1, '   mi sta h</w>': 2, '   si sta h</w>': 1, '   i re y</w>': 1, '   nu tt in</w>': 2, '   w i</w>': 3, '   d ere</w>': 1, '   ob eah</w>': 2, '   p le tely</w>': 1, '   ro ti ss er i e</w>': 1, '   ab a ted</w>': 1, '   c of</w>': 1, '   pi ci ous</w>': 1, '   for ti tu de</w>': 3, '   a im less</w>': 1, '   du l ging</w>': 1, '   com man de er ed</w>': 1, '   pu ta tive</w>': 1, '   u sa ble</w>': 2, ' po on ing</w>': 1, '   bu s i</w>': 2, '   per m is</w>': 1, '   sion</w>': 2, '   che w y</w>': 2, '   r ary</w>': 1, '   me m</w>': 1, '   gra tu la tions</w>': 1, ' ra p ture</w>': 2, ' pa ssion</w>': 2, '   im par ting</w>': 1, '   c er</w>': 1, '   tain</w>': 1, '   a or ta</w>': 5, '   ir re par ab </w>': 1, '   pro g no sti ca tions</w>': 1, '   i li tes</w>': 1, '   g sta a d</w>': 1, ' h re w</w>': 1, '   di sc on</w>': 1, '   cer ted</w>': 1, '   p an ting</w>': 2, '   qu es</w>': 2, '   minu e ts</w>': 1, '   mi ll en ni u ms</w>': 1, '   a e ons</w>': 1, '   comp oun ded</w>': 1, '   pi qu ed</w>': 1, ' ob ses</w>': 1, ' e li ber a tely</w>': 1, '   z a g ged</w>': 1, '   fin ni sh</w>': 3, '   cu s to</w>': 1, '   di ans</w>': 1, '   ce i ve</w>': 1, '   di ke</w>': 1, '   re ce de</w>': 2, ' ce le bra tion</w>': 1, '   fo ge y</w>': 1, '   ver b o</w>': 1, ' li gh t ning</w>': 2, '   cor in th</w>': 1, '   pri cked</w>': 1, '   wh ar ton</w>': 1, '   de li ri ou s ly</w>': 3, '   u p most</w>': 1, '   der vi sh</w>': 1, '   ti t mi ce</w>': 1, '   sh ar p sh oo t</w>': 1, '   6 6 </w>': 1, ' \t \t       </w>': 12, ' \t \t \t   </w>': 10, ' \t \t \t \t \t \t \t       </w>': 4, ' \t \t \t \t \t \t \t     </w>': 3, ' \t \t \t \t \t \t \t   </w>': 3, '   3 0 4 </w>': 3, ' \t \t \t \t </w>': 22, ' \t \t \t \t \t \t \t </w>': 7, '   g ran t z</w>': 3, ' \t \t \t \t     </w>': 2, ' \t \t \t \t \t \t   </w>': 2, ' \t \t \t \t \t \t </w>': 9, '   do d d</w>': 5, '   b li ss fu lly</w>': 2, ' \t \t     </w>': 7, '   \t \t \t \t \t \t \t     </w>': 1, ' \t \t \t     </w>': 4, ' \t \t \t       </w>': 3, '   j an k is</w>': 5, '   \t \t \t \t \t     </w>': 1, ' \t \t \t \t \t \t     </w>': 2, '   ga mm ell</w>': 3, ' \t \t \t \t \t \t       </w>': 2, ' \t \t \t \t   </w>': 3, ' \t \t \t \t \t       </w>': 1, ' \t \t \t \t \t   </w>': 1, ' \t \t \t \t       </w>': 1, ' le ar ning</w>': 2, '   a mp he ta m ine</w>': 1, '   na mi bi a</w>': 1, '   go tt in g en</w>': 1, '   tru dy</w>': 5, '   tri age</w>': 1, '   ar ra ys</w>': 2, '   imp ort ers</w>': 1, '   ma gu da</w>': 3, '   jo s</w>': 18, '   y er o</w>': 4, '   fee b</w>': 3, ' pa tri a ted</w>': 1, '   al on z o</w>': 11, '   pre cu r sor s</w>': 3, '   g un fi ght</w>': 2, '   gu a j ir a</w>': 1, '   pen in su l a</w>': 2, '   lon d on o</w>': 1, '   p ru den ti al</w>': 1, '   a ll sta te</w>': 1, ' fin an ci al</w>': 1, '   vi e j o</w>': 2, '   sa mb a</w>': 1, '   k i</w>': 10, '   gr in go s</w>': 1, '   mo j i ta s</w>': 5, '   ther af ore</w>': 1, ' prob able</w>': 1, ' app li es</w>': 1, '   ru bi o</w>': 1, '   lu c in da</w>': 1, '   lu an da</w>': 1, '   ma l i</w>': 2, '   e co le</w>': 1, '   po l y te ch ni qu e</w>': 1, '   mb a</w>': 2, '   m ac on</w>': 1, '   an go l an</w>': 1, '   mo z a m bi qu e</w>': 1, '   ver da do</w>': 3, '   co b b le st on es</w>': 1, '   bo de gu i ta</w>': 1, '   me di o</w>': 1, '   de f ying</w>': 3, '   gra t is</w>': 1, '   su c cu mb ed</w>': 1, ' ga ll on</w>': 1, '   tr an ss hi p</w>': 1, ' ex tra s</w>': 2, '   o ver town</w>': 1, '   a qu a ti c</w>': 2, ' co ch i</w>': 1, '   s wi te k</w>': 3, '   ro man c ing</w>': 3, '   tr an ss hi p ment</w>': 3, '   j it</w>': 1, '   me th a mp he ta m ine</w>': 1, '   af g ha n</w>': 3, ' o c ean</w>': 3, '   tr an ss hi p men ts</w>': 2, ' ton na ge</w>': 1, '   rea s su ran ces</w>': 2, '   a wa c s</w>': 2, ' a u</w>': 1, ' pr in ce</w>': 1, '   tw ea k ers</w>': 4, '   do p ers</w>': 1, '   hi j o</w>': 1, '   po ll ack</w>': 1, '   a v g as</w>': 1, ' ru ssi ans</w>': 1, '   tr an sp o</w>': 1, '   supp li ers</w>': 3, '   re ci di vi st</w>': 1, '   bl ack ma il ed</w>': 2, '   du ba i</w>': 1, ' lon d on o</w>': 1, '   n l r</w>': 3, '   om gs</w>': 1, '   mon go ls</w>': 1, '   h r t</w>': 1, '   at y pi cal</w>': 1, '   en gen d ers</w>': 1, ' for e bo ding</w>': 1, '   inter a gen cy</w>': 2, ' 9 2 </w>': 1, '   fu j im a</w>': 1, '   sta te s vi ll e</w>': 1, '   h or o s co p es</w>': 2, '   fran c ine</w>': 1, '   ri c car do</w>': 1, '   n ar co tra f fi cking</w>': 1, '   inter di c tions</w>': 1, '   fr on ted</w>': 1, '   su pre m ac i st</w>': 2, '   i dent</w>': 2, '   te le f l or a</w>': 1, ' sa lu ta tions</w>': 1, ' c ri me</w>': 1, ' re lo ca ted</w>': 2, '   ni ger i an</w>': 2, '   for go</w>': 1, '   car te l</w>': 13, '   pa ki stan i</w>': 3, '   tr an ss hi pped</w>': 1, '   pre cu r s or</w>': 1, '   ra s</w>': 1, '   t an u r a</w>': 1, ' a y ma h</w>': 1, '   f al s</w>': 1, '   i gu a z u</w>': 1, '   con ve y an ce</w>': 2, '   ca y man</w>': 1, '   lan d f all</w>': 1, '   b la d d ers</w>': 2, '   imp or ta tion</w>': 1, '   pre de ter min ed</w>': 1, '   lon gs</w>': 1, '   la ts</w>': 1, '   1 4 0 0</w>': 1, '   pa y lo a d</w>': 2, '   car a v el s</w>': 1, '   7 2 7 </w>': 3, ' sp ea k ers</w>': 1, '   z o d da</w>': 1, '   li vi g</w>': 1, '   mor ey</w>': 1, '   fo l ding</w>': 3, '   bi t sy</w>': 1, '   mu d d l ed</w>': 3, '   ke e</w>': 4, ' ri st</w>': 4, '   f re u di an</w>': 1, '   sc ri b ba ge</w>': 1, '   sh e e</w>': 7, '   ra t so</w>': 8, '   be d bu g</w>': 1, '   un pl u g</w>': 3, '   win i f red</w>': 6, '   t wi st y</w>': 1, '   s ki mp y</w>': 1, '   gi mp</w>': 2, '   h un ch back</w>': 2, '   f lu e</w>': 2, '   tw at</w>': 1, '   pa go da s</w>': 1, '   per go la s</w>': 1, '   en ri c o</w>': 2, '   s lo b b er</w>': 3, '   ne di cks</w>': 1, '   me e</w>': 1, ' i d dle</w>': 1, '   jo y less</w>': 1, '   lon e s om en ess</w>': 1, '   for ni ca tor</w>': 1, '   pen i</w>': 1, ' ci ll in</w>': 1, ' bo pp er</w>': 1, '   z i at</w>': 4, '   h er ni a</w>': 3, '   as in a</w>': 1, '   su b st ru c ture</w>': 1, '   pi ck po ck e ts</w>': 1, '   ha mi d ou </w>': 1, ' fi g g ers</w>': 1, '   o l su n</w>': 2, '   er i ch</w>': 2, '   ge tch m is</w>': 2, '   ha ps</w>': 1, '   ti me ta b les</w>': 2, '   ca t ac o om b s</w>': 1, ' wee ps</w>': 1, '   ga st ro head</w>': 1, '   ne c d it</w>': 2, '   ye si l</w>': 6, '   min e fi el ds</w>': 2, '   c y p ru s</w>': 1, '   sho pp e</w>': 2, '   pa le st in i an</w>': 1, '   as l an</w>': 1, ' su l a</w>': 1, ' bu l a</w>': 1, '   hi sh ra d y o</w>': 1, '   k o gu s</w>': 2, '   d ou b le time</w>': 1, '   na mi d ou </w>': 1, ' ca t ac om b s</w>': 1, '   b ac la v as</w>': 1, ' er i can</w>': 1, '   su k</w>': 1, '   hi yes</w>': 1, '   a y i p</w>': 2, '   vi l y u m</w>': 1, ' ye si l</w>': 1, '   y un an</w>': 1, '   fr on ting</w>': 3, '   he mor ra ge</w>': 1, '   t b</w>': 1, ' e pi de mi c</w>': 1, '   car bo h y d ra tes</w>': 4, '   s ould</w>': 1, '   bl in t z es</w>': 2, '   bur r ows</w>': 1, '   s wi tch room</w>': 1, '   bu tt side</w>': 1, '   ch u y</w>': 2, '   man ti ds</w>': 1, '   mi mi c</w>': 4, '   si r i</w>': 3, '   po t ro a st</w>': 2, '   a p ter i ds</w>': 1, '   me ta x on y ch a</w>': 1, '   god man i</w>': 1, '   men op au sa l</w>': 1, ' oo the c a</w>': 1, '   e g g case</w>': 1, ' we ir d bu gs</w>': 1, '   ba ster</w>': 2, '   s an d b last</w>': 1, '   ex cre tions</w>': 1, '   win d th or n e</w>': 6, ' i g g</w>': 2, ' g land</w>': 1, '   cer a mi c</w>': 1, '   can d le l it</w>': 1, '   cle ary</w>': 2, ' ex per im en tal</w>': 1, ' cha p ter</w>': 2, '   co ck a do o die</w>': 1, ' bra kes</w>': 1, ' c li ff</w>': 1, ' han g ers</w>': 1, '   ba k er s fi e ld</w>': 2, '   gra ve di g ger</w>': 1, '   han d made</w>': 2, '   cor ra sa ble</w>': 1, '   s mu d g es</w>': 1, ' gra in ed</w>': 1, '   mi me o</w>': 1, ' f oo l er</w>': 1, '   sa le s la dy</w>': 1, '   no v ri l</w>': 2, '   ca ss er o le</w>': 4, '   g ri ff in</w>': 5, '   ki d der</w>': 2, '   1 8 7 1 </w>': 1, '   cha sta in</w>': 4, '   tra mp y</w>': 1, ' sc ra mb l ed</w>': 1, '   s lu m</w>': 2, '   den ts</w>': 3, '   sin d ell</w>': 1, ' di stan ce</w>': 1, ' le st</w>': 1, '   au to bi o gra p hi cal</w>': 1, '   k ni ck</w>': 1, '   ru m ma ging</w>': 1, ' po ck et</w>': 1, '   ne k h or vi ch</w>': 13, '   sc ru t in i z ing</w>': 1, ' sp li tting</w>': 1, '   nu an ce</w>': 1, '   au ssi es</w>': 1, '   n y ah</w>': 19, '   u l ri ch</w>': 1, '   be ll in is</w>': 1, ' hi e f</w>': 1, ' cha l k</w>': 1, '   bi o c y te</w>': 7, '   be ll er op h en</w>': 1, '   pe tr i</w>': 1, '   cu r b ing</w>': 1, '   por tra y</w>': 1, '   a tri u m</w>': 1, '   mi s di re ction</w>': 2, ' wa ll is</w>': 1, '   wa ll is</w>': 1, '   be ll er op h on</w>': 11, '   in tu i tions</w>': 1, '   b lo kes</w>': 5, '   o sc or</w>': 1, '   p in po in ting</w>': 2, '   o s ci ll a ting</w>': 1, '   coun ter sur ve i ll an ce</w>': 1, '   g ps</w>': 1, '   ba ir d</w>': 1, '   in ven ts</w>': 1, ' n y ah</w>': 1, '   e than</w>': 77, '   char m ers</w>': 2, '   au ssi e</w>': 1, '   for e war ned</w>': 1, '   for ear med</w>': 1, '   q an ta s</w>': 1, '   2 7 3 5 </w>': 1, ' de sti tu te</w>': 1, ' y o o</w>': 1, ' na d</w>': 1, '   co er c ing</w>': 1, '   in sin u ate</w>': 1, ' p in ch ed</w>': 1, '   ad ver se</w>': 2, ' ar en a</w>': 1, ' se tting</w>': 1, '   sen ors</w>': 1, '   dis con er ting</w>': 1, '   ex p un ged</w>': 2, '   c ani ster</w>': 2, '   de bri e f ing</w>': 2, ' di f fi cu lt</w>': 2, ' er r ori sts</w>': 1, '   pi lling</w>': 1, '   gra d s k i</w>': 5, ' be ll er op h on</w>': 1, '   k li ck</w>': 1, '   en ve lo p ed</w>': 1, '   e pi de mi c s</w>': 1, '   br un y</w>': 2, '   re com b in ing</w>': 1, '   an ti bi o ti c</w>': 3, '   b ac ter i u m</w>': 1, '   stra ins</w>': 3, '   in f lu en z as</w>': 1, '   an a to ly</w>': 1, '   mi e di ev </w>': 3, ' di mi tr i</w>': 2, '   k a si mo v </w>': 2, '   la p to ps</w>': 2, '   t g v </w>': 2, '   no c</w>': 10, '   mi red</w>': 2, '   k ri e ger</w>': 6, '   l ac he z</w>': 1, ' a gu e u e</w>': 1, '   ki tt ri dge</w>': 11, ' ki tt ri dge</w>': 2, ' vi en s</w>': 1, '   d ev i ation</w>': 1, '   vi sc o</w>': 1, '   di sa v ow ed</w>': 3, '   re con si der ing</w>': 1, '   im f</w>': 5, '   ki tt er i dge</w>': 2, '   mo le h un t</w>': 2, ' go li t sy n</w>': 1, '   go li t sy n</w>': 5, '   3 1 4 </w>': 2, '   bl ow back</w>': 1, '   b ow ti e</w>': 2, '   ex fi l tra tion</w>': 1, '   dis re pu ta ble</w>': 1, '   p hel ps</w>': 1, '   re in sta te</w>': 1, '   f la v our</w>': 1, '   6 8 6 </w>': 1, '   pro to t y p es</w>': 2, '   cra y</w>': 1, '   sti ck ell</w>': 2, '   ph in ea s</w>': 1, '   gh o st co m</w>': 1, '   pi e ce m ea l</w>': 1, ' en ti re</w>': 1, '   r f</w>': 1, '   ten able</w>': 1, '   ex fi l</w>': 1, '   tu b s</w>': 1, ' ea st er n</w>': 1, '   su pre me ly</w>': 1, ' thou </w>': 5, '   gi de ons</w>': 1, '   en r ou te</w>': 1, '   u p gra ding</w>': 1, '   com pu ter i ze</w>': 2, '   a li as es</w>': 2, '   f l y fi sh ing</w>': 1, '   ou gh ter ard</w>': 1, '   s l ough</w>': 1, '   ki ev </w>': 2, '   ch un ne l</w>': 1, '   mo ir a</w>': 1, '   sha w fi sh</w>': 1, '   ti c in o</w>': 3, '   r on ny</w>': 11, '   ca m mar er i</w>': 8, '   ca st or in i</w>': 2, '   s n ow f la kes</w>': 1, '   st or y bo o ks</w>': 1, '   ha ve ta</w>': 1, '   cha g all</w>': 2, '   ru int</w>': 2, '   s li c er</w>': 3, '   chri ss y</w>': 1, '   man i co tta</w>': 1, '   chi r o</w>': 2, '   ro s ar i es</w>': 1, '   m our n ers</w>': 1, '   ro te</w>': 2, '   mu l ti p li ca tion</w>': 1, ' be d room</w>': 1, '   fu ri o so</w>': 1, '   er r</w>': 2, '   be d ev ere</w>': 4, '   p lo ver</w>': 1, '   te m per ate</w>': 1, '   mer ce a</w>': 2, '   bri t ons</w>': 8, '   de f ea tor</w>': 1, '   g or ge</w>': 1, '   cla d</w>': 2, '   sa mi te</w>': 1, '   der i ves</w>': 2, '   f ar ci cal</w>': 1, '   sh ru b ber y</w>': 10, '   pen g</w>': 2, '   w u m</w>': 1, ' n i</w>': 4, '   en chan ter</w>': 3, '   nee e</w>': 2, '   w om</w>': 1, '   tri u mp h s</w>': 1, '   ha v n</w>': 1, '   ba tt le men ts</w>': 1, '   ta un ter</w>': 1, '   lo i mb ard</w>': 1, ' a llo</w>': 1, '   e et</w>': 1, '   a a a a ar r r r r r g g gh h h</w>': 3, '   la un ce lot</w>': 6, '   ga la had</w>': 9, '   t in der</w>': 1, '   mo i st en ed</w>': 1, '   b int</w>': 1, '   lo b bed</w>': 1, '   an ar ch o</w>': 1, ' sy n di ca li st</w>': 1, '   cour ti ers</w>': 1, '   out da ted</w>': 2, '   im per i a li st</w>': 1, '   per pe tu a tes</w>': 1, ' den n is</w>': 1, '   p sa l ms</w>': 1, '   a ver ting</w>': 1, '   w art</w>': 5, '   n in e p ence</w>': 2, '   bri d ge kee per</w>': 2, '   as sy ri a</w>': 1, '   app ro ac he th</w>': 1, '   ei gh t sc ore</w>': 1, '   und re ss ing</w>': 2, '   z oo t</w>': 2, '   ne sc ess</w>': 1, '   rea li sed</w>': 4, '   a a a a ar gh h h</w>': 1, '   w we</w>': 1, '   a a a ar gh h</w>': 1, '   ar m our</w>': 1, '   ga ll an tly</w>': 2, '   af oo t</w>': 3, '   p ra tt l er</w>': 1, '   t ar t an</w>': 1, '   p ra lin es</w>': 1, '   sp on g es</w>': 1, '   b al mor al</w>': 6, '   sho ve lling</w>': 1, '   p ou l ti ces</w>': 1, '   gh i lli e</w>': 1, '   tru ss</w>': 1, '   p on son by</w>': 4, '   ro y au m</w>': 1, '   fa v ori ti s m</w>': 1, '   ar ri vi st e</w>': 2, '   mar qu in o</w>': 1, '   win d s or</w>': 7, '   ber ti e</w>': 2, '   ir re sp on si bi li ty</w>': 1, '   in f er na lly</w>': 1, '   a m ba s sa d ors</w>': 1, '   ta u to lo g y</w>': 1, '   s lu r</w>': 2, '   gla sa lt</w>': 1, '   e qu er ri es</w>': 1, '   e qu er ry</w>': 1, '   brea k fa sts</w>': 2, '   wa tch ful</w>': 1, '   ma l con ten ts</w>': 1, '   re pu b li c ani s m</w>': 1, '   no b le st</w>': 1, '   lo ch na g ar</w>': 1, '   ba ir n s</w>': 1, '   de e side</w>': 1, '   f en i ans</w>': 1, '   app re ci a ting</w>': 1, '   ten n y son</w>': 1, ' hi gh land</w>': 1, '   cra ob ha n</w>': 2, ' ge an m ch no</w>': 2, ' f hi ad ha i ch</w>': 1, '   e pi cen ter</w>': 1, '   go ver n an ce</w>': 1, '   sp ate</w>': 3, '   ab er de en</w>': 3, '   bo a st ing</w>': 1, '   dis su ad ed</w>': 1, '   pro tr ac ted</w>': 1, '   er y si pe la s</w>': 1, '   sig na t ori es</w>': 1, '   dr un k en ess</w>': 1, '   un che er ed</w>': 1, '   un gu i ded</w>': 1, '   se c lu ded</w>': 2, '   cl ar en don</w>': 10, '   c ow es</w>': 1, '   ve er ing</w>': 1, '   pro gre ssi ve ly</w>': 2, ' char ac ter</w>': 1, '   pa tt er ned</w>': 2, '   v ere</w>': 3, '   di f for d</w>': 1, ' su b li ma ting</w>': 1, '   fr ac tal s</w>': 1, ' re p li ca ting</w>': 1, ' e s ca pe</w>': 1, '   rea c ts</w>': 2, '   a da p ts</w>': 2, '   d ow gi e</w>': 1, '   y o de l</w>': 1, '   bea ch w o od</w>': 1, '   under stan d in</w>': 2, ' tru ly</w>': 1, '   ke sh er</w>': 4, '   o ver d ra w n</w>': 1, '   co c o</w>': 8, '   b on ner</w>': 2, '   ri ta s</w>': 1, ' ti ts</w>': 3, '   el ms</w>': 4, '   se l w y n</w>': 4, ' wa ll ace</w>': 1, ' s co t t</w>': 1, '   han d sti tch ed</w>': 1, '   ex pe di en cy</w>': 2, '   un for seen</w>': 1, '   m c ca y</w>': 9, '   gir der</w>': 1, '   o ver s</w>': 1, '   de ci ma te</w>': 1, '   co pi er</w>': 1, '   per son el</w>': 1, '   de p le tes</w>': 1, '   bo ff ing</w>': 3, '   mu r der land</w>': 3, '   ir on ed</w>': 1, '   un pa id</w>': 4, '   pi p s qu ea k</w>': 3, '   d un ge ons</w>': 2, '   wa b bi ts</w>': 1, '   imp al ed</w>': 1, '   k o ja k</w>': 2, '   ri g li on i</w>': 1, '   h y d ra u li c s</w>': 3, '   s qu ir re lly</w>': 1, '   ton i gh ts</w>': 1, '   z ac har y</w>': 1, '   be lli ger ent</w>': 2, ' par k</w>': 5, ' gu e sts</w>': 4, '   e m be z z le</w>': 1, ' ho b bi es</w>': 1, '   a qui re</w>': 1, '   z ac h ory</w>': 1, '   chi ces</w>': 1, ' be l ch</w>': 1, '   c ro cks</w>': 1, '   m c cra y</w>': 1, '   t or te ll i</w>': 1, '   e p co t</w>': 2, '   u r in ation</w>': 1, '   e mp ha ti c</w>': 2, '   m c nu g get</w>': 1, '   f lin t st on es</w>': 1, '   ex p lo i ta tion</w>': 8, '   hu mp h</w>': 1, ' le c tur es</w>': 1, '   no t ori ou s ly</w>': 1, '   mu l do v an</w>': 4, '   car ac as</w>': 3, '   ou tw ei gh s</w>': 1, '   v ad a</w>': 81, ' s lea z o id</w>': 1, '   p om mer o y</w>': 6, '   t an a k a</w>': 1, '   ho l en be ck</w>': 1, ' u m m</w>': 5, ' e h h</w>': 2, '   su l ten fu ss</w>': 10, '   com m uni ca tes</w>': 1, ' n r</w>': 1, '   ger n al d i</w>': 1, '   f l ow er ed</w>': 2, '   whi r l w ind</w>': 1, '   b lu r ted</w>': 1, '   ce ce</w>': 1, '   bi e der me y er</w>': 1, '   m ower</w>': 4, '   gra m mo o</w>': 9, '   lu r ks</w>': 1, '   hi ll ary</w>': 4, ' w o ah</w>': 2, ' au di en ces</w>': 1, ' ma g gi e</w>': 1, '   hel bur n</w>': 4, '   be i der me y er</w>': 3, '   under ta k er</w>': 3, '   b loo d lin es</w>': 1, ' ac ci den ta lly</w>': 1, '   c li ft</w>': 1, '   o wh </w>': 1, '   p in ki e</w>': 1, '   bo tt om less</w>': 2, '   f ra g ran ce</w>': 2, '   chi li do gs</w>': 1, '   b in din gs</w>': 1, '   ye ar bo o ks</w>': 1, '   x k</w>': 9, '   po di a tri st</w>': 4, '   rea li g ned</w>': 1, '   ro ta ted</w>': 2, '   a li g ned</w>': 1, '   tra di tion ally</w>': 1, '   s mo ck</w>': 4, '   car e ss ing</w>': 1, ' ge stu r ing</w>': 1, '   bl in k er</w>': 3, '   li z t</w>': 1, '   li s z t</w>': 1, ' gen er ally</w>': 1, '   com men ting</w>': 3, '   ma t th a u</w>': 1, '   per m ea ting</w>': 1, ' su pre mes</w>': 1, '   ro sen fe ld</w>': 2, '   ci tru s</w>': 1, '   ch er ice</w>': 1, '   b on king</w>': 1, '   w el ty</w>': 3, '   supp ers</w>': 1, ' b on e head</w>': 1, '   wi d d man</w>': 1, '   f ru gu e</w>': 1, '   ma x i mi ze</w>': 2, '   ba der</w>': 1, '   l or en z o</w>': 3, '   bab ri t z i o</w>': 1, '   mar m</w>': 2, ' ma ke up</w>': 1, '   d ev o to</w>': 2, '   co s me to lo gi st</w>': 1, ' din o</w>': 1, '   g ru mp</w>': 2, '   can u l a</w>': 1, '   c ru e ll a</w>': 1, '   d ev i ll e</w>': 1, '   ja un di c ed</w>': 1, ' m ea sure</w>': 1, '   bri que tt es</w>': 2, '   wom ani z er</w>': 1, '   bi x l er</w>': 7, '   r on da</w>': 1, ' c rea m</w>': 4, '   ba ff l ed</w>': 2, '   ne ph ri t is</w>': 2, '   a ar r g gh</w>': 1, ' on es</w>': 1, '   ee ee u u u w w</w>': 1, '   sen ne t t</w>': 1, '   st rea m er</w>': 1, '   ph r en o lo g y</w>': 1, '   u uh</w>': 1, '   c ani ev al</w>': 1, '   tr ou s er</w>': 1, '   man i cu re</w>': 1, '   ne o sp or in</w>': 1, '   se le c ting</w>': 3, '   an no y in g ly</w>': 1, '   do p p</w>': 1, '   st ea d fa st</w>': 1, '   ne w s day</w>': 1, '   che f s</w>': 1, '   in sin cer i ty</w>': 1, '   c happ ed</w>': 1, '   im comp at</w>': 1, '   lo v in g ly</w>': 1, '   r en e ged</w>': 3, '   in n s</w>': 1, '   c li pp ers</w>': 1, '   ro o t less</w>': 1, '   en sh r in ed</w>': 1, '   un as sa il able</w>': 1, '   wh i</w>': 1, '   s na ff le</w>': 1, '   sy mp hon i c</w>': 1, '   fa h r en he it</w>': 1, '   cl ev el</w>': 1, '   sch mu cks</w>': 3, '   g ro om s men</w>': 3, '   com i s k ey</w>': 1, '   some wh </w>': 2, '   dis qu a li f ying</w>': 1, '   men age</w>': 1, '   bur den some</w>': 1, '   fa v </w>': 1, '   ju li an n e</w>': 4, '   e gre gi ous</w>': 1, '   in con se qu en ti al s</w>': 1, '   i sa ac son</w>': 1, '   mu cou s</w>': 2, '   c ru ds</w>': 3, '   con f li c ted</w>': 1, '   so ph om or es</w>': 2, '   ju i lli ard</w>': 1, '   p ow er bo ok</w>': 1, '   da z z ling</w>': 4, '   o pp or tu n</w>': 1, '   de sp on dent</w>': 2, '   e ma s cu l a</w>': 1, '   p ers</w>': 2, '   ve sp a</w>': 1, '   fir en ze</w>': 1, ' per son able</w>': 1, '   ad mi r es</w>': 1, '   en de ar in g ly</w>': 1, '   li v ab i li ty</w>': 1, '   s om et</w>': 1, '   j en n y le e</w>': 1, '   cha u v in i st</w>': 1, ' mi k ey</w>': 1, ' b in k y</w>': 1, '   e un u ch</w>': 1, '   so i ree</w>': 2, '   har bor in</w>': 1, '   u s da</w>': 1, '   l ar ge ly</w>': 3, '   ex ce p ted</w>': 1, '   de f en se man</w>': 1, '   op r y land</w>': 1, '   win ni f red</w>': 1, '   so ci al s</w>': 2, '   o ver ha u l</w>': 2, '   tri p le tt e</w>': 7, '   ta mm any</w>': 2, '   ad ver ti se men ts</w>': 1, '   ter re</w>': 1, '   ha u te</w>': 1, '   n y qui l</w>': 1, '   d y spe p ti c</w>': 1, '   e m be z z l ed</w>': 2, ' lin ne a</w>': 1, '   lin ne a</w>': 3, '   cour t land</w>': 1, '   su e le en</w>': 3, '   dri pp y</w>': 1, '   g an d i</w>': 1, '   dam med</w>': 1, '   ar gu ably</w>': 1, '   je et</w>': 3, '   k un e</w>': 3, '   con du c t ors</w>': 2, '   wh a tch m ac a ll it</w>': 1, '   ma st er min ded</w>': 2, '   k no x s</w>': 3, '   ga y le</w>': 11, '   m c c lu s k y</w>': 3, ' h ow down</w>': 1, '   mo ja ve</w>': 1, '   ex ter min ation</w>': 3, ' u per co p</w>': 1, '   shi pp in</w>': 2, '   n y st ro m</w>': 1, '   he ll ho les</w>': 1, '   so le da de</w>': 1, '   pen i ten ti ar i es</w>': 1, '   de wi ght</w>': 2, ' el d or a do</w>': 1, '   stan do ff</w>': 1, '   ho o a a a</w>': 1, '   co l or ad a</w>': 1, '   h om o l k a</w>': 1, ' d ev e lo p ed</w>': 2, '   k in d da</w>': 1, ' win ner</w>': 1, '   mu r row</w>': 2, ' spe c ted</w>': 1, '   pre f er r able</w>': 1, '   mi ll house</w>': 1, '   ma ss ac r ing</w>': 1, '   a a a h h h</w>': 1, '   mu l ber ry</w>': 1, ' a ll ory</w>': 1, ' ead</w>': 1, '   st e in s ma</w>': 1, '   re in gh old</w>': 2, '   lo bo tom i es</w>': 1, '   ra il ro a ding</w>': 1, '   a ver t</w>': 1, '   g ac y</w>': 3, '   mar ch in</w>': 1, ' ad mi re</w>': 1, '   sch war t z en e g ger</w>': 2, '   f er i g no</w>': 3, ' con qu er ing</w>': 1, '   h un s</w>': 1, '   sa w in</w>': 1, ' pu mp ing</w>': 1, '   h y p no ti z ing</w>': 2, '   pi t ne y</w>': 1, '   na gr a</w>': 1, '   bur r</w>': 1, ' se x u a li ty</w>': 1, '   h in den ber g</w>': 1, '   tru f fa u t</w>': 1, '   hi tch co ck</w>': 2, '   cap a</w>': 1, '   ber n sti en</w>': 1, '   re id</w>': 2, '   may s les</w>': 1, '   al ta mon t</w>': 1, '   y a tes</w>': 1, '   b in g ha m</w>': 1, '   ta sti c</w>': 1, '   a do be</w>': 1, ' e dge</w>': 1, '   me s mer i sing</w>': 1, '   p hi ll</w>': 1, '   ne i gh b our ho od</w>': 2, '   t wi g</w>': 3, '   w u r li t z er</w>': 3, '   per ch ed</w>': 1, ' u kes</w>': 2, '   ha z z ard</w>': 2, ' o il ed</w>': 1, ' ro ger</w>': 1, '   fri en d lin ess</w>': 1, '   j i v in</w>': 2, '   pi ck an in ni es</w>': 1, '   hou se bro k en</w>': 1, '   l y n ch ing</w>': 1, '   ans w er in</w>': 1, ' h ow l ers</w>': 1, '   sig ni f y in</w>': 1, '   win d in</w>': 1, '   ni gra h s</w>': 1, '   sto op in</w>': 1, ' ee ms</w>': 1, '   cour t in</w>': 3, '   h b t</w>': 1, '   w inter mu te</w>': 7, '   t ar ted</w>': 1, '   gu ar d s men</w>': 2, '   stra y li ght</w>': 3, '   de f en ces</w>': 4, '   an ti to x in</w>': 2, '   te ssi ers</w>': 1, '   mo le st ing</w>': 2, '   te ssi er</w>': 5, ' con tri bu ting</w>': 1, '   de lin qu en cy</w>': 1, '   sta tu t ory</w>': 3, '   ar mi ta ge</w>': 13, '   wi fe ly</w>': 1, '   f re e side</w>': 4, '   cen tr ed</w>': 1, '   u p st rea m</w>': 1, '   of f world</w>': 1, ' w inter mu te</w>': 2, '   ac ce ss es</w>': 1, '   pre pp ing</w>': 1, '   je op ar di sing</w>': 1, '   sp li ces</w>': 3, '   di gi ti sed</w>': 1, '   f la t lin ed</w>': 2, ' wi red</w>': 1, '   s la gs</w>': 2, '   s ac s</w>': 1, '   a bu s er</w>': 1, '   as se mb ling</w>': 3, '   cor por a tely</w>': 1, '   pl under ing</w>': 1, '   ye sho to</w>': 1, '   chi b a</w>': 4, '   ca the d ra ls</w>': 1, '   di st ru st ing</w>': 1, '   ar ti st e</w>': 3, '   ra t z</w>': 1, '   fir m w are</w>': 1, '   su k u r a</w>': 2, '   jo e boy</w>': 1, '   ma el co m</w>': 5, '   ju mp ship</w>': 1, '   ber n e</w>': 1, '   mu ta ting</w>': 1, '   tw ee z ers</w>': 1, '   pa tr on i se</w>': 1, '   ch in o s</w>': 1, '   ke ta m ine</w>': 1, '   fu l fi l</w>': 1, '   pen e tra ting</w>': 1, ' ice</w>': 2, '   in sti ga te</w>': 1, '   sp ok e sp er son</w>': 1, '   app r en ti c ed</w>': 1, '   v in g ti e me</w>': 1, '   si e c le</w>': 1, ' fee ls</w>': 2, '   ho sa k a</w>': 1, '   f la t lin es</w>': 2, '   f lu c tu a tes</w>': 1, '   fa g an</w>': 1, '   k in ked</w>': 1, '   lo ca les</w>': 1, '   h y p ed</w>': 1, '   nor th g l en</w>': 1, '   s ou th g l en</w>': 6, '   al d ys</w>': 3, '   com pe tes</w>': 1, '   den om in at ors</w>': 2, '   lo gar i th m</w>': 1, '   ca l cu l us</w>': 1, '   1 9 7 4 </w>': 1, '   f la u ti st</w>': 1, '   ma x x</w>': 1, '   sy ne c do c he</w>': 1, '   h y per bo le</w>': 1, '   har le qu in</w>': 2, '   le m min gs</w>': 1, '   cha in ing</w>': 1, ' ni ta</w>': 1, '   ac ce s sor i z ing</w>': 1, '   cu u u u te</w>': 1, '   bri d ge work</w>': 1, '   ye at</w>': 1, '   e sp ad ri ll es</w>': 1, '   qui z z ed</w>': 1, '   ver b s</w>': 1, '   ri g for t</w>': 5, '   ca pri o</w>': 2, '   ki r st en</w>': 2, '   tr an si tion ed</w>': 3, '   o pp o si tes</w>': 2, '   c run ch ed</w>': 1, '   a i gu e</w>': 1, '   ga u c he</w>': 2, '   mo ck er y</w>': 1, '   bo tch es</w>': 1, '   cou l son</w>': 3, '   im mor a li ty</w>': 1, '   ga g a</w>': 2, '   pl u ra l</w>': 1, '   u r m</w>': 1, '   co o l ers</w>': 1, '   sch oo l ers</w>': 2, '   sc an da ls</w>': 2, '   pi men to</w>': 1, ' me ss y</w>': 1, ' f l ack</w>': 1, '   f l un ki es</w>': 2, '   ca ho on</w>': 2, ' ho pe fu lly</w>': 2, '   ad ver b</w>': 1, '   se p tu p le ts</w>': 1, '   in la id</w>': 1, '   pa t in es</w>': 1, '   r h y med</w>': 1, ' ri s k y</w>': 1, '   g ro ssi e</w>': 2, '   ho t ti es</w>': 1, '   ex p o</w>': 2, '   s k o ki e</w>': 2, ' b orrow</w>': 1, '   mon o</w>': 1, '   lu a u</w>': 6, '   la ke sh ore</w>': 1, '   hu b ca p</w>': 1, '   ok a a a ay</w>': 1, ' sur pri se</w>': 2, ' pro m</w>': 1, ' pro men a de</w>': 1, ' y i kes</w>': 1, '   du mp ty</w>': 2, '   tw ee dle</w>': 2, '   du m</w>': 1, '   y oo oo o ow</w>': 1, '   f er r is</w>': 1, ' e u r d or a</w>': 1, '   ca th ar s is</w>': 1, '   en ti ces</w>': 1, '   co h or ts</w>': 1, '   di sc ar ding</w>': 1, '   c rea tions</w>': 1, '   di st r ac ts</w>': 2, '   ni gh t sha de</w>': 1, '   chri st ma s land</w>': 1, '                           </w>': 2, '                                   </w>': 5, '                               </w>': 1, '   shi r l</w>': 1, '   co di tion ing</w>': 1, '   po le tt i</w>': 1, '   pre tt y bo ys</w>': 1, '   e s ca pe es</w>': 1, ' e ss en ti al</w>': 1, '   bab </w>': 1, ' sy na p ses</w>': 1, '   ne u rong</w>': 1, '   any ting</w>': 1, '   ne ve</w>': 1, ' mar ch</w>': 1, ' un p ac king</w>': 1, '   y or</w>': 1, '   s mar k</w>': 1, '   st ran gen ess</w>': 1, '   p sy chi a ti ri sts</w>': 1, '   pri ve le g es</w>': 1, '   go l d man</w>': 1, '   st it</w>': 1, '   p le e se</w>': 1, '   so em</w>': 1, '   si m m</w>': 1, '   ch ro mo s om es</w>': 1, '   h y p no c y l</w>': 4, '   p sy cho ac tive</w>': 1, '   con ci ou s ne ss</w>': 1, '   wi ling</w>': 1, '   fa ir vi e w</w>': 1, ' bo o ge y man</w>': 1, ' sur vi v ors</w>': 1, '   be d we tting</w>': 1, '   ju vi e</w>': 1, '   stan ge</w>': 1, '   un inter ru p ted</w>': 3, '   o ver t</w>': 1, '   un qui et</w>': 1, '   car ver</w>': 6, '   supp re ss ant</w>': 1, '   o c cu r en ces</w>': 1, '   el ri c</w>': 1, '   el ves</w>': 3, ' om orrow</w>': 1, ' ar se</w>': 1, ' ad vi sed</w>': 1, '   an or ex i a</w>': 1, ' bo tt om ed</w>': 1, '   cl ou d cu ck oo land</w>': 1, '   w o de house</w>': 1, '   no tting</w>': 2, '   per spe c ti ves</w>': 1, '   ti m bu k tu </w>': 1, '   gi l da</w>': 2, '   th s</w>': 1, ' car tw right</w>': 1, '   tr s</w>': 1, '   k ft</w>': 1, ' e s sa ge</w>': 1, '   h ks</w>': 1, '   gra in y</w>': 1, ' who op si da i si es</w>': 3, '   r in g le ts</w>': 1, '   who op si da i si es</w>': 1, '   shi t ti ty</w>': 1, '   bri c ki tty</w>': 1, '   f la t man</w>': 1, '   th ack er</w>': 3, '   bo ok sho p</w>': 1, '   l ow point</w>': 1, ' u r real</w>': 1, '   a pri co ts</w>': 3, '   su g ary</w>': 1, '   spi ck</w>': 1, ' pro sti tu te</w>': 1, '   s wa y ze</w>': 1, '   k ni gh t s bri dge</w>': 1, '   go ddy</w>': 1, '   de pre s sin g ly</w>': 1, '   a se x u al</w>': 1, '   ch u b bi er</w>': 1, '   e st ab li sh es</w>': 1, ' ac hi ev ers</w>': 1, '   bo ll o ck sed</w>': 1, '   g li mp sed</w>': 2, '   car dy</w>': 1, '   li qui ds</w>': 1, '   wh ack er</w>': 1, '   bl ac ki sh</w>': 1, '   star r</w>': 2, '   to po l</w>': 3, ' fi d d l er</w>': 1, '   to pp y</w>': 1, ' 3 4 7 </w>': 1, '   gra sp ing</w>': 1, '   ha mp st ead</w>': 2, '   hea th</w>': 1, '   h en ce for w ard</w>': 2, '   im pre ssi ve ly</w>': 1, '   f la t ma te</w>': 1, '   c lu tt er ing</w>': 1, '   par s ni p</w>': 1, ' ev i den tly</w>': 1, ' h er o in</w>': 1, '   c ru el ti es</w>': 1, ' f lin t st one</w>': 3, '   go g g les</w>': 2, '   sp ac ey</w>': 1, '   fe st</w>': 2, '   ca do g an</w>': 1, ' tr ou s ers</w>': 1, '   m c mu r ph y</w>': 45, '   ra tch ed</w>': 19, '   con fr on ting</w>': 2, '   a ll u sion</w>': 2, '   spe cu la ted</w>': 1, ' p l</w>': 4, ' mu h</w>': 9, ' m c mu r ph y</w>': 1, '   lu h</w>': 8, ' b r</w>': 1, ' ce li a</w>': 1, ' la u gh ing</w>': 1, ' l u</w>': 2, ' tal ks</w>': 3, '   se fe lt</w>': 4, ' vo te</w>': 2, '   lin gs</w>': 1, '   s wi tch in</w>': 1, '   ra tion ed</w>': 1, '   pi l b ow</w>': 2, '   or ally</w>': 1, '   sta in ing</w>': 1, '   ea s in</w>': 1, '   gr ow ed</w>': 4, '   s lo sh ing</w>': 2, '   bro m d en</w>': 4, ' ra tch ed</w>': 1, ' can dy</w>': 2, ' li gh ten ed</w>': 1, '   he ll a v a</w>': 1, ' wee l</w>': 1, '   won t t</w>': 1, ' a ee e</w>': 1, '   ran dle</w>': 1, ' har ding</w>': 1, ' p r</w>': 1, ' pa ti en ts</w>': 1, ' coun ci l</w>': 1, ' he ar ts</w>': 2, '   p ss st</w>': 1, '   cra z i es</w>': 2, '   che s wi ck</w>': 8, '   sc an l on</w>': 7, '   ga ff in</w>': 1, '   ni ck les</w>': 1, '   war m in</w>': 2, '   ho o e e</w>': 1, '   h ome ly</w>': 3, '   pe ck in</w>': 2, '   pe cking</w>': 3, '   y o ong</w>': 1, '   pe cks</w>': 3, ' ce du re</w>': 1, ' p y</w>': 1, '   sh in di gs</w>': 1, '   ta b er</w>': 4, '   t re y</w>': 1, '   th u m</w>': 2, '   d ru d ger y</w>': 1, '   in sti ga ting</w>': 1, '   ha bi tu al</w>': 2, '   ha ss l er</w>': 1, '   sh ee u t</w>': 1, '   lo g ger</w>': 1, '   har d wor kin</w>': 1, '   s lan t wi se</w>': 1, '   per se cu tes</w>': 1, '   ho o pl a</w>': 2, '   ca t s k in ner</w>': 1, '   g y p o</w>': 1, '   lo g g in</w>': 1, '   de po e</w>': 1, '   po on t an g</w>': 1, '   lo v v a</w>': 1, '   f re dri ck son</w>': 1, '   du mb wai ter</w>': 1, '   le b enty</w>': 1, '   le b en</w>': 1, '   tur k le</w>': 7, '   un god ly</w>': 2, '   i t su </w>': 2, '   who ow e</w>': 1, '   e g g pl ant</w>': 1, '   ba tt er in</w>': 1, '   bo o ger ed</w>': 1, '   as sa u l tive</w>': 1, '   cle an lin ess</w>': 2, ' a v ev a</w>': 1, '   bra d le y</w>': 8, '   bu on a</w>': 1, '   no tt e</w>': 1, '   dam on</w>': 16, ' se par ate</w>': 1, '   e cho ed</w>': 1, ' com m uni ca tion</w>': 1, '   p in sle y</w>': 3, '   na v on a</w>': 1, '   me z z al un a</w>': 1, '   do v </w>': 1, '   ok a a a a a y y y y</w>': 1, '   g lo ss y</w>': 4, '   g ro t to</w>': 1, '   si r en use</w>': 1, '   stu b bed</w>': 4, '   tra d i</w>': 1, '   s ca mp i</w>': 2, '   br un e llo</w>': 1, '   mon tal c in o</w>': 1, ' nu cu le ar</w>': 1, ' ir re gar d less</w>': 1, '   what ar e you doing</w>': 1, ' i i i</w>': 1, '   so ci op a th s</w>': 1, '   he ee ee er r re</w>': 1, '   li ter</w>': 1, '   c ro s by</w>': 1, ' g ri tty</w>': 1, '   ho st el</w>': 1, '   b ack p ack ers</w>': 1, '   v a por e tt o s</w>': 1, '   char min gs</w>': 1, '   al p</w>': 1, ' th ri lls</w>': 1, '   d on a hu e</w>': 2, '   a man a</w>': 3, '   me ta ph ori ca lly</w>': 1, '   ca s an o v as</w>': 1, '   por c o</w>': 2, '   au c tions</w>': 1, '   ye ar ning</w>': 2, '   s k y wri ting</w>': 2, ' no l an</w>': 1, '   s wa ll ow in</w>': 1, '   fi sh ba lls</w>': 1, '   pi l f er age</w>': 1, '   tw en t y six</w>': 1, '   s qu ea l in</w>': 1, '   fa tt est</w>': 2, '   que en s bu ry</w>': 1, '   pa loo k a vi ll e</w>': 1, ' t an k er</w>': 3, '   lo a der</w>': 2, '   st oo ling</w>': 5, '   f ni sh ed</w>': 1, '   da i sy land</w>': 2, '   har p in</w>': 1, '   j in g l in</w>': 1, '   pe d d l ed</w>': 1, '   cha s ers</w>': 1, '   ru l ers</w>': 2, '   fo x ed</w>': 1, '   beau t ee ful</w>': 1, '   bra i ds</w>': 2, '   par o chi al</w>': 2, '   bra in y</w>': 2, '   bo tt le baby</w>': 1, '   ru m mi es</w>': 1, '   ma ll o ys</w>': 1, '   li f t in</w>': 1, '   t re mb les</w>': 1, '   as sy ri ans</w>': 1, '   ab y s sin i ans</w>': 1, '   as sy ri an</w>': 2, '   ab y s sin i an</w>': 2, '   so f the ar ted</w>': 1, '   ch ok in</w>': 1, '   bor ried</w>': 2, '   k a y o</w>': 4, '   ra tting</w>': 1, '   loo p ing</w>': 2, '   p un ch er</w>': 1, '   who p</w>': 1, ' who p</w>': 1, '   u pp er cu t</w>': 1, '   lon g sh ore</w>': 1, ' sho e ing</w>': 1, '   ma ted</w>': 2, '   s qu ab s</w>': 1, '   che ck er</w>': 3, '   ra tt in</w>': 1, '   pi sto l er o s</w>': 1, '   w run g</w>': 2, '   go of of f s</w>': 1, '   se v </w>': 1, '   f ar e ll a</w>': 1, '   ro t gu t</w>': 1, '   ti pp i</w>': 4, ' ti pp i</w>': 2, ' ti m</w>': 3, '   to a sts</w>': 1, '   g un ne ls</w>': 1, '   ta in ment</w>': 1, '   in fini te ssi ma l</w>': 1, '   f lu ff</w>': 2, '   ha h h</w>': 1, ' de c ree</w>': 1, '   pa in le ss ly</w>': 1, '   wor shi pp er</w>': 1, '   di a</w>': 1, '   mu er to s</w>': 1, '   ma ta d or</w>': 1, '   re pu l se</w>': 1, '   a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a</w>': 1, '   a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a</w>': 1, '   cu ri ou si ty</w>': 1, '   ha ha h h n n n</w>': 1, '   ha h n n n</w>': 1, '   a an n n na a h h n n</w>': 2, '   a a a a an n n n nu u un n nu h h h h h</w>': 1, '   a a a h h n n na h n n n n</w>': 1, '   a a ar r r g g ga a h h</w>': 1, '   a a h h h h n n na a a ha h n n</w>': 1, '   a a a a a a h h h ho o ow w w oo oo oo oo oo oo oo oo oo oo o o</w>': 1, ' s li my</w>': 1, '   a a a a h h h h n n na h h h n n n g g n n n</w>': 1, '   a a a a a h h n n n ha a a a a an n n h h h</w>': 1, '   lo m po c</w>': 8, '   ca f es</w>': 1, '   b on n ev i ll e</w>': 1, '   may i sha n</w>': 2, '   man u el</w>': 2, ' ju mp er</w>': 2, '   si sc o</w>': 6, '   r en</w>': 1, '   sp u r red</w>': 2, '   k r on k</w>': 1, '   tr ans for ms</w>': 1, '   li l ac </w>': 2, ' s have</w>': 1, ' whi tes</w>': 1, '   chi r in o</w>': 3, '   ha ll an da le</w>': 1, '   de li s i</w>': 1, '   ti ll man</w>': 4, '   bur don</w>': 5, '   en du ran ce</w>': 2, '   wa ck job</w>': 1, '   pri z e fi gh ter</w>': 1, '   in v a si ons</w>': 1, '   y on e ll e</w>': 1, '   e man u el</w>': 1, '   k ep</w>': 1, '   he ar n s</w>': 1, '   hi me y</w>': 1, ' to h</w>': 1, '   sha ke d ow n s</w>': 1, '   cu lly</w>': 2, '   whi pla sh</w>': 1, '   ch ea t ers</w>': 1, '   d un away</w>': 3, '   con d or</w>': 1, '   bea tty</w>': 2, '   1 9 3 4 </w>': 3, '   gi b s land</w>': 1, '   u s p</w>': 1, '   s li d ell</w>': 1, '   te l ev an ge li st</w>': 1, '   st un k</w>': 2, '   sha ck l ed</w>': 4, '   mo t or m ou th</w>': 1, '   mi dge</w>': 13, '   du mb fuck</w>': 1, ' re la x ed</w>': 1, '   mi s ju d ged</w>': 4, '   k nu te</w>': 1, '   ro ck n e</w>': 1, '   bo o t stra ps</w>': 1, '   tal mu d</w>': 2, '   ki b bu t z</w>': 1, '   he j ir a</w>': 2, '   mo ha mm ed</w>': 4, '   6 2 2 </w>': 1, '   i s la mi c</w>': 1, ' he j ir a</w>': 1, '   b lo om fi e ld</w>': 1, '   con sp ir ing</w>': 2, '   mo se ll e</w>': 2, '   the m self</w>': 1, '   de i on</w>': 1, '   ban dan na</w>': 1, '   lin ar es</w>': 2, '   tu ss l ed</w>': 1, '   tu ff y</w>': 4, ' mi d d le we ight</w>': 1, '   ban ta m</w>': 1, '   inter lu de</w>': 2, ' na pped</w>': 1, '   sha ck les</w>': 1, ' si g</w>': 1, ' sa u er</w>': 1, ' ight</w>': 5, '   e s ca pe e</w>': 1, '   ba u s ch</w>': 1, '   lo mb </w>': 1, '   sa l ine</w>': 2, ' bra d</w>': 1, ' st re et</w>': 1, '   sh an ked</w>': 1, '   bro ad st re et</w>': 1, '   ge es</w>': 2, '   nu mb nu ts</w>': 2, '   si i ing</w>': 1, ' g ran ge</w>': 1, '   st r</w>': 1, '   ra fe</w>': 17, '   m c ca w le y</w>': 4, '   ha le i w a</w>': 2, ' pu ked</w>': 1, ' su ck in</w>': 1, '   bro om sti cks</w>': 1, ' mu g</w>': 1, '   w u mp</w>': 1, '   ri ck sha w</w>': 2, '   mu sh u</w>': 1, '   sc or ch ed</w>': 2, '   bu n ch ed</w>': 1, ' pa ved</w>': 1, '   f ru ga l</w>': 1, ' lu sh u</w>': 1, '   me g w a</w>': 1, '   fu g i</w>': 1, ' pr ac ti c ing</w>': 1, ' de cks</w>': 1, ' ta ke off</w>': 1, '   g un n er y</w>': 1, ' ho ma ge</w>': 1, '   d ori e</w>': 3, '   bo x in</w>': 1, ' sur vi ve</w>': 1, '   de po ts</w>': 1, '   un char ged</w>': 1, '   bea con s</w>': 2, '   f oo k</w>': 2, '   do o ble</w>': 1, '   lea p in</w>': 1, '   y a ma mo to</w>': 2, '   ca l cu la tion</w>': 3, '   r hu bar b</w>': 1, '   li ll a</w>': 1, '   bri de y</w>': 1, '   sti mu la tion</w>': 1, '   di on</w>': 2, '   be l mon ts</w>': 1, '   fi t z si m mon s</w>': 7, '   bo d ell</w>': 4, '   a ja x</w>': 1, '   fran chi sing</w>': 1, '   mon o ton es</w>': 1, '   s cu z z b all</w>': 1, '   nor vi k</w>': 2, '   un wor k able</w>': 1, '   st ac ca to</w>': 1, '   v ant</w>': 2, '   k el ch er</w>': 3, '   o ver e mo tion al</w>': 1, '   gar g le</w>': 2, '   un for med</w>': 2, '   fe tal</w>': 1, '   t re ble</w>': 1, '   lea sh es</w>': 1, '   po l y ga my</w>': 1, '   on en ess</w>': 1, '   imp al a</w>': 1, '   gra ze</w>': 1, '   g ro o ving</w>': 1, '   gi l f on d</w>': 1, '   dar n de st</w>': 1, '   c rea m si c le</w>': 1, '   l un ac y</w>': 2, '   der e gu late</w>': 1, '   ac qu ir ing</w>': 2, '   s lu mp s</w>': 1, '   sh e er o t ar ds</w>': 1, '   le o t ar ds</w>': 1, '   kn ea ds</w>': 1, '   k ri spi e</w>': 1, '   fe sti ve</w>': 1, '   tra in ers</w>': 2, '   ei gh te en th</w>': 5, '   h y p no ti c</w>': 1, '   ca l cu la t ors</w>': 2, '   wal k man s</w>': 1, ' st y les</w>': 1, '   w b at</w>': 1, '   sp as ma ti ci an</w>': 1, '   dis cont in u ent</w>': 1, '   pre co g ni tion</w>': 1, '   con fu ci ous</w>': 1, '   ne w ton i an</w>': 1, '   af fir med</w>': 1, '   s n ell</w>': 1, '   y o lan der</w>': 3, '   har g ro ve</w>': 1, ' dre w</w>': 1, '   so m</w>': 3, '   ge off</w>': 8, '   z ow i e</w>': 12, '   fu r ba lls</w>': 1, '   g an ged</w>': 1, '   mi st y p ed</w>': 1, '   br ack man</w>': 1, '   re ps</w>': 2, '   m ac ro sy st e ms</w>': 3, '   co p y right</w>': 2, '   en ac ted</w>': 1, '   mi s fi l ed</w>': 1, ' win n ers</w>': 1, ' vi c ti ms</w>': 2, '   man a g ea ble</w>': 1, ' be g in</w>': 2, '   clo tt ed</w>': 2, '   ca the ter</w>': 2, '   re vi ta li ze</w>': 2, '   a z t</w>': 2, '   lon g st re et</w>': 1, '   m ac ro sy st em</w>': 1, '   sp rea d sh e et</w>': 1, '   under so ld</w>': 1, '   in fr in ge ment</w>': 1, '   k l en st e in</w>': 1, '   ks</w>': 1, '   co lon o s co p y</w>': 1, ' ac cu ra te</w>': 1, '   w y ant</w>': 8, '   en c y clo pe di c</w>': 1, '   li ti ga tor</w>': 1, '   pro ba te</w>': 5, '   p hi lli es</w>': 1, '   en er gi z ed</w>': 1, '   par r t y y</w>': 1, '   ex on er ation</w>': 1, '   ar l ine</w>': 2, '   k en ton</w>': 2, '   par a le ga l</w>': 2, '   ra y i sh a</w>': 2, '   he ll er man</w>': 1, '   te t l ow</w>': 1, '   en sh r ou ded</w>': 1, ' car r ying</w>': 1, '   pe sti l ent</w>': 1, '   af ter school</w>': 1, '   fi an ces</w>': 1, ' por no gra p hi c</w>': 1, '   s an so m</w>': 1, '   gi ll man</w>': 2, ' ce lls</w>': 1, '   a ll ev i ate</w>': 1, '   pla te le ts</w>': 1, ' e th ni c</w>': 1, '   gar i sh</w>': 2, '   g ri ev ous</w>': 3, ' who op s</w>': 1, '   la ir d</w>': 2, ' th ri lled</w>': 1, '   un f oun ded</w>': 1, '   g ee sh</w>': 1, '   fa g go ty</w>': 2, '   wi l ting</w>': 1, '   pa mp ers</w>': 1, '   un con s ci ou s ly</w>': 2, ' vi be</w>': 1, ' brea the</w>': 1, '   h ori z on tal</w>': 4, '   na u s ea ting</w>': 1, ' le s bi an</w>': 1, '   can t well</w>': 1, '   ca b in e ts</w>': 4, '   fin le y</w>': 2, '   vo y a g es</w>': 1, '   har d wor king</w>': 1, '   app e ti tes</w>': 1, ' in st in c ts</w>': 1, '   der sh ow i t z</w>': 1, ' di re ct</w>': 2, ' ne g li g ence</w>': 1, '   shi l ts</w>': 1, '   he ll oo o o</w>': 1, '   fo g g in ess</w>': 1, '   re l ev an cy</w>': 1, '   ca m ac </w>': 1, '   as sa i ling</w>': 2, '   fu lls</w>': 1, '   ri d di ck</w>': 19, '   con fu s in</w>': 1, ' be ar ers</w>': 1, '   hu d d les</w>': 1, '   tri ck er ation</w>': 1, '   s la m li ght</w>': 1, '   sh in ed</w>': 1, '   gu sh er</w>': 1, '   co pp er y</w>': 1, '   u mb re ll as</w>': 1, '   han d li ght</w>': 1, '   le s go</w>': 3, ' ma lt</w>': 2, '   fo i e</w>': 1, ' du ck</w>': 1, '   st ea m in</w>': 1, '   to ad shit</w>': 1, ' pe ct</w>': 3, ' ri d di ck</w>': 1, ' sp o t</w>': 5, ' en din gs</w>': 1, ' st op s</w>': 1, '   beli ev in</w>': 2, ' gi gs</w>': 1, '   gi gs</w>': 2, ' gi g</w>': 1, '   s ys</w>': 1, ' j ac kin</w>': 2, '   pi sto l a</w>': 1, ' ow en s</w>': 1, '   de h y d ra tes</w>': 1, '   con g a</w>': 1, ' s la m</w>': 1, '   cre wi es</w>': 1, '   i ma m</w>': 2, ' lan e</w>': 2, '   1 5 5 0</w>': 1, '   mi lli b ars</w>': 1, '   mb </w>': 1, '   ch r on o</w>': 1, '   ha s an</w>': 1, '   ha j j </w>': 2, '   chri s la m</w>': 2, ' t y p es</w>': 1, '   su le i man</w>': 2, '   cor ing</w>': 1, '   p ho bi c</w>': 1, ' ra ft</w>': 1, '   mar a th a</w>': 1, ' pi cks</w>': 2, '   w oo ten</w>': 1, ' sh ar ks</w>': 1, '   e m be lli sh ment</w>': 1, ' tra w l</w>': 1, '   hel li fi ed</w>': 1, ' tri age</w>': 1, '   gon a ds</w>': 1, '   shi v s</w>': 1, '   ra t ba g</w>': 1, '   e ss en ti al s</w>': 1, ' d art</w>': 1, '   pa pu a</w>': 1, '   un cra te</w>': 1, '   la w gi ver</w>': 2, '   ja w b one</w>': 2, '   ab so lu tes</w>': 1, '   con te st able</w>': 1, '   un con te st able</w>': 1, '   t rea son able</w>': 1, '   a po sta te</w>': 1, '   sc ro lls</w>': 6, '   ma x im us</w>': 2, '   re con st ru c ting</w>': 1, '   re m n an ts</w>': 2, '   di g g in gs</w>': 1, '   un ev o l ved</w>': 1, ' o dge</w>': 1, '   i on i z ation</w>': 1, '   h y dro car b ons</w>': 1, '   ni tra tes</w>': 1, '   t x</w>': 1, '   ge i ger</w>': 2, '   se p ti c</w>': 1, '   com m uni ca ble</w>': 1, '   a mp hi th ea ter</w>': 1, '   hon ori us</w>': 2, '   sc ar e c r ows</w>': 1, '   vo l can o</w>': 4, '   ha ss le in</w>': 1, '   be ll a tri x</w>': 1, '   lu ci us</w>': 3, '   f oo li sh ly</w>': 3, '   di si ll u si on ed</w>': 2, '   cu sto di al</w>': 1, '   n on a pe</w>': 1, '   in ven tive</w>': 1, '   ear th man</w>': 1, '   un re cor ded</w>': 1, '   ex on er ate</w>': 1, '   out g un ned</w>': 2, '   de ser ve d ly</w>': 1, '   ve ter in ary</w>': 1, '   e ma s cu la tion</w>': 1, '   pre or da in ed</w>': 3, ' fin al</w>': 1, '   ac qui tt al</w>': 1, ' lan don</w>': 1, ' oo ls</w>': 1, '   si rs</w>': 1, '   rea s se mb l ed</w>': 1, '   as ser tion</w>': 1, '   e li as</w>': 14, ' li as</w>': 5, '   spi der ho les</w>': 1, '   cour t mar ti al</w>': 1, '   cla y mor es</w>': 2, '   ba a d</w>': 1, '   k li ks</w>': 1, '   bab y talk</w>': 1, '   whi te boy</w>': 1, '   li ma</w>': 1, '   i se</w>': 1, '   di d i</w>': 2, '   mo ther fuck ah</w>': 2, '   tw e en</w>': 1, '   y o self</w>': 2, ' p ac i fi sts</w>': 1, '   r ha h</w>': 3, '   ba a a a a a a a a</w>': 1, '   ba a a</w>': 1, '   l er ner</w>': 2, '   da i</w>': 2, '   c ro ss b re ed</w>': 1, '   su per li f er</w>': 1, ' ga in</w>': 1, '   ch u cks</w>': 1, ' i d i</w>': 1, '   c ro ss m oun ted</w>': 1, '   9 2 </w>': 4, '   der o s</w>': 1, '   wa ke up</w>': 1, '   p in o c chi o</w>': 2, ' mo ther fuck ah</w>': 1, '   le t up</w>': 1, '   f ra gs</w>': 2, '   sh en g</w>': 1, '   i a</w>': 2, '   ce es</w>': 1, '   k li k</w>': 1, '   du g ged</w>': 1, '   cl ac king</w>': 1, '   sh ee it</w>': 1, '   hu tch es</w>': 1, '   ba a a a</w>': 2, '   shi t ban g</w>': 1, '   ba a a d</w>': 1, '   ba a a a a a a</w>': 1, '   en u ff</w>': 1, '   may fi e ld</w>': 29, '   en ti t les</w>': 1, '   ten sen ess</w>': 1, '   s mar ted</w>': 1, '   ki ll a ine</w>': 21, '   fin is</w>': 1, '   b lea ch ed</w>': 2, '   sha u gh ne ss y</w>': 1, '   v al k y ri e</w>': 1, '   pe k in ge se</w>': 1, '   k need</w>': 2, '   su spe c ting</w>': 1, '   th under st or m</w>': 1, '   mi t che lls</w>': 1, '   na tu ra li z ed</w>': 1, ' to ss</w>': 1, '   ma g no li as</w>': 1, ' wa ves</w>': 1, '   in let</w>': 1, '   gr ou se</w>': 1, '   plea s an ter</w>': 2, '   s ea for th</w>': 1, '   na sti ly</w>': 1, '   go ble</w>': 8, '   nu t sh e lls</w>': 1, '   per mi ssi ble</w>': 1, '   de li gh t fu lly</w>': 2, '   e f fe te</w>': 1, '   mer ci es</w>': 1, '   s ar d on i c</w>': 2, '   pa ss ke ys</w>': 1, '   re fin ed</w>': 4, '   fri z z ling</w>': 1, ' dro ps</w>': 1, '   gi g i</w>': 1, '   s lan ted</w>': 2, '   bea st ly</w>': 2, '   co l d st rea m</w>': 2, '   gla z ed</w>': 1, '   ge sur es</w>': 1, '   wor thin ess</w>': 1, '   1 8 8 </w>': 1, ' ma i ling</w>': 1, '   to lli son</w>': 1, '   gre en wa ter</w>': 2, ' su per in ten dent</w>': 2, '   m ck e ch ni e</w>': 2, '   re pl ying</w>': 1, '   k in so l ving</w>': 3, '   pre si ding</w>': 1, '   2 6 8 4 </w>': 1, '   n in com po o p</w>': 1, '   un so p hi sti ca ted</w>': 2, '   in ten sion</w>': 1, '   in er ti a</w>': 1, '   see di est</w>': 1, '   du ff ers</w>': 1, '   bri st les</w>': 1, '   b m</w>': 1, '   char ti s m</w>': 2, ' wor shi pped</w>': 1, '   at le e</w>': 1, '   ear n ers</w>': 1, '   ga i t s kill</w>': 1, '   han s ard</w>': 1, '   su e z</w>': 11, '   hi st ori ans</w>': 1, '   do z ing</w>': 1, '   b on fir es</w>': 2, '   for ge t fu l ne ss</w>': 1, '   k under a</w>': 1, '   pen fi e ld</w>': 9, '   a po lo gi ses</w>': 1, '   min e fi e ld</w>': 1, ' su e z</w>': 2, '   di li g ent</w>': 2, '   loo ts</w>': 1, '   f al k lan ds</w>': 1, '   han d ou ts</w>': 2, '   inter vi e w er</w>': 2, '   rea der ship</w>': 2, '   co ll a g es</w>': 1, '   sa m bu c c a</w>': 1, '   di sh on our able</w>': 1, '   con ser v a ti ves</w>': 1, '   mor a li sing</w>': 1, '   con tri bu t ors</w>': 1, '   bri gh ton</w>': 3, '   spe ci a li se</w>': 1, '   mon o lo gu es</w>': 1, '   wom ani s er</w>': 1, '   y o b</w>': 1, '   nu mb ne ss</w>': 1, '   su m mar i es</w>': 1, '   bu ll e t ins</w>': 1, '   pr in t ou ts</w>': 1, '   pro fe ssi on a li s m</w>': 1, '   a ver si ons</w>': 1, ' sha pe less</w>': 1, '   wa d j a</w>': 1, '   tu ck e t t</w>': 1, '   wa j da</w>': 1, '   ba je es</w>': 2, '   f re i gh ting</w>': 1, '   ar gi e</w>': 1, '   uni on i sts</w>': 2, ' dan c ed</w>': 1, '   f al k land</w>': 1, '   pu r dy</w>': 2, '   j in go</w>': 2, '   h or o s co pe</w>': 1, '   ce le b s</w>': 1, '   ce le bra tes</w>': 1, '   ar gi es</w>': 2, '   ac cla im ed</w>': 2, '   ch ort ling</w>': 1, '   b la s</w>': 2, '   e pi cen e</w>': 2, ' com m uni sts</w>': 1, '   c ro m we lls</w>': 1, '   vi i i</w>': 3, '   vi i</w>': 1, '   mp s</w>': 2, '   we st min ster</w>': 1, '   ra ma ge</w>': 1, '   pl ou gh man</w>': 3, ' o o</w>': 2, ' a d</w>': 1, '   pro gra m mes</w>': 1, '   su sy</w>': 1, '   bo d hi</w>': 13, '   e ff</w>': 1, '   bra h</w>': 3, '   bl an k ne ss</w>': 1, '   wh oo o a a ah</w>': 2, '   sh hi i i tt t</w>': 1, '   sp ee d st ar</w>': 1, '   el ev a tes</w>': 1, '   dr on es</w>': 1, '   wai me a</w>': 1, '   wi ess</w>': 1, '   war chi ld</w>': 1, '   nu ked</w>': 1, '   co ck a po o</w>': 1, '   i di om</w>': 1, ' f la me</w>': 1, ' sc en e</w>': 1, '   1 3 2 2 </w>': 1, '   sh in ny</w>': 1, '   mu no z</w>': 3, '   s co p ing</w>': 1, '   bab b it</w>': 1, '   ju v ey</w>': 1, '   po st gra du ate</w>': 1, '   p c b s</w>': 1, '   wa x ing</w>': 1, '   s cu ff s</w>': 2, '   a sp ha lt</w>': 3, '   car nu b a</w>': 1, ' f la m er</w>': 1, '   uni ta s</w>': 1, '   k ne el ing</w>': 1, '   be ta d y n e</w>': 1, '   k a mi k a ze</w>': 2, '   mo on i es</w>': 1, '   bo d hi sa tt v a</w>': 1, '   ph i</w>': 1, '   fi re fi ght</w>': 1, '   ca u ter i z ed</w>': 3, '   b la in</w>': 2, '   ca m ou f la ged</w>': 1, '   p on ch o</w>': 1, '   s qu ir re ly</w>': 1, '   wa x ed</w>': 2, '   ad vi sor s</w>': 1, '   mm m n n n</w>': 1, ' h or ri d</w>': 1, '   vi vi an</w>': 34, '   e sc ar got</w>': 1, '   k ro ss</w>': 14, '   s mo ke st ac ks</w>': 1, '   el ses</w>': 1, ' ex pe c ted</w>': 2, '   con ve ys</w>': 1, '   ru do l ph o</w>': 1, '   b en o it</w>': 1, '   stu ck ey</w>': 2, ' app o in t ment</w>': 1, '   n an ni es</w>': 3, '   bri d get</w>': 1, '   bi ga ss</w>': 1, '   sc re w in</w>': 3, '   f ar m bo ys</w>': 1, ' o c cu pa tion al</w>': 1, ' sp in</w>': 1, '   t our gu i de</w>': 1, ' ni e ce</w>': 1, '   ca ving</w>': 1, '   vi v </w>': 4, '   du mp ster</w>': 2, '   con su el o</w>': 10, '   be d sp read</w>': 1, ' hou se kee p ing</w>': 1, '   v on der</w>': 1, '   vi ch</w>': 1, ' cu me</w>': 1, '   with stan ds</w>': 1, '                                                                                                                     </w>': 26, '   w ea k ling</w>': 2, '   i were</w>': 1, '   du l c et</w>': 1, '   f l or in</w>': 7, ' re ven ge</w>': 3, '   sur pa ss ing</w>': 1, '   hu m per din ck</w>': 14, '   i o can e</w>': 3, '   tri f l ed</w>': 1, '                                                                                                                   </w>': 2, ' gen tly</w>': 1, '   ter r ors</w>': 1, '   qu ir ks</w>': 1, '   mar au ding</w>': 1, '   sin ged</w>': 1, '   sp ort s man like</w>': 1, '   vi z z in i</w>': 7, '   co lo s sus</w>': 1, '   ve er</w>': 1, '   hi pp o po ta mi c</w>': 1, '   gu il der i ans</w>': 1, '   gu il der</w>': 9, '   fe z z i k</w>': 7, '   in i go</w>': 5, '   pe ster</w>': 2, '   wh ee l bar row</w>': 1, '   9 1 </w>': 1, '   mon to y as</w>': 1, '   out thought</w>': 1, '   st ea di ly</w>': 2, '   r ac ed</w>': 2, '   ra v ine</w>': 1, ' le eve</w>': 1, '   pa sti mes</w>': 2, '   mor gen st er n</w>': 1, '   en for c ers</w>': 1, '   in ha bi t ant</w>': 2, '   en for c er</w>': 1, ' ho g</w>': 1, '   di lly</w>': 3, '   da ll ying</w>': 2, '   ru g en</w>': 1, '   of a</w>': 1, '   hu mi li a tions</w>': 1, '   g al ore</w>': 1, '   m lt</w>': 1, '   mu tt on</w>': 2, '   per k y</w>': 4, '   b la ve</w>': 2, '   be ll ows</w>': 1, '   f er r o</w>': 1, '   s wor d play</w>': 1, '   si x fin ger ed</w>': 2, '   s wor d ma k er</w>': 1, ' fin ger ed</w>': 2, '   sp ani ar ds</w>': 1, '   sp ani ard</w>': 2, '   da mp er</w>': 1, '   pe o pl ed</w>': 1, '   o d or less</w>': 2, '   so cra tes</w>': 1, '   co a ting</w>': 1, '   ex pi r ing</w>': 1, ' oo o</w>': 1, '   bo om er an g</w>': 5, '   br is</w>': 5, '   a a a h gh h h</w>': 1, ' hon e y mo on</w>': 2, '   c ro on</w>': 5, '   si l very</w>': 3, '   mo oo o on</w>': 1, '   li e b kind</w>': 9, '   o h h h h h h h h h</w>': 1, '   bi al y sto ck</w>': 18, '   nu t sy</w>': 1, '   fa g in</w>': 1, '   u ll a</w>': 2, ' oo oh</w>': 1, '   w en t wor th</w>': 1, '   re s ni ck</w>': 1, '   bi d d le com be</w>': 1, ' sp r in g time</w>': 2, '   pi st ac hi o s</w>': 1, '   li gh th ea ded</w>': 2, '   de p le ting</w>': 1, ' s mi le</w>': 1, '   a a a a a a a a a a</w>': 1, '   po pp e a</w>': 2, '   f lo pped</w>': 2, '   de ci ma l</w>': 2, '   mi sh kin</w>': 1, '   k off</w>': 2, '   har ru mp h</w>': 1, '   con de m</w>': 1, '   oo oo oo op s</w>': 2, '   oo oo oo o op s</w>': 1, '   gar go y le</w>': 3, '   v h y</w>': 2, '   q vi et</w>': 2, '   ver da mp ter</w>': 2, '   fu h r er</w>': 5, '   st e er ed</w>': 3, '   v here</w>': 2, ' see k</w>': 1, '   c r in ging</w>': 1, '   v a a a at</w>': 1, '   hi t l ers</w>': 1, '   v ould</w>': 1, '   vi lling</w>': 1, '   v hi ch</w>': 1, '   al li ed</w>': 3, '   v or d</w>': 1, '   v it</w>': 7, '   n ar z is</w>': 2, '   n ar z i es</w>': 1, '   ac h h h</w>': 1, '   ber ch te s gar ten</w>': 1, '   do o dle</w>': 3, '   v a ves</w>': 1, '   b lu m</w>': 6, '   da g</w>': 2, '   s ma sh er o o</w>': 2, '   con te ss a</w>': 3, '   ru do l f o</w>': 3, '   ba w dy</w>': 1, '   bi a ly</w>': 1, '   che ck e e</w>': 1, '   pla ye es</w>': 1, '   che ck e es</w>': 1, '   he il</w>': 4, '   a do l p h</w>': 5, '   gra f</w>': 3, '   spe e</w>': 2, '   ni e t z c he</w>': 3, ' de u tch land</w>': 2, '   de u tch land</w>': 2, '   u b er</w>': 2, '   v el t</w>': 1, '   ber t z</w>': 1, '   he in t z</w>': 1, '   du bo is</w>': 2, '   go e b be ls</w>': 19, '   re t rea ts</w>': 1, '   g ro o ves</w>': 3, '   a spi c</w>': 1, '   je lling</w>': 1, '   pi c tur ing</w>': 2, '   da mp ed</w>': 1, ' mar i e</w>': 1, ' sa mu el s</w>': 1, '   re mar king</w>': 1, '   ar bo ga st</w>': 12, '   da mp ne ss</w>': 2, '   b le ary</w>': 1, '   v ac an ci es</w>': 3, '   l ow er y</w>': 4, '   gre en la w n</w>': 1, '   na ke d f ac ed</w>': 1, ' an x i ous</w>': 1, '   fa ir v a le</w>': 5, '   su b tr ac ted</w>': 1, '   h er mi ting</w>': 1, ' r en ting</w>': 1, '   ten ses</w>': 1, ' i ll u si ons</w>': 1, '   pro cra st in a ting</w>': 1, '   stra i gh ten s</w>': 2, '   e lo p ed</w>': 2, '   wh er e with al</w>': 1, '   te m per an ce</w>': 1, '   re spe c ta b ly</w>': 1, '   man te l</w>': 1, '   bro il</w>': 2, ' de du c ti ble</w>': 1, '   ri di cu les</w>': 1, '   la ze</w>': 1, ' some place</w>': 1, '   sa w d ust</w>': 1, '   f al si e</w>': 1, ' ba tes</w>': 1, '   t in k er be ll</w>': 1, '   per i win k le</w>': 1, '   bar e f ac ed</w>': 1, '   s co o ted</w>': 1, '   con ten ted</w>': 1, '   ni gh t work</w>': 1, '   beau t</w>': 1, '   au to s</w>': 1, ' me ir sch u l t z</w>': 1, '   s er</w>': 1, '   i ous</w>': 1, '   win n in g ly</w>': 1, ' per fe c tly</w>': 1, '   me ir sch u l t z</w>': 4, '   p le</w>': 2, ' ha ten est</w>': 1, ' thous an ds</w>': 1, '   mu r</w>': 2, '   der e ss es</w>': 1, '   cou </w>': 1, '   ried</w>': 1, '   en do c r in o lo g y</w>': 1, ' ; ] </w>': 1, '   der ed</w>': 1, '   pre par es</w>': 3, '   f ra g ment</w>': 1, '   en fee bl ed</w>': 1, '   p al er</w>': 2, '   s wi ve l</w>': 1, '   i g or</w>': 12, '   wa x wor ks</w>': 4, '   ro tter</w>': 1, '   du m mi es</w>': 2, '   re qui si te</w>': 2, '   i g or o t</w>': 1, '   ca ll ou sed</w>': 1, ' at ti c</w>': 1, '   pa w k</w>': 1, '   win t ons</w>': 1, ' mon e y ed</w>': 1, '   vi car i ous</w>': 1, '   ho b bl ed</w>': 2, '   s wa y ed</w>': 2, '   ch in ning</w>': 1, ' di me</w>': 2, '   stu pen d ous</w>': 1, '   pe d d les</w>': 1, '   brea th less</w>': 1, ' po k ey</w>': 1, '   wi se c r ack</w>': 1, '   in k well</w>': 1, '   si l</w>': 1, '   ma t ti e</w>': 8, '   p ru ssi c</w>': 1, '   wh oo ey</w>': 1, '   c ru sa ding</w>': 1, '   j er k er</w>': 1, '   c happ i e</w>': 2, '   or na men tal</w>': 1, '   ho tch a</w>': 3, '   s wee ti es</w>': 1, '   st r in g in</w>': 1, '   sc ar c er</w>': 1, '   f l o</w>': 4, ' t ack</w>': 1, '   h or se f ea th ers</w>': 3, ' bu n ch</w>': 1, '   im mor ta li z ing</w>': 1, '   rea s su red</w>': 1, '   te x ture</w>': 2, '   wal st on</w>': 1, '   tw op ence</w>': 1, '   s li m ne ss</w>': 1, '   si de show</w>': 1, '   e li z a be than</w>': 1, '   dar c ey</w>': 1, '   par qu et</w>': 2, '   nu r ture</w>': 2, '   m ea su re men ts</w>': 1, '   sp o of</w>': 1, '   d y s fun c tion al</w>': 1, '   e le c tr ons</w>': 1, '   im pa s sa ble</w>': 1, '   bu d d hi sts</w>': 1, ' at om i c</w>': 1, '   hi ll y er</w>': 32, '   l un e tt e</w>': 2, '   der n</w>': 6, '   mar t ins</w>': 26, '   stu m mi ck</w>': 2, '   un c ri ti cal</w>': 2, '   so le m n</w>': 3, '   a st in</w>': 1, '   g ri st le</w>': 1, '   can tal ou pe</w>': 2, ' ra l</w>': 1, '   wi g g l ed</w>': 3, '   wa s k i</w>': 11, '   f la ked</w>': 1, '   who se ver</w>': 1, '   bo cks</w>': 1, ' da m</w>': 3, '   ga h da m</w>': 1, '   ge c c ch</w>': 1, '   ye h h</w>': 1, '   gu h h h h</w>': 1, '   ca lo me l</w>': 1, '   g l en vi ll e</w>': 4, '   re con fir med</w>': 2, '   du mb ne ss</w>': 1, ' r r</w>': 2, '   br r ro ther</w>': 1, '   tri u mp h ed</w>': 1, '   fi b bed</w>': 1, '   out sta y ed</w>': 1, '   in cen sed</w>': 2, '   ki kes</w>': 2, '   n y mp h om ani ac </w>': 4, '   wi l ki e</w>': 5, '   e pi z oo ti c s</w>': 6, '   sor gh u m</w>': 1, '   h om in y</w>': 1, '   dre ss ma king</w>': 1, '   o ver se x ed</w>': 3, '   sp ay</w>': 1, '   po lea x ed</w>': 1, '   com po s</w>': 1, '   men t is</w>': 1, '   ca ta stra st ro ke</w>': 1, '   f lu m mer y</w>': 2, '   p ea k y</w>': 1, '   s lin gs</w>': 1, '   un wa ver in g ly</w>': 1, ' in te ll e c tu al</w>': 1, '   mar t in son</w>': 1, '   h ow e</w>': 2, '   fi st fi gh ting</w>': 1, '   su l k y</w>': 2, '   c ru di ty</w>': 2, '   ga d s d en</w>': 1, ' i ke</w>': 1, '   supp er time</w>': 2, '   h ort on</w>': 1, '   cor n f la kes</w>': 1, ' h m mu h h</w>': 1, '   ther mo p y la e</w>': 4, '   ro se bi r d</w>': 1, '   ro se baby</w>': 1, '   y a ms</w>': 1, '   p ea ch bi r d</w>': 1, '   sc ha pi r o</w>': 2, ' sa le m</w>': 1, '   la st in</w>': 1, '   ever la st in</w>': 1, '   whi sp er in</w>': 1, ' n an ce</w>': 1, '   ta tt le ta le</w>': 1, '   de li la h</w>': 3, '   fi d d l ed</w>': 1, '   re d ha i red</w>': 1, '   p sy ch on e u ro ti c</w>': 2, '   e th i ca lly</w>': 1, '   co ar sen ing</w>': 1, '   go b b le d y go ok</w>': 1, '   na u s ea tes</w>': 1, '   c y st</w>': 2, '   nee d le ss ly</w>': 1, '   ma li g n an cy</w>': 1, '   gon or r he a</w>': 2, ' z o o</w>': 2, ' ti c s</w>': 2, ' e p i</w>': 1, ' o t</w>': 1, '   c ome ly</w>': 1, '   sp or a di ca lly</w>': 1, '   ho ok wor m</w>': 1, '   pe ll a gr a</w>': 1, '   dea th ly</w>': 1, '   imp ac ted</w>': 2, ' w a</w>': 1, '   pr ac t i</w>': 1, ' c ly</w>': 1, '   ton ks</w>': 1, '   for war d in</w>': 1, '   sa in tly</w>': 1, '   ca l tr ust</w>': 2, '   z im mer man n</w>': 11, '   ki ttle</w>': 8, '   ki ck stand</w>': 1, '   k in ks</w>': 3, ' ca ke</w>': 1, '   in se par able</w>': 1, '   run a ing</w>': 1, '   ca l cu la tor</w>': 2, '   z im n er man n</w>': 1, ' fi st</w>': 1, ' s wa pp ing</w>': 1, '   ca d di es</w>': 1, ' pu sh ers</w>': 2, '   n en a</w>': 2, '   bi cen ten ni al</w>': 9, ' jo y</w>': 1, '   fi l th i est</w>': 1, '   sti mu la tes</w>': 1, ' spi ri ted</w>': 1, ' stu pi di ty</w>': 1, '   wi l d fi re</w>': 1, '   hu mp ers</w>': 1, '   oo ing</w>': 1, '   sha k ers</w>': 1, ' le o</w>': 2, '   vi g or ous</w>': 1, '   shi r tt a i ls</w>': 1, ' bo o z e h ound</w>': 1, '   re sig na tions</w>': 1, '   fi re work</w>': 1, '   fi z z les</w>': 1, '   b le u</w>': 1, ' ro a sted</w>': 1, ' br un ch</w>': 1, ' ri gh ti ous</w>': 1, '   har as s men ts</w>': 1, '   s ea t be l ts</w>': 1, ' pe d d l er</w>': 1, '   p ac ing</w>': 1, '   s wee t pe a</w>': 2, '   me men to s</w>': 1, '   wri tt em</w>': 1, '   1 7 8 7 </w>': 1, ' nu ts</w>': 6, ' sp an g l ed</w>': 1, ' beli ever</w>': 1, ' sc or pi o</w>': 1, ' pe d d ling</w>': 1, '   the dea th</w>': 1, '   c ru i s ers</w>': 1, '   an a kin</w>': 2, '   li gh t sa b er</w>': 2, '   en d or</w>': 2, '   su ll ust</w>': 3, '   ta an ab </w>': 1, '   de i ty</w>': 1, '   d un e</w>': 2, '   car k o on</w>': 1, ' p ow er ful</w>': 2, '   s ar l ac c</w>': 1, '   ex al te d ne ss</w>': 1, '   di sp lea sed</w>': 3, '   we dge</w>': 4, ' i ma g es</w>': 1, '   di sin te gra ted</w>': 2, '   inter pre ter</w>': 2, '   rea di ly</w>': 2, ' th re e pi o</w>': 1, ' cy</w>': 1, '   th ro b bed</w>': 1, '   ex ci te d ly</w>': 1, '   ha p sc ha t t</w>': 1, '   ce le bra tions</w>': 1, '   mo t or c y c li st</w>': 1, '   s ki l ful</w>': 1, '   tr an se x u al</w>': 3, '   e mi tting</w>': 1, ' tr an sy l v ani a</w>': 1, ' na m ea ble</w>': 1, '   re f ra ins</w>': 1, ' dr en ch ed</w>': 1, ' an dro gen ous</w>': 1, '   tra in in</w>': 2, '   s k a ted</w>': 1, '   y a self</w>': 2, '   ga z z o</w>': 11, ' pa u lie</w>': 2, '   in sta ma ti c ly</w>': 1, '   si ck a</w>': 1, '   s ou th pa w</w>': 4, ' a po llo</w>': 1, '   sh hi i i</w>': 1, '   b al bo a</w>': 10, '   f ou ls</w>': 1, '   ho pe fu ls</w>': 1, '   sp ort in</w>': 4, '   pu b li ci z ed</w>': 2, '   wi l co x son</w>': 1, '   th under bi r d</w>': 1, '   pu gs</w>': 1, '   ti m in</w>': 1, '   le f than ded</w>': 1, '   s k at in</w>': 1, '   op er at in</w>': 1, '   s lu g fe st</w>': 1, '   sp r in ts</w>': 1, '   fe in ts</w>': 1, '   k no ck down</w>': 2, ' cla ss es</w>': 1, '   j ab s</w>': 1, '   im per son ation</w>': 1, '   ca ve man</w>': 2, '   k no ck ou ts</w>': 1, ' com men ta tor</w>': 1, '   sp ar</w>': 2, '   di pp er</w>': 5, '   sp ar r ing</w>': 1, '   ga ll a ger</w>': 1, '   gu in n ea s</w>': 1, '   b le ss in</w>': 1, '   ca pp o l i</w>': 2, '   sp ar r in</w>': 8, '   j er gen s</w>': 2, '   ge tta</w>': 1, '   be com in</w>': 1, ' sh re w d</w>': 1, '   cu t man</w>': 1, '   f oo t work</w>': 1, '   di sh ed</w>': 1, '   to ck er</w>': 1, ' g in ny</w>': 1, '   1 9 2 3 </w>': 2, '   fir p o</w>': 1, '   de mp se y</w>': 1, '   u st a</w>': 4, '   re ti r in</w>': 1, ' to ma to</w>': 1, '   sa pp in</w>': 1, '   bu ll ma sti ff</w>': 1, '   bu t k us</w>': 1, '   in ven t in</w>': 1, '   th a tta</w>': 1, '   ba ll in</w>': 1, '   dr y in</w>': 1, '   than k s gi v in</w>': 1, ' ad ri an</w>': 1, '   pri z e fi ght</w>': 1, '   di ab l o</w>': 1, '   din ki e</w>': 1, '   ac me</w>': 23, '   to on</w>': 16, '   to on town</w>': 13, '   to ons</w>': 7, '   jo cu l ar i ty</w>': 1, ' bo o p</w>': 1, '   bo o p</w>': 1, ' do o p</w>': 1, '   tw ee ting</w>': 2, '   clo ver lea f</w>': 6, '   mo s l er</w>': 1, '   c ack</w>': 1, '   stu ff in</w>': 1, ' da me</w>': 4, '   inter tw in es</w>': 1, '   f re e ways</w>': 1, '   di ver se</w>': 1, '   ma g n an im ous</w>': 1, '   ju ri st</w>': 1, ' to on</w>': 2, '   b on go</w>': 1, '   ch ee ta h</w>': 1, '   wa ll a</w>': 2, '   k ok om o</w>': 1, '   w ea s les</w>': 2, ' ro o</w>': 1, '   al tru i sti c</w>': 1, '   pa tt y ca ke</w>': 3, '   ac e t one</w>': 1, '   ru l in</w>': 1, '   ga ss er</w>': 1, '   co o ing</w>': 1, '   ca la mar i</w>': 1, '   no t son e w</w>': 1, '   c ro ck er</w>': 2, '   sa f es</w>': 1, '   bu n n y si tter</w>': 1, ' rea pp ear ing</w>': 1, '   s qu ir ted</w>': 2, ' h r ough</w>': 1, '   d ra f ting</w>': 1, '   sha mu ses</w>': 1, '   e m bi tt er ing</w>': 1, '   no se pl u gs</w>': 1, '   fun er ea l</w>': 2, '   ra i son</w>': 1, '   ri s kin</w>': 1, '   spe ci ous</w>': 1, '   ha be us</w>': 1, '   th i ck en ing</w>': 1, '   ru in in</w>': 1, ' a side</w>': 2, ' b al th sa s r</w>': 1, '   ca pe l</w>': 1, '   g ri ev en ces</w>': 1, '   ca pu le ts</w>': 1, '   con fin es</w>': 2, '   cla ps</w>': 1, '   min i m</w>': 1, '   du el li st</w>': 2, '   pa s sa do</w>': 1, '   p un to</w>': 1, '   re ver so</w>': 1, '   t y b al t</w>': 6, ' s ong</w>': 1, '   c left</w>': 1, ' sha ft</w>': 1, '   k in s man</w>': 3, '   ca pu let</w>': 6, '   ro sa l ine</w>': 12, '   hu m ou rs</w>': 1, '   po per in</w>': 1, '   tru ck le</w>': 1, '   for be ar</w>': 1, '   mer cu ti o</w>': 11, '   mi s gi ves</w>': 1, '   st e er age</w>': 2, '   lu st y</w>': 1, '   be take</w>': 1, '   whi p p</w>': 1, '   bi de</w>': 4, ' se du c ing</w>': 1, '   mu ff l ed</w>': 1, '   f ra y</w>': 2, '   li gh t ne ss</w>': 2, '   mi ss ha p en</w>': 2, ' see m ing</w>': 1, ' b en vo li o</w>': 1, '   t y ran n ous</w>': 1, '   l en g th en s</w>': 2, ' mor row</w>': 4, '   di sp ar a ge ment</w>': 1, '   mon ta gu e</w>': 7, '   wh er e fore</w>': 4, '   f le er</w>': 1, '   so le m ni ty</w>': 1, '   ma ye st</w>': 1, '   con sen ts</w>': 1, '   li ve st</w>': 1, '   r ou se</w>': 2, '   man tu a</w>': 5, '   di sti lling</w>': 1, '   pen si ve</w>': 2, '   un ac cu st om ed</w>': 1, '   d rea mp t</w>': 1, '   b al th as ar</w>': 1, '   hi e</w>': 3, '   th a u</w>': 1, '   w en t st</w>': 1, '   la men ta tion</w>': 1, '   so j our n</w>': 1, '   un than k fu l ne ss</w>': 1, ' ter med</w>': 1, '   en a m ou red</w>': 1, '   b ani sh ment</w>': 3, '   wa ver er</w>': 1, '   r ac h or</w>': 1, '   ch de</w>': 1, '   do ting</w>': 1, '   chi d</w>': 1, '   ri d d ling</w>': 2, '   sh ri ft</w>': 1, '   reme i di es</w>': 1, '   ph y si c</w>': 1, '   gh o st ly</w>': 1, '   ar gu es</w>': 1, '   di st e mp er</w>': 1, '   b en e di ci te</w>': 1, '   sa lu de th</w>': 1, '   ra i le st</w>': 1, '   mi sa p li ed</w>': 1, '   r ind</w>': 1, '   s la ys</w>': 1, '   e m po ssed</w>': 1, '   en ca mp</w>': 1, '   wor s er</w>': 1, '   pre d om in ant</w>': 1, '   can k er</w>': 2, '   ge th</w>': 1, '   cu lled</w>': 1, '   ne ce ss ar i es</w>': 1, '   be ho ve ful</w>': 1, '   jo y ful</w>': 2, '   be sh re w</w>': 1, '   sp ea ke st</w>': 1, '   re p li est</w>': 1, '   ha d st</w>': 1, ' we ary</w>': 1, '   st ri e f</w>': 1, '   hon our able</w>': 2, '   pro cu re</w>': 1, '   ba de</w>': 1, ' di v in ing</w>': 1, '   dis cour ses</w>': 1, '   a di e u</w>': 1, '   nee de st</w>': 1, ' br ow ned</w>': 1, '   un ad vi sed</w>': 1, '   li gh ten s</w>': 1, '   ri pen ing</w>': 1, '   beau te ous</w>': 1, '   in con st ant</w>': 2, '   or b</w>': 1, '   be pa int</w>': 1, '   fa in</w>': 3, '   may st</w>': 1, '   pr or o gu ed</w>': 1, ' per ch</w>': 1, '   st on y</w>': 2, '   k in s men</w>': 2, '   ca me st</w>': 1, '   do ff</w>': 1, '   ve st al</w>': 1, '   s wee tly</w>': 2, '   man n er ly</w>': 1, '   un wor th i est</w>': 1, '   beau ti f y</w>': 1, '   wa st</w>': 1, '   wi t t</w>': 1, '   con stra ins</w>': 1, '   sig ni or</w>': 1, '   sa lu ta tion</w>': 1, '   be got</w>': 1, '   w oo es</w>': 1, '   th ence</w>': 1, '   ma b</w>': 2, '   aga te</w>': 1, ' st one</w>': 1, ' fin ger</w>': 2, '   at om i es</w>': 1, ' nu t</w>': 1, '   wa gon er</w>': 1, ' co a ted</w>': 1, '   ga ll op s</w>': 1, '   dri ve th</w>': 1, '   fri gh ted</w>': 1, '   bo i st er ous</w>': 1, '   ni mb le</w>': 1, '   w ou l d st</w>': 1, '   ra t ca tch er</w>': 1, '   min st re ls</w>': 2, '   di sc or ds</w>': 1, '   fi d d le sti ck</w>': 1, '   z oun ds</w>': 1, '   con sor te st</w>': 1, '   go o day</w>': 1, '   sh ri ved</w>': 1, '   ir a</w>': 9, '   un bri d l ed</w>': 3, '   s qu a g</w>': 1, '   de ir d re</w>': 7, '   pa tri ate</w>': 1, '   b lo c</w>': 1, '   su b ter fu ge</w>': 1, '   an way</w>': 1, '   te le com m uni ca tions</w>': 1, '   re la ying</w>': 1, '   sa ti f ac t ory</w>': 1, '   mi k hi</w>': 2, '   under e sti ma ting</w>': 1, ' pi er re</w>': 3, '   sa mu ra i</w>': 9, '   bu shi do</w>': 1, ' un plea s an t ne ss</w>': 1, '   di ck less</w>': 2, '   ki d na p ing</w>': 1, '   sh ru b</w>': 1, ' sp oo k</w>': 1, '   st as i</w>': 1, '   ju j i t su </w>': 1, '   ga e li c</w>': 2, '   pre e min ent</w>': 1, '   fe ll a h s</w>': 3, '   o ver char ge</w>': 1, '   au te u r</w>': 1, '   fi re br and</w>': 1, '   b re thr en</w>': 2, '   un g lu ed</w>': 2, '   in c ri min a tes</w>': 1, '   g an g ban ger</w>': 2, '   po s su m</w>': 5, '   t ow n ers</w>': 1, '   in ti mi da t in</w>': 1, '   h or ni est</w>': 1, '   ob ser ves</w>': 1, '   he ll ll l p</w>': 3, '   ma x ell</w>': 1, '   den ning</w>': 6, '   wi re ta p</w>': 1, ' pla ted</w>': 2, '   sp ee d well</w>': 1, '   sh er i ff ing</w>': 1, ' ta i ling</w>': 1, '   in au di ble</w>': 1, '   au di ble</w>': 1, '   r ac y</w>': 1, '   hi gh ta il</w>': 2, '   ar mp it</w>': 2, '   for king</w>': 1, '   o ver ex ten ded</w>': 1, '   tu mb le we ed</w>': 2, '   se g un do</w>': 1, '   b tu </w>': 1, '   g ory</w>': 2, '   un lea ded</w>': 2, '   o c t an e</w>': 1, '   u p gra d es</w>': 1, '   su per f rea k</w>': 1, '   ta i</w>': 1, '   k o o</w>': 1, '   z ing</w>': 1, '   su per bi lls</w>': 2, '   in ta g li o</w>': 2, '   min ow</w>': 1, '   min now</w>': 1, '   mar t in e z</w>': 1, '   e li z on do</w>': 1, '   mo lin a</w>': 1, '   mu bu tu </w>': 1, '   ma s sa g es</w>': 1, '   coun ter fe i t ers</w>': 2, ' pa o</w>': 1, ' sh u</w>': 11, '   dar re ll</w>': 19, ' can g</w>': 4, '   pa o</w>': 2, ' sh h h</w>': 1, '   cla s p</w>': 1, '   ri ch whi te men</w>': 1, '   ge fi l te</w>': 1, '   lo x</w>': 3, ' a a a a h h h</w>': 1, '   a a a a h h</w>': 1, '   ri ch whi te man</w>': 1, ' cu st om s</w>': 1, ' ri ck y</w>': 1, '   tri a ds</w>': 4, '   ti to</w>': 1, '   ro o st</w>': 1, '   al to id</w>': 1, '   d y na st y</w>': 1, '   su per in den dent</w>': 1, ' go at</w>': 2, '   m ac a o</w>': 1, '   su per bi ll</w>': 1, '   qu ac king</w>': 1, '   be j ing</w>': 1, '   see ding</w>': 2, '   p oun da ge</w>': 1, '   b ac kin</w>': 1, ' a w w</w>': 1, '   h h h</w>': 1, ' s cu se</w>': 1, '   j un ta o</w>': 4, '   st oo ly</w>': 1, '   y o lan da</w>': 1, '   su per f ly</w>': 1, '   la f on ta ine</w>': 2, '   for te</w>': 1, '   pa tt in</w>': 1, '   pu t t</w>': 4, '   f ab ri ca te</w>': 2, ' n g b</w>': 1, '   ta ke out</w>': 2, '   re im bur se ment</w>': 1, ' spe c</w>': 1, '   h inter lan ds</w>': 1, '   v in o</w>': 2, '   su p p</w>': 1, '   lo c ust</w>': 1, '   lo cu sts</w>': 2, '   c ac op hon y</w>': 2, '   to o ta lo o</w>': 1, '   pre ss man</w>': 2, '   sch u lli an</w>': 1, ' ha ter</w>': 1, '   ph le m less</w>': 1, '   ph le m ing</w>': 1, '   ge w ga w</w>': 1, '   g ro om s</w>': 2, '   o of</w>': 1, '   i ta li c s</w>': 1, '   an na pu ma</w>': 2, '   hea d dre ss</w>': 1, '   ri p k en</w>': 1, '   app la u ded</w>': 2, ' si l ence</w>': 1, '   cha u v in i sti c</w>': 1, '   un gr oun ded</w>': 1, '   sh er p as</w>': 1, '   wa h ine</w>': 1, '   bi r ch w o od</w>': 1, '   du ck bi ll</w>': 1, '   pla t y p us</w>': 1, '   ir re ver si b ly</w>': 2, '   sp a z</w>': 1, ' do g g</w>': 1, '   ru de st</w>': 1, '   hon e y mo on ing</w>': 1, '   s ca l ed</w>': 1, '   re mo de ls</w>': 1, '   hou se bo a ts</w>': 1, '   re po s se ssed</w>': 1, '   ro x</w>': 1, '   du lu th</w>': 1, '   sc ar le t t</w>': 1, '   pe l men i</w>': 1, '   a a p</w>': 1, ' hu gh es</w>': 1, '   s we lls</w>': 1, ' be t sy</w>': 1, ' sle e p less</w>': 2, '   no b</w>': 1, ' v ac ant</w>': 1, '   f la sh er</w>': 1, '   can n on b all</w>': 1, ' co s mi ca lly</w>': 1, ' h ow ever</w>': 1, '   su b con s ci ou s ly</w>': 1, '   my sti cal</w>': 4, '   fa ted</w>': 1, '   vo t y p k a</w>': 1, '   m ac a da mi a</w>': 1, '   re done</w>': 1, '   ri gh</w>': 1, '   ti r a</w>': 2, '   mi su </w>': 2, '   cl en da</w>': 1, '   we i gh t li f ter</w>': 1, '   ta x i ca b s</w>': 1, '   cl ar i se</w>': 1, '   bar re tt e</w>': 1, ' ba se man</w>': 1, '   mar j or am</w>': 1, '   mar co s</w>': 1, '   k er r</w>': 1, '   re bri cked</w>': 1, '   what d ya</w>': 1, '   qu inter o</w>': 3, '   co v ey</w>': 1, '   pi ck e ting</w>': 2, '   vi da l</w>': 5, '   k al in s k y</w>': 2, '   ore</w>': 2, '   ne go ti at in</w>': 1, '   e st el li ta</w>': 1, '   e sp er an z a</w>': 3, '   out last</w>': 1, '   co ts</w>': 1, '   or g ani z er</w>': 3, '   e st e ll a</w>': 1, '   shu ff le</w>': 9, '   fo ll ower</w>': 1, '   s lu g ged</w>': 1, ' cen te</w>': 1, '   st e war ds</w>': 4, '   who pped</w>': 1, '   s ca b s</w>': 1, '   tra i d or</w>': 1, '   gen te</w>': 1, '   ro m pe hu el g a</w>': 1, '   de s gr ac i a do</w>': 1, '   si m my</w>': 1, '   co le tt e</w>': 4, '   qu in c ey</w>': 3, '   su t c li f fe</w>': 1, '   k u j o</w>': 1, ' na ps</w>': 1, '   tw ea k er</w>': 4, '   te m</w>': 1, '   fu u u u u ck</w>': 1, '   tw ea kin</w>': 1, '   g ack</w>': 1, '   dis gr ac ing</w>': 1, '   ver n e</w>': 1, '   st e u b ing</w>': 1, ' s li ck er</w>': 1, '   4 1 1 </w>': 2, '   gar ce tt i</w>': 6, '   what a</w>': 2, '   bl ab b er</w>': 1, '   f l y n n e</w>': 10, '   b ack fu ll</w>': 1, '   ta tt s</w>': 1, '   c ran k vi ll e</w>': 1, '   men ac ing</w>': 1, '   co or din a ting</w>': 1, '   pu ssi fi ed</w>': 1, ' chi ck en shit</w>': 1, '   z app ing</w>': 1, '   c r ack l er</w>': 1, '   h om i es</w>': 1, '   me x i ca l i</w>': 1, '   tw ea king</w>': 1, '   f ea der</w>': 1, '   d on ner</w>': 3, '   r en di tion</w>': 1, '   cra po l a</w>': 1, '   ra me ll e</w>': 12, '   de bar k ation</w>': 1, '   ca en</w>': 16, ' fi ves</w>': 2, '   f our ty</w>': 1, '   we h r m ac h t</w>': 1, '   p on te</w>': 1, '   ho c</w>': 1, ' g ro o ve</w>': 1, '   ri f ling</w>': 1, '   fo l d in</w>': 1, '   ha ss et</w>': 1, '   jo in in</w>': 1, '   min i st er i al</w>': 1, '   ru st l in</w>': 1, '   re i b en</w>': 15, '   lo ck ja w</w>': 1, '   ad d le y</w>': 3, '   mo ti v a tor</w>': 1, '   sur pre ss ing</w>': 1, ' ei gh ts</w>': 3, ' d ab </w>': 1, '   mi sa ll o ca tion</w>': 1, '   ga v in</w>': 7, ' wa lls</w>': 2, '   ro ad ster</w>': 1, '   u p ha m</w>': 8, '   g un n ers</w>': 2, '   cho let</w>': 1, '   si beli us</w>': 1, '   c r ou ch es</w>': 1, '   ne ar ne ss</w>': 1, '   br ac ke ting</w>': 1, '   in ho spi ta ble</w>': 1, '   pro pri e t ary</w>': 1, '   de u ces</w>': 1, '   b lu ff ed</w>': 2, '   s qu a w ked</w>': 1, '   si pped</w>': 1, '   ve c chi o</w>': 2, '   su l f a</w>': 3, '   por t s m ou th</w>': 1, '   f l y bo ys</w>': 1, '   ber n ay</w>': 1, '   tal bo t</w>': 3, '   ru mb le</w>': 2, '   ad v an c ing</w>': 2, '   s an d ba gs</w>': 1, ' ta gs</w>': 1, '   mo tion less</w>': 1, '   qui ver s</w>': 1, '   un c li ps</w>': 1, '   wa d es</w>': 1, '   k ab ack</w>': 1, '   fa u l kn er</w>': 1, '   te d di es</w>': 1, '   sh e ar</w>': 1, '   ne g li ge es</w>': 1, '   wai sts</w>': 1, '   di sc oun ts</w>': 1, '   g ran d fa th ers</w>': 2, '   p in ea pp les</w>': 2, '   2 2 0</w>': 1, ' re tur ned</w>': 1, '   ga me show</w>': 1, ' wi re</w>': 1, ' co ver ed</w>': 2, ' t an k</w>': 1, '   su ture</w>': 1, '   per son a li sed</w>': 1, '   ca s se tt es</w>': 3, '   d ow n shi ft</w>': 1, '   a ge i s m</w>': 1, '   che ck li st</w>': 1, ' ll o y d</w>': 1, '   ba v ar i an</w>': 1, '   pre t z el s</w>': 3, '   v al here</w>': 2, '   sh ow p on y</w>': 1, '   ki ck bo x ing</w>': 3, '   do bl er</w>': 4, ' 1 3 4 2 </w>': 1, '   gra ph s</w>': 2, '   oo o p</w>': 1, '   fi re bi r d</w>': 1, '   ar ma men ts</w>': 2, '   k la us</w>': 1, '   ta u b er</w>': 1, '   b re s la u</w>': 3, '   a m on</w>': 1, '   go e th</w>': 2, '   c r ac ow</w>': 1, '   sch er ner</w>': 2, '   c z u r da</w>': 1, '   po l de k</w>': 1, '   mi l a</w>': 1, '   gra ys</w>': 1, '   ra s ch</w>': 1, '   e mi lie</w>': 1, '   mi s ca li bra ting</w>': 1, '   with ho l ding</w>': 2, '   m ac h in i st</w>': 3, '   u sa g es</w>': 1, '   hi mb ry</w>': 4, '   re gi st ar</w>': 1, '   ta tu m</w>': 17, '   gu tt ed</w>': 4, ' ca mp us</w>': 1, '   wh a a a at t</w>': 1, '   ep</w>': 3, '   ci c i</w>': 6, '   da w ni e</w>': 1, '   an ci lli ary</w>': 1, '   h er i di t ary</w>': 1, '   in cou ra ged</w>': 1, '   te mp late</w>': 1, '   v u l ture</w>': 2, '   of ff ff f</w>': 1, '   jo y y y</w>': 1, '   ha lli e</w>': 8, '   w oo d s bor o</w>': 13, '   co d d ling</w>': 1, '   or g in al</w>': 1, '   po st in gs</w>': 1, ' ma ll ory</w>': 1, ' w oo dy</w>': 1, '   har re l son</w>': 1, ' ju li e tt e</w>': 1, '   qui z z i cal</w>': 1, '   ss sh h</w>': 3, '   ro o ma tes</w>': 1, '   b al ding</w>': 2, '   see ps</w>': 1, '   f on d l ed</w>': 1, '   in ex per i ence</w>': 3, '   ther e by</w>': 2, ' de pu ty</w>': 2, '   oo z ed</w>': 1, '   fi fi sh</w>': 1, '   ke y r ing</w>': 1, '   de l ta s</w>': 2, '   att ack er</w>': 2, '   i i i i i i i i i ye i i i i i</w>': 1, '   oo oo o ow u oo oo oo o o</w>': 1, '   ro mp er</w>': 1, '   de sp ar ate</w>': 1, '   gir le tta</w>': 1, '   do c u</w>': 1, '   ca l cu la s</w>': 1, '   c lu tch ing</w>': 3, ' k ni fe</w>': 1, ' d ra gs</w>': 1, ' ma s ked</w>': 1, '   bi ll ow ing</w>': 1, ' har mon i c a</w>': 1, '   ru b ber ed</w>': 1, '   ke g ger</w>': 1, '   sh ow gir ls</w>': 2, '   dri pped</w>': 1, '   in su la tes</w>': 1, '   me lin da</w>': 6, '   m c gra w</w>': 1, '   com mi sh</w>': 1, ' mo vi es</w>': 1, '   he y day</w>': 1, ' a ma z om bi es</w>': 1, ' p ace</w>': 1, ' c rea tur es</w>': 1, '   the s an</w>': 1, '   an d rea s</w>': 4, '   r in a</w>': 6, '   pre s co t t</w>': 18, '   ex e c s</w>': 1, ' ta b</w>': 4, '   pr in ze</w>': 2, '   m ac her</w>': 2, '   s oun d sta ge</w>': 1, '   tri lo gi es</w>': 3, '   ri e lly</w>': 5, '   an ge lin a</w>': 3, '   su per hu man</w>': 3, '   o h mi god</w>': 5, '   co l che ck</w>': 1, ' vi de o</w>': 1, '   s la sh in gs</w>': 1, '   pa sts</w>': 1, '   to be</w>': 1, '   ra p p</w>': 1, '   t or i</w>': 2, '   tri lo g y</w>': 1, '   re in er</w>': 1, '   t ar en t in o</w>': 1, '   k y</w>': 1, '   mon e y ba gs</w>': 1, '   s n l</w>': 1, '   k en ni son</w>': 1, '   ye ll er</w>': 1, '   po l ans k i</w>': 1, '   bri d ger</w>': 2, '   jo lli e</w>': 1, '   wa tch out</w>': 1, '   re p ent</w>': 1, ' r ti go</w>': 1, '   u m h m m</w>': 1, '   m c nu g ge ts</w>': 1, '   fri g g en</w>': 1, ' wri tes</w>': 1, '   b ack lot</w>': 1, '   to the</w>': 1, '   fuck ra g</w>': 1, '   s b i</w>': 1, '   jo die</w>': 1, ' ab sor bed</w>': 1, ' bu mp</w>': 1, '   n c</w>': 1, '   so t</w>': 1, '   v or he es</w>': 1, ' si tt ers</w>': 1, '   si tt ers</w>': 2, '   an ti chri st</w>': 1, '   me te or o lo gi st</w>': 1, '   ra di o ed</w>': 3, '   de mo gra p hi c</w>': 1, ' re f er en c ed</w>': 1, '   a ir com p</w>': 1, '   k ro ger</w>': 1, '   wal mar t</w>': 2, '   g li tch ed</w>': 1, '   ri ck i</w>': 1, '   t ro ll o p</w>': 1, '   lea ve m ea l one</w>': 1, '   un ori g in al</w>': 1, '   brea sted</w>': 1, '   ra le i gh</w>': 3, '   lea ther face</w>': 1, '   ga u ged</w>': 1, '   re vi si ted</w>': 1, '   ho ll ow ed</w>': 1, ' p sy cho ti c</w>': 3, '   sa t ani cal</w>': 1, '   go o de ve</w>': 3, '   ma tur ing</w>': 1, '   an or ex i c</w>': 1, '   g ere</w>': 1, ' ger bi l</w>': 1, '   sle e po ver</w>': 1, '   j ani t ors</w>': 2, '   pa pi ll ary</w>': 1, '   ar ra i g ned</w>': 1, '   with our</w>': 1, '   de cl in ing</w>': 2, '   g lu tt on y</w>': 5, '   bi ding</w>': 1, '   o be se</w>': 1, '   s war r</w>': 1, '   t ou gh en ing</w>': 1, '   har d co ver</w>': 1, '   ju r g en</w>': 1, '   f an u</w>': 1, '   ma st er work</w>': 1, '   di sc er ni ble</w>': 1, '   a qu in as</w>': 1, '   a q u</w>': 1, '   a qu in</w>': 1, '   ori g ins</w>': 2, '   de e ms</w>': 1, '   me th o di cal</w>': 4, '   gon ing</w>': 1, '   dis mi ssi ve</w>': 1, '   un re ven ged</w>': 1, '   g ould</w>': 6, '   att ri tion</w>': 2, '   ter r ac es</w>': 1, '   pu r ga tion</w>': 1, '   ti gh ten ed</w>': 1, '   hon e y mo on ers</w>': 1, '   sc ra p in gs</w>': 2, '   do d der in</w>': 1, '   ja mi s ons</w>': 1, '   s qu a ll in</w>': 1, '   c lu mp</w>': 1, '   c rea kin</w>': 1, '   mo se</w>': 6, '   dam y an ke es</w>': 1, '   s ca ld</w>': 1, ' me b be</w>': 1, ' m ac cor ry</w>': 1, '   pu ta ti ous</w>': 1, '   s ca l ps</w>': 4, ' m oun t</w>': 1, '   ho g back</w>': 1, ' no pe</w>': 1, '   fu tt er man</w>': 6, '   com an ch es</w>': 8, ' ver end</w>': 1, ' ev en in</w>': 1, '   com an ch</w>': 2, '   sta m pe d in</w>': 1, '   c ome th</w>': 1, '   de sc ri p tions</w>': 1, '   sur r en der in</w>': 1, '   pl ou gh sh are</w>': 1, '   ho ll ers</w>': 1, ' t ro o p</w>': 1, '   pa s se l</w>': 1, '   te x i can</w>': 1, ' s ke da d dle</w>': 1, ' hea d in</w>': 1, '   ma la pa i</w>': 1, ' kee ps</w>': 1, ' gre en hi ll</w>': 1, '   gre en hi ll</w>': 5, ' com man ding</w>': 1, '   un n t</w>': 2, ' m eah</w>': 2, ' e than</w>': 2, ' che w ed</w>': 1, '   man gi er</w>': 1, '   win ga te</w>': 1, '   na w ye ck y</w>': 1, '   s qu a w s</w>': 1, '   kee f er</w>': 1, '   car b ine</w>': 2, '   b lu e be lli es</w>': 1, '   na w ye ck a</w>': 1, ' che y en n e</w>': 1, '   ba sh ed</w>': 2, '   do d g in</w>': 1, '   ci ca tri z</w>': 2, '   en jo y in</w>': 1, '   f er n an de z</w>': 3, '   k in fo l k</w>': 1, ' ki ll in</w>': 1, ' bi st</w>': 1, '   p ab o</w>': 1, '   ta i b o</w>': 1, '   f l out in</w>': 1, '   ch un k head</w>': 1, '   na w ye ck as</w>': 1, '   wh a tch u</w>': 3, ' r oun d about</w>': 1, ' na w ye ck a</w>': 1, '   car tri d g es</w>': 1, ' j or gen s en</w>': 1, ' la u ri e</w>': 3, ' fin ding</w>': 1, '   gra m pa w</w>': 1, '   f ra z z le</w>': 1, ' un c le</w>': 1, '   com an ch er o s</w>': 1, '   bu sh el</w>': 1, ' com an c he</w>': 1, '   ca li c o</w>': 2, '   b la m in</w>': 2, ' mar t in</w>': 2, ' mar ti e</w>': 1, '   la u ry</w>': 2, '   an ch es</w>': 1, '   mar ti e</w>': 4, ' re ck on</w>': 1, ' sc our</w>': 1, '   sa d d l in</w>': 1, '   ge l ding</w>': 1, '   bra z o s</w>': 1, '   ga un ted</w>': 1, '   tom m i</w>': 5, '   bi r d bra ins</w>': 1, ' ev en ts</w>': 2, '   app ar i tions</w>': 1, '   di sh ar mon i es</w>': 1, '   or dea ls</w>': 1, '   au to gra ph s</w>': 1, '   so l sti ce</w>': 1, '   an ti qui ty</w>': 1, '   mu se</w>': 4, '   win d pi pe</w>': 1, '   e li sa be th</w>': 2, '   k o sin s k i</w>': 1, '   mo or</w>': 2, '   stra u b</w>': 3, '   lin d ner</w>': 2, '   o ver st run g</w>': 1, '   te t an us</w>': 1, ' stra u b</w>': 1, '   war wi ck shi re</w>': 1, '   le s se ps</w>': 4, '   un man ned</w>': 2, '   un men ded</w>': 1, '   un made</w>': 2, '   mar l ow e</w>': 10, '   a ll e y n</w>': 1, '   so ver ei g n s</w>': 2, '   de p t for d</w>': 1, '   h en s l ow e</w>': 9, '   ti l ne y</w>': 1, '   bur ba ge</w>': 10, '   s wor d s man</w>': 4, '   ke mp e</w>': 1, '   ar den s</w>': 1, '   o ver thr ows</w>': 1, '   bri m st on es</w>': 1, '   n un n er y</w>': 1, '   a po the car y</w>': 6, '   f en n y man</w>': 6, '   ta m bur la ine</w>': 1, '   shi p w re ck</w>': 2, '   in sur m oun ta ble</w>': 1, '   ba ga te ll e</w>': 1, ' ti ck l er</w>': 1, ' mi sta k en</w>': 1, '   a a a a gh</w>': 1, '   \x85 </w>': 2, '   e mb ar k</w>': 1, '   o ver par ted</w>': 1, '   ran t ers</w>': 1, '   stu tt er ers</w>': 1, '   no l</w>': 2, '   c ro ok back</w>': 1, ' y ar ds</w>': 1, '   ba g got</w>': 1, '   fa u st us</w>': 1, '   pi e ty</w>': 2, '   pi ous</w>': 2, '   m oun te ban k</w>': 1, '   fo st er ed</w>': 1, '   si l vi a</w>': 3, '   un bi d d able</w>': 1, '   un go ver n able</w>': 1, ' mon i ed</w>': 1, '   we s se x</w>': 15, '   pla y hou ses</w>': 2, '   pe t ti co a ts</w>': 1, '   for e told</w>': 2, '   a su n der</w>': 1, '   st ri fe</w>': 1, ' ma da m</w>': 2, ' par ting</w>': 1, '   bri gh t ne ss</w>': 1, '   u r g</w>': 1, ' th us</w>': 1, '   pu r g</w>': 1, '   ru bi es</w>': 1, '   sa d d le ba g</w>': 1, '   plan ta tions</w>': 2, '   pre f er ment</w>': 1, '   d ow ry</w>': 2, '   an dr on i c us</w>': 2, '   or sin o</w>': 1, ' or sin o</w>': 1, '   se ver ing</w>': 1, '   jo c un d</w>': 1, ' wi lt</w>': 1, '   ni gh tly</w>': 2, '   p ome g ran ate</w>': 1, '   un s an c ti fi ed</w>': 2, '   un chan g ea ble</w>': 1, '   sp ea k able</w>': 1, '   l ow ly</w>': 1, ' a la s</w>': 1, '   pla y wri ting</w>': 1, '   pla y ac ting</w>': 1, '   su b mi ssi ve</w>': 1, '   re pu te</w>': 1, '   we s se x es</w>': 1, '   ri ver ban k</w>': 1, '   b li gh ts</w>': 1, '   ban k side</w>': 1, '   pi pp ins</w>': 1, '   lu te</w>': 1, '   t wi tt er ing</w>': 1, '   l ar ks</w>': 1, '   b ani sh</w>': 1, '   ni gh t in g al es</w>': 1, '   se ar</w>': 1, '   for tu i t ous</w>': 3, '   wi l hel min a</w>': 1, '   ra ven s</w>': 1, '   ne c ce ss ar i ly</w>': 1, '   ma l en ess</w>': 1, '   a si ck</w>': 1, '   re ci ever</w>': 1, '   d ev al u e</w>': 1, '   bra u n</w>': 1, '   f re e ba se maybe</w>': 1, '   a ll like</w>': 1, '   thin k i</w>': 1, '   un f ea si ble</w>': 1, '   f la t ma tes</w>': 1, '   sh ou l d i</w>': 1, '   to point</w>': 1, '   li as on</w>': 2, '   th i se</w>': 1, '   lo ks</w>': 1, '   di lli g ent</w>': 1, '   s ne er</w>': 1, '   s ne er ing</w>': 1, '   s ne er ed</w>': 3, '   no ve list s</w>': 1, '   pa ta gon i a</w>': 1, '   bo got</w>': 2, '   in ton ation</w>': 1, '   d ev r a</w>': 3, '   f li gh ty</w>': 1, '   fe li ci a</w>': 10, '   nu mm ers</w>': 1, '   per co d an</w>': 1, '   le ss er man</w>': 1, '   r oun dy</w>': 3, '   f h a</w>': 1, '   shu man n</w>': 1, '   bu f fu ms</w>': 3, '   cho l o</w>': 1, '   nor wal k</w>': 1, '   ber co vi c i</w>': 1, '   k wan g</w>': 1, '   bri llo</w>': 1, '   o ver po pu la ted</w>': 1, '   e pi to me</w>': 2, ' e st ab li sh ment</w>': 1, '   f lu ff s</w>': 1, '   la y er ed</w>': 2, '   bi st r o</w>': 1, ' car ed</w>': 1, '   ha y n es</w>': 2, '   y or ki es</w>': 1, '   k ar p f</w>': 3, '   s lu tes</w>': 1, '   we st w o od</w>': 1, ' k le i s er</w>': 1, '   ki d di el an ds</w>': 1, '   su n de ck</w>': 2, ' hu h h</w>': 1, '   si l ver man</w>': 1, '   he mm ed</w>': 1, '   s n ow b ound</w>': 2, '   can n ab i li s m</w>': 2, '   si er ra s</w>': 1, '   2 3 7 </w>': 6, '   b ou l der</w>': 1, '   t or ran ce</w>': 21, '   ad vo ca at</w>': 2, '   u ll man</w>': 4, '   t or ran ces</w>': 2, ' o c</w>': 1, '   win ni e</w>': 2, '   ti m bu c too</w>': 1, '   sho ve ll in gs</w>': 1, '   dri ve ways</w>': 1, '   sc ar ey</w>': 1, '   e g</w>': 1, '   cla u st ro p ho bi c</w>': 1, '   de pre ci ation</w>': 1, '   re fu r bi sh</w>': 1, '   con st able</w>': 10, '   ha ll er</w>': 4, ' ju mp ed</w>': 1, ' b al d hea ded</w>': 1, '   s no t bra ins</w>': 1, '   ru st ling</w>': 1, '   gla s sp ack</w>': 1, '   de ar i es</w>': 1, '   fo a m ing</w>': 1, '   o ti c</w>': 2, '   e ye wa sh</w>': 2, '   n on pre sc ri p tion</w>': 1, '   k no p f l er</w>': 1, '   pe l t z er</w>': 1, '   cor n ea l</w>': 1, ' pa tch</w>': 2, '   so p ran o</w>': 3, '   p li ars</w>': 1, '   win e s bur g</w>': 1, '   ad ju st able</w>': 1, ' wi ves</w>': 1, '   hou k</w>': 1, '   bab co ck</w>': 1, ' car l ton</w>': 1, '   sti ck ne y</w>': 1, '   ever more</w>': 1, '   com m in</w>': 1, '   s lea z e b all</w>': 1, '   ex com m uni ca ted</w>': 1, '   an ton e ll i</w>': 1, ' lan g</w>': 4, '   re jo i c ing</w>': 1, '   u p be at</w>': 2, '   hea d lin ers</w>': 1, '   a ll over</w>': 1, '   he d d les</w>': 2, '   bea ter</w>': 1, ' supp or ting</w>': 1, '   con ven ts</w>': 2, ' han ging</w>': 1, '   r on e tt es</w>': 1, '   car ti er</w>': 3, '   po p es</w>': 1, '   lo af er</w>': 1, '   shi re ll es</w>': 1, '   j ab b er</w>': 2, '   d ow n he ar ted</w>': 1, '   la y per son</w>': 1, '   bo b s</w>': 1, '   o il ed</w>': 3, '   pe w s</w>': 1, '   h y m n al s</w>': 2, '   sp an de x</w>': 2, '   ev i den c ed</w>': 1, '   s k y di ving</w>': 1, '   ta mb our ine</w>': 2, ' har a</w>': 1, '   to ta li t ar i an</w>': 1, '   wa y w ard</w>': 3, '   e c ci e si a st es</w>': 1, '   wa st re l</w>': 1, '   n on ex i st ent</w>': 1, '   p ong</w>': 2, '   lo y o l a</w>': 1, '   ca pi s ce</w>': 1, '   he e d less</w>': 1, '   s r o</w>': 1, '   g ru el</w>': 2, '   al p o</w>': 1, '   q on na</w>': 1, '   qui l ted</w>': 1, '   j er u sa le m</w>': 1, '   e ter ni ti es</w>': 1, '   w ea ving</w>': 1, '   b en e di c t ine</w>': 2, '   ch or d</w>': 1, '   cho ir mi st re ss</w>': 1, '   under side</w>': 2, '   st e en w y ck</w>': 2, '   re d man</w>': 2, '   l en a pe</w>': 5, '   b al tu s</w>': 7, '   ta s se l</w>': 9, '   clo tting</w>': 1, '   ma s ba th</w>': 7, '   bro m</w>': 3, '   be he mo th</w>': 1, '   ga w k</w>': 1, '   b list er ing</w>': 1, '   mu r der in gs</w>': 1, '   a du l ter er</w>': 1, '   p hi li p se</w>': 3, '   har den bro ok</w>': 2, '   ne ther</w>': 2, '   po l ter ge i sts</w>': 1, '   de se cra tions</w>': 1, '   ex per im en ta tions</w>': 1, ' me th o ds</w>': 1, '   lo pped</w>': 1, '   li ber a li s m</w>': 3, '   e qu ani mi ty</w>': 1, '   fe der a li st</w>': 1, '   pri ed</w>': 1, '   pu r su ant</w>': 1, '   sta tu tes</w>': 1, '   re mar ry</w>': 1, '   du ll ar ds</w>': 1, '   con st ab les</w>': 1, '   ta li s man</w>': 1, '   gar re tt s</w>': 1, ' pa as ch</w>': 1, ' pi e ter</w>': 1, ' po s</w>': 1, '   in te sta te</w>': 3, '   s la u gh t ers</w>': 1, '   c r one</w>': 2, '   bo g g les</w>': 1, '   rea p ing</w>': 1, '   ci ca da s</w>': 1, '   mon i es</w>': 1, '   bor r ows</w>': 1, '   bl ack en ed</w>': 1, '   sh el ter ing</w>': 1, '   p le be i an</w>': 1, '   bu tch er y</w>': 2, '   om ni po ten tly</w>': 1, '   b loo d less</w>': 1, '   st al k in gs</w>': 1, '   in ex p li ca ble</w>': 1, '   re ta ins</w>': 1, '   te ar dro p</w>': 1, '   mi sp l ac es</w>': 1, '   p ow w ows</w>': 1, '   mi lled</w>': 1, '   ten et</w>': 1, '   go o du n</w>': 2, '   pe ck ers</w>': 1, '   ti ll er</w>': 2, '   po ta t ers</w>': 5, '   no on time</w>': 3, '   m ow ers</w>': 2, '   wh ea t le y</w>': 5, '   l ar g es</w>': 1, '   for gi v in</w>': 2, '   ph y li ss</w>': 1, '   po or house</w>': 2, '   war sh</w>': 1, '   war sh ing</w>': 1, '   ea sy going</w>': 1, '   ac qu a in t an ces</w>': 3, '   wa gon ful</w>': 1, '   cra v in</w>': 1, '   do or na il</w>': 2, '   tra i p s in</w>': 1, '   wa dn</w>': 7, '   dro o ls</w>': 1, '   dro o l in</w>': 1, '   sto l</w>': 1, '   an them</w>': 1, '   b ow ed</w>': 3, '   mi z z</w>': 4, '   o g le t ree</w>': 1, '   chi l d ers</w>': 3, '   gr un t in</w>': 1, '   har gra ves</w>': 2, '   be g at</w>': 5, '   sa ye th</w>': 1, '   or t</w>': 12, '   li qu or ed</w>': 1, '   bor ned</w>': 3, ' do in</w>': 2, '   sto b</w>': 1, '   t ou ch d ow n s</w>': 2, '   fo ll er in</w>': 1, '   wor med</w>': 1, ' ra g g in</w>': 1, '   chi l der n</w>': 1, '   he ar ed</w>': 1, '   ta k en ed</w>': 1, '   war sh ers</w>': 1, '   t in g les</w>': 1, '   fo ll er</w>': 1, '   thou gh ty</w>': 1, '   a wal kin</w>': 1, '   so die</w>': 1, ' te ll in</w>': 1, '   ra in in</w>': 3, '   mar on ey</w>': 2, ' e ms</w>': 2, ' pla y in</w>': 1, '   i d l in</w>': 1, '   la und r y m at</w>': 1, '   a loo se</w>': 1, ' car r y in</w>': 1, '   mi ll s bur g</w>': 3, '   war sh in</w>': 1, '   s om m ers</w>': 1, '   he p</w>': 2, '   la w n m ow ers</w>': 1, '   wh ea tly</w>': 1, '   w oo l ri dge</w>': 2, '   h or ned</w>': 1, '   d wi g g ins</w>': 2, '   re ev al u a ted</w>': 1, '   to o l sh ed</w>': 2, '   su b sti tu tes</w>': 1, '   le d better</w>': 1, '   th re e s om es</w>': 2, '   me l vi ll e</w>': 3, '   a m bi gu i ti es</w>': 1, '   al es</w>': 2, '   ra shi d</w>': 9, '   fuck us</w>': 1, '   as sus</w>': 1, '   bo b b se y</w>': 1, '   mar i mb a</w>': 1, '   as sa u l tu s</w>': 1, '   inter ru p tu s</w>': 1, '   w r en</w>': 1, '   wee k days</w>': 1, '   o ver co a ts</w>': 4, '   ar ti s to</w>': 1, '   sc hi m me l pen n in cks</w>': 2, '   t ins</w>': 2, '   b lu b b er</w>': 1, '   tu g bo at</w>': 1, '   fe li ci ty</w>': 4, '   m c nu t t</w>': 2, ' happ in ess</w>': 2, '   out la y</w>': 1, '   mon te c ri sto s</w>': 1, '   le gi s late</w>': 1, '   ti p to p</w>': 1, '   or chi d</w>': 1, '   st in k ers</w>': 2, '   cor set</w>': 4, '   ni mb ly</w>': 1, '   c y ru s</w>': 12, '   im bi bed</w>': 1, '   p ee k s kill</w>': 1, '   bo er u m</w>': 1, '   v a il</w>': 1, '   8 4 6 </w>': 1, ' c y ru s</w>': 1, '   bu l ging</w>': 1, '   so ck e ts</w>': 2, '   p lo ps</w>': 1, '   sp r ou ted</w>': 1, '   sh ei k</w>': 1, '   s ea side</w>': 1, '   cre e per</w>': 8, '   l en in gra d</w>': 1, '   ba k h t in</w>': 1, '   co ff ers</w>': 1, '   s ki er</w>': 1, '   o ver do</w>': 1, ' ca sh ing</w>': 2, '   cle m m</w>': 1, '   go l d ba u m</w>': 1, '   sc hi m me l pen n in ck</w>': 1, '   me di as</w>': 1, '   c lu m sin ess</w>': 1, '   gre y lo ck</w>': 1, '   m om si e</w>': 2, '   po p si e</w>': 2, '   oo d les</w>': 3, '   s lo sh</w>': 1, '   he mo to lo gi st</w>': 2, ' on to</w>': 1, '   he ine</w>': 9, '   de f en</w>': 1, '   mi y a mo to</w>': 12, ' ge ther</w>': 1, '   pi ck er</w>': 1, ' 4 5 0 0</w>': 2, ' 7 0 0 0</w>': 1, '   ju r gen s en</w>': 2, '   gi ll an d ers</w>': 1, ' ad v an ce</w>': 1, ' p un ch ed</w>': 1, ' ne tter</w>': 1, ' o ver jo y ed</w>': 2, ' pa ssed</w>': 3, '   k a bu o</w>': 2, '   do l</w>': 2, '   be ll ea u</w>': 1, '   mi l ho ll and</w>': 3, '   a g no sti c s</w>': 1, '   k en do</w>': 3, '   in f er ence</w>': 1, '   cle at</w>': 2, ' st r and</w>': 1, '   b ow lin ed</w>': 1, ' im me di ate</w>': 2, '   en mi ty</w>': 1, '   no ssi r</w>': 1, ' fi gh ting</w>': 1, ' k en do</w>': 1, '   d ra ma tur g y</w>': 1, '   su b du e</w>': 1, ' pro ve</w>': 1, ' app ea ling</w>': 1, '   ful</w>': 1, '   gu e ss work</w>': 1, '   g un ne l</w>': 5, '   spe te mber</w>': 1, '   hea lt</w>': 1, ' in i ti ally</w>': 1, '   in i ti ally</w>': 1, '   lo a st</w>': 1, '   s mo l t z</w>': 1, '   do g wa tch</w>': 1, '   sh or th ard</w>': 1, '   tr ans m is</w>': 1, '   si ons</w>': 1, '   see p ed</w>': 1, '   la sh ed</w>': 2, ' car l</w>': 1, '   h mm n</w>': 2, '   gu d m und s son</w>': 1, '   o th</w>': 1, '   f lan ge</w>': 2, '   in f er red</w>': 1, '   re fi tt ed</w>': 1, ' de f en d ant</w>': 2, '   m ea sur ing</w>': 1, '   chan d l er y</w>': 1, '   ab so l</w>': 1, '   fa ir l ead</w>': 1, ' dr ow ning</w>': 2, '   mu c us</w>': 1, ' fo am</w>': 1, '   at ti tu </w>': 1, '   me k u m</w>': 9, '   ph y si ca li ty</w>': 1, '   ru bri ck</w>': 3, '   fr ow ning</w>': 1, ' a q </w>': 1, '   me l ton</w>': 6, '   s mi r ks</w>': 1, '   st on e f ac ed</w>': 1, ' pla t for m</w>': 2, '   bl in ks</w>': 2, '   f lin ch es</w>': 1, ' te sted</w>': 1, '   wee ell</w>': 1, '   bi en sto ck</w>': 6, '   ro se ll a</w>': 1, '   o l g a</w>': 1, '   con ser v at ory</w>': 3, '   u p sy</w>': 2, ' da i sy</w>': 1, '   ber th s</w>': 2, '   ti ck li sh</w>': 3, '   man ha tt ans</w>': 2, '   cla m ba ke</w>': 1, '   o s good</w>': 12, '   e lo pe</w>': 2, '   con ven tions</w>': 2, '   y ac h ts</w>': 2, '   re pu b li c s</w>': 1, '   pre ten ses</w>': 1, '   v a ll e e</w>': 2, '   mo t or bo at</w>': 3, '   tri c ki est</w>': 2, '   s qu ea l ed</w>': 1, '   na ve l</w>': 1, '   k ow al c z y k</w>': 2, '   che sts</w>': 1, '   e cla ir s</w>': 1, '   t ar ts</w>': 1, '   wa ba sh</w>': 2, '   1 0 9 8 </w>': 1, ' fi d dle</w>': 1, '   u r ban a</w>': 3, '   kn ack w u r st</w>': 2, '   o ver f l ows</w>': 1, '   pi ck for d</w>': 1, '   sho o</w>': 2, '   de li ca te ss en</w>': 2, '   in la y</w>': 1, '   v as s ar</w>': 3, '   ba th house</w>': 1, '   c r ow ding</w>': 1, '   u k u le le</w>': 4, '   poli a k off</w>': 2, '   te ar oo m</w>': 1, '   ac ro ba ti c</w>': 1, '   cont or tion i st</w>': 1, ' se a</w>': 1, '   sha pe ly</w>': 1, '   ca le d on i a</w>': 2, '   ha tt er as</w>': 2, '   we in me y er</w>': 1, '   mar ce lled</w>': 1, '   re jo in ing</w>': 1, '   sh e bo y g an</w>': 2, '   lo lli po p</w>': 5, ' go o d ni cks</w>': 1, '   a we i gh</w>': 1, '   ven e z u el an</w>': 2, '   ad vi s ers</w>': 1, '   su i ci da lly</w>': 1, '   ba z a ars</w>': 2, '   ca pu t t</w>': 1, '   bar be cu ing</w>': 1, '   in ha ling</w>': 1, '   no vo ca ine</w>': 1, '   hu p mo bi le</w>': 1, ' sh oo ing</w>': 1, '   fo g ged</w>': 1, '   gr ab b ers</w>': 1, '   co ti lli ons</w>': 1, '   sy n co pa t ors</w>': 2, '   co le s la w</w>': 1, '   f lo cks</w>': 2, '   cu r dle</w>': 1, ' pi mp ly</w>': 1, '   pa ll be ar ers</w>': 1, '   de mi ta s se</w>': 1, '   v u l c ani z ing</w>': 1, '   i g nor a m us</w>': 1, '   mo z ar e ll a</w>': 2, '   out sh ine</w>': 1, '   b ac ar d i</w>': 1, '   1 5 1 </w>': 1, '   fr ow ny</w>': 1, '   g ru mp le sti l s kin</w>': 1, '   le mon dro p</w>': 1, '   bu g gi es</w>': 1, '   pi a f</w>': 1, '   re uni ted</w>': 1, '   p inter e llo</w>': 1, '   cen ter pi e ces</w>': 1, '   car po o ling</w>': 1, ' w oo ten s</w>': 1, '   re fin ing</w>': 1, '   d day</w>': 1, '   la si k</w>': 1, '   op ti ci an</w>': 1, '   ne tt les</w>': 1, '   ex gir l friend</w>': 1, ' fee ly</w>': 1, '   c ri k ey</w>': 1, '   ti ra mi su </w>': 1, ' t ou ch ing</w>': 1, '   com bu st</w>': 1, ' mor ally</w>': 1, '   pa pi ll o ma vi ru s</w>': 1, '   un sin g le</w>': 1, '   un mi ser able</w>': 1, ' ba i le ys</w>': 1, '   ti p to es</w>': 1, '   m ac ra m</w>': 1, '   ma tch y</w>': 1, ' ma tch y</w>': 1, '   b lea ch er</w>': 1, '   g y ne co lo gi sts</w>': 1, '   a ho y</w>': 1, ' dr un k er</w>': 2, '   e mu la ting</w>': 1, '   sa fe gu ard</w>': 1, '   m ac ca ll u m</w>': 1, '   wh e w w w w</w>': 1, '   s oun der</w>': 9, '   bor der da le</w>': 1, '   a g li tter</w>': 1, ' fin er</w>': 1, '   gu er don</w>': 1, '   bo a tw right</w>': 17, '   ju st a</w>': 1, '   wh ee e w</w>': 1, ' cor d in</w>': 1, '   af ta</w>': 4, '   cla y bur n</w>': 1, '   na w w</w>': 1, '   po s su ms</w>': 1, '   ha mb on es</w>': 1, '   wom en fo l k</w>': 2, '   p out</w>': 2, '   sh ar e c ro p</w>': 1, '   c ro pp in</w>': 3, '   com mi ss ary</w>': 2, ' s ea s ons</w>': 1, '   re c ti f y</w>': 1, '   lan d s down</w>': 1, '   u so</w>': 1, '   ch u t z pa h</w>': 1, '   bi t c he</w>': 1, '   dam m i</w>': 1, ' h or se fuck er</w>': 1, ' pi llow</w>': 1, ' on t an a</w>': 1, ' o g g y</w>': 1, '   4 5 3 </w>': 1, ' chi p</w>': 6, '   j i z</w>': 1, '   car t man</w>': 15, '   aga gh</w>': 1, '   a a gh gh</w>': 1, '   ter ran ce</w>': 30, '   z ey</w>': 2, '   wa ga ga h gh gh</w>': 1, '   gla ddy</w>': 1, '   me ani e</w>': 1, '   che d d ar</w>': 2, '   ph lli p</w>': 1, '   un cle fuck a</w>': 6, '   p hi li i p</w>': 1, '   m r ph mm mp h</w>': 1, '   aga ga h</w>': 1, '   mp h</w>': 15, '   r mp h</w>': 16, '   r m</w>': 17, '   pro of s</w>': 3, '   s n ack y</w>': 3, '   r mp r mm h</w>': 1, '   r mp m h</w>': 1, '   bro v lo f s k i</w>': 3, '   m k ay</w>': 6, '   ra per</w>': 3, '   pri ck fuck</w>': 2, '   re ha bi li ta ted</w>': 2, '   m pa a</w>': 1, '   pe in ci pa l</w>': 1, '   si m i</w>': 1, '   da da</w>': 3, '   do k y</w>': 1, '   ma p ing</w>': 1, '   de e der</w>': 1, '   pu r p re</w>': 1, '   mm mm mp h ph ph ph p</w>': 1, '   m r p h</w>': 3, '   mp r h m</w>': 1, '   mp r p h</w>': 1, '   bo o bi e</w>': 1, '   g w pa a p a</w>': 2, '   bu t for</w>': 2, '   re vo lu tion i sts</w>': 1, '   z em</w>': 2, '   be e tch</w>': 1, '   pri z on ers</w>': 1, '   di ck su ck er</w>': 1, '   co ck ma ster</w>': 1, ' c li t or is</w>': 1, '   v u l v a</w>': 2, '   cu ss es</w>': 1, '   di ck ho le</w>': 1, '   w o a</w>': 1, '   shi t ea ter</w>': 1, ' fi st ing</w>': 1, ' can ad a</w>': 1, '   in to l er an ce</w>': 1, '   der sh wi t z</w>': 1, '   imp ea ch ing</w>': 1, ' er ran ce</w>': 1, '   un tal en ted</w>': 1, '   un fun ny</w>': 1, '   g ev al t</w>': 1, '   we en ed</w>': 1, '   pu r ve y or</w>': 1, '   sc ro tu m</w>': 1, '   as s li cking</w>': 1, '   pi g fuck er</w>': 1, '   e mb ar as sed</w>': 1, '   r en e ge</w>': 1, '   h ows</w>': 1, '   ab oo t</w>': 1, '   fu ton</w>': 2, '   cho co la te y</w>': 2, ' n ack y</w>': 1, '   cen sor ship</w>': 1, '   c run ch y</w>': 1, '   p in ki es</w>': 1, '   z is</w>': 2, '   k or o s</w>': 3, ' k or o s</w>': 1, '   sen su ally</w>': 1, '   \t \t \t \t </w>': 1, ' dis sa ti s f ac tion</w>': 1, '   dis sa ti s f ac tion</w>': 2, '   fu ses</w>': 2, '   der e gu la tion</w>': 1, '   li fe po ds</w>': 2, '   je t ti son ed</w>': 1, '   of f l ine</w>': 1, '   c happy</w>': 1, '   st ea ms</w>': 2, '   s che ma ti c</w>': 1, '   he ll l p</w>': 1, '   comp li es</w>': 1, '   ear th ri se</w>': 1, ' re c ent</w>': 1, '   tr an qui lli ty</w>': 1, '   dre x l er</w>': 1, '   sh ei ks</w>': 1, ' o x y gen a ted</w>': 1, '   f lu en tly</w>': 1, '   s k u l ks</w>': 1, '   k ac ked</w>': 3, '   af fi x</w>': 1, '   2 4 1 </w>': 1, ' ser ve</w>': 2, ' pu tt s</w>': 1, ' go l f er</w>': 1, '   fa z</w>': 8, '   fa z el i</w>': 20, '   ma pl es</w>': 1, '   cra d les</w>': 2, '   s ni g ger</w>': 1, '   pa tch es</w>': 1, '   k a st le</w>': 25, '   in cor ru p ti ble</w>': 1, '   me ga lo mi lli on a ir es</w>': 1, '   da la l a</w>': 1, '   car u so</w>': 1, '   s lea z e ba lls</w>': 4, '   b lu r t</w>': 2, '   b ru d da s</w>': 1, ' cu es</w>': 1, '   e er i ly</w>': 1, '   ac com mo da ted</w>': 1, '   re mo l ds</w>': 1, '   de l r in</w>': 1, '   bar n ac le</w>': 1, ' pu t t</w>': 2, '   ta d po les</w>': 1, '   g ru m pi ly</w>': 1, '   r on k on k om a</w>': 1, '   s k in ner</w>': 6, '   un ta in ted</w>': 1, ' fa z el i</w>': 1, '   v an de mar k</w>': 1, '   mor a li sti c</w>': 1, ' bo di es</w>': 2, ' com er</w>': 1, '   la m ba da</w>': 1, '   h ow z i t go in</w>': 1, '   b oun cy</w>': 6, ' b oun cy</w>': 2, '   shi sh</w>': 3, '   b lu e face</w>': 3, '   c r in g es</w>': 1, '   lu bri ca tion</w>': 2, '   stra i t j ack et</w>': 1, ' b ow ling</w>': 3, '   mu r mu r</w>': 1, ' bl ow t or ch</w>': 1, '   p ss sh h t</w>': 1, '   ke bab s</w>': 1, '   r ac </w>': 1, '   au to ma te</w>': 1, '   gu pp i es</w>': 1, ' cha mp</w>': 1, '   p in f all</w>': 1, '   mo to c ro ss</w>': 2, '   im pa ssi ve ly</w>': 1, '   1 5 4 </w>': 1, ' b ow l er in a</w>': 1, ' k a st le</w>': 1, '   st ro kin</w>': 1, '   h y d ran t</w>': 3, '   fif t y i sh</w>': 1, '   w ea ves</w>': 1, '   ga p ing</w>': 1, '   s sh he ei i ll a a a</w>': 1, '   re v s</w>': 2, '   p ea ls</w>': 1, '   m oun ts</w>': 2, '   th ro b</w>': 2, '   ru mb les</w>': 1, '   in sta ted</w>': 1, '   s an est</w>': 1, '   ti ck l ers</w>': 1, '   si de bur n s</w>': 2, '   d ra w b ac ks</w>': 1, '   mo t or bi ke</w>': 1, '   pro ffer ed</w>': 1, '   d war f s ca m</w>': 1, '   k in g p in</w>': 1, '   gu i de lin es</w>': 1, '   fa ta li ty</w>': 3, '   bu sy boy</w>': 2, '   bi o che mi st</w>': 1, '   y ea sts</w>': 2, '   kn ow in g ly</w>': 1, '   ac kn ow le d ge ment</w>': 1, '   imp r t ant</w>': 1, '   dea th wi sh</w>': 1, '   cor ro si ve</w>': 1, '   pre s su ri ze</w>': 1, '   he li u m</w>': 4, '   u lf</w>': 3, '   sur vi ver s</w>': 1, '   o pp ort un in ty</w>': 1, '   re t y pe</w>': 1, '   spi ra lling</w>': 1, '   an th ro p om or p hi c</w>': 1, '   re s in</w>': 1, '   hon e y com b</w>': 1, ' de la y</w>': 1, '   as c ends</w>': 1, '   mi mi cking</w>': 1, '   de e p s at</w>': 1, '   su b stan tive</w>': 1, '   je ll y fi sh</w>': 11, '   v in di c ti ven ess</w>': 1, '   beli ve</w>': 3, '   con spi ra t ori ally</w>': 1, ' kee p ing</w>': 3, '   j on e ses</w>': 1, '   man i fe st ing</w>': 2, '   di u r na l</w>': 1, '   par a so lu tr ine</w>': 2, '   par ac in</w>': 2, '   sur pre ssed</w>': 1, '   tri ch l ori de</w>': 1, '   ch l or a mp h en i co l</w>': 1, '   ri or d an</w>': 1, '   ha lu c in a tions</w>': 1, '   dr ow sin ess</w>': 3, '   t ar a z ine</w>': 1, '   an a lo gu e</w>': 1, '   sin ta g</w>': 1, '   v al d om et</w>': 1, '   h y dro ch l ori de</w>': 1, '   di ph en y l</w>': 1, '   par l en e</w>': 1, '   cre w man</w>': 2, '   na tu ra list s</w>': 1, '   s qui ds</w>': 2, '   re fri d ger a tor</w>': 3, ' de e p se a</w>': 1, '   u sn</w>': 1, '   s an d pa per</w>': 1, '   i tch ed</w>': 1, '   st in ging</w>': 1, '   ex ha le</w>': 1, '   ac com o da tions</w>': 1, '   s ci e ti sts</w>': 1, '   sa g an</w>': 1, '   pa tting</w>': 1, '   can ni ba li s m</w>': 1, '   ne an der th al s</w>': 1, '   du ff le ba g</w>': 1, '   ke en er</w>': 1, '   op tom i sti c</w>': 1, '   ter re sti al</w>': 1, ' sp here</w>': 1, ' hi ding</w>': 1, '   su b sti tu ting</w>': 1, '   spi ra ls</w>': 1, '   su b sti tu tion</w>': 2, ' ar ri ved</w>': 1, '   we en y</w>': 1, '   pre sen ting</w>': 1, '   ton n es</w>': 1, '   ma the ma ti ci ans</w>': 1, '   a ir cra sh</w>': 1, '   i s lo a ted</w>': 1, ' im po t ent</w>': 1, '   ki b er</w>': 5, '   a qui la e</w>': 1, '   or g an a</w>': 6, '   k en ob i</w>': 4, '   star ki ll er</w>': 5, '   al der a an</w>': 3, '   si th</w>': 8, '   bo g an</w>': 7, '   sp ee der</w>': 1, '   a mp li f y</w>': 2, '   con da w n</w>': 1, '   ei sle y</w>': 1, '   sp ac e por t</w>': 2, '   ra di a tes</w>': 1, '   o be ys</w>': 1, '   mar v el s</w>': 1, '   b en d u</w>': 3, '   star t ro op er</w>': 2, ' i ary</w>': 1, '   an ni kin</w>': 1, '   ro a med</w>': 1, '   tu s k en s</w>': 2, '   star for ce</w>': 1, '   cor el li an</w>': 1, '   s ke w</w>': 1, '   y a v in</w>': 2, '   shi p y ar ds</w>': 1, ' 8 8 </w>': 1, '   tra w l er</w>': 1, '   dea d ends</w>': 1, ' mo tes</w>': 1, '   bo o ter</w>': 1, '   out po sts</w>': 1, '   du e ly</w>': 1, ' ro i ds</w>': 3, '   con den sing</w>': 2, '   ma sh ers</w>': 1, '   2 1 8 7 </w>': 1, '   coun ter part</w>': 1, '   de too</w>': 1, '   an ch or head</w>': 1, '   r ou gh ne cks</w>': 4, '   bar ca l ow</w>': 2, '   f la ts</w>': 4, '   s ki mm ers</w>': 2, '   loo k ou ts</w>': 1, '   ther te</w>': 1, '   nu t bu ster</w>': 1, '   ra sc z a k</w>': 7, '   in f an ter y</w>': 4, '   y a a a a a a a a</w>': 1, '   wh a d</w>': 3, '   1 2 4 0</w>': 1, '   mar au der</w>': 2, '   t ro op r es</w>': 1, '   ex c ru ci at in g ly</w>': 1, '   man e u ver ing</w>': 5, '   star side</w>': 2, '   se ma ph ore</w>': 1, '   dis re ga ed</w>': 1, '   co po se ti c</w>': 1, '   de la di er</w>': 1, '   re p lo tt ed</w>': 1, '   in s ru ctor</w>': 1, '   ac i vi li an</w>': 1, '   ter e sh k o v a</w>': 1, '   a a a a a a a a a a a a a a</w>': 1, '   wi l ya</w>': 1, '   ve sti ga l</w>': 1, '   ga st ri c</w>': 1, '   ca e cal</w>': 1, '   ee u ch</w>': 1, '   c r w o ded</w>': 1, '   i ban e z</w>': 4, '   ro d ger</w>': 5, '   a mi sta ke</w>': 1, '   ar ac h ni d</w>': 1, '   r ou gh ne ck</w>': 3, '   ac i ti z en</w>': 2, '   z e ge ma</w>': 3, ' ci ti z en s</w>': 1, '   s an t or i</w>': 1, '   dis cou ra g es</w>': 1, '   a ter m</w>': 1, '   a a a a a a a a a a a</w>': 1, '   gi ll e spi e</w>': 3, '   mi co m</w>': 1, '   on si te</w>': 1, '   0 8 2 1 </w>': 1, ' op er a tion al</w>': 2, '   im me di e tly</w>': 1, '   figu re you</w>': 1, '   com le te</w>': 1, '   re ce ice</w>': 1, '   wa t k ins</w>': 1, '   y op u</w>': 1, ' ci vi c</w>': 1, '   in fe st</w>': 1, '   pro c ee de</w>': 2, ' c rea tor</w>': 1, '   in fe st ing</w>': 1, '   me l ds</w>': 1, '   le ar n able</w>': 2, ' wh om</w>': 1, '   pro ce de</w>': 1, '   re p re</w>': 1, '   le ss en ing</w>': 1, ' be lli ger en cy</w>': 1, '   de ck er</w>': 17, ' secon ds</w>': 2, ' un war ran ted</w>': 1, '   mi sin ter</w>': 1, '   pre ted</w>': 1, '   un fami li ar i ty</w>': 1, '   e mb ar</w>': 1, '   ra ssed</w>': 1, '   re de sig n</w>': 2, '   coun ter man ded</w>': 1, '   son a k</w>': 1, ' la tion ship</w>': 1, '   du p li ca tes</w>': 1, ' pre ci se</w>': 1, '   du p li ca ted</w>': 1, ' me chan i s m</w>': 1, '   na vi ga tion al</w>': 1, '   de f le c t ors</w>': 1, '   di re c tion al</w>': 1, '   con i c</w>': 1, '   ce li b ac y</w>': 1, ' un tri ed</w>': 1, '   un tri ed</w>': 1, '   re fi tting</w>': 1, ' me ld</w>': 1, '   ev o l ving</w>': 1, '   a du l th o od</w>': 1, ' i t self</w>': 1, '   de ma ter i a li z ed</w>': 1, ' en s or</w>': 1, ' s war ms</w>': 1, '   m ea sur ably</w>': 1, '   p ow er fi e ld</w>': 2, ' in sig ni fi can ce</w>': 1, ' pu z z le ment</w>': 2, '   in v ad es</w>': 2, '   ab sen ting</w>': 1, '   no gu r a</w>': 1, '   app ri sed</w>': 1, ' u r g ent</w>': 1, '   de c el er a ting</w>': 1, '   hu s ki es</w>': 7, '   di e t ary</w>': 2, ' bl ack b all</w>': 1, '   fi l m fo l k</w>': 1, '   pr in t sho p</w>': 3, '   pe l ting</w>': 1, '   wa ter for d</w>': 6, '   ye s su h</w>': 1, ' pu ri ty</w>': 4, ' in fi de li ti es</w>': 1, ' rea li z es</w>': 1, '   fi sh ho ok</w>': 1, '   t y pi st</w>': 4, '   c l</w>': 2, '   pa per c li p</w>': 3, ' fi re house</w>': 2, ' con f li ct</w>': 1, '   ch h h</w>': 1, ' th a ts</w>': 1, ' s an c ti ty</w>': 1, '   v ou ch sa f ed</w>': 1, ' se es</w>': 2, ' mer cy</w>': 1, ' p ra y ed</w>': 2, '   ma t in e es</w>': 1, '   s ar d ine</w>': 1, '   da l ma tion</w>': 2, '   da l ma ti a</w>': 1, '   s ar din i a</w>': 1, '   re ver sa ls</w>': 1, '   un lin ed</w>': 1, ' an gu i sh</w>': 1, '   sp ru c ed</w>': 1, '   t y pe wri tter</w>': 1, '   fi an</w>': 1, '   bu d ge ted</w>': 1, '   m ea s l</w>': 1, '   gr in ds</w>': 1, '   th ea tri ca ls</w>': 1, ' com mo di ty</w>': 1, '   no b by</w>': 1, '   a ll a su d d en</w>': 2, '   bar r en ger</w>': 16, '   per ni ci ous</w>': 1, ' ca f fe ine</w>': 1, '   sc r</w>': 2, '   h ow i e</w>': 3, '   re wri tes</w>': 1, ' bi mb o</w>': 1, ' d on k ey</w>': 1, '   ro ss en</w>': 3, ' ha gu e</w>': 1, ' b le e ding</w>': 1, '   g r</w>': 1, ' o ver come</w>': 1, ' u p set</w>': 1, ' sch oo l night</w>': 1, ' wai t re ss</w>': 1, ' a pr on</w>': 1, ' ar ri v al</w>': 1, '   in con ven i en c ed</w>': 1, ' wi l ds</w>': 1, ' er r or</w>': 1, ' ma tt ers</w>': 1, '   ma t z oh</w>': 1, ' fee lin gs</w>': 2, '   inter pre tive</w>': 1, ' to il</w>': 1, ' ani ma ls</w>': 1, '   cha gr in ed</w>': 1, ' mor ti fi ed</w>': 1, '   el an or a</w>': 1, ' d use</w>': 1, '   1 9 0 5 </w>': 1, '   li l ac s</w>': 5, '   cont r</w>': 1, ' li gi on</w>': 1, ' nu de</w>': 1, ' ca mer a man</w>': 1, '   che ss y</w>': 1, '   sp osed</w>': 1, ' gen i tal s</w>': 1, '   o ver a g es</w>': 1, '   y un ti f</w>': 1, ' possi b ly</w>': 1, '   tur p</w>': 1, '   co ll u sion</w>': 1, '   per fi dy</w>': 1, '   ca ll sh e et</w>': 2, ' mar ty</w>': 3, ' mi d wife</w>': 1, ' to ta lly</w>': 2, ' su spi ci ons</w>': 1, ' li ter ally</w>': 1, '   cu r l ed</w>': 2, '   ge t ti m</w>': 2, '   ma t z o h s</w>': 1, '   what ser name</w>': 1, ' no sh</w>': 1, '   mo v </w>': 1, ' fa u lt</w>': 2, '   sy m bo li ze</w>': 2, ' ha ying</w>': 1, ' che st</w>': 1, ' si d down</w>': 1, ' me dea</w>': 1, ' tu e s day</w>': 1, '   1 8 3 5 </w>': 2, ' ki tch en</w>': 1, ' sta tu t ory</w>': 1, ' un supp or ted</w>': 1, ' ba z o om er</w>': 2, ' do t</w>': 1, '   com any</w>': 1, ' sh ee p</w>': 2, '   z i z</w>': 1, ' ye ar ly</w>': 1, '   re de cor ate</w>': 1, '   sh er</w>': 1, ' ex c lu si ve</w>': 1, '   la z y boy</w>': 2, ' re qui re men ts</w>': 1, ' per m it</w>': 1, ' bo ther</w>': 1, '   y ea up</w>': 1, ' as se mb ly</w>': 1, ' won der ed</w>': 1, '   ga g n on</w>': 1, ' l even</w>': 1, ' fi re hu t</w>': 1, '   gi v v em</w>': 1, ' r ac he</w>': 2, '   r ac he</w>': 9, '   go di v a</w>': 3, ' stra ight</w>': 3, ' in st ant</w>': 1, '   f ar t face</w>': 1, ' d or k face</w>': 1, ' c r ow d</w>': 1, ' h en ch men</w>': 1, '   in hu man i ty</w>': 1, '   f lan n er y</w>': 1, '   th u r s d</w>': 1, ' th u r s day</w>': 1, '   a li se</w>': 1, '   u ran us</w>': 1, '   a ti ca lly</w>': 1, ' su i ta ble</w>': 1, ' pre p</w>': 1, ' whi p</w>': 1, ' so l ves</w>': 1, ' la u gh ed</w>': 1, ' st e p mother</w>': 1, ' bo ss ing</w>': 1, ' w ea k</w>': 1, ' par en ts</w>': 2, '   fro g f ac ed</w>': 1, '   f oo t men</w>': 1, '   b en n n n</w>': 1, '   sa tur d</w>': 1, ' a ll er gi c</w>': 1, '   a a a a g g g gh h h h</w>': 1, '   b en n n n n</w>': 1, ' comp le tely</w>': 2, '   f la k y</w>': 1, ' e g g z ac tly</w>': 1, ' thin ks</w>': 1, '   fi re per son</w>': 1, ' he ld</w>': 1, ' su ff o ca te</w>': 1, '   st e p m om my</w>': 1, '   lu i g i</w>': 1, ' la mb </w>': 1, ' sp a gh e tt i</w>': 1, '   m om mi es</w>': 1, '   ha t ea ble</w>': 1, '   exac </w>': 1, '   my st er i</w>': 1, ' ra di ate</w>': 1, ' lo se</w>': 1, ' a gen ci es</w>': 1, '   an n ab </w>': 2, '   for ce fe ed</w>': 1, ' cou s cou s</w>': 1, '   com mi t m</w>': 1, ' s lu gs</w>': 1, ' t re es</w>': 1, '   pen ed</w>': 1, '   al m o</w>': 1, '   mo le hi ll</w>': 1, ' com a</w>': 1, '   bur ni sh es</w>': 1, '   sto lie</w>': 1, ' plea ding</w>': 1, ' ca ving</w>': 1, ' l ow er ing</w>': 1, ' de f en ding</w>': 1, ' li mp</w>': 1, ' n ow ba lling</w>': 1, '   in v is</w>': 1, '   i ble</w>': 1, '   t rea d mi ll</w>': 1, '   f ere</w>': 1, '   be d t i</w>': 1, ' pri m o</w>': 1, ' be ll bo tt om s</w>': 1, '   k o vi t s k y</w>': 2, '   s wa mp ing</w>': 1, '   d rea m bo at</w>': 1, '   de p o</w>': 1, '   un de ser ving</w>': 1, ' mar r ying</w>': 1, ' g rea te st</w>': 1, '   w ow i e</w>': 1, '   ex pen s</w>': 1, '   dr i</w>': 1, '   re st ro om s</w>': 1, '   reme mb </w>': 1, '   co d d l ed</w>': 1, ' bra ts</w>': 1, ' in vo l ved</w>': 1, '   im per i ous</w>': 1, ' for gi ven ess</w>': 1, ' in app ro pri ate</w>': 1, ' h in gs</w>': 1, '   sur pr i</w>': 1, '   pa e ll a</w>': 1, '   su i</w>': 1, ' s ki ing</w>': 1, ' cl o</w>': 4, ' ha ff</w>': 1, ' per son ally</w>': 2, '   sch in d l er</w>': 1, '   s co o b</w>': 1, '   z e da</w>': 1, '   s co o by</w>': 17, '   mu er to</w>': 1, ' in vi te</w>': 1, '   pro vo ca tive</w>': 2, '   ca mer a work</w>': 1, '   f ac i le</w>': 1, '   5 5 0</w>': 1, '   5 2 0</w>': 1, ' na p k ins</w>': 1, '   tw in ke ys</w>': 1, ' e u ro p ean</w>': 1, '   re ca ta lo gu e</w>': 1, '   k in k in ess</w>': 1, ' ra w</w>': 1, '   con for mi s m</w>': 1, '   e mb le ma ti c</w>': 1, '   di si ll u si on ment</w>': 1, '   re con ce i ve</w>': 1, '   so ci o e con om i c</w>': 1, ' g ger</w>': 2, ' under gr ound</w>': 2, '   der ri da</w>': 1, '   o x man</w>': 1, '   har w ard</w>': 1, ' 7 1 0</w>': 1, '   pr n ce ton</w>': 1, '   su n dan ce</w>': 2, '   af ter ma th</w>': 2, '   co lu mb ine</w>': 1, '   fi l more</w>': 7, '   con gre ss woman</w>': 3, '   che m c o</w>': 5, '   ch u mp s</w>': 1, ' vi o la ting</w>': 1, '   a m ba ss</w>': 1, '   ad or</w>': 1, '   le i z bur g</w>': 1, ' poli ti ca lly</w>': 1, ' con gre ss woman</w>': 1, '   bu n s en</w>': 1, ' an ch or</w>': 1, '   to pp ing</w>': 1, '   pi pe lin es</w>': 2, '   co di fi ed</w>': 3, '   sh el ved</w>': 1, ' gra s p</w>': 1, '   as sa s sin a tes</w>': 1, '   in ca l cu l able</w>': 1, '   pi p ing</w>': 3, '   k z</w>': 2, ' nu mer i ca lly</w>': 1, '   k om p ong</w>': 1, '   7 0 7 0 9 </w>': 1, '   c y ni ca lly</w>': 1, '   h y po c ri ti c</w>': 1, '   dis ar ma ment</w>': 6, '   g ru d d</w>': 1, '   j ac ke ted</w>': 1, ' th ori u m</w>': 2, '   en sh r ou d</w>': 1, ' min i a ture</w>': 1, '   l ar ge men ts</w>': 1, '   k u l ni ck</w>': 1, '   au th en ti ca te</w>': 1, '   mo ff o</w>': 1, '   ori fi ces</w>': 2, '   un tri g ger</w>': 3, '   un al ter able</w>': 1, '   tri g ger ing</w>': 1, '   ri k i</w>': 2, '   ta v i</w>': 1, ' ta v i</w>': 1, ' b at</w>': 2, '   7 0 1 </w>': 1, ' sta tion</w>': 1, '   e dad</w>': 1, '   8 4 3 </w>': 4, '   pi e tra s z ki e wi c z</w>': 3, '   ce i da</w>': 2, '   te le pr in ter</w>': 2, '   bur pe l son</w>': 3, '   p sy cho es</w>': 1, '   p un tri ch</w>': 1, '   re com en ded</w>': 1, '   sa bo te u rs</w>': 3, '   com m un i</w>': 1, '   ca tions</w>': 1, '   you d don</w>': 1, '   in do c tr in a ted</w>': 1, '   in spe c t ors</w>': 2, '   bl ab </w>': 1, '   in ff act</w>': 1, '   fa sc in ate</w>': 1, '   mu ssed</w>': 1, '   do y your happ en</w>': 1, '   cle men c ea u</w>': 1, '   c r m</w>': 12, '   z l at</w>': 2, ' z a g ging</w>': 1, '   la pu ta</w>': 2, ' le per</w>': 1, '   f ac e man</w>': 1, '   con co m</w>': 1, '   ac kn ow le d ge men ts</w>': 1, '   c ru d le y</w>': 1, '   bu l di ke</w>': 1, '   a g gre ss or</w>': 2, '   e ch el on</w>': 3, ' bor n e</w>': 2, '   e ch el ons</w>': 1, '   pre si ent</w>': 1, '   b in k y</w>': 2, '   bro m din g na</w>': 2, '   lo th ar</w>': 15, '   ba ll mu ff</w>': 1, '   e c m</w>': 3, '   fi re b all</w>': 3, ' an g le</w>': 1, '   qui ff er</w>': 1, '   to e j am</w>': 4, '   rea d i</w>': 1, ' inter na tion al</w>': 1, '   sig n al ing</w>': 2, '   bi m</w>': 2, '   k ar na k</w>': 2, '   de co d es</w>': 1, ' pla y ers</w>': 1, '   de u ter i u m</w>': 1, '   li e u ten t ant</w>': 1, '   z o g g</w>': 2, '   au th en ti ca tion</w>': 1, ' re call</w>': 1, '   op e</w>': 1, '   ar ro wh ead</w>': 1, '   lo that</w>': 1, '   fu sing</w>': 2, '   se on d</w>': 1, '   he ll n no</w>': 1, '   an gu i sh ed</w>': 1, '   re co ver able</w>': 1, ' ba lled</w>': 1, '   gre en hou ses</w>': 1, '   min e si tes</w>': 1, '   ac com o da te d d</w>': 1, '   a da p ta ble</w>': 1, '   ra di o ac ti vi ty</w>': 2, '   hi gh ga te</w>': 1, '   out pa ti ent</w>': 1, ' bur ger</w>': 2, '   m n</w>': 1, '   di a le c ti cal</w>': 1, '   du b es</w>': 1, '   du be</w>': 1, '   sch ro om s</w>': 1, '   bur n fi e ld</w>': 9, '   so o ze</w>': 19, '   gi lli g an</w>': 2, '   li te</w>': 2, '   bo d y su it</w>': 1, '   je ff ster</w>': 1, '   bu mm ing</w>': 1, '   or e o s</w>': 1, '   ban gla de sh i</w>': 1, '   co a gu la ted</w>': 1, ' be e</w>': 5, ' d rea m</w>': 2, '   h ore</w>': 1, '   sta di u ms</w>': 1, '   mo y ni ha n</w>': 2, '   ne o fa s ci st</w>': 1, '   pa kee s a</w>': 2, '   tra sh in</w>': 1, '   lin e man</w>': 1, '   sta ir ma ster</w>': 1, '   v ani e ty</w>': 1, '   do cu men t ar i es</w>': 1, ' an n e</w>': 1, '   s an da ls</w>': 1, '   loo s er</w>': 1, '   nu d ging</w>': 1, '   ch en ow s k y</w>': 1, '   je ffer y</w>': 1, '   sp ar er i b</w>': 1, '   ber hard</w>': 1, '   ro ck st ar</w>': 1, '   s ar din es</w>': 1, '   na z e er</w>': 3, '   cha u d ry</w>': 1, '   nu di st</w>': 2, '   ni car a gu a</w>': 3, '   mo sh</w>': 1, '   con su mer i s m</w>': 1, '   con ce p tu al</w>': 2, ' e c</w>': 1, '   t ar p it</w>': 1, '   mu si cl and</w>': 1, ' dro ve</w>': 1, '   min i v an</w>': 2, '   de p p</w>': 1, '   ber n hard</w>': 1, '   ho l bro ok</w>': 1, '   g rea se ca ke</w>': 1, ' st al k er</w>': 1, '   sa fe way</w>': 1, ' ga h</w>': 1, ' bu h</w>': 1, '   bi on i c</w>': 1, '   cle o</w>': 5, '   ke an u</w>': 1, '   en ab les</w>': 1, '   po t ti es</w>': 1, ' ha a a a</w>': 1, ' wee e</w>': 1, ' ha a a</w>': 1, '   ni ck na mes</w>': 1, '   gre t z k y</w>': 1, '   ha a a a a</w>': 1, '   ga ss y</w>': 1, ' d rea ms</w>': 1, ' pre g n an cy</w>': 1, '   si t z</w>': 1, ' na tal</w>': 1, '   cho cu l a</w>': 1, '   fran k en ber ry</w>': 1, ' st in k y</w>': 1, '   ven de l a</w>': 1, ' bo di ed</w>': 1, ' nee dle</w>': 1, '   no oo oo o o</w>': 1, '   si ph on ing</w>': 1, '   han d sp r ing</w>': 1, ' st e w art</w>': 1, ' pa pp a</w>': 1, ' ck</w>': 1, ' i tch y</w>': 1, '   p hi lli pe</w>': 1, '   sp r in ger</w>': 1, '   ne ther world</w>': 1, '   pe t uni a</w>': 1, ' m ou th y</w>': 1, '   m ou th y</w>': 1, ' lin ks</w>': 2, ' car at</w>': 1, ' ho p</w>': 1, '   a do l p he</w>': 1, '   men j ou </w>': 1, '   gi ll is</w>': 18, '   car e fu ll</w>': 1, '   sy no p s is</w>': 1, '   pa la z z o s</w>': 1, '   oo sts</w>': 1, '   f re e h ly</w>': 1, '   la und red</w>': 1, '   k er chi e f s</w>': 1, '   st en o gra p hi c</w>': 1, '   di ction</w>': 1, '   e le c</w>': 1, '   tri ci an</w>': 1, '   mu l ti mi lli on a i re</w>': 1, '   da y times</w>': 1, '   o our se</w>': 1, '   th rea d b are</w>': 1, '   w pr ry</w>': 1, ' so le</w>': 1, '   sp r in g bo ard</w>': 1, '   w y ck</w>': 1, '   la mp sha de</w>': 1, '   cre st vi e w</w>': 1, '   you l</w>': 1, '   bur me se</w>': 1, '   bar ri ca ding</w>': 1, '   re ga tta</w>': 1, '   sc ha e ter</w>': 1, '   do st oo s v s k y</w>': 1, '   lin d ber gh</w>': 1, '   car el</w>': 1, '   out fo x</w>': 1, '   li t t</w>': 1, '   sa lo me</w>': 5, '   pre vo st</w>': 1, '   nor m and</w>': 1, '   a st ro lo g ers</w>': 2, '   con ju ction</w>': 1, '   per for</w>': 1, '   man ce</w>': 1, '   tra vi at a</w>': 1, '   th use</w>': 1, '   i so tta</w>': 2, ' f ra sch in i</w>': 1, ' f ra sch in is</w>': 1, '   de du ct</w>': 2, '   ni do</w>': 1, '   sa gi tt ar i ans</w>': 1, '   ve i ls</w>': 1, '   imp or</w>': 1, '   busin e ss l</w>': 1, '   gu r g les</w>': 1, '   fa ir ban k ses</w>': 1, '   cha pl ins</w>': 1, '   gi l ber ts</w>': 1, '   v al en t in o s</w>': 1, '   no bo di es</w>': 2, '   c ro a king</w>': 1, ' po sh l</w>': 1, '   wi t n</w>': 1, '   un en du r</w>': 1, '   ri r st</w>': 1, '   g ri r ri th</w>': 1, '   may er ling</w>': 1, '   pi o ture</w>': 1, ' ad ers</w>': 1, '   po st mar ks</w>': 1, '   ri ck e ty</w>': 1, '   ven e ti an</w>': 1, '                                 </w>': 6, '   man i</w>': 1, '   cu ri st</w>': 1, '   h y der ab a d</w>': 1, '   3 4 7 </w>': 2, '   ro man off</w>': 2, '   mo ca mb o</w>': 1, '   z al t ar</w>': 7, '   spi k y</w>': 1, '   an g ri ly</w>': 1, '   en clo ses</w>': 1, '   al u r a</w>': 2, '   li mi ting</w>': 1, '   g li tt er y</w>': 1, '   su per girl</w>': 4, '   \t \t \t \t \t </w>': 1, ' g ran d mother</w>': 3, '   ha ck saw</w>': 4, '   hu ss y</w>': 1, '   un sh ea th ed</w>': 1, '   co ff er</w>': 3, '   se l en a</w>': 14, '   un pl u g ged</w>': 1, '   pa pa y as</w>': 1, '   a ma l ga ma ted</w>': 1, '   pu r r ing</w>': 3, '   pa ssi on f ru it</w>': 1, '   s mo o the e</w>': 1, '   t or na do</w>': 4, '   dan ver s</w>': 7, '   z or</w>': 1, '   m c clo s k ey</w>': 1, '   z on ked</w>': 1, '   pu ck</w>': 1, '   t an ni s ro o t</w>': 1, '   sc or pi o s</w>': 1, '   ba le fi re</w>': 1, '   no wh er e s vi ll e</w>': 1, '   app r en ti ce ship</w>': 1, '   mi su sed</w>': 1, '   y ow ling</w>': 1, '   su per p ow ers</w>': 2, ' d ra w</w>': 1, '   my se</w>': 1, ' mi d</w>': 1, '   sh ou l d n t</w>': 1, '   s ma ll vi ll e</w>': 8, '   ro e bu sh</w>': 2, '   be h in ds</w>': 1, '   any day</w>': 1, '   st e w ed</w>': 1, '   b lo om er</w>': 1, ' cl ar k</w>': 2, '   o a k y</w>': 1, '   ab di ca ted</w>': 1, '   st rea m ers</w>': 1, '   cha ir per son</w>': 1, '   la pe l</w>': 1, '   rea ly</w>': 2, ' r an</w>': 1, '   g lo p</w>': 1, ' cen t ses</w>': 2, '   th or ou gh b red</w>': 1, ' 1 4 3 </w>': 3, '   2 2 5 </w>': 1, '   son o fa gu n</w>': 3, '   mu th a</w>': 2, ' di ll er</w>': 1, ' su n day</w>': 1, ' ba s k et</w>': 1, '   whi te ma il</w>': 1, '   si tt en</w>': 1, '   spi ll in</w>': 2, '   ir re ver si ble</w>': 2, '   pi p in</w>': 1, '   de li ver in</w>': 1, ' du per</w>': 1, '   k r y p ton i te</w>': 6, '   r in k y</w>': 1, '   de ca f fe in a ted</w>': 1, '   a le u ti an</w>': 1, '   gu am</w>': 1, '   bo li vi a</w>': 1, '   g ab on</w>': 1, '   we b c o</w>': 2, '   ra p ers</w>': 2, '   l or e le i</w>': 2, '   f lu st er ed</w>': 2, '   mer cu ro ch ro me</w>': 1, '   fa v or ing</w>': 1, '   1 7 8 </w>': 1, '   cen sure</w>': 1, '   ab sta in ing</w>': 1, ' en er g y</w>': 1, '   un questi on ed</w>': 1, ' ex pen ses</w>': 1, '   the ses</w>': 2, ' dar n</w>': 1, ' ver a</w>': 1, '   f loo ds</w>': 1, ' list en ed</w>': 1, '   ri di c</w>': 2, '   spe c t ac les</w>': 3, ' comp li men t ary</w>': 2, '   cor sa ge</w>': 1, '   mi lli gra ms</w>': 1, '   te sch m ac her</w>': 11, '   co ck er</w>': 1, ' le x</w>': 1, ' cu r l</w>': 1, '   ar te m is</w>': 1, '   pt</w>': 1, '   pre ss room</w>': 1, '   k al</w>': 4, '   ear th ling</w>': 1, '   ex ha u sti ve</w>': 1, ' c ri min al s</w>': 1, '   lu th or</w>': 38, '   bo y sc out</w>': 1, '   bea ch fr on t</w>': 1, '   fu ll ne ss</w>': 1, '   por to</w>': 1, '   ear th lin gs</w>': 1, '   \t \t \t \t \t \t \t \t </w>': 1, ' u l ti ma te</w>': 1, '   \t \t \t \t \t \t \t </w>': 2, ' gre ed</w>': 1, '   z od</w>': 3, ' pro te c ting</w>': 2, '   \t </w>': 1, ' e t ro po l is</w>': 1, ' f an</w>': 1, '   a m i</w>': 2, '   t y co on</w>': 3, '   re gar de z</w>': 1, ' app ro pri ate</w>': 1, '   sh r in er</w>': 1, '   sen sa tion a li s m</w>': 1, '   f rea k o</w>': 2, '   bra w n</w>': 1, '   o st er i z ed</w>': 1, '   re ar m ing</w>': 1, '   de lu si on ary</w>': 1, '   hur l ed</w>': 4, ' 1 0 1 </w>': 5, ' t ar ge ted</w>': 1, '   w er ner</w>': 4, ' i ssi le</w>': 1, '   si lo s</w>': 1, '   no se c one</w>': 1, ' poli sh er</w>': 1, '   bu n k ers</w>': 2, '   hu b b ard</w>': 1, '   sh ow of f s</w>': 1, ' st un ts</w>': 1, '   tra m po l ine</w>': 1, ' gra y</w>': 1, '   clo ber t</w>': 1, '   hea dy</w>': 1, '   ve su vi us</w>': 1, '   wri thing</w>': 1, '   e th i op i an</w>': 1, '   sh men dri ck</w>': 1, '   ra pped</w>': 1, '   mi s sp ent</w>': 1, ' pla y boy</w>': 1, '   me te ori tes</w>': 2, '   de du c tive</w>': 1, '   ne ar ing</w>': 1, '   c ri ers</w>': 1, '   com pla in ers</w>': 1, '   mo an ers</w>': 1, '   g ro an ers</w>': 1, '   e ke</w>': 1, ' bu ster</w>': 2, '   be g in n in gs</w>': 2, ' me mber</w>': 3, '   ir re de e ma ble</w>': 1, '   v on d</w>': 1, ' sta te men ts</w>': 1, '   su n ning</w>': 1, '   b loo d le tting</w>': 2, ' j uni or</w>': 1, '   ho o ts</w>': 1, '   ga p in</w>': 1, '   ph y si o lo gi ca lly</w>': 1, '   den s er</w>': 1, '   ab bo t</w>': 2, '   x en o</w>': 1, '   im per vi ous</w>': 1, '   1 9 5 </w>': 1, '   pe j or a tive</w>': 1, '   la ve</w>': 1, ' f la sh li ght</w>': 1, ' re gre ssed</w>': 1, '   pu r ses</w>': 1, '   e li min a tes</w>': 1, '   po pu l ous</w>': 1, '   st e ph en s</w>': 16, '   k l ar a</w>': 1, '   l ab ou red</w>': 1, '   su n n y ri dge</w>': 1, '   an se l</w>': 11, '   ri s a</w>': 5, '   o tt o s</w>': 10, '   wal k ers</w>': 6, '   bur n ell</w>': 1, ' out gr own</w>': 1, '   di ck er ing</w>': 1, '   gu ar d ra il</w>': 2, '   ser vi c ed</w>': 1, ' att en tive</w>': 1, '   go ers</w>': 1, '   har t le y</w>': 2, '   v a ts</w>': 1, '   la d les</w>': 1, '   sp ra ts</w>': 1, '   cha ts</w>': 1, '   s qu ea king</w>': 2, '   sh ar ps</w>': 1, '   di tty</w>': 2, '   ha me l in</w>': 1, '   we s er</w>': 1, '   spi ed</w>': 1, '   min i mi ze</w>': 1, ' un lu ck y</w>': 1, ' cle ar ly</w>': 1, '   pre s co ts</w>': 1, '   w oo d lot</w>': 1, '   o ver done</w>': 1, '   v ar i an ce</w>': 1, ' cour t</w>': 1, '   la mb st on</w>': 1, '   s n ow ban k</w>': 1, '   sp ack le</w>': 1, '   bi lo dea us</w>': 1, '   at wa t ers</w>': 1, '   char l en e</w>': 2, '   ha mi l t ons</w>': 1, '   h un se ck er</w>': 17, '   mar g in al</w>': 2, ' su i tor</w>': 1, ' si d ne y</w>': 2, '   bar th a</w>': 4, '   en dan g ers</w>': 1, '   cle ve</w>': 4, '   ro b ard</w>': 2, '   e go ti sti c</w>': 1, ' h un se ck er</w>': 1, '   ho ok ey</w>': 1, '   pa tri o ti c s</w>': 1, '   p ra t t</w>': 1, ' de ars</w>': 1, '   c ru de st</w>': 1, '   mi l g ri m</w>': 1, ' ca tting</w>': 1, '   s qu ar er</w>': 1, ' un b al an c ed</w>': 1, ' be g an</w>': 1, '   co lu m ni sts</w>': 2, ' in for med</w>': 1, ' d ow n town</w>': 1, '   b on na</w>': 1, '   s qu oo sh y</w>': 1, ' li ter ary</w>': 1, '   ru mp us</w>': 1, '   el well</w>': 4, ' in te g ri ty</w>': 2, '   ti z zy</w>': 1, ' pre sti ge</w>': 1, '   fin a g le</w>': 1, '   sp a hi sh</w>': 1, ' spi gs</w>': 1, '   ga gs</w>': 1, '   si da le e</w>': 1, ' si sted</w>': 1, '   ge st es</w>': 1, ' an ge l o</w>': 3, ' inter vi e w ed</w>': 1, '   k ow</w>': 1, ' t ow</w>': 1, '   m ea ty</w>': 1, '   su b sc ri b er</w>': 1, '   pa in ed</w>': 1, '   w o l f h ound</w>': 1, ' c rea m ing</w>': 1, '   u ti c a</w>': 1, '   fran n is</w>': 1, '   por ti s an</w>': 1, ' stu tter</w>': 1, '   cra w</w>': 1, '   ro se land</w>': 1, '   pu l ver i z ing</w>': 1, '   ha sen p fe ff er</w>': 1, '   c y ber d y n e</w>': 6, '   je t l ine</w>': 1, '   th i i is</w>': 1, '   ma tri x ing</w>': 1, '   hi er ar chi es</w>': 1, '   d y son</w>': 7, '   ra di ca lly</w>': 1, '   rea c qui re</w>': 2, '   al ri ight</w>': 1, ' ha st a</w>': 2, ' af fir ma tive</w>': 1, '   mi me ti c</w>': 1, '   po l y a ll o y</w>': 1, '   c y ber ne ti c</w>': 2, '   en do s ke le ton</w>': 1, '   pe sc e der o</w>': 1, '   k a la sh ni k o v </w>': 1, '   s k y n et</w>': 8, '   si l ber man</w>': 2, '   i c b ms</w>': 1, '   ge ome tri c</w>': 1, ' a w are</w>': 1, '   mi r co pro ce ss or</w>': 1, '   pre se ts</w>': 1, '   fu r t w</w>': 27, ' n g l er</w>': 27, '   vi o lin i st</w>': 6, '   b la me less</w>': 1, '   p hi l har mon i c</w>': 6, '   to sc an in i</w>': 4, '   e mm i</w>': 13, '   stra u be</w>': 3, ' se mi ti c</w>': 2, '   pro cu red</w>': 1, '   hel mu th</w>': 12, '   h in k el</w>': 10, '   d y m shi t z</w>': 3, '   or che stra ted</w>': 1, '   mor ti ci ans</w>': 1, '   le i p z i g</w>': 1, '   k om man da tur a</w>': 2, '   ban d lea der</w>': 4, '   de gen er a tes</w>': 2, '   sch u ber t</w>': 1, '   un e qu a lled</w>': 1, '   a da gi o</w>': 2, '   b ru ck ner</w>': 3, '   questi on na ir es</w>': 1, '   wi e s ba d en</w>': 1, '   ra ven s b ru ck</w>': 1, '   cre ma t ori a</w>': 1, '   or che stra s</w>': 2, '   han g men</w>': 1, '   st ru tt ed</w>': 1, '   s wa g ger ed</w>': 1, '   sch on ber g</w>': 1, ' je wi sh</w>': 2, ' car ri er</w>': 1, '   go er ing</w>': 8, '   a ge ing</w>': 1, '   supp lan ted</w>': 1, '   k ar a j an</w>': 6, '   st al in gra d</w>': 2, '   nu l</w>': 1, '   c ri ti ci se</w>': 1, '   con sc ri p ted</w>': 2, '   nu ell</w>': 1, '   coun ci ll or</w>': 6, '   hi der</w>': 1, '   h ome land</w>': 1, '   k le m per er</w>': 1, '   sch o en ber g</w>': 1, '   h er man n</w>': 3, '   la men ting</w>': 1, '   mu si k k a mm er</w>': 1, '   hea p ing</w>': 1, '   hon ou rs</w>': 1, '   gu sta v </w>': 1, '   er n st</w>': 1, '   1 8 8 6 </w>': 1, '   or che stra l</w>': 1, '   den a z i fi ca tion</w>': 2, '   ra ved</w>': 1, '   fi l th i er</w>': 1, '   chan ce ll er y</w>': 1, '   cre sc en do</w>': 1, '   as se ss or</w>': 1, '   o bo e</w>': 1, '   wa t t</w>': 1, '   be i ge</w>': 3, '   z er o ed</w>': 2, '   bo g g ling</w>': 1, '   si bi li ti es</w>': 1, ' dis ne y land</w>': 1, ' ma t in e es</w>': 1, '   h ow to</w>': 1, '   un in v ent</w>': 1, ' s k y n et</w>': 1, ' dre ss ing</w>': 1, ' ar ah</w>': 2, ' or g ani ze</w>': 1, '   in te ll i</w>': 1, '   g ence</w>': 1, '   mi c ro secon d</w>': 1, ' ex ter min ation</w>': 1, ' every thin g is</w>': 1, '   rea son ed</w>': 1, '   c y bor gs</w>': 1, '   dn</w>': 1, ' 3 8 4 1 6 </w>': 1, '   ma in f ra mes</w>': 1, ' di sp l ac e ment</w>': 1, '   su m ner</w>': 1, ' nor a d</w>': 1, '   2 0 2 7 </w>': 1, '   re d lan ds</w>': 1, '   si l b er</w>': 1, '   i den t i</w>': 1, '   fi ed</w>': 1, '   v u k o vi ch</w>': 1, '   we st b ound</w>': 1, ' st ro king</w>': 1, '   mo pped</w>': 1, '   i tive</w>': 1, '   ba er</w>': 6, '   su ther land</w>': 1, '   comp en sa tions</w>': 1, ' g ran t</w>': 2, ' st e pp ing</w>': 1, ' pen ni es</w>': 1, ' wh er ea s</w>': 1, '   k u do s</w>': 1, '   ort on</w>': 8, '   ju lli ard</w>': 1, '   s qui b s</w>': 1, '   fe in go ld</w>': 2, '   a ll ge me ine</w>': 1, '   ca mp ton</w>': 2, '   war n s</w>': 1, ' p in at a</w>': 1, '   me di ction</w>': 1, ' con su m er</w>': 1, '   re c r rea tion</w>': 1, '   la y o ver s</w>': 1, '   re de cor a ted</w>': 1, '   pe di a tri ci an</w>': 2, '   g y na e co lo gi st</w>': 1, ' me th</w>': 1, '   fe ti shi st</w>': 2, ' d is</w>': 1, ' sa ti s fi ed</w>': 1, '   cu r sor y</w>': 2, ' cou gh</w>': 1, ' te sts</w>': 1, '   ho y</w>': 1, '   tu x e do s</w>': 1, '   lu d well</w>': 1, '   li mer i ck</w>': 1, '   h in ch ber ger</w>': 1, '   fi t z wi lli am</w>': 1, ' fin est</w>': 1, ' hea l th</w>': 1, '   vi c t ori an</w>': 3, '   au bo ch on</w>': 1, '   mo de lling</w>': 2, '   in du st ri a li st</w>': 1, '   cra in</w>': 10, '   wor ri er</w>': 1, '   to ss er</w>': 1, ' tur ner</w>': 1, '   du d le ys</w>': 1, '   ad da m</w>': 1, '   la mb re tta</w>': 2, '   tal k y</w>': 1, '   vi o lin i sts</w>': 1, '   in su la tion</w>': 1, '   o h ms</w>': 1, '   hu mm ed</w>': 2, '   l ab or ers</w>': 1, '   a po lo ge ti c</w>': 2, '   ha w s er</w>': 1, '   si de bo ard</w>': 1, '   win d st or m</w>': 1, '   pe di cu re</w>': 2, '   da vi i i i id</w>': 1, '   bu o ys</w>': 1, '   re ti cu lu m</w>': 1, '   ad har a</w>': 1, '   ni tty</w>': 1, '   shi t ki ck ers</w>': 1, ' g an g land</w>': 1, ' car ry</w>': 3, '   c q c</w>': 1, '   wa t for d</w>': 1, '   bo o z er</w>': 1, ' v a le ts</w>': 1, '   d on ca ster</w>': 1, '   wal li es</w>': 1, '   p on c ed</w>': 1, '   min d ers</w>': 1, '   s ki ved</w>': 1, '   di sp en sing</w>': 1, '   bu n g</w>': 1, '   cor cor an</w>': 4, '   ab ou ts</w>': 1, '   ra ma</w>': 3, '   ca ll b ac ks</w>': 1, ' wai ver</w>': 1, ' co ll ea gu es</w>': 1, '   sc ar per ed</w>': 1, '   bu s man</w>': 1, '   t wi g ged</w>': 1, '   i ago</w>': 1, '   in cre men ts</w>': 1, ' fr on ts</w>': 1, '   o de on</w>': 1, '   mu s well</w>': 1, ' du st ers</w>': 1, ' u r ri can es</w>': 1, ' ar d ly</w>': 2, ' app en</w>': 1, '   je we ll er</w>': 1, '   t un ne lled</w>': 1, '   z a g at</w>': 1, '   ma i d st one</w>': 1, '   bri x ton</w>': 1, '   t wi d d l in</w>': 1, '   lin d gr en</w>': 5, '   lon g stan ding</w>': 1, '   par k hur st</w>': 1, '   mu ck ers</w>': 1, '   f oo t loo se</w>': 1, '   do d dle</w>': 1, '   hon ing</w>': 2, '   e pi so di c</w>': 1, '   e ff ace</w>': 1, '   chi su m</w>': 1, '   s ar k y</w>': 1, '   we mb le y</w>': 1, '   sta du i m</w>': 1, '   ma il ba gs</w>': 2, '   com men su ra te</w>': 1, '   s k int</w>': 1, '   a ll o tt ed</w>': 1, '   p ra m</w>': 1, ' la tely</w>': 1, ' sta te ment</w>': 1, '   e m i</w>': 2, '   lu mb er ed</w>': 1, ' a very</w>': 1, '   j i gs</w>': 1, ' pla u si ble</w>': 1, '   a ir b ru sh ed</w>': 1, '   e s se l in</w>': 1, '   chi pp i es</w>': 1, '   bi r dy</w>': 8, '   en th u si a st</w>': 1, '   car can o gu es</w>': 1, '   sch er z o</w>': 1, ' p ing</w>': 1, ' pla ys</w>': 1, '   z t in ks</w>': 2, '   fa h z er</w>': 1, '   cre i gh ton</w>': 1, '   e sc ri ture</w>': 1, '   to lli ver</w>': 1, ' u su ally</w>': 1, '   re vo lu tion i ze</w>': 1, '   ha ir pi e ce</w>': 1, '   ca pp er</w>': 1, '   sh r in k age</w>': 1, '   ni r d lin ger</w>': 3, '   ca pi ta li z ation</w>': 1, '   re ta i ling</w>': 1, '   gar son</w>': 1, '   ha ber da sh er y</w>': 1, ' ca su al</w>': 1, '   comp t ro ll er</w>': 2, '   bar ber sho p</w>': 4, ' pro hi bi ted</w>': 1, ' d or is</w>': 2, '   ri e den sch ne i der</w>': 7, '   ir re gu l ar i ti es</w>': 1, ' gra tu la tions</w>': 1, '   gar r ow ay</w>': 3, '   in f la me</w>': 1, '   ma st er mind</w>': 1, '   vi c ti mi ze</w>': 1, '   bo at y ard</w>': 1, ' fe llow</w>': 1, '   dis gr un t l ed</w>': 1, ' ju g g ling</w>': 1, '   under ling</w>': 1, '   tri mm er</w>': 1, '   tu ran do t</w>': 2, '   me t ro po le</w>': 1, ' ob je ct</w>': 1, ' pro bi ty</w>': 1, '   pro bi ty</w>': 1, '   i ff y</w>': 1, '   a bu n da s</w>': 1, '   son a ta s</w>': 1, '   wh u d d ya</w>': 1, '   ma ti ck less</w>': 1, '   he in i e</w>': 1, '   m ac a da m</w>': 1, '   di e dri ck son</w>': 1, '   se cu r ing</w>': 2, '   de ce dent</w>': 1, '   pen der ga st</w>': 19, '   cu th ber t</w>': 10, '   beau re g ard</w>': 8, '   k a wa ki ta</w>': 3, '   pi ran e s i</w>': 1, ' a go st a</w>': 3, '   i pp o li to</w>': 3, '   re d und an ci es</w>': 1, '   v as es</w>': 1, '   w o ve</w>': 1, '   ex tr ac ted</w>': 3, ' brea sted</w>': 1, '   mer g ans er</w>': 1, '   mb w u n</w>': 9, '   whi tt le sle y</w>': 12, '   tra m pl ing</w>': 1, '   th a la m us</w>': 3, '   g ri z z ly</w>': 1, '   pre ter na tu ra l</w>': 1, '   lo com o tor</w>': 1, '   chi as m</w>': 1, ' ex ter na l</w>': 1, '   mu co id</w>': 1, ' mor p ho lo gi cal</w>': 1, '   a g gre ssi ven ess</w>': 1, '   qu ad ru pe da l</w>': 1, '   gen us</w>': 1, '   ph y lu m</w>': 1, '   k g</w>': 1, ' qu ad ru pe da l</w>': 1, ' g l y co te tra g l y c ine</w>': 1, '   co ll a gen o id</w>': 1, '   we in st e in</w>': 2, '   mon o x y to c in</w>': 1, '   su pre ss in</w>': 1, '   bur ls</w>': 1, '   a mb y lo id</w>': 1, '   re o vi ru s</w>': 1, '   te pu i</w>': 6, '   k o th o g a</w>': 2, '   f l ou ri sh ed</w>': 2, '   un im pre ssi ve</w>': 1, '   in fu sion</w>': 1, '   au c tion ing</w>': 1, '   be ch u an al and</w>': 1, '   ph ar m ac o lo g y</w>': 2, '   pro min en tly</w>': 1, '   inter me di a tes</w>': 1, '   sha man s</w>': 1, '   t ac t fu lly</w>': 2, '   sta ir ca ses</w>': 1, '   att ri bu tes</w>': 2, '   bi pe da li a</w>': 1, '   in sig ni a</w>': 1, '   mor i ar ty</w>': 1, '   ma m ma ls</w>': 1, '   1 0 1 2 </w>': 2, '   ca ll it</w>': 1, '   s nu ff l ed</w>': 1, '   cu r v y</w>': 1, '   ex tr ac ts</w>': 1, ' mon o x y to c in</w>': 1, '   ma m ma li an</w>': 1, '   ti din gs</w>': 1, '   ex tra po la tor</w>': 2, '   fr ac tal</w>': 1, '   de ce p ti ve ly</w>': 1, '   p t ar mi g an</w>': 1, '   fe i g ning</w>': 1, '   un cont ro ll ably</w>': 1, '   ca li s to</w>': 1, '   hi gh s</w>': 1, '   l ows</w>': 1, '   be le m</w>': 1, '   vi t i</w>': 1, '   l ev u</w>': 1, ' fi j i</w>': 1, ' wa g ner</w>': 1, '   ti d ying</w>': 1, '   clo ck ca m</w>': 1, '   ha gu e</w>': 2, '   chri st of</w>': 2, '   ca me o</w>': 1, '   s ea haven</w>': 4, ' tru man</w>': 4, '   gen er a tes</w>': 2, ' cra dle</w>': 1, '   vi de o gra p her</w>': 1, '   al ter na ti ve ly</w>': 1, '   re le ga te</w>': 1, '   e m mi es</w>': 1, '   con te mp la tion</w>': 1, '   vo ye u rs</w>': 1, ' s ea haven</w>': 1, '   k a i s er</w>': 1, ' mi x ed</w>': 1, '   ha lo g en</w>': 1, ' bur ban k</w>': 1, ' sp are</w>': 1, '   y u b a</w>': 2, '   co co as</w>': 1, '   mo co co a</w>': 1, '   s wee ten ers</w>': 1, '   ban k c ard</w>': 1, ' h y po c ri te</w>': 1, '   mar d i</w>': 1, '   bar be qu e</w>': 1, '   ar ra h</w>': 1, ' che f</w>': 1, '   di c er</w>': 1, '   p ee l er</w>': 1, '   un con qu er ed</w>': 1, '   as c ent</w>': 2, '   cra mp ons</w>': 1, ' ri s ks</w>': 1, '   ad ju ster</w>': 1, ' sp o ok y</w>': 1, '   mu l der</w>': 41, '   po op y</w>': 1, '   b y ers</w>': 1, '   c ran i o tom y</w>': 1, '   su b du ra l</w>': 2, '   per for ation</w>': 1, '   g lan c ed</w>': 1, '   att ri bu ta ble</w>': 1, '   a ll e g es</w>': 1, '   ru bri c</w>': 1, '   han ta</w>': 3, '   si z ea ble</w>': 1, '   ge sta tes</w>': 1, '   co in ci dent</w>': 1, '   k u r t z we il</w>': 10, '   st ru gh old</w>': 1, '   t un is</w>': 1, '   fe ma</w>': 6, '   tr ans gen i c</w>': 1, '   ex ca v a ted</w>': 1, '   un an ti ci pa ted</w>': 1, '   ti me ta ble</w>': 2, '   in di sc ri min ate</w>': 1, '   ven a li ty</w>': 1, '   bi go ted</w>': 1, '   te m pl ar</w>': 1, '   bi l der bur g</w>': 1, '   on e world</w>': 1, '   mi cha u d</w>': 4, '   dr in k y</w>': 1, '   th ready</w>': 1, '   lan c in a ting</w>': 1, '   ra tion a li s m</w>': 1, '   de bu n k</w>': 1, '   de ba ted</w>': 1, '   ar cha e o lo gi sts</w>': 1, '   im mu no hi sto che mi cal</w>': 1, '   ca u sa tive</w>': 1, '   mi c ro be</w>': 1, '   di g si te</w>': 1, '   ca te g ori z ed</w>': 2, '   e de ma t ous</w>': 1, '   op r</w>': 2, '   reme di ation</w>': 1, '   m ca d die</w>': 3, '   p st n</w>': 1, '   re si du es</w>': 1, '   as si du ou s ly</w>': 1, '   v ac c in a ted</w>': 1, '   h y bri ds</w>': 1, '   ge sta te</w>': 1, '   co op er a ti ve ly</w>': 1, '   f ac i li ta ting</w>': 1, '   se l fi sh ne ss</w>': 1, '   pa th o g en</w>': 1, '   re con sti tu ted</w>': 1, '   co lon i ze</w>': 1, '   e bo l a</w>': 1, '   mu ta tes</w>': 3, '   to il er</w>': 1, '   f ac i li ta te</w>': 1, '   di ge sti ves</w>': 2, '   re po pu la tion</w>': 1, ' ev al u a tions</w>': 1, '   rea s se ss</w>': 1, '   re fi lled</w>': 1, '   z y pre x a</w>': 1, '   a ta v an</w>': 1, '   no v a k</w>': 3, '   con v in c in g ly</w>': 2, '   lu ci en</w>': 1, '   mo ck y</w>': 5, ' mo ck y</w>': 2, '   bo ge y man</w>': 2, '   star gh er</w>': 14, '   hi ck son</w>': 2, '   gi sh</w>': 1, ' re mor se</w>': 1, ' c ru el</w>': 1, '   i dea li z ed</w>': 1, ' tr ou bl ed</w>': 1, '   wh al en</w>': 1, '   de lan o</w>': 2, '   la t ti m er</w>': 1, '   per ver se ly</w>': 1, ' re ce i ver s</w>': 1, '   pre v ac id</w>': 1, '   mi ran di ze</w>': 1, '   wai ve</w>': 1, '   le ga li ti es</w>': 1, '   vi ck se y</w>': 1, '   la be t z k i</w>': 1, '   war s che in li ch</w>': 1, '   wor r en</w>': 1, '   no oh</w>': 1, '   z ei ten</w>': 1, '   hi er</w>': 2, '   s ind</w>': 1, '   f ru her</w>': 1, '   fu r st en</w>': 1, '   un t</w>': 1, '   e in ge g an ger</w>': 1, '   so g ar</w>': 1, '   ver ke h r t</w>': 1, '   sa y en</w>': 1, '   le u ten</w>': 2, '   au ch</w>': 2, '   wi ss en</w>': 4, '   mu ss en</w>': 2, '   do ch</w>': 3, '   me ine</w>': 2, '   o h n e</w>': 1, '   gr un d</w>': 1, '   wi r d</w>': 1, '   poli t z e i</w>': 2, '   k om me</w>': 1, '   s chan de</w>': 1, ' qu ar re l</w>': 1, '   ca ll ow ay</w>': 15, ' e in</w>': 1, '   hu ri g an</w>': 1, '   ge st er n</w>': 1, '   he u ri g en</w>': 1, '   win k el</w>': 5, '   en t sch u l di g en</w>': 1, '   ni ch ts</w>': 1, '   dan ke</w>': 1, '   ke in</w>': 2, '   har b in</w>': 5, '   au st ri ans</w>': 2, ' au st ri an</w>': 1, ' ok la h om a</w>': 1, '   tur n in gs</w>': 1, '   ca ll a g ha n</w>': 4, '   r ac ke te er</w>': 2, ' ex pen se</w>': 1, ' poli ce men</w>': 1, ' n ch</w>': 1, '   sch mo l k a</w>': 1, '   k r on ers</w>': 1, '   s ac her</w>': 1, '   z an e</w>': 1, '   c r ab b in</w>': 2, '   st ri p t ea se</w>': 1, '   go a d</w>': 1, '   po pe sc u</w>': 2, '   r ou man i an</w>': 3, '   di sc re di ta ble</w>': 1, '   z we i</w>': 2, '   z war t ze</w>': 1, '   v in k el</w>': 2, ' co ll e ction</w>': 1, '   da v on</w>': 1, '   w en n</w>': 1, '   f re und li ch</w>': 1, '   au s lan der n</w>': 1, '   mi r</w>': 2, '   ge h en</w>': 1, '   z u r</w>': 1, '   g an sa lt</w>': 1, '   ab er</w>': 1, '   mi ch</w>': 2, '   la ss en</w>': 1, '   m ac h t</w>': 1, '   no ch</w>': 2, '   de pp er t</w>': 1, '   b lo d su m</w>': 1, '   s an g en</w>': 1, '   g le i ch</w>': 3, '   bra u ch en</w>': 1, '   an g st</w>': 1, ' w art</w>': 1, ' war te in</w>': 1, '   ca s</w>': 2, '   so ll</w>': 1, '   se h en</w>': 1, '   un ter</w>': 1, '   un ten</w>': 1, '   pa ssi er t</w>': 2, '   k an n</w>': 1, '   a ll e</w>': 1, '   k en n en</w>': 1, ' po pe sc u</w>': 1, '   w un sch en</w>': 1, '   sti ff ga s se</w>': 1, '   pa t p ong</w>': 1, '   e ti en n e</w>': 2, '   ru ck s ac ks</w>': 1, '   fran co i se</w>': 1, '   b on so ir</w>': 2, '   mor i ah</w>': 1, '   e sa u</w>': 1, '   mi d ra sh</w>': 2, '   sha lo m</w>': 6, '   spi e ge l</w>': 1, '   ha sh em</w>': 1, '   t or ah</w>': 14, '   re b be</w>': 1, '   z i on i sts</w>': 2, '   je w ry</w>': 1, '   sa br a</w>': 1, '   sha t ti l a</w>': 1, '   mi li t ar i sti c</w>': 1, '   bu ll y bo ys</w>': 1, '   z i on i st</w>': 2, '   te fi ll in</w>': 1, '   t si t s is</w>': 1, '   sho t ne ss</w>': 1, '   k ad di sh</w>': 4, '   ki d du sh</w>': 3, '   ei ch man n</w>': 5, '   mi sh na h</w>': 1, '   my sti c s</w>': 1, '   fa u ri s son</w>': 1, ' me in</w>': 1, ' ex e mp t</w>': 1, '   ki pp u r</w>': 2, '   ma tt er ing</w>': 1, ' ju da i s m</w>': 1, ' a si te m</w>': 2, '   le ch em</w>': 1, '   pe s se l</w>': 1, ' mon at</w>': 1, '   pa y n</w>': 1, '   ta sh</w>': 1, ' chi ton</w>': 1, ' as hi y te m</w>': 1, '   v u v </w>': 1, ' z a y in</w>': 1, ' ch et</w>': 1, ' t et</w>': 1, '   ch u ma sh</w>': 1, ' gi me l</w>': 1, ' da li d</w>': 1, '   v ow el</w>': 1, '   v ow el s</w>': 1, '   a le p h</w>': 1, '   gen ti les</w>': 1, '   gen ti le</w>': 3, '   a v ra m</w>': 2, ' y om er</w>': 2, '   ad on a i</w>': 3, '   le ch a</w>': 1, '   may ar t z ch a</w>': 1, ' mi mo h la d</w>': 1, ' tch a</w>': 1, ' ba y t</w>': 1, '   a v a y ch a</w>': 1, ' er te z</w>': 1, '   a sh er</w>': 1, '   ar e ch a</w>': 1, '   mo l der ing</w>': 1, ' un d</w>': 1, '   wor te</w>': 1, '   z er fi el en</w>': 1, '   m un de</w>': 1, '   mo dri ge</w>': 1, '   pi l ze</w>': 1, '   vi ri li o</w>': 1, '   to y n be e</w>': 1, '   b al int</w>': 5, ' m ee ting</w>': 1, '   sh e ma</w>': 1, '   y i s ra el</w>': 1, '   e lo h en u</w>': 1, '   e ch od</w>': 1, '   i li o</w>': 5, '   man z e tt i</w>': 12, '   se mi ti s m</w>': 1, '   s wa sti k as</w>': 1, ' man z e tt i</w>': 1, ' so ci o bi o lo g y</w>': 1, ' z i on i s m</w>': 1, ' m uni t ar i an</w>': 1, '   co ck bur n</w>': 1, '   c r ou ch</w>': 1, '   sha ha ck</w>': 1, '   mo e bi us</w>': 6, '   stra u ss</w>': 1, ' in su l ting</w>': 1, ' ca li b er</w>': 1, '   e mi gra ted</w>': 1, ' to l er an ce</w>': 1, '   de cen tra li z ed</w>': 1, '   n on vi o l ent</w>': 1, '   an ti ab or tion</w>': 1, ' im mi gra tion</w>': 1, '   re si st ers</w>': 1, '   li ber t ar i ans</w>': 2, '   st r en g th en ed</w>': 1, '   na de l man</w>': 2, '   mi t z v a h ed</w>': 2, '   e lo him</w>': 1, '   je wi sh ne ss</w>': 1, '   d om in a tes</w>': 1, ' uni ver sa li z es</w>': 1, '   co s mo poli t an</w>': 4, '   e mer ged</w>': 1, '   gh e tt o s</w>': 1, '   der ac in ate</w>': 2, '   go e the</w>': 1, '   i bo s</w>': 1, '   ban tu s</w>': 1, '   man din go s</w>': 1, '   mu l ti cu l tu ra l</w>': 1, '   e ga li t ar i an</w>': 1, '   r ac i a li st</w>': 1, '   z a mp f</w>': 2, '   v h at</w>': 1, '   v u n</w>': 2, '   ab o ve gr ound</w>': 2, ' for m er</w>': 1, '   sch war z chi ld</w>': 1, '   mar g in a li ze</w>': 1, '   r h in el and</w>': 1, '   see the</w>': 1, '   bl v d</w>': 1, '   dan i el s en</w>': 1, '   da ven</w>': 1, '   inter ning</w>': 1, '   j ts</w>': 2, '   r ab b in i c</w>': 1, '   shu l</w>': 1, '   d or f man n</w>': 1, '   coun ten an ce</w>': 1, '   vi ru l ent</w>': 1, '   pro p on en ts</w>': 1, ' ran king</w>': 2, '   jo han a</w>': 13, '   b on i ta</w>': 3, '   mi t z vo h ed</w>': 1, '   car do z a</w>': 1, ' f ru tt i</w>': 1, '   st re tch ers</w>': 1, '   ta or min a</w>': 2, '   mo lin ar i</w>': 2, '   may o l</w>': 8, '   ro s ary</w>': 1, '   s an t in i</w>': 1, '   por th o le</w>': 2, '   no i re u ter</w>': 1, '   den te</w>': 1, '   mer ma i ds</w>': 2, '   a hi</w>': 2, '   ma mi a</w>': 1, '   stu pi do</w>': 1, '   com pe ti tions</w>': 1, '   1 2 7 </w>': 1, '   d om en i c o</w>': 1, '   a mor go s</w>': 1, '   h en ri e tta</w>': 1, '   ber li o z</w>': 1, '   fr en ch y</w>': 1, '   b lo om in g da les</w>': 1, '   per u vi an</w>': 1, '   bur g l ed</w>': 1, '   3 3 0</w>': 1, ' o x y g en</w>': 1, '   f lu or o s co pe</w>': 2, '   cre v as se</w>': 1, '   ra him</w>': 1, '   mor de cha i</w>': 14, '   h om en ta sh en</w>': 1, '   y en t l</w>': 1, '   po to k</w>': 1, '   pro ta gon i st</w>': 1, '   cha l ant</w>': 1, '   j ee ps</w>': 1, '   j d l</w>': 3, '   han u k k ah</w>': 13, '   sh man u k k ah</w>': 1, '   pu ri m</w>': 1, '   wh or ing</w>': 1, '   se mi ti c</w>': 2, '   as ser tive</w>': 1, '   ba t ya</w>': 1, '   chi c ke</w>': 1, '   la t ke</w>': 1, '   v ey</w>': 1, '   da mi an</w>': 1, '   sh to op s</w>': 1, '   b lo om en ber gen st e in en th al</w>': 2, '   sh to op ing</w>': 1, '   ma z el</w>': 3, '   to v </w>': 3, '   bo o je e</w>': 3, '   la t kes</w>': 1, '   dre i de ls</w>': 1, '   men or ah</w>': 1, '   kn ow l</w>': 1, '   sha bo s</w>': 1, '   tu ch us</w>': 1, '   ra ton</w>': 1, '   sch le p ing</w>': 1, '   sha b at</w>': 2, '   man i s che wi t z</w>': 2, '   ti k v a</w>': 1, '   of f sc re en</w>': 1, '   de te c ts</w>': 1, '   x p</w>': 1, '   com pu ta t ors</w>': 1, '   vi lli an</w>': 1, '   bi o gra p hi cal</w>': 1, '   ke y st ro kes</w>': 1, '   a ve en o oh</w>': 1, '   a le ch em</w>': 1, '   k wan z a a</w>': 1, '   mor de</w>': 1, ' cha i</w>': 1, '   k lf</w>': 1, '   sh le pped</w>': 1, '   no sh</w>': 1, '   un plea s an t ne ss</w>': 1, '   sh pi el</w>': 1, '   go y</w>': 1, '   k c</w>': 1, '   chi lling</w>': 1, '   d om in gu e z</w>': 1, '   cl o</w>': 7, '   con tr er as</w>': 1, '   g al bra i th</w>': 13, '   le op ar ds</w>': 4, '   ma m ac i ta</w>': 1, '   bl ack c ard</w>': 1, '   ro si ta</w>': 1, ' fi ts</w>': 1, '   con qui sta d or es</w>': 1, ' man g l ed</w>': 1, '   k ink</w>': 1, ' z a z a</w>': 1, '   b ack fi re</w>': 1, '   na tu ra li st</w>': 2, '   z oo lo g y</w>': 1, '   ki k i</w>': 11, '   cou g ars</w>': 1, '   ma u ls</w>': 1, '   figu ra ti ve ly</w>': 1, '   ten der f oo t</w>': 1, '   e lo i se</w>': 2, '   be l mon te</w>': 1, '   pro ce ssi on i st</w>': 1, '   hu d d ling</w>': 1, ' so a p</w>': 1, ' al ar m</w>': 1, ' 9 3 9 </w>': 1, '   ga w ks</w>': 2, '   t wi r l er</w>': 2, '   mor a es</w>': 31, '   spi t z</w>': 14, '   ma j or d om o</w>': 1, '   f l un k ey</w>': 2, '   sa o</w>': 1, '   pa u l o</w>': 1, '   pu r cha s er</w>': 1, '   u g lin ess</w>': 2, '   lo mb ard</w>': 46, '   comp li ca ting</w>': 1, '   re ver ted</w>': 1, '   ex ten u a ting</w>': 1, '   mi s be ha ved</w>': 1, '   tra f fi ck ers</w>': 3, '   du p ing</w>': 1, '   la mon t</w>': 8, '   man h un ter</w>': 1, '   e pp ing</w>': 1, ' j un ki e</w>': 1, '   b lu st er y</w>': 1, ' du lling</w>': 1, '   sur pl us</w>': 2, '   f ru i t fu lly</w>': 1, '   se per ate</w>': 1, '   de l ving</w>': 1, '   mon dri an</w>': 4, ' 7 5 4 8 </w>': 1, '   5 0 4 </w>': 2, '   2 6 6 </w>': 1, '   7 5 4 8 </w>': 1, '   min i mi z es</w>': 1, '   spe ci a li sing</w>': 1, '   ex t or tion er</w>': 1, '   po ses</w>': 3, '   pro cu r ers</w>': 2, '   un wi se ly</w>': 1, '   pa e do p hi le</w>': 1, '   re ta i ls</w>': 1, '   bra ved</w>': 1, '   pen r h y n de u d ra e th</w>': 1, '   7 7 0</w>': 3, '   4 7 1 </w>': 3, '   0 1 7 6 6 </w>': 2, '   cre sc en ts</w>': 2, '   c ro i ss an ts</w>': 4, '   tra f fi ck er</w>': 2, '   no m</w>': 1, '   3 9 5 </w>': 1, '   na th a lie</w>': 5, '   fa e ces</w>': 1, '   mor ea u</w>': 2, '   ma l t rea t ment</w>': 1, '   pro vi sion</w>': 1, '   wi l s ons</w>': 1, '   fin s bu ry</w>': 2, '   je e</w>': 1, '   g lu ck</w>': 5, '   h y at t</w>': 4, '   sa vi e er</w>': 1, '   di tes</w>': 1, '   p our ri e z</w>': 1, '   a ve z</w>': 1, '   re com m and</w>': 1, ' mes</w>': 1, ' qu el q u</w>': 1, ' ce m ment</w>': 1, '   re mer ci e</w>': 1, '   f an c ying</w>': 1, '   co ll e c tion oo se</w>': 2, '   re ck ons</w>': 1, '   un di vi ded</w>': 1, ' spi ri te d ne ss</w>': 1, ' a is</w>': 1, '   r hi an</w>': 7, '   re ta il ed</w>': 1, '   re par ation</w>': 1, '   f oo t pa th</w>': 1, '   bar st ow</w>': 1, '   con st ru ed</w>': 2, '   un app re ci a tive</w>': 1, '   ci vi li sa tion</w>': 1, '   wor d ly</w>': 1, ' ac h at</w>': 1, '   qu at re</w>': 1, '   c in q </w>': 2, '   p our ra it</w>': 1, '   v ra i ment</w>': 1, '   tru c</w>': 1, ' e m ba ll age</w>': 1, '   qu o i</w>': 1, '   as se z</w>': 1, '   f ou tu e</w>': 1, ' ta it</w>': 1, '   sur ve ys</w>': 1, '   tr ou ver</w>': 1, '   pe u r</w>': 1, '   fa u dr a</w>': 1, '   att en d re</w>': 1, '   a v ant</w>': 1, '   com bi en</w>': 1, '   au tri chi en</w>': 1, '   ne go ci ant</w>': 1, '   pu be sc en ts</w>': 1, ' ce</w>': 1, '   fi g li o</w>': 1, '   pu tt an a</w>': 1, '   mo s sa d</w>': 3, '   spi t z es</w>': 2, '   sha dy</w>': 1, '   sa lu t</w>': 1, '   ou a is</w>': 1, '   co st li est</w>': 2, '   t re mo i ll e</w>': 5, '   re g na u lt</w>': 2, '   y o lan de</w>': 1, '   com pi e g n e</w>': 6, '   gi ll es</w>': 1, '   au l on</w>': 2, ' au l on</w>': 2, '   pu ri f y</w>': 1, '   b ru sh w o od</w>': 1, '   i lli ter ate</w>': 1, ' sur r en der</w>': 1, '   di sp lea sing</w>': 2, '   clo v is</w>': 4, '   r he i ms</w>': 2, '   an o in ting</w>': 1, '   ca u ch on</w>': 2, '   r ou en</w>': 2, '   e c cle si a sti cal</w>': 1, '   pre ven ts</w>': 2, '   pre la tes</w>': 1, '   ear ne st ly</w>': 1, '   ad mon i tion</w>': 1, '   d om re my</w>': 2, '   beau v a is</w>': 1, '   di o ce se</w>': 1, '   bur g un di an</w>': 2, ' pro pi ti ous</w>': 1, '   a st ro lo gi cal</w>': 1, '   char la t ans</w>': 1, '   ba tt er ing</w>': 2, ' ra ms</w>': 2, '   t ou re ll es</w>': 6, ' b ows</w>': 1, '   c ro ss b ows</w>': 1, '   cu l ver ins</w>': 1, '   du no is</w>': 7, '   u sur p ed</w>': 1, '   re c ro ss</w>': 1, '   al en c on</w>': 1, ' c ry</w>': 1, '   x a in tra i ll es</w>': 2, '   a g in cour t</w>': 2, '   beau re vo ir</w>': 1, '   st ee ds</w>': 1, '   ha ck ne ys</w>': 1, ' st e ed</w>': 1, '   w la de k</w>': 18, '   d or o ta</w>': 4, '   lu c z a k</w>': 1, '   w la d y s la w</w>': 2, '   s z pi l man</w>': 8, '   mar e k</w>': 1, '   ge b c z y n s k i</w>': 1, '   y u re k</w>': 1, '   mi cha l</w>': 1, '   d z i ki e wi c z</w>': 2, '   ju re k</w>': 7, '   j an in a</w>': 2, '   an dr z e j </w>': 2, '   ha lin a</w>': 1, '   h en r y k</w>': 5, '   y i t z cha k</w>': 1, '   su ffer in gs</w>': 1, '   ra s z e j a</w>': 1, '   z lo ty</w>': 1, '   car t lo a ds</w>': 1, '   z lo t ys</w>': 2, '   do sto ev s k y</w>': 1, '   ma j or e k</w>': 3, '   re se ttle</w>': 1, '   un de si r ab les</w>': 1, '   je hu da</w>': 2, '   k har k ho v </w>': 1, '   pu pp y do g</w>': 1, '   z e la z na</w>': 1, '   bra ma</w>': 1, '   god le w s k a</w>': 1, '   bo gu ck i</w>': 1, '   re se tt le ment</w>': 1, '   z y g m un t</w>': 1, '   so k o l ow</w>': 1, '   ra il wa y man</w>': 1, '   t re bl in k a</w>': 3, '   for ked</w>': 1, '   j il ted</w>': 1, '   ja un di ce</w>': 2, '   ban es</w>': 1, '   ne ssi e</w>': 2, '   ad a</w>': 7, '   en th u si a sti ca lly</w>': 1, '   lu x e m bur g</w>': 2, '   pe in i</w>': 5, '   ha ere</w>': 1, '   a tu </w>': 1, '   r n an a</w>': 1, '   ju n</w>': 1, '   k a w n ar na</w>': 1, '   par a i ke te</w>': 1, '   ta hi</w>': 1, '   ha w lie</w>': 1, '   wh en na</w>': 1, '   u c i</w>': 1, '   un su b ti t l ed</w>': 1, '   f la ti sh</w>': 1, '   ta p u</w>': 1, '   st un ted</w>': 1, '   f l or a</w>': 2, '   pr ou d ly</w>': 1, '   jo y hou ses</w>': 1, '   la m st ers</w>': 1, '   bu n c o</w>': 4, '   fa s an e ll a</w>': 2, '   lon g sho ts</w>': 1, '   gon d or ff</w>': 14, '   st en ner</w>': 1, '   p al tr ow</w>': 1, '   fu re y</w>': 1, '   fi s kin</w>': 1, '   li me house</w>': 1, '   du k y</w>': 1, '   in si de man</w>': 1, '   ha l f si es</w>': 1, '   e mb ar ra ss in</w>': 1, '   sp ra y in</w>': 1, '   sa lin o</w>': 3, '   ev an st on</w>': 1, '   ja y ce es</w>': 1, '   com b s</w>': 2, '   lon ne g an</w>': 17, '   com mi tt in</w>': 1, '   ei ri e</w>': 2, '   g ri f t in</w>': 3, '   gi an e ll i</w>': 1, '   car ne llo</w>': 1, '   mo tt o l a</w>': 2, '   an en ber g</w>': 1, '   bl ack bo ar ds</w>': 1, '   ti ck er</w>': 3, '   p ac kin</w>': 4, '   ta il in</w>': 1, '   c ro a ked</w>': 1, '   la mm ed</w>': 1, '   chan to o z i e</w>': 1, '   sy ph on</w>': 4, '   hi al eah</w>': 1, '   ca den z a</w>': 1, '   du c er</w>': 1, '   hon ni g an</w>': 1, '   lon ne man</w>': 1, '   ma x i es</w>': 1, '   sh op li f ter</w>': 1, '   ra g g le</w>': 1, '   sor e head</w>': 1, '   mi g th</w>': 1, '   6 6 0</w>': 1, '   li qui da ting</w>': 3, '   ti pp in</w>': 1, '   ch u mm in</w>': 1, '   ro se b re en</w>': 3, '   w y n ant</w>': 48, '   se x o ger ar i an</w>': 1, '   se x o gen ar i an</w>': 1, '   we l</w>': 2, '   nor man die</w>': 1, '   m ac au la y</w>': 14, '   st r en g then</w>': 1, '   c in ch es</w>': 1, '   n un he i m</w>': 7, '   mor e ll i</w>': 5, ' s n ea k</w>': 1, '   gra y i sh</w>': 1, '   re d di sh</w>': 1, ' ha tting</w>': 1, '   a st a</w>': 11, ' mer ry</w>': 1, '   a ll en town</w>': 1, '   gu age</w>': 1, ' sle u thing</w>': 1, '   g y pp ing</w>': 1, '   stu d sy</w>': 2, ' stu d sy</w>': 1, '   sh in</w>': 1, ' f oo ting</w>': 1, '   wa g</w>': 2, '   ki r b ys</w>': 1, '   chi se ling</w>': 1, ' i tes</w>': 8, '   na vi ca m</w>': 1, '   na vi co m</w>': 1, '   g un ship</w>': 1, '   e ye ful</w>': 1, '   hu m ve e</w>': 2, '   i ra q is</w>': 27, '   k hur ds</w>': 2, '   v a ll or o</w>': 1, '   g m</w>': 1, ' w es</w>': 1, '   sa u d is</w>': 1, ' e pp s</w>': 1, '   fri en d li es</w>': 2, '   lan ter n on</w>': 1, '   ear th a</w>': 1, '   ki t t</w>': 1, ' fu se</w>': 1, '   we ll ll ll</w>': 1, '   o tt om an</w>': 1, '   pi ll a ged</w>': 1, '   sch war z k op f</w>': 1, '   to po gra p hi cal</w>': 1, ' i te</w>': 1, ' na ga f</w>': 1, '   ar ab land</w>': 1, '   af g ani st an</w>': 1, '   re gi on ally</w>': 1, '   te t an ty</w>': 1, '   vo ca si ty</w>': 1, '   lo ba l</w>': 1, '   sin ex </w>': 3, '   4 7 3 2 </w>': 1, '   t ac tal</w>': 1, '   re ten tion</w>': 2, '   hu mm m m</w>': 1, '   per m ant</w>': 1, '   as sor ted</w>': 2, ' se x act</w>': 1, '   su per st ru c ture</w>': 1, '   cer vi x</w>': 1, ' 2 9 7 </w>': 2, '   s en</w>': 5, '   sh e ll d we ll er</w>': 1, '   e con om i ca lly</w>': 1, ' mo bi li ty</w>': 1, '   1 4 6 </w>': 1, '   con di tion al s</w>': 1, '   fami li ar i z ation</w>': 1, '   di sc oun ted</w>': 1, '   on a</w>': 1, ' se le ction</w>': 1, '   com pu ted</w>': 1, '   ven d able</w>': 1, '   5 5 5 5 </w>': 1, '   s r t</w>': 1, '   en tr on</w>': 1, '   1 1 3 8 </w>': 1, '   9 4 1 0 7 </w>': 1, '   6 4 2 1 </w>': 1, '   3 4 1 7 </w>': 1, '   ma in way</w>': 1, ' ro om s</w>': 1, '   con for m ing</w>': 1, '   s p</w>': 3, '   sc o</w>': 1, '   1 2 0 2 </w>': 1, '   9 0 4 </w>': 1, '   be v </w>': 3, '   2 6 0 0</w>': 1, '   la ke fr on t</w>': 1, '   nu t jo b s</w>': 1, ' til</w>': 1, ' ce ll u lo se</w>': 1, '   m ac g y ver</w>': 1, '   re d st one</w>': 4, '   ti ck ers</w>': 3, ' p un ks</w>': 2, '   di t k a</w>': 1, ' d ev ice</w>': 1, ' d ev i ces</w>': 1, '   sh un ned</w>': 2, '   m ou se tra p</w>': 1, '   su n cre st</w>': 1, '   l ev ea u</w>': 1, '   v a g ran ts</w>': 2, '   pl u ch in s k y</w>': 2, '   ki d do s</w>': 1, '   se m t re x</w>': 1, '   sch no z</w>': 1, '   pro po si tion ed</w>': 1, '   cla ssi f ying</w>': 1, ' p ho o ey</w>': 1, ' i e u</w>': 51, '   la v al</w>': 21, '   gir on</w>': 16, '   fi li b a</w>': 7, '   co let</w>': 11, '   ga u ti er</w>': 3, '   mar i e tt e</w>': 3, ' ton si l i</w>': 1, ' ni en te</w>': 1, ' k in ds</w>': 1, '   su l t ans</w>': 1, '   pa s has</w>': 1, ' comp are</w>': 1, '   o gi l vi e</w>': 1, '   o g le th or pe</w>': 1, '   mon e sc u</w>': 5, ' mon e sc u</w>': 1, '   in sin u a ting</w>': 6, '   la v al s</w>': 3, ' re por t</w>': 1, ' ma da me</w>': 1, ' han d ba g</w>': 1, ' b lu ff</w>': 1, ' rea li ze</w>': 1, ' co let</w>': 1, ' au f</w>': 2, ' wi e der se h n</w>': 2, ' ber lin er</w>': 1, ' z u g</w>': 1, ' z w o e lf</w>': 1, ' u h r</w>': 1, ' g ro ss ar ti g</w>': 1, ' k o lo s sa l</w>': 1, '   i g n ac i o</w>': 2, '   l y ons</w>': 1, ' ys</w>': 1, ' fi li b a</w>': 1, '   s win dle</w>': 1, '   gi go lo s</w>': 1, ' je w el ry</w>': 2, ' fran c</w>': 1, ' pr in ces</w>': 1, '   mar che s a</w>': 2, '   cha mb r o</w>': 1, '   al con i a</w>': 1, ' di st in gu i sh ed</w>': 1, ' a do l p h</w>': 1, ' ki ss</w>': 2, '   mi s la y</w>': 1, '   s co l ding</w>': 1, '   n ou v ea u x</w>': 1, '   e mb ar ra ss es</w>': 1, '   ni c he</w>': 4, '   fo y er</w>': 2, '   f al con i er</w>': 1, ' ma de mo i se ll e</w>': 1, '   bu l k hea ds</w>': 1, '   for e p ea k</w>': 1, '   da vi ts</w>': 1, '   c lu tt er ed</w>': 1, ' ru l ed</w>': 1, '   ca l ver t</w>': 4, '   bu k a ter</w>': 3, '   br ow be at</w>': 1, '   su b mer si b les</w>': 1, '   mu r do ch</w>': 1, '   mar t in et</w>': 1, '   ho ck le y</w>': 7, '   s qu a li d</w>': 1, '   un sin k able</w>': 2, '   man ser v ant</w>': 1, '   ex er tions</w>': 1, '   ex au st ing</w>': 1, '   co e u r</w>': 2, '   m er</w>': 2, '   pro pe ll ers</w>': 2, '   pu d d les</w>': 1, '   s ou p c on</w>': 1, '   im pro ves</w>': 1, '   re ga le</w>': 2, '   de st in i o</w>': 1, '   ca pi to</w>': 1, '   ra ga z z o</w>': 1, '   por c a</w>': 1, '   ye ea a a a a</w>': 1, '   cu l o</w>': 1, '   f ab ri z i o</w>': 3, '   ni en te</w>': 1, ' p in k er ton</w>': 1, ' or re e ble</w>': 1, '   ss s sh h</w>': 1, '   ni ck e lo de on</w>': 1, '   ro ll er co a ster</w>': 2, '   i sa d or a</w>': 1, '   par e e</w>': 2, '   mon i k er</w>': 1, '   wi s so ta</w>': 1, '   li gh to ll er</w>': 1, '   1 9 1 2 </w>': 1, '   ca le don</w>': 1, '   cla i ment</w>': 1, '   1 7 9 2 </w>': 1, ' po i son</w>': 1, ' per su a de</w>': 1, '   ro s son</w>': 4, '   ear ac he</w>': 1, '   lo ther</w>': 8, ' gla d ly</w>': 1, ' sp ent</w>': 1, ' hon est</w>': 2, '   pu r s er</w>': 6, ' c ro ok ed</w>': 1, ' rea ch ing</w>': 1, '   den by</w>': 10, '   m ck in ne y</w>': 7, '   sh or ti e</w>': 6, '   tr ou p ers</w>': 1, '   ro se bu sh</w>': 1, '   s qu a w kin</w>': 1, '   mm mm mm mm m h m</w>': 1, '   ti ck ling</w>': 1, '   sle u thing</w>': 1, ' h ers</w>': 1, '   f our some</w>': 1, ' j un e</w>': 1, '   ca b le gra m</w>': 1, ' k no ts</w>': 1, ' den by</w>': 1, '   bi ll f old</w>': 1, ' st e w ard</w>': 1, '   k ni ck kn ac ks</w>': 2, ' tri ed</w>': 1, ' ned</w>': 1, ' in ex per i en c ed</w>': 1, ' h ome made</w>': 1, '   au tom at</w>': 1, ' ex ce p ting</w>': 1, ' sp en ding</w>': 1, '   tr an sa t lan ti c</w>': 1, ' cu te</w>': 1, ' ro cking</w>': 1, ' lo ther</w>': 1, '   s co tch man</w>': 2, '   pu ll man</w>': 1, '   bi ar ri t z</w>': 1, '   ex tra di te</w>': 2, '   ti po ff</w>': 1, ' s mo o th</w>': 1, '   l un ger</w>': 3, '   si mp er</w>': 1, '   ho lli day</w>': 4, '   t ow n lot</w>': 1, '   co tt a g es</w>': 1, '   ear p</w>': 7, '   be ha n</w>': 4, '   s wa g ger</w>': 1, '   f ar o</w>': 3, ' be hold</w>': 1, '   me ssi can</w>': 1, '   co wh an ds</w>': 1, '   la w ing</w>': 1, '   gon n</w>': 1, '   re qui e sc at</w>': 1, '   e cen tu s</w>': 1, '   stu l t or u m</w>': 1, '   ma gi ster</w>': 1, '   cre d at</w>': 1, '   ju da e us</w>': 1, '   a pe ll a</w>': 1, '   a g is</w>': 1, '   pi sto le er</w>': 1, '   hi lt</w>': 1, '   f al li ble</w>': 1, '   b en i gh ted</w>': 1, '   with al</w>': 1, '   f ra il ti es</w>': 1, '   de fi le</w>': 1, ' hu ed</w>': 1, '   you self</w>': 1, '   for sa king</w>': 1, '   b la y lo ck</w>': 1, '   f re der i c</w>': 1, ' ch op in</w>': 1, '   no c tur n e</w>': 1, '   su s an na</w>': 1, ' st in kin</w>': 1, ' fo ster</w>': 1, '   so p hi sti ca tes</w>': 1, '   cra w fi sh ed</w>': 1, '   b la z in</w>': 1, '   b re e z ed</w>': 1, '   m c ma st ers</w>': 2, '   p rea ch ers</w>': 1, '   sta p p</w>': 1, '   un la d y like</w>': 1, '   bar ged</w>': 1, '   s la pp in</w>': 2, '   bu mm ers</w>': 1, '   dro ver s</w>': 1, ' op er a tor</w>': 1, '   re in</w>': 1, '   ce li a</w>': 1, '   cl an ton</w>': 5, ' ca tch ers</w>': 1, '   mor g</w>': 5, '   c lu m</w>': 1, ' sh oo t in</w>': 1, '   vi r ge</w>': 8, '   spi ri tu a li s m</w>': 1, '   he el ed</w>': 2, '   bu l ge</w>': 1, '   po s se man</w>': 1, '   sh ea f</w>': 3, '   n er v y</w>': 1, '   cla i bor n e</w>': 1, '   loo t in</w>': 1, ' inter est</w>': 1, '   do ted</w>': 1, '   fr ow ner</w>': 1, ' b last</w>': 1, '   ho st e ss es</w>': 1, ' ro ll ers</w>': 1, '   0 0 7 </w>': 18, '   bu k har in</w>': 1, ' as si stan ce</w>': 1, ' cha sing</w>': 1, ' o y d</w>': 1, '   wal ther</w>': 1, '   pp k</w>': 1, '   dis se min ate</w>': 1, '   har m s way</w>': 12, '   lu m pu r</w>': 5, '   pla y bo ys</w>': 1, ' pi lo ts</w>': 1, '   ma l ac c a</w>': 1, '   mon e y pen ny</w>': 4, '   s ar ga s so</w>': 1, '   t s i</w>': 3, '   ti en</w>': 3, '   chi en</w>': 1, '   g ran d da u gh t ers</w>': 1, '   re fin e men ts</w>': 1, '   in st ru men ta tion</w>': 1, '   me te or o lo g</w>': 1, '   i cal</w>': 2, '   con tri bu tor</w>': 1, '   de sc en ded</w>': 2, ' ab er de en</w>': 1, ' k g b</w>': 1, '   s cu r ri l ous</w>': 1, '   mo gu l</w>': 1, '   mi c ro pro ce ss or</w>': 2, ' k no tty</w>': 1, '   ci ti ban k</w>': 1, '   re fin an ce</w>': 1, '   che ong</w>': 1, '   b lo om s</w>': 1, '   af fir m ing</w>': 1, ' r ust</w>': 1, ' e du ce</w>': 1, '   t ar o</w>': 1, '   fri ga te</w>': 1, '   ro e bu ck</w>': 1, '   joh n st one</w>': 1, '   ever har t</w>': 1, '   sa k w a</w>': 1, ' sh w a</w>': 1, '   t s way</w>': 1, '   a gr ound</w>': 1, '   ti mber</w>': 1, '   m un cy</w>': 1, '   gr in d ers</w>': 1, '   fu el ed</w>': 1, '   sta mp er</w>': 1, '   s b</w>': 1, '   ki lo ton</w>': 1, '   da e</w>': 1, '   y un g</w>': 1, '   w oo d sy</w>': 1, '   ca d ence</w>': 1, '   bea le</w>': 1, '   e mor y</w>': 8, '   ho ck er</w>': 6, '   what no ts</w>': 1, '   ir re ver ent</w>': 1, '   sin fu l ne ss</w>': 1, '   ho ll er ed</w>': 1, ' sin ning</w>': 1, '   chi t lin s</w>': 1, '   bab ying</w>': 1, '   ha t ti e</w>': 3, '   pl ough</w>': 1, '   re de e m ing</w>': 1, '   per si mm on</w>': 2, '   lu ll a</w>': 2, '   c ome u pp an ce</w>': 1, '   af ter bi r th</w>': 1, '   ra f fi sh</w>': 2, '   to l er ably</w>': 1, ' h mm mm m</w>': 1, '   wh oo oo o ah</w>': 1, '   a ac h</w>': 1, '   li gh t year</w>': 7, '   wh oo sh</w>': 1, ' bu z z</w>': 3, ' wa sted</w>': 1, '   ne s b it</w>': 1, '   dar j ee ling</w>': 1, '   har ne ss es</w>': 1, '   sp ac e f re i gh ter</w>': 1, '   je t ti s ons</w>': 1, '   na v a</w>': 1, '   sp ac e man</w>': 1, '   li gh t s n ack</w>': 1, '   li gh t be er</w>': 1, '   ter i lli u m</w>': 1, ' car b on i c</w>': 1, '   fu el s</w>': 1, '   c r y sta li c</w>': 1, ' y a a a h h</w>': 1, '   win g sp an</w>': 1, '   sh ee pi sh</w>': 1, '   sp u d head</w>': 1, ' e pped</w>': 1, ' op er a tions</w>': 1, '   ma tt el</w>': 2, '   pla y s k ool</w>': 1, ' st ab b in</w>': 1, ' en v y</w>': 1, '   l un ch bo x</w>': 2, '   ro o a a a ar r</w>': 1, '   con fr on ta tions</w>': 1, '   bo ar d ga me</w>': 2, '   s ar g ent</w>': 1, '   ye ss ss</w>': 2, '   f on de st</w>': 1, '   j on a th on</w>': 3, '   jo ve</w>': 1, '   fe ster</w>': 1, ' fu ri ous</w>': 1, '   pla i ts</w>': 1, '   pl ou gh ed</w>': 1, '   wa ke fi e ld</w>': 2, '   sa la z ar</w>': 8, '   ma dri ga l</w>': 7, '   ob re g on</w>': 6, '   hon ed</w>': 1, '   la und ro ma ts</w>': 1, '   ja y wal k</w>': 1, ' re cor ded</w>': 1, '   le i c a</w>': 1, '   me di ca ting</w>': 1, '   su per fun d</w>': 1, '   c z ar in a</w>': 1, '   pla ti tu de</w>': 1, '   le ga li z ed</w>': 1, '   le ga li z ation</w>': 1, '   le ga li ze</w>': 1, '   le ga li z ing</w>': 1, '   fin a li st</w>': 1, '   the spi an</w>': 1, '   f re e ba se</w>': 1, '   mar ad on a</w>': 2, '   mon te l</w>': 2, '   me t z ger</w>': 1, '   te ll tal es</w>': 1, '   pu t na m</w>': 1, '   en t re pr en e u ri al</w>': 1, '   h y dro p on i c</w>': 1, '   ra sp ber ri es</w>': 1, '   ti g ri llo</w>': 2, ' mo l ded</w>': 1, '   e sp a sti c o</w>': 1, '   j ac ob o</w>': 1, '   a y al a</w>': 1, '   mar que z</w>': 2, '   ja v i</w>': 2, '   gu z man</w>': 1, ' co ke head</w>': 1, '   ja vi er</w>': 2, '   car te ls</w>': 2, '   ju ar a z</w>': 1, '   cha in sa w s</w>': 1, '   te ch ni ca li ti es</w>': 1, '   na f ta</w>': 1, ' tra il ers</w>': 1, '   lan d ry</w>': 2, '   au to cra ti c</w>': 1, ' de f ea ting</w>': 1, '   par ti s an</w>': 1, ' ver sed</w>': 1, '   inter di ction</w>': 1, '   ma pp ing</w>': 1, '   gen o me</w>': 1, '   n in te en</w>': 1, '   do d gi est</w>': 1, '   s k a g</w>': 6, '   be g bi e</w>': 2, '   sa u gh ton</w>': 1, '   sp u d</w>': 7, '   bu f ti e</w>': 1, '   ke mp ton</w>': 1, '   p un t</w>': 3, '   z i g g y</w>': 1, '   ge m mi ll</w>': 1, '   to x op la s mo s is</w>': 2, '   r en ton</w>': 1, '   cra i g ne w ton</w>': 1, '   f oo t ba ll er</w>': 1, '   d ss</w>': 1, '   gir o</w>': 1, '   p un ting</w>': 2, '   s wan ne y</w>': 7, ' ho p ers</w>': 1, '   d ra f t p ac ks</w>': 1, '   s che mi es</w>': 1, '   co in ing</w>': 1, '   e u gh h</w>': 1, '   m cl aren</w>': 1, '   ro a ld</w>': 2, '   da h l</w>': 2, '   pi sh ed</w>': 1, '   go l d fin ger</w>': 1, '   th under b all</w>': 1, '   no ta ble</w>': 1, '   ten n ers</w>': 1, '   fi b re</w>': 1, '   wan king</w>': 1, '   ra dge</w>': 1, '   pr in ci pa lly</w>': 1, '   gu b bed</w>': 1, '   under took</w>': 1, '   qu al</w>': 1, '   shu tt le cra ft</w>': 2, '   pre ar ran ged</w>': 1, '   con su ls</w>': 1, '   in for ma lly</w>': 1, '   bl ow sc re en</w>': 1, '   ve i w sc re en</w>': 1, '   star let</w>': 1, '   sy bo k</w>': 16, '   bo o ster</w>': 1, '   me l ons</w>': 1, '   un er r ing</w>': 1, '   shi p ma tes</w>': 1, '   k or r d</w>': 4, '   f oo t spe ed</w>': 1, '   8 5 6 3 </w>': 1, '   stra te gi es</w>': 1, ' row</w>': 4, '   mer ri ly</w>': 4, '   sin g al ong</w>': 1, '   y o se mi te</w>': 1, '   rea l ty</w>': 1, '   s cour ce</w>': 1, ' e d en</w>': 1, ' qu i</w>': 1, ' v or ta</w>': 1, '   v or</w>': 1, '   an d ori an</w>': 1, '   un pr on oun c ea ble</w>': 1, ' ra i se</w>': 1, '   cu r ran t</w>': 1, '   bor gu s</w>': 1, '   tr on</w>': 13, '   par an o i ds</w>': 3, '   ye pp er</w>': 1, '   i co m</w>': 4, '   wa y back</w>': 1, ' e m be z z ling</w>': 1, '   si ph on</w>': 1, '   l or a</w>': 1, '   r oun d tri p</w>': 1, '   di sin te gra ting</w>': 2, '   di gi ti z ing</w>': 1, '   d ow n time</w>': 1, '   n yet</w>': 1, '   t in t y pe</w>': 1, '   v d t</w>': 1, ' 4 1 5 </w>': 1, '   app ro pri a ted</w>': 1, '   tri c ki er</w>': 1, '   un called</w>': 1, ' re z z ed</w>': 3, '   du mon t</w>': 3, '   y or i</w>': 5, '   s ar k</w>': 6, '   l ev e lled</w>': 1, '   ra kin</w>': 1, '   do ma ins</w>': 1, '   re co g ni z er</w>': 2, ' se ctor</w>': 1, '   re co g ni z ers</w>': 1, '   an nu i ty</w>': 1, '   lo ca tes</w>': 1, '   cle an cu t</w>': 1, ' ac kn ow le dge</w>': 1, '   mo x i e</w>': 1, '   c rea m pu ff s</w>': 1, '   n an o secon ds</w>': 1, '   ju mp start</w>': 1, '   hon e y be ar</w>': 1, '   r en qui st</w>': 6, ' \t \t \t \t \t \t \t \t \t </w>': 2, '   wor k up</w>': 1, '   bo p</w>': 1, '   fa i si l</w>': 1, '   har ri ers</w>': 1, '   of f er r ing</w>': 1, '   ta s k er</w>': 8, ' cu be</w>': 1, '   fun dam en ta li st</w>': 1, ' b om b in gs</w>': 1, '   a bu </w>': 1, '   k a le em</w>': 1, '   ma li k</w>': 1, '   su per sp y</w>': 1, '   k ha l ed</w>': 1, ' na il</w>': 1, '   bo in kin</w>': 1, '   a x l</w>': 1, '   b om bar d ment</w>': 1, ' bri e f</w>': 1, '   tal k ra di o</w>': 1, ' vo cal</w>': 1, '   sa mi r</w>': 3, '   ex por t</w>': 1, '   t ow tru ck</w>': 1, '   con ve ying</w>': 1, '   tr ou b le sh oo ting</w>': 1, '   6 8 0</w>': 2, '   b on e head</w>': 1, '   el u de</w>': 1, '   ye ea o ow w w</w>': 1, '   so oo o o</w>': 1, ' t ea ch ers</w>': 1, '   ve tt e</w>': 1, '   j ack al</w>': 1, '   ke tt le man</w>': 1, '   m c gra th</w>': 1, '   mar ri o t</w>': 1, '   el b ow ed</w>': 1, '   wa m</w>': 1, ' cl ar ence</w>': 1, '   can co on</w>': 3, '   tur t le do ve</w>': 1, '   s ca tch</w>': 1, '   f ru stra t in</w>': 1, '   dr ow n in</w>': 1, '   so a p y</w>': 1, '   chi ll in</w>': 1, '   bab al ou ey</w>': 1, '   qui ck d ra w</w>': 1, '   dre x l</w>': 14, '   por sch es</w>': 1, '   sto ck bro k ers</w>': 1, '   be e p ers</w>': 1, '   p un ch in</w>': 1, '   st om ch</w>': 1, ' al ab a ma</w>': 1, '   mi s con ce p tions</w>': 1, '   j an is</w>': 1, ' ro o ki es</w>': 1, '   bor g n ine</w>': 1, ' spi der man</w>': 1, '   cou l d da</w>': 1, '   r our ke</w>': 1, ' st re e t fi gh ter</w>': 1, ' p ac ked</w>': 1, '   go ob ers</w>': 1, '   as sa s in</w>': 1, '   c lu m si est</w>': 1, '   con tr ac tions</w>': 1, '   z on k ers</w>': 1, ' dri ver</w>': 2, '   wal do</w>': 3, '   da u gh ter ly</w>': 1, '   s m ack er o o</w>': 1, '   for a ger</w>': 1, '   ok e e</w>': 2, ' do ke e</w>': 2, ' do g gi e</w>': 1, ' d on gs</w>': 1, '   thin ke th</w>': 1, ' ki ss in</w>': 1, '   at la s</w>': 1, '   un ha un ted</w>': 1, '   wan na be e</w>': 1, '   d on ow i t z</w>': 7, '   ca ll back</w>': 1, '   b re ck</w>': 2, '   hea vi es</w>': 1, '   f re e lo ad in</w>': 1, '   b li t z er</w>': 1, '   en chi la da</w>': 1, '   s k in ni er</w>': 1, '   cl ar</w>': 1, '   st re e t fi gh ter</w>': 3, ' si ll</w>': 1, '   la w re ys</w>': 1, '   nor ms</w>': 1, ' i v ory</w>': 1, '   un wa t cha ble</w>': 1, '   un rea d able</w>': 1, ' so p hi e</w>': 1, ' or din ary</w>': 1, ' k ra m er</w>': 1, ' g an d hi</w>': 1, ' a po ca l y p se</w>': 1, '   z hi v ago</w>': 4, '   di st ri bu t ers</w>': 1, '   tw on</w>': 1, '   bo y le</w>': 3, '   de fini tly</w>': 1, '   fo st ers</w>': 1, '   w op s</w>': 1, '   che st er fi el ds</w>': 1, '   hi gh ta il ed</w>': 1, '   w ou l d da</w>': 1, '   pi mp in</w>': 1, '   che st er fi e ld</w>': 1, '   per son i fi ed</w>': 1, '   co c co tt i</w>': 2, '   m c t ea gu e</w>': 1, '   mi li ta</w>': 1, '   th under bi r ds</w>': 1, '   m c que en</w>': 1, '   b lu ff in</w>': 2, ' bu st ers</w>': 1, ' p an ny</w>': 1, '   s om thing</w>': 1, '   in tru si ons</w>': 1, ' s li tting</w>': 1, '   du mp in</w>': 1, '   k an d i</w>': 1, ' ma tes</w>': 2, '   be e ch w o od</w>': 1, '   car de ll a</w>': 4, '   nu t s ack</w>': 1, ' di men si ons</w>': 1, '   z on ed</w>': 1, ' mi s for t un es</w>': 1, '   k in sha s a</w>': 2, '   k ar ac hi</w>': 2, ' di st or ted</w>': 1, ' be come</w>': 1, ' d rea med</w>': 1, '   di ver g ent</w>': 2, ' men ta lly</w>': 1, ' fu z zy</w>': 1, '   vi ro lo gi st</w>': 2, '   z s</w>': 1, '   1 9 8 9 </w>': 5, ' st ran ge st</w>': 1, '   ra i lly</w>': 1, '   bu li shit</w>': 1, ' de se cra tion</w>': 1, '   cha tt er ed</w>': 1, '   go in es</w>': 1, ' im m ac u late</w>': 2, '   per i ls</w>': 1, '   w oo oo o</w>': 1, '   do sed</w>': 1, ' pro per ly</w>': 1, '   di sin fe c t an ts</w>': 1, ' se m me l we i ss</w>': 1, '   wh ad da you</w>': 1, ' ger ms</w>': 1, '   wh ack o s</w>': 1, '   lo on</w>': 1, ' to i let</w>': 1, '   bl en d ers</w>': 1, '   e le c tri ca lly</w>': 1, '   what da ya</w>': 1, ' jo se</w>': 1, '   re stra in ed</w>': 1, '   wri g g ling</w>': 1, ' d ev e lo p</w>': 1, '   pa the ti ca lly</w>': 1, '   pu si ll an im ous</w>': 1, ' pre t end</w>': 1, ' p sy chi a tri st</w>': 1, '   co les</w>': 1, '   st e ll a z ine</w>': 1, ' mon ke ys</w>': 1, '   stra ying</w>': 2, '   to ss in</w>': 2, '   in fir m</w>': 2, '   pe tt in</w>': 2, '   pe pp a</w>': 1, '   mo v a do</w>': 1, '   gu mp tion</w>': 2, '   li pped</w>': 1, '   hea d li ght</w>': 2, '   ma l d en</w>': 1, '   li c ki ty</w>': 2, '   sha st a</w>': 2, '   fin ger less</w>': 1, '   to e less</w>': 1, '   mon t ro se</w>': 1, '   ha ld</w>': 1, '   bra g ged</w>': 1, '   bra gs</w>': 1, '   re el ed</w>': 2, '   li v el in ess</w>': 2, '   run ne th</w>': 1, '   beau t i</w>': 1, '   se x ing</w>': 1, '   o ver v al u e</w>': 1, '   ju ke bo x</w>': 2, '   ju ke</w>': 3, '   ve sc i</w>': 4, ' doll ars</w>': 2, '   sh rea ds</w>': 1, '   fro om</w>': 1, '   ja mi ll a</w>': 4, '   ha bl ar</w>': 2, '   ear th ma k er</w>': 1, '   ki ss y</w>': 3, '   fu ff on i u m</w>': 1, '   fuck on on i u m</w>': 1, '   as son on i u m</w>': 1, ' om</w>': 1, '   v ye e</w>': 1, ' ir on e e</w>': 1, '   u p fr on t</w>': 1, '   son u f ab i tch</w>': 3, '   fi g g ers</w>': 1, '   do o dad</w>': 1, '   sch no o z</w>': 1, '   ga sh ed</w>': 1, ' ja ke</w>': 1, '   ba i ting</w>': 1, ' sh er i ff</w>': 1, '   ha l f b re ed</w>': 1, '   el k har t</w>': 1, '   bi a</w>': 1, '   ar k a dy</w>': 2, ' sho t gu n</w>': 1, '   hi ck town</w>': 1, ' ro b bed</w>': 1, ' ar k a dy</w>': 1, '   ar k ad in</w>': 1, '   dan ged</w>': 1, ' con c ea l ed</w>': 1, '   te ll e tu b bi es</w>': 1, '   in ver so</w>': 1, '   st ru tt in</w>': 2, '   wi g g in</w>': 1, '   cor n er back</w>': 1, '   e qui v al en ts</w>': 1, '   der a i ls</w>': 1, '   at ro ph y</w>': 1, ' 7 8 </w>': 1, '   p al si ed</w>': 1, '   beau cha mp</w>': 6, '   vi ll a in ous</w>': 1, '   r oun d t ree</w>': 1, '   bu ll whi ps</w>': 1, '   no ti c in</w>': 1, '   p ea ce ma k ers</w>': 1, '   da g ge t t</w>': 2, '   ro an</w>': 1, '   s qu ir re l ed</w>': 1, '   ha m st r ing</w>': 1, '   la w ful</w>': 1, '   h en der sho t</w>': 4, '   re ce p t ac les</w>': 1, '   w want</w>': 1, '   hur r y in</w>': 1, '   whi z z in</w>': 1, '   a i ms</w>': 2, '   in su l t in</w>': 1, '   tw o gu n</w>': 1, '   b la z ed</w>': 1, '   si x g un s</w>': 1, '   de pi c ting</w>': 1, ' du ke</w>': 2, ' w r</w>': 2, '   w r wri te</w>': 1, '   le m me e</w>': 2, '   sch o fi e ld</w>': 6, '   b on ne y</w>': 1, '   a li s son</w>': 1, ' 7 3 </w>': 1, '   mi x in</w>': 1, '   gu </w>': 1, '   your ow n self</w>': 1, '   ri per</w>': 1, '   c ro t che ty</w>': 1, '   o ga ll al a</w>': 1, '   ni o br ar a</w>': 1, '   god dam n de st</w>': 1, '   so th ow</w>': 2, '   har be y</w>': 1, '   m un ny</w>': 3, '   sin g le han ded</w>': 1, '   dro ver</w>': 1, '   o ver fri en d ly</w>': 1, '   c un ny</w>': 1, '   ke y s er</w>': 20, '   so ze</w>': 24, ' lt</w>': 18, ' g t</w>': 18, ' ke y s er</w>': 1, '   for ti er</w>': 1, '   hi j ack er</w>': 1, '   k int</w>': 8, '   f en ster</w>': 5, '   lo gi sti cal</w>': 1, '   re d f oo t</w>': 5, '   fin n er an</w>': 7, '   shi v ved</w>': 2, '   ar ra i g n</w>': 1, ' s la pped</w>': 1, '                                     </w>': 1, '   k u j an</w>': 7, '   ch ea pe st</w>': 1, ' c ri pp le</w>': 1, ' fe tch ed</w>': 1, '   co er ci on</w>': 1, '   pu ck er</w>': 1, '   5 2 </w>': 2, '                                                                                                                           </w>': 1, '   k o v as h</w>': 1, '   me t z he i s er</w>': 1, '   ri d g ly</w>': 1, '   de e m er</w>': 2, '   ta il b one</w>': 1, '   p al sy</w>': 1, ' f ac e less</w>': 1, ' n ev ins</w>': 1, ' mar x</w>': 1, ' me th o do lo g y</w>': 2, '   in com pe te m t</w>': 1, ' ex pe ct</w>': 1, '   f our years</w>': 1, ' mi st</w>': 1, ' lo y al ty</w>': 1, ' g al v in</w>': 1, '   car lo tta</w>': 18, '   v al d es</w>': 6, '   bri gh tly</w>': 1, '   p sy cho an al y st</w>': 1, '   shi p bu il ding</w>': 2, '   ac ro p ho bi a</w>': 4, '   tr an ces</w>': 1, '   el ster</w>': 7, '   ran so ho ff</w>': 1, '   sa lin a</w>': 2, '   ma g n in</w>': 2, '   ga ll up</w>': 1, '   f er gu son</w>': 4, '   ba u ti st a</w>': 1, '   clo i ster</w>': 1, '   car ri a g es</w>': 2, '   mi r r or ed</w>': 1, '   se qu o i a</w>': 1, '   se m per vi r en s</w>': 1, '   co it</w>': 2, '   r ow bo a ts</w>': 1, ' fa lling</w>': 1, '   pre si di o</w>': 1, '   under cu r r ent</w>': 1, '   le i be l</w>': 4, '   e mb ar ca der o</w>': 2, '   1 8 7 9 </w>': 2, '   ar go sy</w>': 1, '   u p li ft</w>': 1, '   can ti l ever</w>': 1, '   cha ir bor n e</w>': 1, '   di sa pp ro ving</w>': 1, '   cor se ts</w>': 1, '   un con fin ed</w>': 1, '   ca bar et</w>': 1, '   i ves</w>': 2, '   tw e ed</w>': 1, '   ac ce sor y</w>': 2, '   s ac ri le gi ous</w>': 1, '   pa ton</w>': 1, '   ea ger ly</w>': 1, '   st r en ght</w>': 1, '   re uni te</w>': 1, '   su ti ca ses</w>': 1, '   l b</w>': 1, '   mo ps</w>': 1, '   mi ch ea l</w>': 29, ' op tion</w>': 1, '   star bur sts</w>': 3, '   me ll ow ing</w>': 1, ' s che du l ed</w>': 1, '   go osed</w>': 7, '   k er ou back</w>': 1, '   spe ci fi ca tion</w>': 1, '   su per se d es</w>': 1, '   fa t bur ger</w>': 1, ' fi e ld</w>': 1, '   d win d ling</w>': 1, '   st ee l er</w>': 1, '   im m ac u late</w>': 1, '   ri co ch et</w>': 2, '   re sur re c ting</w>': 1, ' bo y d</w>': 1, ' re f und able</w>': 1, ' dar l i</w>': 1, '   dar l i</w>': 1, ' tri mm ed</w>': 1, '   com mon ed</w>': 1, ' op ra h</w>': 1, '   c r ow ning</w>': 1, '   in stan ts</w>': 1, '   ma il bo x es</w>': 1, '   cu t b ac ks</w>': 1, '   li gh t f oo t</w>': 1, '   thin ked</w>': 1, '   wa ho o</w>': 3, '   om</w>': 1, '   sp ar k l er</w>': 1, ' wa ho o</w>': 1, '   z z z z z z z z z z z z z z z z</w>': 1, '   br en n</w>': 2, ' 3 2 1 </w>': 1, ' 2 5 6 </w>': 1, '   3 4 3 2 </w>': 1, ' 3 4 3 </w>': 1, ' pu ck er ing</w>': 1, ' p up</w>': 5, ' sh el m er</w>': 1, '   sh el m er</w>': 2, ' at la s</w>': 1, '   sh ru g ged</w>': 1, '   ac ou sti c</w>': 1, ' ve g as</w>': 1, ' re gre ssion</w>': 1, ' e mb ar ra ss ing</w>': 1, '   de gen er ation</w>': 1, '   a ma li o</w>': 3, '   re fu gi o</w>': 1, '   en e din a</w>': 2, '   b al con i es</w>': 1, '   s ca mp</w>': 1, '   ma i ze</w>': 1, '   man u r ing</w>': 1, '   vi ri di an a</w>': 4, '   pe se ta s</w>': 1, '   can el o</w>': 1, '   r ab bi ting</w>': 1, '   le pro sy</w>': 1, '   mon ch o</w>': 3, '   d or mi t ori es</w>': 1, '   gu t ful</w>': 1, '   lin den me y er</w>': 18, '   lo ca ter</w>': 1, '   par o le e</w>': 1, '   to lu c a</w>': 1, '   af f lu ent</w>': 1, '   le t ac </w>': 3, '   de an e</w>': 3, '   d on le y</w>': 1, '   co ll e c ti ve ly</w>': 1, '   un ac ce p ta b ly</w>': 1, '   de ton a tes</w>': 1, '   au th ori z es</w>': 1, '   par ti ci p an ts</w>': 1, '   con ne c t ors</w>': 2, '   ca li bra ted</w>': 1, '   v r</w>': 3, '   co l d b loo ded</w>': 1, '   po l y m er</w>': 1, '   1 8 3 </w>': 1, '   gi ve th</w>': 1, '   ta ke th</w>': 1, ' a da p ting</w>': 1, ' e du ca ted</w>': 1, '   k ar in</w>': 1, '   can ni ba li ze</w>': 1, '   l ab i an c a</w>': 2, ' en ac ting</w>': 1, '   b y stan der</w>': 1, '   dis se c ted</w>': 1, '   pre di sp o si tion</w>': 1, '   par ti ci pa ted</w>': 1, '   fin a le</w>': 1, ' ro bo ti c</w>': 1, '   o ver pa ying</w>': 3, '   sch u man n</w>': 17, '   t ar ri es</w>': 1, ' lu re</w>': 1, ' e f fe c ts</w>': 2, '   me d i</w>': 1, ' pi tch</w>': 2, '   3 0 3 </w>': 4, '   co ked</w>': 1, ' gi lls</w>': 2, ' ma ter i al</w>': 1, ' o pp ort uni ty</w>': 2, ' b ack story</w>': 1, '   ye ea a h h h</w>': 1, '   in au gu ra l</w>': 6, '   al b ani an</w>': 21, ' sta tion ma ster</w>': 1, ' ca li c o</w>': 2, ' char t</w>': 1, ' re ta in er</w>': 1, '   le tt em</w>': 1, '   gir l sc out</w>': 4, '   b ack la sh</w>': 1, ' hi ll bi lly</w>': 1, ' wi ll ya</w>': 1, ' al b ani a</w>': 3, ' ye ssi r</w>': 1, ' pro du c ing</w>': 1, '   a a a and</w>': 1, '   oo o okay</w>': 1, '   ge o poli ti ca lly</w>': 1, '   gre en c ard</w>': 1, '   mi l it</w>': 1, ' e m my</w>': 1, ' wh a le shit</w>': 1, ' ra in for est</w>': 1, ' con vi ct</w>': 1, '   p ea c ni k</w>': 1, '   se g men ts</w>': 1, ' f l ower</w>': 1, ' pen ta g on</w>': 1, ' cu sto dy</w>': 1, ' secon d ly</w>': 1, ' tur n co at</w>': 1, ' me di ca tion</w>': 1, ' la und ry</w>': 1, ' d d d</w>': 1, ' bra z en</w>': 1, ' plan e</w>': 1, '   be l t way</w>': 2, '   poli ti k</w>': 1, '   ma tch ma k er</w>': 2, '   l ev in s k y</w>': 3, '   sa u l</w>': 1, ' na sh vi ll e</w>': 1, ' pa tri o t</w>': 1, ' to go</w>': 2, '   pu tt em</w>': 3, ' wee p</w>': 1, '   sti lling</w>': 1, '   sa in ted</w>': 1, '   la de e s n gen n l men</w>': 1, '   al b ani a</w>': 15, ' al b ani an</w>': 1, '   te m pe st</w>': 1, ' o c cu r ing</w>': 1, ' shi f ty</w>': 1, ' b om b er</w>': 1, ' le go s</w>': 1, ' app ear an ce</w>': 2, '   ge m me</w>': 4, ' di st r act</w>': 1, '   h h h h</w>': 2, ' bo e ing</w>': 1, '   brea n</w>': 10, ' brea ks</w>': 1, '   a ll e ging</w>': 1, '   un t ow ard</w>': 2, ' re ve la tion</w>': 1, '   h inter land</w>': 1, '   pi c tu re s qu e</w>': 1, ' e mp lo y</w>': 1, ' pi i i i ll ll l ll</w>': 1, '   can ni st ers</w>': 1, ' be ans</w>': 1, ' s wa y ed</w>': 1, '   sch w n</w>': 1, ' re v ea l</w>': 1, ' s ki lls</w>': 1, '   pre s sc or p</w>': 1, '   wi de m ou th ed</w>': 1, '   di ssi den ts</w>': 3, '   nu c</w>': 1, ' con ta o t</w>': 1, ' cre di ts</w>': 1, '   a ir k or ce</w>': 1, '   ban d ma ster</w>': 1, '   do o d ling</w>': 1, '   po o l cha ir</w>': 1, ' i ta ly</w>': 2, ' bo o t</w>': 2, ' gi v v em</w>': 1, ' ab i li ty</w>': 1, ' nu cle ar</w>': 2, '   un be kn ow n st</w>': 1, ' go ver n men ts</w>': 1, ' pre par ed</w>': 1, ' dri ll</w>': 1, '   we ll e s ly</w>': 1, '   gir l s cou ts</w>': 2, '   di sp la y ed</w>': 1, '   a h out</w>': 1, ' de p lo y</w>': 1, ' a tions</w>': 1, ' fi l m school</w>': 1, ' cre d it</w>': 3, '   pa al ll</w>': 1, ' wh ad da ya</w>': 1, ' ju dge</w>': 2, ' im mi g ran t</w>': 1, ' mo per y</w>': 1, ' nu n</w>': 1, '   sch u ster</w>': 1, '   p ack in g cra te</w>': 1, ' sh h</w>': 1, '   sa ya</w>': 1, '   nu m</w>': 1, ' pre sc ri p tion</w>': 1, ' cra te</w>': 1, '   he l</w>': 2, ' a m bu lan ce</w>': 1, ' wh e at</w>': 1, ' tw ins</w>': 1, ' gi v em</w>': 1, ' k in g do m</w>': 1, ' vo ting</w>': 1, '   z t</w>': 1, ' lin dy</w>': 1, ' re ti re ment</w>': 1, '   l co k it</w>': 1, '   ge tt em</w>': 1, ' s qu ee ze</w>': 1, ' ja w s</w>': 1, ' t ea se</w>': 1, '   sc bu man n</w>': 1, '   com mu t ers</w>': 1, '   ki ss en ger</w>': 1, '   p rea k ne ss</w>': 1, ' cor ny</w>': 2, ' ha ts</w>': 1, ' s w ea ter</w>': 1, '   no t re</w>': 1, '   no le</w>': 1, '   hu gs</w>': 1, '   h or der</w>': 1, '   m c dan i el</w>': 1, '   in e f fe c tive</w>': 1, '   t ea s er</w>': 2, ' y o h a</w>': 1, '   y o h a</w>': 2, ' sta ged</w>': 1, ' pa ge ant</w>': 2, ' po stu re</w>': 1, '   s lo g ans</w>': 1, ' w ars</w>': 1, '   su ri b ac hi</w>': 1, '   ti pp e can o e</w>': 1, ' wh ad day</w>': 1, ' n u</w>': 1, ' sh re ds</w>': 1, ' ac e ta te</w>': 1, '   sch u man</w>': 1, ' 2 8 4 1 </w>': 1, ' 2 6 2 </w>': 1, ' pro gra ms</w>': 1, ' sch u ster</w>': 1, '   ser vi ce man</w>': 2, ' shu ck</w>': 1, '   ma p le lea f s</w>': 1, ' je ans</w>': 1, ' ter r or</w>': 1, '   d ra pe</w>': 1, ' ber et</w>': 1, ' c ru sh</w>': 2, '   m oun ti e</w>': 2, '   m oun ti es</w>': 2, ' ter r ori sts</w>': 1, ' sta ging</w>': 1, '   w ran g l er</w>': 1, ' ki tt en</w>': 2, ' mo bi li z ing</w>': 1, ' de f end</w>': 1, '   mo bi li z ing</w>': 1, '   bu t s k y</w>': 1, ' bor d ers</w>': 1, '   be lu sh i</w>': 3, ' w oo ds</w>': 1, ' pi ll</w>': 1, ' 1 9 9 0</w>': 1, ' y body</w>': 1, ' gh e tt o s</w>': 1, '   under stan d ab </w>': 1, ' pu tt in</w>': 1, ' ph ar m ac i st</w>': 1, ' su si e</w>': 1, ' p on d</w>': 1, '   b lu e st ar</w>': 18, '   ar b s</w>': 1, '   to a ds</w>': 1, ' in sure</w>': 1, '   sch tu pp ing</w>': 1, '   b al king</w>': 1, '   ar b</w>': 1, '   y u ro vi ch</w>': 2, '   ho li dea ls</w>': 1, '   ac cu mu la tion</w>': 2, '   mar ni er</w>': 2, '   an ac o t t</w>': 4, '   op t or e c tom i es</w>': 1, '   op t or e c tom y</w>': 1, ' de w ey</w>': 1, '   no stra dam us</w>': 1, '   di p sti ck</w>': 1, '   re ce ssion</w>': 1, '   th u d der</w>': 1, '   wi cks</w>': 1, '   st ee pl es</w>': 1, '   ge k k o</w>': 25, '   be ar i sh</w>': 1, '   wi l more</w>': 1, '   di vi d end</w>': 1, '   mi sc hi ev ous</w>': 1, '   su gar man</w>': 1, '   ro sc o</w>': 1, ' da vi do ff</w>': 1, ' st rong</w>': 1, '   he w li t t</w>': 1, '   lu te ce</w>': 1, '   ro get</w>': 1, '   fa ir chi ld</w>': 1, '   te l d ar</w>': 4, '   dar i en</w>': 12, '   wa ter s k i</w>': 1, '   w re ck able</w>': 1, '   th wi ck</w>': 1, '   hou r ly</w>': 2, ' 8 5 0</w>': 1, '   o ver f und ed</w>': 1, '   wi l d man</w>': 6, '   d ev on</w>': 2, ' a st on i sh</w>': 1, '   e u ro f la sh</w>': 1, '   g q </w>': 1, '   der e gu la ted</w>': 1, '   re ev al u ate</w>': 1, ' g an g</w>': 1, ' a w right</w>': 1, '   o ver he ars</w>': 1, '   x y z</w>': 2, '   r d l</w>': 1, '   ra i d ers</w>': 1, '   t ar af ly</w>': 1, '   wa sp s</w>': 2, ' fini sh</w>': 1, '   com er</w>': 1, '   whi te w o od</w>': 1, '   sy n di ca m</w>': 1, '   com pu tes</w>': 1, '   sy sto li c</w>': 1, '   di a sto li c</w>': 1, '   l c d</w>': 1, '   ki mo sa be</w>': 1, '   f ru pp i e</w>': 1, '   l y ce e</w>': 1, '   fran ca i se</w>': 1, ' de mo li sh ed</w>': 1, '   di a i a on d</w>': 1, '   hou se pl ant</w>': 1, '   ta bri z</w>': 1, '   ce la don</w>': 1, '   cu shi ons</w>': 1, '   din g y</w>': 1, ' co lu m bi an</w>': 1, '   wi l d w o od</w>': 1, '   app ra i s er</w>': 1, ' j an et</w>': 1, '   su gar pi e</w>': 1, '   pu t ne y</w>': 1, ' s cu m</w>': 1, ' de di ca ted</w>': 1, '   nor th st ar</w>': 1, '   ph ar o ah</w>': 1, '   comp ac tor</w>': 1, '   be e z er</w>': 1, '   j an son</w>': 2, '   re ar don</w>': 1, '   c n x</w>': 1, '   je ss m on</w>': 1, '   li qui da tion</w>': 1, '   mo der ni ze</w>': 1, '   r ar er</w>': 1, '   lu ger</w>': 1, ' man ned</w>': 1, '   go b bl ed</w>': 1, ' j e</w>': 1, '   n es</w>': 1, '   qu a</w>': 1, '   co ff</w>': 1, '   lo f t more</w>': 1, '   wa x work</w>': 4, '   v ell</w>': 1, '   v a x ver k</w>': 1, '   w oo o</w>': 1, ' e s qu e</w>': 1, '   w er e bea st</w>': 1, ' fi ed</w>': 1, '   ca ff ine</w>': 1, '   f ac e ti ous</w>': 1, ' ne st</w>': 1, '   in tu it</w>': 1, '   in tru si ven ess</w>': 1, '   ma d die</w>': 1, '   a li ght</w>': 1, ' par tly</w>': 1, '   se l a</w>': 2, '   fe u r</w>': 7, '   me f</w>': 2, '   plan che tt e</w>': 2, '   ni i ice</w>': 1, ' ac he</w>': 1, '   we ll ne ss</w>': 1, '   k a m bu ch a</w>': 1, '   a a and</w>': 1, '   ma u du h</w>': 1, '   mm mu h</w>': 1, '   sa y i</w>': 1, ' sy mp to m</w>': 1, '   fe u rs</w>': 1, '   in t ro du ces</w>': 1, '   sch u m way</w>': 1, '   cu ri e</w>': 1, ' whi sp er ing</w>': 1, '   mm n p h</w>': 1, '   tom es</w>': 1, '   my se l</w>': 1, '   su g</w>': 1, '   me chan i s ms</w>': 1, '   e u k ar y o ti c</w>': 1, '   di e d re</w>': 7, '   st un ting</w>': 1, '   la sh ers</w>': 1, '   la sh er</w>': 17, '   he ar t bea ts</w>': 1, '   sha d ow ing</w>': 1, '   sp ac i al</w>': 1, '   li e ten</w>': 1, '   sc ar e dy</w>': 1, '   ta la ma sc a</w>': 1, '   may fa ir</w>': 16, '   c ro v d</w>': 1, '   m oun th s</w>': 2, '   gi f for d</w>': 3, '   may ta ir</w>': 1, '   de e de e</w>': 1, '   gi ff</w>': 2, '   shu tt up</w>': 1, '   an th a</w>': 2, '   la mp li ght</w>': 1, '   t mp c ssi ble</w>': 1, '   be t ore</w>': 1, ' t and</w>': 1, ' car lo tta</w>': 1, ' i e d re</w>': 1, ' e ga l</w>': 1, '   h er ni a ting</w>': 1, '   ba t ore</w>': 1, '   t re ph ine</w>': 1, '   in tu ba ted</w>': 1, '   ha ma tom a</w>': 1, '   ev ac u at</w>': 1, '   b p</w>': 1, ' i en</w>': 1, '   may fa ir s</w>': 1, '   w ea ved</w>': 1, '   pe t y r</w>': 1, '   ti lled</w>': 1, '   in der stand</w>': 1, '   su z an n</w>': 1, '   b re er es</w>': 1, '   mi cha e ll ll l ll</w>': 1, '   do ve ta i ls</w>': 1, '   y you</w>': 1, '   gi tting</w>': 1, '   dar m</w>': 1, '   b en e fi c ence</w>': 2, '   wa ter lo g ged</w>': 1, '   na s is</w>': 1, '   al li ga t ors</w>': 2, '   re c ti tu de</w>': 1, '   min d rea der</w>': 2, '   le tt s</w>': 1, '   r n ar ried</w>': 1, '   mi cha e</w>': 1, '   p on char tra in</w>': 1, '   y at i</w>': 1, '   in tri g ned</w>': 1, '   in c rea di ble</w>': 1, '   gen te l</w>': 1, '   re ee ea l</w>': 1, ' he e y ah</w>': 1, '   bu st ers</w>': 1, '   y an g</w>': 1, '   in sp ar ation</w>': 1, '   do in ging</w>': 1, '   k ne lt</w>': 1, '   ye ss ss ss</w>': 1, '   an al y si a</w>': 1, ' star ved</w>': 1, '   sig n po sts</w>': 1, '   happen in q </w>': 1, '   t ee l</w>': 1, '   mar gar i ta vi ll e</w>': 1, '   de ca dan ce</w>': 1, '   nor mi e</w>': 1, '   lon n n n n g</w>': 1, '   on t</w>': 1, ' whi st le</w>': 1, '   su t</w>': 1, '   ev e t y thing</w>': 1, '   co di ci l</w>': 1, '   en jo ins</w>': 1, '   fi ll et</w>': 1, '   si mp s ons</w>': 1, '   st e ck l er</w>': 3, '   v in ni es</w>': 1, '   g ri f fi th s</w>': 1, '   ma d der</w>': 1, '   mo ther ed</w>': 1, ' he ir</w>': 1, '   ex t ro ver ts</w>': 1, '   co l ou rs</w>': 1, '   ac cu ses</w>': 1, ' so ci e ty</w>': 1, '   sa v a ged</w>': 1, '   hi k er</w>': 1, '   la y by</w>': 1, ' fe ma les</w>': 1, '   mer k</w>': 8, '   sp r in g y</w>': 2, '   x en i a</w>': 11, '   5 0 1 </w>': 1, '   a st r on om i cal</w>': 1, '   re t ar da tion</w>': 1, '   re in v ent</w>': 2, '   sy m bo li s m</w>': 1, '   st in g y</w>': 1, '   lu be</w>': 1, ' ten der i z ing</w>': 1, '   an a lo gi es</w>': 2, '   sa li v ate</w>': 1, '   ma lon es</w>': 1, '   ha un ts</w>': 1, '   hu sh pu pp i es</w>': 1, '   por ter house</w>': 1, '   bu ck e ye</w>': 1, '   r hi z op us</w>': 1, ' le gs</w>': 1, '   da y li gh ts</w>': 1, '   st af f ers</w>': 1, '   m c do le</w>': 1, ' g ro c ers</w>': 1, ' bra in er</w>': 1, '   f ac si mi le</w>': 1, '   pe ter s en</w>': 2, ' co x s wa in</w>': 1, '   ma d den ing</w>': 1, ' co s m o</w>': 1, '   re f re sh in g ly</w>': 1, '   go o se bu mp</w>': 1, '   be tra ys</w>': 1, '   s mo k y</w>': 1, ' pro d ded</w>': 1, '   d om in a tri x</w>': 1, '   le mon s</w>': 1, '   ra ff les</w>': 1, '   br un ch es</w>': 1, '   t wi tt er pa ted</w>': 1, '   pi ck a x</w>': 1, '   su n n y bro ok</w>': 1, ' p ea ch y</w>': 1, '   ex po s</w>': 2, '   c r o</w>': 1, ' ma g n on</w>': 1, ' dar ed</w>': 1, '   sti le t to</w>': 1, '   ke ds</w>': 1, '   po m</w>': 1, '   p om s</w>': 1, '   du mp st ers</w>': 1, '   sp r in k les</w>': 1, '   con tra sted</w>': 1, '   c ri spi e</w>': 1, '   s ni ck er do o d les</w>': 1, '   ne p t un es</w>': 1, '   z u k er man</w>': 1, '   mer kin</w>': 2, '   ex ci ses</w>': 1, '   p an der ing</w>': 1, ' p ea ch es</w>': 1, '   le g g y</w>': 1, '   l ha ma</w>': 1, ' cla ssi c</w>': 1, ' me di a</w>': 1, '   ti p n</w>': 1, '   na tu re l</w>': 1, '   a pr es</w>': 1, '   di ps</w>': 1, '   b lu r ter</w>': 1, ' di pp ing</w>': 1, '   ti p to ed</w>': 1, '   du que tt e</w>': 7, '   to ll er</w>': 7, '   pa go da</w>': 1, '   con ing</w>': 2, '   ch u mp ed</w>': 1, '   sin k er</w>': 1, '   lon bar do</w>': 1, '   whi ff in</w>': 1, ' bo d y gu ar ds</w>': 1, '   man s ons</w>': 1, '   de fa ma tion</w>': 1, '   e du ca tor</w>': 1, ' t ine</w>': 1, '   per e z</w>': 1, '   bu st l ine</w>': 1, '   cu to ff s</w>': 1, '   k no th o les</w>': 1, '   ar te m us</w>': 4, '   bu ck bo ar ds</w>': 1, '   co o t</w>': 1, ' dam n ably</w>': 1, '   lo ve less</w>': 8, '   pi th</w>': 1, '   pe mb er ton</w>': 6, '   w y l er</w>': 1, ' gu e ssed</w>': 1, '   bra in ed</w>': 1, '   m c ne il</w>': 1, '   l y ri c</w>': 1, '   ll e we ll y n</w>': 1, '   li me st one</w>': 2, '   w ran g ling</w>': 1, '   de sp er a does</w>': 1, '   e mi ts</w>': 2, ' ga d get</w>': 1, '   me ta ll u r g y</w>': 1, '   h y d ra u li ca lly</w>': 1, '   i mb ro g li o</w>': 1, '   f re sc o</w>': 1, '   wor d s mi th</w>': 1, '   di se mb ar k</w>': 1, '   t ar an tu l a</w>': 1, '   ir on cla d</w>': 1, '   con ne mar a</w>': 1, '   por ta tion</w>': 1, '   han de d ly</w>': 1, '   com pe ten tly</w>': 1, '   ba m bo o z le</w>': 1, ' e m per or</w>': 1, '   mi gu e li to</w>': 1, '   pre o c cu pa tion</w>': 1, ' r are</w>': 1, '   mar ti g an</w>': 1, '   cou n</w>': 1, '   a ir k</w>': 6, '   ma d mar ti g an</w>': 13, '   k a el</w>': 1, '   ba v mor da</w>': 7, '   no ck</w>': 1, '   ma ar</w>': 1, '   ma g pi e</w>': 1, '   ma d mar t i</w>': 1, '   g an</w>': 1, '   pr s ence</w>': 1, '   ne l w y n</w>': 4, '   cer ess</w>': 2, '   ra z i el</w>': 10, '   ti r</w>': 7, '   as le en</w>': 7, '   u f good</w>': 2, '   bur g le k u t t</w>': 1, '   da i k in i</w>': 4, '   st ru c</w>': 1, '   al d w in</w>': 2, '   sor sh a</w>': 5, '   i c</w>': 1, '   el or a</w>': 4, '   dan an</w>': 2, '   a g ri cu l</w>': 1, '   sor cer ess</w>': 3, '   do sn</w>': 1, '   bi es</w>': 1, ' po le</w>': 1, '   ru pt</w>': 1, '   w oo d cu tter</w>': 1, '   da i k in is</w>': 2, '   ran on</w>': 1, '   r gu ard</w>': 1, '   in vo ca tion</w>': 1, '   en er</w>': 1, '   g y</w>': 1, '   ex x ence</w>': 1, '   con ce tra te</w>': 1, '   cre at</w>': 1, '   uni ver s</w>': 1, '   pr in</w>': 1, '   c ess</w>': 1, '   ro o l</w>': 2, '   t ee m o</w>': 1, '   hu g g</w>': 1, '   li ev ed</w>': 1, '   ab sen tly</w>': 1, '   ch er lin dre a</w>': 3, '   co sha ir m</w>': 1, '   te at</w>': 1, '   pl u g ging</w>': 1, '   la p p</w>': 3, '   ho ch st e t l er</w>': 2, '   sto l t z fu s</w>': 5, '   sc ha e ff er</w>': 3, '   z en o vi ch</w>': 3, '   z en o vi tch</w>': 2, ' pe ed</w>': 1, '   a mi sh man</w>': 1, '   g lo t z k op p</w>': 1, '   sa l t z bur g</w>': 2, '   g un th ers</w>': 1, '   men n on i te</w>': 1, '   ho ch mu t s n ar r</w>': 1, '   ho ch mu t</w>': 2, '   di en er</w>': 1, '   t s chan t z</w>': 1, '   poli ces</w>': 1, '   1 7 9 0</w>': 1, '   co al m ine</w>': 1, '   o ver sta ted</w>': 1, '   bu ll whi p</w>': 1, '   a g ani shi sh</w>': 1, '   aga ani si sh</w>': 1, '   or d n un g</w>': 2, '   re pen t ant</w>': 1, '   bar n s</w>': 1, '   li e b ch en</w>': 1, '   da w die</w>': 1, '   gu l ch</w>': 8, '   c ru ll ers</w>': 1, '   con tra p tion</w>': 1, '   hi ck ory</w>': 2, ' wa pp ing</w>': 1, '   hi st</w>': 1, '   re ga lly</w>': 1, ' re s our ce ful</w>': 1, ' o z</w>': 1, '   re ta in ing</w>': 1, '   pl u ri bu s</w>': 1, '   un u m</w>': 2, '   ch u ck l ed</w>': 1, '   lu x e</w>': 1, '   hu m bu g</w>': 2, ' to to</w>': 1, '   m un ch k in land</w>': 2, '   su l p hu r</w>': 1, '   po pp i es</w>': 2, '   car e wor n</w>': 1, '   as car ed</w>': 1, ' o il</w>': 1, '   hur tch a</w>': 1, '   ro ar er</w>': 1, '   ce ll op h ant</w>': 1, '   th ra sh</w>': 2, '   bo tt o ma m us</w>': 1, '   hi pp o po ta m us</w>': 1, '   no how</w>': 2, '   vi m</w>': 1, '   m ou </w>': 1, ' ess</w>': 1, '   den y in</w>': 1, '   dan de</w>': 1, ' li on</w>': 1, '   fi re ba lls</w>': 1, '   wh h ho o op s</w>': 1, '   g lin da</w>': 1, ' ting</w>': 1, ' ex ce ll ence</w>': 1, '   stra to sp h er i c</w>': 1, '   stu f fin gs</w>': 1, '   for es</w>': 1, '   br on to sa u ru s</w>': 1, '   who z at</w>': 2, '   what z is</w>': 2, '   im po ss er ous</w>': 1, ' j un k</w>': 1, '   who z is</w>': 2, '   p al pi ta tion</w>': 1, '   j i j i k</w>': 1, '   t in s mi th</w>': 1, '   mer i t ori ous</w>': 1, '   uni ver si ta tu s</w>': 1, '   com mi tt e ea tu m</w>': 1, '   pl u r b is</w>': 1, '   g al v ani z ed</w>': 1, '   ob st ru c ted</w>': 1, '   un w ra pped</w>': 1, '   bo a</w>': 1, '   z u k o v s k y</w>': 2, '   e le k tr a</w>': 11, '   ir ra di a ted</w>': 1, '   mi ra to m</w>': 2, '   da vi do v </w>': 3, '   ro ad si d es</w>': 1, '   me du ll a</w>': 1, '   o bl on ga ta</w>': 1, '   ser ra u lt</w>': 1, '   go l d ru sh</w>': 1, '   ma tch less</w>': 1, '   lan d lo cked</w>': 1, '   re d ra w</w>': 1, ' r en ard</w>': 1, '   g ab or</w>': 1, '   r en o v a tions</w>': 1, '   e ye li d</w>': 1, ' pl u ton i c</w>': 1, '   par as k i</w>': 1, '   c y ri lli c</w>': 1, '   fi sh er y</w>': 1, '   war l or ds</w>': 1, '   con sor ti u ms</w>': 1, '   b on d ja me s b on d</w>': 1, '   su b mar in es</w>': 1, '   y ev gen y</w>': 1, '   a dam son</w>': 1, '   for e ca sts</w>': 1, '   qu ir k</w>': 1, '   c tu </w>': 9, '   h n n r r</w>': 1, '   ve i d t</w>': 6, '   un qu an ti fi able</w>': 1, '   dre i ber g</w>': 3, '   la ther</w>': 1, ' st al king</w>': 1, '   wa tch men</w>': 4, '   v a ll es</w>': 1, '   mar in er is</w>': 1, ' s ci en ti st</w>': 1, '   ja y wal k ers</w>': 1, '   ti me lin es</w>': 1, '   gi l a</w>': 2, '   b in g es</w>': 1, '   ri gh ting</w>': 1, '   ow l ship</w>': 1, '   lu tch er</w>': 1, '   hi gh ta i ling</w>': 1, '   si c c ed</w>': 1, '   o st er man</w>': 1, '   a mi d</w>': 1, ' al ter ed</w>': 1, '   ju spe c z y k</w>': 3, '   su per ac c el er a tor</w>': 1, '   mo lo ch</w>': 1, '   ha st en ed</w>': 1, '   gi b b ons</w>': 4, '   a ha b</w>': 3, ' cla ssi fi ed</w>': 1, '   gr un g y</w>': 1, ' a ha b</w>': 3, ' ex t re me</w>': 1, '   dar e d ev il</w>': 1, '   b ack story</w>': 1, '   pe gs</w>': 1, '   t ro lling</w>': 1, '   n ar c s</w>': 1, ' sa le s man</w>': 1, ' tru ck er</w>': 1, ' wal ked</w>': 1, '   ga ff le</w>': 1, '   mu r de</w>': 1, '   de s k to p</w>': 1, '   e se</w>': 1, ' an ar ch y</w>': 5, '   y or g i</w>': 10, '   ma fi ya</w>': 1, '   ki ri ll</w>': 2, '   che ch n ya</w>': 1, '   s n ow bo ar ding</w>': 2, '   to a di es</w>': 1, '   be tt er ment</w>': 1, '   bor a</w>': 4, '   x an der</w>': 4, '   pe tr a</w>': 5, '   i v ans</w>': 1, '   w w f</w>': 1, '   ni hi li sti c</w>': 1, '   f re el an c ing</w>': 1, '   ga ther er</w>': 1, '   o ver vi e w</w>': 1, '   re t ar d ant</w>': 1, '   de p lo ys</w>': 1, '   bea sti e</w>': 1, '   ca ms</w>': 1, '   o z z fe st</w>': 1, '   ru ss ki e</w>': 1, '   sha ver s</w>': 2, '   po l y ne si a</w>': 1, '   mer sh</w>': 1, '   do g ging</w>': 1, '   s lo v o</w>': 1, '   da w ning</w>': 1, '   t shi r t</w>': 1, '   di sh on or ably</w>': 1, '   v an da ls</w>': 1, '   g ro z ny</w>': 1, '   la mb or gh in i</w>': 1, '   gen o a</w>': 1, '   pi z da</w>': 1, '   t un g st en</w>': 1, '   fi la ment</w>': 1, ' fi l ter</w>': 1, '   re gu la ting</w>': 1, '   cer e br o</w>': 4, '   mu ta tor</w>': 3, '   re p li ca te</w>': 1, '   a dam an ti u m</w>': 4, '   so li der</w>': 1, '   se ca u c us</w>': 1, '   al ter ca tion</w>': 1, '   a th le ti c s</w>': 1, '   we st che ster</w>': 2, '   h er ded</w>': 1, '   gu y ri ch</w>': 1, '   w o l ver ine</w>': 1, '   ab sor b s</w>': 1, '   de fini ti ve ly</w>': 1, '   f al k st e in</w>': 5, '   po pp y co ck</w>': 1, '   ru m ous</w>': 1, ' fr on k on st e en</w>': 2, '   fr on k on st e en</w>': 5, '   ca u li f l ower</w>': 3, '   di sin fe c t ant</w>': 1, '   c el er y</w>': 1, '   cer e b ru m</w>': 1, '   in cor ri gi ble</w>': 2, '   ba ack</w>': 2, '   un as ha me d ly</w>': 1, '   de l b ru ck</w>': 6, '   t ac h la s</w>': 1, '   di r</w>': 1, ' a ster</w>': 2, '   a ye g or</w>': 3, '   in g a</w>': 5, '   ger har t</w>': 1, '   fr on</w>': 4, '   st e en</w>': 5, '   men ding</w>': 1, ' ani ma tion</w>': 1, '   ver mi ce ll i</w>': 2, ' f le x i ve</w>': 1, ' vo l un t ary</w>': 1, '   g al v ani s m</w>': 1, '   e qu a li ze</w>': 1, '   cer e bro sp in al</w>': 1, '   ma de in</w>': 1, '   fro der i ck</w>': 1, '   bl in din g ly</w>': 1, ' y u m my</w>': 2, '   de ss er ts</w>': 1, '   bu char est</w>': 2, '   im possi bi li ti es</w>': 1, '   re ce ss es</w>': 1, '   as c end</w>': 1, '   th und ers</w>': 1, '   ki tes</w>': 1, '   co ok bo o ks</w>': 1, '   di a g no sti ci an</w>': 1, ' der i ck</w>': 1, ' g or</w>': 1, '   so ck ers</w>': 1, '   fo ck ers</w>': 1, '   w o ck ers</w>': 4, '   k no ck ers</w>': 2, '   g or</w>': 2, '   f re der e ck</w>': 1, '   fro der e ck</w>': 1, '   fr on k on</w>': 1, '   er e ck</w>': 1, '   der e ck</w>': 1, '   mm mm mm mm m</w>': 3, '   mm mm mm mm m m</w>': 1, '   mm mm mm mm mm m m</w>': 2, '   mm mm mm mm mm mm mm mm mm mm mm mm mm mm m</w>': 2, ' fe ll ers</w>': 1, '   mm mm mm mm mm mm mm mm mm mm mm mm m</w>': 1, '   pa h</w>': 1, '   tr ou per</w>': 1, '   t mm m</w>': 2, '   an n gh</w>': 2, '   mm mm mm mm mm m</w>': 1, '   mm mm mm mm mm mm m n n n n n n mm mm mm m m</w>': 1, '   tr an sy l v ani an</w>': 1, '   a p fe l st ru de l</w>': 1, '   mm mm mm mm mm mm mm m</w>': 1, '   fu ch s m ac h en</w>': 1, '   li e be</w>': 1, ' sion</w>': 1, '   n en</w>': 1, ' tal</w>': 3, ' n en</w>': 2, '   dan c</w>': 1, '   u l</w>': 1, ' tr a</w>': 1, ' t le</w>': 1, ' i la tion</w>': 1, '   s ac ra l</w>': 1, '   minu ten ess</w>': 1, '   un p ac king</w>': 1, '   b lu ch er</w>': 2, '   sch wan z stu ck er</w>': 1, '   gr r r h mm n n n j k j mm m n n</w>': 1, '   r ou gh hou sing</w>': 1, '   f oo oo oo od</w>': 1, '   z u l us</w>': 4, '   a fe ar ed</w>': 2, '   as se ga is</w>': 2, '   z u lu land</w>': 2, '   do o ty</w>': 1, '   li fe bu o y</w>': 1, '   sa i j </w>': 1, '   pe i fe c l</w>': 1, '   me se if</w>': 1, '   we ll in g ton</w>': 1, '   du m for d</w>': 4, '   pu ll e ine</w>': 3, '   l or j </w>': 1, '   no g gs</w>': 2, '   ce t sh wa y o</w>': 1, '   imp is</w>': 2, '   man o e u v re</w>': 1, '   in de e d l did</w>': 1, '   my l or d</w>': 1, '   i t was</w>': 1, '   of the</w>': 1, ' ne w man</w>': 2, '   u l und i</w>': 2, '   ver e k er</w>': 4, '   ad c</w>': 1, '   of na tal</w>': 1, '   c rea lo ck</w>': 3, '   su b al ter n</w>': 1, '   du r n for d</w>': 3, '   si k al i</w>': 3, '   h or se man ship</w>': 1, '   sp l en di l</w>': 1, '   k ra al</w>': 2, '   ch el m s for d</w>': 4, '   b om bar di er</w>': 1, '   co gh i ll</w>': 3, '   imp i</w>': 1, '   ba su to s</w>': 1}

In [50]:
def build_vocab(final_vocabs, max_vocab_size=1000):
	subword_counts = {}
	for word, freq in final_vocabs.items():
		for subword in word.split():
			subword_counts[subword] = subword_counts.get(subword,0) + freq
	
	# sort subwords by frequency and keep Only the top 2000 most common ones
	top_subwords = sorted(subword_counts, key=subword_counts.get, reverse=True)[:max_vocab_size]

	# Everything else gets mapped to an "unknown" token token
	if '[UNK]' not in top_subwords:
		top_subwords.append('[UNK]')
		
	stoi = {s:i for i, s in enumerate(top_subwords)}
	itos = {i:s for s, i in stoi.items()}
	
	return stoi, itos


# def encode_text(text, stoi, final_vocab):
#     tokens = pretokenization(text)
#     token_ids = []
#     for token in tokens:
#         # Convert to the space-separated BPE form
#         space_word = " " + " ".join(list(token)) + " </w>"
#         # Apply BPE merges by checking if the merged form is in vocab
#         # Simplest: look up the final_vocab representation
#         if space_word in final_vocab:
#             subwords = space_word.split()  # already merged after training
#         else:
#             subwords = list(token) + ['</w>']  # fall back to characters
#         for sw in subwords:
#             token_ids.append(stoi.get(sw, stoi['[UNK]']))
#     return token_ids


print("Starting build_vocab...")
stoi, itos = build_vocab(final_vocab, 1000)
print(f"Vocab built. Size: {len(stoi)}")

# print("Starting encode_text...")
# token_ids = encode_text(text, stoi, final_vocab)
# print(f"Encoding complete. Length: {len(token_ids)}")

# print(f"First 10 token_ids: {token_ids[:10]}")

Starting build_vocab...
Vocab built. Size: 1001


In [51]:
# 1. Create token_ids by parsing your final_vocab weights
token_ids = []

for word_structure, freq in final_vocab.items():
	# word_structure looks like: "th ey </w>" or "  you </w>"
	subwords = word_structure.split()
	
	# Convert subword strings to their integer IDs using stoi
	# If a subword was cut from the top 2000, fall back to '[UNK]'
	word_ids = [stoi.get(sub, stoi['[UNK]']) for sub in subwords]
	
	# Since 'freq' is how many times that exact word appeared in your script,
	# we repeat those token IDs 'freq' times to match the real dataset statistics!
	for _ in range(freq):
		token_ids.extend(word_ids)
print(token_ids[:100])

[49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49]


In [39]:
# feed BPE tokens into bigram Model.
import torch

vocab_size = len(stoi) # 2001

N = torch.ones((vocab_size, vocab_size), dtype=torch.int32)

for i in range(len(token_ids)-1):
	ix1 = token_ids[i]
	ix2 = token_ids[i+1]
	N[ix1, ix2] += 1

P = N.float()
P /= P.sum(dim=1, keepdim=True)


In [52]:
import re
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

if device.type == 'mps':
	g = torch.Generator(device='mps').manual_seed(2147483647)
elif device.type == 'cuda':
	g = torch.Generator(device='cuda').manual_seed(2147483647)
else:
	g = torch.Generator(device='cpu').manual_seed(2147483647)
vocab_size = len(stoi)

P = P.to(device)

print(f"Locked vocabulary size: {vocab_size}")
print("--- Generating Text ---")

# Lower temperature (e.g., 0.7) makes the model more confident and coherent
# Higher temperature (e.g., 1.2) makes it more random/creative
temperature = 0.2

for _ in range(5):
	out = []
	start_token = stoi.get('  </w>', stoi.get(' </w>', 0))
	ix = start_token
   
	while True:
		p = P[ix]
		
		# Apply temperature scaling:
		# 1. Raise to the power of 1/T to adjust confidence
		# 2. Re-normalize so the probabilities still sum to 1
		p = p.pow(1.0 / temperature)
		p = p / p.sum()
		
		ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
		ix = min(ix, len(itos)-1)
		
		token_str = itos.get(ix)
		
		# Stop if we hit a dead end
		if token_str is None or token_str == '[UNK]':
			break
		
		# Clean and append
		clean_token = token_str.replace("</w>", ' ')
		out.append(clean_token)

		# Break on sentence enders
		if any(punct in clean_token for punct in ['.', '?', '!']):
			break

		if len(out) > 40:
			break
			
	raw_sentence = "".join(out)
	clean_sentence = re.sub(r'\s+', ' ', raw_sentence).strip()
	print(clean_sentence)
	
	# Check what the model thinks should follow a common word like "you"
token = 'you</w>'
if token in stoi:
	idx = stoi[token]
	probs = P[idx]
	top_probs, top_indices = torch.topk(probs, 5)
	print(f"Top followers for '{token}':")
	for prob, i in zip(top_probs, top_indices):
		print(f"  {itos.get(i.item(), '[UNK]')}: {prob.item():.4f}")

Locked vocabulary size: 1001
--- Generating Text ---
maro jesus knsomess on serpleanothing try something pekes even dge menlooks t chosed sir five on z outword off room bormakes does be try use acd everything es today kid
ing talking doctor looks being god lanfrom ins problefungotta y
late p see killed gaaltetive anbetter listis side sy driinto to'd cuausays ass around say questiset business disouble hege clayears wethought tchdinre can
cuyourself gugarcal eve okay enough found real tomhave impmight burand thanks qugone ven by sit mpbig ess 30
burtor ve woman sed tting ctor gs deawhat give maybe or thanks ys possied irsy 'm omehuh problecomtch sinmuch than listen houme
Top followers for 'you</w>':
  you</w>: 0.0010
  i</w>: 0.0010
  </w>: 0.0010
  the</w>: 0.0010
  a</w>: 0.0010


In [ ]:
# def create_token_ids(final_vocabs, max_vocab_size=2000):
#     # 1. Count how often every single subword token actually appears
#     subword_counts = {}
#     for word, freq in final_vocabs.items():
#         for subword in word.split():
#             subword_counts[subword] = subword_counts.get(subword, 0) + freq
			
#     # 2. Isolate the base characters (length 1, or matching your special tags)
#     # This guarantees the model can always spell out unknown words letter-by-letter!
#     base_alphabet = {subword for subword in subword_counts.keys() if len(subword) == 1 or subword == '</w>'}
	
#     # 3. Isolate the remaining multi-character subwords/merges
#     merged_subwords = {subword: count for subword, count in subword_counts.items() if subword not in base_alphabet}
	
#     # 4. Sort the merges by frequency to find the most valuable ones
#     most_frequent_merges = sorted(merged_subwords, key=merged_subwords.get, reverse=True)
	
#     # 5. Build the final vocabulary list
#     # Start with the fallback alphabet, then add top merges until we hit our cap
#     final_tokens = list(base_alphabet)
	
#     # Calculate how many extra slots we have left for merged words
#     remaining_slots = max_vocab_size - len(final_tokens) - 1 # saving 1 slot for [UNK]
	
#     final_tokens.extend(most_frequent_merges[:remaining_slots])
	
#     # 6. Append the unknown safety token
#     if '[UNK]' not in final_tokens:
#         final_tokens.append('[UNK]')
		
#     # 7. Create lookups (Sort them cleanly so the index IDs are ordered)
#     sorted_tokens = sorted(final_tokens)
#     stoi = {token: token_id for token_id, token in enumerate(sorted_tokens)}
#     itos = {token_id: token for token, token_id in stoi.items()}
	
#     return stoi, itos
# x = create_token_ids(final_vocab, 2000)
# print(x)